In [ ]:
# ============================================================
# SETUP — Notebook 12 / Stage12 Residual Edge Recovery
# Purpose:
#   Initialize the Stage12 notebook and verify the frozen
#   Stage11/Stage12 artifacts already produced.
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

SRC = PROJECT / "src"
SCRIPTS = PROJECT / "scripts"

REPORTS = PROJECT / "reports"
S11 = REPORTS / "stage11_division_probe"
S12 = REPORTS / "stage12_residual_audit"

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

CV_PATH = PROJECT / "configs/cv_manifest.csv"


sys.path[:0] = [
    str(SRC),
    str(SCRIPTS),
]


# ------------------------------------------------------------
# Frozen Stage12 artifacts
# ------------------------------------------------------------

STAGE12_FILES = {
    "edge_residual_summary":
        S12 / "stage12_edge_residual_summary.json",

    "mutual_cont_policy_lock":
        S12 / "stage12_mutual_cont_v1_policy_lock.json",

    "fold0_action_lock":
        S12 / "stage12_fold0_mutual_cont_v1_action_lock.json",

    "fold0_outer_summary":
        S12 / "stage12_fold0_mutual_cont_v1_outer_summary.json",

    "full_hybrid_summary":
        S12 / "stage12_full_hybrid_mutual_cont_v1_summary.json",

    "current_best_checkpoint":
        S12 / "stage12_current_best_checkpoint.json",
}


missing = [
    name
    for name, path in STAGE12_FILES.items()
    if not path.exists()
]

assert not missing, (
    f"Missing Stage12 artifacts: {missing}"
)


checkpoint = json.loads(
    STAGE12_FILES[
        "current_best_checkpoint"
    ].read_text(
        encoding="utf-8"
    )
)


print(
    "========== NOTEBOOK 12 =========="
)

print(
    "Stage12 residual edge recovery"
)

print(
    "\nCurrent pipeline:"
)

print(
    checkpoint["checkpoint_name"]
)

print(
    "\nCurrent Fold0 metrics:"
)

for key, value in (
    checkpoint["fold0_metrics"].items()
):
    print(
        f"{key:18s}: {value:.9f}"
    )


print(
    "\n========== SETUP DECISION =========="
)

print(
    "NOTEBOOK12_STAGE12_SETUP: PASS"
)

# Stage 12 — Residual Edge Recovery

## Starting checkpoint

Stage 12 starts from the frozen Stage 11 hybrid pipeline:

- **44b6:** Sparse9995 seed1640 → Frozen V9
- **6bba:** Baseline256 → Frozen V9 → MORPH-DIV-V2

Stage 11 full 40-dataset checkpoint:

- Edge Jaccard: **0.781996647**
- Adjusted Edge Jaccard: **0.774285647**
- Division Jaccard: **0.046875000**
- Node Recall: **0.959822345**
- Score: **0.778973147**

## Stage 12 residual audit

Initial edge residual census:

- Edge TP / FP / FN: **22387 / 2397 / 3844**
- Detection-limited FN: **1554 (40.43%)**
- Linking-limited FN: **2290 (59.57%)**

Association replay showed that most linking failures were not missing detection candidates.  
For the 6bba raw-association-miss population, the correct continuation frequently remained near the top of the dense transformer association matrix.

This motivated a deployment-valid continuation recovery rule.

## MUTUAL-CONT-V1

Locked rule:

- Final source outdegree = **0**
- Final target indegree = **0**
- Probability row rank = **1**
- Probability column rank = **1**
- Candidate probability ≥ **0.25**
- Add exactly one continuation edge
- No node changes
- No division creation

Development folds 1–4:

- Baseline score: **0.781893060**
- MUTUAL-CONT-V1 score: **0.788841246**
- Δ score: **+0.006948186**

Locked Fold0 outer test:

- Baseline score: **0.780480353**
- MUTUAL-CONT-V1 score: **0.786581383**
- Δ score: **+0.006101030**
- Edge TP: **+288**
- Edge FP: **+185**
- Edge FN: **−288**
- Division Jaccard: unchanged
- Dataset effect: **23 improved / 2 worsened / 0 unchanged**

## Current Stage 12 checkpoint

Full 40-dataset integration:

- Edge TP / FP / FN: **22675 / 2582 / 3556**
- Edge Jaccard: **0.786971159**
- Adjusted Edge Jaccard: **0.779233490**
- Division Jaccard: **0.046875000**
- Node Recall: **0.959822345**
- Score: **0.783920990**

Improvement over Stage 11:

**Δ score = +0.004947842**

`MUTUAL-CONT-V1` is frozen and promoted. Its threshold or rank/topology gates must not be retuned using Fold0.

In [ ]:
# ============================================================
# RESULT — Stage 12.05A Post-MUTUAL-CONT Residual Budget
# Purpose:
#   Lock the residual edge-FN budget after MUTUAL-CONT-V1.
#
#   MUTUAL-CONT-V1 changes edges only and does not change nodes.
#   Therefore the previously measured detection-limited FN
#   population remains invariant, while all recovered FNs come
#   from the linking-limited population.
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO NEW GT READ
# Keep after running: YES
# ============================================================

from pathlib import Path
import json


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports/stage12_residual_audit"
)

SUMMARY_PATH = (
    S12
    / "stage12_full_hybrid_mutual_cont_v1_summary.json"
)


current = json.loads(
    SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


# Frozen Stage 12.00D residual census.
OLD_FN = 3844
DETECTION_LIMITED_FN = 1554
OLD_LINKING_LIMITED_FN = 2290


new_fn = int(
    current["new"]["edge_fn"]
)

recovered_fn = (
    OLD_FN
    - new_fn
)

new_detection_fn = (
    DETECTION_LIMITED_FN
)

new_linking_fn = (
    new_fn
    - new_detection_fn
)


assert new_fn == 3556
assert recovered_fn == 288
assert new_linking_fn == 2002

assert (
    new_detection_fn
    + new_linking_fn
    == new_fn
)


print(
    "========== CURRENT CHECKPOINT =========="
)

print(
    "score:",
    f"{current['new']['score']:.9f}"
)


print(
    "\n========== RESIDUAL EDGE FN BUDGET =========="
)

print(
    "total FN          :",
    new_fn
)

print(
    "detection-limited :",
    new_detection_fn,
    f"({new_detection_fn/new_fn:.3%})"
)

print(
    "linking-limited   :",
    new_linking_fn,
    f"({new_linking_fn/new_fn:.3%})"
)


print(
    "\n========== MUTUAL-CONT-V1 RECOVERY =========="
)

print(
    "FN recovered:",
    recovered_fn
)

print(
    "old linking FN recovered:",
    f"{recovered_fn/OLD_LINKING_LIMITED_FN:.3%}"
)


print(
    "\n========== 12.05A DECISION =========="
)

print(
    "POST_MUTUAL_CONT_RESIDUAL_BUDGET_LOCKED: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.05C Exact Post-V1 Linking Residuals
# Purpose:
#   Reconstruct exactly which old linking FNs were recovered by
#   MUTUAL-CONT-V1, and derive the topology of the 2002 linking
#   FNs that remain after V1.
#
#   No association inference or new GT evaluation is required.
#
# Changes files: YES
#   reports/stage12_residual_audit/
#       stage12_post_v1_linking_residuals.pkl
#       stage12_post_v1_linking_residual_summary.json
#
# Runs training/inference: NO
# Uses GT: NO NEW GT READ
#   Uses previously frozen GT-derived residual artifacts.
# Keep after running: YES
# ============================================================

import json
import numpy as np
import pandas as pd


FN_PATH = (
    S12 / "stage12_edge_fn_residuals.pkl"
)

PROV_PATH = (
    S12 / "stage12_linking_fn_raw_edge_provenance.pkl"
)

ACTION_PATH = (
    S12 / "stage12_fold0_mutual_cont_v1_actions_locked.pkl"
)

OUT = (
    S12 / "stage12_post_v1_linking_residuals.pkl"
)

SUMMARY_OUT = (
    S12 / "stage12_post_v1_linking_residual_summary.json"
)


fn = pd.read_pickle(FN_PATH)

link = (
    fn[
        fn["root_cause"].eq("LINKING_LIMITED")
    ]
    .copy()
)

prov = pd.read_pickle(PROV_PATH)
actions = pd.read_pickle(ACTION_PATH)


assert len(link) == 2290
assert len(prov) == 2290
assert len(actions) == 10107


# ------------------------------------------------------------
# 1. Exact V1 edge recovery
# ------------------------------------------------------------

action_edges = set(
    zip(
        actions["dataset"],
        actions["source_pred_node_id"].astype(int),
        actions["target_pred_node_id"].astype(int),
    )
)

link["recovered_by_v1"] = [
    (
        r.dataset,
        int(r.pred_source_id),
        int(r.pred_target_id),
    ) in action_edges
    for r in link.itertuples(index=False)
]


# ------------------------------------------------------------
# 2. Derive endpoint occupation AFTER V1
# ------------------------------------------------------------

v1_sources = set(
    zip(
        actions["dataset"],
        actions["source_pred_node_id"].astype(int),
    )
)

v1_targets = set(
    zip(
        actions["dataset"],
        actions["target_pred_node_id"].astype(int),
    )
)


link["v1_uses_source"] = [
    (
        r.dataset,
        int(r.pred_source_id),
    ) in v1_sources
    for r in link.itertuples(index=False)
]

link["v1_uses_target"] = [
    (
        r.dataset,
        int(r.pred_target_id),
    ) in v1_targets
    for r in link.itertuples(index=False)
]


link["post_v1_source_outdegree"] = (
    link["pred_source_outdegree"]
    + link["v1_uses_source"].astype(int)
)

link["post_v1_target_indegree"] = (
    link["pred_target_indegree"]
    + link["v1_uses_target"].astype(int)
)


def topology(row):

    s = int(row.post_v1_source_outdegree)
    t = int(row.post_v1_target_indegree)

    return (
        f"SRC_OUT{s}_"
        + (
            "TARGET_FREE"
            if t == 0
            else "TARGET_OCCUPIED"
        )
    )


link["post_v1_topology"] = link.apply(
    topology,
    axis=1,
)


# ------------------------------------------------------------
# 3. Join frozen raw-edge provenance
# ------------------------------------------------------------

KEY = [
    "dataset",
    "gt_source_id",
    "gt_target_id",
]

prov_small = prov[
    KEY + [
        "provenance",
        "best_raw_prob",
    ]
]

link = link.merge(
    prov_small,
    on=KEY,
    how="left",
    validate="one_to_one",
)


# ------------------------------------------------------------
# 4. Exact accounting
# ------------------------------------------------------------

recovered = link[
    link["recovered_by_v1"]
]

remaining = link[
    ~link["recovered_by_v1"]
].copy()


assert len(recovered) == 288
assert len(remaining) == 2002

assert (
    len(recovered)
    + len(remaining)
    == 2290
)


print(
    "========== V1 EXACT FN RECOVERY =========="
)

print(
    "old linking FN :",
    len(link)
)

print(
    "recovered      :",
    len(recovered)
)

print(
    "remaining      :",
    len(remaining)
)

print(
    "recovery rate  :",
    f"{len(recovered)/len(link):.3%}"
)


print(
    "\n========== RECOVERED BY PREFIX =========="
)

print(
    recovered.groupby(
        "prefix"
    ).size().to_string()
)


print(
    "\n========== POST-V1 REMAINING TOPOLOGY =========="
)

print(
    remaining[
        "post_v1_topology"
    ]
    .value_counts()
    .to_string()
)


print(
    "\n========== POST-V1 REMAINING PROVENANCE =========="
)

print(
    remaining[
        "provenance"
    ]
    .value_counts()
    .to_string()
)


free_free = remaining[
    (remaining["post_v1_source_outdegree"] == 0)
    &
    (remaining["post_v1_target_indegree"] == 0)
]


print(
    "\n========== SECOND-LAYER FREE/FREE CEILING =========="
)

print(
    "remaining linking FN:",
    len(remaining)
)

print(
    "post-V1 source0 + target-free:",
    len(free_free),
    f"({len(free_free)/len(remaining):.3%})"
)

print(
    "\nby prefix:"
)

print(
    free_free.groupby(
        "prefix"
    ).size().to_string()
)

print(
    "\nby provenance:"
)

print(
    free_free[
        "provenance"
    ]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 5. Persist
# ------------------------------------------------------------

link.to_pickle(
    OUT
)

payload = {
    "stage": "12.05C",
    "old_linking_fn": int(len(link)),
    "recovered_by_mutual_cont_v1": int(len(recovered)),
    "remaining_linking_fn": int(len(remaining)),
    "post_v1_free_free_fn": int(len(free_free)),
    "post_v1_free_free_fraction":
        float(len(free_free) / len(remaining)),
}

SUMMARY_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    OUT
)

print(
    "written:",
    SUMMARY_OUT
)


print(
    "\n========== 12.05C DECISION =========="
)

print(
    "POST_V1_LINKING_RESIDUALS_RECONSTRUCTED: PASS"
)

In [ ]:
# ============================================================
# SETUP — Stage 12.06A-P1 Reciprocal Helper Verification
# Purpose:
#   Verify that the reusable Stage12 reciprocal helper imports.
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO
# Keep after running: YES
# ============================================================

import importlib

import stage12_iterative_reciprocal as ir

importlib.reload(ir)


required = [
    "fast_association_pair",
    "residual_reciprocal_pairs",
]

for name in required:
    assert hasattr(ir, name)


print(
    "========== 12.06A-P1 DECISION =========="
)

print(
    "ITERATIVE_RECIPROCAL_HELPER_READY: PASS"
)

In [ ]:
# ============================================================
# RUN — Stage 12.06A-S1 Iterative Reciprocal Round-2 Smoke
# Purpose:
#   On one folds1–4 pilot dataset per fold:
#
#   1. Verify the new helper reproduces the exact locked
#      MUTUAL-CONT-V1 actions.
#   2. Apply the locked V1 endpoint occupancy.
#   3. Recompute mutual-best only inside the remaining
#      free-source × free-target submatrix.
#   4. Measure the observable Round-2 candidate universe.
#
# Changes files: NO
# Runs training: NO
# Runs inference: YES — 4 pilot datasets, fast replay
# Uses GT: NO
# Changes frozen policy: NO
# Keep after running: NO
#   Delete after Round-2 contract is verified.
# ============================================================

from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import torch


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

P995 = (
    PROJECT
    / "predictions/heqiuyan/"
      "stage11_fullgraph_baseline256_folds1to4_det995/"
      "split_0"
)

P9975 = (
    PROJECT
    / "predictions/heqiuyan/"
      "stage11_fullgraph_baseline256_folds1to4_det9975/"
      "split_0"
)

CKPT = (
    PROJECT
    / "models/official_baseline/"
      "stage10_pilot_baseline_256/"
      "split_0/edge_predictor_best.pth"
)

CV_PATH = PROJECT / "configs/cv_manifest.csv"

S11 = PROJECT / "reports/stage11_division_probe"
S12 = PROJECT / "reports/stage12_residual_audit"


sys.path[:0] = [
    str(PROJECT / "src"),
    str(PROJECT / "scripts"),
]

from biohub_cell_tracking import frozen_v9 as fv9

import run_stage11_multiframe_extract as runner
import stage12_iterative_reciprocal as ir

from replay_stage11_association import DatasetReplay


# ============================================================
# 1. Four fixed pilot datasets
# ============================================================

cv = pd.read_csv(CV_PATH)

corp = (
    cv[
        cv["fold"].isin([1, 2, 3, 4])
        &
        cv["dataset"].str.startswith("6bba_")
    ]
    .sort_values(["fold", "dataset"])
    .reset_index(drop=True)
)

pilot = (
    corp.groupby("fold", as_index=False)
    .first()
    .sort_values("fold")
)

assert len(pilot) == 4

names = pilot["dataset"].tolist()

prefix_map = dict(
    zip(
        pilot["dataset"],
        pilot["prefix"],
    )
)

fold_map = dict(
    zip(
        pilot["dataset"],
        pilot["fold"],
    )
)


print("========== PILOT DATASETS ==========")
print(
    pilot[
        ["fold", "dataset"]
    ].to_string(index=False)
)


# ============================================================
# 2. Locked Stage11 + V1 artifacts
# ============================================================

morph_all = pd.read_pickle(
    S11
    / "stage11_deployable_final_v9_policy_grid_actions.pkl"
)

morph = (
    morph_all[
        morph_all["config_id"].eq("K3_R3_P100")
    ]
    .copy()
)

universe = pd.read_pickle(
    S12
    / "stage12_folds14_mutual_best_universe.pkl"
)

v1 = (
    universe[
        universe["candidate_prob"].ge(0.25)
    ]
    .copy()
)

assert len(v1) == 46951


policies = fv9.build_frozen_v9_policies(
    names,
    P995,
    P9975,
    TRAIN_ROOT,
    prefix_map=prefix_map,
    config=fv9.FROZEN_V9_CONFIG,
    progress=False,
)


# ============================================================
# 3. Exact Stage11 final topology
# ============================================================

def stage11_topology(name):

    g, _, _ = fv9.apply_frozen_v9(
        name,
        P995,
        P9975,
        policies,
        TRAIN_ROOT,
        config=fv9.FROZEN_V9_CONFIG,
    )

    aa = (
        morph[
            morph["dataset"].eq(name)
        ]
        .sort_values(
            [
                "morph_rank",
                "t",
                "source_pred_node_id",
                "target_pred_node_id",
            ]
        )
    )

    nodes = (
        g.node_attrs(
            attr_keys=[
                "node_id", "t", "z", "y", "x"
            ]
        )
        .to_pandas()
        .set_index("node_id")
    )

    edges = (
        g.edge_attrs(unpack=True)
        .to_pandas()
    )

    outdeg = (
        edges.groupby("source_id")
        .size()
        .astype(int)
        .to_dict()
    )

    indeg = (
        edges.groupby("target_id")
        .size()
        .astype(int)
        .to_dict()
    )

    for a in aa.itertuples(index=False):

        s = int(a.source_pred_node_id)
        q = int(a.target_pred_node_id)

        assert outdeg.get(s, 0) == 1
        assert indeg.get(q, 0) == 0

        outdeg[s] = 2
        indeg[q] = 1

    return nodes, outdeg, indeg


# ============================================================
# 4. Model
# ============================================================

device = runner.resolve_device("mps")

model, window_size, downsample = (
    runner.load_model(
        CKPT,
        device,
    )
)

assert window_size == 2


# ============================================================
# 5. Round-1 parity + Round-2 extraction
# ============================================================

summary = []
round2_records = []

started = time.time()


for name in names:

    final_nodes, outdeg, indeg = (
        stage11_topology(name)
    )

    final_ids = set(
        map(int, final_nodes.index)
    )


    raw_graph = runner.load_graph(
        P995 / f"{name}.geff"
    )

    raw_nodes = (
        raw_graph.node_attrs(
            attr_keys=[
                "node_id", "t", "z", "y", "x"
            ]
        )
        .to_pandas()
    )


    replay = DatasetReplay(
        name,
        TRAIN_ROOT,
        model,
        window_size,
        downsample,
        device,
    )


    expected_v1 = (
        v1[
            v1["dataset"].eq(name)
        ]
    )

    expected_set = set(
        zip(
            expected_v1[
                "source_pred_node_id"
            ].astype(int),

            expected_v1[
                "target_pred_node_id"
            ].astype(int),
        )
    )


    observed_v1 = set()
    pair_cache = {}


    # --------------------------------------------------------
    # Reconstruct Round 1 exactly
    # --------------------------------------------------------

    for t in range(replay.T - 1):

        src_ids, tgt_ids, probs = (
            ir.fast_association_pair(
                replay,
                raw_nodes,
                t,
                model,
                device,
            )
        )

        pair_cache[t] = (
            src_ids,
            tgt_ids,
            probs,
        )

        if probs is None:
            continue

        row_top = np.argmax(
            probs,
            axis=1,
        )

        col_top = np.argmax(
            probs,
            axis=0,
        )

        for i, j in enumerate(row_top):

            if int(col_top[j]) != i:
                continue

            s = int(src_ids[i])
            q = int(tgt_ids[j])

            if (
                s not in final_ids
                or q not in final_ids
            ):
                continue

            if outdeg.get(s, 0) != 0:
                continue

            if indeg.get(q, 0) != 0:
                continue

            if float(probs[i, j]) < 0.25:
                continue

            observed_v1.add(
                (s, q)
            )


    parity = (
        observed_v1
        == expected_set
    )

    assert parity, (
        name,
        len(observed_v1),
        len(expected_set),
        len(observed_v1 ^ expected_set),
    )


    # --------------------------------------------------------
    # Apply locked V1 endpoint occupancy
    # --------------------------------------------------------

    post_out = dict(outdeg)
    post_in = dict(indeg)

    for s, q in expected_set:

        assert post_out.get(s, 0) == 0
        assert post_in.get(q, 0) == 0

        post_out[s] = 1
        post_in[q] = 1


    # --------------------------------------------------------
    # Round 2: reciprocal ranking only on FREE × FREE
    # --------------------------------------------------------

    n_round2 = 0

    for t, (
        src_ids,
        tgt_ids,
        probs,
    ) in pair_cache.items():

        if probs is None:
            continue


        free_sources = {
            int(s)
            for s in src_ids
            if (
                int(s) in final_ids
                and post_out.get(
                    int(s), 0
                ) == 0
            )
        }

        free_targets = {
            int(q)
            for q in tgt_ids
            if (
                int(q) in final_ids
                and post_in.get(
                    int(q), 0
                ) == 0
            )
        }


        pairs = (
            ir.residual_reciprocal_pairs(
                src_ids,
                tgt_ids,
                probs,
                free_sources,
                free_targets,
            )
        )


        for s, q, p in pairs:

            round2_records.append({
                "dataset":
                    name,

                "fold":
                    int(fold_map[name]),

                "t":
                    int(t),

                "source_pred_node_id":
                    int(s),

                "target_pred_node_id":
                    int(q),

                "candidate_prob":
                    float(p),
            })

        n_round2 += len(pairs)


    summary.append({
        "fold":
            int(fold_map[name]),

        "dataset":
            name,

        "expected_v1":
            len(expected_set),

        "observed_v1":
            len(observed_v1),

        "v1_exact":
            parity,

        "round2_candidates":
            n_round2,
    })


    if torch.backends.mps.is_available():
        torch.mps.empty_cache()


res = pd.DataFrame(summary)
r2 = pd.DataFrame(round2_records)


# ============================================================
# 6. Report
# ============================================================

print(
    "\n========== ROUND-1 PARITY =========="
)

print(
    res.to_string(
        index=False
    )
)


print(
    "\n========== ROUND-2 SMOKE =========="
)

print(
    "datasets:",
    r2["dataset"].nunique()
)

print(
    "candidates:",
    len(r2)
)

print(
    "median / dataset:",
    f"{r2.groupby('dataset').size().median():.1f}"
)


print(
    "\n========== ROUND-2 PROBABILITY =========="
)

print(
    "q10/q25/q50/q75/q90:",
    np.round(
        r2["candidate_prob"]
        .quantile(
            [.10, .25, .50, .75, .90]
        )
        .to_numpy(),
        6,
    )
)

for p in [
    .10,
    .20,
    .25,
    .30,
    .35,
    .40,
    .45,
]:

    n = int(
        r2["candidate_prob"]
        .ge(p)
        .sum()
    )

    print(
        f"p >= {p:.2f}: "
        f"{n:5d} "
        f"({n/len(r2):.3%})"
    )


print(
    "\n========== RUNTIME =========="
)

print(
    "elapsed min:",
    f"{(time.time()-started)/60:.1f}"
)


assert res["v1_exact"].all()
assert len(r2) > 0


print(
    "\n========== 12.06A-S1 DECISION =========="
)

print(
    "ITERATIVE_RECIPROCAL_ROUND2_SMOKE: PASS"
)

In [ ]:
# ============================================================
# RUN — Stage 12.06C Round-2 Probability Grid
# Purpose:
#   Evaluate the predeclared Round-2 probability policy family
#   on folds1–4 development data.
#
# Changes files: YES — Stage12 grid artifacts + log
# Runs training/inference: NO
# Uses GT: YES — folds1–4 only
# Uses Fold0: NO
# Keep after running: NO
# ============================================================

import subprocess
import sys


SCRIPT = (
    PROJECT
    / "scripts/run_stage12_round2_policy_grid.py"
)

LOG = (
    S12
    / "stage12_round2_policy_grid.log"
)


with LOG.open("w") as f:

    result = subprocess.run(
        [
            sys.executable,
            str(SCRIPT),
        ],
        stdout=f,
        stderr=subprocess.STDOUT,
        text=True,
    )


assert result.returncode == 0, (
    f"Grid failed. Inspect {LOG}"
)


text = LOG.read_text(
    encoding="utf-8"
)

print(
    "\n".join(
        text.splitlines()[-60:]
    )
)

In [ ]:
# ============================================================
# RESULT — Stage 12.06D Lock MUTUAL-CONT-V2
# Purpose:
#   Freeze the second-round reciprocal continuation policy
#   selected only from the predeclared folds1–4 grid.
#
# Policy:
#   Start after locked MUTUAL-CONT-V1.
#   Remove all already-occupied source/target endpoints.
#   Recompute reciprocal Top1 inside the remaining
#   free-source × free-target association submatrix.
#   Add candidate iff original association probability >= 0.25.
#
# Changes files: YES
#   reports/stage12_residual_audit/
#       stage12_mutual_cont_v2_policy_lock.json
#
# Runs training/inference: NO
# Uses GT: NO NEW GT READ
# Keep after running: YES
# ============================================================

from pathlib import Path
import hashlib
import json

import pandas as pd


GRID_PATH = (
    S12
    / "stage12_round2_probability_grid.pkl"
)

UNIVERSE_PATH = (
    S12
    / "stage12_folds14_round2_reciprocal_universe.pkl"
)

OUT = (
    S12
    / "stage12_mutual_cont_v2_policy_lock.json"
)


grid = pd.read_pickle(
    GRID_PATH
)

universe = pd.read_pickle(
    UNIVERSE_PATH
)


row = (
    grid[
        grid["config_id"].eq("R2_P25")
    ]
    .iloc[0]
)


assert abs(
    float(row["threshold"]) - 0.25
) < 1e-12

assert int(
    row["actions"]
) == 18511

assert abs(
    float(row["score"])
    - 0.791940513
) <= 1e-9

assert abs(
    float(row["delta_score"])
    - 0.003099268
) <= 1e-9


# ------------------------------------------------------------
# Canonical development action set
# ------------------------------------------------------------

dev_actions = (
    universe[
        universe["candidate_prob"].ge(0.25)
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ]
    )
    .reset_index(drop=True)
)

assert len(dev_actions) == 18511


LOCK_COLS = [
    "dataset",
    "fold",
    "t",
    "source_pred_node_id",
    "target_pred_node_id",
    "candidate_prob",
]

blob = (
    dev_actions[
        LOCK_COLS
    ]
    .to_csv(
        index=False,
        float_format="%.17g",
    )
    .encode()
)

action_sha = hashlib.sha256(
    blob
).hexdigest()


payload = {
    "stage":
        "12.06D",

    "policy_name":
        "MUTUAL-CONT-V2",

    "status":
        "LOCKED_BEFORE_FOLD0_CONFIRMATORY_EVALUATION",

    "predecessor":
        "MUTUAL-CONT-V1",

    "domain":
        "6bba",

    "rule": {
        "starting_topology":
            "POST_MUTUAL_CONT_V1",

        "source_post_v1_outdegree":
            0,

        "target_post_v1_indegree":
            0,

        "ranking_space":
            "FREE_SOURCE_X_FREE_TARGET_SUBMATRIX",

        "round2_row_rank":
            1,

        "round2_column_rank":
            1,

        "candidate_prob_min":
            0.25,

        "operation":
            "ADD_SINGLE_CONTINUATION_EDGE",

        "extra_margin_gate":
            None,

        "distance_gate":
            None,

        "original_rank_gate":
            None,
    },

    "development": {
        "folds":
            [1, 2, 3, 4],

        "actions":
            int(len(dev_actions)),

        "baseline_score":
            0.788841246,

        "policy_score":
            float(row["score"]),

        "delta_score":
            float(row["delta_score"]),

        "action_sha256":
            action_sha,
    },

    "prohibited_after_lock": [
        "change probability threshold",
        "add or change distance gate",
        "add or change margin gate",
        "add or change original-rank gate",
        "change reciprocal ranking space",
        "change source or target topology gate",
    ],
}


OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "========== MUTUAL-CONT-V2 POLICY =========="
)

print(
    "development actions:",
    len(dev_actions)
)

print(
    "development score  :",
    f"{row['score']:.9f}"
)

print(
    "development delta  :",
    f"{row['delta_score']:+.9f}"
)

print(
    "dev action SHA256  :",
    action_sha
)


print(
    "\n========== POLICY CONTRACT =========="
)

print(
    "predecessor             = MUTUAL-CONT-V1"
)

print(
    "post-V1 source outdeg   = 0"
)

print(
    "post-V1 target indeg    = 0"
)

print(
    "ranking space           = free-source x free-target"
)

print(
    "Round-2 row rank        = 1"
)

print(
    "Round-2 column rank     = 1"
)

print(
    "candidate_prob          >= 0.25"
)

print(
    "distance/margin/rank gate = NONE"
)


print(
    "\nwritten:",
    OUT
)


print(
    "\n========== 12.06D DECISION =========="
)

print(
    "MUTUAL_CONT_V2_LOCKED_BEFORE_FOLD0_CONFIRMATION: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.07C Promote MUTUAL-CONT-V2
#          + Full 40-Dataset Integration
# Purpose:
#   Promote locked MUTUAL-CONT-V2 after successful development
#   and Fold0 confirmatory replication.
#
#   44b6 remains unchanged.
#
#   6bba:
#       Baseline256
#       -> Frozen V9
#       -> MORPH-DIV-V2
#       -> MUTUAL-CONT-V1
#       -> MUTUAL-CONT-V2
#
#   First reproduce the V1 40-dataset checkpoint exactly,
#   then substitute the locked V2 6bba per-dataset metrics.
#
# Changes files: YES
#   reports/stage12_residual_audit/
#       stage12_full_hybrid_mutual_cont_v2_datasets.pkl
#       stage12_full_hybrid_mutual_cont_v2_summary.json
#       stage12_current_best_checkpoint.json
#
# Runs training/inference: NO
# Uses GT: NO NEW GT READ
#   Reuses already frozen official metric artifacts.
# Keep after running: YES
# ============================================================

from pathlib import Path
import json
import sys

import pandas as pd


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports/stage12_residual_audit"
)


V1_FULL_PATH = (
    S12
    / "stage12_full_hybrid_mutual_cont_v1_datasets.pkl"
)

V2_CONFIRM_PATH = (
    S12
    / "stage12_fold0_mutual_cont_v2_confirmatory_datasets.pkl"
)

V2_ACTION_LOCK_PATH = (
    S12
    / "stage12_fold0_mutual_cont_v2_action_lock.json"
)

V2_POLICY_LOCK_PATH = (
    S12
    / "stage12_mutual_cont_v2_policy_lock.json"
)

DATASET_OUT = (
    S12
    / "stage12_full_hybrid_mutual_cont_v2_datasets.pkl"
)

SUMMARY_OUT = (
    S12
    / "stage12_full_hybrid_mutual_cont_v2_summary.json"
)

CHECKPOINT_OUT = (
    S12
    / "stage12_current_best_checkpoint.json"
)


sys.path.insert(
    0,
    str(PROJECT / "src"),
)

from biohub_cell_tracking import frozen_v9 as fv9


# ============================================================
# 1. Load immutable artifacts
# ============================================================

v1_full = pd.read_pickle(
    V1_FULL_PATH
)

v2 = pd.read_pickle(
    V2_CONFIRM_PATH
)


assert len(v1_full) == 40
assert len(v2) == 25

assert (
    v1_full["dataset"]
    .str.startswith("44b6_")
    .sum()
    == 15
)

assert (
    v1_full["dataset"]
    .str.startswith("6bba_")
    .sum()
    == 25
)


action_lock = json.loads(
    V2_ACTION_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)

policy_lock = json.loads(
    V2_POLICY_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    action_lock["action_sha256"]
    ==
    "e59cad5260e7b222e067b6b4e1dbddb95f04541978e30e548ab662bc9f62af35"
)

assert (
    policy_lock["policy_name"]
    == "MUTUAL-CONT-V2"
)


# ============================================================
# 2. Metric contract
# ============================================================

METRIC_KEYS = [
    "edge_tp",
    "edge_fp",
    "edge_fn",
    "division_tp",
    "division_fp",
    "division_fn",
    "num_pred_nodes",
    "node_recall",
    "total_node_ratio",
    "edge_jaccard",
    "adj_edge_jaccard",
]


# ============================================================
# 3. Reproduce current V1 full checkpoint
# ============================================================

old_rows = (
    v1_full[
        [
            "dataset",
            "route",
            "actions",
            *METRIC_KEYS,
        ]
    ]
    .to_dict("records")
)


old = fv9.summarise(
    [
        {
            k: r[k]
            for k in METRIC_KEYS
        }
        for r in old_rows
    ]
)


print(
    "========== V1 CURRENT-BEST REPRODUCTION =========="
)

print(
    "edge J      :",
    f"{old['edge_jaccard']:.9f}"
)

print(
    "adj edge J  :",
    f"{old['adj_edge_jaccard']:.9f}"
)

print(
    "division J  :",
    f"{old['division_jaccard']:.9f}"
)

print(
    "node recall :",
    f"{old['node_recall']:.9f}"
)

print(
    "score       :",
    f"{old['score']:.9f}"
)


assert abs(
    old["edge_jaccard"]
    - 0.786971159
) <= 1e-9

assert abs(
    old["adj_edge_jaccard"]
    - 0.779233490
) <= 1e-9

assert abs(
    old["division_jaccard"]
    - 0.046875000
) <= 1e-9

assert abs(
    old["node_recall"]
    - 0.959822345
) <= 1e-9

assert abs(
    old["score"]
    - 0.783920990
) <= 1e-9


# ============================================================
# 4. Keep 44b6 unchanged
# ============================================================

rows44 = (
    v1_full[
        v1_full["dataset"]
        .str.startswith("44b6_")
    ]
    [
        [
            "dataset",
            "route",
            "actions",
            *METRIC_KEYS,
        ]
    ]
    .to_dict("records")
)

assert len(rows44) == 15


# ============================================================
# 5. Replace 6bba with locked V2 metrics
# ============================================================

rows6 = []


for r in v2.itertuples(
    index=False
):

    rows6.append({
        "dataset":
            r.dataset,

        "route":
            (
                "6bba_BASELINE256_FROZEN_V9_"
                "MORPHDIV_V2_MUTUAL_CONT_V1_"
                "MUTUAL_CONT_V2"
            ),

        # V2 actions only.
        "actions":
            int(r.actions),

        **{
            key:
                getattr(
                    r,
                    f"new_{key}",
                )
            for key in METRIC_KEYS
        },
    })


assert len(rows6) == 25


# ============================================================
# 6. Exact new 40-dataset aggregate
# ============================================================

new_rows = (
    rows44
    + rows6
)

assert len(new_rows) == 40


new = fv9.summarise(
    [
        {
            k: r[k]
            for k in METRIC_KEYS
        }
        for r in new_rows
    ]
)


old_tp = sum(
    int(r["edge_tp"])
    for r in old_rows
)

old_fp = sum(
    int(r["edge_fp"])
    for r in old_rows
)

old_fn = sum(
    int(r["edge_fn"])
    for r in old_rows
)


new_tp = sum(
    int(r["edge_tp"])
    for r in new_rows
)

new_fp = sum(
    int(r["edge_fp"])
    for r in new_rows
)

new_fn = sum(
    int(r["edge_fn"])
    for r in new_rows
)


assert new_tp - old_tp == 125
assert new_fp - old_fp == 77
assert new_fn - old_fn == -125


print(
    "\n========== NEW FULL HYBRID + MUTUAL-CONT-V2 =========="
)

print(
    "edge TP/FP/FN:",
    new_tp,
    "/",
    new_fp,
    "/",
    new_fn,
)

print(
    "edge J      :",
    f"{new['edge_jaccard']:.9f}"
)

print(
    "adj edge J  :",
    f"{new['adj_edge_jaccard']:.9f}"
)

print(
    "division J  :",
    f"{new['division_jaccard']:.9f}"
)

print(
    "node recall :",
    f"{new['node_recall']:.9f}"
)

print(
    "score       :",
    f"{new['score']:.9f}"
)


print(
    "\n========== FULL 40-DATASET V2 DELTAS =========="
)

print(
    "edge TP:",
    f"{new_tp-old_tp:+d}"
)

print(
    "edge FP:",
    f"{new_fp-old_fp:+d}"
)

print(
    "edge FN:",
    f"{new_fn-old_fn:+d}"
)

print(
    "edge J:",
    f"{new['edge_jaccard']-old['edge_jaccard']:+.9f}"
)

print(
    "adj edge J:",
    f"{new['adj_edge_jaccard']-old['adj_edge_jaccard']:+.9f}"
)

print(
    "division J:",
    f"{new['division_jaccard']-old['division_jaccard']:+.9f}"
)

print(
    "node recall:",
    f"{new['node_recall']-old['node_recall']:+.9f}"
)

print(
    "score:",
    f"{new['score']-old['score']:+.9f}"
)


assert new["score"] > old["score"]


# ============================================================
# 7. Persist new current-best
# ============================================================

final_df = (
    pd.DataFrame(new_rows)
    .sort_values("dataset")
    .reset_index(drop=True)
)

final_df.to_pickle(
    DATASET_OUT
)


payload = {
    "stage":
        "12.07C",

    "pipeline":
        (
            "FULL_HYBRID_PLUS_MORPH_DIV_V2_"
            "PLUS_MUTUAL_CONT_V1_PLUS_MUTUAL_CONT_V2"
        ),

    "v2_status":
        "PROMOTED",

    "v2_fold0_actions":
        int(
            action_lock["actions"]
        ),

    "v2_fold0_action_sha256":
        action_lock["action_sha256"],

    "old": {
        "edge_tp":
            int(old_tp),

        "edge_fp":
            int(old_fp),

        "edge_fn":
            int(old_fn),

        "edge_jaccard":
            float(old["edge_jaccard"]),

        "adj_edge_jaccard":
            float(old["adj_edge_jaccard"]),

        "division_jaccard":
            float(old["division_jaccard"]),

        "node_recall":
            float(old["node_recall"]),

        "score":
            float(old["score"]),
    },

    "new": {
        "edge_tp":
            int(new_tp),

        "edge_fp":
            int(new_fp),

        "edge_fn":
            int(new_fn),

        "edge_jaccard":
            float(new["edge_jaccard"]),

        "adj_edge_jaccard":
            float(new["adj_edge_jaccard"]),

        "division_jaccard":
            float(new["division_jaccard"]),

        "node_recall":
            float(new["node_recall"]),

        "score":
            float(new["score"]),
    },

    "delta_score":
        float(
            new["score"]
            - old["score"]
        ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


checkpoint = {
    "checkpoint_name":
        (
            "FULL_HYBRID_PLUS_MORPH_DIV_V2_"
            "PLUS_MUTUAL_CONT_V1_PLUS_MUTUAL_CONT_V2"
        ),

    "status":
        "STAGE12_CURRENT_BEST",

    "fold0_metrics": {
        "score":
            float(new["score"]),

        "edge_jaccard":
            float(new["edge_jaccard"]),

        "adj_edge_jaccard":
            float(new["adj_edge_jaccard"]),

        "division_jaccard":
            float(new["division_jaccard"]),

        "node_recall":
            float(new["node_recall"]),
    },

    "edge_counts": {
        "tp":
            int(new_tp),

        "fp":
            int(new_fp),

        "fn":
            int(new_fn),
    },

    "policies": {
        "MORPH-DIV-V2":
            "LOCKED",

        "MUTUAL-CONT-V1":
            "LOCKED_AND_PROMOTED",

        "MUTUAL-CONT-V2":
            "LOCKED_AND_PROMOTED",
    },

    "mutual_cont_v2": {
        "threshold":
            0.25,

        "ranking_space":
            "POST_V1_FREE_SOURCE_X_FREE_TARGET",

        "row_rank":
            1,

        "column_rank":
            1,

        "fold0_actions":
            int(
                action_lock["actions"]
            ),

        "fold0_action_sha256":
            action_lock["action_sha256"],

        "development_action_sha256":
            policy_lock[
                "development"
            ][
                "action_sha256"
            ],
    },
}


CHECKPOINT_OUT.write_text(
    json.dumps(
        checkpoint,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    DATASET_OUT
)

print(
    "written:",
    SUMMARY_OUT
)

print(
    "written:",
    CHECKPOINT_OUT
)


print(
    "\n========== 12.07C DECISION =========="
)

print(
    "MUTUAL_CONT_V2_PROMOTED_AND_INTEGRATED: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.08A Exact Post-V2 Linking Residuals
# Purpose:
#   Reconstruct exactly which post-V1 linking FNs were recovered
#   by MUTUAL-CONT-V2 and derive the remaining topology after V2.
#
#   This determines whether a Round-3 reciprocal continuation
#   branch still has enough theoretical ceiling to justify
#   another full association replay.
#
# Changes files: YES
#   reports/stage12_residual_audit/
#       stage12_post_v2_linking_residuals.pkl
#       stage12_post_v2_linking_residual_summary.json
#
# Runs training/inference: NO
# Uses GT: NO NEW GT READ
#   Reuses previously frozen GT-derived residual artifacts.
# Keep after running: YES
# ============================================================

import json
import pandas as pd


POST_V1_PATH = (
    S12
    / "stage12_post_v1_linking_residuals.pkl"
)

V2_ACTION_PATH = (
    S12
    / "stage12_fold0_mutual_cont_v2_actions_locked.pkl"
)

OUT = (
    S12
    / "stage12_post_v2_linking_residuals.pkl"
)

SUMMARY_OUT = (
    S12
    / "stage12_post_v2_linking_residual_summary.json"
)


# ============================================================
# 1. Load frozen artifacts
# ============================================================

post_v1 = pd.read_pickle(
    POST_V1_PATH
)

v2 = pd.read_pickle(
    V2_ACTION_PATH
)


remaining_v1 = (
    post_v1[
        ~post_v1["recovered_by_v1"]
    ]
    .copy()
)

assert len(remaining_v1) == 2002
assert len(v2) == 4076


# ============================================================
# 2. Exact V2 edge identities
# ============================================================

v2_edges = set(
    zip(
        v2["dataset"],
        v2["source_pred_node_id"].astype(int),
        v2["target_pred_node_id"].astype(int),
    )
)


remaining_v1["recovered_by_v2"] = [
    (
        r.dataset,
        int(r.pred_source_id),
        int(r.pred_target_id),
    ) in v2_edges
    for r in remaining_v1.itertuples(
        index=False
    )
]


recovered_v2 = (
    remaining_v1[
        remaining_v1["recovered_by_v2"]
    ]
)

remaining_v2 = (
    remaining_v1[
        ~remaining_v1["recovered_by_v2"]
    ]
    .copy()
)


# Exact confirmatory accounting.
assert len(recovered_v2) == 125
assert len(remaining_v2) == 1877


# ============================================================
# 3. Endpoint occupation introduced by V2
# ============================================================

v2_sources = set(
    zip(
        v2["dataset"],
        v2["source_pred_node_id"].astype(int),
    )
)

v2_targets = set(
    zip(
        v2["dataset"],
        v2["target_pred_node_id"].astype(int),
    )
)


remaining_v1["v2_uses_source"] = [
    (
        r.dataset,
        int(r.pred_source_id),
    ) in v2_sources
    for r in remaining_v1.itertuples(
        index=False
    )
]

remaining_v1["v2_uses_target"] = [
    (
        r.dataset,
        int(r.pred_target_id),
    ) in v2_targets
    for r in remaining_v1.itertuples(
        index=False
    )
]


remaining_v1["post_v2_source_outdegree"] = (
    remaining_v1["post_v1_source_outdegree"]
    +
    remaining_v1["v2_uses_source"].astype(int)
)

remaining_v1["post_v2_target_indegree"] = (
    remaining_v1["post_v1_target_indegree"]
    +
    remaining_v1["v2_uses_target"].astype(int)
)


def post_v2_topology(row):

    s = int(
        row.post_v2_source_outdegree
    )

    t = int(
        row.post_v2_target_indegree
    )

    return (
        f"SRC_OUT{s}_"
        +
        (
            "TARGET_FREE"
            if t == 0
            else "TARGET_OCCUPIED"
        )
    )


remaining_v1["post_v2_topology"] = (
    remaining_v1.apply(
        post_v2_topology,
        axis=1,
    )
)


remaining_v2 = (
    remaining_v1[
        ~remaining_v1["recovered_by_v2"]
    ]
    .copy()
)


# ============================================================
# 4. Round-3 theoretical free/free ceiling
# ============================================================

free_free_v2 = (
    remaining_v2[
        (
            remaining_v2[
                "post_v2_source_outdegree"
            ].eq(0)
        )
        &
        (
            remaining_v2[
                "post_v2_target_indegree"
            ].eq(0)
        )
    ]
)


# ============================================================
# 5. Exact accounting
# ============================================================

assert (
    len(recovered_v2)
    + len(remaining_v2)
    == 2002
)

assert len(remaining_v2) == 1877


print(
    "========== V2 EXACT FN RECOVERY =========="
)

print(
    "post-V1 linking FN:",
    len(remaining_v1)
)

print(
    "recovered by V2  :",
    len(recovered_v2)
)

print(
    "remaining        :",
    len(remaining_v2)
)

print(
    "V2 recovery rate :",
    f"{len(recovered_v2)/len(remaining_v1):.3%}"
)


print(
    "\n========== RECOVERED BY PREFIX =========="
)

print(
    recovered_v2.groupby(
        "prefix"
    )
    .size()
    .to_string()
)


print(
    "\n========== POST-V2 REMAINING TOPOLOGY =========="
)

print(
    remaining_v2[
        "post_v2_topology"
    ]
    .value_counts()
    .to_string()
)


print(
    "\n========== POST-V2 REMAINING PROVENANCE =========="
)

print(
    remaining_v2[
        "provenance"
    ]
    .value_counts()
    .to_string()
)


print(
    "\n========== ROUND-3 FREE/FREE CEILING =========="
)

print(
    "remaining linking FN:",
    len(remaining_v2)
)

print(
    "post-V2 source0 + target-free:",
    len(free_free_v2),
    f"({len(free_free_v2)/len(remaining_v2):.3%})"
)


print(
    "\nby prefix:"
)

print(
    free_free_v2.groupby(
        "prefix"
    )
    .size()
    .to_string()
)


print(
    "\nby provenance:"
)

print(
    free_free_v2[
        "provenance"
    ]
    .value_counts()
    .to_string()
)


print(
    "\n========== CURRENT RESIDUAL BUDGET =========="
)

print(
    "total edge FN      :",
    3431
)

print(
    "detection-limited  :",
    1554,
    f"({1554/3431:.3%})"
)

print(
    "linking-limited    :",
    1877,
    f"({1877/3431:.3%})"
)


# ============================================================
# 6. Persist
# ============================================================

remaining_v1.to_pickle(
    OUT
)


payload = {
    "stage":
        "12.08A",

    "post_v1_linking_fn":
        2002,

    "recovered_by_mutual_cont_v2":
        int(len(recovered_v2)),

    "post_v2_linking_fn":
        int(len(remaining_v2)),

    "post_v2_free_free_fn":
        int(len(free_free_v2)),

    "post_v2_free_free_fraction":
        float(
            len(free_free_v2)
            / len(remaining_v2)
        ),

    "current_total_edge_fn":
        3431,

    "detection_limited_fn":
        1554,

    "linking_limited_fn":
        1877,
}


SUMMARY_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    OUT
)

print(
    "written:",
    SUMMARY_OUT
)


print(
    "\n========== 12.08A DECISION =========="
)

print(
    "POST_V2_LINKING_RESIDUALS_RECONSTRUCTED: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.08B Post-V2 Rewiring Actionability Census
# Purpose:
#   Quantify the remaining post-V2 linking residuals that are
#   structurally compatible with source replacement, target
#   replacement, or double rewiring.
#
#   No new policy is constructed here.
#
# Changes files: YES
#   reports/stage12_residual_audit/
#       stage12_post_v2_rewiring_actionability.pkl
#       stage12_post_v2_rewiring_actionability_summary.json
#
# Runs training/inference: NO
# Uses GT: NO NEW GT READ
#   Reuses frozen GT-derived residual diagnostics.
# Keep after running: YES
# ============================================================

import json
import pandas as pd


POST_V2_PATH = (
    S12
    / "stage12_post_v2_linking_residuals.pkl"
)

OUT = (
    S12
    / "stage12_post_v2_rewiring_actionability.pkl"
)

SUMMARY_OUT = (
    S12
    / "stage12_post_v2_rewiring_actionability_summary.json"
)


# ============================================================
# 1. Remaining post-V2 linking FNs
# ============================================================

df = pd.read_pickle(
    POST_V2_PATH
)

remaining = (
    df[
        ~df["recovered_by_v2"]
    ]
    .copy()
)

assert len(remaining) == 1877


# ============================================================
# 2. Structural repair class
# ============================================================

def repair_class(row):

    s = int(
        row.post_v2_source_outdegree
    )

    t = int(
        row.post_v2_target_indegree
    )

    if s == 0 and t == 0:
        return "FREE_FREE"

    if s >= 1 and t == 0:
        return "SOURCE_REPLACE"

    if s == 0 and t >= 1:
        return "TARGET_REPLACE"

    if s >= 1 and t >= 1:
        return "DOUBLE_REWIRE"

    return "OTHER"


remaining["repair_class"] = (
    remaining.apply(
        repair_class,
        axis=1,
    )
)


# ============================================================
# 3. Existing official-valid-FP support
# ============================================================

remaining["source_replace_supported"] = (
    remaining["source_has_valid_fp_out"]
    .fillna(False)
    .astype(bool)
)

remaining["target_replace_supported"] = (
    remaining["target_has_valid_fp_in"]
    .fillna(False)
    .astype(bool)
)

remaining["double_rewire_supported"] = (
    remaining["source_replace_supported"]
    &
    remaining["target_replace_supported"]
)


def actionable(row):

    if row.repair_class == "SOURCE_REPLACE":
        return bool(
            row.source_replace_supported
        )

    if row.repair_class == "TARGET_REPLACE":
        return bool(
            row.target_replace_supported
        )

    if row.repair_class == "DOUBLE_REWIRE":
        return bool(
            row.double_rewire_supported
        )

    return False


remaining["structurally_actionable"] = (
    remaining.apply(
        actionable,
        axis=1,
    )
)


# ============================================================
# 4. Reports
# ============================================================

print(
    "========== POST-V2 REPAIR CLASSES =========="
)

print(
    remaining[
        "repair_class"
    ]
    .value_counts()
    .to_string()
)


print(
    "\n========== GT PARENT TYPE BY REPAIR CLASS =========="
)

print(
    pd.crosstab(
        remaining[
            "repair_class"
        ],
        remaining[
            "gt_parent_outdegree"
        ],
    ).to_string()
)


print(
    "\n========== VALID-FP SUPPORT =========="
)

for cls in [
    "SOURCE_REPLACE",
    "TARGET_REPLACE",
    "DOUBLE_REWIRE",
]:

    rr = remaining[
        remaining[
            "repair_class"
        ].eq(cls)
    ]

    aa = rr[
        rr[
            "structurally_actionable"
        ]
    ]

    print(
        f"{cls:16s}: "
        f"{len(aa):4d} / {len(rr):4d} "
        f"({len(aa)/len(rr):.3%})"
    )


actionable = remaining[
    remaining[
        "structurally_actionable"
    ]
]


print(
    "\n========== ACTIONABLE RESIDUAL CEILING =========="
)

print(
    "remaining linking FN:",
    len(remaining)
)

print(
    "structurally actionable:",
    len(actionable),
    f"({len(actionable)/len(remaining):.3%})"
)


print(
    "\nby repair class:"
)

print(
    actionable[
        "repair_class"
    ]
    .value_counts()
    .to_string()
)


print(
    "\nby prefix:"
)

print(
    actionable[
        "prefix"
    ]
    .value_counts()
    .to_string()
)


print(
    "\nby provenance:"
)

print(
    actionable[
        "provenance"
    ]
    .value_counts()
    .to_string()
)


print(
    "\n========== ORDINARY-CONTINUATION ACTIONABILITY =========="
)

ordinary = actionable[
    actionable[
        "gt_parent_outdegree"
    ].eq(1)
]

print(
    "actionable with GT parent outdegree=1:",
    len(ordinary),
    f"({len(ordinary)/len(actionable):.3%})"
)

print(
    ordinary[
        "repair_class"
    ]
    .value_counts()
    .to_string()
)


# ============================================================
# 5. Persist
# ============================================================

remaining.to_pickle(
    OUT
)


payload = {
    "stage":
        "12.08B",

    "remaining_linking_fn":
        int(len(remaining)),

    "repair_class_counts": {
        str(k): int(v)
        for k, v in (
            remaining[
                "repair_class"
            ]
            .value_counts()
            .items()
        )
    },

    "structurally_actionable":
        int(len(actionable)),

    "structurally_actionable_fraction":
        float(
            len(actionable)
            / len(remaining)
        ),

    "ordinary_actionable":
        int(len(ordinary)),
}


SUMMARY_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    OUT
)

print(
    "written:",
    SUMMARY_OUT
)


print(
    "\n========== 12.08B DECISION =========="
)

print(
    "POST_V2_REWIRING_ACTIONABILITY_CENSUS: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.09C GT-Blind Rewire Evidence Barrier Audit
# Purpose:
#   Characterize whether single-rewire candidates have enough
#   observable evidence to challenge their currently retained
#   conflict edges.
#
#   SOURCE_REPLACE:
#       candidate and conflict share the SOURCE but have
#       different target columns. Probability is column-softmax,
#       so raw-logit difference is the primary comparison.
#
#   TARGET_REPLACE:
#       candidate and conflict share the TARGET column, so both
#       probability and raw-logit differences are meaningful.
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO
# Executes rewiring: NO
# Keep after running: YES
# ============================================================

import numpy as np
import pandas as pd


UNIVERSE_PATH = (
    S12
    / "stage12_folds14_single_rewire_universe.pkl"
)

u = pd.read_pickle(
    UNIVERSE_PATH
)

assert len(u) == 261249


# ============================================================
# 1. Global evidence barriers
# ============================================================

print(
    "========== REWIRE EVIDENCE BARRIERS =========="
)

for cls in [
    "SOURCE_REPLACE",
    "TARGET_REPLACE",
]:

    x = u[
        u["repair_class"].eq(cls)
    ]

    print(
        f"\n{cls}"
    )

    print(
        "n:",
        len(x)
    )

    print(
        "candidate raw >= conflict raw:",
        int(
            x[
                "candidate_minus_conflict_raw"
            ].ge(0).sum()
        ),
        f"({x['candidate_minus_conflict_raw'].ge(0).mean():.3%})"
    )

    print(
        "candidate raw >= conflict raw - 0.25:",
        int(
            x[
                "candidate_minus_conflict_raw"
            ].ge(-0.25).sum()
        ),
        f"({x['candidate_minus_conflict_raw'].ge(-0.25).mean():.3%})"
    )

    print(
        "candidate raw >= conflict raw - 0.50:",
        int(
            x[
                "candidate_minus_conflict_raw"
            ].ge(-0.50).sum()
        ),
        f"({x['candidate_minus_conflict_raw'].ge(-0.50).mean():.3%})"
    )

    print(
        "candidate raw >= conflict raw - 1.00:",
        int(
            x[
                "candidate_minus_conflict_raw"
            ].ge(-1.00).sum()
        ),
        f"({x['candidate_minus_conflict_raw'].ge(-1.00).mean():.3%})"
    )

    print(
        "candidate no farther than conflict:",
        int(
            x[
                "candidate_minus_conflict_distance_um"
            ].le(0).sum()
        ),
        f"({x['candidate_minus_conflict_distance_um'].le(0).mean():.3%})"
    )


# ============================================================
# 2. Evidence by conflict origin
# ============================================================

rows = []


for (
    cls,
    origin,
), x in u.groupby(
    [
        "repair_class",
        "conflict_origin",
    ]
):

    rows.append({
        "repair_class":
            cls,

        "conflict_origin":
            origin,

        "n":
            len(x),

        "raw_ge_0":
            int(
                x[
                    "candidate_minus_conflict_raw"
                ].ge(0).sum()
            ),

        "raw_ge_m025":
            int(
                x[
                    "candidate_minus_conflict_raw"
                ].ge(-0.25).sum()
            ),

        "raw_ge_m050":
            int(
                x[
                    "candidate_minus_conflict_raw"
                ].ge(-0.50).sum()
            ),

        "candidate_shorter":
            int(
                x[
                    "candidate_minus_conflict_distance_um"
                ].le(0).sum()
            ),

        "median_delta_raw":
            float(
                x[
                    "candidate_minus_conflict_raw"
                ].median()
            ),

        "median_delta_distance":
            float(
                x[
                    "candidate_minus_conflict_distance_um"
                ].median()
            ),
    })


origin = pd.DataFrame(
    rows
)


for col in [
    "raw_ge_0",
    "raw_ge_m025",
    "raw_ge_m050",
    "candidate_shorter",
]:

    origin[
        f"{col}_rate"
    ] = (
        origin[col]
        / origin["n"]
    )


print(
    "\n========== EVIDENCE BY CONFLICT ORIGIN =========="
)

print(
    origin[
        [
            "repair_class",
            "conflict_origin",
            "n",
            "raw_ge_0_rate",
            "raw_ge_m025_rate",
            "raw_ge_m050_rate",
            "candidate_shorter_rate",
            "median_delta_raw",
            "median_delta_distance",
        ]
    ]
    .sort_values(
        [
            "repair_class",
            "conflict_origin",
        ]
    )
    .to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}",
    )
)


# ============================================================
# 3. Strict observable frontier
#
# SOURCE:
#   raw competition is primary.
#
# TARGET:
#   same target column -> probability competition is also valid.
# ============================================================

source = u[
    u["repair_class"].eq(
        "SOURCE_REPLACE"
    )
].copy()

target = u[
    u["repair_class"].eq(
        "TARGET_REPLACE"
    )
].copy()


print(
    "\n========== SOURCE-REPLACE FRONTIER =========="
)

for dr in [
    0.0,
    -0.25,
    -0.50,
    -1.00,
]:

    mask = (
        source[
            "candidate_minus_conflict_raw"
        ].ge(dr)
    )

    print(
        f"delta_raw >= {dr:+.2f}: "
        f"{int(mask.sum()):6d} "
        f"({mask.mean():.3%})"
    )


print(
    "\n========== TARGET-REPLACE FRONTIER =========="
)

for dp in [
    0.0,
    -0.05,
    -0.10,
    -0.20,
]:

    mask = (
        target[
            "candidate_minus_conflict_prob"
        ].ge(dp)
    )

    print(
        f"delta_prob >= {dp:+.2f}: "
        f"{int(mask.sum()):6d} "
        f"({mask.mean():.3%})"
    )


# ============================================================
# 4. Safety-relevant TARGET_REPLACE subset
# ============================================================

safe_target = target[
    ~target[
        "conflict_parent_is_division"
    ]
]


print(
    "\n========== TARGET-REPLACE DIVISION SAFETY =========="
)

print(
    "all target-replace:",
    len(target)
)

print(
    "division-parent conflicts:",
    int(
        target[
            "conflict_parent_is_division"
        ].sum()
    )
)

print(
    "non-division conflict parent:",
    len(safe_target)
)


print(
    "\n========== 12.09C DECISION =========="
)

print(
    "SINGLE_REWIRE_OBSERVABLE_BARRIERS_AUDITED: PASS"
)

## 12.09D — Single-rewire branch closed

The predeclared single-rewire development family failed.

- V2 development baseline: **0.791940513**
- All non-empty SOURCE_REPLACE policies reduced score.
- Strict SOURCE_REPLACE (`Δraw ≥ 0`) was negative on all four folds.
- TARGET_REPLACE was negative overall and strongly fold-unstable.
- No Fold0 evaluation was performed.
- No post-hoc threshold, distance, margin, origin, or fold-specific rule was introduced.

**Decision:** reject single-rewire and do not proceed to double-rewire.

Current promoted Stage12 pipeline remains:

`MORPH-DIV-V2 -> MUTUAL-CONT-V1 -> MUTUAL-CONT-V2`

In [ ]:
# ============================================================
# RESULT — Stage 12.10A Detection-Limited Endpoint Anatomy
# Purpose:
#   Convert the invariant 1,554 detection-limited edge FNs
#   into unique missing GT nodes using the verified sentinel:
#
#       pred_*_id == -1  <=>  GT endpoint has no matched
#                              predicted node.
#
#   Quantify how many edge FNs each missing GT node causes.
#
# Changes files: YES
#   reports/stage12_residual_audit/
#       stage12_detection_limited_missing_nodes.pkl
#       stage12_detection_limited_endpoint_anatomy.json
#
# Runs training/inference: NO
# Uses GT: NO NEW GT READ
#   Reuses frozen GT-derived Stage12 residuals.
# Keep after running: YES
# ============================================================

import json
import pandas as pd


FN_PATH = (
    S12
    / "stage12_edge_fn_residuals.pkl"
)

OUT = (
    S12
    / "stage12_detection_limited_missing_nodes.pkl"
)

SUMMARY_OUT = (
    S12
    / "stage12_detection_limited_endpoint_anatomy.json"
)


# ============================================================
# 1. Frozen detection-limited residuals
# ============================================================

fn = pd.read_pickle(FN_PATH)

det = (
    fn[
        fn["root_cause"].eq(
            "DETECTION_LIMITED"
        )
    ]
    .copy()
)

assert len(det) == 1554


src_missing = (
    det["pred_source_id"] < 0
)

tgt_missing = (
    det["pred_target_id"] < 0
)

assert (
    src_missing
    | tgt_missing
).all()


# Verified encoding contract.
assert (
    (
        src_missing
        &
        tgt_missing
    ).sum()
    == 689
)

assert (
    (
        src_missing
        &
        ~tgt_missing
    ).sum()
    == 435
)

assert (
    (
        ~src_missing
        &
        tgt_missing
    ).sum()
    == 430
)


det["missing_pattern"] = "BOTH_MISSING"

det.loc[
    src_missing & ~tgt_missing,
    "missing_pattern",
] = "SOURCE_ONLY"

det.loc[
    ~src_missing & tgt_missing,
    "missing_pattern",
] = "TARGET_ONLY"


# ============================================================
# 2. Edge FNs -> missing GT-node incidences
# ============================================================

records = []


for r in det.itertuples(index=False):

    if int(r.pred_source_id) < 0:

        records.append({
            "dataset":
                r.dataset,

            "prefix":
                r.prefix,

            "gt_node_id":
                int(r.gt_source_id),

            "role":
                "SOURCE",
        })


    if int(r.pred_target_id) < 0:

        records.append({
            "dataset":
                r.dataset,

            "prefix":
                r.prefix,

            "gt_node_id":
                int(r.gt_target_id),

            "role":
                "TARGET",
        })


inc = pd.DataFrame(records)

assert len(inc) == (
    435
    + 430
    + 2 * 689
)


# ============================================================
# 3. Collapse repeated edge incidences to missing GT nodes
# ============================================================

nodes = (
    inc.groupby(
        [
            "dataset",
            "prefix",
            "gt_node_id",
        ]
    )
    .agg(
        fn_incidence=(
            "role",
            "size",
        ),

        source_incidence=(
            "role",
            lambda x:
                int(
                    (x == "SOURCE")
                    .sum()
                ),
        ),

        target_incidence=(
            "role",
            lambda x:
                int(
                    (x == "TARGET")
                    .sum()
                ),
        ),
    )
    .reset_index()
)


# A missing interior node appears as target of an incoming edge
# and source of at least one outgoing edge.
nodes["missing_node_type"] = "ONE_SIDED"

nodes.loc[
    (
        nodes[
            "source_incidence"
        ].gt(0)
        &
        nodes[
            "target_incidence"
        ].gt(0)
    ),
    "missing_node_type",
] = "INTERIOR"


# ============================================================
# 4. Reports
# ============================================================

print(
    "========== DETECTION-LIMITED EDGE FN =========="
)

print(
    "edge FN:",
    len(det)
)

print(
    det[
        "missing_pattern"
    ]
    .value_counts()
    .to_string()
)


print(
    "\n========== UNIQUE MISSING GT NODES =========="
)

print(
    "unique missing nodes:",
    len(nodes)
)

print(
    nodes[
        "missing_node_type"
    ]
    .value_counts()
    .to_string()
)


print(
    "\n========== FN INCIDENCE PER MISSING NODE =========="
)

print(
    nodes[
        "fn_incidence"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

print(
    "nodes causing >=2 FN:",
    int(
        nodes[
            "fn_incidence"
        ].ge(2)
        .sum()
    )
)

print(
    "nodes causing >=3 FN:",
    int(
        nodes[
            "fn_incidence"
        ].ge(3)
        .sum()
    )
)


print(
    "\n========== DETECTION LOSS BY PREFIX =========="
)

prefix_summary = (
    nodes.groupby(
        "prefix"
    )
    .agg(
        missing_nodes=(
            "gt_node_id",
            "size",
        ),

        interior_nodes=(
            "missing_node_type",
            lambda x:
                int(
                    (x == "INTERIOR")
                    .sum()
                ),
        ),

        high_leverage_nodes=(
            "fn_incidence",
            lambda x:
                int(
                    (x >= 2)
                    .sum()
                ),
        ),

        fn_incidence=(
            "fn_incidence",
            "sum",
        ),
    )
)

print(
    prefix_summary.to_string()
)


print(
    "\n========== HIGH-LEVERAGE MISSING NODES =========="
)

high = (
    nodes[
        nodes[
            "fn_incidence"
        ].ge(2)
    ]
)

print(
    "total:",
    len(high)
)

print(
    high.groupby(
        "prefix"
    )
    .size()
    .to_string()
)


# ============================================================
# 5. Persist
# ============================================================

nodes.to_pickle(OUT)


payload = {
    "stage":
        "12.10A",

    "missing_id_encoding":
        "pred_node_id < 0",

    "detection_limited_edge_fn":
        int(len(det)),

    "unique_missing_gt_nodes":
        int(len(nodes)),

    "interior_missing_nodes":
        int(
            nodes[
                "missing_node_type"
            ]
            .eq("INTERIOR")
            .sum()
        ),

    "nodes_causing_ge2_fn":
        int(
            nodes[
                "fn_incidence"
            ]
            .ge(2)
            .sum()
        ),

    "nodes_causing_ge3_fn":
        int(
            nodes[
                "fn_incidence"
            ]
            .ge(3)
            .sum()
        ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    OUT
)

print(
    "written:",
    SUMMARY_OUT
)


print(
    "\n========== 12.10A DECISION =========="
)

print(
    "DETECTION_LIMITED_ENDPOINT_ANATOMY: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.10B Missing-Gap Component Anatomy
# Purpose:
#   Collapse detection-limited missing GT nodes into connected
#   gap components using BOTH_MISSING GT edges.
#
#   Measure:
#       missing nodes / component
#       internal missing-missing edges
#       observed-boundary edges
#       unique detection FN edges / component
#
# Changes files: YES
#   reports/stage12_residual_audit/
#       stage12_detection_gap_components.pkl
#       stage12_detection_gap_component_summary.json
#
# Runs training/inference: NO
# Uses GT: NO NEW GT READ
#   Reuses frozen GT-derived residual artifacts.
# Keep after running: YES
# ============================================================

import json
import pandas as pd


FN_PATH = (
    S12
    / "stage12_edge_fn_residuals.pkl"
)

OUT = (
    S12
    / "stage12_detection_gap_components.pkl"
)

SUMMARY_OUT = (
    S12
    / "stage12_detection_gap_component_summary.json"
)


fn = pd.read_pickle(FN_PATH)

det = (
    fn[
        fn["root_cause"].eq(
            "DETECTION_LIMITED"
        )
    ]
    .copy()
)

assert len(det) == 1554


src_missing = det["pred_source_id"].lt(0)
tgt_missing = det["pred_target_id"].lt(0)

assert (src_missing | tgt_missing).all()


# ============================================================
# 1. Union-find over missing GT nodes
# ============================================================

missing_nodes = set()

for r in det.itertuples(index=False):

    if int(r.pred_source_id) < 0:
        missing_nodes.add(
            (
                r.dataset,
                int(r.gt_source_id),
            )
        )

    if int(r.pred_target_id) < 0:
        missing_nodes.add(
            (
                r.dataset,
                int(r.gt_target_id),
            )
        )


assert len(missing_nodes) == 1194


parent = {
    x: x
    for x in missing_nodes
}


def find(x):

    while parent[x] != x:

        parent[x] = parent[
            parent[x]
        ]

        x = parent[x]

    return x


def union(a, b):

    ra = find(a)
    rb = find(b)

    if ra != rb:
        parent[rb] = ra


# BOTH_MISSING edges connect nodes inside one gap.
both = det[
    src_missing
    &
    tgt_missing
]

assert len(both) == 689


for r in both.itertuples(index=False):

    union(
        (
            r.dataset,
            int(r.gt_source_id),
        ),
        (
            r.dataset,
            int(r.gt_target_id),
        ),
    )


# ============================================================
# 2. Component membership
# ============================================================

membership = pd.DataFrame(
    [
        {
            "dataset": ds,
            "gt_node_id": node_id,
            "root": find(
                (ds, node_id)
            ),
        }
        for ds, node_id
        in missing_nodes
    ]
)


root_to_id = {
    root: i
    for i, root in enumerate(
        sorted(
            membership["root"].unique()
        )
    )
}


membership["component_id"] = (
    membership["root"]
    .map(root_to_id)
    .astype(int)
)


node_to_component = {
    (
        r.dataset,
        int(r.gt_node_id),
    ): int(r.component_id)

    for r in membership.itertuples(
        index=False
    )
}


# ============================================================
# 3. Assign each unique detection FN edge to its gap component
# ============================================================

edge_records = []


for r in det.itertuples(index=False):

    sm = int(r.pred_source_id) < 0
    tm = int(r.pred_target_id) < 0


    if sm:

        cid = node_to_component[
            (
                r.dataset,
                int(r.gt_source_id),
            )
        ]

    else:

        cid = node_to_component[
            (
                r.dataset,
                int(r.gt_target_id),
            )
        ]


    if sm and tm:

        other = node_to_component[
            (
                r.dataset,
                int(r.gt_target_id),
            )
        ]

        assert other == cid

        edge_type = "INTERNAL"


    elif sm:

        edge_type = "OUT_BOUNDARY"


    else:

        edge_type = "IN_BOUNDARY"


    edge_records.append({
        "dataset":
            r.dataset,

        "prefix":
            r.prefix,

        "component_id":
            cid,

        "gt_source_id":
            int(r.gt_source_id),

        "gt_target_id":
            int(r.gt_target_id),

        "edge_type":
            edge_type,
    })


edges = pd.DataFrame(
    edge_records
)

assert len(edges) == 1554


# ============================================================
# 4. Component table
# ============================================================

node_counts = (
    membership.groupby(
        [
            "dataset",
            "component_id",
        ]
    )
    .size()
    .rename("missing_nodes")
)


edge_counts = (
    edges.groupby(
        [
            "dataset",
            "component_id",
            "edge_type",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)


components = (
    node_counts.to_frame()
    .join(edge_counts)
    .reset_index()
)


for col in [
    "INTERNAL",
    "IN_BOUNDARY",
    "OUT_BOUNDARY",
]:

    if col not in components:
        components[col] = 0


components["boundary_edges"] = (
    components["IN_BOUNDARY"]
    +
    components["OUT_BOUNDARY"]
)

components["detection_fn_edges"] = (
    components["INTERNAL"]
    +
    components["boundary_edges"]
)


prefix_map = (
    det[
        ["dataset", "prefix"]
    ]
    .drop_duplicates()
    .set_index("dataset")[
        "prefix"
    ]
    .to_dict()
)

components["prefix"] = (
    components["dataset"]
    .map(prefix_map)
)


components["boundary_type"] = "NO_BOUNDARY"

components.loc[
    components["boundary_edges"].eq(1),
    "boundary_type",
] = "ONE_SIDED"

components.loc[
    (
        components["IN_BOUNDARY"].gt(0)
        &
        components["OUT_BOUNDARY"].gt(0)
    ),
    "boundary_type",
] = "TWO_SIDED"


# ============================================================
# 5. Forest contract
# ============================================================

assert len(components) == (
    1194 - 689
)

assert len(components) == 505

assert (
    components[
        "missing_nodes"
    ].sum()
    == 1194
)

assert (
    components[
        "INTERNAL"
    ].sum()
    == 689
)

assert (
    components[
        "boundary_edges"
    ].sum()
    == 865
)

assert (
    components[
        "detection_fn_edges"
    ].sum()
    == 1554
)


# ============================================================
# 6. Report
# ============================================================

print(
    "========== MISSING-GAP COMPONENTS =========="
)

print(
    "components:",
    len(components)
)

print(
    "missing nodes:",
    int(
        components[
            "missing_nodes"
        ].sum()
    )
)

print(
    "internal FN edges:",
    int(
        components[
            "INTERNAL"
        ].sum()
    )
)

print(
    "boundary FN edges:",
    int(
        components[
            "boundary_edges"
        ].sum()
    )
)

print(
    "total detection FN:",
    int(
        components[
            "detection_fn_edges"
        ].sum()
    )
)


print(
    "\n========== COMPONENT SIZE DISTRIBUTION =========="
)

print(
    components[
        "missing_nodes"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\n========== FN EDGES PER COMPONENT =========="
)

print(
    components[
        "detection_fn_edges"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\n========== COMPONENT BOUNDARY TYPE =========="
)

print(
    components[
        "boundary_type"
    ]
    .value_counts()
    .to_string()
)


print(
    "\n========== COMPONENTS BY PREFIX =========="
)

print(
    components.groupby(
        "prefix"
    )
    .agg(
        components=(
            "component_id",
            "size",
        ),

        missing_nodes=(
            "missing_nodes",
            "sum",
        ),

        detection_fn_edges=(
            "detection_fn_edges",
            "sum",
        ),

        median_gap_size=(
            "missing_nodes",
            "median",
        ),

        max_gap_size=(
            "missing_nodes",
            "max",
        ),
    )
    .to_string()
)


print(
    "\n========== HIGH-YIELD GAP COMPONENTS =========="
)

for threshold in [
    2,
    3,
    4,
    5,
]:

    x = components[
        components[
            "detection_fn_edges"
        ].ge(threshold)
    ]

    print(
        f"components with >= {threshold} FN edges: "
        f"{len(x)} "
        f"({len(x)/len(components):.3%})"
    )


# ============================================================
# 7. Persist
# ============================================================

components.to_pickle(
    OUT
)


payload = {
    "stage":
        "12.10B",

    "components":
        int(len(components)),

    "missing_nodes":
        1194,

    "internal_missing_edges":
        689,

    "boundary_missing_edges":
        865,

    "detection_fn_edges":
        1554,

    "mean_missing_nodes_per_component":
        float(
            components[
                "missing_nodes"
            ].mean()
        ),

    "mean_fn_edges_per_component":
        float(
            components[
                "detection_fn_edges"
            ].mean()
        ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    OUT
)

print(
    "written:",
    SUMMARY_OUT
)


print(
    "\n========== 12.10B DECISION =========="
)

print(
    "DETECTION_GAP_COMPONENTS_RECONSTRUCTED: PASS"
)

In [ ]:
# ============================================================
# SETUP — Stage 12.10G Fold0 6bba Confirmation Corpus
# Purpose:
#   Freeze the Fold0 6bba corpus for independent R1P50
#   confirmation and verify both Baseline256 prediction roots.
#
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import pandas as pd


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

CV_PATH = (
    PROJECT
    / "configs/cv_manifest.csv"
)

P995 = (
    PROJECT
    / "predictions/heqiuyan/"
      "stage10_infer_baseline_256_det995/split_0"
)

P9975 = (
    PROJECT
    / "predictions/heqiuyan/"
      "stage10_infer_baseline_256_det9975/split_0"
)


cv = pd.read_csv(CV_PATH)


fold0 = (
    cv[
        cv["fold"].eq(0)
    ]
    .sort_values(
        ["prefix", "dataset"]
    )
    .reset_index(drop=True)
)


fold0_6bba = (
    fold0[
        fold0[
            "dataset"
        ].str.startswith("6bba_")
    ]
    .copy()
    .reset_index(drop=True)
)


names = (
    fold0_6bba[
        "dataset"
    ].tolist()
)


missing_995 = [
    name
    for name in names
    if not (
        P995
        / f"{name}.geff"
    ).exists()
]


missing_9975 = [
    name
    for name in names
    if not (
        P9975
        / f"{name}.geff"
    ).exists()
]


assert len(fold0) == 40

assert len(names) > 0

assert not missing_995, (
    "Missing P995 predictions: "
    f"{missing_995}"
)

assert not missing_9975, (
    "Missing P9975 predictions: "
    f"{missing_9975}"
)


print(
    "========== FOLD0 CONFIRMATION CORPUS =========="
)

print(
    "Fold0 total datasets:",
    len(fold0)
)

print(
    "Fold0 6bba datasets:",
    len(fold0_6bba)
)

print(
    "Fold0 44b6 datasets:",
    len(fold0) - len(fold0_6bba)
)


print(
    "\n========== BASELINE256 ROUTE =========="
)

print(
    "P995 complete:",
    len(names),
    "/",
    len(names),
)

print(
    "P9975 complete:",
    len(names),
    "/",
    len(names),
)


print(
    "\n========== FOLD0 6BBA DATASETS =========="
)

print(
    fold0_6bba[
        [
            "dataset",
            "estimated_nodes",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n========== 12.10G DECISION =========="
)

print(
    "FOLD0_R1P50_CONFIRMATION_CORPUS_LOCKED: PASS"
)

In [ ]:
# ============================================================
# RUN — Stage 12.10G1 Fold0 Suppressed-Peak Universe
# Purpose:
#   Extract the GT-blind p>.90 suppressed detector-peak
#   universe for all 25 Fold0 6bba datasets.
#
#   Hard fidelity gate:
#       reconstructed P995 coordinates must exactly reproduce
#       persisted Fold0 Baseline256 P995 predictions.
#
# Changes files: YES — Fold0 Stage12 artifacts + log
# Runs training: NO
# Runs inference: YES — 25 datasets
# Uses GT: NO
# Keep after running: NO
#   Delete after successful extraction.
# ============================================================

from pathlib import Path
import subprocess
import sys


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

SCRIPT = (
    PROJECT
    / "scripts"
    / "run_stage12_fold0_suppressed_peak_universe.py"
)

LOG = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
    / "stage12_fold0_suppressed_peak_universe.log"
)


# ------------------------------------------------------------
# Verify exact Fold0 runner identity
# ------------------------------------------------------------

assert SCRIPT.exists(), SCRIPT

source = SCRIPT.read_text(
    encoding="utf-8",
)

assert 'cv["fold"].eq(0)' in source
assert "assert len(corp) == 25" in source
assert "stage10_infer_baseline_256_det995" in source
assert "FLOOR = 0.90" in source
assert "PERSISTED_THRESHOLD = 0.995" in source
assert "FOLD0_SUPPRESSED_PEAK_UNIVERSE_EXTRACTED" in source


print("RUNNING:")
print(SCRIPT)

print()
print("FOLD0 6bba DATASETS: 25")
print("CANDIDATE FLOOR: 0.90")
print("PERSISTED THRESHOLD: 0.995")
print("GT USED: NO")

print()
print("LOG:")
print(LOG)


# ------------------------------------------------------------
# Run
# ------------------------------------------------------------

with LOG.open(
    "w",
    encoding="utf-8",
) as f:

    result = subprocess.run(
        [
            sys.executable,
            str(SCRIPT),
        ],
        stdout=f,
        stderr=subprocess.STDOUT,
        text=True,
    )


# ------------------------------------------------------------
# Show concise final output
# ------------------------------------------------------------

text = LOG.read_text(
    encoding="utf-8",
    errors="replace",
)

print(
    "\n".join(
        text.splitlines()[-100:]
    )
)


assert result.returncode == 0, (
    "\nStage 12.10G1 failed.\n"
    f"Inspect log: {LOG}"
)

In [ ]:
# ============================================================
# SETUP — Stage 12.10G3 Build Fold0 Association Runner
# Purpose:
#   Derive the Fold0 augmented-association scorer from the
#   already-validated Stage 12.10E full scorer.
#
#   Locked changes only:
#     development B12 (15074) -> Fold0 B12 (3174)
#     folds1–4 corpus -> Fold0 25-dataset corpus
#     development P995 route -> Fold0 Baseline256 P995 route
#     artifact namespace -> fold0
#
#   Transformer scoring / TTA / first-seen / augmentation
#   semantics remain unchanged.
#
# Changes files: YES — creates Fold0 association runner
# Runs training/inference: NO
# Uses GT: NO
# Keep after running: YES
# ============================================================

from pathlib import Path


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

SOURCE = (
    PROJECT
    / "scripts"
    / "run_stage12_bridge_association_universe.py"
)

TARGET = (
    PROJECT
    / "scripts"
    / "run_stage12_fold0_bridge_association_universe.py"
)


assert SOURCE.exists(), SOURCE

text = SOURCE.read_text(
    encoding="utf-8"
)


# ------------------------------------------------------------
# 1. Verify validated 12.10E source identity
# ------------------------------------------------------------

required = [
    "FULL_AUGMENTED_ASSOCIATION_EVIDENCE_EXTRACTED",
    "stage12_folds14_bridge_geometry_b12.pkl",
    "stage12_folds14_bridge_association_evidence.pkl",
    "assert len(evidence) == 15074",
    "score_context",
    "encode_windows_with_tta",
]

for marker in required:
    assert marker in text, marker


# ------------------------------------------------------------
# 2. Fold0 prediction + artifact paths
# ------------------------------------------------------------

replacements = {
    "stage11_fullgraph_baseline256_folds1to4_det995/split_0":
        "stage10_infer_baseline_256_det995/split_0",

    "stage12_folds14_bridge_geometry_b12.pkl":
        "stage12_fold0_bridge_geometry_b12.pkl",

    "stage12_folds14_bridge_association_evidence.pkl":
        "stage12_fold0_bridge_association_evidence.pkl",

    "stage12_folds14_bridge_association_dataset_summary.pkl":
        "stage12_fold0_bridge_association_dataset_summary.pkl",

    "stage12_folds14_bridge_association_summary.json":
        "stage12_fold0_bridge_association_summary.json",
}

for old, new in replacements.items():

    assert old in text, old

    text = text.replace(
        old,
        new,
    )


# ------------------------------------------------------------
# 3. Replace entire validated Section 2 with Fold0 contract
# ------------------------------------------------------------

section2_start = text.index(
    "# 2. Locked B12 universe"
)

section3_start = text.index(
    "# 3. Frozen model"
)


fold0_section2 = '''# 2. Locked Fold0 B12 universe
# ============================================================

b12 = pd.read_pickle(
    B12_PATH
)

assert len(b12) == 3174
assert b12["dataset"].nunique() == 25


cv = pd.read_csv(
    PROJECT
    / "configs/cv_manifest.csv"
)

corp = (
    cv[
        cv["fold"].eq(0)
        &
        cv["dataset"].str.startswith("6bba_")
    ]
    .sort_values(
        ["dataset"]
    )
    .reset_index(drop=True)
)

assert len(corp) == 25
assert corp["dataset"].nunique() == 25


all_names = (
    corp["dataset"]
    .tolist()
)

fold_map = dict(
    zip(
        corp["dataset"],
        corp["fold"],
    )
)


b12 = (
    b12
    .sort_values(
        [
            "dataset",
            "t",
            "z_ds",
            "y_ds",
            "x_ds",
        ]
    )
    .reset_index(drop=True)
)


key_cols = [
    "dataset",
    "t",
    "z_ds",
    "y_ds",
    "x_ds",
]


assert (
    b12[
        key_cols
    ]
    .duplicated()
    .sum()
    == 0
)


assert set(
    b12["dataset"]
) == set(
    all_names
)


names = all_names

zero_b12_names = []


print(
    "Fold0 development corpus:",
    len(all_names),
)

print(
    "Candidate-bearing datasets:",
    len(names),
)

print(
    "Zero-B12 datasets:",
    zero_b12_names,
)


# ============================================================
'''

text = (
    text[:section2_start]
    + fold0_section2
    + text[section3_start:]
)


# ------------------------------------------------------------
# 4. Remove development-only zero-B12 bypass block
# ------------------------------------------------------------

zero_marker = (
    "# Zero-B12 datasets belong to the 103-dataset development"
)

section6_marker = (
    "# 6. Assemble exact evidence table"
)

if zero_marker in text:

    z0 = text.index(
        zero_marker
    )

    z1 = text.index(
        section6_marker,
        z0,
    )

    text = (
        text[:z0]
        + text[z1:]
    )


# ------------------------------------------------------------
# 5. Fold0 evidence row contract
# ------------------------------------------------------------

assert (
    "assert len(evidence) == 15074"
    in text
)

text = text.replace(
    "assert len(evidence) == 15074",
    "assert len(evidence) == 3174",
    1,
)


# ------------------------------------------------------------
# 6. Replace final parity report with Fold0 25/25 contract
# ------------------------------------------------------------

parity_start = text.index(
    'print(\n    "\\n========== P995 FRAME PARITY =========="'
)

persist_section = text.index(
    "# 8. Persist",
    parity_start,
)


fold0_parity = '''print(
    "\\n========== P995 FRAME PARITY =========="
)

dataset_summary = pd.DataFrame(
    dataset_rows
)

assert len(dataset_summary) == 25
assert dataset_summary["dataset"].nunique() == 25

exact_scored = (
    dataset_summary[
        "parity_frames"
    ]
    ==
    dataset_summary[
        "total_frames"
    ]
)

assert bool(
    exact_scored.all()
)

print(
    "scored datasets exact:",
    int(
        exact_scored.sum()
    ),
    "/",
    len(
        dataset_summary
    ),
)

print(
    "full Fold0 corpus:",
    len(
        dataset_summary
    ),
    "/ 25",
)


# ============================================================
'''

text = (
    text[:parity_start]
    + fold0_parity
    + text[persist_section:]
)


# ------------------------------------------------------------
# 7. Fold0 metadata / stage marker
# ------------------------------------------------------------

text = text.replace(
    '"stage":\n        "12.10E"',
    '"stage":\n        "12.10G3"',
    1,
)


# Patched development scorer stores this metadata.
text = text.replace(
    '"development_datasets":\n        103',
    '"development_datasets":\n        25',
    1,
)


text = text.replace(
    "FULL BRIDGE ASSOCIATION EVIDENCE",
    "FOLD0 BRIDGE ASSOCIATION EVIDENCE",
    1,
)


text = text.replace(
    "FULL_AUGMENTED_ASSOCIATION_EVIDENCE_EXTRACTED",
    "FOLD0_AUGMENTED_ASSOCIATION_EVIDENCE_EXTRACTED",
    1,
)


# ------------------------------------------------------------
# 8. Write + syntax check
# ------------------------------------------------------------

TARGET.write_text(
    text,
    encoding="utf-8",
)

compiled = TARGET.read_text(
    encoding="utf-8",
)

compile(
    compiled,
    str(TARGET),
    "exec",
)


# ------------------------------------------------------------
# 9. Hard Fold0 identity checks
# ------------------------------------------------------------

assert "assert len(b12) == 3174" in compiled
assert 'cv["fold"].eq(0)' in compiled
assert "assert len(corp) == 25" in compiled

assert (
    "stage10_infer_baseline_256_det995/split_0"
    in compiled
)

assert (
    "stage12_fold0_bridge_geometry_b12.pkl"
    in compiled
)

assert (
    "stage12_fold0_bridge_association_evidence.pkl"
    in compiled
)

assert (
    "stage12_fold0_bridge_association_dataset_summary.pkl"
    in compiled
)

assert (
    "stage12_fold0_bridge_association_summary.json"
    in compiled
)

assert (
    "assert len(evidence) == 3174"
    in compiled
)

assert (
    "FOLD0_AUGMENTED_ASSOCIATION_EVIDENCE_EXTRACTED"
    in compiled
)


# Critical fidelity mechanisms must remain present.
for marker in [
    "encode_windows_with_tta",
    "_detect_cells_pooled",
    "score_context",
    "torch.softmax",
    "in_col_rank",
    "out_col_rank",
    "bridge_min_prob",
]:

    assert marker in compiled, marker


print(
    "========== FOLD0 ASSOCIATION RUNNER =========="
)

print(TARGET)


print(
    "\n========== CONTRACT =========="
)

print(
    "Fold0 datasets: 25"
)

print(
    "B12 candidates: 3174"
)

print(
    "P995 route: stage10_infer_baseline_256_det995"
)

print(
    "augmentation semantics: unchanged from validated 12.10E"
)

print(
    "GT used: NO"
)


print(
    "\n========== OUTPUTS =========="
)

print(
    "stage12_fold0_bridge_association_evidence.pkl"
)

print(
    "stage12_fold0_bridge_association_dataset_summary.pkl"
)

print(
    "stage12_fold0_bridge_association_summary.json"
)


print(
    "\n========== 12.10G3 DECISION =========="
)

print(
    "FOLD0_ASSOCIATION_RUNNER_BUILT: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.10G6 Full40 R1P50 Promotion
# Purpose:
#   Promote the independently confirmed R1P50 component into
#   the current full Fold0 40-dataset hybrid:
#
#     15 x 44b6: CURRENT_V2 unchanged
#     25 x 6bba: CURRENT_V2 -> confirmed R1P50
#
#   Recompute the exact official aggregate score and persist
#   the promoted full40 checkpoint artifacts.
#
# Changes files: YES — promoted full40 artifacts
# Runs training/inference: NO
# Uses GT: YES — previously evaluated metric rows only
# Keep after running: YES
# ============================================================

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports/stage12_residual_audit"
)

sys.path[:0] = [
    str(PROJECT / "src"),
    str(PROJECT / "scripts"),
]

from biohub_cell_tracking import frozen_v9 as fv9


BASE_PATH = (
    S12
    / "stage12_full_hybrid_mutual_cont_v2_datasets.pkl"
)

CONFIRM_PATH = (
    S12
    / "stage12_fold0_r1p50_confirmation_datasets.pkl"
)

OUT_DATASETS = (
    S12
    / "stage12_full_hybrid_mutual_cont_v2_r1p50_datasets.pkl"
)

OUT_SUMMARY = (
    S12
    / "stage12_full_hybrid_mutual_cont_v2_r1p50_summary.json"
)


# ------------------------------------------------------------
# 1. Current full40 V2 baseline
# ------------------------------------------------------------

base40 = pd.read_pickle(
    BASE_PATH
).copy()

assert len(base40) == 40
assert base40["dataset"].nunique() == 40


baseline_summary = fv9.summarise(
    base40.to_dict("records")
)


assert np.isclose(
    baseline_summary["score"],
    0.786144721,
    atol=5e-7,
    rtol=0.0,
)


# ------------------------------------------------------------
# 2. Confirmed Fold0 R1P50 rows
# ------------------------------------------------------------

confirm = pd.read_pickle(
    CONFIRM_PATH
)

r1 = (
    confirm[
        confirm["policy_id"].eq(
            "R1P50"
        )
    ]
    .copy()
)


assert len(r1) == 25
assert r1["dataset"].nunique() == 25


r1_names = set(
    r1["dataset"]
)

assert all(
    name.startswith("6bba_")
    for name in r1_names
)


# ------------------------------------------------------------
# 3. Keep 15 Fold0 44b6 rows unchanged
# ------------------------------------------------------------

unchanged = (
    base40[
        ~base40["dataset"].isin(
            r1_names
        )
    ]
    .copy()
)


assert len(unchanged) == 15

assert all(
    name.startswith("44b6_")
    for name in unchanged["dataset"]
)


# ------------------------------------------------------------
# 4. Required metric schema
# ------------------------------------------------------------

required = [
    "edge_tp",
    "edge_fp",
    "edge_fn",
    "division_tp",
    "division_fp",
    "division_fn",
    "node_recall",
    "adj_edge_jaccard",
]


assert not [
    c
    for c in required
    if c not in r1.columns
]


# ------------------------------------------------------------
# 5. Build promoted Full40 metric table
# ------------------------------------------------------------

promoted = pd.concat(
    [
        unchanged,
        r1,
    ],
    ignore_index=True,
    sort=False,
)


assert len(promoted) == 40
assert promoted["dataset"].nunique() == 40

assert (
    promoted["dataset"]
    .str.startswith("44b6_")
    .sum()
    == 15
)

assert (
    promoted["dataset"]
    .str.startswith("6bba_")
    .sum()
    == 25
)


# ------------------------------------------------------------
# 6. Official aggregate
# ------------------------------------------------------------

summary = fv9.summarise(
    promoted.to_dict("records")
)

delta = (
    float(summary["score"])
    -
    float(baseline_summary["score"])
)


print(
    "========== CURRENT FULL40 V2 =========="
)

print(
    "edge:",
    f"{baseline_summary['edge_jaccard']:.9f}"
)

print(
    "adj:",
    f"{baseline_summary['adj_edge_jaccard']:.9f}"
)

print(
    "division:",
    f"{baseline_summary['division_jaccard']:.9f}"
)

print(
    "node recall:",
    f"{baseline_summary['node_recall']:.9f}"
)

print(
    "score:",
    f"{baseline_summary['score']:.9f}"
)


print(
    "\n========== FULL40 + R1P50 =========="
)

print(
    "edge:",
    f"{summary['edge_jaccard']:.9f}"
)

print(
    "adj:",
    f"{summary['adj_edge_jaccard']:.9f}"
)

print(
    "division:",
    f"{summary['division_jaccard']:.9f}"
)

print(
    "node recall:",
    f"{summary['node_recall']:.9f}"
)

print(
    "score:",
    f"{summary['score']:.9f}"
)

print(
    "delta score:",
    f"{delta:+.9f}"
)


assert delta > 0


# ------------------------------------------------------------
# 7. Persist promoted checkpoint
# ------------------------------------------------------------

promoted.to_pickle(
    OUT_DATASETS
)


payload = {
    "stage":
        "12.10G6",

    "pipeline":
        (
            "FULL_HYBRID_PLUS_MORPH_DIV_V2"
            "_PLUS_MUTUAL_CONT_V1"
            "_PLUS_MUTUAL_CONT_V2"
            "_PLUS_DET_BRIDGE_R1P50"
        ),

    "full40_datasets":
        40,

    "r1p50_datasets":
        25,

    "r1p50_fold0_action_sha256":
        (
            "270fff00e3016403b378fddc14cdbd651"
            "b300f387167c0bfb050bc87b37e597d"
        ),

    "baseline_score":
        float(
            baseline_summary["score"]
        ),

    "score":
        float(
            summary["score"]
        ),

    "delta_score":
        float(delta),

    "edge_jaccard":
        float(
            summary["edge_jaccard"]
        ),

    "adj_edge_jaccard":
        float(
            summary["adj_edge_jaccard"]
        ),

    "division_jaccard":
        float(
            summary["division_jaccard"]
        ),

    "node_recall":
        float(
            summary["node_recall"]
        ),

    "promotion":
        "CONFIRMED",
}


OUT_SUMMARY.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    OUT_DATASETS
)

print(
    "written:",
    OUT_SUMMARY
)


print(
    "\n========== 12.10G6 DECISION =========="
)

print(
    "R1P50_PROMOTED_TO_FULL40: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.10G7 Freeze Current Best Checkpoint
# Purpose:
#   Promote confirmed DET-BRIDGE-R1P50 into the Stage12
#   current-best checkpoint and record exact Full40 metrics.
#
# Changes files: YES — stage12_current_best_checkpoint.json
# Runs training/inference: NO
# Uses GT: NO — reads already-evaluated cached metric rows
# Keep after running: YES
# ============================================================

from pathlib import Path
import json
import shutil
import sys

import pandas as pd


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports/stage12_residual_audit"
)

DATASETS = (
    S12
    / "stage12_full_hybrid_mutual_cont_v2_r1p50_datasets.pkl"
)

SUMMARY = (
    S12
    / "stage12_full_hybrid_mutual_cont_v2_r1p50_summary.json"
)

CHECKPOINT = (
    S12
    / "stage12_current_best_checkpoint.json"
)

BACKUP = (
    S12
    / "stage12_current_best_checkpoint.pre_r1p50.json"
)


sys.path[:0] = [
    str(PROJECT / "src"),
    str(PROJECT / "scripts"),
]

from biohub_cell_tracking import frozen_v9 as fv9


# ------------------------------------------------------------
# 1. Recompute exact Full40 summary from frozen metric rows
# ------------------------------------------------------------

df = pd.read_pickle(
    DATASETS
)

assert len(df) == 40
assert df["dataset"].nunique() == 40


summary = fv9.summarise(
    df.to_dict("records")
)


assert abs(
    summary["score"]
    - 0.786344478
) < 5e-7


# ------------------------------------------------------------
# 2. Exact aggregate edge counts
# ------------------------------------------------------------

edge_tp = int(
    df["edge_tp"].sum()
)

edge_fp = int(
    df["edge_fp"].sum()
)

edge_fn = int(
    df["edge_fn"].sum()
)

division_tp = int(
    df["division_tp"].sum()
)

division_fp = int(
    df["division_fp"].sum()
)

division_fn = int(
    df["division_fn"].sum()
)


# ------------------------------------------------------------
# 3. Preserve previous checkpoint once
# ------------------------------------------------------------

if (
    CHECKPOINT.exists()
    and not BACKUP.exists()
):

    shutil.copy2(
        CHECKPOINT,
        BACKUP,
    )


# ------------------------------------------------------------
# 4. Freeze promoted checkpoint
# ------------------------------------------------------------

payload = {
    "stage":
        "12.10G7",

    "status":
        "CURRENT_BEST",

    "pipeline":
        (
            "FULL_HYBRID_PLUS_MORPH_DIV_V2"
            "_PLUS_MUTUAL_CONT_V1"
            "_PLUS_MUTUAL_CONT_V2"
            "_PLUS_DET_BRIDGE_R1P50"
        ),

    "score":
        float(
            summary["score"]
        ),

    "edge_jaccard":
        float(
            summary["edge_jaccard"]
        ),

    "adj_edge_jaccard":
        float(
            summary["adj_edge_jaccard"]
        ),

    "division_jaccard":
        float(
            summary["division_jaccard"]
        ),

    "node_recall":
        float(
            summary["node_recall"]
        ),

    "edge_tp":
        edge_tp,

    "edge_fp":
        edge_fp,

    "edge_fn":
        edge_fn,

    "division_tp":
        division_tp,

    "division_fp":
        division_fp,

    "division_fn":
        division_fn,

    "previous_score":
        0.786144721,

    "promotion_delta_score":
        float(
            summary["score"]
            - 0.786144721
        ),

    "det_bridge_policy": {
        "policy_id":
            "R1P50",

        "geometry":
            "B12",

        "incoming_col_rank":
            1,

        "outgoing_col_rank":
            1,

        "bridge_min_prob":
            0.50,

        "development_action_sha256":
            (
                "2d80d9b1cda3c4f0593aced5e8010051"
                "cc49500ef64868541365737dcbe1592f"
            ),

        "fold0_action_sha256":
            (
                "270fff00e3016403b378fddc14cdbd651"
                "b300f387167c0bfb050bc87b37e597d"
            ),

        "fold0_raw_eligible":
            152,

        "fold0_accepted_actions":
            152,

        "fold0_confirmation_delta":
            0.000246305,

        "fold0_leave_one_out_positive":
            "25/25",
    },

    "source_datasets_artifact":
        str(
            DATASETS.relative_to(
                PROJECT
            )
        ),

    "source_summary_artifact":
        str(
            SUMMARY.relative_to(
                PROJECT
            )
        ),

    "branch_decision":
        (
            "SINGLE_FRAME_SUPPRESSED_NODE_BRIDGE_"
            "R1P50_CONFIRMED_AND_PROMOTED"
        ),
}


CHECKPOINT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 5. Report
# ------------------------------------------------------------

print(
    "========== NEW CURRENT BEST =========="
)

print(
    "pipeline:",
    payload["pipeline"]
)

print(
    "score:",
    f"{payload['score']:.9f}"
)

print(
    "edge:",
    f"{payload['edge_jaccard']:.9f}"
)

print(
    "adj:",
    f"{payload['adj_edge_jaccard']:.9f}"
)

print(
    "division:",
    f"{payload['division_jaccard']:.9f}"
)

print(
    "node recall:",
    f"{payload['node_recall']:.9f}"
)

print(
    "edge TP/FP/FN:",
    edge_tp,
    edge_fp,
    edge_fn,
)

print(
    "promotion delta:",
    f"{payload['promotion_delta_score']:+.9f}"
)


print(
    "\nwritten:",
    CHECKPOINT
)

if BACKUP.exists():

    print(
        "previous checkpoint backup:",
        BACKUP
    )


print(
    "\n========== STAGE 12.10 DECISION =========="
)

print(
    "DET_BRIDGE_R1P50_PROMOTED_AND_STAGE12_10_CLOSED: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11B CURRENT_BEST Detection Gap Components
# Purpose:
#   Recompute detection-limited missing-node components under
#   the new R1P50 CURRENT_BEST residual census.
#
#   Quantify the remaining ceiling for:
#       single-frame gaps
#       multi-frame / multi-node gaps
#       internal missing->missing edges
#       boundary edges
#
# Changes files: YES — updated gap-component artifacts
# Runs training/inference: NO
# Uses GT: NO NEW GT READ
#   Reuses frozen CURRENT_BEST GT-derived residual artifact.
# Keep after running: YES
# ============================================================

from pathlib import Path
import json
import pandas as pd


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)

FN_PATH = (
    S12
    / "stage12_current_best_edge_fn_residuals.pkl"
)

OUT = (
    S12
    / "stage12_current_best_detection_gap_components.pkl"
)

SUMMARY_OUT = (
    S12
    / "stage12_current_best_detection_gap_component_summary.json"
)


# ------------------------------------------------------------
# 1. Exact CURRENT_BEST detection FNs
# ------------------------------------------------------------

fn = pd.read_pickle(FN_PATH)

assert len(fn) == 3425


det = (
    fn[
        fn["root_cause"].eq(
            "DETECTION_LIMITED"
        )
    ]
    .copy()
)

assert len(det) == 1550


src_missing = det["pred_source_id"].lt(0)
tgt_missing = det["pred_target_id"].lt(0)

assert (src_missing | tgt_missing).all()

assert int(
    (src_missing & tgt_missing).sum()
) == 689

assert int(
    (src_missing & ~tgt_missing).sum()
) == 433

assert int(
    (~src_missing & tgt_missing).sum()
) == 428


# ------------------------------------------------------------
# 2. Missing GT-node universe
# ------------------------------------------------------------

missing_nodes = set()

for r in det.itertuples(index=False):

    if int(r.pred_source_id) < 0:
        missing_nodes.add(
            (
                r.dataset,
                int(r.gt_source_id),
            )
        )

    if int(r.pred_target_id) < 0:
        missing_nodes.add(
            (
                r.dataset,
                int(r.gt_target_id),
            )
        )


# ------------------------------------------------------------
# 3. Union-find on BOTH_MISSING edges
# ------------------------------------------------------------

parent = {
    x: x
    for x in missing_nodes
}


def find(x):

    while parent[x] != x:
        parent[x] = parent[
            parent[x]
        ]
        x = parent[x]

    return x


def union(a, b):

    ra = find(a)
    rb = find(b)

    if ra != rb:
        parent[rb] = ra


both = det[
    src_missing
    & tgt_missing
]

assert len(both) == 689


for r in both.itertuples(index=False):

    union(
        (
            r.dataset,
            int(r.gt_source_id),
        ),
        (
            r.dataset,
            int(r.gt_target_id),
        ),
    )


# ------------------------------------------------------------
# 4. Stable component IDs
# ------------------------------------------------------------

membership = pd.DataFrame([
    {
        "dataset": ds,
        "gt_node_id": node_id,
        "root": find(
            (ds, node_id)
        ),
    }
    for ds, node_id
    in sorted(missing_nodes)
])


roots = sorted(
    set(membership["root"])
)

root_to_id = {
    root: i
    for i, root in enumerate(roots)
}

membership["component_id"] = (
    membership["root"]
    .map(root_to_id)
    .astype(int)
)


node_to_component = {
    (
        r.dataset,
        int(r.gt_node_id),
    ): int(r.component_id)

    for r in membership.itertuples(
        index=False
    )
}


# ------------------------------------------------------------
# 5. Assign every detection FN to a component
# ------------------------------------------------------------

edge_rows = []


for r in det.itertuples(index=False):

    sm = int(r.pred_source_id) < 0
    tm = int(r.pred_target_id) < 0

    if sm:

        cid = node_to_component[
            (
                r.dataset,
                int(r.gt_source_id),
            )
        ]

    else:

        cid = node_to_component[
            (
                r.dataset,
                int(r.gt_target_id),
            )
        ]


    if sm and tm:

        assert cid == node_to_component[
            (
                r.dataset,
                int(r.gt_target_id),
            )
        ]

        edge_type = "INTERNAL"

    elif sm:

        edge_type = "OUT_BOUNDARY"

    else:

        edge_type = "IN_BOUNDARY"


    edge_rows.append({
        "dataset":
            r.dataset,

        "component_id":
            cid,

        "edge_type":
            edge_type,
    })


edges = pd.DataFrame(edge_rows)

assert len(edges) == 1550


# ------------------------------------------------------------
# 6. Component table
# ------------------------------------------------------------

node_counts = (
    membership
    .groupby(
        ["dataset", "component_id"]
    )
    .size()
    .rename("missing_nodes")
)


edge_counts = (
    edges
    .groupby(
        [
            "dataset",
            "component_id",
            "edge_type",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)


components = (
    node_counts
    .to_frame()
    .join(
        edge_counts,
        how="left",
    )
    .fillna(0)
    .reset_index()
)


for col in [
    "INTERNAL",
    "IN_BOUNDARY",
    "OUT_BOUNDARY",
]:

    if col not in components:
        components[col] = 0

    components[col] = (
        components[col]
        .astype(int)
    )


components["boundary_edges"] = (
    components["IN_BOUNDARY"]
    +
    components["OUT_BOUNDARY"]
)

components["detection_fn_edges"] = (
    components["INTERNAL"]
    +
    components["boundary_edges"]
)


prefix_map = (
    det[
        ["dataset", "prefix"]
    ]
    .drop_duplicates()
    .set_index("dataset")["prefix"]
    .to_dict()
)

components["prefix"] = (
    components["dataset"]
    .map(prefix_map)
)


assert int(
    components["missing_nodes"].sum()
) == len(missing_nodes)

assert int(
    components["INTERNAL"].sum()
) == 689

assert int(
    components["detection_fn_edges"].sum()
) == 1550


# ------------------------------------------------------------
# 7. Single vs multi-node ceiling
# ------------------------------------------------------------

single = components[
    components["missing_nodes"].eq(1)
]

multi = components[
    components["missing_nodes"].ge(2)
]

long_gap = components[
    components["missing_nodes"].ge(3)
]


print(
    "========== CURRENT_BEST GAP COMPONENTS =========="
)

print(
    "components:",
    len(components)
)

print(
    "missing nodes:",
    len(missing_nodes)
)

print(
    "internal missing-missing FN:",
    int(
        components["INTERNAL"].sum()
    )
)

print(
    "boundary FN:",
    int(
        components[
            "boundary_edges"
        ].sum()
    )
)

print(
    "total detection FN:",
    int(
        components[
            "detection_fn_edges"
        ].sum()
    )
)


print(
    "\n========== GAP SIZE DISTRIBUTION =========="
)

print(
    components[
        "missing_nodes"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\n========== SINGLE VS MULTI-NODE CEILING =========="
)

print(
    "single-node components:",
    len(single)
)

print(
    "single-node detection FNs:",
    int(
        single[
            "detection_fn_edges"
        ].sum()
    )
)

print(
    "multi-node components:",
    len(multi)
)

print(
    "multi-node missing nodes:",
    int(
        multi[
            "missing_nodes"
        ].sum()
    )
)

print(
    "multi-node detection FNs:",
    int(
        multi[
            "detection_fn_edges"
        ].sum()
    ),
    f"({multi['detection_fn_edges'].sum()/1550:.3%})"
)

print(
    ">=3-node components:",
    len(long_gap)
)

print(
    ">=3-node detection FNs:",
    int(
        long_gap[
            "detection_fn_edges"
        ].sum()
    )
)


print(
    "\n========== COMPONENTS BY PREFIX =========="
)

print(
    components.groupby(
        "prefix"
    )
    .agg(
        components=(
            "component_id",
            "size",
        ),
        missing_nodes=(
            "missing_nodes",
            "sum",
        ),
        detection_fn_edges=(
            "detection_fn_edges",
            "sum",
        ),
        internal_edges=(
            "INTERNAL",
            "sum",
        ),
        median_gap_size=(
            "missing_nodes",
            "median",
        ),
        max_gap_size=(
            "missing_nodes",
            "max",
        ),
    )
    .to_string()
)


print(
    "\n========== HIGH-YIELD COMPONENT CEILING =========="
)

for n in [
    2,
    3,
    4,
    5,
]:

    x = components[
        components[
            "detection_fn_edges"
        ].ge(n)
    ]

    print(
        f">= {n} detection FNs: "
        f"{len(x)} components | "
        f"{int(x['detection_fn_edges'].sum())} FNs"
    )


# ------------------------------------------------------------
# 8. Persist
# ------------------------------------------------------------

components.to_pickle(OUT)


payload = {
    "stage":
        "12.11B",

    "current_best_detection_fn":
        1550,

    "components":
        int(len(components)),

    "missing_nodes":
        int(len(missing_nodes)),

    "internal_missing_edges":
        int(
            components[
                "INTERNAL"
            ].sum()
        ),

    "boundary_missing_edges":
        int(
            components[
                "boundary_edges"
            ].sum()
        ),

    "single_node_components":
        int(len(single)),

    "multi_node_components":
        int(len(multi)),

    "multi_node_detection_fn":
        int(
            multi[
                "detection_fn_edges"
            ].sum()
        ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    OUT
)

print(
    "written:",
    SUMMARY_OUT
)


print(
    "\n========== 12.11B DECISION =========="
)

print(
    "CURRENT_BEST_DETECTION_GAP_COMPONENTS_LOCKED: PASS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11D Two-Sided Gap Interpolation Ceiling
# Purpose:
#   Test whether CURRENT_BEST multi-node detection gaps can
#   theoretically be reconstructed from their two observed
#   temporal boundaries by simple linear motion interpolation.
#
#   This is an intrinsic-motion ceiling:
#       GT boundary positions -> interpolate -> missing GT nodes
#
#   It answers whether interpolation is geometrically viable
#   BEFORE introducing prediction-boundary noise or designing
#   a deployment policy.
#
# Changes files: YES — interpolation ceiling artifacts
# Runs training/inference: NO
# Uses GT: YES — research ceiling only
# Defines/tunes deployment policy: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

FN_PATH = (
    S12
    / "stage12_current_best_edge_fn_residuals.pkl"
)

OUT = (
    S12
    / "stage12_two_sided_gap_interpolation_ceiling.pkl"
)

SUMMARY_OUT = (
    S12
    / "stage12_two_sided_gap_interpolation_ceiling_summary.json"
)


sys.path.insert(
    0,
    str(PROJECT / "src"),
)

from biohub_cell_tracking import frozen_v9 as fv9


# Physical voxel scale of original image coordinates:
# z = 1.625 um, y/x = 0.40625 um.
SCALE = np.asarray(
    [1.625, 0.40625, 0.40625],
    dtype=np.float64,
)


# ============================================================
# 1. CURRENT_BEST 6bba detection residuals
# ============================================================

fn = pd.read_pickle(
    FN_PATH
)

assert len(fn) == 3425


det = (
    fn[
        fn["root_cause"].eq(
            "DETECTION_LIMITED"
        )
        &
        fn["prefix"].eq(
            "6bba"
        )
    ]
    .copy()
)


# Locked CURRENT_BEST census.
assert len(det) == 1303


missing_keys = set()


for r in det.itertuples(
    index=False
):

    if int(r.pred_source_id) < 0:

        missing_keys.add(
            (
                r.dataset,
                int(r.gt_source_id),
            )
        )


    if int(r.pred_target_id) < 0:

        missing_keys.add(
            (
                r.dataset,
                int(r.gt_target_id),
            )
        )


# Locked Stage12.11B anatomy.
assert len(missing_keys) == 1021


print(
    "========== CURRENT_BEST DETECTION POPULATION =========="
)

print(
    "6bba detection FN:",
    len(det)
)

print(
    "6bba unique missing GT nodes:",
    len(missing_keys)
)


# ============================================================
# 2. Reconstruct missing-node components
# ============================================================

parent = {
    key: key
    for key in missing_keys
}


def find(x):

    while parent[x] != x:

        parent[x] = parent[
            parent[x]
        ]

        x = parent[x]

    return x


def union(a, b):

    ra = find(a)
    rb = find(b)

    if ra != rb:
        parent[rb] = ra


both = det[
    det["pred_source_id"].lt(0)
    &
    det["pred_target_id"].lt(0)
]


# Locked 6bba internal missing->missing edges.
assert len(both) == 614


for r in both.itertuples(
    index=False
):

    union(
        (
            r.dataset,
            int(r.gt_source_id),
        ),
        (
            r.dataset,
            int(r.gt_target_id),
        ),
    )


# Freeze canonical roots after union-find.
root_map = {
    key: find(key)
    for key in missing_keys
}


# ============================================================
# 3. Exact GT node coordinates
# ============================================================

datasets = sorted(
    det["dataset"].unique()
)


gt_nodes = {}


for name in datasets:

    gt = fv9.load_graph(
        TRAIN_ROOT
        / f"{name}.geff"
    )


    nodes = (
        gt.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
    )


    nodes["node_id"] = (
        nodes["node_id"]
        .astype(int)
    )

    nodes["t"] = (
        nodes["t"]
        .astype(int)
    )


    assert not nodes[
        "node_id"
    ].duplicated().any()


    gt_nodes[name] = (
        nodes.set_index(
            "node_id"
        )
    )


print(
    "GT datasets loaded:",
    len(gt_nodes)
)


# ============================================================
# 4. Exact component membership
# ============================================================

membership_rows = []


for dataset, node_id in sorted(
    missing_keys
):

    node = (
        gt_nodes[
            dataset
        ]
        .loc[
            node_id
        ]
    )


    root = root_map[
        (
            dataset,
            node_id,
        )
    ]


    membership_rows.append({
        "dataset":
            dataset,

        # Tuple-valued canonical root.
        "component_root":
            root,

        "gt_node_id":
            int(node_id),

        "t":
            int(node["t"]),

        "z":
            float(node["z"]),

        "y":
            float(node["y"]),

        "x":
            float(node["x"]),
    })


membership = pd.DataFrame(
    membership_rows
)


assert len(membership) == 1021


component_sizes = (
    membership.groupby(
        [
            "dataset",
            "component_root",
        ],
        sort=False,
    )
    .size()
)


assert len(component_sizes) == 407


multi_component_sizes = (
    component_sizes[
        component_sizes.ge(2)
    ]
)


# Locked Stage12.11C-P1 anatomy.
assert len(multi_component_sizes) == 183

assert int(
    multi_component_sizes.sum()
) == 797


print(
    "6bba components:",
    len(component_sizes)
)

print(
    "multi-node components:",
    len(multi_component_sizes)
)

print(
    "multi-node missing nodes:",
    int(
        multi_component_sizes.sum()
    )
)


# ============================================================
# 5. Locate observed GT temporal boundaries
# ============================================================

boundary_rows = []


for r in det.itertuples(
    index=False
):

    sm = (
        int(
            r.pred_source_id
        )
        < 0
    )

    tm = (
        int(
            r.pred_target_id
        )
        < 0
    )


    # --------------------------------------------------------
    # Observed source -> missing target
    #
    # GT source is the observed boundary immediately before
    # this missing component.
    # --------------------------------------------------------

    if (
        not sm
        and tm
    ):

        root = root_map[
            (
                r.dataset,
                int(r.gt_target_id),
            )
        ]


        boundary_rows.append({
            "dataset":
                r.dataset,

            "component_root":
                root,

            "boundary_type":
                "IN",

            "gt_boundary_node_id":
                int(
                    r.gt_source_id
                ),
        })


    # --------------------------------------------------------
    # Missing source -> observed target
    #
    # GT target is the observed boundary immediately after
    # this missing component.
    # --------------------------------------------------------

    if (
        sm
        and not tm
    ):

        root = root_map[
            (
                r.dataset,
                int(r.gt_source_id),
            )
        ]


        boundary_rows.append({
            "dataset":
                r.dataset,

            "component_root":
                root,

            "boundary_type":
                "OUT",

            "gt_boundary_node_id":
                int(
                    r.gt_target_id
                ),
        })


boundaries = pd.DataFrame(
    boundary_rows
)


assert len(boundaries) > 0


print(
    "boundary records:",
    len(boundaries)
)


# ============================================================
# 6. Restrict to clean two-sided ordinary chains
# ============================================================

records = []

component_records = []


for (
    dataset,
    root
), mm in membership.groupby(
    [
        "dataset",
        "component_root",
    ],
    sort=False,
):


    # --------------------------------------------------------
    # Only multi-node gaps.
    # --------------------------------------------------------

    if len(mm) < 2:
        continue


    # --------------------------------------------------------
    # IMPORTANT PATCH:
    #
    # component_root is a Python tuple.
    #
    # Pandas Series.eq(root) treats tuple as array-like and
    # raises:
    #
    #     ValueError: Lengths must be equal
    #
    # Therefore comparison must be scalar row-by-row.
    # --------------------------------------------------------

    root_mask = (
        boundaries[
            "component_root"
        ]
        .map(
            lambda x:
                x == root
        )
    )


    bb = boundaries[
        boundaries[
            "dataset"
        ].eq(
            dataset
        )
        &
        root_mask
    ].copy()


    ins = (
        bb[
            bb[
                "boundary_type"
            ].eq(
                "IN"
            )
        ][
            "gt_boundary_node_id"
        ]
        .astype(int)
        .unique()
    )


    outs = (
        bb[
            bb[
                "boundary_type"
            ].eq(
                "OUT"
            )
        ][
            "gt_boundary_node_id"
        ]
        .astype(int)
        .unique()
    )


    # --------------------------------------------------------
    # Clean two-sided chain only:
    #
    # exactly one observed boundary before the gap
    # exactly one observed boundary after the gap
    # --------------------------------------------------------

    if (
        len(ins) != 1
        or len(outs) != 1
    ):
        continue


    in_id = int(
        ins[0]
    )

    out_id = int(
        outs[0]
    )


    nodes = gt_nodes[
        dataset
    ]


    assert in_id in nodes.index
    assert out_id in nodes.index


    a = nodes.loc[
        in_id
    ]

    b = nodes.loc[
        out_id
    ]


    ta = int(
        a["t"]
    )

    tb = int(
        b["t"]
    )


    # Must have at least one intermediate frame.
    if tb <= ta + 1:
        continue


    mm = (
        mm.sort_values(
            "t"
        )
        .reset_index(
            drop=True
        )
    )


    times = (
        mm[
            "t"
        ]
        .astype(int)
        .to_numpy()
    )


    # --------------------------------------------------------
    # Missing nodes must all lie strictly between boundaries.
    # --------------------------------------------------------

    if not (
        (times > ta).all()
        and
        (times < tb).all()
    ):
        continue


    # --------------------------------------------------------
    # Clean ordinary continuation:
    #
    # exactly one missing node at EVERY intermediate frame.
    #
    # This deliberately excludes branching/division topology.
    # --------------------------------------------------------

    expected_times = np.arange(
        ta + 1,
        tb,
        dtype=int,
    )


    if not np.array_equal(
        times,
        expected_times,
    ):
        continue


    gap_size = int(
        len(mm)
    )


    assert (
        gap_size
        ==
        tb - ta - 1
    )


    # --------------------------------------------------------
    # Boundary coordinates
    # --------------------------------------------------------

    pa = np.asarray(
        [
            float(
                a["z"]
            ),
            float(
                a["y"]
            ),
            float(
                a["x"]
            ),
        ],
        dtype=np.float64,
    )


    pb = np.asarray(
        [
            float(
                b["z"]
            ),
            float(
                b["y"]
            ),
            float(
                b["x"]
            ),
        ],
        dtype=np.float64,
    )


    component_errors = []


    # --------------------------------------------------------
    # Linear temporal interpolation
    # --------------------------------------------------------

    for r in mm.itertuples(
        index=False
    ):

        alpha = (
            (
                int(r.t)
                -
                ta
            )
            /
            (
                tb
                -
                ta
            )
        )


        assert (
            0.0
            <
            alpha
            <
            1.0
        )


        interp = (
            (1.0 - alpha)
            * pa
            +
            alpha
            * pb
        )


        truth = np.asarray(
            [
                float(r.z),
                float(r.y),
                float(r.x),
            ],
            dtype=np.float64,
        )


        error_um = float(
            np.linalg.norm(
                (
                    interp
                    -
                    truth
                )
                *
                SCALE
            )
        )


        component_errors.append(
            error_um
        )


        records.append({
            "dataset":
                dataset,

            "component_root":
                root,

            "gap_size":
                gap_size,

            "boundary_in_gt_node_id":
                in_id,

            "boundary_out_gt_node_id":
                out_id,

            "boundary_t0":
                ta,

            "boundary_t1":
                tb,

            "gt_node_id":
                int(
                    r.gt_node_id
                ),

            "t":
                int(
                    r.t
                ),

            "alpha":
                float(
                    alpha
                ),

            "truth_z":
                float(
                    r.z
                ),

            "truth_y":
                float(
                    r.y
                ),

            "truth_x":
                float(
                    r.x
                ),

            "interp_z":
                float(
                    interp[0]
                ),

            "interp_y":
                float(
                    interp[1]
                ),

            "interp_x":
                float(
                    interp[2]
                ),

            "error_um":
                error_um,
        })


    component_records.append({
        "dataset":
            dataset,

        "component_root":
            root,

        "gap_size":
            gap_size,

        "boundary_in_gt_node_id":
            in_id,

        "boundary_out_gt_node_id":
            out_id,

        "boundary_t0":
            ta,

        "boundary_t1":
            tb,

        "missing_nodes":
            gap_size,

        "mean_error_um":
            float(
                np.mean(
                    component_errors
                )
            ),

        "median_error_um":
            float(
                np.median(
                    component_errors
                )
            ),

        "max_error_um":
            float(
                np.max(
                    component_errors
                )
            ),
    })


audit = pd.DataFrame(
    records
)

components = pd.DataFrame(
    component_records
)


assert len(audit) > 0
assert len(components) > 0


assert int(
    components[
        "missing_nodes"
    ].sum()
) == len(audit)


assert (
    components[
        "gap_size"
    ]
    ==
    components[
        "missing_nodes"
    ]
).all()


# Every selected component must be one of the
# 183 multi-node components.
assert len(components) <= 183
assert len(audit) <= 797


# ============================================================
# 7. Report — clean two-sided population
# ============================================================

print(
    "\n========== CLEAN TWO-SIDED MULTI-NODE GAPS =========="
)

print(
    "components:",
    len(components)
)

print(
    "missing nodes:",
    len(audit)
)

print(
    "fraction of all 6bba multi-node components:",
    f"{len(components)/183:.3%}"
)

print(
    "fraction of all 6bba multi-node missing nodes:",
    f"{len(audit)/797:.3%}"
)


# ============================================================
# 8. Node-level interpolation error
# ============================================================

print(
    "\n========== INTERPOLATION ERROR QUANTILES =========="
)

print(
    audit[
        "error_um"
    ]
    .quantile(
        [
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_string()
)


print(
    "\n========== NODE RECOVERABILITY BY ERROR =========="
)


for radius in [
    2.0,
    3.0,
    5.0,
    7.0,
]:

    n = int(
        audit[
            "error_um"
        ]
        .le(
            radius
        )
        .sum()
    )


    print(
        f"error <= {radius:.1f} um:",
        n,
        "/",
        len(audit),
        f"({n/len(audit):.3%})",
    )


# ============================================================
# 9. Component-level interpolation error
# ============================================================

print(
    "\n========== FULL-COMPONENT RECOVERABILITY =========="
)


for radius in [
    2.0,
    3.0,
    5.0,
    7.0,
]:

    n = int(
        components[
            "max_error_um"
        ]
        .le(
            radius
        )
        .sum()
    )


    print(
        f"all nodes <= {radius:.1f} um:",
        n,
        "/",
        len(components),
        f"({n/len(components):.3%})",
    )


# ============================================================
# 10. Anatomy by gap size
# ============================================================

print(
    "\n========== INTERPOLATION BY GAP SIZE =========="
)


table = (
    components.groupby(
        "gap_size"
    )
    .agg(
        components=(
            "component_root",
            "size",
        ),

        missing_nodes=(
            "missing_nodes",
            "sum",
        ),

        median_mean_error_um=(
            "mean_error_um",
            "median",
        ),

        median_max_error_um=(
            "max_error_um",
            "median",
        ),

        full_2um=(
            "max_error_um",
            lambda s:
                int(
                    (
                        s <= 2.0
                    ).sum()
                ),
        ),

        full_3um=(
            "max_error_um",
            lambda s:
                int(
                    (
                        s <= 3.0
                    ).sum()
                ),
        ),

        full_5um=(
            "max_error_um",
            lambda s:
                int(
                    (
                        s <= 5.0
                    ).sum()
                ),
        ),

        full_7um=(
            "max_error_um",
            lambda s:
                int(
                    (
                        s <= 7.0
                    ).sum()
                ),
        ),
    )
)


print(
    table.to_string(
        float_format=lambda x:
            f"{x:.3f}"
    )
)


# ============================================================
# 11. Persist
# ============================================================

audit.to_pickle(
    OUT
)


payload = {
    "stage":
        "12.11D",

    "status":
        "PASS",

    "audit_type":
        "GT_INTRINSIC_MOTION_CEILING",

    "deployment_policy_defined":
        False,

    "dataset_family":
        "6bba",

    "current_best_detection_fn_6bba":
        1303,

    "missing_gt_nodes_6bba":
        1021,

    "all_multi_node_components":
        183,

    "all_multi_node_missing_nodes":
        797,

    "clean_two_sided_components":
        int(
            len(components)
        ),

    "clean_two_sided_missing_nodes":
        int(
            len(audit)
        ),

    "clean_two_sided_component_fraction":
        float(
            len(components)
            /
            183
        ),

    "clean_two_sided_node_fraction":
        float(
            len(audit)
            /
            797
        ),

    "median_interpolation_error_um":
        float(
            audit[
                "error_um"
            ].median()
        ),

    "nodes_error_le_2um":
        int(
            audit[
                "error_um"
            ]
            .le(
                2.0
            )
            .sum()
        ),

    "nodes_error_le_3um":
        int(
            audit[
                "error_um"
            ]
            .le(
                3.0
            )
            .sum()
        ),

    "nodes_error_le_5um":
        int(
            audit[
                "error_um"
            ]
            .le(
                5.0
            )
            .sum()
        ),

    "nodes_error_le_7um":
        int(
            audit[
                "error_um"
            ]
            .le(
                7.0
            )
            .sum()
        ),

    "components_full_le_2um":
        int(
            components[
                "max_error_um"
            ]
            .le(
                2.0
            )
            .sum()
        ),

    "components_full_le_3um":
        int(
            components[
                "max_error_um"
            ]
            .le(
                3.0
            )
            .sum()
        ),

    "components_full_le_5um":
        int(
            components[
                "max_error_um"
            ]
            .le(
                5.0
            )
            .sum()
        ),

    "components_full_le_7um":
        int(
            components[
                "max_error_um"
            ]
            .le(
                7.0
            )
            .sum()
        ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    OUT
)

print(
    "written:",
    SUMMARY_OUT
)


print(
    "\n========== 12.11D DECISION =========="
)

print(
    "TWO_SIDED_INTERPOLATION_CEILING_AUDITED: PASS"
)

In [ ]:
# ============================================================
# RUN — Stage 12.11F
# Predicted-Path Suppressed-Peak Support Audit
#
# LOCKED INPUT POPULATION:
#   116 clean-gap components
#   406 missing GT nodes
#   19 datasets
#
# PURPOSE:
#   Test whether Stage 12.10C suppressed detector peaks provide
#   GT-blind image-supported candidates near Stage 12.11E
#   predicted-boundary interpolation paths.
#
# CANDIDATE SELECTION:
#   same dataset
#   same frame
#   nearest suppressed peak to predicted interpolation coordinate
#
# IMPORTANT:
#   GT is used ONLY AFTER candidate selection for evaluation.
#
# FIXED RADII:
#   2 / 3 / 5 / 7 um
#
# NO:
#   detector rerun
#   threshold tuning
#   radius tuning
#   graph modification
#   model training
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

RPT = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


NODE_IN = (
    RPT
    / "stage12_predicted_boundary_interpolation_audit.pkl"
)

COMP_IN = (
    RPT
    / "stage12_predicted_boundary_interpolation_components.pkl"
)

PEAK_IN = (
    RPT
    / "stage12_folds14_suppressed_peak_universe.pkl"
)


NODE_OUT = (
    RPT
    / "stage12_predicted_path_suppressed_peak_support_audit.pkl"
)

COMP_OUT = (
    RPT
    / "stage12_predicted_path_suppressed_peak_support_components.pkl"
)

SUMMARY_OUT = (
    RPT
    / "stage12_predicted_path_suppressed_peak_support_summary.json"
)


for p in [
    NODE_IN,
    COMP_IN,
    PEAK_IN,
]:
    assert p.exists(), (
        f"Missing required artifact:\n{p}"
    )


# ============================================================
# 1. Load locked inputs
# ============================================================

nodes = pd.read_pickle(
    NODE_IN
).copy()

components_prev = pd.read_pickle(
    COMP_IN
).copy()

peaks = pd.read_pickle(
    PEAK_IN
).copy()


print(
    "========== 12.11F INPUTS =========="
)

print(
    "12.11E missing nodes :",
    len(nodes),
)

print(
    "12.11E components    :",
    len(components_prev),
)

print(
    "datasets             :",
    nodes["dataset"].nunique(),
)

print(
    "12.10C suppressed peaks:",
    len(peaks),
)


# ============================================================
# 2. Lock actual 12.11E schema
# ============================================================

GT_COLS = (
    "truth_z",
    "truth_y",
    "truth_x",
)

INTRINSIC_COLS = (
    "interp_z",
    "interp_y",
    "interp_x",
)

PRED_COLS = (
    "pred_interp_z",
    "pred_interp_y",
    "pred_interp_x",
)

COMPONENT_COL = "component_root"

INTRINSIC_ERROR_COL = "error_um"

PRED_ERROR_COL = "pred_interp_error_um"


required_node_cols = {
    "dataset",
    "component_root",
    "gap_size",
    "gt_node_id",
    "t",

    "truth_z",
    "truth_y",
    "truth_x",

    "interp_z",
    "interp_y",
    "interp_x",

    "error_um",

    "pred_interp_z",
    "pred_interp_y",
    "pred_interp_x",

    "pred_interp_error_um",
}


missing_node_cols = (
    required_node_cols
    - set(nodes.columns)
)

assert not missing_node_cols, (
    "12.11E schema mismatch. Missing:\n"
    f"{sorted(missing_node_cols)}"
)


required_peak_cols = {
    "dataset",
    "t",
    "z",
    "y",
    "x",
    "detector_prob",
}


missing_peak_cols = (
    required_peak_cols
    - set(peaks.columns)
)

assert not missing_peak_cols, (
    "12.10C suppressed-peak schema mismatch. Missing:\n"
    f"{sorted(missing_peak_cols)}"
)


print(
    "\n========== LOCKED SCHEMA =========="
)

print(
    "GT:",
    GT_COLS,
)

print(
    "intrinsic interpolation:",
    INTRINSIC_COLS,
)

print(
    "predicted interpolation:",
    PRED_COLS,
)

print(
    "component:",
    COMPONENT_COL,
)


# ============================================================
# 3. Locked population fidelity
# ============================================================

assert len(nodes) == 406, (
    f"Expected 406 missing nodes; got {len(nodes)}"
)

assert len(components_prev) == 116, (
    f"Expected 116 components; got {len(components_prev)}"
)

assert nodes["dataset"].nunique() == 19, (
    "Expected 19 datasets"
)


n_components = (
    nodes[
        [
            "dataset",
            COMPONENT_COL,
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


print(
    "\n========== LOCKED CLEAN-GAP POPULATION =========="
)

print(
    "components:",
    n_components,
)

print(
    "missing nodes:",
    len(nodes),
)

print(
    "datasets:",
    nodes["dataset"].nunique(),
)


assert n_components == 116


# Suppressed universe sanity.
assert peaks["detector_prob"].gt(
    0.90
).all(), (
    "Suppressed peak universe contains detector_prob <= 0.90"
)

assert not peaks[
    [
        "dataset",
        "t",
        "z",
        "y",
        "x",
    ]
].duplicated().any(), (
    "Duplicate suppressed peak coordinates detected"
)


# ============================================================
# 4. Infer physical coordinate scale directly from 12.11E
#
# We know:
#
#   error_um^2
#     =
#       (dz * sz)^2
#     + (dy * sy)^2
#     + (dx * sx)^2
#
# Therefore:
#
#   error_um^2
#     =
#       dz^2 * sz^2
#     + dy^2 * sy^2
#     + dx^2 * sx^2
#
# This is linear in:
#
#   beta = [sz^2, sy^2, sx^2]
#
# We use BOTH:
#   intrinsic interpolation errors
#   predicted-boundary interpolation errors
#
# No scale is guessed.
# ============================================================

def build_scale_equations(
    df,
):

    truth = df[
        list(GT_COLS)
    ].to_numpy(
        dtype=float
    )

    intrinsic = df[
        list(INTRINSIC_COLS)
    ].to_numpy(
        dtype=float
    )

    predicted = df[
        list(PRED_COLS)
    ].to_numpy(
        dtype=float
    )

    intrinsic_err = df[
        INTRINSIC_ERROR_COL
    ].to_numpy(
        dtype=float
    )

    predicted_err = df[
        PRED_ERROR_COL
    ].to_numpy(
        dtype=float
    )


    delta_intrinsic_sq = (
        intrinsic
        - truth
    ) ** 2

    delta_predicted_sq = (
        predicted
        - truth
    ) ** 2


    X = np.vstack(
        [
            delta_intrinsic_sq,
            delta_predicted_sq,
        ]
    )

    y = np.concatenate(
        [
            intrinsic_err ** 2,
            predicted_err ** 2,
        ]
    )


    finite = (
        np.isfinite(X).all(
            axis=1
        )
        & np.isfinite(y)
    )

    return (
        X[finite],
        y[finite],
    )


def infer_scale(
    df,
):

    X, y = build_scale_equations(
        df
    )

    rank = np.linalg.matrix_rank(
        X
    )

    if rank < 3:
        return None

    beta, _, _, _ = np.linalg.lstsq(
        X,
        y,
        rcond=None,
    )

    if (
        not np.isfinite(beta).all()
        or (beta <= 0).any()
    ):
        return None

    scale = np.sqrt(
        beta
    )

    return scale


# Global fallback candidate.
global_scale = infer_scale(
    nodes
)

assert global_scale is not None, (
    "Could not infer even a global physical scale "
    "from the locked 12.11E errors."
)


def physical_error_from_scale(
    df,
    coord_cols,
    scale,
):

    truth = df[
        list(GT_COLS)
    ].to_numpy(
        dtype=float
    )

    coord = df[
        list(coord_cols)
    ].to_numpy(
        dtype=float
    )

    return np.linalg.norm(
        (
            coord
            - truth
        )
        * np.asarray(
            scale,
            dtype=float,
        ).reshape(
            1,
            3,
        ),
        axis=1,
    )


def validate_scale(
    df,
    scale,
):

    pred_recomputed = (
        physical_error_from_scale(
            df,
            PRED_COLS,
            scale,
        )
    )

    intr_recomputed = (
        physical_error_from_scale(
            df,
            INTRINSIC_COLS,
            scale,
        )
    )


    pred_stored = df[
        PRED_ERROR_COL
    ].to_numpy(
        dtype=float
    )

    intr_stored = df[
        INTRINSIC_ERROR_COL
    ].to_numpy(
        dtype=float
    )


    pred_abs = np.abs(
        pred_recomputed
        - pred_stored
    )

    intr_abs = np.abs(
        intr_recomputed
        - intr_stored
    )


    return {
        "pred_median_abs":
            float(
                np.median(
                    pred_abs
                )
            ),

        "pred_max_abs":
            float(
                np.max(
                    pred_abs
                )
            ),

        "intr_median_abs":
            float(
                np.median(
                    intr_abs
                )
            ),

        "intr_max_abs":
            float(
                np.max(
                    intr_abs
                )
            ),
    }


# Dataset-specific scales where identifiable.
scale_map = {}

scale_validation = {}


for dataset, g in nodes.groupby(
    "dataset",
    sort=False,
):

    local_scale = infer_scale(
        g
    )

    if local_scale is not None:

        stats = validate_scale(
            g,
            local_scale,
        )

        # Prefer local scale if it faithfully reproduces
        # persisted Stage 12.11E micron errors.
        if (
            stats[
                "pred_max_abs"
            ] <= 0.01
            and stats[
                "intr_max_abs"
            ] <= 0.01
        ):
            scale_map[
                dataset
            ] = local_scale

            scale_validation[
                dataset
            ] = {
                "source":
                    "dataset_specific",
                **stats,
            }

            continue


    # Otherwise use global scale ONLY if it faithfully
    # reproduces this dataset's persisted errors.
    stats = validate_scale(
        g,
        global_scale,
    )

    if (
        stats[
            "pred_max_abs"
        ] > 0.05
        or stats[
            "intr_max_abs"
        ] > 0.05
    ):
        raise RuntimeError(
            "\nCould not infer a trustworthy physical scale "
            f"for dataset {dataset}.\n"
            f"global scale={global_scale}\n"
            f"validation={stats}"
        )


    scale_map[
        dataset
    ] = global_scale.copy()

    scale_validation[
        dataset
    ] = {
        "source":
            "global_fallback",
        **stats,
    }


print(
    "\n========== INFERRED PHYSICAL SCALE =========="
)

for dataset in sorted(
    scale_map
):

    s = scale_map[
        dataset
    ]

    src = scale_validation[
        dataset
    ][
        "source"
    ]

    print(
        f"{dataset}: "
        f"z={s[0]:.8f}, "
        f"y={s[1]:.8f}, "
        f"x={s[2]:.8f} um/unit "
        f"[{src}]"
    )


# ============================================================
# 5. Reconstruct 12.11E errors and HARD validate
# ============================================================

scales_per_node = np.vstack(
    [
        scale_map[
            dataset
        ]
        for dataset
        in nodes[
            "dataset"
        ]
    ]
)


truth_raw = nodes[
    list(GT_COLS)
].to_numpy(
    dtype=float
)

intrinsic_raw = nodes[
    list(INTRINSIC_COLS)
].to_numpy(
    dtype=float
)

pred_raw = nodes[
    list(PRED_COLS)
].to_numpy(
    dtype=float
)


intrinsic_recomputed = np.linalg.norm(
    (
        intrinsic_raw
        - truth_raw
    )
    * scales_per_node,
    axis=1,
)

pred_recomputed = np.linalg.norm(
    (
        pred_raw
        - truth_raw
    )
    * scales_per_node,
    axis=1,
)


intrinsic_stored = nodes[
    INTRINSIC_ERROR_COL
].to_numpy(
    dtype=float
)

pred_stored = nodes[
    PRED_ERROR_COL
].to_numpy(
    dtype=float
)


print(
    "\n========== 12.11E FIDELITY RECHECK =========="
)

print(
    "intrinsic stored median   :",
    f"{np.median(intrinsic_stored):.6f}",
)

print(
    "intrinsic recomputed median:",
    f"{np.median(intrinsic_recomputed):.6f}",
)

print(
    "pred-boundary stored median:",
    f"{np.median(pred_stored):.6f}",
)

print(
    "pred-boundary recomputed   :",
    f"{np.median(pred_recomputed):.6f}",
)


print(
    "max intrinsic abs mismatch:",
    f"{np.max(np.abs(intrinsic_recomputed - intrinsic_stored)):.8f}",
)

print(
    "max predicted abs mismatch:",
    f"{np.max(np.abs(pred_recomputed - pred_stored)):.8f}",
)


assert np.allclose(
    intrinsic_recomputed,
    intrinsic_stored,
    atol=0.05,
    rtol=1e-4,
), (
    "Physical scale does not reproduce intrinsic Stage 12.11E errors"
)

assert np.allclose(
    pred_recomputed,
    pred_stored,
    atol=0.05,
    rtol=1e-4,
), (
    "Physical scale does not reproduce predicted Stage 12.11E errors"
)


# Known Stage 12.11E distribution.
assert abs(
    float(
        np.median(
            intrinsic_stored
        )
    )
    - 1.692700
) < 0.01


assert abs(
    float(
        np.median(
            pred_stored
        )
    )
    - 4.737669
) < 0.01


# Canonical raw error remains the persisted Stage 12.11E result.
nodes[
    "raw_interp_gt_error_um"
] = pred_stored


# Node fidelity counts from 12.11E.
expected_node_counts = {
    2.0: 49,
    3.0: 81,
    5.0: 222,
    7.0: 335,
}


print(
    "\n========== 12.11E NODE FIDELITY =========="
)

for threshold, expected in expected_node_counts.items():

    observed = int(
        (
            nodes[
                "raw_interp_gt_error_um"
            ]
            <= threshold
        ).sum()
    )

    print(
        f"error <= {threshold:.1f} um:",
        observed,
        "/",
        len(nodes),
    )

    assert observed == expected, (
        threshold,
        observed,
        expected,
    )


# ============================================================
# 6. Build physical predicted/GT coordinates
# ============================================================

truth_um = (
    truth_raw
    * scales_per_node
)

pred_interp_um = (
    pred_raw
    * scales_per_node
)


# ============================================================
# 7. Index suppressed peaks by dataset + frame
# ============================================================

peak_groups = {}


for (
    dataset,
    t,
), g in peaks.groupby(
    [
        "dataset",
        "t",
    ],
    sort=False,
):

    peak_groups[
        (
            dataset,
            int(t),
        )
    ] = g.reset_index(
        drop=True
    )


# ============================================================
# 8. GT-blind nearest-peak search
# ============================================================

RADIUS_LIST = [
    2.0,
    3.0,
    5.0,
    7.0,
]


N = len(nodes)


nearest_distance_um = np.full(
    N,
    np.inf,
    dtype=float,
)

nearest_peak_z = np.full(
    N,
    np.nan,
)

nearest_peak_y = np.full(
    N,
    np.nan,
)

nearest_peak_x = np.full(
    N,
    np.nan,
)

nearest_peak_prob = np.full(
    N,
    np.nan,
)

nearest_peak_logit = np.full(
    N,
    np.nan,
)

nearest_peak_gt_error_um = np.full(
    N,
    np.nan,
)


candidate_counts = {
    radius: np.zeros(
        N,
        dtype=np.int64,
    )
    for radius
    in RADIUS_LIST
}


for i, row in enumerate(
    nodes.itertuples(
        index=False
    )
):

    dataset = row.dataset
    t = int(
        row.t
    )

    g = peak_groups.get(
        (
            dataset,
            t,
        )
    )

    if (
        g is None
        or len(g) == 0
    ):
        continue


    scale = scale_map[
        dataset
    ]


    peak_raw = g[
        [
            "z",
            "y",
            "x",
        ]
    ].to_numpy(
        dtype=float
    )

    peak_um = (
        peak_raw
        * scale.reshape(
            1,
            3,
        )
    )


    # ========================================================
    # STRICTLY GT-BLIND SELECTION
    #
    # Only interpolation coordinate is used here.
    # GT coordinate is NOT involved.
    # ========================================================

    d_interp = np.linalg.norm(
        peak_um
        - pred_interp_um[
            i
        ].reshape(
            1,
            3,
        ),
        axis=1,
    )


    j = int(
        np.argmin(
            d_interp
        )
    )


    nearest_distance_um[
        i
    ] = float(
        d_interp[
            j
        ]
    )


    nearest_peak_z[
        i
    ] = float(
        peak_raw[
            j,
            0
        ]
    )

    nearest_peak_y[
        i
    ] = float(
        peak_raw[
            j,
            1
        ]
    )

    nearest_peak_x[
        i
    ] = float(
        peak_raw[
            j,
            2
        ]
    )


    nearest_peak_prob[
        i
    ] = float(
        g.iloc[
            j
        ][
            "detector_prob"
        ]
    )


    if (
        "detector_logit"
        in g.columns
    ):
        nearest_peak_logit[
            i
        ] = float(
            g.iloc[
                j
            ][
                "detector_logit"
            ]
        )


    # ========================================================
    # GT ONLY ENTERS HERE, AFTER CANDIDATE HAS BEEN SELECTED.
    # ========================================================

    nearest_peak_gt_error_um[
        i
    ] = float(
        np.linalg.norm(
            peak_um[
                j
            ]
            - truth_um[
                i
            ]
        )
    )


    for radius in RADIUS_LIST:

        candidate_counts[
            radius
        ][
            i
        ] = int(
            (
                d_interp
                <= radius
            ).sum()
        )


nodes[
    "nearest_suppressed_peak_distance_um"
] = nearest_distance_um

nodes[
    "nearest_peak_z"
] = nearest_peak_z

nodes[
    "nearest_peak_y"
] = nearest_peak_y

nodes[
    "nearest_peak_x"
] = nearest_peak_x

nodes[
    "nearest_peak_detector_prob"
] = nearest_peak_prob

nodes[
    "nearest_peak_detector_logit"
] = nearest_peak_logit

nodes[
    "nearest_peak_gt_error_um"
] = nearest_peak_gt_error_um


nodes[
    "snap_improvement_um"
] = (
    nodes[
        "raw_interp_gt_error_um"
    ]
    - nodes[
        "nearest_peak_gt_error_um"
    ]
)


for radius in RADIUS_LIST:

    tag = int(
        radius
    )

    nodes[
        f"n_suppressed_peaks_within_{tag}um"
    ] = candidate_counts[
        radius
    ]

    nodes[
        f"has_suppressed_peak_within_{tag}um"
    ] = (
        nearest_distance_um
        <= radius
    )


# ============================================================
# 9. Suppressed-peak support
# ============================================================

print(
    "\n========== SUPPRESSED-PEAK SUPPORT =========="
)


support_summary = {}


for radius in RADIUS_LIST:

    mask = (
        nearest_distance_um
        <= radius
    )

    n = int(
        mask.sum()
    )


    support_summary[
        str(radius)
    ] = {
        "nodes":
            n,

        "fraction":
            float(
                n
                / N
            ),
    }


    print(
        f"within {radius:.1f} um:",
        n,
        "/",
        N,
        f"({n/N:.3%})",
    )


# ============================================================
# 10. Candidate ambiguity
# ============================================================

print(
    "\n========== CANDIDATE AMBIGUITY =========="
)


ambiguity_summary = {}


for radius in RADIUS_LIST:

    counts = candidate_counts[
        radius
    ]

    zero = int(
        (
            counts == 0
        ).sum()
    )

    one = int(
        (
            counts == 1
        ).sum()
    )

    multiple = int(
        (
            counts >= 2
        ).sum()
    )


    ambiguity_summary[
        str(radius)
    ] = {
        "zero":
            zero,

        "one":
            one,

        "multiple":
            multiple,
    }


    print(
        f"\nradius <= {radius:.1f} um"
    )

    print(
        "0 peaks :",
        zero,
        f"({zero/N:.3%})",
    )

    print(
        "1 peak  :",
        one,
        f"({one/N:.3%})",
    )

    print(
        "2+ peaks:",
        multiple,
        f"({multiple/N:.3%})",
    )


# ============================================================
# 11. Node-level raw vs snapped — official R=7 support
# ============================================================

R_MAIN = 7.0


supported7_mask = (
    nodes[
        "nearest_suppressed_peak_distance_um"
    ]
    <= R_MAIN
)


supported7 = nodes[
    supported7_mask
].copy()


print(
    "\n========== RAW VS PEAK-SNAPPED — SUPPORT <= 7 UM =========="
)

print(
    "supported nodes:",
    len(
        supported7
    ),
    "/",
    len(nodes),
    f"({len(supported7)/len(nodes):.3%})",
)


if len(
    supported7
) > 0:

    quantiles = [
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]


    comparison = pd.DataFrame(
        {
            "raw_interpolation":
                supported7[
                    "raw_interp_gt_error_um"
                ].quantile(
                    quantiles
                ),

            "peak_snapped":
                supported7[
                    "nearest_peak_gt_error_um"
                ].quantile(
                    quantiles
                ),
        }
    )


    print(
        comparison.to_string()
    )


# ============================================================
# 12. Node recoverability among R7-supported nodes
# ============================================================

print(
    "\n========== NODE RECOVERABILITY — SUPPORT <= 7 UM =========="
)


node_recoverability = {}


for threshold in [
    2.0,
    3.0,
    5.0,
    7.0,
]:

    raw_n = int(
        (
            supported7[
                "raw_interp_gt_error_um"
            ]
            <= threshold
        ).sum()
    )

    snap_n = int(
        (
            supported7[
                "nearest_peak_gt_error_um"
            ]
            <= threshold
        ).sum()
    )


    denom = len(
        supported7
    )


    node_recoverability[
        str(threshold)
    ] = {
        "raw":
            raw_n,

        "snapped":
            snap_n,

        "denominator":
            denom,
    }


    if denom:

        print(
            f"error <= {threshold:.1f} um | "
            f"raw {raw_n}/{denom} "
            f"({raw_n/denom:.3%}) | "
            f"snap {snap_n}/{denom} "
            f"({snap_n/denom:.3%})"
        )

    else:

        print(
            f"error <= {threshold:.1f} um | no R7-supported nodes"
        )


# ============================================================
# 13. Improvement / degradation
# ============================================================

print(
    "\n========== SNAP EFFECT — SUPPORT <= 7 UM =========="
)


if len(
    supported7
) > 0:

    delta = supported7[
        "snap_improvement_um"
    ].dropna()

    EPS = 1e-9


    n_improve = int(
        (
            delta > EPS
        ).sum()
    )

    n_same = int(
        (
            np.abs(
                delta
            )
            <= EPS
        ).sum()
    )

    n_worse = int(
        (
            delta < -EPS
        ).sum()
    )


    print(
        "improves:",
        n_improve,
        "/",
        len(delta),
        f"({n_improve/len(delta):.3%})",
    )

    print(
        "unchanged:",
        n_same,
        "/",
        len(delta),
        f"({n_same/len(delta):.3%})",
    )

    print(
        "worsens :",
        n_worse,
        "/",
        len(delta),
        f"({n_worse/len(delta):.3%})",
    )


    print(
        "\nraw - snap improvement quantiles (um):"
    )

    print(
        delta.quantile(
            [
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
            ]
        ).to_string()
    )

else:

    delta = pd.Series(
        dtype=float
    )

    n_improve = 0
    n_same = 0
    n_worse = 0


# ============================================================
# 14. Official 7-um transition table
#
# Among nodes with a suppressed peak within interpolation R7:
#
# A raw bad  -> snap good
# B raw good -> snap good
# C raw good -> snap bad
# D raw bad  -> snap bad
# ============================================================

print(
    "\n========== 7-UM TRANSITION TABLE — R7 SUPPORTED =========="
)


raw_good = (
    supported7[
        "raw_interp_gt_error_um"
    ]
    <= 7.0
)

snap_good = (
    supported7[
        "nearest_peak_gt_error_um"
    ]
    <= 7.0
)


transition = {
    "A_raw_bad_snap_good":
        int(
            (
                ~raw_good
                & snap_good
            ).sum()
        ),

    "B_raw_good_snap_good":
        int(
            (
                raw_good
                & snap_good
            ).sum()
        ),

    "C_raw_good_snap_bad":
        int(
            (
                raw_good
                & ~snap_good
            ).sum()
        ),

    "D_raw_bad_snap_bad":
        int(
            (
                ~raw_good
                & ~snap_good
            ).sum()
        ),
}


for key, value in transition.items():

    print(
        f"{key:24s}:",
        value,
    )


# ============================================================
# 15. Detector-probability descriptive audit
#
# AUDIT ONLY.
# Do NOT choose a new detector threshold from this cell.
# ============================================================

print(
    "\n========== DETECTOR PROBABILITY AUDIT =========="
)


probability_audit = pd.DataFrame()


if len(
    supported7
) > 0:

    prob_bins = [
        0.90,
        0.95,
        0.97,
        0.98,
        0.99,
        0.995,
        np.inf,
    ]

    prob_labels = [
        "[0.90,0.95)",
        "[0.95,0.97)",
        "[0.97,0.98)",
        "[0.98,0.99)",
        "[0.99,0.995)",
        "[0.995,+inf)",
    ]


    supported7[
        "detector_probability_band"
    ] = pd.cut(
        supported7[
            "nearest_peak_detector_prob"
        ],
        bins=prob_bins,
        labels=prob_labels,
        right=False,
        include_lowest=True,
    )


    probability_audit = (
        supported7
        .groupby(
            "detector_probability_band",
            observed=True,
        )
        .agg(
            n=(
                "nearest_peak_gt_error_um",
                "size",
            ),

            median_gt_error_um=(
                "nearest_peak_gt_error_um",
                "median",
            ),

            mean_gt_error_um=(
                "nearest_peak_gt_error_um",
                "mean",
            ),

            good_7um=(
                "nearest_peak_gt_error_um",
                lambda x:
                    int(
                        (
                            x <= 7.0
                        ).sum()
                    ),
            ),
        )
    )


    probability_audit[
        "p_gt_error_le_7um"
    ] = (
        probability_audit[
            "good_7um"
        ]
        / probability_audit[
            "n"
        ]
    )


    print(
        probability_audit.to_string()
    )

else:

    print(
        "No R7-supported nodes."
    )


# ============================================================
# 16. Component reconstruction
# ============================================================

component_rows = []


for (
    dataset,
    component_root,
), g in nodes.groupby(
    [
        "dataset",
        COMPONENT_COL,
    ],
    sort=False,
):

    gap_sizes = g[
        "gap_size"
    ].dropna().unique()

    assert len(
        gap_sizes
    ) == 1, (
        dataset,
        component_root,
        gap_sizes,
    )


    gap_size = int(
        gap_sizes[
            0
        ]
    )


    assert len(
        g
    ) == gap_size, (
        "Locked clean gap should contain exactly gap_size "
        "missing nodes",
        dataset,
        component_root,
        gap_size,
        len(g),
    )


    row = {
        "dataset":
            dataset,

        "component_root":
            component_root,

        "gap_size":
            gap_size,

        "missing_nodes":
            int(
                len(g)
            ),

        "raw_mean_error_um":
            float(
                g[
                    "raw_interp_gt_error_um"
                ].mean()
            ),

        "raw_max_error_um":
            float(
                g[
                    "raw_interp_gt_error_um"
                ].max()
            ),
    }


    for radius in RADIUS_LIST:

        tag = int(
            radius
        )


        support_col = (
            f"has_suppressed_peak_within_{tag}um"
        )


        all_supported = bool(
            g[
                support_col
            ].all()
        )


        row[
            f"all_nodes_supported_r{tag}"
        ] = all_supported


        if all_supported:

            snap_errors = g[
                "nearest_peak_gt_error_um"
            ].to_numpy(
                dtype=float
            )


            row[
                f"snap_mean_error_r{tag}_um"
            ] = float(
                np.mean(
                    snap_errors
                )
            )


            row[
                f"snap_max_error_r{tag}_um"
            ] = float(
                np.max(
                    snap_errors
                )
            )


            for threshold in [
                2,
                3,
                5,
                7,
            ]:

                row[
                    f"r{tag}_full_{threshold}um"
                ] = bool(
                    np.all(
                        snap_errors
                        <= threshold
                    )
                )


        else:

            row[
                f"snap_mean_error_r{tag}_um"
            ] = np.nan

            row[
                f"snap_max_error_r{tag}_um"
            ] = np.nan


            for threshold in [
                2,
                3,
                5,
                7,
            ]:

                row[
                    f"r{tag}_full_{threshold}um"
                ] = False


    component_rows.append(
        row
    )


component_df = pd.DataFrame(
    component_rows
)


assert len(
    component_df
) == 116


# ============================================================
# 17. HARD component fidelity against 12.11E
# ============================================================

expected_component_counts = {
    2.0: 10,
    3.0: 20,
    5.0: 53,
    7.0: 90,
}


print(
    "\n========== 12.11E COMPONENT FIDELITY =========="
)


for threshold, expected in expected_component_counts.items():

    observed = int(
        (
            component_df[
                "raw_max_error_um"
            ]
            <= threshold
        ).sum()
    )


    print(
        f"all nodes <= {threshold:.1f} um:",
        observed,
        "/",
        116,
        f"({observed/116:.3%})",
    )


    assert observed == expected, (
        "12.11E component fidelity failed",
        threshold,
        observed,
        expected,
    )


# ============================================================
# 18. Full-component suppressed-peak support
# ============================================================

print(
    "\n========== PEAK-SNAPPED FULL-COMPONENT RECOVERABILITY =========="
)


component_support_summary = {}


for radius in RADIUS_LIST:

    tag = int(
        radius
    )


    all_supported_n = int(
        component_df[
            f"all_nodes_supported_r{tag}"
        ].sum()
    )


    print(
        f"\nsearch radius <= {radius:.1f} um"
    )


    print(
        "all missing nodes supported:",
        all_supported_n,
        "/",
        116,
        f"({all_supported_n/116:.3%})",
    )


    radius_summary = {
        "all_nodes_supported":
            all_supported_n,
    }


    for threshold in [
        2,
        3,
        5,
        7,
    ]:

        n = int(
            component_df[
                f"r{tag}_full_{threshold}um"
            ].sum()
        )


        radius_summary[
            f"full_{threshold}um"
        ] = n


        print(
            f"all nodes <= {threshold}.0 um:",
            n,
            "/",
            116,
            f"({n/116:.3%})",
        )


    component_support_summary[
        str(radius)
    ] = radius_summary


# ============================================================
# 19. Direct component comparison at official R7
# ============================================================

print(
    "\n========== RAW VS PEAK-SNAPPED COMPONENTS — SEARCH R7 =========="
)


component_comparison = {}


print(
    f"{'criterion':15s}"
    f"{'raw':>15s}"
    f"{'snap@R7':>15s}"
)


for threshold in [
    2,
    3,
    5,
    7,
]:

    raw_n = int(
        (
            component_df[
                "raw_max_error_um"
            ]
            <= threshold
        ).sum()
    )


    snap_n = int(
        component_df[
            f"r7_full_{threshold}um"
        ].sum()
    )


    component_comparison[
        str(threshold)
    ] = {
        "raw":
            raw_n,

        "snap_r7":
            snap_n,
    }


    print(
        f"full <= {threshold}um"
        f"{raw_n:>8d}/116"
        f"{snap_n:>8d}/116"
    )


# ============================================================
# 20. Gap-size stratification
# ============================================================

gap_summary = (
    component_df
    .groupby(
        "gap_size"
    )
    .agg(
        components=(
            "component_root",
            "size",
        ),

        missing_nodes=(
            "missing_nodes",
            "sum",
        ),

        raw_median_mean_error_um=(
            "raw_mean_error_um",
            "median",
        ),

        raw_median_max_error_um=(
            "raw_max_error_um",
            "median",
        ),

        r7_all_supported=(
            "all_nodes_supported_r7",
            "sum",
        ),

        r7_full_3um=(
            "r7_full_3um",
            "sum",
        ),

        r7_full_5um=(
            "r7_full_5um",
            "sum",
        ),

        r7_full_7um=(
            "r7_full_7um",
            "sum",
        ),
    )
)


print(
    "\n========== PEAK-SNAPPED BY GAP SIZE =========="
)

print(
    gap_summary.to_string()
)


# ============================================================
# 21. Most important rescue/risk cases
# ============================================================

nodes[
    "r7_supported"
] = (
    nodes[
        "nearest_suppressed_peak_distance_um"
    ]
    <= 7.0
)


nodes[
    "raw_good_7um"
] = (
    nodes[
        "raw_interp_gt_error_um"
    ]
    <= 7.0
)


nodes[
    "snap_good_7um"
] = (
    nodes[
        "nearest_peak_gt_error_um"
    ]
    <= 7.0
)


nodes[
    "transition_class"
] = "UNSUPPORTED"


mask = nodes[
    "r7_supported"
]


nodes.loc[
    mask
    & ~nodes[
        "raw_good_7um"
    ]
    & nodes[
        "snap_good_7um"
    ],
    "transition_class",
] = "A_RAW_BAD_SNAP_GOOD"


nodes.loc[
    mask
    & nodes[
        "raw_good_7um"
    ]
    & nodes[
        "snap_good_7um"
    ],
    "transition_class",
] = "B_RAW_GOOD_SNAP_GOOD"


nodes.loc[
    mask
    & nodes[
        "raw_good_7um"
    ]
    & ~nodes[
        "snap_good_7um"
    ],
    "transition_class",
] = "C_RAW_GOOD_SNAP_BAD"


nodes.loc[
    mask
    & ~nodes[
        "raw_good_7um"
    ]
    & ~nodes[
        "snap_good_7um"
    ],
    "transition_class",
] = "D_RAW_BAD_SNAP_BAD"


print(
    "\n========== TRANSITION CLASS COUNTS =========="
)

print(
    nodes[
        "transition_class"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


# ============================================================
# 22. Persist audit artifacts
# ============================================================

NODE_OUT.parent.mkdir(
    parents=True,
    exist_ok=True,
)


nodes.to_pickle(
    NODE_OUT
)

component_df.to_pickle(
    COMP_OUT
)


# ============================================================
# 23. JSON summary
# ============================================================

def json_default(
    obj
):

    if isinstance(
        obj,
        np.integer,
    ):
        return int(
            obj
        )

    if isinstance(
        obj,
        np.floating,
    ):
        return float(
            obj
        )

    if isinstance(
        obj,
        np.bool_,
    ):
        return bool(
            obj
        )

    if isinstance(
        obj,
        np.ndarray,
    ):
        return obj.tolist()

    raise TypeError(
        f"Not JSON serializable: {type(obj)}"
    )


scale_json = {
    str(dataset): {
        "z_um_per_unit":
            float(
                scale[
                    0
                ]
            ),

        "y_um_per_unit":
            float(
                scale[
                    1
                ]
            ),

        "x_um_per_unit":
            float(
                scale[
                    2
                ]
            ),

        "source":
            scale_validation[
                dataset
            ][
                "source"
            ],
    }
    for dataset, scale
    in scale_map.items()
}


probability_records = []

if not probability_audit.empty:

    tmp = (
        probability_audit
        .reset_index()
        .copy()
    )

    tmp[
        "detector_probability_band"
    ] = tmp[
        "detector_probability_band"
    ].astype(
        str
    )

    probability_records = tmp.to_dict(
        orient="records"
    )


gap_records = (
    gap_summary
    .reset_index()
    .to_dict(
        orient="records"
    )
)


summary = {
    "stage":
        "12.11F",

    "decision":
        "PREDICTED_PATH_SUPPRESSED_PEAK_SUPPORT_AUDITED",

    "population": {
        "components":
            116,

        "missing_nodes":
            406,

        "datasets":
            19,
    },

    "candidate_selection": {
        "gt_blind":
            True,

        "same_dataset":
            True,

        "same_frame":
            True,

        "selection":
            (
                "nearest Stage 12.10C suppressed peak "
                "to Stage 12.11E predicted interpolation coordinate"
            ),

        "gt_usage":
            "evaluation only",
    },

    "fixed_search_radii_um":
        RADIUS_LIST,

    "physical_scale":
        scale_json,

    "stage12_11e_fidelity": {
        "intrinsic_median_error_um":
            float(
                np.median(
                    intrinsic_stored
                )
            ),

        "predicted_boundary_median_error_um":
            float(
                np.median(
                    pred_stored
                )
            ),

        "node_full_counts": {
            str(k):
                int(v)
            for k, v
            in expected_node_counts.items()
        },

        "component_full_counts": {
            str(k):
                int(v)
            for k, v
            in expected_component_counts.items()
        },
    },

    "support_by_radius":
        support_summary,

    "ambiguity_by_radius":
        ambiguity_summary,

    "node_recoverability_r7_supported":
        node_recoverability,

    "snap_effect_r7_supported": {
        "supported_nodes":
            int(
                len(
                    supported7
                )
            ),

        "improved":
            int(
                n_improve
            ),

        "unchanged":
            int(
                n_same
            ),

        "worsened":
            int(
                n_worse
            ),

        "median_raw_minus_snap_um":
            (
                float(
                    delta.median()
                )
                if len(
                    delta
                )
                else None
            ),
    },

    "transition_r7_supported":
        transition,

    "component_support_by_radius":
        component_support_summary,

    "component_raw_vs_snap_r7":
        component_comparison,

    "detector_probability_audit":
        probability_records,

    "gap_size_audit":
        gap_records,

    "prohibitions": {
        "detector_rerun":
            False,

        "threshold_tuning":
            False,

        "radius_tuning":
            False,

        "graph_modification":
            False,

        "training":
            False,
    },
}


SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
        default=json_default,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    NODE_OUT
)

print(
    "written:",
    COMP_OUT
)

print(
    "written:",
    SUMMARY_OUT
)


# ============================================================
# 24. Final compact summary
# ============================================================

r7_supported_nodes = int(
    (
        nodes[
            "nearest_suppressed_peak_distance_um"
        ]
        <= 7.0
    ).sum()
)

r7_supported_components = int(
    component_df[
        "all_nodes_supported_r7"
    ].sum()
)

r7_snap_full7 = int(
    component_df[
        "r7_full_7um"
    ].sum()
)


print(
    "\n========== 12.11F KEY RESULT =========="
)

print(
    "nodes with suppressed-peak support <=7um:",
    r7_supported_nodes,
    "/ 406",
    f"({r7_supported_nodes/406:.3%})",
)

print(
    "components with complete R7 peak support:",
    r7_supported_components,
    "/ 116",
    f"({r7_supported_components/116:.3%})",
)

print(
    "raw interpolation full-component <=7um:",
    "90 / 116",
    f"({90/116:.3%})",
)

print(
    "peak-snapped full-component <=7um:",
    r7_snap_full7,
    "/ 116",
    f"({r7_snap_full7/116:.3%})",
)

print(
    "raw bad -> snap good nodes:",
    transition[
        "A_raw_bad_snap_good"
    ],
)

print(
    "raw good -> snap bad nodes:",
    transition[
        "C_raw_good_snap_bad"
    ],
)


print(
    "\n========== 12.11F DECISION =========="
)

print(
    "PREDICTED_PATH_SUPPRESSED_PEAK_SUPPORT_AUDITED: PASS"
)

print(
    "candidate selection: GT-BLIND"
)

print(
    "GT use: AUDIT ONLY"
)

print(
    "detector rerun: NO"
)

print(
    "threshold tuning: NO"
)

print(
    "radius tuning: NO"
)

print(
    "prediction graph modification: NO"
)

In [ ]:
# ============================================================
# RUN — Stage 12.11F-X
# Suppressed-Peak Coordinate / Key Consistency Audit
#
# PURPOSE:
#   Diagnose why Stage 12.11F found:
#
#       0 / 406 nodes with suppressed peak <= 7 um
#
# This cell does NOT:
#   - change any graph
#   - tune thresholds
#   - rerun detector
#   - alter Stage 12.11F artifacts
#
# It checks:
#   1. dataset-key overlap
#   2. (dataset, frame) overlap
#   3. +/- frame indexing mismatch
#   4. coordinate ranges
#   5. nearest-peak distance distribution
#   6. possible unit mismatch
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

RPT = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)

NODE_IN = (
    RPT
    / "stage12_predicted_boundary_interpolation_audit.pkl"
)

PEAK_IN = (
    RPT
    / "stage12_folds14_suppressed_peak_universe.pkl"
)

assert NODE_IN.exists(), NODE_IN
assert PEAK_IN.exists(), PEAK_IN


# ============================================================
# 1. Load
# ============================================================

nodes = pd.read_pickle(
    NODE_IN
).copy()

peaks = pd.read_pickle(
    PEAK_IN
).copy()


SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


print(
    "========== 12.11F-X INPUT =========="
)

print(
    "nodes:",
    len(nodes)
)

print(
    "node datasets:",
    nodes["dataset"].nunique()
)

print(
    "suppressed peaks:",
    len(peaks)
)

print(
    "peak datasets:",
    peaks["dataset"].nunique()
)


# ============================================================
# 2. Dataset overlap
# ============================================================

node_datasets = set(
    nodes[
        "dataset"
    ].astype(str)
)

peak_datasets = set(
    peaks[
        "dataset"
    ].astype(str)
)


common_datasets = (
    node_datasets
    & peak_datasets
)

node_only = (
    node_datasets
    - peak_datasets
)

peak_only = (
    peak_datasets
    - node_datasets
)


print(
    "\n========== DATASET KEY OVERLAP =========="
)

print(
    "node datasets:",
    len(node_datasets)
)

print(
    "peak datasets:",
    len(peak_datasets)
)

print(
    "exact overlap:",
    len(common_datasets)
)

print(
    "node-only:",
    len(node_only)
)

print(
    "peak-only:",
    len(peak_only)
)


if node_only:

    print(
        "\nnode-only examples:"
    )

    for x in sorted(
        node_only
    )[:20]:
        print(
            " ",
            repr(x)
        )


if peak_only:

    print(
        "\npeak-only examples:"
    )

    for x in sorted(
        peak_only
    )[:20]:
        print(
            " ",
            repr(x)
        )


# ============================================================
# 3. Exact (dataset, frame) overlap
# ============================================================

node_keys = set(
    zip(
        nodes[
            "dataset"
        ].astype(str),
        nodes[
            "t"
        ].astype(int),
    )
)

peak_keys = set(
    zip(
        peaks[
            "dataset"
        ].astype(str),
        peaks[
            "t"
        ].astype(int),
    )
)


exact_key_overlap = (
    node_keys
    & peak_keys
)


print(
    "\n========== DATASET + FRAME OVERLAP =========="
)

print(
    "unique node (dataset,t):",
    len(node_keys)
)

print(
    "unique peak (dataset,t):",
    len(peak_keys)
)

print(
    "exact key overlap:",
    len(exact_key_overlap)
)


node_exact_available = np.array(
    [
        (
            str(dataset),
            int(t),
        )
        in peak_keys

        for dataset, t
        in zip(
            nodes[
                "dataset"
            ],
            nodes[
                "t"
            ],
        )
    ]
)


print(
    "node rows with exact same-frame peak universe:",
    int(
        node_exact_available.sum()
    ),
    "/",
    len(nodes),
    f"({node_exact_available.mean():.3%})",
)


# ============================================================
# 4. Audit possible frame offset
#
# Diagnostic ONLY.
# We are NOT choosing an offset here.
# ============================================================

print(
    "\n========== FRAME OFFSET COVERAGE =========="
)


offset_coverage = {}


for offset in [
    -3,
    -2,
    -1,
    0,
    1,
    2,
    3,
]:

    available = np.array(
        [
            (
                str(dataset),
                int(t) + offset,
            )
            in peak_keys

            for dataset, t
            in zip(
                nodes[
                    "dataset"
                ],
                nodes[
                    "t"
                ],
            )
        ]
    )


    n = int(
        available.sum()
    )


    offset_coverage[
        offset
    ] = n


    print(
        f"peak frame = node t {offset:+d}:",
        n,
        "/",
        len(nodes),
        f"({n/len(nodes):.3%})",
    )


# ============================================================
# 5. Frame ranges by common dataset
# ============================================================

print(
    "\n========== FRAME RANGE COMPARISON =========="
)


frame_rows = []


for dataset in sorted(
    common_datasets
):

    ng = nodes[
        nodes[
            "dataset"
        ].astype(str)
        == dataset
    ]

    pg = peaks[
        peaks[
            "dataset"
        ].astype(str)
        == dataset
    ]


    frame_rows.append(
        {
            "dataset":
                dataset,

            "node_t_min":
                int(
                    ng[
                        "t"
                    ].min()
                ),

            "node_t_max":
                int(
                    ng[
                        "t"
                    ].max()
                ),

            "peak_t_min":
                int(
                    pg[
                        "t"
                    ].min()
                ),

            "peak_t_max":
                int(
                    pg[
                        "t"
                    ].max()
                ),

            "node_rows":
                int(
                    len(ng)
                ),

            "peak_rows":
                int(
                    len(pg)
                ),
        }
    )


frame_df = pd.DataFrame(
    frame_rows
)


if len(
    frame_df
):

    print(
        frame_df.to_string(
            index=False
        )
    )

else:

    print(
        "NO COMMON DATASETS"
    )


# ============================================================
# 6. Coordinate-range comparison
#
# Compare RAW coordinate spaces first.
# This tells us whether z/y/x are even expressed in comparable
# numerical ranges.
# ============================================================

def describe_axes(
    df,
    cols,
):

    out = {}

    for col in cols:

        x = df[
            col
        ].to_numpy(
            dtype=float
        )

        out[
            col
        ] = {
            "min":
                float(
                    np.min(
                        x
                    )
                ),

            "p01":
                float(
                    np.quantile(
                        x,
                        0.01
                    )
                ),

            "median":
                float(
                    np.median(
                        x
                    )
                ),

            "p99":
                float(
                    np.quantile(
                        x,
                        0.99
                    )
                ),

            "max":
                float(
                    np.max(
                        x
                    )
                ),
        }

    return pd.DataFrame(
        out
    ).T


print(
    "\n========== RAW COORDINATE RANGE — PREDICTED INTERP =========="
)

print(
    describe_axes(
        nodes,
        [
            "pred_interp_z",
            "pred_interp_y",
            "pred_interp_x",
        ],
    ).to_string()
)


print(
    "\n========== RAW COORDINATE RANGE — SUPPRESSED PEAKS =========="
)

print(
    describe_axes(
        peaks,
        [
            "z",
            "y",
            "x",
        ],
    ).to_string()
)


# ============================================================
# 7. Index exact same-frame suppressed peaks
# ============================================================

peak_groups = {
    (
        str(dataset),
        int(t),
    ):
        g.reset_index(
            drop=True
        )

    for (
        dataset,
        t,
    ), g

    in peaks.groupby(
        [
            "dataset",
            "t",
        ],
        sort=False,
    )
}


# ============================================================
# 8. Nearest distances under several unit hypotheses
#
# H0:
#   both coordinates are voxel units
#   -> multiply both by physical voxel scale
#
# H1:
#   suppressed peak coordinates are already microns
#   -> peak raw vs pred*scale
#
# H2:
#   predicted coordinates already microns
#   -> peak*scale vs pred raw
#
# These are diagnostics only.
# ============================================================

nearest_H0 = np.full(
    len(nodes),
    np.nan
)

nearest_H1 = np.full(
    len(nodes),
    np.nan
)

nearest_H2 = np.full(
    len(nodes),
    np.nan
)

same_frame_peak_count = np.zeros(
    len(nodes),
    dtype=int,
)


for i, row in enumerate(
    nodes.itertuples(
        index=False
    )
):

    key = (
        str(
            row.dataset
        ),
        int(
            row.t
        ),
    )

    g = peak_groups.get(
        key
    )


    if (
        g is None
        or len(g) == 0
    ):
        continue


    same_frame_peak_count[
        i
    ] = len(
        g
    )


    pred_raw = np.array(
        [
            row.pred_interp_z,
            row.pred_interp_y,
            row.pred_interp_x,
        ],
        dtype=float,
    )


    peak_raw = g[
        [
            "z",
            "y",
            "x",
        ]
    ].to_numpy(
        dtype=float
    )


    # ----------------------------------------
    # H0: both are voxel coordinates
    # ----------------------------------------

    d0 = np.linalg.norm(
        (
            peak_raw
            - pred_raw.reshape(
                1,
                3,
            )
        )
        * SCALE.reshape(
            1,
            3,
        ),
        axis=1,
    )


    nearest_H0[
        i
    ] = float(
        np.min(
            d0
        )
    )


    # ----------------------------------------
    # H1: peaks already in microns
    # ----------------------------------------

    pred_um = (
        pred_raw
        * SCALE
    )


    d1 = np.linalg.norm(
        peak_raw
        - pred_um.reshape(
            1,
            3,
        ),
        axis=1,
    )


    nearest_H1[
        i
    ] = float(
        np.min(
            d1
        )
    )


    # ----------------------------------------
    # H2: predicted interpolation already um
    # ----------------------------------------

    peak_um = (
        peak_raw
        * SCALE.reshape(
            1,
            3,
        )
    )


    d2 = np.linalg.norm(
        peak_um
        - pred_raw.reshape(
            1,
            3,
        ),
        axis=1,
    )


    nearest_H2[
        i
    ] = float(
        np.min(
            d2
        )
    )


# ============================================================
# 9. Distance summaries
# ============================================================

def distance_report(
    name,
    arr,
):

    finite = arr[
        np.isfinite(
            arr
        )
    ]


    print(
        f"\n---------- {name} ----------"
    )

    print(
        "finite nodes:",
        len(
            finite
        ),
        "/",
        len(arr),
    )


    if len(
        finite
    ) == 0:

        return


    qs = [
        0.00,
        0.01,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        1.00,
    ]


    for q, value in zip(
        qs,
        np.quantile(
            finite,
            qs,
        ),
    ):

        print(
            f"q={q:>4.2f}:",
            f"{value:.6f}"
        )


    for radius in [
        2,
        3,
        5,
        7,
        10,
        15,
        20,
        30,
        50,
        100,
    ]:

        n = int(
            (
                finite
                <= radius
            ).sum()
        )

        print(
            f"<= {radius:3d}:",
            n,
            "/",
            len(finite),
            f"({n/len(finite):.3%})",
        )


print(
    "\n========== SAME-FRAME PEAK COUNTS =========="
)

print(
    "nodes with >=1 same-frame suppressed peak:",
    int(
        (
            same_frame_peak_count > 0
        ).sum()
    ),
    "/",
    len(nodes),
)

if (
    same_frame_peak_count
    > 0
).any():

    positive_counts = same_frame_peak_count[
        same_frame_peak_count
        > 0
    ]

    print(
        "same-frame peak-count median:",
        float(
            np.median(
                positive_counts
            )
        )
    )

    print(
        "same-frame peak-count p90:",
        float(
            np.quantile(
                positive_counts,
                0.90
            )
        )
    )


print(
    "\n========== NEAREST-PEAK DISTANCE DIAGNOSTICS =========="
)

distance_report(
    "H0 BOTH VOXEL -> PHYSICAL SCALE (CURRENT 12.11F ASSUMPTION)",
    nearest_H0,
)

distance_report(
    "H1 PEAK RAW ASSUMED MICRONS",
    nearest_H1,
)

distance_report(
    "H2 PRED RAW ASSUMED MICRONS",
    nearest_H2,
)


# ============================================================
# 10. Per-dataset nearest-distance summary under H0
# ============================================================

diag = nodes[
    [
        "dataset",
        "t",
        "component_root",
        "gt_node_id",
        "pred_interp_z",
        "pred_interp_y",
        "pred_interp_x",
    ]
].copy()


diag[
    "same_frame_peak_count"
] = same_frame_peak_count

diag[
    "nearest_H0_um"
] = nearest_H0

diag[
    "nearest_H1"
] = nearest_H1

diag[
    "nearest_H2"
] = nearest_H2


per_dataset = (
    diag
    .groupby(
        "dataset"
    )
    .agg(
        nodes=(
            "gt_node_id",
            "size",
        ),

        frames_with_candidates=(
            "same_frame_peak_count",
            lambda x:
                int(
                    (
                        x > 0
                    ).sum()
                ),
        ),

        median_same_frame_candidates=(
            "same_frame_peak_count",
            "median",
        ),

        median_nearest_H0_um=(
            "nearest_H0_um",
            "median",
        ),

        min_nearest_H0_um=(
            "nearest_H0_um",
            "min",
        ),

        median_nearest_H1=(
            "nearest_H1",
            "median",
        ),

        median_nearest_H2=(
            "nearest_H2",
            "median",
        ),
    )
)


print(
    "\n========== PER-DATASET DIAGNOSTIC =========="
)

print(
    per_dataset.to_string()
)


# ============================================================
# 11. Final diagnosis hints
# ============================================================

n_exact = int(
    node_exact_available.sum()
)

finite_H0 = nearest_H0[
    np.isfinite(
        nearest_H0
    )
]


print(
    "\n========== 12.11F-X DIAGNOSTIC SUMMARY =========="
)


if len(
    common_datasets
) == 0:

    print(
        "DIAGNOSIS: DATASET_KEY_MISMATCH"
    )


elif n_exact == 0:

    print(
        "DIAGNOSIS: FRAME_KEY_MISMATCH"
    )

    best_offset = max(
        offset_coverage,
        key=offset_coverage.get,
    )

    print(
        "best diagnostic frame offset:",
        best_offset,
        "coverage:",
        offset_coverage[
            best_offset
        ],
        "/",
        len(nodes),
    )


elif len(
    finite_H0
) == 0:

    print(
        "DIAGNOSIS: NO_SAME_FRAME_CANDIDATE_DATA"
    )


else:

    med = float(
        np.median(
            finite_H0
        )
    )

    q10 = float(
        np.quantile(
            finite_H0,
            0.10
        )
    )

    min_d = float(
        np.min(
            finite_H0
        )
    )


    print(
        "exact same-frame rows:",
        n_exact,
        "/",
        len(nodes),
    )

    print(
        "H0 nearest distance minimum:",
        f"{min_d:.6f}",
        "um",
    )

    print(
        "H0 nearest distance p10:",
        f"{q10:.6f}",
        "um",
    )

    print(
        "H0 nearest distance median:",
        f"{med:.6f}",
        "um",
    )


    if min_d > 100:

        print(
            "DIAGNOSIS: STRONG_COORDINATE_FRAME_MISMATCH"
        )

    elif min_d > 20:

        print(
            "DIAGNOSIS: LIKELY_COORDINATE_OR_CROP_MISMATCH"
        )

    elif (
        nearest_H1[
            np.isfinite(
                nearest_H1
            )
        ].size
        and
        np.nanmedian(
            nearest_H1
        )
        < med * 0.5
    ):

        print(
            "DIAGNOSIS: POSSIBLE_PEAK_UNIT_MISMATCH"
        )

    elif (
        nearest_H2[
            np.isfinite(
                nearest_H2
            )
        ].size
        and
        np.nanmedian(
            nearest_H2
        )
        < med * 0.5
    ):

        print(
            "DIAGNOSIS: POSSIBLE_PREDICTED_COORDINATE_UNIT_MISMATCH"
        )

    else:

        print(
            "DIAGNOSIS: COORDINATES_APPEAR_COMPARABLE; "
            "ZERO_R7_SUPPORT_MAY_BE_REAL"
        )


print(
    "\n12.11F-X COMPLETE"
)

In [ ]:
# ============================================================
# RUN — Stage 12.11F0
# Development / Fold0 Corpus Provenance Gate
#
# Purpose:
#   Formally prove why Stage 12.11F was non-evaluable:
#
#       12.11E clean gaps = Fold0
#       12.10C suppressed peaks = folds1-4
#
#   No training/inference.
#   No GT use.
#   No graph modification.
# ============================================================

from pathlib import Path
import json

import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

RPT = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)

GAP_PATH = (
    RPT
    / "stage12_predicted_boundary_interpolation_audit.pkl"
)

PEAK_PATH = (
    RPT
    / "stage12_folds14_suppressed_peak_universe.pkl"
)

OUT = (
    RPT
    / "stage12_11f0_corpus_provenance_summary.json"
)


for p in [
    CV_PATH,
    GAP_PATH,
    PEAK_PATH,
]:
    assert p.exists(), p


# ============================================================
# 1. Load
# ============================================================

cv = pd.read_csv(
    CV_PATH
)

gaps = pd.read_pickle(
    GAP_PATH
)

peaks = pd.read_pickle(
    PEAK_PATH
)


assert {
    "dataset",
    "fold",
}.issubset(
    cv.columns
)


fold_map = dict(
    zip(
        cv["dataset"],
        cv["fold"],
    )
)


# ============================================================
# 2. Dataset populations
# ============================================================

gap_datasets = sorted(
    gaps[
        "dataset"
    ]
    .astype(str)
    .unique()
)

peak_datasets = sorted(
    peaks[
        "dataset"
    ]
    .astype(str)
    .unique()
)


assert len(
    gap_datasets
) == 19

assert len(
    peak_datasets
) == 103


unknown_gap = [
    d
    for d in gap_datasets
    if d not in fold_map
]

unknown_peak = [
    d
    for d in peak_datasets
    if d not in fold_map
]


assert not unknown_gap, unknown_gap
assert not unknown_peak, unknown_peak


gap_fold_table = (
    pd.DataFrame(
        {
            "dataset":
                gap_datasets,
            "fold":
                [
                    int(
                        fold_map[d]
                    )
                    for d
                    in gap_datasets
                ],
        }
    )
    .sort_values(
        [
            "fold",
            "dataset",
        ]
    )
    .reset_index(
        drop=True
    )
)


peak_fold_table = (
    pd.DataFrame(
        {
            "dataset":
                peak_datasets,
            "fold":
                [
                    int(
                        fold_map[d]
                    )
                    for d
                    in peak_datasets
                ],
        }
    )
    .sort_values(
        [
            "fold",
            "dataset",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 3. Report
# ============================================================

print(
    "========== 12.11E CLEAN-GAP FOLDS =========="
)

print(
    gap_fold_table[
        "fold"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\n========== 12.10C SUPPRESSED-PEAK FOLDS =========="
)

print(
    peak_fold_table[
        "fold"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


gap_set = set(
    gap_datasets
)

peak_set = set(
    peak_datasets
)

overlap = (
    gap_set
    & peak_set
)


print(
    "\n========== CORPUS OVERLAP =========="
)

print(
    "12.11E datasets:",
    len(
        gap_set
    )
)

print(
    "12.10C datasets:",
    len(
        peak_set
    )
)

print(
    "overlap:",
    len(
        overlap
    )
)


# ============================================================
# 4. Hard provenance gates
# ============================================================

gap_folds = set(
    gap_fold_table[
        "fold"
    ].astype(int)
)

peak_folds = set(
    peak_fold_table[
        "fold"
    ].astype(int)
)


print(
    "\n12.11E folds:",
    sorted(
        gap_folds
    )
)

print(
    "12.10C folds:",
    sorted(
        peak_folds
    )
)


assert gap_folds == {
    0
}, (
    "Expected 12.11E population to be exclusively Fold0; "
    f"found {gap_folds}"
)

assert peak_folds == {
    1,
    2,
    3,
    4,
}, (
    "Expected 12.10C population to be folds1-4; "
    f"found {peak_folds}"
)

assert len(
    overlap
) == 0


# ============================================================
# 5. Persist formal decision
# ============================================================

payload = {
    "stage":
        "12.11F0",

    "stage12_11e": {
        "datasets":
            len(
                gap_set
            ),

        "folds":
            sorted(
                int(x)
                for x
                in gap_folds
            ),

        "role":
            "locked_fold0_outer_population",
    },

    "stage12_10c": {
        "datasets":
            len(
                peak_set
            ),

        "folds":
            sorted(
                int(x)
                for x
                in peak_folds
            ),

        "role":
            "development_suppressed_peak_universe",
    },

    "dataset_overlap":
        len(
            overlap
        ),

    "stage12_11f_status":
        (
            "NON_EVALUABLE_CORPUS_MISMATCH"
        ),

    "scientific_interpretation": (
        "Zero suppressed-peak support in the original "
        "12.11F run was caused by disjoint evaluation corpora "
        "and is not evidence that suppressed detector peaks "
        "are absent near clean-gap trajectories."
    ),

    "next_required_stage": (
        "construct folds1-4 development clean-gap population "
        "before suppressed-peak policy research"
    ),
}


OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    OUT
)


print(
    "\n========== 12.11F0 DECISION =========="
)

print(
    "CORPUS_PROVENANCE_AUDITED: PASS"
)

print(
    "12.11E: FOLD0"
)

print(
    "12.10C: FOLDS1TO4"
)

print(
    "original 12.11F:"
)

print(
    "NON_EVALUABLE_CORPUS_MISMATCH"
)

print(
    "next:"
)

print(
    "BUILD_FOLDS14_CLEAN_GAP_POPULATION"
)

In [ ]:
# ============================================================
# RUN — Stage 12.11F-R
# Fold0 Predicted-Path Suppressed-Peak Support Audit
#
# Corrected audit:
#
#   12.11E:
#       Fold0 clean-gap population
#       116 components
#       406 missing nodes
#
#   candidate source:
#       stage12_fold0_suppressed_peak_universe.pkl
#
# This is CONFIRMATORY ONLY.
# No policy tuning is permitted from Fold0 GT.
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

RPT = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


NODE_IN = (
    RPT
    / "stage12_predicted_boundary_interpolation_audit.pkl"
)

COMP_IN = (
    RPT
    / "stage12_predicted_boundary_interpolation_components.pkl"
)

PEAK_IN = (
    RPT
    / "stage12_fold0_suppressed_peak_universe.pkl"
)

PEAK_SUMMARY_IN = (
    RPT
    / "stage12_fold0_suppressed_peak_universe_summary.json"
)

PROVENANCE_IN = (
    RPT
    / "stage12_11f0_corpus_provenance_summary.json"
)


NODE_OUT = (
    RPT
    / "stage12_fold0_predicted_path_suppressed_peak_support_audit.pkl"
)

COMP_OUT = (
    RPT
    / "stage12_fold0_predicted_path_suppressed_peak_support_components.pkl"
)

SUMMARY_OUT = (
    RPT
    / "stage12_fold0_predicted_path_suppressed_peak_support_summary.json"
)


for p in [
    NODE_IN,
    COMP_IN,
    PEAK_IN,
    PEAK_SUMMARY_IN,
    PROVENANCE_IN,
]:
    assert p.exists(), p


# ============================================================
# 1. Load
# ============================================================

nodes = pd.read_pickle(
    NODE_IN
).copy()

components_prev = pd.read_pickle(
    COMP_IN
).copy()

peaks = pd.read_pickle(
    PEAK_IN
).copy()

peak_summary = json.loads(
    PEAK_SUMMARY_IN.read_text(
        encoding="utf-8"
    )
)

provenance = json.loads(
    PROVENANCE_IN.read_text(
        encoding="utf-8"
    )
)


print(
    "========== 12.11F-R INPUT =========="
)

print(
    "12.11E missing nodes:",
    len(nodes)
)

print(
    "12.11E components:",
    len(components_prev)
)

print(
    "12.11E datasets:",
    nodes["dataset"].nunique()
)

print(
    "Fold0 suppressed peaks:",
    len(peaks)
)

print(
    "Fold0 peak datasets:",
    peaks["dataset"].nunique()
)


# ============================================================
# 2. Fixed schemas
# ============================================================

GT_COLS = [
    "truth_z",
    "truth_y",
    "truth_x",
]

PRED_COLS = [
    "pred_interp_z",
    "pred_interp_y",
    "pred_interp_x",
]

COMPONENT_COL = "component_root"

RAW_ERROR_COL = "pred_interp_error_um"


required_node_cols = {
    "dataset",
    "component_root",
    "gap_size",
    "gt_node_id",
    "t",
    "truth_z",
    "truth_y",
    "truth_x",
    "pred_interp_z",
    "pred_interp_y",
    "pred_interp_x",
    "pred_interp_error_um",
}


required_peak_cols = {
    "dataset",
    "fold",
    "t",
    "z",
    "y",
    "x",
    "detector_prob",
}


assert required_node_cols.issubset(
    nodes.columns
)

assert required_peak_cols.issubset(
    peaks.columns
)


# ============================================================
# 3. Locked population gates
# ============================================================

assert len(nodes) == 406
assert len(components_prev) == 116
assert nodes["dataset"].nunique() == 19


n_components = (
    nodes[
        [
            "dataset",
            COMPONENT_COL,
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

assert n_components == 116


# All candidate data must be Fold0.
assert set(
    peaks[
        "fold"
    ]
    .dropna()
    .astype(int)
    .unique()
) == {
    0
}


node_datasets = set(
    nodes[
        "dataset"
    ].astype(str)
)

peak_datasets = set(
    peaks[
        "dataset"
    ].astype(str)
)


overlap = (
    node_datasets
    & peak_datasets
)


print(
    "\n========== CORRECTED CORPUS OVERLAP =========="
)

print(
    "clean-gap datasets:",
    len(node_datasets)
)

print(
    "Fold0 peak datasets:",
    len(peak_datasets)
)

print(
    "overlap:",
    len(overlap)
)

print(
    "clean-gap coverage:",
    f"{len(overlap)}/{len(node_datasets)}"
)


assert overlap == node_datasets, (
    "Fold0 suppressed-peak artifact does not cover "
    "all 12.11E clean-gap datasets."
)


# Formal proof that previous run was known invalid.
assert provenance[
    "stage12_11f_status"
] == "NON_EVALUABLE_CORPUS_MISMATCH"


# ============================================================
# 4. Suppressed-universe integrity
# ============================================================

assert peaks[
    "detector_prob"
].gt(
    0.90
).all()


assert not peaks[
    [
        "dataset",
        "t",
        "z",
        "y",
        "x",
    ]
].duplicated().any()


print(
    "\n========== FOLD0 PEAK UNIVERSE =========="
)

print(
    "candidate floor:",
    peak_summary.get(
        "candidate_floor"
    )
)

print(
    "production threshold:",
    peak_summary.get(
        "production_threshold"
    )
)

print(
    "GT used during peak extraction:",
    peak_summary.get(
        "gt_used"
    )
)


# ============================================================
# 5. Physical scale
#
# Previously recovered exactly from Stage 12.11E:
#
#   z = 1.625 um / voxel
#   y = 0.40625 um / voxel
#   x = 0.40625 um / voxel
#
# Revalidate before candidate matching.
# ============================================================

SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


truth_raw = nodes[
    GT_COLS
].to_numpy(
    dtype=float
)

pred_raw = nodes[
    PRED_COLS
].to_numpy(
    dtype=float
)


recomputed_raw_error = np.linalg.norm(
    (
        pred_raw
        - truth_raw
    )
    * SCALE.reshape(
        1,
        3,
    ),
    axis=1,
)


stored_raw_error = nodes[
    RAW_ERROR_COL
].to_numpy(
    dtype=float
)


print(
    "\n========== 12.11E FIDELITY =========="
)

print(
    "stored median:",
    f"{np.median(stored_raw_error):.6f}"
)

print(
    "recomputed median:",
    f"{np.median(recomputed_raw_error):.6f}"
)

print(
    "max abs mismatch:",
    f"{np.max(np.abs(recomputed_raw_error - stored_raw_error)):.10f}"
)


assert np.allclose(
    recomputed_raw_error,
    stored_raw_error,
    atol=1e-6,
    rtol=1e-6,
)


assert abs(
    np.median(
        stored_raw_error
    )
    - 4.737669
) < 0.01


nodes[
    "raw_interp_gt_error_um"
] = stored_raw_error


# ============================================================
# 6. Reproduce locked 12.11E recoverability
# ============================================================

expected_node_counts = {
    2.0: 49,
    3.0: 81,
    5.0: 222,
    7.0: 335,
}


print(
    "\n========== LOCKED RAW NODE RECOVERABILITY =========="
)

for threshold, expected in expected_node_counts.items():

    observed = int(
        (
            nodes[
                "raw_interp_gt_error_um"
            ]
            <= threshold
        ).sum()
    )

    print(
        f"<= {threshold:.1f} um:",
        observed,
        "/ 406"
    )

    assert observed == expected


# ============================================================
# 7. Build exact same-frame peak index
# ============================================================

peak_groups = {
    (
        str(dataset),
        int(t),
    ):
        g.reset_index(
            drop=True
        )

    for (
        dataset,
        t,
    ), g in peaks.groupby(
        [
            "dataset",
            "t",
        ],
        sort=False,
    )
}


same_frame_available = np.array(
    [
        (
            str(dataset),
            int(t),
        )
        in peak_groups

        for dataset, t
        in zip(
            nodes[
                "dataset"
            ],
            nodes[
                "t"
            ],
        )
    ]
)


print(
    "\n========== SAME-FRAME CORPUS COVERAGE =========="
)

print(
    "nodes with candidate frame:",
    int(
        same_frame_available.sum()
    ),
    "/",
    len(nodes),
    f"({same_frame_available.mean():.3%})",
)


assert same_frame_available.all(), (
    "Some clean-gap frames are absent from Fold0 "
    "suppressed-peak universe."
)


# ============================================================
# 8. GT-blind nearest suppressed peak search
# ============================================================

RADIUS_LIST = [
    2.0,
    3.0,
    5.0,
    7.0,
]


N = len(nodes)


pred_um = (
    pred_raw
    * SCALE.reshape(
        1,
        3,
    )
)

truth_um = (
    truth_raw
    * SCALE.reshape(
        1,
        3,
    )
)


nearest_distance_um = np.full(
    N,
    np.inf
)

nearest_peak_z = np.full(
    N,
    np.nan
)

nearest_peak_y = np.full(
    N,
    np.nan
)

nearest_peak_x = np.full(
    N,
    np.nan
)

nearest_peak_prob = np.full(
    N,
    np.nan
)

nearest_peak_logit = np.full(
    N,
    np.nan
)

nearest_peak_gt_error_um = np.full(
    N,
    np.nan
)


candidate_counts = {
    radius:
        np.zeros(
            N,
            dtype=int,
        )

    for radius
    in RADIUS_LIST
}


for i, row in enumerate(
    nodes.itertuples(
        index=False
    )
):

    key = (
        str(
            row.dataset
        ),
        int(
            row.t
        ),
    )


    g = peak_groups[
        key
    ]


    peak_raw = g[
        [
            "z",
            "y",
            "x",
        ]
    ].to_numpy(
        dtype=float
    )


    peak_um = (
        peak_raw
        * SCALE.reshape(
            1,
            3,
        )
    )


    # --------------------------------------------------------
    # Candidate selection is GT-BLIND.
    #
    # Only predicted interpolation location is used.
    # --------------------------------------------------------

    d_interp = np.linalg.norm(
        peak_um
        - pred_um[
            i
        ].reshape(
            1,
            3,
        ),
        axis=1,
    )


    j = int(
        np.argmin(
            d_interp
        )
    )


    nearest_distance_um[
        i
    ] = float(
        d_interp[
            j
        ]
    )


    nearest_peak_z[
        i
    ] = float(
        peak_raw[
            j,
            0
        ]
    )

    nearest_peak_y[
        i
    ] = float(
        peak_raw[
            j,
            1
        ]
    )

    nearest_peak_x[
        i
    ] = float(
        peak_raw[
            j,
            2
        ]
    )


    nearest_peak_prob[
        i
    ] = float(
        g.iloc[
            j
        ][
            "detector_prob"
        ]
    )


    if (
        "detector_logit"
        in g.columns
    ):

        nearest_peak_logit[
            i
        ] = float(
            g.iloc[
                j
            ][
                "detector_logit"
            ]
        )


    # --------------------------------------------------------
    # GT enters only AFTER candidate selection.
    # --------------------------------------------------------

    nearest_peak_gt_error_um[
        i
    ] = float(
        np.linalg.norm(
            peak_um[
                j
            ]
            - truth_um[
                i
            ]
        )
    )


    for radius in RADIUS_LIST:

        candidate_counts[
            radius
        ][
            i
        ] = int(
            (
                d_interp
                <= radius
            ).sum()
        )


# ============================================================
# 9. Store node-level audit
# ============================================================

nodes[
    "nearest_suppressed_peak_distance_um"
] = nearest_distance_um

nodes[
    "nearest_peak_z"
] = nearest_peak_z

nodes[
    "nearest_peak_y"
] = nearest_peak_y

nodes[
    "nearest_peak_x"
] = nearest_peak_x

nodes[
    "nearest_peak_detector_prob"
] = nearest_peak_prob

nodes[
    "nearest_peak_detector_logit"
] = nearest_peak_logit

nodes[
    "nearest_peak_gt_error_um"
] = nearest_peak_gt_error_um


nodes[
    "snap_improvement_um"
] = (
    nodes[
        "raw_interp_gt_error_um"
    ]
    - nodes[
        "nearest_peak_gt_error_um"
    ]
)


for radius in RADIUS_LIST:

    tag = int(
        radius
    )

    nodes[
        f"n_suppressed_peaks_within_{tag}um"
    ] = candidate_counts[
        radius
    ]

    nodes[
        f"has_suppressed_peak_within_{tag}um"
    ] = (
        nearest_distance_um
        <= radius
    )


# ============================================================
# 10. Suppressed-peak support
# ============================================================

print(
    "\n========== CORRECTED SUPPRESSED-PEAK SUPPORT =========="
)


support_summary = {}


for radius in RADIUS_LIST:

    supported = (
        nearest_distance_um
        <= radius
    )

    n = int(
        supported.sum()
    )


    support_summary[
        str(radius)
    ] = {
        "nodes":
            n,

        "fraction":
            float(
                n
                / N
            ),
    }


    print(
        f"within {radius:.1f} um:",
        n,
        "/",
        N,
        f"({n/N:.3%})",
    )


# ============================================================
# 11. Candidate ambiguity
# ============================================================

print(
    "\n========== CANDIDATE AMBIGUITY =========="
)


ambiguity_summary = {}


for radius in RADIUS_LIST:

    counts = candidate_counts[
        radius
    ]


    zero = int(
        (
            counts == 0
        ).sum()
    )

    one = int(
        (
            counts == 1
        ).sum()
    )

    multiple = int(
        (
            counts >= 2
        ).sum()
    )


    ambiguity_summary[
        str(radius)
    ] = {
        "zero":
            zero,
        "one":
            one,
        "multiple":
            multiple,
    }


    print(
        f"\nradius <= {radius:.1f} um"
    )

    print(
        "0 peaks :",
        zero,
        f"({zero/N:.3%})"
    )

    print(
        "1 peak  :",
        one,
        f"({one/N:.3%})"
    )

    print(
        "2+ peaks:",
        multiple,
        f"({multiple/N:.3%})"
    )


# ============================================================
# 12. R7-supported raw vs snapped
# ============================================================

R_MAIN = 7.0


r7_mask = (
    nodes[
        "nearest_suppressed_peak_distance_um"
    ]
    <= R_MAIN
)


r7 = nodes[
    r7_mask
].copy()


print(
    "\n========== RAW VS PEAK-SNAPPED — R7 SUPPORT =========="
)

print(
    "supported nodes:",
    len(r7),
    "/",
    N,
    f"({len(r7)/N:.3%})",
)


QUANTILES = [
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
]


if len(
    r7
):

    comparison = pd.DataFrame(
        {
            "raw_interpolation":
                r7[
                    "raw_interp_gt_error_um"
                ].quantile(
                    QUANTILES
                ),

            "peak_snapped":
                r7[
                    "nearest_peak_gt_error_um"
                ].quantile(
                    QUANTILES
                ),
        }
    )

    print(
        comparison.to_string()
    )


# ============================================================
# 13. Node recoverability
# ============================================================

print(
    "\n========== NODE RECOVERABILITY — R7 SUPPORTED =========="
)


node_recoverability = {}


for threshold in [
    2.0,
    3.0,
    5.0,
    7.0,
]:

    raw_n = int(
        (
            r7[
                "raw_interp_gt_error_um"
            ]
            <= threshold
        ).sum()
    )

    snap_n = int(
        (
            r7[
                "nearest_peak_gt_error_um"
            ]
            <= threshold
        ).sum()
    )


    denom = len(
        r7
    )


    node_recoverability[
        str(threshold)
    ] = {
        "raw":
            raw_n,
        "snapped":
            snap_n,
        "denominator":
            denom,
    }


    if denom:

        print(
            f"<= {threshold:.1f} um | "
            f"raw {raw_n}/{denom} "
            f"({raw_n/denom:.3%}) | "
            f"snap {snap_n}/{denom} "
            f"({snap_n/denom:.3%})"
        )


# ============================================================
# 14. Improvement / degradation
# ============================================================

print(
    "\n========== SNAP EFFECT =========="
)


if len(
    r7
):

    delta = r7[
        "snap_improvement_um"
    ].dropna()


    n_improve = int(
        (
            delta > 1e-9
        ).sum()
    )

    n_same = int(
        (
            np.abs(
                delta
            )
            <= 1e-9
        ).sum()
    )

    n_worse = int(
        (
            delta < -1e-9
        ).sum()
    )


    print(
        "improves:",
        n_improve,
        "/",
        len(delta),
        f"({n_improve/len(delta):.3%})"
    )

    print(
        "unchanged:",
        n_same,
        "/",
        len(delta),
        f"({n_same/len(delta):.3%})"
    )

    print(
        "worsens:",
        n_worse,
        "/",
        len(delta),
        f"({n_worse/len(delta):.3%})"
    )


    print(
        "\nraw - snap improvement quantiles (um):"
    )

    print(
        delta.quantile(
            [
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
            ]
        ).to_string()
    )


else:

    delta = pd.Series(
        dtype=float
    )

    n_improve = 0
    n_same = 0
    n_worse = 0


# ============================================================
# 15. 7-um transition table
# ============================================================

raw_good = (
    r7[
        "raw_interp_gt_error_um"
    ]
    <= 7.0
)

snap_good = (
    r7[
        "nearest_peak_gt_error_um"
    ]
    <= 7.0
)


transition = {
    "A_raw_bad_snap_good":
        int(
            (
                ~raw_good
                & snap_good
            ).sum()
        ),

    "B_raw_good_snap_good":
        int(
            (
                raw_good
                & snap_good
            ).sum()
        ),

    "C_raw_good_snap_bad":
        int(
            (
                raw_good
                & ~snap_good
            ).sum()
        ),

    "D_raw_bad_snap_bad":
        int(
            (
                ~raw_good
                & ~snap_good
            ).sum()
        ),
}


print(
    "\n========== 7-UM TRANSITION TABLE =========="
)


for k, v in transition.items():

    print(
        f"{k:24s}:",
        v
    )


# ============================================================
# 16. Detector-probability descriptive audit
#
# IMPORTANT:
# This is descriptive only.
# Do NOT select a threshold from Fold0.
# ============================================================

probability_audit = pd.DataFrame()


if len(
    r7
):

    bins = [
        0.90,
        0.95,
        0.97,
        0.98,
        0.99,
        0.995,
        np.inf,
    ]

    labels = [
        "[0.90,0.95)",
        "[0.95,0.97)",
        "[0.97,0.98)",
        "[0.98,0.99)",
        "[0.99,0.995)",
        "[0.995,+inf)",
    ]


    r7[
        "detector_probability_band"
    ] = pd.cut(
        r7[
            "nearest_peak_detector_prob"
        ],
        bins=bins,
        labels=labels,
        right=False,
        include_lowest=True,
    )


    probability_audit = (
        r7
        .groupby(
            "detector_probability_band",
            observed=True,
        )
        .agg(
            n=(
                "nearest_peak_gt_error_um",
                "size",
            ),

            median_gt_error_um=(
                "nearest_peak_gt_error_um",
                "median",
            ),

            good_7um=(
                "nearest_peak_gt_error_um",
                lambda x:
                    int(
                        (
                            x <= 7.0
                        ).sum()
                    ),
            ),
        )
    )


    probability_audit[
        "p_good_7um"
    ] = (
        probability_audit[
            "good_7um"
        ]
        / probability_audit[
            "n"
        ]
    )


print(
    "\n========== DETECTOR PROBABILITY — DESCRIPTIVE ONLY =========="
)

if len(
    probability_audit
):

    print(
        probability_audit.to_string()
    )

else:

    print(
        "No R7-supported nodes."
    )


# ============================================================
# 17. Component-level results
# ============================================================

component_rows = []


for (
    dataset,
    component_root,
), g in nodes.groupby(
    [
        "dataset",
        COMPONENT_COL,
    ],
    sort=False,
):

    gap_size = int(
        g[
            "gap_size"
        ].iloc[
            0
        ]
    )


    assert len(
        g
    ) == gap_size


    row = {
        "dataset":
            dataset,

        "component_root":
            component_root,

        "gap_size":
            gap_size,

        "missing_nodes":
            len(g),

        "raw_mean_error_um":
            float(
                g[
                    "raw_interp_gt_error_um"
                ].mean()
            ),

        "raw_max_error_um":
            float(
                g[
                    "raw_interp_gt_error_um"
                ].max()
            ),
    }


    for radius in RADIUS_LIST:

        tag = int(
            radius
        )


        all_supported = bool(
            g[
                f"has_suppressed_peak_within_{tag}um"
            ].all()
        )


        row[
            f"all_nodes_supported_r{tag}"
        ] = all_supported


        if all_supported:

            e = g[
                "nearest_peak_gt_error_um"
            ].to_numpy(
                dtype=float
            )


            row[
                f"snap_mean_error_r{tag}_um"
            ] = float(
                np.mean(
                    e
                )
            )

            row[
                f"snap_max_error_r{tag}_um"
            ] = float(
                np.max(
                    e
                )
            )


            for threshold in [
                2,
                3,
                5,
                7,
            ]:

                row[
                    f"r{tag}_full_{threshold}um"
                ] = bool(
                    np.all(
                        e
                        <= threshold
                    )
                )


        else:

            row[
                f"snap_mean_error_r{tag}_um"
            ] = np.nan

            row[
                f"snap_max_error_r{tag}_um"
            ] = np.nan


            for threshold in [
                2,
                3,
                5,
                7,
            ]:

                row[
                    f"r{tag}_full_{threshold}um"
                ] = False


    component_rows.append(
        row
    )


component_df = pd.DataFrame(
    component_rows
)


assert len(
    component_df
) == 116


# ============================================================
# 18. Reproduce raw component counts
# ============================================================

RAW_COMPONENT_EXPECTED = {
    2: 10,
    3: 20,
    5: 53,
    7: 90,
}


print(
    "\n========== RAW COMPONENT FIDELITY =========="
)


for threshold, expected in RAW_COMPONENT_EXPECTED.items():

    observed = int(
        (
            component_df[
                "raw_max_error_um"
            ]
            <= threshold
        ).sum()
    )


    print(
        f"full <= {threshold}um:",
        observed,
        "/ 116"
    )

    assert observed == expected


# ============================================================
# 19. Peak-supported components
# ============================================================

print(
    "\n========== CORRECTED FULL-COMPONENT RECOVERABILITY =========="
)


component_support = {}


for radius in RADIUS_LIST:

    tag = int(
        radius
    )


    all_supported = int(
        component_df[
            f"all_nodes_supported_r{tag}"
        ].sum()
    )


    print(
        f"\nsearch radius <= {radius:.1f} um"
    )

    print(
        "all nodes have support:",
        all_supported,
        "/ 116",
        f"({all_supported/116:.3%})"
    )


    radius_result = {
        "all_supported":
            all_supported
    }


    for threshold in [
        2,
        3,
        5,
        7,
    ]:

        n = int(
            component_df[
                f"r{tag}_full_{threshold}um"
            ].sum()
        )


        radius_result[
            f"full_{threshold}um"
        ] = n


        print(
            f"snapped full <= {threshold}um:",
            n,
            "/ 116",
            f"({n/116:.3%})"
        )


    component_support[
        str(radius)
    ] = radius_result


# ============================================================
# 20. Direct R7 component comparison
# ============================================================

print(
    "\n========== RAW VS SNAP COMPONENTS — R7 =========="
)


component_comparison = {}


for threshold in [
    2,
    3,
    5,
    7,
]:

    raw_n = int(
        (
            component_df[
                "raw_max_error_um"
            ]
            <= threshold
        ).sum()
    )


    snap_n = int(
        component_df[
            f"r7_full_{threshold}um"
        ].sum()
    )


    component_comparison[
        str(threshold)
    ] = {
        "raw":
            raw_n,
        "snap":
            snap_n,
    }


    print(
        f"full <= {threshold}um | "
        f"raw {raw_n}/116 | "
        f"snap {snap_n}/116"
    )


# ============================================================
# 21. By gap size
# ============================================================

gap_summary = (
    component_df
    .groupby(
        "gap_size"
    )
    .agg(
        components=(
            "component_root",
            "size",
        ),

        missing_nodes=(
            "missing_nodes",
            "sum",
        ),

        raw_median_mean_error_um=(
            "raw_mean_error_um",
            "median",
        ),

        raw_median_max_error_um=(
            "raw_max_error_um",
            "median",
        ),

        r7_all_supported=(
            "all_nodes_supported_r7",
            "sum",
        ),

        r7_full_3um=(
            "r7_full_3um",
            "sum",
        ),

        r7_full_5um=(
            "r7_full_5um",
            "sum",
        ),

        r7_full_7um=(
            "r7_full_7um",
            "sum",
        ),
    )
)


print(
    "\n========== CORRECTED RESULTS BY GAP SIZE =========="
)

print(
    gap_summary.to_string()
)


# ============================================================
# 22. Transition classes
# ============================================================

nodes[
    "r7_supported"
] = (
    nodes[
        "nearest_suppressed_peak_distance_um"
    ]
    <= 7.0
)

nodes[
    "raw_good_7um"
] = (
    nodes[
        "raw_interp_gt_error_um"
    ]
    <= 7.0
)

nodes[
    "snap_good_7um"
] = (
    nodes[
        "nearest_peak_gt_error_um"
    ]
    <= 7.0
)


nodes[
    "transition_class"
] = "UNSUPPORTED"


m = nodes[
    "r7_supported"
]


nodes.loc[
    m
    & ~nodes[
        "raw_good_7um"
    ]
    & nodes[
        "snap_good_7um"
    ],
    "transition_class",
] = "A_RAW_BAD_SNAP_GOOD"


nodes.loc[
    m
    & nodes[
        "raw_good_7um"
    ]
    & nodes[
        "snap_good_7um"
    ],
    "transition_class",
] = "B_RAW_GOOD_SNAP_GOOD"


nodes.loc[
    m
    & nodes[
        "raw_good_7um"
    ]
    & ~nodes[
        "snap_good_7um"
    ],
    "transition_class",
] = "C_RAW_GOOD_SNAP_BAD"


nodes.loc[
    m
    & ~nodes[
        "raw_good_7um"
    ]
    & ~nodes[
        "snap_good_7um"
    ],
    "transition_class",
] = "D_RAW_BAD_SNAP_BAD"


print(
    "\n========== TRANSITION CLASS COUNTS =========="
)

print(
    nodes[
        "transition_class"
    ]
    .value_counts()
    .to_string()
)


# ============================================================
# 23. Persist corrected artifacts
# ============================================================

nodes.to_pickle(
    NODE_OUT
)

component_df.to_pickle(
    COMP_OUT
)


def native(
    obj
):

    if isinstance(
        obj,
        np.integer,
    ):
        return int(
            obj
        )

    if isinstance(
        obj,
        np.floating,
    ):
        return float(
            obj
        )

    if isinstance(
        obj,
        np.bool_,
    ):
        return bool(
            obj
        )

    raise TypeError(
        type(
            obj
        )
    )


summary = {
    "stage":
        "12.11F-R",

    "status":
        "CORRECTED_FOLD0_CONFIRMATORY_AUDIT",

    "previous_stage12_11f": {
        "status":
            "NON_EVALUABLE_CORPUS_MISMATCH",

        "reason":
            (
                "Original run paired Fold0 clean gaps "
                "with folds1-4 suppressed peaks."
            ),
    },

    "population": {
        "components":
            116,

        "missing_nodes":
            406,

        "datasets":
            19,
    },

    "candidate_source": {
        "artifact":
            str(
                PEAK_IN
            ),

        "fold":
            0,

        "candidates":
            int(
                len(peaks)
            ),

        "datasets":
            int(
                peaks[
                    "dataset"
                ].nunique()
            ),
    },

    "selection": {
        "gt_blind":
            True,

        "same_dataset":
            True,

        "same_frame":
            True,

        "nearest_to_predicted_interpolation":
            True,

        "gt_used_after_selection_only":
            True,
    },

    "protocol": {
        "confirmatory_only":
            True,

        "policy_tuning_permitted":
            False,

        "threshold_tuning_permitted":
            False,

        "radius_tuning_permitted":
            False,

        "fold0_gt_used_for_evaluation":
            True,
    },

    "support_by_radius":
        support_summary,

    "ambiguity_by_radius":
        ambiguity_summary,

    "node_recoverability_r7":
        node_recoverability,

    "snap_effect_r7": {
        "supported":
            int(
                len(r7)
            ),

        "improved":
            int(
                n_improve
            ),

        "unchanged":
            int(
                n_same
            ),

        "worsened":
            int(
                n_worse
            ),

        "median_improvement_um":
            (
                float(
                    delta.median()
                )
                if len(delta)
                else None
            ),
    },

    "transition_r7":
        transition,

    "component_support":
        component_support,

    "component_raw_vs_snap_r7":
        component_comparison,

    "next_if_signal_exists":
        (
            "Reconstruct the same clean-gap experiment "
            "on development folds 1-4 before designing any policy."
        ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
        default=native,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    NODE_OUT
)

print(
    "written:",
    COMP_OUT
)

print(
    "written:",
    SUMMARY_OUT
)


# ============================================================
# 24. Final result
# ============================================================

r7_nodes = int(
    nodes[
        "r7_supported"
    ].sum()
)

r7_components = int(
    component_df[
        "all_nodes_supported_r7"
    ].sum()
)

r7_full7 = int(
    component_df[
        "r7_full_7um"
    ].sum()
)


print(
    "\n========== 12.11F-R KEY RESULT =========="
)

print(
    "nodes with R7 suppressed-peak support:",
    r7_nodes,
    "/ 406",
    f"({r7_nodes/406:.3%})"
)

print(
    "components with complete R7 support:",
    r7_components,
    "/ 116",
    f"({r7_components/116:.3%})"
)

print(
    "raw full-component <=7um:",
    "90 / 116",
    f"({90/116:.3%})"
)

print(
    "snapped full-component <=7um:",
    r7_full7,
    "/ 116",
    f"({r7_full7/116:.3%})"
)

print(
    "raw bad -> snap good:",
    transition[
        "A_raw_bad_snap_good"
    ]
)

print(
    "raw good -> snap bad:",
    transition[
        "C_raw_good_snap_bad"
    ]
)


print(
    "\n========== 12.11F-R DECISION =========="
)

print(
    "FOLD0_SUPPRESSED_PEAK_SUPPORT_CONFIRMATORY_AUDIT: PASS"
)

print(
    "artifact corpus: CORRECT"
)

print(
    "candidate selection: GT-BLIND"
)

print(
    "Fold0 GT: EVALUATION ONLY"
)

print(
    "policy tuning permitted: NO"
)

print(
    "threshold tuning permitted: NO"
)

print(
    "radius tuning permitted: NO"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G1
# Folds1-4 CURRENT_BEST 6bba Edge Residual Census
#
# Purpose:
#   Reconstruct the exact promoted Stage12 CURRENT_BEST pipeline
#   on the 103 development 6bba datasets:
#
#       Baseline256
#       -> Frozen V9
#       -> MORPH-DIV-V2 (K3_R3_P100)
#       -> MUTUAL-CONT-V1
#       -> MUTUAL-CONT-V2
#       -> DET-BRIDGE-R1P50
#
#   Then apply the SAME official node matching / edge accounting
#   used by the Fold0 CURRENT_BEST residual census.
#
# This establishes the GT-derived folds1-4 residual population
# required for clean-gap policy development.
#
# Changes files: YES — development residual artifacts
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — development research only
# Uses Fold0: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

PRED_ROOT = (
    PROJECT
    / "predictions/heqiuyan"
)

S11 = (
    PROJECT
    / "reports"
    / "stage11_division_probe"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


# ------------------------------------------------------------
# Development Baseline256 production routes
# ------------------------------------------------------------

B256_995 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det995"
    / "split_0"
)

B256_9975 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det9975"
    / "split_0"
)


# ------------------------------------------------------------
# Development policy/action artifacts
# ------------------------------------------------------------

MORPH_GRID_PATH = (
    S11
    / "stage11_deployable_final_v9_policy_grid_actions.pkl"
)

V1_UNIVERSE_PATH = (
    S12
    / "stage12_folds14_mutual_best_universe.pkl"
)

V2_UNIVERSE_PATH = (
    S12
    / "stage12_folds14_round2_reciprocal_universe.pkl"
)

BRIDGE_ACTION_PATH = (
    S12
    / "stage12_detection_bridge_policy_actions.pkl"
)


# ------------------------------------------------------------
# Locked policy metadata
# ------------------------------------------------------------

V1_LOCK_PATH = (
    S12
    / "stage12_mutual_cont_v1_policy_lock.json"
)

V2_LOCK_PATH = (
    S12
    / "stage12_mutual_cont_v2_policy_lock.json"
)

BRIDGE_SUMMARY_PATH = (
    S12
    / "stage12_detection_bridge_policy_grid_summary.json"
)


# ------------------------------------------------------------
# New development CURRENT_BEST residual artifacts
# ------------------------------------------------------------

FN_OUT = (
    S12
    / "stage12_folds14_current_best_edge_fn_residuals.pkl"
)

FP_OUT = (
    S12
    / "stage12_folds14_current_best_edge_fp_residuals.pkl"
)

DATASET_OUT = (
    S12
    / "stage12_folds14_current_best_edge_residual_datasets.pkl"
)

SUMMARY_OUT = (
    S12
    / "stage12_folds14_current_best_edge_residual_summary.json"
)


for p in [
    TRAIN_ROOT,
    B256_995,
    B256_9975,
    MORPH_GRID_PATH,
    V1_UNIVERSE_PATH,
    V2_UNIVERSE_PATH,
    BRIDGE_ACTION_PATH,
    V1_LOCK_PATH,
    V2_LOCK_PATH,
    BRIDGE_SUMMARY_PATH,
]:
    assert p.exists(), p


S12.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. Imports from project / official metric package
# ============================================================

if str(
    PROJECT / "src"
) not in sys.path:
    sys.path.insert(
        0,
        str(
            PROJECT / "src"
        ),
    )

if str(
    PROJECT / "external/official/src"
) not in sys.path:
    sys.path.insert(
        0,
        str(
            PROJECT
            / "external/official/src"
        ),
    )


from biohub_cell_tracking import frozen_v9 as fv9
from tracking_cellmot.metrics import _evaluate_matched_graph


# ============================================================
# 2. Development corpus
# ============================================================

cv = pd.read_csv(
    CV_PATH
)


corp = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(corp) == 103
assert corp["dataset"].nunique() == 103

assert set(
    corp[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}


names = (
    corp[
        "dataset"
    ]
    .tolist()
)


name_set = set(
    names
)


prefix_map = dict(
    zip(
        corp[
            "dataset"
        ],
        corp[
            "prefix"
        ],
    )
)


fold_map = dict(
    zip(
        corp[
            "dataset"
        ],
        corp[
            "fold"
        ].astype(int),
    )
)


estimated_nodes = dict(
    zip(
        corp[
            "dataset"
        ],
        corp[
            "estimated_nodes"
        ].astype(float),
    )
)


print(
    "========== FOLDS1-4 CURRENT_BEST CORPUS =========="
)

print(
    "datasets:",
    len(names)
)

print(
    "\nfold counts:"
)

print(
    corp[
        "fold"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


# ------------------------------------------------------------
# Hard Fold0 exclusion
# ------------------------------------------------------------

selected_cv = cv[
    cv[
        "dataset"
    ].isin(
        name_set
    )
]

assert len(
    selected_cv
) == 103

assert not selected_cv[
    "fold"
].eq(
    0
).any()


# ------------------------------------------------------------
# Prediction coverage
# ------------------------------------------------------------

missing_995 = [
    name
    for name in names
    if not (
        B256_995
        / f"{name}.geff"
    ).exists()
]

missing_9975 = [
    name
    for name in names
    if not (
        B256_9975
        / f"{name}.geff"
    ).exists()
]


assert not missing_995, (
    "Missing development P995 predictions: "
    f"{missing_995[:20]}"
)

assert not missing_9975, (
    "Missing development P9975 predictions: "
    f"{missing_9975[:20]}"
)


print(
    "\nP995 coverage:",
    len(names),
    "/",
    len(names),
)

print(
    "P9975 coverage:",
    len(names),
    "/",
    len(names),
)


# ============================================================
# 3. Load exact frozen development actions
# ============================================================

morph_all = pd.read_pickle(
    MORPH_GRID_PATH
)

assert "config_id" in morph_all.columns


morph_actions = (
    morph_all[
        morph_all[
            "config_id"
        ].eq(
            "K3_R3_P100"
        )
    ]
    .copy()
)


v1_universe = pd.read_pickle(
    V1_UNIVERSE_PATH
)


v1_actions = (
    v1_universe[
        v1_universe[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


v2_universe = pd.read_pickle(
    V2_UNIVERSE_PATH
)


v2_actions = (
    v2_universe[
        v2_universe[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


bridge_all = pd.read_pickle(
    BRIDGE_ACTION_PATH
)


r1p50_actions = (
    bridge_all[
        bridge_all[
            "policy_id"
        ].eq(
            "R1P50"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ============================================================
# 4. Exact action-contract gates
# ============================================================

assert len(v1_actions) == 46951
assert len(v2_actions) == 18511
assert len(r1p50_actions) == 805


for table_name, table in [
    (
        "MORPH",
        morph_actions,
    ),
    (
        "V1",
        v1_actions,
    ),
    (
        "V2",
        v2_actions,
    ),
    (
        "R1P50",
        r1p50_actions,
    ),
]:

    assert set(
        table[
            "dataset"
        ]
    ).issubset(
        name_set
    ), (
        table_name,
        "contains non-development dataset(s)",
    )


# ------------------------------------------------------------
# V1 observable contract
# ------------------------------------------------------------

assert (
    v1_actions[
        "source_final_outdegree"
    ]
    .eq(0)
    .all()
)

assert (
    v1_actions[
        "target_final_indegree"
    ]
    .eq(0)
    .all()
)

assert (
    v1_actions[
        "row_rank"
    ]
    .eq(1)
    .all()
)

assert (
    v1_actions[
        "column_rank"
    ]
    .eq(1)
    .all()
)

assert (
    v1_actions[
        "candidate_prob"
    ]
    .ge(
        0.25
    )
    .all()
)


# ------------------------------------------------------------
# V2 probability contract
# Reciprocal/free-free ranking was already encoded in universe.
# ------------------------------------------------------------

assert (
    v2_actions[
        "candidate_prob"
    ]
    .ge(
        0.25
    )
    .all()
)


# ------------------------------------------------------------
# R1P50 contract
# ------------------------------------------------------------

assert (
    r1p50_actions[
        "policy_id"
    ]
    .eq(
        "R1P50"
    )
    .all()
)

assert (
    r1p50_actions[
        "bridge_min_prob"
    ]
    .ge(
        0.50
    )
    .all()
)


print(
    "\n========== DEVELOPMENT ACTION CONTRACT =========="
)

print(
    "MORPH-DIV-V2 actions:",
    len(
        morph_actions
    ),
)

print(
    "V1 actions:",
    len(
        v1_actions
    ),
)

print(
    "V2 actions:",
    len(
        v2_actions
    ),
)

print(
    "R1P50 actions:",
    len(
        r1p50_actions
    ),
)


# ============================================================
# 5. Revalidate policy metadata + action hashes
# ============================================================

v1_lock = json.loads(
    V1_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)

v2_lock = json.loads(
    V2_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)

bridge_summary = json.loads(
    BRIDGE_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    v1_lock[
        "policy_name"
    ]
    == "MUTUAL-CONT-V1"
)

assert int(
    v1_lock[
        "development"
    ][
        "actions"
    ]
) == 46951


assert (
    v2_lock[
        "policy_name"
    ]
    == "MUTUAL-CONT-V2"
)

assert int(
    v2_lock[
        "development"
    ][
        "actions"
    ]
) == 18511


# ------------------------------------------------------------
# Reproduce V1 locked action hash
# ------------------------------------------------------------

V1_LOCK_COLS = [
    "dataset",
    "fold",
    "t",
    "source_pred_node_id",
    "target_pred_node_id",
    "candidate_prob",
]


v1_blob = (
    v1_actions[
        V1_LOCK_COLS
    ]
    .to_csv(
        index=False,
        float_format="%.17g",
    )
    .encode()
)


v1_sha = hashlib.sha256(
    v1_blob
).hexdigest()


assert (
    v1_sha
    ==
    v1_lock[
        "development"
    ][
        "action_sha256"
    ]
), (
    v1_sha,
    v1_lock[
        "development"
    ][
        "action_sha256"
    ],
)


# ------------------------------------------------------------
# Reproduce V2 locked action hash
# ------------------------------------------------------------

V2_LOCK_COLS = [
    "dataset",
    "fold",
    "t",
    "source_pred_node_id",
    "target_pred_node_id",
    "candidate_prob",
]


v2_blob = (
    v2_actions[
        V2_LOCK_COLS
    ]
    .to_csv(
        index=False,
        float_format="%.17g",
    )
    .encode()
)


v2_sha = hashlib.sha256(
    v2_blob
).hexdigest()


assert (
    v2_sha
    ==
    v2_lock[
        "development"
    ][
        "action_sha256"
    ]
), (
    v2_sha,
    v2_lock[
        "development"
    ][
        "action_sha256"
    ],
)


# ------------------------------------------------------------
# R1P50 historical development score
# ------------------------------------------------------------

r1_summary = [
    r
    for r in bridge_summary[
        "aggregate"
    ]
    if r[
        "policy_id"
    ] == "R1P50"
]


assert len(
    r1_summary
) == 1


r1_summary = r1_summary[
    0
]


assert int(
    r1_summary[
        "accepted_actions"
    ]
) == 805


EXPECTED_SCORE = float(
    r1_summary[
        "score"
    ]
)


assert abs(
    EXPECTED_SCORE
    -
    0.7919774336227487
) <= 1e-12


assert (
    bridge_summary[
        "action_hashes"
    ][
        "R1P50"
    ]
    ==
    "2d80d9b1cda3c4f0593aced5e8010051"
    "cc49500ef64868541365737dcbe1592f"
)


print(
    "\n========== POLICY LOCK FIDELITY =========="
)

print(
    "V1 SHA256:",
    v1_sha
)

print(
    "V2 SHA256:",
    v2_sha
)

print(
    "R1P50 expected SHA256:",
    bridge_summary[
        "action_hashes"
    ][
        "R1P50"
    ]
)

print(
    "expected CURRENT_BEST dev score:",
    f"{EXPECTED_SCORE:.12f}"
)


# ============================================================
# 6. Build Frozen-V9 policies
# ============================================================

print(
    "\n========== BUILD FROZEN-V9 DEVELOPMENT POLICIES =========="
)


policies = fv9.build_frozen_v9_policies(
    names,
    B256_995,
    B256_9975,
    TRAIN_ROOT,
    prefix_map=prefix_map,
    config=fv9.FROZEN_V9_CONFIG,
    progress=False,
)


# IMPORTANT:
# FrozenV9Policies is a structured policy object,
# not a list/dict and therefore has no len().
print(
    "Frozen-V9 policies object:",
    type(
        policies
    ).__name__,
)


assert (
    type(
        policies
    ).__name__
    == "FrozenV9Policies"
)


# ============================================================
# 7. Graph-state helper
# ============================================================

def graph_state(
    graph,
):

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )


    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    edge_set = set(
        zip(
            edges[
                "source_id"
            ].astype(int),

            edges[
                "target_id"
            ].astype(int),
        )
    )


    return (
        nodes,
        outdeg,
        indeg,
        edge_set,
    )


# ============================================================
# 8. Exact continuation insertion semantics
# ============================================================

def add_continuation_actions(
    graph,
    nodes,
    table,
    name,
):

    ds = table[
        table[
            "dataset"
        ].eq(
            name
        )
    ]


    for a in ds.itertuples(
        index=False
    ):

        src = int(
            a.source_pred_node_id
        )

        tgt = int(
            a.target_pred_node_id
        )


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(
                        dist
                    ),

                "edge_prob":
                    float(
                        a.candidate_prob
                    ),
            },
            validate_keys=True,
        )


# ============================================================
# 9. Exact folds1-4 CURRENT_BEST graph builder
# ============================================================

def build_dev_current_best(
    name,
):

    graph, scale, _ = (
        fv9.apply_frozen_v9(
            name,
            B256_995,
            B256_9975,
            policies,
            TRAIN_ROOT,
            config=fv9.FROZEN_V9_CONFIG,
        )
    )


    nodes, outdeg, indeg, edge_set = (
        graph_state(
            graph
        )
    )


    # --------------------------------------------------------
    # MORPH-DIV-V2
    # Exact Stage11 insertion semantics
    # --------------------------------------------------------

    morph = (
        morph_actions[
            morph_actions[
                "dataset"
            ].eq(
                name
            )
        ]
        .sort_values(
            [
                "morph_rank",
                "t",
                "source_pred_node_id",
                "target_pred_node_id",
            ]
        )
    )


    for a in morph.itertuples(
        index=False
    ):

        src = int(
            a.source_pred_node_id
        )

        tgt = int(
            a.target_pred_node_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 1

        assert indeg.get(
            tgt,
            0,
        ) == 0

        assert (
            src,
            tgt,
        ) not in edge_set


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(
                        dist
                    ),

                "edge_prob":
                    float(
                        a.candidate_prob
                    ),
            },
            validate_keys=True,
        )


        outdeg[
            src
        ] = 2

        indeg[
            tgt
        ] = 1

        edge_set.add(
            (
                src,
                tgt,
            )
        )


    # --------------------------------------------------------
    # MUTUAL-CONT-V1
    # --------------------------------------------------------

    add_continuation_actions(
        graph,
        nodes,
        v1_actions,
        name,
    )


    # --------------------------------------------------------
    # MUTUAL-CONT-V2
    # --------------------------------------------------------

    add_continuation_actions(
        graph,
        nodes,
        v2_actions,
        name,
    )


    # --------------------------------------------------------
    # Re-read exact post-V2 state
    # --------------------------------------------------------

    nodes, outdeg, indeg, edge_set = (
        graph_state(
            graph
        )
    )


    # --------------------------------------------------------
    # DET-BRIDGE-R1P50
    # Exact Fold0 CURRENT runner insertion semantics
    # --------------------------------------------------------

    bridge = (
        r1p50_actions[
            r1p50_actions[
                "dataset"
            ].eq(
                name
            )
        ]
    )


    for a in bridge.itertuples(
        index=False
    ):

        src = int(
            a.source_id
        )

        tgt = int(
            a.target_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 0

        assert indeg.get(
            tgt,
            0,
        ) == 0


        new_id = graph.add_node(
            {
                "t":
                    int(
                        a.t
                    ),

                "z":
                    float(
                        a.z
                    ),

                "y":
                    float(
                        a.y
                    ),

                "x":
                    float(
                        a.x
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            src,
            new_id,
            {
                "edge_dist":
                    float(
                        a.src_dist_um
                    ),

                "edge_prob":
                    float(
                        a.in_prob
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            new_id,
            tgt,
            {
                "edge_dist":
                    float(
                        a.tgt_dist_um
                    ),

                "edge_prob":
                    float(
                        a.out_prob
                    ),
            },
            validate_keys=True,
        )


        # Locked development actions are deterministic/conflict-free.
        outdeg[
            src
        ] = 1

        indeg[
            new_id
        ] = 1

        outdeg[
            new_id
        ] = 1

        indeg[
            tgt
        ] = 1


    return (
        graph,
        scale,
    )


# ============================================================
# 10. Official residual census
# ============================================================

metric_records = []

dataset_rows = []

fn_rows = []

fp_rows = []


print(
    "\n========== RUN DEVELOPMENT RESIDUAL CENSUS =========="
)


for i, name in enumerate(
    names,
    start=1,
):

    prefix = prefix_map[
        name
    ]

    fold = int(
        fold_map[
            name
        ]
    )


    graph, scale = (
        build_dev_current_best(
            name
        )
    )


    gt = fv9.load_graph(
        TRAIN_ROOT
        / f"{name}.geff"
    )


    # --------------------------------------------------------
    # Official evaluation.
    #
    # IMPORTANT:
    # evaluate_graph() performs official node matching in-place
    # on the prediction graph.
    # --------------------------------------------------------

    metric = fv9.evaluate_graph(
        graph,
        gt,
        scale,
        estimated_nodes[
            name
        ],
        fv9.FROZEN_V9_CONFIG,
    )


    metric_records.append(
        metric
    )


    # --------------------------------------------------------
    # Exact official matched-edge accounting
    # --------------------------------------------------------

    edge_eval = (
        _evaluate_matched_graph(
            graph,
            gt,
        )
        .to_pandas()
    )


    pred_nodes = (
        graph.node_attrs(
            unpack=True
        )
        .to_pandas()
    )


    pred_edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    gt_edges = (
        gt.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    # --------------------------------------------------------
    # Node matching maps
    # --------------------------------------------------------

    pred_nodes[
        "match_node_id"
    ] = (
        pd.to_numeric(
            pred_nodes[
                "match_node_id"
            ],
            errors="coerce",
        )
        .fillna(
            -1
        )
        .astype(int)
    )


    matched_nodes = (
        pred_nodes[
            pred_nodes[
                "match_node_id"
            ] >= 0
        ]
    )


    # Official matching is one-to-one.
    assert not matched_nodes[
        "match_node_id"
    ].duplicated().any()


    pred_to_gt = dict(
        zip(
            matched_nodes[
                "node_id"
            ].astype(int),

            matched_nodes[
                "match_node_id"
            ].astype(int),
        )
    )


    gt_to_pred = {
        gt_id:
            pred_id

        for pred_id, gt_id
        in pred_to_gt.items()
    }


    # --------------------------------------------------------
    # Exact official TP / valid FP
    # --------------------------------------------------------

    matched_mask = (
        edge_eval[
            "matched_edge_mask"
        ].astype(bool)
    )


    valid_mask = (
        edge_eval[
            "pred_valid"
        ].astype(bool)
    )


    tp_edges = (
        edge_eval[
            matched_mask
        ]
    )


    valid_fp_edges = (
        edge_eval[
            valid_mask
            &
            ~matched_mask
        ]
    )


    assert len(
        tp_edges
    ) == int(
        metric[
            "edge_tp"
        ]
    )


    assert len(
        valid_fp_edges
    ) == int(
        metric[
            "edge_fp"
        ]
    )


    # --------------------------------------------------------
    # Convert TP prediction edges back to GT identities
    # --------------------------------------------------------

    matched_gt_pairs = set()


    for e in tp_edges.itertuples(
        index=False
    ):

        ps = int(
            e.source_id
        )

        pt = int(
            e.target_id
        )


        assert ps in pred_to_gt
        assert pt in pred_to_gt


        matched_gt_pairs.add(
            (
                pred_to_gt[
                    ps
                ],

                pred_to_gt[
                    pt
                ],
            )
        )


    assert (
        len(
            matched_gt_pairs
        )
        ==
        int(
            metric[
                "edge_tp"
            ]
        )
    )


    gt_edge_set = set(
        zip(
            gt_edges[
                "source_id"
            ].astype(int),

            gt_edges[
                "target_id"
            ].astype(int),
        )
    )


    fn_pairs = (
        gt_edge_set
        -
        matched_gt_pairs
    )


    assert len(
        fn_pairs
    ) == int(
        metric[
            "edge_fn"
        ]
    )


    # --------------------------------------------------------
    # Degree context
    # --------------------------------------------------------

    pred_outdeg = (
        pred_edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    pred_indeg = (
        pred_edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    gt_outdeg = (
        gt_edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    valid_fp_out = (
        valid_fp_edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    valid_fp_in = (
        valid_fp_edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    # --------------------------------------------------------
    # FN root-cause decomposition
    # --------------------------------------------------------

    for gsrc, gtgt in sorted(
        fn_pairs
    ):

        ps = gt_to_pred.get(
            gsrc
        )

        pt = gt_to_pred.get(
            gtgt
        )


        src_present = (
            ps is not None
        )

        tgt_present = (
            pt is not None
        )


        if (
            not src_present
            and
            not tgt_present
        ):

            cause = (
                "DETECTION_BOTH_MISSING"
            )

        elif not src_present:

            cause = (
                "DETECTION_SOURCE_MISSING"
            )

        elif not tgt_present:

            cause = (
                "DETECTION_TARGET_MISSING"
            )

        else:

            cause = (
                "LINKING_LIMITED"
            )


        if cause == "LINKING_LIMITED":

            src_out = int(
                pred_outdeg.get(
                    ps,
                    0,
                )
            )

            tgt_in = int(
                pred_indeg.get(
                    pt,
                    0,
                )
            )


            if tgt_in == 0:

                topology = (
                    f"SRC_OUT{src_out}_TARGET_FREE"
                )

            else:

                topology = (
                    f"SRC_OUT{src_out}_TARGET_OCCUPIED"
                )


            src_fp = int(
                valid_fp_out.get(
                    ps,
                    0,
                )
            )

            tgt_fp = int(
                valid_fp_in.get(
                    pt,
                    0,
                )
            )

        else:

            src_out = (
                int(
                    pred_outdeg.get(
                        ps,
                        0,
                    )
                )
                if ps is not None
                else -1
            )


            tgt_in = (
                int(
                    pred_indeg.get(
                        pt,
                        0,
                    )
                )
                if pt is not None
                else -1
            )


            topology = (
                "DETECTION_LIMITED"
            )

            src_fp = 0
            tgt_fp = 0


        fn_rows.append(
            {
                "dataset":
                    name,

                "fold":
                    fold,

                "prefix":
                    prefix,

                "gt_source_id":
                    int(
                        gsrc
                    ),

                "gt_target_id":
                    int(
                        gtgt
                    ),

                "pred_source_id":
                    (
                        int(
                            ps
                        )
                        if ps is not None
                        else -1
                    ),

                "pred_target_id":
                    (
                        int(
                            pt
                        )
                        if pt is not None
                        else -1
                    ),

                "cause":
                    cause,

                "gt_parent_outdegree":
                    int(
                        gt_outdeg.get(
                            gsrc,
                            0,
                        )
                    ),

                "pred_source_outdegree":
                    src_out,

                "pred_target_indegree":
                    tgt_in,

                "link_topology":
                    topology,

                "source_has_valid_fp_out":
                    bool(
                        src_fp > 0
                    ),

                "target_has_valid_fp_in":
                    bool(
                        tgt_fp > 0
                    ),

                "swap_supported":
                    bool(
                        src_fp > 0
                        or tgt_fp > 0
                    ),
            }
        )


    # --------------------------------------------------------
    # Exact official FP endpoint-support decomposition
    # --------------------------------------------------------

    for e in valid_fp_edges.itertuples(
        index=False
    ):

        ps = int(
            e.source_id
        )

        pt = int(
            e.target_id
        )


        sg = pred_to_gt.get(
            ps
        )

        tg = pred_to_gt.get(
            pt
        )


        if (
            sg is not None
            and
            tg is not None
        ):

            support = (
                "BOTH_ENDPOINTS_GT_MATCHED"
            )

        elif sg is not None:

            support = (
                "SOURCE_ONLY_GT_MATCHED"
            )

        elif tg is not None:

            support = (
                "TARGET_ONLY_GT_MATCHED"
            )

        else:

            support = (
                "NEITHER_ENDPOINT_GT_MATCHED"
            )


        fp_rows.append(
            {
                "dataset":
                    name,

                "fold":
                    fold,

                "prefix":
                    prefix,

                "source_pred_node_id":
                    ps,

                "target_pred_node_id":
                    pt,

                "source_gt_node_id":
                    (
                        int(
                            sg
                        )
                        if sg is not None
                        else -1
                    ),

                "target_gt_node_id":
                    (
                        int(
                            tg
                        )
                        if tg is not None
                        else -1
                    ),

                "endpoint_support":
                    support,

                "pred_source_outdegree":
                    int(
                        pred_outdeg.get(
                            ps,
                            0,
                        )
                    ),

                "pred_target_indegree":
                    int(
                        pred_indeg.get(
                            pt,
                            0,
                        )
                    ),
            }
        )


    # --------------------------------------------------------
    # Dataset metric row
    # --------------------------------------------------------

    dataset_rows.append(
        {
            "dataset":
                name,

            "fold":
                fold,

            "prefix":
                prefix,

            "edge_tp":
                int(
                    metric[
                        "edge_tp"
                    ]
                ),

            "edge_fp":
                int(
                    metric[
                        "edge_fp"
                    ]
                ),

            "edge_fn":
                int(
                    metric[
                        "edge_fn"
                    ]
                ),

            "division_tp":
                int(
                    metric[
                        "division_tp"
                    ]
                ),

            "division_fp":
                int(
                    metric[
                        "division_fp"
                    ]
                ),

            "division_fn":
                int(
                    metric[
                        "division_fn"
                    ]
                ),

            "num_pred_nodes":
                int(
                    metric[
                        "num_pred_nodes"
                    ]
                ),

            "node_recall":
                float(
                    metric[
                        "node_recall"
                    ]
                ),

            "total_node_ratio":
                float(
                    metric[
                        "total_node_ratio"
                    ]
                ),

            "edge_jaccard":
                float(
                    metric[
                        "edge_jaccard"
                    ]
                ),

            "adj_edge_jaccard":
                float(
                    metric[
                        "adj_edge_jaccard"
                    ]
                ),
        }
    )


    if (
        i == 1
        or i % 10 == 0
        or i == len(
            names
        )
    ):

        print(
            f"{i:3d}/{len(names)}"
            f" | fold={fold}"
            f" | {name}"
        )


# ============================================================
# 11. Build census tables
# ============================================================

fn_df = pd.DataFrame(
    fn_rows
)

fp_df = pd.DataFrame(
    fp_rows
)

dataset_df = pd.DataFrame(
    dataset_rows
)


assert len(
    dataset_df
) == 103

assert dataset_df[
    "dataset"
].nunique() == 103


total_tp = int(
    dataset_df[
        "edge_tp"
    ].sum()
)

total_fp = int(
    dataset_df[
        "edge_fp"
    ].sum()
)

total_fn = int(
    dataset_df[
        "edge_fn"
    ].sum()
)


assert len(
    fn_df
) == total_fn

assert len(
    fp_df
) == total_fp


# ============================================================
# 12. Exact aggregate fidelity gate
# ============================================================

summary_metric = fv9.summarise(
    metric_records
)


print(
    "\n========== DEVELOPMENT CURRENT_BEST REPRODUCTION =========="
)

print(
    "edge TP/FP/FN:",
    total_tp,
    "/",
    total_fp,
    "/",
    total_fn,
)

print(
    "edge J       :",
    f"{summary_metric['edge_jaccard']:.9f}",
)

print(
    "adj edge J   :",
    f"{summary_metric['adj_edge_jaccard']:.9f}",
)

print(
    "division J   :",
    f"{summary_metric['division_jaccard']:.9f}",
)

print(
    "node recall  :",
    f"{summary_metric['node_recall']:.9f}",
)

print(
    "score        :",
    f"{summary_metric['score']:.9f}",
)

print(
    "expected     :",
    f"{EXPECTED_SCORE:.9f}",
)


assert abs(
    float(
        summary_metric[
            "score"
        ]
    )
    -
    EXPECTED_SCORE
) <= 5e-7, (
    summary_metric[
        "score"
    ],
    EXPECTED_SCORE,
)


# ============================================================
# 13. Per-fold fidelity gates
# ============================================================

EXPECTED_FOLD_SCORE = {
    1:
        0.8096252480135975,

    2:
        0.8063069126596746,

    3:
        0.7910271037079211,

    4:
        0.7616436652667813,
}


print(
    "\n========== CURRENT_BEST BY DEVELOPMENT FOLD =========="
)


fold_metrics = {}


for fold in [
    1,
    2,
    3,
    4,
]:

    fold_rows = (
        dataset_df[
            dataset_df[
                "fold"
            ].eq(
                fold
            )
        ]
        .to_dict(
            "records"
        )
    )


    fm = fv9.summarise(
        fold_rows
    )


    fold_metrics[
        fold
    ] = fm


    print(
        f"fold {fold}: "
        f"{fm['score']:.12f} "
        f"(expected "
        f"{EXPECTED_FOLD_SCORE[fold]:.12f})"
    )


    assert abs(
        float(
            fm[
                "score"
            ]
        )
        -
        EXPECTED_FOLD_SCORE[
            fold
        ]
    ) <= 5e-7, (
        fold,
        fm[
            "score"
        ],
        EXPECTED_FOLD_SCORE[
            fold
        ],
    )


# ============================================================
# 14. Root-cause classification
# ============================================================

fn_df[
    "root_cause"
] = np.where(
    fn_df[
        "cause"
    ].eq(
        "LINKING_LIMITED"
    ),
    "LINKING_LIMITED",
    "DETECTION_LIMITED",
)


root_counts = (
    fn_df[
        "root_cause"
    ]
    .value_counts()
)


n_detection = int(
    root_counts.get(
        "DETECTION_LIMITED",
        0,
    )
)

n_linking = int(
    root_counts.get(
        "LINKING_LIMITED",
        0,
    )
)


assert (
    n_detection
    +
    n_linking
    ==
    total_fn
)


print(
    "\n========== DEVELOPMENT EDGE FN ROOT CAUSE =========="
)

print(
    "DETECTION_LIMITED:",
    n_detection,
    f"({n_detection/total_fn:.3%})",
)

print(
    "LINKING_LIMITED  :",
    n_linking,
    f"({n_linking/total_fn:.3%})",
)


print(
    "\n========== DETECTION-LIMITED DETAIL =========="
)

print(
    fn_df.loc[
        fn_df[
            "root_cause"
        ].eq(
            "DETECTION_LIMITED"
        ),
        "cause",
    ]
    .value_counts()
    .to_string()
)


print(
    "\n========== ROOT CAUSE BY FOLD =========="
)

print(
    pd.crosstab(
        fn_df[
            "fold"
        ],
        fn_df[
            "root_cause"
        ],
    )
    .to_string()
)


# ============================================================
# 15. Detection missing-node census
# ============================================================

det = (
    fn_df[
        fn_df[
            "root_cause"
        ].eq(
            "DETECTION_LIMITED"
        )
    ]
    .copy()
)


src_missing = (
    det[
        "pred_source_id"
    ]
    .lt(
        0
    )
)

tgt_missing = (
    det[
        "pred_target_id"
    ]
    .lt(
        0
    )
)


assert (
    src_missing
    |
    tgt_missing
).all()


missing_nodes = set()


for r in det.itertuples(
    index=False
):

    if int(
        r.pred_source_id
    ) < 0:

        missing_nodes.add(
            (
                r.dataset,
                int(
                    r.gt_source_id
                ),
            )
        )


    if int(
        r.pred_target_id
    ) < 0:

        missing_nodes.add(
            (
                r.dataset,
                int(
                    r.gt_target_id
                ),
            )
        )


both_missing = int(
    (
        src_missing
        &
        tgt_missing
    )
    .sum()
)


source_missing_only = int(
    (
        src_missing
        &
        ~tgt_missing
    )
    .sum()
)


target_missing_only = int(
    (
        ~src_missing
        &
        tgt_missing
    )
    .sum()
)


print(
    "\n========== DEVELOPMENT DETECTION GAP INPUT =========="
)

print(
    "detection-limited FN:",
    len(
        det
    )
)

print(
    "unique missing GT nodes:",
    len(
        missing_nodes
    )
)

print(
    "BOTH_MISSING edges:",
    both_missing
)

print(
    "SOURCE_MISSING only:",
    source_missing_only
)

print(
    "TARGET_MISSING only:",
    target_missing_only
)


assert (
    both_missing
    +
    source_missing_only
    +
    target_missing_only
    ==
    len(
        det
    )
)


# ============================================================
# 16. Additional dataset/fold diagnostics
# ============================================================

print(
    "\n========== RESIDUAL COUNTS BY FOLD =========="
)

fold_residual = (
    fn_df.groupby(
        [
            "fold",
            "root_cause",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print(
    fold_residual.to_string()
)


print(
    "\n========== DETECTION FN BY DATASET — SUMMARY =========="
)

det_by_dataset = (
    det.groupby(
        "dataset"
    )
    .size()
)

print(
    det_by_dataset.describe().to_string()
)


# ============================================================
# 17. Persist canonical development census
# ============================================================

fn_df.to_pickle(
    FN_OUT
)

fp_df.to_pickle(
    FP_OUT
)

dataset_df.to_pickle(
    DATASET_OUT
)


payload = {
    "stage":
        "12.11G1",

    "status":
        "FOLDS14_CURRENT_BEST_RESIDUAL_CENSUS",

    "corpus":
        "folds1-4 6bba",

    "folds":
        [
            1,
            2,
            3,
            4,
        ],

    "datasets":
        103,

    "pipeline": (
        "BASELINE256_FROZEN_V9"
        "_PLUS_MORPH_DIV_V2"
        "_PLUS_MUTUAL_CONT_V1"
        "_PLUS_MUTUAL_CONT_V2"
        "_PLUS_DET_BRIDGE_R1P50"
    ),

    "actions": {
        "morph_div_v2":
            int(
                len(
                    morph_actions
                )
            ),

        "mutual_cont_v1":
            int(
                len(
                    v1_actions
                )
            ),

        "mutual_cont_v2":
            int(
                len(
                    v2_actions
                )
            ),

        "det_bridge_r1p50":
            int(
                len(
                    r1p50_actions
                )
            ),
    },

    "policy_fidelity": {
        "mutual_cont_v1_action_sha256":
            v1_sha,

        "mutual_cont_v2_action_sha256":
            v2_sha,

        "det_bridge_r1p50_expected_action_sha256":
            bridge_summary[
                "action_hashes"
            ][
                "R1P50"
            ],
    },

    "metrics": {
        "score":
            float(
                summary_metric[
                    "score"
                ]
            ),

        "edge_jaccard":
            float(
                summary_metric[
                    "edge_jaccard"
                ]
            ),

        "adj_edge_jaccard":
            float(
                summary_metric[
                    "adj_edge_jaccard"
                ]
            ),

        "division_jaccard":
            float(
                summary_metric[
                    "division_jaccard"
                ]
            ),

        "node_recall":
            float(
                summary_metric[
                    "node_recall"
                ]
            ),

        "edge_tp":
            total_tp,

        "edge_fp":
            total_fp,

        "edge_fn":
            total_fn,
    },

    "expected_development_score":
        float(
            EXPECTED_SCORE
        ),

    "residual": {
        "detection_limited_fn":
            n_detection,

        "linking_limited_fn":
            n_linking,

        "unique_missing_gt_nodes":
            int(
                len(
                    missing_nodes
                )
            ),

        "both_missing_edges":
            both_missing,

        "source_missing_only_edges":
            source_missing_only,

        "target_missing_only_edges":
            target_missing_only,
    },

    "fold_scores": {
        str(
            fold
        ):
            float(
                fold_metrics[
                    fold
                ][
                    "score"
                ]
            )

        for fold in [
            1,
            2,
            3,
            4,
        ]
    },

    "expected_fold_scores": {
        str(
            fold
        ):
            float(
                EXPECTED_FOLD_SCORE[
                    fold
                ]
            )

        for fold in [
            1,
            2,
            3,
            4,
        ]
    },

    "gt_used":
        True,

    "fold0_used":
        False,

    "deployment_policy_changed":
        False,
}


SUMMARY_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    FN_OUT
)

print(
    "written:",
    FP_OUT
)

print(
    "written:",
    DATASET_OUT
)

print(
    "written:",
    SUMMARY_OUT
)


# ============================================================
# 18. Final artifact integrity
# ============================================================

assert FN_OUT.exists()
assert FP_OUT.exists()
assert DATASET_OUT.exists()
assert SUMMARY_OUT.exists()


fn_check = pd.read_pickle(
    FN_OUT
)

fp_check = pd.read_pickle(
    FP_OUT
)

dataset_check = pd.read_pickle(
    DATASET_OUT
)


assert len(
    fn_check
) == total_fn

assert len(
    fp_check
) == total_fp

assert len(
    dataset_check
) == 103


assert set(
    dataset_check[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}


# ============================================================
# 19. Final decision
# ============================================================

print(
    "\n========== 12.11G1 DECISION =========="
)

print(
    "FOLDS14_CURRENT_BEST_RESIDUAL_CENSUS: PASS"
)

print(
    "development score reproduced:",
    f"{summary_metric['score']:.9f}",
)

print(
    "expected score:",
    f"{EXPECTED_SCORE:.9f}",
)

print(
    "score absolute mismatch:",
    f"{abs(summary_metric['score'] - EXPECTED_SCORE):.12g}",
)

print(
    "Fold0 used: NO"
)

print(
    "GT used: DEVELOPMENT EVALUATION ONLY"
)

print(
    "policy changed: NO"
)

print(
    "artifacts written: YES"
)

print(
    "next:"
)

print(
    "BUILD_FOLDS14_CLEAN_TWO_SIDED_PREDICTED_BOUNDARY_GAPS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G1
# Folds1-4 CURRENT_BEST 6bba Edge Residual Census
#
# Purpose:
#   Reconstruct the exact promoted Stage12 CURRENT_BEST pipeline
#   on the 103 development 6bba datasets:
#
#       Baseline256
#       -> Frozen V9
#       -> MORPH-DIV-V2 (K3_R3_P100)
#       -> MUTUAL-CONT-V1
#       -> MUTUAL-CONT-V2
#       -> DET-BRIDGE-R1P50
#
#   Then apply the SAME official node matching / edge accounting
#   used by the Fold0 CURRENT_BEST residual census.
#
# This establishes the GT-derived folds1-4 residual population
# required for clean-gap policy development.
#
# Changes files: YES — development residual artifacts
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — development research only
# Uses Fold0: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

PRED_ROOT = (
    PROJECT
    / "predictions/heqiuyan"
)

S11 = (
    PROJECT
    / "reports"
    / "stage11_division_probe"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


# ------------------------------------------------------------
# Development Baseline256 production routes
# ------------------------------------------------------------

B256_995 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det995"
    / "split_0"
)

B256_9975 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det9975"
    / "split_0"
)


# ------------------------------------------------------------
# Development policy/action artifacts
# ------------------------------------------------------------

MORPH_GRID_PATH = (
    S11
    / "stage11_deployable_final_v9_policy_grid_actions.pkl"
)

V1_UNIVERSE_PATH = (
    S12
    / "stage12_folds14_mutual_best_universe.pkl"
)

V2_UNIVERSE_PATH = (
    S12
    / "stage12_folds14_round2_reciprocal_universe.pkl"
)

BRIDGE_ACTION_PATH = (
    S12
    / "stage12_detection_bridge_policy_actions.pkl"
)


# ------------------------------------------------------------
# Locked policy metadata
# ------------------------------------------------------------

V1_LOCK_PATH = (
    S12
    / "stage12_mutual_cont_v1_policy_lock.json"
)

V2_LOCK_PATH = (
    S12
    / "stage12_mutual_cont_v2_policy_lock.json"
)

BRIDGE_SUMMARY_PATH = (
    S12
    / "stage12_detection_bridge_policy_grid_summary.json"
)


# ------------------------------------------------------------
# New development CURRENT_BEST residual artifacts
# ------------------------------------------------------------

FN_OUT = (
    S12
    / "stage12_folds14_current_best_edge_fn_residuals.pkl"
)

FP_OUT = (
    S12
    / "stage12_folds14_current_best_edge_fp_residuals.pkl"
)

DATASET_OUT = (
    S12
    / "stage12_folds14_current_best_edge_residual_datasets.pkl"
)

SUMMARY_OUT = (
    S12
    / "stage12_folds14_current_best_edge_residual_summary.json"
)


for p in [
    TRAIN_ROOT,
    B256_995,
    B256_9975,
    MORPH_GRID_PATH,
    V1_UNIVERSE_PATH,
    V2_UNIVERSE_PATH,
    BRIDGE_ACTION_PATH,
    V1_LOCK_PATH,
    V2_LOCK_PATH,
    BRIDGE_SUMMARY_PATH,
]:
    assert p.exists(), p


S12.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. Imports from project / official metric package
# ============================================================

if str(
    PROJECT / "src"
) not in sys.path:
    sys.path.insert(
        0,
        str(
            PROJECT / "src"
        ),
    )

if str(
    PROJECT / "external/official/src"
) not in sys.path:
    sys.path.insert(
        0,
        str(
            PROJECT
            / "external/official/src"
        ),
    )


from biohub_cell_tracking import frozen_v9 as fv9
from tracking_cellmot.metrics import _evaluate_matched_graph


# ============================================================
# 2. Development corpus
# ============================================================

cv = pd.read_csv(
    CV_PATH
)


corp = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(corp) == 103
assert corp["dataset"].nunique() == 103

assert set(
    corp[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}


names = (
    corp[
        "dataset"
    ]
    .tolist()
)


name_set = set(
    names
)


prefix_map = dict(
    zip(
        corp[
            "dataset"
        ],
        corp[
            "prefix"
        ],
    )
)


fold_map = dict(
    zip(
        corp[
            "dataset"
        ],
        corp[
            "fold"
        ].astype(int),
    )
)


estimated_nodes = dict(
    zip(
        corp[
            "dataset"
        ],
        corp[
            "estimated_nodes"
        ].astype(float),
    )
)


print(
    "========== FOLDS1-4 CURRENT_BEST CORPUS =========="
)

print(
    "datasets:",
    len(names)
)

print(
    "\nfold counts:"
)

print(
    corp[
        "fold"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


# ------------------------------------------------------------
# Hard Fold0 exclusion
# ------------------------------------------------------------

selected_cv = cv[
    cv[
        "dataset"
    ].isin(
        name_set
    )
]

assert len(
    selected_cv
) == 103

assert not selected_cv[
    "fold"
].eq(
    0
).any()


# ------------------------------------------------------------
# Prediction coverage
# ------------------------------------------------------------

missing_995 = [
    name
    for name in names
    if not (
        B256_995
        / f"{name}.geff"
    ).exists()
]

missing_9975 = [
    name
    for name in names
    if not (
        B256_9975
        / f"{name}.geff"
    ).exists()
]


assert not missing_995, (
    "Missing development P995 predictions: "
    f"{missing_995[:20]}"
)

assert not missing_9975, (
    "Missing development P9975 predictions: "
    f"{missing_9975[:20]}"
)


print(
    "\nP995 coverage:",
    len(names),
    "/",
    len(names),
)

print(
    "P9975 coverage:",
    len(names),
    "/",
    len(names),
)


# ============================================================
# 3. Load exact frozen development actions
# ============================================================

morph_all = pd.read_pickle(
    MORPH_GRID_PATH
)

assert "config_id" in morph_all.columns


morph_actions = (
    morph_all[
        morph_all[
            "config_id"
        ].eq(
            "K3_R3_P100"
        )
    ]
    .copy()
)


v1_universe = pd.read_pickle(
    V1_UNIVERSE_PATH
)


v1_actions = (
    v1_universe[
        v1_universe[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


v2_universe = pd.read_pickle(
    V2_UNIVERSE_PATH
)


v2_actions = (
    v2_universe[
        v2_universe[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


bridge_all = pd.read_pickle(
    BRIDGE_ACTION_PATH
)


r1p50_actions = (
    bridge_all[
        bridge_all[
            "policy_id"
        ].eq(
            "R1P50"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ============================================================
# 4. Exact action-contract gates
# ============================================================

assert len(v1_actions) == 46951
assert len(v2_actions) == 18511
assert len(r1p50_actions) == 805


for table_name, table in [
    (
        "MORPH",
        morph_actions,
    ),
    (
        "V1",
        v1_actions,
    ),
    (
        "V2",
        v2_actions,
    ),
    (
        "R1P50",
        r1p50_actions,
    ),
]:

    assert set(
        table[
            "dataset"
        ]
    ).issubset(
        name_set
    ), (
        table_name,
        "contains non-development dataset(s)",
    )


# ------------------------------------------------------------
# V1 observable contract
# ------------------------------------------------------------

assert (
    v1_actions[
        "source_final_outdegree"
    ]
    .eq(0)
    .all()
)

assert (
    v1_actions[
        "target_final_indegree"
    ]
    .eq(0)
    .all()
)

assert (
    v1_actions[
        "row_rank"
    ]
    .eq(1)
    .all()
)

assert (
    v1_actions[
        "column_rank"
    ]
    .eq(1)
    .all()
)

assert (
    v1_actions[
        "candidate_prob"
    ]
    .ge(
        0.25
    )
    .all()
)


# ------------------------------------------------------------
# V2 probability contract
# Reciprocal/free-free ranking was already encoded in universe.
# ------------------------------------------------------------

assert (
    v2_actions[
        "candidate_prob"
    ]
    .ge(
        0.25
    )
    .all()
)


# ------------------------------------------------------------
# R1P50 contract
# ------------------------------------------------------------

assert (
    r1p50_actions[
        "policy_id"
    ]
    .eq(
        "R1P50"
    )
    .all()
)

assert (
    r1p50_actions[
        "bridge_min_prob"
    ]
    .ge(
        0.50
    )
    .all()
)


print(
    "\n========== DEVELOPMENT ACTION CONTRACT =========="
)

print(
    "MORPH-DIV-V2 actions:",
    len(
        morph_actions
    ),
)

print(
    "V1 actions:",
    len(
        v1_actions
    ),
)

print(
    "V2 actions:",
    len(
        v2_actions
    ),
)

print(
    "R1P50 actions:",
    len(
        r1p50_actions
    ),
)


# ============================================================
# 5. Revalidate policy metadata + action hashes
# ============================================================

v1_lock = json.loads(
    V1_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)

v2_lock = json.loads(
    V2_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)

bridge_summary = json.loads(
    BRIDGE_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    v1_lock[
        "policy_name"
    ]
    == "MUTUAL-CONT-V1"
)

assert int(
    v1_lock[
        "development"
    ][
        "actions"
    ]
) == 46951


assert (
    v2_lock[
        "policy_name"
    ]
    == "MUTUAL-CONT-V2"
)

assert int(
    v2_lock[
        "development"
    ][
        "actions"
    ]
) == 18511


# ------------------------------------------------------------
# Reproduce V1 locked action hash
# ------------------------------------------------------------

V1_LOCK_COLS = [
    "dataset",
    "fold",
    "t",
    "source_pred_node_id",
    "target_pred_node_id",
    "candidate_prob",
]


v1_blob = (
    v1_actions[
        V1_LOCK_COLS
    ]
    .to_csv(
        index=False,
        float_format="%.17g",
    )
    .encode()
)


v1_sha = hashlib.sha256(
    v1_blob
).hexdigest()


assert (
    v1_sha
    ==
    v1_lock[
        "development"
    ][
        "action_sha256"
    ]
), (
    v1_sha,
    v1_lock[
        "development"
    ][
        "action_sha256"
    ],
)


# ------------------------------------------------------------
# Reproduce V2 locked action hash
# ------------------------------------------------------------

V2_LOCK_COLS = [
    "dataset",
    "fold",
    "t",
    "source_pred_node_id",
    "target_pred_node_id",
    "candidate_prob",
]


v2_blob = (
    v2_actions[
        V2_LOCK_COLS
    ]
    .to_csv(
        index=False,
        float_format="%.17g",
    )
    .encode()
)


v2_sha = hashlib.sha256(
    v2_blob
).hexdigest()


assert (
    v2_sha
    ==
    v2_lock[
        "development"
    ][
        "action_sha256"
    ]
), (
    v2_sha,
    v2_lock[
        "development"
    ][
        "action_sha256"
    ],
)


# ------------------------------------------------------------
# R1P50 historical development score
# ------------------------------------------------------------

r1_summary = [
    r
    for r in bridge_summary[
        "aggregate"
    ]
    if r[
        "policy_id"
    ] == "R1P50"
]


assert len(
    r1_summary
) == 1


r1_summary = r1_summary[
    0
]


assert int(
    r1_summary[
        "accepted_actions"
    ]
) == 805


EXPECTED_SCORE = float(
    r1_summary[
        "score"
    ]
)


assert abs(
    EXPECTED_SCORE
    -
    0.7919774336227487
) <= 1e-12


assert (
    bridge_summary[
        "action_hashes"
    ][
        "R1P50"
    ]
    ==
    "2d80d9b1cda3c4f0593aced5e8010051"
    "cc49500ef64868541365737dcbe1592f"
)


print(
    "\n========== POLICY LOCK FIDELITY =========="
)

print(
    "V1 SHA256:",
    v1_sha
)

print(
    "V2 SHA256:",
    v2_sha
)

print(
    "R1P50 expected SHA256:",
    bridge_summary[
        "action_hashes"
    ][
        "R1P50"
    ]
)

print(
    "expected CURRENT_BEST dev score:",
    f"{EXPECTED_SCORE:.12f}"
)


# ============================================================
# 6. Build Frozen-V9 policies
# ============================================================

print(
    "\n========== BUILD FROZEN-V9 DEVELOPMENT POLICIES =========="
)


policies = fv9.build_frozen_v9_policies(
    names,
    B256_995,
    B256_9975,
    TRAIN_ROOT,
    prefix_map=prefix_map,
    config=fv9.FROZEN_V9_CONFIG,
    progress=False,
)


# IMPORTANT:
# FrozenV9Policies is a structured policy object,
# not a list/dict and therefore has no len().
print(
    "Frozen-V9 policies object:",
    type(
        policies
    ).__name__,
)


assert (
    type(
        policies
    ).__name__
    == "FrozenV9Policies"
)


# ============================================================
# 7. Graph-state helper
# ============================================================

def graph_state(
    graph,
):

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )


    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    edge_set = set(
        zip(
            edges[
                "source_id"
            ].astype(int),

            edges[
                "target_id"
            ].astype(int),
        )
    )


    return (
        nodes,
        outdeg,
        indeg,
        edge_set,
    )


# ============================================================
# 8. Exact continuation insertion semantics
# ============================================================

def add_continuation_actions(
    graph,
    nodes,
    table,
    name,
):

    ds = table[
        table[
            "dataset"
        ].eq(
            name
        )
    ]


    for a in ds.itertuples(
        index=False
    ):

        src = int(
            a.source_pred_node_id
        )

        tgt = int(
            a.target_pred_node_id
        )


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(
                        dist
                    ),

                "edge_prob":
                    float(
                        a.candidate_prob
                    ),
            },
            validate_keys=True,
        )


# ============================================================
# 9. Exact folds1-4 CURRENT_BEST graph builder
# ============================================================

def build_dev_current_best(
    name,
):

    graph, scale, _ = (
        fv9.apply_frozen_v9(
            name,
            B256_995,
            B256_9975,
            policies,
            TRAIN_ROOT,
            config=fv9.FROZEN_V9_CONFIG,
        )
    )


    nodes, outdeg, indeg, edge_set = (
        graph_state(
            graph
        )
    )


    # --------------------------------------------------------
    # MORPH-DIV-V2
    # Exact Stage11 insertion semantics
    # --------------------------------------------------------

    morph = (
        morph_actions[
            morph_actions[
                "dataset"
            ].eq(
                name
            )
        ]
        .sort_values(
            [
                "morph_rank",
                "t",
                "source_pred_node_id",
                "target_pred_node_id",
            ]
        )
    )


    for a in morph.itertuples(
        index=False
    ):

        src = int(
            a.source_pred_node_id
        )

        tgt = int(
            a.target_pred_node_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 1

        assert indeg.get(
            tgt,
            0,
        ) == 0

        assert (
            src,
            tgt,
        ) not in edge_set


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(
                        dist
                    ),

                "edge_prob":
                    float(
                        a.candidate_prob
                    ),
            },
            validate_keys=True,
        )


        outdeg[
            src
        ] = 2

        indeg[
            tgt
        ] = 1

        edge_set.add(
            (
                src,
                tgt,
            )
        )


    # --------------------------------------------------------
    # MUTUAL-CONT-V1
    # --------------------------------------------------------

    add_continuation_actions(
        graph,
        nodes,
        v1_actions,
        name,
    )


    # --------------------------------------------------------
    # MUTUAL-CONT-V2
    # --------------------------------------------------------

    add_continuation_actions(
        graph,
        nodes,
        v2_actions,
        name,
    )


    # --------------------------------------------------------
    # Re-read exact post-V2 state
    # --------------------------------------------------------

    nodes, outdeg, indeg, edge_set = (
        graph_state(
            graph
        )
    )


    # --------------------------------------------------------
    # DET-BRIDGE-R1P50
    # Exact Fold0 CURRENT runner insertion semantics
    # --------------------------------------------------------

    bridge = (
        r1p50_actions[
            r1p50_actions[
                "dataset"
            ].eq(
                name
            )
        ]
    )


    for a in bridge.itertuples(
        index=False
    ):

        src = int(
            a.source_id
        )

        tgt = int(
            a.target_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 0

        assert indeg.get(
            tgt,
            0,
        ) == 0


        new_id = graph.add_node(
            {
                "t":
                    int(
                        a.t
                    ),

                "z":
                    float(
                        a.z
                    ),

                "y":
                    float(
                        a.y
                    ),

                "x":
                    float(
                        a.x
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            src,
            new_id,
            {
                "edge_dist":
                    float(
                        a.src_dist_um
                    ),

                "edge_prob":
                    float(
                        a.in_prob
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            new_id,
            tgt,
            {
                "edge_dist":
                    float(
                        a.tgt_dist_um
                    ),

                "edge_prob":
                    float(
                        a.out_prob
                    ),
            },
            validate_keys=True,
        )


        # Locked development actions are deterministic/conflict-free.
        outdeg[
            src
        ] = 1

        indeg[
            new_id
        ] = 1

        outdeg[
            new_id
        ] = 1

        indeg[
            tgt
        ] = 1


    return (
        graph,
        scale,
    )


# ============================================================
# 10. Official residual census
# ============================================================

metric_records = []

dataset_rows = []

fn_rows = []

fp_rows = []


print(
    "\n========== RUN DEVELOPMENT RESIDUAL CENSUS =========="
)


for i, name in enumerate(
    names,
    start=1,
):

    prefix = prefix_map[
        name
    ]

    fold = int(
        fold_map[
            name
        ]
    )


    graph, scale = (
        build_dev_current_best(
            name
        )
    )


    gt = fv9.load_graph(
        TRAIN_ROOT
        / f"{name}.geff"
    )


    # --------------------------------------------------------
    # Official evaluation.
    #
    # IMPORTANT:
    # evaluate_graph() performs official node matching in-place
    # on the prediction graph.
    # --------------------------------------------------------

    metric = fv9.evaluate_graph(
        graph,
        gt,
        scale,
        estimated_nodes[
            name
        ],
        fv9.FROZEN_V9_CONFIG,
    )


    metric_records.append(
        metric
    )


    # --------------------------------------------------------
    # Exact official matched-edge accounting
    # --------------------------------------------------------

    edge_eval = (
        _evaluate_matched_graph(
            graph,
            gt,
        )
        .to_pandas()
    )


    pred_nodes = (
        graph.node_attrs(
            unpack=True
        )
        .to_pandas()
    )


    pred_edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    gt_edges = (
        gt.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    # --------------------------------------------------------
    # Node matching maps
    # --------------------------------------------------------

    pred_nodes[
        "match_node_id"
    ] = (
        pd.to_numeric(
            pred_nodes[
                "match_node_id"
            ],
            errors="coerce",
        )
        .fillna(
            -1
        )
        .astype(int)
    )


    matched_nodes = (
        pred_nodes[
            pred_nodes[
                "match_node_id"
            ] >= 0
        ]
    )


    # Official matching is one-to-one.
    assert not matched_nodes[
        "match_node_id"
    ].duplicated().any()


    pred_to_gt = dict(
        zip(
            matched_nodes[
                "node_id"
            ].astype(int),

            matched_nodes[
                "match_node_id"
            ].astype(int),
        )
    )


    gt_to_pred = {
        gt_id:
            pred_id

        for pred_id, gt_id
        in pred_to_gt.items()
    }


    # --------------------------------------------------------
    # Exact official TP / valid FP
    # --------------------------------------------------------

    matched_mask = (
        edge_eval[
            "matched_edge_mask"
        ].astype(bool)
    )


    valid_mask = (
        edge_eval[
            "pred_valid"
        ].astype(bool)
    )


    tp_edges = (
        edge_eval[
            matched_mask
        ]
    )


    valid_fp_edges = (
        edge_eval[
            valid_mask
            &
            ~matched_mask
        ]
    )


    assert len(
        tp_edges
    ) == int(
        metric[
            "edge_tp"
        ]
    )


    assert len(
        valid_fp_edges
    ) == int(
        metric[
            "edge_fp"
        ]
    )


    # --------------------------------------------------------
    # Convert TP prediction edges back to GT identities
    # --------------------------------------------------------

    matched_gt_pairs = set()


    for e in tp_edges.itertuples(
        index=False
    ):

        ps = int(
            e.source_id
        )

        pt = int(
            e.target_id
        )


        assert ps in pred_to_gt
        assert pt in pred_to_gt


        matched_gt_pairs.add(
            (
                pred_to_gt[
                    ps
                ],

                pred_to_gt[
                    pt
                ],
            )
        )


    assert (
        len(
            matched_gt_pairs
        )
        ==
        int(
            metric[
                "edge_tp"
            ]
        )
    )


    gt_edge_set = set(
        zip(
            gt_edges[
                "source_id"
            ].astype(int),

            gt_edges[
                "target_id"
            ].astype(int),
        )
    )


    fn_pairs = (
        gt_edge_set
        -
        matched_gt_pairs
    )


    assert len(
        fn_pairs
    ) == int(
        metric[
            "edge_fn"
        ]
    )


    # --------------------------------------------------------
    # Degree context
    # --------------------------------------------------------

    pred_outdeg = (
        pred_edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    pred_indeg = (
        pred_edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    gt_outdeg = (
        gt_edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    valid_fp_out = (
        valid_fp_edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    valid_fp_in = (
        valid_fp_edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    # --------------------------------------------------------
    # FN root-cause decomposition
    # --------------------------------------------------------

    for gsrc, gtgt in sorted(
        fn_pairs
    ):

        ps = gt_to_pred.get(
            gsrc
        )

        pt = gt_to_pred.get(
            gtgt
        )


        src_present = (
            ps is not None
        )

        tgt_present = (
            pt is not None
        )


        if (
            not src_present
            and
            not tgt_present
        ):

            cause = (
                "DETECTION_BOTH_MISSING"
            )

        elif not src_present:

            cause = (
                "DETECTION_SOURCE_MISSING"
            )

        elif not tgt_present:

            cause = (
                "DETECTION_TARGET_MISSING"
            )

        else:

            cause = (
                "LINKING_LIMITED"
            )


        if cause == "LINKING_LIMITED":

            src_out = int(
                pred_outdeg.get(
                    ps,
                    0,
                )
            )

            tgt_in = int(
                pred_indeg.get(
                    pt,
                    0,
                )
            )


            if tgt_in == 0:

                topology = (
                    f"SRC_OUT{src_out}_TARGET_FREE"
                )

            else:

                topology = (
                    f"SRC_OUT{src_out}_TARGET_OCCUPIED"
                )


            src_fp = int(
                valid_fp_out.get(
                    ps,
                    0,
                )
            )

            tgt_fp = int(
                valid_fp_in.get(
                    pt,
                    0,
                )
            )

        else:

            src_out = (
                int(
                    pred_outdeg.get(
                        ps,
                        0,
                    )
                )
                if ps is not None
                else -1
            )


            tgt_in = (
                int(
                    pred_indeg.get(
                        pt,
                        0,
                    )
                )
                if pt is not None
                else -1
            )


            topology = (
                "DETECTION_LIMITED"
            )

            src_fp = 0
            tgt_fp = 0


        fn_rows.append(
            {
                "dataset":
                    name,

                "fold":
                    fold,

                "prefix":
                    prefix,

                "gt_source_id":
                    int(
                        gsrc
                    ),

                "gt_target_id":
                    int(
                        gtgt
                    ),

                "pred_source_id":
                    (
                        int(
                            ps
                        )
                        if ps is not None
                        else -1
                    ),

                "pred_target_id":
                    (
                        int(
                            pt
                        )
                        if pt is not None
                        else -1
                    ),

                "cause":
                    cause,

                "gt_parent_outdegree":
                    int(
                        gt_outdeg.get(
                            gsrc,
                            0,
                        )
                    ),

                "pred_source_outdegree":
                    src_out,

                "pred_target_indegree":
                    tgt_in,

                "link_topology":
                    topology,

                "source_has_valid_fp_out":
                    bool(
                        src_fp > 0
                    ),

                "target_has_valid_fp_in":
                    bool(
                        tgt_fp > 0
                    ),

                "swap_supported":
                    bool(
                        src_fp > 0
                        or tgt_fp > 0
                    ),
            }
        )


    # --------------------------------------------------------
    # Exact official FP endpoint-support decomposition
    # --------------------------------------------------------

    for e in valid_fp_edges.itertuples(
        index=False
    ):

        ps = int(
            e.source_id
        )

        pt = int(
            e.target_id
        )


        sg = pred_to_gt.get(
            ps
        )

        tg = pred_to_gt.get(
            pt
        )


        if (
            sg is not None
            and
            tg is not None
        ):

            support = (
                "BOTH_ENDPOINTS_GT_MATCHED"
            )

        elif sg is not None:

            support = (
                "SOURCE_ONLY_GT_MATCHED"
            )

        elif tg is not None:

            support = (
                "TARGET_ONLY_GT_MATCHED"
            )

        else:

            support = (
                "NEITHER_ENDPOINT_GT_MATCHED"
            )


        fp_rows.append(
            {
                "dataset":
                    name,

                "fold":
                    fold,

                "prefix":
                    prefix,

                "source_pred_node_id":
                    ps,

                "target_pred_node_id":
                    pt,

                "source_gt_node_id":
                    (
                        int(
                            sg
                        )
                        if sg is not None
                        else -1
                    ),

                "target_gt_node_id":
                    (
                        int(
                            tg
                        )
                        if tg is not None
                        else -1
                    ),

                "endpoint_support":
                    support,

                "pred_source_outdegree":
                    int(
                        pred_outdeg.get(
                            ps,
                            0,
                        )
                    ),

                "pred_target_indegree":
                    int(
                        pred_indeg.get(
                            pt,
                            0,
                        )
                    ),
            }
        )


    # --------------------------------------------------------
    # Dataset metric row
    # --------------------------------------------------------

    dataset_rows.append(
        {
            "dataset":
                name,

            "fold":
                fold,

            "prefix":
                prefix,

            "edge_tp":
                int(
                    metric[
                        "edge_tp"
                    ]
                ),

            "edge_fp":
                int(
                    metric[
                        "edge_fp"
                    ]
                ),

            "edge_fn":
                int(
                    metric[
                        "edge_fn"
                    ]
                ),

            "division_tp":
                int(
                    metric[
                        "division_tp"
                    ]
                ),

            "division_fp":
                int(
                    metric[
                        "division_fp"
                    ]
                ),

            "division_fn":
                int(
                    metric[
                        "division_fn"
                    ]
                ),

            "num_pred_nodes":
                int(
                    metric[
                        "num_pred_nodes"
                    ]
                ),

            "node_recall":
                float(
                    metric[
                        "node_recall"
                    ]
                ),

            "total_node_ratio":
                float(
                    metric[
                        "total_node_ratio"
                    ]
                ),

            "edge_jaccard":
                float(
                    metric[
                        "edge_jaccard"
                    ]
                ),

            "adj_edge_jaccard":
                float(
                    metric[
                        "adj_edge_jaccard"
                    ]
                ),
        }
    )


    if (
        i == 1
        or i % 10 == 0
        or i == len(
            names
        )
    ):

        print(
            f"{i:3d}/{len(names)}"
            f" | fold={fold}"
            f" | {name}"
        )


# ============================================================
# 11. Build census tables
# ============================================================

fn_df = pd.DataFrame(
    fn_rows
)

fp_df = pd.DataFrame(
    fp_rows
)

dataset_df = pd.DataFrame(
    dataset_rows
)


assert len(
    dataset_df
) == 103

assert dataset_df[
    "dataset"
].nunique() == 103


total_tp = int(
    dataset_df[
        "edge_tp"
    ].sum()
)

total_fp = int(
    dataset_df[
        "edge_fp"
    ].sum()
)

total_fn = int(
    dataset_df[
        "edge_fn"
    ].sum()
)


assert len(
    fn_df
) == total_fn

assert len(
    fp_df
) == total_fp


# ============================================================
# 12. Exact aggregate fidelity gate
# ============================================================

summary_metric = fv9.summarise(
    metric_records
)


print(
    "\n========== DEVELOPMENT CURRENT_BEST REPRODUCTION =========="
)

print(
    "edge TP/FP/FN:",
    total_tp,
    "/",
    total_fp,
    "/",
    total_fn,
)

print(
    "edge J       :",
    f"{summary_metric['edge_jaccard']:.9f}",
)

print(
    "adj edge J   :",
    f"{summary_metric['adj_edge_jaccard']:.9f}",
)

print(
    "division J   :",
    f"{summary_metric['division_jaccard']:.9f}",
)

print(
    "node recall  :",
    f"{summary_metric['node_recall']:.9f}",
)

print(
    "score        :",
    f"{summary_metric['score']:.9f}",
)

print(
    "expected     :",
    f"{EXPECTED_SCORE:.9f}",
)


assert abs(
    float(
        summary_metric[
            "score"
        ]
    )
    -
    EXPECTED_SCORE
) <= 5e-7, (
    summary_metric[
        "score"
    ],
    EXPECTED_SCORE,
)


# ============================================================
# 13. Per-fold fidelity gates
# ============================================================

EXPECTED_FOLD_SCORE = {
    1:
        0.8096252480135975,

    2:
        0.8063069126596746,

    3:
        0.7910271037079211,

    4:
        0.7616436652667813,
}


print(
    "\n========== CURRENT_BEST BY DEVELOPMENT FOLD =========="
)


fold_metrics = {}


for fold in [
    1,
    2,
    3,
    4,
]:

    fold_rows = (
        dataset_df[
            dataset_df[
                "fold"
            ].eq(
                fold
            )
        ]
        .to_dict(
            "records"
        )
    )


    fm = fv9.summarise(
        fold_rows
    )


    fold_metrics[
        fold
    ] = fm


    print(
        f"fold {fold}: "
        f"{fm['score']:.12f} "
        f"(expected "
        f"{EXPECTED_FOLD_SCORE[fold]:.12f})"
    )


    assert abs(
        float(
            fm[
                "score"
            ]
        )
        -
        EXPECTED_FOLD_SCORE[
            fold
        ]
    ) <= 5e-7, (
        fold,
        fm[
            "score"
        ],
        EXPECTED_FOLD_SCORE[
            fold
        ],
    )


# ============================================================
# 14. Root-cause classification
# ============================================================

fn_df[
    "root_cause"
] = np.where(
    fn_df[
        "cause"
    ].eq(
        "LINKING_LIMITED"
    ),
    "LINKING_LIMITED",
    "DETECTION_LIMITED",
)


root_counts = (
    fn_df[
        "root_cause"
    ]
    .value_counts()
)


n_detection = int(
    root_counts.get(
        "DETECTION_LIMITED",
        0,
    )
)

n_linking = int(
    root_counts.get(
        "LINKING_LIMITED",
        0,
    )
)


assert (
    n_detection
    +
    n_linking
    ==
    total_fn
)


print(
    "\n========== DEVELOPMENT EDGE FN ROOT CAUSE =========="
)

print(
    "DETECTION_LIMITED:",
    n_detection,
    f"({n_detection/total_fn:.3%})",
)

print(
    "LINKING_LIMITED  :",
    n_linking,
    f"({n_linking/total_fn:.3%})",
)


print(
    "\n========== DETECTION-LIMITED DETAIL =========="
)

print(
    fn_df.loc[
        fn_df[
            "root_cause"
        ].eq(
            "DETECTION_LIMITED"
        ),
        "cause",
    ]
    .value_counts()
    .to_string()
)


print(
    "\n========== ROOT CAUSE BY FOLD =========="
)

print(
    pd.crosstab(
        fn_df[
            "fold"
        ],
        fn_df[
            "root_cause"
        ],
    )
    .to_string()
)


# ============================================================
# 15. Detection missing-node census
# ============================================================

det = (
    fn_df[
        fn_df[
            "root_cause"
        ].eq(
            "DETECTION_LIMITED"
        )
    ]
    .copy()
)


src_missing = (
    det[
        "pred_source_id"
    ]
    .lt(
        0
    )
)

tgt_missing = (
    det[
        "pred_target_id"
    ]
    .lt(
        0
    )
)


assert (
    src_missing
    |
    tgt_missing
).all()


missing_nodes = set()


for r in det.itertuples(
    index=False
):

    if int(
        r.pred_source_id
    ) < 0:

        missing_nodes.add(
            (
                r.dataset,
                int(
                    r.gt_source_id
                ),
            )
        )


    if int(
        r.pred_target_id
    ) < 0:

        missing_nodes.add(
            (
                r.dataset,
                int(
                    r.gt_target_id
                ),
            )
        )


both_missing = int(
    (
        src_missing
        &
        tgt_missing
    )
    .sum()
)


source_missing_only = int(
    (
        src_missing
        &
        ~tgt_missing
    )
    .sum()
)


target_missing_only = int(
    (
        ~src_missing
        &
        tgt_missing
    )
    .sum()
)


print(
    "\n========== DEVELOPMENT DETECTION GAP INPUT =========="
)

print(
    "detection-limited FN:",
    len(
        det
    )
)

print(
    "unique missing GT nodes:",
    len(
        missing_nodes
    )
)

print(
    "BOTH_MISSING edges:",
    both_missing
)

print(
    "SOURCE_MISSING only:",
    source_missing_only
)

print(
    "TARGET_MISSING only:",
    target_missing_only
)


assert (
    both_missing
    +
    source_missing_only
    +
    target_missing_only
    ==
    len(
        det
    )
)


# ============================================================
# 16. Additional dataset/fold diagnostics
# ============================================================

print(
    "\n========== RESIDUAL COUNTS BY FOLD =========="
)

fold_residual = (
    fn_df.groupby(
        [
            "fold",
            "root_cause",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print(
    fold_residual.to_string()
)


print(
    "\n========== DETECTION FN BY DATASET — SUMMARY =========="
)

det_by_dataset = (
    det.groupby(
        "dataset"
    )
    .size()
)

print(
    det_by_dataset.describe().to_string()
)


# ============================================================
# 17. Persist canonical development census
# ============================================================

fn_df.to_pickle(
    FN_OUT
)

fp_df.to_pickle(
    FP_OUT
)

dataset_df.to_pickle(
    DATASET_OUT
)


payload = {
    "stage":
        "12.11G1",

    "status":
        "FOLDS14_CURRENT_BEST_RESIDUAL_CENSUS",

    "corpus":
        "folds1-4 6bba",

    "folds":
        [
            1,
            2,
            3,
            4,
        ],

    "datasets":
        103,

    "pipeline": (
        "BASELINE256_FROZEN_V9"
        "_PLUS_MORPH_DIV_V2"
        "_PLUS_MUTUAL_CONT_V1"
        "_PLUS_MUTUAL_CONT_V2"
        "_PLUS_DET_BRIDGE_R1P50"
    ),

    "actions": {
        "morph_div_v2":
            int(
                len(
                    morph_actions
                )
            ),

        "mutual_cont_v1":
            int(
                len(
                    v1_actions
                )
            ),

        "mutual_cont_v2":
            int(
                len(
                    v2_actions
                )
            ),

        "det_bridge_r1p50":
            int(
                len(
                    r1p50_actions
                )
            ),
    },

    "policy_fidelity": {
        "mutual_cont_v1_action_sha256":
            v1_sha,

        "mutual_cont_v2_action_sha256":
            v2_sha,

        "det_bridge_r1p50_expected_action_sha256":
            bridge_summary[
                "action_hashes"
            ][
                "R1P50"
            ],
    },

    "metrics": {
        "score":
            float(
                summary_metric[
                    "score"
                ]
            ),

        "edge_jaccard":
            float(
                summary_metric[
                    "edge_jaccard"
                ]
            ),

        "adj_edge_jaccard":
            float(
                summary_metric[
                    "adj_edge_jaccard"
                ]
            ),

        "division_jaccard":
            float(
                summary_metric[
                    "division_jaccard"
                ]
            ),

        "node_recall":
            float(
                summary_metric[
                    "node_recall"
                ]
            ),

        "edge_tp":
            total_tp,

        "edge_fp":
            total_fp,

        "edge_fn":
            total_fn,
    },

    "expected_development_score":
        float(
            EXPECTED_SCORE
        ),

    "residual": {
        "detection_limited_fn":
            n_detection,

        "linking_limited_fn":
            n_linking,

        "unique_missing_gt_nodes":
            int(
                len(
                    missing_nodes
                )
            ),

        "both_missing_edges":
            both_missing,

        "source_missing_only_edges":
            source_missing_only,

        "target_missing_only_edges":
            target_missing_only,
    },

    "fold_scores": {
        str(
            fold
        ):
            float(
                fold_metrics[
                    fold
                ][
                    "score"
                ]
            )

        for fold in [
            1,
            2,
            3,
            4,
        ]
    },

    "expected_fold_scores": {
        str(
            fold
        ):
            float(
                EXPECTED_FOLD_SCORE[
                    fold
                ]
            )

        for fold in [
            1,
            2,
            3,
            4,
        ]
    },

    "gt_used":
        True,

    "fold0_used":
        False,

    "deployment_policy_changed":
        False,
}


SUMMARY_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    FN_OUT
)

print(
    "written:",
    FP_OUT
)

print(
    "written:",
    DATASET_OUT
)

print(
    "written:",
    SUMMARY_OUT
)


# ============================================================
# 18. Final artifact integrity
# ============================================================

assert FN_OUT.exists()
assert FP_OUT.exists()
assert DATASET_OUT.exists()
assert SUMMARY_OUT.exists()


fn_check = pd.read_pickle(
    FN_OUT
)

fp_check = pd.read_pickle(
    FP_OUT
)

dataset_check = pd.read_pickle(
    DATASET_OUT
)


assert len(
    fn_check
) == total_fn

assert len(
    fp_check
) == total_fp

assert len(
    dataset_check
) == 103


assert set(
    dataset_check[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}


# ============================================================
# 19. Final decision
# ============================================================

print(
    "\n========== 12.11G1 DECISION =========="
)

print(
    "FOLDS14_CURRENT_BEST_RESIDUAL_CENSUS: PASS"
)

print(
    "development score reproduced:",
    f"{summary_metric['score']:.9f}",
)

print(
    "expected score:",
    f"{EXPECTED_SCORE:.9f}",
)

print(
    "score absolute mismatch:",
    f"{abs(summary_metric['score'] - EXPECTED_SCORE):.12g}",
)

print(
    "Fold0 used: NO"
)

print(
    "GT used: DEVELOPMENT EVALUATION ONLY"
)

print(
    "policy changed: NO"
)

print(
    "artifacts written: YES"
)

print(
    "next:"
)

print(
    "BUILD_FOLDS14_CLEAN_TWO_SIDED_PREDICTED_BOUNDARY_GAPS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G2
# Folds1-4 Clean Two-Sided Predicted-Boundary Interpolation
#
# Purpose:
#
#   Starting from the exact Stage 12.11G1 development
#   CURRENT_BEST official residual census:
#
#       1. Reconstruct detection-limited missing-node components.
#       2. Select clean multi-node two-sided ordinary GT chains.
#       3. Require both observed predicted boundary nodes to come
#          from the original Baseline256 P995 detection graph.
#       4. Compare:
#
#              GT-boundary interpolation
#                     vs
#              predicted-boundary interpolation
#
#          against the missing GT-node positions.
#
# IMPORTANT:
#
#   This is a GT-defined DEVELOPMENT ORACLE AUDIT.
#
#   It measures geometric feasibility only.
#   It does NOT define a deployable gap detector.
#   It does NOT modify any prediction graph.
#   It does NOT tune any policy using Fold0.
#
# Changes files: YES
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — folds1-4 development audit only
# Uses Fold0: NO
# Changes policy: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths / constants
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

P995_ROOT = (
    PROJECT
    / "predictions/heqiuyan"
    / "stage11_fullgraph_baseline256_folds1to4_det995"
    / "split_0"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


FN_PATH = (
    S12
    / "stage12_folds14_current_best_edge_fn_residuals.pkl"
)

G1_SUMMARY_PATH = (
    S12
    / "stage12_folds14_current_best_edge_residual_summary.json"
)


# Optional parity universe already built in Stage12.10C.
SUPPRESSED_PATH = (
    S12
    / "stage12_folds14_suppressed_peak_universe.pkl"
)


NODE_OUT = (
    S12
    / "stage12_folds14_predicted_boundary_interpolation_audit.pkl"
)

COMP_OUT = (
    S12
    / "stage12_folds14_predicted_boundary_interpolation_components.pkl"
)

SUMMARY_OUT = (
    S12
    / "stage12_folds14_predicted_boundary_interpolation_summary.json"
)


for p in [
    TRAIN_ROOT,
    P995_ROOT,
    FN_PATH,
    G1_SUMMARY_PATH,
    CV_PATH,
]:
    assert p.exists(), p


# Physical voxel size used throughout Stage12.
SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


RADII = [
    2.0,
    3.0,
    5.0,
    7.0,
]


# ============================================================
# 1. Imports
# ============================================================

import sys


if str(
    PROJECT / "src"
) not in sys.path:

    sys.path.insert(
        0,
        str(
            PROJECT / "src"
        ),
    )


from biohub_cell_tracking import frozen_v9 as fv9


# ============================================================
# 2. Revalidate Stage 12.11G1
# ============================================================

g1_summary = json.loads(
    G1_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    g1_summary[
        "status"
    ]
    ==
    "FOLDS14_CURRENT_BEST_RESIDUAL_CENSUS"
)

assert g1_summary[
    "folds"
] == [
    1,
    2,
    3,
    4,
]

assert int(
    g1_summary[
        "datasets"
    ]
) == 103

assert (
    g1_summary[
        "fold0_used"
    ]
    is False
)


fn = pd.read_pickle(
    FN_PATH
)


assert len(fn) == 10254

assert fn[
    "dataset"
].nunique() == 103

assert set(
    fn[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}

assert fn[
    "prefix"
].eq(
    "6bba"
).all()


print(
    "========== 12.11G2 INPUT =========="
)

print(
    "CURRENT_BEST edge FN:",
    len(fn)
)

print(
    "datasets:",
    fn[
        "dataset"
    ].nunique()
)

print(
    "folds:",
    sorted(
        fn[
            "fold"
        ]
        .unique()
        .tolist()
    )
)


# ============================================================
# 3. Exact detection-limited population
# ============================================================

det = (
    fn[
        fn[
            "root_cause"
        ].eq(
            "DETECTION_LIMITED"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


assert len(det) == 4097


src_missing = (
    det[
        "pred_source_id"
    ].lt(
        0
    )
)

tgt_missing = (
    det[
        "pred_target_id"
    ].lt(
        0
    )
)


assert (
    src_missing
    |
    tgt_missing
).all()


assert int(
    (
        src_missing
        &
        tgt_missing
    ).sum()
) == 1390

assert int(
    (
        src_missing
        &
        ~tgt_missing
    ).sum()
) == 1365

assert int(
    (
        ~src_missing
        &
        tgt_missing
    ).sum()
) == 1342


print(
    "\n========== DETECTION-LIMITED INPUT =========="
)

print(
    "FN:",
    len(det)
)

print(
    "BOTH_MISSING:",
    int(
        (
            src_missing
            &
            tgt_missing
        ).sum()
    )
)

print(
    "SOURCE_MISSING:",
    int(
        (
            src_missing
            &
            ~tgt_missing
        ).sum()
    )
)

print(
    "TARGET_MISSING:",
    int(
        (
            ~src_missing
            &
            tgt_missing
        ).sum()
    )
)


# ============================================================
# 4. Missing GT-node universe
# ============================================================

missing_nodes = set()


for r in det.itertuples(
    index=False
):

    if int(
        r.pred_source_id
    ) < 0:

        missing_nodes.add(
            (
                r.dataset,
                int(
                    r.gt_source_id
                ),
            )
        )


    if int(
        r.pred_target_id
    ) < 0:

        missing_nodes.add(
            (
                r.dataset,
                int(
                    r.gt_target_id
                ),
            )
        )


assert len(
    missing_nodes
) == 2985


# ============================================================
# 5. Union-find using BOTH_MISSING GT edges
# ============================================================

parent = {
    x:
        x

    for x in missing_nodes
}


def find(
    x,
):

    while parent[
        x
    ] != x:

        parent[
            x
        ] = parent[
            parent[
                x
            ]
        ]

        x = parent[
            x
        ]

    return x


def union(
    a,
    b,
):

    ra = find(
        a
    )

    rb = find(
        b
    )


    if ra != rb:

        parent[
            rb
        ] = ra


both = det[
    src_missing
    &
    tgt_missing
]


assert len(
    both
) == 1390


for r in both.itertuples(
    index=False
):

    a = (
        r.dataset,
        int(
            r.gt_source_id
        ),
    )

    b = (
        r.dataset,
        int(
            r.gt_target_id
        ),
    )


    assert a in parent
    assert b in parent


    union(
        a,
        b,
    )


# ============================================================
# 6. Stable component membership
# ============================================================

membership = pd.DataFrame(
    [
        {
            "dataset":
                ds,

            "gt_node_id":
                int(
                    node_id
                ),

            "root":
                find(
                    (
                        ds,
                        node_id,
                    )
                ),
        }

        for ds, node_id
        in sorted(
            missing_nodes
        )
    ]
)


roots = sorted(
    set(
        membership[
            "root"
        ]
    )
)


root_to_component = {
    root:
        i

    for i, root
    in enumerate(
        roots
    )
}


membership[
    "component_id"
] = (
    membership[
        "root"
    ]
    .map(
        root_to_component
    )
    .astype(int)
)


node_to_component = {
    (
        r.dataset,
        int(
            r.gt_node_id
        ),
    ):
        int(
            r.component_id
        )

    for r in membership.itertuples(
        index=False
    )
}


assert len(
    membership
) == 2985


print(
    "\n========== RAW MISSING-NODE COMPONENTS =========="
)

print(
    "components:",
    membership[
        "component_id"
    ].nunique()
)

print(
    "missing nodes:",
    len(
        membership
    )
)


component_sizes = (
    membership.groupby(
        [
            "dataset",
            "component_id",
        ]
    )
    .size()
)


print(
    "multi-node components:",
    int(
        (
            component_sizes
            >= 2
        ).sum()
    )
)

print(
    "multi-node missing nodes:",
    int(
        component_sizes[
            component_sizes
            >= 2
        ].sum()
    )
)


# ============================================================
# 7. Assign every detection FN to its missing component
# ============================================================

component_edge_rows = []


for r in det.itertuples(
    index=False
):

    sm = (
        int(
            r.pred_source_id
        ) < 0
    )

    tm = (
        int(
            r.pred_target_id
        ) < 0
    )


    if sm:

        cid = node_to_component[
            (
                r.dataset,
                int(
                    r.gt_source_id
                ),
            )
        ]

    else:

        cid = node_to_component[
            (
                r.dataset,
                int(
                    r.gt_target_id
                ),
            )
        ]


    if sm and tm:

        other_cid = node_to_component[
            (
                r.dataset,
                int(
                    r.gt_target_id
                ),
            )
        ]

        assert cid == other_cid

        edge_type = (
            "INTERNAL"
        )


    elif sm:

        # Missing source -> observed target.
        edge_type = (
            "OUT_BOUNDARY"
        )


    else:

        # Observed source -> missing target.
        edge_type = (
            "IN_BOUNDARY"
        )


    component_edge_rows.append(
        {
            "dataset":
                r.dataset,

            "fold":
                int(
                    r.fold
                ),

            "component_id":
                int(
                    cid
                ),

            "edge_type":
                edge_type,

            "gt_source_id":
                int(
                    r.gt_source_id
                ),

            "gt_target_id":
                int(
                    r.gt_target_id
                ),

            "pred_source_id":
                int(
                    r.pred_source_id
                ),

            "pred_target_id":
                int(
                    r.pred_target_id
                ),
        }
    )


component_edges = pd.DataFrame(
    component_edge_rows
)


assert len(
    component_edges
) == 4097


# ============================================================
# 8. Development corpus / Fold0 exclusion
# ============================================================

cv = pd.read_csv(
    CV_PATH
)


dev = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .copy()
)


assert len(dev) == 103


dev_names = set(
    dev[
        "dataset"
    ]
)


assert set(
    membership[
        "dataset"
    ]
).issubset(
    dev_names
)


fold_map = dict(
    zip(
        dev[
            "dataset"
        ],
        dev[
            "fold"
        ].astype(int),
    )
)


# ============================================================
# 9. Optional parity against the Stage12.10C corpus
# ============================================================

if SUPPRESSED_PATH.exists():

    suppressed = pd.read_pickle(
        SUPPRESSED_PATH
    )

    assert suppressed[
        "dataset"
    ].nunique() == 103

    assert set(
        suppressed[
            "dataset"
        ]
    ) == dev_names

    assert set(
        suppressed[
            "fold"
        ].astype(int)
    ) == {
        1,
        2,
        3,
        4,
    }


    print(
        "\n========== DEVELOPMENT CORPUS PARITY =========="
    )

    print(
        "G2 development datasets:",
        len(
            dev_names
        )
    )

    print(
        "suppressed-peak universe datasets:",
        suppressed[
            "dataset"
        ].nunique()
    )

    print(
        "dataset universe exact:",
        True
    )


# ============================================================
# 10. Clean-chain reconstruction
# ============================================================

audit_rows = []

component_rows = []

selection_counts = {
    "raw_components":
        int(
            membership[
                "component_id"
            ].nunique()
        ),

    "multi_node":
        0,

    "two_sided":
        0,

    "temporal_chain":
        0,

    "ordinary_gt_chain":
        0,

    "p995_boundaries":
        0,
}


# Process one dataset at a time so graphs are only loaded once.
component_groups = (
    membership.groupby(
        [
            "dataset",
            "component_id",
        ]
    )
)


datasets_with_components = sorted(
    membership[
        "dataset"
    ].unique()
)


for ds_i, name in enumerate(
    datasets_with_components,
    start=1,
):

    # --------------------------------------------------------
    # GT graph
    # --------------------------------------------------------

    gt = fv9.load_graph(
        TRAIN_ROOT
        / f"{name}.geff"
    )


    gt_nodes = (
        gt.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )


    gt_edges = (
        gt.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    gt_indeg = (
        gt_edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    gt_outdeg = (
        gt_edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    # --------------------------------------------------------
    # Raw production P995 graph
    #
    # This is used only for:
    #   1. proving boundary origin = B256_DET995
    #   2. reading predicted boundary coordinates
    #
    # G1 official matching already gave us the pred node IDs.
    # --------------------------------------------------------

    p995 = fv9.load_graph(
        P995_ROOT
        / f"{name}.geff"
    )


    p995_nodes = (
        p995.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )


    ds_membership = membership[
        membership[
            "dataset"
        ].eq(
            name
        )
    ]


    for cid in sorted(
        ds_membership[
            "component_id"
        ].unique()
    ):

        mem = (
            ds_membership[
                ds_membership[
                    "component_id"
                ].eq(
                    cid
                )
            ]
            .copy()
        )


        gap_size = len(
            mem
        )


        # 12.11D clean-gap audit was multi-node only.
        if gap_size < 2:

            continue


        selection_counts[
            "multi_node"
        ] += 1


        edges = component_edges[
            (
                component_edges[
                    "dataset"
                ].eq(
                    name
                )
            )
            &
            (
                component_edges[
                    "component_id"
                ].eq(
                    cid
                )
            )
        ]


        in_edges = edges[
            edges[
                "edge_type"
            ].eq(
                "IN_BOUNDARY"
            )
        ]


        out_edges = edges[
            edges[
                "edge_type"
            ].eq(
                "OUT_BOUNDARY"
            )
        ]


        internal_edges = edges[
            edges[
                "edge_type"
            ].eq(
                "INTERNAL"
            )
        ]


        # Exact two-sided boundary requirement.
        if (
            len(
                in_edges
            ) != 1
            or
            len(
                out_edges
            ) != 1
        ):

            continue


        selection_counts[
            "two_sided"
        ] += 1


        in_edge = in_edges.iloc[
            0
        ]

        out_edge = out_edges.iloc[
            0
        ]


        boundary_in_gt = int(
            in_edge[
                "gt_source_id"
            ]
        )

        boundary_out_gt = int(
            out_edge[
                "gt_target_id"
            ]
        )


        assert boundary_in_gt in gt_nodes.index
        assert boundary_out_gt in gt_nodes.index


        ta = int(
            gt_nodes.loc[
                boundary_in_gt,
                "t",
            ]
        )

        tb = int(
            gt_nodes.loc[
                boundary_out_gt,
                "t",
            ]
        )


        # Must contain at least one intermediate frame.
        if tb <= ta + 1:

            continue


        missing_ids = (
            mem[
                "gt_node_id"
            ]
            .astype(int)
            .tolist()
        )


        missing_info = []


        for node_id in missing_ids:

            assert node_id in gt_nodes.index

            row = gt_nodes.loc[
                node_id
            ]


            missing_info.append(
                (
                    int(
                        row[
                            "t"
                        ]
                    ),
                    int(
                        node_id
                    ),
                )
            )


        missing_info = sorted(
            missing_info
        )


        missing_times = np.array(
            [
                t
                for t, _
                in missing_info
            ],
            dtype=int,
        )


        expected_times = np.arange(
            ta + 1,
            tb,
            dtype=int,
        )


        # ----------------------------------------------------
        # Exact temporal clean-gap condition:
        #
        # one and only one missing node at EVERY intermediate
        # frame and no missing node outside boundary interval.
        # ----------------------------------------------------

        if not np.array_equal(
            missing_times,
            expected_times,
        ):

            continue


        if gap_size != (
            tb
            -
            ta
            -
            1
        ):

            continue


        # Exact chain identities.
        time_to_missing = {
            t:
                node_id

            for t, node_id
            in missing_info
        }


        first_missing = int(
            time_to_missing[
                ta + 1
            ]
        )

        last_missing = int(
            time_to_missing[
                tb - 1
            ]
        )


        if int(
            in_edge[
                "gt_target_id"
            ]
        ) != first_missing:

            continue


        if int(
            out_edge[
                "gt_source_id"
            ]
        ) != last_missing:

            continue


        expected_internal_pairs = set()


        for t in range(
            ta + 1,
            tb - 1,
        ):

            expected_internal_pairs.add(
                (
                    int(
                        time_to_missing[
                            t
                        ]
                    ),
                    int(
                        time_to_missing[
                            t + 1
                        ]
                    ),
                )
            )


        observed_internal_pairs = set(
            zip(
                internal_edges[
                    "gt_source_id"
                ].astype(int),

                internal_edges[
                    "gt_target_id"
                ].astype(int),
            )
        )


        if (
            observed_internal_pairs
            !=
            expected_internal_pairs
        ):

            continue


        selection_counts[
            "temporal_chain"
        ] += 1


        # ----------------------------------------------------
        # Ordinary non-division/non-branch topology.
        # ----------------------------------------------------

        ordinary = True


        # Observed incoming GT boundary cannot be a division.
        if int(
            gt_outdeg.get(
                boundary_in_gt,
                0,
            )
        ) != 1:

            ordinary = False


        # Observed outgoing GT boundary must have one parent.
        if int(
            gt_indeg.get(
                boundary_out_gt,
                0,
            )
        ) != 1:

            ordinary = False


        # Every missing GT node must be an ordinary chain node.
        for node_id in missing_ids:

            if int(
                gt_indeg.get(
                    node_id,
                    0,
                )
            ) != 1:

                ordinary = False
                break


            if int(
                gt_outdeg.get(
                    node_id,
                    0,
                )
            ) != 1:

                ordinary = False
                break


        if not ordinary:

            continue


        selection_counts[
            "ordinary_gt_chain"
        ] += 1


        # ----------------------------------------------------
        # Predicted boundaries from the exact G1 matching map.
        # ----------------------------------------------------

        boundary_in_pred = int(
            in_edge[
                "pred_source_id"
            ]
        )

        boundary_out_pred = int(
            out_edge[
                "pred_target_id"
            ]
        )


        assert boundary_in_pred >= 0
        assert boundary_out_pred >= 0


        # Exact Stage12.11E boundary-origin contract:
        # both must be original B256 DET995 detections.
        if (
            boundary_in_pred
            not in p995_nodes.index
            or
            boundary_out_pred
            not in p995_nodes.index
        ):

            continue


        selection_counts[
            "p995_boundaries"
        ] += 1


        # Predicted boundary time must match GT boundary time.
        pred_ta = int(
            p995_nodes.loc[
                boundary_in_pred,
                "t",
            ]
        )

        pred_tb = int(
            p995_nodes.loc[
                boundary_out_pred,
                "t",
            ]
        )


        assert pred_ta == ta, (
            name,
            cid,
            pred_ta,
            ta,
        )

        assert pred_tb == tb, (
            name,
            cid,
            pred_tb,
            tb,
        )


        # ----------------------------------------------------
        # Boundary coordinates
        # ----------------------------------------------------

        gt_in = (
            gt_nodes.loc[
                boundary_in_gt,
                [
                    "z",
                    "y",
                    "x",
                ],
            ]
            .to_numpy(
                dtype=float
            )
        )


        gt_out = (
            gt_nodes.loc[
                boundary_out_gt,
                [
                    "z",
                    "y",
                    "x",
                ],
            ]
            .to_numpy(
                dtype=float
            )
        )


        pred_in = (
            p995_nodes.loc[
                boundary_in_pred,
                [
                    "z",
                    "y",
                    "x",
                ],
            ]
            .to_numpy(
                dtype=float
            )
        )


        pred_out = (
            p995_nodes.loc[
                boundary_out_pred,
                [
                    "z",
                    "y",
                    "x",
                ],
            ]
            .to_numpy(
                dtype=float
            )
        )


        boundary_in_error = float(
            np.linalg.norm(
                (
                    pred_in
                    -
                    gt_in
                )
                *
                SCALE
            )
        )


        boundary_out_error = float(
            np.linalg.norm(
                (
                    pred_out
                    -
                    gt_out
                )
                *
                SCALE
            )
        )


        # ----------------------------------------------------
        # Missing-node interpolation rows
        # ----------------------------------------------------

        component_node_rows = []


        for t, gt_node_id in missing_info:

            alpha = float(
                (
                    t
                    -
                    ta
                )
                /
                (
                    tb
                    -
                    ta
                )
            )


            truth = (
                gt_nodes.loc[
                    gt_node_id,
                    [
                        "z",
                        "y",
                        "x",
                    ],
                ]
                .to_numpy(
                    dtype=float
                )
            )


            # GT-boundary intrinsic interpolation.
            interp = (
                (
                    1.0
                    -
                    alpha
                )
                *
                gt_in
                +
                alpha
                *
                gt_out
            )


            intrinsic_error = float(
                np.linalg.norm(
                    (
                        interp
                        -
                        truth
                    )
                    *
                    SCALE
                )
            )


            # Predicted-boundary interpolation.
            pred_interp = (
                (
                    1.0
                    -
                    alpha
                )
                *
                pred_in
                +
                alpha
                *
                pred_out
            )


            pred_interp_error = float(
                np.linalg.norm(
                    (
                        pred_interp
                        -
                        truth
                    )
                    *
                    SCALE
                )
            )


            row = {
                "dataset":
                    name,

                "fold":
                    int(
                        fold_map[
                            name
                        ]
                    ),

                "component_id":
                    int(
                        cid
                    ),

                "gap_size":
                    int(
                        gap_size
                    ),

                "boundary_in_gt_node_id":
                    int(
                        boundary_in_gt
                    ),

                "boundary_out_gt_node_id":
                    int(
                        boundary_out_gt
                    ),

                "boundary_t0":
                    int(
                        ta
                    ),

                "boundary_t1":
                    int(
                        tb
                    ),

                "gt_node_id":
                    int(
                        gt_node_id
                    ),

                "t":
                    int(
                        t
                    ),

                "alpha":
                    float(
                        alpha
                    ),

                "truth_z":
                    float(
                        truth[
                            0
                        ]
                    ),

                "truth_y":
                    float(
                        truth[
                            1
                        ]
                    ),

                "truth_x":
                    float(
                        truth[
                            2
                        ]
                    ),

                "interp_z":
                    float(
                        interp[
                            0
                        ]
                    ),

                "interp_y":
                    float(
                        interp[
                            1
                        ]
                    ),

                "interp_x":
                    float(
                        interp[
                            2
                        ]
                    ),

                "error_um":
                    float(
                        intrinsic_error
                    ),

                "boundary_in_pred_node_id":
                    int(
                        boundary_in_pred
                    ),

                "boundary_out_pred_node_id":
                    int(
                        boundary_out_pred
                    ),

                "pred_in_z":
                    float(
                        pred_in[
                            0
                        ]
                    ),

                "pred_in_y":
                    float(
                        pred_in[
                            1
                        ]
                    ),

                "pred_in_x":
                    float(
                        pred_in[
                            2
                        ]
                    ),

                "pred_out_z":
                    float(
                        pred_out[
                            0
                        ]
                    ),

                "pred_out_y":
                    float(
                        pred_out[
                            1
                        ]
                    ),

                "pred_out_x":
                    float(
                        pred_out[
                            2
                        ]
                    ),

                "pred_in_origin":
                    "B256_DET995",

                "pred_out_origin":
                    "B256_DET995",

                "pred_interp_z":
                    float(
                        pred_interp[
                            0
                        ]
                    ),

                "pred_interp_y":
                    float(
                        pred_interp[
                            1
                        ]
                    ),

                "pred_interp_x":
                    float(
                        pred_interp[
                            2
                        ]
                    ),

                "pred_interp_error_um":
                    float(
                        pred_interp_error
                    ),
            }


            audit_rows.append(
                row
            )

            component_node_rows.append(
                row
            )


        # ----------------------------------------------------
        # Component summary
        # ----------------------------------------------------

        cdf = pd.DataFrame(
            component_node_rows
        )


        comp = {
            "dataset":
                name,

            "fold":
                int(
                    fold_map[
                        name
                    ]
                ),

            "component_id":
                int(
                    cid
                ),

            "gap_size":
                int(
                    gap_size
                ),

            "boundary_t0":
                int(
                    ta
                ),

            "boundary_t1":
                int(
                    tb
                ),

            "boundary_in_gt_node_id":
                int(
                    boundary_in_gt
                ),

            "boundary_out_gt_node_id":
                int(
                    boundary_out_gt
                ),

            "boundary_in_pred_node_id":
                int(
                    boundary_in_pred
                ),

            "boundary_out_pred_node_id":
                int(
                    boundary_out_pred
                ),

            "pred_in_origin":
                "B256_DET995",

            "pred_out_origin":
                "B256_DET995",

            "boundary_in_error_um":
                float(
                    boundary_in_error
                ),

            "boundary_out_error_um":
                float(
                    boundary_out_error
                ),

            "intrinsic_mean_error_um":
                float(
                    cdf[
                        "error_um"
                    ].mean()
                ),

            "intrinsic_median_error_um":
                float(
                    cdf[
                        "error_um"
                    ].median()
                ),

            "intrinsic_max_error_um":
                float(
                    cdf[
                        "error_um"
                    ].max()
                ),

            "pred_mean_error_um":
                float(
                    cdf[
                        "pred_interp_error_um"
                    ].mean()
                ),

            "pred_median_error_um":
                float(
                    cdf[
                        "pred_interp_error_um"
                    ].median()
                ),

            "pred_max_error_um":
                float(
                    cdf[
                        "pred_interp_error_um"
                    ].max()
                ),
        }


        for radius in RADII:

            suffix = int(
                radius
            )


            comp[
                f"intrinsic_full_{suffix}um"
            ] = bool(
                cdf[
                    "error_um"
                ].le(
                    radius
                ).all()
            )


            comp[
                f"pred_full_{suffix}um"
            ] = bool(
                cdf[
                    "pred_interp_error_um"
                ].le(
                    radius
                ).all()
            )


        component_rows.append(
            comp
        )


    if (
        ds_i == 1
        or ds_i % 10 == 0
        or ds_i == len(
            datasets_with_components
        )
    ):

        print(
            f"{ds_i:3d}/"
            f"{len(datasets_with_components)}"
            f" | {name}"
        )


# ============================================================
# 11. Canonical tables
# ============================================================

audit = pd.DataFrame(
    audit_rows
)

components = pd.DataFrame(
    component_rows
)


assert len(
    audit
) > 0

assert len(
    components
) > 0


assert audit[
    "dataset"
].nunique() <= 103

assert set(
    audit[
        "fold"
    ].astype(int)
).issubset(
    {
        1,
        2,
        3,
        4,
    }
)


assert audit[
    "pred_in_origin"
].eq(
    "B256_DET995"
).all()

assert audit[
    "pred_out_origin"
].eq(
    "B256_DET995"
).all()


assert int(
    components[
        "gap_size"
    ].sum()
) == len(
    audit
)


# Every clean component must have exactly one node per gap frame.
for r in components.itertuples(
    index=False
):

    n = int(
        (
            (
                audit[
                    "dataset"
                ].eq(
                    r.dataset
                )
            )
            &
            (
                audit[
                    "component_id"
                ].eq(
                    int(
                        r.component_id
                    )
                )
            )
        ).sum()
    )


    assert n == int(
        r.gap_size
    )


# ============================================================
# 12. Selection funnel
# ============================================================

print(
    "\n========== CLEAN-GAP SELECTION FUNNEL =========="
)


for key, value in selection_counts.items():

    print(
        f"{key:22s}:",
        value
    )


print(
    "\nfinal clean components:",
    len(
        components
    )
)

print(
    "final missing nodes:",
    len(
        audit
    )
)

print(
    "datasets:",
    audit[
        "dataset"
    ].nunique()
)


# ============================================================
# 13. Boundary localization
# ============================================================

boundary_errors = np.concatenate(
    [
        components[
            "boundary_in_error_um"
        ].to_numpy(
            dtype=float
        ),

        components[
            "boundary_out_error_um"
        ].to_numpy(
            dtype=float
        ),
    ]
)


print(
    "\n========== BOUNDARY LOCALIZATION =========="
)


boundary_q = [
    .10,
    .25,
    .50,
    .75,
    .90,
    .95,
    .99,
]


for q, value in zip(
    boundary_q,
    np.quantile(
        boundary_errors,
        boundary_q,
    ),
):

    print(
        f"{q:.2f}:",
        f"{value:.6f}"
    )


# ============================================================
# 14. Intrinsic vs predicted-boundary geometry
# ============================================================

print(
    "\n========== INTERPOLATION ERROR QUANTILES =========="
)


quantiles = [
    .10,
    .25,
    .50,
    .75,
    .90,
    .95,
    .99,
]


q_table = pd.DataFrame(
    {
        "intrinsic_gt_boundary":
            audit[
                "error_um"
            ].quantile(
                quantiles
            ),

        "predicted_boundary":
            audit[
                "pred_interp_error_um"
            ].quantile(
                quantiles
            ),
    }
)


print(
    q_table.to_string()
)


print(
    "\nmedian intrinsic:",
    f"{audit['error_um'].median():.6f}",
    "um"
)

print(
    "median predicted-boundary:",
    f"{audit['pred_interp_error_um'].median():.6f}",
    "um"
)


# ============================================================
# 15. Node recoverability
# ============================================================

print(
    "\n========== NODE RECOVERABILITY =========="
)


node_recovery = {}


for radius in RADII:

    intrinsic_good = int(
        audit[
            "error_um"
        ]
        .le(
            radius
        )
        .sum()
    )


    pred_good = int(
        audit[
            "pred_interp_error_um"
        ]
        .le(
            radius
        )
        .sum()
    )


    node_recovery[
        str(
            int(
                radius
            )
        )
    ] = {
        "intrinsic":
            intrinsic_good,

        "predicted_boundary":
            pred_good,
    }


    print(
        f"<= {radius:.0f} um | "
        f"intrinsic "
        f"{intrinsic_good}/{len(audit)} "
        f"({intrinsic_good/len(audit):.3%}) | "
        f"predicted "
        f"{pred_good}/{len(audit)} "
        f"({pred_good/len(audit):.3%})"
    )


# ============================================================
# 16. Full-component recoverability
# ============================================================

print(
    "\n========== FULL-COMPONENT RECOVERABILITY =========="
)


component_recovery = {}


for radius in RADII:

    suffix = int(
        radius
    )


    intrinsic_col = (
        f"intrinsic_full_{suffix}um"
    )

    pred_col = (
        f"pred_full_{suffix}um"
    )


    intrinsic_good = int(
        components[
            intrinsic_col
        ].sum()
    )


    pred_good = int(
        components[
            pred_col
        ].sum()
    )


    component_recovery[
        str(
            suffix
        )
    ] = {
        "intrinsic":
            intrinsic_good,

        "predicted_boundary":
            pred_good,
    }


    print(
        f"full <= {radius:.0f} um | "
        f"intrinsic "
        f"{intrinsic_good}/{len(components)} "
        f"({intrinsic_good/len(components):.3%}) | "
        f"predicted "
        f"{pred_good}/{len(components)} "
        f"({pred_good/len(components):.3%})"
    )


# ============================================================
# 17. Results by fold
# ============================================================

print(
    "\n========== RESULTS BY FOLD =========="
)


fold_table = (
    components.groupby(
        "fold"
    )
    .agg(
        components=(
            "component_id",
            "size",
        ),

        missing_nodes=(
            "gap_size",
            "sum",
        ),

        median_gap_size=(
            "gap_size",
            "median",
        ),

        median_boundary_in_error_um=(
            "boundary_in_error_um",
            "median",
        ),

        median_boundary_out_error_um=(
            "boundary_out_error_um",
            "median",
        ),

        median_pred_error_um=(
            "pred_median_error_um",
            "median",
        ),

        pred_full_3um=(
            "pred_full_3um",
            "sum",
        ),

        pred_full_5um=(
            "pred_full_5um",
            "sum",
        ),

        pred_full_7um=(
            "pred_full_7um",
            "sum",
        ),
    )
)


print(
    fold_table.to_string()
)


# ============================================================
# 18. Results by gap size
# ============================================================

print(
    "\n========== RESULTS BY GAP SIZE =========="
)


gap_table = (
    components.groupby(
        "gap_size"
    )
    .agg(
        components=(
            "component_id",
            "size",
        ),

        missing_nodes=(
            "gap_size",
            "sum",
        ),

        intrinsic_median_mean_error_um=(
            "intrinsic_mean_error_um",
            "median",
        ),

        pred_median_mean_error_um=(
            "pred_mean_error_um",
            "median",
        ),

        pred_median_max_error_um=(
            "pred_max_error_um",
            "median",
        ),

        pred_full_3um=(
            "pred_full_3um",
            "sum",
        ),

        pred_full_5um=(
            "pred_full_5um",
            "sum",
        ),

        pred_full_7um=(
            "pred_full_7um",
            "sum",
        ),
    )
)


print(
    gap_table.to_string()
)


# ============================================================
# 19. Persist
# ============================================================

audit = (
    audit.sort_values(
        [
            "fold",
            "dataset",
            "component_id",
            "t",
        ]
    )
    .reset_index(
        drop=True
    )
)


components = (
    components.sort_values(
        [
            "fold",
            "dataset",
            "component_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


audit.to_pickle(
    NODE_OUT
)

components.to_pickle(
    COMP_OUT
)


payload = {
    "stage":
        "12.11G2",

    "status":
        "FOLDS14_PREDICTED_BOUNDARY_INTERPOLATION_AUDITED",

    "audit_type":
        "GT_DEFINED_DEVELOPMENT_ORACLE_GEOMETRY",

    "corpus":
        "folds1-4 6bba CURRENT_BEST",

    "folds":
        [
            1,
            2,
            3,
            4,
        ],

    "fold0_used":
        False,

    "gt_used":
        True,

    "policy_changed":
        False,

    "matching_source":
        (
            "stage12_folds14_current_best_"
            "edge_fn_residuals.pkl"
        ),

    "boundary_prediction_source":
        (
            "stage11_fullgraph_baseline256_"
            "folds1to4_det995"
        ),

    "boundary_origin_required":
        "B256_DET995",

    "physical_scale_um_per_voxel":
        SCALE.tolist(),

    "input": {
        "current_best_edge_fn":
            int(
                len(
                    fn
                )
            ),

        "detection_limited_fn":
            int(
                len(
                    det
                )
            ),

        "unique_missing_gt_nodes":
            int(
                len(
                    missing_nodes
                )
            ),

        "both_missing_edges":
            int(
                len(
                    both
                )
            ),
    },

    "selection_funnel":
        {
            k:
                int(
                    v
                )

            for k, v
            in selection_counts.items()
        },

    "clean_population": {
        "components":
            int(
                len(
                    components
                )
            ),

        "missing_nodes":
            int(
                len(
                    audit
                )
            ),

        "datasets":
            int(
                audit[
                    "dataset"
                ].nunique()
            ),
    },

    "geometry": {
        "median_intrinsic_error_um":
            float(
                audit[
                    "error_um"
                ].median()
            ),

        "median_predicted_boundary_error_um":
            float(
                audit[
                    "pred_interp_error_um"
                ].median()
            ),

        "node_recoverability":
            node_recovery,

        "component_recoverability":
            component_recovery,
    },

    "scientific_constraint": (
        "Clean-gap membership is GT-defined and therefore "
        "cannot itself be used as a deployment trigger. "
        "A separate GT-blind observable gap-candidate universe "
        "must be constructed before policy development."
    ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        payload,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    NODE_OUT
)

print(
    "written:",
    COMP_OUT
)

print(
    "written:",
    SUMMARY_OUT
)


# ============================================================
# 20. Artifact integrity
# ============================================================

assert NODE_OUT.exists()
assert COMP_OUT.exists()
assert SUMMARY_OUT.exists()


audit_check = pd.read_pickle(
    NODE_OUT
)

components_check = pd.read_pickle(
    COMP_OUT
)


assert len(
    audit_check
) == len(
    audit
)

assert len(
    components_check
) == len(
    components
)

assert int(
    components_check[
        "gap_size"
    ].sum()
) == len(
    audit_check
)


assert set(
    audit_check[
        "fold"
    ].astype(int)
).issubset(
    {
        1,
        2,
        3,
        4,
    }
)


assert audit_check[
    "pred_in_origin"
].eq(
    "B256_DET995"
).all()

assert audit_check[
    "pred_out_origin"
].eq(
    "B256_DET995"
).all()


# ============================================================
# 21. Final decision
# ============================================================

print(
    "\n========== 12.11G2 DECISION =========="
)

print(
    "FOLDS14_PREDICTED_BOUNDARY_INTERPOLATION_AUDIT: PASS"
)

print(
    "clean components:",
    len(
        components
    )
)

print(
    "clean missing nodes:",
    len(
        audit
    )
)

print(
    "median intrinsic error:",
    f"{audit['error_um'].median():.6f}",
    "um"
)

print(
    "median predicted-boundary error:",
    f"{audit['pred_interp_error_um'].median():.6f}",
    "um"
)

print(
    "Fold0 used: NO"
)

print(
    "GT used: DEVELOPMENT ORACLE AUDIT ONLY"
)

print(
    "deployment trigger learned: NO"
)

print(
    "policy changed: NO"
)

print(
    "next:"
)

print(
    "BUILD_GT_BLIND_DEPLOYABLE_GAP_CANDIDATE_UNIVERSE"
)

In [ ]:
# ============================================================
# 1. Self-contained CURRENT_BEST graph-builder bootstrap
#
# Purpose:
#   Make Stage 12.11G3A reproducible even after kernel restart.
#
#   This reconstructs ONLY the prediction-side CURRENT_BEST
#   graph builder. It does NOT rerun G1 GT residual evaluation.
#
# GT used here: NO
# Fold0 used here: NO
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

PRED_ROOT = (
    PROJECT
    / "predictions/heqiuyan"
)

S11 = (
    PROJECT
    / "reports"
    / "stage11_division_probe"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


# ------------------------------------------------------------
# Development Baseline256 prediction roots
# ------------------------------------------------------------

B256_995 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det995"
    / "split_0"
)

B256_9975 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det9975"
    / "split_0"
)


# ------------------------------------------------------------
# Exact promoted development action sources
# ------------------------------------------------------------

MORPH_GRID_PATH = (
    S11
    / "stage11_deployable_final_v9_policy_grid_actions.pkl"
)

V1_UNIVERSE_PATH = (
    S12
    / "stage12_folds14_mutual_best_universe.pkl"
)

V2_UNIVERSE_PATH = (
    S12
    / "stage12_folds14_round2_reciprocal_universe.pkl"
)

BRIDGE_ACTION_PATH = (
    S12
    / "stage12_detection_bridge_policy_actions.pkl"
)


for path in [
    TRAIN_ROOT,
    CV_PATH,
    B256_995,
    B256_9975,
    MORPH_GRID_PATH,
    V1_UNIVERSE_PATH,
    V2_UNIVERSE_PATH,
    BRIDGE_ACTION_PATH,
]:

    assert path.exists(), path


# ------------------------------------------------------------
# Project import
# ------------------------------------------------------------

SRC_PATH = str(
    PROJECT
    / "src"
)

if SRC_PATH not in sys.path:

    sys.path.insert(
        0,
        SRC_PATH,
    )


from biohub_cell_tracking import frozen_v9 as fv9


# ------------------------------------------------------------
# Development 6bba corpus
# ------------------------------------------------------------

bootstrap_cv = pd.read_csv(
    CV_PATH
)


bootstrap_corp = (
    bootstrap_cv[
        bootstrap_cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        bootstrap_cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(
    bootstrap_corp
) == 103

assert bootstrap_corp[
    "dataset"
].nunique() == 103

assert not bootstrap_corp[
    "fold"
].eq(
    0
).any()


bootstrap_names = (
    bootstrap_corp[
        "dataset"
    ]
    .tolist()
)


bootstrap_name_set = set(
    bootstrap_names
)


bootstrap_prefix_map = dict(
    zip(
        bootstrap_corp[
            "dataset"
        ],
        bootstrap_corp[
            "prefix"
        ],
    )
)


# ------------------------------------------------------------
# Exact MORPH-DIV-V2 development actions
# ------------------------------------------------------------

bootstrap_morph_all = pd.read_pickle(
    MORPH_GRID_PATH
)


assert (
    "config_id"
    in bootstrap_morph_all.columns
)


bootstrap_morph_actions = (
    bootstrap_morph_all[
        bootstrap_morph_all[
            "config_id"
        ].eq(
            "K3_R3_P100"
        )
    ]
    .copy()
)


assert set(
    bootstrap_morph_actions[
        "dataset"
    ]
).issubset(
    bootstrap_name_set
)


# ------------------------------------------------------------
# Exact MUTUAL-CONT-V1 development actions
# ------------------------------------------------------------

bootstrap_v1_universe = pd.read_pickle(
    V1_UNIVERSE_PATH
)


bootstrap_v1_actions = (
    bootstrap_v1_universe[
        bootstrap_v1_universe[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    bootstrap_v1_actions
) == 46951


assert set(
    bootstrap_v1_actions[
        "dataset"
    ]
).issubset(
    bootstrap_name_set
)


# ------------------------------------------------------------
# Exact MUTUAL-CONT-V2 development actions
# ------------------------------------------------------------

bootstrap_v2_universe = pd.read_pickle(
    V2_UNIVERSE_PATH
)


bootstrap_v2_actions = (
    bootstrap_v2_universe[
        bootstrap_v2_universe[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    bootstrap_v2_actions
) == 18511


assert set(
    bootstrap_v2_actions[
        "dataset"
    ]
).issubset(
    bootstrap_name_set
)


# ------------------------------------------------------------
# Exact DET-BRIDGE-R1P50 development actions
# ------------------------------------------------------------

bootstrap_bridge_all = pd.read_pickle(
    BRIDGE_ACTION_PATH
)


bootstrap_r1p50_actions = (
    bootstrap_bridge_all[
        bootstrap_bridge_all[
            "policy_id"
        ].eq(
            "R1P50"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


assert len(
    bootstrap_r1p50_actions
) == 805


assert bootstrap_r1p50_actions[
    "bridge_min_prob"
].ge(
    0.50
).all()


assert set(
    bootstrap_r1p50_actions[
        "dataset"
    ]
).issubset(
    bootstrap_name_set
)


print(
    "========== G3A CURRENT_BEST BOOTSTRAP =========="
)

print(
    "development datasets:",
    len(
        bootstrap_names
    )
)

print(
    "MORPH-DIV-V2:",
    len(
        bootstrap_morph_actions
    )
)

print(
    "MUTUAL-CONT-V1:",
    len(
        bootstrap_v1_actions
    )
)

print(
    "MUTUAL-CONT-V2:",
    len(
        bootstrap_v2_actions
    )
)

print(
    "DET-BRIDGE-R1P50:",
    len(
        bootstrap_r1p50_actions
    )
)


# ------------------------------------------------------------
# Frozen-V9 policy object
# ------------------------------------------------------------

bootstrap_policies = (
    fv9.build_frozen_v9_policies(
        bootstrap_names,
        B256_995,
        B256_9975,
        TRAIN_ROOT,
        prefix_map=
            bootstrap_prefix_map,
        config=
            fv9.FROZEN_V9_CONFIG,
        progress=False,
    )
)


assert (
    type(
        bootstrap_policies
    ).__name__
    ==
    "FrozenV9Policies"
)


print(
    "Frozen-V9 policy object:",
    type(
        bootstrap_policies
    ).__name__,
)


# ============================================================
# Graph-state helper
# ============================================================

def graph_state(
    graph,
):

    state_nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )


    state_edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    state_outdeg = (
        state_edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    state_indeg = (
        state_edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    state_edge_set = set(
        zip(
            state_edges[
                "source_id"
            ].astype(int),

            state_edges[
                "target_id"
            ].astype(int),
        )
    )


    return (
        state_nodes,
        state_outdeg,
        state_indeg,
        state_edge_set,
    )


# ============================================================
# Continuation action helper
# ============================================================

def add_continuation_actions(
    graph,
    nodes,
    table,
    name,
):

    action_rows = table[
        table[
            "dataset"
        ].eq(
            name
        )
    ]


    for action in action_rows.itertuples(
        index=False
    ):

        src = int(
            action.source_pred_node_id
        )

        tgt = int(
            action.target_pred_node_id
        )


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(
                        dist
                    ),

                "edge_prob":
                    float(
                        action.candidate_prob
                    ),
            },
            validate_keys=True,
        )


# ============================================================
# Exact development CURRENT_BEST builder
# ============================================================

def build_dev_current_best(
    name,
):

    assert name in bootstrap_name_set


    graph, scale, _ = (
        fv9.apply_frozen_v9(
            name,
            B256_995,
            B256_9975,
            bootstrap_policies,
            TRAIN_ROOT,
            config=
                fv9.FROZEN_V9_CONFIG,
        )
    )


    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = graph_state(
        graph
    )


    # --------------------------------------------------------
    # MORPH-DIV-V2
    # --------------------------------------------------------

    morph = (
        bootstrap_morph_actions[
            bootstrap_morph_actions[
                "dataset"
            ].eq(
                name
            )
        ]
        .sort_values(
            [
                "morph_rank",
                "t",
                "source_pred_node_id",
                "target_pred_node_id",
            ],
            kind="stable",
        )
    )


    for action in morph.itertuples(
        index=False
    ):

        src = int(
            action.source_pred_node_id
        )

        tgt = int(
            action.target_pred_node_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 1


        assert indeg.get(
            tgt,
            0,
        ) == 0


        assert (
            src,
            tgt,
        ) not in edge_set


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(
                        dist
                    ),

                "edge_prob":
                    float(
                        action.candidate_prob
                    ),
            },
            validate_keys=True,
        )


        outdeg[
            src
        ] = 2

        indeg[
            tgt
        ] = 1

        edge_set.add(
            (
                src,
                tgt,
            )
        )


    # --------------------------------------------------------
    # MUTUAL-CONT-V1
    # --------------------------------------------------------

    add_continuation_actions(
        graph,
        nodes,
        bootstrap_v1_actions,
        name,
    )


    # --------------------------------------------------------
    # MUTUAL-CONT-V2
    # --------------------------------------------------------

    add_continuation_actions(
        graph,
        nodes,
        bootstrap_v2_actions,
        name,
    )


    # --------------------------------------------------------
    # Re-read exact post-V2 graph state
    # --------------------------------------------------------

    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = graph_state(
        graph
    )


    # --------------------------------------------------------
    # DET-BRIDGE-R1P50
    # --------------------------------------------------------

    bridge = (
        bootstrap_r1p50_actions[
            bootstrap_r1p50_actions[
                "dataset"
            ].eq(
                name
            )
        ]
    )


    for action in bridge.itertuples(
        index=False
    ):

        src = int(
            action.source_id
        )

        tgt = int(
            action.target_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 0


        assert indeg.get(
            tgt,
            0,
        ) == 0


        new_id = graph.add_node(
            {
                "t":
                    int(
                        action.t
                    ),

                "z":
                    float(
                        action.z
                    ),

                "y":
                    float(
                        action.y
                    ),

                "x":
                    float(
                        action.x
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            src,
            new_id,
            {
                "edge_dist":
                    float(
                        action.src_dist_um
                    ),

                "edge_prob":
                    float(
                        action.in_prob
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            new_id,
            tgt,
            {
                "edge_dist":
                    float(
                        action.tgt_dist_um
                    ),

                "edge_prob":
                    float(
                        action.out_prob
                    ),
            },
            validate_keys=True,
        )


        outdeg[
            src
        ] = 1

        indeg[
            new_id
        ] = 1

        outdeg[
            new_id
        ] = 1

        indeg[
            tgt
        ] = 1


    return (
        graph,
        scale,
    )


# ============================================================
# Bootstrap integrity
# ============================================================

assert callable(
    build_dev_current_best
)


print(
    "\n========== G3A BOOTSTRAP DECISION =========="
)

print(
    "SELF_CONTAINED_CURRENT_BEST_BUILDER: PASS"
)

print(
    "GT used: NO"
)

print(
    "Fold0 used: NO"
)

print(
    "ready for GT-blind candidate generation: YES"
)

In [ ]:
# ============================================================
# Stage 12.11G3A
# GT-Blind Free/Free Gap Endpoint Candidate Universe
#
# SELF-CONTAINED VERSION
#
# PART A:
#   Build CURRENT_BEST prediction graph on folds1-4.
#   Generate GT-BLIND free-source -> free-target gap candidates.
#   Persist candidate universe BEFORE reading any G2 GT artifact.
#
# PART B:
#   Read development G2 clean-gap oracle only after Part A freeze.
#   Measure whether true clean gaps exist in the deployable universe.
#
# No Fold0.
# No policy tuning.
# No threshold selection.
# No graph changes written to predictions.
# No synthetic nodes inserted as a new policy.
#
# Keep after running: YES
# ============================================================

from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS / CONSTANTS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

PRED_ROOT = (
    PROJECT
    / "predictions/heqiuyan"
)

S11 = (
    PROJECT
    / "reports"
    / "stage11_division_probe"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


B256_995 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det995"
    / "split_0"
)

B256_9975 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det9975"
    / "split_0"
)


MORPH_GRID_PATH = (
    S11
    / "stage11_deployable_final_v9_policy_grid_actions.pkl"
)

V1_UNIVERSE_PATH = (
    S12
    / "stage12_folds14_mutual_best_universe.pkl"
)

V2_UNIVERSE_PATH = (
    S12
    / "stage12_folds14_round2_reciprocal_universe.pkl"
)

BRIDGE_ACTION_PATH = (
    S12
    / "stage12_detection_bridge_policy_actions.pkl"
)


G2_COMPONENT_PATH = (
    S12
    / "stage12_folds14_predicted_boundary_interpolation_components.pkl"
)


CANDIDATE_OUT = (
    S12
    / "stage12_folds14_gap_endpoint_candidate_universe.pkl"
)

CANDIDATE_SUMMARY_OUT = (
    S12
    / "stage12_folds14_gap_endpoint_candidate_universe_summary.json"
)

ORACLE_OUT = (
    S12
    / "stage12_folds14_gap_endpoint_candidate_oracle_coverage.pkl"
)

ORACLE_SUMMARY_OUT = (
    S12
    / "stage12_folds14_gap_endpoint_candidate_oracle_summary.json"
)


MIN_GAP_SIZE = 2
MAX_GAP_SIZE = 32


SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


for path in [
    TRAIN_ROOT,
    CV_PATH,
    B256_995,
    B256_9975,
    MORPH_GRID_PATH,
    V1_UNIVERSE_PATH,
    V2_UNIVERSE_PATH,
    BRIDGE_ACTION_PATH,
]:

    assert path.exists(), path


S12.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. IMPORT PROJECT
# ============================================================

SRC = str(
    PROJECT / "src"
)

if SRC not in sys.path:
    sys.path.insert(
        0,
        SRC,
    )


from biohub_cell_tracking import frozen_v9 as fv9


# ============================================================
# 2. DEVELOPMENT CORPUS
# ============================================================

cv = pd.read_csv(
    CV_PATH
)


corp = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(corp) == 103
assert corp["dataset"].nunique() == 103
assert not corp["fold"].eq(0).any()

assert set(
    corp[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}


names = (
    corp[
        "dataset"
    ]
    .tolist()
)


name_set = set(
    names
)


fold_map = dict(
    zip(
        corp[
            "dataset"
        ],
        corp[
            "fold"
        ].astype(int),
    )
)


prefix_map = dict(
    zip(
        corp[
            "dataset"
        ],
        corp[
            "prefix"
        ],
    )
)


print(
    "========== 12.11G3A DEVELOPMENT CORPUS =========="
)

print(
    "datasets:",
    len(names)
)

print(
    "folds:",
    sorted(
        corp[
            "fold"
        ]
        .astype(int)
        .unique()
        .tolist()
    )
)

print(
    "gap horizon:",
    MIN_GAP_SIZE,
    "to",
    MAX_GAP_SIZE,
    "missing frames",
)


# ============================================================
# 3. LOAD EXACT CURRENT_BEST ACTION CONTRACTS
# ============================================================

morph_all = pd.read_pickle(
    MORPH_GRID_PATH
)


morph_actions = (
    morph_all[
        morph_all[
            "config_id"
        ].eq(
            "K3_R3_P100"
        )
    ]
    .copy()
)


v1_universe = pd.read_pickle(
    V1_UNIVERSE_PATH
)


v1_actions = (
    v1_universe[
        v1_universe[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


v2_universe = pd.read_pickle(
    V2_UNIVERSE_PATH
)


v2_actions = (
    v2_universe[
        v2_universe[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


bridge_all = pd.read_pickle(
    BRIDGE_ACTION_PATH
)


r1p50_actions = (
    bridge_all[
        bridge_all[
            "policy_id"
        ].eq(
            "R1P50"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


assert len(morph_actions) == 96
assert len(v1_actions) == 46951
assert len(v2_actions) == 18511
assert len(r1p50_actions) == 805


for label, table in [
    (
        "MORPH",
        morph_actions,
    ),
    (
        "V1",
        v1_actions,
    ),
    (
        "V2",
        v2_actions,
    ),
    (
        "R1P50",
        r1p50_actions,
    ),
]:

    assert set(
        table[
            "dataset"
        ]
    ).issubset(
        name_set
    ), label


assert r1p50_actions[
    "bridge_min_prob"
].ge(
    0.50
).all()


print(
    "\n========== G3A CURRENT_BEST BOOTSTRAP =========="
)

print(
    "development datasets:",
    len(names)
)

print(
    "MORPH-DIV-V2:",
    len(morph_actions)
)

print(
    "MUTUAL-CONT-V1:",
    len(v1_actions)
)

print(
    "MUTUAL-CONT-V2:",
    len(v2_actions)
)

print(
    "DET-BRIDGE-R1P50:",
    len(r1p50_actions)
)


# ============================================================
# 4. BUILD FROZEN V9 POLICY OBJECT
# ============================================================

policies = fv9.build_frozen_v9_policies(
    names,
    B256_995,
    B256_9975,
    TRAIN_ROOT,
    prefix_map=prefix_map,
    config=fv9.FROZEN_V9_CONFIG,
    progress=False,
)


assert (
    type(
        policies
    ).__name__
    ==
    "FrozenV9Policies"
)


print(
    "Frozen-V9 policy object:",
    type(
        policies
    ).__name__,
)


# ============================================================
# 5. CURRENT_BEST GRAPH HELPERS
# ============================================================

def graph_state(
    graph,
):

    state_nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )


    state_edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    outdeg = (
        state_edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = (
        state_edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = {
        int(k):
            int(v)

        for k, v
        in outdeg.items()
    }


    indeg = {
        int(k):
            int(v)

        for k, v
        in indeg.items()
    }


    edge_set = set(
        zip(
            state_edges[
                "source_id"
            ].astype(int),

            state_edges[
                "target_id"
            ].astype(int),
        )
    )


    return (
        state_nodes,
        outdeg,
        indeg,
        edge_set,
    )


def add_continuation_actions(
    graph,
    nodes,
    table,
    dataset,
):

    rows = table[
        table[
            "dataset"
        ].eq(
            dataset
        )
    ]


    for action in rows.itertuples(
        index=False
    ):

        src = int(
            action.source_pred_node_id
        )

        tgt = int(
            action.target_pred_node_id
        )


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(
                        dist
                    ),

                "edge_prob":
                    float(
                        action.candidate_prob
                    ),
            },
            validate_keys=True,
        )


def build_dev_current_best(
    dataset,
):

    assert dataset in name_set


    graph, scale, _ = fv9.apply_frozen_v9(
        dataset,
        B256_995,
        B256_9975,
        policies,
        TRAIN_ROOT,
        config=fv9.FROZEN_V9_CONFIG,
    )


    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = graph_state(
        graph
    )


    # --------------------------------------------------------
    # MORPH-DIV-V2
    # --------------------------------------------------------

    morph = (
        morph_actions[
            morph_actions[
                "dataset"
            ].eq(
                dataset
            )
        ]
        .sort_values(
            [
                "morph_rank",
                "t",
                "source_pred_node_id",
                "target_pred_node_id",
            ],
            kind="stable",
        )
    )


    for action in morph.itertuples(
        index=False
    ):

        src = int(
            action.source_pred_node_id
        )

        tgt = int(
            action.target_pred_node_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 1

        assert indeg.get(
            tgt,
            0,
        ) == 0

        assert (
            src,
            tgt,
        ) not in edge_set


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(
                        dist
                    ),

                "edge_prob":
                    float(
                        action.candidate_prob
                    ),
            },
            validate_keys=True,
        )


        outdeg[
            src
        ] = 2

        indeg[
            tgt
        ] = 1

        edge_set.add(
            (
                src,
                tgt,
            )
        )


    # --------------------------------------------------------
    # MUTUAL-CONT-V1
    # --------------------------------------------------------

    add_continuation_actions(
        graph,
        nodes,
        v1_actions,
        dataset,
    )


    # --------------------------------------------------------
    # MUTUAL-CONT-V2
    # --------------------------------------------------------

    add_continuation_actions(
        graph,
        nodes,
        v2_actions,
        dataset,
    )


    # --------------------------------------------------------
    # Exact post-V2 state
    # --------------------------------------------------------

    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = graph_state(
        graph
    )


    # --------------------------------------------------------
    # R1P50
    # --------------------------------------------------------

    bridge = (
        r1p50_actions[
            r1p50_actions[
                "dataset"
            ].eq(
                dataset
            )
        ]
    )


    for action in bridge.itertuples(
        index=False
    ):

        src = int(
            action.source_id
        )

        tgt = int(
            action.target_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 0

        assert indeg.get(
            tgt,
            0,
        ) == 0


        new_id = graph.add_node(
            {
                "t":
                    int(
                        action.t
                    ),

                "z":
                    float(
                        action.z
                    ),

                "y":
                    float(
                        action.y
                    ),

                "x":
                    float(
                        action.x
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            src,
            new_id,
            {
                "edge_dist":
                    float(
                        action.src_dist_um
                    ),

                "edge_prob":
                    float(
                        action.in_prob
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            new_id,
            tgt,
            {
                "edge_dist":
                    float(
                        action.tgt_dist_um
                    ),

                "edge_prob":
                    float(
                        action.out_prob
                    ),
            },
            validate_keys=True,
        )


        outdeg[
            src
        ] = 1

        indeg[
            new_id
        ] = 1

        outdeg[
            new_id
        ] = 1

        indeg[
            tgt
        ] = 1


    return (
        graph,
        scale,
    )


assert callable(
    build_dev_current_best
)


print(
    "\n========== G3A BOOTSTRAP DECISION =========="
)

print(
    "SELF_CONTAINED_CURRENT_BEST_BUILDER: PASS"
)

print(
    "GT used: NO"
)

print(
    "Fold0 used: NO"
)

print(
    "ready for GT-blind candidate generation: YES"
)


# ============================================================
# 6. FEATURE HELPERS
# ============================================================

def physical_xyz(
    row,
):

    return (
        np.array(
            [
                float(
                    row[
                        "z"
                    ]
                ),
                float(
                    row[
                        "y"
                    ]
                ),
                float(
                    row[
                        "x"
                    ]
                ),
            ],
            dtype=float,
        )
        *
        SCALE
    )


def vector_norm(
    x,
):

    return float(
        np.linalg.norm(
            np.asarray(
                x,
                dtype=float,
            )
        )
    )


def safe_angle_deg(
    a,
    b,
):

    a = np.asarray(
        a,
        dtype=float,
    )

    b = np.asarray(
        b,
        dtype=float,
    )


    na = float(
        np.linalg.norm(
            a
        )
    )

    nb = float(
        np.linalg.norm(
            b
        )
    )


    if (
        na <= 1e-12
        or
        nb <= 1e-12
    ):

        return np.nan


    cosv = float(
        np.dot(
            a,
            b,
        )
        /
        (
            na
            *
            nb
        )
    )


    cosv = float(
        np.clip(
            cosv,
            -1.0,
            1.0,
        )
    )


    return float(
        np.degrees(
            np.arccos(
                cosv
            )
        )
    )


# ============================================================
# PART A
# GT-BLIND CANDIDATE GENERATION
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART A — GT-BLIND CANDIDATE GENERATION"
)

print(
    "============================================================"
)


candidate_frames = []

endpoint_summary_rows = []

node_state = {}


# ============================================================
# 7. BUILD PREDICTION-SIDE ENDPOINT UNIVERSE
# ============================================================

for ds_i, dataset in enumerate(
    names,
    start=1,
):

    fold = int(
        fold_map[
            dataset
        ]
    )


    graph, graph_scale = (
        build_dev_current_best(
            dataset
        )
    )


    graph_scale = np.asarray(
        graph_scale,
        dtype=float,
    )


    assert np.allclose(
        graph_scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    )


    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = (
        nodes[
            "node_id"
        ]
        .astype(int)
    )


    nodes[
        "t"
    ] = (
        nodes[
            "t"
        ]
        .astype(int)
    )


    # FIX:
    # sort BEFORE node_id becomes an index.
    nodes = (
        nodes
        .sort_values(
            [
                "t",
                "node_id",
            ],
            kind="stable",
        )
        .set_index(
            "node_id",
            drop=False,
        )
    )


    # --------------------------------------------------------
    # Raw production P995 node identity = origin feature.
    # --------------------------------------------------------

    p995 = fv9.load_graph(
        B256_995
        / f"{dataset}.geff"
    )


    p995_ids = set(
        p995.node_attrs(
            attr_keys=[
                "node_id",
            ]
        )
        .to_pandas()[
            "node_id"
        ]
        .astype(int)
        .tolist()
    )


    # --------------------------------------------------------
    # Degrees
    # --------------------------------------------------------

    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = {
        int(k):
            int(v)

        for k, v
        in outdeg.items()
    }


    indeg = {
        int(k):
            int(v)

        for k, v
        in indeg.items()
    }


    node_ids = (
        nodes[
            "node_id"
        ]
        .astype(int)
        .tolist()
    )


    # --------------------------------------------------------
    # Unique predecessor / successor
    # --------------------------------------------------------

    incoming = (
        edges.groupby(
            "target_id"
        )[
            "source_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    outgoing = (
        edges.groupby(
            "source_id"
        )[
            "target_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    pred_map = {}

    succ_map = {}


    for nid in node_ids:

        inc = incoming.get(
            nid,
            [],
        )

        out = outgoing.get(
            nid,
            [],
        )


        if len(inc) == 1:

            pred_map[
                nid
            ] = int(
                inc[
                    0
                ]
            )


        if len(out) == 1:

            succ_map[
                nid
            ] = int(
                out[
                    0
                ]
            )


    # --------------------------------------------------------
    # Coordinates / time / origin
    # --------------------------------------------------------

    phys = {}

    time_map = {}

    origin_map = {}


    for nid, row in nodes.iterrows():

        nid = int(
            nid
        )


        phys[
            nid
        ] = physical_xyz(
            row
        )


        time_map[
            nid
        ] = int(
            row[
                "t"
            ]
        )


        origin_map[
            nid
        ] = (
            "B256_DET995"
            if nid in p995_ids
            else
            "POST_P995_ADDED"
        )


    # --------------------------------------------------------
    # Ordinary history length
    # --------------------------------------------------------

    ordered_ids = sorted(
        node_ids,
        key=lambda nid: (
            time_map[
                nid
            ],
            nid,
        ),
    )


    back_len = {}


    for nid in ordered_ids:

        predecessor = pred_map.get(
            nid
        )


        if predecessor is None:

            back_len[
                nid
            ] = 0

            continue


        if outdeg.get(
            predecessor,
            0,
        ) != 1:

            back_len[
                nid
            ] = 0

            continue


        if time_map[
            predecessor
        ] >= time_map[
            nid
        ]:

            back_len[
                nid
            ] = 0

            continue


        back_len[
            nid
        ] = (
            back_len.get(
                predecessor,
                0,
            )
            +
            1
        )


    forward_len = {}


    for nid in reversed(
        ordered_ids
    ):

        successor = succ_map.get(
            nid
        )


        if successor is None:

            forward_len[
                nid
            ] = 0

            continue


        if indeg.get(
            successor,
            0,
        ) != 1:

            forward_len[
                nid
            ] = 0

            continue


        if time_map[
            successor
        ] <= time_map[
            nid
        ]:

            forward_len[
                nid
            ] = 0

            continue


        forward_len[
            nid
        ] = (
            forward_len.get(
                successor,
                0,
            )
            +
            1
        )


    # --------------------------------------------------------
    # Local endpoint velocity
    # --------------------------------------------------------

    past_velocity = {}

    future_velocity = {}

    prev_disp_um = {}

    next_disp_um = {}


    for nid in node_ids:

        predecessor = pred_map.get(
            nid
        )


        if predecessor is not None:

            dt = (
                time_map[
                    nid
                ]
                -
                time_map[
                    predecessor
                ]
            )


            if dt > 0:

                velocity = (
                    phys[
                        nid
                    ]
                    -
                    phys[
                        predecessor
                    ]
                ) / float(
                    dt
                )


                past_velocity[
                    nid
                ] = velocity


                prev_disp_um[
                    nid
                ] = vector_norm(
                    phys[
                        nid
                    ]
                    -
                    phys[
                        predecessor
                    ]
                )


        successor = succ_map.get(
            nid
        )


        if successor is not None:

            dt = (
                time_map[
                    successor
                ]
                -
                time_map[
                    nid
                ]
            )


            if dt > 0:

                velocity = (
                    phys[
                        successor
                    ]
                    -
                    phys[
                        nid
                    ]
                ) / float(
                    dt
                )


                future_velocity[
                    nid
                ] = velocity


                next_disp_um[
                    nid
                ] = vector_norm(
                    phys[
                        successor
                    ]
                    -
                    phys[
                        nid
                    ]
                )


    # --------------------------------------------------------
    # Save prediction topology for later oracle lookup.
    # Still GT-blind.
    # --------------------------------------------------------

    for nid in node_ids:

        node_state[
            (
                dataset,
                int(
                    nid
                ),
            )
        ] = {
            "t":
                int(
                    time_map[
                        nid
                    ]
                ),

            "indegree":
                int(
                    indeg.get(
                        nid,
                        0,
                    )
                ),

            "outdegree":
                int(
                    outdeg.get(
                        nid,
                        0,
                    )
                ),

            "origin":
                origin_map[
                    nid
                ],
        }


    # --------------------------------------------------------
    # Free endpoints
    # --------------------------------------------------------

    source_ids = [
        nid

        for nid in node_ids

        if outdeg.get(
            nid,
            0,
        ) == 0
    ]


    target_ids = [
        nid

        for nid in node_ids

        if indeg.get(
            nid,
            0,
        ) == 0
    ]


    targets_by_t = {}


    for tgt in target_ids:

        targets_by_t.setdefault(
            time_map[
                tgt
            ],
            [],
        ).append(
            tgt
        )


    endpoint_summary_rows.append(
        {
            "dataset":
                dataset,

            "fold":
                fold,

            "nodes":
                len(
                    node_ids
                ),

            "free_sources":
                len(
                    source_ids
                ),

            "free_targets":
                len(
                    target_ids
                ),
        }
    )


    dataset_frames = []


    # --------------------------------------------------------
    # Enumerate ALL free/free pairs in temporal horizon.
    # --------------------------------------------------------

    for src in source_ids:

        source_t = int(
            time_map[
                src
            ]
        )


        source_pos = phys[
            src
        ]


        source_v = past_velocity.get(
            src
        )


        for gap_size in range(
            MIN_GAP_SIZE,
            MAX_GAP_SIZE + 1,
        ):

            delta_t = gap_size + 1

            target_t = (
                source_t
                +
                delta_t
            )


            tgts = targets_by_t.get(
                target_t,
                [],
            )


            if not tgts:

                continue


            tgt_pos = np.vstack(
                [
                    phys[
                        tgt
                    ]

                    for tgt in tgts
                ]
            )


            displacement_vec = (
                tgt_pos
                -
                source_pos[
                    None,
                    :
                ]
            )


            displacement_um = np.linalg.norm(
                displacement_vec,
                axis=1,
            )


            straight_velocity = (
                displacement_vec
                /
                float(
                    delta_t
                )
            )


            avg_speed = (
                displacement_um
                /
                float(
                    delta_t
                )
            )


            # ------------------------------------------------
            # Source-side CV
            # ------------------------------------------------

            if source_v is not None:

                source_pred_target = (
                    source_pos
                    +
                    float(
                        delta_t
                    )
                    *
                    source_v
                )


                source_cv = np.linalg.norm(
                    tgt_pos
                    -
                    source_pred_target[
                        None,
                        :
                    ],
                    axis=1,
                )


                source_speed_mismatch = np.linalg.norm(
                    straight_velocity
                    -
                    source_v[
                        None,
                        :
                    ],
                    axis=1,
                )

            else:

                source_cv = np.full(
                    len(
                        tgts
                    ),
                    np.nan,
                )


                source_speed_mismatch = np.full(
                    len(
                        tgts
                    ),
                    np.nan,
                )


            # ------------------------------------------------
            # Target-side CV
            # ------------------------------------------------

            target_cv = []

            target_speed_mismatch = []

            velocity_disagreement = []

            velocity_angle = []

            target_has_future_velocity = []


            for j, tgt in enumerate(
                tgts
            ):

                target_v = future_velocity.get(
                    tgt
                )


                if target_v is None:

                    target_has_future_velocity.append(
                        False
                    )

                    target_cv.append(
                        np.nan
                    )

                    target_speed_mismatch.append(
                        np.nan
                    )

                    velocity_disagreement.append(
                        np.nan
                    )

                    velocity_angle.append(
                        np.nan
                    )

                    continue


                target_has_future_velocity.append(
                    True
                )


                predicted_source = (
                    phys[
                        tgt
                    ]
                    -
                    float(
                        delta_t
                    )
                    *
                    target_v
                )


                target_cv.append(
                    vector_norm(
                        source_pos
                        -
                        predicted_source
                    )
                )


                target_speed_mismatch.append(
                    vector_norm(
                        straight_velocity[
                            j
                        ]
                        -
                        target_v
                    )
                )


                if source_v is not None:

                    velocity_disagreement.append(
                        vector_norm(
                            source_v
                            -
                            target_v
                        )
                    )


                    velocity_angle.append(
                        safe_angle_deg(
                            source_v,
                            target_v,
                        )
                    )

                else:

                    velocity_disagreement.append(
                        np.nan
                    )

                    velocity_angle.append(
                        np.nan
                    )


            target_cv = np.asarray(
                target_cv,
                dtype=float,
            )

            target_speed_mismatch = np.asarray(
                target_speed_mismatch,
                dtype=float,
            )

            velocity_disagreement = np.asarray(
                velocity_disagreement,
                dtype=float,
            )

            velocity_angle = np.asarray(
                velocity_angle,
                dtype=float,
            )


            # ------------------------------------------------
            # Symmetric CV residual
            # ------------------------------------------------

            cv_stack = np.vstack(
                [
                    source_cv,
                    target_cv,
                ]
            )


            available = np.isfinite(
                cv_stack
            )


            n_available = available.sum(
                axis=0
            )


            symmetric_cv = np.full(
                len(
                    tgts
                ),
                np.nan,
                dtype=float,
            )


            valid = (
                n_available
                >
                0
            )


            symmetric_cv[
                valid
            ] = (
                np.nansum(
                    cv_stack[
                        :,
                        valid
                    ],
                    axis=0,
                )
                /
                n_available[
                    valid
                ]
            )


            frame = pd.DataFrame(
                {
                    "dataset":
                        dataset,

                    "fold":
                        fold,

                    "source_id":
                        int(
                            src
                        ),

                    "target_id":
                        [
                            int(
                                tgt
                            )

                            for tgt
                            in tgts
                        ],

                    "source_t":
                        source_t,

                    "target_t":
                        target_t,

                    "gap_size":
                        gap_size,

                    "delta_t":
                        delta_t,

                    "source_z":
                        float(
                            nodes.loc[
                                src,
                                "z",
                            ]
                        ),

                    "source_y":
                        float(
                            nodes.loc[
                                src,
                                "y",
                            ]
                        ),

                    "source_x":
                        float(
                            nodes.loc[
                                src,
                                "x",
                            ]
                        ),

                    "target_z":
                        [
                            float(
                                nodes.loc[
                                    tgt,
                                    "z",
                                ]
                            )

                            for tgt in tgts
                        ],

                    "target_y":
                        [
                            float(
                                nodes.loc[
                                    tgt,
                                    "y",
                                ]
                            )

                            for tgt in tgts
                        ],

                    "target_x":
                        [
                            float(
                                nodes.loc[
                                    tgt,
                                    "x",
                                ]
                            )

                            for tgt in tgts
                        ],

                    "source_origin":
                        origin_map[
                            src
                        ],

                    "target_origin":
                        [
                            origin_map[
                                tgt
                            ]

                            for tgt in tgts
                        ],

                    "source_indegree":
                        int(
                            indeg.get(
                                src,
                                0,
                            )
                        ),

                    "source_outdegree":
                        0,

                    "target_indegree":
                        0,

                    "target_outdegree":
                        [
                            int(
                                outdeg.get(
                                    tgt,
                                    0,
                                )
                            )

                            for tgt in tgts
                        ],

                    "source_back_track_len":
                        int(
                            back_len.get(
                                src,
                                0,
                            )
                        ),

                    "target_forward_track_len":
                        [
                            int(
                                forward_len.get(
                                    tgt,
                                    0,
                                )
                            )

                            for tgt in tgts
                        ],

                    "source_has_past_velocity":
                        bool(
                            source_v
                            is not None
                        ),

                    "target_has_future_velocity":
                        target_has_future_velocity,

                    "source_prev_disp_um":
                        float(
                            prev_disp_um.get(
                                src,
                                np.nan,
                            )
                        ),

                    "target_next_disp_um":
                        [
                            float(
                                next_disp_um.get(
                                    tgt,
                                    np.nan,
                                )
                            )

                            for tgt in tgts
                        ],

                    "displacement_um":
                        displacement_um,

                    "avg_speed_um_per_frame":
                        avg_speed,

                    "source_cv_residual_um":
                        source_cv,

                    "target_cv_residual_um":
                        target_cv,

                    "symmetric_cv_residual_um":
                        symmetric_cv,

                    "source_speed_mismatch_um_per_frame":
                        source_speed_mismatch,

                    "target_speed_mismatch_um_per_frame":
                        target_speed_mismatch,

                    "velocity_disagreement_um_per_frame":
                        velocity_disagreement,

                    "velocity_angle_deg":
                        velocity_angle,
                }
            )


            dataset_frames.append(
                frame
            )


    if dataset_frames:

        candidate_frames.append(
            pd.concat(
                dataset_frames,
                ignore_index=True,
            )
        )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            names
        )
    ):

        pair_count = sum(
            len(
                frame
            )

            for frame
            in dataset_frames
        )


        print(
            f"{ds_i:3d}/{len(names)}"
            f" | fold={fold}"
            f" | {dataset}"
            f" | src={len(source_ids)}"
            f" | tgt={len(target_ids)}"
            f" | pairs={pair_count}"
        )


# ============================================================
# 8. CANONICAL CANDIDATE TABLE
# ============================================================

assert candidate_frames


candidates = pd.concat(
    candidate_frames,
    ignore_index=True,
)


assert len(
    candidates
) > 0


PAIR_KEY = [
    "dataset",
    "source_id",
    "target_id",
]


assert not candidates[
    PAIR_KEY
].duplicated().any()


assert candidates[
    "source_outdegree"
].eq(
    0
).all()


assert candidates[
    "target_indegree"
].eq(
    0
).all()


assert candidates[
    "gap_size"
].between(
    MIN_GAP_SIZE,
    MAX_GAP_SIZE,
).all()


# ============================================================
# 9. GT-BLIND RANK FEATURES
# ============================================================

SOURCE_GROUP = [
    "dataset",
    "source_id",
    "target_t",
]

TARGET_GROUP = [
    "dataset",
    "target_id",
    "source_t",
]


def add_rank(
    df,
    value_col,
    output_col,
    group_cols,
):

    df[
        output_col
    ] = (
        df.groupby(
            group_cols
        )[
            value_col
        ]
        .rank(
            method="min",
            ascending=True,
        )
    )


for value_col, prefix in [
    (
        "displacement_um",
        "displacement",
    ),
    (
        "source_cv_residual_um",
        "source_cv",
    ),
    (
        "target_cv_residual_um",
        "target_cv",
    ),
    (
        "symmetric_cv_residual_um",
        "symmetric_cv",
    ),
]:

    add_rank(
        candidates,
        value_col,
        prefix
        +
        "_source_rank",
        SOURCE_GROUP,
    )


    add_rank(
        candidates,
        value_col,
        prefix
        +
        "_target_rank",
        TARGET_GROUP,
    )


candidates[
    "displacement_reciprocal_top1"
] = (
    candidates[
        "displacement_source_rank"
    ].eq(
        1
    )
    &
    candidates[
        "displacement_target_rank"
    ].eq(
        1
    )
)


candidates[
    "symmetric_cv_reciprocal_top1"
] = (
    candidates[
        "symmetric_cv_source_rank"
    ].eq(
        1
    )
    &
    candidates[
        "symmetric_cv_target_rank"
    ].eq(
        1
    )
)


candidates[
    "source_has_history"
] = (
    candidates[
        "source_back_track_len"
    ]
    >
    0
)


candidates[
    "target_has_future"
] = (
    candidates[
        "target_forward_track_len"
    ]
    >
    0
)


candidates[
    "both_boundaries_original_p995"
] = (
    candidates[
        "source_origin"
    ].eq(
        "B256_DET995"
    )
    &
    candidates[
        "target_origin"
    ].eq(
        "B256_DET995"
    )
)


# ============================================================
# 10. STABLE SORT + HASH
# ============================================================

candidates = (
    candidates
    .sort_values(
        [
            "fold",
            "dataset",
            "source_t",
            "source_id",
            "target_t",
            "target_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


HASH_COLS = [
    "dataset",
    "fold",
    "source_id",
    "target_id",
    "source_t",
    "target_t",
    "gap_size",
]


candidate_sha = hashlib.sha256(
    candidates[
        HASH_COLS
    ]
    .to_csv(
        index=False,
    )
    .encode()
).hexdigest()


endpoint_summary = pd.DataFrame(
    endpoint_summary_rows
)


# ============================================================
# 11. GT-BLIND DIAGNOSTICS
# ============================================================

print(
    "\n========== GT-BLIND FREE/FREE UNIVERSE =========="
)

print(
    "datasets:",
    candidates[
        "dataset"
    ].nunique()
)

print(
    "candidates:",
    len(
        candidates
    )
)

print(
    "free sources:",
    int(
        endpoint_summary[
            "free_sources"
        ].sum()
    )
)

print(
    "free targets:",
    int(
        endpoint_summary[
            "free_targets"
        ].sum()
    )
)

print(
    "candidate SHA256:",
    candidate_sha
)


print(
    "\n========== CANDIDATES BY GAP SIZE =========="
)

print(
    candidates[
        "gap_size"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


source_velocity_fraction = float(
    candidates[
        "source_has_past_velocity"
    ].mean()
)


target_velocity_fraction = float(
    candidates[
        "target_has_future_velocity"
    ].mean()
)


both_velocity_fraction = float(
    (
        candidates[
            "source_has_past_velocity"
        ].astype(bool)
        &
        candidates[
            "target_has_future_velocity"
        ].astype(bool)
    ).mean()
)


both_p995_fraction = float(
    candidates[
        "both_boundaries_original_p995"
    ].mean()
)


print(
    "\n========== FEATURE AVAILABILITY =========="
)

print(
    "source past velocity:",
    f"{source_velocity_fraction:.3%}"
)

print(
    "target future velocity:",
    f"{target_velocity_fraction:.3%}"
)

print(
    "both velocity sides:",
    f"{both_velocity_fraction:.3%}"
)

print(
    "both original P995:",
    f"{both_p995_fraction:.3%}"
)


source_ambiguity = (
    candidates.groupby(
        SOURCE_GROUP
    )
    .size()
)


target_ambiguity = (
    candidates.groupby(
        TARGET_GROUP
    )
    .size()
)


print(
    "\n========== CANDIDATE AMBIGUITY =========="
)

print(
    "targets per source/gap-frame:"
)

print(
    source_ambiguity.describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    ).to_string()
)


print(
    "\nsources per target/source-frame:"
)

print(
    target_ambiguity.describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    ).to_string()
)


# ============================================================
# 12. PERSIST GT-BLIND UNIVERSE
#
# IMPORTANT:
# This occurs BEFORE reading G2 GT artifact.
# ============================================================

candidates.to_pickle(
    CANDIDATE_OUT
)


candidate_summary = {
    "stage":
        "12.11G3A",

    "artifact":
        "GT_BLIND_FREE_FREE_GAP_ENDPOINT_UNIVERSE",

    "corpus":
        "folds1-4 6bba CURRENT_BEST",

    "folds":
        [
            1,
            2,
            3,
            4,
        ],

    "datasets":
        int(
            candidates[
                "dataset"
            ].nunique()
        ),

    "candidate_generation": {
        "source_requirement":
            "CURRENT_BEST_OUTDEGREE_0",

        "target_requirement":
            "CURRENT_BEST_INDEGREE_0",

        "min_missing_frames":
            MIN_GAP_SIZE,

        "max_missing_frames":
            MAX_GAP_SIZE,

        "all_pairs_within_temporal_horizon":
            True,
    },

    "candidates":
        int(
            len(
                candidates
            )
        ),

    "free_sources":
        int(
            endpoint_summary[
                "free_sources"
            ].sum()
        ),

    "free_targets":
        int(
            endpoint_summary[
                "free_targets"
            ].sum()
        ),

    "candidate_key_sha256":
        candidate_sha,

    "gt_used_during_generation":
        False,

    "fold0_used":
        False,

    "graph_modified":
        False,

    "policy_selected":
        False,

    "threshold_selected":
        False,
}


CANDIDATE_SUMMARY_OUT.write_text(
    json.dumps(
        candidate_summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert CANDIDATE_OUT.exists()
assert CANDIDATE_SUMMARY_OUT.exists()


print(
    "\nwritten GT-blind universe:",
    CANDIDATE_OUT
)

print(
    "written GT-blind summary:",
    CANDIDATE_SUMMARY_OUT
)

print(
    "GT-BLIND UNIVERSE FROZEN BEFORE GT READ: YES"
)


# ============================================================
# PART B
# DEVELOPMENT ORACLE COVERAGE
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — DEVELOPMENT ORACLE COVERAGE AUDIT"
)

print(
    "============================================================"
)


# ============================================================
# 13. ONLY NOW READ G2 GT ORACLE
# ============================================================

assert G2_COMPONENT_PATH.exists(), (
    G2_COMPONENT_PATH
)


oracle = pd.read_pickle(
    G2_COMPONENT_PATH
)


assert len(
    oracle
) == 369


assert int(
    oracle[
        "gap_size"
    ].sum()
) == 1189


# ============================================================
# 14. ORACLE IDENTITIES
# ============================================================

oracle_eval = oracle[
    [
        "dataset",
        "fold",
        "component_id",
        "gap_size",
        "boundary_t0",
        "boundary_t1",
        "boundary_in_pred_node_id",
        "boundary_out_pred_node_id",
        "pred_full_3um",
        "pred_full_5um",
        "pred_full_7um",
        "pred_median_error_um",
        "pred_max_error_um",
    ]
].copy()


oracle_eval = oracle_eval.rename(
    columns={
        "boundary_in_pred_node_id":
            "source_id",

        "boundary_out_pred_node_id":
            "target_id",

        "boundary_t0":
            "source_t",

        "boundary_t1":
            "target_t",
    }
)


oracle_eval[
    "expected_gap_size"
] = (
    oracle_eval[
        "target_t"
    ]
    -
    oracle_eval[
        "source_t"
    ]
    -
    1
)


assert (
    oracle_eval[
        "expected_gap_size"
    ]
    ==
    oracle_eval[
        "gap_size"
    ]
).all()


# ============================================================
# 15. CURRENT_BEST TOPOLOGY OF TRUE BOUNDARIES
# ============================================================

def lookup_state(
    dataset,
    node_id,
):

    key = (
        dataset,
        int(
            node_id
        ),
    )


    assert key in node_state, key


    return node_state[
        key
    ]


oracle_eval[
    "source_current_outdegree"
] = [
    int(
        lookup_state(
            row.dataset,
            row.source_id,
        )[
            "outdegree"
        ]
    )

    for row in oracle_eval.itertuples(
        index=False
    )
]


oracle_eval[
    "source_current_indegree"
] = [
    int(
        lookup_state(
            row.dataset,
            row.source_id,
        )[
            "indegree"
        ]
    )

    for row in oracle_eval.itertuples(
        index=False
    )
]


oracle_eval[
    "target_current_indegree"
] = [
    int(
        lookup_state(
            row.dataset,
            row.target_id,
        )[
            "indegree"
        ]
    )

    for row in oracle_eval.itertuples(
        index=False
    )
]


oracle_eval[
    "target_current_outdegree"
] = [
    int(
        lookup_state(
            row.dataset,
            row.target_id,
        )[
            "outdegree"
        ]
    )

    for row in oracle_eval.itertuples(
        index=False
    )
]


oracle_eval[
    "free_free_topology"
] = (
    oracle_eval[
        "source_current_outdegree"
    ].eq(
        0
    )
    &
    oracle_eval[
        "target_current_indegree"
    ].eq(
        0
    )
)


oracle_eval[
    "within_temporal_horizon"
] = (
    oracle_eval[
        "gap_size"
    ].between(
        MIN_GAP_SIZE,
        MAX_GAP_SIZE,
    )
)


# ============================================================
# 16. JOIN TRUE PAIR TO FROZEN GT-BLIND UNIVERSE
# ============================================================

candidate_join_cols = [
    "dataset",
    "source_id",
    "target_id",

    "displacement_um",
    "avg_speed_um_per_frame",

    "source_back_track_len",
    "target_forward_track_len",

    "source_has_past_velocity",
    "target_has_future_velocity",

    "source_cv_residual_um",
    "target_cv_residual_um",
    "symmetric_cv_residual_um",

    "source_speed_mismatch_um_per_frame",
    "target_speed_mismatch_um_per_frame",

    "velocity_disagreement_um_per_frame",
    "velocity_angle_deg",

    "displacement_source_rank",
    "displacement_target_rank",

    "source_cv_source_rank",
    "source_cv_target_rank",

    "target_cv_source_rank",
    "target_cv_target_rank",

    "symmetric_cv_source_rank",
    "symmetric_cv_target_rank",

    "displacement_reciprocal_top1",
    "symmetric_cv_reciprocal_top1",

    "both_boundaries_original_p995",
]


oracle_eval = oracle_eval.merge(
    candidates[
        candidate_join_cols
    ],
    on=[
        "dataset",
        "source_id",
        "target_id",
    ],
    how="left",
    validate="one_to_one",
    indicator=True,
)


oracle_eval[
    "in_gt_blind_universe"
] = (
    oracle_eval[
        "_merge"
    ].eq(
        "both"
    )
)


oracle_eval = oracle_eval.drop(
    columns=[
        "_merge"
    ]
)


# ============================================================
# 17. EXPLAIN MISSES
# ============================================================

def coverage_reason(
    row,
):

    if bool(
        row.in_gt_blind_universe
    ):

        return (
            "IN_FREE_FREE_UNIVERSE"
        )


    if not bool(
        row.within_temporal_horizon
    ):

        return (
            "OUTSIDE_TEMPORAL_HORIZON"
        )


    src_busy = (
        int(
            row.source_current_outdegree
        )
        != 0
    )


    tgt_busy = (
        int(
            row.target_current_indegree
        )
        != 0
    )


    if (
        src_busy
        and
        tgt_busy
    ):

        return (
            "BOTH_BOUNDARIES_OCCUPIED"
        )


    if src_busy:

        return (
            "SOURCE_BOUNDARY_OCCUPIED"
        )


    if tgt_busy:

        return (
            "TARGET_BOUNDARY_OCCUPIED"
        )


    return (
        "UNEXPECTED_UNIVERSE_MISS"
    )


oracle_eval[
    "coverage_reason"
] = oracle_eval.apply(
    coverage_reason,
    axis=1,
)


unexpected = oracle_eval[
    oracle_eval[
        "coverage_reason"
    ].eq(
        "UNEXPECTED_UNIVERSE_MISS"
    )
]


assert len(
    unexpected
) == 0, unexpected


# ============================================================
# 18. ORACLE COVERAGE
# ============================================================

n_oracle = len(
    oracle_eval
)


n_covered = int(
    oracle_eval[
        "in_gt_blind_universe"
    ].sum()
)


coverage_fraction = float(
    n_covered
    /
    n_oracle
)


print(
    "\n========== ORACLE FREE/FREE COVERAGE =========="
)

print(
    "clean components:",
    n_oracle
)

print(
    "captured:",
    n_covered,
    "/",
    n_oracle,
    f"({coverage_fraction:.3%})",
)


print(
    "\n========== ORACLE COVERAGE REASONS =========="
)

print(
    oracle_eval[
        "coverage_reason"
    ]
    .value_counts()
    .to_string()
)


# ============================================================
# 19. COVERAGE BY FOLD
# ============================================================

print(
    "\n========== ORACLE COVERAGE BY FOLD =========="
)


fold_cov = (
    oracle_eval.groupby(
        "fold"
    )
    .agg(
        clean_components=(
            "component_id",
            "size",
        ),

        free_free_components=(
            "in_gt_blind_universe",
            "sum",
        ),
    )
)


fold_cov[
    "coverage"
] = (
    fold_cov[
        "free_free_components"
    ]
    /
    fold_cov[
        "clean_components"
    ]
)


print(
    fold_cov.to_string()
)


# ============================================================
# 20. COVERAGE BY GAP SIZE
# ============================================================

print(
    "\n========== ORACLE COVERAGE BY GAP SIZE =========="
)


gap_cov = (
    oracle_eval.groupby(
        "gap_size"
    )
    .agg(
        clean_components=(
            "component_id",
            "size",
        ),

        free_free_components=(
            "in_gt_blind_universe",
            "sum",
        ),
    )
)


gap_cov[
    "coverage"
] = (
    gap_cov[
        "free_free_components"
    ]
    /
    gap_cov[
        "clean_components"
    ]
)


print(
    gap_cov.to_string()
)


# ============================================================
# 21. TRUE-PAIR RANK DISTRIBUTIONS
# ============================================================

covered = (
    oracle_eval[
        oracle_eval[
            "in_gt_blind_universe"
        ]
    ]
    .copy()
)


print(
    "\n========== TRUE-PAIR RANK DISTRIBUTION =========="
)


rank_cols = [
    "displacement_source_rank",
    "displacement_target_rank",

    "source_cv_source_rank",
    "source_cv_target_rank",

    "target_cv_source_rank",
    "target_cv_target_rank",

    "symmetric_cv_source_rank",
    "symmetric_cv_target_rank",
]


for col in rank_cols:

    valid = (
        covered[
            col
        ]
        .dropna()
    )


    print(
        "\n" + col
    )


    if len(
        valid
    ) == 0:

        print(
            "no valid values"
        )

        continue


    print(
        valid.describe(
            percentiles=[
                0.50,
                0.75,
                0.90,
                0.95,
                0.99,
            ]
        ).to_string()
    )


# ============================================================
# 22. DESCRIPTIVE TOP1 SUPPORT
# ============================================================

print(
    "\n========== TRUE PAIR — DESCRIPTIVE TOP1 SUPPORT =========="
)


if len(
    covered
) > 0:

    disp_top1 = int(
        covered[
            "displacement_reciprocal_top1"
        ]
        .fillna(
            False
        )
        .sum()
    )


    sym_top1 = int(
        covered[
            "symmetric_cv_reciprocal_top1"
        ]
        .fillna(
            False
        )
        .sum()
    )


    disp_fraction = float(
        disp_top1
        /
        len(
            covered
        )
    )


    sym_fraction = float(
        sym_top1
        /
        len(
            covered
        )
    )


    print(
        "displacement reciprocal Top1:",
        disp_top1,
        "/",
        len(
            covered
        ),
        f"({disp_fraction:.3%})",
    )


    print(
        "symmetric-CV reciprocal Top1:",
        sym_top1,
        "/",
        len(
            covered
        ),
        f"({sym_fraction:.3%})",
    )

else:

    print(
        "No true clean gaps are inside free/free universe."
    )


# ============================================================
# 23. INTERPOLATION CEILING WITHIN DEPLOYABLE TOPOLOGY
# ============================================================

print(
    "\n========== INTERPOLATION CEILING INSIDE FREE/FREE =========="
)


for radius in [
    3,
    5,
    7,
]:

    col = (
        f"pred_full_{radius}um"
    )


    good = int(
        covered[
            col
        ].sum()
    )


    if len(
        covered
    ) > 0:

        good_fraction = float(
            good
            /
            len(
                covered
            )
        )


        text = (
            f"{good_fraction:.3%}"
        )

    else:

        text = (
            "NA"
        )


    print(
        f"full <= {radius} um:",
        good,
        "/",
        len(
            covered
        ),
        "(" + text + ")",
    )


# ============================================================
# 24. WRITE ORACLE AUDIT
# ============================================================

oracle_eval = (
    oracle_eval
    .sort_values(
        [
            "fold",
            "dataset",
            "component_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


oracle_eval.to_pickle(
    ORACLE_OUT
)


oracle_summary = {
    "stage":
        "12.11G3A",

    "audit":
        "DEVELOPMENT_ORACLE_COVERAGE_OF_FROZEN_GT_BLIND_UNIVERSE",

    "candidate_artifact_sha256":
        candidate_sha,

    "clean_components":
        int(
            n_oracle
        ),

    "free_free_captured":
        int(
            n_covered
        ),

    "free_free_coverage":
        coverage_fraction,

    "coverage_reasons":
        {
            str(k):
                int(v)

            for k, v
            in oracle_eval[
                "coverage_reason"
            ]
            .value_counts()
            .items()
        },

    "candidate_generation_gt_blind":
        True,

    "oracle_gt_used":
        True,

    "oracle_gt_used_only_after_candidate_freeze":
        True,

    "fold0_used":
        False,

    "policy_selected":
        False,

    "threshold_selected":
        False,

    "graph_modified":
        False,
}


ORACLE_SUMMARY_OUT.write_text(
    json.dumps(
        oracle_summary,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten oracle coverage:",
    ORACLE_OUT
)

print(
    "written oracle summary:",
    ORACLE_SUMMARY_OUT
)


# ============================================================
# 25. FINAL INTEGRITY
# ============================================================

assert CANDIDATE_OUT.exists()
assert CANDIDATE_SUMMARY_OUT.exists()

assert ORACLE_OUT.exists()
assert ORACLE_SUMMARY_OUT.exists()


candidate_check = pd.read_pickle(
    CANDIDATE_OUT
)


oracle_check = pd.read_pickle(
    ORACLE_OUT
)


assert len(
    candidate_check
) == len(
    candidates
)


assert len(
    oracle_check
) == 369


assert not candidate_check[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


assert candidate_check[
    "source_outdegree"
].eq(
    0
).all()


assert candidate_check[
    "target_indegree"
].eq(
    0
).all()


assert candidate_check[
    "gap_size"
].between(
    MIN_GAP_SIZE,
    MAX_GAP_SIZE,
).all()


assert set(
    candidate_check[
        "fold"
    ].astype(int)
).issubset(
    {
        1,
        2,
        3,
        4,
    }
)


# ============================================================
# 26. DECISION
# ============================================================

print(
    "\n========== 12.11G3A DECISION =========="
)

print(
    "GT_BLIND_FREE_FREE_GAP_ENDPOINT_UNIVERSE: PASS"
)

print(
    "candidate generation GT-blind: YES"
)

print(
    "candidate pairs:",
    len(
        candidates
    )
)

print(
    "candidate SHA256:",
    candidate_sha
)

print(
    "oracle clean components:",
    n_oracle
)

print(
    "free/free oracle captured:",
    n_covered,
    "/",
    n_oracle,
    f"({coverage_fraction:.3%})",
)

print(
    "Fold0 used: NO"
)

print(
    "policy selected: NO"
)

print(
    "threshold selected: NO"
)

print(
    "synthetic nodes inserted: NO"
)

print(
    "graph modified: NO"
)

print(
    "next:"
)

print(
    "AUDIT_GT_BLIND_FEATURE_SEPARATION_AND_POLICY_FEASIBILITY"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3B
# Occupied-Boundary FP / Rewire Feasibility Audit
#
# Purpose:
#
#   Stage 12.11G3A showed:
#
#       clean oracle components = 369
#       free/free                = 116
#       occupied at >=1 side     = 253
#
#   This audit asks:
#
#       Are the CURRENT_BEST edges occupying those clean-gap
#       boundary nodes already identifiable as official
#       development FP edges?
#
# If yes, then the major remaining route is not simply:
#
#       free/free synthetic insertion
#
# but:
#
#       wrong-edge removal / rewire
#           +
#       synthetic missing-node reconstruction
#
# IMPORTANT:
#
#   - Uses development oracle identity only for AUDIT.
#   - Uses exact G1 official valid-FP residual census.
#   - Does NOT define a deployment rule.
#   - Does NOT remove edges.
#   - Does NOT insert synthetic nodes.
#   - Does NOT use Fold0.
#
# Changes files: YES
# Runs training: NO
# Runs inference: NO
# Modifies graph: NO
# Fold0: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


G3A_ORACLE_PATH = (
    S12
    / "stage12_folds14_gap_endpoint_candidate_oracle_coverage.pkl"
)

G3A_ORACLE_SUMMARY_PATH = (
    S12
    / "stage12_folds14_gap_endpoint_candidate_oracle_summary.json"
)

G1_FP_PATH = (
    S12
    / "stage12_folds14_current_best_edge_fp_residuals.pkl"
)

G1_SUMMARY_PATH = (
    S12
    / "stage12_folds14_current_best_edge_residual_summary.json"
)


OUT = (
    S12
    / "stage12_folds14_occupied_boundary_rewire_feasibility.pkl"
)

SUMMARY_OUT = (
    S12
    / "stage12_folds14_occupied_boundary_rewire_feasibility_summary.json"
)


for path in [
    G3A_ORACLE_PATH,
    G3A_ORACLE_SUMMARY_PATH,
    G1_FP_PATH,
    G1_SUMMARY_PATH,
]:
    assert path.exists(), path


# ============================================================
# 1. Load canonical artifacts
# ============================================================

oracle = pd.read_pickle(
    G3A_ORACLE_PATH
)


fp = pd.read_pickle(
    G1_FP_PATH
)


g3a_summary = json.loads(
    G3A_ORACLE_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


g1_summary = json.loads(
    G1_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


# ============================================================
# 2. Input fidelity gates
# ============================================================

assert len(
    oracle
) == 369


assert int(
    oracle[
        "in_gt_blind_universe"
    ].sum()
) == 116


assert (
    oracle[
        "fold"
    ]
    .astype(int)
    .isin(
        [
            1,
            2,
            3,
            4,
        ]
    )
    .all()
)


assert len(
    fp
) == 9398


assert int(
    g1_summary[
        "metrics"
    ][
        "edge_fp"
    ]
) == 9398


assert int(
    g3a_summary[
        "clean_components"
    ]
) == 369


assert int(
    g3a_summary[
        "free_free_captured"
    ]
) == 116


print(
    "========== 12.11G3B INPUT =========="
)

print(
    "clean components:",
    len(
        oracle
    )
)

print(
    "free/free:",
    int(
        oracle[
            "in_gt_blind_universe"
        ].sum()
    )
)

print(
    "occupied >=1 side:",
    int(
        (
            ~oracle[
                "in_gt_blind_universe"
            ]
        ).sum()
    )
)

print(
    "official valid FP edges:",
    len(
        fp
    )
)


# ============================================================
# 3. Normalize FP endpoint IDs
# ============================================================

required_fp_cols = [
    "dataset",
    "source_pred_node_id",
    "target_pred_node_id",
    "endpoint_support",
]


missing = [
    col

    for col in required_fp_cols

    if col not in fp.columns
]


assert not missing, (
    "Missing G1 FP columns:",
    missing,
)


fp[
    "source_pred_node_id"
] = (
    fp[
        "source_pred_node_id"
    ]
    .astype(int)
)


fp[
    "target_pred_node_id"
] = (
    fp[
        "target_pred_node_id"
    ]
    .astype(int)
)


# ============================================================
# 4. Count official FP occupancy per predicted boundary node
# ============================================================

fp_out_count = (
    fp.groupby(
        [
            "dataset",
            "source_pred_node_id",
        ]
    )
    .size()
    .to_dict()
)


fp_in_count = (
    fp.groupby(
        [
            "dataset",
            "target_pred_node_id",
        ]
    )
    .size()
    .to_dict()
)


def lookup_fp_out(
    dataset,
    node_id,
):

    return int(
        fp_out_count.get(
            (
                dataset,
                int(
                    node_id
                ),
            ),
            0,
        )
    )


def lookup_fp_in(
    dataset,
    node_id,
):

    return int(
        fp_in_count.get(
            (
                dataset,
                int(
                    node_id
                ),
            ),
            0,
        )
    )


oracle[
    "source_valid_fp_out_count"
] = [
    lookup_fp_out(
        row.dataset,
        row.source_id,
    )

    for row in oracle.itertuples(
        index=False
    )
]


oracle[
    "target_valid_fp_in_count"
] = [
    lookup_fp_in(
        row.dataset,
        row.target_id,
    )

    for row in oracle.itertuples(
        index=False
    )
]


# ============================================================
# 5. Basic degree consistency
# ============================================================

assert (
    oracle[
        "source_valid_fp_out_count"
    ]
    <=
    oracle[
        "source_current_outdegree"
    ]
).all()


assert (
    oracle[
        "target_valid_fp_in_count"
    ]
    <=
    oracle[
        "target_current_indegree"
    ]
).all()


# ============================================================
# 6. Boundary occupancy topology
# ============================================================

oracle[
    "source_occupied"
] = (
    oracle[
        "source_current_outdegree"
    ]
    >
    0
)


oracle[
    "target_occupied"
] = (
    oracle[
        "target_current_indegree"
    ]
    >
    0
)


def topology_class(
    row,
):

    src = bool(
        row.source_occupied
    )

    tgt = bool(
        row.target_occupied
    )


    if (
        not src
        and
        not tgt
    ):

        return (
            "FREE_FREE"
        )


    if (
        src
        and
        not tgt
    ):

        return (
            "SOURCE_OCCUPIED"
        )


    if (
        not src
        and
        tgt
    ):

        return (
            "TARGET_OCCUPIED"
        )


    return (
        "BOTH_OCCUPIED"
    )


oracle[
    "topology_class"
] = oracle.apply(
    topology_class,
    axis=1,
)


print(
    "\n========== ORACLE TOPOLOGY =========="
)

print(
    oracle[
        "topology_class"
    ]
    .value_counts()
    .to_string()
)


assert int(
    oracle[
        "topology_class"
    ]
    .eq(
        "FREE_FREE"
    )
    .sum()
) == 116


assert int(
    oracle[
        "topology_class"
    ]
    .eq(
        "SOURCE_OCCUPIED"
    )
    .sum()
) == 66


assert int(
    oracle[
        "topology_class"
    ]
    .eq(
        "TARGET_OCCUPIED"
    )
    .sum()
) == 75


assert int(
    oracle[
        "topology_class"
    ]
    .eq(
        "BOTH_OCCUPIED"
    )
    .sum()
) == 112


# ============================================================
# 7. Is ALL current occupancy explained by official FP edges?
# ============================================================
#
# For a free side, there is nothing to remove.
#
# For an occupied side:
#
#       valid FP count == current degree
#
# means every current blocking edge on that boundary is already
# represented in the official valid-FP residual census.
#
# This is an ORACLE AUDIT statement, not a deployable rule.
# ============================================================

oracle[
    "source_occupancy_fully_fp_explained"
] = np.where(
    ~oracle[
        "source_occupied"
    ],

    True,

    (
        oracle[
            "source_valid_fp_out_count"
        ]
        ==
        oracle[
            "source_current_outdegree"
        ]
    ),
)


oracle[
    "target_occupancy_fully_fp_explained"
] = np.where(
    ~oracle[
        "target_occupied"
    ],

    True,

    (
        oracle[
            "target_valid_fp_in_count"
        ]
        ==
        oracle[
            "target_current_indegree"
        ]
    ),
)


oracle[
    "all_blocking_occupancy_fp_explained"
] = (
    oracle[
        "source_occupancy_fully_fp_explained"
    ].astype(bool)
    &
    oracle[
        "target_occupancy_fully_fp_explained"
    ].astype(bool)
)


# ============================================================
# 8. Occupied subset
# ============================================================

occupied = (
    oracle[
        oracle[
            "topology_class"
        ].ne(
            "FREE_FREE"
        )
    ]
    .copy()
)


assert len(
    occupied
) == 253


print(
    "\n========== OCCUPIED BOUNDARY FP EXPLANATION =========="
)


source_occ = occupied[
    occupied[
        "source_occupied"
    ]
]


target_occ = occupied[
    occupied[
        "target_occupied"
    ]
]


source_full = int(
    source_occ[
        "source_occupancy_fully_fp_explained"
    ].sum()
)


target_full = int(
    target_occ[
        "target_occupancy_fully_fp_explained"
    ].sum()
)


all_full = int(
    occupied[
        "all_blocking_occupancy_fp_explained"
    ].sum()
)


print(
    "occupied components:",
    len(
        occupied
    )
)


print(
    "occupied source boundaries:",
    len(
        source_occ
    )
)


print(
    "source fully explained by valid FP:",
    source_full,
    "/",
    len(
        source_occ
    ),
    f"({source_full/len(source_occ):.3%})",
)


print(
    "occupied target boundaries:",
    len(
        target_occ
    )
)


print(
    "target fully explained by valid FP:",
    target_full,
    "/",
    len(
        target_occ
    ),
    f"({target_full/len(target_occ):.3%})",
)


print(
    "ALL blocking occupancy fully FP-explained:",
    all_full,
    "/",
    len(
        occupied
    ),
    f"({all_full/len(occupied):.3%})",
)


# ============================================================
# 9. Breakdown by topology class
# ============================================================

print(
    "\n========== FP EXPLANATION BY TOPOLOGY =========="
)


topology_summary = (
    occupied.groupby(
        "topology_class"
    )
    .agg(
        components=(
            "component_id",
            "size",
        ),

        source_fp_explained=(
            "source_occupancy_fully_fp_explained",
            "sum",
        ),

        target_fp_explained=(
            "target_occupancy_fully_fp_explained",
            "sum",
        ),

        all_fp_explained=(
            "all_blocking_occupancy_fp_explained",
            "sum",
        ),

        median_gap_size=(
            "gap_size",
            "median",
        ),

        pred_full_5um=(
            "pred_full_5um",
            "sum",
        ),

        pred_full_7um=(
            "pred_full_7um",
            "sum",
        ),
    )
)


print(
    topology_summary.to_string()
)


# ============================================================
# 10. Current degree distributions
# ============================================================

print(
    "\n========== OCCUPIED DEGREE DISTRIBUTION =========="
)


print(
    "source current outdegree:"
)

print(
    source_occ[
        "source_current_outdegree"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\ntarget current indegree:"
)

print(
    target_occ[
        "target_current_indegree"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\nsource valid FP-out count:"
)

print(
    source_occ[
        "source_valid_fp_out_count"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\ntarget valid FP-in count:"
)

print(
    target_occ[
        "target_valid_fp_in_count"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


# ============================================================
# 11. Collect the actual incumbent valid-FP edges
# ============================================================

source_boundary_keys = set(
    zip(
        source_occ[
            "dataset"
        ],
        source_occ[
            "source_id"
        ].astype(int),
    )
)


target_boundary_keys = set(
    zip(
        target_occ[
            "dataset"
        ],
        target_occ[
            "target_id"
        ].astype(int),
    )
)


fp[
    "blocks_oracle_source"
] = [
    (
        row.dataset,
        int(
            row.source_pred_node_id
        ),
    )
    in source_boundary_keys

    for row in fp.itertuples(
        index=False
    )
]


fp[
    "blocks_oracle_target"
] = [
    (
        row.dataset,
        int(
            row.target_pred_node_id
        ),
    )
    in target_boundary_keys

    for row in fp.itertuples(
        index=False
    )
]


blocking_fp = (
    fp[
        fp[
            "blocks_oracle_source"
        ]
        |
        fp[
            "blocks_oracle_target"
        ]
    ]
    .copy()
)


print(
    "\n========== BLOCKING OFFICIAL FP EDGES =========="
)

print(
    "unique blocking valid-FP edges:",
    len(
        blocking_fp
    )
)


print(
    "\nendpoint support:"
)

print(
    blocking_fp[
        "endpoint_support"
    ]
    .value_counts()
    .to_string()
)


# ============================================================
# 12. Does one blocking edge touch multiple oracle boundaries?
# ============================================================
#
# Important for a future greedy rewire policy:
#
# one wrong edge may simultaneously occupy:
#     source boundary of one gap
#     target boundary of another gap
# ============================================================

blocking_fp[
    "blocks_both_types"
] = (
    blocking_fp[
        "blocks_oracle_source"
    ]
    &
    blocking_fp[
        "blocks_oracle_target"
    ]
)


n_dual_block = int(
    blocking_fp[
        "blocks_both_types"
    ].sum()
)


print(
    "\n========== BLOCKING EDGE INTERACTION =========="
)

print(
    "blocks both an oracle source and target:",
    n_dual_block,
    "/",
    len(
        blocking_fp
    ),
)


# ============================================================
# 13. Boundary-sharing between different oracle clean gaps
# ============================================================

all_source_keys = set(
    zip(
        oracle[
            "dataset"
        ],
        oracle[
            "source_id"
        ].astype(int),
    )
)


all_target_keys = set(
    zip(
        oracle[
            "dataset"
        ],
        oracle[
            "target_id"
        ].astype(int),
    )
)


oracle[
    "source_is_other_oracle_target"
] = [
    (
        row.dataset,
        int(
            row.source_id
        ),
    )
    in all_target_keys

    for row in oracle.itertuples(
        index=False
    )
]


oracle[
    "target_is_other_oracle_source"
] = [
    (
        row.dataset,
        int(
            row.target_id
        ),
    )
    in all_source_keys

    for row in oracle.itertuples(
        index=False
    )
]


shared_boundary_components = int(
    (
        oracle[
            "source_is_other_oracle_target"
        ]
        |
        oracle[
            "target_is_other_oracle_source"
        ]
    ).sum()
)


print(
    "\n========== ORACLE BOUNDARY SHARING =========="
)

print(
    "components sharing a boundary with another clean gap:",
    shared_boundary_components,
    "/",
    len(
        oracle
    ),
)


# ============================================================
# 14. Geometry ceiling for occupied topology
# ============================================================

print(
    "\n========== OCCUPIED-TOPOLOGY GEOMETRY CEILING =========="
)


for radius in [
    3,
    5,
    7,
]:

    col = (
        f"pred_full_{radius}um"
    )


    good = int(
        occupied[
            col
        ].sum()
    )


    frac = float(
        good
        /
        len(
            occupied
        )
    )


    print(
        f"full <= {radius} um:",
        good,
        "/",
        len(
            occupied
        ),
        f"({frac:.3%})",
    )


# ============================================================
# 15. Fully FP-explained occupied subset geometry
# ============================================================

rewire_oracle = (
    occupied[
        occupied[
            "all_blocking_occupancy_fp_explained"
        ]
    ]
    .copy()
)


print(
    "\n========== FP-EXPLAINED REWIRE ORACLE =========="
)

print(
    "components:",
    len(
        rewire_oracle
    )
)


for radius in [
    3,
    5,
    7,
]:

    col = (
        f"pred_full_{radius}um"
    )


    good = int(
        rewire_oracle[
            col
        ].sum()
    )


    if len(
        rewire_oracle
    ) > 0:

        frac = float(
            good
            /
            len(
                rewire_oracle
            )
        )


        frac_text = (
            f"{frac:.3%}"
        )

    else:

        frac_text = (
            "NA"
        )


    print(
        f"full <= {radius} um:",
        good,
        "/",
        len(
            rewire_oracle
        ),
        "(" + frac_text + ")",
    )


# ============================================================
# 16. Combined free/free + FP-explained ceiling
# ============================================================
#
# This is NOT a deployable policy.
#
# It answers:
#
#   How many clean development gaps are topologically reachable
#   by either:
#
#       A) direct free/free insertion
#       B) removing blocking edges already known (by GT audit)
#          to be official FPs
#
# ============================================================

oracle[
    "oracle_topology_reachable"
] = (
    oracle[
        "topology_class"
    ].eq(
        "FREE_FREE"
    )
    |
    oracle[
        "all_blocking_occupancy_fp_explained"
    ]
)


reachable = int(
    oracle[
        "oracle_topology_reachable"
    ].sum()
)


reachable_fraction = float(
    reachable
    /
    len(
        oracle
    )
)


print(
    "\n========== COMBINED ORACLE TOPOLOGY CEILING =========="
)

print(
    "free/free + fully FP-explained occupied:",
    reachable,
    "/",
    len(
        oracle
    ),
    f"({reachable_fraction:.3%})",
)


reachable_df = oracle[
    oracle[
        "oracle_topology_reachable"
    ]
]


for radius in [
    3,
    5,
    7,
]:

    col = (
        f"pred_full_{radius}um"
    )


    good = int(
        reachable_df[
            col
        ].sum()
    )


    frac = float(
        good
        /
        len(
            reachable_df
        )
    )


    print(
        f"reachable full <= {radius} um:",
        good,
        "/",
        len(
            reachable_df
        ),
        f"({frac:.3%})",
    )


# ============================================================
# 17. By fold
# ============================================================

print(
    "\n========== REWIRE FEASIBILITY BY FOLD =========="
)


fold_table = (
    oracle.groupby(
        "fold"
    )
    .agg(
        clean_components=(
            "component_id",
            "size",
        ),

        free_free=(
            "in_gt_blind_universe",
            "sum",
        ),

        fully_fp_explained=(
            "all_blocking_occupancy_fp_explained",
            "sum",
        ),

        topology_reachable=(
            "oracle_topology_reachable",
            "sum",
        ),

        pred_full_5um=(
            "pred_full_5um",
            "sum",
        ),

        pred_full_7um=(
            "pred_full_7um",
            "sum",
        ),
    )
)


fold_table[
    "reachable_fraction"
] = (
    fold_table[
        "topology_reachable"
    ]
    /
    fold_table[
        "clean_components"
    ]
)


print(
    fold_table.to_string()
)


# ============================================================
# 18. Persist audit
# ============================================================

oracle = (
    oracle.sort_values(
        [
            "fold",
            "dataset",
            "component_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


oracle.to_pickle(
    OUT
)


summary = {
    "stage":
        "12.11G3B",

    "status":
        "OCCUPIED_BOUNDARY_REWIRE_FEASIBILITY_AUDITED",

    "clean_components":
        int(
            len(
                oracle
            )
        ),

    "free_free_components":
        116,

    "occupied_components":
        253,

    "occupied_topology": {
        "source_only":
            66,

        "target_only":
            75,

        "both":
            112,
    },

    "official_valid_fp_edges":
        int(
            len(
                fp
            )
        ),

    "occupied_source_boundaries":
        int(
            len(
                source_occ
            )
        ),

    "occupied_source_fully_fp_explained":
        int(
            source_full
        ),

    "occupied_target_boundaries":
        int(
            len(
                target_occ
            )
        ),

    "occupied_target_fully_fp_explained":
        int(
            target_full
        ),

    "occupied_components_all_blocking_fp_explained":
        int(
            all_full
        ),

    "occupied_components_all_blocking_fp_explained_fraction":
        float(
            all_full
            /
            len(
                occupied
            )
        ),

    "unique_blocking_valid_fp_edges":
        int(
            len(
                blocking_fp
            )
        ),

    "blocking_edges_touching_both_oracle_boundary_types":
        int(
            n_dual_block
        ),

    "oracle_components_with_shared_boundary":
        int(
            shared_boundary_components
        ),

    "combined_oracle_topology_reachable":
        int(
            reachable
        ),

    "combined_oracle_topology_reachable_fraction":
        float(
            reachable_fraction
        ),

    "gt_used":
        True,

    "gt_role":
        "DEVELOPMENT_ORACLE_AUDIT_ONLY",

    "fold0_used":
        False,

    "policy_selected":
        False,

    "threshold_selected":
        False,

    "graph_modified":
        False,
}


SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nwritten:",
    OUT
)

print(
    "written:",
    SUMMARY_OUT
)


# ============================================================
# 19. Integrity
# ============================================================

assert OUT.exists()
assert SUMMARY_OUT.exists()


check = pd.read_pickle(
    OUT
)


assert len(
    check
) == 369


assert int(
    check[
        "topology_class"
    ]
    .eq(
        "FREE_FREE"
    )
    .sum()
) == 116


assert int(
    (
        check[
            "topology_class"
        ]
        .ne(
            "FREE_FREE"
        )
    ).sum()
) == 253


# ============================================================
# 20. Decision
# ============================================================

print(
    "\n========== 12.11G3B DECISION =========="
)

print(
    "OCCUPIED_BOUNDARY_REWIRE_FEASIBILITY_AUDIT: PASS"
)

print(
    "clean components:",
    len(
        oracle
    )
)

print(
    "free/free:",
    116
)

print(
    "occupied:",
    253
)

print(
    "occupied fully FP-explained:",
    all_full,
    "/",
    253,
)

print(
    "combined oracle topology reachable:",
    reachable,
    "/",
    369,
    f"({reachable_fraction:.3%})",
)

print(
    "Fold0 used: NO"
)

print(
    "policy selected: NO"
)

print(
    "graph modified: NO"
)

print(
    "next:"
)

print(
    "BUILD_GT_BLIND_REWIRE_PLUS_SYNTHETIC_CANDIDATE_UNIVERSE"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3C
# GT-Blind CURRENT_BEST Incumbent Edge Feature Universe
# + Development Blocking-FP Separation Audit
#
# Goal:
#
#   Before building a rewire policy, determine whether the
#   incumbent edges that block clean missing-detection gaps can
#   be recognized using ONLY deployment-visible features.
#
# PART A:
#   - exact CURRENT_BEST graph reconstruction
#   - GT-blind incumbent-edge features
#   - artifact is persisted BEFORE labels are loaded
#
# PART B:
#   - load G1 official valid-FP census
#   - load G3B clean-gap occupancy oracle
#   - label:
#       official FP
#       clean-gap blocking FP
#   - audit scalar separation only
#
# NO policy threshold.
# NO classifier training.
# NO graph modification.
# NO Fold0.
#
# Keep after running: YES
# ============================================================

from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

PRED_ROOT = (
    PROJECT
    / "predictions/heqiuyan"
)

S11 = (
    PROJECT
    / "reports"
    / "stage11_division_probe"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


B256_995 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det995"
    / "split_0"
)

B256_9975 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det9975"
    / "split_0"
)


MORPH_PATH = (
    S11
    / "stage11_deployable_final_v9_policy_grid_actions.pkl"
)

V1_PATH = (
    S12
    / "stage12_folds14_mutual_best_universe.pkl"
)

V2_PATH = (
    S12
    / "stage12_folds14_round2_reciprocal_universe.pkl"
)

BRIDGE_PATH = (
    S12
    / "stage12_detection_bridge_policy_actions.pkl"
)


# ------------------------------------------------------------
# GT-blind outputs
# ------------------------------------------------------------

FEATURE_OUT = (
    S12
    / "stage12_folds14_current_best_incumbent_edge_features.pkl"
)

FEATURE_SUMMARY_OUT = (
    S12
    / "stage12_folds14_current_best_incumbent_edge_features_summary.json"
)


# ------------------------------------------------------------
# Labels loaded ONLY AFTER FEATURE_OUT is frozen
# ------------------------------------------------------------

G1_FP_PATH = (
    S12
    / "stage12_folds14_current_best_edge_fp_residuals.pkl"
)

G3B_PATH = (
    S12
    / "stage12_folds14_occupied_boundary_rewire_feasibility.pkl"
)


LABEL_OUT = (
    S12
    / "stage12_folds14_incumbent_edge_separation_audit.pkl"
)

AUDIT_SUMMARY_OUT = (
    S12
    / "stage12_folds14_incumbent_edge_separation_summary.json"
)


SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


for path in [
    TRAIN_ROOT,
    CV_PATH,
    B256_995,
    B256_9975,
    MORPH_PATH,
    V1_PATH,
    V2_PATH,
    BRIDGE_PATH,
]:

    assert path.exists(), path


# ============================================================
# 1. IMPORT
# ============================================================

SRC = str(
    PROJECT
    / "src"
)

if SRC not in sys.path:
    sys.path.insert(
        0,
        SRC,
    )


from biohub_cell_tracking import frozen_v9 as fv9


# ============================================================
# 2. DEVELOPMENT CORPUS
# ============================================================

cv = pd.read_csv(
    CV_PATH
)


corp = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(corp) == 103
assert corp["dataset"].nunique() == 103
assert not corp["fold"].eq(0).any()


names = corp[
    "dataset"
].tolist()


name_set = set(
    names
)


fold_map = dict(
    zip(
        corp[
            "dataset"
        ],
        corp[
            "fold"
        ].astype(int),
    )
)


prefix_map = dict(
    zip(
        corp[
            "dataset"
        ],
        corp[
            "prefix"
        ],
    )
)


# ============================================================
# 3. EXACT CURRENT_BEST ACTIONS
# ============================================================

morph_all = pd.read_pickle(
    MORPH_PATH
)


morph_actions = (
    morph_all[
        morph_all[
            "config_id"
        ].eq(
            "K3_R3_P100"
        )
    ]
    .copy()
)


v1_all = pd.read_pickle(
    V1_PATH
)


v1_actions = (
    v1_all[
        v1_all[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


v2_all = pd.read_pickle(
    V2_PATH
)


v2_actions = (
    v2_all[
        v2_all[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


bridge_all = pd.read_pickle(
    BRIDGE_PATH
)


r1p50_actions = (
    bridge_all[
        bridge_all[
            "policy_id"
        ].eq(
            "R1P50"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


assert len(morph_actions) == 96
assert len(v1_actions) == 46951
assert len(v2_actions) == 18511
assert len(r1p50_actions) == 805


print(
    "========== 12.11G3C CURRENT_BEST CONTRACT =========="
)

print(
    "datasets:",
    len(names)
)

print(
    "MORPH:",
    len(morph_actions)
)

print(
    "V1:",
    len(v1_actions)
)

print(
    "V2:",
    len(v2_actions)
)

print(
    "R1P50:",
    len(r1p50_actions)
)


# ============================================================
# 4. FROZEN V9
# ============================================================

policies = fv9.build_frozen_v9_policies(
    names,
    B256_995,
    B256_9975,
    TRAIN_ROOT,
    prefix_map=prefix_map,
    config=fv9.FROZEN_V9_CONFIG,
    progress=False,
)


assert (
    type(
        policies
    ).__name__
    ==
    "FrozenV9Policies"
)


# ============================================================
# 5. GRAPH BUILDER HELPERS
# ============================================================

def graph_state(
    graph,
):

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )


    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = {
        int(k):
            int(v)

        for k, v in outdeg.items()
    }


    indeg = {
        int(k):
            int(v)

        for k, v in indeg.items()
    }


    edge_set = set(
        zip(
            edges[
                "source_id"
            ].astype(int),
            edges[
                "target_id"
            ].astype(int),
        )
    )


    return (
        nodes,
        outdeg,
        indeg,
        edge_set,
    )


def add_continuation_actions(
    graph,
    nodes,
    table,
    dataset,
):

    rows = table[
        table[
            "dataset"
        ].eq(
            dataset
        )
    ]


    for action in rows.itertuples(
        index=False
    ):

        src = int(
            action.source_pred_node_id
        )

        tgt = int(
            action.target_pred_node_id
        )


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(dist),

                "edge_prob":
                    float(
                        action.candidate_prob
                    ),
            },
            validate_keys=True,
        )


def build_current_best(
    dataset,
):

    graph, scale, _ = fv9.apply_frozen_v9(
        dataset,
        B256_995,
        B256_9975,
        policies,
        TRAIN_ROOT,
        config=fv9.FROZEN_V9_CONFIG,
    )


    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = graph_state(
        graph
    )


    # --------------------------------------------------------
    # MORPH
    # --------------------------------------------------------

    rows = (
        morph_actions[
            morph_actions[
                "dataset"
            ].eq(
                dataset
            )
        ]
        .sort_values(
            [
                "morph_rank",
                "t",
                "source_pred_node_id",
                "target_pred_node_id",
            ],
            kind="stable",
        )
    )


    for action in rows.itertuples(
        index=False
    ):

        src = int(
            action.source_pred_node_id
        )

        tgt = int(
            action.target_pred_node_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 1

        assert indeg.get(
            tgt,
            0,
        ) == 0


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(dist),

                "edge_prob":
                    float(
                        action.candidate_prob
                    ),
            },
            validate_keys=True,
        )


        outdeg[
            src
        ] = 2

        indeg[
            tgt
        ] = 1


    # --------------------------------------------------------
    # V1 / V2
    # --------------------------------------------------------

    add_continuation_actions(
        graph,
        nodes,
        v1_actions,
        dataset,
    )


    add_continuation_actions(
        graph,
        nodes,
        v2_actions,
        dataset,
    )


    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = graph_state(
        graph
    )


    # --------------------------------------------------------
    # R1P50
    # --------------------------------------------------------

    rows = r1p50_actions[
        r1p50_actions[
            "dataset"
        ].eq(
            dataset
        )
    ]


    for action in rows.itertuples(
        index=False
    ):

        src = int(
            action.source_id
        )

        tgt = int(
            action.target_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 0

        assert indeg.get(
            tgt,
            0,
        ) == 0


        new_id = graph.add_node(
            {
                "t":
                    int(
                        action.t
                    ),

                "z":
                    float(
                        action.z
                    ),

                "y":
                    float(
                        action.y
                    ),

                "x":
                    float(
                        action.x
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            src,
            new_id,
            {
                "edge_dist":
                    float(
                        action.src_dist_um
                    ),

                "edge_prob":
                    float(
                        action.in_prob
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            new_id,
            tgt,
            {
                "edge_dist":
                    float(
                        action.tgt_dist_um
                    ),

                "edge_prob":
                    float(
                        action.out_prob
                    ),
            },
            validate_keys=True,
        )


        outdeg[
            src
        ] = 1

        indeg[
            new_id
        ] = 1

        outdeg[
            new_id
        ] = 1

        indeg[
            tgt
        ] = 1


    return (
        graph,
        scale,
    )


# ============================================================
# 6. FEATURE HELPERS
# ============================================================

def norm(
    x,
):

    return float(
        np.linalg.norm(
            np.asarray(
                x,
                dtype=float,
            )
        )
    )


def angle_deg(
    a,
    b,
):

    a = np.asarray(
        a,
        dtype=float,
    )

    b = np.asarray(
        b,
        dtype=float,
    )


    na = norm(a)
    nb = norm(b)


    if (
        na <= 1e-12
        or
        nb <= 1e-12
    ):
        return np.nan


    value = float(
        np.dot(
            a,
            b,
        )
        /
        (
            na
            *
            nb
        )
    )


    value = np.clip(
        value,
        -1.0,
        1.0,
    )


    return float(
        np.degrees(
            np.arccos(
                value
            )
        )
    )


# ============================================================
# PART A
# GT-BLIND EDGE FEATURE GENERATION
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART A — GT-BLIND INCUMBENT EDGE FEATURES"
)

print(
    "============================================================"
)


feature_frames = []


for i, dataset in enumerate(
    names,
    start=1,
):

    fold = int(
        fold_map[
            dataset
        ]
    )


    graph, scale = build_current_best(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    )


    # --------------------------------------------------------
    # Nodes
    # --------------------------------------------------------

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = nodes[
        "node_id"
    ].astype(int)


    nodes[
        "t"
    ] = nodes[
        "t"
    ].astype(int)


    # Sort BEFORE set_index.
    nodes = (
        nodes
        .sort_values(
            [
                "t",
                "node_id",
            ],
            kind="stable",
        )
        .set_index(
            "node_id",
            drop=False,
        )
    )


    # --------------------------------------------------------
    # Edges
    # --------------------------------------------------------

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = edges[
        "source_id"
    ].astype(int)


    edges[
        "target_id"
    ] = edges[
        "target_id"
    ].astype(int)


    # --------------------------------------------------------
    # P995 origins
    # --------------------------------------------------------

    p995 = fv9.load_graph(
        B256_995
        / f"{dataset}.geff"
    )


    p995_ids = set(
        p995.node_attrs(
            attr_keys=[
                "node_id",
            ]
        )
        .to_pandas()[
            "node_id"
        ]
        .astype(int)
        .tolist()
    )


    # --------------------------------------------------------
    # Degree maps
    # --------------------------------------------------------

    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = {
        int(k):
            int(v)

        for k, v in outdeg.items()
    }


    indeg = {
        int(k):
            int(v)

        for k, v in indeg.items()
    }


    # --------------------------------------------------------
    # Unique predecessor/successor
    # --------------------------------------------------------

    incoming = (
        edges.groupby(
            "target_id"
        )[
            "source_id"
        ]
        .apply(list)
        .to_dict()
    )


    outgoing = (
        edges.groupby(
            "source_id"
        )[
            "target_id"
        ]
        .apply(list)
        .to_dict()
    )


    pred_map = {}

    succ_map = {}


    node_ids = nodes[
        "node_id"
    ].astype(int).tolist()


    for nid in node_ids:

        inc = incoming.get(
            nid,
            [],
        )

        out = outgoing.get(
            nid,
            [],
        )


        if len(inc) == 1:
            pred_map[
                nid
            ] = int(
                inc[
                    0
                ]
            )


        if len(out) == 1:
            succ_map[
                nid
            ] = int(
                out[
                    0
                ]
            )


    # --------------------------------------------------------
    # Physical coordinates
    # --------------------------------------------------------

    phys = {}

    time_map = {}


    for nid, row in nodes.iterrows():

        nid = int(nid)


        phys[
            nid
        ] = (
            np.array(
                [
                    float(
                        row["z"]
                    ),
                    float(
                        row["y"]
                    ),
                    float(
                        row["x"]
                    ),
                ],
                dtype=float,
            )
            *
            SCALE
        )


        time_map[
            nid
        ] = int(
            row[
                "t"
            ]
        )


    # --------------------------------------------------------
    # Endpoint local velocities
    # --------------------------------------------------------

    past_velocity = {}

    future_velocity = {}

    previous_displacement = {}

    next_displacement = {}


    for nid in node_ids:

        predecessor = pred_map.get(
            nid
        )


        if predecessor is not None:

            dt = (
                time_map[
                    nid
                ]
                -
                time_map[
                    predecessor
                ]
            )


            if dt > 0:

                vec = (
                    phys[
                        nid
                    ]
                    -
                    phys[
                        predecessor
                    ]
                )


                past_velocity[
                    nid
                ] = (
                    vec
                    /
                    float(dt)
                )


                previous_displacement[
                    nid
                ] = norm(
                    vec
                )


        successor = succ_map.get(
            nid
        )


        if successor is not None:

            dt = (
                time_map[
                    successor
                ]
                -
                time_map[
                    nid
                ]
            )


            if dt > 0:

                vec = (
                    phys[
                        successor
                    ]
                    -
                    phys[
                        nid
                    ]
                )


                future_velocity[
                    nid
                ] = (
                    vec
                    /
                    float(dt)
                )


                next_displacement[
                    nid
                ] = norm(
                    vec
                )


    # --------------------------------------------------------
    # Edge attr columns
    # --------------------------------------------------------

    edge_prob_col = (
        "edge_prob"
        if "edge_prob" in edges.columns
        else None
    )


    edge_dist_col = (
        "edge_dist"
        if "edge_dist" in edges.columns
        else None
    )


    rows = []


    # --------------------------------------------------------
    # GT-blind feature construction
    # --------------------------------------------------------

    for edge in edges.itertuples(
        index=False
    ):

        src = int(
            edge.source_id
        )

        tgt = int(
            edge.target_id
        )


        if (
            src not in phys
            or
            tgt not in phys
        ):
            continue


        src_t = int(
            time_map[
                src
            ]
        )

        tgt_t = int(
            time_map[
                tgt
            ]
        )


        dt = (
            tgt_t
            -
            src_t
        )


        displacement_vec = (
            phys[
                tgt
            ]
            -
            phys[
                src
            ]
        )


        displacement = norm(
            displacement_vec
        )


        if dt > 0:

            straight_velocity = (
                displacement_vec
                /
                float(dt)
            )

            avg_speed = (
                displacement
                /
                float(dt)
            )

        else:

            straight_velocity = np.full(
                3,
                np.nan,
            )

            avg_speed = np.nan


        source_v = past_velocity.get(
            src
        )

        target_v = future_velocity.get(
            tgt
        )


        # ----------------------------------------------------
        # Source forward residual
        # ----------------------------------------------------

        if (
            source_v is not None
            and
            dt > 0
        ):

            predicted_target = (
                phys[
                    src
                ]
                +
                float(dt)
                *
                source_v
            )


            source_cv = norm(
                phys[
                    tgt
                ]
                -
                predicted_target
            )


            source_speed_mismatch = norm(
                straight_velocity
                -
                source_v
            )

        else:

            source_cv = np.nan

            source_speed_mismatch = np.nan


        # ----------------------------------------------------
        # Target backward residual
        # ----------------------------------------------------

        if (
            target_v is not None
            and
            dt > 0
        ):

            predicted_source = (
                phys[
                    tgt
                ]
                -
                float(dt)
                *
                target_v
            )


            target_cv = norm(
                phys[
                    src
                ]
                -
                predicted_source
            )


            target_speed_mismatch = norm(
                straight_velocity
                -
                target_v
            )

        else:

            target_cv = np.nan

            target_speed_mismatch = np.nan


        available_cv = [
            value

            for value in [
                source_cv,
                target_cv,
            ]

            if np.isfinite(
                value
            )
        ]


        if available_cv:

            symmetric_cv = float(
                np.mean(
                    available_cv
                )
            )

        else:

            symmetric_cv = np.nan


        if (
            source_v is not None
            and
            target_v is not None
        ):

            velocity_disagreement = norm(
                source_v
                -
                target_v
            )


            velocity_angle = angle_deg(
                source_v,
                target_v,
            )

        else:

            velocity_disagreement = np.nan

            velocity_angle = np.nan


        prev_disp = previous_displacement.get(
            src,
            np.nan,
        )


        next_disp = next_displacement.get(
            tgt,
            np.nan,
        )


        if (
            np.isfinite(
                prev_disp
            )
            and
            prev_disp > 1e-12
        ):

            displacement_prev_ratio = (
                displacement
                /
                prev_disp
            )

        else:

            displacement_prev_ratio = np.nan


        if (
            np.isfinite(
                next_disp
            )
            and
            next_disp > 1e-12
        ):

            displacement_next_ratio = (
                displacement
                /
                next_disp
            )

        else:

            displacement_next_ratio = np.nan


        # ----------------------------------------------------
        # Extract edge attrs robustly
        # ----------------------------------------------------

        edge_dict = edge._asdict()


        edge_prob = (
            float(
                edge_dict[
                    edge_prob_col
                ]
            )
            if (
                edge_prob_col is not None
                and
                edge_dict.get(
                    edge_prob_col
                ) is not None
            )
            else np.nan
        )


        edge_dist = (
            float(
                edge_dict[
                    edge_dist_col
                ]
            )
            if (
                edge_dist_col is not None
                and
                edge_dict.get(
                    edge_dist_col
                ) is not None
            )
            else np.nan
        )


        row = {
            "dataset":
                dataset,

            "fold":
                fold,

            "source_id":
                src,

            "target_id":
                tgt,

            "source_t":
                src_t,

            "target_t":
                tgt_t,

            "delta_t":
                int(dt),

            "source_indegree":
                int(
                    indeg.get(
                        src,
                        0,
                    )
                ),

            "source_outdegree":
                int(
                    outdeg.get(
                        src,
                        0,
                    )
                ),

            "target_indegree":
                int(
                    indeg.get(
                        tgt,
                        0,
                    )
                ),

            "target_outdegree":
                int(
                    outdeg.get(
                        tgt,
                        0,
                    )
                ),

            # candidate future removal eligibility
            "source_cuttable":
                bool(
                    outdeg.get(
                        src,
                        0,
                    )
                    ==
                    1
                ),

            "target_cuttable":
                bool(
                    indeg.get(
                        tgt,
                        0,
                    )
                    ==
                    1
                ),

            "ordinary_1to1":
                bool(
                    outdeg.get(
                        src,
                        0,
                    )
                    ==
                    1
                    and
                    indeg.get(
                        tgt,
                        0,
                    )
                    ==
                    1
                ),

            "source_origin":
                (
                    "B256_DET995"
                    if src in p995_ids
                    else
                    "POST_P995_ADDED"
                ),

            "target_origin":
                (
                    "B256_DET995"
                    if tgt in p995_ids
                    else
                    "POST_P995_ADDED"
                ),

            "edge_prob":
                edge_prob,

            "edge_dist":
                edge_dist,

            "displacement_um":
                float(
                    displacement
                ),

            "avg_speed_um_per_frame":
                float(
                    avg_speed
                ),

            "source_has_past_velocity":
                bool(
                    source_v
                    is not None
                ),

            "target_has_future_velocity":
                bool(
                    target_v
                    is not None
                ),

            "source_prev_disp_um":
                float(
                    prev_disp
                ),

            "target_next_disp_um":
                float(
                    next_disp
                ),

            "source_cv_residual_um":
                float(
                    source_cv
                ),

            "target_cv_residual_um":
                float(
                    target_cv
                ),

            "symmetric_cv_residual_um":
                float(
                    symmetric_cv
                ),

            "source_speed_mismatch_um_per_frame":
                float(
                    source_speed_mismatch
                ),

            "target_speed_mismatch_um_per_frame":
                float(
                    target_speed_mismatch
                ),

            "velocity_disagreement_um_per_frame":
                float(
                    velocity_disagreement
                ),

            "velocity_angle_deg":
                float(
                    velocity_angle
                ),

            "displacement_prev_ratio":
                float(
                    displacement_prev_ratio
                ),

            "displacement_next_ratio":
                float(
                    displacement_next_ratio
                ),
        }


        rows.append(
            row
        )


    feature_frames.append(
        pd.DataFrame(
            rows
        )
    )


    if (
        i == 1
        or
        i % 10 == 0
        or
        i == len(names)
    ):

        print(
            f"{i:3d}/{len(names)}"
            f" | fold={fold}"
            f" | {dataset}"
            f" | edges={len(rows)}"
        )


# ============================================================
# 7. FREEZE GT-BLIND FEATURE UNIVERSE
# ============================================================

features = pd.concat(
    feature_frames,
    ignore_index=True,
)


assert len(
    features
) > 0


assert not features[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


features = (
    features
    .sort_values(
        [
            "fold",
            "dataset",
            "source_t",
            "source_id",
            "target_t",
            "target_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


HASH_COLS = [
    "dataset",
    "fold",
    "source_id",
    "target_id",
    "source_t",
    "target_t",
]


feature_sha = hashlib.sha256(
    features[
        HASH_COLS
    ]
    .to_csv(
        index=False,
    )
    .encode()
).hexdigest()


features.to_pickle(
    FEATURE_OUT
)


feature_summary = {
    "stage":
        "12.11G3C",

    "artifact":
        "GT_BLIND_CURRENT_BEST_INCUMBENT_EDGE_FEATURES",

    "datasets":
        103,

    "edges":
        int(
            len(
                features
            )
        ),

    "ordinary_1to1_edges":
        int(
            features[
                "ordinary_1to1"
            ].sum()
        ),

    "sha256":
        feature_sha,

    "gt_used_during_feature_generation":
        False,

    "fold0_used":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


FEATURE_SUMMARY_OUT.write_text(
    json.dumps(
        feature_summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert FEATURE_OUT.exists()


print(
    "\n========== GT-BLIND INCUMBENT EDGE UNIVERSE =========="
)

print(
    "edges:",
    len(
        features
    )
)

print(
    "ordinary 1->1:",
    int(
        features[
            "ordinary_1to1"
        ].sum()
    )
)

print(
    "SHA256:",
    feature_sha
)

print(
    "GT-BLIND FEATURE UNIVERSE FROZEN: YES"
)


# ============================================================
# PART B
# DEVELOPMENT LABEL AUDIT
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — DEVELOPMENT FP / BLOCKING LABEL AUDIT"
)

print(
    "============================================================"
)


# ============================================================
# 8. LOAD LABEL ARTIFACTS ONLY NOW
# ============================================================

for path in [
    G1_FP_PATH,
    G3B_PATH,
]:

    assert path.exists(), path


fp = pd.read_pickle(
    G1_FP_PATH
)


g3b = pd.read_pickle(
    G3B_PATH
)


assert len(fp) == 9398
assert len(g3b) == 369


# ============================================================
# 9. OFFICIAL FP LABEL
# ============================================================

fp_keys = set(
    zip(
        fp[
            "dataset"
        ],
        fp[
            "source_pred_node_id"
        ].astype(int),
        fp[
            "target_pred_node_id"
        ].astype(int),
    )
)


features[
    "official_valid_fp"
] = [
    (
        row.dataset,
        int(
            row.source_id
        ),
        int(
            row.target_id
        ),
    )
    in fp_keys

    for row in features.itertuples(
        index=False
    )
]


assert int(
    features[
        "official_valid_fp"
    ].sum()
) == 9398


# ============================================================
# 10. CLEAN-GAP BLOCKING EDGE LABELS
# ============================================================

occupied_source = g3b[
    g3b[
        "source_occupied"
    ]
]


occupied_target = g3b[
    g3b[
        "target_occupied"
    ]
]


source_boundary_keys = set(
    zip(
        occupied_source[
            "dataset"
        ],
        occupied_source[
            "source_id"
        ].astype(int),
    )
)


target_boundary_keys = set(
    zip(
        occupied_target[
            "dataset"
        ],
        occupied_target[
            "target_id"
        ].astype(int),
    )
)


features[
    "blocks_clean_gap_source"
] = [
    (
        row.dataset,
        int(
            row.source_id
        ),
    )
    in source_boundary_keys

    for row in features.itertuples(
        index=False
    )
]


features[
    "blocks_clean_gap_target"
] = [
    (
        row.dataset,
        int(
            row.target_id
        ),
    )
    in target_boundary_keys

    for row in features.itertuples(
        index=False
    )
]


features[
    "blocks_clean_gap"
] = (
    features[
        "blocks_clean_gap_source"
    ]
    |
    features[
        "blocks_clean_gap_target"
    ]
)


n_blocking = int(
    features[
        "blocks_clean_gap"
    ].sum()
)


assert n_blocking == 365


# G3B proved every blocking edge is official FP.
assert features.loc[
    features[
        "blocks_clean_gap"
    ],
    "official_valid_fp",
].all()


print(
    "\n========== LABEL FIDELITY =========="
)

print(
    "official valid FP:",
    int(
        features[
            "official_valid_fp"
        ].sum()
    )
)

print(
    "clean-gap blocking edges:",
    n_blocking
)

print(
    "all blocking edges are official FP:",
    True
)


# ============================================================
# 11. REWIRE-RELEVANT ORDINARY 1->1 UNIVERSE
# ============================================================

ordinary = (
    features[
        features[
            "ordinary_1to1"
        ]
    ]
    .copy()
)


blocking = (
    ordinary[
        ordinary[
            "blocks_clean_gap"
        ]
    ]
    .copy()
)


other_fp = (
    ordinary[
        ordinary[
            "official_valid_fp"
        ]
        &
        ~ordinary[
            "blocks_clean_gap"
        ]
    ]
    .copy()
)


non_fp = (
    ordinary[
        ~ordinary[
            "official_valid_fp"
        ]
    ]
    .copy()
)


print(
    "\n========== REWIRE-RELEVANT EDGE POPULATION =========="
)

print(
    "ordinary 1->1 edges:",
    len(
        ordinary
    )
)

print(
    "blocking clean-gap FP:",
    len(
        blocking
    )
)

print(
    "other official FP:",
    len(
        other_fp
    )
)

print(
    "non-FP ordinary edges:",
    len(
        non_fp
    )
)


blocking_prevalence = float(
    len(
        blocking
    )
    /
    len(
        ordinary
    )
)


print(
    "blocking prevalence:",
    f"{blocking_prevalence:.6%}"
)


# ============================================================
# 12. FEATURE QUANTILE TABLE
# ============================================================

FEATURE_COLS = [
    "edge_prob",
    "edge_dist",
    "displacement_um",
    "avg_speed_um_per_frame",

    "source_cv_residual_um",
    "target_cv_residual_um",
    "symmetric_cv_residual_um",

    "source_speed_mismatch_um_per_frame",
    "target_speed_mismatch_um_per_frame",

    "velocity_disagreement_um_per_frame",
    "velocity_angle_deg",

    "displacement_prev_ratio",
    "displacement_next_ratio",
]


def median_table(
    frame,
):

    result = {}


    for col in FEATURE_COLS:

        values = (
            frame[
                col
            ]
            .replace(
                [
                    np.inf,
                    -np.inf,
                ],
                np.nan,
            )
            .dropna()
        )


        result[
            col
        ] = (
            float(
                values.median()
            )
            if len(values)
            else np.nan
        )


    return result


median_comparison = pd.DataFrame(
    {
        "blocking_clean_gap_fp":
            median_table(
                blocking
            ),

        "other_official_fp":
            median_table(
                other_fp
            ),

        "ordinary_non_fp":
            median_table(
                non_fp
            ),
    }
)


print(
    "\n========== FEATURE MEDIANS =========="
)

print(
    median_comparison.to_string()
)


# ============================================================
# 13. BLOCKING-FP QUANTILES
# ============================================================

print(
    "\n========== BLOCKING FP FEATURE QUANTILES =========="
)


for col in FEATURE_COLS:

    values = (
        blocking[
            col
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
    )


    if len(values) == 0:
        continue


    q = values.quantile(
        [
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )


    print(
        "\n" + col
    )

    print(
        q.to_string()
    )


# ============================================================
# 14. UNIVARIATE SEPARATION AUC
#
# Merely diagnostic:
#   blocking clean-gap FP
#       vs
#   ordinary non-FP
#
# max(AUC, 1-AUC) reports direction-free separability.
#
# NO threshold is selected.
# ============================================================

try:

    from sklearn.metrics import roc_auc_score

    auc_rows = []


    for col in FEATURE_COLS:

        pos = (
            blocking[
                col
            ]
            .replace(
                [
                    np.inf,
                    -np.inf,
                ],
                np.nan,
            )
            .dropna()
        )


        neg = (
            non_fp[
                col
            ]
            .replace(
                [
                    np.inf,
                    -np.inf,
                ],
                np.nan,
            )
            .dropna()
        )


        if (
            len(pos) < 2
            or
            len(neg) < 2
        ):
            continue


        values = np.concatenate(
            [
                pos.to_numpy(
                    dtype=float
                ),
                neg.to_numpy(
                    dtype=float
                ),
            ]
        )


        labels = np.concatenate(
            [
                np.ones(
                    len(pos),
                    dtype=int,
                ),
                np.zeros(
                    len(neg),
                    dtype=int,
                ),
            ]
        )


        auc = float(
            roc_auc_score(
                labels,
                values,
            )
        )


        separation = max(
            auc,
            1.0 - auc,
        )


        direction = (
            "HIGHER_BLOCKING"
            if auc >= 0.5
            else
            "LOWER_BLOCKING"
        )


        auc_rows.append(
            {
                "feature":
                    col,

                "auc":
                    auc,

                "direction_free_auc":
                    separation,

                "direction":
                    direction,

                "blocking_n":
                    int(
                        len(pos)
                    ),

                "non_fp_n":
                    int(
                        len(neg)
                    ),
            }
        )


    auc_df = (
        pd.DataFrame(
            auc_rows
        )
        .sort_values(
            "direction_free_auc",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )


    print(
        "\n========== UNIVARIATE BLOCKING-FP SEPARATION =========="
    )

    print(
        auc_df.to_string(
            index=False
        )
    )

except Exception as exc:

    auc_df = pd.DataFrame()

    print(
        "\nAUC diagnostic unavailable:",
        repr(
            exc
        )
    )


# ============================================================
# 15. FEATURE AVAILABILITY ON BLOCKING EDGES
# ============================================================

print(
    "\n========== BLOCKING EDGE FEATURE AVAILABILITY =========="
)


for col in [
    "source_cv_residual_um",
    "target_cv_residual_um",
    "symmetric_cv_residual_um",
    "velocity_disagreement_um_per_frame",
    "velocity_angle_deg",
]:

    available = float(
        blocking[
            col
        ]
        .notna()
        .mean()
    )


    print(
        col + ":",
        f"{available:.3%}"
    )


# ============================================================
# 16. BLOCKING FP BY FOLD
# ============================================================

print(
    "\n========== BLOCKING FP BY FOLD =========="
)


fold_table = (
    ordinary.groupby(
        "fold"
    )
    .agg(
        ordinary_edges=(
            "source_id",
            "size",
        ),

        official_fp=(
            "official_valid_fp",
            "sum",
        ),

        blocking_fp=(
            "blocks_clean_gap",
            "sum",
        ),
    )
)


fold_table[
    "blocking_prevalence"
] = (
    fold_table[
        "blocking_fp"
    ]
    /
    fold_table[
        "ordinary_edges"
    ]
)


print(
    fold_table.to_string()
)


# ============================================================
# 17. PERSIST LABEL AUDIT
# ============================================================

features.to_pickle(
    LABEL_OUT
)


summary = {
    "stage":
        "12.11G3C",

    "status":
        "INCUMBENT_EDGE_FEATURE_SEPARATION_AUDITED",

    "gt_blind_feature_sha256":
        feature_sha,

    "all_edges":
        int(
            len(
                features
            )
        ),

    "ordinary_1to1_edges":
        int(
            len(
                ordinary
            )
        ),

    "official_valid_fp_edges":
        int(
            features[
                "official_valid_fp"
            ].sum()
        ),

    "clean_gap_blocking_edges":
        int(
            n_blocking
        ),

    "other_ordinary_fp_edges":
        int(
            len(
                other_fp
            )
        ),

    "ordinary_non_fp_edges":
        int(
            len(
                non_fp
            )
        ),

    "blocking_prevalence":
        blocking_prevalence,

    "features_gt_blind":
        True,

    "labels_gt_derived":
        True,

    "labels_loaded_only_after_feature_freeze":
        True,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,

    "top_univariate_features":
        (
            auc_df[
                [
                    "feature",
                    "direction_free_auc",
                    "direction",
                ]
            ]
            .head(
                10
            )
            .to_dict(
                "records"
            )
            if len(
                auc_df
            )
            else []
        ),
}


AUDIT_SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert LABEL_OUT.exists()
assert AUDIT_SUMMARY_OUT.exists()


# ============================================================
# 18. DECISION
# ============================================================

print(
    "\n========== 12.11G3C DECISION =========="
)

print(
    "INCUMBENT_EDGE_FEATURE_SEPARATION_AUDIT: PASS"
)

print(
    "GT-blind features frozen before labels: YES"
)

print(
    "blocking FP edges:",
    n_blocking
)

print(
    "Fold0 used: NO"
)

print(
    "classifier trained: NO"
)

print(
    "threshold selected: NO"
)

print(
    "graph modified: NO"
)

print(
    "next:"
)

print(
    "BUILD_GT_BLIND_REWIRE_PAIR_PROPOSAL_UNIVERSE"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3C
# GT-Blind CURRENT_BEST Incumbent Edge Feature Universe
# + Development Blocking-FP Separation Audit
#
# Goal:
#
#   Before building a rewire policy, determine whether the
#   incumbent edges that block clean missing-detection gaps can
#   be recognized using ONLY deployment-visible features.
#
# PART A:
#   - exact CURRENT_BEST graph reconstruction
#   - GT-blind incumbent-edge features
#   - artifact is persisted BEFORE labels are loaded
#
# PART B:
#   - load G1 official valid-FP census
#   - load G3B clean-gap occupancy oracle
#   - label:
#       official FP
#       clean-gap blocking FP
#   - audit scalar separation only
#
# NO policy threshold.
# NO classifier training.
# NO graph modification.
# NO Fold0.
#
# Keep after running: YES
# ============================================================

from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

PRED_ROOT = (
    PROJECT
    / "predictions/heqiuyan"
)

S11 = (
    PROJECT
    / "reports"
    / "stage11_division_probe"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


B256_995 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det995"
    / "split_0"
)

B256_9975 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det9975"
    / "split_0"
)


MORPH_PATH = (
    S11
    / "stage11_deployable_final_v9_policy_grid_actions.pkl"
)

V1_PATH = (
    S12
    / "stage12_folds14_mutual_best_universe.pkl"
)

V2_PATH = (
    S12
    / "stage12_folds14_round2_reciprocal_universe.pkl"
)

BRIDGE_PATH = (
    S12
    / "stage12_detection_bridge_policy_actions.pkl"
)


# ------------------------------------------------------------
# GT-blind outputs
# ------------------------------------------------------------

FEATURE_OUT = (
    S12
    / "stage12_folds14_current_best_incumbent_edge_features.pkl"
)

FEATURE_SUMMARY_OUT = (
    S12
    / "stage12_folds14_current_best_incumbent_edge_features_summary.json"
)


# ------------------------------------------------------------
# Labels loaded ONLY AFTER FEATURE_OUT is frozen
# ------------------------------------------------------------

G1_FP_PATH = (
    S12
    / "stage12_folds14_current_best_edge_fp_residuals.pkl"
)

G3B_PATH = (
    S12
    / "stage12_folds14_occupied_boundary_rewire_feasibility.pkl"
)


LABEL_OUT = (
    S12
    / "stage12_folds14_incumbent_edge_separation_audit.pkl"
)

AUDIT_SUMMARY_OUT = (
    S12
    / "stage12_folds14_incumbent_edge_separation_summary.json"
)


SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


for path in [
    TRAIN_ROOT,
    CV_PATH,
    B256_995,
    B256_9975,
    MORPH_PATH,
    V1_PATH,
    V2_PATH,
    BRIDGE_PATH,
]:

    assert path.exists(), path


# ============================================================
# 1. IMPORT
# ============================================================

SRC = str(
    PROJECT
    / "src"
)

if SRC not in sys.path:
    sys.path.insert(
        0,
        SRC,
    )


from biohub_cell_tracking import frozen_v9 as fv9


# ============================================================
# 2. DEVELOPMENT CORPUS
# ============================================================

cv = pd.read_csv(
    CV_PATH
)


corp = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(corp) == 103
assert corp["dataset"].nunique() == 103
assert not corp["fold"].eq(0).any()


names = corp[
    "dataset"
].tolist()


name_set = set(
    names
)


fold_map = dict(
    zip(
        corp[
            "dataset"
        ],
        corp[
            "fold"
        ].astype(int),
    )
)


prefix_map = dict(
    zip(
        corp[
            "dataset"
        ],
        corp[
            "prefix"
        ],
    )
)


# ============================================================
# 3. EXACT CURRENT_BEST ACTIONS
# ============================================================

morph_all = pd.read_pickle(
    MORPH_PATH
)


morph_actions = (
    morph_all[
        morph_all[
            "config_id"
        ].eq(
            "K3_R3_P100"
        )
    ]
    .copy()
)


v1_all = pd.read_pickle(
    V1_PATH
)


v1_actions = (
    v1_all[
        v1_all[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


v2_all = pd.read_pickle(
    V2_PATH
)


v2_actions = (
    v2_all[
        v2_all[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


bridge_all = pd.read_pickle(
    BRIDGE_PATH
)


r1p50_actions = (
    bridge_all[
        bridge_all[
            "policy_id"
        ].eq(
            "R1P50"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


assert len(morph_actions) == 96
assert len(v1_actions) == 46951
assert len(v2_actions) == 18511
assert len(r1p50_actions) == 805


print(
    "========== 12.11G3C CURRENT_BEST CONTRACT =========="
)

print(
    "datasets:",
    len(names)
)

print(
    "MORPH:",
    len(morph_actions)
)

print(
    "V1:",
    len(v1_actions)
)

print(
    "V2:",
    len(v2_actions)
)

print(
    "R1P50:",
    len(r1p50_actions)
)


# ============================================================
# 4. FROZEN V9
# ============================================================

policies = fv9.build_frozen_v9_policies(
    names,
    B256_995,
    B256_9975,
    TRAIN_ROOT,
    prefix_map=prefix_map,
    config=fv9.FROZEN_V9_CONFIG,
    progress=False,
)


assert (
    type(
        policies
    ).__name__
    ==
    "FrozenV9Policies"
)


# ============================================================
# 5. GRAPH BUILDER HELPERS
# ============================================================

def graph_state(
    graph,
):

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )


    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = {
        int(k):
            int(v)

        for k, v in outdeg.items()
    }


    indeg = {
        int(k):
            int(v)

        for k, v in indeg.items()
    }


    edge_set = set(
        zip(
            edges[
                "source_id"
            ].astype(int),
            edges[
                "target_id"
            ].astype(int),
        )
    )


    return (
        nodes,
        outdeg,
        indeg,
        edge_set,
    )


def add_continuation_actions(
    graph,
    nodes,
    table,
    dataset,
):

    rows = table[
        table[
            "dataset"
        ].eq(
            dataset
        )
    ]


    for action in rows.itertuples(
        index=False
    ):

        src = int(
            action.source_pred_node_id
        )

        tgt = int(
            action.target_pred_node_id
        )


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(dist),

                "edge_prob":
                    float(
                        action.candidate_prob
                    ),
            },
            validate_keys=True,
        )


def build_current_best(
    dataset,
):

    graph, scale, _ = fv9.apply_frozen_v9(
        dataset,
        B256_995,
        B256_9975,
        policies,
        TRAIN_ROOT,
        config=fv9.FROZEN_V9_CONFIG,
    )


    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = graph_state(
        graph
    )


    # --------------------------------------------------------
    # MORPH
    # --------------------------------------------------------

    rows = (
        morph_actions[
            morph_actions[
                "dataset"
            ].eq(
                dataset
            )
        ]
        .sort_values(
            [
                "morph_rank",
                "t",
                "source_pred_node_id",
                "target_pred_node_id",
            ],
            kind="stable",
        )
    )


    for action in rows.itertuples(
        index=False
    ):

        src = int(
            action.source_pred_node_id
        )

        tgt = int(
            action.target_pred_node_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 1

        assert indeg.get(
            tgt,
            0,
        ) == 0


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(dist),

                "edge_prob":
                    float(
                        action.candidate_prob
                    ),
            },
            validate_keys=True,
        )


        outdeg[
            src
        ] = 2

        indeg[
            tgt
        ] = 1


    # --------------------------------------------------------
    # V1 / V2
    # --------------------------------------------------------

    add_continuation_actions(
        graph,
        nodes,
        v1_actions,
        dataset,
    )


    add_continuation_actions(
        graph,
        nodes,
        v2_actions,
        dataset,
    )


    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = graph_state(
        graph
    )


    # --------------------------------------------------------
    # R1P50
    # --------------------------------------------------------

    rows = r1p50_actions[
        r1p50_actions[
            "dataset"
        ].eq(
            dataset
        )
    ]


    for action in rows.itertuples(
        index=False
    ):

        src = int(
            action.source_id
        )

        tgt = int(
            action.target_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 0

        assert indeg.get(
            tgt,
            0,
        ) == 0


        new_id = graph.add_node(
            {
                "t":
                    int(
                        action.t
                    ),

                "z":
                    float(
                        action.z
                    ),

                "y":
                    float(
                        action.y
                    ),

                "x":
                    float(
                        action.x
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            src,
            new_id,
            {
                "edge_dist":
                    float(
                        action.src_dist_um
                    ),

                "edge_prob":
                    float(
                        action.in_prob
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            new_id,
            tgt,
            {
                "edge_dist":
                    float(
                        action.tgt_dist_um
                    ),

                "edge_prob":
                    float(
                        action.out_prob
                    ),
            },
            validate_keys=True,
        )


        outdeg[
            src
        ] = 1

        indeg[
            new_id
        ] = 1

        outdeg[
            new_id
        ] = 1

        indeg[
            tgt
        ] = 1


    return (
        graph,
        scale,
    )


# ============================================================
# 6. FEATURE HELPERS
# ============================================================

def norm(
    x,
):

    return float(
        np.linalg.norm(
            np.asarray(
                x,
                dtype=float,
            )
        )
    )


def angle_deg(
    a,
    b,
):

    a = np.asarray(
        a,
        dtype=float,
    )

    b = np.asarray(
        b,
        dtype=float,
    )


    na = norm(a)
    nb = norm(b)


    if (
        na <= 1e-12
        or
        nb <= 1e-12
    ):
        return np.nan


    value = float(
        np.dot(
            a,
            b,
        )
        /
        (
            na
            *
            nb
        )
    )


    value = np.clip(
        value,
        -1.0,
        1.0,
    )


    return float(
        np.degrees(
            np.arccos(
                value
            )
        )
    )


# ============================================================
# PART A
# GT-BLIND EDGE FEATURE GENERATION
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART A — GT-BLIND INCUMBENT EDGE FEATURES"
)

print(
    "============================================================"
)


feature_frames = []


for i, dataset in enumerate(
    names,
    start=1,
):

    fold = int(
        fold_map[
            dataset
        ]
    )


    graph, scale = build_current_best(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    )


    # --------------------------------------------------------
    # Nodes
    # --------------------------------------------------------

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = nodes[
        "node_id"
    ].astype(int)


    nodes[
        "t"
    ] = nodes[
        "t"
    ].astype(int)


    # Sort BEFORE set_index.
    nodes = (
        nodes
        .sort_values(
            [
                "t",
                "node_id",
            ],
            kind="stable",
        )
        .set_index(
            "node_id",
            drop=False,
        )
    )


    # --------------------------------------------------------
    # Edges
    # --------------------------------------------------------

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = edges[
        "source_id"
    ].astype(int)


    edges[
        "target_id"
    ] = edges[
        "target_id"
    ].astype(int)


    # --------------------------------------------------------
    # P995 origins
    # --------------------------------------------------------

    p995 = fv9.load_graph(
        B256_995
        / f"{dataset}.geff"
    )


    p995_ids = set(
        p995.node_attrs(
            attr_keys=[
                "node_id",
            ]
        )
        .to_pandas()[
            "node_id"
        ]
        .astype(int)
        .tolist()
    )


    # --------------------------------------------------------
    # Degree maps
    # --------------------------------------------------------

    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = {
        int(k):
            int(v)

        for k, v in outdeg.items()
    }


    indeg = {
        int(k):
            int(v)

        for k, v in indeg.items()
    }


    # --------------------------------------------------------
    # Unique predecessor/successor
    # --------------------------------------------------------

    incoming = (
        edges.groupby(
            "target_id"
        )[
            "source_id"
        ]
        .apply(list)
        .to_dict()
    )


    outgoing = (
        edges.groupby(
            "source_id"
        )[
            "target_id"
        ]
        .apply(list)
        .to_dict()
    )


    pred_map = {}

    succ_map = {}


    node_ids = nodes[
        "node_id"
    ].astype(int).tolist()


    for nid in node_ids:

        inc = incoming.get(
            nid,
            [],
        )

        out = outgoing.get(
            nid,
            [],
        )


        if len(inc) == 1:
            pred_map[
                nid
            ] = int(
                inc[
                    0
                ]
            )


        if len(out) == 1:
            succ_map[
                nid
            ] = int(
                out[
                    0
                ]
            )


    # --------------------------------------------------------
    # Physical coordinates
    # --------------------------------------------------------

    phys = {}

    time_map = {}


    for nid, row in nodes.iterrows():

        nid = int(nid)


        phys[
            nid
        ] = (
            np.array(
                [
                    float(
                        row["z"]
                    ),
                    float(
                        row["y"]
                    ),
                    float(
                        row["x"]
                    ),
                ],
                dtype=float,
            )
            *
            SCALE
        )


        time_map[
            nid
        ] = int(
            row[
                "t"
            ]
        )


    # --------------------------------------------------------
    # Endpoint local velocities
    # --------------------------------------------------------

    past_velocity = {}

    future_velocity = {}

    previous_displacement = {}

    next_displacement = {}


    for nid in node_ids:

        predecessor = pred_map.get(
            nid
        )


        if predecessor is not None:

            dt = (
                time_map[
                    nid
                ]
                -
                time_map[
                    predecessor
                ]
            )


            if dt > 0:

                vec = (
                    phys[
                        nid
                    ]
                    -
                    phys[
                        predecessor
                    ]
                )


                past_velocity[
                    nid
                ] = (
                    vec
                    /
                    float(dt)
                )


                previous_displacement[
                    nid
                ] = norm(
                    vec
                )


        successor = succ_map.get(
            nid
        )


        if successor is not None:

            dt = (
                time_map[
                    successor
                ]
                -
                time_map[
                    nid
                ]
            )


            if dt > 0:

                vec = (
                    phys[
                        successor
                    ]
                    -
                    phys[
                        nid
                    ]
                )


                future_velocity[
                    nid
                ] = (
                    vec
                    /
                    float(dt)
                )


                next_displacement[
                    nid
                ] = norm(
                    vec
                )


    # --------------------------------------------------------
    # Edge attr columns
    # --------------------------------------------------------

    edge_prob_col = (
        "edge_prob"
        if "edge_prob" in edges.columns
        else None
    )


    edge_dist_col = (
        "edge_dist"
        if "edge_dist" in edges.columns
        else None
    )


    rows = []


    # --------------------------------------------------------
    # GT-blind feature construction
    # --------------------------------------------------------

    for edge in edges.itertuples(
        index=False
    ):

        src = int(
            edge.source_id
        )

        tgt = int(
            edge.target_id
        )


        if (
            src not in phys
            or
            tgt not in phys
        ):
            continue


        src_t = int(
            time_map[
                src
            ]
        )

        tgt_t = int(
            time_map[
                tgt
            ]
        )


        dt = (
            tgt_t
            -
            src_t
        )


        displacement_vec = (
            phys[
                tgt
            ]
            -
            phys[
                src
            ]
        )


        displacement = norm(
            displacement_vec
        )


        if dt > 0:

            straight_velocity = (
                displacement_vec
                /
                float(dt)
            )

            avg_speed = (
                displacement
                /
                float(dt)
            )

        else:

            straight_velocity = np.full(
                3,
                np.nan,
            )

            avg_speed = np.nan


        source_v = past_velocity.get(
            src
        )

        target_v = future_velocity.get(
            tgt
        )


        # ----------------------------------------------------
        # Source forward residual
        # ----------------------------------------------------

        if (
            source_v is not None
            and
            dt > 0
        ):

            predicted_target = (
                phys[
                    src
                ]
                +
                float(dt)
                *
                source_v
            )


            source_cv = norm(
                phys[
                    tgt
                ]
                -
                predicted_target
            )


            source_speed_mismatch = norm(
                straight_velocity
                -
                source_v
            )

        else:

            source_cv = np.nan

            source_speed_mismatch = np.nan


        # ----------------------------------------------------
        # Target backward residual
        # ----------------------------------------------------

        if (
            target_v is not None
            and
            dt > 0
        ):

            predicted_source = (
                phys[
                    tgt
                ]
                -
                float(dt)
                *
                target_v
            )


            target_cv = norm(
                phys[
                    src
                ]
                -
                predicted_source
            )


            target_speed_mismatch = norm(
                straight_velocity
                -
                target_v
            )

        else:

            target_cv = np.nan

            target_speed_mismatch = np.nan


        available_cv = [
            value

            for value in [
                source_cv,
                target_cv,
            ]

            if np.isfinite(
                value
            )
        ]


        if available_cv:

            symmetric_cv = float(
                np.mean(
                    available_cv
                )
            )

        else:

            symmetric_cv = np.nan


        if (
            source_v is not None
            and
            target_v is not None
        ):

            velocity_disagreement = norm(
                source_v
                -
                target_v
            )


            velocity_angle = angle_deg(
                source_v,
                target_v,
            )

        else:

            velocity_disagreement = np.nan

            velocity_angle = np.nan


        prev_disp = previous_displacement.get(
            src,
            np.nan,
        )


        next_disp = next_displacement.get(
            tgt,
            np.nan,
        )


        if (
            np.isfinite(
                prev_disp
            )
            and
            prev_disp > 1e-12
        ):

            displacement_prev_ratio = (
                displacement
                /
                prev_disp
            )

        else:

            displacement_prev_ratio = np.nan


        if (
            np.isfinite(
                next_disp
            )
            and
            next_disp > 1e-12
        ):

            displacement_next_ratio = (
                displacement
                /
                next_disp
            )

        else:

            displacement_next_ratio = np.nan


        # ----------------------------------------------------
        # Extract edge attrs robustly
        # ----------------------------------------------------

        edge_dict = edge._asdict()


        edge_prob = (
            float(
                edge_dict[
                    edge_prob_col
                ]
            )
            if (
                edge_prob_col is not None
                and
                edge_dict.get(
                    edge_prob_col
                ) is not None
            )
            else np.nan
        )


        edge_dist = (
            float(
                edge_dict[
                    edge_dist_col
                ]
            )
            if (
                edge_dist_col is not None
                and
                edge_dict.get(
                    edge_dist_col
                ) is not None
            )
            else np.nan
        )


        row = {
            "dataset":
                dataset,

            "fold":
                fold,

            "source_id":
                src,

            "target_id":
                tgt,

            "source_t":
                src_t,

            "target_t":
                tgt_t,

            "delta_t":
                int(dt),

            "source_indegree":
                int(
                    indeg.get(
                        src,
                        0,
                    )
                ),

            "source_outdegree":
                int(
                    outdeg.get(
                        src,
                        0,
                    )
                ),

            "target_indegree":
                int(
                    indeg.get(
                        tgt,
                        0,
                    )
                ),

            "target_outdegree":
                int(
                    outdeg.get(
                        tgt,
                        0,
                    )
                ),

            # candidate future removal eligibility
            "source_cuttable":
                bool(
                    outdeg.get(
                        src,
                        0,
                    )
                    ==
                    1
                ),

            "target_cuttable":
                bool(
                    indeg.get(
                        tgt,
                        0,
                    )
                    ==
                    1
                ),

            "ordinary_1to1":
                bool(
                    outdeg.get(
                        src,
                        0,
                    )
                    ==
                    1
                    and
                    indeg.get(
                        tgt,
                        0,
                    )
                    ==
                    1
                ),

            "source_origin":
                (
                    "B256_DET995"
                    if src in p995_ids
                    else
                    "POST_P995_ADDED"
                ),

            "target_origin":
                (
                    "B256_DET995"
                    if tgt in p995_ids
                    else
                    "POST_P995_ADDED"
                ),

            "edge_prob":
                edge_prob,

            "edge_dist":
                edge_dist,

            "displacement_um":
                float(
                    displacement
                ),

            "avg_speed_um_per_frame":
                float(
                    avg_speed
                ),

            "source_has_past_velocity":
                bool(
                    source_v
                    is not None
                ),

            "target_has_future_velocity":
                bool(
                    target_v
                    is not None
                ),

            "source_prev_disp_um":
                float(
                    prev_disp
                ),

            "target_next_disp_um":
                float(
                    next_disp
                ),

            "source_cv_residual_um":
                float(
                    source_cv
                ),

            "target_cv_residual_um":
                float(
                    target_cv
                ),

            "symmetric_cv_residual_um":
                float(
                    symmetric_cv
                ),

            "source_speed_mismatch_um_per_frame":
                float(
                    source_speed_mismatch
                ),

            "target_speed_mismatch_um_per_frame":
                float(
                    target_speed_mismatch
                ),

            "velocity_disagreement_um_per_frame":
                float(
                    velocity_disagreement
                ),

            "velocity_angle_deg":
                float(
                    velocity_angle
                ),

            "displacement_prev_ratio":
                float(
                    displacement_prev_ratio
                ),

            "displacement_next_ratio":
                float(
                    displacement_next_ratio
                ),
        }


        rows.append(
            row
        )


    feature_frames.append(
        pd.DataFrame(
            rows
        )
    )


    if (
        i == 1
        or
        i % 10 == 0
        or
        i == len(names)
    ):

        print(
            f"{i:3d}/{len(names)}"
            f" | fold={fold}"
            f" | {dataset}"
            f" | edges={len(rows)}"
        )


# ============================================================
# 7. FREEZE GT-BLIND FEATURE UNIVERSE
# ============================================================

features = pd.concat(
    feature_frames,
    ignore_index=True,
)


assert len(
    features
) > 0


assert not features[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


features = (
    features
    .sort_values(
        [
            "fold",
            "dataset",
            "source_t",
            "source_id",
            "target_t",
            "target_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


HASH_COLS = [
    "dataset",
    "fold",
    "source_id",
    "target_id",
    "source_t",
    "target_t",
]


feature_sha = hashlib.sha256(
    features[
        HASH_COLS
    ]
    .to_csv(
        index=False,
    )
    .encode()
).hexdigest()


features.to_pickle(
    FEATURE_OUT
)


feature_summary = {
    "stage":
        "12.11G3C",

    "artifact":
        "GT_BLIND_CURRENT_BEST_INCUMBENT_EDGE_FEATURES",

    "datasets":
        103,

    "edges":
        int(
            len(
                features
            )
        ),

    "ordinary_1to1_edges":
        int(
            features[
                "ordinary_1to1"
            ].sum()
        ),

    "sha256":
        feature_sha,

    "gt_used_during_feature_generation":
        False,

    "fold0_used":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


FEATURE_SUMMARY_OUT.write_text(
    json.dumps(
        feature_summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert FEATURE_OUT.exists()


print(
    "\n========== GT-BLIND INCUMBENT EDGE UNIVERSE =========="
)

print(
    "edges:",
    len(
        features
    )
)

print(
    "ordinary 1->1:",
    int(
        features[
            "ordinary_1to1"
        ].sum()
    )
)

print(
    "SHA256:",
    feature_sha
)

print(
    "GT-BLIND FEATURE UNIVERSE FROZEN: YES"
)


# ============================================================
# PART B
# DEVELOPMENT LABEL AUDIT
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — DEVELOPMENT FP / BLOCKING LABEL AUDIT"
)

print(
    "============================================================"
)


# ============================================================
# 8. LOAD LABEL ARTIFACTS ONLY NOW
# ============================================================

for path in [
    G1_FP_PATH,
    G3B_PATH,
]:

    assert path.exists(), path


fp = pd.read_pickle(
    G1_FP_PATH
)


g3b = pd.read_pickle(
    G3B_PATH
)


assert len(fp) == 9398
assert len(g3b) == 369


# ============================================================
# 9. OFFICIAL FP LABEL
# ============================================================

fp_keys = set(
    zip(
        fp[
            "dataset"
        ],
        fp[
            "source_pred_node_id"
        ].astype(int),
        fp[
            "target_pred_node_id"
        ].astype(int),
    )
)


features[
    "official_valid_fp"
] = [
    (
        row.dataset,
        int(
            row.source_id
        ),
        int(
            row.target_id
        ),
    )
    in fp_keys

    for row in features.itertuples(
        index=False
    )
]


assert int(
    features[
        "official_valid_fp"
    ].sum()
) == 9398


# ============================================================
# 10. CLEAN-GAP BLOCKING EDGE LABELS
# ============================================================

occupied_source = g3b[
    g3b[
        "source_occupied"
    ]
]


occupied_target = g3b[
    g3b[
        "target_occupied"
    ]
]


source_boundary_keys = set(
    zip(
        occupied_source[
            "dataset"
        ],
        occupied_source[
            "source_id"
        ].astype(int),
    )
)


target_boundary_keys = set(
    zip(
        occupied_target[
            "dataset"
        ],
        occupied_target[
            "target_id"
        ].astype(int),
    )
)


features[
    "blocks_clean_gap_source"
] = [
    (
        row.dataset,
        int(
            row.source_id
        ),
    )
    in source_boundary_keys

    for row in features.itertuples(
        index=False
    )
]


features[
    "blocks_clean_gap_target"
] = [
    (
        row.dataset,
        int(
            row.target_id
        ),
    )
    in target_boundary_keys

    for row in features.itertuples(
        index=False
    )
]


features[
    "blocks_clean_gap"
] = (
    features[
        "blocks_clean_gap_source"
    ]
    |
    features[
        "blocks_clean_gap_target"
    ]
)


n_blocking = int(
    features[
        "blocks_clean_gap"
    ].sum()
)


assert n_blocking == 365


# G3B proved every blocking edge is official FP.
assert features.loc[
    features[
        "blocks_clean_gap"
    ],
    "official_valid_fp",
].all()


print(
    "\n========== LABEL FIDELITY =========="
)

print(
    "official valid FP:",
    int(
        features[
            "official_valid_fp"
        ].sum()
    )
)

print(
    "clean-gap blocking edges:",
    n_blocking
)

print(
    "all blocking edges are official FP:",
    True
)


# ============================================================
# 11. REWIRE-RELEVANT ORDINARY 1->1 UNIVERSE
# ============================================================

ordinary = (
    features[
        features[
            "ordinary_1to1"
        ]
    ]
    .copy()
)


blocking = (
    ordinary[
        ordinary[
            "blocks_clean_gap"
        ]
    ]
    .copy()
)


other_fp = (
    ordinary[
        ordinary[
            "official_valid_fp"
        ]
        &
        ~ordinary[
            "blocks_clean_gap"
        ]
    ]
    .copy()
)


non_fp = (
    ordinary[
        ~ordinary[
            "official_valid_fp"
        ]
    ]
    .copy()
)


print(
    "\n========== REWIRE-RELEVANT EDGE POPULATION =========="
)

print(
    "ordinary 1->1 edges:",
    len(
        ordinary
    )
)

print(
    "blocking clean-gap FP:",
    len(
        blocking
    )
)

print(
    "other official FP:",
    len(
        other_fp
    )
)

print(
    "non-FP ordinary edges:",
    len(
        non_fp
    )
)


blocking_prevalence = float(
    len(
        blocking
    )
    /
    len(
        ordinary
    )
)


print(
    "blocking prevalence:",
    f"{blocking_prevalence:.6%}"
)


# ============================================================
# 12. FEATURE QUANTILE TABLE
# ============================================================

FEATURE_COLS = [
    "edge_prob",
    "edge_dist",
    "displacement_um",
    "avg_speed_um_per_frame",

    "source_cv_residual_um",
    "target_cv_residual_um",
    "symmetric_cv_residual_um",

    "source_speed_mismatch_um_per_frame",
    "target_speed_mismatch_um_per_frame",

    "velocity_disagreement_um_per_frame",
    "velocity_angle_deg",

    "displacement_prev_ratio",
    "displacement_next_ratio",
]


def median_table(
    frame,
):

    result = {}


    for col in FEATURE_COLS:

        values = (
            frame[
                col
            ]
            .replace(
                [
                    np.inf,
                    -np.inf,
                ],
                np.nan,
            )
            .dropna()
        )


        result[
            col
        ] = (
            float(
                values.median()
            )
            if len(values)
            else np.nan
        )


    return result


median_comparison = pd.DataFrame(
    {
        "blocking_clean_gap_fp":
            median_table(
                blocking
            ),

        "other_official_fp":
            median_table(
                other_fp
            ),

        "ordinary_non_fp":
            median_table(
                non_fp
            ),
    }
)


print(
    "\n========== FEATURE MEDIANS =========="
)

print(
    median_comparison.to_string()
)


# ============================================================
# 13. BLOCKING-FP QUANTILES
# ============================================================

print(
    "\n========== BLOCKING FP FEATURE QUANTILES =========="
)


for col in FEATURE_COLS:

    values = (
        blocking[
            col
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
    )


    if len(values) == 0:
        continue


    q = values.quantile(
        [
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )


    print(
        "\n" + col
    )

    print(
        q.to_string()
    )


# ============================================================
# 14. UNIVARIATE SEPARATION AUC
#
# Merely diagnostic:
#   blocking clean-gap FP
#       vs
#   ordinary non-FP
#
# max(AUC, 1-AUC) reports direction-free separability.
#
# NO threshold is selected.
# ============================================================

try:

    from sklearn.metrics import roc_auc_score

    auc_rows = []


    for col in FEATURE_COLS:

        pos = (
            blocking[
                col
            ]
            .replace(
                [
                    np.inf,
                    -np.inf,
                ],
                np.nan,
            )
            .dropna()
        )


        neg = (
            non_fp[
                col
            ]
            .replace(
                [
                    np.inf,
                    -np.inf,
                ],
                np.nan,
            )
            .dropna()
        )


        if (
            len(pos) < 2
            or
            len(neg) < 2
        ):
            continue


        values = np.concatenate(
            [
                pos.to_numpy(
                    dtype=float
                ),
                neg.to_numpy(
                    dtype=float
                ),
            ]
        )


        labels = np.concatenate(
            [
                np.ones(
                    len(pos),
                    dtype=int,
                ),
                np.zeros(
                    len(neg),
                    dtype=int,
                ),
            ]
        )


        auc = float(
            roc_auc_score(
                labels,
                values,
            )
        )


        separation = max(
            auc,
            1.0 - auc,
        )


        direction = (
            "HIGHER_BLOCKING"
            if auc >= 0.5
            else
            "LOWER_BLOCKING"
        )


        auc_rows.append(
            {
                "feature":
                    col,

                "auc":
                    auc,

                "direction_free_auc":
                    separation,

                "direction":
                    direction,

                "blocking_n":
                    int(
                        len(pos)
                    ),

                "non_fp_n":
                    int(
                        len(neg)
                    ),
            }
        )


    auc_df = (
        pd.DataFrame(
            auc_rows
        )
        .sort_values(
            "direction_free_auc",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )


    print(
        "\n========== UNIVARIATE BLOCKING-FP SEPARATION =========="
    )

    print(
        auc_df.to_string(
            index=False
        )
    )

except Exception as exc:

    auc_df = pd.DataFrame()

    print(
        "\nAUC diagnostic unavailable:",
        repr(
            exc
        )
    )


# ============================================================
# 15. FEATURE AVAILABILITY ON BLOCKING EDGES
# ============================================================

print(
    "\n========== BLOCKING EDGE FEATURE AVAILABILITY =========="
)


for col in [
    "source_cv_residual_um",
    "target_cv_residual_um",
    "symmetric_cv_residual_um",
    "velocity_disagreement_um_per_frame",
    "velocity_angle_deg",
]:

    available = float(
        blocking[
            col
        ]
        .notna()
        .mean()
    )


    print(
        col + ":",
        f"{available:.3%}"
    )


# ============================================================
# 16. BLOCKING FP BY FOLD
# ============================================================

print(
    "\n========== BLOCKING FP BY FOLD =========="
)


fold_table = (
    ordinary.groupby(
        "fold"
    )
    .agg(
        ordinary_edges=(
            "source_id",
            "size",
        ),

        official_fp=(
            "official_valid_fp",
            "sum",
        ),

        blocking_fp=(
            "blocks_clean_gap",
            "sum",
        ),
    )
)


fold_table[
    "blocking_prevalence"
] = (
    fold_table[
        "blocking_fp"
    ]
    /
    fold_table[
        "ordinary_edges"
    ]
)


print(
    fold_table.to_string()
)


# ============================================================
# 17. PERSIST LABEL AUDIT
# ============================================================

features.to_pickle(
    LABEL_OUT
)


summary = {
    "stage":
        "12.11G3C",

    "status":
        "INCUMBENT_EDGE_FEATURE_SEPARATION_AUDITED",

    "gt_blind_feature_sha256":
        feature_sha,

    "all_edges":
        int(
            len(
                features
            )
        ),

    "ordinary_1to1_edges":
        int(
            len(
                ordinary
            )
        ),

    "official_valid_fp_edges":
        int(
            features[
                "official_valid_fp"
            ].sum()
        ),

    "clean_gap_blocking_edges":
        int(
            n_blocking
        ),

    "other_ordinary_fp_edges":
        int(
            len(
                other_fp
            )
        ),

    "ordinary_non_fp_edges":
        int(
            len(
                non_fp
            )
        ),

    "blocking_prevalence":
        blocking_prevalence,

    "features_gt_blind":
        True,

    "labels_gt_derived":
        True,

    "labels_loaded_only_after_feature_freeze":
        True,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,

    "top_univariate_features":
        (
            auc_df[
                [
                    "feature",
                    "direction_free_auc",
                    "direction",
                ]
            ]
            .head(
                10
            )
            .to_dict(
                "records"
            )
            if len(
                auc_df
            )
            else []
        ),
}


AUDIT_SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert LABEL_OUT.exists()
assert AUDIT_SUMMARY_OUT.exists()


# ============================================================
# 18. DECISION
# ============================================================

print(
    "\n========== 12.11G3C DECISION =========="
)

print(
    "INCUMBENT_EDGE_FEATURE_SEPARATION_AUDIT: PASS"
)

print(
    "GT-blind features frozen before labels: YES"
)

print(
    "blocking FP edges:",
    n_blocking
)

print(
    "Fold0 used: NO"
)

print(
    "classifier trained: NO"
)

print(
    "threshold selected: NO"
)

print(
    "graph modified: NO"
)

print(
    "next:"
)

print(
    "BUILD_GT_BLIND_REWIRE_PAIR_PROPOSAL_UNIVERSE"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3D1
# GT-Blind Incumbent Suspicion Rank / Shortlist Feasibility
#
# Purpose:
#
#   G3C established:
#
#       ordinary incumbent edges = 1,759,090
#       clean-gap blocking FP     = 365
#       prevalence                = 0.020749%
#
#   Edge-only deletion is too risky.
#
#   Before constructing the NEW multi-frame
#   rewire + synthetic candidate universe, determine how broad
#   a GT-blind incumbent-edge anchor shortlist must be to retain
#   the blocking edges.
#
# Method:
#
#   PART A:
#       read GT-blind G3C feature artifact ONLY
#       compute deterministic suspicion ranks
#       freeze ranked universe
#
#   PART B:
#       only after freeze, read G3C labelled audit
#       measure blocking-edge recall at broad shortlist sizes
#
# No classifier.
# No policy threshold selected.
# No graph modification.
# No Fold0.
#
# Keep after running: YES
# ============================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


# GT-BLIND feature artifact produced before G3C labels.
FEATURE_PATH = (
    S12
    / "stage12_folds14_current_best_incumbent_edge_features.pkl"
)


# Labelled G3C artifact.
# MUST NOT be read until ranked universe is frozen.
LABEL_PATH = (
    S12
    / "stage12_folds14_incumbent_edge_separation_audit.pkl"
)


RANK_OUT = (
    S12
    / "stage12_folds14_incumbent_edge_suspicion_ranks.pkl"
)

RANK_SUMMARY_OUT = (
    S12
    / "stage12_folds14_incumbent_edge_suspicion_ranks_summary.json"
)


AUDIT_OUT = (
    S12
    / "stage12_folds14_incumbent_edge_shortlist_feasibility.pkl"
)

AUDIT_SUMMARY_OUT = (
    S12
    / "stage12_folds14_incumbent_edge_shortlist_feasibility_summary.json"
)


assert FEATURE_PATH.exists(), FEATURE_PATH
assert LABEL_PATH.exists(), LABEL_PATH


# ============================================================
# PART A
# GT-BLIND SUSPICION RANK GENERATION
# ============================================================

print(
    "============================================================"
)

print(
    "PART A — GT-BLIND INCUMBENT SUSPICION RANKING"
)

print(
    "============================================================"
)


features = pd.read_pickle(
    FEATURE_PATH
)


assert len(features) == 1763948


# Rewire-relevant ordinary graph edges.
ordinary = (
    features[
        features[
            "ordinary_1to1"
        ].astype(bool)
    ]
    .copy()
)


assert len(ordinary) == 1759090


assert not ordinary[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


assert set(
    ordinary[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}


print(
    "========== G3D1 GT-BLIND INPUT =========="
)

print(
    "all CURRENT_BEST edges:",
    len(features)
)

print(
    "ordinary 1->1 edges:",
    len(ordinary)
)

print(
    "datasets:",
    ordinary[
        "dataset"
    ].nunique()
)


# ============================================================
# 1. Deterministic rank helpers
# ============================================================
#
# Suspicious direction:
#
#   low edge_prob                       -> suspicious
#   high edge_dist                      -> suspicious
#   high displacement                   -> suspicious
#   high symmetric CV residual          -> suspicious
#   high source CV residual             -> suspicious
#   high target CV residual             -> suspicious
#   high previous displacement ratio    -> suspicious
#   high next displacement ratio        -> suspicious
#
# IMPORTANT:
#
# These are generic deployment-visible disagreement measures.
# No GT labels are being read here.
# ============================================================

def percentile_suspicion(
    frame,
    value_col,
    higher_is_suspicious,
):

    values = (
        frame[
            value_col
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )


    if higher_is_suspicious:

        # Larger value => larger suspicion percentile.
        ranks = (
            values.groupby(
                frame[
                    "dataset"
                ]
            )
            .rank(
                method="average",
                pct=True,
                ascending=True,
            )
        )

    else:

        # Smaller value => larger suspicion percentile.
        ranks = (
            values.groupby(
                frame[
                    "dataset"
                ]
            )
            .rank(
                method="average",
                pct=True,
                ascending=False,
            )
        )


    return ranks.astype(
        float
    )


# ============================================================
# 2. Individual suspicion percentiles
# ============================================================

rank_specs = [
    (
        "edge_prob",
        False,
        "susp_edge_prob",
    ),
    (
        "edge_dist",
        True,
        "susp_edge_dist",
    ),
    (
        "displacement_um",
        True,
        "susp_displacement",
    ),
    (
        "symmetric_cv_residual_um",
        True,
        "susp_symmetric_cv",
    ),
    (
        "source_cv_residual_um",
        True,
        "susp_source_cv",
    ),
    (
        "target_cv_residual_um",
        True,
        "susp_target_cv",
    ),
    (
        "displacement_prev_ratio",
        True,
        "susp_prev_ratio",
    ),
    (
        "displacement_next_ratio",
        True,
        "susp_next_ratio",
    ),
]


for (
    value_col,
    higher_is_suspicious,
    output_col,
) in rank_specs:

    ordinary[
        output_col
    ] = percentile_suspicion(
        ordinary,
        value_col,
        higher_is_suspicious,
    )


# ============================================================
# 3. GT-blind composite ranks
# ============================================================
#
# We deliberately create more than one generic composite family.
# This stage does NOT choose among them.
#
# A. geometry composite:
#
#     edge distance
#     displacement
#     symmetric CV
#
# B. local continuity composite:
#
#     symmetric CV
#     prev ratio
#     next ratio
#
# C. broad composite:
#
#     probability
#     distance
#     displacement
#     symmetric CV
#     prev ratio
#     next ratio
#
# nanmean allows missing local trajectory features.
# ============================================================

def row_nanmean(
    frame,
    cols,
):

    arr = (
        frame[
            cols
        ]
        .to_numpy(
            dtype=float
        )
    )


    with np.errstate(
        invalid="ignore",
    ):

        result = np.nanmean(
            arr,
            axis=1,
        )


    return result


ordinary[
    "susp_geometry_composite"
] = row_nanmean(
    ordinary,
    [
        "susp_edge_dist",
        "susp_displacement",
        "susp_symmetric_cv",
    ],
)


ordinary[
    "susp_continuity_composite"
] = row_nanmean(
    ordinary,
    [
        "susp_symmetric_cv",
        "susp_prev_ratio",
        "susp_next_ratio",
    ],
)


ordinary[
    "susp_broad_composite"
] = row_nanmean(
    ordinary,
    [
        "susp_edge_prob",
        "susp_edge_dist",
        "susp_displacement",
        "susp_symmetric_cv",
        "susp_prev_ratio",
        "susp_next_ratio",
    ],
)


# ============================================================
# 4. Rank the composites per dataset
# ============================================================

for col in [
    "susp_geometry_composite",
    "susp_continuity_composite",
    "susp_broad_composite",
]:

    ordinary[
        col
        +
        "_pct"
    ] = (
        ordinary.groupby(
            "dataset"
        )[
            col
        ]
        .rank(
            method="average",
            pct=True,
            ascending=True,
        )
    )


# ============================================================
# 5. Generic rank-union score
#
# max = suspicious if ANY broad family considers the edge
# highly suspicious.
#
# Useful for high-recall anchor generation.
# ============================================================

union_cols = [
    "susp_edge_prob",
    "susp_edge_dist",
    "susp_displacement",
    "susp_symmetric_cv",
    "susp_prev_ratio",
    "susp_next_ratio",
]


ordinary[
    "susp_rank_union"
] = (
    ordinary[
        union_cols
    ]
    .max(
        axis=1,
        skipna=True,
    )
)


ordinary[
    "susp_rank_union_pct"
] = (
    ordinary.groupby(
        "dataset"
    )[
        "susp_rank_union"
    ]
    .rank(
        method="average",
        pct=True,
        ascending=True,
    )
)


# ============================================================
# 6. Stable ordering + hash
# ============================================================

ordinary = (
    ordinary
    .sort_values(
        [
            "fold",
            "dataset",
            "source_t",
            "source_id",
            "target_t",
            "target_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


HASH_COLS = [
    "dataset",
    "fold",
    "source_id",
    "target_id",
    "source_t",
    "target_t",
]


rank_sha = hashlib.sha256(
    ordinary[
        HASH_COLS
    ]
    .to_csv(
        index=False,
    )
    .encode()
).hexdigest()


# ============================================================
# 7. Persist BEFORE labels
# ============================================================

ordinary.to_pickle(
    RANK_OUT
)


rank_summary = {
    "stage":
        "12.11G3D1",

    "artifact":
        "GT_BLIND_INCUMBENT_EDGE_SUSPICION_RANKS",

    "ordinary_edges":
        int(
            len(
                ordinary
            )
        ),

    "datasets":
        int(
            ordinary[
                "dataset"
            ].nunique()
        ),

    "sha256":
        rank_sha,

    "rank_features": [
        spec[
            2
        ]
        for spec in rank_specs
    ],

    "composites": [
        "susp_geometry_composite_pct",
        "susp_continuity_composite_pct",
        "susp_broad_composite_pct",
        "susp_rank_union_pct",
    ],

    "gt_used_during_ranking":
        False,

    "fold0_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


RANK_SUMMARY_OUT.write_text(
    json.dumps(
        rank_summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert RANK_OUT.exists()
assert RANK_SUMMARY_OUT.exists()


print(
    "\n========== GT-BLIND SUSPICION UNIVERSE =========="
)

print(
    "ordinary edges:",
    len(
        ordinary
    )
)

print(
    "SHA256:",
    rank_sha
)

print(
    "GT-BLIND RANK ARTIFACT FROZEN BEFORE LABEL READ: YES"
)


# ============================================================
# PART B
# DEVELOPMENT BLOCKING-EDGE RECALL
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — DEVELOPMENT SHORTLIST FEASIBILITY"
)

print(
    "============================================================"
)


labels = pd.read_pickle(
    LABEL_PATH
)


blocking = (
    labels[
        labels[
            "blocks_clean_gap"
        ].astype(bool)
    ]
    [
        [
            "dataset",
            "fold",
            "source_id",
            "target_id",
        ]
    ]
    .drop_duplicates()
    .copy()
)


assert len(
    blocking
) == 365


blocking[
    "is_blocking"
] = True


audit = ordinary.merge(
    blocking,
    on=[
        "dataset",
        "fold",
        "source_id",
        "target_id",
    ],
    how="left",
    validate="one_to_one",
)


audit[
    "is_blocking"
] = (
    audit[
        "is_blocking"
    ]
    .fillna(
        False
    )
    .astype(bool)
)


assert int(
    audit[
        "is_blocking"
    ].sum()
) == 365


# ============================================================
# 8. Candidate suspicion families
# ============================================================

FAMILIES = {
    "EDGE_PROB":
        "susp_edge_prob",

    "EDGE_DIST":
        "susp_edge_dist",

    "DISPLACEMENT":
        "susp_displacement",

    "SYMMETRIC_CV":
        "susp_symmetric_cv",

    "PREV_RATIO":
        "susp_prev_ratio",

    "NEXT_RATIO":
        "susp_next_ratio",

    "GEOMETRY_COMPOSITE":
        "susp_geometry_composite_pct",

    "CONTINUITY_COMPOSITE":
        "susp_continuity_composite_pct",

    "BROAD_COMPOSITE":
        "susp_broad_composite_pct",

    "RANK_UNION":
        "susp_rank_union_pct",
}


# ============================================================
# 9. Broad shortlist percentages
#
# Still diagnostic only.
#
# A percentile 0.95 means:
#
#       top suspicious 5%
# ============================================================

SHORTLIST_FRACTIONS = [
    0.01,
    0.02,
    0.05,
    0.10,
    0.20,
    0.30,
    0.50,
]


rows = []


for family, score_col in FAMILIES.items():

    for fraction in SHORTLIST_FRACTIONS:

        cutoff = (
            1.0
            -
            float(
                fraction
            )
        )


        selected = (
            audit[
                score_col
            ]
            .ge(
                cutoff
            )
        )


        n_selected = int(
            selected.sum()
        )


        blocking_selected = int(
            (
                selected
                &
                audit[
                    "is_blocking"
                ]
            ).sum()
        )


        recall = float(
            blocking_selected
            /
            365.0
        )


        precision = float(
            blocking_selected
            /
            n_selected
        ) if n_selected else 0.0


        rows.append(
            {
                "family":
                    family,

                "score_col":
                    score_col,

                "shortlist_fraction_target":
                    float(
                        fraction
                    ),

                "percentile_cutoff":
                    float(
                        cutoff
                    ),

                "selected_edges":
                    n_selected,

                "selected_fraction_actual":
                    float(
                        n_selected
                        /
                        len(
                            audit
                        )
                    ),

                "blocking_selected":
                    blocking_selected,

                "blocking_total":
                    365,

                "blocking_recall":
                    recall,

                "blocking_precision":
                    precision,
            }
        )


shortlist = pd.DataFrame(
    rows
)


# ============================================================
# 10. Overall table
# ============================================================

print(
    "\n========== BLOCKING RECALL BY SHORTLIST SIZE =========="
)


pretty = (
    shortlist[
        [
            "family",
            "shortlist_fraction_target",
            "selected_edges",
            "blocking_selected",
            "blocking_recall",
        ]
    ]
    .copy()
)


pretty[
    "shortlist_pct"
] = (
    100.0
    *
    pretty[
        "shortlist_fraction_target"
    ]
)


pretty[
    "blocking_recall_pct"
] = (
    100.0
    *
    pretty[
        "blocking_recall"
    ]
)


print(
    pretty[
        [
            "family",
            "shortlist_pct",
            "selected_edges",
            "blocking_selected",
            "blocking_recall_pct",
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 11. Best family at each shortlist size
# ============================================================

best_by_size = (
    shortlist.sort_values(
        [
            "shortlist_fraction_target",
            "blocking_recall",
            "selected_edges",
        ],
        ascending=[
            True,
            False,
            True,
        ],
    )
    .groupby(
        "shortlist_fraction_target",
        as_index=False,
    )
    .first()
)


print(
    "\n========== BEST FAMILY BY SHORTLIST SIZE =========="
)


print(
    best_by_size[
        [
            "shortlist_fraction_target",
            "family",
            "selected_edges",
            "blocking_selected",
            "blocking_recall",
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 12. Blocking-edge suspicion quantiles
# ============================================================

blocking_rows = audit[
    audit[
        "is_blocking"
    ]
]


print(
    "\n========== BLOCKING EDGE SUSPICION QUANTILES =========="
)


for family, score_col in FAMILIES.items():

    values = (
        blocking_rows[
            score_col
        ]
        .dropna()
    )


    print(
        "\n" + family
    )


    if len(
        values
    ) == 0:

        print(
            "NO VALUES"
        )

        continue


    print(
        values.quantile(
            [
                0.01,
                0.05,
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99,
            ]
        ).to_string()
    )


# ============================================================
# 13. Fold stability
#
# Evaluate the broad 5%, 10%, 20%, 30% cases.
# ============================================================

FOLD_FRACTIONS = [
    0.05,
    0.10,
    0.20,
    0.30,
]


fold_rows = []


for family, score_col in FAMILIES.items():

    for fraction in FOLD_FRACTIONS:

        cutoff = (
            1.0
            -
            fraction
        )


        for fold in [
            1,
            2,
            3,
            4,
        ]:

            sub = audit[
                audit[
                    "fold"
                ].eq(
                    fold
                )
            ]


            selected = sub[
                score_col
            ].ge(
                cutoff
            )


            blocking_total = int(
                sub[
                    "is_blocking"
                ].sum()
            )


            blocking_selected = int(
                (
                    selected
                    &
                    sub[
                        "is_blocking"
                    ]
                ).sum()
            )


            recall = (
                float(
                    blocking_selected
                    /
                    blocking_total
                )
                if blocking_total
                else np.nan
            )


            fold_rows.append(
                {
                    "family":
                        family,

                    "fraction":
                        fraction,

                    "fold":
                        fold,

                    "selected_edges":
                        int(
                            selected.sum()
                        ),

                    "blocking_total":
                        blocking_total,

                    "blocking_selected":
                        blocking_selected,

                    "blocking_recall":
                        recall,
                }
            )


fold_audit = pd.DataFrame(
    fold_rows
)


print(
    "\n========== FOLD STABILITY — RANK UNION =========="
)


rank_union_fold = fold_audit[
    fold_audit[
        "family"
    ].eq(
        "RANK_UNION"
    )
]


print(
    rank_union_fold.to_string(
        index=False
    )
)


print(
    "\n========== FOLD STABILITY — BROAD COMPOSITE =========="
)


broad_fold = fold_audit[
    fold_audit[
        "family"
    ].eq(
        "BROAD_COMPOSITE"
    )
]


print(
    broad_fold.to_string(
        index=False
    )
)


# ============================================================
# 14. Rank-union absolute anchor counts
# ============================================================

print(
    "\n========== RANK-UNION ANCHOR COUNTS =========="
)


for fraction in SHORTLIST_FRACTIONS:

    cutoff = (
        1.0
        -
        fraction
    )


    selected = audit[
        "susp_rank_union_pct"
    ].ge(
        cutoff
    )


    unique_sources = audit.loc[
        selected,
        [
            "dataset",
            "source_id",
        ],
    ].drop_duplicates()


    unique_targets = audit.loc[
        selected,
        [
            "dataset",
            "target_id",
        ],
    ].drop_duplicates()


    recall = float(
        (
            selected
            &
            audit[
                "is_blocking"
            ]
        ).sum()
        /
        365.0
    )


    print(
        f"top {fraction:.0%}"
        f" | edges={int(selected.sum())}"
        f" | source anchors={len(unique_sources)}"
        f" | target anchors={len(unique_targets)}"
        f" | blocking recall={recall:.3%}"
    )


# ============================================================
# 15. Persist audit
# ============================================================

shortlist.to_pickle(
    AUDIT_OUT
)


summary = {
    "stage":
        "12.11G3D1",

    "status":
        "INCUMBENT_SHORTLIST_FEASIBILITY_AUDITED",

    "gt_blind_rank_sha256":
        rank_sha,

    "ordinary_edges":
        int(
            len(
                audit
            )
        ),

    "blocking_edges":
        365,

    "shortlist_fractions":
        SHORTLIST_FRACTIONS,

    "families":
        list(
            FAMILIES.keys()
        ),

    "best_by_shortlist_size":
        best_by_size[
            [
                "shortlist_fraction_target",
                "family",
                "selected_edges",
                "blocking_selected",
                "blocking_recall",
            ]
        ].to_dict(
            "records"
        ),

    "rank_generation_gt_blind":
        True,

    "labels_loaded_only_after_rank_freeze":
        True,

    "gt_role":
        "DEVELOPMENT AUDIT ONLY",

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


AUDIT_SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert AUDIT_OUT.exists()
assert AUDIT_SUMMARY_OUT.exists()


# ============================================================
# 16. Decision
# ============================================================

print(
    "\n========== 12.11G3D1 DECISION =========="
)

print(
    "INCUMBENT_SHORTLIST_FEASIBILITY_AUDIT: PASS"
)

print(
    "ordinary edges:",
    len(
        audit
    )
)

print(
    "blocking edges:",
    365
)

print(
    "GT-blind ranking frozen before labels: YES"
)

print(
    "Fold0 used: NO"
)

print(
    "classifier trained: NO"
)

print(
    "threshold selected: NO"
)

print(
    "graph modified: NO"
)

print(
    "next:"
)

print(
    "FREEZE_BROAD_REWIRE_ANCHOR_CONTRACT_AND_BUILD_MULTIFRAME_TOPK_PROPOSALS"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3D2
# Development Oracle Replacement-vs-Incumbent Advantage Audit
#
# Scientific question:
#
#   G3D1 showed incumbent-only suspicion is insufficient.
#
#   For occupied clean gaps, compare the TRUE multi-frame
#   replacement pair against the blocking CURRENT_BEST edge:
#
#       replacement motion consistency
#               versus
#       incumbent motion consistency
#
#   This determines whether the next candidate generator should
#   be proposal-first and use relative improvement features.
#
# IMPORTANT:
#
#   - G3B clean-gap identity is GT-derived DEVELOPMENT oracle.
#   - This cell is DIAGNOSTIC / DESIGN AUDIT only.
#   - No deployment rule is selected.
#   - No threshold is selected.
#   - No classifier is trained.
#   - No graph is modified.
#   - No Fold0.
#
# Changes files: YES
# Runs training: NO
# Runs inference: NO
# Fold0: NO
# Graph modification: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


G3B_PATH = (
    S12
    / "stage12_folds14_occupied_boundary_rewire_feasibility.pkl"
)

G3C_PATH = (
    S12
    / "stage12_folds14_incumbent_edge_separation_audit.pkl"
)


OUT = (
    S12
    / "stage12_folds14_oracle_replacement_vs_incumbent_advantage.pkl"
)

SUMMARY_OUT = (
    S12
    / "stage12_folds14_oracle_replacement_vs_incumbent_advantage_summary.json"
)


assert G3B_PATH.exists(), G3B_PATH
assert G3C_PATH.exists(), G3C_PATH


SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


# ============================================================
# 1. Exact CURRENT_BEST builder contract
#
# Reuse retained exact builder.
#
# Prefer G3C:
#     build_current_best
#
# Otherwise G3A:
#     build_dev_current_best
#
# Do NOT reconstruct an ad-hoc graph pipeline here.
# ============================================================

if (
    "build_current_best"
    in globals()
    and
    callable(
        globals()[
            "build_current_best"
        ]
    )
):

    current_best_builder = (
        globals()[
            "build_current_best"
        ]
    )

    builder_name = (
        "build_current_best"
    )

elif (
    "build_dev_current_best"
    in globals()
    and
    callable(
        globals()[
            "build_dev_current_best"
        ]
    )
):

    current_best_builder = (
        globals()[
            "build_dev_current_best"
        ]
    )

    builder_name = (
        "build_dev_current_best"
    )

else:

    raise AssertionError(
        "Exact CURRENT_BEST builder is not present in the current "
        "kernel. Re-run retained G3C or the self-contained G3A "
        "before running Stage 12.11G3D2."
    )


print(
    "========== 12.11G3D2 INPUT =========="
)

print(
    "CURRENT_BEST builder:",
    builder_name
)


# ============================================================
# 2. Load development oracle + incumbent-edge audit
# ============================================================

g3b = pd.read_pickle(
    G3B_PATH
)


g3c = pd.read_pickle(
    G3C_PATH
)


# ------------------------------------------------------------
# G3B fidelity
# ------------------------------------------------------------

assert len(
    g3b
) == 369


assert set(
    g3b[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}


assert int(
    g3b[
        "topology_class"
    ]
    .eq(
        "FREE_FREE"
    )
    .sum()
) == 116


# ------------------------------------------------------------
# Occupied clean-gap subset
# ------------------------------------------------------------

occupied = (
    g3b[
        g3b[
            "topology_class"
        ].ne(
            "FREE_FREE"
        )
    ]
    .copy()
)


assert len(
    occupied
) == 253


assert set(
    occupied[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}


assert (
    occupied[
        "source_occupied"
    ].astype(bool)
    |
    occupied[
        "target_occupied"
    ].astype(bool)
).all()


# ------------------------------------------------------------
# Blocking edges from G3C
# ------------------------------------------------------------

blocking_edges = (
    g3c[
        g3c[
            "blocks_clean_gap"
        ].astype(bool)
    ]
    .copy()
)


assert len(
    blocking_edges
) == 365


assert blocking_edges[
    "official_valid_fp"
].astype(bool).all()


assert set(
    blocking_edges[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}


print(
    "clean components:",
    len(
        g3b
    )
)

print(
    "occupied clean components:",
    len(
        occupied
    )
)

print(
    "blocking incumbent edges:",
    len(
        blocking_edges
    )
)


# ============================================================
# 3. Correct occupied-dataset corpus
#
# IMPORTANT FIX:
#
# G3B overall contains 75 clean-gap datasets.
# G3D2 uses only the 253 OCCUPIED components.
#
# Therefore occupied dataset count is not required to equal 75.
# ============================================================

datasets = (
    occupied[
        "dataset"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)


assert len(
    datasets
) > 0


assert (
    len(
        datasets
    )
    ==
    occupied[
        "dataset"
    ].nunique()
)


assert set(
    datasets
).issubset(
    set(
        g3b[
            "dataset"
        ].unique()
    )
)


print(
    "occupied-component datasets:",
    len(
        datasets
    )
)

print(
    "all G3B clean-gap datasets:",
    g3b[
        "dataset"
    ].nunique()
)


# ============================================================
# 4. Helpers
# ============================================================

def vec_norm(
    x,
):

    return float(
        np.linalg.norm(
            np.asarray(
                x,
                dtype=float,
            )
        )
    )


def safe_ratio(
    numerator,
    denominator,
):

    if (
        numerator is None
        or
        denominator is None
    ):

        return np.nan


    numerator = float(
        numerator
    )

    denominator = float(
        denominator
    )


    if (
        not np.isfinite(
            numerator
        )
        or
        not np.isfinite(
            denominator
        )
        or
        abs(
            denominator
        )
        <= 1e-12
    ):

        return np.nan


    return float(
        numerator
        /
        denominator
    )


def safe_improvement_fraction(
    incumbent,
    replacement,
):

    if (
        not np.isfinite(
            incumbent
        )
        or
        not np.isfinite(
            replacement
        )
    ):

        return np.nan


    if (
        incumbent
        <=
        1e-12
    ):

        return np.nan


    return float(
        (
            incumbent
            -
            replacement
        )
        /
        incumbent
    )


def safe_angle_deg(
    a,
    b,
):

    if (
        a is None
        or
        b is None
    ):

        return np.nan


    a = np.asarray(
        a,
        dtype=float,
    )

    b = np.asarray(
        b,
        dtype=float,
    )


    na = float(
        np.linalg.norm(
            a
        )
    )

    nb = float(
        np.linalg.norm(
            b
        )
    )


    if (
        na <= 1e-12
        or
        nb <= 1e-12
    ):

        return np.nan


    cosine = float(
        np.dot(
            a,
            b,
        )
        /
        (
            na
            *
            nb
        )
    )


    cosine = float(
        np.clip(
            cosine,
            -1.0,
            1.0,
        )
    )


    return float(
        np.degrees(
            np.arccos(
                cosine
            )
        )
    )


# ============================================================
# 5. Blocking-edge lookup
# ============================================================

blocking_lookup = {}


for row in blocking_edges.itertuples(
    index=False
):

    key = (
        row.dataset,
        int(
            row.source_id
        ),
        int(
            row.target_id
        ),
    )


    assert key not in blocking_lookup, (
        "Duplicate blocking edge key:",
        key,
    )


    blocking_lookup[
        key
    ] = row


assert len(
    blocking_lookup
) == 365


# ============================================================
# 6. Build oracle comparison rows dataset-by-dataset
# ============================================================

comparison_rows = []


print(
    "\n============================================================"
)

print(
    "BUILD ORACLE REPLACEMENT-vs-INCUMBENT COMPARISONS"
)

print(
    "============================================================"
)


for ds_i, dataset in enumerate(
    datasets,
    start=1,
):

    graph, scale = current_best_builder(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    ), (
        dataset,
        scale,
        SCALE,
    )


    # --------------------------------------------------------
    # Nodes
    # --------------------------------------------------------

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = (
        nodes[
            "node_id"
        ]
        .astype(int)
    )


    nodes[
        "t"
    ] = (
        nodes[
            "t"
        ]
        .astype(int)
    )


    # Sort BEFORE set_index to avoid pandas node_id ambiguity.
    nodes = (
        nodes
        .sort_values(
            [
                "t",
                "node_id",
            ],
            kind="stable",
        )
        .set_index(
            "node_id",
            drop=False,
        )
    )


    # --------------------------------------------------------
    # Edges / topology
    # --------------------------------------------------------

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = (
        edges[
            "source_id"
        ]
        .astype(int)
    )


    edges[
        "target_id"
    ] = (
        edges[
            "target_id"
        ]
        .astype(int)
    )


    incoming = (
        edges.groupby(
            "target_id"
        )[
            "source_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    outgoing = (
        edges.groupby(
            "source_id"
        )[
            "target_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    # --------------------------------------------------------
    # Physical node state
    # --------------------------------------------------------

    phys = {}

    time_map = {}


    for nid, row in nodes.iterrows():

        nid = int(
            nid
        )


        phys[
            nid
        ] = (
            np.array(
                [
                    float(
                        row[
                            "z"
                        ]
                    ),
                    float(
                        row[
                            "y"
                        ]
                    ),
                    float(
                        row[
                            "x"
                        ]
                    ),
                ],
                dtype=float,
            )
            *
            SCALE
        )


        time_map[
            nid
        ] = int(
            row[
                "t"
            ]
        )


    # --------------------------------------------------------
    # Occupied clean-gap components in this dataset
    # --------------------------------------------------------

    ds_components = (
        occupied[
            occupied[
                "dataset"
            ].eq(
                dataset
            )
        ]
        .copy()
    )


    assert len(
        ds_components
    ) > 0


    for comp in ds_components.itertuples(
        index=False
    ):

        src = int(
            comp.source_id
        )

        tgt = int(
            comp.target_id
        )


        assert src in phys, (
            dataset,
            "missing source",
            src,
        )

        assert tgt in phys, (
            dataset,
            "missing target",
            tgt,
        )


        source_t = int(
            time_map[
                src
            ]
        )


        target_t = int(
            time_map[
                tgt
            ]
        )


        delta_t = (
            target_t
            -
            source_t
        )


        expected_dt = (
            int(
                comp.gap_size
            )
            +
            1
        )


        assert delta_t == expected_dt, (
            dataset,
            comp.component_id,
            delta_t,
            expected_dt,
        )


        assert delta_t > 1


        # ====================================================
        # TRUE replacement geometry
        #
        # DEVELOPMENT ORACLE only.
        # ====================================================

        replacement_vec = (
            phys[
                tgt
            ]
            -
            phys[
                src
            ]
        )


        replacement_displacement = vec_norm(
            replacement_vec
        )


        replacement_velocity = (
            replacement_vec
            /
            float(
                delta_t
            )
        )


        replacement_speed = vec_norm(
            replacement_velocity
        )


        # ====================================================
        # Source past trajectory
        # ====================================================

        src_incoming = incoming.get(
            src,
            [],
        )


        source_past_velocity = None

        source_prev_disp = np.nan


        if len(
            src_incoming
        ) == 1:

            src_prev = int(
                src_incoming[
                    0
                ]
            )


            if src_prev in phys:

                dt_prev = (
                    source_t
                    -
                    int(
                        time_map[
                            src_prev
                        ]
                    )
                )


                if dt_prev > 0:

                    source_prev_vec = (
                        phys[
                            src
                        ]
                        -
                        phys[
                            src_prev
                        ]
                    )


                    source_past_velocity = (
                        source_prev_vec
                        /
                        float(
                            dt_prev
                        )
                    )


                    source_prev_disp = vec_norm(
                        source_prev_vec
                    )


        # ====================================================
        # Target future trajectory
        # ====================================================

        tgt_outgoing = outgoing.get(
            tgt,
            [],
        )


        target_future_velocity = None

        target_next_disp = np.nan


        if len(
            tgt_outgoing
        ) == 1:

            tgt_next = int(
                tgt_outgoing[
                    0
                ]
            )


            if tgt_next in phys:

                dt_next = (
                    int(
                        time_map[
                            tgt_next
                        ]
                    )
                    -
                    target_t
                )


                if dt_next > 0:

                    target_next_vec = (
                        phys[
                            tgt_next
                        ]
                        -
                        phys[
                            tgt
                        ]
                    )


                    target_future_velocity = (
                        target_next_vec
                        /
                        float(
                            dt_next
                        )
                    )


                    target_next_disp = vec_norm(
                        target_next_vec
                    )


        # ====================================================
        # Replacement motion residuals
        # ====================================================

        if source_past_velocity is not None:

            source_pred_target = (
                phys[
                    src
                ]
                +
                float(
                    delta_t
                )
                *
                source_past_velocity
            )


            replacement_source_cv = vec_norm(
                phys[
                    tgt
                ]
                -
                source_pred_target
            )


            replacement_source_speed_mismatch = (
                vec_norm(
                    replacement_velocity
                    -
                    source_past_velocity
                )
            )

        else:

            replacement_source_cv = np.nan

            replacement_source_speed_mismatch = np.nan


        if target_future_velocity is not None:

            target_pred_source = (
                phys[
                    tgt
                ]
                -
                float(
                    delta_t
                )
                *
                target_future_velocity
            )


            replacement_target_cv = vec_norm(
                phys[
                    src
                ]
                -
                target_pred_source
            )


            replacement_target_speed_mismatch = (
                vec_norm(
                    replacement_velocity
                    -
                    target_future_velocity
                )
            )

        else:

            replacement_target_cv = np.nan

            replacement_target_speed_mismatch = np.nan


        replacement_cv_values = [
            value

            for value in [
                replacement_source_cv,
                replacement_target_cv,
            ]

            if np.isfinite(
                value
            )
        ]


        replacement_symmetric_cv = (
            float(
                np.mean(
                    replacement_cv_values
                )
            )
            if replacement_cv_values
            else np.nan
        )


        if (
            source_past_velocity is not None
            and
            target_future_velocity is not None
        ):

            replacement_velocity_disagreement = (
                vec_norm(
                    source_past_velocity
                    -
                    target_future_velocity
                )
            )


            replacement_velocity_angle = (
                safe_angle_deg(
                    source_past_velocity,
                    target_future_velocity,
                )
            )

        else:

            replacement_velocity_disagreement = np.nan

            replacement_velocity_angle = np.nan


        # ----------------------------------------------------
        # Compare per-frame replacement speed against adjacent
        # observed per-frame displacements.
        # ----------------------------------------------------

        replacement_prev_ratio = safe_ratio(
            replacement_speed,
            source_prev_disp,
        )


        replacement_next_ratio = safe_ratio(
            replacement_speed,
            target_next_disp,
        )


        # ====================================================
        # Source-side blocking incumbent
        # ====================================================

        source_incumbent = None


        if bool(
            comp.source_occupied
        ):

            current_out = outgoing.get(
                src,
                [],
            )


            assert len(
                current_out
            ) == 1, (
                dataset,
                comp.component_id,
                "source occupied but outdegree != 1",
                current_out,
            )


            incumbent_target = int(
                current_out[
                    0
                ]
            )


            source_key = (
                dataset,
                src,
                incumbent_target,
            )


            assert source_key in blocking_lookup, (
                "Source blocking edge absent from G3C blocking lookup",
                source_key,
            )


            source_incumbent = blocking_lookup[
                source_key
            ]


        # ====================================================
        # Target-side blocking incumbent
        # ====================================================

        target_incumbent = None


        if bool(
            comp.target_occupied
        ):

            current_in = incoming.get(
                tgt,
                [],
            )


            assert len(
                current_in
            ) == 1, (
                dataset,
                comp.component_id,
                "target occupied but indegree != 1",
                current_in,
            )


            incumbent_source = int(
                current_in[
                    0
                ]
            )


            target_key = (
                dataset,
                incumbent_source,
                tgt,
            )


            assert target_key in blocking_lookup, (
                "Target blocking edge absent from G3C blocking lookup",
                target_key,
            )


            target_incumbent = blocking_lookup[
                target_key
            ]


        # ====================================================
        # Incumbent edge collection
        # ====================================================

        incumbent_rows = [
            item

            for item in [
                source_incumbent,
                target_incumbent,
            ]

            if item is not None
        ]


        assert len(
            incumbent_rows
        ) in {
            1,
            2,
        }, (
            dataset,
            comp.component_id,
            len(
                incumbent_rows
            ),
        )


        # ====================================================
        # Incumbent mean helper
        # ====================================================

        def incumbent_mean(
            column,
        ):

            values = []


            for incumbent_row in incumbent_rows:

                value = getattr(
                    incumbent_row,
                    column
                )


                try:

                    value = float(
                        value
                    )

                except (
                    TypeError,
                    ValueError,
                ):

                    continue


                if np.isfinite(
                    value
                ):

                    values.append(
                        value
                    )


            if not values:

                return np.nan


            return float(
                np.mean(
                    values
                )
            )


        # ====================================================
        # Incumbent aggregate features
        # ====================================================

        incumbent_edge_prob = incumbent_mean(
            "edge_prob"
        )


        incumbent_edge_dist = incumbent_mean(
            "edge_dist"
        )


        incumbent_displacement = incumbent_mean(
            "displacement_um"
        )


        incumbent_symmetric_cv = incumbent_mean(
            "symmetric_cv_residual_um"
        )


        incumbent_source_cv = incumbent_mean(
            "source_cv_residual_um"
        )


        incumbent_target_cv = incumbent_mean(
            "target_cv_residual_um"
        )


        incumbent_velocity_disagreement = incumbent_mean(
            "velocity_disagreement_um_per_frame"
        )


        # ====================================================
        # Relative advantages
        #
        # Positive:
        #     TRUE replacement has LOWER residual than incumbent
        # ====================================================

        if (
            np.isfinite(
                incumbent_symmetric_cv
            )
            and
            np.isfinite(
                replacement_symmetric_cv
            )
        ):

            delta_symmetric_cv = (
                incumbent_symmetric_cv
                -
                replacement_symmetric_cv
            )

        else:

            delta_symmetric_cv = np.nan


        if (
            np.isfinite(
                incumbent_source_cv
            )
            and
            np.isfinite(
                replacement_source_cv
            )
        ):

            delta_source_cv = (
                incumbent_source_cv
                -
                replacement_source_cv
            )

        else:

            delta_source_cv = np.nan


        if (
            np.isfinite(
                incumbent_target_cv
            )
            and
            np.isfinite(
                replacement_target_cv
            )
        ):

            delta_target_cv = (
                incumbent_target_cv
                -
                replacement_target_cv
            )

        else:

            delta_target_cv = np.nan


        if (
            np.isfinite(
                incumbent_velocity_disagreement
            )
            and
            np.isfinite(
                replacement_velocity_disagreement
            )
        ):

            delta_velocity_disagreement = (
                incumbent_velocity_disagreement
                -
                replacement_velocity_disagreement
            )

        else:

            delta_velocity_disagreement = np.nan


        if (
            np.isfinite(
                incumbent_displacement
            )
        ):

            replacement_minus_incumbent_displacement = (
                replacement_displacement
                -
                incumbent_displacement
            )

        else:

            replacement_minus_incumbent_displacement = np.nan


        # ====================================================
        # Persist comparison row
        # ====================================================

        comparison_rows.append(
            {
                "dataset":
                    dataset,

                "fold":
                    int(
                        comp.fold
                    ),

                "component_id":
                    comp.component_id,

                "topology_class":
                    comp.topology_class,

                "gap_size":
                    int(
                        comp.gap_size
                    ),

                "source_id":
                    src,

                "target_id":
                    tgt,

                "source_t":
                    source_t,

                "target_t":
                    target_t,

                "source_occupied":
                    bool(
                        comp.source_occupied
                    ),

                "target_occupied":
                    bool(
                        comp.target_occupied
                    ),

                "n_blocking_edges":
                    int(
                        len(
                            incumbent_rows
                        )
                    ),

                # --------------------------------------------
                # TRUE replacement
                # --------------------------------------------

                "replacement_displacement_um":
                    float(
                        replacement_displacement
                    ),

                "replacement_speed_um_per_frame":
                    float(
                        replacement_speed
                    ),

                "replacement_source_cv_um":
                    float(
                        replacement_source_cv
                    ),

                "replacement_target_cv_um":
                    float(
                        replacement_target_cv
                    ),

                "replacement_symmetric_cv_um":
                    float(
                        replacement_symmetric_cv
                    ),

                "replacement_source_speed_mismatch":
                    float(
                        replacement_source_speed_mismatch
                    ),

                "replacement_target_speed_mismatch":
                    float(
                        replacement_target_speed_mismatch
                    ),

                "replacement_velocity_disagreement":
                    float(
                        replacement_velocity_disagreement
                    ),

                "replacement_velocity_angle_deg":
                    float(
                        replacement_velocity_angle
                    ),

                "replacement_prev_speed_ratio":
                    float(
                        replacement_prev_ratio
                    ),

                "replacement_next_speed_ratio":
                    float(
                        replacement_next_ratio
                    ),

                # --------------------------------------------
                # Incumbent means
                # --------------------------------------------

                "incumbent_edge_prob_mean":
                    float(
                        incumbent_edge_prob
                    ),

                "incumbent_edge_dist_mean":
                    float(
                        incumbent_edge_dist
                    ),

                "incumbent_displacement_um_mean":
                    float(
                        incumbent_displacement
                    ),

                "incumbent_source_cv_um_mean":
                    float(
                        incumbent_source_cv
                    ),

                "incumbent_target_cv_um_mean":
                    float(
                        incumbent_target_cv
                    ),

                "incumbent_symmetric_cv_um_mean":
                    float(
                        incumbent_symmetric_cv
                    ),

                "incumbent_velocity_disagreement_mean":
                    float(
                        incumbent_velocity_disagreement
                    ),

                # --------------------------------------------
                # Relative advantages
                # --------------------------------------------

                "replacement_minus_incumbent_displacement":
                    float(
                        replacement_minus_incumbent_displacement
                    ),

                "symmetric_cv_improvement_um":
                    float(
                        delta_symmetric_cv
                    ),

                "source_cv_improvement_um":
                    float(
                        delta_source_cv
                    ),

                "target_cv_improvement_um":
                    float(
                        delta_target_cv
                    ),

                "velocity_disagreement_improvement":
                    float(
                        delta_velocity_disagreement
                    ),

                "symmetric_cv_improvement_fraction":
                    float(
                        safe_improvement_fraction(
                            incumbent_symmetric_cv,
                            replacement_symmetric_cv,
                        )
                    ),

                "source_cv_improvement_fraction":
                    float(
                        safe_improvement_fraction(
                            incumbent_source_cv,
                            replacement_source_cv,
                        )
                    ),

                "target_cv_improvement_fraction":
                    float(
                        safe_improvement_fraction(
                            incumbent_target_cv,
                            replacement_target_cv,
                        )
                    ),

                # --------------------------------------------
                # Existing G2 interpolation quality
                # --------------------------------------------

                "pred_full_3um":
                    bool(
                        comp.pred_full_3um
                    ),

                "pred_full_5um":
                    bool(
                        comp.pred_full_5um
                    ),

                "pred_full_7um":
                    bool(
                        comp.pred_full_7um
                    ),
            }
        )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            datasets
        )
    ):

        print(
            f"{ds_i:2d}/{len(datasets)}"
            f" | {dataset}"
            f" | oracle_components={len(ds_components)}"
        )


# ============================================================
# 7. Canonical audit table
# ============================================================

audit = pd.DataFrame(
    comparison_rows
)


assert len(
    audit
) == 253


assert not audit[
    [
        "dataset",
        "component_id",
    ]
].duplicated().any()


assert set(
    audit[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}


assert int(
    audit[
        "n_blocking_edges"
    ].sum()
) == 365


print(
    "\n========== REPLACEMENT-vs-INCUMBENT POPULATION =========="
)

print(
    "occupied components:",
    len(
        audit
    )
)

print(
    "datasets:",
    audit[
        "dataset"
    ].nunique()
)

print(
    "topology:"
)

print(
    audit[
        "topology_class"
    ]
    .value_counts()
    .to_string()
)


print(
    "\nblocking edges per component:"
)

print(
    audit[
        "n_blocking_edges"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


# ============================================================
# 8. Replacement feature availability
# ============================================================

print(
    "\n========== REPLACEMENT FEATURE AVAILABILITY =========="
)


availability_cols = [
    "replacement_source_cv_um",
    "replacement_target_cv_um",
    "replacement_symmetric_cv_um",
    "replacement_velocity_disagreement",

    "symmetric_cv_improvement_um",
    "source_cv_improvement_um",
    "target_cv_improvement_um",
    "velocity_disagreement_improvement",
]


for col in availability_cols:

    fraction = float(
        audit[
            col
        ]
        .notna()
        .mean()
    )


    print(
        col + ":",
        f"{fraction:.3%}"
    )


# ============================================================
# 9. Absolute replacement vs incumbent medians
# ============================================================

comparison_pairs = [
    (
        "replacement_displacement_um",
        "incumbent_displacement_um_mean",
    ),
    (
        "replacement_source_cv_um",
        "incumbent_source_cv_um_mean",
    ),
    (
        "replacement_target_cv_um",
        "incumbent_target_cv_um_mean",
    ),
    (
        "replacement_symmetric_cv_um",
        "incumbent_symmetric_cv_um_mean",
    ),
    (
        "replacement_velocity_disagreement",
        "incumbent_velocity_disagreement_mean",
    ),
]


print(
    "\n========== REPLACEMENT vs INCUMBENT MEDIANS =========="
)


for replacement_col, incumbent_col in comparison_pairs:

    replacement_values = (
        audit[
            replacement_col
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
    )


    incumbent_values = (
        audit[
            incumbent_col
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
    )


    replacement_median = (
        float(
            replacement_values.median()
        )
        if len(
            replacement_values
        )
        else np.nan
    )


    incumbent_median = (
        float(
            incumbent_values.median()
        )
        if len(
            incumbent_values
        )
        else np.nan
    )


    print(
        "\n" + replacement_col
    )

    print(
        "replacement median:",
        replacement_median
    )

    print(
        "incumbent median:",
        incumbent_median
    )


# ============================================================
# 10. Relative-improvement distributions
# ============================================================

IMPROVEMENT_COLS = [
    "symmetric_cv_improvement_um",
    "source_cv_improvement_um",
    "target_cv_improvement_um",

    "velocity_disagreement_improvement",

    "symmetric_cv_improvement_fraction",
    "source_cv_improvement_fraction",
    "target_cv_improvement_fraction",
]


print(
    "\n========== RELATIVE IMPROVEMENT QUANTILES =========="
)


for col in IMPROVEMENT_COLS:

    values = (
        audit[
            col
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
    )


    print(
        "\n" + col
    )


    if len(
        values
    ) == 0:

        print(
            "NO VALUES"
        )

        continue


    print(
        values.quantile(
            [
                0.01,
                0.05,
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99,
            ]
        ).to_string()
    )


# ============================================================
# 11. TRUE replacement better-than-incumbent rate
# ============================================================

print(
    "\n========== TRUE REPLACEMENT BETTER-THAN-INCUMBENT RATE =========="
)


better_specs = [
    (
        "symmetric_cv_improvement_um",
        "symmetric CV",
    ),
    (
        "source_cv_improvement_um",
        "source CV",
    ),
    (
        "target_cv_improvement_um",
        "target CV",
    ),
    (
        "velocity_disagreement_improvement",
        "velocity disagreement",
    ),
]


better_rate_rows = []


for col, label in better_specs:

    valid = (
        audit[
            col
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
    )


    if len(
        valid
    ) == 0:

        continue


    better = int(
        valid.gt(
            0
        ).sum()
    )


    fraction = float(
        better
        /
        len(
            valid
        )
    )


    better_rate_rows.append(
        {
            "metric":
                label,

            "valid":
                int(
                    len(
                        valid
                    )
                ),

            "replacement_better":
                better,

            "fraction":
                fraction,
        }
    )


    print(
        label + ":",
        better,
        "/",
        len(
            valid
        ),
        f"({fraction:.3%})",
    )


better_rate_df = pd.DataFrame(
    better_rate_rows
)


# ============================================================
# 12. Joint CV advantage
#
# Missing metrics are NOT counted as better.
# ============================================================

sym_valid = audit[
    "symmetric_cv_improvement_um"
].notna()

src_valid = audit[
    "source_cv_improvement_um"
].notna()

tgt_valid = audit[
    "target_cv_improvement_um"
].notna()


sym_better = (
    sym_valid
    &
    audit[
        "symmetric_cv_improvement_um"
    ].gt(
        0
    )
)


src_better = (
    src_valid
    &
    audit[
        "source_cv_improvement_um"
    ].gt(
        0
    )
)


tgt_better = (
    tgt_valid
    &
    audit[
        "target_cv_improvement_um"
    ].gt(
        0
    )
)


available_count = (
    sym_valid.astype(int)
    +
    src_valid.astype(int)
    +
    tgt_valid.astype(int)
)


better_count = (
    sym_better.astype(int)
    +
    src_better.astype(int)
    +
    tgt_better.astype(int)
)


audit[
    "cv_metrics_available"
] = available_count


audit[
    "cv_metrics_better"
] = better_count


audit[
    "any_cv_better"
] = (
    better_count
    >=
    1
)


audit[
    "two_of_three_cv_better"
] = (
    better_count
    >=
    2
)


audit[
    "all_available_cv_better"
] = (
    (
        available_count
        >
        0
    )
    &
    (
        better_count
        ==
        available_count
    )
)


print(
    "\n========== JOINT CV ADVANTAGE =========="
)


for col in [
    "any_cv_better",
    "two_of_three_cv_better",
    "all_available_cv_better",
]:

    count = int(
        audit[
            col
        ].sum()
    )


    print(
        col + ":",
        count,
        "/",
        len(
            audit
        ),
        f"({count/len(audit):.3%})",
    )


print(
    "\nCV metrics available per component:"
)

print(
    audit[
        "cv_metrics_available"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


# ============================================================
# 13. Advantage by topology
# ============================================================

print(
    "\n========== ADVANTAGE BY TOPOLOGY =========="
)


topology_table = (
    audit.groupby(
        "topology_class"
    )
    .agg(
        components=(
            "component_id",
            "size",
        ),

        median_gap_size=(
            "gap_size",
            "median",
        ),

        median_replacement_symmetric_cv=(
            "replacement_symmetric_cv_um",
            "median",
        ),

        median_incumbent_symmetric_cv=(
            "incumbent_symmetric_cv_um_mean",
            "median",
        ),

        median_symmetric_improvement=(
            "symmetric_cv_improvement_um",
            "median",
        ),

        any_cv_better=(
            "any_cv_better",
            "sum",
        ),

        two_of_three_cv_better=(
            "two_of_three_cv_better",
            "sum",
        ),

        all_available_cv_better=(
            "all_available_cv_better",
            "sum",
        ),

        pred_full_5um=(
            "pred_full_5um",
            "sum",
        ),

        pred_full_7um=(
            "pred_full_7um",
            "sum",
        ),
    )
)


topology_table[
    "any_cv_better_fraction"
] = (
    topology_table[
        "any_cv_better"
    ]
    /
    topology_table[
        "components"
    ]
)


topology_table[
    "two_of_three_fraction"
] = (
    topology_table[
        "two_of_three_cv_better"
    ]
    /
    topology_table[
        "components"
    ]
)


topology_table[
    "all_available_fraction"
] = (
    topology_table[
        "all_available_cv_better"
    ]
    /
    topology_table[
        "components"
    ]
)


print(
    topology_table.to_string()
)


# ============================================================
# 14. Advantage by gap size
# ============================================================

print(
    "\n========== ADVANTAGE BY GAP SIZE =========="
)


gap_table = (
    audit.groupby(
        "gap_size"
    )
    .agg(
        components=(
            "component_id",
            "size",
        ),

        symmetric_valid=(
            "symmetric_cv_improvement_um",
            "count",
        ),

        median_symmetric_improvement=(
            "symmetric_cv_improvement_um",
            "median",
        ),

        any_cv_better=(
            "any_cv_better",
            "sum",
        ),

        two_of_three_cv_better=(
            "two_of_three_cv_better",
            "sum",
        ),

        all_available_cv_better=(
            "all_available_cv_better",
            "sum",
        ),

        pred_full_5um=(
            "pred_full_5um",
            "sum",
        ),

        pred_full_7um=(
            "pred_full_7um",
            "sum",
        ),
    )
)


gap_table[
    "any_cv_better_fraction"
] = (
    gap_table[
        "any_cv_better"
    ]
    /
    gap_table[
        "components"
    ]
)


gap_table[
    "two_of_three_fraction"
] = (
    gap_table[
        "two_of_three_cv_better"
    ]
    /
    gap_table[
        "components"
    ]
)


gap_table[
    "all_available_fraction"
] = (
    gap_table[
        "all_available_cv_better"
    ]
    /
    gap_table[
        "components"
    ]
)


print(
    gap_table.to_string()
)


# ============================================================
# 15. Fold stability
# ============================================================

print(
    "\n========== ADVANTAGE BY FOLD =========="
)


fold_table = (
    audit.groupby(
        "fold"
    )
    .agg(
        components=(
            "component_id",
            "size",
        ),

        symmetric_valid=(
            "symmetric_cv_improvement_um",
            "count",
        ),

        median_symmetric_improvement=(
            "symmetric_cv_improvement_um",
            "median",
        ),

        any_cv_better=(
            "any_cv_better",
            "sum",
        ),

        two_of_three_cv_better=(
            "two_of_three_cv_better",
            "sum",
        ),

        all_available_cv_better=(
            "all_available_cv_better",
            "sum",
        ),
    )
)


fold_table[
    "any_cv_better_fraction"
] = (
    fold_table[
        "any_cv_better"
    ]
    /
    fold_table[
        "components"
    ]
)


fold_table[
    "two_of_three_fraction"
] = (
    fold_table[
        "two_of_three_cv_better"
    ]
    /
    fold_table[
        "components"
    ]
)


fold_table[
    "all_available_fraction"
] = (
    fold_table[
        "all_available_cv_better"
    ]
    /
    fold_table[
        "components"
    ]
)


print(
    fold_table.to_string()
)


# ============================================================
# 16. Advantage vs interpolation quality
# ============================================================

print(
    "\n========== ADVANTAGE vs INTERPOLATION QUALITY =========="
)


for quality_col in [
    "pred_full_3um",
    "pred_full_5um",
    "pred_full_7um",
]:

    quality_mask = audit[
        quality_col
    ].astype(bool)


    good = audit[
        quality_mask
    ]


    bad = audit[
        ~quality_mask
    ]


    print(
        "\n" + quality_col
    )


    print(
        "good n:",
        len(
            good
        )
    )


    good_values = (
        good[
            "symmetric_cv_improvement_um"
        ]
        .dropna()
    )


    bad_values = (
        bad[
            "symmetric_cv_improvement_um"
        ]
        .dropna()
    )


    print(
        "good median symmetric improvement:",
        (
            float(
                good_values.median()
            )
            if len(
                good_values
            )
            else np.nan
        )
    )


    print(
        "good any-CV-better:",
        int(
            good[
                "any_cv_better"
            ].sum()
        ),
        "/",
        len(
            good
        ),
        (
            f"({good['any_cv_better'].mean():.3%})"
            if len(
                good
            )
            else "(NA)"
        ),
    )


    print(
        "bad n:",
        len(
            bad
        )
    )


    print(
        "bad median symmetric improvement:",
        (
            float(
                bad_values.median()
            )
            if len(
                bad_values
            )
            else np.nan
        )
    )


    print(
        "bad any-CV-better:",
        int(
            bad[
                "any_cv_better"
            ].sum()
        ),
        "/",
        len(
            bad
        ),
        (
            f"({bad['any_cv_better'].mean():.3%})"
            if len(
                bad
            )
            else "(NA)"
        ),
    )


# ============================================================
# 17. Persist formal audit artifact
# ============================================================

audit = (
    audit
    .sort_values(
        [
            "fold",
            "dataset",
            "component_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


audit.to_pickle(
    OUT
)


summary = {
    "stage":
        "12.11G3D2",

    "status":
        "ORACLE_REPLACEMENT_VS_INCUMBENT_ADVANTAGE_AUDITED",

    "clean_components_total":
        369,

    "free_free_components":
        116,

    "occupied_components":
        int(
            len(
                audit
            )
        ),

    "occupied_datasets":
        int(
            audit[
                "dataset"
            ].nunique()
        ),

    "all_clean_gap_datasets":
        int(
            g3b[
                "dataset"
            ].nunique()
        ),

    "blocking_incumbent_edges":
        int(
            len(
                blocking_edges
            )
        ),

    "better_rates":
        better_rate_df.to_dict(
            "records"
        ),

    "joint_advantage": {
        "any_cv_better":
            int(
                audit[
                    "any_cv_better"
                ].sum()
            ),

        "two_of_three_cv_better":
            int(
                audit[
                    "two_of_three_cv_better"
                ].sum()
            ),

        "all_available_cv_better":
            int(
                audit[
                    "all_available_cv_better"
                ].sum()
            ),
    },

    "gt_used":
        True,

    "gt_role":
        "DEVELOPMENT_ORACLE_DESIGN_AUDIT_ONLY",

    "fold0_used":
        False,

    "policy_selected":
        False,

    "threshold_selected":
        False,

    "classifier_trained":
        False,

    "graph_modified":
        False,
}


SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert OUT.exists()
assert SUMMARY_OUT.exists()


# ============================================================
# 18. Reload fidelity gate
# ============================================================

check = pd.read_pickle(
    OUT
)


assert len(
    check
) == 253


assert int(
    check[
        "n_blocking_edges"
    ].sum()
) == 365


assert set(
    check[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}


assert not check[
    [
        "dataset",
        "component_id",
    ]
].duplicated().any()


# ============================================================
# 19. Final decision
# ============================================================

print(
    "\n========== 12.11G3D2 DECISION =========="
)

print(
    "ORACLE_REPLACEMENT_VS_INCUMBENT_ADVANTAGE_AUDIT: PASS"
)

print(
    "occupied components:",
    len(
        audit
    )
)

print(
    "occupied datasets:",
    audit[
        "dataset"
    ].nunique()
)

print(
    "blocking incumbent edges:",
    int(
        audit[
            "n_blocking_edges"
        ].sum()
    )
)

print(
    "Fold0 used: NO"
)

print(
    "policy selected: NO"
)

print(
    "threshold selected: NO"
)

print(
    "classifier trained: NO"
)

print(
    "graph modified: NO"
)

print(
    "next:"
)

print(
    "DESIGN_PROPOSAL_FIRST_MULTIFRAME_TOPK_GENERATOR"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3D2-R
# Time-Normalized / Side-Matched
# Replacement-vs-Incumbent Advantage Audit
#
# Why this repair is necessary:
#
#   Original G3D2 compared:
#
#       multi-frame replacement CV extrapolation residual
#
#   directly against:
#
#       mostly one-frame incumbent-edge CV residual
#
#   That comparison is temporally confounded because CV
#   position error naturally accumulates with delta_t.
#
# This repair compares:
#
#   A) CV residual / delta_t
#
#   B) velocity-space mismatch, already measured in um/frame
#
#   C) SOURCE replacement only against SOURCE blocker
#
#   D) TARGET replacement only against TARGET blocker
#
# Development GT oracle is used ONLY to identify the known
# clean-gap replacement and blocking incumbent edge.
#
# No deployment rule.
# No threshold.
# No classifier.
# No graph modification.
# No Fold0.
#
# Keep after running: YES
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


G3D2_PATH = (
    S12
    / "stage12_folds14_oracle_replacement_vs_incumbent_advantage.pkl"
)

G3C_PATH = (
    S12
    / "stage12_folds14_incumbent_edge_separation_audit.pkl"
)


OUT = (
    S12
    / "stage12_folds14_oracle_replacement_vs_incumbent_advantage_normalized.pkl"
)

SUMMARY_OUT = (
    S12
    / "stage12_folds14_oracle_replacement_vs_incumbent_advantage_normalized_summary.json"
)


assert G3D2_PATH.exists(), G3D2_PATH
assert G3C_PATH.exists(), G3C_PATH


# ============================================================
# 1. Load
# ============================================================

oracle = pd.read_pickle(
    G3D2_PATH
)


edges = pd.read_pickle(
    G3C_PATH
)


assert len(oracle) == 253


blocking = (
    edges[
        edges[
            "blocks_clean_gap"
        ].astype(bool)
    ]
    .copy()
)


assert len(blocking) == 365


assert blocking[
    "official_valid_fp"
].astype(bool).all()


print(
    "========== 12.11G3D2-R INPUT =========="
)

print(
    "occupied oracle components:",
    len(oracle)
)

print(
    "blocking incumbent edges:",
    len(blocking)
)

print(
    "folds:",
    sorted(
        oracle[
            "fold"
        ]
        .astype(int)
        .unique()
        .tolist()
    )
)


# ============================================================
# 2. Required feature contract
# ============================================================

required_oracle_cols = [
    "dataset",
    "fold",
    "component_id",
    "topology_class",
    "gap_size",
    "source_id",
    "target_id",
    "source_occupied",
    "target_occupied",

    "replacement_source_cv_um",
    "replacement_target_cv_um",
    "replacement_symmetric_cv_um",

    "replacement_source_speed_mismatch",
    "replacement_target_speed_mismatch",

    "pred_full_3um",
    "pred_full_5um",
    "pred_full_7um",
]


required_edge_cols = [
    "dataset",
    "source_id",
    "target_id",
    "delta_t",

    "blocks_clean_gap",
    "blocks_clean_gap_source",
    "blocks_clean_gap_target",

    "source_cv_residual_um",
    "target_cv_residual_um",
    "symmetric_cv_residual_um",

    "source_speed_mismatch_um_per_frame",
    "target_speed_mismatch_um_per_frame",

    "official_valid_fp",
]


missing_oracle = [
    col
    for col in required_oracle_cols
    if col not in oracle.columns
]


missing_edge = [
    col
    for col in required_edge_cols
    if col not in blocking.columns
]


assert not missing_oracle, (
    "Missing oracle columns:",
    missing_oracle,
)


assert not missing_edge, (
    "Missing blocking-edge columns:",
    missing_edge,
)


# ============================================================
# 3. Helpers
# ============================================================

def finite_or_nan(
    value,
):

    try:
        value = float(value)

    except (
        TypeError,
        ValueError,
    ):
        return np.nan


    if not np.isfinite(value):
        return np.nan


    return value


def safe_divide(
    value,
    denominator,
):

    value = finite_or_nan(value)
    denominator = finite_or_nan(denominator)


    if (
        not np.isfinite(value)
        or
        not np.isfinite(denominator)
        or
        denominator <= 0
    ):
        return np.nan


    return float(
        value
        /
        denominator
    )


def improvement(
    incumbent,
    replacement,
):

    incumbent = finite_or_nan(
        incumbent
    )

    replacement = finite_or_nan(
        replacement
    )


    if (
        not np.isfinite(incumbent)
        or
        not np.isfinite(replacement)
    ):
        return np.nan


    # Positive = replacement is better / lower-cost.
    return float(
        incumbent
        -
        replacement
    )


def improvement_fraction(
    incumbent,
    replacement,
):

    incumbent = finite_or_nan(
        incumbent
    )

    replacement = finite_or_nan(
        replacement
    )


    if (
        not np.isfinite(incumbent)
        or
        not np.isfinite(replacement)
        or
        incumbent <= 1e-12
    ):
        return np.nan


    return float(
        (
            incumbent
            -
            replacement
        )
        /
        incumbent
    )


def mean_finite(
    values,
):

    clean = [
        finite_or_nan(v)
        for v in values
    ]


    clean = [
        v
        for v in clean
        if np.isfinite(v)
    ]


    if not clean:
        return np.nan


    return float(
        np.mean(clean)
    )


# ============================================================
# 4. Blocking edge lookup
# ============================================================

source_blocking = (
    blocking[
        blocking[
            "blocks_clean_gap_source"
        ].astype(bool)
    ]
    .copy()
)


target_blocking = (
    blocking[
        blocking[
            "blocks_clean_gap_target"
        ].astype(bool)
    ]
    .copy()
)


print(
    "\n========== BLOCKING EDGE ROLES =========="
)

print(
    "source-side blockers:",
    len(source_blocking)
)

print(
    "target-side blockers:",
    len(target_blocking)
)


# G3B contract:
# 178 occupied sources + 187 occupied targets = 365 blockers.
assert len(source_blocking) == 178
assert len(target_blocking) == 187


source_lookup = {}


for row in source_blocking.itertuples(
    index=False
):

    key = (
        row.dataset,
        int(row.source_id),
    )


    assert key not in source_lookup, (
        "Duplicate source-side blocking edge:",
        key,
    )


    source_lookup[key] = row


target_lookup = {}


for row in target_blocking.itertuples(
    index=False
):

    key = (
        row.dataset,
        int(row.target_id),
    )


    assert key not in target_lookup, (
        "Duplicate target-side blocking edge:",
        key,
    )


    target_lookup[key] = row


# ============================================================
# 5. Construct SIDE-MATCHED normalized comparisons
# ============================================================

rows = []


for comp in oracle.itertuples(
    index=False
):

    dataset = comp.dataset

    source_id = int(
        comp.source_id
    )

    target_id = int(
        comp.target_id
    )


    replacement_dt = (
        int(
            comp.gap_size
        )
        +
        1
    )


    assert replacement_dt >= 3


    # --------------------------------------------------------
    # Replacement normalized CV costs
    # --------------------------------------------------------

    replacement_source_cv_pf = safe_divide(
        comp.replacement_source_cv_um,
        replacement_dt,
    )


    replacement_target_cv_pf = safe_divide(
        comp.replacement_target_cv_um,
        replacement_dt,
    )


    replacement_symmetric_cv_pf = mean_finite(
        [
            replacement_source_cv_pf,
            replacement_target_cv_pf,
        ]
    )


    replacement_source_speed_cost = finite_or_nan(
        comp.replacement_source_speed_mismatch
    )


    replacement_target_speed_cost = finite_or_nan(
        comp.replacement_target_speed_mismatch
    )


    # --------------------------------------------------------
    # SOURCE blocker
    # --------------------------------------------------------

    source_blocker = None


    if bool(
        comp.source_occupied
    ):

        source_key = (
            dataset,
            source_id,
        )


        assert source_key in source_lookup, (
            "Missing source blocker:",
            source_key,
            comp.component_id,
        )


        source_blocker = source_lookup[
            source_key
        ]


        source_blocker_dt = int(
            source_blocker.delta_t
        )


        assert source_blocker_dt > 0


        incumbent_source_cv_pf = safe_divide(
            source_blocker.source_cv_residual_um,
            source_blocker_dt,
        )


        incumbent_source_speed_cost = finite_or_nan(
            source_blocker.source_speed_mismatch_um_per_frame
        )


        source_cv_improvement_pf = improvement(
            incumbent_source_cv_pf,
            replacement_source_cv_pf,
        )


        source_speed_improvement = improvement(
            incumbent_source_speed_cost,
            replacement_source_speed_cost,
        )


        source_cv_improvement_fraction = improvement_fraction(
            incumbent_source_cv_pf,
            replacement_source_cv_pf,
        )


        source_speed_improvement_fraction = improvement_fraction(
            incumbent_source_speed_cost,
            replacement_source_speed_cost,
        )

    else:

        source_blocker_dt = np.nan

        incumbent_source_cv_pf = np.nan
        incumbent_source_speed_cost = np.nan

        source_cv_improvement_pf = np.nan
        source_speed_improvement = np.nan

        source_cv_improvement_fraction = np.nan
        source_speed_improvement_fraction = np.nan


    # --------------------------------------------------------
    # TARGET blocker
    # --------------------------------------------------------

    target_blocker = None


    if bool(
        comp.target_occupied
    ):

        target_key = (
            dataset,
            target_id,
        )


        assert target_key in target_lookup, (
            "Missing target blocker:",
            target_key,
            comp.component_id,
        )


        target_blocker = target_lookup[
            target_key
        ]


        target_blocker_dt = int(
            target_blocker.delta_t
        )


        assert target_blocker_dt > 0


        incumbent_target_cv_pf = safe_divide(
            target_blocker.target_cv_residual_um,
            target_blocker_dt,
        )


        incumbent_target_speed_cost = finite_or_nan(
            target_blocker.target_speed_mismatch_um_per_frame
        )


        target_cv_improvement_pf = improvement(
            incumbent_target_cv_pf,
            replacement_target_cv_pf,
        )


        target_speed_improvement = improvement(
            incumbent_target_speed_cost,
            replacement_target_speed_cost,
        )


        target_cv_improvement_fraction = improvement_fraction(
            incumbent_target_cv_pf,
            replacement_target_cv_pf,
        )


        target_speed_improvement_fraction = improvement_fraction(
            incumbent_target_speed_cost,
            replacement_target_speed_cost,
        )

    else:

        target_blocker_dt = np.nan

        incumbent_target_cv_pf = np.nan
        incumbent_target_speed_cost = np.nan

        target_cv_improvement_pf = np.nan
        target_speed_improvement = np.nan

        target_cv_improvement_fraction = np.nan
        target_speed_improvement_fraction = np.nan


    # --------------------------------------------------------
    # Joint side-matched cost
    #
    # Only compare sides that actually require cutting.
    # --------------------------------------------------------

    replacement_relevant_cv_cost = mean_finite(
        [
            (
                replacement_source_cv_pf
                if bool(comp.source_occupied)
                else np.nan
            ),
            (
                replacement_target_cv_pf
                if bool(comp.target_occupied)
                else np.nan
            ),
        ]
    )


    incumbent_relevant_cv_cost = mean_finite(
        [
            incumbent_source_cv_pf,
            incumbent_target_cv_pf,
        ]
    )


    joint_cv_improvement_pf = improvement(
        incumbent_relevant_cv_cost,
        replacement_relevant_cv_cost,
    )


    replacement_relevant_speed_cost = mean_finite(
        [
            (
                replacement_source_speed_cost
                if bool(comp.source_occupied)
                else np.nan
            ),
            (
                replacement_target_speed_cost
                if bool(comp.target_occupied)
                else np.nan
            ),
        ]
    )


    incumbent_relevant_speed_cost = mean_finite(
        [
            incumbent_source_speed_cost,
            incumbent_target_speed_cost,
        ]
    )


    joint_speed_improvement = improvement(
        incumbent_relevant_speed_cost,
        replacement_relevant_speed_cost,
    )


    joint_cv_improvement_fraction = improvement_fraction(
        incumbent_relevant_cv_cost,
        replacement_relevant_cv_cost,
    )


    joint_speed_improvement_fraction = improvement_fraction(
        incumbent_relevant_speed_cost,
        replacement_relevant_speed_cost,
    )


    rows.append(
        {
            "dataset":
                dataset,

            "fold":
                int(
                    comp.fold
                ),

            "component_id":
                comp.component_id,

            "topology_class":
                comp.topology_class,

            "gap_size":
                int(
                    comp.gap_size
                ),

            "replacement_delta_t":
                replacement_dt,

            "source_id":
                source_id,

            "target_id":
                target_id,

            "source_occupied":
                bool(
                    comp.source_occupied
                ),

            "target_occupied":
                bool(
                    comp.target_occupied
                ),

            # --------------------------------------------
            # Replacement normalized CV
            # --------------------------------------------

            "replacement_source_cv_per_frame":
                replacement_source_cv_pf,

            "replacement_target_cv_per_frame":
                replacement_target_cv_pf,

            "replacement_symmetric_cv_per_frame":
                replacement_symmetric_cv_pf,

            # --------------------------------------------
            # Replacement velocity-space costs
            # --------------------------------------------

            "replacement_source_speed_mismatch":
                replacement_source_speed_cost,

            "replacement_target_speed_mismatch":
                replacement_target_speed_cost,

            # --------------------------------------------
            # Source blocker
            # --------------------------------------------

            "source_blocker_delta_t":
                source_blocker_dt,

            "incumbent_source_cv_per_frame":
                incumbent_source_cv_pf,

            "incumbent_source_speed_mismatch":
                incumbent_source_speed_cost,

            "source_cv_improvement_per_frame":
                source_cv_improvement_pf,

            "source_speed_improvement":
                source_speed_improvement,

            "source_cv_improvement_fraction":
                source_cv_improvement_fraction,

            "source_speed_improvement_fraction":
                source_speed_improvement_fraction,

            # --------------------------------------------
            # Target blocker
            # --------------------------------------------

            "target_blocker_delta_t":
                target_blocker_dt,

            "incumbent_target_cv_per_frame":
                incumbent_target_cv_pf,

            "incumbent_target_speed_mismatch":
                incumbent_target_speed_cost,

            "target_cv_improvement_per_frame":
                target_cv_improvement_pf,

            "target_speed_improvement":
                target_speed_improvement,

            "target_cv_improvement_fraction":
                target_cv_improvement_fraction,

            "target_speed_improvement_fraction":
                target_speed_improvement_fraction,

            # --------------------------------------------
            # Joint side-matched comparison
            # --------------------------------------------

            "replacement_relevant_cv_cost":
                replacement_relevant_cv_cost,

            "incumbent_relevant_cv_cost":
                incumbent_relevant_cv_cost,

            "joint_cv_improvement_per_frame":
                joint_cv_improvement_pf,

            "joint_cv_improvement_fraction":
                joint_cv_improvement_fraction,

            "replacement_relevant_speed_cost":
                replacement_relevant_speed_cost,

            "incumbent_relevant_speed_cost":
                incumbent_relevant_speed_cost,

            "joint_speed_improvement":
                joint_speed_improvement,

            "joint_speed_improvement_fraction":
                joint_speed_improvement_fraction,

            # --------------------------------------------
            # Existing interpolation quality
            # --------------------------------------------

            "pred_full_3um":
                bool(
                    comp.pred_full_3um
                ),

            "pred_full_5um":
                bool(
                    comp.pred_full_5um
                ),

            "pred_full_7um":
                bool(
                    comp.pred_full_7um
                ),
        }
    )


audit = pd.DataFrame(
    rows
)


# ============================================================
# 6. Fidelity
# ============================================================

assert len(
    audit
) == 253


assert not audit[
    [
        "dataset",
        "component_id",
    ]
].duplicated().any()


assert set(
    audit[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}


assert int(
    audit[
        "source_occupied"
    ].sum()
) == 178


assert int(
    audit[
        "target_occupied"
    ].sum()
) == 187


print(
    "\n========== NORMALIZED COMPARISON POPULATION =========="
)

print(
    "components:",
    len(
        audit
    )
)

print(
    "source occupied:",
    int(
        audit[
            "source_occupied"
        ].sum()
    )
)

print(
    "target occupied:",
    int(
        audit[
            "target_occupied"
        ].sum()
    )
)


# ============================================================
# 7. Blocker delta_t contract
# ============================================================

print(
    "\n========== INCUMBENT BLOCKER DELTA_T =========="
)


source_dt = (
    audit[
        "source_blocker_delta_t"
    ]
    .dropna()
    .astype(int)
)


target_dt = (
    audit[
        "target_blocker_delta_t"
    ]
    .dropna()
    .astype(int)
)


print(
    "source blockers:"
)

print(
    source_dt
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\ntarget blockers:"
)

print(
    target_dt
    .value_counts()
    .sort_index()
    .to_string()
)


# ============================================================
# 8. Side-matched medians
# ============================================================

print(
    "\n========== SIDE-MATCHED NORMALIZED MEDIANS =========="
)


comparison_specs = [
    (
        "SOURCE CV/frame",
        "replacement_source_cv_per_frame",
        "incumbent_source_cv_per_frame",
        "source_occupied",
    ),
    (
        "TARGET CV/frame",
        "replacement_target_cv_per_frame",
        "incumbent_target_cv_per_frame",
        "target_occupied",
    ),
    (
        "SOURCE speed mismatch",
        "replacement_source_speed_mismatch",
        "incumbent_source_speed_mismatch",
        "source_occupied",
    ),
    (
        "TARGET speed mismatch",
        "replacement_target_speed_mismatch",
        "incumbent_target_speed_mismatch",
        "target_occupied",
    ),
    (
        "JOINT CV/frame",
        "replacement_relevant_cv_cost",
        "incumbent_relevant_cv_cost",
        None,
    ),
    (
        "JOINT speed mismatch",
        "replacement_relevant_speed_cost",
        "incumbent_relevant_speed_cost",
        None,
    ),
]


for (
    label,
    replacement_col,
    incumbent_col,
    occupancy_col,
) in comparison_specs:

    if occupancy_col is None:

        sub = audit

    else:

        sub = audit[
            audit[
                occupancy_col
            ].astype(bool)
        ]


    paired = sub[
        [
            replacement_col,
            incumbent_col,
        ]
    ].dropna()


    print(
        "\n" + label
    )

    print(
        "paired n:",
        len(
            paired
        )
    )


    if len(
        paired
    ) == 0:
        continue


    print(
        "replacement median:",
        float(
            paired[
                replacement_col
            ].median()
        )
    )


    print(
        "incumbent median:",
        float(
            paired[
                incumbent_col
            ].median()
        )
    )


# ============================================================
# 9. Improvement quantiles
# ============================================================

IMPROVEMENT_COLS = [
    "source_cv_improvement_per_frame",
    "target_cv_improvement_per_frame",

    "source_speed_improvement",
    "target_speed_improvement",

    "joint_cv_improvement_per_frame",
    "joint_speed_improvement",

    "joint_cv_improvement_fraction",
    "joint_speed_improvement_fraction",
]


print(
    "\n========== NORMALIZED IMPROVEMENT QUANTILES =========="
)


for col in IMPROVEMENT_COLS:

    values = (
        audit[
            col
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
    )


    print(
        "\n" + col
    )


    if len(
        values
    ) == 0:

        print(
            "NO VALUES"
        )

        continue


    print(
        values.quantile(
            [
                0.01,
                0.05,
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99,
            ]
        ).to_string()
    )


# ============================================================
# 10. Better-than-incumbent rates
# ============================================================

BETTER_COLS = [
    (
        "source_cv_improvement_per_frame",
        "source CV/frame",
    ),
    (
        "target_cv_improvement_per_frame",
        "target CV/frame",
    ),
    (
        "source_speed_improvement",
        "source speed mismatch",
    ),
    (
        "target_speed_improvement",
        "target speed mismatch",
    ),
    (
        "joint_cv_improvement_per_frame",
        "joint CV/frame",
    ),
    (
        "joint_speed_improvement",
        "joint speed mismatch",
    ),
]


better_rows = []


print(
    "\n========== NORMALIZED BETTER-THAN-INCUMBENT RATE =========="
)


for col, label in BETTER_COLS:

    valid = (
        audit[
            col
        ]
        .dropna()
    )


    better = int(
        valid.gt(
            0
        ).sum()
    )


    fraction = (
        float(
            better
            /
            len(
                valid
            )
        )
        if len(
            valid
        )
        else np.nan
    )


    better_rows.append(
        {
            "metric":
                label,

            "valid":
                int(
                    len(
                        valid
                    )
                ),

            "better":
                better,

            "fraction":
                fraction,
        }
    )


    print(
        label + ":",
        better,
        "/",
        len(
            valid
        ),
        (
            f"({fraction:.3%})"
            if np.isfinite(
                fraction
            )
            else "(NA)"
        ),
    )


better_df = pd.DataFrame(
    better_rows
)


# ============================================================
# 11. Joint evidence patterns
# ============================================================

cv_better = (
    audit[
        "joint_cv_improvement_per_frame"
    ].gt(
        0
    )
)


speed_better = (
    audit[
        "joint_speed_improvement"
    ].gt(
        0
    )
)


cv_valid = audit[
    "joint_cv_improvement_per_frame"
].notna()


speed_valid = audit[
    "joint_speed_improvement"
].notna()


audit[
    "joint_any_better"
] = (
    (
        cv_valid
        &
        cv_better
    )
    |
    (
        speed_valid
        &
        speed_better
    )
)


audit[
    "joint_both_better"
] = (
    cv_valid
    &
    speed_valid
    &
    cv_better
    &
    speed_better
)


audit[
    "joint_both_available"
] = (
    cv_valid
    &
    speed_valid
)


print(
    "\n========== JOINT NORMALIZED ADVANTAGE =========="
)


any_count = int(
    audit[
        "joint_any_better"
    ].sum()
)


both_count = int(
    audit[
        "joint_both_better"
    ].sum()
)


both_available = int(
    audit[
        "joint_both_available"
    ].sum()
)


print(
    "any normalized advantage:",
    any_count,
    "/",
    len(
        audit
    ),
    f"({any_count/len(audit):.3%})",
)


print(
    "both CV/frame + speed advantage:",
    both_count,
    "/",
    both_available,
    (
        f"({both_count/both_available:.3%})"
        if both_available
        else "(NA)"
    ),
)


# ============================================================
# 12. By gap size
# ============================================================

print(
    "\n========== NORMALIZED ADVANTAGE BY GAP SIZE =========="
)


gap_table = (
    audit.groupby(
        "gap_size"
    )
    .agg(
        components=(
            "component_id",
            "size",
        ),

        median_joint_cv_improvement=(
            "joint_cv_improvement_per_frame",
            "median",
        ),

        median_joint_speed_improvement=(
            "joint_speed_improvement",
            "median",
        ),

        joint_any_better=(
            "joint_any_better",
            "sum",
        ),

        joint_both_better=(
            "joint_both_better",
            "sum",
        ),

        pred_full_5um=(
            "pred_full_5um",
            "sum",
        ),

        pred_full_7um=(
            "pred_full_7um",
            "sum",
        ),
    )
)


gap_table[
    "any_better_fraction"
] = (
    gap_table[
        "joint_any_better"
    ]
    /
    gap_table[
        "components"
    ]
)


gap_table[
    "both_better_fraction"
] = (
    gap_table[
        "joint_both_better"
    ]
    /
    gap_table[
        "components"
    ]
)


print(
    gap_table.to_string()
)


# ============================================================
# 13. By topology
# ============================================================

print(
    "\n========== NORMALIZED ADVANTAGE BY TOPOLOGY =========="
)


topology_table = (
    audit.groupby(
        "topology_class"
    )
    .agg(
        components=(
            "component_id",
            "size",
        ),

        median_joint_cv_improvement=(
            "joint_cv_improvement_per_frame",
            "median",
        ),

        median_joint_speed_improvement=(
            "joint_speed_improvement",
            "median",
        ),

        joint_any_better=(
            "joint_any_better",
            "sum",
        ),

        joint_both_better=(
            "joint_both_better",
            "sum",
        ),

        pred_full_5um=(
            "pred_full_5um",
            "sum",
        ),

        pred_full_7um=(
            "pred_full_7um",
            "sum",
        ),
    )
)


topology_table[
    "any_better_fraction"
] = (
    topology_table[
        "joint_any_better"
    ]
    /
    topology_table[
        "components"
    ]
)


topology_table[
    "both_better_fraction"
] = (
    topology_table[
        "joint_both_better"
    ]
    /
    topology_table[
        "components"
    ]
)


print(
    topology_table.to_string()
)


# ============================================================
# 14. Fold stability
# ============================================================

print(
    "\n========== NORMALIZED ADVANTAGE BY FOLD =========="
)


fold_table = (
    audit.groupby(
        "fold"
    )
    .agg(
        components=(
            "component_id",
            "size",
        ),

        median_joint_cv_improvement=(
            "joint_cv_improvement_per_frame",
            "median",
        ),

        median_joint_speed_improvement=(
            "joint_speed_improvement",
            "median",
        ),

        joint_any_better=(
            "joint_any_better",
            "sum",
        ),

        joint_both_better=(
            "joint_both_better",
            "sum",
        ),
    )
)


fold_table[
    "any_better_fraction"
] = (
    fold_table[
        "joint_any_better"
    ]
    /
    fold_table[
        "components"
    ]
)


fold_table[
    "both_better_fraction"
] = (
    fold_table[
        "joint_both_better"
    ]
    /
    fold_table[
        "components"
    ]
)


print(
    fold_table.to_string()
)


# ============================================================
# 15. Relation to interpolation quality
# ============================================================

print(
    "\n========== NORMALIZED ADVANTAGE vs INTERPOLATION QUALITY =========="
)


for quality_col in [
    "pred_full_3um",
    "pred_full_5um",
    "pred_full_7um",
]:

    quality = audit[
        quality_col
    ].astype(bool)


    good = audit[
        quality
    ]


    bad = audit[
        ~quality
    ]


    print(
        "\n" + quality_col
    )


    print(
        "good n:",
        len(
            good
        )
    )


    print(
        "good median joint CV improvement:",
        float(
            good[
                "joint_cv_improvement_per_frame"
            ].median()
        )
        if good[
            "joint_cv_improvement_per_frame"
        ].notna().any()
        else np.nan
    )


    print(
        "good median joint speed improvement:",
        float(
            good[
                "joint_speed_improvement"
            ].median()
        )
        if good[
            "joint_speed_improvement"
        ].notna().any()
        else np.nan
    )


    good_any = int(
        good[
            "joint_any_better"
        ].sum()
    )


    print(
        "good any advantage:",
        good_any,
        "/",
        len(
            good
        ),
        (
            f"({good_any/len(good):.3%})"
            if len(
                good
            )
            else "(NA)"
        ),
    )


    print(
        "bad n:",
        len(
            bad
        )
    )


    print(
        "bad median joint CV improvement:",
        float(
            bad[
                "joint_cv_improvement_per_frame"
            ].median()
        )
        if bad[
            "joint_cv_improvement_per_frame"
        ].notna().any()
        else np.nan
    )


    print(
        "bad median joint speed improvement:",
        float(
            bad[
                "joint_speed_improvement"
            ].median()
        )
        if bad[
            "joint_speed_improvement"
        ].notna().any()
        else np.nan
    )


    bad_any = int(
        bad[
            "joint_any_better"
        ].sum()
    )


    print(
        "bad any advantage:",
        bad_any,
        "/",
        len(
            bad
        ),
        (
            f"({bad_any/len(bad):.3%})"
            if len(
                bad
            )
            else "(NA)"
        ),
    )


# ============================================================
# 16. Persist
# ============================================================

audit = (
    audit
    .sort_values(
        [
            "fold",
            "dataset",
            "component_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


audit.to_pickle(
    OUT
)


summary = {
    "stage":
        "12.11G3D2-R",

    "status":
        "TIME_NORMALIZED_SIDE_MATCHED_REPLACEMENT_ADVANTAGE_AUDITED",

    "methodological_repair":
        (
            "Original G3D2 raw CV residual comparison was "
            "temporally confounded. This audit compares CV/delta_t "
            "and velocity-space side-matched costs."
        ),

    "occupied_components":
        int(
            len(
                audit
            )
        ),

    "source_occupied_components":
        int(
            audit[
                "source_occupied"
            ].sum()
        ),

    "target_occupied_components":
        int(
            audit[
                "target_occupied"
            ].sum()
        ),

    "better_rates":
        better_df.to_dict(
            "records"
        ),

    "joint_any_better":
        int(
            audit[
                "joint_any_better"
            ].sum()
        ),

    "joint_both_better":
        int(
            audit[
                "joint_both_better"
            ].sum()
        ),

    "gt_used":
        True,

    "gt_role":
        "DEVELOPMENT_ORACLE_DESIGN_AUDIT_ONLY",

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert OUT.exists()
assert SUMMARY_OUT.exists()


# ============================================================
# 17. Reload integrity
# ============================================================

check = pd.read_pickle(
    OUT
)


assert len(
    check
) == 253


assert int(
    check[
        "source_occupied"
    ].sum()
) == 178


assert int(
    check[
        "target_occupied"
    ].sum()
) == 187


assert not check[
    [
        "dataset",
        "component_id",
    ]
].duplicated().any()


# ============================================================
# 18. Decision
# ============================================================

print(
    "\n========== 12.11G3D2-R DECISION =========="
)

print(
    "TIME_NORMALIZED_SIDE_MATCHED_ADVANTAGE_AUDIT: PASS"
)

print(
    "occupied components:",
    len(
        audit
    )
)

print(
    "raw G3D2 temporal confounding repaired: YES"
)

print(
    "Fold0 used: NO"
)

print(
    "classifier trained: NO"
)

print(
    "threshold selected: NO"
)

print(
    "policy selected: NO"
)

print(
    "graph modified: NO"
)

print(
    "next:"
)

print(
    "DECIDE_PROPOSAL_FIRST_MULTIFRAME_GENERATOR_FROM_NORMALIZED_EVIDENCE"
)

In [ ]:
# ============================================================
# ONE-TIME CLEANUP
# Remove ONLY failed/partial Stage 12.11G3E-R artifacts
#
# Safe scope:
#   - G3E-R proposal shard directory
#   - G3E-R manifest
#   - G3E-R oracle outputs, if any partial copies exist
#
# Does NOT touch:
#   G3A / G3B / G3C / G3D1 / G3D2 / G3D2-R
# ============================================================

from pathlib import Path
import shutil


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


RUN_DIR = (
    S12
    / "stage12_folds14_multiframe_rewire_cvtop5_dominance_any_v1"
)

MANIFEST_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_cvtop5_dominance_any_v1_manifest.json"
)

ORACLE_OUT = (
    S12
    / "stage12_folds14_multiframe_rewire_cvtop5_dominance_any_v1_oracle.pkl"
)

ORACLE_SUMMARY_OUT = (
    S12
    / "stage12_folds14_multiframe_rewire_cvtop5_dominance_any_v1_oracle_summary.json"
)


print(
    "========== G3E-R PARTIAL CLEANUP =========="
)


# ------------------------------------------------------------
# Proposal shard directory
# ------------------------------------------------------------

if RUN_DIR.exists():

    print(
        "removing partial directory:"
    )

    print(
        RUN_DIR
    )


    shutil.rmtree(
        RUN_DIR
    )


else:

    print(
        "partial directory not present:",
        RUN_DIR
    )


# ------------------------------------------------------------
# Manifest
# ------------------------------------------------------------

if MANIFEST_PATH.exists():

    print(
        "removing manifest:"
    )

    print(
        MANIFEST_PATH
    )


    MANIFEST_PATH.unlink()


else:

    print(
        "manifest not present"
    )


# ------------------------------------------------------------
# Oracle artifacts
#
# Normally these do not exist because Part A did not complete.
# Remove them if a partial/stale attempt created them.
# ------------------------------------------------------------

for path in [
    ORACLE_OUT,
    ORACLE_SUMMARY_OUT,
]:

    if path.exists():

        print(
            "removing partial/stale output:"
        )

        print(
            path
        )


        path.unlink()


# ------------------------------------------------------------
# Fidelity
# ------------------------------------------------------------

assert not RUN_DIR.exists()

assert not MANIFEST_PATH.exists()

assert not ORACLE_OUT.exists()

assert not ORACLE_SUMMARY_OUT.exists()


print(
    "\n========== CLEANUP DECISION =========="
)

print(
    "FAILED_G3E_R_PARTIAL_ARTIFACTS_REMOVED: PASS"
)

print(
    "G3A-G3D2-R touched: NO"
)

print(
    "ready to rerun G3E-R: YES"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3E-R
#
# Streaming GT-Blind Sparse Multi-Frame
# Reciprocal-CV + Relative-Dominance Proposal Universe
#
# FIXED VERSION
#
# Fix:
#   - REMOVED invalid top-level `nonlocal`
#   - streaming flush now uses explicit arguments + return values
#   - endpoint state records ALL graph nodes for exact miss audit
#
# Scientific motivation:
#
#   Original G3E:
#       >62M proposals by dataset 80/103
#       kernel memory failure
#
#   G3D1:
#       incumbent-only suspicion cannot provide high-recall anchors
#
#   G3D2-R:
#       normalized TRUE replacement outperforms incumbent on
#       ~73% of comparable occupied clean gaps
#
# Therefore:
#
#       proposal FIRST
#           ->
#       sparse reciprocal CV candidate
#           ->
#       incumbent-relative dominance
#
# Generator:
#
#   source:
#       CURRENT_BEST outdegree <= 1
#       past velocity available
#
#   target:
#       CURRENT_BEST indegree <= 1
#       future velocity available
#
#   gap_size:
#       2 ... 12
#
#   proposal family:
#       reciprocal forward/backward CV Top-K
#
#   K_MAX:
#       5
#
#   pruning:
#
#       FREE/FREE:
#           retain reciprocal proposal
#
#       OCCUPIED:
#           retain iff replacement normalized CV is better
#           than incumbent on >=1 side requiring a cut
#
# Streaming:
#
#   - at most CHUNK_ROWS proposal records in Python memory
#   - write pickle shard
#   - release memory
#   - full proposal universe is NEVER concatenated in RAM
#
# PART A:
#   GT-blind generation
#   write shards
#   freeze manifest + hash
#
# PART B:
#   ONLY AFTER manifest freeze
#   read G3B development oracle
#   scan shards for the 369 oracle pairs
#
# IMPORTANT:
#
#   - Candidate generation is GT-blind.
#   - Dominance threshold is zero improvement.
#   - Zero was not selected by GT parameter sweep.
#   - No Fold0.
#   - No classifier.
#   - No final deployment policy.
#   - No graph modification.
#
# Keep after running: YES
# ============================================================

from pathlib import Path
import gc
import hashlib
import json
import sys

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree


# ============================================================
# 0. CONFIG
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

PRED_ROOT = (
    PROJECT
    / "predictions/heqiuyan"
)

S11 = (
    PROJECT
    / "reports"
    / "stage11_division_probe"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


B256_995 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det995"
    / "split_0"
)

B256_9975 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det9975"
    / "split_0"
)


MORPH_PATH = (
    S11
    / "stage11_deployable_final_v9_policy_grid_actions.pkl"
)

V1_PATH = (
    S12
    / "stage12_folds14_mutual_best_universe.pkl"
)

V2_PATH = (
    S12
    / "stage12_folds14_round2_reciprocal_universe.pkl"
)

BRIDGE_PATH = (
    S12
    / "stage12_detection_bridge_policy_actions.pkl"
)


# ------------------------------------------------------------
# GT DEVELOPMENT ORACLE
#
# IMPORTANT:
# File existence may be checked now.
# Contents MUST NOT be read until Part A is frozen.
# ------------------------------------------------------------

G3B_PATH = (
    S12
    / "stage12_folds14_occupied_boundary_rewire_feasibility.pkl"
)


# ------------------------------------------------------------
# Versioned G3E-R outputs
# ------------------------------------------------------------

RUN_DIR = (
    S12
    / "stage12_folds14_multiframe_rewire_cvtop5_dominance_any_v1"
)

MANIFEST_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_cvtop5_dominance_any_v1_manifest.json"
)

ORACLE_OUT = (
    S12
    / "stage12_folds14_multiframe_rewire_cvtop5_dominance_any_v1_oracle.pkl"
)

ORACLE_SUMMARY_OUT = (
    S12
    / "stage12_folds14_multiframe_rewire_cvtop5_dominance_any_v1_oracle_summary.json"
)


MIN_GAP_SIZE = 2
MAX_GAP_SIZE = 12

K_MAX = 5

K_AUDIT = [
    1,
    2,
    3,
    5,
]


# Maximum retained Python proposal rows before disk flush.
CHUNK_ROWS = 100_000


SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


# ============================================================
# 0A. Input existence gates
# ============================================================

for path in [
    TRAIN_ROOT,
    CV_PATH,
    B256_995,
    B256_9975,
    MORPH_PATH,
    V1_PATH,
    V2_PATH,
    BRIDGE_PATH,
    G3B_PATH,
]:

    assert path.exists(), path


# ============================================================
# 0B. Fresh-run protection
#
# Do not accidentally mix shards from different runs.
# ============================================================

if RUN_DIR.exists():

    raise RuntimeError(
        "\nG3E-R output directory already exists:\n"
        f"{RUN_DIR}\n\n"
        "If it belongs to a FAILED/PARTIAL G3E-R run, delete ONLY "
        "this G3E-R directory and its G3E-R manifest, then rerun.\n"
        "Do NOT delete G3A-G3D2-R artifacts."
    )


if MANIFEST_PATH.exists():

    raise RuntimeError(
        "\nFrozen G3E-R manifest already exists:\n"
        f"{MANIFEST_PATH}\n\n"
        "Do not overwrite a completed/frozen proposal universe."
    )


RUN_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# 1. PROJECT IMPORT
# ============================================================

SRC = str(
    PROJECT
    / "src"
)


if SRC not in sys.path:

    sys.path.insert(
        0,
        SRC,
    )


from biohub_cell_tracking import frozen_v9 as fv9


# ============================================================
# 2. DEVELOPMENT CORPUS
# ============================================================

cv = pd.read_csv(
    CV_PATH
)


corp = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    corp
) == 103


assert corp[
    "dataset"
].nunique() == 103


assert not corp[
    "fold"
].eq(
    0
).any()


names = corp[
    "dataset"
].tolist()


fold_map = dict(
    zip(
        corp[
            "dataset"
        ],
        corp[
            "fold"
        ].astype(int),
    )
)


prefix_map = dict(
    zip(
        corp[
            "dataset"
        ],
        corp[
            "prefix"
        ],
    )
)


# ============================================================
# 3. EXACT CURRENT_BEST ACTION CONTRACT
# ============================================================

morph_all = pd.read_pickle(
    MORPH_PATH
)


morph_actions = (
    morph_all[
        morph_all[
            "config_id"
        ].eq(
            "K3_R3_P100"
        )
    ]
    .copy()
)


v1_all = pd.read_pickle(
    V1_PATH
)


v1_actions = (
    v1_all[
        v1_all[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


v2_all = pd.read_pickle(
    V2_PATH
)


v2_actions = (
    v2_all[
        v2_all[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


bridge_all = pd.read_pickle(
    BRIDGE_PATH
)


r1p50_actions = (
    bridge_all[
        bridge_all[
            "policy_id"
        ].eq(
            "R1P50"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


assert len(
    morph_actions
) == 96


assert len(
    v1_actions
) == 46951


assert len(
    v2_actions
) == 18511


assert len(
    r1p50_actions
) == 805


print(
    "========== G3E-R CURRENT_BEST CONTRACT =========="
)

print(
    "datasets:",
    len(
        names
    )
)

print(
    "MORPH:",
    len(
        morph_actions
    )
)

print(
    "V1:",
    len(
        v1_actions
    )
)

print(
    "V2:",
    len(
        v2_actions
    )
)

print(
    "R1P50:",
    len(
        r1p50_actions
    )
)


# ============================================================
# 4. FROZEN V9
# ============================================================

policies = fv9.build_frozen_v9_policies(
    names,
    B256_995,
    B256_9975,
    TRAIN_ROOT,
    prefix_map=prefix_map,
    config=fv9.FROZEN_V9_CONFIG,
    progress=False,
)


assert (
    type(
        policies
    ).__name__
    ==
    "FrozenV9Policies"
)


# ============================================================
# 5. EXACT CURRENT_BEST BUILDER
# ============================================================

def graph_state(
    graph,
):

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )


    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = {
        int(k):
            int(v)

        for k, v in outdeg.items()
    }


    indeg = {
        int(k):
            int(v)

        for k, v in indeg.items()
    }


    edge_set = set(
        zip(
            edges[
                "source_id"
            ].astype(int),

            edges[
                "target_id"
            ].astype(int),
        )
    )


    return (
        nodes,
        outdeg,
        indeg,
        edge_set,
    )


def add_continuation_actions(
    graph,
    nodes,
    table,
    dataset,
):

    rows = table[
        table[
            "dataset"
        ].eq(
            dataset
        )
    ]


    for action in rows.itertuples(
        index=False
    ):

        src = int(
            action.source_pred_node_id
        )

        tgt = int(
            action.target_pred_node_id
        )


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(
                        dist
                    ),

                "edge_prob":
                    float(
                        action.candidate_prob
                    ),
            },
            validate_keys=True,
        )


def build_current_best(
    dataset,
):

    graph, scale, _ = fv9.apply_frozen_v9(
        dataset,
        B256_995,
        B256_9975,
        policies,
        TRAIN_ROOT,
        config=fv9.FROZEN_V9_CONFIG,
    )


    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = graph_state(
        graph
    )


    # --------------------------------------------------------
    # MORPH-DIV-V2
    # --------------------------------------------------------

    rows = (
        morph_actions[
            morph_actions[
                "dataset"
            ].eq(
                dataset
            )
        ]
        .sort_values(
            [
                "morph_rank",
                "t",
                "source_pred_node_id",
                "target_pred_node_id",
            ],
            kind="stable",
        )
    )


    for action in rows.itertuples(
        index=False
    ):

        src = int(
            action.source_pred_node_id
        )

        tgt = int(
            action.target_pred_node_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 1


        assert indeg.get(
            tgt,
            0,
        ) == 0


        assert (
            src,
            tgt,
        ) not in edge_set


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(
                        dist
                    ),

                "edge_prob":
                    float(
                        action.candidate_prob
                    ),
            },
            validate_keys=True,
        )


        outdeg[
            src
        ] = 2


        indeg[
            tgt
        ] = 1


        edge_set.add(
            (
                src,
                tgt,
            )
        )


    # --------------------------------------------------------
    # MUTUAL-CONT-V1
    # --------------------------------------------------------

    add_continuation_actions(
        graph,
        nodes,
        v1_actions,
        dataset,
    )


    # --------------------------------------------------------
    # MUTUAL-CONT-V2
    # --------------------------------------------------------

    add_continuation_actions(
        graph,
        nodes,
        v2_actions,
        dataset,
    )


    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = graph_state(
        graph
    )


    # --------------------------------------------------------
    # DET-BRIDGE-R1P50
    # --------------------------------------------------------

    rows = r1p50_actions[
        r1p50_actions[
            "dataset"
        ].eq(
            dataset
        )
    ]


    for action in rows.itertuples(
        index=False
    ):

        src = int(
            action.source_id
        )

        tgt = int(
            action.target_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 0


        assert indeg.get(
            tgt,
            0,
        ) == 0


        new_id = graph.add_node(
            {
                "t":
                    int(
                        action.t
                    ),

                "z":
                    float(
                        action.z
                    ),

                "y":
                    float(
                        action.y
                    ),

                "x":
                    float(
                        action.x
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            src,
            new_id,
            {
                "edge_dist":
                    float(
                        action.src_dist_um
                    ),

                "edge_prob":
                    float(
                        action.in_prob
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            new_id,
            tgt,
            {
                "edge_dist":
                    float(
                        action.tgt_dist_um
                    ),

                "edge_prob":
                    float(
                        action.out_prob
                    ),
            },
            validate_keys=True,
        )


        outdeg[
            src
        ] = 1


        indeg[
            new_id
        ] = 1


        outdeg[
            new_id
        ] = 1


        indeg[
            tgt
        ] = 1


    return (
        graph,
        scale,
    )


print(
    "SELF_CONTAINED_CURRENT_BEST_BUILDER: PASS"
)


# ============================================================
# 6. NUMERIC / KNN HELPERS
# ============================================================

def vec_norm(
    x,
):

    return float(
        np.linalg.norm(
            np.asarray(
                x,
                dtype=float,
            )
        )
    )


def finite_mean(
    values,
):

    clean = []


    for value in values:

        try:

            value = float(
                value
            )

        except (
            TypeError,
            ValueError,
        ):

            continue


        if np.isfinite(
            value
        ):

            clean.append(
                value
            )


    if not clean:

        return np.nan


    return float(
        np.mean(
            clean
        )
    )


def query_knn(
    query_points,
    candidate_points,
    k,
):

    query_points = np.asarray(
        query_points,
        dtype=float,
    )

    candidate_points = np.asarray(
        candidate_points,
        dtype=float,
    )


    assert query_points.ndim == 2
    assert candidate_points.ndim == 2

    assert query_points.shape[
        1
    ] == 3

    assert candidate_points.shape[
        1
    ] == 3


    if (
        len(
            query_points
        ) == 0
        or
        len(
            candidate_points
        ) == 0
    ):

        return (
            np.empty(
                (
                    len(
                        query_points
                    ),
                    0,
                ),
                dtype=float,
            ),

            np.empty(
                (
                    len(
                        query_points
                    ),
                    0,
                ),
                dtype=int,
            ),
        )


    k_eff = min(
        int(
            k
        ),
        int(
            len(
                candidate_points
            )
        ),
    )


    tree = cKDTree(
        candidate_points
    )


    distances, indices = tree.query(
        query_points,
        k=k_eff,
    )


    if k_eff == 1:

        distances = distances[
            :,
            None
        ]

        indices = indices[
            :,
            None
        ]


    return (
        np.asarray(
            distances,
            dtype=float,
        ),

        np.asarray(
            indices,
            dtype=int,
        ),
    )


def make_rank_map(
    query_ids,
    candidate_ids,
    distances,
    indices,
    reverse_pair=False,
):

    result = {}


    for query_i, query_id in enumerate(
        query_ids
    ):

        for rank_i in range(
            indices.shape[
                1
            ]
        ):

            candidate_id = int(
                candidate_ids[
                    indices[
                        query_i,
                        rank_i
                    ]
                ]
            )


            if reverse_pair:

                pair = (
                    candidate_id,
                    int(
                        query_id
                    ),
                )

            else:

                pair = (
                    int(
                        query_id
                    ),
                    candidate_id,
                )


            rank = int(
                rank_i
                +
                1
            )


            distance = float(
                distances[
                    query_i,
                    rank_i
                ]
            )


            if pair not in result:

                result[
                    pair
                ] = (
                    rank,
                    distance,
                )


    return result


# ============================================================
# 7. STREAMING OUTPUT SCHEMA
# ============================================================

COLUMNS = [
    "dataset",
    "fold",

    "source_id",
    "target_id",

    "source_t",
    "target_t",

    "gap_size",
    "delta_t",

    "source_outdegree",
    "target_indegree",

    "n_required_cuts",

    "source_cv_rank",
    "target_cv_rank",

    "proposal_min_k",

    "source_cv_per_frame",
    "target_cv_per_frame",
    "symmetric_cv_per_frame",

    "source_incumbent_cv_per_frame",
    "target_incumbent_cv_per_frame",

    "source_relative_improvement",
    "target_relative_improvement",

    "joint_relative_improvement",

    "dominance_any",
    "dominance_all",
]


KEY_COLS = [
    "dataset",
    "source_id",
    "target_id",

    "source_t",
    "target_t",

    "gap_size",
    "proposal_min_k",
]


# ============================================================
# 8. FIXED STREAM FLUSH FUNCTION
#
# IMPORTANT:
# No `nonlocal`.
#
# State is passed explicitly and updated state is returned.
# ============================================================

def flush_proposal_chunk(
    chunk_rows,
    chunk_index,
    dataset_total,
    chunk_files,
    dataset_dir,
    global_hasher,
):

    if not chunk_rows:

        return (
            [],
            chunk_index,
            dataset_total,
            chunk_files,
        )


    chunk_df = pd.DataFrame.from_records(
        chunk_rows,
        columns=COLUMNS,
    )


    assert len(
        chunk_df
    ) > 0


    chunk_df = (
        chunk_df
        .sort_values(
            [
                "source_t",
                "source_id",
                "target_t",
                "target_id",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


    # Every source-target pair corresponds to one temporal gap.
    assert not chunk_df[
        [
            "dataset",
            "source_id",
            "target_id",
        ]
    ].duplicated().any()


    chunk_name = (
        f"proposal_chunk_{chunk_index:04d}.pkl"
    )


    chunk_path = (
        dataset_dir
        /
        chunk_name
    )


    chunk_df.to_pickle(
        chunk_path
    )


    assert chunk_path.exists()


    # --------------------------------------------------------
    # Stable stream-order proposal identity hash
    # --------------------------------------------------------

    key_bytes = (
        chunk_df[
            KEY_COLS
        ]
        .to_csv(
            index=False,
        )
        .encode()
    )


    global_hasher.update(
        key_bytes
    )


    chunk_sha = hashlib.sha256(
        key_bytes
    ).hexdigest()


    chunk_files = list(
        chunk_files
    )


    chunk_files.append(
        {
            "file":
                chunk_name,

            "rows":
                int(
                    len(
                        chunk_df
                    )
                ),

            "sha256":
                chunk_sha,
        }
    )


    new_dataset_total = (
        int(
            dataset_total
        )
        +
        int(
            len(
                chunk_df
            )
        )
    )


    new_chunk_index = (
        int(
            chunk_index
        )
        +
        1
    )


    # Explicit cleanup.
    del chunk_df
    del key_bytes


    gc.collect()


    return (
        [],
        new_chunk_index,
        new_dataset_total,
        chunk_files,
    )


# ============================================================
# 9. PART A — GT-BLIND STREAMING GENERATION
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART A — STREAMING GT-BLIND RECIPROCAL-CV GENERATION"
)

print(
    "============================================================"
)


global_hasher = hashlib.sha256()


manifest_datasets = []


global_total = 0

global_free_free = 0
global_source_cut = 0
global_target_cut = 0
global_both_cut = 0


for ds_i, dataset in enumerate(
    names,
    start=1,
):

    fold = int(
        fold_map[
            dataset
        ]
    )


    dataset_dir = (
        RUN_DIR
        /
        dataset
    )


    dataset_dir.mkdir(
        parents=True,
        exist_ok=False,
    )


    # ========================================================
    # 9A. Exact CURRENT_BEST
    # ========================================================

    graph, scale = build_current_best(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    ), (
        dataset,
        scale,
        SCALE,
    )


    # ========================================================
    # 9B. Nodes
    # ========================================================

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = (
        nodes[
            "node_id"
        ]
        .astype(int)
    )


    nodes[
        "t"
    ] = (
        nodes[
            "t"
        ]
        .astype(int)
    )


    nodes = (
        nodes
        .sort_values(
            [
                "t",
                "node_id",
            ],
            kind="stable",
        )
        .set_index(
            "node_id",
            drop=False,
        )
    )


    # ========================================================
    # 9C. Edges
    # ========================================================

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = (
        edges[
            "source_id"
        ]
        .astype(int)
    )


    edges[
        "target_id"
    ] = (
        edges[
            "target_id"
        ]
        .astype(int)
    )


    edge_set = set(
        zip(
            edges[
                "source_id"
            ].astype(int),

            edges[
                "target_id"
            ].astype(int),
        )
    )


    incoming = (
        edges.groupby(
            "target_id"
        )[
            "source_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    outgoing = (
        edges.groupby(
            "source_id"
        )[
            "target_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = {
        int(k):
            int(v)

        for k, v in indeg.items()
    }


    outdeg = {
        int(k):
            int(v)

        for k, v in outdeg.items()
    }


    # ========================================================
    # 9D. Physical state
    # ========================================================

    phys = {}

    time_map = {}


    for nid, row in nodes.iterrows():

        nid = int(
            nid
        )


        phys[
            nid
        ] = (
            np.array(
                [
                    float(
                        row[
                            "z"
                        ]
                    ),

                    float(
                        row[
                            "y"
                        ]
                    ),

                    float(
                        row[
                            "x"
                        ]
                    ),
                ],
                dtype=float,
            )
            *
            SCALE
        )


        time_map[
            nid
        ] = int(
            row[
                "t"
            ]
        )


    node_ids = sorted(
        phys.keys()
    )


    # ========================================================
    # 9E. Local past/future velocities
    # ========================================================

    past_velocity = {}

    future_velocity = {}


    for nid in node_ids:

        # ----------------------------------------------------
        # Past velocity
        # ----------------------------------------------------

        inc = incoming.get(
            nid,
            [],
        )


        if len(
            inc
        ) == 1:

            prev_id = int(
                inc[
                    0
                ]
            )


            if prev_id in phys:

                dt = (
                    time_map[
                        nid
                    ]
                    -
                    time_map[
                        prev_id
                    ]
                )


                if dt > 0:

                    past_velocity[
                        nid
                    ] = (
                        phys[
                            nid
                        ]
                        -
                        phys[
                            prev_id
                        ]
                    ) / float(
                        dt
                    )


        # ----------------------------------------------------
        # Future velocity
        # ----------------------------------------------------

        out = outgoing.get(
            nid,
            [],
        )


        if len(
            out
        ) == 1:

            next_id = int(
                out[
                    0
                ]
            )


            if next_id in phys:

                dt = (
                    time_map[
                        next_id
                    ]
                    -
                    time_map[
                        nid
                    ]
                )


                if dt > 0:

                    future_velocity[
                        nid
                    ] = (
                        phys[
                            next_id
                        ]
                        -
                        phys[
                            nid
                        ]
                    ) / float(
                        dt
                    )


    # ========================================================
    # 9F. Endpoint eligibility
    # ========================================================

    source_eligible = [
        nid

        for nid in node_ids

        if (
            outdeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid
            in
            past_velocity
        )
    ]


    target_eligible = [
        nid

        for nid in node_ids

        if (
            indeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid
            in
            future_velocity
        )
    ]


    source_eligible_set = set(
        source_eligible
    )


    target_eligible_set = set(
        target_eligible
    )


    # ========================================================
    # 9G. Endpoint-state artifact
    #
    # FIX:
    # Store ALL nodes, not only eligible-union nodes.
    #
    # This makes later oracle miss classification exact.
    # ========================================================

    endpoint_rows = []


    for nid in node_ids:

        endpoint_rows.append(
            {
                "dataset":
                    dataset,

                "fold":
                    fold,

                "node_id":
                    int(
                        nid
                    ),

                "t":
                    int(
                        time_map[
                            nid
                        ]
                    ),

                "indegree":
                    int(
                        indeg.get(
                            nid,
                            0,
                        )
                    ),

                "outdegree":
                    int(
                        outdeg.get(
                            nid,
                            0,
                        )
                    ),

                "has_past_velocity":
                    bool(
                        nid
                        in
                        past_velocity
                    ),

                "has_future_velocity":
                    bool(
                        nid
                        in
                        future_velocity
                    ),

                "source_eligible":
                    bool(
                        nid
                        in
                        source_eligible_set
                    ),

                "target_eligible":
                    bool(
                        nid
                        in
                        target_eligible_set
                    ),
            }
        )


    endpoint_df = pd.DataFrame(
        endpoint_rows
    )


    assert len(
        endpoint_df
    ) == len(
        node_ids
    )


    assert not endpoint_df[
        [
            "dataset",
            "node_id",
        ]
    ].duplicated().any()


    endpoint_path = (
        dataset_dir
        /
        "endpoint_state.pkl"
    )


    endpoint_df.to_pickle(
        endpoint_path
    )


    assert endpoint_path.exists()


    # ========================================================
    # 9H. Incumbent normalized source-side CV costs
    # ========================================================

    source_incumbent_cost = {}


    for src in source_eligible:

        if outdeg.get(
            src,
            0,
        ) != 1:

            continue


        current_out = outgoing.get(
            src,
            [],
        )


        if len(
            current_out
        ) != 1:

            continue


        old_tgt = int(
            current_out[
                0
            ]
        )


        if old_tgt not in phys:

            continue


        dt_old = (
            time_map[
                old_tgt
            ]
            -
            time_map[
                src
            ]
        )


        if dt_old <= 0:

            continue


        predicted_target = (
            phys[
                src
            ]
            +
            float(
                dt_old
            )
            *
            past_velocity[
                src
            ]
        )


        residual_um = vec_norm(
            phys[
                old_tgt
            ]
            -
            predicted_target
        )


        source_incumbent_cost[
            src
        ] = (
            residual_um
            /
            float(
                dt_old
            )
        )


    # ========================================================
    # 9I. Incumbent normalized target-side CV costs
    # ========================================================

    target_incumbent_cost = {}


    for tgt in target_eligible:

        if indeg.get(
            tgt,
            0,
        ) != 1:

            continue


        current_in = incoming.get(
            tgt,
            [],
        )


        if len(
            current_in
        ) != 1:

            continue


        old_src = int(
            current_in[
                0
            ]
        )


        if old_src not in phys:

            continue


        dt_old = (
            time_map[
                tgt
            ]
            -
            time_map[
                old_src
            ]
        )


        if dt_old <= 0:

            continue


        predicted_source = (
            phys[
                tgt
            ]
            -
            float(
                dt_old
            )
            *
            future_velocity[
                tgt
            ]
        )


        residual_um = vec_norm(
            phys[
                old_src
            ]
            -
            predicted_source
        )


        target_incumbent_cost[
            tgt
        ] = (
            residual_um
            /
            float(
                dt_old
            )
        )


    # ========================================================
    # 9J. Group eligible endpoints by frame
    # ========================================================

    source_by_t = {}

    target_by_t = {}


    for nid in source_eligible:

        source_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for nid in target_eligible:

        target_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for t in source_by_t:

        source_by_t[
            t
        ] = sorted(
            source_by_t[
                t
            ]
        )


    for t in target_by_t:

        target_by_t[
            t
        ] = sorted(
            target_by_t[
                t
            ]
        )


    # ========================================================
    # 9K. Dataset streaming state
    # ========================================================

    chunk_rows = []

    chunk_index = 0

    dataset_total = 0


    dataset_free_free = 0
    dataset_source_cut = 0
    dataset_target_cut = 0
    dataset_both_cut = 0


    chunk_files = []


    # ========================================================
    # 9L. Reciprocal CV generator
    # ========================================================

    for gap_size in range(
        MIN_GAP_SIZE,
        MAX_GAP_SIZE + 1,
    ):

        delta_t = (
            gap_size
            +
            1
        )


        for source_t in sorted(
            source_by_t.keys()
        ):

            target_t = (
                source_t
                +
                delta_t
            )


            if target_t not in target_by_t:

                continue


            source_ids = source_by_t[
                source_t
            ]


            target_ids = target_by_t[
                target_t
            ]


            if (
                len(
                    source_ids
                ) == 0
                or
                len(
                    target_ids
                ) == 0
            ):

                continue


            # ------------------------------------------------
            # Positions and velocities
            # ------------------------------------------------

            source_pos = np.vstack(
                [
                    phys[
                        nid
                    ]

                    for nid in source_ids
                ]
            )


            target_pos = np.vstack(
                [
                    phys[
                        nid
                    ]

                    for nid in target_ids
                ]
            )


            source_vel = np.vstack(
                [
                    past_velocity[
                        nid
                    ]

                    for nid in source_ids
                ]
            )


            target_vel = np.vstack(
                [
                    future_velocity[
                        nid
                    ]

                    for nid in target_ids
                ]
            )


            # ------------------------------------------------
            # Forward source CV query
            # ------------------------------------------------

            source_predicted_target = (
                source_pos
                +
                float(
                    delta_t
                )
                *
                source_vel
            )


            (
                source_cv_dist,
                source_cv_idx,
            ) = query_knn(
                source_predicted_target,
                target_pos,
                K_MAX,
            )


            source_cv_map = make_rank_map(
                source_ids,
                target_ids,
                source_cv_dist,
                source_cv_idx,
                reverse_pair=False,
            )


            # ------------------------------------------------
            # Backward target CV query
            # ------------------------------------------------

            target_predicted_source = (
                target_pos
                -
                float(
                    delta_t
                )
                *
                target_vel
            )


            (
                target_cv_dist,
                target_cv_idx,
            ) = query_knn(
                target_predicted_source,
                source_pos,
                K_MAX,
            )


            target_cv_map = make_rank_map(
                target_ids,
                source_ids,
                target_cv_dist,
                target_cv_idx,
                reverse_pair=True,
            )


            # ------------------------------------------------
            # Reciprocal Top-K intersection
            # ------------------------------------------------

            reciprocal_pairs = (
                set(
                    source_cv_map.keys()
                )
                &
                set(
                    target_cv_map.keys()
                )
            )


            # Deterministic pair iteration.
            for (
                src,
                tgt,
            ) in sorted(
                reciprocal_pairs
            ):

                # Existing edge: no missing-gap repair needed.
                if (
                    src,
                    tgt,
                ) in edge_set:

                    continue


                # --------------------------------------------
                # Reciprocal ranks / residuals
                # --------------------------------------------

                (
                    source_rank,
                    source_residual_um,
                ) = source_cv_map[
                    (
                        src,
                        tgt,
                    )
                ]


                (
                    target_rank,
                    target_residual_um,
                ) = target_cv_map[
                    (
                        src,
                        tgt,
                    )
                ]


                proposal_min_k = max(
                    int(
                        source_rank
                    ),
                    int(
                        target_rank
                    ),
                )


                assert (
                    1
                    <=
                    proposal_min_k
                    <=
                    K_MAX
                )


                # --------------------------------------------
                # Normalize multi-frame position residual by dt
                # --------------------------------------------

                source_cv_per_frame = (
                    float(
                        source_residual_um
                    )
                    /
                    float(
                        delta_t
                    )
                )


                target_cv_per_frame = (
                    float(
                        target_residual_um
                    )
                    /
                    float(
                        delta_t
                    )
                )


                symmetric_cv_per_frame = (
                    0.5
                    *
                    (
                        source_cv_per_frame
                        +
                        target_cv_per_frame
                    )
                )


                # --------------------------------------------
                # Current topology
                # --------------------------------------------

                source_degree = int(
                    outdeg.get(
                        src,
                        0,
                    )
                )


                target_degree = int(
                    indeg.get(
                        tgt,
                        0,
                    )
                )


                assert source_degree <= 1
                assert target_degree <= 1


                n_required_cuts = (
                    int(
                        source_degree == 1
                    )
                    +
                    int(
                        target_degree == 1
                    )
                )


                # --------------------------------------------
                # Incumbent costs
                # --------------------------------------------

                source_incumbent = float(
                    source_incumbent_cost.get(
                        src,
                        np.nan,
                    )
                )


                target_incumbent = float(
                    target_incumbent_cost.get(
                        tgt,
                        np.nan,
                    )
                )


                # --------------------------------------------
                # Side-specific relative improvement
                #
                # Positive means replacement is better.
                # --------------------------------------------

                if (
                    source_degree == 1
                    and
                    np.isfinite(
                        source_incumbent
                    )
                ):

                    source_improvement = (
                        source_incumbent
                        -
                        source_cv_per_frame
                    )

                else:

                    source_improvement = np.nan


                if (
                    target_degree == 1
                    and
                    np.isfinite(
                        target_incumbent
                    )
                ):

                    target_improvement = (
                        target_incumbent
                        -
                        target_cv_per_frame
                    )

                else:

                    target_improvement = np.nan


                # --------------------------------------------
                # Required-side dominance
                # --------------------------------------------

                required_improvements = []


                if source_degree == 1:

                    required_improvements.append(
                        source_improvement
                    )


                if target_degree == 1:

                    required_improvements.append(
                        target_improvement
                    )


                if n_required_cuts == 0:

                    # Free/free needs no incumbent deletion.
                    dominance_any = True
                    dominance_all = True

                else:

                    finite_required = [
                        value

                        for value in required_improvements

                        if np.isfinite(
                            value
                        )
                    ]


                    # All required-side incumbent comparisons
                    # must be available before this GT-blind
                    # dominance test is allowed.
                    if (
                        len(
                            finite_required
                        )
                        !=
                        len(
                            required_improvements
                        )
                    ):

                        dominance_any = False
                        dominance_all = False

                    else:

                        dominance_any = any(
                            value > 0

                            for value in finite_required
                        )


                        dominance_all = all(
                            value > 0

                            for value in finite_required
                        )


                # =================================================
                # GT-BLIND DOMINANCE PRUNE
                # =================================================

                if not dominance_any:

                    continue


                # --------------------------------------------
                # Joint relative cost
                # --------------------------------------------

                incumbent_relevant_cost = finite_mean(
                    [
                        (
                            source_incumbent
                            if source_degree == 1
                            else np.nan
                        ),

                        (
                            target_incumbent
                            if target_degree == 1
                            else np.nan
                        ),
                    ]
                )


                replacement_relevant_cost = finite_mean(
                    [
                        (
                            source_cv_per_frame
                            if source_degree == 1
                            else np.nan
                        ),

                        (
                            target_cv_per_frame
                            if target_degree == 1
                            else np.nan
                        ),
                    ]
                )


                if (
                    n_required_cuts > 0
                    and
                    np.isfinite(
                        incumbent_relevant_cost
                    )
                    and
                    np.isfinite(
                        replacement_relevant_cost
                    )
                ):

                    joint_relative_improvement = (
                        incumbent_relevant_cost
                        -
                        replacement_relevant_cost
                    )

                else:

                    joint_relative_improvement = np.nan


                # --------------------------------------------
                # Append MINIMAL proposal record
                # --------------------------------------------

                chunk_rows.append(
                    (
                        dataset,
                        fold,

                        int(
                            src
                        ),

                        int(
                            tgt
                        ),

                        int(
                            source_t
                        ),

                        int(
                            target_t
                        ),

                        int(
                            gap_size
                        ),

                        int(
                            delta_t
                        ),

                        int(
                            source_degree
                        ),

                        int(
                            target_degree
                        ),

                        int(
                            n_required_cuts
                        ),

                        int(
                            source_rank
                        ),

                        int(
                            target_rank
                        ),

                        int(
                            proposal_min_k
                        ),

                        float(
                            source_cv_per_frame
                        ),

                        float(
                            target_cv_per_frame
                        ),

                        float(
                            symmetric_cv_per_frame
                        ),

                        float(
                            source_incumbent
                        ),

                        float(
                            target_incumbent
                        ),

                        float(
                            source_improvement
                        ),

                        float(
                            target_improvement
                        ),

                        float(
                            joint_relative_improvement
                        ),

                        bool(
                            dominance_any
                        ),

                        bool(
                            dominance_all
                        ),
                    )
                )


                # --------------------------------------------
                # Topology counters
                # --------------------------------------------

                if n_required_cuts == 0:

                    dataset_free_free += 1


                elif (
                    source_degree == 1
                    and
                    target_degree == 0
                ):

                    dataset_source_cut += 1


                elif (
                    source_degree == 0
                    and
                    target_degree == 1
                ):

                    dataset_target_cut += 1


                else:

                    assert (
                        source_degree == 1
                        and
                        target_degree == 1
                    )


                    dataset_both_cut += 1


                # --------------------------------------------
                # Streaming flush
                # --------------------------------------------

                if len(
                    chunk_rows
                ) >= CHUNK_ROWS:

                    (
                        chunk_rows,
                        chunk_index,
                        dataset_total,
                        chunk_files,
                    ) = flush_proposal_chunk(
                        chunk_rows=chunk_rows,
                        chunk_index=chunk_index,
                        dataset_total=dataset_total,
                        chunk_files=chunk_files,
                        dataset_dir=dataset_dir,
                        global_hasher=global_hasher,
                    )


            # ------------------------------------------------
            # Release frame-pair KNN structures
            # ------------------------------------------------

            del source_pos
            del target_pos

            del source_vel
            del target_vel

            del source_predicted_target
            del target_predicted_source

            del source_cv_dist
            del source_cv_idx

            del target_cv_dist
            del target_cv_idx

            del source_cv_map
            del target_cv_map

            del reciprocal_pairs


    # ========================================================
    # 9M. Final flush for this dataset
    # ========================================================

    (
        chunk_rows,
        chunk_index,
        dataset_total,
        chunk_files,
    ) = flush_proposal_chunk(
        chunk_rows=chunk_rows,
        chunk_index=chunk_index,
        dataset_total=dataset_total,
        chunk_files=chunk_files,
        dataset_dir=dataset_dir,
        global_hasher=global_hasher,
    )


    assert len(
        chunk_rows
    ) == 0


    # ========================================================
    # 9N. Dataset accounting
    # ========================================================

    topology_sum = (
        dataset_free_free
        +
        dataset_source_cut
        +
        dataset_target_cut
        +
        dataset_both_cut
    )


    assert topology_sum == dataset_total, (
        dataset,
        topology_sum,
        dataset_total,
    )


    global_total += int(
        dataset_total
    )


    global_free_free += int(
        dataset_free_free
    )


    global_source_cut += int(
        dataset_source_cut
    )


    global_target_cut += int(
        dataset_target_cut
    )


    global_both_cut += int(
        dataset_both_cut
    )


    manifest_datasets.append(
        {
            "dataset":
                dataset,

            "fold":
                int(
                    fold
                ),

            "node_count":
                int(
                    len(
                        node_ids
                    )
                ),

            "source_eligible":
                int(
                    len(
                        source_eligible
                    )
                ),

            "target_eligible":
                int(
                    len(
                        target_eligible
                    )
                ),

            "proposal_rows":
                int(
                    dataset_total
                ),

            "topology_counts": {
                "free_free":
                    int(
                        dataset_free_free
                    ),

                "source_cut":
                    int(
                        dataset_source_cut
                    ),

                "target_cut":
                    int(
                        dataset_target_cut
                    ),

                "both_cut":
                    int(
                        dataset_both_cut
                    ),
            },

            "chunks":
                chunk_files,

            "endpoint_state_file":
                str(
                    endpoint_path.relative_to(
                        RUN_DIR
                    )
                ),
        }
    )


    print(
        f"{ds_i:3d}/{len(names)}"
        f" | fold={fold}"
        f" | {dataset}"
        f" | nodes={len(node_ids)}"
        f" | src={len(source_eligible)}"
        f" | tgt={len(target_eligible)}"
        f" | retained={dataset_total}"
        f" | chunks={len(chunk_files)}"
        f" | cumulative={global_total}"
    )


    # ========================================================
    # 9O. Aggressive dataset cleanup
    # ========================================================

    del graph

    del nodes
    del edges

    del edge_set

    del incoming
    del outgoing

    del indeg
    del outdeg

    del phys
    del time_map
    del node_ids

    del past_velocity
    del future_velocity

    del source_eligible
    del target_eligible

    del source_eligible_set
    del target_eligible_set

    del source_incumbent_cost
    del target_incumbent_cost

    del source_by_t
    del target_by_t

    del endpoint_rows
    del endpoint_df

    del chunk_files

    gc.collect()


# ============================================================
# 10. Global topology accounting
# ============================================================

assert (
    global_free_free
    +
    global_source_cut
    +
    global_target_cut
    +
    global_both_cut
) == global_total


# ============================================================
# 11. FREEZE MANIFEST BEFORE GT READ
# ============================================================

proposal_sha = (
    global_hasher.hexdigest()
)


manifest = {
    "stage":
        "12.11G3E-R",

    "artifact":
        "STREAMING_GT_BLIND_RECIPROCAL_CV_DOMINANCE_ANY_PROPOSAL_UNIVERSE",

    "generator_version":
        "CVTOP5_DOMINANCE_ANY_V1",

    "repair_of":
        "12.11G3E_MEMORY_NONVIABLE",

    "development_datasets":
        int(
            len(
                names
            )
        ),

    "min_gap_size":
        int(
            MIN_GAP_SIZE
        ),

    "max_gap_size":
        int(
            MAX_GAP_SIZE
        ),

    "k_max":
        int(
            K_MAX
        ),

    "chunk_rows":
        int(
            CHUNK_ROWS
        ),

    "proposal_rows":
        int(
            global_total
        ),

    "topology_counts": {
        "free_free":
            int(
                global_free_free
            ),

        "source_cut":
            int(
                global_source_cut
            ),

        "target_cut":
            int(
                global_target_cut
            ),

        "both_cut":
            int(
                global_both_cut
            ),
    },

    "proposal_key_sha256":
        proposal_sha,

    "datasets":
        manifest_datasets,

    "generator": {
        "source_outdegree_max":
            1,

        "target_indegree_max":
            1,

        "source_requires_past_velocity":
            True,

        "target_requires_future_velocity":
            True,

        "proposal_family":
            "RECIPROCAL_FORWARD_BACKWARD_CV_TOPK",

        "distance_union":
            False,

        "free_free_gate":
            "KEEP_RECIPROCAL_TOPK",

        "occupied_gate":
            (
                "AT_LEAST_ONE_REQUIRED_SIDE_HAS_"
                "NORMALIZED_CV_IMPROVEMENT_GT_0"
            ),
    },

    "candidate_generation_gt_blind":
        True,

    "gt_read_before_manifest_freeze":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "threshold_tuned":
        False,

    "policy_selected":
        False,

    "synthetic_nodes_inserted":
        False,

    "graph_modified":
        False,
}


MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)


assert MANIFEST_PATH.exists()


print(
    "\n========== G3E-R GT-BLIND UNIVERSE =========="
)

print(
    "proposal rows:",
    global_total
)

print(
    "free/free:",
    global_free_free
)

print(
    "source cut:",
    global_source_cut
)

print(
    "target cut:",
    global_target_cut
)

print(
    "both cut:",
    global_both_cut
)

print(
    "proposal SHA256:",
    proposal_sha
)

print(
    "GT-BLIND MANIFEST FROZEN BEFORE GT READ: YES"
)


# ============================================================
# PART B — DEVELOPMENT ORACLE AUDIT
#
# GT MAY BE READ ONLY BELOW THIS LINE.
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — DEVELOPMENT ORACLE COVERAGE"
)

print(
    "============================================================"
)


# ============================================================
# 12. Load G3B only AFTER Part A freeze
# ============================================================

oracle = pd.read_pickle(
    G3B_PATH
)


assert len(
    oracle
) == 369


assert set(
    oracle[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}


# ============================================================
# 13. Oracle base
# ============================================================

oracle_eval = oracle[
    [
        "dataset",
        "fold",
        "component_id",

        "gap_size",

        "source_id",
        "target_id",

        "topology_class",

        "source_current_outdegree",
        "target_current_indegree",

        "pred_full_3um",
        "pred_full_5um",
        "pred_full_7um",
    ]
].copy()


oracle_eval[
    "within_generator_horizon"
] = (
    oracle_eval[
        "gap_size"
    ].between(
        MIN_GAP_SIZE,
        MAX_GAP_SIZE,
    )
)


oracle_dataset_set = set(
    oracle_eval[
        "dataset"
    ].unique()
)


# ============================================================
# 14. Read endpoint-state shards only for oracle datasets
# ============================================================

endpoint_lookup_frames = []


for dataset_meta in manifest_datasets:

    dataset = dataset_meta[
        "dataset"
    ]


    if dataset not in oracle_dataset_set:

        continue


    endpoint_path = (
        RUN_DIR
        /
        dataset_meta[
            "endpoint_state_file"
        ]
    )


    assert endpoint_path.exists(), endpoint_path


    endpoint_df = pd.read_pickle(
        endpoint_path
    )


    endpoint_lookup_frames.append(
        endpoint_df
    )


assert endpoint_lookup_frames


endpoint_oracle_state = pd.concat(
    endpoint_lookup_frames,
    ignore_index=True,
)


assert not endpoint_oracle_state[
    [
        "dataset",
        "node_id",
    ]
].duplicated().any()


# ============================================================
# 15. Source state
# ============================================================

source_state = (
    endpoint_oracle_state[
        [
            "dataset",
            "node_id",

            "source_eligible",
            "has_past_velocity",

            "outdegree",
        ]
    ]
    .rename(
        columns={
            "node_id":
                "source_id",

            "has_past_velocity":
                "source_has_past_velocity",

            "outdegree":
                "generator_source_outdegree",
        }
    )
)


# ============================================================
# 16. Target state
# ============================================================

target_state = (
    endpoint_oracle_state[
        [
            "dataset",
            "node_id",

            "target_eligible",
            "has_future_velocity",

            "indegree",
        ]
    ]
    .rename(
        columns={
            "node_id":
                "target_id",

            "has_future_velocity":
                "target_has_future_velocity",

            "indegree":
                "generator_target_indegree",
        }
    )
)


# ============================================================
# 17. Attach endpoint eligibility
# ============================================================

oracle_eval = oracle_eval.merge(
    source_state,
    on=[
        "dataset",
        "source_id",
    ],
    how="left",
    validate="many_to_one",
)


oracle_eval = oracle_eval.merge(
    target_state,
    on=[
        "dataset",
        "target_id",
    ],
    how="left",
    validate="many_to_one",
)


# Because endpoint state stores ALL graph nodes,
# oracle boundaries should resolve exactly.
assert oracle_eval[
    "source_eligible"
].notna().all()


assert oracle_eval[
    "target_eligible"
].notna().all()


assert oracle_eval[
    "source_has_past_velocity"
].notna().all()


assert oracle_eval[
    "target_has_future_velocity"
].notna().all()


# ============================================================
# 18. Build tiny oracle key tables by dataset
# ============================================================

oracle_key_by_dataset = {}


for dataset, sub in oracle_eval.groupby(
    "dataset"
):

    oracle_key_by_dataset[
        dataset
    ] = (
        sub[
            [
                "source_id",
                "target_id",
            ]
        ]
        .drop_duplicates()
        .copy()
    )


# ============================================================
# 19. Scan proposal shards and extract ONLY oracle pairs
#
# Full universe never enters RAM.
# ============================================================

oracle_matches = []


for dataset_meta in manifest_datasets:

    dataset = dataset_meta[
        "dataset"
    ]


    if dataset not in oracle_key_by_dataset:

        continue


    oracle_keys = oracle_key_by_dataset[
        dataset
    ]


    for chunk_meta in dataset_meta[
        "chunks"
    ]:

        chunk_path = (
            RUN_DIR
            /
            dataset
            /
            chunk_meta[
                "file"
            ]
        )


        assert chunk_path.exists(), chunk_path


        chunk = pd.read_pickle(
            chunk_path
        )


        matched = chunk.merge(
            oracle_keys,
            on=[
                "source_id",
                "target_id",
            ],
            how="inner",
            validate="many_to_many",
        )


        if len(
            matched
        ) > 0:

            oracle_matches.append(
                matched
            )


        del chunk
        del matched


    gc.collect()


if oracle_matches:

    matched_pairs = pd.concat(
        oracle_matches,
        ignore_index=True,
    )


else:

    matched_pairs = pd.DataFrame(
        columns=COLUMNS
    )


assert not matched_pairs[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


# ============================================================
# 20. Attach frozen proposal characteristics
# ============================================================

join_cols = [
    "dataset",
    "source_id",
    "target_id",

    "proposal_min_k",

    "source_cv_rank",
    "target_cv_rank",

    "source_outdegree",
    "target_indegree",

    "n_required_cuts",

    "source_cv_per_frame",
    "target_cv_per_frame",
    "symmetric_cv_per_frame",

    "source_relative_improvement",
    "target_relative_improvement",

    "joint_relative_improvement",

    "dominance_any",
    "dominance_all",
]


oracle_eval = oracle_eval.merge(
    matched_pairs[
        join_cols
    ],
    on=[
        "dataset",
        "source_id",
        "target_id",
    ],
    how="left",
    validate="one_to_one",
    indicator=True,
)


oracle_eval[
    "in_frozen_proposal_universe"
] = (
    oracle_eval[
        "_merge"
    ].eq(
        "both"
    )
)


oracle_eval = oracle_eval.drop(
    columns=[
        "_merge"
    ]
)


# ============================================================
# 21. Miss classification
# ============================================================

def coverage_reason(
    row,
):

    if bool(
        row.in_frozen_proposal_universe
    ):

        return (
            "IN_FROZEN_PROPOSAL_UNIVERSE"
        )


    if not bool(
        row.within_generator_horizon
    ):

        return (
            "OUTSIDE_GENERATOR_HORIZON"
        )


    if not bool(
        row.source_has_past_velocity
    ):

        return (
            "SOURCE_NO_PAST_VELOCITY"
        )


    if int(
        row.generator_source_outdegree
    ) > 1:

        return (
            "SOURCE_OUTDEGREE_GT1"
        )


    if not bool(
        row.source_eligible
    ):

        return (
            "SOURCE_NOT_ELIGIBLE_OTHER"
        )


    if not bool(
        row.target_has_future_velocity
    ):

        return (
            "TARGET_NO_FUTURE_VELOCITY"
        )


    if int(
        row.generator_target_indegree
    ) > 1:

        return (
            "TARGET_INDEGREE_GT1"
        )


    if not bool(
        row.target_eligible
    ):

        return (
            "TARGET_NOT_ELIGIBLE_OTHER"
        )


    # At this point:
    #
    #   - temporal horizon valid
    #   - source endpoint eligible
    #   - target endpoint eligible
    #
    # Therefore miss is caused by:
    #   reciprocal Top-K failure
    #       OR
    #   occupied dominance-any failure.
    #
    # These are separated in a later diagnostic if needed.
    return (
        "RECIPROCAL_TOPK_OR_DOMINANCE_MISS"
    )


oracle_eval[
    "coverage_reason"
] = oracle_eval.apply(
    coverage_reason,
    axis=1,
)


# ============================================================
# 22. Overall oracle coverage
# ============================================================

captured = int(
    oracle_eval[
        "in_frozen_proposal_universe"
    ].sum()
)


print(
    "\n========== G3E-R ORACLE COVERAGE =========="
)

print(
    "clean components:",
    len(
        oracle_eval
    )
)

print(
    "captured:",
    captured,
    "/",
    len(
        oracle_eval
    ),
    f"({captured/len(oracle_eval):.3%})",
)


print(
    "\n========== G3E-R MISS REASONS =========="
)

print(
    oracle_eval[
        "coverage_reason"
    ]
    .value_counts()
    .to_string()
)


# ============================================================
# 23. Nested K coverage
#
# proposal_min_k <= K
# ============================================================

k_rows = []


print(
    "\n========== G3E-R COVERAGE BY K =========="
)


for k in K_AUDIT:

    captured_k_mask = (
        oracle_eval[
            "proposal_min_k"
        ]
        .le(
            k
        )
        .fillna(
            False
        )
    )


    captured_k = int(
        captured_k_mask.sum()
    )


    # --------------------------------------------------------
    # Within generator temporal horizon
    # --------------------------------------------------------

    horizon_mask = (
        oracle_eval[
            "within_generator_horizon"
        ]
        .astype(bool)
    )


    horizon_total = int(
        horizon_mask.sum()
    )


    horizon_captured = int(
        (
            captured_k_mask
            &
            horizon_mask
        ).sum()
    )


    # --------------------------------------------------------
    # Interpolation <=7 μm ceiling subset
    # --------------------------------------------------------

    full7_mask = (
        oracle_eval[
            "pred_full_7um"
        ]
        .astype(bool)
    )


    full7_total = int(
        full7_mask.sum()
    )


    full7_captured = int(
        (
            captured_k_mask
            &
            full7_mask
        ).sum()
    )


    k_rows.append(
        {
            "k":
                int(
                    k
                ),

            "captured":
                captured_k,

            "total":
                int(
                    len(
                        oracle_eval
                    )
                ),

            "coverage":
                float(
                    captured_k
                    /
                    len(
                        oracle_eval
                    )
                ),

            "horizon_total":
                horizon_total,

            "horizon_captured":
                horizon_captured,

            "horizon_coverage":
                float(
                    horizon_captured
                    /
                    horizon_total
                ),

            "full7_total":
                full7_total,

            "full7_captured":
                full7_captured,

            "full7_coverage":
                float(
                    full7_captured
                    /
                    full7_total
                ),
        }
    )


    print(
        f"K={k}"
        f" | all={captured_k}/{len(oracle_eval)}"
        f" ({captured_k/len(oracle_eval):.3%})"
        f" | horizon={horizon_captured}/{horizon_total}"
        f" ({horizon_captured/horizon_total:.3%})"
        f" | full<=7={full7_captured}/{full7_total}"
        f" ({full7_captured/full7_total:.3%})"
    )


k_table = pd.DataFrame(
    k_rows
)


# ============================================================
# 24. Coverage by topology
# ============================================================

print(
    "\n========== G3E-R COVERAGE BY TOPOLOGY =========="
)


topology_table = (
    oracle_eval.groupby(
        "topology_class"
    )
    .agg(
        components=(
            "component_id",
            "size",
        ),

        captured=(
            "in_frozen_proposal_universe",
            "sum",
        ),

        pred_full_7um=(
            "pred_full_7um",
            "sum",
        ),
    )
)


topology_table[
    "coverage"
] = (
    topology_table[
        "captured"
    ]
    /
    topology_table[
        "components"
    ]
)


print(
    topology_table.to_string()
)


# ============================================================
# 25. Coverage by gap size
# ============================================================

print(
    "\n========== G3E-R COVERAGE BY GAP SIZE =========="
)


gap_table = (
    oracle_eval.groupby(
        "gap_size"
    )
    .agg(
        components=(
            "component_id",
            "size",
        ),

        captured=(
            "in_frozen_proposal_universe",
            "sum",
        ),

        pred_full_7um=(
            "pred_full_7um",
            "sum",
        ),
    )
)


gap_table[
    "coverage"
] = (
    gap_table[
        "captured"
    ]
    /
    gap_table[
        "components"
    ]
)


print(
    gap_table.to_string()
)


# ============================================================
# 26. Captured TRUE-pair characteristics
# ============================================================

true_pairs = (
    oracle_eval[
        oracle_eval[
            "in_frozen_proposal_universe"
        ].astype(bool)
    ]
    .copy()
)


print(
    "\n========== G3E-R TRUE-PAIR CHARACTERISTICS =========="
)


print(
    "captured true pairs:",
    len(
        true_pairs
    )
)


if len(
    true_pairs
) > 0:

    print(
        "\nproposal_min_k:"
    )


    print(
        true_pairs[
            "proposal_min_k"
        ]
        .value_counts()
        .sort_index()
        .to_string()
    )


    print(
        "\nrequired cuts:"
    )


    print(
        true_pairs[
            "n_required_cuts"
        ]
        .value_counts()
        .sort_index()
        .to_string()
    )


    print(
        "\ndominance_all:"
    )


    print(
        true_pairs[
            "dominance_all"
        ]
        .value_counts()
        .to_string()
    )


    print(
        "\nsymmetric CV/frame:"
    )


    print(
        true_pairs[
            "symmetric_cv_per_frame"
        ]
        .describe(
            percentiles=[
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99,
            ]
        )
        .to_string()
    )


    occupied_true = true_pairs[
        true_pairs[
            "n_required_cuts"
        ].gt(
            0
        )
    ]


    print(
        "\noccupied true-pair joint relative improvement:"
    )


    if len(
        occupied_true
    ) > 0:

        print(
            occupied_true[
                "joint_relative_improvement"
            ]
            .describe(
                percentiles=[
                    0.10,
                    0.25,
                    0.50,
                    0.75,
                    0.90,
                    0.95,
                    0.99,
                ]
            )
            .to_string()
        )

    else:

        print(
            "NO OCCUPIED TRUE PAIRS"
        )


else:

    print(
        "NO TRUE PAIRS CAPTURED"
    )


# ============================================================
# 27. Persist formal oracle audit
# ============================================================

oracle_eval = (
    oracle_eval
    .sort_values(
        [
            "fold",
            "dataset",
            "component_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


oracle_eval.to_pickle(
    ORACLE_OUT
)


oracle_summary = {
    "stage":
        "12.11G3E-R",

    "status":
        "STREAMING_GT_BLIND_RECIPROCAL_CV_DOMINANCE_ANY_UNIVERSE_AUDITED",

    "generator_version":
        "CVTOP5_DOMINANCE_ANY_V1",

    "proposal_rows":
        int(
            global_total
        ),

    "proposal_sha256":
        proposal_sha,

    "clean_components":
        int(
            len(
                oracle_eval
            )
        ),

    "captured":
        int(
            captured
        ),

    "coverage":
        float(
            captured
            /
            len(
                oracle_eval
            )
        ),

    "coverage_by_k":
        k_table.to_dict(
            "records"
        ),

    "miss_reasons":
        {
            str(
                key
            ):
                int(
                    value
                )

            for (
                key,
                value
            )
            in oracle_eval[
                "coverage_reason"
            ]
            .value_counts()
            .items()
        },

    "candidate_generation_gt_blind":
        True,

    "oracle_loaded_after_manifest_freeze":
        True,

    "full_proposal_universe_loaded_in_ram":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "threshold_tuned":
        False,

    "policy_selected":
        False,

    "synthetic_nodes_inserted":
        False,

    "graph_modified":
        False,
}


ORACLE_SUMMARY_OUT.write_text(
    json.dumps(
        oracle_summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert ORACLE_OUT.exists()
assert ORACLE_SUMMARY_OUT.exists()


# ============================================================
# 28. Reload integrity
# ============================================================

oracle_check = pd.read_pickle(
    ORACLE_OUT
)


assert len(
    oracle_check
) == 369


assert not oracle_check[
    [
        "dataset",
        "component_id",
    ]
].duplicated().any()


assert set(
    oracle_check[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}


# ============================================================
# 29. FINAL DECISION
# ============================================================

print(
    "\n========== 12.11G3E-R DECISION =========="
)

print(
    "STREAMING_GT_BLIND_MULTIFRAME_PROPOSAL_UNIVERSE: PASS"
)

print(
    "proposal rows:",
    global_total
)

print(
    "proposal SHA256:",
    proposal_sha
)

print(
    "oracle captured:",
    captured,
    "/",
    len(
        oracle_eval
    ),
    f"({captured/len(oracle_eval):.3%})",
)

print(
    "original G3E memory failure repaired: YES"
)

print(
    "nonlocal scope bug repaired: YES"
)

print(
    "full universe ever loaded into RAM: NO"
)

print(
    "GT loaded after proposal freeze: YES"
)

print(
    "Fold0 used: NO"
)

print(
    "classifier trained: NO"
)

print(
    "threshold tuned: NO"
)

print(
    "policy selected: NO"
)

print(
    "synthetic nodes inserted: NO"
)

print(
    "graph modified: NO"
)

print(
    "next:"
)

print(
    "AUDIT_RECIPROCAL_VS_DOMINANCE_MISS_AND_CROSS_FOLD_POLICY_FEASIBILITY"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3F0-R
#
# Exact-K5 Reciprocal-vs-Dominance Miss Decomposition
#
# REPAIR:
#
#   Old G3F0 incorrectly inferred original K=5 membership from
#   a K=64 cKDTree query using rank <= 5.
#
#   This is NOT exact when distance ties exist.
#
# This repair performs TWO INDEPENDENT queries:
#
#   A. EXACT K=5
#      -> reconstruct frozen G3E-R exactly
#
#   B. DIAGNOSTIC K=64
#      -> estimate wider retrieval ceiling only
#
# Also reproduces:
#
#   - temporal horizon
#   - endpoint eligibility
#   - existing-edge exclusion
#   - dominance-any
#
# Scientific purpose:
#
#   Decompose the 149 G3E-R unresolved misses into:
#
#       RECIPROCAL_ONLY_MISS
#       DOMINANCE_ONLY_MISS
#       RECIPROCAL_AND_DOMINANCE_MISS
#
# No new proposal universe.
# No Fold0.
# No classifier.
# No threshold selection.
# No policy selection.
# No graph modification.
#
# Keep after running: YES
# ============================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree


# ============================================================
# 0. Paths / contract
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


G3B_PATH = (
    S12
    / "stage12_folds14_occupied_boundary_rewire_feasibility.pkl"
)


G3ER_ORACLE_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_cvtop5_dominance_any_v1_oracle.pkl"
)


OUT = (
    S12
    / "stage12_folds14_multiframe_rewire_gate_miss_decomposition_exactk5.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_folds14_multiframe_rewire_gate_miss_decomposition_exactk5_summary.json"
)


assert G3B_PATH.exists(), G3B_PATH
assert G3ER_ORACLE_PATH.exists(), G3ER_ORACLE_PATH


assert (
    "build_current_best"
    in globals()
    and
    callable(
        globals()[
            "build_current_best"
        ]
    )
), (
    "build_current_best is not present in the current kernel. "
    "Re-run the retained successful G3E-R cell first."
)


current_best_builder = globals()[
    "build_current_best"
]


SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


MIN_GAP_SIZE = 2
MAX_GAP_SIZE = 12

EXACT_K = 5
DIAG_K_MAX = 64


K_AUDIT = [
    1,
    2,
    3,
    5,
    8,
    12,
    16,
    24,
    32,
    64,
]


# ============================================================
# 1. Load development oracle artifacts
# ============================================================

g3b = pd.read_pickle(
    G3B_PATH
)


frozen = pd.read_pickle(
    G3ER_ORACLE_PATH
)


assert len(
    g3b
) == 369


assert len(
    frozen
) == 369


assert not frozen[
    [
        "dataset",
        "component_id",
    ]
].duplicated().any()


assert int(
    frozen[
        "in_frozen_proposal_universe"
    ].sum()
) == 185


assert int(
    frozen[
        "coverage_reason"
    ]
    .eq(
        "RECIPROCAL_TOPK_OR_DOMINANCE_MISS"
    )
    .sum()
) == 149


print(
    "========== 12.11G3F0-R INPUT =========="
)

print(
    "clean components:",
    len(
        frozen
    )
)

print(
    "frozen G3E-R captured:",
    int(
        frozen[
            "in_frozen_proposal_universe"
        ].sum()
    )
)

print(
    "previous mixed misses:",
    int(
        frozen[
            "coverage_reason"
        ]
        .eq(
            "RECIPROCAL_TOPK_OR_DOMINANCE_MISS"
        )
        .sum()
    )
)

print(
    "exact reconstruction K:",
    EXACT_K
)

print(
    "diagnostic rank K:",
    DIAG_K_MAX
)


# ============================================================
# 2. Canonical oracle
# ============================================================

oracle = (
    g3b[
        [
            "dataset",
            "fold",
            "component_id",

            "gap_size",

            "source_id",
            "target_id",

            "topology_class",

            "source_occupied",
            "target_occupied",

            "pred_full_3um",
            "pred_full_5um",
            "pred_full_7um",
        ]
    ]
    .copy()
)


oracle = oracle.merge(
    frozen[
        [
            "dataset",
            "component_id",

            "in_frozen_proposal_universe",
            "coverage_reason",
        ]
    ],
    on=[
        "dataset",
        "component_id",
    ],
    how="left",
    validate="one_to_one",
)


assert oracle[
    "coverage_reason"
].notna().all()


datasets = (
    oracle[
        "dataset"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)


assert len(
    datasets
) == 75


print(
    "oracle datasets:",
    len(
        datasets
    )
)


# ============================================================
# 3. Helpers
# ============================================================

def vec_norm(
    x,
):

    return float(
        np.linalg.norm(
            np.asarray(
                x,
                dtype=float,
            )
        )
    )


def query_knn(
    query_points,
    candidate_points,
    k,
):

    query_points = np.asarray(
        query_points,
        dtype=float,
    )

    candidate_points = np.asarray(
        candidate_points,
        dtype=float,
    )


    assert query_points.ndim == 2
    assert candidate_points.ndim == 2


    if (
        len(
            query_points
        ) == 0
        or
        len(
            candidate_points
        ) == 0
    ):

        return (
            np.empty(
                (
                    len(
                        query_points
                    ),
                    0,
                ),
                dtype=float,
            ),

            np.empty(
                (
                    len(
                        query_points
                    ),
                    0,
                ),
                dtype=int,
            ),
        )


    k_eff = min(
        int(
            k
        ),
        int(
            len(
                candidate_points
            )
        ),
    )


    tree = cKDTree(
        candidate_points
    )


    distances, indices = tree.query(
        query_points,
        k=k_eff,
    )


    if k_eff == 1:

        distances = distances[
            :,
            None
        ]

        indices = indices[
            :,
            None
        ]


    return (
        np.asarray(
            distances,
            dtype=float,
        ),

        np.asarray(
            indices,
            dtype=int,
        ),
    )


def find_rank(
    indices_row,
    candidate_ids,
    true_id,
):

    for rank_i, candidate_index in enumerate(
        indices_row
    ):

        candidate_id = int(
            candidate_ids[
                int(
                    candidate_index
                )
            ]
        )


        if candidate_id == int(
            true_id
        ):

            return int(
                rank_i
                +
                1
            )


    return np.nan


# ============================================================
# 4. Reconstruct each oracle dataset
# ============================================================

records = []


print(
    "\n============================================================"
)

print(
    "EXACT K=5 RECONSTRUCTION + K=64 DIAGNOSTIC"
)

print(
    "============================================================"
)


for ds_i, dataset in enumerate(
    datasets,
    start=1,
):

    graph, scale = current_best_builder(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    )


    # ========================================================
    # 4A. Nodes
    # ========================================================

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = (
        nodes[
            "node_id"
        ]
        .astype(int)
    )


    nodes[
        "t"
    ] = (
        nodes[
            "t"
        ]
        .astype(int)
    )


    nodes = (
        nodes
        .sort_values(
            [
                "t",
                "node_id",
            ],
            kind="stable",
        )
        .set_index(
            "node_id",
            drop=False,
        )
    )


    # ========================================================
    # 4B. Edges
    # ========================================================

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = (
        edges[
            "source_id"
        ]
        .astype(int)
    )


    edges[
        "target_id"
    ] = (
        edges[
            "target_id"
        ]
        .astype(int)
    )


    edge_set = set(
        zip(
            edges[
                "source_id"
            ],
            edges[
                "target_id"
            ],
        )
    )


    incoming = (
        edges.groupby(
            "target_id"
        )[
            "source_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    outgoing = (
        edges.groupby(
            "source_id"
        )[
            "target_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = {
        int(k):
            int(v)

        for k, v in indeg.items()
    }


    outdeg = {
        int(k):
            int(v)

        for k, v in outdeg.items()
    }


    # ========================================================
    # 4C. Physical state
    # ========================================================

    phys = {}

    time_map = {}


    for nid, node in nodes.iterrows():

        nid = int(
            nid
        )


        phys[
            nid
        ] = (
            np.array(
                [
                    float(
                        node[
                            "z"
                        ]
                    ),

                    float(
                        node[
                            "y"
                        ]
                    ),

                    float(
                        node[
                            "x"
                        ]
                    ),
                ],
                dtype=float,
            )
            *
            SCALE
        )


        time_map[
            nid
        ] = int(
            node[
                "t"
            ]
        )


    node_ids = sorted(
        phys.keys()
    )


    # ========================================================
    # 4D. Local velocities
    # ========================================================

    past_velocity = {}

    future_velocity = {}


    for nid in node_ids:

        inc = incoming.get(
            nid,
            [],
        )


        if len(
            inc
        ) == 1:

            prev_id = int(
                inc[
                    0
                ]
            )


            if prev_id in phys:

                dt = (
                    time_map[
                        nid
                    ]
                    -
                    time_map[
                        prev_id
                    ]
                )


                if dt > 0:

                    past_velocity[
                        nid
                    ] = (
                        phys[
                            nid
                        ]
                        -
                        phys[
                            prev_id
                        ]
                    ) / float(
                        dt
                    )


        out = outgoing.get(
            nid,
            [],
        )


        if len(
            out
        ) == 1:

            next_id = int(
                out[
                    0
                ]
            )


            if next_id in phys:

                dt = (
                    time_map[
                        next_id
                    ]
                    -
                    time_map[
                        nid
                    ]
                )


                if dt > 0:

                    future_velocity[
                        nid
                    ] = (
                        phys[
                            next_id
                        ]
                        -
                        phys[
                            nid
                        ]
                    ) / float(
                        dt
                    )


    # ========================================================
    # 4E. EXACT G3E-R eligibility
    # ========================================================

    source_eligible = [
        nid

        for nid in node_ids

        if (
            outdeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid
            in
            past_velocity
        )
    ]


    target_eligible = [
        nid

        for nid in node_ids

        if (
            indeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid
            in
            future_velocity
        )
    ]


    source_eligible_set = set(
        source_eligible
    )


    target_eligible_set = set(
        target_eligible
    )


    source_by_t = {}

    target_by_t = {}


    for nid in source_eligible:

        source_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for nid in target_eligible:

        target_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for t in source_by_t:

        source_by_t[
            t
        ] = sorted(
            source_by_t[
                t
            ]
        )


    for t in target_by_t:

        target_by_t[
            t
        ] = sorted(
            target_by_t[
                t
            ]
        )


    # ========================================================
    # 4F. Incumbent normalized costs
    # ========================================================

    source_incumbent_cost = {}

    target_incumbent_cost = {}


    for src in source_eligible:

        if outdeg.get(
            src,
            0,
        ) != 1:

            continue


        current_out = outgoing.get(
            src,
            [],
        )


        if len(
            current_out
        ) != 1:

            continue


        old_tgt = int(
            current_out[
                0
            ]
        )


        if old_tgt not in phys:

            continue


        dt_old = (
            time_map[
                old_tgt
            ]
            -
            time_map[
                src
            ]
        )


        if dt_old <= 0:

            continue


        predicted_target = (
            phys[
                src
            ]
            +
            float(
                dt_old
            )
            *
            past_velocity[
                src
            ]
        )


        source_incumbent_cost[
            src
        ] = (
            vec_norm(
                phys[
                    old_tgt
                ]
                -
                predicted_target
            )
            /
            float(
                dt_old
            )
        )


    for tgt in target_eligible:

        if indeg.get(
            tgt,
            0,
        ) != 1:

            continue


        current_in = incoming.get(
            tgt,
            [],
        )


        if len(
            current_in
        ) != 1:

            continue


        old_src = int(
            current_in[
                0
            ]
        )


        if old_src not in phys:

            continue


        dt_old = (
            time_map[
                tgt
            ]
            -
            time_map[
                old_src
            ]
        )


        if dt_old <= 0:

            continue


        predicted_source = (
            phys[
                tgt
            ]
            -
            float(
                dt_old
            )
            *
            future_velocity[
                tgt
            ]
        )


        target_incumbent_cost[
            tgt
        ] = (
            vec_norm(
                phys[
                    old_src
                ]
                -
                predicted_source
            )
            /
            float(
                dt_old
            )
        )


    # ========================================================
    # 4G. Prepare oracle records
    # ========================================================

    ds_oracle = (
        oracle[
            oracle[
                "dataset"
            ].eq(
                dataset
            )
        ]
        .copy()
    )


    local_records = {}


    frame_groups = {}


    for comp in ds_oracle.itertuples(
        index=False
    ):

        src = int(
            comp.source_id
        )

        tgt = int(
            comp.target_id
        )

        gap_size = int(
            comp.gap_size
        )


        within_horizon = (
            MIN_GAP_SIZE
            <=
            gap_size
            <=
            MAX_GAP_SIZE
        )


        source_has_past = (
            src
            in
            past_velocity
        )


        target_has_future = (
            tgt
            in
            future_velocity
        )


        source_is_eligible = (
            src
            in
            source_eligible_set
        )


        target_is_eligible = (
            tgt
            in
            target_eligible_set
        )


        pair_existing = (
            (
                src,
                tgt,
            )
            in
            edge_set
        )


        key = (
            dataset,
            comp.component_id,
        )


        local_records[
            key
        ] = {
            "dataset":
                dataset,

            "fold":
                int(
                    comp.fold
                ),

            "component_id":
                comp.component_id,

            "gap_size":
                gap_size,

            "source_id":
                src,

            "target_id":
                tgt,

            "topology_class":
                comp.topology_class,

            "source_occupied":
                bool(
                    comp.source_occupied
                ),

            "target_occupied":
                bool(
                    comp.target_occupied
                ),

            "pred_full_3um":
                bool(
                    comp.pred_full_3um
                ),

            "pred_full_5um":
                bool(
                    comp.pred_full_5um
                ),

            "pred_full_7um":
                bool(
                    comp.pred_full_7um
                ),

            "within_horizon":
                bool(
                    within_horizon
                ),

            "source_has_past_velocity":
                bool(
                    source_has_past
                ),

            "target_has_future_velocity":
                bool(
                    target_has_future
                ),

            "source_eligible":
                bool(
                    source_is_eligible
                ),

            "target_eligible":
                bool(
                    target_is_eligible
                ),

            "pair_existing_edge":
                bool(
                    pair_existing
                ),

            # --------------------------------------------
            # EXACT K=5 reconstruction
            # --------------------------------------------

            "exact_source_rank_k5":
                np.nan,

            "exact_target_rank_k5":
                np.nan,

            "exact_proposal_min_k":
                np.nan,

            "exact_reciprocal_top5":
                False,

            # --------------------------------------------
            # K=64 diagnostic only
            # --------------------------------------------

            "diag_source_rank_k64":
                np.nan,

            "diag_target_rank_k64":
                np.nan,

            "diag_reciprocal_min_k":
                np.nan,

            # --------------------------------------------
            # Candidate costs / dominance
            # --------------------------------------------

            "source_cv_per_frame":
                np.nan,

            "target_cv_per_frame":
                np.nan,

            "source_incumbent_cv_per_frame":
                np.nan,

            "target_incumbent_cv_per_frame":
                np.nan,

            "source_relative_improvement":
                np.nan,

            "target_relative_improvement":
                np.nan,

            "dominance_any":
                False,

            "dominance_all":
                False,
        }


        if (
            within_horizon
            and
            source_is_eligible
            and
            target_is_eligible
        ):

            source_t = int(
                time_map[
                    src
                ]
            )


            target_t = int(
                time_map[
                    tgt
                ]
            )


            delta_t = (
                target_t
                -
                source_t
            )


            assert delta_t == (
                gap_size
                +
                1
            )


            frame_groups.setdefault(
                (
                    source_t,
                    target_t,
                ),
                [],
            ).append(
                key
            )


    # ========================================================
    # 4H. Exact frame-pair KNN queries
    # ========================================================

    for (
        source_t,
        target_t,
    ), record_keys in frame_groups.items():

        source_ids = source_by_t[
            source_t
        ]


        target_ids = target_by_t[
            target_t
        ]


        delta_t = (
            target_t
            -
            source_t
        )


        source_pos = np.vstack(
            [
                phys[
                    nid
                ]

                for nid in source_ids
            ]
        )


        target_pos = np.vstack(
            [
                phys[
                    nid
                ]

                for nid in target_ids
            ]
        )


        source_vel = np.vstack(
            [
                past_velocity[
                    nid
                ]

                for nid in source_ids
            ]
        )


        target_vel = np.vstack(
            [
                future_velocity[
                    nid
                ]

                for nid in target_ids
            ]
        )


        source_predicted_target = (
            source_pos
            +
            float(
                delta_t
            )
            *
            source_vel
        )


        target_predicted_source = (
            target_pos
            -
            float(
                delta_t
            )
            *
            target_vel
        )


        # ====================================================
        # CRITICAL:
        # exact original K=5 query
        # ====================================================

        (
            exact_source_dist,
            exact_source_idx,
        ) = query_knn(
            source_predicted_target,
            target_pos,
            EXACT_K,
        )


        (
            exact_target_dist,
            exact_target_idx,
        ) = query_knn(
            target_predicted_source,
            source_pos,
            EXACT_K,
        )


        # ====================================================
        # Separate K=64 diagnostic query
        #
        # Never used for exact G3E-R reconstruction.
        # ====================================================

        (
            diag_source_dist,
            diag_source_idx,
        ) = query_knn(
            source_predicted_target,
            target_pos,
            DIAG_K_MAX,
        )


        (
            diag_target_dist,
            diag_target_idx,
        ) = query_knn(
            target_predicted_source,
            source_pos,
            DIAG_K_MAX,
        )


        source_index = {
            int(
                nid
            ):
                i

            for i, nid
            in enumerate(
                source_ids
            )
        }


        target_index = {
            int(
                nid
            ):
                i

            for i, nid
            in enumerate(
                target_ids
            )
        }


        # ====================================================
        # 4I. Resolve each TRUE pair
        # ====================================================

        for key in record_keys:

            rec = local_records[
                key
            ]


            src = int(
                rec[
                    "source_id"
                ]
            )


            tgt = int(
                rec[
                    "target_id"
                ]
            )


            src_i = source_index[
                src
            ]


            tgt_i = target_index[
                tgt
            ]


            # ------------------------------------------------
            # EXACT K=5 ranks
            # ------------------------------------------------

            exact_source_rank = find_rank(
                exact_source_idx[
                    src_i
                ],
                target_ids,
                tgt,
            )


            exact_target_rank = find_rank(
                exact_target_idx[
                    tgt_i
                ],
                source_ids,
                src,
            )


            rec[
                "exact_source_rank_k5"
            ] = exact_source_rank


            rec[
                "exact_target_rank_k5"
            ] = exact_target_rank


            if (
                np.isfinite(
                    exact_source_rank
                )
                and
                np.isfinite(
                    exact_target_rank
                )
            ):

                rec[
                    "exact_reciprocal_top5"
                ] = True


                rec[
                    "exact_proposal_min_k"
                ] = max(
                    int(
                        exact_source_rank
                    ),
                    int(
                        exact_target_rank
                    ),
                )


            # ------------------------------------------------
            # Independent K=64 diagnostic ranks
            # ------------------------------------------------

            diag_source_rank = find_rank(
                diag_source_idx[
                    src_i
                ],
                target_ids,
                tgt,
            )


            diag_target_rank = find_rank(
                diag_target_idx[
                    tgt_i
                ],
                source_ids,
                src,
            )


            rec[
                "diag_source_rank_k64"
            ] = diag_source_rank


            rec[
                "diag_target_rank_k64"
            ] = diag_target_rank


            if (
                np.isfinite(
                    diag_source_rank
                )
                and
                np.isfinite(
                    diag_target_rank
                )
            ):

                rec[
                    "diag_reciprocal_min_k"
                ] = max(
                    int(
                        diag_source_rank
                    ),
                    int(
                        diag_target_rank
                    ),
                )


            # ------------------------------------------------
            # TRUE replacement normalized CV
            # ------------------------------------------------

            source_residual_um = vec_norm(
                phys[
                    tgt
                ]
                -
                (
                    phys[
                        src
                    ]
                    +
                    float(
                        delta_t
                    )
                    *
                    past_velocity[
                        src
                    ]
                )
            )


            target_residual_um = vec_norm(
                phys[
                    src
                ]
                -
                (
                    phys[
                        tgt
                    ]
                    -
                    float(
                        delta_t
                    )
                    *
                    future_velocity[
                        tgt
                    ]
                )
            )


            source_cv_pf = (
                source_residual_um
                /
                float(
                    delta_t
                )
            )


            target_cv_pf = (
                target_residual_um
                /
                float(
                    delta_t
                )
            )


            rec[
                "source_cv_per_frame"
            ] = float(
                source_cv_pf
            )


            rec[
                "target_cv_per_frame"
            ] = float(
                target_cv_pf
            )


            # ------------------------------------------------
            # Incumbent-relative dominance
            # ------------------------------------------------

            source_degree = int(
                outdeg.get(
                    src,
                    0,
                )
            )


            target_degree = int(
                indeg.get(
                    tgt,
                    0,
                )
            )


            required = []


            if source_degree == 1:

                src_incumbent = float(
                    source_incumbent_cost.get(
                        src,
                        np.nan,
                    )
                )


                rec[
                    "source_incumbent_cv_per_frame"
                ] = src_incumbent


                if np.isfinite(
                    src_incumbent
                ):

                    source_improvement = (
                        src_incumbent
                        -
                        source_cv_pf
                    )


                    rec[
                        "source_relative_improvement"
                    ] = float(
                        source_improvement
                    )


                    required.append(
                        source_improvement
                    )


                else:

                    required.append(
                        np.nan
                    )


            if target_degree == 1:

                tgt_incumbent = float(
                    target_incumbent_cost.get(
                        tgt,
                        np.nan,
                    )
                )


                rec[
                    "target_incumbent_cv_per_frame"
                ] = tgt_incumbent


                if np.isfinite(
                    tgt_incumbent
                ):

                    target_improvement = (
                        tgt_incumbent
                        -
                        target_cv_pf
                    )


                    rec[
                        "target_relative_improvement"
                    ] = float(
                        target_improvement
                    )


                    required.append(
                        target_improvement
                    )


                else:

                    required.append(
                        np.nan
                    )


            if len(
                required
            ) == 0:

                rec[
                    "dominance_any"
                ] = True


                rec[
                    "dominance_all"
                ] = True


            else:

                finite_required = [
                    value

                    for value in required

                    if np.isfinite(
                        value
                    )
                ]


                if (
                    len(
                        finite_required
                    )
                    ==
                    len(
                        required
                    )
                ):

                    rec[
                        "dominance_any"
                    ] = any(
                        value > 0

                        for value in finite_required
                    )


                    rec[
                        "dominance_all"
                    ] = all(
                        value > 0

                        for value in finite_required
                    )


        # ====================================================
        # Release frame-pair arrays
        # ====================================================

        del source_pos
        del target_pos

        del source_vel
        del target_vel

        del source_predicted_target
        del target_predicted_source

        del exact_source_dist
        del exact_source_idx

        del exact_target_dist
        del exact_target_idx

        del diag_source_dist
        del diag_source_idx

        del diag_target_dist
        del diag_target_idx


    records.extend(
        local_records.values()
    )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            datasets
        )
    ):

        processable = sum(
            (
                bool(
                    rec[
                        "within_horizon"
                    ]
                )
                and
                bool(
                    rec[
                        "source_eligible"
                    ]
                )
                and
                bool(
                    rec[
                        "target_eligible"
                    ]
                )
            )

            for rec
            in local_records.values()
        )


        print(
            f"{ds_i:2d}/{len(datasets)}"
            f" | {dataset}"
            f" | oracle={len(ds_oracle)}"
            f" | processable={processable}"
        )


    # ========================================================
    # Cleanup
    # ========================================================

    del graph
    del nodes
    del edges

    del edge_set

    del incoming
    del outgoing

    del indeg
    del outdeg

    del phys
    del time_map

    del node_ids

    del past_velocity
    del future_velocity

    del source_eligible
    del target_eligible

    del source_eligible_set
    del target_eligible_set

    del source_by_t
    del target_by_t

    del source_incumbent_cost
    del target_incumbent_cost

    del local_records
    del frame_groups

    gc.collect()


# ============================================================
# 5. Canonical reconstructed table
# ============================================================

audit = pd.DataFrame(
    records
)


assert len(
    audit
) == 369


assert not audit[
    [
        "dataset",
        "component_id",
    ]
].duplicated().any()


# ============================================================
# 6. Attach frozen G3E-R result
# ============================================================

audit = audit.merge(
    frozen[
        [
            "dataset",
            "component_id",

            "in_frozen_proposal_universe",
            "coverage_reason",
        ]
    ],
    on=[
        "dataset",
        "component_id",
    ],
    how="left",
    validate="one_to_one",
)


assert audit[
    "coverage_reason"
].notna().all()


# ============================================================
# 7. EXACT frozen G3E-R reconstruction
# ============================================================

audit[
    "reconstructed_g3er_capture"
] = (
    audit[
        "within_horizon"
    ].astype(bool)
    &
    audit[
        "source_eligible"
    ].astype(bool)
    &
    audit[
        "target_eligible"
    ].astype(bool)
    &
    ~audit[
        "pair_existing_edge"
    ].astype(bool)
    &
    audit[
        "exact_reciprocal_top5"
    ].astype(bool)
    &
    audit[
        "dominance_any"
    ].astype(bool)
)


audit[
    "reconstruction_match"
] = (
    audit[
        "reconstructed_g3er_capture"
    ]
    ==
    audit[
        "in_frozen_proposal_universe"
    ].astype(bool)
)


mismatches = (
    audit[
        ~audit[
            "reconstruction_match"
        ]
    ]
    .copy()
)


print(
    "\n========== G3E-R EXACT-K5 RECONSTRUCTION FIDELITY =========="
)

print(
    "matches:",
    int(
        audit[
            "reconstruction_match"
        ].sum()
    ),
    "/",
    len(
        audit
    )
)

print(
    "mismatches:",
    len(
        mismatches
    )
)


if len(
    mismatches
) > 0:

    print(
        "\nMismatch diagnostics:"
    )


    print(
        mismatches[
            [
                "dataset",
                "component_id",

                "coverage_reason",

                "within_horizon",
                "source_eligible",
                "target_eligible",

                "pair_existing_edge",

                "exact_source_rank_k5",
                "exact_target_rank_k5",
                "exact_proposal_min_k",

                "exact_reciprocal_top5",

                "dominance_any",

                "reconstructed_g3er_capture",
                "in_frozen_proposal_universe",

                "diag_source_rank_k64",
                "diag_target_rank_k64",
                "diag_reciprocal_min_k",
            ]
        ]
        .head(
            50
        )
        .to_string(
            index=False
        )
    )


assert len(
    mismatches
) == 0, (
    "Exact K=5 reconstruction still differs from frozen G3E-R. "
    "Use printed mismatch diagnostics before changing any policy."
)


print(
    "EXACT_G3ER_RECONSTRUCTION: PASS"
)


# ============================================================
# 8. Exact gate classification
# ============================================================

def classify_gate_state(
    row,
):

    if not bool(
        row.within_horizon
    ):

        return (
            "OUTSIDE_HORIZON"
        )


    if not bool(
        row.source_has_past_velocity
    ):

        return (
            "SOURCE_NO_PAST_VELOCITY"
        )


    if not bool(
        row.target_has_future_velocity
    ):

        return (
            "TARGET_NO_FUTURE_VELOCITY"
        )


    if not bool(
        row.source_eligible
    ):

        return (
            "SOURCE_NOT_ELIGIBLE"
        )


    if not bool(
        row.target_eligible
    ):

        return (
            "TARGET_NOT_ELIGIBLE"
        )


    if bool(
        row.pair_existing_edge
    ):

        return (
            "PAIR_ALREADY_EXISTING_EDGE"
        )


    reciprocal = bool(
        row.exact_reciprocal_top5
    )


    dominance = bool(
        row.dominance_any
    )


    if (
        reciprocal
        and
        dominance
    ):

        return (
            "PASS_BOTH"
        )


    if (
        reciprocal
        and
        not dominance
    ):

        return (
            "DOMINANCE_ONLY_MISS"
        )


    if (
        not reciprocal
        and
        dominance
    ):

        return (
            "RECIPROCAL_ONLY_MISS"
        )


    return (
        "RECIPROCAL_AND_DOMINANCE_MISS"
    )


audit[
    "gate_state"
] = audit.apply(
    classify_gate_state,
    axis=1,
)


print(
    "\n========== EXACT GATE MISS DECOMPOSITION =========="
)

print(
    audit[
        "gate_state"
    ]
    .value_counts()
    .to_string()
)


# ============================================================
# 9. Previous 149 mixed misses
# ============================================================

mixed_previous = (
    audit[
        audit[
            "coverage_reason"
        ].eq(
            "RECIPROCAL_TOPK_OR_DOMINANCE_MISS"
        )
    ]
    .copy()
)


assert len(
    mixed_previous
) == 149


print(
    "\n========== PREVIOUS 149 MIXED MISSES =========="
)


print(
    mixed_previous[
        "gate_state"
    ]
    .value_counts()
    .to_string()
)


# ============================================================
# 10. Exact Top-5 versus K64-derived Top-5
#
# Quantify the tie-order issue that broke old G3F0.
# ============================================================

audit[
    "diag_rank_le5"
] = (
    audit[
        "diag_reciprocal_min_k"
    ]
    .le(
        5
    )
    .fillna(
        False
    )
)


audit[
    "k5_vs_k64_top5_agree"
] = (
    audit[
        "exact_reciprocal_top5"
    ]
    ==
    audit[
        "diag_rank_le5"
    ]
)


print(
    "\n========== K=5 vs K=64 TOP-5 FIDELITY =========="
)

print(
    "agree:",
    int(
        audit[
            "k5_vs_k64_top5_agree"
        ].sum()
    ),
    "/",
    len(
        audit
    )
)

print(
    "disagree:",
    int(
        (
            ~audit[
                "k5_vs_k64_top5_agree"
            ]
        ).sum()
    )
)


tie_disagreement = audit[
    ~audit[
        "k5_vs_k64_top5_agree"
    ]
]


if len(
    tie_disagreement
):

    print(
        "\nExamples:"
    )


    print(
        tie_disagreement[
            [
                "dataset",
                "component_id",

                "exact_source_rank_k5",
                "exact_target_rank_k5",

                "diag_source_rank_k64",
                "diag_target_rank_k64",

                "exact_reciprocal_top5",
                "diag_rank_le5",
            ]
        ]
        .head(
            30
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 11. K64 diagnostic retrieval population
# ============================================================

eligible_horizon = (
    audit[
        audit[
            "within_horizon"
        ].astype(bool)
        &
        audit[
            "source_eligible"
        ].astype(bool)
        &
        audit[
            "target_eligible"
        ].astype(bool)
        &
        ~audit[
            "pair_existing_edge"
        ].astype(bool)
    ]
    .copy()
)


print(
    "\n========== DIAGNOSTIC RECIPROCAL RETRIEVAL CEILING =========="
)

print(
    "eligible+horizon:",
    len(
        eligible_horizon
    )
)


# ============================================================
# 12. Retrieval ceiling by K
#
# IMPORTANT:
#
# K=1/2/3/5 below uses K64 diagnostic ranking and therefore is
# approximate under ties.
#
# EXACT K=5 is separately printed from exact query.
#
# K>5 is a diagnostic ceiling, not a proposed generator.
# ============================================================

retrieval_rows = []


# ------------------------------------------------------------
# Exact frozen K=5 first
# ------------------------------------------------------------

exact5_count = int(
    eligible_horizon[
        "exact_reciprocal_top5"
    ].sum()
)


exact5_full7_mask = (
    eligible_horizon[
        "pred_full_7um"
    ].astype(bool)
)


exact5_full7_total = int(
    exact5_full7_mask.sum()
)


exact5_full7_captured = int(
    (
        eligible_horizon[
            "exact_reciprocal_top5"
        ].astype(bool)
        &
        exact5_full7_mask
    ).sum()
)


print(
    "\nEXACT K=5:"
)

print(
    f"eligible={exact5_count}/{len(eligible_horizon)}"
    f" ({exact5_count/len(eligible_horizon):.3%})"
    f" | full<=7={exact5_full7_captured}/{exact5_full7_total}"
    f" ({exact5_full7_captured/exact5_full7_total:.3%})"
)


print(
    "\nK64 diagnostic rank ceilings:"
)


for k in K_AUDIT:

    pass_k = (
        eligible_horizon[
            "diag_reciprocal_min_k"
        ]
        .le(
            k
        )
        .fillna(
            False
        )
    )


    captured_k = int(
        pass_k.sum()
    )


    full7_mask = (
        eligible_horizon[
            "pred_full_7um"
        ].astype(bool)
    )


    full7_total = int(
        full7_mask.sum()
    )


    full7_captured = int(
        (
            pass_k
            &
            full7_mask
        ).sum()
    )


    retrieval_rows.append(
        {
            "k":
                int(
                    k
                ),

            "eligible_total":
                int(
                    len(
                        eligible_horizon
                    )
                ),

            "captured":
                captured_k,

            "coverage":
                float(
                    captured_k
                    /
                    len(
                        eligible_horizon
                    )
                ),

            "full7_total":
                full7_total,

            "full7_captured":
                full7_captured,

            "full7_coverage":
                float(
                    full7_captured
                    /
                    full7_total
                ),
        }
    )


    print(
        f"K={k:2d}"
        f" | eligible={captured_k}/{len(eligible_horizon)}"
        f" ({captured_k/len(eligible_horizon):.3%})"
        f" | full<=7={full7_captured}/{full7_total}"
        f" ({full7_captured/full7_total:.3%})"
    )


retrieval_table = pd.DataFrame(
    retrieval_rows
)


# ============================================================
# 13. Combined diagnostic ceiling with dominance-any
# ============================================================

print(
    "\n========== COMBINED DIAGNOSTIC CEILING WITH CURRENT DOMINANCE =========="
)


combined_rows = []


# Exact K=5.
exact_combined = (
    audit[
        "within_horizon"
    ].astype(bool)
    &
    audit[
        "source_eligible"
    ].astype(bool)
    &
    audit[
        "target_eligible"
    ].astype(bool)
    &
    ~audit[
        "pair_existing_edge"
    ].astype(bool)
    &
    audit[
        "exact_reciprocal_top5"
    ].astype(bool)
    &
    audit[
        "dominance_any"
    ].astype(bool)
)


exact_combined_count = int(
    exact_combined.sum()
)


assert exact_combined_count == 185


print(
    "EXACT K=5:",
    exact_combined_count,
    "/",
    len(
        audit
    ),
    f"({exact_combined_count/len(audit):.3%})"
)


for k in K_AUDIT:

    pass_k = (
        audit[
            "within_horizon"
        ].astype(bool)
        &
        audit[
            "source_eligible"
        ].astype(bool)
        &
        audit[
            "target_eligible"
        ].astype(bool)
        &
        ~audit[
            "pair_existing_edge"
        ].astype(bool)
        &
        audit[
            "diag_reciprocal_min_k"
        ]
        .le(
            k
        )
        .fillna(
            False
        )
        &
        audit[
            "dominance_any"
        ].astype(bool)
    )


    count = int(
        pass_k.sum()
    )


    full7_mask = (
        audit[
            "pred_full_7um"
        ].astype(bool)
    )


    full7_total = int(
        full7_mask.sum()
    )


    full7_captured = int(
        (
            pass_k
            &
            full7_mask
        ).sum()
    )


    combined_rows.append(
        {
            "k":
                int(
                    k
                ),

            "captured":
                count,

            "coverage":
                float(
                    count
                    /
                    len(
                        audit
                    )
                ),

            "full7_total":
                full7_total,

            "full7_captured":
                full7_captured,

            "full7_coverage":
                float(
                    full7_captured
                    /
                    full7_total
                ),
        }
    )


    print(
        f"K={k:2d}"
        f" | all={count}/{len(audit)}"
        f" ({count/len(audit):.3%})"
        f" | full<=7={full7_captured}/{full7_total}"
        f" ({full7_captured/full7_total:.3%})"
    )


combined_table = pd.DataFrame(
    combined_rows
)


# ============================================================
# 14. Diagnostic reciprocal-rank distribution
# ============================================================

print(
    "\n========== TRUE-PAIR DIAGNOSTIC RECIPROCAL RANK =========="
)


resolved_rank = (
    eligible_horizon[
        "diag_reciprocal_min_k"
    ]
    .dropna()
)


print(
    "resolved within K<=64:",
    len(
        resolved_rank
    ),
    "/",
    len(
        eligible_horizon
    )
)


if len(
    resolved_rank
):

    print(
        resolved_rank.describe(
            percentiles=[
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99,
            ]
        ).to_string()
    )


# ============================================================
# 15. Dominance pass rate
# ============================================================

print(
    "\n========== DOMINANCE PASS RATE =========="
)


dominance_any_count = int(
    eligible_horizon[
        "dominance_any"
    ].sum()
)


dominance_all_count = int(
    eligible_horizon[
        "dominance_all"
    ].sum()
)


print(
    "dominance-any:",
    dominance_any_count,
    "/",
    len(
        eligible_horizon
    ),
    f"({dominance_any_count/len(eligible_horizon):.3%})",
)


print(
    "dominance-all:",
    dominance_all_count,
    "/",
    len(
        eligible_horizon
    ),
    f"({dominance_all_count/len(eligible_horizon):.3%})",
)


# ============================================================
# 16. Dominance by topology
# ============================================================

print(
    "\n========== DOMINANCE BY TOPOLOGY =========="
)


dominance_topology = (
    eligible_horizon.groupby(
        "topology_class"
    )
    .agg(
        components=(
            "component_id",
            "size",
        ),

        exact_reciprocal_top5=(
            "exact_reciprocal_top5",
            "sum",
        ),

        dominance_any=(
            "dominance_any",
            "sum",
        ),

        dominance_all=(
            "dominance_all",
            "sum",
        ),
    )
)


dominance_topology[
    "reciprocal_top5_fraction"
] = (
    dominance_topology[
        "exact_reciprocal_top5"
    ]
    /
    dominance_topology[
        "components"
    ]
)


dominance_topology[
    "dominance_any_fraction"
] = (
    dominance_topology[
        "dominance_any"
    ]
    /
    dominance_topology[
        "components"
    ]
)


print(
    dominance_topology.to_string()
)


# ============================================================
# 17. Gate state by topology
# ============================================================

print(
    "\n========== GATE STATE BY TOPOLOGY =========="
)


gate_topology = pd.crosstab(
    audit[
        "topology_class"
    ],
    audit[
        "gate_state"
    ],
)


print(
    gate_topology.to_string()
)


# ============================================================
# 18. Gate state by gap size
# ============================================================

print(
    "\n========== GATE STATE BY GAP SIZE =========="
)


gate_gap = pd.crosstab(
    audit[
        "gap_size"
    ],
    audit[
        "gate_state"
    ],
)


print(
    gate_gap.to_string()
)


# ============================================================
# 19. Gate state by fold
# ============================================================

print(
    "\n========== GATE STATE BY FOLD =========="
)


gate_fold = pd.crosstab(
    audit[
        "fold"
    ],
    audit[
        "gate_state"
    ],
)


print(
    gate_fold.to_string()
)


# ============================================================
# 20. Persist formal corrected audit
# ============================================================

audit = (
    audit
    .sort_values(
        [
            "fold",
            "dataset",
            "component_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


audit.to_pickle(
    OUT
)


summary = {
    "stage":
        "12.11G3F0-R",

    "status":
        "EXACT_K5_RECIPROCAL_VS_DOMINANCE_MISS_DECOMPOSED",

    "repair":
        (
            "Original G3F0 incorrectly inferred K5 membership "
            "from a K64 cKDTree query. Exact K5 and K64 "
            "diagnostic queries are now independent."
        ),

    "clean_components":
        369,

    "frozen_g3er_captured":
        185,

    "previous_mixed_misses":
        149,

    "exact_reconstruction_mismatches":
        int(
            len(
                mismatches
            )
        ),

    "k5_vs_k64_top5_disagreements":
        int(
            (
                ~audit[
                    "k5_vs_k64_top5_agree"
                ]
            ).sum()
        ),

    "gate_state_counts": {
        str(k):
            int(v)

        for (
            k,
            v
        )
        in audit[
            "gate_state"
        ]
        .value_counts()
        .items()
    },

    "diagnostic_retrieval_by_k":
        retrieval_table.to_dict(
            "records"
        ),

    "diagnostic_combined_by_k":
        combined_table.to_dict(
            "records"
        ),

    "exact_g3er_reconstruction":
        True,

    "new_proposal_universe_generated":
        False,

    "gt_used":
        True,

    "gt_role":
        "DEVELOPMENT_ORACLE_DIAGNOSTIC_ONLY",

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "synthetic_nodes_inserted":
        False,

    "graph_modified":
        False,
}


SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert OUT.exists()
assert SUMMARY_OUT.exists()


# ============================================================
# 21. Reload integrity
# ============================================================

check = pd.read_pickle(
    OUT
)


assert len(
    check
) == 369


assert check[
    "reconstruction_match"
].all()


assert int(
    check[
        "reconstructed_g3er_capture"
    ].sum()
) == 185


# ============================================================
# 22. Decision
# ============================================================

print(
    "\n========== 12.11G3F0-R DECISION =========="
)

print(
    "EXACT_K5_RECIPROCAL_VS_DOMINANCE_MISS_DECOMPOSITION: PASS"
)

print(
    "clean components:",
    len(
        audit
    )
)

print(
    "exact frozen G3E-R reconstruction:",
    "YES"
)

print(
    "reconstruction mismatches:",
    len(
        mismatches
    )
)

print(
    "K5 and K64 queries separated:",
    "YES"
)

print(
    "new proposal universe generated:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "classifier trained:",
    "NO"
)

print(
    "threshold selected:",
    "NO"
)

print(
    "policy selected:",
    "NO"
)

print(
    "graph modified:",
    "NO"
)

print(
    "next:"
)

print(
    "DECIDE_RETRIEVAL_VS_DOMINANCE_REDESIGN_FROM_EXACT_GATE_DECOMPOSITION"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3F1
#
# GT-Blind Asymmetric Retrieval Contract Cardinality Audit
# + Development Oracle Coverage Audit
#
# Motivation:
#
#   G3F0-R exact decomposition:
#
#       RECIPROCAL_ONLY_MISS             101
#       RECIPROCAL_AND_DOMINANCE_MISS     28
#       DOMINANCE_ONLY_MISS               20
#
#   Therefore retrieval is the primary bottleneck.
#
# BUT:
#
#   G3E-R K=5 stored 23,012,742 proposals.
#   Naively increasing full reciprocal K would be too expensive.
#
# We therefore test asymmetric retrieval contracts:
#
#   ASYM(a,b):
#
#       (source_rank <= a AND target_rank <= b)
#         OR
#       (target_rank <= a AND source_rank <= b)
#
# This requires one side to be a strong local anchor while
# allowing the opposite trajectory-consistency rank more slack.
#
# PART A:
#   - 103 development datasets
#   - GT blind
#   - count candidate cardinality only
#   - DO NOT persist candidate rows
#   - freeze contract-count artifact before GT read
#
# PART B:
#   - only after Part A freeze
#   - development oracle rank audit
#   - exact K16 query for K<=16 family
#   - BASE_EXACT_5x5 uses retained exact-K5 G3F0-R result
#   - combine with existing dominance-any only diagnostically
#
# No new proposal universe.
# No Fold0.
# No classifier.
# No final threshold.
# No policy selection.
# No graph modification.
#
# Keep after running: YES
# ============================================================

from pathlib import Path
import gc
import hashlib
import json

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


G3F0R_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_gate_miss_decomposition_exactk5.pkl"
)


CARDINALITY_OUT = (
    S12
    / "stage12_folds14_multiframe_asymmetric_retrieval_cardinality_v1.json"
)


ORACLE_OUT = (
    S12
    / "stage12_folds14_multiframe_asymmetric_retrieval_oracle_v1.pkl"
)


ORACLE_SUMMARY_OUT = (
    S12
    / "stage12_folds14_multiframe_asymmetric_retrieval_oracle_v1_summary.json"
)


assert G3F0R_PATH.exists(), G3F0R_PATH


# ============================================================
# 1. Existing exact CURRENT_BEST contract
# ============================================================

assert (
    "build_current_best"
    in globals()
    and
    callable(
        globals()[
            "build_current_best"
        ]
    )
), (
    "build_current_best missing. "
    "Re-run retained successful G3E-R first."
)


current_best_builder = globals()[
    "build_current_best"
]


# ------------------------------------------------------------
# Development corpus
# ------------------------------------------------------------

if (
    "names"
    in globals()
    and
    len(
        globals()[
            "names"
        ]
    ) == 103
):

    development_names = list(
        globals()[
            "names"
        ]
    )

else:

    CV_PATH = (
        PROJECT
        / "configs"
        / "cv_manifest.csv"
    )


    cv = pd.read_csv(
        CV_PATH
    )


    development_names = (
        cv[
            cv[
                "fold"
            ].isin(
                [
                    1,
                    2,
                    3,
                    4,
                ]
            )
            &
            cv[
                "dataset"
            ].str.startswith(
                "6bba_"
            )
        ]
        .sort_values(
            [
                "fold",
                "dataset",
            ],
            kind="stable",
        )[
            "dataset"
        ]
        .tolist()
    )


assert len(
    development_names
) == 103


# ------------------------------------------------------------
# Fold map
# ------------------------------------------------------------

if (
    "fold_map"
    in globals()
):

    development_fold_map = dict(
        globals()[
            "fold_map"
        ]
    )

else:

    CV_PATH = (
        PROJECT
        / "configs"
        / "cv_manifest.csv"
    )


    cv = pd.read_csv(
        CV_PATH
    )


    dev = cv[
        cv[
            "dataset"
        ].isin(
            development_names
        )
    ]


    development_fold_map = dict(
        zip(
            dev[
                "dataset"
            ],
            dev[
                "fold"
            ].astype(int),
        )
    )


SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


MIN_GAP_SIZE = 2
MAX_GAP_SIZE = 12

EXACT_BASE_K = 5
MAX_RANK_K = 16


# ============================================================
# 2. Frozen retrieval-contract family
#
# IMPORTANT:
# Defined BEFORE development oracle is read.
# ============================================================

CONTRACTS = {
    "BASE_EXACT_5x5": {
        "type":
            "BASE_EXACT",

        "a":
            5,

        "b":
            5,
    },

    "ASYM_3x8": {
        "type":
            "ASYM",

        "a":
            3,

        "b":
            8,
    },

    "ASYM_5x8": {
        "type":
            "ASYM",

        "a":
            5,

        "b":
            8,
    },

    "ASYM_3x12": {
        "type":
            "ASYM",

        "a":
            3,

        "b":
            12,
    },

    "ASYM_5x12": {
        "type":
            "ASYM",

        "a":
            5,

        "b":
            12,
    },

    "ASYM_3x16": {
        "type":
            "ASYM",

        "a":
            3,

        "b":
            16,
    },

    "ASYM_5x16": {
        "type":
            "ASYM",

        "a":
            5,

        "b":
            16,
    },

    "RECIP_8x8": {
        "type":
            "RECIP",

        "a":
            8,

        "b":
            8,
    },

    "RECIP_12x12": {
        "type":
            "RECIP",

        "a":
            12,

        "b":
            12,
    },

    "RECIP_16x16": {
        "type":
            "RECIP",

        "a":
            16,

        "b":
            16,
    },
}


print(
    "========== 12.11G3F1 CONTRACT FAMILY =========="
)

print(
    "development datasets:",
    len(
        development_names
    )
)

print(
    "gap horizon:",
    MIN_GAP_SIZE,
    "to",
    MAX_GAP_SIZE
)

print(
    "max exact rank query:",
    MAX_RANK_K
)

print(
    "contracts:"
)


for name, spec in CONTRACTS.items():

    print(
        name,
        spec
    )


# ============================================================
# 3. Helpers
# ============================================================

def build_rank_map(
    query_ids,
    candidate_ids,
    indices,
):

    result = {}


    for query_i, query_id in enumerate(
        query_ids
    ):

        for rank_i in range(
            indices.shape[
                1
            ]
        ):

            candidate_id = int(
                candidate_ids[
                    int(
                        indices[
                            query_i,
                            rank_i
                        ]
                    )
                ]
            )


            pair = (
                int(
                    query_id
                ),
                candidate_id,
            )


            if pair not in result:

                result[
                    pair
                ] = int(
                    rank_i
                    +
                    1
                )


    return result


def build_reverse_rank_map(
    query_ids,
    candidate_ids,
    indices,
):

    result = {}


    for query_i, query_id in enumerate(
        query_ids
    ):

        for rank_i in range(
            indices.shape[
                1
            ]
        ):

            candidate_id = int(
                candidate_ids[
                    int(
                        indices[
                            query_i,
                            rank_i
                        ]
                    )
                ]
            )


            # candidate is source;
            # query is target.
            pair = (
                candidate_id,
                int(
                    query_id
                ),
            )


            if pair not in result:

                result[
                    pair
                ] = int(
                    rank_i
                    +
                    1
                )


    return result


def query_tree(
    tree,
    query_points,
    candidate_count,
    k,
):

    k_eff = min(
        int(
            k
        ),
        int(
            candidate_count
        ),
    )


    distances, indices = tree.query(
        query_points,
        k=k_eff,
    )


    if k_eff == 1:

        distances = distances[
            :,
            None
        ]

        indices = indices[
            :,
            None
        ]


    return (
        np.asarray(
            distances,
            dtype=float,
        ),

        np.asarray(
            indices,
            dtype=int,
        ),
    )


def contract_pass(
    contract_name,
    source_rank,
    target_rank,
):

    spec = CONTRACTS[
        contract_name
    ]


    contract_type = spec[
        "type"
    ]


    a = int(
        spec[
            "a"
        ]
    )


    b = int(
        spec[
            "b"
        ]
    )


    if (
        not np.isfinite(
            source_rank
        )
        or
        not np.isfinite(
            target_rank
        )
    ):

        return False


    source_rank = int(
        source_rank
    )

    target_rank = int(
        target_rank
    )


    if contract_type == "RECIP":

        return bool(
            source_rank <= a
            and
            target_rank <= b
        )


    if contract_type == "ASYM":

        return bool(
            (
                source_rank <= a
                and
                target_rank <= b
            )
            or
            (
                target_rank <= a
                and
                source_rank <= b
            )
        )


    raise ValueError(
        (
            contract_name,
            spec,
        )
    )


# ============================================================
# PART A
# GT-BLIND CARDINALITY SCAN
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART A — GT-BLIND RETRIEVAL CARDINALITY"
)

print(
    "============================================================"
)


# ------------------------------------------------------------
# Global count structure
# ------------------------------------------------------------

counts = {}


for contract_name in CONTRACTS:

    counts[
        contract_name
    ] = {
        "rows":
            0,

        "cuts_0":
            0,

        "cuts_1":
            0,

        "cuts_2":
            0,
    }


dataset_counts = []


# ============================================================
# 4. Scan 103 datasets
# ============================================================

for ds_i, dataset in enumerate(
    development_names,
    start=1,
):

    fold = int(
        development_fold_map[
            dataset
        ]
    )


    graph, scale = current_best_builder(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    )


    # ========================================================
    # Nodes
    # ========================================================

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = (
        nodes[
            "node_id"
        ]
        .astype(int)
    )


    nodes[
        "t"
    ] = (
        nodes[
            "t"
        ]
        .astype(int)
    )


    nodes = (
        nodes
        .sort_values(
            [
                "t",
                "node_id",
            ],
            kind="stable",
        )
        .set_index(
            "node_id",
            drop=False,
        )
    )


    # ========================================================
    # Edges / topology
    # ========================================================

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = (
        edges[
            "source_id"
        ]
        .astype(int)
    )


    edges[
        "target_id"
    ] = (
        edges[
            "target_id"
        ]
        .astype(int)
    )


    edge_set = set(
        zip(
            edges[
                "source_id"
            ],
            edges[
                "target_id"
            ],
        )
    )


    incoming = (
        edges.groupby(
            "target_id"
        )[
            "source_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    outgoing = (
        edges.groupby(
            "source_id"
        )[
            "target_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = {
        int(k):
            int(v)

        for k, v
        in indeg.items()
    }


    outdeg = {
        int(k):
            int(v)

        for k, v
        in outdeg.items()
    }


    # ========================================================
    # Physical state
    # ========================================================

    phys = {}

    time_map = {}


    for nid, row in nodes.iterrows():

        nid = int(
            nid
        )


        phys[
            nid
        ] = (
            np.array(
                [
                    float(
                        row[
                            "z"
                        ]
                    ),

                    float(
                        row[
                            "y"
                        ]
                    ),

                    float(
                        row[
                            "x"
                        ]
                    ),
                ],
                dtype=float,
            )
            *
            SCALE
        )


        time_map[
            nid
        ] = int(
            row[
                "t"
            ]
        )


    node_ids = sorted(
        phys
    )


    # ========================================================
    # Local velocities
    # ========================================================

    past_velocity = {}

    future_velocity = {}


    for nid in node_ids:

        inc = incoming.get(
            nid,
            [],
        )


        if len(
            inc
        ) == 1:

            prev_id = int(
                inc[
                    0
                ]
            )


            if prev_id in phys:

                dt = (
                    time_map[
                        nid
                    ]
                    -
                    time_map[
                        prev_id
                    ]
                )


                if dt > 0:

                    past_velocity[
                        nid
                    ] = (
                        phys[
                            nid
                        ]
                        -
                        phys[
                            prev_id
                        ]
                    ) / float(
                        dt
                    )


        out = outgoing.get(
            nid,
            [],
        )


        if len(
            out
        ) == 1:

            next_id = int(
                out[
                    0
                ]
            )


            if next_id in phys:

                dt = (
                    time_map[
                        next_id
                    ]
                    -
                    time_map[
                        nid
                    ]
                )


                if dt > 0:

                    future_velocity[
                        nid
                    ] = (
                        phys[
                            next_id
                        ]
                        -
                        phys[
                            nid
                        ]
                    ) / float(
                        dt
                    )


    # ========================================================
    # Eligibility
    # ========================================================

    source_eligible = [
        nid

        for nid in node_ids

        if (
            outdeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid
            in
            past_velocity
        )
    ]


    target_eligible = [
        nid

        for nid in node_ids

        if (
            indeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid
            in
            future_velocity
        )
    ]


    source_by_t = {}

    target_by_t = {}


    for nid in source_eligible:

        source_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for nid in target_eligible:

        target_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for t in source_by_t:

        source_by_t[
            t
        ] = sorted(
            source_by_t[
                t
            ]
        )


    for t in target_by_t:

        target_by_t[
            t
        ] = sorted(
            target_by_t[
                t
            ]
        )


    # ========================================================
    # Per-dataset counters
    # ========================================================

    local_counts = {}


    for contract_name in CONTRACTS:

        local_counts[
            contract_name
        ] = {
            "rows":
                0,

            "cuts_0":
                0,

            "cuts_1":
                0,

            "cuts_2":
                0,
        }


    # ========================================================
    # Frame-pair retrieval
    # ========================================================

    for gap_size in range(
        MIN_GAP_SIZE,
        MAX_GAP_SIZE + 1,
    ):

        delta_t = (
            gap_size
            +
            1
        )


        for source_t in sorted(
            source_by_t
        ):

            target_t = (
                source_t
                +
                delta_t
            )


            if target_t not in target_by_t:

                continue


            source_ids = source_by_t[
                source_t
            ]


            target_ids = target_by_t[
                target_t
            ]


            if (
                len(
                    source_ids
                ) == 0
                or
                len(
                    target_ids
                ) == 0
            ):

                continue


            source_pos = np.vstack(
                [
                    phys[
                        nid
                    ]

                    for nid in source_ids
                ]
            )


            target_pos = np.vstack(
                [
                    phys[
                        nid
                    ]

                    for nid in target_ids
                ]
            )


            source_vel = np.vstack(
                [
                    past_velocity[
                        nid
                    ]

                    for nid in source_ids
                ]
            )


            target_vel = np.vstack(
                [
                    future_velocity[
                        nid
                    ]

                    for nid in target_ids
                ]
            )


            source_predicted_target = (
                source_pos
                +
                float(
                    delta_t
                )
                *
                source_vel
            )


            target_predicted_source = (
                target_pos
                -
                float(
                    delta_t
                )
                *
                target_vel
            )


            target_tree = cKDTree(
                target_pos
            )


            source_tree = cKDTree(
                source_pos
            )


            # =================================================
            # Exact original K=5 query
            # =================================================

            (
                _,
                src_idx_k5,
            ) = query_tree(
                target_tree,
                source_predicted_target,
                len(
                    target_ids
                ),
                EXACT_BASE_K,
            )


            (
                _,
                tgt_idx_k5,
            ) = query_tree(
                source_tree,
                target_predicted_source,
                len(
                    source_ids
                ),
                EXACT_BASE_K,
            )


            src_map_k5 = build_rank_map(
                source_ids,
                target_ids,
                src_idx_k5,
            )


            tgt_map_k5 = build_reverse_rank_map(
                target_ids,
                source_ids,
                tgt_idx_k5,
            )


            base_pairs = (
                set(
                    src_map_k5
                )
                &
                set(
                    tgt_map_k5
                )
            )


            for (
                src,
                tgt,
            ) in base_pairs:

                if (
                    src,
                    tgt,
                ) in edge_set:

                    continue


                cuts = (
                    int(
                        outdeg.get(
                            src,
                            0,
                        )
                        == 1
                    )
                    +
                    int(
                        indeg.get(
                            tgt,
                            0,
                        )
                        == 1
                    )
                )


                local_counts[
                    "BASE_EXACT_5x5"
                ][
                    "rows"
                ] += 1


                local_counts[
                    "BASE_EXACT_5x5"
                ][
                    f"cuts_{cuts}"
                ] += 1


            # =================================================
            # Exact K=16 family query
            # =================================================

            (
                _,
                src_idx_k16,
            ) = query_tree(
                target_tree,
                source_predicted_target,
                len(
                    target_ids
                ),
                MAX_RANK_K,
            )


            (
                _,
                tgt_idx_k16,
            ) = query_tree(
                source_tree,
                target_predicted_source,
                len(
                    source_ids
                ),
                MAX_RANK_K,
            )


            src_map_k16 = build_rank_map(
                source_ids,
                target_ids,
                src_idx_k16,
            )


            tgt_map_k16 = build_reverse_rank_map(
                target_ids,
                source_ids,
                tgt_idx_k16,
            )


            mutual16_pairs = (
                set(
                    src_map_k16
                )
                &
                set(
                    tgt_map_k16
                )
            )


            for (
                src,
                tgt,
            ) in mutual16_pairs:

                if (
                    src,
                    tgt,
                ) in edge_set:

                    continue


                source_rank = src_map_k16[
                    (
                        src,
                        tgt,
                    )
                ]


                target_rank = tgt_map_k16[
                    (
                        src,
                        tgt,
                    )
                ]


                cuts = (
                    int(
                        outdeg.get(
                            src,
                            0,
                        )
                        == 1
                    )
                    +
                    int(
                        indeg.get(
                            tgt,
                            0,
                        )
                        == 1
                    )
                )


                for contract_name in CONTRACTS:

                    if contract_name == "BASE_EXACT_5x5":

                        continue


                    if contract_pass(
                        contract_name,
                        source_rank,
                        target_rank,
                    ):

                        local_counts[
                            contract_name
                        ][
                            "rows"
                        ] += 1


                        local_counts[
                            contract_name
                        ][
                            f"cuts_{cuts}"
                        ] += 1


            # =================================================
            # Frame-pair cleanup
            # =================================================

            del source_pos
            del target_pos

            del source_vel
            del target_vel

            del source_predicted_target
            del target_predicted_source

            del target_tree
            del source_tree

            del src_idx_k5
            del tgt_idx_k5

            del src_map_k5
            del tgt_map_k5
            del base_pairs

            del src_idx_k16
            del tgt_idx_k16

            del src_map_k16
            del tgt_map_k16
            del mutual16_pairs


    # ========================================================
    # Aggregate dataset counts
    # ========================================================

    dataset_record = {
        "dataset":
            dataset,

        "fold":
            fold,
    }


    for contract_name in CONTRACTS:

        local = local_counts[
            contract_name
        ]


        assert (
            local[
                "cuts_0"
            ]
            +
            local[
                "cuts_1"
            ]
            +
            local[
                "cuts_2"
            ]
        ) == local[
            "rows"
        ]


        for key in [
            "rows",
            "cuts_0",
            "cuts_1",
            "cuts_2",
        ]:

            counts[
                contract_name
            ][
                key
            ] += int(
                local[
                    key
                ]
            )


        dataset_record[
            contract_name
        ] = {
            key:
                int(
                    local[
                        key
                    ]
                )

            for key in [
                "rows",
                "cuts_0",
                "cuts_1",
                "cuts_2",
            ]
        }


    dataset_counts.append(
        dataset_record
    )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            development_names
        )
    ):

        print(
            f"{ds_i:3d}/{len(development_names)}"
            f" | fold={fold}"
            f" | {dataset}"
            f" | base5={local_counts['BASE_EXACT_5x5']['rows']}"
            f" | a5x16={local_counts['ASYM_5x16']['rows']}"
            f" | r16={local_counts['RECIP_16x16']['rows']}"
        )


    # ========================================================
    # Dataset cleanup
    # ========================================================

    del graph
    del nodes
    del edges

    del edge_set

    del incoming
    del outgoing

    del indeg
    del outdeg

    del phys
    del time_map
    del node_ids

    del past_velocity
    del future_velocity

    del source_eligible
    del target_eligible

    del source_by_t
    del target_by_t

    del local_counts

    gc.collect()


# ============================================================
# 5. GT-blind cardinality table
# ============================================================

cardinality_rows = []


base_raw_rows = int(
    counts[
        "BASE_EXACT_5x5"
    ][
        "rows"
    ]
)


assert base_raw_rows > 0


for contract_name, spec in CONTRACTS.items():

    row = {
        "contract":
            contract_name,

        "type":
            spec[
                "type"
            ],

        "a":
            int(
                spec[
                    "a"
                ]
            ),

        "b":
            int(
                spec[
                    "b"
                ]
            ),

        "rows":
            int(
                counts[
                    contract_name
                ][
                    "rows"
                ]
            ),

        "cuts_0":
            int(
                counts[
                    contract_name
                ][
                    "cuts_0"
                ]
            ),

        "cuts_1":
            int(
                counts[
                    contract_name
                ][
                    "cuts_1"
                ]
            ),

        "cuts_2":
            int(
                counts[
                    contract_name
                ][
                    "cuts_2"
                ]
            ),
    }


    row[
        "raw_cardinality_ratio_vs_exact5"
    ] = float(
        row[
            "rows"
        ]
        /
        base_raw_rows
    )


    cardinality_rows.append(
        row
    )


cardinality_table = pd.DataFrame(
    cardinality_rows
)


cardinality_table = (
    cardinality_table
    .sort_values(
        [
            "rows",
            "contract",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n========== GT-BLIND RETRIEVAL CARDINALITY =========="
)


print(
    cardinality_table.to_string(
        index=False
    )
)


# ============================================================
# 6. Freeze GT-blind contract/cardinality artifact
#
# IMPORTANT:
# G3F0-R oracle has NOT been read in this cell yet.
# ============================================================

cardinality_payload = {
    "stage":
        "12.11G3F1",

    "part":
        "GT_BLIND_RETRIEVAL_CONTRACT_CARDINALITY",

    "development_datasets":
        103,

    "min_gap_size":
        MIN_GAP_SIZE,

    "max_gap_size":
        MAX_GAP_SIZE,

    "exact_base_k":
        EXACT_BASE_K,

    "max_rank_query":
        MAX_RANK_K,

    "contracts":
        CONTRACTS,

    "counts":
        {
            row[
                "contract"
            ]:
                {
                    "rows":
                        int(
                            row[
                                "rows"
                            ]
                        ),

                    "cuts_0":
                        int(
                            row[
                                "cuts_0"
                            ]
                        ),

                    "cuts_1":
                        int(
                            row[
                                "cuts_1"
                            ]
                        ),

                    "cuts_2":
                        int(
                            row[
                                "cuts_2"
                            ]
                        ),

                    "raw_cardinality_ratio_vs_exact5":
                        float(
                            row[
                                "raw_cardinality_ratio_vs_exact5"
                            ]
                        ),
                }

            for row
            in cardinality_rows
        },

    "dataset_counts":
        dataset_counts,

    "candidate_rows_persisted":
        False,

    "gt_used":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


canonical_contract_bytes = json.dumps(
    cardinality_payload,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
).encode(
    "utf-8"
)


contract_sha = hashlib.sha256(
    canonical_contract_bytes
).hexdigest()


cardinality_payload[
    "contract_cardinality_sha256"
] = contract_sha


CARDINALITY_OUT.write_text(
    json.dumps(
        cardinality_payload,
        indent=2,
    ),
    encoding="utf-8",
)


assert CARDINALITY_OUT.exists()


print(
    "\nGT-BLIND RETRIEVAL CONTRACT FAMILY FROZEN BEFORE ORACLE READ: YES"
)

print(
    "contract/cardinality SHA256:",
    contract_sha
)


# ============================================================
# PART B
#
# DEVELOPMENT ORACLE READ IS ALLOWED ONLY BELOW THIS LINE.
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — DEVELOPMENT ORACLE CONTRACT COVERAGE"
)

print(
    "============================================================"
)


# ============================================================
# 7. Load retained exact G3F0-R oracle
# ============================================================

g3f0 = pd.read_pickle(
    G3F0R_PATH
)


assert len(
    g3f0
) == 369


assert g3f0[
    "reconstruction_match"
].all()


assert int(
    g3f0[
        "reconstructed_g3er_capture"
    ].sum()
) == 185


oracle = (
    g3f0
    .copy()
)


oracle_datasets = (
    oracle[
        "dataset"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)


assert len(
    oracle_datasets
) == 75


# ============================================================
# 8. Exact K16 rank fields for oracle true pairs
#
# We cannot infer exact K16 ranks from K64 because of tie order.
# So rerun exact K16 query only on oracle datasets.
# ============================================================

oracle_rank_rows = []


for ds_i, dataset in enumerate(
    oracle_datasets,
    start=1,
):

    graph, scale = current_best_builder(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    )


    # ========================================================
    # Nodes
    # ========================================================

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = (
        nodes[
            "node_id"
        ]
        .astype(int)
    )


    nodes[
        "t"
    ] = (
        nodes[
            "t"
        ]
        .astype(int)
    )


    nodes = (
        nodes
        .sort_values(
            [
                "t",
                "node_id",
            ],
            kind="stable",
        )
        .set_index(
            "node_id",
            drop=False,
        )
    )


    # ========================================================
    # Edges
    # ========================================================

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = (
        edges[
            "source_id"
        ]
        .astype(int)
    )


    edges[
        "target_id"
    ] = (
        edges[
            "target_id"
        ]
        .astype(int)
    )


    incoming = (
        edges.groupby(
            "target_id"
        )[
            "source_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    outgoing = (
        edges.groupby(
            "source_id"
        )[
            "target_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = {
        int(k):
            int(v)

        for k, v
        in indeg.items()
    }


    outdeg = {
        int(k):
            int(v)

        for k, v
        in outdeg.items()
    }


    # ========================================================
    # Physical state
    # ========================================================

    phys = {}

    time_map = {}


    for nid, row in nodes.iterrows():

        nid = int(
            nid
        )


        phys[
            nid
        ] = (
            np.array(
                [
                    float(
                        row[
                            "z"
                        ]
                    ),

                    float(
                        row[
                            "y"
                        ]
                    ),

                    float(
                        row[
                            "x"
                        ]
                    ),
                ],
                dtype=float,
            )
            *
            SCALE
        )


        time_map[
            nid
        ] = int(
            row[
                "t"
            ]
        )


    node_ids = sorted(
        phys
    )


    # ========================================================
    # Velocities
    # ========================================================

    past_velocity = {}

    future_velocity = {}


    for nid in node_ids:

        inc = incoming.get(
            nid,
            [],
        )


        if len(
            inc
        ) == 1:

            prev_id = int(
                inc[
                    0
                ]
            )


            if prev_id in phys:

                dt = (
                    time_map[
                        nid
                    ]
                    -
                    time_map[
                        prev_id
                    ]
                )


                if dt > 0:

                    past_velocity[
                        nid
                    ] = (
                        phys[
                            nid
                        ]
                        -
                        phys[
                            prev_id
                        ]
                    ) / float(
                        dt
                    )


        out = outgoing.get(
            nid,
            [],
        )


        if len(
            out
        ) == 1:

            next_id = int(
                out[
                    0
                ]
            )


            if next_id in phys:

                dt = (
                    time_map[
                        next_id
                    ]
                    -
                    time_map[
                        nid
                    ]
                )


                if dt > 0:

                    future_velocity[
                        nid
                    ] = (
                        phys[
                            next_id
                        ]
                        -
                        phys[
                            nid
                        ]
                    ) / float(
                        dt
                    )


    # ========================================================
    # Same eligibility
    # ========================================================

    source_eligible = [
        nid

        for nid in node_ids

        if (
            outdeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid
            in
            past_velocity
        )
    ]


    target_eligible = [
        nid

        for nid in node_ids

        if (
            indeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid
            in
            future_velocity
        )
    ]


    source_eligible_set = set(
        source_eligible
    )


    target_eligible_set = set(
        target_eligible
    )


    source_by_t = {}

    target_by_t = {}


    for nid in source_eligible:

        source_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for nid in target_eligible:

        target_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for t in source_by_t:

        source_by_t[
            t
        ] = sorted(
            source_by_t[
                t
            ]
        )


    for t in target_by_t:

        target_by_t[
            t
        ] = sorted(
            target_by_t[
                t
            ]
        )


    # ========================================================
    # Oracle pairs for this dataset
    # ========================================================

    ds_oracle = oracle[
        oracle[
            "dataset"
        ].eq(
            dataset
        )
    ]


    local = {}


    frame_groups = {}


    for row in ds_oracle.itertuples(
        index=False
    ):

        key = (
            dataset,
            row.component_id,
        )


        src = int(
            row.source_id
        )


        tgt = int(
            row.target_id
        )


        local[
            key
        ] = {
            "dataset":
                dataset,

            "component_id":
                row.component_id,

            "exact_source_rank_k16":
                np.nan,

            "exact_target_rank_k16":
                np.nan,
        }


        if not (
            bool(
                row.within_horizon
            )
            and
            bool(
                row.source_eligible
            )
            and
            bool(
                row.target_eligible
            )
            and
            not bool(
                row.pair_existing_edge
            )
        ):

            continue


        # Cross-check fresh graph state.
        assert src in source_eligible_set
        assert tgt in target_eligible_set


        source_t = int(
            time_map[
                src
            ]
        )


        target_t = int(
            time_map[
                tgt
            ]
        )


        frame_groups.setdefault(
            (
                source_t,
                target_t,
            ),
            [],
        ).append(
            key
        )


    # ========================================================
    # Exact K16 query
    # ========================================================

    for (
        source_t,
        target_t,
    ), keys in frame_groups.items():

        source_ids = source_by_t[
            source_t
        ]


        target_ids = target_by_t[
            target_t
        ]


        delta_t = (
            target_t
            -
            source_t
        )


        source_pos = np.vstack(
            [
                phys[
                    nid
                ]

                for nid in source_ids
            ]
        )


        target_pos = np.vstack(
            [
                phys[
                    nid
                ]

                for nid in target_ids
            ]
        )


        source_vel = np.vstack(
            [
                past_velocity[
                    nid
                ]

                for nid in source_ids
            ]
        )


        target_vel = np.vstack(
            [
                future_velocity[
                    nid
                ]

                for nid in target_ids
            ]
        )


        source_predicted_target = (
            source_pos
            +
            float(
                delta_t
            )
            *
            source_vel
        )


        target_predicted_source = (
            target_pos
            -
            float(
                delta_t
            )
            *
            target_vel
        )


        target_tree = cKDTree(
            target_pos
        )


        source_tree = cKDTree(
            source_pos
        )


        (
            _,
            source_idx,
        ) = query_tree(
            target_tree,
            source_predicted_target,
            len(
                target_ids
            ),
            MAX_RANK_K,
        )


        (
            _,
            target_idx,
        ) = query_tree(
            source_tree,
            target_predicted_source,
            len(
                source_ids
            ),
            MAX_RANK_K,
        )


        source_position = {
            int(
                nid
            ):
                i

            for i, nid
            in enumerate(
                source_ids
            )
        }


        target_position = {
            int(
                nid
            ):
                i

            for i, nid
            in enumerate(
                target_ids
            )
        }


        for key in keys:

            row = local[
                key
            ]


            oracle_row = ds_oracle[
                ds_oracle[
                    "component_id"
                ].eq(
                    key[
                        1
                    ]
                )
            ].iloc[
                0
            ]


            src = int(
                oracle_row[
                    "source_id"
                ]
            )


            tgt = int(
                oracle_row[
                    "target_id"
                ]
            )


            src_i = source_position[
                src
            ]


            tgt_i = target_position[
                tgt
            ]


            source_rank = np.nan


            for rank_i, candidate_idx in enumerate(
                source_idx[
                    src_i
                ]
            ):

                candidate_id = int(
                    target_ids[
                        int(
                            candidate_idx
                        )
                    ]
                )


                if candidate_id == tgt:

                    source_rank = int(
                        rank_i
                        +
                        1
                    )

                    break


            target_rank = np.nan


            for rank_i, candidate_idx in enumerate(
                target_idx[
                    tgt_i
                ]
            ):

                candidate_id = int(
                    source_ids[
                        int(
                            candidate_idx
                        )
                    ]
                )


                if candidate_id == src:

                    target_rank = int(
                        rank_i
                        +
                        1
                    )

                    break


            row[
                "exact_source_rank_k16"
            ] = source_rank


            row[
                "exact_target_rank_k16"
            ] = target_rank


        del source_pos
        del target_pos

        del source_vel
        del target_vel

        del source_predicted_target
        del target_predicted_source

        del target_tree
        del source_tree

        del source_idx
        del target_idx


    oracle_rank_rows.extend(
        local.values()
    )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            oracle_datasets
        )
    ):

        print(
            f"{ds_i:2d}/{len(oracle_datasets)}"
            f" | {dataset}"
            f" | oracle={len(ds_oracle)}"
        )


    del graph
    del nodes
    del edges

    del incoming
    del outgoing

    del indeg
    del outdeg

    del phys
    del time_map

    del node_ids

    del past_velocity
    del future_velocity

    del source_eligible
    del target_eligible

    del source_eligible_set
    del target_eligible_set

    del source_by_t
    del target_by_t

    del local
    del frame_groups

    gc.collect()


# ============================================================
# 9. Attach exact K16 ranks
# ============================================================

rank_table = pd.DataFrame(
    oracle_rank_rows
)


assert len(
    rank_table
) == 369


oracle = oracle.merge(
    rank_table,
    on=[
        "dataset",
        "component_id",
    ],
    how="left",
    validate="one_to_one",
)


# ============================================================
# 10. Contract oracle masks
# ============================================================

print(
    "\n========== ORACLE RETRIEVAL CONTRACT COVERAGE =========="
)


oracle_rows = []


processable = (
    oracle[
        "within_horizon"
    ].astype(bool)
    &
    oracle[
        "source_eligible"
    ].astype(bool)
    &
    oracle[
        "target_eligible"
    ].astype(bool)
    &
    ~oracle[
        "pair_existing_edge"
    ].astype(bool)
)


processable_total = int(
    processable.sum()
)


processable_full7 = (
    processable
    &
    oracle[
        "pred_full_7um"
    ].astype(bool)
)


processable_full7_total = int(
    processable_full7.sum()
)


for contract_name in CONTRACTS:

    if contract_name == "BASE_EXACT_5x5":

        retrieval_pass = (
            processable
            &
            oracle[
                "exact_reciprocal_top5"
            ].astype(bool)
        )

    else:

        spec = CONTRACTS[
            contract_name
        ]


        contract_type = spec[
            "type"
        ]


        a = int(
            spec[
                "a"
            ]
        )


        b = int(
            spec[
                "b"
            ]
        )


        source_rank = oracle[
            "exact_source_rank_k16"
        ]


        target_rank = oracle[
            "exact_target_rank_k16"
        ]


        if contract_type == "RECIP":

            rank_pass = (
                source_rank.le(
                    a
                )
                &
                target_rank.le(
                    b
                )
            )


        elif contract_type == "ASYM":

            rank_pass = (
                (
                    source_rank.le(
                        a
                    )
                    &
                    target_rank.le(
                        b
                    )
                )
                |
                (
                    target_rank.le(
                        a
                    )
                    &
                    source_rank.le(
                        b
                    )
                )
            )


        else:

            raise ValueError(
                spec
            )


        retrieval_pass = (
            processable
            &
            rank_pass.fillna(
                False
            )
        )


    # --------------------------------------------------------
    # Retrieval-only coverage
    # --------------------------------------------------------

    retrieval_n = int(
        retrieval_pass.sum()
    )


    retrieval_full7_n = int(
        (
            retrieval_pass
            &
            oracle[
                "pred_full_7um"
            ].astype(bool)
        ).sum()
    )


    # --------------------------------------------------------
    # Combined with CURRENT dominance-any
    # --------------------------------------------------------

    combined_pass = (
        retrieval_pass
        &
        oracle[
            "dominance_any"
        ].astype(bool)
    )


    combined_n = int(
        combined_pass.sum()
    )


    combined_full7_n = int(
        (
            combined_pass
            &
            oracle[
                "pred_full_7um"
            ].astype(bool)
        ).sum()
    )


    raw_rows = int(
        counts[
            contract_name
        ][
            "rows"
        ]
    )


    ratio = float(
        raw_rows
        /
        base_raw_rows
    )


    oracle_rows.append(
        {
            "contract":
                contract_name,

            "raw_candidate_rows":
                raw_rows,

            "raw_ratio_vs_exact5":
                ratio,

            "processable_total":
                processable_total,

            "retrieval_captured":
                retrieval_n,

            "retrieval_coverage":
                float(
                    retrieval_n
                    /
                    processable_total
                ),

            "processable_full7_total":
                processable_full7_total,

            "retrieval_full7_captured":
                retrieval_full7_n,

            "retrieval_full7_coverage":
                float(
                    retrieval_full7_n
                    /
                    processable_full7_total
                ),

            "combined_captured":
                combined_n,

            "combined_overall_coverage":
                float(
                    combined_n
                    /
                    len(
                        oracle
                    )
                ),

            "combined_full7_captured":
                combined_full7_n,

            "combined_full7_overall_coverage":
                float(
                    combined_full7_n
                    /
                    int(
                        oracle[
                            "pred_full_7um"
                        ].sum()
                    )
                ),
        }
    )


oracle_contract_table = pd.DataFrame(
    oracle_rows
)


oracle_contract_table = (
    oracle_contract_table
    .sort_values(
        [
            "raw_candidate_rows",
            "contract",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


print(
    oracle_contract_table.to_string(
        index=False
    )
)


# ============================================================
# 11. Relative efficiency table
#
# Oracle benefit per cardinality multiplier.
#
# DIAGNOSTIC ONLY.
# ============================================================

baseline_row = (
    oracle_contract_table[
        oracle_contract_table[
            "contract"
        ].eq(
            "BASE_EXACT_5x5"
        )
    ]
    .iloc[
        0
    ]
)


base_retrieval = int(
    baseline_row[
        "retrieval_captured"
    ]
)


base_combined = int(
    baseline_row[
        "combined_captured"
    ]
)


efficiency = (
    oracle_contract_table
    .copy()
)


efficiency[
    "extra_retrieval_vs_base"
] = (
    efficiency[
        "retrieval_captured"
    ]
    -
    base_retrieval
)


efficiency[
    "extra_combined_vs_base"
] = (
    efficiency[
        "combined_captured"
    ]
    -
    base_combined
)


efficiency[
    "candidate_multiplier_minus_1"
] = (
    efficiency[
        "raw_ratio_vs_exact5"
    ]
    -
    1.0
)


print(
    "\n========== RETRIEVAL CARDINALITY / COVERAGE TRADEOFF =========="
)


print(
    efficiency[
        [
            "contract",

            "raw_candidate_rows",
            "raw_ratio_vs_exact5",

            "retrieval_captured",
            "extra_retrieval_vs_base",

            "combined_captured",
            "extra_combined_vs_base",

            "retrieval_full7_captured",
            "combined_full7_captured",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 12. Coverage by topology for each contract
# ============================================================

print(
    "\n========== CONTRACT COVERAGE BY TOPOLOGY =========="
)


topology_rows = []


for contract_name in CONTRACTS:

    if contract_name == "BASE_EXACT_5x5":

        retrieval_pass = (
            processable
            &
            oracle[
                "exact_reciprocal_top5"
            ].astype(bool)
        )

    else:

        spec = CONTRACTS[
            contract_name
        ]


        source_rank = oracle[
            "exact_source_rank_k16"
        ]


        target_rank = oracle[
            "exact_target_rank_k16"
        ]


        if spec[
            "type"
        ] == "RECIP":

            rank_pass = (
                source_rank.le(
                    int(
                        spec[
                            "a"
                        ]
                    )
                )
                &
                target_rank.le(
                    int(
                        spec[
                            "b"
                        ]
                    )
                )
            )


        else:

            a = int(
                spec[
                    "a"
                ]
            )

            b = int(
                spec[
                    "b"
                ]
            )


            rank_pass = (
                (
                    source_rank.le(
                        a
                    )
                    &
                    target_rank.le(
                        b
                    )
                )
                |
                (
                    target_rank.le(
                        a
                    )
                    &
                    source_rank.le(
                        b
                    )
                )
            )


        retrieval_pass = (
            processable
            &
            rank_pass.fillna(
                False
            )
        )


    combined_pass = (
        retrieval_pass
        &
        oracle[
            "dominance_any"
        ].astype(bool)
    )


    for topology, sub_index in oracle.groupby(
        "topology_class"
    ).groups.items():

        sub_index = list(
            sub_index
        )


        total = len(
            sub_index
        )


        retrieval_n = int(
            retrieval_pass.loc[
                sub_index
            ].sum()
        )


        combined_n = int(
            combined_pass.loc[
                sub_index
            ].sum()
        )


        topology_rows.append(
            {
                "contract":
                    contract_name,

                "topology":
                    topology,

                "total":
                    int(
                        total
                    ),

                "retrieval_captured":
                    retrieval_n,

                "combined_captured":
                    combined_n,
            }
        )


topology_table = pd.DataFrame(
    topology_rows
)


for contract_name in CONTRACTS:

    print(
        "\n" + contract_name
    )


    print(
        topology_table[
            topology_table[
                "contract"
            ].eq(
                contract_name
            )
        ][
            [
                "topology",
                "total",
                "retrieval_captured",
                "combined_captured",
            ]
        ]
        .to_string(
            index=False
        )
    )


# ============================================================
# 13. Persist oracle audit
# ============================================================

for contract_name in CONTRACTS:

    if contract_name == "BASE_EXACT_5x5":

        mask = (
            processable
            &
            oracle[
                "exact_reciprocal_top5"
            ].astype(bool)
        )

    else:

        spec = CONTRACTS[
            contract_name
        ]


        sr = oracle[
            "exact_source_rank_k16"
        ]

        tr = oracle[
            "exact_target_rank_k16"
        ]


        if spec[
            "type"
        ] == "RECIP":

            mask = (
                processable
                &
                sr.le(
                    int(
                        spec[
                            "a"
                        ]
                    )
                )
                &
                tr.le(
                    int(
                        spec[
                            "b"
                        ]
                    )
                )
            )


        else:

            a = int(
                spec[
                    "a"
                ]
            )

            b = int(
                spec[
                    "b"
                ]
            )


            mask = (
                processable
                &
                (
                    (
                        sr.le(
                            a
                        )
                        &
                        tr.le(
                            b
                        )
                    )
                    |
                    (
                        tr.le(
                            a
                        )
                        &
                        sr.le(
                            b
                        )
                    )
                )
            )


        mask = mask.fillna(
            False
        )


    oracle[
        f"retrieval_{contract_name}"
    ] = mask.astype(
        bool
    )


    oracle[
        f"combined_{contract_name}"
    ] = (
        mask.astype(bool)
        &
        oracle[
            "dominance_any"
        ].astype(bool)
    )


oracle = (
    oracle
    .sort_values(
        [
            "fold",
            "dataset",
            "component_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


oracle.to_pickle(
    ORACLE_OUT
)


oracle_summary = {
    "stage":
        "12.11G3F1",

    "status":
        "ASYMMETRIC_RETRIEVAL_CONTRACT_CARDINALITY_AND_ORACLE_AUDITED",

    "contract_cardinality_sha256":
        contract_sha,

    "gt_blind_cardinality_frozen_before_oracle":
        True,

    "contracts":
        CONTRACTS,

    "contract_results":
        oracle_contract_table.to_dict(
            "records"
        ),

    "clean_components":
        369,

    "processable_components":
        processable_total,

    "no_new_proposal_universe":
        True,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


ORACLE_SUMMARY_OUT.write_text(
    json.dumps(
        oracle_summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert ORACLE_OUT.exists()
assert ORACLE_SUMMARY_OUT.exists()


# ============================================================
# 14. Integrity
# ============================================================

check = pd.read_pickle(
    ORACLE_OUT
)


assert len(
    check
) == 369


assert not check[
    [
        "dataset",
        "component_id",
    ]
].duplicated().any()


# Baseline retrieval should exactly match G3F0-R exact K5.
assert (
    check[
        "retrieval_BASE_EXACT_5x5"
    ]
    ==
    (
        check[
            "within_horizon"
        ].astype(bool)
        &
        check[
            "source_eligible"
        ].astype(bool)
        &
        check[
            "target_eligible"
        ].astype(bool)
        &
        ~check[
            "pair_existing_edge"
        ].astype(bool)
        &
        check[
            "exact_reciprocal_top5"
        ].astype(bool)
    )
).all()


assert int(
    check[
        "combined_BASE_EXACT_5x5"
    ].sum()
) == 185


# ============================================================
# 15. Decision
# ============================================================

print(
    "\n========== 12.11G3F1 DECISION =========="
)

print(
    "ASYMMETRIC_RETRIEVAL_CONTRACT_AUDIT: PASS"
)

print(
    "GT-blind cardinality frozen before oracle:",
    "YES"
)

print(
    "new proposal universe generated:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "classifier trained:",
    "NO"
)

print(
    "threshold selected:",
    "NO"
)

print(
    "policy selected:",
    "NO"
)

print(
    "graph modified:",
    "NO"
)

print(
    "next:"
)

print(
    "SELECT_LOW_CARDINALITY_RETRIEVAL_FRONTIER_THEN_FREEZE_GENERATOR"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3F1
#
# GT-Blind Asymmetric Retrieval Contract Cardinality Audit
# + Development Oracle Coverage Audit
#
# Motivation:
#
#   G3F0-R exact decomposition:
#
#       RECIPROCAL_ONLY_MISS             101
#       RECIPROCAL_AND_DOMINANCE_MISS     28
#       DOMINANCE_ONLY_MISS               20
#
#   Therefore retrieval is the primary bottleneck.
#
# BUT:
#
#   G3E-R K=5 stored 23,012,742 proposals.
#   Naively increasing full reciprocal K would be too expensive.
#
# We therefore test asymmetric retrieval contracts:
#
#   ASYM(a,b):
#
#       (source_rank <= a AND target_rank <= b)
#         OR
#       (target_rank <= a AND source_rank <= b)
#
# This requires one side to be a strong local anchor while
# allowing the opposite trajectory-consistency rank more slack.
#
# PART A:
#   - 103 development datasets
#   - GT blind
#   - count candidate cardinality only
#   - DO NOT persist candidate rows
#   - freeze contract-count artifact before GT read
#
# PART B:
#   - only after Part A freeze
#   - development oracle rank audit
#   - exact K16 query for K<=16 family
#   - BASE_EXACT_5x5 uses retained exact-K5 G3F0-R result
#   - combine with existing dominance-any only diagnostically
#
# No new proposal universe.
# No Fold0.
# No classifier.
# No final threshold.
# No policy selection.
# No graph modification.
#
# Keep after running: YES
# ============================================================

from pathlib import Path
import gc
import hashlib
import json

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


G3F0R_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_gate_miss_decomposition_exactk5.pkl"
)


CARDINALITY_OUT = (
    S12
    / "stage12_folds14_multiframe_asymmetric_retrieval_cardinality_v1.json"
)


ORACLE_OUT = (
    S12
    / "stage12_folds14_multiframe_asymmetric_retrieval_oracle_v1.pkl"
)


ORACLE_SUMMARY_OUT = (
    S12
    / "stage12_folds14_multiframe_asymmetric_retrieval_oracle_v1_summary.json"
)


assert G3F0R_PATH.exists(), G3F0R_PATH


# ============================================================
# 1. Existing exact CURRENT_BEST contract
# ============================================================

assert (
    "build_current_best"
    in globals()
    and
    callable(
        globals()[
            "build_current_best"
        ]
    )
), (
    "build_current_best missing. "
    "Re-run retained successful G3E-R first."
)


current_best_builder = globals()[
    "build_current_best"
]


# ------------------------------------------------------------
# Development corpus
# ------------------------------------------------------------

if (
    "names"
    in globals()
    and
    len(
        globals()[
            "names"
        ]
    ) == 103
):

    development_names = list(
        globals()[
            "names"
        ]
    )

else:

    CV_PATH = (
        PROJECT
        / "configs"
        / "cv_manifest.csv"
    )


    cv = pd.read_csv(
        CV_PATH
    )


    development_names = (
        cv[
            cv[
                "fold"
            ].isin(
                [
                    1,
                    2,
                    3,
                    4,
                ]
            )
            &
            cv[
                "dataset"
            ].str.startswith(
                "6bba_"
            )
        ]
        .sort_values(
            [
                "fold",
                "dataset",
            ],
            kind="stable",
        )[
            "dataset"
        ]
        .tolist()
    )


assert len(
    development_names
) == 103


# ------------------------------------------------------------
# Fold map
# ------------------------------------------------------------

if (
    "fold_map"
    in globals()
):

    development_fold_map = dict(
        globals()[
            "fold_map"
        ]
    )

else:

    CV_PATH = (
        PROJECT
        / "configs"
        / "cv_manifest.csv"
    )


    cv = pd.read_csv(
        CV_PATH
    )


    dev = cv[
        cv[
            "dataset"
        ].isin(
            development_names
        )
    ]


    development_fold_map = dict(
        zip(
            dev[
                "dataset"
            ],
            dev[
                "fold"
            ].astype(int),
        )
    )


SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


MIN_GAP_SIZE = 2
MAX_GAP_SIZE = 12

EXACT_BASE_K = 5
MAX_RANK_K = 16


# ============================================================
# 2. Frozen retrieval-contract family
#
# IMPORTANT:
# Defined BEFORE development oracle is read.
# ============================================================

CONTRACTS = {
    "BASE_EXACT_5x5": {
        "type":
            "BASE_EXACT",

        "a":
            5,

        "b":
            5,
    },

    "ASYM_3x8": {
        "type":
            "ASYM",

        "a":
            3,

        "b":
            8,
    },

    "ASYM_5x8": {
        "type":
            "ASYM",

        "a":
            5,

        "b":
            8,
    },

    "ASYM_3x12": {
        "type":
            "ASYM",

        "a":
            3,

        "b":
            12,
    },

    "ASYM_5x12": {
        "type":
            "ASYM",

        "a":
            5,

        "b":
            12,
    },

    "ASYM_3x16": {
        "type":
            "ASYM",

        "a":
            3,

        "b":
            16,
    },

    "ASYM_5x16": {
        "type":
            "ASYM",

        "a":
            5,

        "b":
            16,
    },

    "RECIP_8x8": {
        "type":
            "RECIP",

        "a":
            8,

        "b":
            8,
    },

    "RECIP_12x12": {
        "type":
            "RECIP",

        "a":
            12,

        "b":
            12,
    },

    "RECIP_16x16": {
        "type":
            "RECIP",

        "a":
            16,

        "b":
            16,
    },
}


print(
    "========== 12.11G3F1 CONTRACT FAMILY =========="
)

print(
    "development datasets:",
    len(
        development_names
    )
)

print(
    "gap horizon:",
    MIN_GAP_SIZE,
    "to",
    MAX_GAP_SIZE
)

print(
    "max exact rank query:",
    MAX_RANK_K
)

print(
    "contracts:"
)


for name, spec in CONTRACTS.items():

    print(
        name,
        spec
    )


# ============================================================
# 3. Helpers
# ============================================================

def build_rank_map(
    query_ids,
    candidate_ids,
    indices,
):

    result = {}


    for query_i, query_id in enumerate(
        query_ids
    ):

        for rank_i in range(
            indices.shape[
                1
            ]
        ):

            candidate_id = int(
                candidate_ids[
                    int(
                        indices[
                            query_i,
                            rank_i
                        ]
                    )
                ]
            )


            pair = (
                int(
                    query_id
                ),
                candidate_id,
            )


            if pair not in result:

                result[
                    pair
                ] = int(
                    rank_i
                    +
                    1
                )


    return result


def build_reverse_rank_map(
    query_ids,
    candidate_ids,
    indices,
):

    result = {}


    for query_i, query_id in enumerate(
        query_ids
    ):

        for rank_i in range(
            indices.shape[
                1
            ]
        ):

            candidate_id = int(
                candidate_ids[
                    int(
                        indices[
                            query_i,
                            rank_i
                        ]
                    )
                ]
            )


            # candidate is source;
            # query is target.
            pair = (
                candidate_id,
                int(
                    query_id
                ),
            )


            if pair not in result:

                result[
                    pair
                ] = int(
                    rank_i
                    +
                    1
                )


    return result


def query_tree(
    tree,
    query_points,
    candidate_count,
    k,
):

    k_eff = min(
        int(
            k
        ),
        int(
            candidate_count
        ),
    )


    distances, indices = tree.query(
        query_points,
        k=k_eff,
    )


    if k_eff == 1:

        distances = distances[
            :,
            None
        ]

        indices = indices[
            :,
            None
        ]


    return (
        np.asarray(
            distances,
            dtype=float,
        ),

        np.asarray(
            indices,
            dtype=int,
        ),
    )


def contract_pass(
    contract_name,
    source_rank,
    target_rank,
):

    spec = CONTRACTS[
        contract_name
    ]


    contract_type = spec[
        "type"
    ]


    a = int(
        spec[
            "a"
        ]
    )


    b = int(
        spec[
            "b"
        ]
    )


    if (
        not np.isfinite(
            source_rank
        )
        or
        not np.isfinite(
            target_rank
        )
    ):

        return False


    source_rank = int(
        source_rank
    )

    target_rank = int(
        target_rank
    )


    if contract_type == "RECIP":

        return bool(
            source_rank <= a
            and
            target_rank <= b
        )


    if contract_type == "ASYM":

        return bool(
            (
                source_rank <= a
                and
                target_rank <= b
            )
            or
            (
                target_rank <= a
                and
                source_rank <= b
            )
        )


    raise ValueError(
        (
            contract_name,
            spec,
        )
    )


# ============================================================
# PART A
# GT-BLIND CARDINALITY SCAN
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART A — GT-BLIND RETRIEVAL CARDINALITY"
)

print(
    "============================================================"
)


# ------------------------------------------------------------
# Global count structure
# ------------------------------------------------------------

counts = {}


for contract_name in CONTRACTS:

    counts[
        contract_name
    ] = {
        "rows":
            0,

        "cuts_0":
            0,

        "cuts_1":
            0,

        "cuts_2":
            0,
    }


dataset_counts = []


# ============================================================
# 4. Scan 103 datasets
# ============================================================

for ds_i, dataset in enumerate(
    development_names,
    start=1,
):

    fold = int(
        development_fold_map[
            dataset
        ]
    )


    graph, scale = current_best_builder(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    )


    # ========================================================
    # Nodes
    # ========================================================

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = (
        nodes[
            "node_id"
        ]
        .astype(int)
    )


    nodes[
        "t"
    ] = (
        nodes[
            "t"
        ]
        .astype(int)
    )


    nodes = (
        nodes
        .sort_values(
            [
                "t",
                "node_id",
            ],
            kind="stable",
        )
        .set_index(
            "node_id",
            drop=False,
        )
    )


    # ========================================================
    # Edges / topology
    # ========================================================

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = (
        edges[
            "source_id"
        ]
        .astype(int)
    )


    edges[
        "target_id"
    ] = (
        edges[
            "target_id"
        ]
        .astype(int)
    )


    edge_set = set(
        zip(
            edges[
                "source_id"
            ],
            edges[
                "target_id"
            ],
        )
    )


    incoming = (
        edges.groupby(
            "target_id"
        )[
            "source_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    outgoing = (
        edges.groupby(
            "source_id"
        )[
            "target_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = {
        int(k):
            int(v)

        for k, v
        in indeg.items()
    }


    outdeg = {
        int(k):
            int(v)

        for k, v
        in outdeg.items()
    }


    # ========================================================
    # Physical state
    # ========================================================

    phys = {}

    time_map = {}


    for nid, row in nodes.iterrows():

        nid = int(
            nid
        )


        phys[
            nid
        ] = (
            np.array(
                [
                    float(
                        row[
                            "z"
                        ]
                    ),

                    float(
                        row[
                            "y"
                        ]
                    ),

                    float(
                        row[
                            "x"
                        ]
                    ),
                ],
                dtype=float,
            )
            *
            SCALE
        )


        time_map[
            nid
        ] = int(
            row[
                "t"
            ]
        )


    node_ids = sorted(
        phys
    )


    # ========================================================
    # Local velocities
    # ========================================================

    past_velocity = {}

    future_velocity = {}


    for nid in node_ids:

        inc = incoming.get(
            nid,
            [],
        )


        if len(
            inc
        ) == 1:

            prev_id = int(
                inc[
                    0
                ]
            )


            if prev_id in phys:

                dt = (
                    time_map[
                        nid
                    ]
                    -
                    time_map[
                        prev_id
                    ]
                )


                if dt > 0:

                    past_velocity[
                        nid
                    ] = (
                        phys[
                            nid
                        ]
                        -
                        phys[
                            prev_id
                        ]
                    ) / float(
                        dt
                    )


        out = outgoing.get(
            nid,
            [],
        )


        if len(
            out
        ) == 1:

            next_id = int(
                out[
                    0
                ]
            )


            if next_id in phys:

                dt = (
                    time_map[
                        next_id
                    ]
                    -
                    time_map[
                        nid
                    ]
                )


                if dt > 0:

                    future_velocity[
                        nid
                    ] = (
                        phys[
                            next_id
                        ]
                        -
                        phys[
                            nid
                        ]
                    ) / float(
                        dt
                    )


    # ========================================================
    # Eligibility
    # ========================================================

    source_eligible = [
        nid

        for nid in node_ids

        if (
            outdeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid
            in
            past_velocity
        )
    ]


    target_eligible = [
        nid

        for nid in node_ids

        if (
            indeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid
            in
            future_velocity
        )
    ]


    source_by_t = {}

    target_by_t = {}


    for nid in source_eligible:

        source_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for nid in target_eligible:

        target_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for t in source_by_t:

        source_by_t[
            t
        ] = sorted(
            source_by_t[
                t
            ]
        )


    for t in target_by_t:

        target_by_t[
            t
        ] = sorted(
            target_by_t[
                t
            ]
        )


    # ========================================================
    # Per-dataset counters
    # ========================================================

    local_counts = {}


    for contract_name in CONTRACTS:

        local_counts[
            contract_name
        ] = {
            "rows":
                0,

            "cuts_0":
                0,

            "cuts_1":
                0,

            "cuts_2":
                0,
        }


    # ========================================================
    # Frame-pair retrieval
    # ========================================================

    for gap_size in range(
        MIN_GAP_SIZE,
        MAX_GAP_SIZE + 1,
    ):

        delta_t = (
            gap_size
            +
            1
        )


        for source_t in sorted(
            source_by_t
        ):

            target_t = (
                source_t
                +
                delta_t
            )


            if target_t not in target_by_t:

                continue


            source_ids = source_by_t[
                source_t
            ]


            target_ids = target_by_t[
                target_t
            ]


            if (
                len(
                    source_ids
                ) == 0
                or
                len(
                    target_ids
                ) == 0
            ):

                continue


            source_pos = np.vstack(
                [
                    phys[
                        nid
                    ]

                    for nid in source_ids
                ]
            )


            target_pos = np.vstack(
                [
                    phys[
                        nid
                    ]

                    for nid in target_ids
                ]
            )


            source_vel = np.vstack(
                [
                    past_velocity[
                        nid
                    ]

                    for nid in source_ids
                ]
            )


            target_vel = np.vstack(
                [
                    future_velocity[
                        nid
                    ]

                    for nid in target_ids
                ]
            )


            source_predicted_target = (
                source_pos
                +
                float(
                    delta_t
                )
                *
                source_vel
            )


            target_predicted_source = (
                target_pos
                -
                float(
                    delta_t
                )
                *
                target_vel
            )


            target_tree = cKDTree(
                target_pos
            )


            source_tree = cKDTree(
                source_pos
            )


            # =================================================
            # Exact original K=5 query
            # =================================================

            (
                _,
                src_idx_k5,
            ) = query_tree(
                target_tree,
                source_predicted_target,
                len(
                    target_ids
                ),
                EXACT_BASE_K,
            )


            (
                _,
                tgt_idx_k5,
            ) = query_tree(
                source_tree,
                target_predicted_source,
                len(
                    source_ids
                ),
                EXACT_BASE_K,
            )


            src_map_k5 = build_rank_map(
                source_ids,
                target_ids,
                src_idx_k5,
            )


            tgt_map_k5 = build_reverse_rank_map(
                target_ids,
                source_ids,
                tgt_idx_k5,
            )


            base_pairs = (
                set(
                    src_map_k5
                )
                &
                set(
                    tgt_map_k5
                )
            )


            for (
                src,
                tgt,
            ) in base_pairs:

                if (
                    src,
                    tgt,
                ) in edge_set:

                    continue


                cuts = (
                    int(
                        outdeg.get(
                            src,
                            0,
                        )
                        == 1
                    )
                    +
                    int(
                        indeg.get(
                            tgt,
                            0,
                        )
                        == 1
                    )
                )


                local_counts[
                    "BASE_EXACT_5x5"
                ][
                    "rows"
                ] += 1


                local_counts[
                    "BASE_EXACT_5x5"
                ][
                    f"cuts_{cuts}"
                ] += 1


            # =================================================
            # Exact K=16 family query
            # =================================================

            (
                _,
                src_idx_k16,
            ) = query_tree(
                target_tree,
                source_predicted_target,
                len(
                    target_ids
                ),
                MAX_RANK_K,
            )


            (
                _,
                tgt_idx_k16,
            ) = query_tree(
                source_tree,
                target_predicted_source,
                len(
                    source_ids
                ),
                MAX_RANK_K,
            )


            src_map_k16 = build_rank_map(
                source_ids,
                target_ids,
                src_idx_k16,
            )


            tgt_map_k16 = build_reverse_rank_map(
                target_ids,
                source_ids,
                tgt_idx_k16,
            )


            mutual16_pairs = (
                set(
                    src_map_k16
                )
                &
                set(
                    tgt_map_k16
                )
            )


            for (
                src,
                tgt,
            ) in mutual16_pairs:

                if (
                    src,
                    tgt,
                ) in edge_set:

                    continue


                source_rank = src_map_k16[
                    (
                        src,
                        tgt,
                    )
                ]


                target_rank = tgt_map_k16[
                    (
                        src,
                        tgt,
                    )
                ]


                cuts = (
                    int(
                        outdeg.get(
                            src,
                            0,
                        )
                        == 1
                    )
                    +
                    int(
                        indeg.get(
                            tgt,
                            0,
                        )
                        == 1
                    )
                )


                for contract_name in CONTRACTS:

                    if contract_name == "BASE_EXACT_5x5":

                        continue


                    if contract_pass(
                        contract_name,
                        source_rank,
                        target_rank,
                    ):

                        local_counts[
                            contract_name
                        ][
                            "rows"
                        ] += 1


                        local_counts[
                            contract_name
                        ][
                            f"cuts_{cuts}"
                        ] += 1


            # =================================================
            # Frame-pair cleanup
            # =================================================

            del source_pos
            del target_pos

            del source_vel
            del target_vel

            del source_predicted_target
            del target_predicted_source

            del target_tree
            del source_tree

            del src_idx_k5
            del tgt_idx_k5

            del src_map_k5
            del tgt_map_k5
            del base_pairs

            del src_idx_k16
            del tgt_idx_k16

            del src_map_k16
            del tgt_map_k16
            del mutual16_pairs


    # ========================================================
    # Aggregate dataset counts
    # ========================================================

    dataset_record = {
        "dataset":
            dataset,

        "fold":
            fold,
    }


    for contract_name in CONTRACTS:

        local = local_counts[
            contract_name
        ]


        assert (
            local[
                "cuts_0"
            ]
            +
            local[
                "cuts_1"
            ]
            +
            local[
                "cuts_2"
            ]
        ) == local[
            "rows"
        ]


        for key in [
            "rows",
            "cuts_0",
            "cuts_1",
            "cuts_2",
        ]:

            counts[
                contract_name
            ][
                key
            ] += int(
                local[
                    key
                ]
            )


        dataset_record[
            contract_name
        ] = {
            key:
                int(
                    local[
                        key
                    ]
                )

            for key in [
                "rows",
                "cuts_0",
                "cuts_1",
                "cuts_2",
            ]
        }


    dataset_counts.append(
        dataset_record
    )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            development_names
        )
    ):

        print(
            f"{ds_i:3d}/{len(development_names)}"
            f" | fold={fold}"
            f" | {dataset}"
            f" | base5={local_counts['BASE_EXACT_5x5']['rows']}"
            f" | a5x16={local_counts['ASYM_5x16']['rows']}"
            f" | r16={local_counts['RECIP_16x16']['rows']}"
        )


    # ========================================================
    # Dataset cleanup
    # ========================================================

    del graph
    del nodes
    del edges

    del edge_set

    del incoming
    del outgoing

    del indeg
    del outdeg

    del phys
    del time_map
    del node_ids

    del past_velocity
    del future_velocity

    del source_eligible
    del target_eligible

    del source_by_t
    del target_by_t

    del local_counts

    gc.collect()


# ============================================================
# 5. GT-blind cardinality table
# ============================================================

cardinality_rows = []


base_raw_rows = int(
    counts[
        "BASE_EXACT_5x5"
    ][
        "rows"
    ]
)


assert base_raw_rows > 0


for contract_name, spec in CONTRACTS.items():

    row = {
        "contract":
            contract_name,

        "type":
            spec[
                "type"
            ],

        "a":
            int(
                spec[
                    "a"
                ]
            ),

        "b":
            int(
                spec[
                    "b"
                ]
            ),

        "rows":
            int(
                counts[
                    contract_name
                ][
                    "rows"
                ]
            ),

        "cuts_0":
            int(
                counts[
                    contract_name
                ][
                    "cuts_0"
                ]
            ),

        "cuts_1":
            int(
                counts[
                    contract_name
                ][
                    "cuts_1"
                ]
            ),

        "cuts_2":
            int(
                counts[
                    contract_name
                ][
                    "cuts_2"
                ]
            ),
    }


    row[
        "raw_cardinality_ratio_vs_exact5"
    ] = float(
        row[
            "rows"
        ]
        /
        base_raw_rows
    )


    cardinality_rows.append(
        row
    )


cardinality_table = pd.DataFrame(
    cardinality_rows
)


cardinality_table = (
    cardinality_table
    .sort_values(
        [
            "rows",
            "contract",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n========== GT-BLIND RETRIEVAL CARDINALITY =========="
)


print(
    cardinality_table.to_string(
        index=False
    )
)


# ============================================================
# 6. Freeze GT-blind contract/cardinality artifact
#
# IMPORTANT:
# G3F0-R oracle has NOT been read in this cell yet.
# ============================================================

cardinality_payload = {
    "stage":
        "12.11G3F1",

    "part":
        "GT_BLIND_RETRIEVAL_CONTRACT_CARDINALITY",

    "development_datasets":
        103,

    "min_gap_size":
        MIN_GAP_SIZE,

    "max_gap_size":
        MAX_GAP_SIZE,

    "exact_base_k":
        EXACT_BASE_K,

    "max_rank_query":
        MAX_RANK_K,

    "contracts":
        CONTRACTS,

    "counts":
        {
            row[
                "contract"
            ]:
                {
                    "rows":
                        int(
                            row[
                                "rows"
                            ]
                        ),

                    "cuts_0":
                        int(
                            row[
                                "cuts_0"
                            ]
                        ),

                    "cuts_1":
                        int(
                            row[
                                "cuts_1"
                            ]
                        ),

                    "cuts_2":
                        int(
                            row[
                                "cuts_2"
                            ]
                        ),

                    "raw_cardinality_ratio_vs_exact5":
                        float(
                            row[
                                "raw_cardinality_ratio_vs_exact5"
                            ]
                        ),
                }

            for row
            in cardinality_rows
        },

    "dataset_counts":
        dataset_counts,

    "candidate_rows_persisted":
        False,

    "gt_used":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


canonical_contract_bytes = json.dumps(
    cardinality_payload,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
).encode(
    "utf-8"
)


contract_sha = hashlib.sha256(
    canonical_contract_bytes
).hexdigest()


cardinality_payload[
    "contract_cardinality_sha256"
] = contract_sha


CARDINALITY_OUT.write_text(
    json.dumps(
        cardinality_payload,
        indent=2,
    ),
    encoding="utf-8",
)


assert CARDINALITY_OUT.exists()


print(
    "\nGT-BLIND RETRIEVAL CONTRACT FAMILY FROZEN BEFORE ORACLE READ: YES"
)

print(
    "contract/cardinality SHA256:",
    contract_sha
)


# ============================================================
# PART B
#
# DEVELOPMENT ORACLE READ IS ALLOWED ONLY BELOW THIS LINE.
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — DEVELOPMENT ORACLE CONTRACT COVERAGE"
)

print(
    "============================================================"
)


# ============================================================
# 7. Load retained exact G3F0-R oracle
# ============================================================

g3f0 = pd.read_pickle(
    G3F0R_PATH
)


assert len(
    g3f0
) == 369


assert g3f0[
    "reconstruction_match"
].all()


assert int(
    g3f0[
        "reconstructed_g3er_capture"
    ].sum()
) == 185


oracle = (
    g3f0
    .copy()
)


oracle_datasets = (
    oracle[
        "dataset"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)


assert len(
    oracle_datasets
) == 75


# ============================================================
# 8. Exact K16 rank fields for oracle true pairs
#
# We cannot infer exact K16 ranks from K64 because of tie order.
# So rerun exact K16 query only on oracle datasets.
# ============================================================

oracle_rank_rows = []


for ds_i, dataset in enumerate(
    oracle_datasets,
    start=1,
):

    graph, scale = current_best_builder(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    )


    # ========================================================
    # Nodes
    # ========================================================

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = (
        nodes[
            "node_id"
        ]
        .astype(int)
    )


    nodes[
        "t"
    ] = (
        nodes[
            "t"
        ]
        .astype(int)
    )


    nodes = (
        nodes
        .sort_values(
            [
                "t",
                "node_id",
            ],
            kind="stable",
        )
        .set_index(
            "node_id",
            drop=False,
        )
    )


    # ========================================================
    # Edges
    # ========================================================

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = (
        edges[
            "source_id"
        ]
        .astype(int)
    )


    edges[
        "target_id"
    ] = (
        edges[
            "target_id"
        ]
        .astype(int)
    )


    incoming = (
        edges.groupby(
            "target_id"
        )[
            "source_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    outgoing = (
        edges.groupby(
            "source_id"
        )[
            "target_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = {
        int(k):
            int(v)

        for k, v
        in indeg.items()
    }


    outdeg = {
        int(k):
            int(v)

        for k, v
        in outdeg.items()
    }


    # ========================================================
    # Physical state
    # ========================================================

    phys = {}

    time_map = {}


    for nid, row in nodes.iterrows():

        nid = int(
            nid
        )


        phys[
            nid
        ] = (
            np.array(
                [
                    float(
                        row[
                            "z"
                        ]
                    ),

                    float(
                        row[
                            "y"
                        ]
                    ),

                    float(
                        row[
                            "x"
                        ]
                    ),
                ],
                dtype=float,
            )
            *
            SCALE
        )


        time_map[
            nid
        ] = int(
            row[
                "t"
            ]
        )


    node_ids = sorted(
        phys
    )


    # ========================================================
    # Velocities
    # ========================================================

    past_velocity = {}

    future_velocity = {}


    for nid in node_ids:

        inc = incoming.get(
            nid,
            [],
        )


        if len(
            inc
        ) == 1:

            prev_id = int(
                inc[
                    0
                ]
            )


            if prev_id in phys:

                dt = (
                    time_map[
                        nid
                    ]
                    -
                    time_map[
                        prev_id
                    ]
                )


                if dt > 0:

                    past_velocity[
                        nid
                    ] = (
                        phys[
                            nid
                        ]
                        -
                        phys[
                            prev_id
                        ]
                    ) / float(
                        dt
                    )


        out = outgoing.get(
            nid,
            [],
        )


        if len(
            out
        ) == 1:

            next_id = int(
                out[
                    0
                ]
            )


            if next_id in phys:

                dt = (
                    time_map[
                        next_id
                    ]
                    -
                    time_map[
                        nid
                    ]
                )


                if dt > 0:

                    future_velocity[
                        nid
                    ] = (
                        phys[
                            next_id
                        ]
                        -
                        phys[
                            nid
                        ]
                    ) / float(
                        dt
                    )


    # ========================================================
    # Same eligibility
    # ========================================================

    source_eligible = [
        nid

        for nid in node_ids

        if (
            outdeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid
            in
            past_velocity
        )
    ]


    target_eligible = [
        nid

        for nid in node_ids

        if (
            indeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid
            in
            future_velocity
        )
    ]


    source_eligible_set = set(
        source_eligible
    )


    target_eligible_set = set(
        target_eligible
    )


    source_by_t = {}

    target_by_t = {}


    for nid in source_eligible:

        source_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for nid in target_eligible:

        target_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for t in source_by_t:

        source_by_t[
            t
        ] = sorted(
            source_by_t[
                t
            ]
        )


    for t in target_by_t:

        target_by_t[
            t
        ] = sorted(
            target_by_t[
                t
            ]
        )


    # ========================================================
    # Oracle pairs for this dataset
    # ========================================================

    ds_oracle = oracle[
        oracle[
            "dataset"
        ].eq(
            dataset
        )
    ]


    local = {}


    frame_groups = {}


    for row in ds_oracle.itertuples(
        index=False
    ):

        key = (
            dataset,
            row.component_id,
        )


        src = int(
            row.source_id
        )


        tgt = int(
            row.target_id
        )


        local[
            key
        ] = {
            "dataset":
                dataset,

            "component_id":
                row.component_id,

            "exact_source_rank_k16":
                np.nan,

            "exact_target_rank_k16":
                np.nan,
        }


        if not (
            bool(
                row.within_horizon
            )
            and
            bool(
                row.source_eligible
            )
            and
            bool(
                row.target_eligible
            )
            and
            not bool(
                row.pair_existing_edge
            )
        ):

            continue


        # Cross-check fresh graph state.
        assert src in source_eligible_set
        assert tgt in target_eligible_set


        source_t = int(
            time_map[
                src
            ]
        )


        target_t = int(
            time_map[
                tgt
            ]
        )


        frame_groups.setdefault(
            (
                source_t,
                target_t,
            ),
            [],
        ).append(
            key
        )


    # ========================================================
    # Exact K16 query
    # ========================================================

    for (
        source_t,
        target_t,
    ), keys in frame_groups.items():

        source_ids = source_by_t[
            source_t
        ]


        target_ids = target_by_t[
            target_t
        ]


        delta_t = (
            target_t
            -
            source_t
        )


        source_pos = np.vstack(
            [
                phys[
                    nid
                ]

                for nid in source_ids
            ]
        )


        target_pos = np.vstack(
            [
                phys[
                    nid
                ]

                for nid in target_ids
            ]
        )


        source_vel = np.vstack(
            [
                past_velocity[
                    nid
                ]

                for nid in source_ids
            ]
        )


        target_vel = np.vstack(
            [
                future_velocity[
                    nid
                ]

                for nid in target_ids
            ]
        )


        source_predicted_target = (
            source_pos
            +
            float(
                delta_t
            )
            *
            source_vel
        )


        target_predicted_source = (
            target_pos
            -
            float(
                delta_t
            )
            *
            target_vel
        )


        target_tree = cKDTree(
            target_pos
        )


        source_tree = cKDTree(
            source_pos
        )


        (
            _,
            source_idx,
        ) = query_tree(
            target_tree,
            source_predicted_target,
            len(
                target_ids
            ),
            MAX_RANK_K,
        )


        (
            _,
            target_idx,
        ) = query_tree(
            source_tree,
            target_predicted_source,
            len(
                source_ids
            ),
            MAX_RANK_K,
        )


        source_position = {
            int(
                nid
            ):
                i

            for i, nid
            in enumerate(
                source_ids
            )
        }


        target_position = {
            int(
                nid
            ):
                i

            for i, nid
            in enumerate(
                target_ids
            )
        }


        for key in keys:

            row = local[
                key
            ]


            oracle_row = ds_oracle[
                ds_oracle[
                    "component_id"
                ].eq(
                    key[
                        1
                    ]
                )
            ].iloc[
                0
            ]


            src = int(
                oracle_row[
                    "source_id"
                ]
            )


            tgt = int(
                oracle_row[
                    "target_id"
                ]
            )


            src_i = source_position[
                src
            ]


            tgt_i = target_position[
                tgt
            ]


            source_rank = np.nan


            for rank_i, candidate_idx in enumerate(
                source_idx[
                    src_i
                ]
            ):

                candidate_id = int(
                    target_ids[
                        int(
                            candidate_idx
                        )
                    ]
                )


                if candidate_id == tgt:

                    source_rank = int(
                        rank_i
                        +
                        1
                    )

                    break


            target_rank = np.nan


            for rank_i, candidate_idx in enumerate(
                target_idx[
                    tgt_i
                ]
            ):

                candidate_id = int(
                    source_ids[
                        int(
                            candidate_idx
                        )
                    ]
                )


                if candidate_id == src:

                    target_rank = int(
                        rank_i
                        +
                        1
                    )

                    break


            row[
                "exact_source_rank_k16"
            ] = source_rank


            row[
                "exact_target_rank_k16"
            ] = target_rank


        del source_pos
        del target_pos

        del source_vel
        del target_vel

        del source_predicted_target
        del target_predicted_source

        del target_tree
        del source_tree

        del source_idx
        del target_idx


    oracle_rank_rows.extend(
        local.values()
    )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            oracle_datasets
        )
    ):

        print(
            f"{ds_i:2d}/{len(oracle_datasets)}"
            f" | {dataset}"
            f" | oracle={len(ds_oracle)}"
        )


    del graph
    del nodes
    del edges

    del incoming
    del outgoing

    del indeg
    del outdeg

    del phys
    del time_map

    del node_ids

    del past_velocity
    del future_velocity

    del source_eligible
    del target_eligible

    del source_eligible_set
    del target_eligible_set

    del source_by_t
    del target_by_t

    del local
    del frame_groups

    gc.collect()


# ============================================================
# 9. Attach exact K16 ranks
# ============================================================

rank_table = pd.DataFrame(
    oracle_rank_rows
)


assert len(
    rank_table
) == 369


oracle = oracle.merge(
    rank_table,
    on=[
        "dataset",
        "component_id",
    ],
    how="left",
    validate="one_to_one",
)


# ============================================================
# 10. Contract oracle masks
# ============================================================

print(
    "\n========== ORACLE RETRIEVAL CONTRACT COVERAGE =========="
)


oracle_rows = []


processable = (
    oracle[
        "within_horizon"
    ].astype(bool)
    &
    oracle[
        "source_eligible"
    ].astype(bool)
    &
    oracle[
        "target_eligible"
    ].astype(bool)
    &
    ~oracle[
        "pair_existing_edge"
    ].astype(bool)
)


processable_total = int(
    processable.sum()
)


processable_full7 = (
    processable
    &
    oracle[
        "pred_full_7um"
    ].astype(bool)
)


processable_full7_total = int(
    processable_full7.sum()
)


for contract_name in CONTRACTS:

    if contract_name == "BASE_EXACT_5x5":

        retrieval_pass = (
            processable
            &
            oracle[
                "exact_reciprocal_top5"
            ].astype(bool)
        )

    else:

        spec = CONTRACTS[
            contract_name
        ]


        contract_type = spec[
            "type"
        ]


        a = int(
            spec[
                "a"
            ]
        )


        b = int(
            spec[
                "b"
            ]
        )


        source_rank = oracle[
            "exact_source_rank_k16"
        ]


        target_rank = oracle[
            "exact_target_rank_k16"
        ]


        if contract_type == "RECIP":

            rank_pass = (
                source_rank.le(
                    a
                )
                &
                target_rank.le(
                    b
                )
            )


        elif contract_type == "ASYM":

            rank_pass = (
                (
                    source_rank.le(
                        a
                    )
                    &
                    target_rank.le(
                        b
                    )
                )
                |
                (
                    target_rank.le(
                        a
                    )
                    &
                    source_rank.le(
                        b
                    )
                )
            )


        else:

            raise ValueError(
                spec
            )


        retrieval_pass = (
            processable
            &
            rank_pass.fillna(
                False
            )
        )


    # --------------------------------------------------------
    # Retrieval-only coverage
    # --------------------------------------------------------

    retrieval_n = int(
        retrieval_pass.sum()
    )


    retrieval_full7_n = int(
        (
            retrieval_pass
            &
            oracle[
                "pred_full_7um"
            ].astype(bool)
        ).sum()
    )


    # --------------------------------------------------------
    # Combined with CURRENT dominance-any
    # --------------------------------------------------------

    combined_pass = (
        retrieval_pass
        &
        oracle[
            "dominance_any"
        ].astype(bool)
    )


    combined_n = int(
        combined_pass.sum()
    )


    combined_full7_n = int(
        (
            combined_pass
            &
            oracle[
                "pred_full_7um"
            ].astype(bool)
        ).sum()
    )


    raw_rows = int(
        counts[
            contract_name
        ][
            "rows"
        ]
    )


    ratio = float(
        raw_rows
        /
        base_raw_rows
    )


    oracle_rows.append(
        {
            "contract":
                contract_name,

            "raw_candidate_rows":
                raw_rows,

            "raw_ratio_vs_exact5":
                ratio,

            "processable_total":
                processable_total,

            "retrieval_captured":
                retrieval_n,

            "retrieval_coverage":
                float(
                    retrieval_n
                    /
                    processable_total
                ),

            "processable_full7_total":
                processable_full7_total,

            "retrieval_full7_captured":
                retrieval_full7_n,

            "retrieval_full7_coverage":
                float(
                    retrieval_full7_n
                    /
                    processable_full7_total
                ),

            "combined_captured":
                combined_n,

            "combined_overall_coverage":
                float(
                    combined_n
                    /
                    len(
                        oracle
                    )
                ),

            "combined_full7_captured":
                combined_full7_n,

            "combined_full7_overall_coverage":
                float(
                    combined_full7_n
                    /
                    int(
                        oracle[
                            "pred_full_7um"
                        ].sum()
                    )
                ),
        }
    )


oracle_contract_table = pd.DataFrame(
    oracle_rows
)


oracle_contract_table = (
    oracle_contract_table
    .sort_values(
        [
            "raw_candidate_rows",
            "contract",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


print(
    oracle_contract_table.to_string(
        index=False
    )
)


# ============================================================
# 11. Relative efficiency table
#
# Oracle benefit per cardinality multiplier.
#
# DIAGNOSTIC ONLY.
# ============================================================

baseline_row = (
    oracle_contract_table[
        oracle_contract_table[
            "contract"
        ].eq(
            "BASE_EXACT_5x5"
        )
    ]
    .iloc[
        0
    ]
)


base_retrieval = int(
    baseline_row[
        "retrieval_captured"
    ]
)


base_combined = int(
    baseline_row[
        "combined_captured"
    ]
)


efficiency = (
    oracle_contract_table
    .copy()
)


efficiency[
    "extra_retrieval_vs_base"
] = (
    efficiency[
        "retrieval_captured"
    ]
    -
    base_retrieval
)


efficiency[
    "extra_combined_vs_base"
] = (
    efficiency[
        "combined_captured"
    ]
    -
    base_combined
)


efficiency[
    "candidate_multiplier_minus_1"
] = (
    efficiency[
        "raw_ratio_vs_exact5"
    ]
    -
    1.0
)


print(
    "\n========== RETRIEVAL CARDINALITY / COVERAGE TRADEOFF =========="
)


print(
    efficiency[
        [
            "contract",

            "raw_candidate_rows",
            "raw_ratio_vs_exact5",

            "retrieval_captured",
            "extra_retrieval_vs_base",

            "combined_captured",
            "extra_combined_vs_base",

            "retrieval_full7_captured",
            "combined_full7_captured",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 12. Coverage by topology for each contract
# ============================================================

print(
    "\n========== CONTRACT COVERAGE BY TOPOLOGY =========="
)


topology_rows = []


for contract_name in CONTRACTS:

    if contract_name == "BASE_EXACT_5x5":

        retrieval_pass = (
            processable
            &
            oracle[
                "exact_reciprocal_top5"
            ].astype(bool)
        )

    else:

        spec = CONTRACTS[
            contract_name
        ]


        source_rank = oracle[
            "exact_source_rank_k16"
        ]


        target_rank = oracle[
            "exact_target_rank_k16"
        ]


        if spec[
            "type"
        ] == "RECIP":

            rank_pass = (
                source_rank.le(
                    int(
                        spec[
                            "a"
                        ]
                    )
                )
                &
                target_rank.le(
                    int(
                        spec[
                            "b"
                        ]
                    )
                )
            )


        else:

            a = int(
                spec[
                    "a"
                ]
            )

            b = int(
                spec[
                    "b"
                ]
            )


            rank_pass = (
                (
                    source_rank.le(
                        a
                    )
                    &
                    target_rank.le(
                        b
                    )
                )
                |
                (
                    target_rank.le(
                        a
                    )
                    &
                    source_rank.le(
                        b
                    )
                )
            )


        retrieval_pass = (
            processable
            &
            rank_pass.fillna(
                False
            )
        )


    combined_pass = (
        retrieval_pass
        &
        oracle[
            "dominance_any"
        ].astype(bool)
    )


    for topology, sub_index in oracle.groupby(
        "topology_class"
    ).groups.items():

        sub_index = list(
            sub_index
        )


        total = len(
            sub_index
        )


        retrieval_n = int(
            retrieval_pass.loc[
                sub_index
            ].sum()
        )


        combined_n = int(
            combined_pass.loc[
                sub_index
            ].sum()
        )


        topology_rows.append(
            {
                "contract":
                    contract_name,

                "topology":
                    topology,

                "total":
                    int(
                        total
                    ),

                "retrieval_captured":
                    retrieval_n,

                "combined_captured":
                    combined_n,
            }
        )


topology_table = pd.DataFrame(
    topology_rows
)


for contract_name in CONTRACTS:

    print(
        "\n" + contract_name
    )


    print(
        topology_table[
            topology_table[
                "contract"
            ].eq(
                contract_name
            )
        ][
            [
                "topology",
                "total",
                "retrieval_captured",
                "combined_captured",
            ]
        ]
        .to_string(
            index=False
        )
    )


# ============================================================
# 13. Persist oracle audit
# ============================================================

for contract_name in CONTRACTS:

    if contract_name == "BASE_EXACT_5x5":

        mask = (
            processable
            &
            oracle[
                "exact_reciprocal_top5"
            ].astype(bool)
        )

    else:

        spec = CONTRACTS[
            contract_name
        ]


        sr = oracle[
            "exact_source_rank_k16"
        ]

        tr = oracle[
            "exact_target_rank_k16"
        ]


        if spec[
            "type"
        ] == "RECIP":

            mask = (
                processable
                &
                sr.le(
                    int(
                        spec[
                            "a"
                        ]
                    )
                )
                &
                tr.le(
                    int(
                        spec[
                            "b"
                        ]
                    )
                )
            )


        else:

            a = int(
                spec[
                    "a"
                ]
            )

            b = int(
                spec[
                    "b"
                ]
            )


            mask = (
                processable
                &
                (
                    (
                        sr.le(
                            a
                        )
                        &
                        tr.le(
                            b
                        )
                    )
                    |
                    (
                        tr.le(
                            a
                        )
                        &
                        sr.le(
                            b
                        )
                    )
                )
            )


        mask = mask.fillna(
            False
        )


    oracle[
        f"retrieval_{contract_name}"
    ] = mask.astype(
        bool
    )


    oracle[
        f"combined_{contract_name}"
    ] = (
        mask.astype(bool)
        &
        oracle[
            "dominance_any"
        ].astype(bool)
    )


oracle = (
    oracle
    .sort_values(
        [
            "fold",
            "dataset",
            "component_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


oracle.to_pickle(
    ORACLE_OUT
)


oracle_summary = {
    "stage":
        "12.11G3F1",

    "status":
        "ASYMMETRIC_RETRIEVAL_CONTRACT_CARDINALITY_AND_ORACLE_AUDITED",

    "contract_cardinality_sha256":
        contract_sha,

    "gt_blind_cardinality_frozen_before_oracle":
        True,

    "contracts":
        CONTRACTS,

    "contract_results":
        oracle_contract_table.to_dict(
            "records"
        ),

    "clean_components":
        369,

    "processable_components":
        processable_total,

    "no_new_proposal_universe":
        True,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


ORACLE_SUMMARY_OUT.write_text(
    json.dumps(
        oracle_summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert ORACLE_OUT.exists()
assert ORACLE_SUMMARY_OUT.exists()


# ============================================================
# 14. Integrity
# ============================================================

check = pd.read_pickle(
    ORACLE_OUT
)


assert len(
    check
) == 369


assert not check[
    [
        "dataset",
        "component_id",
    ]
].duplicated().any()


# Baseline retrieval should exactly match G3F0-R exact K5.
assert (
    check[
        "retrieval_BASE_EXACT_5x5"
    ]
    ==
    (
        check[
            "within_horizon"
        ].astype(bool)
        &
        check[
            "source_eligible"
        ].astype(bool)
        &
        check[
            "target_eligible"
        ].astype(bool)
        &
        ~check[
            "pair_existing_edge"
        ].astype(bool)
        &
        check[
            "exact_reciprocal_top5"
        ].astype(bool)
    )
).all()


assert int(
    check[
        "combined_BASE_EXACT_5x5"
    ].sum()
) == 185


# ============================================================
# 15. Decision
# ============================================================

print(
    "\n========== 12.11G3F1 DECISION =========="
)

print(
    "ASYMMETRIC_RETRIEVAL_CONTRACT_AUDIT: PASS"
)

print(
    "GT-blind cardinality frozen before oracle:",
    "YES"
)

print(
    "new proposal universe generated:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "classifier trained:",
    "NO"
)

print(
    "threshold selected:",
    "NO"
)

print(
    "policy selected:",
    "NO"
)

print(
    "graph modified:",
    "NO"
)

print(
    "next:"
)

print(
    "SELECT_LOW_CARDINALITY_RETRIEVAL_FRONTIER_THEN_FREEZE_GENERATOR"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3F2
#
# GT-Blind Post-Dominance Cardinality Audit
# for the Low-Cardinality Retrieval Frontier
#
# G3F1 frontier retained for further study:
#
#   BASE_EXACT_5x5
#   ASYM_3x8
#   ASYM_3x12
#   ASYM_3x16
#
# Why:
#
#   - ASYM_5x8 is dominated by ASYM_3x12
#   - RECIP_8x8 is dominated by ASYM_3x16
#   - larger contracts show rapidly diminishing returns
#
# IMPORTANT:
#
#   This shortlist is DEVELOPMENT-GT-INFORMED from G3F1.
#
#   However, every actual generator feature and pruning operation
#   evaluated in Part A below is GT-blind.
#
# PART A:
#
#   Scan all 103 development datasets.
#
#   For each frontier contract:
#
#       raw retrieval cardinality
#           ->
#       incumbent-relative dominance-any
#           ->
#       final retained cardinality
#
#   NO proposal rows are persisted.
#
#   Freeze post-dominance cardinality artifact.
#
# PART B:
#
#   Only after Part A artifact freeze:
#       read G3F1 oracle artifact
#       attach known development oracle coverage
#       construct engineering tradeoff table
#
# No final generator selected in this cell.
# No Fold0.
# No classifier.
# No policy.
# No graph modification.
#
# Keep after running: YES
# ============================================================

from pathlib import Path
import gc
import hashlib
import json

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


G3F1_CARDINALITY_PATH = (
    S12
    / "stage12_folds14_multiframe_asymmetric_retrieval_cardinality_v1.json"
)


# GT-derived development oracle.
# DO NOT READ until Part A has been frozen.
G3F1_ORACLE_PATH = (
    S12
    / "stage12_folds14_multiframe_asymmetric_retrieval_oracle_v1.pkl"
)


OUT = (
    S12
    / "stage12_folds14_multiframe_postdominance_frontier_v1.json"
)


SUMMARY_OUT = (
    S12
    / "stage12_folds14_multiframe_postdominance_frontier_oracle_v1.json"
)


assert G3F1_CARDINALITY_PATH.exists(), G3F1_CARDINALITY_PATH
assert G3F1_ORACLE_PATH.exists(), G3F1_ORACLE_PATH


# ============================================================
# 1. CURRENT_BEST contract
# ============================================================

assert (
    "build_current_best"
    in globals()
    and
    callable(
        globals()[
            "build_current_best"
        ]
    )
), (
    "build_current_best is missing. "
    "Re-run retained successful G3E-R before G3F2."
)


current_best_builder = globals()[
    "build_current_best"
]


# ============================================================
# 2. Development corpus
# ============================================================

if (
    "names"
    in globals()
    and
    len(
        globals()[
            "names"
        ]
    ) == 103
):

    development_names = list(
        globals()[
            "names"
        ]
    )

else:

    CV_PATH = (
        PROJECT
        / "configs"
        / "cv_manifest.csv"
    )


    cv = pd.read_csv(
        CV_PATH
    )


    dev = (
        cv[
            cv[
                "fold"
            ].isin(
                [
                    1,
                    2,
                    3,
                    4,
                ]
            )
            &
            cv[
                "dataset"
            ].str.startswith(
                "6bba_"
            )
        ]
        .sort_values(
            [
                "fold",
                "dataset",
            ],
            kind="stable",
        )
    )


    development_names = dev[
        "dataset"
    ].tolist()


assert len(
    development_names
) == 103


if "fold_map" in globals():

    fold_map_local = dict(
        globals()[
            "fold_map"
        ]
    )

else:

    CV_PATH = (
        PROJECT
        / "configs"
        / "cv_manifest.csv"
    )


    cv = pd.read_csv(
        CV_PATH
    )


    dev = cv[
        cv[
            "dataset"
        ].isin(
            development_names
        )
    ]


    fold_map_local = dict(
        zip(
            dev[
                "dataset"
            ],
            dev[
                "fold"
            ].astype(int),
        )
    )


# ============================================================
# 3. Fixed geometry / retrieval contract
# ============================================================

SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


MIN_GAP_SIZE = 2
MAX_GAP_SIZE = 12

BASE_K = 5
ASYM_QUERY_K = 16


# ============================================================
# 4. DEVELOPMENT-INFORMED frontier
#
# No further oracle read occurs before Part A freeze.
# ============================================================

FRONTIER = {
    "BASE_EXACT_5x5": {
        "type":
            "BASE",

        "a":
            5,

        "b":
            5,
    },

    "ASYM_3x8": {
        "type":
            "ASYM",

        "a":
            3,

        "b":
            8,
    },

    "ASYM_3x12": {
        "type":
            "ASYM",

        "a":
            3,

        "b":
            12,
    },

    "ASYM_3x16": {
        "type":
            "ASYM",

        "a":
            3,

        "b":
            16,
    },
}


print(
    "========== 12.11G3F2 FRONTIER CONTRACT =========="
)

print(
    "development datasets:",
    len(
        development_names
    )
)

print(
    "gap horizon:",
    MIN_GAP_SIZE,
    "to",
    MAX_GAP_SIZE
)

print(
    "frontier:"
)


for contract, spec in FRONTIER.items():

    print(
        contract,
        spec
    )


# ============================================================
# 5. Load prior GT-BLIND G3F1 cardinality only
#
# Safe:
# This artifact was created before oracle read.
# ============================================================

prior_cardinality = json.loads(
    G3F1_CARDINALITY_PATH.read_text(
        encoding="utf-8"
    )
)


prior_counts = prior_cardinality[
    "counts"
]


for contract in FRONTIER:

    assert contract in prior_counts


# ============================================================
# 6. Helpers
# ============================================================

def vec_norm(
    value,
):

    return float(
        np.linalg.norm(
            np.asarray(
                value,
                dtype=float,
            )
        )
    )


def query_knn(
    tree,
    query_points,
    candidate_count,
    k,
):

    k_eff = min(
        int(
            k
        ),
        int(
            candidate_count
        ),
    )


    distances, indices = tree.query(
        query_points,
        k=k_eff,
    )


    if k_eff == 1:

        distances = distances[
            :,
            None
        ]

        indices = indices[
            :,
            None
        ]


    return (
        np.asarray(
            distances,
            dtype=float,
        ),

        np.asarray(
            indices,
            dtype=int,
        ),
    )


def rank_distance_map(
    query_ids,
    candidate_ids,
    distances,
    indices,
    reverse=False,
):

    result = {}


    for query_i, query_id in enumerate(
        query_ids
    ):

        for rank_i in range(
            indices.shape[
                1
            ]
        ):

            candidate_id = int(
                candidate_ids[
                    int(
                        indices[
                            query_i,
                            rank_i
                        ]
                    )
                ]
            )


            if reverse:

                pair = (
                    candidate_id,
                    int(
                        query_id
                    ),
                )

            else:

                pair = (
                    int(
                        query_id
                    ),
                    candidate_id,
                )


            if pair not in result:

                result[
                    pair
                ] = (
                    int(
                        rank_i
                        +
                        1
                    ),

                    float(
                        distances[
                            query_i,
                            rank_i
                        ]
                    ),
                )


    return result


def asym_pass(
    source_rank,
    target_rank,
    a,
    b,
):

    return bool(
        (
            source_rank <= a
            and
            target_rank <= b
        )
        or
        (
            target_rank <= a
            and
            source_rank <= b
        )
    )


# ============================================================
# 7. Global counters
# ============================================================

stats = {}


for contract in FRONTIER:

    stats[
        contract
    ] = {
        "raw_rows":
            0,

        "retained_rows":
            0,

        "raw_cuts_0":
            0,

        "raw_cuts_1":
            0,

        "raw_cuts_2":
            0,

        "retained_cuts_0":
            0,

        "retained_cuts_1":
            0,

        "retained_cuts_2":
            0,

        "folds": {
            "1": {
                "raw":
                    0,

                "retained":
                    0,
            },

            "2": {
                "raw":
                    0,

                "retained":
                    0,
            },

            "3": {
                "raw":
                    0,

                "retained":
                    0,
            },

            "4": {
                "raw":
                    0,

                "retained":
                    0,
            },
        },
    }


dataset_stats = []


# ============================================================
# 8. PART A
# GT-BLIND post-dominance cardinality
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART A — GT-BLIND POST-DOMINANCE CARDINALITY"
)

print(
    "============================================================"
)


for ds_i, dataset in enumerate(
    development_names,
    start=1,
):

    fold = int(
        fold_map_local[
            dataset
        ]
    )


    graph, scale = current_best_builder(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    )


    # ========================================================
    # Nodes
    # ========================================================

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = (
        nodes[
            "node_id"
        ]
        .astype(int)
    )


    nodes[
        "t"
    ] = (
        nodes[
            "t"
        ]
        .astype(int)
    )


    nodes = (
        nodes
        .sort_values(
            [
                "t",
                "node_id",
            ],
            kind="stable",
        )
        .set_index(
            "node_id",
            drop=False,
        )
    )


    # ========================================================
    # Edges / topology
    # ========================================================

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = (
        edges[
            "source_id"
        ]
        .astype(int)
    )


    edges[
        "target_id"
    ] = (
        edges[
            "target_id"
        ]
        .astype(int)
    )


    edge_set = set(
        zip(
            edges[
                "source_id"
            ],
            edges[
                "target_id"
            ],
        )
    )


    incoming = (
        edges.groupby(
            "target_id"
        )[
            "source_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    outgoing = (
        edges.groupby(
            "source_id"
        )[
            "target_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = {
        int(k):
            int(v)

        for k, v in indeg.items()
    }


    outdeg = {
        int(k):
            int(v)

        for k, v in outdeg.items()
    }


    # ========================================================
    # Physical coordinates
    # ========================================================

    phys = {}

    time_map = {}


    for nid, row in nodes.iterrows():

        nid = int(
            nid
        )


        phys[
            nid
        ] = (
            np.array(
                [
                    float(
                        row[
                            "z"
                        ]
                    ),

                    float(
                        row[
                            "y"
                        ]
                    ),

                    float(
                        row[
                            "x"
                        ]
                    ),
                ],
                dtype=float,
            )
            *
            SCALE
        )


        time_map[
            nid
        ] = int(
            row[
                "t"
            ]
        )


    node_ids = sorted(
        phys
    )


    # ========================================================
    # Past / future velocities
    # ========================================================

    past_velocity = {}

    future_velocity = {}


    for nid in node_ids:

        inc = incoming.get(
            nid,
            [],
        )


        if len(
            inc
        ) == 1:

            prev_id = int(
                inc[
                    0
                ]
            )


            if prev_id in phys:

                dt = (
                    time_map[
                        nid
                    ]
                    -
                    time_map[
                        prev_id
                    ]
                )


                if dt > 0:

                    past_velocity[
                        nid
                    ] = (
                        phys[
                            nid
                        ]
                        -
                        phys[
                            prev_id
                        ]
                    ) / float(
                        dt
                    )


        out = outgoing.get(
            nid,
            [],
        )


        if len(
            out
        ) == 1:

            next_id = int(
                out[
                    0
                ]
            )


            if next_id in phys:

                dt = (
                    time_map[
                        next_id
                    ]
                    -
                    time_map[
                        nid
                    ]
                )


                if dt > 0:

                    future_velocity[
                        nid
                    ] = (
                        phys[
                            next_id
                        ]
                        -
                        phys[
                            nid
                        ]
                    ) / float(
                        dt
                    )


    # ========================================================
    # Endpoint eligibility
    # ========================================================

    source_eligible = [
        nid

        for nid in node_ids

        if (
            outdeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid
            in
            past_velocity
        )
    ]


    target_eligible = [
        nid

        for nid in node_ids

        if (
            indeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid
            in
            future_velocity
        )
    ]


    source_by_t = {}

    target_by_t = {}


    for nid in source_eligible:

        source_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for nid in target_eligible:

        target_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for t in source_by_t:

        source_by_t[
            t
        ] = sorted(
            source_by_t[
                t
            ]
        )


    for t in target_by_t:

        target_by_t[
            t
        ] = sorted(
            target_by_t[
                t
            ]
        )


    # ========================================================
    # Incumbent normalized CV costs
    # ========================================================

    source_incumbent_cost = {}

    target_incumbent_cost = {}


    for src in source_eligible:

        if outdeg.get(
            src,
            0,
        ) != 1:

            continue


        old_targets = outgoing.get(
            src,
            [],
        )


        if len(
            old_targets
        ) != 1:

            continue


        old_tgt = int(
            old_targets[
                0
            ]
        )


        if old_tgt not in phys:

            continue


        dt_old = (
            time_map[
                old_tgt
            ]
            -
            time_map[
                src
            ]
        )


        if dt_old <= 0:

            continue


        predicted_target = (
            phys[
                src
            ]
            +
            float(
                dt_old
            )
            *
            past_velocity[
                src
            ]
        )


        source_incumbent_cost[
            src
        ] = (
            vec_norm(
                phys[
                    old_tgt
                ]
                -
                predicted_target
            )
            /
            float(
                dt_old
            )
        )


    for tgt in target_eligible:

        if indeg.get(
            tgt,
            0,
        ) != 1:

            continue


        old_sources = incoming.get(
            tgt,
            [],
        )


        if len(
            old_sources
        ) != 1:

            continue


        old_src = int(
            old_sources[
                0
            ]
        )


        if old_src not in phys:

            continue


        dt_old = (
            time_map[
                tgt
            ]
            -
            time_map[
                old_src
            ]
        )


        if dt_old <= 0:

            continue


        predicted_source = (
            phys[
                tgt
            ]
            -
            float(
                dt_old
            )
            *
            future_velocity[
                tgt
            ]
        )


        target_incumbent_cost[
            tgt
        ] = (
            vec_norm(
                phys[
                    old_src
                ]
                -
                predicted_source
            )
            /
            float(
                dt_old
            )
        )


    # ========================================================
    # Dataset counters
    # ========================================================

    local = {}


    for contract in FRONTIER:

        local[
            contract
        ] = {
            "raw":
                0,

            "retained":
                0,

            "raw_cuts_0":
                0,

            "raw_cuts_1":
                0,

            "raw_cuts_2":
                0,

            "retained_cuts_0":
                0,

            "retained_cuts_1":
                0,

            "retained_cuts_2":
                0,
        }


    # ========================================================
    # Proposal dominance helper
    # ========================================================

    def evaluate_pair(
        src,
        tgt,
        delta_t,
    ):

        source_degree = int(
            outdeg.get(
                src,
                0,
            )
        )


        target_degree = int(
            indeg.get(
                tgt,
                0,
            )
        )


        assert source_degree <= 1
        assert target_degree <= 1


        cuts = (
            int(
                source_degree == 1
            )
            +
            int(
                target_degree == 1
            )
        )


        # ----------------------------------------------------
        # Free/free:
        # dominance is automatically satisfied.
        # ----------------------------------------------------

        if cuts == 0:

            return (
                cuts,
                True,
            )


        # ----------------------------------------------------
        # Candidate source normalized CV
        # ----------------------------------------------------

        source_prediction = (
            phys[
                src
            ]
            +
            float(
                delta_t
            )
            *
            past_velocity[
                src
            ]
        )


        source_cv_pf = (
            vec_norm(
                phys[
                    tgt
                ]
                -
                source_prediction
            )
            /
            float(
                delta_t
            )
        )


        # ----------------------------------------------------
        # Candidate target normalized CV
        # ----------------------------------------------------

        target_prediction = (
            phys[
                tgt
            ]
            -
            float(
                delta_t
            )
            *
            future_velocity[
                tgt
            ]
        )


        target_cv_pf = (
            vec_norm(
                phys[
                    src
                ]
                -
                target_prediction
            )
            /
            float(
                delta_t
            )
        )


        required = []


        if source_degree == 1:

            incumbent = float(
                source_incumbent_cost.get(
                    src,
                    np.nan,
                )
            )


            if np.isfinite(
                incumbent
            ):

                required.append(
                    incumbent
                    -
                    source_cv_pf
                )

            else:

                required.append(
                    np.nan
                )


        if target_degree == 1:

            incumbent = float(
                target_incumbent_cost.get(
                    tgt,
                    np.nan,
                )
            )


            if np.isfinite(
                incumbent
            ):

                required.append(
                    incumbent
                    -
                    target_cv_pf
                )

            else:

                required.append(
                    np.nan
                )


        finite_required = [
            value

            for value in required

            if np.isfinite(
                value
            )
        ]


        if (
            len(
                finite_required
            )
            !=
            len(
                required
            )
        ):

            return (
                cuts,
                False,
            )


        dominance_any = any(
            value > 0

            for value in finite_required
        )


        return (
            cuts,
            bool(
                dominance_any
            ),
        )


    # ========================================================
    # Frame-pair enumeration
    # ========================================================

    for gap_size in range(
        MIN_GAP_SIZE,
        MAX_GAP_SIZE + 1,
    ):

        delta_t = (
            gap_size
            +
            1
        )


        for source_t in sorted(
            source_by_t
        ):

            target_t = (
                source_t
                +
                delta_t
            )


            if target_t not in target_by_t:

                continue


            source_ids = source_by_t[
                source_t
            ]


            target_ids = target_by_t[
                target_t
            ]


            if (
                len(
                    source_ids
                ) == 0
                or
                len(
                    target_ids
                ) == 0
            ):

                continue


            source_pos = np.vstack(
                [
                    phys[
                        nid
                    ]

                    for nid in source_ids
                ]
            )


            target_pos = np.vstack(
                [
                    phys[
                        nid
                    ]

                    for nid in target_ids
                ]
            )


            source_vel = np.vstack(
                [
                    past_velocity[
                        nid
                    ]

                    for nid in source_ids
                ]
            )


            target_vel = np.vstack(
                [
                    future_velocity[
                        nid
                    ]

                    for nid in target_ids
                ]
            )


            source_pred = (
                source_pos
                +
                float(
                    delta_t
                )
                *
                source_vel
            )


            target_pred = (
                target_pos
                -
                float(
                    delta_t
                )
                *
                target_vel
            )


            target_tree = cKDTree(
                target_pos
            )


            source_tree = cKDTree(
                source_pos
            )


            # =================================================
            # BASE EXACT K=5
            # =================================================

            (
                src_dist5,
                src_idx5,
            ) = query_knn(
                target_tree,
                source_pred,
                len(
                    target_ids
                ),
                BASE_K,
            )


            (
                tgt_dist5,
                tgt_idx5,
            ) = query_knn(
                source_tree,
                target_pred,
                len(
                    source_ids
                ),
                BASE_K,
            )


            src_map5 = rank_distance_map(
                source_ids,
                target_ids,
                src_dist5,
                src_idx5,
                reverse=False,
            )


            tgt_map5 = rank_distance_map(
                target_ids,
                source_ids,
                tgt_dist5,
                tgt_idx5,
                reverse=True,
            )


            base_pairs = (
                set(
                    src_map5
                )
                &
                set(
                    tgt_map5
                )
            )


            for (
                src,
                tgt,
            ) in base_pairs:

                if (
                    src,
                    tgt,
                ) in edge_set:

                    continue


                (
                    cuts,
                    dominance,
                ) = evaluate_pair(
                    src,
                    tgt,
                    delta_t,
                )


                local[
                    "BASE_EXACT_5x5"
                ][
                    "raw"
                ] += 1


                local[
                    "BASE_EXACT_5x5"
                ][
                    f"raw_cuts_{cuts}"
                ] += 1


                if dominance:

                    local[
                        "BASE_EXACT_5x5"
                    ][
                        "retained"
                    ] += 1


                    local[
                        "BASE_EXACT_5x5"
                    ][
                        f"retained_cuts_{cuts}"
                    ] += 1


            # =================================================
            # Exact K16 rank universe for asymmetric contracts
            # =================================================

            (
                src_dist16,
                src_idx16,
            ) = query_knn(
                target_tree,
                source_pred,
                len(
                    target_ids
                ),
                ASYM_QUERY_K,
            )


            (
                tgt_dist16,
                tgt_idx16,
            ) = query_knn(
                source_tree,
                target_pred,
                len(
                    source_ids
                ),
                ASYM_QUERY_K,
            )


            src_map16 = rank_distance_map(
                source_ids,
                target_ids,
                src_dist16,
                src_idx16,
                reverse=False,
            )


            tgt_map16 = rank_distance_map(
                target_ids,
                source_ids,
                tgt_dist16,
                tgt_idx16,
                reverse=True,
            )


            mutual16 = (
                set(
                    src_map16
                )
                &
                set(
                    tgt_map16
                )
            )


            for (
                src,
                tgt,
            ) in mutual16:

                if (
                    src,
                    tgt,
                ) in edge_set:

                    continue


                source_rank = int(
                    src_map16[
                        (
                            src,
                            tgt,
                        )
                    ][
                        0
                    ]
                )


                target_rank = int(
                    tgt_map16[
                        (
                            src,
                            tgt,
                        )
                    ][
                        0
                    ]
                )


                passed_contracts = []


                for contract in [
                    "ASYM_3x8",
                    "ASYM_3x12",
                    "ASYM_3x16",
                ]:

                    spec = FRONTIER[
                        contract
                    ]


                    if asym_pass(
                        source_rank,
                        target_rank,
                        int(
                            spec[
                                "a"
                            ]
                        ),
                        int(
                            spec[
                                "b"
                            ]
                        ),
                    ):

                        passed_contracts.append(
                            contract
                        )


                if not passed_contracts:

                    continue


                (
                    cuts,
                    dominance,
                ) = evaluate_pair(
                    src,
                    tgt,
                    delta_t,
                )


                for contract in passed_contracts:

                    local[
                        contract
                    ][
                        "raw"
                    ] += 1


                    local[
                        contract
                    ][
                        f"raw_cuts_{cuts}"
                    ] += 1


                    if dominance:

                        local[
                            contract
                        ][
                            "retained"
                        ] += 1


                        local[
                            contract
                        ][
                            f"retained_cuts_{cuts}"
                        ] += 1


            # =================================================
            # Frame cleanup
            # =================================================

            del source_pos
            del target_pos

            del source_vel
            del target_vel

            del source_pred
            del target_pred

            del target_tree
            del source_tree

            del src_dist5
            del src_idx5

            del tgt_dist5
            del tgt_idx5

            del src_map5
            del tgt_map5

            del base_pairs

            del src_dist16
            del src_idx16

            del tgt_dist16
            del tgt_idx16

            del src_map16
            del tgt_map16

            del mutual16


    # ========================================================
    # Dataset accounting
    # ========================================================

    dataset_record = {
        "dataset":
            dataset,

        "fold":
            fold,

        "contracts":
            {},
    }


    for contract in FRONTIER:

        entry = local[
            contract
        ]


        assert (
            entry[
                "raw_cuts_0"
            ]
            +
            entry[
                "raw_cuts_1"
            ]
            +
            entry[
                "raw_cuts_2"
            ]
        ) == entry[
            "raw"
        ]


        assert (
            entry[
                "retained_cuts_0"
            ]
            +
            entry[
                "retained_cuts_1"
            ]
            +
            entry[
                "retained_cuts_2"
            ]
        ) == entry[
            "retained"
        ]


        assert (
            entry[
                "retained"
            ]
            <=
            entry[
                "raw"
            ]
        )


        stats[
            contract
        ][
            "raw_rows"
        ] += int(
            entry[
                "raw"
            ]
        )


        stats[
            contract
        ][
            "retained_rows"
        ] += int(
            entry[
                "retained"
            ]
        )


        for cuts in [
            0,
            1,
            2,
        ]:

            stats[
                contract
            ][
                f"raw_cuts_{cuts}"
            ] += int(
                entry[
                    f"raw_cuts_{cuts}"
                ]
            )


            stats[
                contract
            ][
                f"retained_cuts_{cuts}"
            ] += int(
                entry[
                    f"retained_cuts_{cuts}"
                ]
            )


        stats[
            contract
        ][
            "folds"
        ][
            str(
                fold
            )
        ][
            "raw"
        ] += int(
            entry[
                "raw"
            ]
        )


        stats[
            contract
        ][
            "folds"
        ][
            str(
                fold
            )
        ][
            "retained"
        ] += int(
            entry[
                "retained"
            ]
        )


        dataset_record[
            "contracts"
        ][
            contract
        ] = {
            key:
                int(
                    value
                )

            for (
                key,
                value
            )
            in entry.items()
        }


    dataset_stats.append(
        dataset_record
    )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            development_names
        )
    ):

        print(
            f"{ds_i:3d}/{len(development_names)}"
            f" | fold={fold}"
            f" | {dataset}"
            f" | base={local['BASE_EXACT_5x5']['retained']}"
            f" | a3x8={local['ASYM_3x8']['retained']}"
            f" | a3x12={local['ASYM_3x12']['retained']}"
            f" | a3x16={local['ASYM_3x16']['retained']}"
        )


    # ========================================================
    # Dataset cleanup
    # ========================================================

    del graph
    del nodes
    del edges

    del edge_set

    del incoming
    del outgoing

    del indeg
    del outdeg

    del phys
    del time_map
    del node_ids

    del past_velocity
    del future_velocity

    del source_eligible
    del target_eligible

    del source_by_t
    del target_by_t

    del source_incumbent_cost
    del target_incumbent_cost

    del local

    gc.collect()


# ============================================================
# 9. Fidelity against G3F1 RAW cardinality
# ============================================================

print(
    "\n========== G3F1 RAW-CARDINALITY REPRODUCTION =========="
)


for contract in FRONTIER:

    expected_raw = int(
        prior_counts[
            contract
        ][
            "rows"
        ]
    )


    reproduced_raw = int(
        stats[
            contract
        ][
            "raw_rows"
        ]
    )


    print(
        contract,
        reproduced_raw,
        "/",
        expected_raw,
        "match=",
        reproduced_raw == expected_raw,
    )


    assert reproduced_raw == expected_raw, (
        contract,
        reproduced_raw,
        expected_raw,
    )


print(
    "RAW_CARDINALITY_REPRODUCTION: PASS"
)


# ============================================================
# 10. Post-dominance table
# ============================================================

rows = []


for contract in FRONTIER:

    entry = stats[
        contract
    ]


    raw_rows = int(
        entry[
            "raw_rows"
        ]
    )


    retained_rows = int(
        entry[
            "retained_rows"
        ]
    )


    rows.append(
        {
            "contract":
                contract,

            "raw_rows":
                raw_rows,

            "retained_rows":
                retained_rows,

            "retention_fraction":
                float(
                    retained_rows
                    /
                    raw_rows
                ),

            "raw_cuts_0":
                int(
                    entry[
                        "raw_cuts_0"
                    ]
                ),

            "raw_cuts_1":
                int(
                    entry[
                        "raw_cuts_1"
                    ]
                ),

            "raw_cuts_2":
                int(
                    entry[
                        "raw_cuts_2"
                    ]
                ),

            "retained_cuts_0":
                int(
                    entry[
                        "retained_cuts_0"
                    ]
                ),

            "retained_cuts_1":
                int(
                    entry[
                        "retained_cuts_1"
                    ]
                ),

            "retained_cuts_2":
                int(
                    entry[
                        "retained_cuts_2"
                    ]
                ),
        }
    )


postdom_table = pd.DataFrame(
    rows
)


base_retained = int(
    postdom_table[
        postdom_table[
            "contract"
        ].eq(
            "BASE_EXACT_5x5"
        )
    ][
        "retained_rows"
    ].iloc[
        0
    ]
)


postdom_table[
    "retained_multiplier_vs_base"
] = (
    postdom_table[
        "retained_rows"
    ]
    /
    float(
        base_retained
    )
)


print(
    "\n========== GT-BLIND POST-DOMINANCE CARDINALITY =========="
)


print(
    postdom_table.to_string(
        index=False
    )
)


# ============================================================
# 11. Fold stability of retained cardinality
# ============================================================

print(
    "\n========== POST-DOMINANCE CARDINALITY BY FOLD =========="
)


fold_rows = []


for contract in FRONTIER:

    for fold in [
        1,
        2,
        3,
        4,
    ]:

        entry = stats[
            contract
        ][
            "folds"
        ][
            str(
                fold
            )
        ]


        fold_rows.append(
            {
                "contract":
                    contract,

                "fold":
                    fold,

                "raw_rows":
                    int(
                        entry[
                            "raw"
                        ]
                    ),

                "retained_rows":
                    int(
                        entry[
                            "retained"
                        ]
                    ),

                "retention_fraction":
                    float(
                        entry[
                            "retained"
                        ]
                        /
                        entry[
                            "raw"
                        ]
                    )
                    if entry[
                        "raw"
                    ]
                    else np.nan,
            }
        )


fold_table = pd.DataFrame(
    fold_rows
)


for contract in FRONTIER:

    print(
        "\n" + contract
    )


    print(
        fold_table[
            fold_table[
                "contract"
            ].eq(
                contract
            )
        ].to_string(
            index=False
        )
    )


# ============================================================
# 12. Freeze Part A before reading G3F1 oracle
# ============================================================

part_a_payload = {
    "stage":
        "12.11G3F2",

    "part":
        "GT_BLIND_POST_DOMINANCE_CARDINALITY",

    "frontier_selection":
        "DEVELOPMENT_GT_INFORMED_FROM_G3F1",

    "frontier_contracts":
        FRONTIER,

    "development_datasets":
        103,

    "gap_horizon": {
        "min_gap_size":
            MIN_GAP_SIZE,

        "max_gap_size":
            MAX_GAP_SIZE,
    },

    "dominance_gate":
        (
            "FREE_FREE_ALWAYS_PASS; "
            "OCCUPIED_REQUIRES_AT_LEAST_ONE_REQUIRED_SIDE_"
            "NORMALIZED_CV_IMPROVEMENT_GT_0"
        ),

    "results":
        {
            row[
                "contract"
            ]:
                {
                    key:
                        (
                            int(
                                value
                            )
                            if isinstance(
                                value,
                                (
                                    np.integer,
                                    int,
                                )
                            )
                            else float(
                                value
                            )
                        )

                    for (
                        key,
                        value
                    )
                    in row.items()

                    if key != "contract"
                }

            for row
            in rows
        },

    "fold_results":
        fold_table.to_dict(
            "records"
        ),

    "dataset_results":
        dataset_stats,

    "proposal_rows_persisted":
        False,

    "gt_used_during_part_a":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


canonical_bytes = json.dumps(
    part_a_payload,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
).encode(
    "utf-8"
)


part_a_sha = hashlib.sha256(
    canonical_bytes
).hexdigest()


part_a_payload[
    "sha256"
] = part_a_sha


OUT.write_text(
    json.dumps(
        part_a_payload,
        indent=2,
    ),
    encoding="utf-8",
)


assert OUT.exists()


print(
    "\nPOST-DOMINANCE CARDINALITY ARTIFACT FROZEN: YES"
)

print(
    "Part-A SHA256:",
    part_a_sha
)


# ============================================================
# PART B
#
# G3F1 oracle can now be read.
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — ENGINEERING FRONTIER WITH DEVELOPMENT ORACLE"
)

print(
    "============================================================"
)


oracle = pd.read_pickle(
    G3F1_ORACLE_PATH
)


assert len(
    oracle
) == 369


# ============================================================
# 13. Oracle coverage + retained-cardinality tradeoff
# ============================================================

tradeoff_rows = []


for contract in FRONTIER:

    retrieval_col = (
        f"retrieval_{contract}"
    )


    combined_col = (
        f"combined_{contract}"
    )


    assert retrieval_col in oracle.columns
    assert combined_col in oracle.columns


    retrieval_count = int(
        oracle[
            retrieval_col
        ].sum()
    )


    combined_count = int(
        oracle[
            combined_col
        ].sum()
    )


    full7_mask = (
        oracle[
            "pred_full_7um"
        ].astype(bool)
    )


    retrieval_full7 = int(
        (
            oracle[
                retrieval_col
            ].astype(bool)
            &
            full7_mask
        ).sum()
    )


    combined_full7 = int(
        (
            oracle[
                combined_col
            ].astype(bool)
            &
            full7_mask
        ).sum()
    )


    postdom_row = postdom_table[
        postdom_table[
            "contract"
        ].eq(
            contract
        )
    ].iloc[
        0
    ]


    tradeoff_rows.append(
        {
            "contract":
                contract,

            "retained_rows":
                int(
                    postdom_row[
                        "retained_rows"
                    ]
                ),

            "retained_multiplier_vs_base":
                float(
                    postdom_row[
                        "retained_multiplier_vs_base"
                    ]
                ),

            "retrieval_captured":
                retrieval_count,

            "combined_captured":
                combined_count,

            "combined_overall_coverage":
                float(
                    combined_count
                    /
                    369
                ),

            "retrieval_full7":
                retrieval_full7,

            "combined_full7":
                combined_full7,

            "combined_full7_overall_coverage":
                float(
                    combined_full7
                    /
                    int(
                        full7_mask.sum()
                    )
                ),
        }
    )


tradeoff = pd.DataFrame(
    tradeoff_rows
)


base_combined = int(
    tradeoff[
        tradeoff[
            "contract"
        ].eq(
            "BASE_EXACT_5x5"
        )
    ][
        "combined_captured"
    ].iloc[
        0
    ]
)


base_full7 = int(
    tradeoff[
        tradeoff[
            "contract"
        ].eq(
            "BASE_EXACT_5x5"
        )
    ][
        "combined_full7"
    ].iloc[
        0
    ]
)


tradeoff[
    "extra_combined_vs_base"
] = (
    tradeoff[
        "combined_captured"
    ]
    -
    base_combined
)


tradeoff[
    "extra_full7_vs_base"
] = (
    tradeoff[
        "combined_full7"
    ]
    -
    base_full7
)


tradeoff[
    "extra_retained_rows_vs_base"
] = (
    tradeoff[
        "retained_rows"
    ]
    -
    base_retained
)


tradeoff[
    "extra_million_rows_per_extra_combined"
] = np.where(
    tradeoff[
        "extra_combined_vs_base"
    ] > 0,

    (
        tradeoff[
            "extra_retained_rows_vs_base"
        ]
        /
        1_000_000.0
    )
    /
    tradeoff[
        "extra_combined_vs_base"
    ],

    np.nan,
)


print(
    "\n========== POST-DOMINANCE ENGINEERING TRADEOFF =========="
)


print(
    tradeoff.to_string(
        index=False
    )
)


# ============================================================
# 14. Incremental frontier steps
# ============================================================

ORDER = [
    "BASE_EXACT_5x5",
    "ASYM_3x8",
    "ASYM_3x12",
    "ASYM_3x16",
]


ordered = (
    tradeoff
    .set_index(
        "contract"
    )
    .loc[
        ORDER
    ]
    .reset_index()
)


increment_rows = []


for i in range(
    1,
    len(
        ordered
    ),
):

    previous = ordered.iloc[
        i - 1
    ]


    current = ordered.iloc[
        i
    ]


    row_delta = (
        int(
            current[
                "retained_rows"
            ]
        )
        -
        int(
            previous[
                "retained_rows"
            ]
        )
    )


    combined_delta = (
        int(
            current[
                "combined_captured"
            ]
        )
        -
        int(
            previous[
                "combined_captured"
            ]
        )
    )


    full7_delta = (
        int(
            current[
                "combined_full7"
            ]
        )
        -
        int(
            previous[
                "combined_full7"
            ]
        )
    )


    increment_rows.append(
        {
            "from":
                previous[
                    "contract"
                ],

            "to":
                current[
                    "contract"
                ],

            "extra_retained_rows":
                row_delta,

            "extra_combined_oracle":
                combined_delta,

            "extra_full7_oracle":
                full7_delta,

            "million_rows_per_extra_combined":
                (
                    row_delta
                    /
                    1_000_000.0
                    /
                    combined_delta
                )
                if combined_delta > 0
                else np.nan,
        }
    )


increment_table = pd.DataFrame(
    increment_rows
)


print(
    "\n========== INCREMENTAL FRONTIER COST =========="
)


print(
    increment_table.to_string(
        index=False
    )
)


# ============================================================
# 15. Save Part B summary
# ============================================================

summary = {
    "stage":
        "12.11G3F2",

    "status":
        "POST_DOMINANCE_RETRIEVAL_FRONTIER_AUDITED",

    "part_a_sha256":
        part_a_sha,

    "frontier_contracts":
        FRONTIER,

    "tradeoff":
        tradeoff.to_dict(
            "records"
        ),

    "incremental_frontier":
        increment_table.to_dict(
            "records"
        ),

    "frontier_selection_origin":
        "DEVELOPMENT_GT_INFORMED_G3F1",

    "actual_generator_features_gt_blind":
        True,

    "final_generator_selected":
        False,

    "proposal_universe_generated":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert SUMMARY_OUT.exists()


# ============================================================
# 16. Decision
# ============================================================

print(
    "\n========== 12.11G3F2 DECISION =========="
)

print(
    "POST_DOMINANCE_RETRIEVAL_FRONTIER_AUDIT: PASS"
)

print(
    "raw G3F1 cardinality exactly reproduced:",
    "YES"
)

print(
    "post-dominance cardinality frozen before Part-B oracle read:",
    "YES"
)

print(
    "frontier shortlist development-GT-informed:",
    "YES"
)

print(
    "actual generator features GT-blind:",
    "YES"
)

print(
    "final generator selected:",
    "NO"
)

print(
    "new proposal universe generated:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "classifier trained:",
    "NO"
)

print(
    "policy selected:",
    "NO"
)

print(
    "graph modified:",
    "NO"
)

print(
    "next:"
)

print(
    "FREEZE_ONE_FRONTIER_GENERATOR_FROM_POST_DOMINANCE_COST_BENEFIT"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3F3
#
# Freeze the selected deployable GT-blind retrieval generator:
#
#       ASYM_3x8 + DOMINANCE_ANY
#
# Selection rationale:
#
#   BASE 5x5:
#       retained = 23,012,742
#       combined oracle = 185
#       full<=7 = 153
#
#   ASYM 3x8:
#       retained = 29,227,030
#       multiplier = 1.270037
#       combined oracle = 212
#       full<=7 = 178
#
# Increment:
#
#       +6,214,288 retained proposals
#       +27 oracle components
#       +25 full<=7 components
#
#       0.230159 million rows / extra oracle
#
# Later frontier increments:
#
#   3x8 -> 3x12:
#       0.734069 M rows / extra oracle
#
#   3x12 -> 3x16:
#       0.836710 M rows / extra oracle
#
# Therefore ASYM_3x8 is frozen as the engineering knee.
#
# IMPORTANT:
#
#   Generator selection is DEVELOPMENT-GT-INFORMED.
#
#   Actual inference features remain GT-blind.
#
#   The 29M proposal rows are NOT materialized here.
#   We freeze the deterministic generator CONTRACT instead.
#
#   Future policy/scoring work MUST use this frozen generator
#   without changing:
#
#       - temporal horizon
#       - endpoint eligibility
#       - rank contract
#       - dominance gate
#
# No Fold0.
# No graph modification.
# No classifier.
# No deployment policy yet.
#
# Keep after running: YES
# ============================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


# ------------------------------------------------------------
# Parent GT-blind retrieval cardinality
# ------------------------------------------------------------

G3F1_CARDINALITY_PATH = (
    S12
    / "stage12_folds14_multiframe_asymmetric_retrieval_cardinality_v1.json"
)


# ------------------------------------------------------------
# Parent GT-blind post-dominance cardinality
#
# This was frozen BEFORE G3F2 Part-B oracle read.
# ------------------------------------------------------------

G3F2_PARTA_PATH = (
    S12
    / "stage12_folds14_multiframe_postdominance_frontier_v1.json"
)


# ------------------------------------------------------------
# Development-GT-informed engineering tradeoff
# ------------------------------------------------------------

G3F2_SUMMARY_PATH = (
    S12
    / "stage12_folds14_multiframe_postdominance_frontier_oracle_v1.json"
)


# ------------------------------------------------------------
# Oracle artifact for post-freeze confirmation only
# ------------------------------------------------------------

G3F1_ORACLE_PATH = (
    S12
    / "stage12_folds14_multiframe_asymmetric_retrieval_oracle_v1.pkl"
)


# ------------------------------------------------------------
# Frozen outputs
# ------------------------------------------------------------

FROZEN_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_generator_asym3x8_v1_frozen.json"
)


CONFIRMATION_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_generator_asym3x8_v1_confirmation.json"
)


for path in [
    G3F1_CARDINALITY_PATH,
    G3F2_PARTA_PATH,
    G3F2_SUMMARY_PATH,
    G3F1_ORACLE_PATH,
]:

    assert path.exists(), path


if FROZEN_PATH.exists():

    raise RuntimeError(
        "\nFrozen ASYM_3x8 generator already exists:\n"
        f"{FROZEN_PATH}\n\n"
        "Do not overwrite a frozen generator contract."
    )


# ============================================================
# 1. Helper
# ============================================================

def file_sha256(
    path,
):

    hasher = hashlib.sha256()


    with open(
        path,
        "rb",
    ) as handle:

        while True:

            block = handle.read(
                1024 * 1024
            )


            if not block:

                break


            hasher.update(
                block
            )


    return hasher.hexdigest()


# ============================================================
# 2. Read parent artifacts
# ============================================================

g3f1_cardinality = json.loads(
    G3F1_CARDINALITY_PATH.read_text(
        encoding="utf-8"
    )
)


g3f2_part_a = json.loads(
    G3F2_PARTA_PATH.read_text(
        encoding="utf-8"
    )
)


g3f2_summary = json.loads(
    G3F2_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


# ============================================================
# 3. Parent artifact fidelity
# ============================================================

assert g3f1_cardinality[
    "stage"
] == "12.11G3F1"


assert g3f2_part_a[
    "stage"
] == "12.11G3F2"


assert g3f2_summary[
    "stage"
] == "12.11G3F2"


assert g3f2_part_a[
    "gt_used_during_part_a"
] is False


assert g3f2_part_a[
    "fold0_used"
] is False


assert g3f2_summary[
    "fold0_used"
] is False


# ============================================================
# 4. Extract exact post-dominance results
# ============================================================

part_a_results = g3f2_part_a[
    "results"
]


assert "BASE_EXACT_5x5" in part_a_results
assert "ASYM_3x8" in part_a_results
assert "ASYM_3x12" in part_a_results
assert "ASYM_3x16" in part_a_results


base_part_a = part_a_results[
    "BASE_EXACT_5x5"
]


a38_part_a = part_a_results[
    "ASYM_3x8"
]


a312_part_a = part_a_results[
    "ASYM_3x12"
]


a316_part_a = part_a_results[
    "ASYM_3x16"
]


# Exact G3F2 reproduced values.
assert int(
    base_part_a[
        "retained_rows"
    ]
) == 23_012_742


assert int(
    a38_part_a[
        "raw_rows"
    ]
) == 36_473_971


assert int(
    a38_part_a[
        "retained_rows"
    ]
) == 29_227_030


assert int(
    a38_part_a[
        "retained_cuts_0"
    ]
) == 269_936


assert int(
    a38_part_a[
        "retained_cuts_1"
    ]
) == 3_118_738


assert int(
    a38_part_a[
        "retained_cuts_2"
    ]
) == 25_838_356


assert (
    int(
        a38_part_a[
            "retained_cuts_0"
        ]
    )
    +
    int(
        a38_part_a[
            "retained_cuts_1"
        ]
    )
    +
    int(
        a38_part_a[
            "retained_cuts_2"
        ]
    )
) == 29_227_030


# ============================================================
# 5. Extract engineering tradeoff
# ============================================================

tradeoff = pd.DataFrame(
    g3f2_summary[
        "tradeoff"
    ]
)


increments = pd.DataFrame(
    g3f2_summary[
        "incremental_frontier"
    ]
)


assert set(
    tradeoff[
        "contract"
    ]
) == {
    "BASE_EXACT_5x5",
    "ASYM_3x8",
    "ASYM_3x12",
    "ASYM_3x16",
}


def tradeoff_row(
    contract,
):

    row = tradeoff[
        tradeoff[
            "contract"
        ].eq(
            contract
        )
    ]


    assert len(
        row
    ) == 1


    return row.iloc[
        0
    ]


base = tradeoff_row(
    "BASE_EXACT_5x5"
)


a38 = tradeoff_row(
    "ASYM_3x8"
)


a312 = tradeoff_row(
    "ASYM_3x12"
)


a316 = tradeoff_row(
    "ASYM_3x16"
)


# ============================================================
# 6. Exact selection evidence
# ============================================================

assert int(
    base[
        "combined_captured"
    ]
) == 185


assert int(
    base[
        "combined_full7"
    ]
) == 153


assert int(
    a38[
        "combined_captured"
    ]
) == 212


assert int(
    a38[
        "combined_full7"
    ]
) == 178


assert int(
    a312[
        "combined_captured"
    ]
) == 223


assert int(
    a312[
        "combined_full7"
    ]
) == 185


assert int(
    a316[
        "combined_captured"
    ]
) == 230


assert int(
    a316[
        "combined_full7"
    ]
) == 190


assert np.isclose(
    float(
        a38[
            "retained_multiplier_vs_base"
        ]
    ),
    1.270037,
    atol=1e-6,
)


# ------------------------------------------------------------
# Incremental frontier
# ------------------------------------------------------------

step_base_to_38 = increments[
    increments[
        "to"
    ].eq(
        "ASYM_3x8"
    )
].iloc[
    0
]


step_38_to_312 = increments[
    increments[
        "to"
    ].eq(
        "ASYM_3x12"
    )
].iloc[
    0
]


step_312_to_316 = increments[
    increments[
        "to"
    ].eq(
        "ASYM_3x16"
    )
].iloc[
    0
]


assert int(
    step_base_to_38[
        "extra_retained_rows"
    ]
) == 6_214_288


assert int(
    step_base_to_38[
        "extra_combined_oracle"
    ]
) == 27


assert int(
    step_base_to_38[
        "extra_full7_oracle"
    ]
) == 25


assert np.isclose(
    float(
        step_base_to_38[
            "million_rows_per_extra_combined"
        ]
    ),
    0.230159,
    atol=1e-6,
)


assert float(
    step_38_to_312[
        "million_rows_per_extra_combined"
    ]
) > 0.70


assert float(
    step_312_to_316[
        "million_rows_per_extra_combined"
    ]
) > 0.80


print(
    "========== 12.11G3F3 SELECTION EVIDENCE =========="
)

print(
    "BASE retained:",
    int(
        base[
            "retained_rows"
        ]
    )
)

print(
    "BASE combined:",
    int(
        base[
            "combined_captured"
        ]
    )
)

print(
    "BASE full<=7:",
    int(
        base[
            "combined_full7"
        ]
    )
)


print(
    "\nASYM_3x8 retained:",
    int(
        a38[
            "retained_rows"
        ]
    )
)

print(
    "ASYM_3x8 multiplier:",
    float(
        a38[
            "retained_multiplier_vs_base"
        ]
    )
)

print(
    "ASYM_3x8 combined:",
    int(
        a38[
            "combined_captured"
        ]
    )
)

print(
    "ASYM_3x8 full<=7:",
    int(
        a38[
            "combined_full7"
        ]
    )
)


print(
    "\nBASE -> 3x8 rows:",
    int(
        step_base_to_38[
            "extra_retained_rows"
        ]
    )
)

print(
    "BASE -> 3x8 oracle gain:",
    int(
        step_base_to_38[
            "extra_combined_oracle"
        ]
    )
)

print(
    "BASE -> 3x8 full7 gain:",
    int(
        step_base_to_38[
            "extra_full7_oracle"
        ]
    )
)

print(
    "BASE -> 3x8 M rows / oracle:",
    float(
        step_base_to_38[
            "million_rows_per_extra_combined"
        ]
    )
)


print(
    "\n3x8 -> 3x12 M rows / oracle:",
    float(
        step_38_to_312[
            "million_rows_per_extra_combined"
        ]
    )
)

print(
    "3x12 -> 3x16 M rows / oracle:",
    float(
        step_312_to_316[
            "million_rows_per_extra_combined"
        ]
    )
)


# ============================================================
# 7. Freeze exact generator contract
# ============================================================

generator_contract = {
    "stage":
        "12.11G3F3",

    "status":
        "FROZEN",

    "generator_id":
        "MULTIFRAME_REWIRE_ASYM3X8_DOMINANCE_ANY_V1",

    "selected_contract":
        "ASYM_3x8",

    # --------------------------------------------------------
    # Scientific provenance
    # --------------------------------------------------------

    "selection_origin":
        "DEVELOPMENT_GT_INFORMED_G3F1_G3F2",

    "selection_role":
        (
            "ENGINEERING_RETRIEVAL_BREADTH_SELECTION_ONLY; "
            "ACTUAL_GENERATOR_FEATURES_ARE_GT_BLIND"
        ),

    "selection_reason":
        (
            "ASYM_3x8 is the low-cardinality engineering knee: "
            "+27 combined oracle components and +25 full<=7 "
            "components for +27.0% retained cardinality versus "
            "BASE_EXACT_5x5. Later frontier expansions have "
            "substantially worse marginal row cost."
        ),

    # --------------------------------------------------------
    # Frozen graph state
    # --------------------------------------------------------

    "input_graph":
        "STAGE12_CURRENT_BEST",

    "pipeline":
        (
            "FULL_HYBRID_PLUS_MORPH_DIV_V2_PLUS_"
            "MUTUAL_CONT_V1_PLUS_MUTUAL_CONT_V2_PLUS_"
            "DET_BRIDGE_R1P50"
        ),

    # --------------------------------------------------------
    # Frozen temporal contract
    # --------------------------------------------------------

    "temporal": {
        "min_gap_size":
            2,

        "max_gap_size":
            12,

        "delta_t_definition":
            "gap_size + 1",
    },

    # --------------------------------------------------------
    # Frozen source endpoint contract
    # --------------------------------------------------------

    "source_endpoint": {
        "max_outdegree":
            1,

        "requires_unique_past_predecessor":
            True,

        "requires_past_velocity":
            True,

        "past_velocity_definition":
            (
                "(source_position - unique_predecessor_position) "
                "/ source_predecessor_delta_t"
            ),
    },

    # --------------------------------------------------------
    # Frozen target endpoint contract
    # --------------------------------------------------------

    "target_endpoint": {
        "max_indegree":
            1,

        "requires_unique_future_successor":
            True,

        "requires_future_velocity":
            True,

        "future_velocity_definition":
            (
                "(unique_successor_position - target_position) "
                "/ target_successor_delta_t"
            ),
    },

    # --------------------------------------------------------
    # Frozen retrieval
    # --------------------------------------------------------

    "retrieval": {
        "family":
            "ASYMMETRIC_FORWARD_BACKWARD_CONSTANT_VELOCITY_KNN",

        "source_rank_metric":
            "FORWARD_CV_ENDPOINT_DISTANCE",

        "target_rank_metric":
            "BACKWARD_CV_ENDPOINT_DISTANCE",

        "strong_rank":
            3,

        "weak_rank":
            8,

        "pass_rule":
            (
                "(source_rank <= 3 AND target_rank <= 8) OR "
                "(target_rank <= 3 AND source_rank <= 8)"
            ),

        "exact_knn_query_k":
            8,

        "existing_current_edge_excluded":
            True,
    },

    # --------------------------------------------------------
    # Frozen physical scaling
    # --------------------------------------------------------

    "physical_scale_zyx_um":
        [
            1.625,
            0.40625,
            0.40625,
        ],

    # --------------------------------------------------------
    # Frozen candidate CV costs
    # --------------------------------------------------------

    "candidate_cost": {
        "source_cv_per_frame":
            (
                "norm(target - "
                "(source + delta_t * source_past_velocity)) "
                "/ delta_t"
            ),

        "target_cv_per_frame":
            (
                "norm(source - "
                "(target - delta_t * target_future_velocity)) "
                "/ delta_t"
            ),

        "time_normalized":
            True,
    },

    # --------------------------------------------------------
    # Frozen incumbent comparison
    # --------------------------------------------------------

    "incumbent_comparison": {
        "source_side":
            (
                "if source_outdegree == 1, compare candidate "
                "source_cv_per_frame against CURRENT_BEST "
                "source incumbent normalized CV cost"
            ),

        "target_side":
            (
                "if target_indegree == 1, compare candidate "
                "target_cv_per_frame against CURRENT_BEST "
                "target incumbent normalized CV cost"
            ),

        "free_free":
            "NO_DOMINANCE_COMPARISON_REQUIRED",
    },

    # --------------------------------------------------------
    # Frozen dominance-any gate
    # --------------------------------------------------------

    "dominance_gate": {
        "id":
            "DOMINANCE_ANY_V1",

        "free_free_pass":
            True,

        "occupied_pass_rule":
            (
                "AT_LEAST_ONE_REQUIRED_SIDE_RELATIVE_"
                "NORMALIZED_CV_IMPROVEMENT_GT_0"
            ),

        "source_improvement":
            (
                "source_incumbent_cv_per_frame "
                "- candidate_source_cv_per_frame"
            ),

        "target_improvement":
            (
                "target_incumbent_cv_per_frame "
                "- candidate_target_cv_per_frame"
            ),

        "missing_required_incumbent_cost":
            "FAIL",
    },

    # --------------------------------------------------------
    # Frozen topology semantics
    # --------------------------------------------------------

    "topology": {
        "free_free":
            "REMOVE_0_INCUMBENT_EDGES",

        "source_occupied":
            "REMOVE_SOURCE_UNIQUE_OUTGOING_EDGE",

        "target_occupied":
            "REMOVE_TARGET_UNIQUE_INCOMING_EDGE",

        "both_occupied":
            "REMOVE_BOTH_UNIQUE_INCUMBENT_EDGES",

        "division_edge_removal_allowed":
            False,

        "source_outdegree_gt1_allowed":
            False,

        "target_indegree_gt1_allowed":
            False,
    },

    # --------------------------------------------------------
    # Frozen cardinality
    # --------------------------------------------------------

    "development_cardinality": {
        "datasets":
            103,

        "raw_rows":
            36_473_971,

        "retained_rows":
            29_227_030,

        "retention_fraction":
            float(
                a38_part_a[
                    "retention_fraction"
                ]
            ),

        "retained_multiplier_vs_base":
            float(
                a38[
                    "retained_multiplier_vs_base"
                ]
            ),

        "retained_topology": {
            "free_free":
                269_936,

            "one_cut":
                3_118_738,

            "two_cut":
                25_838_356,
        },
    },

    # --------------------------------------------------------
    # Development oracle evidence
    #
    # This does NOT become inference input.
    # --------------------------------------------------------

    "development_oracle_evidence": {
        "clean_components":
            369,

        "retrieval_captured":
            int(
                a38[
                    "retrieval_captured"
                ]
            ),

        "combined_captured":
            212,

        "combined_overall_coverage":
            float(
                a38[
                    "combined_overall_coverage"
                ]
            ),

        "full7_combined":
            178,

        "full7_total":
            285,

        "full7_combined_coverage":
            float(
                a38[
                    "combined_full7_overall_coverage"
                ]
            ),
    },

    # --------------------------------------------------------
    # Parent provenance
    # --------------------------------------------------------

    "parents": {
        "g3f1_cardinality_path":
            str(
                G3F1_CARDINALITY_PATH
            ),

        "g3f1_cardinality_sha256":
            file_sha256(
                G3F1_CARDINALITY_PATH
            ),

        "g3f2_part_a_path":
            str(
                G3F2_PARTA_PATH
            ),

        "g3f2_part_a_sha256":
            file_sha256(
                G3F2_PARTA_PATH
            ),

        "g3f2_part_a_internal_sha256":
            g3f2_part_a[
                "sha256"
            ],

        "g3f2_summary_path":
            str(
                G3F2_SUMMARY_PATH
            ),

        "g3f2_summary_sha256":
            file_sha256(
                G3F2_SUMMARY_PATH
            ),
    },

    # --------------------------------------------------------
    # Leakage / policy state
    # --------------------------------------------------------

    "actual_generator_features_gt_blind":
        True,

    "generator_selected_with_development_gt":
        True,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "scoring_policy_selected":
        False,

    "rewire_policy_selected":
        False,

    "synthetic_nodes_inserted":
        False,

    "graph_modified":
        False,

    # --------------------------------------------------------
    # Materialization policy
    # --------------------------------------------------------

    "materialization": {
        "full_29m_row_universe_persisted":
            False,

        "reason":
            (
                "Freeze deterministic generator contract rather "
                "than materializing a multi-gigabyte intermediate "
                "table. Future stages regenerate proposals "
                "streamingly under this exact frozen contract."
            ),

        "future_generator_changes_allowed":
            False,
    },
}


# ============================================================
# 8. Canonical generator SHA
# ============================================================

canonical_payload = json.dumps(
    generator_contract,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
).encode(
    "utf-8"
)


generator_sha = hashlib.sha256(
    canonical_payload
).hexdigest()


generator_contract[
    "generator_contract_sha256"
] = generator_sha


FROZEN_PATH.write_text(
    json.dumps(
        generator_contract,
        indent=2,
    ),
    encoding="utf-8",
)


assert FROZEN_PATH.exists()


print(
    "\n========== FROZEN GENERATOR =========="
)

print(
    "generator ID:",
    generator_contract[
        "generator_id"
    ]
)

print(
    "contract:",
    generator_contract[
        "selected_contract"
    ]
)

print(
    "retained rows:",
    generator_contract[
        "development_cardinality"
    ][
        "retained_rows"
    ]
)

print(
    "generator SHA256:",
    generator_sha
)

print(
    "FROZEN BEFORE ANY FURTHER POLICY DESIGN: YES"
)


# ============================================================
# 9. Post-freeze development confirmation
#
# This is NOT a new tuning step.
#
# Generator is already frozen above.
# ============================================================

oracle = pd.read_pickle(
    G3F1_ORACLE_PATH
)


assert len(
    oracle
) == 369


RETRIEVAL_COL = (
    "retrieval_ASYM_3x8"
)

COMBINED_COL = (
    "combined_ASYM_3x8"
)


assert RETRIEVAL_COL in oracle.columns
assert COMBINED_COL in oracle.columns


retrieval_count = int(
    oracle[
        RETRIEVAL_COL
    ].sum()
)


combined_count = int(
    oracle[
        COMBINED_COL
    ].sum()
)


full7 = oracle[
    "pred_full_7um"
].astype(bool)


full7_total = int(
    full7.sum()
)


retrieval_full7 = int(
    (
        oracle[
            RETRIEVAL_COL
        ].astype(bool)
        &
        full7
    ).sum()
)


combined_full7 = int(
    (
        oracle[
            COMBINED_COL
        ].astype(bool)
        &
        full7
    ).sum()
)


assert retrieval_count == 236
assert combined_count == 212

assert full7_total == 285

assert retrieval_full7 == 201
assert combined_full7 == 178


# ============================================================
# 10. Fold confirmation
# ============================================================

fold_rows = []


for fold in [
    1,
    2,
    3,
    4,
]:

    sub = oracle[
        oracle[
            "fold"
        ].eq(
            fold
        )
    ]


    sub_full7 = sub[
        "pred_full_7um"
    ].astype(bool)


    fold_rows.append(
        {
            "fold":
                fold,

            "components":
                int(
                    len(
                        sub
                    )
                ),

            "retrieval_captured":
                int(
                    sub[
                        RETRIEVAL_COL
                    ].sum()
                ),

            "combined_captured":
                int(
                    sub[
                        COMBINED_COL
                    ].sum()
                ),

            "combined_fraction":
                float(
                    sub[
                        COMBINED_COL
                    ].mean()
                ),

            "full7_total":
                int(
                    sub_full7.sum()
                ),

            "combined_full7":
                int(
                    (
                        sub[
                            COMBINED_COL
                        ].astype(bool)
                        &
                        sub_full7
                    ).sum()
                ),
        }
    )


fold_table = pd.DataFrame(
    fold_rows
)


print(
    "\n========== POST-FREEZE DEVELOPMENT CONFIRMATION =========="
)

print(
    "retrieval:",
    retrieval_count,
    "/ 369"
)

print(
    "combined:",
    combined_count,
    "/ 369"
)

print(
    "full<=7 retrieval:",
    retrieval_full7,
    "/",
    full7_total
)

print(
    "full<=7 combined:",
    combined_full7,
    "/",
    full7_total
)


print(
    "\nBy fold:"
)

print(
    fold_table.to_string(
        index=False
    )
)


# ============================================================
# 11. Confirmation artifact
# ============================================================

confirmation = {
    "stage":
        "12.11G3F3",

    "status":
        "FROZEN_GENERATOR_DEVELOPMENT_CONFIRMATION",

    "generator_id":
        generator_contract[
            "generator_id"
        ],

    "generator_contract_sha256":
        generator_sha,

    "retrieval_captured":
        retrieval_count,

    "combined_captured":
        combined_count,

    "full7_total":
        full7_total,

    "retrieval_full7":
        retrieval_full7,

    "combined_full7":
        combined_full7,

    "fold_results":
        fold_table.to_dict(
            "records"
        ),

    "confirmation_is_not_retuning":
        True,

    "generator_changed_after_freeze":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


CONFIRMATION_PATH.write_text(
    json.dumps(
        confirmation,
        indent=2,
    ),
    encoding="utf-8",
)


assert CONFIRMATION_PATH.exists()


# ============================================================
# 12. Reload fidelity
# ============================================================

frozen_check = json.loads(
    FROZEN_PATH.read_text(
        encoding="utf-8"
    )
)


confirmation_check = json.loads(
    CONFIRMATION_PATH.read_text(
        encoding="utf-8"
    )
)


assert frozen_check[
    "generator_contract_sha256"
] == generator_sha


assert frozen_check[
    "selected_contract"
] == "ASYM_3x8"


assert frozen_check[
    "development_cardinality"
][
    "retained_rows"
] == 29_227_030


assert confirmation_check[
    "combined_captured"
] == 212


assert confirmation_check[
    "combined_full7"
] == 178


# ============================================================
# 13. Decision
# ============================================================

print(
    "\n========== 12.11G3F3 DECISION =========="
)

print(
    "MULTIFRAME_REWIRE_ASYM3X8_DOMINANCE_ANY_V1: FROZEN"
)

print(
    "generator contract SHA256:",
    generator_sha
)

print(
    "selection development-GT-informed:",
    "YES"
)

print(
    "actual generator features GT-blind:",
    "YES"
)

print(
    "full proposal universe materialized:",
    "NO"
)

print(
    "deterministic streaming regeneration required:",
    "YES"
)

print(
    "future retrieval retuning allowed:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "classifier trained:",
    "NO"
)

print(
    "rewire policy selected:",
    "NO"
)

print(
    "graph modified:",
    "NO"
)

print(
    "next:"
)

print(
    "DESIGN_SECOND_STAGE_GT_BLIND_SHORTLIST_ON_FROZEN_ASYM3X8_GENERATOR"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3F3-R
#
# Frozen Generator Fidelity Patch
#
# Problem discovered after G3F3:
#
#   G3F1 / G3F2 evidence for ASYM_3x8 was generated using:
#
#       exact cKDTree query K = 16
#
#   then applying:
#
#       (source_rank <= 3 AND target_rank <= 8)
#       OR
#       (target_rank <= 3 AND source_rank <= 8)
#
# But G3F3 frozen JSON incorrectly recorded:
#
#       exact_knn_query_k = 8
#
# Since cKDTree tie ordering can depend on requested K
# (already demonstrated in G3F0-R), K8 cannot be assumed
# equivalent to K16.
#
# Repair:
#
#   - DO NOT retune retrieval breadth.
#   - DO NOT change ASYM_3x8.
#   - DO NOT change dominance-any.
#   - DO NOT change temporal horizon.
#   - DO NOT change any development evidence.
#
# Only correct implementation fidelity:
#
#       exact_knn_query_k: 8 -> 16
#
# Original G3F3 artifact is preserved for provenance and
# superseded by this fidelity-patched contract.
#
# No Fold0.
# No graph modification.
# No policy selection.
#
# Keep after running: YES
# ============================================================

from pathlib import Path
import copy
import hashlib
import json

import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


# ------------------------------------------------------------
# Original frozen generator
# ------------------------------------------------------------

ORIGINAL_FROZEN_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_generator_asym3x8_v1_frozen.json"
)


ORIGINAL_CONFIRMATION_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_generator_asym3x8_v1_confirmation.json"
)


# ------------------------------------------------------------
# Parent evidence
# ------------------------------------------------------------

G3F1_CARDINALITY_PATH = (
    S12
    / "stage12_folds14_multiframe_asymmetric_retrieval_cardinality_v1.json"
)


G3F1_ORACLE_PATH = (
    S12
    / "stage12_folds14_multiframe_asymmetric_retrieval_oracle_v1.pkl"
)


G3F2_PARTA_PATH = (
    S12
    / "stage12_folds14_multiframe_postdominance_frontier_v1.json"
)


G3F2_SUMMARY_PATH = (
    S12
    / "stage12_folds14_multiframe_postdominance_frontier_oracle_v1.json"
)


# ------------------------------------------------------------
# Fidelity-patched authoritative outputs
# ------------------------------------------------------------

PATCHED_FROZEN_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_generator_asym3x8_v1_fidelitypatch1_frozen.json"
)


PATCHED_CONFIRMATION_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_generator_asym3x8_v1_fidelitypatch1_confirmation.json"
)


SUPERSESSION_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_generator_asym3x8_v1_supersession.json"
)


for path in [
    ORIGINAL_FROZEN_PATH,
    ORIGINAL_CONFIRMATION_PATH,
    G3F1_CARDINALITY_PATH,
    G3F1_ORACLE_PATH,
    G3F2_PARTA_PATH,
    G3F2_SUMMARY_PATH,
]:

    assert path.exists(), path


for path in [
    PATCHED_FROZEN_PATH,
    PATCHED_CONFIRMATION_PATH,
    SUPERSESSION_PATH,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3F3-R output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite an existing fidelity patch."
        )


# ============================================================
# 1. Helpers
# ============================================================

def file_sha256(
    path,
):

    hasher = hashlib.sha256()


    with open(
        path,
        "rb",
    ) as handle:

        while True:

            block = handle.read(
                1024 * 1024
            )


            if not block:

                break


            hasher.update(
                block
            )


    return hasher.hexdigest()


def canonical_sha256(
    payload,
):

    canonical = json.dumps(
        payload,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )


    return hashlib.sha256(
        canonical
    ).hexdigest()


# ============================================================
# 2. Load original frozen contract
# ============================================================

original = json.loads(
    ORIGINAL_FROZEN_PATH.read_text(
        encoding="utf-8"
    )
)


original_confirmation = json.loads(
    ORIGINAL_CONFIRMATION_PATH.read_text(
        encoding="utf-8"
    )
)


print(
    "========== 12.11G3F3-R ORIGINAL CONTRACT =========="
)

print(
    "generator ID:",
    original[
        "generator_id"
    ]
)

print(
    "selected contract:",
    original[
        "selected_contract"
    ]
)

print(
    "recorded exact_knn_query_k:",
    original[
        "retrieval"
    ][
        "exact_knn_query_k"
    ]
)

print(
    "original generator SHA:",
    original[
        "generator_contract_sha256"
    ]
)


assert original[
    "stage"
] == "12.11G3F3"


assert original[
    "status"
] == "FROZEN"


assert original[
    "selected_contract"
] == "ASYM_3x8"


assert original[
    "retrieval"
][
    "strong_rank"
] == 3


assert original[
    "retrieval"
][
    "weak_rank"
] == 8


# This is the fidelity bug being repaired.
assert original[
    "retrieval"
][
    "exact_knn_query_k"
] == 8


assert original[
    "generator_contract_sha256"
] == (
    "2c415947d11c65da1e0918b0dd3135094"
    "ad9fa398aaab271ea2d8c6086ea247b"
)


# ============================================================
# 3. Verify actual parent evidence contract
# ============================================================

g3f1_cardinality = json.loads(
    G3F1_CARDINALITY_PATH.read_text(
        encoding="utf-8"
    )
)


g3f2_part_a = json.loads(
    G3F2_PARTA_PATH.read_text(
        encoding="utf-8"
    )
)


g3f2_summary = json.loads(
    G3F2_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


print(
    "\n========== PARENT IMPLEMENTATION EVIDENCE =========="
)


print(
    "G3F1 max_rank_query:",
    g3f1_cardinality[
        "max_rank_query"
    ]
)


assert g3f1_cardinality[
    "stage"
] == "12.11G3F1"


# Critical fidelity fact.
assert int(
    g3f1_cardinality[
        "max_rank_query"
    ]
) == 16


assert g3f1_cardinality[
    "contracts"
][
    "ASYM_3x8"
][
    "type"
] == "ASYM"


assert int(
    g3f1_cardinality[
        "contracts"
    ][
        "ASYM_3x8"
    ][
        "a"
    ]
) == 3


assert int(
    g3f1_cardinality[
        "contracts"
    ][
        "ASYM_3x8"
    ][
        "b"
    ]
) == 8


# ============================================================
# 4. Verify G3F1 raw evidence
# ============================================================

g3f1_a38 = g3f1_cardinality[
    "counts"
][
    "ASYM_3x8"
]


assert int(
    g3f1_a38[
        "rows"
    ]
) == 36_473_971


print(
    "G3F1 ASYM_3x8 raw rows:",
    int(
        g3f1_a38[
            "rows"
        ]
    )
)


# ============================================================
# 5. Verify G3F2 exact reproduction
# ============================================================

g3f2_a38 = g3f2_part_a[
    "results"
][
    "ASYM_3x8"
]


assert int(
    g3f2_a38[
        "raw_rows"
    ]
) == 36_473_971


assert int(
    g3f2_a38[
        "retained_rows"
    ]
) == 29_227_030


assert int(
    g3f2_a38[
        "retained_cuts_0"
    ]
) == 269_936


assert int(
    g3f2_a38[
        "retained_cuts_1"
    ]
) == 3_118_738


assert int(
    g3f2_a38[
        "retained_cuts_2"
    ]
) == 25_838_356


print(
    "G3F2 raw rows:",
    int(
        g3f2_a38[
            "raw_rows"
        ]
    )
)

print(
    "G3F2 retained rows:",
    int(
        g3f2_a38[
            "retained_rows"
        ]
    )
)


# ============================================================
# 6. Verify development selection evidence unchanged
# ============================================================

tradeoff = pd.DataFrame(
    g3f2_summary[
        "tradeoff"
    ]
)


a38_tradeoff = tradeoff[
    tradeoff[
        "contract"
    ].eq(
        "ASYM_3x8"
    )
]


assert len(
    a38_tradeoff
) == 1


a38_tradeoff = a38_tradeoff.iloc[
    0
]


assert int(
    a38_tradeoff[
        "retrieval_captured"
    ]
) == 236


assert int(
    a38_tradeoff[
        "combined_captured"
    ]
) == 212


assert int(
    a38_tradeoff[
        "combined_full7"
    ]
) == 178


assert int(
    original_confirmation[
        "retrieval_captured"
    ]
) == 236


assert int(
    original_confirmation[
        "combined_captured"
    ]
) == 212


assert int(
    original_confirmation[
        "combined_full7"
    ]
) == 178


print(
    "\nselection evidence unchanged:"
)

print(
    "retrieval:",
    236
)

print(
    "combined:",
    212
)

print(
    "full<=7:",
    178
)


# ============================================================
# 7. Construct fidelity-patched contract
# ============================================================

patched = copy.deepcopy(
    original
)


# ------------------------------------------------------------
# New provenance identity
# ------------------------------------------------------------

patched[
    "stage"
] = "12.11G3F3-R"


patched[
    "status"
] = "FROZEN_FIDELITY_PATCHED"


patched[
    "generator_id"
] = (
    "MULTIFRAME_REWIRE_ASYM3X8_"
    "DOMINANCE_ANY_V1_FIDELITYPATCH1"
)


# Remove old hash before canonical hashing.
patched.pop(
    "generator_contract_sha256",
    None,
)


# ============================================================
# 8. Exact fidelity correction
# ============================================================

patched[
    "retrieval"
][
    "exact_knn_query_k"
] = 16


patched[
    "retrieval"
][
    "rank_decision_limits"
] = {
    "strong_rank":
        3,

    "weak_rank":
        8,
}


patched[
    "retrieval"
][
    "implementation_note"
] = (
    "Candidate ranks MUST be obtained from the exact K=16 "
    "cKDTree queries used in G3F1/G3F2 evidence generation. "
    "The ASYM_3x8 acceptance rule then reads only ranks <=3 "
    "and <=8. Do NOT substitute an exact K=8 query because "
    "cKDTree tie ordering may differ across requested K."
)


# ============================================================
# 9. Supersession provenance
# ============================================================

patched[
    "fidelity_patch"
] = {
    "patch_id":
        "FIDELITYPATCH1",

    "reason":
        (
            "G3F3 recorded exact_knn_query_k=8, while all G3F1 "
            "and G3F2 ASYM_3x8 cardinality/oracle evidence was "
            "generated from exact K=16 queries. The correction "
            "restores implementation fidelity without changing "
            "the selected ASYM_3x8 rank rule."
        ),

    "old_value": {
        "exact_knn_query_k":
            8,
    },

    "new_value": {
        "exact_knn_query_k":
            16,
    },

    "retrieval_rank_rule_changed":
        False,

    "strong_rank_changed":
        False,

    "weak_rank_changed":
        False,

    "dominance_gate_changed":
        False,

    "gap_horizon_changed":
        False,

    "endpoint_eligibility_changed":
        False,

    "development_evidence_changed":
        False,

    "generator_selection_changed":
        False,

    "retuning":
        False,
}


patched[
    "supersedes"
] = {
    "generator_id":
        original[
            "generator_id"
        ],

    "generator_contract_sha256":
        original[
            "generator_contract_sha256"
        ],

    "artifact_path":
        str(
            ORIGINAL_FROZEN_PATH
        ),

    "artifact_file_sha256":
        file_sha256(
            ORIGINAL_FROZEN_PATH
        ),

    "reason":
        "IMPLEMENTATION_FIDELITY_CORRECTION",
}


# ============================================================
# 10. Strengthen parent provenance
# ============================================================

patched[
    "parent_implementation_fidelity"
] = {
    "g3f1_max_rank_query":
        int(
            g3f1_cardinality[
                "max_rank_query"
            ]
        ),

    "g3f1_asym3x8_raw_rows":
        int(
            g3f1_a38[
                "rows"
            ]
        ),

    "g3f2_asym3x8_raw_rows":
        int(
            g3f2_a38[
                "raw_rows"
            ]
        ),

    "g3f2_asym3x8_retained_rows":
        int(
            g3f2_a38[
                "retained_rows"
            ]
        ),

    "raw_cardinality_exactly_reproduced":
        True,

    "evidence_query_k":
        16,
}


# ============================================================
# 11. Reassert leakage / deployment state
# ============================================================

patched[
    "actual_generator_features_gt_blind"
] = True


patched[
    "generator_selected_with_development_gt"
] = True


patched[
    "fold0_used"
] = False


patched[
    "classifier_trained"
] = False


patched[
    "scoring_policy_selected"
] = False


patched[
    "rewire_policy_selected"
] = False


patched[
    "synthetic_nodes_inserted"
] = False


patched[
    "graph_modified"
] = False


patched[
    "future_retrieval_retuning_allowed"
] = False


# ============================================================
# 12. New canonical hash
# ============================================================

patched_sha = canonical_sha256(
    patched
)


patched[
    "generator_contract_sha256"
] = patched_sha


PATCHED_FROZEN_PATH.write_text(
    json.dumps(
        patched,
        indent=2,
    ),
    encoding="utf-8",
)


assert PATCHED_FROZEN_PATH.exists()


# ============================================================
# 13. Post-patch oracle confirmation
#
# No tuning. Read only to confirm unchanged evidence.
# ============================================================

oracle = pd.read_pickle(
    G3F1_ORACLE_PATH
)


assert len(
    oracle
) == 369


assert (
    "retrieval_ASYM_3x8"
    in
    oracle.columns
)


assert (
    "combined_ASYM_3x8"
    in
    oracle.columns
)


retrieval = int(
    oracle[
        "retrieval_ASYM_3x8"
    ].sum()
)


combined = int(
    oracle[
        "combined_ASYM_3x8"
    ].sum()
)


full7_mask = oracle[
    "pred_full_7um"
].astype(bool)


full7_total = int(
    full7_mask.sum()
)


retrieval_full7 = int(
    (
        oracle[
            "retrieval_ASYM_3x8"
        ].astype(bool)
        &
        full7_mask
    ).sum()
)


combined_full7 = int(
    (
        oracle[
            "combined_ASYM_3x8"
        ].astype(bool)
        &
        full7_mask
    ).sum()
)


assert retrieval == 236
assert combined == 212

assert full7_total == 285

assert retrieval_full7 == 201
assert combined_full7 == 178


# ============================================================
# 14. Fold confirmation
# ============================================================

fold_rows = []


for fold in [
    1,
    2,
    3,
    4,
]:

    sub = oracle[
        oracle[
            "fold"
        ].eq(
            fold
        )
    ]


    sub_full7 = sub[
        "pred_full_7um"
    ].astype(bool)


    fold_rows.append(
        {
            "fold":
                int(
                    fold
                ),

            "components":
                int(
                    len(
                        sub
                    )
                ),

            "retrieval":
                int(
                    sub[
                        "retrieval_ASYM_3x8"
                    ].sum()
                ),

            "combined":
                int(
                    sub[
                        "combined_ASYM_3x8"
                    ].sum()
                ),

            "combined_fraction":
                float(
                    sub[
                        "combined_ASYM_3x8"
                    ].mean()
                ),

            "full7_total":
                int(
                    sub_full7.sum()
                ),

            "combined_full7":
                int(
                    (
                        sub[
                            "combined_ASYM_3x8"
                        ].astype(bool)
                        &
                        sub_full7
                    ).sum()
                ),
        }
    )


fold_table = pd.DataFrame(
    fold_rows
)


# ============================================================
# 15. Patched confirmation artifact
# ============================================================

confirmation = {
    "stage":
        "12.11G3F3-R",

    "status":
        "FROZEN_GENERATOR_FIDELITY_PATCH_CONFIRMATION",

    "generator_id":
        patched[
            "generator_id"
        ],

    "generator_contract_sha256":
        patched_sha,

    "superseded_generator_id":
        original[
            "generator_id"
        ],

    "superseded_generator_sha256":
        original[
            "generator_contract_sha256"
        ],

    "fidelity_patch":
        "exact_knn_query_k: 8 -> 16",

    "selected_rank_rule":
        "ASYM_3x8",

    "retrieval_captured":
        retrieval,

    "combined_captured":
        combined,

    "full7_total":
        full7_total,

    "retrieval_full7":
        retrieval_full7,

    "combined_full7":
        combined_full7,

    "development_cardinality": {
        "raw_rows":
            36_473_971,

        "retained_rows":
            29_227_030,
    },

    "fold_results":
        fold_table.to_dict(
            "records"
        ),

    "development_evidence_changed":
        False,

    "retrieval_rule_retuned":
        False,

    "confirmation_is_not_retuning":
        True,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


PATCHED_CONFIRMATION_PATH.write_text(
    json.dumps(
        confirmation,
        indent=2,
    ),
    encoding="utf-8",
)


assert PATCHED_CONFIRMATION_PATH.exists()


# ============================================================
# 16. Supersession marker
# ============================================================

supersession = {
    "stage":
        "12.11G3F3-R",

    "status":
        "SUPERSESSION",

    "superseded": {
        "generator_id":
            original[
                "generator_id"
            ],

        "generator_contract_sha256":
            original[
                "generator_contract_sha256"
            ],

        "artifact":
            str(
                ORIGINAL_FROZEN_PATH
            ),

        "preserve_for_provenance":
            True,
    },

    "authoritative": {
        "generator_id":
            patched[
                "generator_id"
            ],

        "generator_contract_sha256":
            patched_sha,

        "artifact":
            str(
                PATCHED_FROZEN_PATH
            ),
    },

    "reason":
        (
            "Original G3F3 frozen implementation recorded "
            "KNN query K=8 despite G3F1/G3F2 evidence being "
            "generated with K=16. Selection semantics remain "
            "ASYM_3x8; only implementation fidelity is repaired."
        ),

    "retuning":
        False,

    "original_artifact_deleted":
        False,
}


SUPERSESSION_PATH.write_text(
    json.dumps(
        supersession,
        indent=2,
    ),
    encoding="utf-8",
)


assert SUPERSESSION_PATH.exists()


# ============================================================
# 17. Reload fidelity
# ============================================================

patched_check = json.loads(
    PATCHED_FROZEN_PATH.read_text(
        encoding="utf-8"
    )
)


assert patched_check[
    "generator_contract_sha256"
] == patched_sha


assert patched_check[
    "retrieval"
][
    "exact_knn_query_k"
] == 16


assert patched_check[
    "retrieval"
][
    "strong_rank"
] == 3


assert patched_check[
    "retrieval"
][
    "weak_rank"
] == 8


assert patched_check[
    "development_cardinality"
][
    "retained_rows"
] == 29_227_030


# ============================================================
# 18. Output
# ============================================================

print(
    "\n========== 12.11G3F3-R FIDELITY PATCH =========="
)

print(
    "original generator:",
    original[
        "generator_id"
    ]
)

print(
    "original SHA:",
    original[
        "generator_contract_sha256"
    ]
)

print(
    "original exact_knn_query_k:",
    original[
        "retrieval"
    ][
        "exact_knn_query_k"
    ]
)


print(
    "\nauthoritative generator:",
    patched[
        "generator_id"
    ]
)

print(
    "patched SHA:",
    patched_sha
)

print(
    "patched exact_knn_query_k:",
    patched[
        "retrieval"
    ][
        "exact_knn_query_k"
    ]
)

print(
    "rank rule:",
    "ASYM_3x8"
)

print(
    "rank rule changed:",
    "NO"
)

print(
    "dominance gate changed:",
    "NO"
)

print(
    "cardinality changed:",
    "NO"
)

print(
    "development evidence changed:",
    "NO"
)


print(
    "\n========== POST-PATCH CONFIRMATION =========="
)

print(
    "retrieval:",
    retrieval,
    "/ 369"
)

print(
    "combined:",
    combined,
    "/ 369"
)

print(
    "full<=7 retrieval:",
    retrieval_full7,
    "/",
    full7_total
)

print(
    "full<=7 combined:",
    combined_full7,
    "/",
    full7_total
)


print(
    "\nBy fold:"
)

print(
    fold_table.to_string(
        index=False
    )
)


print(
    "\n========== 12.11G3F3-R DECISION =========="
)

print(
    "ASYM3X8_FROZEN_GENERATOR_FIDELITY_PATCH: PASS"
)

print(
    "original G3F3 preserved:",
    "YES"
)

print(
    "original G3F3 superseded:",
    "YES"
)

print(
    "authoritative exact query K:",
    16
)

print(
    "selection changed:",
    "NO"
)

print(
    "retuning performed:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "classifier trained:",
    "NO"
)

print(
    "rewire policy selected:",
    "NO"
)

print(
    "graph modified:",
    "NO"
)

print(
    "next:"
)

print(
    "DESIGN_SECOND_STAGE_GT_BLIND_LOCAL_COMPETITION_SHORTLIST_ON_PATCHED_FROZEN_GENERATOR"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3G0-R
#
# PART-B ONLY REPAIR
#
# Repairs:
#
#   TypeError:
#       tuple indices must be integers or slices, not str
#
# Cause:
#
#   ds_oracle.itertuples(index=False) returns namedtuple.
#
# Incorrect:
#       row["combined_ASYM_3x8"]
#
# Correct:
#       getattr(row, "combined_ASYM_3x8")
#
# IMPORTANT:
#
#   G3G0 Part A already completed successfully and was frozen.
#   DO NOT rerun/recreate Part A.
#
# This cell:
#
#   1. reloads frozen G3G0 Part-A cardinality
#   2. validates authoritative frozen generator
#   3. starts directly from development-oracle Part B
#   4. computes true-pair local competition features
#   5. evaluates the already-frozen shortlist family
#   6. writes intended G3G0 oracle + summary artifacts
#
# No GT-blind cardinality is recomputed.
# No Fold0.
# No graph modification.
# No classifier.
# No shortlist selected.
#
# Keep after running: YES
# ============================================================

from pathlib import Path
import gc
import hashlib
import json

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


FROZEN_GENERATOR_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_generator_asym3x8_v1_fidelitypatch1_frozen.json"
)


CARDINALITY_OUT = (
    S12
    / "stage12_folds14_multiframe_local_competition_shortlist_cardinality_v1.json"
)


G3F1_ORACLE_PATH = (
    S12
    / "stage12_folds14_multiframe_asymmetric_retrieval_oracle_v1.pkl"
)


ORACLE_OUT = (
    S12
    / "stage12_folds14_multiframe_local_competition_shortlist_oracle_v1.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_folds14_multiframe_local_competition_shortlist_summary_v1.json"
)


REPAIR_PROVENANCE_OUT = (
    S12
    / "stage12_folds14_multiframe_local_competition_shortlist_partb_repair1.json"
)


for path in [
    FROZEN_GENERATOR_PATH,
    CARDINALITY_OUT,
    G3F1_ORACLE_PATH,
]:

    assert path.exists(), path


# Part B failed before these should have been written.
# Never silently overwrite a completed result.
for path in [
    ORACLE_OUT,
    SUMMARY_OUT,
    REPAIR_PROVENANCE_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3G0-R output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite an existing completed Part-B audit."
        )


# ============================================================
# 1. AUTHORITATIVE GENERATOR
# ============================================================

AUTHORITATIVE_SHA = (
    "da408df857cd34dcb4b1f65f3fae545e"
    "299f9403ef5edb986584caa8f4c3c203"
)


frozen = json.loads(
    FROZEN_GENERATOR_PATH.read_text(
        encoding="utf-8"
    )
)


assert frozen[
    "generator_contract_sha256"
] == AUTHORITATIVE_SHA


assert frozen[
    "generator_id"
] == (
    "MULTIFRAME_REWIRE_ASYM3X8_"
    "DOMINANCE_ANY_V1_FIDELITYPATCH1"
)


assert frozen[
    "retrieval"
][
    "exact_knn_query_k"
] == 16


assert frozen[
    "retrieval"
][
    "strong_rank"
] == 3


assert frozen[
    "retrieval"
][
    "weak_rank"
] == 8


# ============================================================
# 2. LOAD FROZEN G3G0 PART A
#
# This is the GT-blind artifact already produced successfully
# before the failed Part-B tuple access.
# ============================================================

part_a = json.loads(
    CARDINALITY_OUT.read_text(
        encoding="utf-8"
    )
)


assert part_a[
    "stage"
] == "12.11G3G0"


assert part_a[
    "part"
] == (
    "GT_BLIND_LOCAL_COMPETITION_"
    "SHORTLIST_CARDINALITY"
)


assert part_a[
    "authoritative_generator_sha256"
] == AUTHORITATIVE_SHA


assert part_a[
    "gt_used_during_part_a"
] is False


assert part_a[
    "fold0_used"
] is False


EXPECTED_RAW = int(
    part_a[
        "raw_generator_rows"
    ]
)


EXPECTED_RETAINED = int(
    part_a[
        "retained_generator_rows"
    ]
)


assert EXPECTED_RAW == 36_473_971
assert EXPECTED_RETAINED == 29_227_030


SHORTLISTS = part_a[
    "shortlist_family"
]


cardinality_table = pd.DataFrame(
    part_a[
        "cardinality"
    ]
)


assert (
    cardinality_table[
        "shortlist"
    ].nunique()
    ==
    len(
        SHORTLISTS
    )
)


all_frozen_cardinality = int(
    cardinality_table.loc[
        cardinality_table[
            "shortlist"
        ].eq(
            "ALL_FROZEN"
        ),
        "rows",
    ].iloc[
        0
    ]
)


assert all_frozen_cardinality == EXPECTED_RETAINED


print(
    "========== 12.11G3G0-R PART-A FIDELITY =========="
)

print(
    "authoritative generator:",
    frozen[
        "generator_id"
    ]
)

print(
    "generator SHA:",
    AUTHORITATIVE_SHA
)

print(
    "Part-A SHA:",
    part_a[
        "sha256"
    ]
)

print(
    "GT-blind raw generator:",
    EXPECTED_RAW
)

print(
    "GT-blind retained generator:",
    EXPECTED_RETAINED
)

print(
    "shortlist family size:",
    len(
        SHORTLISTS
    )
)

print(
    "Part A reused without recomputation: YES"
)


# ============================================================
# 3. CURRENT_BEST BUILDER
# ============================================================

assert (
    "build_current_best"
    in globals()
    and
    callable(
        globals()[
            "build_current_best"
        ]
    )
), (
    "build_current_best is missing from the current kernel. "
    "Re-run the retained successful G3E-R setup/builder cell, "
    "but DO NOT rerun G3G0 Part A."
)


current_best_builder = globals()[
    "build_current_best"
]


# ============================================================
# 4. FIXED CONTRACT
# ============================================================

SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


QUERY_K = 16
STRONG_RANK = 3
WEAK_RANK = 8

EPS = 1e-12


# ============================================================
# 5. HELPERS
# ============================================================

def safe_ratio(
    candidate,
    best,
):

    candidate = float(
        candidate
    )

    best = float(
        best
    )


    if best <= EPS:

        if candidate <= EPS:

            return 1.0

        return np.inf


    return float(
        candidate
        /
        best
    )


def geometric_mean_ratio(
    a,
    b,
):

    if (
        not np.isfinite(
            a
        )
        or
        not np.isfinite(
            b
        )
    ):

        return np.inf


    return float(
        np.sqrt(
            float(
                a
            )
            *
            float(
                b
            )
        )
    )


def asym3x8_pass(
    source_rank,
    target_rank,
):

    return bool(
        (
            source_rank <= STRONG_RANK
            and
            target_rank <= WEAK_RANK
        )
        or
        (
            target_rank <= STRONG_RANK
            and
            source_rank <= WEAK_RANK
        )
    )


def anchor_peer_ratios(
    source_rank,
    target_rank,
    source_ratio,
    target_ratio,
):

    source_anchor = (
        source_rank
        <=
        STRONG_RANK
    )


    target_anchor = (
        target_rank
        <=
        STRONG_RANK
    )


    assert (
        source_anchor
        or
        target_anchor
    )


    if (
        source_anchor
        and
        target_anchor
    ):

        return (
            min(
                source_ratio,
                target_ratio,
            ),

            max(
                source_ratio,
                target_ratio,
            ),
        )


    if source_anchor:

        return (
            source_ratio,
            target_ratio,
        )


    return (
        target_ratio,
        source_ratio,
    )


def shortlist_pass(
    name,
    features,
):

    spec = SHORTLISTS[
        name
    ]


    typ = spec[
        "type"
    ]


    if typ == "ALL":

        return True


    if typ == "MAX_RATIO":

        return bool(
            features[
                "max_ratio"
            ]
            <=
            float(
                spec[
                    "threshold"
                ]
            )
        )


    if typ == "GMEAN_RATIO":

        return bool(
            features[
                "gmean_ratio"
            ]
            <=
            float(
                spec[
                    "threshold"
                ]
            )
        )


    if typ == "MAX_EXCESS_PF":

        return bool(
            features[
                "max_excess_pf"
            ]
            <=
            float(
                spec[
                    "threshold"
                ]
            )
        )


    if typ == "ANCHOR_PEER_RATIO":

        return bool(
            features[
                "anchor_ratio"
            ]
            <=
            float(
                spec[
                    "anchor_threshold"
                ]
            )
            and
            features[
                "peer_ratio"
            ]
            <=
            float(
                spec[
                    "peer_threshold"
                ]
            )
        )


    raise ValueError(
        (
            name,
            spec,
        )
    )


def query_knn(
    tree,
    query_points,
    candidate_count,
    k,
):

    k_eff = min(
        int(
            k
        ),
        int(
            candidate_count
        ),
    )


    distances, indices = tree.query(
        query_points,
        k=k_eff,
    )


    if k_eff == 1:

        distances = distances[
            :,
            None
        ]

        indices = indices[
            :,
            None
        ]


    return (
        np.asarray(
            distances,
            dtype=float,
        ),

        np.asarray(
            indices,
            dtype=int,
        ),
    )


def build_rank_distance_map(
    query_ids,
    candidate_ids,
    distances,
    indices,
    reverse=False,
):

    result = {}


    for qi, query_id in enumerate(
        query_ids
    ):

        for ri in range(
            indices.shape[
                1
            ]
        ):

            candidate_id = int(
                candidate_ids[
                    int(
                        indices[
                            qi,
                            ri
                        ]
                    )
                ]
            )


            if reverse:

                pair = (
                    candidate_id,
                    int(
                        query_id
                    ),
                )

            else:

                pair = (
                    int(
                        query_id
                    ),
                    candidate_id,
                )


            if pair not in result:

                result[
                    pair
                ] = (
                    int(
                        ri
                        +
                        1
                    ),

                    float(
                        distances[
                            qi,
                            ri
                        ]
                    ),
                )


    return result


# ============================================================
# PART B — DEVELOPMENT ORACLE
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — REPAIRED DEVELOPMENT ORACLE LOCAL COMPETITION AUDIT"
)

print(
    "============================================================"
)


oracle = pd.read_pickle(
    G3F1_ORACLE_PATH
)


assert len(
    oracle
) == 369


assert (
    "combined_ASYM_3x8"
    in
    oracle.columns
)


assert int(
    oracle[
        "combined_ASYM_3x8"
    ].sum()
) == 212


oracle_datasets = (
    oracle[
        "dataset"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)


assert len(
    oracle_datasets
) == 75


# ============================================================
# 6. TRUE-PAIR LOCAL COMPETITION FEATURES
# ============================================================

oracle_feature_rows = []


for ds_i, dataset in enumerate(
    oracle_datasets,
    start=1,
):

    graph, scale = current_best_builder(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    )


    # ========================================================
    # Nodes
    # ========================================================

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = nodes[
        "node_id"
    ].astype(int)


    nodes[
        "t"
    ] = nodes[
        "t"
    ].astype(int)


    nodes = (
        nodes
        .sort_values(
            [
                "t",
                "node_id",
            ],
            kind="stable",
        )
        .set_index(
            "node_id",
            drop=False,
        )
    )


    # ========================================================
    # Edges
    # ========================================================

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = edges[
        "source_id"
    ].astype(int)


    edges[
        "target_id"
    ] = edges[
        "target_id"
    ].astype(int)


    incoming = (
        edges.groupby(
            "target_id"
        )[
            "source_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    outgoing = (
        edges.groupby(
            "source_id"
        )[
            "target_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = {
        int(k):
            int(v)

        for k, v
        in indeg.items()
    }


    outdeg = {
        int(k):
            int(v)

        for k, v
        in outdeg.items()
    }


    # ========================================================
    # Physical coordinates
    # ========================================================

    phys = {}

    time_map = {}


    for nid, row in nodes.iterrows():

        nid = int(
            nid
        )


        phys[
            nid
        ] = (
            np.array(
                [
                    float(
                        row[
                            "z"
                        ]
                    ),

                    float(
                        row[
                            "y"
                        ]
                    ),

                    float(
                        row[
                            "x"
                        ]
                    ),
                ],
                dtype=float,
            )
            *
            SCALE
        )


        time_map[
            nid
        ] = int(
            row[
                "t"
            ]
        )


    node_ids = sorted(
        phys
    )


    # ========================================================
    # Velocities
    # ========================================================

    past_velocity = {}

    future_velocity = {}


    for nid in node_ids:

        inc = incoming.get(
            nid,
            [],
        )


        if len(
            inc
        ) == 1:

            prev_id = int(
                inc[
                    0
                ]
            )


            if prev_id in phys:

                dt = (
                    time_map[
                        nid
                    ]
                    -
                    time_map[
                        prev_id
                    ]
                )


                if dt > 0:

                    past_velocity[
                        nid
                    ] = (
                        phys[
                            nid
                        ]
                        -
                        phys[
                            prev_id
                        ]
                    ) / float(
                        dt
                    )


        out = outgoing.get(
            nid,
            [],
        )


        if len(
            out
        ) == 1:

            next_id = int(
                out[
                    0
                ]
            )


            if next_id in phys:

                dt = (
                    time_map[
                        next_id
                    ]
                    -
                    time_map[
                        nid
                    ]
                )


                if dt > 0:

                    future_velocity[
                        nid
                    ] = (
                        phys[
                            next_id
                        ]
                        -
                        phys[
                            nid
                        ]
                    ) / float(
                        dt
                    )


    # ========================================================
    # Eligibility
    # ========================================================

    source_eligible = [
        nid

        for nid in node_ids

        if (
            outdeg.get(
                nid,
                0,
            )
            <= 1
            and
            nid in past_velocity
        )
    ]


    target_eligible = [
        nid

        for nid in node_ids

        if (
            indeg.get(
                nid,
                0,
            )
            <= 1
            and
            nid in future_velocity
        )
    ]


    source_by_t = {}

    target_by_t = {}


    for nid in source_eligible:

        source_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for nid in target_eligible:

        target_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for t in source_by_t:

        source_by_t[
            t
        ] = sorted(
            source_by_t[
                t
            ]
        )


    for t in target_by_t:

        target_by_t[
            t
        ] = sorted(
            target_by_t[
                t
            ]
        )


    # ========================================================
    # Dataset oracle
    # ========================================================

    ds_oracle = oracle[
        oracle[
            "dataset"
        ].eq(
            dataset
        )
    ]


    local = {}

    frame_groups = {}


    for row in ds_oracle.itertuples(
        index=False
    ):

        key = (
            dataset,
            row.component_id,
        )


        local[
            key
        ] = {
            "dataset":
                dataset,

            "component_id":
                row.component_id,

            "source_ratio":
                np.nan,

            "target_ratio":
                np.nan,

            "max_ratio":
                np.nan,

            "gmean_ratio":
                np.nan,

            "source_excess_pf":
                np.nan,

            "target_excess_pf":
                np.nan,

            "max_excess_pf":
                np.nan,

            "anchor_ratio":
                np.nan,

            "peer_ratio":
                np.nan,
        }


        # ====================================================
        # FIX:
        #
        # itertuples() -> namedtuple
        #
        # OLD INVALID:
        # row["combined_ASYM_3x8"]
        #
        # CORRECT:
        # getattr(row, "combined_ASYM_3x8")
        # ====================================================

        if not bool(
            getattr(
                row,
                "combined_ASYM_3x8"
            )
        ):

            continue


        src = int(
            row.source_id
        )


        tgt = int(
            row.target_id
        )


        # A combined=true pair must be generator-eligible.
        assert src in past_velocity
        assert tgt in future_velocity


        source_t = int(
            time_map[
                src
            ]
        )


        target_t = int(
            time_map[
                tgt
            ]
        )


        assert source_t in source_by_t
        assert target_t in target_by_t


        frame_groups.setdefault(
            (
                source_t,
                target_t,
            ),
            [],
        ).append(
            key
        )


    # ========================================================
    # Exact authoritative K16 frame-pair queries
    # ========================================================

    for (
        source_t,
        target_t,
    ), keys in frame_groups.items():

        source_ids = source_by_t[
            source_t
        ]


        target_ids = target_by_t[
            target_t
        ]


        delta_t = (
            target_t
            -
            source_t
        )


        assert delta_t > 0


        source_pos = np.vstack(
            [
                phys[
                    nid
                ]

                for nid
                in source_ids
            ]
        )


        target_pos = np.vstack(
            [
                phys[
                    nid
                ]

                for nid
                in target_ids
            ]
        )


        source_vel = np.vstack(
            [
                past_velocity[
                    nid
                ]

                for nid
                in source_ids
            ]
        )


        target_vel = np.vstack(
            [
                future_velocity[
                    nid
                ]

                for nid
                in target_ids
            ]
        )


        source_pred = (
            source_pos
            +
            float(
                delta_t
            )
            *
            source_vel
        )


        target_pred = (
            target_pos
            -
            float(
                delta_t
            )
            *
            target_vel
        )


        target_tree = cKDTree(
            target_pos
        )


        source_tree = cKDTree(
            source_pos
        )


        (
            source_dist,
            source_idx,
        ) = query_knn(
            target_tree,
            source_pred,
            len(
                target_ids
            ),
            QUERY_K,
        )


        (
            target_dist,
            target_idx,
        ) = query_knn(
            source_tree,
            target_pred,
            len(
                source_ids
            ),
            QUERY_K,
        )


        source_map = build_rank_distance_map(
            source_ids,
            target_ids,
            source_dist,
            source_idx,
            reverse=False,
        )


        target_map = build_rank_distance_map(
            target_ids,
            source_ids,
            target_dist,
            target_idx,
            reverse=True,
        )


        source_best = {
            int(
                source_ids[
                    i
                ]
            ):
                float(
                    source_dist[
                        i,
                        0
                    ]
                )

            for i
            in range(
                len(
                    source_ids
                )
            )
        }


        target_best = {
            int(
                target_ids[
                    i
                ]
            ):
                float(
                    target_dist[
                        i,
                        0
                    ]
                )

            for i
            in range(
                len(
                    target_ids
                )
            )
        }


        # ====================================================
        # Resolve TRUE pairs
        # ====================================================

        for key in keys:

            rec = local[
                key
            ]


            oracle_match = ds_oracle[
                ds_oracle[
                    "component_id"
                ].eq(
                    key[
                        1
                    ]
                )
            ]


            assert len(
                oracle_match
            ) == 1


            oracle_row = oracle_match.iloc[
                0
            ]


            src = int(
                oracle_row[
                    "source_id"
                ]
            )


            tgt = int(
                oracle_row[
                    "target_id"
                ]
            )


            assert (
                src,
                tgt,
            ) in source_map


            assert (
                src,
                tgt,
            ) in target_map


            (
                source_rank,
                source_candidate_dist,
            ) = source_map[
                (
                    src,
                    tgt,
                )
            ]


            (
                target_rank,
                target_candidate_dist,
            ) = target_map[
                (
                    src,
                    tgt,
                )
            ]


            assert asym3x8_pass(
                source_rank,
                target_rank,
            )


            # ------------------------------------------------
            # Ratio against endpoint-local best candidate
            # ------------------------------------------------

            source_ratio = safe_ratio(
                source_candidate_dist,
                source_best[
                    src
                ],
            )


            target_ratio = safe_ratio(
                target_candidate_dist,
                target_best[
                    tgt
                ],
            )


            # ------------------------------------------------
            # Absolute per-frame excess
            # ------------------------------------------------

            source_excess_pf = max(
                0.0,
                (
                    float(
                        source_candidate_dist
                    )
                    -
                    float(
                        source_best[
                            src
                        ]
                    )
                )
                /
                float(
                    delta_t
                ),
            )


            target_excess_pf = max(
                0.0,
                (
                    float(
                        target_candidate_dist
                    )
                    -
                    float(
                        target_best[
                            tgt
                        ]
                    )
                )
                /
                float(
                    delta_t
                ),
            )


            (
                anchor_ratio,
                peer_ratio,
            ) = anchor_peer_ratios(
                source_rank,
                target_rank,
                source_ratio,
                target_ratio,
            )


            rec[
                "source_ratio"
            ] = float(
                source_ratio
            )


            rec[
                "target_ratio"
            ] = float(
                target_ratio
            )


            rec[
                "max_ratio"
            ] = float(
                max(
                    source_ratio,
                    target_ratio,
                )
            )


            rec[
                "gmean_ratio"
            ] = float(
                geometric_mean_ratio(
                    source_ratio,
                    target_ratio,
                )
            )


            rec[
                "source_excess_pf"
            ] = float(
                source_excess_pf
            )


            rec[
                "target_excess_pf"
            ] = float(
                target_excess_pf
            )


            rec[
                "max_excess_pf"
            ] = float(
                max(
                    source_excess_pf,
                    target_excess_pf,
                )
            )


            rec[
                "anchor_ratio"
            ] = float(
                anchor_ratio
            )


            rec[
                "peer_ratio"
            ] = float(
                peer_ratio
            )


        # ====================================================
        # Frame cleanup
        # ====================================================

        del source_pos
        del target_pos

        del source_vel
        del target_vel

        del source_pred
        del target_pred

        del target_tree
        del source_tree

        del source_dist
        del source_idx

        del target_dist
        del target_idx

        del source_map
        del target_map

        del source_best
        del target_best


    oracle_feature_rows.extend(
        local.values()
    )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            oracle_datasets
        )
    ):

        print(
            f"{ds_i:2d}/{len(oracle_datasets)}"
            f" | {dataset}"
            f" | oracle={len(ds_oracle)}"
            f" | frozen_true="
            f"{int(ds_oracle['combined_ASYM_3x8'].sum())}"
        )


    # ========================================================
    # Dataset cleanup
    # ========================================================

    del graph
    del nodes
    del edges

    del incoming
    del outgoing

    del indeg
    del outdeg

    del phys
    del time_map
    del node_ids

    del past_velocity
    del future_velocity

    del source_eligible
    del target_eligible

    del source_by_t
    del target_by_t

    del local
    del frame_groups

    gc.collect()


# ============================================================
# 7. ATTACH LOCAL FEATURES
# ============================================================

feature_table = pd.DataFrame(
    oracle_feature_rows
)


assert len(
    feature_table
) == 369


assert not feature_table[
    [
        "dataset",
        "component_id",
    ]
].duplicated().any()


oracle = oracle.merge(
    feature_table,
    on=[
        "dataset",
        "component_id",
    ],
    how="left",
    validate="one_to_one",
)


captured_mask = oracle[
    "combined_ASYM_3x8"
].astype(bool)


assert int(
    captured_mask.sum()
) == 212


for col in [
    "source_ratio",
    "target_ratio",
    "max_ratio",
    "gmean_ratio",
    "source_excess_pf",
    "target_excess_pf",
    "max_excess_pf",
    "anchor_ratio",
    "peer_ratio",
]:

    assert oracle.loc[
        captured_mask,
        col,
    ].notna().all(), col


print(
    "\nTRUE_PAIR_LOCAL_FEATURE_RECONSTRUCTION: PASS"
)


# ============================================================
# 8. TRUE-PAIR FEATURE DISTRIBUTIONS
# ============================================================

print(
    "\n========== TRUE-PAIR LOCAL COMPETITION FEATURES =========="
)


FEATURE_COLS = [
    "source_ratio",
    "target_ratio",
    "max_ratio",
    "gmean_ratio",

    "source_excess_pf",
    "target_excess_pf",
    "max_excess_pf",

    "anchor_ratio",
    "peer_ratio",
]


for col in FEATURE_COLS:

    print(
        "\n" + col
    )


    print(
        oracle.loc[
            captured_mask,
            col,
        ]
        .describe(
            percentiles=[
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99,
            ]
        )
        .to_string()
    )


# ============================================================
# 9. APPLY FROZEN SHORTLIST FAMILY
# ============================================================

oracle_results = []


for shortlist_name in SHORTLISTS:

    passes = []


    for row in oracle.itertuples(
        index=False
    ):

        if not bool(
            getattr(
                row,
                "combined_ASYM_3x8"
            )
        ):

            passes.append(
                False
            )

            continue


        features = {
            "max_ratio":
                float(
                    row.max_ratio
                ),

            "gmean_ratio":
                float(
                    row.gmean_ratio
                ),

            "max_excess_pf":
                float(
                    row.max_excess_pf
                ),

            "anchor_ratio":
                float(
                    row.anchor_ratio
                ),

            "peer_ratio":
                float(
                    row.peer_ratio
                ),
        }


        passes.append(
            shortlist_pass(
                shortlist_name,
                features,
            )
        )


    col = (
        "shortlist_"
        +
        shortlist_name
    )


    oracle[
        col
    ] = np.asarray(
        passes,
        dtype=bool,
    )


    captured = int(
        oracle[
            col
        ].sum()
    )


    full7_mask = oracle[
        "pred_full_7um"
    ].astype(bool)


    full7 = int(
        (
            oracle[
                col
            ]
            &
            full7_mask
        ).sum()
    )


    card = cardinality_table[
        cardinality_table[
            "shortlist"
        ].eq(
            shortlist_name
        )
    ]


    assert len(
        card
    ) == 1


    candidate_rows = int(
        card[
            "rows"
        ].iloc[
            0
        ]
    )


    oracle_results.append(
        {
            "shortlist":
                shortlist_name,

            "candidate_rows":
                candidate_rows,

            "candidate_fraction_of_frozen":
                float(
                    candidate_rows
                    /
                    EXPECTED_RETAINED
                ),

            "oracle_captured":
                captured,

            "oracle_fraction_of_frozen_true":
                float(
                    captured
                    /
                    212
                ),

            "oracle_overall_coverage":
                float(
                    captured
                    /
                    369
                ),

            "full7_captured":
                full7,

            "full7_fraction_of_frozen_true":
                float(
                    full7
                    /
                    178
                ),

            "full7_overall_coverage":
                float(
                    full7
                    /
                    285
                ),
        }
    )


oracle_table = pd.DataFrame(
    oracle_results
)


oracle_table = (
    oracle_table
    .sort_values(
        [
            "candidate_rows",
            "shortlist",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n========== LOCAL-COMPETITION ORACLE COVERAGE =========="
)


print(
    oracle_table.to_string(
        index=False
    )
)


# ============================================================
# 10. COMPRESSION / RECALL TRADEOFF
# ============================================================

tradeoff = oracle_table.copy()


tradeoff[
    "compression_factor"
] = (
    EXPECTED_RETAINED
    /
    tradeoff[
        "candidate_rows"
    ]
)


tradeoff[
    "oracle_retention_minus_candidate_retention"
] = (
    tradeoff[
        "oracle_fraction_of_frozen_true"
    ]
    -
    tradeoff[
        "candidate_fraction_of_frozen"
    ]
)


tradeoff[
    "oracle_components_per_million_rows"
] = (
    tradeoff[
        "oracle_captured"
    ]
    /
    (
        tradeoff[
            "candidate_rows"
        ]
        /
        1_000_000.0
    )
)


print(
    "\n========== LOCAL-COMPETITION CARDINALITY / RECALL TRADEOFF =========="
)


print(
    tradeoff[
        [
            "shortlist",

            "candidate_rows",
            "candidate_fraction_of_frozen",
            "compression_factor",

            "oracle_captured",
            "oracle_fraction_of_frozen_true",

            "full7_captured",
            "full7_fraction_of_frozen_true",

            "oracle_retention_minus_candidate_retention",
            "oracle_components_per_million_rows",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 11. ORACLE RETENTION BY FOLD
# ============================================================

oracle_fold_rows = []


for shortlist_name in SHORTLISTS:

    col = (
        "shortlist_"
        +
        shortlist_name
    )


    for fold in [
        1,
        2,
        3,
        4,
    ]:

        sub = oracle[
            oracle[
                "fold"
            ].eq(
                fold
            )
        ]


        frozen_true = int(
            sub[
                "combined_ASYM_3x8"
            ].sum()
        )


        retained_true = int(
            sub[
                col
            ].sum()
        )


        oracle_fold_rows.append(
            {
                "shortlist":
                    shortlist_name,

                "fold":
                    fold,

                "frozen_true":
                    frozen_true,

                "retained_true":
                    retained_true,

                "retention_fraction":
                    float(
                        retained_true
                        /
                        frozen_true
                    )
                    if frozen_true
                    else np.nan,
            }
        )


oracle_fold_table = pd.DataFrame(
    oracle_fold_rows
)


print(
    "\n========== LOCAL-COMPETITION ORACLE RETENTION BY FOLD =========="
)


for shortlist_name in SHORTLISTS:

    print(
        "\n" + shortlist_name
    )


    print(
        oracle_fold_table[
            oracle_fold_table[
                "shortlist"
            ].eq(
                shortlist_name
            )
        ].to_string(
            index=False
        )
    )


# ============================================================
# 12. PERSIST REPAIRED PART-B ORACLE
# ============================================================

oracle = (
    oracle
    .sort_values(
        [
            "fold",
            "dataset",
            "component_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


oracle.to_pickle(
    ORACLE_OUT
)


# ============================================================
# 13. SUMMARY
# ============================================================

summary = {
    "stage":
        "12.11G3G0-R",

    "status":
        "LOCAL_COMPETITION_SHORTLIST_FAMILY_AUDITED_PARTB_REPAIR1",

    "repair_of":
        "12.11G3G0_PARTB_TUPLE_ACCESS_ERROR",

    "repair":
        (
            "Replaced tuple string indexing of "
            "combined_ASYM_3x8 with namedtuple attribute access."
        ),

    "authoritative_generator_id":
        frozen[
            "generator_id"
        ],

    "authoritative_generator_sha256":
        AUTHORITATIVE_SHA,

    "part_a_artifact":
        str(
            CARDINALITY_OUT
        ),

    "part_a_sha256":
        part_a[
            "sha256"
        ],

    "part_a_recomputed":
        False,

    "part_a_gt_blind":
        True,

    "shortlist_family":
        SHORTLISTS,

    "tradeoff":
        tradeoff.to_dict(
            "records"
        ),

    "oracle_fold_retention":
        oracle_fold_table.to_dict(
            "records"
        ),

    "frozen_generator_exact_cardinality":
        {
            "raw":
                EXPECTED_RAW,

            "retained":
                EXPECTED_RETAINED,
        },

    "combined_asym3x8_true_pairs":
        212,

    "shortlist_selected":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert SUMMARY_OUT.exists()


# ============================================================
# 14. REPAIR PROVENANCE
# ============================================================

repair = {
    "stage":
        "12.11G3G0-R",

    "status":
        "PARTB_REPAIR",

    "error":
        (
            "TypeError: tuple indices must be integers "
            "or slices, not str"
        ),

    "root_cause":
        (
            "Pandas itertuples returned namedtuple but original "
            "G3G0 Part B used row['combined_ASYM_3x8']."
        ),

    "fix":
        (
            "Use getattr(row, 'combined_ASYM_3x8') "
            "for namedtuple access."
        ),

    "part_a_preserved":
        True,

    "part_a_recomputed":
        False,

    "part_a_sha256":
        part_a[
            "sha256"
        ],

    "authoritative_generator_sha256":
        AUTHORITATIVE_SHA,

    "oracle_output":
        str(
            ORACLE_OUT
        ),

    "summary_output":
        str(
            SUMMARY_OUT
        ),

    "scientific_contract_changed":
        False,

    "shortlist_family_changed":
        False,

    "thresholds_changed":
        False,

    "fold0_used":
        False,

    "retuning":
        False,
}


REPAIR_PROVENANCE_OUT.write_text(
    json.dumps(
        repair,
        indent=2,
    ),
    encoding="utf-8",
)


assert REPAIR_PROVENANCE_OUT.exists()


# ============================================================
# 15. RELOAD INTEGRITY
# ============================================================

check = pd.read_pickle(
    ORACLE_OUT
)


assert len(
    check
) == 369


assert not check[
    [
        "dataset",
        "component_id",
    ]
].duplicated().any()


assert int(
    check[
        "combined_ASYM_3x8"
    ].sum()
) == 212


assert int(
    check[
        "shortlist_ALL_FROZEN"
    ].sum()
) == 212


for shortlist_name in SHORTLISTS:

    assert (
        "shortlist_"
        +
        shortlist_name
    ) in check.columns


# ============================================================
# 16. DECISION
# ============================================================

print(
    "\n========== 12.11G3G0-R DECISION =========="
)

print(
    "LOCAL_COMPETITION_SHORTLIST_PARTB_REPAIR: PASS"
)

print(
    "original error repaired:",
    "YES"
)

print(
    "Part A reused:",
    "YES"
)

print(
    "Part A recomputed:",
    "NO"
)

print(
    "Part A GT-blind freeze preserved:",
    "YES"
)

print(
    "authoritative generator SHA:",
    AUTHORITATIVE_SHA
)

print(
    "frozen generator cardinality:",
    EXPECTED_RETAINED
)

print(
    "oracle frozen true pairs:",
    212
)

print(
    "shortlist selected:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "classifier trained:",
    "NO"
)

print(
    "rewire policy selected:",
    "NO"
)

print(
    "graph modified:",
    "NO"
)

print(
    "next:"
)

print(
    "SELECT_LOCAL_COMPETITION_PARETO_SHORTLIST_THEN_BUILD_CROSS_FOLD_SCORING"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3G1
#
# Endpoint-wise Competition Shortlist Family Audit
#
# Authoritative frozen generator:
#
#   MULTIFRAME_REWIRE_ASYM3X8_DOMINANCE_ANY_V1_FIDELITYPATCH1
#
#   SHA:
#   da408df857cd34dcb4b1f65f3fae545e299f9403ef5edb986584caa8f4c3c203
#
# ------------------------------------------------------------
# MOTIVATION
# ------------------------------------------------------------
#
# G3G0 showed that global local-competition thresholds are
# useful but not sufficient:
#
#   ~50% candidate retention -> only ~71% true-pair retention
#
# We therefore replace hard global gates with endpoint-wise
# competition.
#
# For every retained frozen-generator proposal, define:
#
#   SYM_CV:
#       0.5 * (source_cv_per_frame + target_cv_per_frame)
#
#   MAX_CV:
#       max(source_cv_per_frame, target_cv_per_frame)
#
#   GMEAN_RATIO:
#       sqrt(source_local_ratio * target_local_ratio)
#
# For each score:
#
#   source competition rank:
#       rank among ALL frozen proposals sharing source_id
#
#   target competition rank:
#       rank among ALL frozen proposals sharing target_id
#
# Shortlist:
#
#       source_rank <= N
#           OR
#       target_rank <= N
#
# for:
#
#       N = 1, 2, 3, 5, 8
#
# ------------------------------------------------------------
# PART A
# ------------------------------------------------------------
#
# GT blind.
#
# - Regenerate authoritative frozen generator streamingly,
#   one dataset at a time.
# - Exact fidelity:
#
#       RAW       = 36,473,971
#       RETAINED  = 29,227,030
#
# - Compute endpoint-wise shortlist cardinality.
# - Persist NO proposal universe.
# - Freeze cardinality before GT read.
#
# ------------------------------------------------------------
# PART B
# ------------------------------------------------------------
#
# Only after Part A freeze:
#
# - read development oracle
# - regenerate only 75 oracle datasets
# - measure true-pair endpoint competition ranks
# - compare candidate cardinality vs oracle retention
#
# No Fold0.
# No classifier.
# No final shortlist selected.
# No graph modification.
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import gc
import hashlib
import json

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


FROZEN_GENERATOR_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_generator_asym3x8_v1_fidelitypatch1_frozen.json"
)


G3F1_ORACLE_PATH = (
    S12
    / "stage12_folds14_multiframe_asymmetric_retrieval_oracle_v1.pkl"
)


CARDINALITY_OUT = (
    S12
    / "stage12_folds14_endpoint_competition_shortlist_cardinality_v1.json"
)


ORACLE_OUT = (
    S12
    / "stage12_folds14_endpoint_competition_shortlist_oracle_v1.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_folds14_endpoint_competition_shortlist_summary_v1.json"
)


assert FROZEN_GENERATOR_PATH.exists()
assert G3F1_ORACLE_PATH.exists()


for path in [
    CARDINALITY_OUT,
    ORACLE_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3G1 output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite a completed G3G1 audit."
        )


# ============================================================
# 1. AUTHORITATIVE GENERATOR
# ============================================================

AUTHORITATIVE_SHA = (
    "da408df857cd34dcb4b1f65f3fae545e"
    "299f9403ef5edb986584caa8f4c3c203"
)


frozen = json.loads(
    FROZEN_GENERATOR_PATH.read_text(
        encoding="utf-8"
    )
)


assert frozen[
    "generator_contract_sha256"
] == AUTHORITATIVE_SHA


assert frozen[
    "generator_id"
] == (
    "MULTIFRAME_REWIRE_ASYM3X8_"
    "DOMINANCE_ANY_V1_FIDELITYPATCH1"
)


assert frozen[
    "retrieval"
][
    "exact_knn_query_k"
] == 16


assert frozen[
    "retrieval"
][
    "strong_rank"
] == 3


assert frozen[
    "retrieval"
][
    "weak_rank"
] == 8


EXPECTED_RAW = 36_473_971
EXPECTED_RETAINED = 29_227_030


assert frozen[
    "development_cardinality"
][
    "raw_rows"
] == EXPECTED_RAW


assert frozen[
    "development_cardinality"
][
    "retained_rows"
] == EXPECTED_RETAINED


print(
    "========== 12.11G3G1 AUTHORITATIVE GENERATOR =========="
)

print(
    "generator:",
    frozen[
        "generator_id"
    ]
)

print(
    "SHA:",
    AUTHORITATIVE_SHA
)

print(
    "query K:",
    16
)

print(
    "retrieval:",
    "ASYM_3x8"
)

print(
    "expected raw:",
    EXPECTED_RAW
)

print(
    "expected retained:",
    EXPECTED_RETAINED
)


# ============================================================
# 2. CURRENT_BEST
# ============================================================

assert (
    "build_current_best"
    in globals()
    and
    callable(
        globals()[
            "build_current_best"
        ]
    )
), (
    "build_current_best missing. "
    "Re-run retained G3E-R builder/setup first."
)


current_best_builder = globals()[
    "build_current_best"
]


# ============================================================
# 3. DEVELOPMENT CORPUS
# ============================================================

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


cv = pd.read_csv(
    CV_PATH
)


dev_manifest = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
)


development_names = (
    dev_manifest[
        "dataset"
    ]
    .tolist()
)


assert len(
    development_names
) == 103


fold_map_local = dict(
    zip(
        dev_manifest[
            "dataset"
        ],
        dev_manifest[
            "fold"
        ].astype(int),
    )
)


# ============================================================
# 4. FROZEN CONSTANTS
# ============================================================

SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


MIN_GAP_SIZE = 2
MAX_GAP_SIZE = 12

QUERY_K = 16

STRONG_RANK = 3
WEAK_RANK = 8

EPS = 1e-12


# ============================================================
# 5. FIXED SHORTLIST FAMILY
#
# Defined before GT read.
# ============================================================

SCORES = [
    "SYM_CV",
    "MAX_CV",
    "GMEAN_RATIO",
]


N_VALUES = [
    1,
    2,
    3,
    5,
    8,
]


SHORTLISTS = [
    f"{score}_TOP{n}"

    for score in SCORES

    for n in N_VALUES
]


print(
    "\n========== FIXED ENDPOINT SHORTLIST FAMILY =========="
)


for name in SHORTLISTS:

    print(
        name
    )


# ============================================================
# 6. HELPERS
# ============================================================

def vec_norm(
    x,
):

    return float(
        np.linalg.norm(
            np.asarray(
                x,
                dtype=float,
            )
        )
    )


def safe_ratio(
    candidate,
    best,
):

    candidate = float(
        candidate
    )

    best = float(
        best
    )


    if best <= EPS:

        if candidate <= EPS:

            return 1.0

        return np.inf


    return float(
        candidate
        /
        best
    )


def query_knn(
    tree,
    query_points,
    candidate_count,
    k,
):

    k_eff = min(
        int(
            k
        ),
        int(
            candidate_count
        ),
    )


    distances, indices = tree.query(
        query_points,
        k=k_eff,
    )


    if k_eff == 1:

        distances = distances[
            :,
            None
        ]

        indices = indices[
            :,
            None
        ]


    return (
        np.asarray(
            distances,
            dtype=float,
        ),

        np.asarray(
            indices,
            dtype=int,
        ),
    )


def build_rank_distance_map(
    query_ids,
    candidate_ids,
    distances,
    indices,
    reverse=False,
):

    result = {}


    for qi, query_id in enumerate(
        query_ids
    ):

        for ri in range(
            indices.shape[
                1
            ]
        ):

            candidate_id = int(
                candidate_ids[
                    int(
                        indices[
                            qi,
                            ri
                        ]
                    )
                ]
            )


            if reverse:

                pair = (
                    candidate_id,
                    int(
                        query_id
                    ),
                )

            else:

                pair = (
                    int(
                        query_id
                    ),
                    candidate_id,
                )


            if pair not in result:

                result[
                    pair
                ] = (
                    int(
                        ri
                        +
                        1
                    ),

                    float(
                        distances[
                            qi,
                            ri
                        ]
                    ),
                )


    return result


def asym3x8_pass(
    source_rank,
    target_rank,
):

    return bool(
        (
            source_rank <= STRONG_RANK
            and
            target_rank <= WEAK_RANK
        )
        or
        (
            target_rank <= STRONG_RANK
            and
            source_rank <= WEAK_RANK
        )
    )


# ============================================================
# 7. GENERATE ONE DATASET'S AUTHORITATIVE RETAINED PROPOSALS
#
# Returns only a compact per-dataset DataFrame.
#
# No GT.
# ============================================================

def generate_dataset_proposals(
    dataset,
):

    graph, scale = current_best_builder(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    )


    # --------------------------------------------------------
    # Nodes
    # --------------------------------------------------------

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = nodes[
        "node_id"
    ].astype(int)


    nodes[
        "t"
    ] = nodes[
        "t"
    ].astype(int)


    nodes = (
        nodes
        .sort_values(
            [
                "t",
                "node_id",
            ],
            kind="stable",
        )
        .set_index(
            "node_id",
            drop=False,
        )
    )


    # --------------------------------------------------------
    # Edges
    # --------------------------------------------------------

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = edges[
        "source_id"
    ].astype(int)


    edges[
        "target_id"
    ] = edges[
        "target_id"
    ].astype(int)


    edge_set = set(
        zip(
            edges[
                "source_id"
            ],
            edges[
                "target_id"
            ],
        )
    )


    incoming = (
        edges.groupby(
            "target_id"
        )[
            "source_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    outgoing = (
        edges.groupby(
            "source_id"
        )[
            "target_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = {
        int(k):
            int(v)

        for k, v
        in indeg.items()
    }


    outdeg = {
        int(k):
            int(v)

        for k, v
        in outdeg.items()
    }


    # --------------------------------------------------------
    # Physical coordinates
    # --------------------------------------------------------

    phys = {}

    time_map = {}


    for nid, row in nodes.iterrows():

        nid = int(
            nid
        )


        phys[
            nid
        ] = (
            np.array(
                [
                    float(
                        row[
                            "z"
                        ]
                    ),

                    float(
                        row[
                            "y"
                        ]
                    ),

                    float(
                        row[
                            "x"
                        ]
                    ),
                ],
                dtype=float,
            )
            *
            SCALE
        )


        time_map[
            nid
        ] = int(
            row[
                "t"
            ]
        )


    node_ids = sorted(
        phys
    )


    # --------------------------------------------------------
    # Local velocities
    # --------------------------------------------------------

    past_velocity = {}

    future_velocity = {}


    for nid in node_ids:

        inc = incoming.get(
            nid,
            [],
        )


        if len(
            inc
        ) == 1:

            prev_id = int(
                inc[
                    0
                ]
            )


            if prev_id in phys:

                dt = (
                    time_map[
                        nid
                    ]
                    -
                    time_map[
                        prev_id
                    ]
                )


                if dt > 0:

                    past_velocity[
                        nid
                    ] = (
                        phys[
                            nid
                        ]
                        -
                        phys[
                            prev_id
                        ]
                    ) / float(
                        dt
                    )


        out = outgoing.get(
            nid,
            [],
        )


        if len(
            out
        ) == 1:

            next_id = int(
                out[
                    0
                ]
            )


            if next_id in phys:

                dt = (
                    time_map[
                        next_id
                    ]
                    -
                    time_map[
                        nid
                    ]
                )


                if dt > 0:

                    future_velocity[
                        nid
                    ] = (
                        phys[
                            next_id
                        ]
                        -
                        phys[
                            nid
                        ]
                    ) / float(
                        dt
                    )


    # --------------------------------------------------------
    # Eligibility
    # --------------------------------------------------------

    source_eligible = [
        nid

        for nid in node_ids

        if (
            outdeg.get(
                nid,
                0,
            )
            <= 1
            and
            nid in past_velocity
        )
    ]


    target_eligible = [
        nid

        for nid in node_ids

        if (
            indeg.get(
                nid,
                0,
            )
            <= 1
            and
            nid in future_velocity
        )
    ]


    source_by_t = {}

    target_by_t = {}


    for nid in source_eligible:

        source_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for nid in target_eligible:

        target_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for t in source_by_t:

        source_by_t[
            t
        ] = sorted(
            source_by_t[
                t
            ]
        )


    for t in target_by_t:

        target_by_t[
            t
        ] = sorted(
            target_by_t[
                t
            ]
        )


    # --------------------------------------------------------
    # Incumbent normalized CV
    # --------------------------------------------------------

    source_incumbent = {}

    target_incumbent = {}


    for src in source_eligible:

        if outdeg.get(
            src,
            0,
        ) != 1:

            continue


        old_targets = outgoing.get(
            src,
            [],
        )


        if len(
            old_targets
        ) != 1:

            continue


        old_tgt = int(
            old_targets[
                0
            ]
        )


        if old_tgt not in phys:

            continue


        dt_old = (
            time_map[
                old_tgt
            ]
            -
            time_map[
                src
            ]
        )


        if dt_old <= 0:

            continue


        predicted = (
            phys[
                src
            ]
            +
            float(
                dt_old
            )
            *
            past_velocity[
                src
            ]
        )


        source_incumbent[
            src
        ] = (
            vec_norm(
                phys[
                    old_tgt
                ]
                -
                predicted
            )
            /
            float(
                dt_old
            )
        )


    for tgt in target_eligible:

        if indeg.get(
            tgt,
            0,
        ) != 1:

            continue


        old_sources = incoming.get(
            tgt,
            [],
        )


        if len(
            old_sources
        ) != 1:

            continue


        old_src = int(
            old_sources[
                0
            ]
        )


        if old_src not in phys:

            continue


        dt_old = (
            time_map[
                tgt
            ]
            -
            time_map[
                old_src
            ]
        )


        if dt_old <= 0:

            continue


        predicted = (
            phys[
                tgt
            ]
            -
            float(
                dt_old
            )
            *
            future_velocity[
                tgt
            ]
        )


        target_incumbent[
            tgt
        ] = (
            vec_norm(
                phys[
                    old_src
                ]
                -
                predicted
            )
            /
            float(
                dt_old
            )
        )


    # --------------------------------------------------------
    # Proposal rows
    # --------------------------------------------------------

    proposal_frames = []

    raw_count = 0

    retained_count = 0


    for gap_size in range(
        MIN_GAP_SIZE,
        MAX_GAP_SIZE + 1,
    ):

        delta_t = (
            gap_size
            +
            1
        )


        for source_t in sorted(
            source_by_t
        ):

            target_t = (
                source_t
                +
                delta_t
            )


            if target_t not in target_by_t:

                continue


            source_ids = source_by_t[
                source_t
            ]


            target_ids = target_by_t[
                target_t
            ]


            if not source_ids or not target_ids:

                continue


            source_pos = np.vstack(
                [
                    phys[
                        nid
                    ]

                    for nid
                    in source_ids
                ]
            )


            target_pos = np.vstack(
                [
                    phys[
                        nid
                    ]

                    for nid
                    in target_ids
                ]
            )


            source_vel = np.vstack(
                [
                    past_velocity[
                        nid
                    ]

                    for nid
                    in source_ids
                ]
            )


            target_vel = np.vstack(
                [
                    future_velocity[
                        nid
                    ]

                    for nid
                    in target_ids
                ]
            )


            source_pred = (
                source_pos
                +
                float(
                    delta_t
                )
                *
                source_vel
            )


            target_pred = (
                target_pos
                -
                float(
                    delta_t
                )
                *
                target_vel
            )


            target_tree = cKDTree(
                target_pos
            )


            source_tree = cKDTree(
                source_pos
            )


            (
                source_dist,
                source_idx,
            ) = query_knn(
                target_tree,
                source_pred,
                len(
                    target_ids
                ),
                QUERY_K,
            )


            (
                target_dist,
                target_idx,
            ) = query_knn(
                source_tree,
                target_pred,
                len(
                    source_ids
                ),
                QUERY_K,
            )


            source_map = build_rank_distance_map(
                source_ids,
                target_ids,
                source_dist,
                source_idx,
                reverse=False,
            )


            target_map = build_rank_distance_map(
                target_ids,
                source_ids,
                target_dist,
                target_idx,
                reverse=True,
            )


            source_best = {
                int(
                    source_ids[
                        i
                    ]
                ):
                    float(
                        source_dist[
                            i,
                            0
                        ]
                    )

                for i
                in range(
                    len(
                        source_ids
                    )
                )
            }


            target_best = {
                int(
                    target_ids[
                        i
                    ]
                ):
                    float(
                        target_dist[
                            i,
                            0
                        ]
                    )

                for i
                in range(
                    len(
                        target_ids
                    )
                )
            }


            mutual_pairs = (
                set(
                    source_map
                )
                &
                set(
                    target_map
                )
            )


            frame_rows = []


            for (
                src,
                tgt,
            ) in mutual_pairs:

                if (
                    src,
                    tgt,
                ) in edge_set:

                    continue


                (
                    source_rank,
                    source_candidate_dist,
                ) = source_map[
                    (
                        src,
                        tgt,
                    )
                ]


                (
                    target_rank,
                    target_candidate_dist,
                ) = target_map[
                    (
                        src,
                        tgt,
                    )
                ]


                if not asym3x8_pass(
                    source_rank,
                    target_rank,
                ):

                    continue


                raw_count += 1


                source_degree = int(
                    outdeg.get(
                        src,
                        0,
                    )
                )


                target_degree = int(
                    indeg.get(
                        tgt,
                        0,
                    )
                )


                cuts = (
                    int(
                        source_degree == 1
                    )
                    +
                    int(
                        target_degree == 1
                    )
                )


                source_cv_pf = (
                    float(
                        source_candidate_dist
                    )
                    /
                    float(
                        delta_t
                    )
                )


                target_cv_pf = (
                    float(
                        target_candidate_dist
                    )
                    /
                    float(
                        delta_t
                    )
                )


                # --------------------------------------------
                # Frozen dominance-any
                # --------------------------------------------

                if cuts == 0:

                    dominance_any = True


                else:

                    improvements = []


                    if source_degree == 1:

                        incumbent = float(
                            source_incumbent.get(
                                src,
                                np.nan,
                            )
                        )


                        improvements.append(
                            (
                                incumbent
                                -
                                source_cv_pf
                            )
                            if np.isfinite(
                                incumbent
                            )
                            else np.nan
                        )


                    if target_degree == 1:

                        incumbent = float(
                            target_incumbent.get(
                                tgt,
                                np.nan,
                            )
                        )


                        improvements.append(
                            (
                                incumbent
                                -
                                target_cv_pf
                            )
                            if np.isfinite(
                                incumbent
                            )
                            else np.nan
                        )


                    finite = [
                        x
                        for x in improvements
                        if np.isfinite(
                            x
                        )
                    ]


                    dominance_any = (
                        len(
                            finite
                        )
                        ==
                        len(
                            improvements
                        )
                        and
                        any(
                            x > 0
                            for x in finite
                        )
                    )


                if not dominance_any:

                    continue


                retained_count += 1


                source_ratio = safe_ratio(
                    source_candidate_dist,
                    source_best[
                        src
                    ],
                )


                target_ratio = safe_ratio(
                    target_candidate_dist,
                    target_best[
                        tgt
                    ],
                )


                sym_cv = (
                    0.5
                    *
                    (
                        source_cv_pf
                        +
                        target_cv_pf
                    )
                )


                max_cv = max(
                    source_cv_pf,
                    target_cv_pf,
                )


                if (
                    np.isfinite(
                        source_ratio
                    )
                    and
                    np.isfinite(
                        target_ratio
                    )
                ):

                    gmean_ratio = float(
                        np.sqrt(
                            source_ratio
                            *
                            target_ratio
                        )
                    )

                else:

                    gmean_ratio = np.inf


                frame_rows.append(
                    (
                        int(
                            src
                        ),

                        int(
                            tgt
                        ),

                        int(
                            gap_size
                        ),

                        int(
                            cuts
                        ),

                        float(
                            sym_cv
                        ),

                        float(
                            max_cv
                        ),

                        float(
                            gmean_ratio
                        ),
                    )
                )


            if frame_rows:

                proposal_frames.append(
                    pd.DataFrame(
                        frame_rows,
                        columns=[
                            "source_id",
                            "target_id",
                            "gap_size",
                            "cuts",
                            "SYM_CV",
                            "MAX_CV",
                            "GMEAN_RATIO",
                        ],
                    )
                )


            del source_pos
            del target_pos

            del source_vel
            del target_vel

            del source_pred
            del target_pred

            del target_tree
            del source_tree

            del source_dist
            del source_idx

            del target_dist
            del target_idx

            del source_map
            del target_map

            del source_best
            del target_best

            del mutual_pairs
            del frame_rows


    # --------------------------------------------------------
    # Compact dataset proposal DataFrame
    # --------------------------------------------------------

    if proposal_frames:

        proposals = pd.concat(
            proposal_frames,
            ignore_index=True,
        )

    else:

        proposals = pd.DataFrame(
            columns=[
                "source_id",
                "target_id",
                "gap_size",
                "cuts",
                "SYM_CV",
                "MAX_CV",
                "GMEAN_RATIO",
            ]
        )


    assert len(
        proposals
    ) == retained_count


    assert not proposals[
        [
            "source_id",
            "target_id",
        ]
    ].duplicated().any()


    # --------------------------------------------------------
    # Release graph structures
    # --------------------------------------------------------

    del graph
    del nodes
    del edges

    del edge_set
    del incoming
    del outgoing

    del indeg
    del outdeg

    del phys
    del time_map
    del node_ids

    del past_velocity
    del future_velocity

    del source_eligible
    del target_eligible

    del source_by_t
    del target_by_t

    del source_incumbent
    del target_incumbent

    del proposal_frames

    gc.collect()


    return (
        proposals,
        int(
            raw_count
        ),
        int(
            retained_count
        ),
    )


# ============================================================
# 8. ADD ENDPOINT COMPETITION RANKS
# ============================================================

def add_endpoint_ranks(
    proposals,
):

    proposals = proposals.copy()


    for score in SCORES:

        # ----------------------------------------------------
        # Source-wise deterministic rank
        #
        # tie break:
        #   score
        #   gap
        #   target
        #   source
        # ----------------------------------------------------

        source_order = (
            proposals[
                [
                    "source_id",
                    "target_id",
                    "gap_size",
                    score,
                ]
            ]
            .sort_values(
                [
                    "source_id",
                    score,
                    "gap_size",
                    "target_id",
                ],
                kind="stable",
            )
        )


        source_order[
            f"{score}_SOURCE_RANK"
        ] = (
            source_order.groupby(
                "source_id",
                sort=False,
            )
            .cumcount()
            +
            1
        )


        proposals[
            f"{score}_SOURCE_RANK"
        ] = (
            source_order[
                f"{score}_SOURCE_RANK"
            ]
            .reindex(
                proposals.index
            )
            .astype(
                np.int32
            )
        )


        # ----------------------------------------------------
        # Target-wise deterministic rank
        # ----------------------------------------------------

        target_order = (
            proposals[
                [
                    "source_id",
                    "target_id",
                    "gap_size",
                    score,
                ]
            ]
            .sort_values(
                [
                    "target_id",
                    score,
                    "gap_size",
                    "source_id",
                ],
                kind="stable",
            )
        )


        target_order[
            f"{score}_TARGET_RANK"
        ] = (
            target_order.groupby(
                "target_id",
                sort=False,
            )
            .cumcount()
            +
            1
        )


        proposals[
            f"{score}_TARGET_RANK"
        ] = (
            target_order[
                f"{score}_TARGET_RANK"
            ]
            .reindex(
                proposals.index
            )
            .astype(
                np.int32
            )
        )


        del source_order
        del target_order


    return proposals


# ============================================================
# 9. PART A — GT-BLIND CARDINALITY
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART A — GT-BLIND ENDPOINT COMPETITION CARDINALITY"
)

print(
    "============================================================"
)


stats = {
    name: {
        "rows":
            0,

        "cuts_0":
            0,

        "cuts_1":
            0,

        "cuts_2":
            0,

        "folds": {
            "1":
                0,

            "2":
                0,

            "3":
                0,

            "4":
                0,
        },
    }

    for name
    in SHORTLISTS
}


dataset_results = []


raw_total = 0
retained_total = 0


for ds_i, dataset in enumerate(
    development_names,
    start=1,
):

    fold = int(
        fold_map_local[
            dataset
        ]
    )


    (
        proposals,
        raw_count,
        retained_count,
    ) = generate_dataset_proposals(
        dataset
    )


    proposals = add_endpoint_ranks(
        proposals
    )


    raw_total += raw_count
    retained_total += retained_count


    local_results = {}


    for score in SCORES:

        source_rank_col = (
            f"{score}_SOURCE_RANK"
        )


        target_rank_col = (
            f"{score}_TARGET_RANK"
        )


        for n in N_VALUES:

            name = (
                f"{score}_TOP{n}"
            )


            mask = (
                proposals[
                    source_rank_col
                ].le(
                    n
                )
                |
                proposals[
                    target_rank_col
                ].le(
                    n
                )
            )


            count = int(
                mask.sum()
            )


            local_results[
                name
            ] = count


            stats[
                name
            ][
                "rows"
            ] += count


            stats[
                name
            ][
                "folds"
            ][
                str(
                    fold
                )
            ] += count


            for cuts in [
                0,
                1,
                2,
            ]:

                cuts_count = int(
                    (
                        mask
                        &
                        proposals[
                            "cuts"
                        ].eq(
                            cuts
                        )
                    ).sum()
                )


                stats[
                    name
                ][
                    f"cuts_{cuts}"
                ] += cuts_count


    dataset_results.append(
        {
            "dataset":
                dataset,

            "fold":
                fold,

            "raw_generator":
                raw_count,

            "retained_generator":
                retained_count,

            "shortlists":
                local_results,
        }
    )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            development_names
        )
    ):

        print(
            f"{ds_i:3d}/{len(development_names)}"
            f" | fold={fold}"
            f" | {dataset}"
            f" | raw={raw_count}"
            f" | frozen={retained_count}"
            f" | SYM1={local_results['SYM_CV_TOP1']}"
            f" | SYM3={local_results['SYM_CV_TOP3']}"
            f" | GM3={local_results['GMEAN_RATIO_TOP3']}"
        )


    del proposals

    gc.collect()


# ============================================================
# 10. GENERATOR FIDELITY
# ============================================================

print(
    "\n========== FROZEN GENERATOR REPRODUCTION =========="
)

print(
    "raw:",
    raw_total,
    "/",
    EXPECTED_RAW
)

print(
    "retained:",
    retained_total,
    "/",
    EXPECTED_RETAINED
)


assert raw_total == EXPECTED_RAW
assert retained_total == EXPECTED_RETAINED


print(
    "AUTHORITATIVE_GENERATOR_REPRODUCTION: PASS"
)


# ============================================================
# 11. CARDINALITY TABLE
# ============================================================

cardinality_rows = []


for name in SHORTLISTS:

    entry = stats[
        name
    ]


    rows = int(
        entry[
            "rows"
        ]
    )


    assert (
        int(
            entry[
                "cuts_0"
            ]
        )
        +
        int(
            entry[
                "cuts_1"
            ]
        )
        +
        int(
            entry[
                "cuts_2"
            ]
        )
    ) == rows


    cardinality_rows.append(
        {
            "shortlist":
                name,

            "rows":
                rows,

            "fraction_of_frozen":
                float(
                    rows
                    /
                    EXPECTED_RETAINED
                ),

            "compression_factor":
                float(
                    EXPECTED_RETAINED
                    /
                    rows
                )
                if rows
                else np.inf,

            "cuts_0":
                int(
                    entry[
                        "cuts_0"
                    ]
                ),

            "cuts_1":
                int(
                    entry[
                        "cuts_1"
                    ]
                ),

            "cuts_2":
                int(
                    entry[
                        "cuts_2"
                    ]
                ),
        }
    )


cardinality_table = (
    pd.DataFrame(
        cardinality_rows
    )
    .sort_values(
        [
            "rows",
            "shortlist",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n========== GT-BLIND ENDPOINT SHORTLIST CARDINALITY =========="
)


print(
    cardinality_table.to_string(
        index=False
    )
)


# ============================================================
# 12. FOLD CARDINALITY
# ============================================================

fold_rows = []


for name in SHORTLISTS:

    for fold in [
        1,
        2,
        3,
        4,
    ]:

        fold_rows.append(
            {
                "shortlist":
                    name,

                "fold":
                    fold,

                "rows":
                    int(
                        stats[
                            name
                        ][
                            "folds"
                        ][
                            str(
                                fold
                            )
                        ]
                    ),
            }
        )


fold_table = pd.DataFrame(
    fold_rows
)


# ============================================================
# 13. FREEZE PART A BEFORE ORACLE READ
# ============================================================

part_a = {
    "stage":
        "12.11G3G1",

    "part":
        "GT_BLIND_ENDPOINT_COMPETITION_CARDINALITY",

    "authoritative_generator_id":
        frozen[
            "generator_id"
        ],

    "authoritative_generator_sha256":
        AUTHORITATIVE_SHA,

    "generator_raw_rows":
        int(
            raw_total
        ),

    "generator_retained_rows":
        int(
            retained_total
        ),

    "score_family":
        SCORES,

    "n_values":
        N_VALUES,

    "shortlist_rule":
        "SOURCE_SCORE_RANK_LE_N_OR_TARGET_SCORE_RANK_LE_N",

    "tie_breaking": {
        "source":
            "score,gap_size,target_id",

        "target":
            "score,gap_size,source_id",
    },

    "cardinality":
        cardinality_table.to_dict(
            "records"
        ),

    "fold_cardinality":
        fold_table.to_dict(
            "records"
        ),

    "dataset_results":
        dataset_results,

    "proposal_rows_persisted":
        False,

    "gt_used_during_part_a":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "shortlist_selected":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


canonical = json.dumps(
    part_a,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
).encode(
    "utf-8"
)


part_a_sha = hashlib.sha256(
    canonical
).hexdigest()


part_a[
    "sha256"
] = part_a_sha


CARDINALITY_OUT.write_text(
    json.dumps(
        part_a,
        indent=2,
    ),
    encoding="utf-8",
)


assert CARDINALITY_OUT.exists()


print(
    "\nENDPOINT SHORTLIST CARDINALITY FROZEN BEFORE GT READ: YES"
)

print(
    "Part-A SHA256:",
    part_a_sha
)


# ============================================================
# PART B — DEVELOPMENT ORACLE
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — DEVELOPMENT ORACLE ENDPOINT COMPETITION"
)

print(
    "============================================================"
)


oracle = pd.read_pickle(
    G3F1_ORACLE_PATH
)


assert len(
    oracle
) == 369


assert int(
    oracle[
        "combined_ASYM_3x8"
    ].sum()
) == 212


for name in SHORTLISTS:

    oracle[
        "shortlist_"
        +
        name
    ] = False


# Additional diagnostic true-pair competition rank fields.
for score in SCORES:

    oracle[
        f"{score}_TRUE_SOURCE_RANK"
    ] = np.nan


    oracle[
        f"{score}_TRUE_TARGET_RANK"
    ] = np.nan


    oracle[
        f"{score}_TRUE_BEST_ENDPOINT_RANK"
    ] = np.nan


oracle_datasets = (
    oracle[
        "dataset"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)


assert len(
    oracle_datasets
) == 75


# ============================================================
# 14. REGENERATE ORACLE DATASETS
# ============================================================

for ds_i, dataset in enumerate(
    oracle_datasets,
    start=1,
):

    (
        proposals,
        _,
        retained_count,
    ) = generate_dataset_proposals(
        dataset
    )


    proposals = add_endpoint_ranks(
        proposals
    )


    ds_mask = oracle[
        "dataset"
    ].eq(
        dataset
    )


    true_mask = (
        ds_mask
        &
        oracle[
            "combined_ASYM_3x8"
        ].astype(bool)
    )


    true_rows = oracle.loc[
        true_mask,
        [
            "component_id",
            "source_id",
            "target_id",
        ],
    ].copy()


    if len(
        true_rows
    ):

        # ----------------------------------------------------
        # Only compact columns needed for oracle matching.
        # ----------------------------------------------------

        rank_cols = [
            "source_id",
            "target_id",
        ]


        for score in SCORES:

            rank_cols.extend(
                [
                    f"{score}_SOURCE_RANK",
                    f"{score}_TARGET_RANK",
                ]
            )


        compact = proposals[
            rank_cols
        ]


        matched = true_rows.merge(
            compact,
            on=[
                "source_id",
                "target_id",
            ],
            how="left",
            validate="one_to_one",
        )


        assert len(
            matched
        ) == len(
            true_rows
        )


        for score in SCORES:

            src_col = (
                f"{score}_SOURCE_RANK"
            )


            tgt_col = (
                f"{score}_TARGET_RANK"
            )


            assert matched[
                src_col
            ].notna().all()


            assert matched[
                tgt_col
            ].notna().all()


            source_rank_map = dict(
                zip(
                    matched[
                        "component_id"
                    ],
                    matched[
                        src_col
                    ].astype(int),
                )
            )


            target_rank_map = dict(
                zip(
                    matched[
                        "component_id"
                    ],
                    matched[
                        tgt_col
                    ].astype(int),
                )
            )


            component_mask = (
                ds_mask
                &
                oracle[
                    "component_id"
                ].isin(
                    matched[
                        "component_id"
                    ]
                )
            )


            oracle.loc[
                component_mask,
                f"{score}_TRUE_SOURCE_RANK",
            ] = (
                oracle.loc[
                    component_mask,
                    "component_id"
                ]
                .map(
                    source_rank_map
                )
                .astype(float)
            )


            oracle.loc[
                component_mask,
                f"{score}_TRUE_TARGET_RANK",
            ] = (
                oracle.loc[
                    component_mask,
                    "component_id"
                ]
                .map(
                    target_rank_map
                )
                .astype(float)
            )


            oracle.loc[
                component_mask,
                f"{score}_TRUE_BEST_ENDPOINT_RANK",
            ] = np.minimum(
                oracle.loc[
                    component_mask,
                    f"{score}_TRUE_SOURCE_RANK",
                ].to_numpy(),

                oracle.loc[
                    component_mask,
                    f"{score}_TRUE_TARGET_RANK",
                ].to_numpy(),
            )


            for n in N_VALUES:

                shortlist_name = (
                    f"{score}_TOP{n}"
                )


                source_pass = (
                    matched[
                        src_col
                    ]
                    .le(
                        n
                    )
                )


                target_pass = (
                    matched[
                        tgt_col
                    ]
                    .le(
                        n
                    )
                )


                surviving_components = set(
                    matched.loc[
                        (
                            source_pass
                            |
                            target_pass
                        ),
                        "component_id",
                    ]
                )


                oracle.loc[
                    (
                        ds_mask
                        &
                        oracle[
                            "component_id"
                        ].isin(
                            surviving_components
                        )
                    ),
                    "shortlist_"
                    +
                    shortlist_name,
                ] = True


        del compact
        del matched


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            oracle_datasets
        )
    ):

        print(
            f"{ds_i:2d}/{len(oracle_datasets)}"
            f" | {dataset}"
            f" | proposals={retained_count}"
            f" | true={int(true_mask.sum())}"
        )


    del proposals
    del true_rows

    gc.collect()


# ============================================================
# 15. TRUE-PAIR COMPETITION RANK DISTRIBUTIONS
# ============================================================

captured_mask = oracle[
    "combined_ASYM_3x8"
].astype(bool)


print(
    "\n========== TRUE-PAIR ENDPOINT COMPETITION RANKS =========="
)


for score in SCORES:

    print(
        "\n" + score
    )


    for col in [
        f"{score}_TRUE_SOURCE_RANK",
        f"{score}_TRUE_TARGET_RANK",
        f"{score}_TRUE_BEST_ENDPOINT_RANK",
    ]:

        assert oracle.loc[
            captured_mask,
            col,
        ].notna().all()


        print(
            "\n" + col
        )


        print(
            oracle.loc[
                captured_mask,
                col,
            ]
            .describe(
                percentiles=[
                    0.10,
                    0.25,
                    0.50,
                    0.75,
                    0.90,
                    0.95,
                    0.99,
                ]
            )
            .to_string()
        )


# ============================================================
# 16. ORACLE COVERAGE TABLE
# ============================================================

oracle_rows = []


full7_mask = oracle[
    "pred_full_7um"
].astype(bool)


for name in SHORTLISTS:

    col = (
        "shortlist_"
        +
        name
    )


    candidate_row = cardinality_table[
        cardinality_table[
            "shortlist"
        ].eq(
            name
        )
    ]


    assert len(
        candidate_row
    ) == 1


    candidate_count = int(
        candidate_row[
            "rows"
        ].iloc[
            0
        ]
    )


    true_count = int(
        oracle[
            col
        ].sum()
    )


    true_full7 = int(
        (
            oracle[
                col
            ]
            &
            full7_mask
        ).sum()
    )


    oracle_rows.append(
        {
            "shortlist":
                name,

            "candidate_rows":
                candidate_count,

            "candidate_fraction_of_frozen":
                float(
                    candidate_count
                    /
                    EXPECTED_RETAINED
                ),

            "compression_factor":
                float(
                    EXPECTED_RETAINED
                    /
                    candidate_count
                ),

            "oracle_captured":
                true_count,

            "oracle_fraction_of_frozen_true":
                float(
                    true_count
                    /
                    212
                ),

            "oracle_overall_coverage":
                float(
                    true_count
                    /
                    369
                ),

            "full7_captured":
                true_full7,

            "full7_fraction_of_frozen_true":
                float(
                    true_full7
                    /
                    178
                ),

            "full7_overall_coverage":
                float(
                    true_full7
                    /
                    285
                ),
        }
    )


oracle_table = (
    pd.DataFrame(
        oracle_rows
    )
    .sort_values(
        [
            "candidate_rows",
            "shortlist",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


oracle_table[
    "oracle_retention_minus_candidate_retention"
] = (
    oracle_table[
        "oracle_fraction_of_frozen_true"
    ]
    -
    oracle_table[
        "candidate_fraction_of_frozen"
    ]
)


oracle_table[
    "oracle_components_per_million_rows"
] = (
    oracle_table[
        "oracle_captured"
    ]
    /
    (
        oracle_table[
            "candidate_rows"
        ]
        /
        1_000_000.0
    )
)


print(
    "\n========== ENDPOINT SHORTLIST ORACLE COVERAGE =========="
)


print(
    oracle_table.to_string(
        index=False
    )
)


# ============================================================
# 17. PARETO VIEW
# ============================================================

print(
    "\n========== ENDPOINT SHORTLIST CARDINALITY / RECALL TRADEOFF =========="
)


print(
    oracle_table[
        [
            "shortlist",

            "candidate_rows",
            "candidate_fraction_of_frozen",
            "compression_factor",

            "oracle_captured",
            "oracle_fraction_of_frozen_true",

            "full7_captured",
            "full7_fraction_of_frozen_true",

            "oracle_retention_minus_candidate_retention",
            "oracle_components_per_million_rows",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 18. ORACLE FOLD STABILITY
# ============================================================

fold_oracle_rows = []


for name in SHORTLISTS:

    col = (
        "shortlist_"
        +
        name
    )


    for fold in [
        1,
        2,
        3,
        4,
    ]:

        sub = oracle[
            oracle[
                "fold"
            ].eq(
                fold
            )
        ]


        frozen_true = int(
            sub[
                "combined_ASYM_3x8"
            ].sum()
        )


        retained_true = int(
            sub[
                col
            ].sum()
        )


        fold_oracle_rows.append(
            {
                "shortlist":
                    name,

                "fold":
                    fold,

                "frozen_true":
                    frozen_true,

                "retained_true":
                    retained_true,

                "retention_fraction":
                    (
                        float(
                            retained_true
                            /
                            frozen_true
                        )
                        if frozen_true
                        else np.nan
                    ),
            }
        )


oracle_fold_table = pd.DataFrame(
    fold_oracle_rows
)


print(
    "\n========== ENDPOINT SHORTLIST ORACLE RETENTION BY FOLD =========="
)


for name in SHORTLISTS:

    print(
        "\n" + name
    )


    print(
        oracle_fold_table[
            oracle_fold_table[
                "shortlist"
            ].eq(
                name
            )
        ].to_string(
            index=False
        )
    )


# ============================================================
# 19. PERSIST ORACLE AUDIT
# ============================================================

oracle = (
    oracle
    .sort_values(
        [
            "fold",
            "dataset",
            "component_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


oracle.to_pickle(
    ORACLE_OUT
)


summary = {
    "stage":
        "12.11G3G1",

    "status":
        "ENDPOINT_COMPETITION_SHORTLIST_FAMILY_AUDITED",

    "authoritative_generator_id":
        frozen[
            "generator_id"
        ],

    "authoritative_generator_sha256":
        AUTHORITATIVE_SHA,

    "part_a_sha256":
        part_a_sha,

    "score_family":
        SCORES,

    "n_values":
        N_VALUES,

    "tradeoff":
        oracle_table.to_dict(
            "records"
        ),

    "fold_oracle_retention":
        oracle_fold_table.to_dict(
            "records"
        ),

    "generator_exactly_reproduced":
        True,

    "part_a_gt_blind":
        True,

    "shortlist_selected":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "rewire_policy_selected":
        False,

    "graph_modified":
        False,
}


SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert ORACLE_OUT.exists()
assert SUMMARY_OUT.exists()


# ============================================================
# 20. RELOAD INTEGRITY
# ============================================================

check = pd.read_pickle(
    ORACLE_OUT
)


assert len(
    check
) == 369


assert int(
    check[
        "combined_ASYM_3x8"
    ].sum()
) == 212


for name in SHORTLISTS:

    assert (
        "shortlist_"
        +
        name
    ) in check.columns


# ============================================================
# 21. DECISION
# ============================================================

print(
    "\n========== 12.11G3G1 DECISION =========="
)

print(
    "GT_BLIND_ENDPOINT_COMPETITION_SHORTLIST_AUDIT: PASS"
)

print(
    "authoritative generator SHA:",
    AUTHORITATIVE_SHA
)

print(
    "raw generator reproduced:",
    raw_total,
    "/",
    EXPECTED_RAW
)

print(
    "retained generator reproduced:",
    retained_total,
    "/",
    EXPECTED_RETAINED
)

print(
    "Part-A cardinality frozen before GT:",
    "YES"
)

print(
    "proposal universe persisted:",
    "NO"
)

print(
    "endpoint shortlist selected:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "classifier trained:",
    "NO"
)

print(
    "rewire policy selected:",
    "NO"
)

print(
    "graph modified:",
    "NO"
)

print(
    "next:"
)

print(
    "SELECT_ENDPOINT_COMPETITION_PARETO_FRONTIER_FOR_CROSS_FOLD_SCORING"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3G1A
#
# Development Oracle Diagnostic:
#
#   GMEAN_RATIO_TOP8 tail analysis
#
# Authoritative frozen generator:
#
#   MULTIFRAME_REWIRE_ASYM3X8_DOMINANCE_ANY_V1_FIDELITYPATCH1
#
# Known G3G1:
#
#   frozen true pairs           = 212
#   GMEAN_RATIO_TOP8 captured   = 183
#   tail misses                 = 29
#
# Purpose:
#
#   Determine WHY those 29 true pairs fall outside endpoint
#   Top8 and whether they can be rescued cheaply by:
#
#       1. modest GMEAN rank expansion
#       2. alternative SYM/MAX score union
#       3. topology-specific rescue
#       4. incumbent-relative improvement
#       5. local ratio / excess evidence
#
# This cell is ORACLE DIAGNOSTIC ONLY.
#
# No new candidate universe.
# No shortlist selected.
# No threshold selected.
# No Fold0.
# No classifier.
# No graph modification.
#
# Keep after running: YES
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


G3G1_ORACLE_PATH = (
    S12
    / "stage12_folds14_endpoint_competition_shortlist_oracle_v1.pkl"
)


G3G0_ORACLE_PATH = (
    S12
    / "stage12_folds14_multiframe_local_competition_shortlist_oracle_v1.pkl"
)


OUT = (
    S12
    / "stage12_folds14_endpoint_competition_gmean_top8_tail_v1.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_folds14_endpoint_competition_gmean_top8_tail_v1_summary.json"
)


assert G3G1_ORACLE_PATH.exists(), G3G1_ORACLE_PATH


for path in [
    OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3G1A output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite an existing diagnostic."
        )


# ============================================================
# 1. Load G3G1 oracle
# ============================================================

oracle = pd.read_pickle(
    G3G1_ORACLE_PATH
)


assert len(
    oracle
) == 369


assert int(
    oracle[
        "combined_ASYM_3x8"
    ].sum()
) == 212


TOP8_COL = (
    "shortlist_GMEAN_RATIO_TOP8"
)


assert TOP8_COL in oracle.columns


assert int(
    oracle[
        TOP8_COL
    ].sum()
) == 183


# ============================================================
# 2. Define frozen true population and Top8 tail
# ============================================================

frozen_true = (
    oracle[
        "combined_ASYM_3x8"
    ].astype(bool)
)


top8_true = (
    frozen_true
    &
    oracle[
        TOP8_COL
    ].astype(bool)
)


tail_mask = (
    frozen_true
    &
    ~oracle[
        TOP8_COL
    ].astype(bool)
)


assert int(
    frozen_true.sum()
) == 212


assert int(
    top8_true.sum()
) == 183


assert int(
    tail_mask.sum()
) == 29


tail = (
    oracle[
        tail_mask
    ]
    .copy()
)


print(
    "========== 12.11G3G1A INPUT =========="
)

print(
    "frozen true:",
    int(
        frozen_true.sum()
    )
)

print(
    "GMEAN Top8 captured:",
    int(
        top8_true.sum()
    )
)

print(
    "GMEAN Top8 tail:",
    len(
        tail
    )
)


# ============================================================
# 3. Full <=7 geometry tail
# ============================================================

full7 = oracle[
    "pred_full_7um"
].astype(bool)


full7_frozen = int(
    (
        frozen_true
        &
        full7
    ).sum()
)


full7_top8 = int(
    (
        top8_true
        &
        full7
    ).sum()
)


full7_tail = int(
    (
        tail_mask
        &
        full7
    ).sum()
)


assert full7_frozen == 178
assert full7_top8 == 155
assert full7_tail == 23


print(
    "\n========== FULL<=7 TAIL =========="
)

print(
    "frozen full<=7:",
    full7_frozen
)

print(
    "Top8 full<=7:",
    full7_top8
)

print(
    "tail full<=7:",
    full7_tail
)


# ============================================================
# 4. Exact GMEAN endpoint-rank ceiling
#
# No new generator is created here.
# Pure oracle diagnostic.
# ============================================================

GMEAN_BEST = (
    "GMEAN_RATIO_TRUE_BEST_ENDPOINT_RANK"
)


GMEAN_SRC = (
    "GMEAN_RATIO_TRUE_SOURCE_RANK"
)


GMEAN_TGT = (
    "GMEAN_RATIO_TRUE_TARGET_RANK"
)


for col in [
    GMEAN_BEST,
    GMEAN_SRC,
    GMEAN_TGT,
]:

    assert col in oracle.columns


    assert oracle.loc[
        frozen_true,
        col,
    ].notna().all()


N_DIAG = [
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    16,
    18,
    20,
    21,
    24,
]


rank_ceiling_rows = []


print(
    "\n========== GMEAN ENDPOINT-RANK ORACLE CEILING =========="
)


for n in N_DIAG:

    pass_n = (
        frozen_true
        &
        oracle[
            GMEAN_BEST
        ].le(
            n
        )
    )


    captured = int(
        pass_n.sum()
    )


    captured_full7 = int(
        (
            pass_n
            &
            full7
        ).sum()
    )


    rank_ceiling_rows.append(
        {
            "N":
                int(
                    n
                ),

            "captured":
                captured,

            "frozen_true_retention":
                float(
                    captured
                    /
                    212
                ),

            "full7_captured":
                captured_full7,

            "full7_retention":
                float(
                    captured_full7
                    /
                    178
                ),
        }
    )


    print(
        f"N={n:2d}"
        f" | true={captured}/212"
        f" ({captured/212:.3%})"
        f" | full<=7={captured_full7}/178"
        f" ({captured_full7/178:.3%})"
    )


rank_ceiling = pd.DataFrame(
    rank_ceiling_rows
)


# ============================================================
# 5. Tail GMEAN rank distribution
# ============================================================

print(
    "\n========== TOP8 TAIL GMEAN RANKS =========="
)


for col in [
    GMEAN_SRC,
    GMEAN_TGT,
    GMEAN_BEST,
]:

    print(
        "\n" + col
    )


    print(
        tail[
            col
        ]
        .describe(
            percentiles=[
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99,
            ]
        )
        .to_string()
    )


print(
    "\nExact best-endpoint-rank counts:"
)


print(
    tail[
        GMEAN_BEST
    ]
    .astype(int)
    .value_counts()
    .sort_index()
    .to_string()
)


# ============================================================
# 6. Alternative-score Top8 rescue
#
# Does a true pair missed by GMEAN Top8 become Top8 under
# SYM_CV or MAX_CV?
# ============================================================

SYM_BEST = (
    "SYM_CV_TRUE_BEST_ENDPOINT_RANK"
)


MAX_BEST = (
    "MAX_CV_TRUE_BEST_ENDPOINT_RANK"
)


assert SYM_BEST in oracle.columns
assert MAX_BEST in oracle.columns


tail[
    "rescue_sym_top8"
] = (
    tail[
        SYM_BEST
    ].le(
        8
    )
)


tail[
    "rescue_max_top8"
] = (
    tail[
        MAX_BEST
    ].le(
        8
    )
)


tail[
    "rescue_sym_or_max_top8"
] = (
    tail[
        "rescue_sym_top8"
    ]
    |
    tail[
        "rescue_max_top8"
    ]
)


print(
    "\n========== ALTERNATIVE SCORE TOP8 RESCUE =========="
)

print(
    "tail:",
    len(
        tail
    )
)

print(
    "SYM Top8 rescues:",
    int(
        tail[
            "rescue_sym_top8"
        ].sum()
    )
)

print(
    "MAX Top8 rescues:",
    int(
        tail[
            "rescue_max_top8"
        ].sum()
    )
)

print(
    "SYM OR MAX rescues:",
    int(
        tail[
            "rescue_sym_or_max_top8"
        ].sum()
    )
)


print(
    "\nFull<=7 rescues:"
)


tail_full7_mask = tail[
    "pred_full_7um"
].astype(bool)


print(
    "SYM:",
    int(
        (
            tail[
                "rescue_sym_top8"
            ]
            &
            tail_full7_mask
        ).sum()
    )
)

print(
    "MAX:",
    int(
        (
            tail[
                "rescue_max_top8"
            ]
            &
            tail_full7_mask
        ).sum()
    )
)

print(
    "SYM OR MAX:",
    int(
        (
            tail[
                "rescue_sym_or_max_top8"
            ]
            &
            tail_full7_mask
        ).sum()
    )
)


# ============================================================
# 7. Tail by fold
# ============================================================

print(
    "\n========== TOP8 TAIL BY FOLD =========="
)


tail_by_fold = (
    tail.groupby(
        "fold"
    )
    .agg(
        tail=(
            "component_id",
            "size",
        ),

        full7=(
            "pred_full_7um",
            "sum",
        ),

        median_best_rank=(
            GMEAN_BEST,
            "median",
        ),

        max_best_rank=(
            GMEAN_BEST,
            "max",
        ),
    )
)


print(
    tail_by_fold.to_string()
)


# ============================================================
# 8. Tail by topology
# ============================================================

assert (
    "topology_class"
    in
    tail.columns
)


print(
    "\n========== TOP8 TAIL BY TOPOLOGY =========="
)


tail_by_topology = (
    tail.groupby(
        "topology_class"
    )
    .agg(
        tail=(
            "component_id",
            "size",
        ),

        full7=(
            "pred_full_7um",
            "sum",
        ),

        median_best_rank=(
            GMEAN_BEST,
            "median",
        ),

        median_gap=(
            "gap_size",
            "median",
        ),
    )
)


print(
    tail_by_topology.to_string()
)


# ============================================================
# 9. Tail by gap
# ============================================================

print(
    "\n========== TOP8 TAIL BY GAP SIZE =========="
)


tail_by_gap = (
    tail.groupby(
        "gap_size"
    )
    .agg(
        tail=(
            "component_id",
            "size",
        ),

        full7=(
            "pred_full_7um",
            "sum",
        ),

        median_best_rank=(
            GMEAN_BEST,
            "median",
        ),
    )
)


print(
    tail_by_gap.to_string()
)


# ============================================================
# 10. Required cuts / occupied status
# ============================================================

if (
    "source_occupied"
    in tail.columns
    and
    "target_occupied"
    in tail.columns
):

    tail[
        "n_required_cuts_diag"
    ] = (
        tail[
            "source_occupied"
        ].astype(int)
        +
        tail[
            "target_occupied"
        ].astype(int)
    )


else:

    topology_to_cuts = {
        "FREE_FREE":
            0,

        "SOURCE_OCCUPIED":
            1,

        "TARGET_OCCUPIED":
            1,

        "BOTH_OCCUPIED":
            2,
    }


    tail[
        "n_required_cuts_diag"
    ] = (
        tail[
            "topology_class"
        ]
        .map(
            topology_to_cuts
        )
    )


assert tail[
    "n_required_cuts_diag"
].notna().all()


print(
    "\n========== TOP8 TAIL BY REQUIRED CUTS =========="
)


tail_by_cuts = (
    tail.groupby(
        "n_required_cuts_diag"
    )
    .agg(
        tail=(
            "component_id",
            "size",
        ),

        full7=(
            "pred_full_7um",
            "sum",
        ),

        median_best_rank=(
            GMEAN_BEST,
            "median",
        ),
    )
)


print(
    tail_by_cuts.to_string()
)


# ============================================================
# 11. Incumbent-relative improvement diagnostic
# ============================================================

IMPROVEMENT_COLS = [
    col

    for col in [
        "source_relative_improvement",
        "target_relative_improvement",
        "joint_relative_improvement",
    ]

    if col in tail.columns
]


print(
    "\n========== TOP8 TAIL RELATIVE IMPROVEMENT =========="
)


if not IMPROVEMENT_COLS:

    print(
        "No retained relative-improvement columns found."
    )


else:

    for col in IMPROVEMENT_COLS:

        print(
            "\n" + col
        )


        values = (
            tail[
                col
            ]
            .replace(
                [
                    np.inf,
                    -np.inf,
                ],
                np.nan,
            )
            .dropna()
        )


        if len(
            values
        ):

            print(
                values.describe(
                    percentiles=[
                        0.10,
                        0.25,
                        0.50,
                        0.75,
                        0.90,
                        0.95,
                    ]
                )
                .to_string()
            )


        else:

            print(
                "no finite values"
            )


# ============================================================
# 12. Merge G3G0 local features if available
# ============================================================

LOCAL_FEATURES = [
    "source_ratio",
    "target_ratio",
    "max_ratio",
    "gmean_ratio",

    "source_excess_pf",
    "target_excess_pf",
    "max_excess_pf",

    "anchor_ratio",
    "peer_ratio",
]


if G3G0_ORACLE_PATH.exists():

    g3g0 = pd.read_pickle(
        G3G0_ORACLE_PATH
    )


    assert len(
        g3g0
    ) == 369


    feature_cols = [
        col

        for col in LOCAL_FEATURES

        if col in g3g0.columns
    ]


    if feature_cols:

        local_feature_table = (
            g3g0[
                [
                    "dataset",
                    "component_id",
                ]
                +
                feature_cols
            ]
            .copy()
        )


        tail = tail.merge(
            local_feature_table,
            on=[
                "dataset",
                "component_id",
            ],
            how="left",
            validate="one_to_one",
            suffixes=(
                "",
                "_g3g0",
            ),
        )


        print(
            "\n========== TOP8 TAIL LOCAL-COMPETITION FEATURES =========="
        )


        for col in feature_cols:

            actual_col = (
                col
                if col in tail.columns
                else col
                +
                "_g3g0"
            )


            values = (
                tail[
                    actual_col
                ]
                .replace(
                    [
                        np.inf,
                        -np.inf,
                    ],
                    np.nan,
                )
                .dropna()
            )


            if len(
                values
            ):

                print(
                    "\n" + actual_col
                )


                print(
                    values.describe(
                        percentiles=[
                            0.10,
                            0.25,
                            0.50,
                            0.75,
                            0.90,
                            0.95,
                        ]
                    )
                    .to_string()
                )


# ============================================================
# 13. Individual tail table
#
# Only 29 rows; useful for exact pattern inspection.
# ============================================================

DISPLAY_COLS = [
    col

    for col in [
        "fold",
        "dataset",
        "component_id",

        "topology_class",
        "gap_size",
        "n_required_cuts_diag",

        "pred_full_3um",
        "pred_full_5um",
        "pred_full_7um",

        GMEAN_SRC,
        GMEAN_TGT,
        GMEAN_BEST,

        SYM_BEST,
        MAX_BEST,

        "rescue_sym_top8",
        "rescue_max_top8",

        "source_relative_improvement",
        "target_relative_improvement",
        "joint_relative_improvement",

        "gmean_ratio",
        "anchor_ratio",
        "peer_ratio",
        "max_excess_pf",
    ]

    if col in tail.columns
]


tail = (
    tail
    .sort_values(
        [
            GMEAN_BEST,
            "fold",
            "dataset",
            "component_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n========== TOP8 TAIL — 29 TRUE PAIRS =========="
)


print(
    tail[
        DISPLAY_COLS
    ].to_string(
        index=False
    )
)


# ============================================================
# 14. Persist formal diagnostic
# ============================================================

tail.to_pickle(
    OUT
)


summary = {
    "stage":
        "12.11G3G1A",

    "status":
        "GMEAN_TOP8_TRUE_PAIR_TAIL_DIAGNOSED",

    "frozen_true":
        212,

    "gmean_top8_captured":
        183,

    "tail":
        29,

    "full7_frozen":
        178,

    "full7_top8":
        155,

    "full7_tail":
        23,

    "rank_ceiling":
        rank_ceiling.to_dict(
            "records"
        ),

    "alternative_top8_rescue": {
        "sym":
            int(
                tail[
                    "rescue_sym_top8"
                ].sum()
            ),

        "max":
            int(
                tail[
                    "rescue_max_top8"
                ].sum()
            ),

        "sym_or_max":
            int(
                tail[
                    "rescue_sym_or_max_top8"
                ].sum()
            ),
    },

    "tail_by_fold":
        (
            tail_by_fold
            .reset_index()
            .to_dict(
                "records"
            )
        ),

    "tail_by_topology":
        (
            tail_by_topology
            .reset_index()
            .to_dict(
                "records"
            )
        ),

    "tail_by_gap":
        (
            tail_by_gap
            .reset_index()
            .to_dict(
                "records"
            )
        ),

    "tail_by_cuts":
        (
            tail_by_cuts
            .reset_index()
            .to_dict(
                "records"
            )
        ),

    "diagnostic_only":
        True,

    "new_candidate_universe_generated":
        False,

    "shortlist_selected":
        False,

    "threshold_selected":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert OUT.exists()
assert SUMMARY_OUT.exists()


# ============================================================
# 15. Decision
# ============================================================

print(
    "\n========== 12.11G3G1A DECISION =========="
)

print(
    "GMEAN_TOP8_TRUE_PAIR_TAIL_DIAGNOSTIC: PASS"
)

print(
    "frozen true:",
    212
)

print(
    "Top8 captured:",
    183
)

print(
    "tail:",
    29
)

print(
    "full<=7 tail:",
    23
)

print(
    "new proposal universe generated:",
    "NO"
)

print(
    "shortlist selected:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "classifier trained:",
    "NO"
)

print(
    "graph modified:",
    "NO"
)

print(
    "next:"
)

print(
    "DESIGN_TARGETED_TOP8_RESCUE_FAMILY_FROM_TAIL_STRUCTURE"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3G1B
#
# Targeted GMEAN Top8 Rescue Frontier
#
# Motivation from G3G1A development oracle diagnostic:
#
#   GMEAN Top8:
#       183 / 212 frozen true
#       155 / 178 full<=7
#
#   Tail:
#       29 true
#       23 full<=7
#
#   Tail topology:
#
#       cuts=0 : 11 true, 10 full7
#       cuts=1 :  8 true,  8 full7
#       cuts=2 : 10 true,  5 full7
#
# Therefore:
#
#   18 / 23 full7 misses are in the relatively cheap
#   zero/one-cut region.
#
# Meanwhile GMEAN Top8 candidate cardinality:
#
#       cuts=0 :    182,339
#       cuts=1 :  1,997,681
#       cuts=2 : 14,176,065
#
# Thus 86.7% of Top8 candidates are two-cut.
#
# ------------------------------------------------------------
# DESIGN
# ------------------------------------------------------------
#
# Base:
#
#   GMEAN endpoint Top8 everywhere.
#
# Targeted low-cut rescue:
#
#   expand only cuts <= 1 to:
#
#       Top14
#       Top20
#
# Optional two-cut rescue:
#
#   after low-cut Top20:
#
#       two-cut GMEAN Top10
#       two-cut GMEAN Top13
#
# Alternative-score two-cut rescue:
#
#       two-cut:
#           SYM Top8 OR MAX Top8
#
# Global Top10/13/20 are included only as engineering
# comparators.
#
# ------------------------------------------------------------
# PART A
# ------------------------------------------------------------
#
# GT blind:
#
#   regenerate authoritative frozen generator
#   compute endpoint competition ranks
#   count each rescue contract
#   persist no proposal universe
#   freeze cardinality before oracle read
#
# ------------------------------------------------------------
# PART B
# ------------------------------------------------------------
#
# read G3G1 oracle only after Part A freeze
# evaluate true-pair retention and fold stability
#
# IMPORTANT:
#
# Rescue FAMILY is development-GT-informed by G3G1A.
# Actual shortlist implementation is GT-blind.
#
# No Fold0.
# No classifier.
# No final shortlist selected.
# No graph modification.
#
# Keep after running: YES
# ============================================================

from pathlib import Path
import gc
import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


G3G1_CARDINALITY_PATH = (
    S12
    / "stage12_folds14_endpoint_competition_shortlist_cardinality_v1.json"
)


G3G1_ORACLE_PATH = (
    S12
    / "stage12_folds14_endpoint_competition_shortlist_oracle_v1.pkl"
)


CARDINALITY_OUT = (
    S12
    / "stage12_folds14_endpoint_competition_targeted_rescue_cardinality_v1.json"
)


ORACLE_OUT = (
    S12
    / "stage12_folds14_endpoint_competition_targeted_rescue_oracle_v1.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_folds14_endpoint_competition_targeted_rescue_summary_v1.json"
)


assert G3G1_CARDINALITY_PATH.exists()
assert G3G1_ORACLE_PATH.exists()


for path in [
    CARDINALITY_OUT,
    ORACLE_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3G1B output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite a completed targeted rescue audit."
        )


# ============================================================
# 1. Required verified G3G1 helpers
# ============================================================

for helper_name in [
    "generate_dataset_proposals",
    "add_endpoint_ranks",
]:

    assert (
        helper_name in globals()
        and
        callable(
            globals()[
                helper_name
            ]
        )
    ), (
        f"{helper_name} missing from kernel. "
        "Re-run the retained G3G1 helper-definition portion first."
    )


# ============================================================
# 2. Load GT-blind G3G1 Part A only
# ============================================================

g3g1_part_a = json.loads(
    G3G1_CARDINALITY_PATH.read_text(
        encoding="utf-8"
    )
)


AUTHORITATIVE_SHA = (
    "da408df857cd34dcb4b1f65f3fae545e"
    "299f9403ef5edb986584caa8f4c3c203"
)


assert g3g1_part_a[
    "authoritative_generator_sha256"
] == AUTHORITATIVE_SHA


assert g3g1_part_a[
    "gt_used_during_part_a"
] is False


assert int(
    g3g1_part_a[
        "generator_raw_rows"
    ]
) == 36_473_971


assert int(
    g3g1_part_a[
        "generator_retained_rows"
    ]
) == 29_227_030


# ============================================================
# 3. Recover exact GMEAN Top8 cardinality
# ============================================================

g3g1_cardinality = pd.DataFrame(
    g3g1_part_a[
        "cardinality"
    ]
)


base_row = g3g1_cardinality[
    g3g1_cardinality[
        "shortlist"
    ].eq(
        "GMEAN_RATIO_TOP8"
    )
]


assert len(
    base_row
) == 1


base_row = base_row.iloc[
    0
]


EXPECTED_BASE_ROWS = int(
    base_row[
        "rows"
    ]
)


EXPECTED_BASE_CUT0 = int(
    base_row[
        "cuts_0"
    ]
)


EXPECTED_BASE_CUT1 = int(
    base_row[
        "cuts_1"
    ]
)


EXPECTED_BASE_CUT2 = int(
    base_row[
        "cuts_2"
    ]
)


assert EXPECTED_BASE_ROWS == 16_356_085

assert EXPECTED_BASE_CUT0 == 182_339
assert EXPECTED_BASE_CUT1 == 1_997_681
assert EXPECTED_BASE_CUT2 == 14_176_065


print(
    "========== 12.11G3G1B INPUT =========="
)

print(
    "authoritative generator SHA:",
    AUTHORITATIVE_SHA
)

print(
    "frozen-generator rows:",
    29_227_030
)

print(
    "GMEAN Top8 rows:",
    EXPECTED_BASE_ROWS
)

print(
    "GMEAN Top8 cuts0:",
    EXPECTED_BASE_CUT0
)

print(
    "GMEAN Top8 cuts1:",
    EXPECTED_BASE_CUT1
)

print(
    "GMEAN Top8 cuts2:",
    EXPECTED_BASE_CUT2
)


# ============================================================
# 4. Development corpus
# ============================================================

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


cv = pd.read_csv(
    CV_PATH
)


dev_manifest = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
)


development_names = dev_manifest[
    "dataset"
].tolist()


assert len(
    development_names
) == 103


fold_map_local = dict(
    zip(
        dev_manifest[
            "dataset"
        ],
        dev_manifest[
            "fold"
        ].astype(int),
    )
)


# ============================================================
# 5. Fixed targeted rescue family
#
# IMPORTANT:
# Family is development-informed by G3G1A.
# It is defined BEFORE this cell reads G3G1 oracle.
# ============================================================

CONTRACTS = {
    # --------------------------------------------------------
    # Existing G3G1 reference
    # --------------------------------------------------------

    "BASE_G8": {
        "type":
            "GLOBAL_GMEAN",

        "gmean_n":
            8,
    },

    # --------------------------------------------------------
    # Global comparators
    # --------------------------------------------------------

    "GLOBAL_G10": {
        "type":
            "GLOBAL_GMEAN",

        "gmean_n":
            10,
    },

    "GLOBAL_G13": {
        "type":
            "GLOBAL_GMEAN",

        "gmean_n":
            13,
    },

    "GLOBAL_G20": {
        "type":
            "GLOBAL_GMEAN",

        "gmean_n":
            20,
    },

    # --------------------------------------------------------
    # Main targeted rescue:
    #
    # G8 everywhere, widen only low-cut proposals.
    # --------------------------------------------------------

    "G8_LOWCUT14": {
        "type":
            "LOWCUT_GMEAN",

        "base_n":
            8,

        "lowcut_n":
            14,
    },

    "G8_LOWCUT20": {
        "type":
            "LOWCUT_GMEAN",

        "base_n":
            8,

        "lowcut_n":
            20,
    },

    # --------------------------------------------------------
    # Add small two-cut rank shells after low-cut Top20.
    # --------------------------------------------------------

    "G8_LOWCUT20_TWOCUT10": {
        "type":
            "LOWCUT_PLUS_TWOCUT_GMEAN",

        "base_n":
            8,

        "lowcut_n":
            20,

        "twocut_n":
            10,
    },

    "G8_LOWCUT20_TWOCUT13": {
        "type":
            "LOWCUT_PLUS_TWOCUT_GMEAN",

        "base_n":
            8,

        "lowcut_n":
            20,

        "twocut_n":
            13,
    },

    # --------------------------------------------------------
    # Alternative-score rescue only for expensive two-cut tail.
    # --------------------------------------------------------

    "G8_LOWCUT20_TWOCUT_ALT8": {
        "type":
            "LOWCUT_PLUS_TWOCUT_ALT",

        "base_n":
            8,

        "lowcut_n":
            20,

        "alt_n":
            8,
    },
}


print(
    "\n========== TARGETED RESCUE FAMILY =========="
)


for name, spec in CONTRACTS.items():

    print(
        name,
        spec
    )


# ============================================================
# 6. Rank helpers
# ============================================================

GM_SRC = (
    "GMEAN_RATIO_SOURCE_RANK"
)

GM_TGT = (
    "GMEAN_RATIO_TARGET_RANK"
)


SYM_SRC = (
    "SYM_CV_SOURCE_RANK"
)

SYM_TGT = (
    "SYM_CV_TARGET_RANK"
)


MAX_SRC = (
    "MAX_CV_SOURCE_RANK"
)

MAX_TGT = (
    "MAX_CV_TARGET_RANK"
)


def add_best_endpoint_ranks(
    proposals,
):

    proposals = proposals.copy()


    proposals[
        "GM_BEST"
    ] = np.minimum(
        proposals[
            GM_SRC
        ].to_numpy(),

        proposals[
            GM_TGT
        ].to_numpy(),
    )


    proposals[
        "SYM_BEST"
    ] = np.minimum(
        proposals[
            SYM_SRC
        ].to_numpy(),

        proposals[
            SYM_TGT
        ].to_numpy(),
    )


    proposals[
        "MAX_BEST"
    ] = np.minimum(
        proposals[
            MAX_SRC
        ].to_numpy(),

        proposals[
            MAX_TGT
        ].to_numpy(),
    )


    return proposals


def contract_mask(
    proposals,
    contract_name,
):

    spec = CONTRACTS[
        contract_name
    ]


    typ = spec[
        "type"
    ]


    gm = proposals[
        "GM_BEST"
    ]


    cuts = proposals[
        "cuts"
    ]


    if typ == "GLOBAL_GMEAN":

        return gm.le(
            int(
                spec[
                    "gmean_n"
                ]
            )
        )


    if typ == "LOWCUT_GMEAN":

        base_n = int(
            spec[
                "base_n"
            ]
        )


        lowcut_n = int(
            spec[
                "lowcut_n"
            ]
        )


        return (
            gm.le(
                base_n
            )
            |
            (
                cuts.le(
                    1
                )
                &
                gm.le(
                    lowcut_n
                )
            )
        )


    if typ == "LOWCUT_PLUS_TWOCUT_GMEAN":

        base_n = int(
            spec[
                "base_n"
            ]
        )


        lowcut_n = int(
            spec[
                "lowcut_n"
            ]
        )


        twocut_n = int(
            spec[
                "twocut_n"
            ]
        )


        return (
            gm.le(
                base_n
            )
            |
            (
                cuts.le(
                    1
                )
                &
                gm.le(
                    lowcut_n
                )
            )
            |
            (
                cuts.eq(
                    2
                )
                &
                gm.le(
                    twocut_n
                )
            )
        )


    if typ == "LOWCUT_PLUS_TWOCUT_ALT":

        base_n = int(
            spec[
                "base_n"
            ]
        )


        lowcut_n = int(
            spec[
                "lowcut_n"
            ]
        )


        alt_n = int(
            spec[
                "alt_n"
            ]
        )


        alternative_pass = (
            proposals[
                "SYM_BEST"
            ].le(
                alt_n
            )
            |
            proposals[
                "MAX_BEST"
            ].le(
                alt_n
            )
        )


        return (
            gm.le(
                base_n
            )
            |
            (
                cuts.le(
                    1
                )
                &
                gm.le(
                    lowcut_n
                )
            )
            |
            (
                cuts.eq(
                    2
                )
                &
                alternative_pass
            )
        )


    raise ValueError(
        (
            contract_name,
            spec,
        )
    )


# ============================================================
# PART A
#
# GT-BLIND cardinality scan
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART A — GT-BLIND TARGETED RESCUE CARDINALITY"
)

print(
    "============================================================"
)


stats = {
    name: {
        "rows":
            0,

        "cuts_0":
            0,

        "cuts_1":
            0,

        "cuts_2":
            0,

        "folds": {
            "1":
                0,

            "2":
                0,

            "3":
                0,

            "4":
                0,
        },
    }

    for name
    in CONTRACTS
}


dataset_results = []


raw_total = 0
retained_total = 0


for ds_i, dataset in enumerate(
    development_names,
    start=1,
):

    fold = int(
        fold_map_local[
            dataset
        ]
    )


    (
        proposals,
        raw_count,
        retained_count,
    ) = generate_dataset_proposals(
        dataset
    )


    proposals = add_endpoint_ranks(
        proposals
    )


    proposals = add_best_endpoint_ranks(
        proposals
    )


    raw_total += int(
        raw_count
    )


    retained_total += int(
        retained_count
    )


    local = {}


    for contract_name in CONTRACTS:

        mask = contract_mask(
            proposals,
            contract_name,
        )


        count = int(
            mask.sum()
        )


        local[
            contract_name
        ] = count


        stats[
            contract_name
        ][
            "rows"
        ] += count


        stats[
            contract_name
        ][
            "folds"
        ][
            str(
                fold
            )
        ] += count


        for cuts in [
            0,
            1,
            2,
        ]:

            cut_count = int(
                (
                    mask
                    &
                    proposals[
                        "cuts"
                    ].eq(
                        cuts
                    )
                ).sum()
            )


            stats[
                contract_name
            ][
                f"cuts_{cuts}"
            ] += cut_count


    dataset_results.append(
        {
            "dataset":
                dataset,

            "fold":
                fold,

            "raw_generator":
                int(
                    raw_count
                ),

            "retained_generator":
                int(
                    retained_count
                ),

            "contracts": {
                key:
                    int(
                        value
                    )

                for key, value
                in local.items()
            },
        }
    )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            development_names
        )
    ):

        print(
            f"{ds_i:3d}/{len(development_names)}"
            f" | fold={fold}"
            f" | {dataset}"
            f" | frozen={retained_count}"
            f" | G8={local['BASE_G8']}"
            f" | LC14={local['G8_LOWCUT14']}"
            f" | LC20={local['G8_LOWCUT20']}"
            f" | LC20+C13={local['G8_LOWCUT20_TWOCUT13']}"
        )


    del proposals

    gc.collect()


# ============================================================
# 7. Generator fidelity
# ============================================================

print(
    "\n========== AUTHORITATIVE GENERATOR REPRODUCTION =========="
)

print(
    "raw:",
    raw_total,
    "/ 36473971"
)

print(
    "retained:",
    retained_total,
    "/ 29227030"
)


assert raw_total == 36_473_971
assert retained_total == 29_227_030


print(
    "AUTHORITATIVE_GENERATOR_REPRODUCTION: PASS"
)


# ============================================================
# 8. GMEAN Top8 fidelity
# ============================================================

assert int(
    stats[
        "BASE_G8"
    ][
        "rows"
    ]
) == EXPECTED_BASE_ROWS


assert int(
    stats[
        "BASE_G8"
    ][
        "cuts_0"
    ]
) == EXPECTED_BASE_CUT0


assert int(
    stats[
        "BASE_G8"
    ][
        "cuts_1"
    ]
) == EXPECTED_BASE_CUT1


assert int(
    stats[
        "BASE_G8"
    ][
        "cuts_2"
    ]
) == EXPECTED_BASE_CUT2


print(
    "\nGMEAN_TOP8_REPRODUCTION: PASS"
)


# ============================================================
# 9. GT-blind cardinality table
# ============================================================

cardinality_rows = []


for name in CONTRACTS:

    entry = stats[
        name
    ]


    rows = int(
        entry[
            "rows"
        ]
    )


    assert (
        int(
            entry[
                "cuts_0"
            ]
        )
        +
        int(
            entry[
                "cuts_1"
            ]
        )
        +
        int(
            entry[
                "cuts_2"
            ]
        )
    ) == rows


    cardinality_rows.append(
        {
            "contract":
                name,

            "rows":
                rows,

            "fraction_of_frozen_generator":
                float(
                    rows
                    /
                    29_227_030
                ),

            "multiplier_vs_g8":
                float(
                    rows
                    /
                    EXPECTED_BASE_ROWS
                ),

            "cuts_0":
                int(
                    entry[
                        "cuts_0"
                    ]
                ),

            "cuts_1":
                int(
                    entry[
                        "cuts_1"
                    ]
                ),

            "cuts_2":
                int(
                    entry[
                        "cuts_2"
                    ]
                ),
        }
    )


cardinality_table = (
    pd.DataFrame(
        cardinality_rows
    )
    .sort_values(
        [
            "rows",
            "contract",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n========== GT-BLIND TARGETED RESCUE CARDINALITY =========="
)


print(
    cardinality_table.to_string(
        index=False
    )
)


# ============================================================
# 10. Fold cardinality
# ============================================================

fold_rows = []


for name in CONTRACTS:

    for fold in [
        1,
        2,
        3,
        4,
    ]:

        fold_rows.append(
            {
                "contract":
                    name,

                "fold":
                    int(
                        fold
                    ),

                "rows":
                    int(
                        stats[
                            name
                        ][
                            "folds"
                        ][
                            str(
                                fold
                            )
                        ]
                    ),
            }
        )


fold_table = pd.DataFrame(
    fold_rows
)


# ============================================================
# 11. Freeze Part A BEFORE oracle read
# ============================================================

part_a = {
    "stage":
        "12.11G3G1B",

    "part":
        "GT_BLIND_TARGETED_ENDPOINT_RESCUE_CARDINALITY",

    "authoritative_generator_sha256":
        AUTHORITATIVE_SHA,

    "generator_raw_rows":
        int(
            raw_total
        ),

    "generator_retained_rows":
        int(
            retained_total
        ),

    "base_gmean_top8_rows":
        EXPECTED_BASE_ROWS,

    "rescue_family_origin":
        "DEVELOPMENT_GT_INFORMED_G3G1A",

    "actual_rescue_features_gt_blind":
        True,

    "contracts":
        CONTRACTS,

    "cardinality":
        cardinality_table.to_dict(
            "records"
        ),

    "fold_cardinality":
        fold_table.to_dict(
            "records"
        ),

    "dataset_results":
        dataset_results,

    "proposal_universe_persisted":
        False,

    "gt_used_during_part_a":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "shortlist_selected":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


canonical = json.dumps(
    part_a,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
).encode(
    "utf-8"
)


part_a_sha = hashlib.sha256(
    canonical
).hexdigest()


part_a[
    "sha256"
] = part_a_sha


CARDINALITY_OUT.write_text(
    json.dumps(
        part_a,
        indent=2,
    ),
    encoding="utf-8",
)


assert CARDINALITY_OUT.exists()


print(
    "\nTARGETED RESCUE CARDINALITY FROZEN BEFORE GT READ: YES"
)

print(
    "Part-A SHA256:",
    part_a_sha
)


# ============================================================
# PART B
#
# Development oracle read only below this line.
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — DEVELOPMENT ORACLE TARGETED RESCUE"
)

print(
    "============================================================"
)


oracle = pd.read_pickle(
    G3G1_ORACLE_PATH
)


assert len(
    oracle
) == 369


frozen_true = oracle[
    "combined_ASYM_3x8"
].astype(bool)


assert int(
    frozen_true.sum()
) == 212


# ============================================================
# 12. Oracle cuts
# ============================================================

topology_to_cuts = {
    "FREE_FREE":
        0,

    "SOURCE_OCCUPIED":
        1,

    "TARGET_OCCUPIED":
        1,

    "BOTH_OCCUPIED":
        2,
}


oracle[
    "n_required_cuts_rescue"
] = (
    oracle[
        "topology_class"
    ]
    .map(
        topology_to_cuts
    )
)


assert oracle[
    "n_required_cuts_rescue"
].notna().all()


# ============================================================
# 13. Oracle best-endpoint ranks
# ============================================================

oracle[
    "GM_BEST"
] = oracle[
    "GMEAN_RATIO_TRUE_BEST_ENDPOINT_RANK"
]


oracle[
    "SYM_BEST"
] = oracle[
    "SYM_CV_TRUE_BEST_ENDPOINT_RANK"
]


oracle[
    "MAX_BEST"
] = oracle[
    "MAX_CV_TRUE_BEST_ENDPOINT_RANK"
]


for col in [
    "GM_BEST",
    "SYM_BEST",
    "MAX_BEST",
]:

    assert oracle.loc[
        frozen_true,
        col,
    ].notna().all()


# ============================================================
# 14. Apply same contracts to true pairs
# ============================================================

def oracle_contract_mask(
    frame,
    contract_name,
):

    spec = CONTRACTS[
        contract_name
    ]


    typ = spec[
        "type"
    ]


    gm = frame[
        "GM_BEST"
    ]


    cuts = frame[
        "n_required_cuts_rescue"
    ]


    if typ == "GLOBAL_GMEAN":

        pass_mask = gm.le(
            int(
                spec[
                    "gmean_n"
                ]
            )
        )


    elif typ == "LOWCUT_GMEAN":

        pass_mask = (
            gm.le(
                int(
                    spec[
                        "base_n"
                    ]
                )
            )
            |
            (
                cuts.le(
                    1
                )
                &
                gm.le(
                    int(
                        spec[
                            "lowcut_n"
                        ]
                    )
                )
            )
        )


    elif typ == "LOWCUT_PLUS_TWOCUT_GMEAN":

        pass_mask = (
            gm.le(
                int(
                    spec[
                        "base_n"
                    ]
                )
            )
            |
            (
                cuts.le(
                    1
                )
                &
                gm.le(
                    int(
                        spec[
                            "lowcut_n"
                        ]
                    )
                )
            )
            |
            (
                cuts.eq(
                    2
                )
                &
                gm.le(
                    int(
                        spec[
                            "twocut_n"
                        ]
                    )
                )
            )
        )


    elif typ == "LOWCUT_PLUS_TWOCUT_ALT":

        alt = (
            frame[
                "SYM_BEST"
            ].le(
                int(
                    spec[
                        "alt_n"
                    ]
                )
            )
            |
            frame[
                "MAX_BEST"
            ].le(
                int(
                    spec[
                        "alt_n"
                    ]
                )
            )
        )


        pass_mask = (
            gm.le(
                int(
                    spec[
                        "base_n"
                    ]
                )
            )
            |
            (
                cuts.le(
                    1
                )
                &
                gm.le(
                    int(
                        spec[
                            "lowcut_n"
                        ]
                    )
                )
            )
            |
            (
                cuts.eq(
                    2
                )
                &
                alt
            )
        )


    else:

        raise ValueError(
            (
                contract_name,
                spec,
            )
        )


    return (
        frozen_true
        &
        pass_mask.fillna(
            False
        )
    )


# ============================================================
# 15. Oracle/cardinality tradeoff
# ============================================================

full7 = oracle[
    "pred_full_7um"
].astype(bool)


oracle_rows = []


for contract_name in CONTRACTS:

    mask = oracle_contract_mask(
        oracle,
        contract_name,
    )


    col = (
        "rescue_"
        +
        contract_name
    )


    oracle[
        col
    ] = mask.astype(
        bool
    )


    candidate_row = cardinality_table[
        cardinality_table[
            "contract"
        ].eq(
            contract_name
        )
    ]


    assert len(
        candidate_row
    ) == 1


    candidate_rows = int(
        candidate_row[
            "rows"
        ].iloc[
            0
        ]
    )


    captured = int(
        mask.sum()
    )


    captured_full7 = int(
        (
            mask
            &
            full7
        ).sum()
    )


    oracle_rows.append(
        {
            "contract":
                contract_name,

            "candidate_rows":
                candidate_rows,

            "candidate_fraction_of_frozen_generator":
                float(
                    candidate_rows
                    /
                    29_227_030
                ),

            "candidate_multiplier_vs_g8":
                float(
                    candidate_rows
                    /
                    EXPECTED_BASE_ROWS
                ),

            "oracle_captured":
                captured,

            "oracle_retention":
                float(
                    captured
                    /
                    212
                ),

            "full7_captured":
                captured_full7,

            "full7_retention":
                float(
                    captured_full7
                    /
                    178
                ),
        }
    )


tradeoff = (
    pd.DataFrame(
        oracle_rows
    )
    .sort_values(
        [
            "candidate_rows",
            "contract",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 16. Incremental efficiency versus G8
# ============================================================

base_tradeoff = tradeoff[
    tradeoff[
        "contract"
    ].eq(
        "BASE_G8"
    )
].iloc[
    0
]


assert int(
    base_tradeoff[
        "oracle_captured"
    ]
) == 183


assert int(
    base_tradeoff[
        "full7_captured"
    ]
) == 155


tradeoff[
    "extra_rows_vs_g8"
] = (
    tradeoff[
        "candidate_rows"
    ]
    -
    int(
        base_tradeoff[
            "candidate_rows"
        ]
    )
)


tradeoff[
    "extra_oracle_vs_g8"
] = (
    tradeoff[
        "oracle_captured"
    ]
    -
    183
)


tradeoff[
    "extra_full7_vs_g8"
] = (
    tradeoff[
        "full7_captured"
    ]
    -
    155
)


tradeoff[
    "million_extra_rows_per_extra_oracle"
] = np.where(
    tradeoff[
        "extra_oracle_vs_g8"
    ] > 0,

    (
        tradeoff[
            "extra_rows_vs_g8"
        ]
        /
        1_000_000.0
    )
    /
    tradeoff[
        "extra_oracle_vs_g8"
    ],

    np.nan,
)


print(
    "\n========== TARGETED RESCUE ORACLE TRADEOFF =========="
)


print(
    tradeoff.to_string(
        index=False
    )
)


# ============================================================
# 17. Pareto-focused view
# ============================================================

print(
    "\n========== TARGETED RESCUE COST / BENEFIT =========="
)


print(
    tradeoff[
        [
            "contract",

            "candidate_rows",
            "candidate_multiplier_vs_g8",

            "oracle_captured",
            "oracle_retention",

            "full7_captured",
            "full7_retention",

            "extra_rows_vs_g8",
            "extra_oracle_vs_g8",
            "extra_full7_vs_g8",

            "million_extra_rows_per_extra_oracle",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 18. Fold stability
# ============================================================

fold_oracle_rows = []


for contract_name in CONTRACTS:

    col = (
        "rescue_"
        +
        contract_name
    )


    for fold in [
        1,
        2,
        3,
        4,
    ]:

        sub = oracle[
            oracle[
                "fold"
            ].eq(
                fold
            )
        ]


        frozen_count = int(
            sub[
                "combined_ASYM_3x8"
            ].sum()
        )


        retained_count = int(
            sub[
                col
            ].sum()
        )


        sub_full7 = (
            sub[
                "combined_ASYM_3x8"
            ].astype(bool)
            &
            sub[
                "pred_full_7um"
            ].astype(bool)
        )


        frozen_full7 = int(
            sub_full7.sum()
        )


        retained_full7 = int(
            (
                sub[
                    col
                ].astype(bool)
                &
                sub[
                    "pred_full_7um"
                ].astype(bool)
            ).sum()
        )


        fold_oracle_rows.append(
            {
                "contract":
                    contract_name,

                "fold":
                    fold,

                "frozen_true":
                    frozen_count,

                "retained_true":
                    retained_count,

                "true_retention":
                    (
                        float(
                            retained_count
                            /
                            frozen_count
                        )
                        if frozen_count
                        else np.nan
                    ),

                "frozen_full7":
                    frozen_full7,

                "retained_full7":
                    retained_full7,

                "full7_retention":
                    (
                        float(
                            retained_full7
                            /
                            frozen_full7
                        )
                        if frozen_full7
                        else np.nan
                    ),
            }
        )


fold_oracle_table = pd.DataFrame(
    fold_oracle_rows
)


print(
    "\n========== TARGETED RESCUE ORACLE RETENTION BY FOLD =========="
)


for contract_name in CONTRACTS:

    print(
        "\n" + contract_name
    )


    print(
        fold_oracle_table[
            fold_oracle_table[
                "contract"
            ].eq(
                contract_name
            )
        ].to_string(
            index=False
        )
    )


# ============================================================
# 19. Persist oracle audit
# ============================================================

oracle = (
    oracle
    .sort_values(
        [
            "fold",
            "dataset",
            "component_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


oracle.to_pickle(
    ORACLE_OUT
)


summary = {
    "stage":
        "12.11G3G1B",

    "status":
        "TARGETED_GMEAN_TOP8_RESCUE_FRONTIER_AUDITED",

    "authoritative_generator_sha256":
        AUTHORITATIVE_SHA,

    "part_a_sha256":
        part_a_sha,

    "rescue_family_origin":
        "DEVELOPMENT_GT_INFORMED_G3G1A",

    "actual_rescue_features_gt_blind":
        True,

    "tradeoff":
        tradeoff.to_dict(
            "records"
        ),

    "fold_oracle_retention":
        fold_oracle_table.to_dict(
            "records"
        ),

    "shortlist_selected":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert ORACLE_OUT.exists()
assert SUMMARY_OUT.exists()


# ============================================================
# 20. Reload integrity
# ============================================================

check = pd.read_pickle(
    ORACLE_OUT
)


assert len(
    check
) == 369


assert int(
    check[
        "combined_ASYM_3x8"
    ].sum()
) == 212


assert int(
    check[
        "rescue_BASE_G8"
    ].sum()
) == 183


# ============================================================
# 21. Decision
# ============================================================

print(
    "\n========== 12.11G3G1B DECISION =========="
)

print(
    "TARGETED_GMEAN_TOP8_RESCUE_FRONTIER_AUDIT: PASS"
)

print(
    "authoritative generator reproduced:",
    "YES"
)

print(
    "GMEAN Top8 reproduced:",
    "YES"
)

print(
    "Part-A cardinality frozen before GT read:",
    "YES"
)

print(
    "rescue-family development-GT-informed:",
    "YES"
)

print(
    "actual rescue implementation GT-blind:",
    "YES"
)

print(
    "proposal universe persisted:",
    "NO"
)

print(
    "shortlist selected:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "classifier trained:",
    "NO"
)

print(
    "rewire policy selected:",
    "NO"
)

print(
    "graph modified:",
    "NO"
)

print(
    "next:"
)

print(
    "FREEZE_TARGETED_RESCUE_SHORTLIST_OR_PROCEED_TO_CROSS_FOLD_SCORING"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3G2
#
# Freeze scorer-input shortlist:
#
#   G8_LOWCUT20
#
# Authoritative parent generator:
#
#   MULTIFRAME_REWIRE_ASYM3X8_DOMINANCE_ANY_V1_FIDELITYPATCH1
#
# Parent generator SHA:
#
#   da408df857cd34dcb4b1f65f3fae545e299f9403ef5edb986584caa8f4c3c203
#
# ------------------------------------------------------------
# FROZEN SHORTLIST RULE
# ------------------------------------------------------------
#
# Start from authoritative post-dominance ASYM3x8 proposals.
#
# Define endpoint competition score:
#
#   GMEAN_RATIO =
#       sqrt(source_local_ratio * target_local_ratio)
#
# where:
#
#   source_local_ratio =
#       candidate forward-CV endpoint residual /
#       best forward-CV endpoint residual for same source
#       and exact source_t -> target_t frame pair
#
#   target_local_ratio =
#       candidate backward-CV endpoint residual /
#       best backward-CV endpoint residual for same target
#       and exact source_t -> target_t frame pair
#
# Rank all retained proposals sharing each endpoint across
# ALL frozen gap sizes within the same dataset.
#
# best_endpoint_rank =
#       min(source_competition_rank,
#           target_competition_rank)
#
# Required cuts:
#
#   0 = FREE_FREE
#   1 = SOURCE_OCCUPIED or TARGET_OCCUPIED
#   2 = BOTH_OCCUPIED
#
# Pass:
#
#   if cuts <= 1:
#       best_endpoint_rank <= 20
#
#   if cuts == 2:
#       best_endpoint_rank <= 8
#
# Equivalent:
#
#       GMEAN Top8 globally
#       +
#       low-cut-only Top20 rescue
#
# ------------------------------------------------------------
# SELECTION EVIDENCE
# ------------------------------------------------------------
#
# candidates:
#       17,501,674
#
# frozen-generator fraction:
#       59.8818%
#
# development oracle:
#       202 / 212 = 95.283%
#
# full<=7:
#       173 / 178 = 97.191%
#
# No further Top-N tuning after this freeze.
#
# IMPORTANT:
#
#   Selection is development-GT-informed.
#   Runtime shortlist features are GT-blind.
#
#   Do NOT materialize 17.5M rows globally.
#   Future scoring must regenerate this shortlist streamingly.
#
# No Fold0.
# No classifier.
# No rewire policy.
# No graph modification.
#
# Keep after running: YES
# ============================================================

from pathlib import Path

import copy
import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


PARENT_GENERATOR_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_generator_asym3x8_v1_fidelitypatch1_frozen.json"
)


G3G1_CARDINALITY_PATH = (
    S12
    / "stage12_folds14_endpoint_competition_shortlist_cardinality_v1.json"
)


G3G1B_CARDINALITY_PATH = (
    S12
    / "stage12_folds14_endpoint_competition_targeted_rescue_cardinality_v1.json"
)


G3G1B_ORACLE_PATH = (
    S12
    / "stage12_folds14_endpoint_competition_targeted_rescue_oracle_v1.pkl"
)


G3G1B_SUMMARY_PATH = (
    S12
    / "stage12_folds14_endpoint_competition_targeted_rescue_summary_v1.json"
)


FROZEN_OUT = (
    S12
    / "stage12_folds14_endpoint_shortlist_g8_lowcut20_v1_frozen.json"
)


CONFIRMATION_OUT = (
    S12
    / "stage12_folds14_endpoint_shortlist_g8_lowcut20_v1_confirmation.json"
)


for path in [
    PARENT_GENERATOR_PATH,
    G3G1_CARDINALITY_PATH,
    G3G1B_CARDINALITY_PATH,
    G3G1B_ORACLE_PATH,
    G3G1B_SUMMARY_PATH,
]:

    assert path.exists(), path


for path in [
    FROZEN_OUT,
    CONFIRMATION_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3G2 frozen shortlist already exists:\n"
            f"{path}\n\n"
            "Do not overwrite a frozen shortlist."
        )


# ============================================================
# 1. Helpers
# ============================================================

def file_sha256(
    path,
):

    hasher = hashlib.sha256()


    with open(
        path,
        "rb",
    ) as handle:

        while True:

            block = handle.read(
                1024 * 1024
            )


            if not block:

                break


            hasher.update(
                block
            )


    return hasher.hexdigest()


def canonical_sha256(
    payload,
):

    data = json.dumps(
        payload,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )


    return hashlib.sha256(
        data
    ).hexdigest()


# ============================================================
# 2. Parent authoritative generator
# ============================================================

AUTHORITATIVE_GENERATOR_SHA = (
    "da408df857cd34dcb4b1f65f3fae545e"
    "299f9403ef5edb986584caa8f4c3c203"
)


parent_generator = json.loads(
    PARENT_GENERATOR_PATH.read_text(
        encoding="utf-8"
    )
)


assert parent_generator[
    "generator_contract_sha256"
] == AUTHORITATIVE_GENERATOR_SHA


assert parent_generator[
    "generator_id"
] == (
    "MULTIFRAME_REWIRE_ASYM3X8_"
    "DOMINANCE_ANY_V1_FIDELITYPATCH1"
)


assert parent_generator[
    "retrieval"
][
    "exact_knn_query_k"
] == 16


assert parent_generator[
    "retrieval"
][
    "strong_rank"
] == 3


assert parent_generator[
    "retrieval"
][
    "weak_rank"
] == 8


assert parent_generator[
    "development_cardinality"
][
    "retained_rows"
] == 29_227_030


print(
    "========== 12.11G3G2 PARENT GENERATOR =========="
)

print(
    "generator:",
    parent_generator[
        "generator_id"
    ]
)

print(
    "SHA:",
    AUTHORITATIVE_GENERATOR_SHA
)

print(
    "post-dominance rows:",
    29_227_030
)


# ============================================================
# 3. G3G1 GT-blind endpoint-ranking provenance
# ============================================================

g3g1 = json.loads(
    G3G1_CARDINALITY_PATH.read_text(
        encoding="utf-8"
    )
)


assert g3g1[
    "stage"
] == "12.11G3G1"


assert g3g1[
    "authoritative_generator_sha256"
] == AUTHORITATIVE_GENERATOR_SHA


assert g3g1[
    "gt_used_during_part_a"
] is False


assert g3g1[
    "shortlist_rule"
] == (
    "SOURCE_SCORE_RANK_LE_N_OR_TARGET_SCORE_RANK_LE_N"
)


# Preserve exact deterministic tie breaking.
source_tie_break = g3g1[
    "tie_breaking"
][
    "source"
]


target_tie_break = g3g1[
    "tie_breaking"
][
    "target"
]


assert source_tie_break == (
    "score,gap_size,target_id"
)


assert target_tie_break == (
    "score,gap_size,source_id"
)


# ============================================================
# 4. G3G1B Part-A fidelity
# ============================================================

g3g1b_part_a = json.loads(
    G3G1B_CARDINALITY_PATH.read_text(
        encoding="utf-8"
    )
)


assert g3g1b_part_a[
    "stage"
] == "12.11G3G1B"


assert g3g1b_part_a[
    "part"
] == (
    "GT_BLIND_TARGETED_ENDPOINT_RESCUE_CARDINALITY"
)


assert g3g1b_part_a[
    "authoritative_generator_sha256"
] == AUTHORITATIVE_GENERATOR_SHA


assert g3g1b_part_a[
    "gt_used_during_part_a"
] is False


assert g3g1b_part_a[
    "fold0_used"
] is False


assert int(
    g3g1b_part_a[
        "generator_raw_rows"
    ]
) == 36_473_971


assert int(
    g3g1b_part_a[
        "generator_retained_rows"
    ]
) == 29_227_030


EXPECTED_G3G1B_PART_A_SHA = (
    "56f0884580fdc34162e01019054d56b1c"
    "4d7ae6bb7d10c4ee20b3dead488f8a9"
)


assert g3g1b_part_a[
    "sha256"
] == EXPECTED_G3G1B_PART_A_SHA


# ============================================================
# 5. Extract G8_LOWCUT20 cardinality
# ============================================================

cardinality = pd.DataFrame(
    g3g1b_part_a[
        "cardinality"
    ]
)


selected_card = cardinality[
    cardinality[
        "contract"
    ].eq(
        "G8_LOWCUT20"
    )
]


assert len(
    selected_card
) == 1


selected_card = selected_card.iloc[
    0
]


SELECTED_ROWS = int(
    selected_card[
        "rows"
    ]
)


SELECTED_CUT0 = int(
    selected_card[
        "cuts_0"
    ]
)


SELECTED_CUT1 = int(
    selected_card[
        "cuts_1"
    ]
)


SELECTED_CUT2 = int(
    selected_card[
        "cuts_2"
    ]
)


assert SELECTED_ROWS == 17_501_674

assert SELECTED_CUT0 == 267_632
assert SELECTED_CUT1 == 3_057_977
assert SELECTED_CUT2 == 14_176_065


assert (
    SELECTED_CUT0
    +
    SELECTED_CUT1
    +
    SELECTED_CUT2
) == SELECTED_ROWS


# ============================================================
# 6. Development selection evidence
# ============================================================

g3g1b_summary = json.loads(
    G3G1B_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert g3g1b_summary[
    "stage"
] == "12.11G3G1B"


assert g3g1b_summary[
    "shortlist_selected"
] is False


assert g3g1b_summary[
    "fold0_used"
] is False


tradeoff = pd.DataFrame(
    g3g1b_summary[
        "tradeoff"
    ]
)


def get_tradeoff(
    contract,
):

    row = tradeoff[
        tradeoff[
            "contract"
        ].eq(
            contract
        )
    ]


    assert len(
        row
    ) == 1


    return row.iloc[
        0
    ]


base_g8 = get_tradeoff(
    "BASE_G8"
)


lowcut14 = get_tradeoff(
    "G8_LOWCUT14"
)


lowcut20 = get_tradeoff(
    "G8_LOWCUT20"
)


twocut10 = get_tradeoff(
    "G8_LOWCUT20_TWOCUT10"
)


twocut13 = get_tradeoff(
    "G8_LOWCUT20_TWOCUT13"
)


# ============================================================
# 7. Exact evidence assertions
# ============================================================

assert int(
    base_g8[
        "candidate_rows"
    ]
) == 16_356_085


assert int(
    base_g8[
        "oracle_captured"
    ]
) == 183


assert int(
    base_g8[
        "full7_captured"
    ]
) == 155


# ------------------------------------------------------------
# Selected contract
# ------------------------------------------------------------

assert int(
    lowcut20[
        "candidate_rows"
    ]
) == 17_501_674


assert int(
    lowcut20[
        "oracle_captured"
    ]
) == 202


assert int(
    lowcut20[
        "full7_captured"
    ]
) == 173


assert np.isclose(
    float(
        lowcut20[
            "oracle_retention"
        ]
    ),
    202 / 212,
    atol=1e-12,
)


assert np.isclose(
    float(
        lowcut20[
            "full7_retention"
        ]
    ),
    173 / 178,
    atol=1e-12,
)


# ------------------------------------------------------------
# Knee evidence: lowcut14 -> lowcut20
# ------------------------------------------------------------

increment_14_to_20_rows = (
    int(
        lowcut20[
            "candidate_rows"
        ]
    )
    -
    int(
        lowcut14[
            "candidate_rows"
        ]
    )
)


increment_14_to_20_true = (
    int(
        lowcut20[
            "oracle_captured"
        ]
    )
    -
    int(
        lowcut14[
            "oracle_captured"
        ]
    )
)


increment_14_to_20_full7 = (
    int(
        lowcut20[
            "full7_captured"
        ]
    )
    -
    int(
        lowcut14[
            "full7_captured"
        ]
    )
)


assert increment_14_to_20_rows == 295_405
assert increment_14_to_20_true == 4
assert increment_14_to_20_full7 == 3


# ------------------------------------------------------------
# Knee evidence: lowcut20 -> two-cut Top10
# ------------------------------------------------------------

increment_20_to_twocut10_rows = (
    int(
        twocut10[
            "candidate_rows"
        ]
    )
    -
    int(
        lowcut20[
            "candidate_rows"
        ]
    )
)


increment_20_to_twocut10_true = (
    int(
        twocut10[
            "oracle_captured"
        ]
    )
    -
    int(
        lowcut20[
            "oracle_captured"
        ]
    )
)


increment_20_to_twocut10_full7 = (
    int(
        twocut10[
            "full7_captured"
        ]
    )
    -
    int(
        lowcut20[
            "full7_captured"
        ]
    )
)


assert increment_20_to_twocut10_rows == 2_671_176
assert increment_20_to_twocut10_true == 2
assert increment_20_to_twocut10_full7 == 1


# ============================================================
# 8. Selection evidence output
# ============================================================

print(
    "\n========== 12.11G3G2 SELECTION EVIDENCE =========="
)

print(
    "BASE G8:"
)

print(
    " rows:",
    int(
        base_g8[
            "candidate_rows"
        ]
    )
)

print(
    " true:",
    int(
        base_g8[
            "oracle_captured"
        ]
    ),
    "/ 212"
)

print(
    " full7:",
    int(
        base_g8[
            "full7_captured"
        ]
    ),
    "/ 178"
)


print(
    "\nG8_LOWCUT20:"
)

print(
    " rows:",
    SELECTED_ROWS
)

print(
    " fraction of frozen generator:",
    SELECTED_ROWS
    /
    29_227_030
)

print(
    " true:",
    202,
    "/ 212",
    "=",
    202 / 212
)

print(
    " full7:",
    173,
    "/ 178",
    "=",
    173 / 178
)


print(
    "\nLOWCUT14 -> LOWCUT20:"
)

print(
    " extra rows:",
    increment_14_to_20_rows
)

print(
    " extra true:",
    increment_14_to_20_true
)

print(
    " extra full7:",
    increment_14_to_20_full7
)

print(
    " rows per extra true:",
    increment_14_to_20_rows
    /
    increment_14_to_20_true
)


print(
    "\nLOWCUT20 -> TWOCUT10:"
)

print(
    " extra rows:",
    increment_20_to_twocut10_rows
)

print(
    " extra true:",
    increment_20_to_twocut10_true
)

print(
    " extra full7:",
    increment_20_to_twocut10_full7
)

print(
    " rows per extra true:",
    increment_20_to_twocut10_rows
    /
    increment_20_to_twocut10_true
)


# ============================================================
# 9. Freeze exact shortlist contract
# ============================================================

frozen_shortlist = {
    "stage":
        "12.11G3G2",

    "status":
        "FROZEN",

    "shortlist_id":
        "ENDPOINT_SHORTLIST_GMEAN_G8_LOWCUT20_V1",

    # --------------------------------------------------------
    # Parent generator
    # --------------------------------------------------------

    "parent_generator": {
        "generator_id":
            parent_generator[
                "generator_id"
            ],

        "generator_contract_sha256":
            AUTHORITATIVE_GENERATOR_SHA,

        "artifact_path":
            str(
                PARENT_GENERATOR_PATH
            ),

        "artifact_file_sha256":
            file_sha256(
                PARENT_GENERATOR_PATH
            ),
    },

    # --------------------------------------------------------
    # Selection provenance
    # --------------------------------------------------------

    "selection_origin":
        "DEVELOPMENT_GT_INFORMED_G3G1_G3G1A_G3G1B",

    "selection_reason":
        (
            "G8_LOWCUT20 is the targeted-rescue engineering "
            "knee. Relative to GMEAN Top8 it recovers 19 "
            "additional frozen-generator true pairs and 18 "
            "additional full<=7 true pairs for only 1,145,589 "
            "additional candidates. Expanding further into "
            "two-cut Top10 requires 2,671,176 additional "
            "candidates for only 2 more true pairs and 1 more "
            "full<=7 pair."
        ),

    "runtime_features_gt_blind":
        True,

    # --------------------------------------------------------
    # Input proposal contract
    # --------------------------------------------------------

    "input": {
        "proposal_source":
            (
                "AUTHORITATIVE_POST_DOMINANCE_"
                "ASYM3X8_FROZEN_GENERATOR"
            ),

        "proposal_count_development":
            29_227_030,

        "full_input_materialized":
            False,

        "streaming_regeneration_required":
            True,
    },

    # --------------------------------------------------------
    # Local GMEAN score
    # --------------------------------------------------------

    "competition_score": {
        "id":
            "GMEAN_RATIO",

        "source_local_ratio":
            (
                "candidate_forward_cv_endpoint_distance / "
                "best_forward_cv_endpoint_distance_for_same_"
                "source_and_exact_frame_pair"
            ),

        "target_local_ratio":
            (
                "candidate_backward_cv_endpoint_distance / "
                "best_backward_cv_endpoint_distance_for_same_"
                "target_and_exact_frame_pair"
            ),

        "score":
            (
                "sqrt(source_local_ratio * "
                "target_local_ratio)"
            ),

        "source_and_target_local_best_origin":
            (
                "EXACT_K16_CKDTREE_QUERY_USED_BY_"
                "AUTHORITATIVE_PARENT_GENERATOR"
            ),

        "nonfinite_ratio_policy":
            (
                "safe_ratio: if best<=1e-12 and candidate<=1e-12 "
                "return 1; if best<=1e-12 and candidate>1e-12 "
                "return +inf"
            ),
    },

    # --------------------------------------------------------
    # Endpoint competition rank contract
    # --------------------------------------------------------

    "endpoint_ranking": {
        "scope":
            (
                "ALL_AUTHORITATIVE_POST_DOMINANCE_PROPOSALS_"
                "WITHIN_SAME_DATASET"
            ),

        "source_group":
            "source_id",

        "target_group":
            "target_id",

        "source_sort":
            [
                "GMEAN_RATIO",
                "gap_size",
                "target_id",
            ],

        "target_sort":
            [
                "GMEAN_RATIO",
                "gap_size",
                "source_id",
            ],

        "sorting_kind":
            "stable",

        "rank_definition":
            "cumcount + 1",

        "best_endpoint_rank":
            (
                "min(source_competition_rank, "
                "target_competition_rank)"
            ),
    },

    # --------------------------------------------------------
    # Topology/cut contract
    # --------------------------------------------------------

    "required_cuts": {
        "definition":
            (
                "int(source_outdegree==1) + "
                "int(target_indegree==1)"
            ),

        "free_free":
            0,

        "source_occupied":
            1,

        "target_occupied":
            1,

        "both_occupied":
            2,

        "topology_state":
            "AUTHORITATIVE_CURRENT_BEST_PRE_REWIRE_GRAPH",
    },

    # --------------------------------------------------------
    # Frozen shortlist rule
    # --------------------------------------------------------

    "pass_rule": {
        "human_readable":
            (
                "GMEAN Top8 everywhere, with Top20 rescue only "
                "for proposals requiring zero or one incumbent "
                "edge cut."
            ),

        "equivalent_logic":
            (
                "(best_endpoint_rank <= 8) OR "
                "(required_cuts <= 1 AND "
                "best_endpoint_rank <= 20)"
            ),

        "rank_cap_by_required_cuts": {
            "0":
                20,

            "1":
                20,

            "2":
                8,
        },
    },

    # --------------------------------------------------------
    # Development cardinality
    # --------------------------------------------------------

    "development_cardinality": {
        "datasets":
            103,

        "rows":
            SELECTED_ROWS,

        "fraction_of_parent_generator":
            float(
                SELECTED_ROWS
                /
                29_227_030
            ),

        "cuts_0":
            SELECTED_CUT0,

        "cuts_1":
            SELECTED_CUT1,

        "cuts_2":
            SELECTED_CUT2,
    },

    # --------------------------------------------------------
    # Development oracle evidence
    #
    # Not inference features.
    # --------------------------------------------------------

    "development_oracle_evidence": {
        "frozen_generator_true_pairs":
            212,

        "captured":
            202,

        "retention":
            float(
                202
                /
                212
            ),

        "frozen_generator_full7_pairs":
            178,

        "full7_captured":
            173,

        "full7_retention":
            float(
                173
                /
                178
            ),
    },

    # --------------------------------------------------------
    # Engineering knee evidence
    # --------------------------------------------------------

    "engineering_knee": {
        "g8_to_selected": {
            "extra_rows":
                1_145_589,

            "extra_true":
                19,

            "extra_full7":
                18,

            "million_rows_per_extra_true":
                float(
                    1_145_589
                    /
                    1_000_000
                    /
                    19
                ),
        },

        "lowcut14_to_selected": {
            "extra_rows":
                increment_14_to_20_rows,

            "extra_true":
                increment_14_to_20_true,

            "extra_full7":
                increment_14_to_20_full7,
        },

        "selected_to_twocut10": {
            "extra_rows":
                increment_20_to_twocut10_rows,

            "extra_true":
                increment_20_to_twocut10_true,

            "extra_full7":
                increment_20_to_twocut10_full7,
        },
    },

    # --------------------------------------------------------
    # Exact provenance
    # --------------------------------------------------------

    "parents": {
        "g3g1_cardinality": {
            "path":
                str(
                    G3G1_CARDINALITY_PATH
                ),

            "file_sha256":
                file_sha256(
                    G3G1_CARDINALITY_PATH
                ),

            "internal_sha256":
                g3g1[
                    "sha256"
                ],
        },

        "g3g1b_cardinality": {
            "path":
                str(
                    G3G1B_CARDINALITY_PATH
                ),

            "file_sha256":
                file_sha256(
                    G3G1B_CARDINALITY_PATH
                ),

            "internal_sha256":
                g3g1b_part_a[
                    "sha256"
                ],
        },

        "g3g1b_summary": {
            "path":
                str(
                    G3G1B_SUMMARY_PATH
                ),

            "file_sha256":
                file_sha256(
                    G3G1B_SUMMARY_PATH
                ),
        },
    },

    # --------------------------------------------------------
    # Protocol state
    # --------------------------------------------------------

    "shortlist_selected_with_development_gt":
        True,

    "runtime_shortlist_features_gt_blind":
        True,

    "future_shortlist_retuning_allowed":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "scoring_policy_selected":
        False,

    "rewire_policy_selected":
        False,

    "synthetic_nodes_inserted":
        False,

    "graph_modified":
        False,
}


# ============================================================
# 10. Canonical shortlist hash
# ============================================================

shortlist_sha = canonical_sha256(
    frozen_shortlist
)


frozen_shortlist[
    "shortlist_contract_sha256"
] = shortlist_sha


FROZEN_OUT.write_text(
    json.dumps(
        frozen_shortlist,
        indent=2,
    ),
    encoding="utf-8",
)


assert FROZEN_OUT.exists()


print(
    "\n========== FROZEN SHORTLIST =========="
)

print(
    "shortlist ID:",
    frozen_shortlist[
        "shortlist_id"
    ]
)

print(
    "rows:",
    SELECTED_ROWS
)

print(
    "parent-generator fraction:",
    SELECTED_ROWS
    /
    29_227_030
)

print(
    "shortlist SHA256:",
    shortlist_sha
)

print(
    "FROZEN BEFORE CROSS-FOLD SCORING: YES"
)


# ============================================================
# 11. Post-freeze development confirmation
#
# This is confirmation only.
# No retuning.
# ============================================================

oracle = pd.read_pickle(
    G3G1B_ORACLE_PATH
)


assert len(
    oracle
) == 369


SELECTED_COL = (
    "rescue_G8_LOWCUT20"
)


assert SELECTED_COL in oracle.columns


assert int(
    oracle[
        "combined_ASYM_3x8"
    ].sum()
) == 212


assert int(
    oracle[
        SELECTED_COL
    ].sum()
) == 202


full7 = oracle[
    "pred_full_7um"
].astype(bool)


selected_full7 = int(
    (
        oracle[
            SELECTED_COL
        ].astype(bool)
        &
        full7
    ).sum()
)


assert selected_full7 == 173


# ============================================================
# 12. Fold confirmation
# ============================================================

fold_rows = []


EXPECTED_FOLD = {
    1: {
        "true":
            35,

        "full7":
            31,
    },

    2: {
        "true":
            56,

        "full7":
            50,
    },

    3: {
        "true":
            64,

        "full7":
            49,
    },

    4: {
        "true":
            47,

        "full7":
            43,
    },
}


for fold in [
    1,
    2,
    3,
    4,
]:

    sub = oracle[
        oracle[
            "fold"
        ].eq(
            fold
        )
    ]


    frozen_true = int(
        sub[
            "combined_ASYM_3x8"
        ].sum()
    )


    selected_true = int(
        sub[
            SELECTED_COL
        ].sum()
    )


    frozen_full7 = int(
        (
            sub[
                "combined_ASYM_3x8"
            ].astype(bool)
            &
            sub[
                "pred_full_7um"
            ].astype(bool)
        ).sum()
    )


    selected_full7_fold = int(
        (
            sub[
                SELECTED_COL
            ].astype(bool)
            &
            sub[
                "pred_full_7um"
            ].astype(bool)
        ).sum()
    )


    assert selected_true == EXPECTED_FOLD[
        fold
    ][
        "true"
    ]


    assert selected_full7_fold == EXPECTED_FOLD[
        fold
    ][
        "full7"
    ]


    fold_rows.append(
        {
            "fold":
                fold,

            "frozen_true":
                frozen_true,

            "selected_true":
                selected_true,

            "true_retention":
                float(
                    selected_true
                    /
                    frozen_true
                ),

            "frozen_full7":
                frozen_full7,

            "selected_full7":
                selected_full7_fold,

            "full7_retention":
                float(
                    selected_full7_fold
                    /
                    frozen_full7
                ),
        }
    )


fold_table = pd.DataFrame(
    fold_rows
)


print(
    "\n========== POST-FREEZE DEVELOPMENT CONFIRMATION =========="
)

print(
    "selected true:",
    202,
    "/ 212"
)

print(
    "selected full<=7:",
    173,
    "/ 178"
)

print(
    "\nBy fold:"
)

print(
    fold_table.to_string(
        index=False
    )
)


# ============================================================
# 13. Confirmation artifact
# ============================================================

confirmation = {
    "stage":
        "12.11G3G2",

    "status":
        "FROZEN_SHORTLIST_DEVELOPMENT_CONFIRMATION",

    "shortlist_id":
        frozen_shortlist[
            "shortlist_id"
        ],

    "shortlist_contract_sha256":
        shortlist_sha,

    "candidate_rows":
        SELECTED_ROWS,

    "captured_true":
        202,

    "frozen_true":
        212,

    "captured_full7":
        173,

    "frozen_full7":
        178,

    "fold_results":
        fold_table.to_dict(
            "records"
        ),

    "confirmation_is_not_retuning":
        True,

    "shortlist_changed_after_freeze":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


CONFIRMATION_OUT.write_text(
    json.dumps(
        confirmation,
        indent=2,
    ),
    encoding="utf-8",
)


assert CONFIRMATION_OUT.exists()


# ============================================================
# 14. Reload integrity
# ============================================================

frozen_check = json.loads(
    FROZEN_OUT.read_text(
        encoding="utf-8"
    )
)


confirmation_check = json.loads(
    CONFIRMATION_OUT.read_text(
        encoding="utf-8"
    )
)


assert frozen_check[
    "shortlist_contract_sha256"
] == shortlist_sha


assert frozen_check[
    "shortlist_id"
] == (
    "ENDPOINT_SHORTLIST_GMEAN_G8_LOWCUT20_V1"
)


assert frozen_check[
    "development_cardinality"
][
    "rows"
] == 17_501_674


assert frozen_check[
    "pass_rule"
][
    "rank_cap_by_required_cuts"
] == {
    "0":
        20,

    "1":
        20,

    "2":
        8,
}


assert confirmation_check[
    "captured_true"
] == 202


assert confirmation_check[
    "captured_full7"
] == 173


# ============================================================
# 15. Decision
# ============================================================

print(
    "\n========== 12.11G3G2 DECISION =========="
)

print(
    "ENDPOINT_SHORTLIST_GMEAN_G8_LOWCUT20_V1: FROZEN"
)

print(
    "shortlist SHA256:",
    shortlist_sha
)

print(
    "selection development-GT-informed:",
    "YES"
)

print(
    "runtime shortlist features GT-blind:",
    "YES"
)

print(
    "candidate rows:",
    17_501_674
)

print(
    "frozen-generator true retention:",
    "202 / 212"
)

print(
    "frozen-generator full7 retention:",
    "173 / 178"
)

print(
    "future shortlist retuning allowed:",
    "NO"
)

print(
    "full shortlist universe materialized:",
    "NO"
)

print(
    "deterministic streaming regeneration required:",
    "YES"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "classifier trained:",
    "NO"
)

print(
    "rewire policy selected:",
    "NO"
)

print(
    "graph modified:",
    "NO"
)

print(
    "next:"
)

print(
    "BUILD_CROSS_FOLD_SCORING_FEATURE_AUDIT_ON_FROZEN_SHORTLIST"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3H0
#
# Cross-Fold Scoring Feature Audit
#
# Authoritative parent generator:
#
#   MULTIFRAME_REWIRE_ASYM3X8_DOMINANCE_ANY_V1_FIDELITYPATCH1
#
# Authoritative frozen scorer-input shortlist:
#
#   ENDPOINT_SHORTLIST_GMEAN_G8_LOWCUT20_V1
#
#   rows = 17,501,674
#
# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Do NOT train a classifier yet.
#
# First freeze a deployable GT-blind feature schema and test:
#
#   - whether true multi-frame repairs separate from shortlist
#     background candidates
#   - whether that separation is stable across folds 1–4
#   - which features are redundant / unstable
#
# ------------------------------------------------------------
# PART A — GT BLIND
# ------------------------------------------------------------
#
# Regenerate exact frozen shortlist.
#
# Build scoring features using ONLY:
#
#   candidate geometry
#   endpoint competition
#   CURRENT_BEST graph topology
#   incumbent-relative CV evidence
#
# No GT.
#
# Persist:
#
#   - frozen feature schema
#   - deterministic 1/64 candidate sample
#
# Sampling is candidate-ID hash based and GT blind.
#
# Exact shortlist cardinality must reproduce:
#
#       17,501,674
#
# ------------------------------------------------------------
# PART B — DEVELOPMENT ORACLE
# ------------------------------------------------------------
#
# Only after Part A freeze:
#
#   - read development oracle
#   - regenerate all 202 selected true-pair feature rows
#   - remove accidental positives from candidate sample
#   - evaluate univariate ROC AUC overall and by fold
#
# IMPORTANT:
#
#   AUC direction is selected globally for diagnostic display
#   only. Per-fold stability uses that SAME global direction.
#
# No classifier.
# No feature subset selected.
# No scoring threshold.
# No rewire actions.
# No Fold0.
#
# Keep after running: YES
# ============================================================

from pathlib import Path

import gc
import hashlib
import json

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score


# ============================================================
# 0. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


FROZEN_SHORTLIST_PATH = (
    S12
    / "stage12_folds14_endpoint_shortlist_g8_lowcut20_v1_frozen.json"
)


G3G1B_CARDINALITY_PATH = (
    S12
    / "stage12_folds14_endpoint_competition_targeted_rescue_cardinality_v1.json"
)


# GT artifact.
# DO NOT READ until Part A has been frozen.
G3G1B_ORACLE_PATH = (
    S12
    / "stage12_folds14_endpoint_competition_targeted_rescue_oracle_v1.pkl"
)


FEATURE_SCHEMA_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_feature_schema_v1_frozen.json"
)


SAMPLE_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_feature_sample_hash64_v1.pkl"
)


PARTA_SUMMARY_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_feature_audit_parta_v1.json"
)


POSITIVE_FEATURES_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_positive_features_v1.pkl"
)


AUDIT_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_feature_audit_v1.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_feature_audit_v1_summary.json"
)


for path in [
    FROZEN_SHORTLIST_PATH,
    G3G1B_CARDINALITY_PATH,
    G3G1B_ORACLE_PATH,
]:

    assert path.exists(), path


for path in [
    FEATURE_SCHEMA_OUT,
    SAMPLE_OUT,
    PARTA_SUMMARY_OUT,
    POSITIVE_FEATURES_OUT,
    AUDIT_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3H0 output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite a completed feature audit."
        )


# ============================================================
# 1. Required retained helpers
# ============================================================

for helper_name in [
    "generate_dataset_proposals",
    "add_endpoint_ranks",
]:

    assert (
        helper_name in globals()
        and
        callable(
            globals()[
                helper_name
            ]
        )
    ), (
        f"{helper_name} missing. "
        "Re-run the retained G3G1 helper-definition section, "
        "but do not rerun prior formal audits."
    )


assert (
    "build_current_best"
    in globals()
    and
    callable(
        globals()[
            "build_current_best"
        ]
    )
)


current_best_builder = globals()[
    "build_current_best"
]


# ============================================================
# 2. Frozen shortlist contract
# ============================================================

SHORTLIST_SHA = (
    "ae61216904bc3187f76b238b4b560d6693a7eb4f5d07bb8ef19021cc1bca1490"
)


GENERATOR_SHA = (
    "da408df857cd34dcb4b1f65f3fae545e"
    "299f9403ef5edb986584caa8f4c3c203"
)


frozen_shortlist = json.loads(
    FROZEN_SHORTLIST_PATH.read_text(
        encoding="utf-8"
    )
)


assert frozen_shortlist[
    "shortlist_contract_sha256"
] == SHORTLIST_SHA


assert frozen_shortlist[
    "shortlist_id"
] == (
    "ENDPOINT_SHORTLIST_GMEAN_G8_LOWCUT20_V1"
)


assert frozen_shortlist[
    "parent_generator"
][
    "generator_contract_sha256"
] == GENERATOR_SHA


EXPECTED_SHORTLIST_ROWS = int(
    frozen_shortlist[
        "development_cardinality"
    ][
        "rows"
    ]
)


assert EXPECTED_SHORTLIST_ROWS == 17_501_674


print(
    "========== 12.11G3H0 FROZEN INPUT =========="
)

print(
    "shortlist:",
    frozen_shortlist[
        "shortlist_id"
    ]
)

print(
    "shortlist SHA:",
    SHORTLIST_SHA
)

print(
    "parent generator SHA:",
    GENERATOR_SHA
)

print(
    "expected shortlist rows:",
    EXPECTED_SHORTLIST_ROWS
)


# ============================================================
# 3. GT-blind G3G1B per-dataset fidelity source
# ============================================================

g3g1b_part_a = json.loads(
    G3G1B_CARDINALITY_PATH.read_text(
        encoding="utf-8"
    )
)


assert g3g1b_part_a[
    "stage"
] == "12.11G3G1B"


assert g3g1b_part_a[
    "gt_used_during_part_a"
] is False


expected_dataset_shortlist = {
    row[
        "dataset"
    ]:
        int(
            row[
                "contracts"
            ][
                "G8_LOWCUT20"
            ]
        )

    for row
    in g3g1b_part_a[
        "dataset_results"
    ]
}


assert len(
    expected_dataset_shortlist
) == 103


assert sum(
    expected_dataset_shortlist.values()
) == EXPECTED_SHORTLIST_ROWS


# ============================================================
# 4. Development corpus
# ============================================================

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


cv = pd.read_csv(
    CV_PATH
)


dev_manifest = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
)


development_names = dev_manifest[
    "dataset"
].tolist()


assert len(
    development_names
) == 103


fold_map_local = dict(
    zip(
        dev_manifest[
            "dataset"
        ],
        dev_manifest[
            "fold"
        ].astype(int),
    )
)


# ============================================================
# 5. Frozen GT-blind feature schema
#
# Defined BEFORE any development GT read.
# ============================================================

FEATURE_COLUMNS = [
    # --------------------------------------------------------
    # temporal / topology
    # --------------------------------------------------------

    "gap_size",
    "delta_t",

    "source_occupied",
    "target_occupied",
    "cuts",

    # --------------------------------------------------------
    # candidate motion consistency
    # --------------------------------------------------------

    "source_cv_pf",
    "target_cv_pf",

    "MIN_CV",
    "SYM_CV",
    "MAX_CV",

    # --------------------------------------------------------
    # original G3G1 local competition
    # --------------------------------------------------------

    "GMEAN_RATIO",

    # --------------------------------------------------------
    # GMEAN endpoint competition ranks
    # --------------------------------------------------------

    "GM_SOURCE_RANK",
    "GM_TARGET_RANK",
    "GM_BEST_RANK",
    "GM_MAX_RANK",
    "GM_RANK_SUM",
    "GM_RANK_LOG_ASYMMETRY",

    # --------------------------------------------------------
    # endpoint competition sizes
    # --------------------------------------------------------

    "SOURCE_COMPETITION_COUNT",
    "TARGET_COMPETITION_COUNT",

    "GM_SOURCE_RANK_FRAC",
    "GM_TARGET_RANK_FRAC",
    "GM_BEST_RANK_FRAC",

    # --------------------------------------------------------
    # SYM/MAX endpoint competition
    # --------------------------------------------------------

    "SYM_BEST_RANK",
    "MAX_BEST_RANK",

    # --------------------------------------------------------
    # incumbent-relative evidence
    # --------------------------------------------------------

    "source_incumbent_cv_pf",
    "target_incumbent_cv_pf",

    "source_relative_improvement",
    "target_relative_improvement",

    "max_required_improvement",
    "min_required_improvement",
    "mean_required_improvement",

    "dominance_all",
]


FEATURE_SCHEMA = {
    "stage":
        "12.11G3H0",

    "schema_id":
        "MULTIFRAME_REWIRE_SCORING_FEATURES_V1",

    "parent_shortlist_id":
        frozen_shortlist[
            "shortlist_id"
        ],

    "parent_shortlist_sha256":
        SHORTLIST_SHA,

    "parent_generator_sha256":
        GENERATOR_SHA,

    "features":
        FEATURE_COLUMNS,

    "forbidden_features": [
        "fold",
        "dataset identity",
        "GT labels",
        "official FP",
        "oracle component geometry",
        "pred_full_3um",
        "pred_full_5um",
        "pred_full_7um",
    ],

    "runtime_features_gt_blind":
        True,

    "feature_schema_defined_before_gt_read":
        True,

    "feature_selection_performed":
        False,

    "classifier_trained":
        False,

    "fold0_used":
        False,
}


schema_bytes = json.dumps(
    FEATURE_SCHEMA,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
).encode(
    "utf-8"
)


FEATURE_SCHEMA_SHA = hashlib.sha256(
    schema_bytes
).hexdigest()


FEATURE_SCHEMA[
    "feature_schema_sha256"
] = FEATURE_SCHEMA_SHA


FEATURE_SCHEMA_OUT.write_text(
    json.dumps(
        FEATURE_SCHEMA,
        indent=2,
    ),
    encoding="utf-8",
)


assert FEATURE_SCHEMA_OUT.exists()


print(
    "\n========== FROZEN FEATURE SCHEMA =========="
)

print(
    "schema ID:",
    FEATURE_SCHEMA[
        "schema_id"
    ]
)

print(
    "feature count:",
    len(
        FEATURE_COLUMNS
    )
)

print(
    "schema SHA:",
    FEATURE_SCHEMA_SHA
)

print(
    "FROZEN BEFORE GT READ: YES"
)


# ============================================================
# 6. Frozen shortlist helper
# ============================================================

def add_shortlist_rank_fields(
    proposals,
):

    proposals = proposals.copy()


    proposals[
        "GM_SOURCE_RANK"
    ] = proposals[
        "GMEAN_RATIO_SOURCE_RANK"
    ].astype(
        np.int32
    )


    proposals[
        "GM_TARGET_RANK"
    ] = proposals[
        "GMEAN_RATIO_TARGET_RANK"
    ].astype(
        np.int32
    )


    proposals[
        "GM_BEST_RANK"
    ] = np.minimum(
        proposals[
            "GM_SOURCE_RANK"
        ].to_numpy(),

        proposals[
            "GM_TARGET_RANK"
        ].to_numpy(),
    ).astype(
        np.int32
    )


    proposals[
        "GM_MAX_RANK"
    ] = np.maximum(
        proposals[
            "GM_SOURCE_RANK"
        ].to_numpy(),

        proposals[
            "GM_TARGET_RANK"
        ].to_numpy(),
    ).astype(
        np.int32
    )


    proposals[
        "SYM_BEST_RANK"
    ] = np.minimum(
        proposals[
            "SYM_CV_SOURCE_RANK"
        ].to_numpy(),

        proposals[
            "SYM_CV_TARGET_RANK"
        ].to_numpy(),
    ).astype(
        np.int32
    )


    proposals[
        "MAX_BEST_RANK"
    ] = np.minimum(
        proposals[
            "MAX_CV_SOURCE_RANK"
        ].to_numpy(),

        proposals[
            "MAX_CV_TARGET_RANK"
        ].to_numpy(),
    ).astype(
        np.int32
    )


    return proposals


def frozen_shortlist_mask(
    proposals,
):

    return (
        proposals[
            "GM_BEST_RANK"
        ].le(
            8
        )
        |
        (
            proposals[
                "cuts"
            ].le(
                1
            )
            &
            proposals[
                "GM_BEST_RANK"
            ].le(
                20
            )
        )
    )


# ============================================================
# 7. Current graph scoring-feature attachment
# ============================================================

SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


def attach_scoring_features(
    dataset,
    proposals,
):

    p = proposals.copy()


    if len(
        p
    ) == 0:

        return p


    graph, scale = current_best_builder(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    )


    # --------------------------------------------------------
    # Nodes
    # --------------------------------------------------------

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = nodes[
        "node_id"
    ].astype(
        int
    )


    nodes[
        "t"
    ] = nodes[
        "t"
    ].astype(
        int
    )


    # --------------------------------------------------------
    # Edges
    # --------------------------------------------------------

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = edges[
        "source_id"
    ].astype(
        int
    )


    edges[
        "target_id"
    ] = edges[
        "target_id"
    ].astype(
        int
    )


    max_node_id = int(
        max(
            nodes[
                "node_id"
            ].max(),

            p[
                "source_id"
            ].max(),

            p[
                "target_id"
            ].max(),
        )
    )


    size = (
        max_node_id
        +
        1
    )


    # --------------------------------------------------------
    # Dense node lookup arrays
    # --------------------------------------------------------

    position = np.full(
        (
            size,
            3,
        ),
        np.nan,
        dtype=np.float64,
    )


    time_arr = np.full(
        size,
        -1,
        dtype=np.int32,
    )


    node_ids = nodes[
        "node_id"
    ].to_numpy(
        dtype=np.int64
    )


    position[
        node_ids
    ] = (
        nodes[
            [
                "z",
                "y",
                "x",
            ]
        ]
        .to_numpy(
            dtype=np.float64
        )
        *
        SCALE
    )


    time_arr[
        node_ids
    ] = nodes[
        "t"
    ].to_numpy(
        dtype=np.int32
    )


    # --------------------------------------------------------
    # Degree / predecessor / successor arrays
    # --------------------------------------------------------

    source_edge_ids = edges[
        "source_id"
    ].to_numpy(
        dtype=np.int64
    )


    target_edge_ids = edges[
        "target_id"
    ].to_numpy(
        dtype=np.int64
    )


    outdegree = np.bincount(
        source_edge_ids,
        minlength=size,
    ).astype(
        np.int32
    )


    indegree = np.bincount(
        target_edge_ids,
        minlength=size,
    ).astype(
        np.int32
    )


    predecessor = np.full(
        size,
        -1,
        dtype=np.int64,
    )


    successor = np.full(
        size,
        -1,
        dtype=np.int64,
    )


    # Multiple assignments are harmless because we only use
    # predecessor/successor when exact degree == 1.
    predecessor[
        target_edge_ids
    ] = source_edge_ids


    successor[
        source_edge_ids
    ] = target_edge_ids


    # --------------------------------------------------------
    # Local velocity arrays
    # --------------------------------------------------------

    past_velocity = np.full(
        (
            size,
            3,
        ),
        np.nan,
        dtype=np.float64,
    )


    future_velocity = np.full(
        (
            size,
            3,
        ),
        np.nan,
        dtype=np.float64,
    )


    unique_in = np.flatnonzero(
        indegree == 1
    )


    valid_pred = predecessor[
        unique_in
    ] >= 0


    unique_in = unique_in[
        valid_pred
    ]


    pred_ids = predecessor[
        unique_in
    ]


    dt_past = (
        time_arr[
            unique_in
        ]
        -
        time_arr[
            pred_ids
        ]
    )


    valid_dt = dt_past > 0


    unique_in = unique_in[
        valid_dt
    ]


    pred_ids = predecessor[
        unique_in
    ]


    dt_past = (
        time_arr[
            unique_in
        ]
        -
        time_arr[
            pred_ids
        ]
    ).astype(
        np.float64
    )


    past_velocity[
        unique_in
    ] = (
        position[
            unique_in
        ]
        -
        position[
            pred_ids
        ]
    ) / dt_past[
        :,
        None
    ]


    unique_out = np.flatnonzero(
        outdegree == 1
    )


    valid_succ = successor[
        unique_out
    ] >= 0


    unique_out = unique_out[
        valid_succ
    ]


    succ_ids = successor[
        unique_out
    ]


    dt_future = (
        time_arr[
            succ_ids
        ]
        -
        time_arr[
            unique_out
        ]
    )


    valid_dt = dt_future > 0


    unique_out = unique_out[
        valid_dt
    ]


    succ_ids = successor[
        unique_out
    ]


    dt_future = (
        time_arr[
            succ_ids
        ]
        -
        time_arr[
            unique_out
        ]
    ).astype(
        np.float64
    )


    future_velocity[
        unique_out
    ] = (
        position[
            succ_ids
        ]
        -
        position[
            unique_out
        ]
    ) / dt_future[
        :,
        None
    ]


    # --------------------------------------------------------
    # Candidate arrays
    # --------------------------------------------------------

    src = p[
        "source_id"
    ].to_numpy(
        dtype=np.int64
    )


    tgt = p[
        "target_id"
    ].to_numpy(
        dtype=np.int64
    )


    source_t = time_arr[
        src
    ]


    target_t = time_arr[
        tgt
    ]


    delta_t = (
        target_t
        -
        source_t
    ).astype(
        np.float64
    )


    assert np.all(
        delta_t
        >
        0
    )


    p[
        "delta_t"
    ] = delta_t.astype(
        np.int16
    )


    # --------------------------------------------------------
    # Candidate forward / backward CV
    # --------------------------------------------------------

    source_predicted_target = (
        position[
            src
        ]
        +
        delta_t[
            :,
            None
        ]
        *
        past_velocity[
            src
        ]
    )


    target_predicted_source = (
        position[
            tgt
        ]
        -
        delta_t[
            :,
            None
        ]
        *
        future_velocity[
            tgt
        ]
    )


    source_cv_pf = (
        np.linalg.norm(
            position[
                tgt
            ]
            -
            source_predicted_target,
            axis=1,
        )
        /
        delta_t
    )


    target_cv_pf = (
        np.linalg.norm(
            position[
                src
            ]
            -
            target_predicted_source,
            axis=1,
        )
        /
        delta_t
    )


    p[
        "source_cv_pf"
    ] = source_cv_pf


    p[
        "target_cv_pf"
    ] = target_cv_pf


    p[
        "MIN_CV"
    ] = np.minimum(
        source_cv_pf,
        target_cv_pf,
    )


    # Exact fidelity against generator-returned features.
    assert np.allclose(
        (
            source_cv_pf
            +
            target_cv_pf
        )
        /
        2.0,

        p[
            "SYM_CV"
        ].to_numpy(
            dtype=float
        ),

        atol=1e-10,
        rtol=1e-10,
    )


    assert np.allclose(
        np.maximum(
            source_cv_pf,
            target_cv_pf,
        ),

        p[
            "MAX_CV"
        ].to_numpy(
            dtype=float
        ),

        atol=1e-10,
        rtol=1e-10,
    )


    # --------------------------------------------------------
    # Occupancy
    # --------------------------------------------------------

    source_occupied = (
        outdegree[
            src
        ]
        ==
        1
    )


    target_occupied = (
        indegree[
            tgt
        ]
        ==
        1
    )


    p[
        "source_occupied"
    ] = source_occupied.astype(
        np.int8
    )


    p[
        "target_occupied"
    ] = target_occupied.astype(
        np.int8
    )


    reconstructed_cuts = (
        source_occupied.astype(
            np.int8
        )
        +
        target_occupied.astype(
            np.int8
        )
    )


    assert np.array_equal(
        reconstructed_cuts,
        p[
            "cuts"
        ].to_numpy(
            dtype=np.int8
        ),
    )


    # --------------------------------------------------------
    # Incumbent normalized CV
    # --------------------------------------------------------

    source_incumbent_cv = np.full(
        len(
            p
        ),
        np.nan,
        dtype=np.float64,
    )


    target_incumbent_cv = np.full(
        len(
            p
        ),
        np.nan,
        dtype=np.float64,
    )


    source_occ_index = np.flatnonzero(
        source_occupied
    )


    if len(
        source_occ_index
    ):

        occ_src = src[
            source_occ_index
        ]


        old_tgt = successor[
            occ_src
        ]


        dt_old = (
            time_arr[
                old_tgt
            ]
            -
            time_arr[
                occ_src
            ]
        ).astype(
            np.float64
        )


        assert np.all(
            dt_old
            >
            0
        )


        old_pred = (
            position[
                occ_src
            ]
            +
            dt_old[
                :,
                None
            ]
            *
            past_velocity[
                occ_src
            ]
        )


        source_incumbent_cv[
            source_occ_index
        ] = (
            np.linalg.norm(
                position[
                    old_tgt
                ]
                -
                old_pred,
                axis=1,
            )
            /
            dt_old
        )


    target_occ_index = np.flatnonzero(
        target_occupied
    )


    if len(
        target_occ_index
    ):

        occ_tgt = tgt[
            target_occ_index
        ]


        old_src = predecessor[
            occ_tgt
        ]


        dt_old = (
            time_arr[
                occ_tgt
            ]
            -
            time_arr[
                old_src
            ]
        ).astype(
            np.float64
        )


        assert np.all(
            dt_old
            >
            0
        )


        old_pred = (
            position[
                occ_tgt
            ]
            -
            dt_old[
                :,
                None
            ]
            *
            future_velocity[
                occ_tgt
            ]
        )


        target_incumbent_cv[
            target_occ_index
        ] = (
            np.linalg.norm(
                position[
                    old_src
                ]
                -
                old_pred,
                axis=1,
            )
            /
            dt_old
        )


    p[
        "source_incumbent_cv_pf"
    ] = source_incumbent_cv


    p[
        "target_incumbent_cv_pf"
    ] = target_incumbent_cv


    source_improvement = (
        source_incumbent_cv
        -
        source_cv_pf
    )


    target_improvement = (
        target_incumbent_cv
        -
        target_cv_pf
    )


    p[
        "source_relative_improvement"
    ] = source_improvement


    p[
        "target_relative_improvement"
    ] = target_improvement


    # --------------------------------------------------------
    # Required-side improvement summaries
    # --------------------------------------------------------

    max_required = np.full(
        len(
            p
        ),
        np.nan,
        dtype=np.float64,
    )


    min_required = np.full(
        len(
            p
        ),
        np.nan,
        dtype=np.float64,
    )


    mean_required = np.full(
        len(
            p
        ),
        np.nan,
        dtype=np.float64,
    )


    dominance_all = np.ones(
        len(
            p
        ),
        dtype=np.int8,
    )


    # source-only occupied
    mask = (
        source_occupied
        &
        ~target_occupied
    )


    max_required[
        mask
    ] = source_improvement[
        mask
    ]


    min_required[
        mask
    ] = source_improvement[
        mask
    ]


    mean_required[
        mask
    ] = source_improvement[
        mask
    ]


    dominance_all[
        mask
    ] = (
        source_improvement[
            mask
        ]
        >
        0
    ).astype(
        np.int8
    )


    # target-only occupied
    mask = (
        ~source_occupied
        &
        target_occupied
    )


    max_required[
        mask
    ] = target_improvement[
        mask
    ]


    min_required[
        mask
    ] = target_improvement[
        mask
    ]


    mean_required[
        mask
    ] = target_improvement[
        mask
    ]


    dominance_all[
        mask
    ] = (
        target_improvement[
            mask
        ]
        >
        0
    ).astype(
        np.int8
    )


    # both occupied
    mask = (
        source_occupied
        &
        target_occupied
    )


    max_required[
        mask
    ] = np.maximum(
        source_improvement[
            mask
        ],
        target_improvement[
            mask
        ],
    )


    min_required[
        mask
    ] = np.minimum(
        source_improvement[
            mask
        ],
        target_improvement[
            mask
        ],
    )


    mean_required[
        mask
    ] = (
        source_improvement[
            mask
        ]
        +
        target_improvement[
            mask
        ]
    ) / 2.0


    dominance_all[
        mask
    ] = (
        (
            source_improvement[
                mask
            ]
            >
            0
        )
        &
        (
            target_improvement[
                mask
            ]
            >
            0
        )
    ).astype(
        np.int8
    )


    p[
        "max_required_improvement"
    ] = max_required


    p[
        "min_required_improvement"
    ] = min_required


    p[
        "mean_required_improvement"
    ] = mean_required


    p[
        "dominance_all"
    ] = dominance_all


    # --------------------------------------------------------
    # Endpoint competition counts
    # --------------------------------------------------------

    source_counts = (
        p.groupby(
            "source_id",
            sort=False,
        )[
            "source_id"
        ]
        .transform(
            "size"
        )
        .to_numpy(
            dtype=np.int32
        )
    )


    target_counts = (
        p.groupby(
            "target_id",
            sort=False,
        )[
            "target_id"
        ]
        .transform(
            "size"
        )
        .to_numpy(
            dtype=np.int32
        )
    )


    p[
        "SOURCE_COMPETITION_COUNT"
    ] = source_counts


    p[
        "TARGET_COMPETITION_COUNT"
    ] = target_counts


    # --------------------------------------------------------
    # GMEAN rank derived features
    # --------------------------------------------------------

    gm_src = p[
        "GM_SOURCE_RANK"
    ].to_numpy(
        dtype=np.float64
    )


    gm_tgt = p[
        "GM_TARGET_RANK"
    ].to_numpy(
        dtype=np.float64
    )


    p[
        "GM_RANK_SUM"
    ] = (
        gm_src
        +
        gm_tgt
    )


    p[
        "GM_RANK_LOG_ASYMMETRY"
    ] = np.abs(
        np.log1p(
            gm_src
        )
        -
        np.log1p(
            gm_tgt
        )
    )


    p[
        "GM_SOURCE_RANK_FRAC"
    ] = (
        gm_src
        /
        source_counts
    )


    p[
        "GM_TARGET_RANK_FRAC"
    ] = (
        gm_tgt
        /
        target_counts
    )


    p[
        "GM_BEST_RANK_FRAC"
    ] = np.minimum(
        p[
            "GM_SOURCE_RANK_FRAC"
        ].to_numpy(),

        p[
            "GM_TARGET_RANK_FRAC"
        ].to_numpy(),
    )


    del graph
    del nodes
    del edges

    del position
    del time_arr

    del outdegree
    del indegree

    del predecessor
    del successor

    del past_velocity
    del future_velocity

    gc.collect()


    return p


# ============================================================
# 8. Stable deterministic 1/64 sampling
# ============================================================

SAMPLE_MODULUS = 64
SAMPLE_BUCKET = 0


MASK64 = np.uint64(
    0xFFFFFFFFFFFFFFFF
)


def dataset_seed_uint64(
    dataset,
):

    digest = hashlib.sha256(
        str(
            dataset
        ).encode(
            "utf-8"
        )
    ).digest()


    return np.uint64(
        int.from_bytes(
            digest[
                :8
            ],
            byteorder="little",
            signed=False,
        )
    )


def splitmix64(
    x,
):

    x = (
        x
        +
        np.uint64(
            0x9E3779B97F4A7C15
        )
    )


    x = (
        (
            x
            ^
            (
                x
                >>
                np.uint64(
                    30
                )
            )
        )
        *
        np.uint64(
            0xBF58476D1CE4E5B9
        )
    )


    x = (
        (
            x
            ^
            (
                x
                >>
                np.uint64(
                    27
                )
            )
        )
        *
        np.uint64(
            0x94D049BB133111EB
        )
    )


    x = (
        x
        ^
        (
            x
            >>
            np.uint64(
                31
            )
        )
    )


    return x


def deterministic_sample_mask(
    dataset,
    frame,
):

    src = frame[
        "source_id"
    ].to_numpy(
        dtype=np.uint64
    )


    tgt = frame[
        "target_id"
    ].to_numpy(
        dtype=np.uint64
    )


    gap = frame[
        "gap_size"
    ].to_numpy(
        dtype=np.uint64
    )


    seed = dataset_seed_uint64(
        dataset
    )


    x = (
        seed
        ^
        (
            src
            *
            np.uint64(
                0xD6E8FEB86659FD93
            )
        )
        ^
        (
            tgt
            *
            np.uint64(
                0xA5A3564E27F8862B
            )
        )
        ^
        (
            gap
            *
            np.uint64(
                0x9E3779B185EBCA87
            )
        )
    )


    h = splitmix64(
        x
    )


    return (
        (
            h
            %
            np.uint64(
                SAMPLE_MODULUS
            )
        )
        ==
        np.uint64(
            SAMPLE_BUCKET
        )
    )


# ============================================================
# PART A
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART A — GT-BLIND SCORING FEATURE SAMPLE"
)

print(
    "============================================================"
)


sample_frames = []

dataset_results = []


shortlist_total = 0


for ds_i, dataset in enumerate(
    development_names,
    start=1,
):

    fold = int(
        fold_map_local[
            dataset
        ]
    )


    (
        proposals,
        _,
        _,
    ) = generate_dataset_proposals(
        dataset
    )


    proposals = add_endpoint_ranks(
        proposals
    )


    proposals = add_shortlist_rank_fields(
        proposals
    )


    shortlist_mask = frozen_shortlist_mask(
        proposals
    )


    shortlist = (
        proposals[
            shortlist_mask
        ]
        .copy()
    )


    expected_count = expected_dataset_shortlist[
        dataset
    ]


    assert len(
        shortlist
    ) == expected_count, (
        dataset,
        len(
            shortlist
        ),
        expected_count,
    )


    shortlist_total += len(
        shortlist
    )


    shortlist = attach_scoring_features(
        dataset,
        shortlist,
    )


    # All frozen scoring features must exist.
    for col in FEATURE_COLUMNS:

        assert col in shortlist.columns, (
            dataset,
            col,
        )


    sample_mask = deterministic_sample_mask(
        dataset,
        shortlist,
    )


    sample = shortlist.loc[
        sample_mask,
        [
            "source_id",
            "target_id",
            "gap_size",
        ]
        +
        FEATURE_COLUMNS
    ].copy()


    sample.insert(
        0,
        "dataset",
        dataset,
    )


    sample.insert(
        1,
        "fold",
        fold,
    )


    sample_frames.append(
        sample
    )


    dataset_results.append(
        {
            "dataset":
                dataset,

            "fold":
                fold,

            "shortlist_rows":
                int(
                    len(
                        shortlist
                    )
                ),

            "sample_rows":
                int(
                    len(
                        sample
                    )
                ),
        }
    )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            development_names
        )
    ):

        print(
            f"{ds_i:3d}/{len(development_names)}"
            f" | fold={fold}"
            f" | {dataset}"
            f" | shortlist={len(shortlist)}"
            f" | sample={len(sample)}"
        )


    del proposals
    del shortlist
    del sample

    gc.collect()


assert shortlist_total == EXPECTED_SHORTLIST_ROWS


sample_table = pd.concat(
    sample_frames,
    ignore_index=True,
)


del sample_frames

gc.collect()


assert not sample_table[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


sample_table.to_pickle(
    SAMPLE_OUT
)


assert SAMPLE_OUT.exists()


print(
    "\n========== PART-A SHORTLIST FIDELITY =========="
)

print(
    "shortlist reproduced:",
    shortlist_total,
    "/",
    EXPECTED_SHORTLIST_ROWS
)

print(
    "deterministic sample rows:",
    len(
        sample_table
    )
)

print(
    "sample fraction:",
    len(
        sample_table
    )
    /
    EXPECTED_SHORTLIST_ROWS
)

print(
    "SHORTLIST_REPRODUCTION: PASS"
)


# ============================================================
# 9. Freeze Part A before GT read
# ============================================================

part_a = {
    "stage":
        "12.11G3H0",

    "part":
        "GT_BLIND_SCORING_FEATURE_SAMPLE",

    "feature_schema_id":
        FEATURE_SCHEMA[
            "schema_id"
        ],

    "feature_schema_sha256":
        FEATURE_SCHEMA_SHA,

    "parent_shortlist_id":
        frozen_shortlist[
            "shortlist_id"
        ],

    "parent_shortlist_sha256":
        SHORTLIST_SHA,

    "shortlist_rows":
        int(
            shortlist_total
        ),

    "sampling": {
        "type":
            "DETERMINISTIC_HASH",

        "modulus":
            SAMPLE_MODULUS,

        "bucket":
            SAMPLE_BUCKET,

        "gt_blind":
            True,
    },

    "sample_rows":
        int(
            len(
                sample_table
            )
        ),

    "dataset_results":
        dataset_results,

    "gt_used":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "feature_subset_selected":
        False,

    "scoring_threshold_selected":
        False,

    "graph_modified":
        False,
}


part_a_bytes = json.dumps(
    part_a,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
).encode(
    "utf-8"
)


PART_A_SHA = hashlib.sha256(
    part_a_bytes
).hexdigest()


part_a[
    "sha256"
] = PART_A_SHA


PARTA_SUMMARY_OUT.write_text(
    json.dumps(
        part_a,
        indent=2,
    ),
    encoding="utf-8",
)


assert PARTA_SUMMARY_OUT.exists()


print(
    "\nGT-BLIND FEATURE SAMPLE FROZEN BEFORE GT READ: YES"
)

print(
    "Part-A SHA256:",
    PART_A_SHA
)


# ============================================================
# PART B
#
# GT read starts only below this line.
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — DEVELOPMENT ORACLE FEATURE AUDIT"
)

print(
    "============================================================"
)


oracle = pd.read_pickle(
    G3G1B_ORACLE_PATH
)


assert len(
    oracle
) == 369


SELECTED_TRUE_COL = (
    "rescue_G8_LOWCUT20"
)


assert SELECTED_TRUE_COL in oracle.columns


positive_oracle = (
    oracle[
        oracle[
            SELECTED_TRUE_COL
        ].astype(bool)
    ][
        [
            "dataset",
            "fold",
            "component_id",
            "source_id",
            "target_id",
            "pred_full_7um",
            "topology_class",
        ]
    ]
    .copy()
)


assert len(
    positive_oracle
) == 202


assert int(
    positive_oracle[
        "pred_full_7um"
    ].sum()
) == 173


# ============================================================
# 10. Regenerate exact features for all 202 positives
# ============================================================

positive_feature_frames = []


positive_datasets = (
    positive_oracle[
        "dataset"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)


for ds_i, dataset in enumerate(
    positive_datasets,
    start=1,
):

    (
        proposals,
        _,
        _,
    ) = generate_dataset_proposals(
        dataset
    )


    proposals = add_endpoint_ranks(
        proposals
    )


    proposals = add_shortlist_rank_fields(
        proposals
    )


    shortlist_mask = frozen_shortlist_mask(
        proposals
    )


    shortlist = (
        proposals[
            shortlist_mask
        ]
        .copy()
    )


    shortlist = attach_scoring_features(
        dataset,
        shortlist,
    )


    ds_pos = positive_oracle[
        positive_oracle[
            "dataset"
        ].eq(
            dataset
        )
    ]


    compact = shortlist[
        [
            "source_id",
            "target_id",
            "gap_size",
        ]
        +
        FEATURE_COLUMNS
    ]


    matched = ds_pos.merge(
        compact,
        on=[
            "source_id",
            "target_id",
        ],
        how="left",
        validate="one_to_one",
    )


    assert len(
        matched
    ) == len(
        ds_pos
    )


    for col in [
        "gap_size",
        "GMEAN_RATIO",
        "GM_BEST_RANK",
    ]:

        assert matched[
            col
        ].notna().all(), (
            dataset,
            col,
        )


    positive_feature_frames.append(
        matched
    )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            positive_datasets
        )
    ):

        print(
            f"{ds_i:2d}/{len(positive_datasets)}"
            f" | {dataset}"
            f" | positives={len(ds_pos)}"
        )


    del proposals
    del shortlist
    del compact
    del matched

    gc.collect()


positive_features = pd.concat(
    positive_feature_frames,
    ignore_index=True,
)


del positive_feature_frames

gc.collect()


assert len(
    positive_features
) == 202


assert not positive_features[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


positive_features.to_pickle(
    POSITIVE_FEATURES_OUT
)


assert POSITIVE_FEATURES_OUT.exists()


print(
    "\nPOSITIVE_FEATURE_RECONSTRUCTION: PASS"
)

print(
    "positive feature rows:",
    len(
        positive_features
    )
)


# ============================================================
# 11. Remove accidental positives from hash sample
# ============================================================

positive_keys = positive_features[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].copy()


positive_keys[
    "__is_positive"
] = True


sample_labeled = sample_table.merge(
    positive_keys,
    on=[
        "dataset",
        "source_id",
        "target_id",
    ],
    how="left",
    validate="one_to_one",
)


sample_negatives = (
    sample_labeled[
        sample_labeled[
            "__is_positive"
        ].isna()
    ]
    .drop(
        columns=[
            "__is_positive",
        ]
    )
    .copy()
)


assert len(
    sample_negatives
) <= len(
    sample_table
)


print(
    "\n========== AUDIT POPULATION =========="
)

print(
    "sample before positive removal:",
    len(
        sample_table
    )
)

print(
    "sample negatives:",
    len(
        sample_negatives
    )
)

print(
    "all positives:",
    len(
        positive_features
    )
)


# ============================================================
# 12. Build audit population
# ============================================================

negative_audit = sample_negatives[
    [
        "dataset",
        "fold",
        "source_id",
        "target_id",
    ]
    +
    FEATURE_COLUMNS
].copy()


negative_audit[
    "label"
] = 0


positive_audit = positive_features[
    [
        "dataset",
        "fold",
        "source_id",
        "target_id",
    ]
    +
    FEATURE_COLUMNS
].copy()


positive_audit[
    "label"
] = 1


audit_population = pd.concat(
    [
        negative_audit,
        positive_audit,
    ],
    ignore_index=True,
)


assert int(
    audit_population[
        "label"
    ].sum()
) == 202


# ============================================================
# 13. Univariate AUC helpers
# ============================================================

def finite_feature_frame(
    frame,
    feature,
):

    values = pd.to_numeric(
        frame[
            feature
        ],
        errors="coerce",
    )


    finite = np.isfinite(
        values.to_numpy(
            dtype=float
        )
    )


    return (
        frame.loc[
            finite,
            [
                "fold",
                "label",
            ]
        ].copy(),

        values.loc[
            finite
        ].to_numpy(
            dtype=float
        ),
    )


def auc_or_none(
    labels,
    values,
):

    labels = np.asarray(
        labels,
        dtype=int,
    )


    values = np.asarray(
        values,
        dtype=float,
    )


    if len(
        np.unique(
            labels
        )
    ) < 2:

        return np.nan


    return float(
        roc_auc_score(
            labels,
            values,
        )
    )


# ============================================================
# 14. Cross-fold univariate feature audit
# ============================================================

feature_rows = []


for feature in FEATURE_COLUMNS:

    sub_labels, values = finite_feature_frame(
        audit_population,
        feature,
    )


    labels = sub_labels[
        "label"
    ].to_numpy(
        dtype=int
    )


    raw_auc = auc_or_none(
        labels,
        values,
    )


    if np.isnan(
        raw_auc
    ):

        direction = "UNDEFINED"
        oriented_auc = np.nan

    elif raw_auc >= 0.5:

        direction = "HIGHER_FOR_TRUE"
        oriented_auc = raw_auc

    else:

        direction = "LOWER_FOR_TRUE"
        oriented_auc = (
            1.0
            -
            raw_auc
        )


    # --------------------------------------------------------
    # Same globally chosen direction used for every fold.
    # --------------------------------------------------------

    fold_aucs = []


    for fold in [
        1,
        2,
        3,
        4,
    ]:

        fold_mask = (
            sub_labels[
                "fold"
            ].eq(
                fold
            )
            .to_numpy()
        )


        fold_labels = labels[
            fold_mask
        ]


        fold_values = values[
            fold_mask
        ]


        raw_fold_auc = auc_or_none(
            fold_labels,
            fold_values,
        )


        if np.isnan(
            raw_fold_auc
        ):

            oriented_fold_auc = np.nan

        elif direction == "HIGHER_FOR_TRUE":

            oriented_fold_auc = raw_fold_auc

        elif direction == "LOWER_FOR_TRUE":

            oriented_fold_auc = (
                1.0
                -
                raw_fold_auc
            )

        else:

            oriented_fold_auc = np.nan


        fold_aucs.append(
            oriented_fold_auc
        )


    fold_auc_array = np.asarray(
        fold_aucs,
        dtype=float,
    )


    pos_values = audit_population.loc[
        (
            audit_population[
                "label"
            ].eq(
                1
            )
            &
            np.isfinite(
                pd.to_numeric(
                    audit_population[
                        feature
                    ],
                    errors="coerce",
                )
            )
        ),
        feature,
    ].astype(float)


    neg_values = audit_population.loc[
        (
            audit_population[
                "label"
            ].eq(
                0
            )
            &
            np.isfinite(
                pd.to_numeric(
                    audit_population[
                        feature
                    ],
                    errors="coerce",
                )
            )
        ),
        feature,
    ].astype(float)


    feature_rows.append(
        {
            "feature":
                feature,

            "direction":
                direction,

            "overall_auc":
                oriented_auc,

            "fold1_auc":
                fold_aucs[
                    0
                ],

            "fold2_auc":
                fold_aucs[
                    1
                ],

            "fold3_auc":
                fold_aucs[
                    2
                ],

            "fold4_auc":
                fold_aucs[
                    3
                ],

            "mean_fold_auc":
                float(
                    np.nanmean(
                        fold_auc_array
                    )
                ),

            "worst_fold_auc":
                float(
                    np.nanmin(
                        fold_auc_array
                    )
                ),

            "fold_auc_std":
                float(
                    np.nanstd(
                        fold_auc_array
                    )
                ),

            "positive_finite_n":
                int(
                    len(
                        pos_values
                    )
                ),

            "negative_finite_n":
                int(
                    len(
                        neg_values
                    )
                ),

            "positive_median":
                (
                    float(
                        pos_values.median()
                    )
                    if len(
                        pos_values
                    )
                    else np.nan
                ),

            "negative_median":
                (
                    float(
                        neg_values.median()
                    )
                    if len(
                        neg_values
                    )
                    else np.nan
                ),

            "positive_q25":
                (
                    float(
                        pos_values.quantile(
                            0.25
                        )
                    )
                    if len(
                        pos_values
                    )
                    else np.nan
                ),

            "positive_q75":
                (
                    float(
                        pos_values.quantile(
                            0.75
                        )
                    )
                    if len(
                        pos_values
                    )
                    else np.nan
                ),

            "negative_q25":
                (
                    float(
                        neg_values.quantile(
                            0.25
                        )
                    )
                    if len(
                        neg_values
                    )
                    else np.nan
                ),

            "negative_q75":
                (
                    float(
                        neg_values.quantile(
                            0.75
                        )
                    )
                    if len(
                        neg_values
                    )
                    else np.nan
                ),
        }
    )


feature_audit = pd.DataFrame(
    feature_rows
)


feature_audit = (
    feature_audit
    .sort_values(
        [
            "worst_fold_auc",
            "overall_auc",
        ],
        ascending=[
            False,
            False,
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n========== CROSS-FOLD UNIVARIATE FEATURE AUDIT =========="
)


print(
    feature_audit.to_string(
        index=False
    )
)


# ============================================================
# 15. Strong/stable diagnostic subset
#
# DIAGNOSTIC ONLY.
# Not frozen as model feature selection.
# ============================================================

stable_signal = feature_audit[
    (
        feature_audit[
            "overall_auc"
        ]
        >=
        0.60
    )
    &
    (
        feature_audit[
            "worst_fold_auc"
        ]
        >=
        0.55
    )
]


print(
    "\n========== STABLE UNIVARIATE SIGNALS (DIAGNOSTIC ONLY) =========="
)


if len(
    stable_signal
):

    print(
        stable_signal[
            [
                "feature",
                "direction",
                "overall_auc",
                "worst_fold_auc",
                "fold_auc_std",
                "positive_finite_n",
                "positive_median",
                "negative_median",
            ]
        ].to_string(
            index=False
        )
    )

else:

    print(
        "No feature satisfies overall AUC>=0.60 "
        "and worst-fold AUC>=0.55."
    )


# ============================================================
# 16. Persist formal audit
# ============================================================

feature_audit.to_pickle(
    AUDIT_OUT
)


summary = {
    "stage":
        "12.11G3H0",

    "status":
        "CROSS_FOLD_SCORING_FEATURE_AUDIT_COMPLETE",

    "parent_shortlist_id":
        frozen_shortlist[
            "shortlist_id"
        ],

    "parent_shortlist_sha256":
        SHORTLIST_SHA,

    "feature_schema_id":
        FEATURE_SCHEMA[
            "schema_id"
        ],

    "feature_schema_sha256":
        FEATURE_SCHEMA_SHA,

    "part_a_sha256":
        PART_A_SHA,

    "shortlist_rows":
        EXPECTED_SHORTLIST_ROWS,

    "hash_sample_rows_before_positive_removal":
        int(
            len(
                sample_table
            )
        ),

    "negative_audit_rows":
        int(
            len(
                sample_negatives
            )
        ),

    "positive_rows":
        202,

    "feature_audit":
        feature_audit.to_dict(
            "records"
        ),

    "stable_signal_diagnostic":
        stable_signal[
            [
                "feature",
                "direction",
                "overall_auc",
                "worst_fold_auc",
                "fold_auc_std",
            ]
        ].to_dict(
            "records"
        ),

    "sampling_gt_blind":
        True,

    "all_positives_reconstructed":
        True,

    "feature_subset_selected":
        False,

    "classifier_trained":
        False,

    "scoring_threshold_selected":
        False,

    "rewire_policy_selected":
        False,

    "fold0_used":
        False,

    "graph_modified":
        False,
}


SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert AUDIT_OUT.exists()
assert SUMMARY_OUT.exists()


# ============================================================
# 17. Integrity
# ============================================================

check_features = pd.read_pickle(
    POSITIVE_FEATURES_OUT
)


assert len(
    check_features
) == 202


check_audit = pd.read_pickle(
    AUDIT_OUT
)


assert len(
    check_audit
) == len(
    FEATURE_COLUMNS
)


# ============================================================
# 18. Decision
# ============================================================

print(
    "\n========== 12.11G3H0 DECISION =========="
)

print(
    "CROSS_FOLD_SCORING_FEATURE_AUDIT: PASS"
)

print(
    "parent shortlist SHA:",
    SHORTLIST_SHA
)

print(
    "feature schema SHA:",
    FEATURE_SCHEMA_SHA
)

print(
    "shortlist cardinality reproduced:",
    shortlist_total,
    "/",
    EXPECTED_SHORTLIST_ROWS
)

print(
    "GT-blind deterministic sample rows:",
    len(
        sample_table
    )
)

print(
    "positive feature rows reconstructed:",
    len(
        positive_features
    )
)

print(
    "feature subset selected:",
    "NO"
)

print(
    "classifier trained:",
    "NO"
)

print(
    "scoring threshold selected:",
    "NO"
)

print(
    "rewire policy selected:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "graph modified:",
    "NO"
)

print(
    "next:"
)

print(
    "DESIGN_LEAVE_ONE_FOLD_OUT_SCORER_BENCHMARK_FROM_FROZEN_FEATURE_SCHEMA"
)

In [ ]:
# ============================================================
# RESULT — Stage 12.11G3H0
#
# RESUME-SAFE COMPLETE VERSION
#
# Cross-Fold Scoring Feature Audit
#
# Authoritative parent generator:
#
#   MULTIFRAME_REWIRE_ASYM3X8_DOMINANCE_ANY_V1_FIDELITYPATCH1
#
# Authoritative frozen scorer-input shortlist:
#
#   ENDPOINT_SHORTLIST_GMEAN_G8_LOWCUT20_V1
#
#   rows = 17,501,674
#
# ------------------------------------------------------------
# RESUME SAFETY
# ------------------------------------------------------------
#
# Current known state:
#
#   FEATURE_SCHEMA_OUT        EXISTS
#   SAMPLE_OUT                MISSING
#   PARTA_SUMMARY_OUT         MISSING
#   POSITIVE_FEATURES_OUT     MISSING
#   AUDIT_OUT                 MISSING
#   SUMMARY_OUT               MISSING
#
# Existing schema SHA:
#
#   0a9f8e9a41c5c80c803c3a95955702dc
#   f4039605a21fc656a154aa05be55b39c
#
# Therefore:
#
#   - preserve existing feature schema
#   - verify exact fidelity
#   - do NOT rewrite schema
#   - safely resume GT-blind Part A
#
# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Do NOT train a classifier yet.
#
# First freeze / reuse a deployable GT-blind feature schema
# and test:
#
#   - whether true multi-frame repairs separate from shortlist
#     background candidates
#
#   - whether separation is stable across folds 1–4
#
#   - which features contain stable ranking signal
#
# IMPORTANT:
#
#   sampled label=0 rows below are BACKGROUND CANDIDATES,
#   not proven false proposals.
#
#   Therefore reported ROC AUC means:
#
#       known-positive vs sampled-background ranking AUC
#
#   NOT:
#
#       confirmed-positive vs confirmed-negative classifier AUC
#
# ------------------------------------------------------------
# PART A — GT BLIND
# ------------------------------------------------------------
#
# Regenerate exact frozen shortlist.
#
# Exact cardinality:
#
#       17,501,674
#
# Build scoring features using only:
#
#   candidate geometry
#   endpoint competition
#   CURRENT_BEST graph topology
#   incumbent-relative CV evidence
#
# Persist:
#
#   - existing frozen feature schema reused
#   - deterministic 1/64 candidate sample
#   - Part-A manifest
#
# ------------------------------------------------------------
# PART B — DEVELOPMENT ORACLE
# ------------------------------------------------------------
#
# Only after Part-A freeze:
#
#   - read development oracle
#   - reconstruct all 202 selected true-pair feature rows
#   - remove known positives from sampled background
#   - evaluate univariate AUC overall and by fold
#
# No classifier.
# No feature subset selected.
# No scoring threshold.
# No rewire actions.
# No Fold0.
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import gc
import hashlib
import json
import warnings

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from sklearn.metrics import roc_auc_score


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


FROZEN_GENERATOR_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_generator_asym3x8_v1_fidelitypatch1_frozen.json"
)


FROZEN_SHORTLIST_PATH = (
    S12
    / "stage12_folds14_endpoint_shortlist_g8_lowcut20_v1_frozen.json"
)


G3G1B_CARDINALITY_PATH = (
    S12
    / "stage12_folds14_endpoint_competition_targeted_rescue_cardinality_v1.json"
)


# GT artifact.
# DO NOT READ until Part A has been frozen.
G3G1B_ORACLE_PATH = (
    S12
    / "stage12_folds14_endpoint_competition_targeted_rescue_oracle_v1.pkl"
)


FEATURE_SCHEMA_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_feature_schema_v1_frozen.json"
)


SAMPLE_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_feature_sample_hash64_v1.pkl"
)


PARTA_SUMMARY_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_feature_audit_parta_v1.json"
)


POSITIVE_FEATURES_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_positive_features_v1.pkl"
)


AUDIT_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_feature_audit_v1.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_feature_audit_v1_summary.json"
)


for path in [
    FROZEN_GENERATOR_PATH,
    FROZEN_SHORTLIST_PATH,
    G3G1B_CARDINALITY_PATH,
    G3G1B_ORACLE_PATH,
]:

    assert path.exists(), path


# ============================================================
# 1. RESUME-AWARE OVERWRITE PROTECTION
#
# Existing FEATURE_SCHEMA_OUT is ALLOWED.
#
# Every downstream artifact must still be absent.
# ============================================================

for path in [
    SAMPLE_OUT,
    PARTA_SUMMARY_OUT,
    POSITIVE_FEATURES_OUT,
    AUDIT_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3H0 downstream output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite an existing Part-A/Part-B artifact."
        )


# ============================================================
# 2. AUTHORITATIVE CONTRACTS
# ============================================================

GENERATOR_SHA = (
    "da408df857cd34dcb4b1f65f3fae545e"
    "299f9403ef5edb986584caa8f4c3c203"
)


SHORTLIST_SHA = (
    "ae61216904bc3187f76b238b4b560d669"
    "3a7eb4f5d07bb8ef19021cc1bca1490"
)


KNOWN_SCHEMA_SHA = (
    "0a9f8e9a41c5c80c803c3a95955702dc"
    "f4039605a21fc656a154aa05be55b39c"
)


# ============================================================
# 3. PARENT GENERATOR
# ============================================================

parent_generator = json.loads(
    FROZEN_GENERATOR_PATH.read_text(
        encoding="utf-8"
    )
)


assert parent_generator[
    "generator_contract_sha256"
] == GENERATOR_SHA


assert parent_generator[
    "generator_id"
] == (
    "MULTIFRAME_REWIRE_ASYM3X8_"
    "DOMINANCE_ANY_V1_FIDELITYPATCH1"
)


assert parent_generator[
    "retrieval"
][
    "exact_knn_query_k"
] == 16


assert parent_generator[
    "retrieval"
][
    "strong_rank"
] == 3


assert parent_generator[
    "retrieval"
][
    "weak_rank"
] == 8


assert parent_generator[
    "development_cardinality"
][
    "retained_rows"
] == 29_227_030


# ============================================================
# 4. FROZEN SHORTLIST
# ============================================================

frozen_shortlist = json.loads(
    FROZEN_SHORTLIST_PATH.read_text(
        encoding="utf-8"
    )
)


assert frozen_shortlist[
    "shortlist_contract_sha256"
] == SHORTLIST_SHA


assert frozen_shortlist[
    "shortlist_id"
] == (
    "ENDPOINT_SHORTLIST_GMEAN_G8_LOWCUT20_V1"
)


assert frozen_shortlist[
    "parent_generator"
][
    "generator_contract_sha256"
] == GENERATOR_SHA


EXPECTED_SHORTLIST_ROWS = int(
    frozen_shortlist[
        "development_cardinality"
    ][
        "rows"
    ]
)


assert EXPECTED_SHORTLIST_ROWS == 17_501_674


print(
    "========== 12.11G3H0 FROZEN INPUT =========="
)

print(
    "parent generator:",
    parent_generator[
        "generator_id"
    ]
)

print(
    "generator SHA:",
    GENERATOR_SHA
)

print(
    "shortlist:",
    frozen_shortlist[
        "shortlist_id"
    ]
)

print(
    "shortlist SHA:",
    SHORTLIST_SHA
)

print(
    "expected shortlist rows:",
    EXPECTED_SHORTLIST_ROWS
)


# ============================================================
# 5. CURRENT_BEST BUILDER
#
# This is the only retained project-specific external helper
# required by this complete cell.
# ============================================================

assert (
    "build_current_best"
    in globals()
    and
    callable(
        globals()[
            "build_current_best"
        ]
    )
), (
    "build_current_best is missing from the kernel. "
    "Re-run the retained successful G3E-R CURRENT_BEST "
    "builder/setup cell first. Do NOT rerun G3H0 artifacts."
)


current_best_builder = globals()[
    "build_current_best"
]


# ============================================================
# 6. GT-BLIND G3G1B PER-DATASET FIDELITY SOURCE
# ============================================================

g3g1b_part_a = json.loads(
    G3G1B_CARDINALITY_PATH.read_text(
        encoding="utf-8"
    )
)


assert g3g1b_part_a[
    "stage"
] == "12.11G3G1B"


assert g3g1b_part_a[
    "gt_used_during_part_a"
] is False


assert g3g1b_part_a[
    "fold0_used"
] is False


assert int(
    g3g1b_part_a[
        "generator_raw_rows"
    ]
) == 36_473_971


assert int(
    g3g1b_part_a[
        "generator_retained_rows"
    ]
) == 29_227_030


expected_dataset_shortlist = {
    row[
        "dataset"
    ]:
        int(
            row[
                "contracts"
            ][
                "G8_LOWCUT20"
            ]
        )

    for row
    in g3g1b_part_a[
        "dataset_results"
    ]
}


assert len(
    expected_dataset_shortlist
) == 103


assert (
    sum(
        expected_dataset_shortlist.values()
    )
    ==
    EXPECTED_SHORTLIST_ROWS
)


# ============================================================
# 7. DEVELOPMENT CORPUS
# ============================================================

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


assert CV_PATH.exists(), CV_PATH


cv = pd.read_csv(
    CV_PATH
)


dev_manifest = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
)


development_names = (
    dev_manifest[
        "dataset"
    ]
    .tolist()
)


assert len(
    development_names
) == 103


fold_map_local = dict(
    zip(
        dev_manifest[
            "dataset"
        ],
        dev_manifest[
            "fold"
        ].astype(int),
    )
)


assert set(
    development_names
) == set(
    expected_dataset_shortlist
)


# ============================================================
# 8. FROZEN GEOMETRY / GENERATOR CONSTANTS
# ============================================================

SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


MIN_GAP_SIZE = 2
MAX_GAP_SIZE = 12

QUERY_K = 16

STRONG_RANK = 3
WEAK_RANK = 8

EPS = 1e-12


# ============================================================
# 9. BASIC HELPERS
# ============================================================

def vec_norm(
    x,
):

    return float(
        np.linalg.norm(
            np.asarray(
                x,
                dtype=float,
            )
        )
    )


def safe_ratio(
    candidate,
    best,
):

    candidate = float(
        candidate
    )

    best = float(
        best
    )


    if best <= EPS:

        if candidate <= EPS:

            return 1.0

        return np.inf


    return float(
        candidate
        /
        best
    )


def query_knn(
    tree,
    query_points,
    candidate_count,
    k,
):

    k_eff = min(
        int(
            k
        ),
        int(
            candidate_count
        ),
    )


    distances, indices = tree.query(
        query_points,
        k=k_eff,
    )


    if k_eff == 1:

        distances = distances[
            :,
            None
        ]

        indices = indices[
            :,
            None
        ]


    return (
        np.asarray(
            distances,
            dtype=float,
        ),

        np.asarray(
            indices,
            dtype=int,
        ),
    )


def build_rank_distance_map(
    query_ids,
    candidate_ids,
    distances,
    indices,
    reverse=False,
):

    result = {}


    for qi, query_id in enumerate(
        query_ids
    ):

        for ri in range(
            indices.shape[
                1
            ]
        ):

            candidate_id = int(
                candidate_ids[
                    int(
                        indices[
                            qi,
                            ri
                        ]
                    )
                ]
            )


            if reverse:

                pair = (
                    candidate_id,
                    int(
                        query_id
                    ),
                )

            else:

                pair = (
                    int(
                        query_id
                    ),
                    candidate_id,
                )


            if pair not in result:

                result[
                    pair
                ] = (
                    int(
                        ri
                        +
                        1
                    ),

                    float(
                        distances[
                            qi,
                            ri
                        ]
                    ),
                )


    return result


def asym3x8_pass(
    source_rank,
    target_rank,
):

    return bool(
        (
            source_rank
            <=
            STRONG_RANK
            and
            target_rank
            <=
            WEAK_RANK
        )
        or
        (
            target_rank
            <=
            STRONG_RANK
            and
            source_rank
            <=
            WEAK_RANK
        )
    )


# ============================================================
# 10. AUTHORITATIVE DATASET PROPOSAL GENERATOR
#
# Exact G3G1/G3G1B semantics:
#
#   CURRENT_BEST
#       ->
#   exact K16 ASYM3x8
#       ->
#   DOMINANCE_ANY
#
# Returns one compact dataset DataFrame only.
#
# No GT.
# ============================================================

def generate_dataset_proposals(
    dataset,
):

    graph, scale = current_best_builder(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    )


    # --------------------------------------------------------
    # Nodes
    # --------------------------------------------------------

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = nodes[
        "node_id"
    ].astype(
        int
    )


    nodes[
        "t"
    ] = nodes[
        "t"
    ].astype(
        int
    )


    nodes = (
        nodes
        .sort_values(
            [
                "t",
                "node_id",
            ],
            kind="stable",
        )
        .set_index(
            "node_id",
            drop=False,
        )
    )


    # --------------------------------------------------------
    # Edges
    # --------------------------------------------------------

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = edges[
        "source_id"
    ].astype(
        int
    )


    edges[
        "target_id"
    ] = edges[
        "target_id"
    ].astype(
        int
    )


    edge_set = set(
        zip(
            edges[
                "source_id"
            ],
            edges[
                "target_id"
            ],
        )
    )


    incoming = (
        edges.groupby(
            "target_id"
        )[
            "source_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    outgoing = (
        edges.groupby(
            "source_id"
        )[
            "target_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = {
        int(k):
            int(v)

        for k, v
        in indeg.items()
    }


    outdeg = {
        int(k):
            int(v)

        for k, v
        in outdeg.items()
    }


    # --------------------------------------------------------
    # Physical coordinates
    # --------------------------------------------------------

    phys = {}

    time_map = {}


    for nid, row in nodes.iterrows():

        nid = int(
            nid
        )


        phys[
            nid
        ] = (
            np.array(
                [
                    float(
                        row[
                            "z"
                        ]
                    ),

                    float(
                        row[
                            "y"
                        ]
                    ),

                    float(
                        row[
                            "x"
                        ]
                    ),
                ],
                dtype=float,
            )
            *
            SCALE
        )


        time_map[
            nid
        ] = int(
            row[
                "t"
            ]
        )


    node_ids = sorted(
        phys
    )


    # --------------------------------------------------------
    # Local velocities
    # --------------------------------------------------------

    past_velocity = {}

    future_velocity = {}


    for nid in node_ids:

        inc = incoming.get(
            nid,
            [],
        )


        if len(
            inc
        ) == 1:

            prev_id = int(
                inc[
                    0
                ]
            )


            if prev_id in phys:

                dt = (
                    time_map[
                        nid
                    ]
                    -
                    time_map[
                        prev_id
                    ]
                )


                if dt > 0:

                    past_velocity[
                        nid
                    ] = (
                        phys[
                            nid
                        ]
                        -
                        phys[
                            prev_id
                        ]
                    ) / float(
                        dt
                    )


        out = outgoing.get(
            nid,
            [],
        )


        if len(
            out
        ) == 1:

            next_id = int(
                out[
                    0
                ]
            )


            if next_id in phys:

                dt = (
                    time_map[
                        next_id
                    ]
                    -
                    time_map[
                        nid
                    ]
                )


                if dt > 0:

                    future_velocity[
                        nid
                    ] = (
                        phys[
                            next_id
                        ]
                        -
                        phys[
                            nid
                        ]
                    ) / float(
                        dt
                    )


    # --------------------------------------------------------
    # Eligibility
    # --------------------------------------------------------

    source_eligible = [
        nid

        for nid
        in node_ids

        if (
            outdeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid in past_velocity
        )
    ]


    target_eligible = [
        nid

        for nid
        in node_ids

        if (
            indeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid in future_velocity
        )
    ]


    source_by_t = {}

    target_by_t = {}


    for nid in source_eligible:

        source_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for nid in target_eligible:

        target_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for t in source_by_t:

        source_by_t[
            t
        ] = sorted(
            source_by_t[
                t
            ]
        )


    for t in target_by_t:

        target_by_t[
            t
        ] = sorted(
            target_by_t[
                t
            ]
        )


    # --------------------------------------------------------
    # Incumbent normalized CV
    # --------------------------------------------------------

    source_incumbent = {}

    target_incumbent = {}


    for src in source_eligible:

        if outdeg.get(
            src,
            0,
        ) != 1:

            continue


        old_targets = outgoing.get(
            src,
            [],
        )


        if len(
            old_targets
        ) != 1:

            continue


        old_tgt = int(
            old_targets[
                0
            ]
        )


        if old_tgt not in phys:

            continue


        dt_old = (
            time_map[
                old_tgt
            ]
            -
            time_map[
                src
            ]
        )


        if dt_old <= 0:

            continue


        predicted = (
            phys[
                src
            ]
            +
            float(
                dt_old
            )
            *
            past_velocity[
                src
            ]
        )


        source_incumbent[
            src
        ] = (
            vec_norm(
                phys[
                    old_tgt
                ]
                -
                predicted
            )
            /
            float(
                dt_old
            )
        )


    for tgt in target_eligible:

        if indeg.get(
            tgt,
            0,
        ) != 1:

            continue


        old_sources = incoming.get(
            tgt,
            [],
        )


        if len(
            old_sources
        ) != 1:

            continue


        old_src = int(
            old_sources[
                0
            ]
        )


        if old_src not in phys:

            continue


        dt_old = (
            time_map[
                tgt
            ]
            -
            time_map[
                old_src
            ]
        )


        if dt_old <= 0:

            continue


        predicted = (
            phys[
                tgt
            ]
            -
            float(
                dt_old
            )
            *
            future_velocity[
                tgt
            ]
        )


        target_incumbent[
            tgt
        ] = (
            vec_norm(
                phys[
                    old_src
                ]
                -
                predicted
            )
            /
            float(
                dt_old
            )
        )


    # --------------------------------------------------------
    # Proposal enumeration
    # --------------------------------------------------------

    proposal_frames = []

    raw_count = 0

    retained_count = 0


    for gap_size in range(
        MIN_GAP_SIZE,
        MAX_GAP_SIZE + 1,
    ):

        delta_t = (
            gap_size
            +
            1
        )


        for source_t in sorted(
            source_by_t
        ):

            target_t = (
                source_t
                +
                delta_t
            )


            if target_t not in target_by_t:

                continue


            source_ids = source_by_t[
                source_t
            ]


            target_ids = target_by_t[
                target_t
            ]


            if (
                len(
                    source_ids
                ) == 0
                or
                len(
                    target_ids
                ) == 0
            ):

                continue


            source_pos = np.vstack(
                [
                    phys[
                        nid
                    ]

                    for nid
                    in source_ids
                ]
            )


            target_pos = np.vstack(
                [
                    phys[
                        nid
                    ]

                    for nid
                    in target_ids
                ]
            )


            source_vel = np.vstack(
                [
                    past_velocity[
                        nid
                    ]

                    for nid
                    in source_ids
                ]
            )


            target_vel = np.vstack(
                [
                    future_velocity[
                        nid
                    ]

                    for nid
                    in target_ids
                ]
            )


            source_pred = (
                source_pos
                +
                float(
                    delta_t
                )
                *
                source_vel
            )


            target_pred = (
                target_pos
                -
                float(
                    delta_t
                )
                *
                target_vel
            )


            target_tree = cKDTree(
                target_pos
            )


            source_tree = cKDTree(
                source_pos
            )


            (
                source_dist,
                source_idx,
            ) = query_knn(
                target_tree,
                source_pred,
                len(
                    target_ids
                ),
                QUERY_K,
            )


            (
                target_dist,
                target_idx,
            ) = query_knn(
                source_tree,
                target_pred,
                len(
                    source_ids
                ),
                QUERY_K,
            )


            source_map = build_rank_distance_map(
                source_ids,
                target_ids,
                source_dist,
                source_idx,
                reverse=False,
            )


            target_map = build_rank_distance_map(
                target_ids,
                source_ids,
                target_dist,
                target_idx,
                reverse=True,
            )


            source_best = {
                int(
                    source_ids[
                        i
                    ]
                ):
                    float(
                        source_dist[
                            i,
                            0
                        ]
                    )

                for i
                in range(
                    len(
                        source_ids
                    )
                )
            }


            target_best = {
                int(
                    target_ids[
                        i
                    ]
                ):
                    float(
                        target_dist[
                            i,
                            0
                        ]
                    )

                for i
                in range(
                    len(
                        target_ids
                    )
                )
            }


            mutual_pairs = (
                set(
                    source_map
                )
                &
                set(
                    target_map
                )
            )


            frame_rows = []


            for (
                src,
                tgt,
            ) in mutual_pairs:

                if (
                    src,
                    tgt,
                ) in edge_set:

                    continue


                (
                    source_rank,
                    source_candidate_dist,
                ) = source_map[
                    (
                        src,
                        tgt,
                    )
                ]


                (
                    target_rank,
                    target_candidate_dist,
                ) = target_map[
                    (
                        src,
                        tgt,
                    )
                ]


                if not asym3x8_pass(
                    source_rank,
                    target_rank,
                ):

                    continue


                raw_count += 1


                source_degree = int(
                    outdeg.get(
                        src,
                        0,
                    )
                )


                target_degree = int(
                    indeg.get(
                        tgt,
                        0,
                    )
                )


                assert source_degree <= 1
                assert target_degree <= 1


                cuts = (
                    int(
                        source_degree == 1
                    )
                    +
                    int(
                        target_degree == 1
                    )
                )


                source_cv_pf = (
                    float(
                        source_candidate_dist
                    )
                    /
                    float(
                        delta_t
                    )
                )


                target_cv_pf = (
                    float(
                        target_candidate_dist
                    )
                    /
                    float(
                        delta_t
                    )
                )


                # --------------------------------------------
                # Frozen dominance-any
                # --------------------------------------------

                if cuts == 0:

                    dominance_any = True


                else:

                    improvements = []


                    if source_degree == 1:

                        incumbent = float(
                            source_incumbent.get(
                                src,
                                np.nan,
                            )
                        )


                        improvements.append(
                            (
                                incumbent
                                -
                                source_cv_pf
                            )
                            if np.isfinite(
                                incumbent
                            )
                            else np.nan
                        )


                    if target_degree == 1:

                        incumbent = float(
                            target_incumbent.get(
                                tgt,
                                np.nan,
                            )
                        )


                        improvements.append(
                            (
                                incumbent
                                -
                                target_cv_pf
                            )
                            if np.isfinite(
                                incumbent
                            )
                            else np.nan
                        )


                    finite_improvements = [
                        value

                        for value
                        in improvements

                        if np.isfinite(
                            value
                        )
                    ]


                    dominance_any = (
                        len(
                            finite_improvements
                        )
                        ==
                        len(
                            improvements
                        )
                        and
                        any(
                            value > 0

                            for value
                            in finite_improvements
                        )
                    )


                if not dominance_any:

                    continue


                retained_count += 1


                source_ratio = safe_ratio(
                    source_candidate_dist,
                    source_best[
                        src
                    ],
                )


                target_ratio = safe_ratio(
                    target_candidate_dist,
                    target_best[
                        tgt
                    ],
                )


                sym_cv = (
                    0.5
                    *
                    (
                        source_cv_pf
                        +
                        target_cv_pf
                    )
                )


                max_cv = max(
                    source_cv_pf,
                    target_cv_pf,
                )


                if (
                    np.isfinite(
                        source_ratio
                    )
                    and
                    np.isfinite(
                        target_ratio
                    )
                ):

                    gmean_ratio = float(
                        np.sqrt(
                            source_ratio
                            *
                            target_ratio
                        )
                    )


                else:

                    gmean_ratio = np.inf


                frame_rows.append(
                    (
                        int(
                            src
                        ),

                        int(
                            tgt
                        ),

                        int(
                            gap_size
                        ),

                        int(
                            cuts
                        ),

                        float(
                            sym_cv
                        ),

                        float(
                            max_cv
                        ),

                        float(
                            gmean_ratio
                        ),
                    )
                )


            if frame_rows:

                proposal_frames.append(
                    pd.DataFrame(
                        frame_rows,
                        columns=[
                            "source_id",
                            "target_id",
                            "gap_size",
                            "cuts",
                            "SYM_CV",
                            "MAX_CV",
                            "GMEAN_RATIO",
                        ],
                    )
                )


            del source_pos
            del target_pos

            del source_vel
            del target_vel

            del source_pred
            del target_pred

            del target_tree
            del source_tree

            del source_dist
            del source_idx

            del target_dist
            del target_idx

            del source_map
            del target_map

            del source_best
            del target_best

            del mutual_pairs
            del frame_rows


    # --------------------------------------------------------
    # Compact dataset proposal frame
    # --------------------------------------------------------

    if proposal_frames:

        proposals = pd.concat(
            proposal_frames,
            ignore_index=True,
        )


    else:

        proposals = pd.DataFrame(
            columns=[
                "source_id",
                "target_id",
                "gap_size",
                "cuts",
                "SYM_CV",
                "MAX_CV",
                "GMEAN_RATIO",
            ]
        )


    assert len(
        proposals
    ) == retained_count


    assert not proposals[
        [
            "source_id",
            "target_id",
        ]
    ].duplicated().any()


    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    del graph
    del nodes
    del edges

    del edge_set
    del incoming
    del outgoing

    del indeg
    del outdeg

    del phys
    del time_map
    del node_ids

    del past_velocity
    del future_velocity

    del source_eligible
    del target_eligible

    del source_by_t
    del target_by_t

    del source_incumbent
    del target_incumbent

    del proposal_frames

    gc.collect()


    return (
        proposals,
        int(
            raw_count
        ),
        int(
            retained_count
        ),
    )


# ============================================================
# 11. ENDPOINT COMPETITION RANKS
#
# Exact G3G1 deterministic tie-breaking.
# ============================================================

SCORES = [
    "SYM_CV",
    "MAX_CV",
    "GMEAN_RATIO",
]


def add_endpoint_ranks(
    proposals,
):

    proposals = proposals.copy()


    for score in SCORES:

        # ----------------------------------------------------
        # Source-wise rank
        # ----------------------------------------------------

        source_order = (
            proposals[
                [
                    "source_id",
                    "target_id",
                    "gap_size",
                    score,
                ]
            ]
            .sort_values(
                [
                    "source_id",
                    score,
                    "gap_size",
                    "target_id",
                ],
                kind="stable",
            )
        )


        source_order[
            f"{score}_SOURCE_RANK"
        ] = (
            source_order.groupby(
                "source_id",
                sort=False,
            )
            .cumcount()
            +
            1
        )


        proposals[
            f"{score}_SOURCE_RANK"
        ] = (
            source_order[
                f"{score}_SOURCE_RANK"
            ]
            .reindex(
                proposals.index
            )
            .astype(
                np.int32
            )
        )


        # ----------------------------------------------------
        # Target-wise rank
        # ----------------------------------------------------

        target_order = (
            proposals[
                [
                    "source_id",
                    "target_id",
                    "gap_size",
                    score,
                ]
            ]
            .sort_values(
                [
                    "target_id",
                    score,
                    "gap_size",
                    "source_id",
                ],
                kind="stable",
            )
        )


        target_order[
            f"{score}_TARGET_RANK"
        ] = (
            target_order.groupby(
                "target_id",
                sort=False,
            )
            .cumcount()
            +
            1
        )


        proposals[
            f"{score}_TARGET_RANK"
        ] = (
            target_order[
                f"{score}_TARGET_RANK"
            ]
            .reindex(
                proposals.index
            )
            .astype(
                np.int32
            )
        )


        del source_order
        del target_order


    return proposals


# ============================================================
# 12. FROZEN SCORING FEATURE SCHEMA
#
# EXACTLY the schema already frozen before the interrupted run.
# ============================================================

FEATURE_COLUMNS = [
    # --------------------------------------------------------
    # temporal / topology
    # --------------------------------------------------------

    "gap_size",
    "delta_t",

    "source_occupied",
    "target_occupied",
    "cuts",

    # --------------------------------------------------------
    # candidate motion consistency
    # --------------------------------------------------------

    "source_cv_pf",
    "target_cv_pf",

    "MIN_CV",
    "SYM_CV",
    "MAX_CV",

    # --------------------------------------------------------
    # local competition
    # --------------------------------------------------------

    "GMEAN_RATIO",

    # --------------------------------------------------------
    # GMEAN endpoint competition ranks
    # --------------------------------------------------------

    "GM_SOURCE_RANK",
    "GM_TARGET_RANK",
    "GM_BEST_RANK",
    "GM_MAX_RANK",
    "GM_RANK_SUM",
    "GM_RANK_LOG_ASYMMETRY",

    # --------------------------------------------------------
    # endpoint competition sizes
    # --------------------------------------------------------

    "SOURCE_COMPETITION_COUNT",
    "TARGET_COMPETITION_COUNT",

    "GM_SOURCE_RANK_FRAC",
    "GM_TARGET_RANK_FRAC",
    "GM_BEST_RANK_FRAC",

    # --------------------------------------------------------
    # SYM/MAX endpoint competition
    # --------------------------------------------------------

    "SYM_BEST_RANK",
    "MAX_BEST_RANK",

    # --------------------------------------------------------
    # incumbent-relative evidence
    # --------------------------------------------------------

    "source_incumbent_cv_pf",
    "target_incumbent_cv_pf",

    "source_relative_improvement",
    "target_relative_improvement",

    "max_required_improvement",
    "min_required_improvement",
    "mean_required_improvement",

    "dominance_all",
]


FEATURE_SCHEMA_TEMPLATE = {
    "stage":
        "12.11G3H0",

    "schema_id":
        "MULTIFRAME_REWIRE_SCORING_FEATURES_V1",

    "parent_shortlist_id":
        frozen_shortlist[
            "shortlist_id"
        ],

    "parent_shortlist_sha256":
        SHORTLIST_SHA,

    "parent_generator_sha256":
        GENERATOR_SHA,

    "features":
        FEATURE_COLUMNS,

    "forbidden_features": [
        "fold",
        "dataset identity",
        "GT labels",
        "official FP",
        "oracle component geometry",
        "pred_full_3um",
        "pred_full_5um",
        "pred_full_7um",
    ],

    "runtime_features_gt_blind":
        True,

    "feature_schema_defined_before_gt_read":
        True,

    "feature_selection_performed":
        False,

    "classifier_trained":
        False,

    "fold0_used":
        False,
}


# ============================================================
# 13. RESUME-AWARE FEATURE SCHEMA HANDLING
# ============================================================

schema_bytes = json.dumps(
    FEATURE_SCHEMA_TEMPLATE,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
).encode(
    "utf-8"
)


EXPECTED_FEATURE_SCHEMA_SHA = hashlib.sha256(
    schema_bytes
).hexdigest()


assert EXPECTED_FEATURE_SCHEMA_SHA == KNOWN_SCHEMA_SHA, (
    EXPECTED_FEATURE_SCHEMA_SHA,
    KNOWN_SCHEMA_SHA,
)


if FEATURE_SCHEMA_OUT.exists():

    existing_schema = json.loads(
        FEATURE_SCHEMA_OUT.read_text(
            encoding="utf-8"
        )
    )


    assert existing_schema[
        "stage"
    ] == "12.11G3H0"


    assert existing_schema[
        "schema_id"
    ] == (
        "MULTIFRAME_REWIRE_SCORING_FEATURES_V1"
    )


    assert existing_schema[
        "parent_shortlist_id"
    ] == (
        "ENDPOINT_SHORTLIST_GMEAN_G8_LOWCUT20_V1"
    )


    assert existing_schema[
        "parent_shortlist_sha256"
    ] == SHORTLIST_SHA


    assert existing_schema[
        "parent_generator_sha256"
    ] == GENERATOR_SHA


    assert existing_schema[
        "features"
    ] == FEATURE_COLUMNS


    assert existing_schema[
        "forbidden_features"
    ] == FEATURE_SCHEMA_TEMPLATE[
        "forbidden_features"
    ]


    assert existing_schema[
        "runtime_features_gt_blind"
    ] is True


    assert existing_schema[
        "feature_schema_defined_before_gt_read"
    ] is True


    recorded_sha = existing_schema[
        "feature_schema_sha256"
    ]


    assert recorded_sha == KNOWN_SCHEMA_SHA


    # Recompute from content without self-hash.
    existing_for_hash = dict(
        existing_schema
    )


    existing_for_hash.pop(
        "feature_schema_sha256",
        None,
    )


    recomputed_existing_sha = hashlib.sha256(
        json.dumps(
            existing_for_hash,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        ).encode(
            "utf-8"
        )
    ).hexdigest()


    assert recomputed_existing_sha == recorded_sha


    FEATURE_SCHEMA = existing_schema

    FEATURE_SCHEMA_SHA = recorded_sha


    print(
        "\n========== FROZEN FEATURE SCHEMA =========="
    )

    print(
        "schema ID:",
        FEATURE_SCHEMA[
            "schema_id"
        ]
    )

    print(
        "feature count:",
        len(
            FEATURE_COLUMNS
        )
    )

    print(
        "schema SHA:",
        FEATURE_SCHEMA_SHA
    )

    print(
        "existing frozen schema reused:",
        "YES"
    )

    print(
        "schema rewritten:",
        "NO"
    )

    print(
        "FROZEN BEFORE GT READ:",
        "YES"
    )


else:

    # Fresh-run fallback only.
    FEATURE_SCHEMA = dict(
        FEATURE_SCHEMA_TEMPLATE
    )


    FEATURE_SCHEMA_SHA = (
        EXPECTED_FEATURE_SCHEMA_SHA
    )


    FEATURE_SCHEMA[
        "feature_schema_sha256"
    ] = FEATURE_SCHEMA_SHA


    FEATURE_SCHEMA_OUT.write_text(
        json.dumps(
            FEATURE_SCHEMA,
            indent=2,
        ),
        encoding="utf-8",
    )


    assert FEATURE_SCHEMA_OUT.exists()


    print(
        "\n========== FROZEN FEATURE SCHEMA =========="
    )

    print(
        "schema ID:",
        FEATURE_SCHEMA[
            "schema_id"
        ]
    )

    print(
        "feature count:",
        len(
            FEATURE_COLUMNS
        )
    )

    print(
        "schema SHA:",
        FEATURE_SCHEMA_SHA
    )

    print(
        "existing frozen schema reused:",
        "NO"
    )

    print(
        "schema rewritten:",
        "YES — fresh-run fallback"
    )

    print(
        "FROZEN BEFORE GT READ:",
        "YES"
    )


# ============================================================
# 14. FROZEN SHORTLIST RANK FIELDS
# ============================================================

def add_shortlist_rank_fields(
    proposals,
):

    proposals = proposals.copy()


    proposals[
        "GM_SOURCE_RANK"
    ] = proposals[
        "GMEAN_RATIO_SOURCE_RANK"
    ].astype(
        np.int32
    )


    proposals[
        "GM_TARGET_RANK"
    ] = proposals[
        "GMEAN_RATIO_TARGET_RANK"
    ].astype(
        np.int32
    )


    proposals[
        "GM_BEST_RANK"
    ] = np.minimum(
        proposals[
            "GM_SOURCE_RANK"
        ].to_numpy(),

        proposals[
            "GM_TARGET_RANK"
        ].to_numpy(),
    ).astype(
        np.int32
    )


    proposals[
        "GM_MAX_RANK"
    ] = np.maximum(
        proposals[
            "GM_SOURCE_RANK"
        ].to_numpy(),

        proposals[
            "GM_TARGET_RANK"
        ].to_numpy(),
    ).astype(
        np.int32
    )


    proposals[
        "SYM_BEST_RANK"
    ] = np.minimum(
        proposals[
            "SYM_CV_SOURCE_RANK"
        ].to_numpy(),

        proposals[
            "SYM_CV_TARGET_RANK"
        ].to_numpy(),
    ).astype(
        np.int32
    )


    proposals[
        "MAX_BEST_RANK"
    ] = np.minimum(
        proposals[
            "MAX_CV_SOURCE_RANK"
        ].to_numpy(),

        proposals[
            "MAX_CV_TARGET_RANK"
        ].to_numpy(),
    ).astype(
        np.int32
    )


    return proposals


def frozen_shortlist_mask(
    proposals,
):

    return (
        proposals[
            "GM_BEST_RANK"
        ].le(
            8
        )
        |
        (
            proposals[
                "cuts"
            ].le(
                1
            )
            &
            proposals[
                "GM_BEST_RANK"
            ].le(
                20
            )
        )
    )


# ============================================================
# 15. SCORING-FEATURE ATTACHMENT
# ============================================================

def attach_scoring_features(
    dataset,
    proposals,
):

    p = proposals.copy()


    if len(
        p
    ) == 0:

        return p


    graph, scale = current_best_builder(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    )


    # --------------------------------------------------------
    # Nodes
    # --------------------------------------------------------

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = nodes[
        "node_id"
    ].astype(
        int
    )


    nodes[
        "t"
    ] = nodes[
        "t"
    ].astype(
        int
    )


    # --------------------------------------------------------
    # Edges
    # --------------------------------------------------------

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = edges[
        "source_id"
    ].astype(
        int
    )


    edges[
        "target_id"
    ] = edges[
        "target_id"
    ].astype(
        int
    )


    max_node_id = int(
        max(
            nodes[
                "node_id"
            ].max(),

            p[
                "source_id"
            ].max(),

            p[
                "target_id"
            ].max(),
        )
    )


    size = (
        max_node_id
        +
        1
    )


    # --------------------------------------------------------
    # Dense lookup arrays
    # --------------------------------------------------------

    position = np.full(
        (
            size,
            3,
        ),
        np.nan,
        dtype=np.float64,
    )


    time_arr = np.full(
        size,
        -1,
        dtype=np.int32,
    )


    node_ids_arr = nodes[
        "node_id"
    ].to_numpy(
        dtype=np.int64
    )


    position[
        node_ids_arr
    ] = (
        nodes[
            [
                "z",
                "y",
                "x",
            ]
        ]
        .to_numpy(
            dtype=np.float64
        )
        *
        SCALE
    )


    time_arr[
        node_ids_arr
    ] = nodes[
        "t"
    ].to_numpy(
        dtype=np.int32
    )


    # --------------------------------------------------------
    # Graph topology arrays
    # --------------------------------------------------------

    source_edge_ids = edges[
        "source_id"
    ].to_numpy(
        dtype=np.int64
    )


    target_edge_ids = edges[
        "target_id"
    ].to_numpy(
        dtype=np.int64
    )


    outdegree = np.bincount(
        source_edge_ids,
        minlength=size,
    ).astype(
        np.int32
    )


    indegree = np.bincount(
        target_edge_ids,
        minlength=size,
    ).astype(
        np.int32
    )


    predecessor = np.full(
        size,
        -1,
        dtype=np.int64,
    )


    successor = np.full(
        size,
        -1,
        dtype=np.int64,
    )


    # Multiple assignment is harmless because predecessor /
    # successor are used only when exact degree == 1.
    predecessor[
        target_edge_ids
    ] = source_edge_ids


    successor[
        source_edge_ids
    ] = target_edge_ids


    # --------------------------------------------------------
    # Velocity arrays
    # --------------------------------------------------------

    past_velocity = np.full(
        (
            size,
            3,
        ),
        np.nan,
        dtype=np.float64,
    )


    future_velocity = np.full(
        (
            size,
            3,
        ),
        np.nan,
        dtype=np.float64,
    )


    unique_in = np.flatnonzero(
        indegree == 1
    )


    unique_in = unique_in[
        predecessor[
            unique_in
        ] >= 0
    ]


    if len(
        unique_in
    ):

        pred_ids = predecessor[
            unique_in
        ]


        dt_past = (
            time_arr[
                unique_in
            ]
            -
            time_arr[
                pred_ids
            ]
        )


        valid = dt_past > 0


        unique_in_valid = unique_in[
            valid
        ]


        pred_ids_valid = predecessor[
            unique_in_valid
        ]


        dt_past_valid = (
            time_arr[
                unique_in_valid
            ]
            -
            time_arr[
                pred_ids_valid
            ]
        ).astype(
            np.float64
        )


        past_velocity[
            unique_in_valid
        ] = (
            position[
                unique_in_valid
            ]
            -
            position[
                pred_ids_valid
            ]
        ) / dt_past_valid[
            :,
            None
        ]


    unique_out = np.flatnonzero(
        outdegree == 1
    )


    unique_out = unique_out[
        successor[
            unique_out
        ] >= 0
    ]


    if len(
        unique_out
    ):

        succ_ids = successor[
            unique_out
        ]


        dt_future = (
            time_arr[
                succ_ids
            ]
            -
            time_arr[
                unique_out
            ]
        )


        valid = dt_future > 0


        unique_out_valid = unique_out[
            valid
        ]


        succ_ids_valid = successor[
            unique_out_valid
        ]


        dt_future_valid = (
            time_arr[
                succ_ids_valid
            ]
            -
            time_arr[
                unique_out_valid
            ]
        ).astype(
            np.float64
        )


        future_velocity[
            unique_out_valid
        ] = (
            position[
                succ_ids_valid
            ]
            -
            position[
                unique_out_valid
            ]
        ) / dt_future_valid[
            :,
            None
        ]


    # --------------------------------------------------------
    # Candidate arrays
    # --------------------------------------------------------

    src = p[
        "source_id"
    ].to_numpy(
        dtype=np.int64
    )


    tgt = p[
        "target_id"
    ].to_numpy(
        dtype=np.int64
    )


    assert np.isfinite(
        past_velocity[
            src
        ]
    ).all()


    assert np.isfinite(
        future_velocity[
            tgt
        ]
    ).all()


    source_t = time_arr[
        src
    ]


    target_t = time_arr[
        tgt
    ]


    delta_t = (
        target_t
        -
        source_t
    ).astype(
        np.float64
    )


    assert np.all(
        delta_t
        >
        0
    )


    p[
        "delta_t"
    ] = delta_t.astype(
        np.int16
    )


    # Frozen temporal semantics.
    assert np.array_equal(
        p[
            "delta_t"
        ].to_numpy(
            dtype=np.int16
        ),

        (
            p[
                "gap_size"
            ].to_numpy(
                dtype=np.int16
            )
            +
            1
        ),
    )


    # --------------------------------------------------------
    # Candidate forward / backward CV
    # --------------------------------------------------------

    source_predicted_target = (
        position[
            src
        ]
        +
        delta_t[
            :,
            None
        ]
        *
        past_velocity[
            src
        ]
    )


    target_predicted_source = (
        position[
            tgt
        ]
        -
        delta_t[
            :,
            None
        ]
        *
        future_velocity[
            tgt
        ]
    )


    source_cv_pf = (
        np.linalg.norm(
            position[
                tgt
            ]
            -
            source_predicted_target,
            axis=1,
        )
        /
        delta_t
    )


    target_cv_pf = (
        np.linalg.norm(
            position[
                src
            ]
            -
            target_predicted_source,
            axis=1,
        )
        /
        delta_t
    )


    assert np.isfinite(
        source_cv_pf
    ).all()


    assert np.isfinite(
        target_cv_pf
    ).all()


    p[
        "source_cv_pf"
    ] = source_cv_pf


    p[
        "target_cv_pf"
    ] = target_cv_pf


    p[
        "MIN_CV"
    ] = np.minimum(
        source_cv_pf,
        target_cv_pf,
    )


    # Exact fidelity against proposal generator.
    assert np.allclose(
        (
            source_cv_pf
            +
            target_cv_pf
        )
        /
        2.0,

        p[
            "SYM_CV"
        ].to_numpy(
            dtype=float
        ),

        atol=1e-10,
        rtol=1e-10,
    )


    assert np.allclose(
        np.maximum(
            source_cv_pf,
            target_cv_pf,
        ),

        p[
            "MAX_CV"
        ].to_numpy(
            dtype=float
        ),

        atol=1e-10,
        rtol=1e-10,
    )


    # --------------------------------------------------------
    # Occupancy / cuts
    # --------------------------------------------------------

    source_occupied = (
        outdegree[
            src
        ]
        ==
        1
    )


    target_occupied = (
        indegree[
            tgt
        ]
        ==
        1
    )


    p[
        "source_occupied"
    ] = source_occupied.astype(
        np.int8
    )


    p[
        "target_occupied"
    ] = target_occupied.astype(
        np.int8
    )


    reconstructed_cuts = (
        source_occupied.astype(
            np.int8
        )
        +
        target_occupied.astype(
            np.int8
        )
    )


    assert np.array_equal(
        reconstructed_cuts,

        p[
            "cuts"
        ].to_numpy(
            dtype=np.int8
        ),
    )


    # --------------------------------------------------------
    # Incumbent normalized CV
    # --------------------------------------------------------

    source_incumbent_cv = np.full(
        len(
            p
        ),
        np.nan,
        dtype=np.float64,
    )


    target_incumbent_cv = np.full(
        len(
            p
        ),
        np.nan,
        dtype=np.float64,
    )


    source_occ_index = np.flatnonzero(
        source_occupied
    )


    if len(
        source_occ_index
    ):

        occ_src = src[
            source_occ_index
        ]


        old_tgt = successor[
            occ_src
        ]


        assert np.all(
            old_tgt
            >=
            0
        )


        dt_old = (
            time_arr[
                old_tgt
            ]
            -
            time_arr[
                occ_src
            ]
        ).astype(
            np.float64
        )


        assert np.all(
            dt_old
            >
            0
        )


        old_pred = (
            position[
                occ_src
            ]
            +
            dt_old[
                :,
                None
            ]
            *
            past_velocity[
                occ_src
            ]
        )


        source_incumbent_cv[
            source_occ_index
        ] = (
            np.linalg.norm(
                position[
                    old_tgt
                ]
                -
                old_pred,
                axis=1,
            )
            /
            dt_old
        )


    target_occ_index = np.flatnonzero(
        target_occupied
    )


    if len(
        target_occ_index
    ):

        occ_tgt = tgt[
            target_occ_index
        ]


        old_src = predecessor[
            occ_tgt
        ]


        assert np.all(
            old_src
            >=
            0
        )


        dt_old = (
            time_arr[
                occ_tgt
            ]
            -
            time_arr[
                old_src
            ]
        ).astype(
            np.float64
        )


        assert np.all(
            dt_old
            >
            0
        )


        old_pred = (
            position[
                occ_tgt
            ]
            -
            dt_old[
                :,
                None
            ]
            *
            future_velocity[
                occ_tgt
            ]
        )


        target_incumbent_cv[
            target_occ_index
        ] = (
            np.linalg.norm(
                position[
                    old_src
                ]
                -
                old_pred,
                axis=1,
            )
            /
            dt_old
        )


    p[
        "source_incumbent_cv_pf"
    ] = source_incumbent_cv


    p[
        "target_incumbent_cv_pf"
    ] = target_incumbent_cv


    source_improvement = (
        source_incumbent_cv
        -
        source_cv_pf
    )


    target_improvement = (
        target_incumbent_cv
        -
        target_cv_pf
    )


    p[
        "source_relative_improvement"
    ] = source_improvement


    p[
        "target_relative_improvement"
    ] = target_improvement


    # --------------------------------------------------------
    # Required-side improvement summaries
    # --------------------------------------------------------

    max_required = np.full(
        len(
            p
        ),
        np.nan,
        dtype=np.float64,
    )


    min_required = np.full(
        len(
            p
        ),
        np.nan,
        dtype=np.float64,
    )


    mean_required = np.full(
        len(
            p
        ),
        np.nan,
        dtype=np.float64,
    )


    # For free/free, dominance_all is vacuously true.
    dominance_all = np.ones(
        len(
            p
        ),
        dtype=np.int8,
    )


    # source only
    mask = (
        source_occupied
        &
        ~target_occupied
    )


    max_required[
        mask
    ] = source_improvement[
        mask
    ]


    min_required[
        mask
    ] = source_improvement[
        mask
    ]


    mean_required[
        mask
    ] = source_improvement[
        mask
    ]


    dominance_all[
        mask
    ] = (
        source_improvement[
            mask
        ]
        >
        0
    ).astype(
        np.int8
    )


    # target only
    mask = (
        ~source_occupied
        &
        target_occupied
    )


    max_required[
        mask
    ] = target_improvement[
        mask
    ]


    min_required[
        mask
    ] = target_improvement[
        mask
    ]


    mean_required[
        mask
    ] = target_improvement[
        mask
    ]


    dominance_all[
        mask
    ] = (
        target_improvement[
            mask
        ]
        >
        0
    ).astype(
        np.int8
    )


    # both occupied
    mask = (
        source_occupied
        &
        target_occupied
    )


    max_required[
        mask
    ] = np.maximum(
        source_improvement[
            mask
        ],
        target_improvement[
            mask
        ],
    )


    min_required[
        mask
    ] = np.minimum(
        source_improvement[
            mask
        ],
        target_improvement[
            mask
        ],
    )


    mean_required[
        mask
    ] = (
        source_improvement[
            mask
        ]
        +
        target_improvement[
            mask
        ]
    ) / 2.0


    dominance_all[
        mask
    ] = (
        (
            source_improvement[
                mask
            ]
            >
            0
        )
        &
        (
            target_improvement[
                mask
            ]
            >
            0
        )
    ).astype(
        np.int8
    )


    p[
        "max_required_improvement"
    ] = max_required


    p[
        "min_required_improvement"
    ] = min_required


    p[
        "mean_required_improvement"
    ] = mean_required


    p[
        "dominance_all"
    ] = dominance_all


    # --------------------------------------------------------
    # Endpoint competition counts within scorer-input shortlist
    # --------------------------------------------------------

    source_counts = (
        p.groupby(
            "source_id",
            sort=False,
        )[
            "source_id"
        ]
        .transform(
            "size"
        )
        .to_numpy(
            dtype=np.int32
        )
    )


    target_counts = (
        p.groupby(
            "target_id",
            sort=False,
        )[
            "target_id"
        ]
        .transform(
            "size"
        )
        .to_numpy(
            dtype=np.int32
        )
    )


    assert np.all(
        source_counts
        >=
        1
    )


    assert np.all(
        target_counts
        >=
        1
    )


    p[
        "SOURCE_COMPETITION_COUNT"
    ] = source_counts


    p[
        "TARGET_COMPETITION_COUNT"
    ] = target_counts


    # --------------------------------------------------------
    # Rank-derived features
    # --------------------------------------------------------

    gm_src = p[
        "GM_SOURCE_RANK"
    ].to_numpy(
        dtype=np.float64
    )


    gm_tgt = p[
        "GM_TARGET_RANK"
    ].to_numpy(
        dtype=np.float64
    )


    p[
        "GM_RANK_SUM"
    ] = (
        gm_src
        +
        gm_tgt
    )


    p[
        "GM_RANK_LOG_ASYMMETRY"
    ] = np.abs(
        np.log1p(
            gm_src
        )
        -
        np.log1p(
            gm_tgt
        )
    )


    p[
        "GM_SOURCE_RANK_FRAC"
    ] = (
        gm_src
        /
        source_counts
    )


    p[
        "GM_TARGET_RANK_FRAC"
    ] = (
        gm_tgt
        /
        target_counts
    )


    p[
        "GM_BEST_RANK_FRAC"
    ] = np.minimum(
        p[
            "GM_SOURCE_RANK_FRAC"
        ].to_numpy(
            dtype=float
        ),

        p[
            "GM_TARGET_RANK_FRAC"
        ].to_numpy(
            dtype=float
        ),
    )


    # --------------------------------------------------------
    # Final schema fidelity
    # --------------------------------------------------------

    for col in FEATURE_COLUMNS:

        assert col in p.columns, (
            dataset,
            col,
        )


    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    del graph
    del nodes
    del edges

    del position
    del time_arr

    del outdegree
    del indegree

    del predecessor
    del successor

    del past_velocity
    del future_velocity

    gc.collect()


    return p


# ============================================================
# 16. DETERMINISTIC GT-BLIND 1/64 HASH SAMPLE
# ============================================================

SAMPLE_MODULUS = 64

SAMPLE_BUCKET = 0


def dataset_seed_uint64(
    dataset,
):

    digest = hashlib.sha256(
        str(
            dataset
        ).encode(
            "utf-8"
        )
    ).digest()


    return np.uint64(
        int.from_bytes(
            digest[
                :8
            ],
            byteorder="little",
            signed=False,
        )
    )


def splitmix64(
    x,
):

    # uint64 overflow is intentional.
    with np.errstate(
        over="ignore",
    ):

        x = (
            x
            +
            np.uint64(
                0x9E3779B97F4A7C15
            )
        )


        x = (
            (
                x
                ^
                (
                    x
                    >>
                    np.uint64(
                        30
                    )
                )
            )
            *
            np.uint64(
                0xBF58476D1CE4E5B9
            )
        )


        x = (
            (
                x
                ^
                (
                    x
                    >>
                    np.uint64(
                        27
                    )
                )
            )
            *
            np.uint64(
                0x94D049BB133111EB
            )
        )


        x = (
            x
            ^
            (
                x
                >>
                np.uint64(
                    31
                )
            )
        )


    return x


def deterministic_sample_mask(
    dataset,
    frame,
):

    src = frame[
        "source_id"
    ].to_numpy(
        dtype=np.uint64
    )


    tgt = frame[
        "target_id"
    ].to_numpy(
        dtype=np.uint64
    )


    gap = frame[
        "gap_size"
    ].to_numpy(
        dtype=np.uint64
    )


    seed = dataset_seed_uint64(
        dataset
    )


    with np.errstate(
        over="ignore",
    ):

        x = (
            seed
            ^
            (
                src
                *
                np.uint64(
                    0xD6E8FEB86659FD93
                )
            )
            ^
            (
                tgt
                *
                np.uint64(
                    0xA5A3564E27F8862B
                )
            )
            ^
            (
                gap
                *
                np.uint64(
                    0x9E3779B185EBCA87
                )
            )
        )


    h = splitmix64(
        x
    )


    return (
        (
            h
            %
            np.uint64(
                SAMPLE_MODULUS
            )
        )
        ==
        np.uint64(
            SAMPLE_BUCKET
        )
    )


# ============================================================
# PART A — GT-BLIND SCORING FEATURE SAMPLE
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART A — GT-BLIND SCORING FEATURE SAMPLE"
)

print(
    "============================================================"
)


sample_frames = []

dataset_results = []


shortlist_total = 0

raw_total = 0

retained_total = 0


# NOTE:
#
# Identity columns do NOT include gap_size here because
# FEATURE_COLUMNS already contains gap_size.
#
# This repairs the duplicate-column bug in the first G3H0
# draft.
SAMPLE_BASE_COLUMNS = [
    "source_id",
    "target_id",
]


for ds_i, dataset in enumerate(
    development_names,
    start=1,
):

    fold = int(
        fold_map_local[
            dataset
        ]
    )


    (
        proposals,
        raw_count,
        retained_count,
    ) = generate_dataset_proposals(
        dataset
    )


    raw_total += int(
        raw_count
    )


    retained_total += int(
        retained_count
    )


    proposals = add_endpoint_ranks(
        proposals
    )


    proposals = add_shortlist_rank_fields(
        proposals
    )


    shortlist_mask = frozen_shortlist_mask(
        proposals
    )


    shortlist = (
        proposals[
            shortlist_mask
        ]
        .copy()
    )


    expected_count = int(
        expected_dataset_shortlist[
            dataset
        ]
    )


    assert len(
        shortlist
    ) == expected_count, (
        dataset,
        len(
            shortlist
        ),
        expected_count,
    )


    shortlist_total += int(
        len(
            shortlist
        )
    )


    shortlist = attach_scoring_features(
        dataset,
        shortlist,
    )


    for col in FEATURE_COLUMNS:

        assert col in shortlist.columns, (
            dataset,
            col,
        )


    sample_mask = deterministic_sample_mask(
        dataset,
        shortlist,
    )


    sample_columns = (
        SAMPLE_BASE_COLUMNS
        +
        FEATURE_COLUMNS
    )


    assert len(
        sample_columns
    ) == len(
        set(
            sample_columns
        )
    ), "Duplicate columns in sample extraction."


    sample = shortlist.loc[
        sample_mask,
        sample_columns,
    ].copy()


    sample.insert(
        0,
        "dataset",
        dataset,
    )


    sample.insert(
        1,
        "fold",
        fold,
    )


    assert not sample.columns.duplicated().any()


    sample_frames.append(
        sample
    )


    dataset_results.append(
        {
            "dataset":
                dataset,

            "fold":
                fold,

            "raw_generator_rows":
                int(
                    raw_count
                ),

            "post_dominance_generator_rows":
                int(
                    retained_count
                ),

            "shortlist_rows":
                int(
                    len(
                        shortlist
                    )
                ),

            "sample_rows":
                int(
                    len(
                        sample
                    )
                ),
        }
    )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            development_names
        )
    ):

        print(
            f"{ds_i:3d}/{len(development_names)}"
            f" | fold={fold}"
            f" | {dataset}"
            f" | raw={raw_count}"
            f" | generator={retained_count}"
            f" | shortlist={len(shortlist)}"
            f" | sample={len(sample)}"
        )


    del proposals
    del shortlist
    del sample

    gc.collect()


# ============================================================
# 17. PART-A FIDELITY
# ============================================================

assert raw_total == 36_473_971, (
    raw_total,
    36_473_971,
)


assert retained_total == 29_227_030, (
    retained_total,
    29_227_030,
)


assert shortlist_total == EXPECTED_SHORTLIST_ROWS, (
    shortlist_total,
    EXPECTED_SHORTLIST_ROWS,
)


sample_table = pd.concat(
    sample_frames,
    ignore_index=True,
)


del sample_frames

gc.collect()


assert not sample_table.columns.duplicated().any()


assert not sample_table[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


sample_table.to_pickle(
    SAMPLE_OUT
)


assert SAMPLE_OUT.exists()


print(
    "\n========== PART-A SHORTLIST FIDELITY =========="
)

print(
    "raw generator reproduced:",
    raw_total,
    "/ 36473971"
)

print(
    "post-dominance generator reproduced:",
    retained_total,
    "/ 29227030"
)

print(
    "shortlist reproduced:",
    shortlist_total,
    "/",
    EXPECTED_SHORTLIST_ROWS
)

print(
    "deterministic sample rows:",
    len(
        sample_table
    )
)

print(
    "sample fraction:",
    len(
        sample_table
    )
    /
    EXPECTED_SHORTLIST_ROWS
)

print(
    "SHORTLIST_REPRODUCTION: PASS"
)


# ============================================================
# 18. FREEZE PART A BEFORE GT READ
# ============================================================

part_a = {
    "stage":
        "12.11G3H0",

    "part":
        "GT_BLIND_SCORING_FEATURE_SAMPLE",

    "resume_mode":
        "VALID_SCHEMA_ONLY_RESUME",

    "existing_schema_reused":
        True,

    "existing_schema_rewritten":
        False,

    "feature_schema_id":
        FEATURE_SCHEMA[
            "schema_id"
        ],

    "feature_schema_sha256":
        FEATURE_SCHEMA_SHA,

    "parent_shortlist_id":
        frozen_shortlist[
            "shortlist_id"
        ],

    "parent_shortlist_sha256":
        SHORTLIST_SHA,

    "parent_generator_sha256":
        GENERATOR_SHA,

    "raw_generator_rows":
        int(
            raw_total
        ),

    "post_dominance_generator_rows":
        int(
            retained_total
        ),

    "shortlist_rows":
        int(
            shortlist_total
        ),

    "sampling": {
        "type":
            "DETERMINISTIC_SPLITMIX64_HASH",

        "modulus":
            SAMPLE_MODULUS,

        "bucket":
            SAMPLE_BUCKET,

        "identity":
            "dataset,source_id,target_id,gap_size",

        "gt_blind":
            True,
    },

    "sample_rows":
        int(
            len(
                sample_table
            )
        ),

    "dataset_results":
        dataset_results,

    "gt_used":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "feature_subset_selected":
        False,

    "scoring_threshold_selected":
        False,

    "rewire_policy_selected":
        False,

    "graph_modified":
        False,
}


part_a_bytes = json.dumps(
    part_a,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
).encode(
    "utf-8"
)


PART_A_SHA = hashlib.sha256(
    part_a_bytes
).hexdigest()


part_a[
    "sha256"
] = PART_A_SHA


PARTA_SUMMARY_OUT.write_text(
    json.dumps(
        part_a,
        indent=2,
    ),
    encoding="utf-8",
)


assert PARTA_SUMMARY_OUT.exists()


print(
    "\nGT-BLIND FEATURE SAMPLE FROZEN BEFORE GT READ: YES"
)

print(
    "Part-A SHA256:",
    PART_A_SHA
)


# ============================================================
# PART B
#
# DEVELOPMENT GT READ STARTS ONLY BELOW THIS LINE
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — DEVELOPMENT ORACLE FEATURE AUDIT"
)

print(
    "============================================================"
)


oracle = pd.read_pickle(
    G3G1B_ORACLE_PATH
)


assert len(
    oracle
) == 369


SELECTED_TRUE_COL = (
    "rescue_G8_LOWCUT20"
)


assert SELECTED_TRUE_COL in oracle.columns


positive_oracle = (
    oracle[
        oracle[
            SELECTED_TRUE_COL
        ].astype(bool)
    ][
        [
            "dataset",
            "fold",
            "component_id",
            "source_id",
            "target_id",
            "pred_full_7um",
            "topology_class",
        ]
    ]
    .copy()
)


assert len(
    positive_oracle
) == 202


assert int(
    positive_oracle[
        "pred_full_7um"
    ].sum()
) == 173


print(
    "development selected true pairs:",
    len(
        positive_oracle
    )
)

print(
    "selected true full<=7:",
    int(
        positive_oracle[
            "pred_full_7um"
        ].sum()
    )
)


# ============================================================
# 19. RECONSTRUCT EXACT FEATURES FOR ALL 202 POSITIVES
# ============================================================

positive_feature_frames = []


positive_datasets = (
    positive_oracle[
        "dataset"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)


POSITIVE_COMPACT_BASE = [
    "source_id",
    "target_id",
]


for ds_i, dataset in enumerate(
    positive_datasets,
    start=1,
):

    (
        proposals,
        _,
        _,
    ) = generate_dataset_proposals(
        dataset
    )


    proposals = add_endpoint_ranks(
        proposals
    )


    proposals = add_shortlist_rank_fields(
        proposals
    )


    shortlist_mask = frozen_shortlist_mask(
        proposals
    )


    shortlist = (
        proposals[
            shortlist_mask
        ]
        .copy()
    )


    assert len(
        shortlist
    ) == expected_dataset_shortlist[
        dataset
    ]


    shortlist = attach_scoring_features(
        dataset,
        shortlist,
    )


    ds_pos = positive_oracle[
        positive_oracle[
            "dataset"
        ].eq(
            dataset
        )
    ]


    compact_columns = (
        POSITIVE_COMPACT_BASE
        +
        FEATURE_COLUMNS
    )


    assert len(
        compact_columns
    ) == len(
        set(
            compact_columns
        )
    ), (
        "Duplicate compact columns."
    )


    compact = shortlist[
        compact_columns
    ].copy()


    assert not compact.columns.duplicated().any()


    matched = ds_pos.merge(
        compact,
        on=[
            "source_id",
            "target_id",
        ],
        how="left",
        validate="one_to_one",
    )


    assert len(
        matched
    ) == len(
        ds_pos
    )


    for col in [
        "gap_size",
        "GMEAN_RATIO",
        "GM_BEST_RANK",
    ]:

        assert matched[
            col
        ].notna().all(), (
            dataset,
            col,
        )


    positive_feature_frames.append(
        matched
    )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            positive_datasets
        )
    ):

        print(
            f"{ds_i:2d}/{len(positive_datasets)}"
            f" | {dataset}"
            f" | positives={len(ds_pos)}"
        )


    del proposals
    del shortlist
    del compact
    del matched

    gc.collect()


positive_features = pd.concat(
    positive_feature_frames,
    ignore_index=True,
)


del positive_feature_frames

gc.collect()


assert len(
    positive_features
) == 202


assert not positive_features.columns.duplicated().any()


assert not positive_features[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


positive_features.to_pickle(
    POSITIVE_FEATURES_OUT
)


assert POSITIVE_FEATURES_OUT.exists()


print(
    "\nPOSITIVE_FEATURE_RECONSTRUCTION: PASS"
)

print(
    "positive feature rows:",
    len(
        positive_features
    )
)


# ============================================================
# 20. REMOVE KNOWN POSITIVES FROM HASH SAMPLE
#
# Remaining rows are called BACKGROUND, not confirmed negatives.
# ============================================================

positive_keys = positive_features[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].copy()


positive_keys[
    "__known_positive"
] = True


sample_labeled = sample_table.merge(
    positive_keys,
    on=[
        "dataset",
        "source_id",
        "target_id",
    ],
    how="left",
    validate="one_to_one",
)


sample_background = (
    sample_labeled[
        sample_labeled[
            "__known_positive"
        ].isna()
    ]
    .drop(
        columns=[
            "__known_positive",
        ]
    )
    .copy()
)


known_positive_sample_overlap = (
    len(
        sample_table
    )
    -
    len(
        sample_background
    )
)


assert known_positive_sample_overlap >= 0


print(
    "\n========== AUDIT POPULATION =========="
)

print(
    "sample before positive removal:",
    len(
        sample_table
    )
)

print(
    "known-positive rows found in sample:",
    known_positive_sample_overlap
)

print(
    "sample background rows:",
    len(
        sample_background
    )
)

print(
    "all reconstructed positives:",
    len(
        positive_features
    )
)

print(
    "background rows are confirmed negatives:",
    "NO"
)

print(
    "interpretation:",
    "known-positive vs sampled-background ranking audit"
)


# ============================================================
# 21. BUILD AUDIT POPULATION
# ============================================================

AUDIT_ID_COLUMNS = [
    "dataset",
    "fold",
    "source_id",
    "target_id",
]


negative_audit = sample_background[
    AUDIT_ID_COLUMNS
    +
    FEATURE_COLUMNS
].copy()


negative_audit[
    "label"
] = 0


negative_audit[
    "label_semantics"
] = "SAMPLED_BACKGROUND"


positive_audit = positive_features[
    AUDIT_ID_COLUMNS
    +
    FEATURE_COLUMNS
].copy()


positive_audit[
    "label"
] = 1


positive_audit[
    "label_semantics"
] = "KNOWN_ORACLE_POSITIVE"


audit_population = pd.concat(
    [
        negative_audit,
        positive_audit,
    ],
    ignore_index=True,
)


assert not audit_population.columns.duplicated().any()


assert int(
    audit_population[
        "label"
    ].sum()
) == 202


# ============================================================
# 22. AUC HELPERS
# ============================================================

def finite_feature_frame(
    frame,
    feature,
):

    values = pd.to_numeric(
        frame[
            feature
        ],
        errors="coerce",
    )


    finite = np.isfinite(
        values.to_numpy(
            dtype=float
        )
    )


    metadata = frame.loc[
        finite,
        [
            "fold",
            "label",
        ],
    ].copy()


    numeric_values = values.loc[
        finite
    ].to_numpy(
        dtype=float
    )


    return (
        metadata,
        numeric_values,
    )


def auc_or_nan(
    labels,
    values,
):

    labels = np.asarray(
        labels,
        dtype=int,
    )


    values = np.asarray(
        values,
        dtype=float,
    )


    if len(
        labels
    ) == 0:

        return np.nan


    if len(
        np.unique(
            labels
        )
    ) < 2:

        return np.nan


    return float(
        roc_auc_score(
            labels,
            values,
        )
    )


def finite_nanmean(
    values,
):

    arr = np.asarray(
        values,
        dtype=float,
    )


    finite = arr[
        np.isfinite(
            arr
        )
    ]


    if len(
        finite
    ) == 0:

        return np.nan


    return float(
        finite.mean()
    )


def finite_nanstd(
    values,
):

    arr = np.asarray(
        values,
        dtype=float,
    )


    finite = arr[
        np.isfinite(
            arr
        )
    ]


    if len(
        finite
    ) == 0:

        return np.nan


    return float(
        finite.std()
    )


def finite_nanmin(
    values,
):

    arr = np.asarray(
        values,
        dtype=float,
    )


    finite = arr[
        np.isfinite(
            arr
        )
    ]


    if len(
        finite
    ) == 0:

        return np.nan


    return float(
        finite.min()
    )


# ============================================================
# 23. CROSS-FOLD UNIVARIATE FEATURE AUDIT
# ============================================================

feature_rows = []


for feature in FEATURE_COLUMNS:

    (
        sub_metadata,
        values,
    ) = finite_feature_frame(
        audit_population,
        feature,
    )


    labels = sub_metadata[
        "label"
    ].to_numpy(
        dtype=int
    )


    raw_auc = auc_or_nan(
        labels,
        values,
    )


    if np.isnan(
        raw_auc
    ):

        direction = "UNDEFINED"

        oriented_auc = np.nan


    elif raw_auc >= 0.5:

        direction = "HIGHER_FOR_TRUE"

        oriented_auc = raw_auc


    else:

        direction = "LOWER_FOR_TRUE"

        oriented_auc = (
            1.0
            -
            raw_auc
        )


    # --------------------------------------------------------
    # Same global direction is used for every fold.
    #
    # No per-fold orientation tuning.
    # --------------------------------------------------------

    fold_aucs = []


    for fold in [
        1,
        2,
        3,
        4,
    ]:

        fold_mask = (
            sub_metadata[
                "fold"
            ].eq(
                fold
            )
            .to_numpy()
        )


        fold_labels = labels[
            fold_mask
        ]


        fold_values = values[
            fold_mask
        ]


        raw_fold_auc = auc_or_nan(
            fold_labels,
            fold_values,
        )


        if np.isnan(
            raw_fold_auc
        ):

            oriented_fold_auc = np.nan


        elif direction == "HIGHER_FOR_TRUE":

            oriented_fold_auc = raw_fold_auc


        elif direction == "LOWER_FOR_TRUE":

            oriented_fold_auc = (
                1.0
                -
                raw_fold_auc
            )


        else:

            oriented_fold_auc = np.nan


        fold_aucs.append(
            oriented_fold_auc
        )


    n_defined_folds = int(
        np.isfinite(
            np.asarray(
                fold_aucs,
                dtype=float,
            )
        ).sum()
    )


    # --------------------------------------------------------
    # Positive / background distributions
    # --------------------------------------------------------

    numeric_all = pd.to_numeric(
        audit_population[
            feature
        ],
        errors="coerce",
    )


    finite_all = np.isfinite(
        numeric_all.to_numpy(
            dtype=float
        )
    )


    positive_mask = (
        audit_population[
            "label"
        ].eq(
            1
        )
        .to_numpy()
        &
        finite_all
    )


    background_mask = (
        audit_population[
            "label"
        ].eq(
            0
        )
        .to_numpy()
        &
        finite_all
    )


    pos_values = numeric_all.loc[
        positive_mask
    ].astype(
        float
    )


    background_values = numeric_all.loc[
        background_mask
    ].astype(
        float
    )


    feature_rows.append(
        {
            "feature":
                feature,

            "direction":
                direction,

            "overall_auc":
                oriented_auc,

            "fold1_auc":
                fold_aucs[
                    0
                ],

            "fold2_auc":
                fold_aucs[
                    1
                ],

            "fold3_auc":
                fold_aucs[
                    2
                ],

            "fold4_auc":
                fold_aucs[
                    3
                ],

            "defined_fold_count":
                n_defined_folds,

            "mean_fold_auc":
                finite_nanmean(
                    fold_aucs
                ),

            "worst_fold_auc":
                finite_nanmin(
                    fold_aucs
                ),

            "fold_auc_std":
                finite_nanstd(
                    fold_aucs
                ),

            "positive_finite_n":
                int(
                    len(
                        pos_values
                    )
                ),

            "background_finite_n":
                int(
                    len(
                        background_values
                    )
                ),

            "positive_median":
                (
                    float(
                        pos_values.median()
                    )
                    if len(
                        pos_values
                    )
                    else np.nan
                ),

            "background_median":
                (
                    float(
                        background_values.median()
                    )
                    if len(
                        background_values
                    )
                    else np.nan
                ),

            "positive_q25":
                (
                    float(
                        pos_values.quantile(
                            0.25
                        )
                    )
                    if len(
                        pos_values
                    )
                    else np.nan
                ),

            "positive_q75":
                (
                    float(
                        pos_values.quantile(
                            0.75
                        )
                    )
                    if len(
                        pos_values
                    )
                    else np.nan
                ),

            "background_q25":
                (
                    float(
                        background_values.quantile(
                            0.25
                        )
                    )
                    if len(
                        background_values
                    )
                    else np.nan
                ),

            "background_q75":
                (
                    float(
                        background_values.quantile(
                            0.75
                        )
                    )
                    if len(
                        background_values
                    )
                    else np.nan
                ),
        }
    )


feature_audit = pd.DataFrame(
    feature_rows
)


feature_audit = (
    feature_audit
    .sort_values(
        [
            "defined_fold_count",
            "worst_fold_auc",
            "overall_auc",
        ],
        ascending=[
            False,
            False,
            False,
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n========== CROSS-FOLD UNIVARIATE FEATURE AUDIT =========="
)


print(
    feature_audit.to_string(
        index=False
    )
)


# ============================================================
# 24. STABLE UNIVARIATE SIGNALS
#
# DIAGNOSTIC ONLY.
#
# This DOES NOT freeze a feature subset.
# ============================================================

stable_signal = feature_audit[
    (
        feature_audit[
            "defined_fold_count"
        ]
        ==
        4
    )
    &
    (
        feature_audit[
            "overall_auc"
        ]
        >=
        0.60
    )
    &
    (
        feature_audit[
            "worst_fold_auc"
        ]
        >=
        0.55
    )
].copy()


print(
    "\n========== STABLE UNIVARIATE SIGNALS (DIAGNOSTIC ONLY) =========="
)


if len(
    stable_signal
):

    print(
        stable_signal[
            [
                "feature",
                "direction",

                "overall_auc",
                "fold1_auc",
                "fold2_auc",
                "fold3_auc",
                "fold4_auc",

                "worst_fold_auc",
                "fold_auc_std",

                "positive_finite_n",
                "background_finite_n",

                "positive_median",
                "background_median",
            ]
        ].to_string(
            index=False
        )
    )


else:

    print(
        "No feature satisfies:"
    )

    print(
        "  defined folds = 4"
    )

    print(
        "  overall AUC >= 0.60"
    )

    print(
        "  worst-fold AUC >= 0.55"
    )


# ============================================================
# 25. TOP SIGNALS BY OVERALL AUC
# ============================================================

print(
    "\n========== TOP FEATURES BY OVERALL AUC =========="
)


top_overall = (
    feature_audit[
        feature_audit[
            "overall_auc"
        ].notna()
    ]
    .sort_values(
        "overall_auc",
        ascending=False,
        kind="stable",
    )
    .head(
        15
    )
)


print(
    top_overall[
        [
            "feature",
            "direction",
            "overall_auc",
            "worst_fold_auc",
            "defined_fold_count",
            "fold_auc_std",
            "positive_median",
            "background_median",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 26. PERSIST FORMAL AUDIT
# ============================================================

feature_audit.to_pickle(
    AUDIT_OUT
)


assert AUDIT_OUT.exists()


summary = {
    "stage":
        "12.11G3H0",

    "status":
        "CROSS_FOLD_SCORING_FEATURE_AUDIT_COMPLETE",

    "resume_mode":
        "VALID_SCHEMA_ONLY_RESUME",

    "parent_generator_id":
        parent_generator[
            "generator_id"
        ],

    "parent_generator_sha256":
        GENERATOR_SHA,

    "parent_shortlist_id":
        frozen_shortlist[
            "shortlist_id"
        ],

    "parent_shortlist_sha256":
        SHORTLIST_SHA,

    "feature_schema_id":
        FEATURE_SCHEMA[
            "schema_id"
        ],

    "feature_schema_sha256":
        FEATURE_SCHEMA_SHA,

    "part_a_sha256":
        PART_A_SHA,

    "raw_generator_rows":
        int(
            raw_total
        ),

    "post_dominance_generator_rows":
        int(
            retained_total
        ),

    "shortlist_rows":
        EXPECTED_SHORTLIST_ROWS,

    "hash_sample_rows_before_positive_removal":
        int(
            len(
                sample_table
            )
        ),

    "known_positive_sample_overlap":
        int(
            known_positive_sample_overlap
        ),

    "background_audit_rows":
        int(
            len(
                sample_background
            )
        ),

    "positive_rows":
        202,

    "positive_full7_rows":
        173,

    "background_label_semantics":
        (
            "SAMPLED_SHORTLIST_BACKGROUND_NOT_CONFIRMED_NEGATIVE"
        ),

    "auc_semantics":
        (
            "KNOWN_DEVELOPMENT_ORACLE_POSITIVE_VS_"
            "GT_BLIND_HASH_SAMPLED_BACKGROUND_RANKING_AUC"
        ),

    "feature_audit":
        feature_audit.to_dict(
            "records"
        ),

    "stable_signal_diagnostic":
        stable_signal[
            [
                "feature",
                "direction",
                "overall_auc",
                "worst_fold_auc",
                "fold_auc_std",
            ]
        ].to_dict(
            "records"
        ),

    "sampling_gt_blind":
        True,

    "all_selected_positives_reconstructed":
        True,

    "feature_subset_selected":
        False,

    "classifier_trained":
        False,

    "scoring_threshold_selected":
        False,

    "rewire_policy_selected":
        False,

    "fold0_used":
        False,

    "graph_modified":
        False,
}


SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert SUMMARY_OUT.exists()


# ============================================================
# 27. RELOAD INTEGRITY
# ============================================================

check_schema = json.loads(
    FEATURE_SCHEMA_OUT.read_text(
        encoding="utf-8"
    )
)


assert check_schema[
    "feature_schema_sha256"
] == FEATURE_SCHEMA_SHA


assert check_schema[
    "feature_schema_sha256"
] == KNOWN_SCHEMA_SHA


check_sample = pd.read_pickle(
    SAMPLE_OUT
)


assert len(
    check_sample
) == len(
    sample_table
)


assert not check_sample.columns.duplicated().any()


check_part_a = json.loads(
    PARTA_SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert check_part_a[
    "shortlist_rows"
] == EXPECTED_SHORTLIST_ROWS


assert check_part_a[
    "gt_used"
] is False


check_positive = pd.read_pickle(
    POSITIVE_FEATURES_OUT
)


assert len(
    check_positive
) == 202


assert not check_positive.columns.duplicated().any()


check_audit = pd.read_pickle(
    AUDIT_OUT
)


assert len(
    check_audit
) == len(
    FEATURE_COLUMNS
)


check_summary = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert check_summary[
    "positive_rows"
] == 202


assert check_summary[
    "fold0_used"
] is False


# ============================================================
# 28. DECISION
# ============================================================

print(
    "\n========== 12.11G3H0 DECISION =========="
)

print(
    "CROSS_FOLD_SCORING_FEATURE_AUDIT: PASS"
)

print(
    "parent generator SHA:",
    GENERATOR_SHA
)

print(
    "parent shortlist SHA:",
    SHORTLIST_SHA
)

print(
    "feature schema SHA:",
    FEATURE_SCHEMA_SHA
)

print(
    "existing feature schema reused:",
    "YES"
)

print(
    "existing feature schema rewritten:",
    "NO"
)

print(
    "raw generator reproduced:",
    raw_total,
    "/ 36473971"
)

print(
    "post-dominance generator reproduced:",
    retained_total,
    "/ 29227030"
)

print(
    "shortlist cardinality reproduced:",
    shortlist_total,
    "/",
    EXPECTED_SHORTLIST_ROWS
)

print(
    "GT-blind deterministic sample rows:",
    len(
        sample_table
    )
)

print(
    "sample background rows:",
    len(
        sample_background
    )
)

print(
    "positive feature rows reconstructed:",
    len(
        positive_features
    )
)

print(
    "stable diagnostic signals:",
    len(
        stable_signal
    )
)

print(
    "background treated as confirmed negatives:",
    "NO"
)

print(
    "feature subset selected:",
    "NO"
)

print(
    "classifier trained:",
    "NO"
)

print(
    "scoring threshold selected:",
    "NO"
)

print(
    "rewire policy selected:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "graph modified:",
    "NO"
)

print(
    "next:"
)

print(
    "DESIGN_LEAVE_ONE_FOLD_OUT_SCORER_BENCHMARK_FROM_FROZEN_FEATURE_SCHEMA"
)

In [ ]:
# ============================================================
# 12.11G3H0-R1
#
# Exact CURRENT_BEST Builder Recovery
#
# Source:
#   retained G3E-R cell 39
#
# Purpose:
#   restore ONLY the exact CURRENT_BEST construction globals
#   and build_current_best() after kernel restart.
#
# IMPORTANT:
#
#   - Does NOT execute G3E-R streaming proposal generation.
#   - Does NOT create/delete G3E-R output directories.
#   - Does NOT read development GT.
#   - Does NOT modify frozen artifacts.
#   - Does NOT use Fold0.
#
# Exact CURRENT_BEST:
#
#   Frozen V9
#       ->
#   MORPH-DIV-V2 / K3_R3_P100
#       ->
#   MUTUAL-CONT-V1 / p>=0.25
#       ->
#   MUTUAL-CONT-V2 / p>=0.25
#       ->
#   DET-BRIDGE-R1P50
#
# Keep after running: YES
# ============================================================

from pathlib import Path

import sys

import numpy as np
import pandas as pd


# ============================================================
# 0. PROJECT PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


TRAIN_ROOT = (
    PROJECT
    / "data"
    / "raw"
    / "competition"
    / "biohub-cell-tracking-during-development"
    / "train"
)


PRED_ROOT = (
    PROJECT
    / "predictions"
    / "heqiuyan"
)


S11 = (
    PROJECT
    / "reports"
    / "stage11_division_probe"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


B256_995 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det995"
    / "split_0"
)


B256_9975 = (
    PRED_ROOT
    / "stage11_fullgraph_baseline256_folds1to4_det9975"
    / "split_0"
)


MORPH_PATH = (
    S11
    / "stage11_deployable_final_v9_policy_grid_actions.pkl"
)


V1_PATH = (
    S12
    / "stage12_folds14_mutual_best_universe.pkl"
)


V2_PATH = (
    S12
    / "stage12_folds14_round2_reciprocal_universe.pkl"
)


BRIDGE_PATH = (
    S12
    / "stage12_detection_bridge_policy_actions.pkl"
)


for path in [
    TRAIN_ROOT,
    CV_PATH,
    B256_995,
    B256_9975,
    MORPH_PATH,
    V1_PATH,
    V2_PATH,
    BRIDGE_PATH,
]:

    assert path.exists(), path


print(
    "========== CURRENT_BEST RECOVERY INPUTS =========="
)

print(
    "TRAIN_ROOT:",
    TRAIN_ROOT
)

print(
    "CV manifest:",
    CV_PATH
)

print(
    "all required artifacts exist:",
    "YES"
)


# ============================================================
# 1. PROJECT IMPORT
# ============================================================

SRC = str(
    PROJECT
    / "src"
)


if SRC not in sys.path:

    sys.path.insert(
        0,
        SRC,
    )


from biohub_cell_tracking import frozen_v9 as fv9


print(
    "\nproject import:",
    "PASS"
)


# ============================================================
# 2. DEVELOPMENT CORPUS
#
# Exact G3E-R corpus:
#   folds 1–4
#   6bba_ datasets
#   103 datasets
#
# No Fold0.
# ============================================================

cv = pd.read_csv(
    CV_PATH
)


corp = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    corp
) == 103


assert corp[
    "dataset"
].nunique() == 103


assert not corp[
    "fold"
].eq(
    0
).any()


names = corp[
    "dataset"
].tolist()


fold_map = dict(
    zip(
        corp[
            "dataset"
        ],

        corp[
            "fold"
        ].astype(
            int
        ),
    )
)


assert (
    "prefix"
    in corp.columns
), (
    "cv_manifest.csv no longer contains the prefix column "
    "required by the authoritative Frozen V9 setup."
)


prefix_map = dict(
    zip(
        corp[
            "dataset"
        ],

        corp[
            "prefix"
        ],
    )
)


print(
    "\n========== DEVELOPMENT CORPUS =========="
)

print(
    "datasets:",
    len(
        names
    )
)

print(
    "folds:",
    sorted(
        set(
            corp[
                "fold"
            ].astype(
                int
            )
        )
    )
)

print(
    "Fold0 used:",
    "NO"
)


# ============================================================
# 3. EXACT CURRENT_BEST ACTION CONTRACT
# ============================================================

morph_all = pd.read_pickle(
    MORPH_PATH
)


morph_actions = (
    morph_all[
        morph_all[
            "config_id"
        ].eq(
            "K3_R3_P100"
        )
    ]
    .copy()
)


v1_all = pd.read_pickle(
    V1_PATH
)


v1_actions = (
    v1_all[
        v1_all[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


v2_all = pd.read_pickle(
    V2_PATH
)


v2_actions = (
    v2_all[
        v2_all[
            "candidate_prob"
        ].ge(
            0.25
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
            "t",
            "source_pred_node_id",
            "target_pred_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


bridge_all = pd.read_pickle(
    BRIDGE_PATH
)


r1p50_actions = (
    bridge_all[
        bridge_all[
            "policy_id"
        ].eq(
            "R1P50"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ============================================================
# 4. HARD CONTRACT COUNTS
# ============================================================

assert len(
    morph_actions
) == 96, len(
    morph_actions
)


assert len(
    v1_actions
) == 46_951, len(
    v1_actions
)


assert len(
    v2_actions
) == 18_511, len(
    v2_actions
)


assert len(
    r1p50_actions
) == 805, len(
    r1p50_actions
)


print(
    "\n========== G3E-R CURRENT_BEST CONTRACT =========="
)

print(
    "datasets:",
    len(
        names
    )
)

print(
    "MORPH K3_R3_P100:",
    len(
        morph_actions
    )
)

print(
    "MUTUAL-CONT-V1:",
    len(
        v1_actions
    )
)

print(
    "MUTUAL-CONT-V2:",
    len(
        v2_actions
    )
)

print(
    "DET-BRIDGE-R1P50:",
    len(
        r1p50_actions
    )
)

print(
    "ACTION CONTRACT COUNTS: PASS"
)


# ============================================================
# 5. FROZEN V9 POLICIES
#
# This reconstructs the deterministic Frozen V9 policy object
# required by fv9.apply_frozen_v9().
#
# No GT.
# ============================================================

policies = fv9.build_frozen_v9_policies(
    names,
    B256_995,
    B256_9975,
    TRAIN_ROOT,
    prefix_map=prefix_map,
    config=fv9.FROZEN_V9_CONFIG,
    progress=False,
)


assert (
    type(
        policies
    ).__name__
    ==
    "FrozenV9Policies"
)


print(
    "\n========== FROZEN V9 POLICIES =========="
)

print(
    "type:",
    type(
        policies
    ).__name__
)

print(
    "FROZEN V9 POLICY RECONSTRUCTION: PASS"
)


# ============================================================
# 6. EXACT GRAPH STATE HELPER
# ============================================================

def graph_state(
    graph,
):

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )


    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(
            int
        )
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(
            int
        )
        .to_dict()
    )


    outdeg = {
        int(
            k
        ):
            int(
                v
            )

        for k, v
        in outdeg.items()
    }


    indeg = {
        int(
            k
        ):
            int(
                v
            )

        for k, v
        in indeg.items()
    }


    edge_set = set(
        zip(
            edges[
                "source_id"
            ].astype(
                int
            ),

            edges[
                "target_id"
            ].astype(
                int
            ),
        )
    )


    return (
        nodes,
        outdeg,
        indeg,
        edge_set,
    )


# ============================================================
# 7. EXACT CONTINUATION ACTION HELPER
# ============================================================

def add_continuation_actions(
    graph,
    nodes,
    table,
    dataset,
):

    rows = table[
        table[
            "dataset"
        ].eq(
            dataset
        )
    ]


    for action in rows.itertuples(
        index=False
    ):

        src = int(
            action.source_pred_node_id
        )


        tgt = int(
            action.target_pred_node_id
        )


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(
                        dist
                    ),

                "edge_prob":
                    float(
                        action.candidate_prob
                    ),
            },
            validate_keys=True,
        )


# ============================================================
# 8. EXACT CURRENT_BEST BUILDER
# ============================================================

def build_current_best(
    dataset,
):

    assert dataset in fold_map, dataset


    graph, scale, _ = fv9.apply_frozen_v9(
        dataset,
        B256_995,
        B256_9975,
        policies,
        TRAIN_ROOT,
        config=fv9.FROZEN_V9_CONFIG,
    )


    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = graph_state(
        graph
    )


    # ========================================================
    # MORPH-DIV-V2
    #
    # Frozen contract:
    #     K3_R3_P100
    # ========================================================

    rows = (
        morph_actions[
            morph_actions[
                "dataset"
            ].eq(
                dataset
            )
        ]
        .sort_values(
            [
                "morph_rank",
                "t",
                "source_pred_node_id",
                "target_pred_node_id",
            ],
            kind="stable",
        )
    )


    for action in rows.itertuples(
        index=False
    ):

        src = int(
            action.source_pred_node_id
        )


        tgt = int(
            action.target_pred_node_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 1


        assert indeg.get(
            tgt,
            0,
        ) == 0


        assert (
            src,
            tgt,
        ) not in edge_set


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(
                        dist
                    ),

                "edge_prob":
                    float(
                        action.candidate_prob
                    ),
            },
            validate_keys=True,
        )


        outdeg[
            src
        ] = 2


        indeg[
            tgt
        ] = 1


        edge_set.add(
            (
                src,
                tgt,
            )
        )


    # ========================================================
    # MUTUAL-CONT-V1
    # ========================================================

    add_continuation_actions(
        graph,
        nodes,
        v1_actions,
        dataset,
    )


    # ========================================================
    # MUTUAL-CONT-V2
    # ========================================================

    add_continuation_actions(
        graph,
        nodes,
        v2_actions,
        dataset,
    )


    # V1/V2 changed graph edges.
    # Refresh exact topology before R1P50.
    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = graph_state(
        graph
    )


    # ========================================================
    # DET-BRIDGE-R1P50
    # ========================================================

    rows = r1p50_actions[
        r1p50_actions[
            "dataset"
        ].eq(
            dataset
        )
    ]


    for action in rows.itertuples(
        index=False
    ):

        src = int(
            action.source_id
        )


        tgt = int(
            action.target_id
        )


        assert outdeg.get(
            src,
            0,
        ) == 0


        assert indeg.get(
            tgt,
            0,
        ) == 0


        new_id = graph.add_node(
            {
                "t":
                    int(
                        action.t
                    ),

                "z":
                    float(
                        action.z
                    ),

                "y":
                    float(
                        action.y
                    ),

                "x":
                    float(
                        action.x
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            src,
            new_id,
            {
                "edge_dist":
                    float(
                        action.src_dist_um
                    ),

                "edge_prob":
                    float(
                        action.in_prob
                    ),
            },
            validate_keys=True,
        )


        graph.add_edge(
            new_id,
            tgt,
            {
                "edge_dist":
                    float(
                        action.tgt_dist_um
                    ),

                "edge_prob":
                    float(
                        action.out_prob
                    ),
            },
            validate_keys=True,
        )


        outdeg[
            src
        ] = 1


        indeg[
            new_id
        ] = 1


        outdeg[
            new_id
        ] = 1


        indeg[
            tgt
        ] = 1


    return (
        graph,
        scale,
    )


print(
    "\nSELF_CONTAINED_CURRENT_BEST_BUILDER: PASS"
)


# ============================================================
# 9. FUNCTION DEPENDENCY GATE
# ============================================================

required_globals = [
    "B256_995",
    "B256_9975",
    "TRAIN_ROOT",

    "fv9",

    "graph_state",
    "add_continuation_actions",

    "morph_actions",
    "v1_actions",
    "v2_actions",
    "r1p50_actions",

    "policies",
]


missing_globals = [
    name

    for name in required_globals

    if name not in globals()
]


assert not missing_globals, missing_globals


assert callable(
    build_current_best
)


print(
    "\n========== BUILDER DEPENDENCY RECOVERY =========="
)

print(
    "required globals:",
    len(
        required_globals
    )
)

print(
    "missing globals:",
    missing_globals
)

print(
    "DEPENDENCY RECOVERY: PASS"
)


# ============================================================
# 10. ONE-DATASET FUNCTIONAL SMOKE TEST
#
# No artifact written.
# ============================================================

TEST_DATASET = (
    "6bba_05b6850b"
)


test_graph, test_scale = build_current_best(
    TEST_DATASET
)


test_scale = np.asarray(
    test_scale,
    dtype=float,
)


EXPECTED_SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


assert np.allclose(
    test_scale,
    EXPECTED_SCALE,
    atol=1e-12,
    rtol=0.0,
)


test_nodes = (
    test_graph.node_attrs(
        attr_keys=[
            "node_id",
            "t",
            "z",
            "y",
            "x",
        ]
    )
    .to_pandas()
)


test_edges = (
    test_graph.edge_attrs(
        unpack=True
    )
    .to_pandas()
)


assert len(
    test_nodes
) > 0


assert len(
    test_edges
) > 0


assert {
    "node_id",
    "t",
    "z",
    "y",
    "x",
}.issubset(
    test_nodes.columns
)


assert {
    "source_id",
    "target_id",
}.issubset(
    test_edges.columns
)


print(
    "\n========== CURRENT_BEST SMOKE TEST =========="
)

print(
    "dataset:",
    TEST_DATASET
)

print(
    "nodes:",
    len(
        test_nodes
    )
)

print(
    "edges:",
    len(
        test_edges
    )
)

print(
    "scale:",
    test_scale.tolist()
)

print(
    "FUNCTIONAL SMOKE TEST: PASS"
)


# ============================================================
# 11. G3H0 ARTIFACT SAFETY
# ============================================================

G3H0_SCHEMA = (
    S12
    / "stage12_folds14_rewire_scoring_feature_schema_v1_frozen.json"
)


G3H0_DOWNSTREAM = [
    S12
    / "stage12_folds14_rewire_scoring_feature_sample_hash64_v1.pkl",

    S12
    / "stage12_folds14_rewire_scoring_feature_audit_parta_v1.json",

    S12
    / "stage12_folds14_rewire_scoring_positive_features_v1.pkl",

    S12
    / "stage12_folds14_rewire_scoring_feature_audit_v1.pkl",

    S12
    / "stage12_folds14_rewire_scoring_feature_audit_v1_summary.json",
]


assert G3H0_SCHEMA.exists(), (
    "Expected already-frozen G3H0 schema is missing."
)


existing_downstream = [
    str(
        path
    )

    for path in G3H0_DOWNSTREAM

    if path.exists()
]


assert not existing_downstream, (
    "Unexpected G3H0 downstream artifacts exist:\n"
    +
    "\n".join(
        existing_downstream
    )
)


print(
    "\n========== G3H0 SAFETY STATE =========="
)

print(
    "existing frozen schema:",
    "YES"
)

print(
    "G3H0 downstream artifacts:",
    "NONE"
)

print(
    "GT read:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "project artifacts modified:",
    "NO"
)


# ============================================================
# 12. FINAL DECISION
# ============================================================

print(
    "\n========== 12.11G3H0-R1 DECISION =========="
)

print(
    "EXACT_CURRENT_BEST_BUILDER_RECOVERY: PASS"
)

print(
    "source provenance:",
    "G3E-R retained cell 39"
)

print(
    "CURRENT_BEST action contract:",
    "PASS"
)

print(
    "Frozen V9 policies:",
    "PASS"
)

print(
    "builder dependencies:",
    "PASS"
)

print(
    "functional smoke test:",
    "PASS"
)

print(
    "G3H0 downstream artifacts:",
    "NONE"
)

print(
    "SAFE_TO_RUN_COMPLETE_G3H0:",
    "YES"
)

In [ ]:
# ============================================================
# Stage 12.11G3H0
#
# Cross-Fold Scoring Feature Audit
# RESUME-SAFE COMPLETE VERSION
#
# Parent generator:
#   MULTIFRAME_REWIRE_ASYM3X8_DOMINANCE_ANY_V1_FIDELITYPATCH1
#
# Frozen scorer-input shortlist:
#   ENDPOINT_SHORTLIST_GMEAN_G8_LOWCUT20_V1
#
# Expected:
#   parent generator raw      = 36,473,971
#   parent generator retained = 29,227,030
#   frozen shortlist          = 17,501,674
#
# Existing frozen feature schema:
#   SHA =
#   0a9f8e9a41c5c80c803c3a95955702dc
#   f4039605a21fc656a154aa05be55b39c
#
# ------------------------------------------------------------
# PART A
#   GT blind
#   exact regeneration
#   frozen shortlist
#   deterministic 1/64 background sample
#
# PART B
#   only after Part-A freeze
#   development oracle read
#   202 positive feature reconstruction
#   univariate cross-fold feature audit
#
# NO:
#   Fold0
#   classifier
#   feature subset selection
#   score threshold
#   rewire policy
#   graph modification
#
# Keep after running: YES
# ============================================================

from pathlib import Path

import gc
import hashlib
import json
import sys

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from sklearn.metrics import roc_auc_score


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


FROZEN_GENERATOR_PATH = (
    S12
    / "stage12_folds14_multiframe_rewire_generator_asym3x8_v1_fidelitypatch1_frozen.json"
)


FROZEN_SHORTLIST_PATH = (
    S12
    / "stage12_folds14_endpoint_shortlist_g8_lowcut20_v1_frozen.json"
)


G3G1B_CARDINALITY_PATH = (
    S12
    / "stage12_folds14_endpoint_competition_targeted_rescue_cardinality_v1.json"
)


# GT — existence may be checked here.
# Contents MUST NOT be read until Part A frozen.
G3G1B_ORACLE_PATH = (
    S12
    / "stage12_folds14_endpoint_competition_targeted_rescue_oracle_v1.pkl"
)


FEATURE_SCHEMA_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_feature_schema_v1_frozen.json"
)


SAMPLE_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_feature_sample_hash64_v1.pkl"
)


PARTA_SUMMARY_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_feature_audit_parta_v1.json"
)


POSITIVE_FEATURES_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_positive_features_v1.pkl"
)


AUDIT_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_feature_audit_v1.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_folds14_rewire_scoring_feature_audit_v1_summary.json"
)


for path in [
    FROZEN_GENERATOR_PATH,
    FROZEN_SHORTLIST_PATH,
    G3G1B_CARDINALITY_PATH,
    G3G1B_ORACLE_PATH,
    FEATURE_SCHEMA_OUT,
]:

    assert path.exists(), path


# ============================================================
# 1. RESUME / OVERWRITE SAFETY
#
# Existing schema is expected.
#
# No later G3H0 artifact may exist.
# ============================================================

for path in [
    SAMPLE_OUT,
    PARTA_SUMMARY_OUT,
    POSITIVE_FEATURES_OUT,
    AUDIT_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nExisting downstream G3H0 artifact:\n"
            f"{path}\n\n"
            "Do not overwrite it."
        )


# ============================================================
# 2. CURRENT_BEST MUST ALREADY BE RECOVERED
# ============================================================

assert (
    "build_current_best"
    in globals()
    and
    callable(
        build_current_best
    )
), (
    "build_current_best is missing. "
    "Run the successful G3H0-R1 builder recovery cell first."
)


print(
    "========== G3H0 CURRENT_BEST DEPENDENCY =========="
)

print(
    "build_current_best:",
    "AVAILABLE"
)


# ============================================================
# 3. FIXED CONTRACTS
# ============================================================

GENERATOR_SHA = (
    "da408df857cd34dcb4b1f65f3fae545e"
    "299f9403ef5edb986584caa8f4c3c203"
)


SHORTLIST_SHA = (
    "ae61216904bc3187f76b238b4b560d669"
    "3a7eb4f5d07bb8ef19021cc1bca1490"
)


FEATURE_SCHEMA_SHA = (
    "0a9f8e9a41c5c80c803c3a95955702dc"
    "f4039605a21fc656a154aa05be55b39c"
)


EXPECTED_RAW_ROWS = 36_473_971
EXPECTED_GENERATOR_ROWS = 29_227_030
EXPECTED_SHORTLIST_ROWS = 17_501_674


SCALE = np.array(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=float,
)


MIN_GAP_SIZE = 2
MAX_GAP_SIZE = 12

QUERY_K = 16

STRONG_RANK = 3
WEAK_RANK = 8

EPS = 1e-12


# ============================================================
# 4. VERIFY FROZEN PARENT GENERATOR
# ============================================================

parent_generator = json.loads(
    FROZEN_GENERATOR_PATH.read_text(
        encoding="utf-8"
    )
)


assert parent_generator[
    "generator_contract_sha256"
] == GENERATOR_SHA


assert parent_generator[
    "generator_id"
] == (
    "MULTIFRAME_REWIRE_ASYM3X8_"
    "DOMINANCE_ANY_V1_FIDELITYPATCH1"
)


assert int(
    parent_generator[
        "retrieval"
    ][
        "exact_knn_query_k"
    ]
) == 16


assert int(
    parent_generator[
        "retrieval"
    ][
        "strong_rank"
    ]
) == 3


assert int(
    parent_generator[
        "retrieval"
    ][
        "weak_rank"
    ]
) == 8


assert int(
    parent_generator[
        "development_cardinality"
    ][
        "retained_rows"
    ]
) == EXPECTED_GENERATOR_ROWS


# ============================================================
# 5. VERIFY FROZEN SHORTLIST
# ============================================================

frozen_shortlist = json.loads(
    FROZEN_SHORTLIST_PATH.read_text(
        encoding="utf-8"
    )
)


assert frozen_shortlist[
    "shortlist_contract_sha256"
] == SHORTLIST_SHA


assert frozen_shortlist[
    "shortlist_id"
] == (
    "ENDPOINT_SHORTLIST_GMEAN_G8_LOWCUT20_V1"
)


assert frozen_shortlist[
    "parent_generator"
][
    "generator_contract_sha256"
] == GENERATOR_SHA


assert int(
    frozen_shortlist[
        "development_cardinality"
    ][
        "rows"
    ]
) == EXPECTED_SHORTLIST_ROWS


# ============================================================
# 6. VERIFY / REUSE FROZEN FEATURE SCHEMA
# ============================================================

feature_schema = json.loads(
    FEATURE_SCHEMA_OUT.read_text(
        encoding="utf-8"
    )
)


assert feature_schema[
    "stage"
] == "12.11G3H0"


assert feature_schema[
    "schema_id"
] == (
    "MULTIFRAME_REWIRE_SCORING_FEATURES_V1"
)


assert feature_schema[
    "parent_shortlist_sha256"
] == SHORTLIST_SHA


assert feature_schema[
    "parent_generator_sha256"
] == GENERATOR_SHA


assert feature_schema[
    "runtime_features_gt_blind"
] is True


recorded_schema_sha = feature_schema[
    "feature_schema_sha256"
]


assert recorded_schema_sha == FEATURE_SCHEMA_SHA


schema_without_hash = dict(
    feature_schema
)

schema_without_hash.pop(
    "feature_schema_sha256",
    None,
)


recomputed_schema_sha = hashlib.sha256(
    json.dumps(
        schema_without_hash,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


assert recomputed_schema_sha == FEATURE_SCHEMA_SHA


FEATURE_COLUMNS = list(
    feature_schema[
        "features"
    ]
)


EXPECTED_FEATURE_COLUMNS = [
    "gap_size",
    "delta_t",

    "source_occupied",
    "target_occupied",
    "cuts",

    "source_cv_pf",
    "target_cv_pf",

    "MIN_CV",
    "SYM_CV",
    "MAX_CV",

    "GMEAN_RATIO",

    "GM_SOURCE_RANK",
    "GM_TARGET_RANK",
    "GM_BEST_RANK",
    "GM_MAX_RANK",
    "GM_RANK_SUM",
    "GM_RANK_LOG_ASYMMETRY",

    "SOURCE_COMPETITION_COUNT",
    "TARGET_COMPETITION_COUNT",

    "GM_SOURCE_RANK_FRAC",
    "GM_TARGET_RANK_FRAC",
    "GM_BEST_RANK_FRAC",

    "SYM_BEST_RANK",
    "MAX_BEST_RANK",

    "source_incumbent_cv_pf",
    "target_incumbent_cv_pf",

    "source_relative_improvement",
    "target_relative_improvement",

    "max_required_improvement",
    "min_required_improvement",
    "mean_required_improvement",

    "dominance_all",
]


assert FEATURE_COLUMNS == EXPECTED_FEATURE_COLUMNS


print(
    "\n========== FROZEN FEATURE SCHEMA =========="
)

print(
    "schema ID:",
    feature_schema[
        "schema_id"
    ]
)

print(
    "feature count:",
    len(
        FEATURE_COLUMNS
    )
)

print(
    "schema SHA:",
    FEATURE_SCHEMA_SHA
)

print(
    "existing frozen schema reused:",
    "YES"
)

print(
    "schema rewritten:",
    "NO"
)

print(
    "FROZEN BEFORE GT READ:",
    "YES"
)


# ============================================================
# 7. DEVELOPMENT CORPUS
# ============================================================

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


cv = pd.read_csv(
    CV_PATH
)


dev_manifest = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    dev_manifest
) == 103


assert dev_manifest[
    "dataset"
].nunique() == 103


assert not dev_manifest[
    "fold"
].eq(
    0
).any()


development_names = (
    dev_manifest[
        "dataset"
    ]
    .tolist()
)


fold_map = dict(
    zip(
        dev_manifest[
            "dataset"
        ],

        dev_manifest[
            "fold"
        ].astype(
            int
        ),
    )
)


# ============================================================
# 8. G3G1B GT-BLIND PER-DATASET FIDELITY
# ============================================================

g3g1b_cardinality = json.loads(
    G3G1B_CARDINALITY_PATH.read_text(
        encoding="utf-8"
    )
)


assert g3g1b_cardinality[
    "stage"
] == "12.11G3G1B"


assert g3g1b_cardinality[
    "gt_used_during_part_a"
] is False


assert int(
    g3g1b_cardinality[
        "generator_raw_rows"
    ]
) == EXPECTED_RAW_ROWS


assert int(
    g3g1b_cardinality[
        "generator_retained_rows"
    ]
) == EXPECTED_GENERATOR_ROWS


expected_dataset_shortlist = {
    row[
        "dataset"
    ]:
        int(
            row[
                "contracts"
            ][
                "G8_LOWCUT20"
            ]
        )

    for row
    in g3g1b_cardinality[
        "dataset_results"
    ]
}


assert len(
    expected_dataset_shortlist
) == 103


assert sum(
    expected_dataset_shortlist.values()
) == EXPECTED_SHORTLIST_ROWS


assert set(
    expected_dataset_shortlist
) == set(
    development_names
)


print(
    "\n========== G3H0 FROZEN INPUT =========="
)

print(
    "development datasets:",
    103
)

print(
    "generator raw expected:",
    EXPECTED_RAW_ROWS
)

print(
    "generator retained expected:",
    EXPECTED_GENERATOR_ROWS
)

print(
    "shortlist expected:",
    EXPECTED_SHORTLIST_ROWS
)

print(
    "Fold0 used:",
    "NO"
)


# ============================================================
# 9. BASIC NUMERIC HELPERS
# ============================================================

def safe_ratio(
    candidate,
    best,
):

    candidate = float(
        candidate
    )

    best = float(
        best
    )


    if best <= EPS:

        if candidate <= EPS:

            return 1.0

        return np.inf


    return float(
        candidate
        /
        best
    )


def query_knn(
    query_points,
    candidate_points,
    k,
):

    query_points = np.asarray(
        query_points,
        dtype=float,
    )


    candidate_points = np.asarray(
        candidate_points,
        dtype=float,
    )


    if (
        len(
            query_points
        ) == 0
        or
        len(
            candidate_points
        ) == 0
    ):

        return (
            np.empty(
                (
                    len(
                        query_points
                    ),
                    0,
                ),
                dtype=float,
            ),

            np.empty(
                (
                    len(
                        query_points
                    ),
                    0,
                ),
                dtype=int,
            ),
        )


    k_eff = min(
        int(
            k
        ),
        int(
            len(
                candidate_points
            )
        ),
    )


    tree = cKDTree(
        candidate_points
    )


    distances, indices = tree.query(
        query_points,
        k=k_eff,
    )


    if k_eff == 1:

        distances = distances[
            :,
            None
        ]

        indices = indices[
            :,
            None
        ]


    return (
        np.asarray(
            distances,
            dtype=float,
        ),

        np.asarray(
            indices,
            dtype=int,
        ),
    )


def make_rank_map(
    query_ids,
    candidate_ids,
    distances,
    indices,
    reverse_pair=False,
):

    result = {}


    for query_i, query_id in enumerate(
        query_ids
    ):

        for rank_i in range(
            indices.shape[
                1
            ]
        ):

            candidate_id = int(
                candidate_ids[
                    indices[
                        query_i,
                        rank_i
                    ]
                ]
            )


            if reverse_pair:

                pair = (
                    candidate_id,
                    int(
                        query_id
                    ),
                )

            else:

                pair = (
                    int(
                        query_id
                    ),
                    candidate_id,
                )


            if pair not in result:

                result[
                    pair
                ] = (
                    int(
                        rank_i
                        +
                        1
                    ),

                    float(
                        distances[
                            query_i,
                            rank_i
                        ]
                    ),
                )


    return result


def asym_3x8(
    source_rank,
    target_rank,
):

    return bool(
        (
            source_rank <= 3
            and
            target_rank <= 8
        )
        or
        (
            target_rank <= 3
            and
            source_rank <= 8
        )
    )


# ============================================================
# 10. AUTHORITATIVE DATASET GENERATOR
#
# CURRENT_BEST
#   -> exact K16
#   -> ASYM3x8
#   -> DOMINANCE_ANY
#
# GT BLIND.
# ============================================================

def generate_dataset_proposals(
    dataset,
):

    graph, scale = build_current_best(
        dataset
    )


    scale = np.asarray(
        scale,
        dtype=float,
    )


    assert np.allclose(
        scale,
        SCALE,
        atol=1e-12,
        rtol=0.0,
    )


    # --------------------------------------------------------
    # Nodes
    # --------------------------------------------------------

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .copy()
    )


    nodes[
        "node_id"
    ] = nodes[
        "node_id"
    ].astype(
        int
    )


    nodes[
        "t"
    ] = nodes[
        "t"
    ].astype(
        int
    )


    nodes = (
        nodes
        .sort_values(
            [
                "t",
                "node_id",
            ],
            kind="stable",
        )
        .set_index(
            "node_id",
            drop=False,
        )
    )


    # --------------------------------------------------------
    # Edges
    # --------------------------------------------------------

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "source_id"
    ] = edges[
        "source_id"
    ].astype(
        int
    )


    edges[
        "target_id"
    ] = edges[
        "target_id"
    ].astype(
        int
    )


    edge_set = set(
        zip(
            edges[
                "source_id"
            ],
            edges[
                "target_id"
            ],
        )
    )


    incoming = (
        edges.groupby(
            "target_id"
        )[
            "source_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    outgoing = (
        edges.groupby(
            "source_id"
        )[
            "target_id"
        ]
        .apply(
            list
        )
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(
            int
        )
        .to_dict()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(
            int
        )
        .to_dict()
    )


    indeg = {
        int(k):
            int(v)

        for k, v
        in indeg.items()
    }


    outdeg = {
        int(k):
            int(v)

        for k, v
        in outdeg.items()
    }


    # --------------------------------------------------------
    # Physical state
    # --------------------------------------------------------

    phys = {}

    time_map = {}


    for nid, row in nodes.iterrows():

        nid = int(
            nid
        )


        phys[
            nid
        ] = (
            np.array(
                [
                    float(
                        row[
                            "z"
                        ]
                    ),
                    float(
                        row[
                            "y"
                        ]
                    ),
                    float(
                        row[
                            "x"
                        ]
                    ),
                ],
                dtype=float,
            )
            *
            SCALE
        )


        time_map[
            nid
        ] = int(
            row[
                "t"
            ]
        )


    node_ids = sorted(
        phys
    )


    # --------------------------------------------------------
    # Past / future velocities
    # --------------------------------------------------------

    past_velocity = {}

    future_velocity = {}


    for nid in node_ids:

        inc = incoming.get(
            nid,
            [],
        )


        if len(
            inc
        ) == 1:

            prev_id = int(
                inc[
                    0
                ]
            )


            if prev_id in phys:

                dt = (
                    time_map[
                        nid
                    ]
                    -
                    time_map[
                        prev_id
                    ]
                )


                if dt > 0:

                    past_velocity[
                        nid
                    ] = (
                        phys[
                            nid
                        ]
                        -
                        phys[
                            prev_id
                        ]
                    ) / float(
                        dt
                    )


        out = outgoing.get(
            nid,
            [],
        )


        if len(
            out
        ) == 1:

            next_id = int(
                out[
                    0
                ]
            )


            if next_id in phys:

                dt = (
                    time_map[
                        next_id
                    ]
                    -
                    time_map[
                        nid
                    ]
                )


                if dt > 0:

                    future_velocity[
                        nid
                    ] = (
                        phys[
                            next_id
                        ]
                        -
                        phys[
                            nid
                        ]
                    ) / float(
                        dt
                    )


    # --------------------------------------------------------
    # Eligible endpoints
    # --------------------------------------------------------

    source_eligible = [
        nid

        for nid
        in node_ids

        if (
            outdeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid in past_velocity
        )
    ]


    target_eligible = [
        nid

        for nid
        in node_ids

        if (
            indeg.get(
                nid,
                0,
            )
            <=
            1
            and
            nid in future_velocity
        )
    ]


    # --------------------------------------------------------
    # Incumbent normalized CV costs
    # --------------------------------------------------------

    source_incumbent = {}

    target_incumbent = {}


    for src in source_eligible:

        if outdeg.get(
            src,
            0,
        ) != 1:

            continue


        current_out = outgoing.get(
            src,
            [],
        )


        assert len(
            current_out
        ) == 1


        old_tgt = int(
            current_out[
                0
            ]
        )


        dt_old = (
            time_map[
                old_tgt
            ]
            -
            time_map[
                src
            ]
        )


        if dt_old <= 0:

            continue


        predicted_target = (
            phys[
                src
            ]
            +
            float(
                dt_old
            )
            *
            past_velocity[
                src
            ]
        )


        source_incumbent[
            src
        ] = (
            float(
                np.linalg.norm(
                    phys[
                        old_tgt
                    ]
                    -
                    predicted_target
                )
            )
            /
            float(
                dt_old
            )
        )


    for tgt in target_eligible:

        if indeg.get(
            tgt,
            0,
        ) != 1:

            continue


        current_in = incoming.get(
            tgt,
            [],
        )


        assert len(
            current_in
        ) == 1


        old_src = int(
            current_in[
                0
            ]
        )


        dt_old = (
            time_map[
                tgt
            ]
            -
            time_map[
                old_src
            ]
        )


        if dt_old <= 0:

            continue


        predicted_source = (
            phys[
                tgt
            ]
            -
            float(
                dt_old
            )
            *
            future_velocity[
                tgt
            ]
        )


        target_incumbent[
            tgt
        ] = (
            float(
                np.linalg.norm(
                    phys[
                        old_src
                    ]
                    -
                    predicted_source
                )
            )
            /
            float(
                dt_old
            )
        )


    # --------------------------------------------------------
    # Group eligible endpoints by time
    # --------------------------------------------------------

    source_by_t = {}

    target_by_t = {}


    for nid in source_eligible:

        source_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for nid in target_eligible:

        target_by_t.setdefault(
            time_map[
                nid
            ],
            [],
        ).append(
            nid
        )


    for t in source_by_t:

        source_by_t[
            t
        ] = sorted(
            source_by_t[
                t
            ]
        )


    for t in target_by_t:

        target_by_t[
            t
        ] = sorted(
            target_by_t[
                t
            ]
        )


    # --------------------------------------------------------
    # Proposal generation
    # --------------------------------------------------------

    proposal_rows = []

    raw_count = 0

    retained_count = 0


    for gap_size in range(
        MIN_GAP_SIZE,
        MAX_GAP_SIZE + 1,
    ):

        delta_t = (
            gap_size
            +
            1
        )


        for source_t in sorted(
            source_by_t
        ):

            target_t = (
                source_t
                +
                delta_t
            )


            if target_t not in target_by_t:

                continue


            source_ids = source_by_t[
                source_t
            ]


            target_ids = target_by_t[
                target_t
            ]


            if (
                len(
                    source_ids
                ) == 0
                or
                len(
                    target_ids
                ) == 0
            ):

                continue


            source_pos = np.vstack(
                [
                    phys[
                        nid
                    ]

                    for nid
                    in source_ids
                ]
            )


            target_pos = np.vstack(
                [
                    phys[
                        nid
                    ]

                    for nid
                    in target_ids
                ]
            )


            source_vel = np.vstack(
                [
                    past_velocity[
                        nid
                    ]

                    for nid
                    in source_ids
                ]
            )


            target_vel = np.vstack(
                [
                    future_velocity[
                        nid
                    ]

                    for nid
                    in target_ids
                ]
            )


            # ------------------------------------------------
            # K16 forward CV
            # ------------------------------------------------

            source_pred = (
                source_pos
                +
                float(
                    delta_t
                )
                *
                source_vel
            )


            (
                source_dist,
                source_idx,
            ) = query_knn(
                source_pred,
                target_pos,
                QUERY_K,
            )


            source_map = make_rank_map(
                source_ids,
                target_ids,
                source_dist,
                source_idx,
                reverse_pair=False,
            )


            # ------------------------------------------------
            # K16 backward CV
            # ------------------------------------------------

            target_pred = (
                target_pos
                -
                float(
                    delta_t
                )
                *
                target_vel
            )


            (
                target_dist,
                target_idx,
            ) = query_knn(
                target_pred,
                source_pos,
                QUERY_K,
            )


            target_map = make_rank_map(
                target_ids,
                source_ids,
                target_dist,
                target_idx,
                reverse_pair=True,
            )


            # ------------------------------------------------
            # Local best residuals
            # ------------------------------------------------

            source_best = {
                int(
                    source_ids[
                        i
                    ]
                ):
                    float(
                        source_dist[
                            i,
                            0
                        ]
                    )

                for i
                in range(
                    len(
                        source_ids
                    )
                )
            }


            target_best = {
                int(
                    target_ids[
                        i
                    ]
                ):
                    float(
                        target_dist[
                            i,
                            0
                        ]
                    )

                for i
                in range(
                    len(
                        target_ids
                    )
                )
            }


            reciprocal_pairs = (
                set(
                    source_map
                )
                &
                set(
                    target_map
                )
            )


            for (
                src,
                tgt,
            ) in sorted(
                reciprocal_pairs
            ):

                if (
                    src,
                    tgt,
                ) in edge_set:

                    continue


                (
                    source_rank,
                    source_residual,
                ) = source_map[
                    (
                        src,
                        tgt,
                    )
                ]


                (
                    target_rank,
                    target_residual,
                ) = target_map[
                    (
                        src,
                        tgt,
                    )
                ]


                if not asym_3x8(
                    source_rank,
                    target_rank,
                ):

                    continue


                # ASYM3x8 raw cardinality.
                raw_count += 1


                source_degree = int(
                    outdeg.get(
                        src,
                        0,
                    )
                )


                target_degree = int(
                    indeg.get(
                        tgt,
                        0,
                    )
                )


                assert source_degree <= 1
                assert target_degree <= 1


                cuts = (
                    int(
                        source_degree
                        ==
                        1
                    )
                    +
                    int(
                        target_degree
                        ==
                        1
                    )
                )


                source_cv_pf = (
                    float(
                        source_residual
                    )
                    /
                    float(
                        delta_t
                    )
                )


                target_cv_pf = (
                    float(
                        target_residual
                    )
                    /
                    float(
                        delta_t
                    )
                )


                # --------------------------------------------
                # Relative incumbent improvements
                # --------------------------------------------

                source_imp = np.nan

                target_imp = np.nan


                if source_degree == 1:

                    incumbent = float(
                        source_incumbent.get(
                            src,
                            np.nan,
                        )
                    )


                    if np.isfinite(
                        incumbent
                    ):

                        source_imp = (
                            incumbent
                            -
                            source_cv_pf
                        )


                if target_degree == 1:

                    incumbent = float(
                        target_incumbent.get(
                            tgt,
                            np.nan,
                        )
                    )


                    if np.isfinite(
                        incumbent
                    ):

                        target_imp = (
                            incumbent
                            -
                            target_cv_pf
                        )


                required = []


                if source_degree == 1:

                    required.append(
                        source_imp
                    )


                if target_degree == 1:

                    required.append(
                        target_imp
                    )


                if cuts == 0:

                    dominance_any = True

                    dominance_all = True


                else:

                    finite_required = [
                        value

                        for value
                        in required

                        if np.isfinite(
                            value
                        )
                    ]


                    if len(
                        finite_required
                    ) != len(
                        required
                    ):

                        dominance_any = False

                        dominance_all = False


                    else:

                        dominance_any = any(
                            value > 0

                            for value
                            in finite_required
                        )


                        dominance_all = all(
                            value > 0

                            for value
                            in finite_required
                        )


                if not dominance_any:

                    continue


                retained_count += 1


                sym_cv = (
                    source_cv_pf
                    +
                    target_cv_pf
                ) / 2.0


                max_cv = max(
                    source_cv_pf,
                    target_cv_pf,
                )


                source_ratio = safe_ratio(
                    source_residual,
                    source_best[
                        src
                    ],
                )


                target_ratio = safe_ratio(
                    target_residual,
                    target_best[
                        tgt
                    ],
                )


                if (
                    np.isfinite(
                        source_ratio
                    )
                    and
                    np.isfinite(
                        target_ratio
                    )
                ):

                    gmean_ratio = float(
                        np.sqrt(
                            source_ratio
                            *
                            target_ratio
                        )
                    )


                else:

                    gmean_ratio = np.inf


                proposal_rows.append(
                    {
                        "source_id":
                            int(
                                src
                            ),

                        "target_id":
                            int(
                                tgt
                            ),

                        "gap_size":
                            int(
                                gap_size
                            ),

                        "cuts":
                            int(
                                cuts
                            ),

                        "SYM_CV":
                            float(
                                sym_cv
                            ),

                        "MAX_CV":
                            float(
                                max_cv
                            ),

                        "GMEAN_RATIO":
                            float(
                                gmean_ratio
                            ),

                        # Exact precomputed quantities retained
                        # for later scoring features.
                        "__source_cv_pf":
                            float(
                                source_cv_pf
                            ),

                        "__target_cv_pf":
                            float(
                                target_cv_pf
                            ),

                        "__source_incumbent":
                            float(
                                source_incumbent.get(
                                    src,
                                    np.nan,
                                )
                            ),

                        "__target_incumbent":
                            float(
                                target_incumbent.get(
                                    tgt,
                                    np.nan,
                                )
                            ),

                        "__source_improvement":
                            float(
                                source_imp
                            ),

                        "__target_improvement":
                            float(
                                target_imp
                            ),

                        "__dominance_all":
                            bool(
                                dominance_all
                            ),
                    }
                )


            del source_pos
            del target_pos

            del source_vel
            del target_vel

            del source_pred
            del target_pred

            del source_dist
            del source_idx

            del target_dist
            del target_idx

            del source_map
            del target_map

            del source_best
            del target_best

            del reciprocal_pairs


    proposals = pd.DataFrame.from_records(
        proposal_rows
    )


    assert len(
        proposals
    ) == retained_count


    assert not proposals[
        [
            "source_id",
            "target_id",
        ]
    ].duplicated().any()


    del graph
    del nodes
    del edges

    del edge_set
    del incoming
    del outgoing

    del indeg
    del outdeg

    del phys
    del time_map

    del node_ids
    del past_velocity
    del future_velocity

    del source_eligible
    del target_eligible

    del source_incumbent
    del target_incumbent

    del source_by_t
    del target_by_t

    del proposal_rows

    gc.collect()


    return (
        proposals,
        int(
            raw_count
        ),
        int(
            retained_count
        ),
    )


# ============================================================
# 11. ENDPOINT COMPETITION RANKS
#
# Same deterministic tie semantics as G3G1.
# ============================================================

RANK_SCORES = [
    "SYM_CV",
    "MAX_CV",
    "GMEAN_RATIO",
]


def add_endpoint_ranks(
    proposals,
):

    proposals = proposals.copy()


    for score in RANK_SCORES:

        source_order = (
            proposals[
                [
                    "source_id",
                    "target_id",
                    "gap_size",
                    score,
                ]
            ]
            .sort_values(
                [
                    "source_id",
                    score,
                    "gap_size",
                    "target_id",
                ],
                kind="stable",
            )
        )


        source_order[
            f"{score}_SOURCE_RANK"
        ] = (
            source_order.groupby(
                "source_id",
                sort=False,
            )
            .cumcount()
            +
            1
        )


        proposals[
            f"{score}_SOURCE_RANK"
        ] = (
            source_order[
                f"{score}_SOURCE_RANK"
            ]
            .reindex(
                proposals.index
            )
            .astype(
                np.int32
            )
        )


        target_order = (
            proposals[
                [
                    "source_id",
                    "target_id",
                    "gap_size",
                    score,
                ]
            ]
            .sort_values(
                [
                    "target_id",
                    score,
                    "gap_size",
                    "source_id",
                ],
                kind="stable",
            )
        )


        target_order[
            f"{score}_TARGET_RANK"
        ] = (
            target_order.groupby(
                "target_id",
                sort=False,
            )
            .cumcount()
            +
            1
        )


        proposals[
            f"{score}_TARGET_RANK"
        ] = (
            target_order[
                f"{score}_TARGET_RANK"
            ]
            .reindex(
                proposals.index
            )
            .astype(
                np.int32
            )
        )


        del source_order
        del target_order


    return proposals


# ============================================================
# 12. SHORTLIST RANK FIELDS
# ============================================================

def add_shortlist_rank_fields(
    proposals,
):

    p = proposals.copy()


    p[
        "GM_SOURCE_RANK"
    ] = p[
        "GMEAN_RATIO_SOURCE_RANK"
    ].astype(
        np.int32
    )


    p[
        "GM_TARGET_RANK"
    ] = p[
        "GMEAN_RATIO_TARGET_RANK"
    ].astype(
        np.int32
    )


    p[
        "GM_BEST_RANK"
    ] = np.minimum(
        p[
            "GM_SOURCE_RANK"
        ],
        p[
            "GM_TARGET_RANK"
        ],
    ).astype(
        np.int32
    )


    p[
        "GM_MAX_RANK"
    ] = np.maximum(
        p[
            "GM_SOURCE_RANK"
        ],
        p[
            "GM_TARGET_RANK"
        ],
    ).astype(
        np.int32
    )


    p[
        "SYM_BEST_RANK"
    ] = np.minimum(
        p[
            "SYM_CV_SOURCE_RANK"
        ],
        p[
            "SYM_CV_TARGET_RANK"
        ],
    ).astype(
        np.int32
    )


    p[
        "MAX_BEST_RANK"
    ] = np.minimum(
        p[
            "MAX_CV_SOURCE_RANK"
        ],
        p[
            "MAX_CV_TARGET_RANK"
        ],
    ).astype(
        np.int32
    )


    return p


def frozen_shortlist_mask(
    proposals,
):

    return (
        proposals[
            "GM_BEST_RANK"
        ].le(
            8
        )
        |
        (
            proposals[
                "cuts"
            ].le(
                1
            )
            &
            proposals[
                "GM_BEST_RANK"
            ].le(
                20
            )
        )
    )


# ============================================================
# 13. ATTACH FROZEN SCORING FEATURES
#
# No GT.
# ============================================================

def attach_scoring_features(
    proposals,
):

    p = proposals.copy()


    # --------------------------------------------------------
    # Temporal
    # --------------------------------------------------------

    p[
        "delta_t"
    ] = (
        p[
            "gap_size"
        ].astype(
            np.int16
        )
        +
        1
    )


    # --------------------------------------------------------
    # Topology
    # --------------------------------------------------------

    p[
        "source_occupied"
    ] = (
        p[
            "__source_incumbent"
        ].notna()
    ).astype(
        np.int8
    )


    p[
        "target_occupied"
    ] = (
        p[
            "__target_incumbent"
        ].notna()
    ).astype(
        np.int8
    )


    reconstructed_cuts = (
        p[
            "source_occupied"
        ]
        +
        p[
            "target_occupied"
        ]
    ).astype(
        np.int8
    )


    assert np.array_equal(
        reconstructed_cuts.to_numpy(),

        p[
            "cuts"
        ].to_numpy(
            dtype=np.int8
        ),
    )


    # --------------------------------------------------------
    # Candidate CV
    # --------------------------------------------------------

    p[
        "source_cv_pf"
    ] = p[
        "__source_cv_pf"
    ].astype(
        float
    )


    p[
        "target_cv_pf"
    ] = p[
        "__target_cv_pf"
    ].astype(
        float
    )


    p[
        "MIN_CV"
    ] = np.minimum(
        p[
            "source_cv_pf"
        ],
        p[
            "target_cv_pf"
        ],
    )


    # --------------------------------------------------------
    # Incumbent-relative
    # --------------------------------------------------------

    p[
        "source_incumbent_cv_pf"
    ] = p[
        "__source_incumbent"
    ].astype(
        float
    )


    p[
        "target_incumbent_cv_pf"
    ] = p[
        "__target_incumbent"
    ].astype(
        float
    )


    p[
        "source_relative_improvement"
    ] = p[
        "__source_improvement"
    ].astype(
        float
    )


    p[
        "target_relative_improvement"
    ] = p[
        "__target_improvement"
    ].astype(
        float
    )


    p[
        "dominance_all"
    ] = p[
        "__dominance_all"
    ].astype(
        np.int8
    )


    # --------------------------------------------------------
    # Required improvement summaries
    # --------------------------------------------------------

    source_occ = p[
        "source_occupied"
    ].astype(
        bool
    ).to_numpy()


    target_occ = p[
        "target_occupied"
    ].astype(
        bool
    ).to_numpy()


    source_imp = p[
        "source_relative_improvement"
    ].to_numpy(
        dtype=float
    )


    target_imp = p[
        "target_relative_improvement"
    ].to_numpy(
        dtype=float
    )


    max_required = np.full(
        len(
            p
        ),
        np.nan,
    )


    min_required = np.full(
        len(
            p
        ),
        np.nan,
    )


    mean_required = np.full(
        len(
            p
        ),
        np.nan,
    )


    mask = (
        source_occ
        &
        ~target_occ
    )


    max_required[
        mask
    ] = source_imp[
        mask
    ]

    min_required[
        mask
    ] = source_imp[
        mask
    ]

    mean_required[
        mask
    ] = source_imp[
        mask
    ]


    mask = (
        ~source_occ
        &
        target_occ
    )


    max_required[
        mask
    ] = target_imp[
        mask
    ]

    min_required[
        mask
    ] = target_imp[
        mask
    ]

    mean_required[
        mask
    ] = target_imp[
        mask
    ]


    mask = (
        source_occ
        &
        target_occ
    )


    max_required[
        mask
    ] = np.maximum(
        source_imp[
            mask
        ],
        target_imp[
            mask
        ],
    )


    min_required[
        mask
    ] = np.minimum(
        source_imp[
            mask
        ],
        target_imp[
            mask
        ],
    )


    mean_required[
        mask
    ] = (
        source_imp[
            mask
        ]
        +
        target_imp[
            mask
        ]
    ) / 2.0


    p[
        "max_required_improvement"
    ] = max_required


    p[
        "min_required_improvement"
    ] = min_required


    p[
        "mean_required_improvement"
    ] = mean_required


    # --------------------------------------------------------
    # Competition counts within frozen shortlist
    # --------------------------------------------------------

    source_count = (
        p.groupby(
            "source_id",
            sort=False,
        )[
            "source_id"
        ]
        .transform(
            "size"
        )
        .astype(
            np.int32
        )
    )


    target_count = (
        p.groupby(
            "target_id",
            sort=False,
        )[
            "target_id"
        ]
        .transform(
            "size"
        )
        .astype(
            np.int32
        )
    )


    p[
        "SOURCE_COMPETITION_COUNT"
    ] = source_count


    p[
        "TARGET_COMPETITION_COUNT"
    ] = target_count


    # --------------------------------------------------------
    # Rank combinations
    # --------------------------------------------------------

    p[
        "GM_RANK_SUM"
    ] = (
        p[
            "GM_SOURCE_RANK"
        ].astype(
            float
        )
        +
        p[
            "GM_TARGET_RANK"
        ].astype(
            float
        )
    )


    p[
        "GM_RANK_LOG_ASYMMETRY"
    ] = np.abs(
        np.log1p(
            p[
                "GM_SOURCE_RANK"
            ].astype(
                float
            )
        )
        -
        np.log1p(
            p[
                "GM_TARGET_RANK"
            ].astype(
                float
            )
        )
    )


    p[
        "GM_SOURCE_RANK_FRAC"
    ] = (
        p[
            "GM_SOURCE_RANK"
        ].astype(
            float
        )
        /
        p[
            "SOURCE_COMPETITION_COUNT"
        ].astype(
            float
        )
    )


    p[
        "GM_TARGET_RANK_FRAC"
    ] = (
        p[
            "GM_TARGET_RANK"
        ].astype(
            float
        )
        /
        p[
            "TARGET_COMPETITION_COUNT"
        ].astype(
            float
        )
    )


    p[
        "GM_BEST_RANK_FRAC"
    ] = np.minimum(
        p[
            "GM_SOURCE_RANK_FRAC"
        ],
        p[
            "GM_TARGET_RANK_FRAC"
        ],
    )


    for col in FEATURE_COLUMNS:

        assert col in p.columns, col


    return p


# ============================================================
# 14. DETERMINISTIC 1/64 SAMPLE
# ============================================================

SAMPLE_MODULUS = 64
SAMPLE_BUCKET = 0


def dataset_seed_uint64(
    dataset,
):

    digest = hashlib.sha256(
        str(
            dataset
        ).encode(
            "utf-8"
        )
    ).digest()


    return np.uint64(
        int.from_bytes(
            digest[
                :8
            ],
            byteorder="little",
            signed=False,
        )
    )


def splitmix64(
    x,
):

    with np.errstate(
        over="ignore"
    ):

        x = (
            x
            +
            np.uint64(
                0x9E3779B97F4A7C15
            )
        )


        x = (
            (
                x
                ^
                (
                    x
                    >>
                    np.uint64(
                        30
                    )
                )
            )
            *
            np.uint64(
                0xBF58476D1CE4E5B9
            )
        )


        x = (
            (
                x
                ^
                (
                    x
                    >>
                    np.uint64(
                        27
                    )
                )
            )
            *
            np.uint64(
                0x94D049BB133111EB
            )
        )


        x = (
            x
            ^
            (
                x
                >>
                np.uint64(
                    31
                )
            )
        )


    return x


def deterministic_sample_mask(
    dataset,
    frame,
):

    src = frame[
        "source_id"
    ].to_numpy(
        dtype=np.uint64
    )


    tgt = frame[
        "target_id"
    ].to_numpy(
        dtype=np.uint64
    )


    gap = frame[
        "gap_size"
    ].to_numpy(
        dtype=np.uint64
    )


    seed = dataset_seed_uint64(
        dataset
    )


    with np.errstate(
        over="ignore"
    ):

        x = (
            seed
            ^
            (
                src
                *
                np.uint64(
                    0xD6E8FEB86659FD93
                )
            )
            ^
            (
                tgt
                *
                np.uint64(
                    0xA5A3564E27F8862B
                )
            )
            ^
            (
                gap
                *
                np.uint64(
                    0x9E3779B185EBCA87
                )
            )
        )


    h = splitmix64(
        x
    )


    return (
        (
            h
            %
            np.uint64(
                SAMPLE_MODULUS
            )
        )
        ==
        np.uint64(
            SAMPLE_BUCKET
        )
    )


# ============================================================
# PART A — GT-BLIND FEATURE SAMPLE
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART A — GT-BLIND SCORING FEATURE SAMPLE"
)

print(
    "============================================================"
)


sample_frames = []

dataset_results = []


raw_total = 0
generator_total = 0
shortlist_total = 0


for ds_i, dataset in enumerate(
    development_names,
    start=1,
):

    fold = int(
        fold_map[
            dataset
        ]
    )


    (
        proposals,
        raw_count,
        retained_count,
    ) = generate_dataset_proposals(
        dataset
    )


    raw_total += int(
        raw_count
    )


    generator_total += int(
        retained_count
    )


    proposals = add_endpoint_ranks(
        proposals
    )


    proposals = add_shortlist_rank_fields(
        proposals
    )


    shortlist = proposals[
        frozen_shortlist_mask(
            proposals
        )
    ].copy()


    expected_dataset_count = int(
        expected_dataset_shortlist[
            dataset
        ]
    )


    assert len(
        shortlist
    ) == expected_dataset_count, (
        "\nPer-dataset shortlist fidelity failure\n"
        f"dataset={dataset}\n"
        f"observed={len(shortlist)}\n"
        f"expected={expected_dataset_count}"
    )


    shortlist_total += int(
        len(
            shortlist
        )
    )


    shortlist = attach_scoring_features(
        shortlist
    )


    sample_mask = deterministic_sample_mask(
        dataset,
        shortlist,
    )


    sample_cols = (
        [
            "source_id",
            "target_id",
        ]
        +
        FEATURE_COLUMNS
    )


    assert len(
        sample_cols
    ) == len(
        set(
            sample_cols
        )
    )


    sample = shortlist.loc[
        sample_mask,
        sample_cols,
    ].copy()


    sample.insert(
        0,
        "dataset",
        dataset,
    )


    sample.insert(
        1,
        "fold",
        fold,
    )


    assert not sample.columns.duplicated().any()


    sample_frames.append(
        sample
    )


    dataset_results.append(
        {
            "dataset":
                dataset,

            "fold":
                fold,

            "raw_rows":
                int(
                    raw_count
                ),

            "generator_rows":
                int(
                    retained_count
                ),

            "shortlist_rows":
                int(
                    len(
                        shortlist
                    )
                ),

            "sample_rows":
                int(
                    len(
                        sample
                    )
                ),
        }
    )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == 103
    ):

        print(
            f"{ds_i:3d}/103"
            f" | fold={fold}"
            f" | {dataset}"
            f" | raw={raw_count}"
            f" | gen={retained_count}"
            f" | shortlist={len(shortlist)}"
            f" | sample={len(sample)}"
        )


    del proposals
    del shortlist
    del sample

    gc.collect()


# ============================================================
# 15. GLOBAL PART-A FIDELITY
# ============================================================

assert raw_total == EXPECTED_RAW_ROWS, (
    raw_total,
    EXPECTED_RAW_ROWS,
)


assert generator_total == EXPECTED_GENERATOR_ROWS, (
    generator_total,
    EXPECTED_GENERATOR_ROWS,
)


assert shortlist_total == EXPECTED_SHORTLIST_ROWS, (
    shortlist_total,
    EXPECTED_SHORTLIST_ROWS,
)


sample_table = pd.concat(
    sample_frames,
    ignore_index=True,
)


del sample_frames

gc.collect()


assert not sample_table.columns.duplicated().any()


assert not sample_table[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


sample_table.to_pickle(
    SAMPLE_OUT
)


assert SAMPLE_OUT.exists()


print(
    "\n========== PART-A SHORTLIST FIDELITY =========="
)

print(
    "raw generator reproduced:",
    raw_total,
    "/",
    EXPECTED_RAW_ROWS
)

print(
    "post-dominance generator reproduced:",
    generator_total,
    "/",
    EXPECTED_GENERATOR_ROWS
)

print(
    "shortlist reproduced:",
    shortlist_total,
    "/",
    EXPECTED_SHORTLIST_ROWS
)

print(
    "deterministic sample rows:",
    len(
        sample_table
    )
)

print(
    "sample fraction:",
    len(
        sample_table
    )
    /
    EXPECTED_SHORTLIST_ROWS
)

print(
    "SHORTLIST_REPRODUCTION: PASS"
)


# ============================================================
# 16. FREEZE PART A BEFORE GT READ
# ============================================================

part_a = {
    "stage":
        "12.11G3H0",

    "part":
        "GT_BLIND_SCORING_FEATURE_SAMPLE",

    "feature_schema_sha256":
        FEATURE_SCHEMA_SHA,

    "parent_generator_sha256":
        GENERATOR_SHA,

    "parent_shortlist_sha256":
        SHORTLIST_SHA,

    "raw_generator_rows":
        int(
            raw_total
        ),

    "generator_rows":
        int(
            generator_total
        ),

    "shortlist_rows":
        int(
            shortlist_total
        ),

    "sampling": {
        "type":
            "DETERMINISTIC_SPLITMIX64",

        "modulus":
            64,

        "bucket":
            0,

        "gt_blind":
            True,
    },

    "sample_rows":
        int(
            len(
                sample_table
            )
        ),

    "dataset_results":
        dataset_results,

    "gt_used":
        False,

    "fold0_used":
        False,

    "classifier_trained":
        False,

    "feature_subset_selected":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "graph_modified":
        False,
}


part_a_sha = hashlib.sha256(
    json.dumps(
        part_a,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


part_a[
    "sha256"
] = part_a_sha


PARTA_SUMMARY_OUT.write_text(
    json.dumps(
        part_a,
        indent=2,
    ),
    encoding="utf-8",
)


assert PARTA_SUMMARY_OUT.exists()


print(
    "\nGT-BLIND FEATURE SAMPLE FROZEN BEFORE GT READ: YES"
)

print(
    "Part-A SHA256:",
    part_a_sha
)


# ============================================================
# PART B — DEVELOPMENT ORACLE
#
# GT READ STARTS HERE.
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — DEVELOPMENT ORACLE FEATURE AUDIT"
)

print(
    "============================================================"
)


oracle = pd.read_pickle(
    G3G1B_ORACLE_PATH
)


assert len(
    oracle
) == 369


assert set(
    oracle[
        "fold"
    ].astype(
        int
    )
) == {
    1,
    2,
    3,
    4,
}


SELECTED_TRUE_COL = (
    "rescue_G8_LOWCUT20"
)


assert SELECTED_TRUE_COL in oracle.columns


positive_oracle = (
    oracle[
        oracle[
            SELECTED_TRUE_COL
        ].astype(
            bool
        )
    ][
        [
            "dataset",
            "fold",
            "component_id",
            "source_id",
            "target_id",
            "pred_full_7um",
            "topology_class",
        ]
    ]
    .copy()
)


assert len(
    positive_oracle
) == 202


assert int(
    positive_oracle[
        "pred_full_7um"
    ].sum()
) == 173


# ============================================================
# 17. RECONSTRUCT ALL 202 POSITIVE FEATURE ROWS
# ============================================================

positive_frames = []


positive_datasets = (
    positive_oracle[
        "dataset"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)


for ds_i, dataset in enumerate(
    positive_datasets,
    start=1,
):

    (
        proposals,
        _,
        _,
    ) = generate_dataset_proposals(
        dataset
    )


    proposals = add_endpoint_ranks(
        proposals
    )


    proposals = add_shortlist_rank_fields(
        proposals
    )


    shortlist = proposals[
        frozen_shortlist_mask(
            proposals
        )
    ].copy()


    assert len(
        shortlist
    ) == expected_dataset_shortlist[
        dataset
    ]


    shortlist = attach_scoring_features(
        shortlist
    )


    ds_positive = positive_oracle[
        positive_oracle[
            "dataset"
        ].eq(
            dataset
        )
    ]


    compact_cols = (
        [
            "source_id",
            "target_id",
        ]
        +
        FEATURE_COLUMNS
    )


    compact = shortlist[
        compact_cols
    ].copy()


    matched = ds_positive.merge(
        compact,
        on=[
            "source_id",
            "target_id",
        ],
        how="left",
        validate="one_to_one",
    )


    assert len(
        matched
    ) == len(
        ds_positive
    )


    assert matched[
        "GMEAN_RATIO"
    ].notna().all()


    assert matched[
        "GM_BEST_RANK"
    ].notna().all()


    positive_frames.append(
        matched
    )


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            positive_datasets
        )
    ):

        print(
            f"{ds_i:3d}/{len(positive_datasets)}"
            f" | {dataset}"
            f" | positives={len(ds_positive)}"
        )


    del proposals
    del shortlist
    del compact
    del matched

    gc.collect()


positive_features = pd.concat(
    positive_frames,
    ignore_index=True,
)


del positive_frames

gc.collect()


assert len(
    positive_features
) == 202


assert not positive_features[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


positive_features.to_pickle(
    POSITIVE_FEATURES_OUT
)


assert POSITIVE_FEATURES_OUT.exists()


print(
    "\nPOSITIVE_FEATURE_RECONSTRUCTION: PASS"
)

print(
    "positive rows:",
    len(
        positive_features
    )
)

print(
    "full<=7 positives:",
    int(
        positive_features[
            "pred_full_7um"
        ].sum()
    )
)


# ============================================================
# 18. REMOVE KNOWN POSITIVES FROM BACKGROUND SAMPLE
#
# label=0 means sampled background,
# NOT confirmed false proposal.
# ============================================================

positive_keys = positive_features[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].copy()


positive_keys[
    "__known_positive"
] = True


sample_check = sample_table.merge(
    positive_keys,
    on=[
        "dataset",
        "source_id",
        "target_id",
    ],
    how="left",
    validate="one_to_one",
)


sample_background = (
    sample_check[
        sample_check[
            "__known_positive"
        ].isna()
    ]
    .drop(
        columns=[
            "__known_positive",
        ]
    )
    .copy()
)


known_positive_overlap = (
    len(
        sample_table
    )
    -
    len(
        sample_background
    )
)


print(
    "\n========== AUDIT POPULATION =========="
)

print(
    "sample before known-positive removal:",
    len(
        sample_table
    )
)

print(
    "known positives inside sample:",
    known_positive_overlap
)

print(
    "sampled background:",
    len(
        sample_background
    )
)

print(
    "all positives:",
    len(
        positive_features
    )
)

print(
    "background are confirmed negatives:",
    "NO"
)

print(
    "AUC interpretation:",
    "KNOWN_POSITIVE_VS_SAMPLED_BACKGROUND"
)


# ============================================================
# 19. AUDIT POPULATION
# ============================================================

ID_COLS = [
    "dataset",
    "fold",
    "source_id",
    "target_id",
]


background_audit = sample_background[
    ID_COLS
    +
    FEATURE_COLUMNS
].copy()


background_audit[
    "label"
] = 0


positive_audit = positive_features[
    ID_COLS
    +
    FEATURE_COLUMNS
].copy()


positive_audit[
    "label"
] = 1


audit_population = pd.concat(
    [
        background_audit,
        positive_audit,
    ],
    ignore_index=True,
)


assert int(
    audit_population[
        "label"
    ].sum()
) == 202


# ============================================================
# 20. AUC HELPERS
# ============================================================

def auc_or_nan(
    labels,
    values,
):

    labels = np.asarray(
        labels,
        dtype=int,
    )


    values = np.asarray(
        values,
        dtype=float,
    )


    if len(
        labels
    ) == 0:

        return np.nan


    if len(
        np.unique(
            labels
        )
    ) < 2:

        return np.nan


    return float(
        roc_auc_score(
            labels,
            values,
        )
    )


def finite_stats(
    series,
):

    s = pd.to_numeric(
        series,
        errors="coerce",
    )


    s = s[
        np.isfinite(
            s.to_numpy(
                dtype=float
            )
        )
    ]


    if len(
        s
    ) == 0:

        return {
            "n":
                0,

            "median":
                np.nan,

            "q25":
                np.nan,

            "q75":
                np.nan,
        }


    return {
        "n":
            int(
                len(
                    s
                )
            ),

        "median":
            float(
                s.median()
            ),

        "q25":
            float(
                s.quantile(
                    0.25
                )
            ),

        "q75":
            float(
                s.quantile(
                    0.75
                )
            ),
    }


# ============================================================
# 21. CROSS-FOLD UNIVARIATE AUDIT
# ============================================================

audit_rows = []


for feature in FEATURE_COLUMNS:

    numeric = pd.to_numeric(
        audit_population[
            feature
        ],
        errors="coerce",
    )


    finite_mask = np.isfinite(
        numeric.to_numpy(
            dtype=float
        )
    )


    frame = audit_population.loc[
        finite_mask,
        [
            "fold",
            "label",
        ]
    ].copy()


    values = numeric.loc[
        finite_mask
    ].to_numpy(
        dtype=float
    )


    labels = frame[
        "label"
    ].to_numpy(
        dtype=int
    )


    raw_auc = auc_or_nan(
        labels,
        values,
    )


    if np.isnan(
        raw_auc
    ):

        direction = "UNDEFINED"

        overall_auc = np.nan


    elif raw_auc >= 0.5:

        direction = "HIGHER_FOR_TRUE"

        overall_auc = raw_auc


    else:

        direction = "LOWER_FOR_TRUE"

        overall_auc = (
            1.0
            -
            raw_auc
        )


    fold_aucs = []


    for fold in [
        1,
        2,
        3,
        4,
    ]:

        fold_mask = (
            frame[
                "fold"
            ].eq(
                fold
            )
            .to_numpy()
        )


        fold_labels = labels[
            fold_mask
        ]


        fold_values = values[
            fold_mask
        ]


        fold_raw = auc_or_nan(
            fold_labels,
            fold_values,
        )


        if np.isnan(
            fold_raw
        ):

            fold_auc = np.nan


        elif direction == "HIGHER_FOR_TRUE":

            fold_auc = fold_raw


        elif direction == "LOWER_FOR_TRUE":

            fold_auc = (
                1.0
                -
                fold_raw
            )


        else:

            fold_auc = np.nan


        fold_aucs.append(
            fold_auc
        )


    finite_fold_aucs = np.asarray(
        [
            x
            for x in fold_aucs
            if np.isfinite(
                x
            )
        ],
        dtype=float,
    )


    positive_stats = finite_stats(
        audit_population.loc[
            audit_population[
                "label"
            ].eq(
                1
            ),
            feature,
        ]
    )


    background_stats = finite_stats(
        audit_population.loc[
            audit_population[
                "label"
            ].eq(
                0
            ),
            feature,
        ]
    )


    audit_rows.append(
        {
            "feature":
                feature,

            "direction":
                direction,

            "overall_auc":
                overall_auc,

            "fold1_auc":
                fold_aucs[
                    0
                ],

            "fold2_auc":
                fold_aucs[
                    1
                ],

            "fold3_auc":
                fold_aucs[
                    2
                ],

            "fold4_auc":
                fold_aucs[
                    3
                ],

            "defined_fold_count":
                int(
                    len(
                        finite_fold_aucs
                    )
                ),

            "mean_fold_auc":
                (
                    float(
                        finite_fold_aucs.mean()
                    )
                    if len(
                        finite_fold_aucs
                    )
                    else np.nan
                ),

            "worst_fold_auc":
                (
                    float(
                        finite_fold_aucs.min()
                    )
                    if len(
                        finite_fold_aucs
                    )
                    else np.nan
                ),

            "fold_auc_std":
                (
                    float(
                        finite_fold_aucs.std()
                    )
                    if len(
                        finite_fold_aucs
                    )
                    else np.nan
                ),

            "positive_finite_n":
                positive_stats[
                    "n"
                ],

            "background_finite_n":
                background_stats[
                    "n"
                ],

            "positive_median":
                positive_stats[
                    "median"
                ],

            "background_median":
                background_stats[
                    "median"
                ],

            "positive_q25":
                positive_stats[
                    "q25"
                ],

            "positive_q75":
                positive_stats[
                    "q75"
                ],

            "background_q25":
                background_stats[
                    "q25"
                ],

            "background_q75":
                background_stats[
                    "q75"
                ],
        }
    )


feature_audit = pd.DataFrame(
    audit_rows
)


feature_audit = (
    feature_audit
    .sort_values(
        [
            "defined_fold_count",
            "worst_fold_auc",
            "overall_auc",
        ],
        ascending=[
            False,
            False,
            False,
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n========== CROSS-FOLD UNIVARIATE FEATURE AUDIT =========="
)


print(
    feature_audit.to_string(
        index=False
    )
)


# ============================================================
# 22. STABLE SIGNAL DIAGNOSTIC
#
# Diagnostic only.
# No feature selection.
# ============================================================

stable_signal = feature_audit[
    (
        feature_audit[
            "defined_fold_count"
        ]
        ==
        4
    )
    &
    (
        feature_audit[
            "overall_auc"
        ]
        >=
        0.60
    )
    &
    (
        feature_audit[
            "worst_fold_auc"
        ]
        >=
        0.55
    )
].copy()


print(
    "\n========== STABLE UNIVARIATE SIGNALS (DIAGNOSTIC ONLY) =========="
)


if len(
    stable_signal
):

    print(
        stable_signal[
            [
                "feature",
                "direction",

                "overall_auc",
                "fold1_auc",
                "fold2_auc",
                "fold3_auc",
                "fold4_auc",

                "worst_fold_auc",
                "fold_auc_std",

                "positive_median",
                "background_median",
            ]
        ].to_string(
            index=False
        )
    )


else:

    print(
        "NONE"
    )


# ============================================================
# 23. TOP OVERALL SIGNALS
# ============================================================

top_features = (
    feature_audit[
        feature_audit[
            "overall_auc"
        ].notna()
    ]
    .sort_values(
        "overall_auc",
        ascending=False,
        kind="stable",
    )
    .head(
        15
    )
)


print(
    "\n========== TOP FEATURES BY OVERALL AUC =========="
)


print(
    top_features[
        [
            "feature",
            "direction",
            "overall_auc",
            "worst_fold_auc",
            "fold_auc_std",
            "positive_median",
            "background_median",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 24. PERSIST FORMAL AUDIT
# ============================================================

feature_audit.to_pickle(
    AUDIT_OUT
)


summary = {
    "stage":
        "12.11G3H0",

    "status":
        "CROSS_FOLD_SCORING_FEATURE_AUDIT_COMPLETE",

    "parent_generator_sha256":
        GENERATOR_SHA,

    "parent_shortlist_sha256":
        SHORTLIST_SHA,

    "feature_schema_sha256":
        FEATURE_SCHEMA_SHA,

    "part_a_sha256":
        part_a_sha,

    "raw_generator_rows":
        int(
            raw_total
        ),

    "generator_rows":
        int(
            generator_total
        ),

    "shortlist_rows":
        int(
            shortlist_total
        ),

    "sample_rows_before_positive_removal":
        int(
            len(
                sample_table
            )
        ),

    "known_positive_sample_overlap":
        int(
            known_positive_overlap
        ),

    "background_rows":
        int(
            len(
                sample_background
            )
        ),

    "positive_rows":
        202,

    "full7_positive_rows":
        173,

    "auc_semantics":
        (
            "KNOWN_ORACLE_POSITIVE_VS_"
            "GT_BLIND_HASH_SAMPLED_BACKGROUND"
        ),

    "background_is_confirmed_negative":
        False,

    "feature_audit":
        feature_audit.to_dict(
            "records"
        ),

    "stable_signal_diagnostic":
        stable_signal[
            [
                "feature",
                "direction",
                "overall_auc",
                "worst_fold_auc",
                "fold_auc_std",
            ]
        ].to_dict(
            "records"
        ),

    "feature_subset_selected":
        False,

    "classifier_trained":
        False,

    "scoring_threshold_selected":
        False,

    "rewire_policy_selected":
        False,

    "fold0_used":
        False,

    "graph_modified":
        False,
}


SUMMARY_OUT.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


assert AUDIT_OUT.exists()
assert SUMMARY_OUT.exists()


# ============================================================
# 25. RELOAD INTEGRITY
# ============================================================

sample_check = pd.read_pickle(
    SAMPLE_OUT
)


assert len(
    sample_check
) == len(
    sample_table
)


parta_check = json.loads(
    PARTA_SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert parta_check[
    "gt_used"
] is False


assert parta_check[
    "shortlist_rows"
] == EXPECTED_SHORTLIST_ROWS


positive_check = pd.read_pickle(
    POSITIVE_FEATURES_OUT
)


assert len(
    positive_check
) == 202


audit_check = pd.read_pickle(
    AUDIT_OUT
)


assert len(
    audit_check
) == len(
    FEATURE_COLUMNS
)


summary_check = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert summary_check[
    "positive_rows"
] == 202


assert summary_check[
    "fold0_used"
] is False


# ============================================================
# 26. FINAL DECISION
# ============================================================

print(
    "\n========== 12.11G3H0 DECISION =========="
)

print(
    "CROSS_FOLD_SCORING_FEATURE_AUDIT: PASS"
)

print(
    "parent generator SHA:",
    GENERATOR_SHA
)

print(
    "parent shortlist SHA:",
    SHORTLIST_SHA
)

print(
    "feature schema SHA:",
    FEATURE_SCHEMA_SHA
)

print(
    "existing schema reused:",
    "YES"
)

print(
    "schema rewritten:",
    "NO"
)

print(
    "raw generator:",
    raw_total,
    "/",
    EXPECTED_RAW_ROWS
)

print(
    "post-dominance generator:",
    generator_total,
    "/",
    EXPECTED_GENERATOR_ROWS
)

print(
    "shortlist:",
    shortlist_total,
    "/",
    EXPECTED_SHORTLIST_ROWS
)

print(
    "background sample rows:",
    len(
        sample_background
    )
)

print(
    "positive feature rows:",
    len(
        positive_features
    )
)

print(
    "stable diagnostic signals:",
    len(
        stable_signal
    )
)

print(
    "background treated as confirmed negatives:",
    "NO"
)

print(
    "feature subset selected:",
    "NO"
)

print(
    "classifier trained:",
    "NO"
)

print(
    "scoring threshold selected:",
    "NO"
)

print(
    "rewire policy selected:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "graph modified:",
    "NO"
)

print(
    "next:"
)

print(
    "DESIGN_LEAVE_ONE_FOLD_OUT_SCORER_BENCHMARK_FROM_FROZEN_FEATURE_SCHEMA"
)

In [ ]:
# ============================================================
# Stage 12.11G3H1
#
# Leave-One-Fold-Out Scorer Benchmark
# on Frozen G3H0 Feature Schema + Frozen Shortlist
#
# ------------------------------------------------------------
# FROZEN UPSTREAM
# ------------------------------------------------------------
#
# Parent generator:
#   MULTIFRAME_REWIRE_ASYM3X8_DOMINANCE_ANY_V1_FIDELITYPATCH1
#
# Parent generator SHA:
#   da408df857cd34dcb4b1f65f3fae545e299f9403ef5edb986584caa8f4c3c203
#
# Frozen shortlist:
#   ENDPOINT_SHORTLIST_GMEAN_G8_LOWCUT20_V1
#
# Shortlist SHA:
#   ae61216904bc3187f76b238b4b560d6693a7eb4f5d07bb8ef19021cc1bca1490
#
# Frozen G3H0 feature schema SHA:
#   0a9f8e9a41c5c80c803c3a95955702dcf4039605a21fc656a154aa05be55b39c
#
# Frozen shortlist rows:
#   17,501,674
#
# ------------------------------------------------------------
# G3H1 GOAL
# ------------------------------------------------------------
#
# Benchmark whether a scorer trained ONLY on the other
# development folds can improve local proposal ranking on the
# held-out development fold.
#
# For held-out fold f:
#
#   train:
#       folds {1,2,3,4} \ {f}
#
#   test/rank:
#       COMPLETE frozen shortlist in fold f
#
# Fold0 is NEVER used.
#
# ------------------------------------------------------------
# TRAINING LABEL SEMANTICS
# ------------------------------------------------------------
#
# Positive:
#   known clean-gap development oracle positive
#
# Background:
#   deterministic GT-blind G3H0 hash sample after removing
#   known positives
#
# IMPORTANT:
#   background is NOT a confirmed-negative population.
#
# Therefore model outputs are:
#
#   SURROGATE RANKING SCORES
#
# NOT calibrated probabilities.
#
# ------------------------------------------------------------
# FIXED MODEL FAMILY
# ------------------------------------------------------------
#
# 0. BASE_GMEAN_RATIO_V1
#      no training
#      score = -GMEAN_RATIO
#
# 1. LOGREG_L2_C1_BALANCED_V1
#      fixed L2 logistic regression
#      C = 1
#
# 2. HGB_D3_L15_BALANCED_V1
#      fixed shallow nonlinear benchmark
#
# No feature selection.
# No hyperparameter sweep.
#
# The entire model family and preprocessing contract are frozen
# BEFORE development positive labels are loaded below.
#
# ------------------------------------------------------------
# PREPROCESSING
# ------------------------------------------------------------
#
# Raw 32 frozen G3H0 features only.
#
# Per training fold:
#
#   +/- inf -> missing
#   training-fold median imputation
#   append explicit missingness indicator for every feature
#
# Logistic:
#   standardize imputed values using training fold only
#
# HGB:
#   unscaled imputed values + same missing indicators
#
# No held-out statistics enter preprocessing.
#
# ------------------------------------------------------------
# PRIMARY OOF METRIC
# ------------------------------------------------------------
#
# FULL<=7 positive:
#
# ASYM_SCORE_1x5:
#
#   (source score-rank <= 1 AND target score-rank <= 5)
#       OR
#   (target score-rank <= 1 AND source score-rank <= 5)
#
# This is fixed before OOF evaluation.
#
# It reflects eventual endpoint-conflict resolution:
# true repair should be extremely competitive at one endpoint
# and still competitive at the opposite endpoint.
#
# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------
#
# G3H1 does NOT:
#
#   - select final scorer
#   - choose score threshold
#   - choose action threshold
#   - cut incumbent edges
#   - modify graph
#   - use Fold0
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import gc
import hashlib
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


FEATURE_SCHEMA_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_feature_schema_v1_frozen.json"
)


SAMPLE_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_feature_sample_hash64_v1.pkl"
)


POSITIVE_FEATURES_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_positive_features_v1.pkl"
)


G3H0_SUMMARY_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_feature_audit_v1_summary.json"
)


FROZEN_SHORTLIST_PATH = (
    S12
    / "stage12_folds14_endpoint_shortlist_g8_lowcut20_v1_frozen.json"
)


# ------------------------------------------------------------
# G3H1 outputs
# ------------------------------------------------------------

CONTRACT_OUT = (
    S12
    / "stage12_folds14_rewire_scorer_benchmark_v1_frozen.json"
)


OOF_POSITIVE_RANKS_OUT = (
    S12
    / "stage12_folds14_rewire_scorer_oof_positive_ranks_v1.pkl"
)


FOLD_METRICS_OUT = (
    S12
    / "stage12_folds14_rewire_scorer_oof_fold_metrics_v1.pkl"
)


MODEL_SUMMARY_OUT = (
    S12
    / "stage12_folds14_rewire_scorer_oof_model_summary_v1.json"
)


for path in [
    FEATURE_SCHEMA_PATH,
    SAMPLE_PATH,
    POSITIVE_FEATURES_PATH,
    G3H0_SUMMARY_PATH,
    FROZEN_SHORTLIST_PATH,
]:

    assert path.exists(), path


for path in [
    CONTRACT_OUT,
    OOF_POSITIVE_RANKS_OUT,
    FOLD_METRICS_OUT,
    MODEL_SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3H1 output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite a completed/partial benchmark."
        )


# ============================================================
# 1. REQUIRED IN-MEMORY G3H0 HELPERS
#
# Kernel must still contain the successful G3H0/G3H0-R1
# functions.
# ============================================================

REQUIRED_HELPERS = [
    "build_current_best",
    "generate_dataset_proposals",
    "add_endpoint_ranks",
    "add_shortlist_rank_fields",
    "frozen_shortlist_mask",
    "attach_scoring_features",
]


missing_helpers = [
    name

    for name in REQUIRED_HELPERS

    if (
        name not in globals()
        or
        not callable(
            globals()[
                name
            ]
        )
    )
]


assert not missing_helpers, (
    "\nRequired G3H0 helpers are missing:\n"
    f"{missing_helpers}\n\n"
    "Do not restart the kernel. "
    "If already restarted, restore G3H0-R1 and G3H0 helper definitions."
)


print(
    "========== G3H1 UPSTREAM HELPERS =========="
)

print(
    "required helpers:",
    len(
        REQUIRED_HELPERS
    )
)

print(
    "missing:",
    missing_helpers
)

print(
    "HELPER FIDELITY GATE: PASS"
)


# ============================================================
# 2. FIXED AUTHORITATIVE SHAs
# ============================================================

GENERATOR_SHA = (
    "da408df857cd34dcb4b1f65f3fae545e"
    "299f9403ef5edb986584caa8f4c3c203"
)


SHORTLIST_SHA = (
    "ae61216904bc3187f76b238b4b560d669"
    "3a7eb4f5d07bb8ef19021cc1bca1490"
)


FEATURE_SCHEMA_SHA = (
    "0a9f8e9a41c5c80c803c3a95955702dc"
    "f4039605a21fc656a154aa05be55b39c"
)


EXPECTED_SHORTLIST_ROWS = 17_501_674

EXPECTED_POSITIVES = 202

EXPECTED_FULL7_POSITIVES = 173


# ============================================================
# 3. VERIFY FEATURE SCHEMA
#
# No GT read.
# ============================================================

feature_schema = json.loads(
    FEATURE_SCHEMA_PATH.read_text(
        encoding="utf-8"
    )
)


assert feature_schema[
    "feature_schema_sha256"
] == FEATURE_SCHEMA_SHA


assert feature_schema[
    "parent_shortlist_sha256"
] == SHORTLIST_SHA


assert feature_schema[
    "parent_generator_sha256"
] == GENERATOR_SHA


FEATURE_COLUMNS = list(
    feature_schema[
        "features"
    ]
)


assert len(
    FEATURE_COLUMNS
) == 32


assert "fold" not in FEATURE_COLUMNS
assert "dataset" not in FEATURE_COLUMNS


# ============================================================
# 4. VERIFY FROZEN SHORTLIST
# ============================================================

frozen_shortlist = json.loads(
    FROZEN_SHORTLIST_PATH.read_text(
        encoding="utf-8"
    )
)


assert frozen_shortlist[
    "shortlist_contract_sha256"
] == SHORTLIST_SHA


assert int(
    frozen_shortlist[
        "development_cardinality"
    ][
        "rows"
    ]
) == EXPECTED_SHORTLIST_ROWS


# ============================================================
# 5. LOAD GT-BLIND SAMPLE ONLY
#
# This contains no oracle labels.
# ============================================================

sample = pd.read_pickle(
    SAMPLE_PATH
)


assert len(
    sample
) == 273_658


assert {
    "dataset",
    "fold",
    "source_id",
    "target_id",
}.issubset(
    sample.columns
)


for feature in FEATURE_COLUMNS:

    assert feature in sample.columns


assert set(
    sample[
        "fold"
    ].astype(
        int
    )
) == {
    1,
    2,
    3,
    4,
}


# ============================================================
# 6. FREEZE G3H1 BENCHMARK CONTRACT
#
# IMPORTANT:
# Positive GT features have NOT been loaded yet.
# ============================================================

MODEL_IDS = [
    "BASE_GMEAN_RATIO_V1",
    "LOGREG_L2_C1_BALANCED_V1",
    "HGB_D3_L15_BALANCED_V1",
]


BENCHMARK_CONTRACT = {
    "stage":
        "12.11G3H1",

    "benchmark_id":
        "LOFO_SCORER_BENCHMARK_V1",

    "parent_generator_sha256":
        GENERATOR_SHA,

    "parent_shortlist_sha256":
        SHORTLIST_SHA,

    "feature_schema_sha256":
        FEATURE_SCHEMA_SHA,

    "feature_count":
        int(
            len(
                FEATURE_COLUMNS
            )
        ),

    "feature_subset_selected":
        False,

    "models": {
        "BASE_GMEAN_RATIO_V1": {
            "type":
                "GT_BLIND_BASELINE",

            "score":
                "NEGATIVE_GMEAN_RATIO",

            "trained":
                False,
        },

        "LOGREG_L2_C1_BALANCED_V1": {
            "type":
                "LOGISTIC_REGRESSION",

            "penalty":
                "l2",

            "C":
                1.0,

            "solver":
                "lbfgs",

            "max_iter":
                2000,

            "class_balance":
                "EXPLICIT_EQUAL_TOTAL_CLASS_WEIGHT",
        },

        "HGB_D3_L15_BALANCED_V1": {
            "type":
                "HIST_GRADIENT_BOOSTING",

            "learning_rate":
                0.05,

            "max_iter":
                120,

            "max_depth":
                3,

            "max_leaf_nodes":
                15,

            "min_samples_leaf":
                20,

            "l2_regularization":
                1.0,

            "early_stopping":
                False,

            "random_state":
                20260904,

            "class_balance":
                "EXPLICIT_EQUAL_TOTAL_CLASS_WEIGHT",
        },
    },

    "preprocessing": {
        "nonfinite":
            "PLUS_MINUS_INF_TO_MISSING",

        "imputation":
            "TRAIN_FOLD_MEDIAN_ONLY",

        "missingness":
            "APPEND_ONE_MISSING_INDICATOR_PER_RAW_FEATURE",

        "logistic_scaling":
            "TRAIN_FOLD_STANDARDIZATION_OF_IMPUTED_RAW_VALUES",

        "hgb_scaling":
            "NONE",

        "heldout_statistics_used":
            False,
    },

    "cross_validation": {
        "type":
            "LEAVE_ONE_DEVELOPMENT_FOLD_OUT",

        "folds": [
            1,
            2,
            3,
            4,
        ],

        "fold0_used":
            False,
    },

    "training_label_semantics": {
        "positive":
            "KNOWN_DEVELOPMENT_CLEAN_GAP_ORACLE_POSITIVE",

        "background":
            "GT_BLIND_HASH_SAMPLED_SHORTLIST_BACKGROUND",

        "background_is_confirmed_negative":
            False,

        "model_score_is_calibrated_probability":
            False,
    },

    "local_rank_tie_break": {
        "primary":
            "MODEL_SCORE_DESC",

        "secondary":
            "GMEAN_RATIO_ASC",

        "tertiary":
            "GAP_SIZE_ASC",

        "final":
            "COUNTERPART_NODE_ID_ASC",
    },

    "primary_metric": {
        "population":
            "PRED_FULL_7UM_TRUE_PAIRS",

        "metric":
            "ASYM_SCORE_1x5_RECALL",

        "definition":
            (
                "(SOURCE_SCORE_RANK<=1 AND TARGET_SCORE_RANK<=5)"
                " OR "
                "(TARGET_SCORE_RANK<=1 AND SOURCE_SCORE_RANK<=5)"
            ),

        "aggregation":
            "MACRO_MEAN_ACROSS_FOLDS",
    },

    "secondary_metrics": [
        "BEST_ENDPOINT_TOP1",
        "BEST_ENDPOINT_TOP3",
        "BEST_ENDPOINT_TOP5",
        "BOTH_ENDPOINT_TOP3",
        "BOTH_ENDPOINT_TOP5",
        "ASYM_SCORE_1x3",
        "ASYM_SCORE_1x5",
        "ASYM_SCORE_3x8",
    ],

    "model_selected":
        False,

    "score_threshold_selected":
        False,

    "rewire_policy_selected":
        False,

    "graph_modified":
        False,
}


contract_bytes = json.dumps(
    BENCHMARK_CONTRACT,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
).encode(
    "utf-8"
)


CONTRACT_SHA = hashlib.sha256(
    contract_bytes
).hexdigest()


BENCHMARK_CONTRACT[
    "benchmark_contract_sha256"
] = CONTRACT_SHA


CONTRACT_OUT.write_text(
    json.dumps(
        BENCHMARK_CONTRACT,
        indent=2,
    ),
    encoding="utf-8",
)


assert CONTRACT_OUT.exists()


print(
    "\n========== G3H1 FROZEN BENCHMARK CONTRACT =========="
)

print(
    "benchmark ID:",
    BENCHMARK_CONTRACT[
        "benchmark_id"
    ]
)

print(
    "contract SHA:",
    CONTRACT_SHA
)

print(
    "models:",
    MODEL_IDS
)

print(
    "feature subset selected:",
    "NO"
)

print(
    "primary metric:",
    "FULL7 ASYM_SCORE_1x5_RECALL"
)

print(
    "CONTRACT FROZEN BEFORE DEVELOPMENT POSITIVE LOAD: YES"
)


# ============================================================
# PART B — DEVELOPMENT LABELS MAY NOW BE READ
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — LOFO TRAINING + FULL-SHORTLIST OOF RANKING"
)

print(
    "============================================================"
)


positive = pd.read_pickle(
    POSITIVE_FEATURES_PATH
)


assert len(
    positive
) == EXPECTED_POSITIVES


assert int(
    positive[
        "pred_full_7um"
    ].sum()
) == EXPECTED_FULL7_POSITIVES


assert set(
    positive[
        "fold"
    ].astype(
        int
    )
) == {
    1,
    2,
    3,
    4,
}


for feature in FEATURE_COLUMNS:

    assert feature in positive.columns


# ============================================================
# 7. REMOVE KNOWN POSITIVES FROM GT-BLIND SAMPLE
# ============================================================

positive_keys = (
    positive[
        [
            "dataset",
            "source_id",
            "target_id",
        ]
    ]
    .drop_duplicates()
    .copy()
)


positive_keys[
    "__positive"
] = 1


sample_background = sample.merge(
    positive_keys,
    on=[
        "dataset",
        "source_id",
        "target_id",
    ],
    how="left",
    validate="one_to_one",
)


known_overlap = int(
    sample_background[
        "__positive"
    ].notna().sum()
)


assert known_overlap == 1


sample_background = (
    sample_background[
        sample_background[
            "__positive"
        ].isna()
    ]
    .drop(
        columns=[
            "__positive",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(
    sample_background
) == 273_657


print(
    "\n========== TRAINING POPULATION =========="
)

print(
    "known positives:",
    len(
        positive
    )
)

print(
    "full<=7 positives:",
    int(
        positive[
            "pred_full_7um"
        ].sum()
    )
)

print(
    "sample background:",
    len(
        sample_background
    )
)

print(
    "background confirmed-negative:",
    "NO"
)


# ============================================================
# 8. FIXED FEATURE SANITIZATION
# ============================================================

def raw_feature_matrix(
    frame,
):

    X = (
        frame[
            FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .to_numpy(
            dtype=np.float64
        )
    )


    X[
        ~np.isfinite(
            X
        )
    ] = np.nan


    return X


def fit_preprocessor(
    frame,
):

    X = raw_feature_matrix(
        frame
    )


    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore",
            category=RuntimeWarning,
        )


        medians = np.nanmedian(
            X,
            axis=0,
        )


    # Any feature missing for the entire training split:
    # deterministic zero fallback.
    medians[
        ~np.isfinite(
            medians
        )
    ] = 0.0


    missing = np.isnan(
        X
    )


    filled = np.where(
        missing,
        medians[
            None,
            :
        ],
        X,
    )


    means = filled.mean(
        axis=0
    )


    stds = filled.std(
        axis=0
    )


    stds[
        (
            ~np.isfinite(
                stds
            )
            |
            (
                stds
                <
                1e-12
            )
        )
    ] = 1.0


    return {
        "medians":
            medians.astype(
                np.float64
            ),

        "means":
            means.astype(
                np.float64
            ),

        "stds":
            stds.astype(
                np.float64
            ),
    }


def transform_features(
    frame,
    preprocessing,
    standardize,
):

    X = raw_feature_matrix(
        frame
    )


    missing = np.isnan(
        X
    )


    filled = np.where(
        missing,
        preprocessing[
            "medians"
        ][
            None,
            :
        ],
        X,
    )


    if standardize:

        values = (
            filled
            -
            preprocessing[
                "means"
            ][
                None,
                :
            ]
        ) / preprocessing[
            "stds"
        ][
            None,
            :
        ]


    else:

        values = filled


    design = np.concatenate(
        [
            values,
            missing.astype(
                np.float64
            ),
        ],
        axis=1,
    )


    assert design.shape[
        1
    ] == (
        2
        *
        len(
            FEATURE_COLUMNS
        )
    )


    assert np.isfinite(
        design
    ).all()


    return design.astype(
        np.float32,
        copy=False,
    )


# ============================================================
# 9. BALANCED SAMPLE WEIGHTS
# ============================================================

def balanced_weights(
    labels,
):

    y = np.asarray(
        labels,
        dtype=np.int8,
    )


    n = len(
        y
    )


    n_pos = int(
        (
            y
            ==
            1
        ).sum()
    )


    n_bg = int(
        (
            y
            ==
            0
        ).sum()
    )


    assert n_pos > 0
    assert n_bg > 0


    pos_weight = (
        n
        /
        (
            2.0
            *
            n_pos
        )
    )


    bg_weight = (
        n
        /
        (
            2.0
            *
            n_bg
        )
    )


    w = np.where(
        y == 1,
        pos_weight,
        bg_weight,
    )


    return w.astype(
        np.float64
    )


# ============================================================
# 10. TRAIN FOUR LOFO MODEL BUNDLES
# ============================================================

fold_bundles = {}

training_rows = []


for heldout_fold in [
    1,
    2,
    3,
    4,
]:

    train_background = sample_background[
        sample_background[
            "fold"
        ].ne(
            heldout_fold
        )
    ]


    train_positive = positive[
        positive[
            "fold"
        ].ne(
            heldout_fold
        )
    ]


    # --------------------------------------------------------
    # Never use held-out fold during training.
    # --------------------------------------------------------

    assert not train_background[
        "fold"
    ].eq(
        heldout_fold
    ).any()


    assert not train_positive[
        "fold"
    ].eq(
        heldout_fold
    ).any()


    train_frame = pd.concat(
        [
            train_background[
                FEATURE_COLUMNS
            ],

            train_positive[
                FEATURE_COLUMNS
            ],
        ],
        ignore_index=True,
    )


    y = np.concatenate(
        [
            np.zeros(
                len(
                    train_background
                ),
                dtype=np.int8,
            ),

            np.ones(
                len(
                    train_positive
                ),
                dtype=np.int8,
            ),
        ]
    )


    weights = balanced_weights(
        y
    )


    preprocessing = fit_preprocessor(
        train_frame
    )


    # --------------------------------------------------------
    # Logistic design
    # --------------------------------------------------------

    X_logreg = transform_features(
        train_frame,
        preprocessing,
        standardize=True,
    )


    logreg = LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=2000,
        random_state=20260904,
    )


    logreg.fit(
        X_logreg,
        y,
        sample_weight=weights,
    )


    # --------------------------------------------------------
    # HGB design
    # --------------------------------------------------------

    X_hgb = transform_features(
        train_frame,
        preprocessing,
        standardize=False,
    )


    hgb = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=120,
        max_leaf_nodes=15,
        max_depth=3,
        min_samples_leaf=20,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=20260904,
    )


    hgb.fit(
        X_hgb,
        y,
        sample_weight=weights,
    )


    fold_bundles[
        heldout_fold
    ] = {
        "preprocessing":
            preprocessing,

        "LOGREG_L2_C1_BALANCED_V1":
            logreg,

        "HGB_D3_L15_BALANCED_V1":
            hgb,
    }


    training_rows.append(
        {
            "heldout_fold":
                int(
                    heldout_fold
                ),

            "background_train_rows":
                int(
                    len(
                        train_background
                    )
                ),

            "positive_train_rows":
                int(
                    len(
                        train_positive
                    )
                ),

            "logreg_n_iter":
                int(
                    np.max(
                        logreg.n_iter_
                    )
                ),
        }
    )


    print(
        f"heldout fold {heldout_fold}"
        f" | background train={len(train_background)}"
        f" | positive train={len(train_positive)}"
        f" | logreg_iter={int(np.max(logreg.n_iter_))}"
        f" | models=2"
    )


    del train_background
    del train_positive

    del train_frame

    del X_logreg
    del X_hgb

    del y
    del weights

    gc.collect()


training_table = pd.DataFrame(
    training_rows
)


print(
    "\n========== LOFO MODEL TRAINING =========="
)

print(
    training_table.to_string(
        index=False
    )
)

print(
    "OOF FITTING: PASS"
)


# ============================================================
# 11. BATCHED MODEL SCORING
# ============================================================

PREDICT_BATCH_ROWS = 50_000


def predict_model_scores(
    frame,
    fold_bundle,
):

    n = len(
        frame
    )


    logreg_scores = np.empty(
        n,
        dtype=np.float64,
    )


    hgb_scores = np.empty(
        n,
        dtype=np.float64,
    )


    for start in range(
        0,
        n,
        PREDICT_BATCH_ROWS,
    ):

        stop = min(
            start
            +
            PREDICT_BATCH_ROWS,
            n,
        )


        batch = frame.iloc[
            start:stop
        ]


        X_logreg = transform_features(
            batch,
            fold_bundle[
                "preprocessing"
            ],
            standardize=True,
        )


        X_hgb = transform_features(
            batch,
            fold_bundle[
                "preprocessing"
            ],
            standardize=False,
        )


        logreg_scores[
            start:stop
        ] = (
            fold_bundle[
                "LOGREG_L2_C1_BALANCED_V1"
            ]
            .predict_proba(
                X_logreg
            )[
                :,
                1
            ]
        )


        hgb_scores[
            start:stop
        ] = (
            fold_bundle[
                "HGB_D3_L15_BALANCED_V1"
            ]
            .predict_proba(
                X_hgb
            )[
                :,
                1
            ]
        )


        del X_logreg
        del X_hgb
        del batch


    return (
        logreg_scores,
        hgb_scores,
    )


# ============================================================
# 12. DETERMINISTIC SCORE ENDPOINT RANK
#
# Higher score = better.
#
# Tie:
#   GMEAN_RATIO ascending
#   gap_size ascending
#   counterpart ID ascending
# ============================================================

def add_score_endpoint_ranks(
    frame,
    score_col,
):

    result = pd.DataFrame(
        index=frame.index
    )


    # --------------------------------------------------------
    # Source side
    # --------------------------------------------------------

    source_order = (
        frame[
            [
                "source_id",
                "target_id",
                "gap_size",
                "GMEAN_RATIO",
                score_col,
            ]
        ]
        .sort_values(
            [
                "source_id",
                score_col,
                "GMEAN_RATIO",
                "gap_size",
                "target_id",
            ],
            ascending=[
                True,
                False,
                True,
                True,
                True,
            ],
            kind="stable",
        )
    )


    source_order[
        "__rank"
    ] = (
        source_order.groupby(
            "source_id",
            sort=False,
        )
        .cumcount()
        +
        1
    )


    result[
        "source_score_rank"
    ] = (
        source_order[
            "__rank"
        ]
        .reindex(
            frame.index
        )
        .astype(
            np.int32
        )
    )


    # --------------------------------------------------------
    # Target side
    # --------------------------------------------------------

    target_order = (
        frame[
            [
                "source_id",
                "target_id",
                "gap_size",
                "GMEAN_RATIO",
                score_col,
            ]
        ]
        .sort_values(
            [
                "target_id",
                score_col,
                "GMEAN_RATIO",
                "gap_size",
                "source_id",
            ],
            ascending=[
                True,
                False,
                True,
                True,
                True,
            ],
            kind="stable",
        )
    )


    target_order[
        "__rank"
    ] = (
        target_order.groupby(
            "target_id",
            sort=False,
        )
        .cumcount()
        +
        1
    )


    result[
        "target_score_rank"
    ] = (
        target_order[
            "__rank"
        ]
        .reindex(
            frame.index
        )
        .astype(
            np.int32
        )
    )


    result[
        "best_score_rank"
    ] = np.minimum(
        result[
            "source_score_rank"
        ],
        result[
            "target_score_rank"
        ],
    ).astype(
        np.int32
    )


    result[
        "max_score_rank"
    ] = np.maximum(
        result[
            "source_score_rank"
        ],
        result[
            "target_score_rank"
        ],
    ).astype(
        np.int32
    )


    del source_order
    del target_order


    return result


# ============================================================
# 13. DEVELOPMENT CORPUS
# ============================================================

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


cv = pd.read_csv(
    CV_PATH
)


dev_manifest = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    dev_manifest
) == 103


development_names = dev_manifest[
    "dataset"
].tolist()


fold_map_local = dict(
    zip(
        dev_manifest[
            "dataset"
        ],
        dev_manifest[
            "fold"
        ].astype(
            int
        ),
    )
)


# ============================================================
# 14. POSITIVE LOOKUP
# ============================================================

POSITIVE_META_COLUMNS = [
    "dataset",
    "fold",
    "component_id",
    "source_id",
    "target_id",
    "pred_full_7um",
    "topology_class",
]


positive_lookup = positive[
    POSITIVE_META_COLUMNS
].copy()


positive_dataset_set = set(
    positive_lookup[
        "dataset"
    ]
)


# ============================================================
# 15. FULL FROZEN SHORTLIST OOF SCORING
#
# Each dataset is regenerated ONCE.
#
# Its fold-specific models were trained on the other 3 folds.
# ============================================================

oof_positive_frames = []

global_shortlist_count = 0


for ds_i, dataset in enumerate(
    development_names,
    start=1,
):

    heldout_fold = int(
        fold_map_local[
            dataset
        ]
    )


    (
        proposals,
        _,
        _,
    ) = generate_dataset_proposals(
        dataset
    )


    proposals = add_endpoint_ranks(
        proposals
    )


    proposals = add_shortlist_rank_fields(
        proposals
    )


    shortlist = (
        proposals[
            frozen_shortlist_mask(
                proposals
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    shortlist = attach_scoring_features(
        shortlist
    )


    global_shortlist_count += int(
        len(
            shortlist
        )
    )


    # --------------------------------------------------------
    # BASELINE
    #
    # lower GMEAN is better -> negative score.
    # inf becomes -inf and ranks last.
    # --------------------------------------------------------

    gmean = pd.to_numeric(
        shortlist[
            "GMEAN_RATIO"
        ],
        errors="coerce",
    ).to_numpy(
        dtype=np.float64
    )


    gmean[
        np.isnan(
            gmean
        )
    ] = np.inf


    shortlist[
        "__score_BASE_GMEAN_RATIO_V1"
    ] = -gmean


    # --------------------------------------------------------
    # LOFO model scores
    # --------------------------------------------------------

    (
        logreg_scores,
        hgb_scores,
    ) = predict_model_scores(
        shortlist,
        fold_bundles[
            heldout_fold
        ],
    )


    shortlist[
        "__score_LOGREG_L2_C1_BALANCED_V1"
    ] = logreg_scores


    shortlist[
        "__score_HGB_D3_L15_BALANCED_V1"
    ] = hgb_scores


    # --------------------------------------------------------
    # Extract positive local ranks.
    # --------------------------------------------------------

    if dataset in positive_dataset_set:

        dataset_positive = positive_lookup[
            positive_lookup[
                "dataset"
            ].eq(
                dataset
            )
        ]


        for model_id in MODEL_IDS:

            score_col = (
                "__score_"
                +
                model_id
            )


            ranks = add_score_endpoint_ranks(
                shortlist,
                score_col,
            )


            local = shortlist[
                [
                    "source_id",
                    "target_id",
                    "gap_size",
                    "cuts",
                    "GMEAN_RATIO",
                    score_col,
                ]
            ].copy()


            local[
                "source_score_rank"
            ] = ranks[
                "source_score_rank"
            ]


            local[
                "target_score_rank"
            ] = ranks[
                "target_score_rank"
            ]


            local[
                "best_score_rank"
            ] = ranks[
                "best_score_rank"
            ]


            local[
                "max_score_rank"
            ] = ranks[
                "max_score_rank"
            ]


            matched = dataset_positive.merge(
                local,
                on=[
                    "source_id",
                    "target_id",
                ],
                how="left",
                validate="one_to_one",
            )


            assert len(
                matched
            ) == len(
                dataset_positive
            )


            assert matched[
                "source_score_rank"
            ].notna().all(), (
                dataset,
                model_id,
            )


            matched[
                "model_id"
            ] = model_id


            matched[
                "model_score"
            ] = matched[
                score_col
            ]


            matched = matched.drop(
                columns=[
                    score_col,
                ]
            )


            oof_positive_frames.append(
                matched
            )


            del ranks
            del local
            del matched


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            development_names
        )
    ):

        npos = int(
            (
                positive_lookup[
                    "dataset"
                ]
                ==
                dataset
            ).sum()
        )


        print(
            f"{ds_i:3d}/103"
            f" | fold={heldout_fold}"
            f" | {dataset}"
            f" | shortlist={len(shortlist)}"
            f" | positives={npos}"
        )


    del proposals
    del shortlist

    del logreg_scores
    del hgb_scores
    del gmean

    gc.collect()


assert global_shortlist_count == EXPECTED_SHORTLIST_ROWS, (
    global_shortlist_count,
    EXPECTED_SHORTLIST_ROWS,
)


oof_positive = pd.concat(
    oof_positive_frames,
    ignore_index=True,
)


del oof_positive_frames

gc.collect()


assert len(
    oof_positive
) == (
    EXPECTED_POSITIVES
    *
    len(
        MODEL_IDS
    )
)


assert set(
    oof_positive[
        "model_id"
    ]
) == set(
    MODEL_IDS
)


assert not oof_positive[
    [
        "model_id",
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


# ============================================================
# 16. LOCAL-RANK BOOLEAN METRICS
# ============================================================

src_rank = oof_positive[
    "source_score_rank"
].astype(
    int
)


tgt_rank = oof_positive[
    "target_score_rank"
].astype(
    int
)


best_rank = oof_positive[
    "best_score_rank"
].astype(
    int
)


max_rank = oof_positive[
    "max_score_rank"
].astype(
    int
)


for k in [
    1,
    2,
    3,
    5,
    8,
]:

    oof_positive[
        f"BEST_TOP{k}"
    ] = (
        best_rank
        <=
        k
    )


    oof_positive[
        f"BOTH_TOP{k}"
    ] = (
        max_rank
        <=
        k
    )


oof_positive[
    "ASYM_1x3"
] = (
    (
        src_rank.le(
            1
        )
        &
        tgt_rank.le(
            3
        )
    )
    |
    (
        tgt_rank.le(
            1
        )
        &
        src_rank.le(
            3
        )
    )
)


oof_positive[
    "ASYM_1x5"
] = (
    (
        src_rank.le(
            1
        )
        &
        tgt_rank.le(
            5
        )
    )
    |
    (
        tgt_rank.le(
            1
        )
        &
        src_rank.le(
            5
        )
    )
)


oof_positive[
    "ASYM_3x8"
] = (
    (
        src_rank.le(
            3
        )
        &
        tgt_rank.le(
            8
        )
    )
    |
    (
        tgt_rank.le(
            3
        )
        &
        src_rank.le(
            8
        )
    )
)


# ============================================================
# 17. FOLD METRICS
# ============================================================

metric_rows = []


def recall_bool(
    frame,
    col,
):

    if len(
        frame
    ) == 0:

        return np.nan


    return float(
        frame[
            col
        ].astype(
            bool
        ).mean()
    )


for model_id in MODEL_IDS:

    model_rows = oof_positive[
        oof_positive[
            "model_id"
        ].eq(
            model_id
        )
    ]


    for fold in [
        1,
        2,
        3,
        4,
    ]:

        fold_rows = model_rows[
            model_rows[
                "fold"
            ].eq(
                fold
            )
        ]


        full7_rows = fold_rows[
            fold_rows[
                "pred_full_7um"
            ].astype(
                bool
            )
        ]


        assert len(
            fold_rows
        ) > 0


        assert len(
            full7_rows
        ) > 0


        metric_rows.append(
            {
                "model_id":
                    model_id,

                "fold":
                    int(
                        fold
                    ),

                "positive_n":
                    int(
                        len(
                            fold_rows
                        )
                    ),

                "full7_n":
                    int(
                        len(
                            full7_rows
                        )
                    ),

                # --------------------------------------------
                # all-positive
                # --------------------------------------------

                "best_top1":
                    recall_bool(
                        fold_rows,
                        "BEST_TOP1",
                    ),

                "best_top3":
                    recall_bool(
                        fold_rows,
                        "BEST_TOP3",
                    ),

                "best_top5":
                    recall_bool(
                        fold_rows,
                        "BEST_TOP5",
                    ),

                "both_top3":
                    recall_bool(
                        fold_rows,
                        "BOTH_TOP3",
                    ),

                "both_top5":
                    recall_bool(
                        fold_rows,
                        "BOTH_TOP5",
                    ),

                "asym_1x3":
                    recall_bool(
                        fold_rows,
                        "ASYM_1x3",
                    ),

                "asym_1x5":
                    recall_bool(
                        fold_rows,
                        "ASYM_1x5",
                    ),

                "asym_3x8":
                    recall_bool(
                        fold_rows,
                        "ASYM_3x8",
                    ),

                # --------------------------------------------
                # FULL<=7 subset
                # --------------------------------------------

                "full7_best_top1":
                    recall_bool(
                        full7_rows,
                        "BEST_TOP1",
                    ),

                "full7_best_top3":
                    recall_bool(
                        full7_rows,
                        "BEST_TOP3",
                    ),

                "full7_best_top5":
                    recall_bool(
                        full7_rows,
                        "BEST_TOP5",
                    ),

                "full7_both_top3":
                    recall_bool(
                        full7_rows,
                        "BOTH_TOP3",
                    ),

                "full7_both_top5":
                    recall_bool(
                        full7_rows,
                        "BOTH_TOP5",
                    ),

                "full7_asym_1x3":
                    recall_bool(
                        full7_rows,
                        "ASYM_1x3",
                    ),

                "full7_asym_1x5":
                    recall_bool(
                        full7_rows,
                        "ASYM_1x5",
                    ),

                "full7_asym_3x8":
                    recall_bool(
                        full7_rows,
                        "ASYM_3x8",
                    ),

                # --------------------------------------------
                # rank diagnostics
                # --------------------------------------------

                "median_source_rank":
                    float(
                        fold_rows[
                            "source_score_rank"
                        ].median()
                    ),

                "median_target_rank":
                    float(
                        fold_rows[
                            "target_score_rank"
                        ].median()
                    ),

                "median_best_rank":
                    float(
                        fold_rows[
                            "best_score_rank"
                        ].median()
                    ),

                "median_max_rank":
                    float(
                        fold_rows[
                            "max_score_rank"
                        ].median()
                    ),

                "full7_median_best_rank":
                    float(
                        full7_rows[
                            "best_score_rank"
                        ].median()
                    ),

                "full7_median_max_rank":
                    float(
                        full7_rows[
                            "max_score_rank"
                        ].median()
                    ),
            }
        )


fold_metrics = pd.DataFrame(
    metric_rows
)


# ============================================================
# 18. MODEL-LEVEL SUMMARY
#
# Primary:
#   macro mean FULL7 ASYM1x5 across folds.
#
# No model is frozen/selected here.
# ============================================================

summary_rows = []


for model_id in MODEL_IDS:

    rows = fold_metrics[
        fold_metrics[
            "model_id"
        ].eq(
            model_id
        )
    ]


    assert len(
        rows
    ) == 4


    primary = rows[
        "full7_asym_1x5"
    ].to_numpy(
        dtype=float
    )


    secondary_asym38 = rows[
        "full7_asym_3x8"
    ].to_numpy(
        dtype=float
    )


    both5 = rows[
        "full7_both_top5"
    ].to_numpy(
        dtype=float
    )


    best1 = rows[
        "full7_best_top1"
    ].to_numpy(
        dtype=float
    )


    summary_rows.append(
        {
            "model_id":
                model_id,

            "primary_full7_asym1x5_macro":
                float(
                    primary.mean()
                ),

            "primary_full7_asym1x5_worst_fold":
                float(
                    primary.min()
                ),

            "primary_full7_asym1x5_std":
                float(
                    primary.std()
                ),

            "full7_asym3x8_macro":
                float(
                    secondary_asym38.mean()
                ),

            "full7_asym3x8_worst_fold":
                float(
                    secondary_asym38.min()
                ),

            "full7_both_top5_macro":
                float(
                    both5.mean()
                ),

            "full7_best_top1_macro":
                float(
                    best1.mean()
                ),

            "median_of_fold_full7_best_rank":
                float(
                    rows[
                        "full7_median_best_rank"
                    ].median()
                ),

            "median_of_fold_full7_max_rank":
                float(
                    rows[
                        "full7_median_max_rank"
                    ].median()
                ),
        }
    )


model_summary = (
    pd.DataFrame(
        summary_rows
    )
    .sort_values(
        [
            "primary_full7_asym1x5_macro",
            "primary_full7_asym1x5_worst_fold",
            "full7_asym3x8_macro",
        ],
        ascending=[
            False,
            False,
            False,
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


leader = str(
    model_summary.iloc[
        0
    ][
        "model_id"
    ]
)


# ============================================================
# 19. PRINT RESULTS
# ============================================================

print(
    "\n========== OOF FOLD METRICS =========="
)


print(
    fold_metrics[
        [
            "model_id",
            "fold",

            "positive_n",
            "full7_n",

            "full7_best_top1",
            "full7_best_top3",
            "full7_both_top3",
            "full7_both_top5",

            "full7_asym_1x3",
            "full7_asym_1x5",
            "full7_asym_3x8",

            "full7_median_best_rank",
            "full7_median_max_rank",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n========== OOF MODEL SUMMARY =========="
)


print(
    model_summary.to_string(
        index=False
    )
)


# ============================================================
# 20. BASELINE DELTAS
# ============================================================

baseline_row = model_summary[
    model_summary[
        "model_id"
    ].eq(
        "BASE_GMEAN_RATIO_V1"
    )
].iloc[
    0
]


comparison_rows = []


for _, row in model_summary.iterrows():

    comparison_rows.append(
        {
            "model_id":
                row[
                    "model_id"
                ],

            "primary_macro":
                float(
                    row[
                        "primary_full7_asym1x5_macro"
                    ]
                ),

            "delta_vs_gmean_primary":
                float(
                    row[
                        "primary_full7_asym1x5_macro"
                    ]
                    -
                    baseline_row[
                        "primary_full7_asym1x5_macro"
                    ]
                ),

            "worst_fold":
                float(
                    row[
                        "primary_full7_asym1x5_worst_fold"
                    ]
                ),

            "delta_vs_gmean_asym3x8":
                float(
                    row[
                        "full7_asym3x8_macro"
                    ]
                    -
                    baseline_row[
                        "full7_asym3x8_macro"
                    ]
                ),

            "delta_vs_gmean_both5":
                float(
                    row[
                        "full7_both_top5_macro"
                    ]
                    -
                    baseline_row[
                        "full7_both_top5_macro"
                    ]
                ),
        }
    )


comparison = pd.DataFrame(
    comparison_rows
)


print(
    "\n========== DELTA VS GMEAN BASELINE =========="
)


print(
    comparison.to_string(
        index=False
    )
)


# ============================================================
# 21. PERSIST
# ============================================================

oof_positive.to_pickle(
    OOF_POSITIVE_RANKS_OUT
)


fold_metrics.to_pickle(
    FOLD_METRICS_OUT
)


MODEL_SUMMARY = {
    "stage":
        "12.11G3H1",

    "status":
        "LEAVE_ONE_FOLD_OUT_SCORER_BENCHMARK_COMPLETE",

    "benchmark_contract_sha256":
        CONTRACT_SHA,

    "parent_generator_sha256":
        GENERATOR_SHA,

    "parent_shortlist_sha256":
        SHORTLIST_SHA,

    "feature_schema_sha256":
        FEATURE_SCHEMA_SHA,

    "shortlist_rows_reproduced":
        int(
            global_shortlist_count
        ),

    "expected_shortlist_rows":
        EXPECTED_SHORTLIST_ROWS,

    "positive_rows":
        EXPECTED_POSITIVES,

    "full7_positive_rows":
        EXPECTED_FULL7_POSITIVES,

    "background_rows":
        int(
            len(
                sample_background
            )
        ),

    "background_is_confirmed_negative":
        False,

    "primary_metric":
        "FULL7_ASYM_SCORE_1x5_MACRO_RECALL",

    "model_summary":
        model_summary.to_dict(
            "records"
        ),

    "delta_vs_gmean":
        comparison.to_dict(
            "records"
        ),

    "diagnostic_leader":
        leader,

    "diagnostic_leader_is_frozen_final_scorer":
        False,

    "model_selected":
        False,

    "feature_subset_selected":
        False,

    "score_threshold_selected":
        False,

    "rewire_policy_selected":
        False,

    "fold0_used":
        False,

    "graph_modified":
        False,

    "next":
        (
            "REVIEW_OOF_LOCAL_RANK_GAIN_AND_"
            "FREEZE_OR_REJECT_SCORER_FAMILY"
        ),
}


MODEL_SUMMARY_OUT.write_text(
    json.dumps(
        MODEL_SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


for path in [
    OOF_POSITIVE_RANKS_OUT,
    FOLD_METRICS_OUT,
    MODEL_SUMMARY_OUT,
]:

    assert path.exists(), path


# ============================================================
# 22. RELOAD INTEGRITY
# ============================================================

oof_check = pd.read_pickle(
    OOF_POSITIVE_RANKS_OUT
)


assert len(
    oof_check
) == (
    202
    *
    len(
        MODEL_IDS
    )
)


fold_check = pd.read_pickle(
    FOLD_METRICS_OUT
)


assert len(
    fold_check
) == (
    4
    *
    len(
        MODEL_IDS
    )
)


summary_check = json.loads(
    MODEL_SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert summary_check[
    "shortlist_rows_reproduced"
] == EXPECTED_SHORTLIST_ROWS


assert summary_check[
    "fold0_used"
] is False


assert summary_check[
    "model_selected"
] is False


# ============================================================
# 23. FINAL DECISION
# ============================================================

print(
    "\n========== 12.11G3H1 DECISION =========="
)

print(
    "LEAVE_ONE_FOLD_OUT_SCORER_BENCHMARK: PASS"
)

print(
    "benchmark contract SHA:",
    CONTRACT_SHA
)

print(
    "full frozen shortlist reproduced:",
    global_shortlist_count,
    "/",
    EXPECTED_SHORTLIST_ROWS
)

print(
    "OOF positives:",
    EXPECTED_POSITIVES
)

print(
    "OOF full<=7 positives:",
    EXPECTED_FULL7_POSITIVES
)

print(
    "diagnostic leader:",
    leader
)

print(
    "diagnostic leader frozen as final scorer:",
    "NO"
)

print(
    "background treated as confirmed negatives:",
    "NO"
)

print(
    "feature subset selected:",
    "NO"
)

print(
    "score threshold selected:",
    "NO"
)

print(
    "rewire policy selected:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "graph modified:",
    "NO"
)

print(
    "next:"
)

print(
    "REVIEW_OOF_LOCAL_RANK_GAIN_AND_FREEZE_OR_REJECT_SCORER_FAMILY"
)

In [ ]:
# ============================================================
# Stage 12.11G3H1-R1
#
# Resume G3H1 after:
#
#   1. read-only NumPy view failure
#   2. contract Fold0 field-path recovery bug
#
# ------------------------------------------------------------
# ORIGINAL G3H1 STATE
# ------------------------------------------------------------
#
# Frozen benchmark contract:
#   LOFO_SCORER_BENCHMARK_V1
#
# Contract SHA:
#   0ab41abd8cdb8efd6cf68d8c299f121078f879175267f915b79e48b1e19e8e36
#
# Original G3H1 already completed:
#
#   - contract freeze
#   - positive/background population construction
#   - four LOFO model fits
#
# Original failure:
#
#   gmean = Series.to_numpy(...)
#   gmean[np.isnan(gmean)] = np.inf
#
#   ValueError:
#       assignment destination is read-only
#
# Repair:
#
#   to_numpy(..., copy=True)
#
# First R1 recovery bug:
#
#   contract["fold0_used"]
#
# Correct frozen-contract location:
#
#   contract["cross_validation"]["fold0_used"]
#
# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------
#
# This cell:
#
#   - reuses already-trained fold_bundles
#   - DOES NOT retrain models
#   - DOES NOT rewrite frozen benchmark contract
#   - resumes full frozen-shortlist OOF scoring
#   - computes local endpoint ranks
#   - computes frozen benchmark metrics
#   - persists G3H1 formal outputs
#
# NO:
#
#   - Fold0
#   - model selection
#   - score threshold
#   - rewire policy
#   - graph modification
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import gc
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


CONTRACT_OUT = (
    S12
    / "stage12_folds14_rewire_scorer_benchmark_v1_frozen.json"
)


OOF_POSITIVE_RANKS_OUT = (
    S12
    / "stage12_folds14_rewire_scorer_oof_positive_ranks_v1.pkl"
)


FOLD_METRICS_OUT = (
    S12
    / "stage12_folds14_rewire_scorer_oof_fold_metrics_v1.pkl"
)


MODEL_SUMMARY_OUT = (
    S12
    / "stage12_folds14_rewire_scorer_oof_model_summary_v1.json"
)


CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


# ============================================================
# 1. FIXED CONTRACTS
# ============================================================

EXPECTED_CONTRACT_SHA = (
    "0ab41abd8cdb8efd6cf68d8c299f1210"
    "78f879175267f915b79e48b1e19e8e36"
)


EXPECTED_SHORTLIST_ROWS = 17_501_674

EXPECTED_POSITIVES = 202

EXPECTED_FULL7_POSITIVES = 173


MODEL_IDS = [
    "BASE_GMEAN_RATIO_V1",
    "LOGREG_L2_C1_BALANCED_V1",
    "HGB_D3_L15_BALANCED_V1",
]


# ============================================================
# 2. FROZEN CONTRACT RECOVERY
#
# Contract must already exist.
#
# IMPORTANT:
# Fold0 lives under:
#
#   contract["cross_validation"]["fold0_used"]
#
# ============================================================

assert CONTRACT_OUT.exists(), (
    "\nFrozen G3H1 benchmark contract is missing:\n"
    f"{CONTRACT_OUT}"
)


contract = json.loads(
    CONTRACT_OUT.read_text(
        encoding="utf-8"
    )
)


assert contract[
    "stage"
] == "12.11G3H1"


assert contract[
    "benchmark_id"
] == "LOFO_SCORER_BENCHMARK_V1"


assert contract[
    "benchmark_contract_sha256"
] == EXPECTED_CONTRACT_SHA


# ------------------------------------------------------------
# No policy/model-selection state may have been introduced.
# ------------------------------------------------------------

assert contract[
    "model_selected"
] is False


assert contract[
    "feature_subset_selected"
] is False


assert contract[
    "score_threshold_selected"
] is False


assert contract[
    "rewire_policy_selected"
] is False


assert contract[
    "graph_modified"
] is False


# ------------------------------------------------------------
# Correct Fold0 contract path
# ------------------------------------------------------------

assert contract[
    "cross_validation"
][
    "fold0_used"
] is False


assert contract[
    "cross_validation"
][
    "type"
] == "LEAVE_ONE_DEVELOPMENT_FOLD_OUT"


assert contract[
    "cross_validation"
][
    "folds"
] == [
    1,
    2,
    3,
    4,
]


# ------------------------------------------------------------
# Label semantics
# ------------------------------------------------------------

assert contract[
    "training_label_semantics"
][
    "background_is_confirmed_negative"
] is False


assert contract[
    "training_label_semantics"
][
    "model_score_is_calibrated_probability"
] is False


print(
    "========== G3H1-R1 CONTRACT RECOVERY =========="
)


print(
    "contract SHA:",
    contract[
        "benchmark_contract_sha256"
    ]
)


print(
    "contract reused:",
    "YES"
)


print(
    "contract rewritten:",
    "NO"
)


print(
    "cross-validation:",
    contract[
        "cross_validation"
    ][
        "type"
    ]
)


print(
    "folds:",
    contract[
        "cross_validation"
    ][
        "folds"
    ]
)


print(
    "Fold0 used:",
    contract[
        "cross_validation"
    ][
        "fold0_used"
    ]
)


print(
    "model selected:",
    contract[
        "model_selected"
    ]
)


print(
    "feature subset selected:",
    contract[
        "feature_subset_selected"
    ]
)


print(
    "score threshold selected:",
    contract[
        "score_threshold_selected"
    ]
)


print(
    "rewire policy selected:",
    contract[
        "rewire_policy_selected"
    ]
)


print(
    "CONTRACT FIDELITY: PASS"
)


# ============================================================
# 3. DOWNSTREAM OUTPUT STATE
#
# Original G3H1 crashed before these were written.
# They must still be absent.
# ============================================================

existing_downstream = [
    path

    for path in [
        OOF_POSITIVE_RANKS_OUT,
        FOLD_METRICS_OUT,
        MODEL_SUMMARY_OUT,
    ]

    if path.exists()
]


if existing_downstream:

    raise RuntimeError(
        "\nUnexpected existing G3H1 downstream artifacts:\n"
        +
        "\n".join(
            str(
                path
            )

            for path
            in existing_downstream
        )
        +
        "\n\nDo not overwrite them."
    )


print(
    "\n========== G3H1-R1 DOWNSTREAM STATE =========="
)


print(
    "OOF positive ranks:",
    "MISSING"
)


print(
    "fold metrics:",
    "MISSING"
)


print(
    "model summary:",
    "MISSING"
)


print(
    "SAFE_TO_RESUME_OOF_SCORING:",
    "YES"
)


# ============================================================
# 4. REQUIRED IN-MEMORY FUNCTIONS
#
# All were already defined before original G3H1 failure.
# ============================================================

REQUIRED_CALLABLES = [
    "generate_dataset_proposals",
    "add_endpoint_ranks",
    "add_shortlist_rank_fields",
    "frozen_shortlist_mask",
    "attach_scoring_features",
    "predict_model_scores",
    "add_score_endpoint_ranks",
]


missing_callables = [
    name

    for name in REQUIRED_CALLABLES

    if (
        name not in globals()
        or
        not callable(
            globals()[
                name
            ]
        )
    )
]


assert not missing_callables, (
    "\nMissing required G3H0/G3H1 functions:\n"
    f"{missing_callables}\n\n"
    "The kernel may have been restarted."
)


# ============================================================
# 5. TRAINED MODEL STATE
#
# DO NOT retrain.
# ============================================================

assert (
    "fold_bundles"
    in globals()
), (
    "\nfold_bundles is missing.\n"
    "The kernel may have been restarted."
)


assert set(
    fold_bundles.keys()
) == {
    1,
    2,
    3,
    4,
}


for fold in [
    1,
    2,
    3,
    4,
]:

    bundle = fold_bundles[
        fold
    ]


    assert (
        "preprocessing"
        in bundle
    )


    assert (
        "LOGREG_L2_C1_BALANCED_V1"
        in bundle
    )


    assert (
        "HGB_D3_L15_BALANCED_V1"
        in bundle
    )


print(
    "\n========== G3H1-R1 TRAINED MODEL STATE =========="
)


print(
    "LOFO bundles:",
    sorted(
        fold_bundles.keys()
    )
)


print(
    "models per fold:",
    [
        "LOGREG_L2_C1_BALANCED_V1",
        "HGB_D3_L15_BALANCED_V1",
    ]
)


print(
    "models retrained:",
    "NO"
)


print(
    "TRAINED MODEL RECOVERY: PASS"
)


# ============================================================
# 6. DEVELOPMENT POSITIVE TABLE
#
# Already loaded during original G3H1.
# ============================================================

assert (
    "positive"
    in globals()
), (
    "\npositive table missing from kernel.\n"
    "The kernel may have been restarted."
)


assert isinstance(
    positive,
    pd.DataFrame,
)


assert len(
    positive
) == EXPECTED_POSITIVES


assert int(
    positive[
        "pred_full_7um"
    ].sum()
) == EXPECTED_FULL7_POSITIVES


assert set(
    positive[
        "fold"
    ].astype(
        int
    )
) == {
    1,
    2,
    3,
    4,
}


print(
    "\n========== G3H1-R1 POSITIVE STATE =========="
)


print(
    "positive rows:",
    len(
        positive
    )
)


print(
    "full<=7 positives:",
    int(
        positive[
            "pred_full_7um"
        ].sum()
    )
)


print(
    "Fold0 positives:",
    int(
        positive[
            "fold"
        ].eq(
            0
        ).sum()
    )
)


assert not positive[
    "fold"
].eq(
    0
).any()


# ============================================================
# 7. DEVELOPMENT CORPUS
# ============================================================

assert CV_PATH.exists(), CV_PATH


cv = pd.read_csv(
    CV_PATH
)


dev_manifest = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    dev_manifest
) == 103


assert dev_manifest[
    "dataset"
].nunique() == 103


assert not dev_manifest[
    "fold"
].eq(
    0
).any()


development_names = (
    dev_manifest[
        "dataset"
    ]
    .tolist()
)


fold_map_local = dict(
    zip(
        dev_manifest[
            "dataset"
        ],

        dev_manifest[
            "fold"
        ].astype(
            int
        ),
    )
)


assert set(
    fold_map_local.values()
) == {
    1,
    2,
    3,
    4,
}


print(
    "\n========== G3H1-R1 DEVELOPMENT CORPUS =========="
)


print(
    "datasets:",
    len(
        development_names
    )
)


print(
    "folds:",
    sorted(
        set(
            fold_map_local.values()
        )
    )
)


print(
    "Fold0 used:",
    "NO"
)


# ============================================================
# 8. POSITIVE LOOKUP
# ============================================================

POSITIVE_META_COLUMNS = [
    "dataset",
    "fold",
    "component_id",
    "source_id",
    "target_id",
    "pred_full_7um",
    "topology_class",
]


for col in POSITIVE_META_COLUMNS:

    assert col in positive.columns, col


positive_lookup = (
    positive[
        POSITIVE_META_COLUMNS
    ]
    .copy()
)


assert len(
    positive_lookup
) == EXPECTED_POSITIVES


assert not positive_lookup[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


positive_dataset_set = set(
    positive_lookup[
        "dataset"
    ]
)


print(
    "positive datasets:",
    len(
        positive_dataset_set
    )
)


# ============================================================
# 9. FULL FROZEN SHORTLIST OOF SCORING
#
# IMPORTANT:
#
# One dataset at a time.
#
# Model for fold f was trained only on folds != f.
#
# Repair:
#
#   to_numpy(..., copy=True)
#
# ============================================================

print(
    "\n============================================================"
)


print(
    "G3H1-R1 — FULL-SHORTLIST OOF SCORING RESUME"
)


print(
    "============================================================"
)


oof_positive_frames = []


global_shortlist_count = 0


for ds_i, dataset in enumerate(
    development_names,
    start=1,
):

    heldout_fold = int(
        fold_map_local[
            dataset
        ]
    )


    assert heldout_fold in {
        1,
        2,
        3,
        4,
    }


    # ========================================================
    # Exact authoritative frozen proposal regeneration
    # ========================================================

    (
        proposals,
        _raw_count,
        _generator_count,
    ) = generate_dataset_proposals(
        dataset
    )


    proposals = add_endpoint_ranks(
        proposals
    )


    proposals = add_shortlist_rank_fields(
        proposals
    )


    shortlist = (
        proposals[
            frozen_shortlist_mask(
                proposals
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    shortlist = attach_scoring_features(
        shortlist
    )


    global_shortlist_count += int(
        len(
            shortlist
        )
    )


    # ========================================================
    # BASELINE:
    #
    # Lower GMEAN_RATIO = better.
    #
    # Score convention everywhere else:
    # higher score = better.
    #
    # Therefore:
    #   score = -GMEAN_RATIO
    #
    # READ-ONLY FIX:
    #   copy=True
    # ========================================================

    gmean = (
        pd.to_numeric(
            shortlist[
                "GMEAN_RATIO"
            ],
            errors="coerce",
        )
        .to_numpy(
            dtype=np.float64,
            copy=True,
        )
    )


    assert gmean.flags.writeable, (
        dataset,
        "GMEAN_RATIO array is unexpectedly read-only."
    )


    # NaN behaves as infinitely bad GMEAN.
    gmean[
        np.isnan(
            gmean
        )
    ] = np.inf


    shortlist[
        "__score_BASE_GMEAN_RATIO_V1"
    ] = -gmean


    # ========================================================
    # Learned LOFO scores
    #
    # Uses ONLY model bundle trained with this fold held out.
    # ========================================================

    (
        logreg_scores,
        hgb_scores,
    ) = predict_model_scores(
        shortlist,
        fold_bundles[
            heldout_fold
        ],
    )


    assert len(
        logreg_scores
    ) == len(
        shortlist
    )


    assert len(
        hgb_scores
    ) == len(
        shortlist
    )


    assert np.isfinite(
        logreg_scores
    ).all()


    assert np.isfinite(
        hgb_scores
    ).all()


    shortlist[
        "__score_LOGREG_L2_C1_BALANCED_V1"
    ] = logreg_scores


    shortlist[
        "__score_HGB_D3_L15_BALANCED_V1"
    ] = hgb_scores


    # ========================================================
    # Extract positive local endpoint ranks
    # ========================================================

    if dataset in positive_dataset_set:

        dataset_positive = (
            positive_lookup[
                positive_lookup[
                    "dataset"
                ].eq(
                    dataset
                )
            ]
            .copy()
        )


        assert len(
            dataset_positive
        ) > 0


        assert dataset_positive[
            "fold"
        ].eq(
            heldout_fold
        ).all()


        for model_id in MODEL_IDS:

            score_col = (
                "__score_"
                +
                model_id
            )


            assert score_col in shortlist.columns


            # ------------------------------------------------
            # Deterministic source/target endpoint ranks
            # ------------------------------------------------

            ranks = add_score_endpoint_ranks(
                shortlist,
                score_col,
            )


            assert len(
                ranks
            ) == len(
                shortlist
            )


            # ------------------------------------------------
            # Compact candidate lookup
            # ------------------------------------------------

            local = shortlist[
                [
                    "source_id",
                    "target_id",
                    "gap_size",
                    "cuts",
                    "GMEAN_RATIO",
                    score_col,
                ]
            ].copy()


            local[
                "source_score_rank"
            ] = ranks[
                "source_score_rank"
            ].to_numpy(
                copy=True
            )


            local[
                "target_score_rank"
            ] = ranks[
                "target_score_rank"
            ].to_numpy(
                copy=True
            )


            local[
                "best_score_rank"
            ] = ranks[
                "best_score_rank"
            ].to_numpy(
                copy=True
            )


            local[
                "max_score_rank"
            ] = ranks[
                "max_score_rank"
            ].to_numpy(
                copy=True
            )


            # ------------------------------------------------
            # Join only known true-pair rows
            # ------------------------------------------------

            matched = dataset_positive.merge(
                local,
                on=[
                    "source_id",
                    "target_id",
                ],
                how="left",
                validate="one_to_one",
            )


            assert len(
                matched
            ) == len(
                dataset_positive
            )


            assert matched[
                "source_score_rank"
            ].notna().all(), (
                dataset,
                model_id,
                "source rank unresolved",
            )


            assert matched[
                "target_score_rank"
            ].notna().all(), (
                dataset,
                model_id,
                "target rank unresolved",
            )


            assert matched[
                "best_score_rank"
            ].notna().all()


            assert matched[
                "max_score_rank"
            ].notna().all()


            matched[
                "model_id"
            ] = model_id


            matched[
                "model_score"
            ] = matched[
                score_col
            ].astype(
                float
            )


            matched = matched.drop(
                columns=[
                    score_col,
                ]
            )


            oof_positive_frames.append(
                matched
            )


            del ranks
            del local
            del matched


        del dataset_positive


    # ========================================================
    # Progress
    # ========================================================

    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == len(
            development_names
        )
    ):

        n_positive = int(
            positive_lookup[
                "dataset"
            ].eq(
                dataset
            ).sum()
        )


        print(
            f"{ds_i:3d}/103"
            f" | fold={heldout_fold}"
            f" | {dataset}"
            f" | raw={_raw_count}"
            f" | generator={_generator_count}"
            f" | shortlist={len(shortlist)}"
            f" | positives={n_positive}"
            f" | cumulative={global_shortlist_count}"
        )


    # ========================================================
    # Dataset cleanup
    # ========================================================

    del proposals
    del shortlist

    del gmean
    del logreg_scores
    del hgb_scores

    gc.collect()


# ============================================================
# 10. FULL-SHORTLIST GLOBAL FIDELITY
# ============================================================

assert global_shortlist_count == EXPECTED_SHORTLIST_ROWS, (
    "\nFull frozen shortlist fidelity failure\n"
    f"observed={global_shortlist_count}\n"
    f"expected={EXPECTED_SHORTLIST_ROWS}"
)


assert oof_positive_frames, (
    "No OOF positive rows were collected."
)


oof_positive = pd.concat(
    oof_positive_frames,
    ignore_index=True,
)


del oof_positive_frames

gc.collect()


EXPECTED_OOF_ROWS = (
    EXPECTED_POSITIVES
    *
    len(
        MODEL_IDS
    )
)


assert EXPECTED_OOF_ROWS == 606


assert len(
    oof_positive
) == EXPECTED_OOF_ROWS, (
    len(
        oof_positive
    ),
    EXPECTED_OOF_ROWS,
)


assert set(
    oof_positive[
        "model_id"
    ]
) == set(
    MODEL_IDS
)


assert not oof_positive[
    [
        "model_id",
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


for model_id in MODEL_IDS:

    assert int(
        oof_positive[
            "model_id"
        ].eq(
            model_id
        ).sum()
    ) == EXPECTED_POSITIVES


print(
    "\n========== G3H1-R1 FULL SHORTLIST FIDELITY =========="
)


print(
    "shortlist reproduced:",
    global_shortlist_count,
    "/",
    EXPECTED_SHORTLIST_ROWS
)


print(
    "OOF positive-model rows:",
    len(
        oof_positive
    ),
    "/",
    EXPECTED_OOF_ROWS
)


print(
    "models:",
    MODEL_IDS
)


print(
    "FULL_SHORTLIST_OOF_SCORING: PASS"
)


# ============================================================
# 11. LOCAL-RANK BOOLEAN METRICS
# ============================================================

src_rank = (
    oof_positive[
        "source_score_rank"
    ]
    .astype(
        int
    )
)


tgt_rank = (
    oof_positive[
        "target_score_rank"
    ]
    .astype(
        int
    )
)


best_rank = (
    oof_positive[
        "best_score_rank"
    ]
    .astype(
        int
    )
)


max_rank = (
    oof_positive[
        "max_score_rank"
    ]
    .astype(
        int
    )
)


# ------------------------------------------------------------
# Symmetric Top-K metrics
# ------------------------------------------------------------

for k in [
    1,
    2,
    3,
    5,
    8,
]:

    oof_positive[
        f"BEST_TOP{k}"
    ] = (
        best_rank
        <=
        k
    )


    oof_positive[
        f"BOTH_TOP{k}"
    ] = (
        max_rank
        <=
        k
    )


# ------------------------------------------------------------
# Frozen asymmetric diagnostics
# ------------------------------------------------------------

oof_positive[
    "ASYM_1x3"
] = (
    (
        src_rank.le(
            1
        )
        &
        tgt_rank.le(
            3
        )
    )
    |
    (
        tgt_rank.le(
            1
        )
        &
        src_rank.le(
            3
        )
    )
)


oof_positive[
    "ASYM_1x5"
] = (
    (
        src_rank.le(
            1
        )
        &
        tgt_rank.le(
            5
        )
    )
    |
    (
        tgt_rank.le(
            1
        )
        &
        src_rank.le(
            5
        )
    )
)


oof_positive[
    "ASYM_3x8"
] = (
    (
        src_rank.le(
            3
        )
        &
        tgt_rank.le(
            8
        )
    )
    |
    (
        tgt_rank.le(
            3
        )
        &
        src_rank.le(
            8
        )
    )
)


# ============================================================
# 12. METRIC HELPER
# ============================================================

def recall_bool(
    frame,
    col,
):

    if len(
        frame
    ) == 0:

        return np.nan


    return float(
        frame[
            col
        ]
        .astype(
            bool
        )
        .mean()
    )


# ============================================================
# 13. OOF FOLD METRICS
# ============================================================

metric_rows = []


for model_id in MODEL_IDS:

    model_rows = oof_positive[
        oof_positive[
            "model_id"
        ].eq(
            model_id
        )
    ]


    assert len(
        model_rows
    ) == EXPECTED_POSITIVES


    for fold in [
        1,
        2,
        3,
        4,
    ]:

        fold_rows = model_rows[
            model_rows[
                "fold"
            ].eq(
                fold
            )
        ]


        full7_rows = fold_rows[
            fold_rows[
                "pred_full_7um"
            ].astype(
                bool
            )
        ]


        assert len(
            fold_rows
        ) > 0


        assert len(
            full7_rows
        ) > 0


        metric_rows.append(
            {
                "model_id":
                    model_id,

                "fold":
                    int(
                        fold
                    ),

                "positive_n":
                    int(
                        len(
                            fold_rows
                        )
                    ),

                "full7_n":
                    int(
                        len(
                            full7_rows
                        )
                    ),

                # =================================================
                # ALL KNOWN TRUE PAIRS
                # =================================================

                "best_top1":
                    recall_bool(
                        fold_rows,
                        "BEST_TOP1",
                    ),

                "best_top3":
                    recall_bool(
                        fold_rows,
                        "BEST_TOP3",
                    ),

                "best_top5":
                    recall_bool(
                        fold_rows,
                        "BEST_TOP5",
                    ),

                "both_top3":
                    recall_bool(
                        fold_rows,
                        "BOTH_TOP3",
                    ),

                "both_top5":
                    recall_bool(
                        fold_rows,
                        "BOTH_TOP5",
                    ),

                "asym_1x3":
                    recall_bool(
                        fold_rows,
                        "ASYM_1x3",
                    ),

                "asym_1x5":
                    recall_bool(
                        fold_rows,
                        "ASYM_1x5",
                    ),

                "asym_3x8":
                    recall_bool(
                        fold_rows,
                        "ASYM_3x8",
                    ),

                # =================================================
                # FULL<=7 TRUE PAIRS
                # =================================================

                "full7_best_top1":
                    recall_bool(
                        full7_rows,
                        "BEST_TOP1",
                    ),

                "full7_best_top3":
                    recall_bool(
                        full7_rows,
                        "BEST_TOP3",
                    ),

                "full7_best_top5":
                    recall_bool(
                        full7_rows,
                        "BEST_TOP5",
                    ),

                "full7_both_top3":
                    recall_bool(
                        full7_rows,
                        "BOTH_TOP3",
                    ),

                "full7_both_top5":
                    recall_bool(
                        full7_rows,
                        "BOTH_TOP5",
                    ),

                "full7_asym_1x3":
                    recall_bool(
                        full7_rows,
                        "ASYM_1x3",
                    ),

                "full7_asym_1x5":
                    recall_bool(
                        full7_rows,
                        "ASYM_1x5",
                    ),

                "full7_asym_3x8":
                    recall_bool(
                        full7_rows,
                        "ASYM_3x8",
                    ),

                # =================================================
                # Rank diagnostics
                # =================================================

                "median_source_rank":
                    float(
                        fold_rows[
                            "source_score_rank"
                        ].median()
                    ),

                "median_target_rank":
                    float(
                        fold_rows[
                            "target_score_rank"
                        ].median()
                    ),

                "median_best_rank":
                    float(
                        fold_rows[
                            "best_score_rank"
                        ].median()
                    ),

                "median_max_rank":
                    float(
                        fold_rows[
                            "max_score_rank"
                        ].median()
                    ),

                "full7_median_best_rank":
                    float(
                        full7_rows[
                            "best_score_rank"
                        ].median()
                    ),

                "full7_median_max_rank":
                    float(
                        full7_rows[
                            "max_score_rank"
                        ].median()
                    ),
            }
        )


fold_metrics = pd.DataFrame(
    metric_rows
)


assert len(
    fold_metrics
) == (
    len(
        MODEL_IDS
    )
    *
    4
)


assert len(
    fold_metrics
) == 12


# ============================================================
# 14. MODEL-LEVEL SUMMARY
#
# Frozen primary benchmark:
#
#   macro mean across folds
#   FULL<=7 ASYM_SCORE_1x5 recall
#
# IMPORTANT:
#
# Sorting determines diagnostic leader only.
# It does NOT freeze a final scorer.
# ============================================================

summary_rows = []


for model_id in MODEL_IDS:

    rows = fold_metrics[
        fold_metrics[
            "model_id"
        ].eq(
            model_id
        )
    ]


    assert len(
        rows
    ) == 4


    primary = (
        rows[
            "full7_asym_1x5"
        ]
        .to_numpy(
            dtype=float
        )
    )


    asym38 = (
        rows[
            "full7_asym_3x8"
        ]
        .to_numpy(
            dtype=float
        )
    )


    both5 = (
        rows[
            "full7_both_top5"
        ]
        .to_numpy(
            dtype=float
        )
    )


    best1 = (
        rows[
            "full7_best_top1"
        ]
        .to_numpy(
            dtype=float
        )
    )


    assert np.isfinite(
        primary
    ).all()


    summary_rows.append(
        {
            "model_id":
                model_id,

            "primary_full7_asym1x5_macro":
                float(
                    primary.mean()
                ),

            "primary_full7_asym1x5_worst_fold":
                float(
                    primary.min()
                ),

            "primary_full7_asym1x5_std":
                float(
                    primary.std()
                ),

            "full7_asym3x8_macro":
                float(
                    asym38.mean()
                ),

            "full7_asym3x8_worst_fold":
                float(
                    asym38.min()
                ),

            "full7_both_top5_macro":
                float(
                    both5.mean()
                ),

            "full7_best_top1_macro":
                float(
                    best1.mean()
                ),

            "median_of_fold_full7_best_rank":
                float(
                    rows[
                        "full7_median_best_rank"
                    ].median()
                ),

            "median_of_fold_full7_max_rank":
                float(
                    rows[
                        "full7_median_max_rank"
                    ].median()
                ),
        }
    )


model_summary = (
    pd.DataFrame(
        summary_rows
    )
    .sort_values(
        [
            "primary_full7_asym1x5_macro",
            "primary_full7_asym1x5_worst_fold",
            "full7_asym3x8_macro",
        ],
        ascending=[
            False,
            False,
            False,
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    model_summary
) == len(
    MODEL_IDS
)


leader = str(
    model_summary.iloc[
        0
    ][
        "model_id"
    ]
)


# ============================================================
# 15. PRINT OOF FOLD RESULTS
# ============================================================

print(
    "\n========== OOF FOLD METRICS =========="
)


print(
    fold_metrics[
        [
            "model_id",
            "fold",

            "positive_n",
            "full7_n",

            "full7_best_top1",
            "full7_best_top3",
            "full7_best_top5",

            "full7_both_top3",
            "full7_both_top5",

            "full7_asym_1x3",
            "full7_asym_1x5",
            "full7_asym_3x8",

            "full7_median_best_rank",
            "full7_median_max_rank",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 16. PRINT MODEL SUMMARY
# ============================================================

print(
    "\n========== OOF MODEL SUMMARY =========="
)


print(
    model_summary.to_string(
        index=False
    )
)


# ============================================================
# 17. DELTA VS FROZEN GMEAN BASELINE
# ============================================================

baseline_match = model_summary[
    model_summary[
        "model_id"
    ].eq(
        "BASE_GMEAN_RATIO_V1"
    )
]


assert len(
    baseline_match
) == 1


baseline_row = baseline_match.iloc[
    0
]


comparison_rows = []


for _, row in model_summary.iterrows():

    comparison_rows.append(
        {
            "model_id":
                str(
                    row[
                        "model_id"
                    ]
                ),

            "primary_macro":
                float(
                    row[
                        "primary_full7_asym1x5_macro"
                    ]
                ),

            "delta_vs_gmean_primary":
                float(
                    row[
                        "primary_full7_asym1x5_macro"
                    ]
                    -
                    baseline_row[
                        "primary_full7_asym1x5_macro"
                    ]
                ),

            "worst_fold":
                float(
                    row[
                        "primary_full7_asym1x5_worst_fold"
                    ]
                ),

            "delta_vs_gmean_asym3x8":
                float(
                    row[
                        "full7_asym3x8_macro"
                    ]
                    -
                    baseline_row[
                        "full7_asym3x8_macro"
                    ]
                ),

            "delta_vs_gmean_both5":
                float(
                    row[
                        "full7_both_top5_macro"
                    ]
                    -
                    baseline_row[
                        "full7_both_top5_macro"
                    ]
                ),

            "delta_vs_gmean_best1":
                float(
                    row[
                        "full7_best_top1_macro"
                    ]
                    -
                    baseline_row[
                        "full7_best_top1_macro"
                    ]
                ),
        }
    )


comparison = pd.DataFrame(
    comparison_rows
)


print(
    "\n========== DELTA VS GMEAN BASELINE =========="
)


print(
    comparison.to_string(
        index=False
    )
)


# ============================================================
# 18. PERSIST FORMAL G3H1 OUTPUTS
# ============================================================

oof_positive.to_pickle(
    OOF_POSITIVE_RANKS_OUT
)


fold_metrics.to_pickle(
    FOLD_METRICS_OUT
)


MODEL_SUMMARY = {
    "stage":
        "12.11G3H1",

    "recovery_stage":
        "12.11G3H1-R1",

    "status":
        "LEAVE_ONE_FOLD_OUT_SCORER_BENCHMARK_COMPLETE",

    "repair_history": [
        (
            "READ_ONLY_GMEAN_NUMPY_VIEW_"
            "FIXED_WITH_EXPLICIT_COPY"
        ),

        (
            "FROZEN_CONTRACT_FOLD0_FIELD_"
            "READ_FROM_CROSS_VALIDATION_SUBOBJECT"
        ),
    ],

    "benchmark_contract_sha256":
        EXPECTED_CONTRACT_SHA,

    "benchmark_contract_rewritten":
        False,

    "models_retrained_during_recovery":
        False,

    "shortlist_rows_reproduced":
        int(
            global_shortlist_count
        ),

    "expected_shortlist_rows":
        int(
            EXPECTED_SHORTLIST_ROWS
        ),

    "positive_rows":
        int(
            EXPECTED_POSITIVES
        ),

    "full7_positive_rows":
        int(
            EXPECTED_FULL7_POSITIVES
        ),

    "primary_metric":
        "FULL7_ASYM_SCORE_1x5_MACRO_RECALL",

    "model_summary":
        model_summary.to_dict(
            "records"
        ),

    "delta_vs_gmean":
        comparison.to_dict(
            "records"
        ),

    "diagnostic_leader":
        leader,

    "diagnostic_leader_is_frozen_final_scorer":
        False,

    "model_selected":
        False,

    "feature_subset_selected":
        False,

    "score_threshold_selected":
        False,

    "rewire_policy_selected":
        False,

    "fold0_used":
        False,

    "graph_modified":
        False,

    "next":
        (
            "REVIEW_OOF_LOCAL_RANK_GAIN_AND_"
            "FREEZE_OR_REJECT_SCORER_FAMILY"
        ),
}


MODEL_SUMMARY_OUT.write_text(
    json.dumps(
        MODEL_SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


for path in [
    OOF_POSITIVE_RANKS_OUT,
    FOLD_METRICS_OUT,
    MODEL_SUMMARY_OUT,
]:

    assert path.exists(), path


# ============================================================
# 19. RELOAD INTEGRITY
# ============================================================

oof_check = pd.read_pickle(
    OOF_POSITIVE_RANKS_OUT
)


assert len(
    oof_check
) == EXPECTED_OOF_ROWS


assert not oof_check[
    [
        "model_id",
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


fold_check = pd.read_pickle(
    FOLD_METRICS_OUT
)


assert len(
    fold_check
) == 12


assert set(
    fold_check[
        "fold"
    ].astype(
        int
    )
) == {
    1,
    2,
    3,
    4,
}


summary_check = json.loads(
    MODEL_SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert summary_check[
    "status"
] == "LEAVE_ONE_FOLD_OUT_SCORER_BENCHMARK_COMPLETE"


assert summary_check[
    "shortlist_rows_reproduced"
] == EXPECTED_SHORTLIST_ROWS


assert summary_check[
    "expected_shortlist_rows"
] == EXPECTED_SHORTLIST_ROWS


assert summary_check[
    "positive_rows"
] == EXPECTED_POSITIVES


assert summary_check[
    "full7_positive_rows"
] == EXPECTED_FULL7_POSITIVES


assert summary_check[
    "fold0_used"
] is False


assert summary_check[
    "model_selected"
] is False


assert summary_check[
    "feature_subset_selected"
] is False


assert summary_check[
    "score_threshold_selected"
] is False


assert summary_check[
    "rewire_policy_selected"
] is False


assert summary_check[
    "graph_modified"
] is False


# ============================================================
# 20. FINAL DECISION
# ============================================================

print(
    "\n========== 12.11G3H1-R1 DECISION =========="
)


print(
    "LEAVE_ONE_FOLD_OUT_SCORER_BENCHMARK: PASS"
)


print(
    "benchmark contract SHA:",
    EXPECTED_CONTRACT_SHA
)


print(
    "benchmark contract preserved:",
    "YES"
)


print(
    "benchmark contract rewritten:",
    "NO"
)


print(
    "read-only NumPy repair:",
    "PASS"
)


print(
    "contract field-path repair:",
    "PASS"
)


print(
    "models retrained during recovery:",
    "NO"
)


print(
    "full frozen shortlist reproduced:",
    global_shortlist_count,
    "/",
    EXPECTED_SHORTLIST_ROWS
)


print(
    "OOF positive-model rows:",
    len(
        oof_positive
    ),
    "/",
    EXPECTED_OOF_ROWS
)


print(
    "OOF positives:",
    EXPECTED_POSITIVES
)


print(
    "OOF full<=7 positives:",
    EXPECTED_FULL7_POSITIVES
)


print(
    "diagnostic leader:",
    leader
)


print(
    "diagnostic leader frozen as final scorer:",
    "NO"
)


print(
    "feature subset selected:",
    "NO"
)


print(
    "score threshold selected:",
    "NO"
)


print(
    "rewire policy selected:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "graph modified:",
    "NO"
)


print(
    "next:"
)


print(
    "REVIEW_OOF_LOCAL_RANK_GAIN_AND_FREEZE_OR_REJECT_SCORER_FAMILY"
)

In [ ]:
# ============================================================
# Stage 12.11G3H2
#
# Scorer Family Freeze + Full-Development Fit
#
# ------------------------------------------------------------
# UPSTREAM DECISION
# ------------------------------------------------------------
#
# G3H1 LOFO benchmark:
#
# LOGREG_L2_C1_BALANCED_V1
#
#   primary FULL7 ASYM1x5 macro:
#       0.794094
#
#   worst fold:
#       0.720930
#
#   FULL7 ASYM3x8 macro:
#       0.955917
#
#   FULL7 BOTH_TOP5 macro:
#       0.924393
#
#   FULL7 BEST_TOP1 macro:
#       0.816638
#
# LogReg beats:
#
#   - GMEAN baseline on all 4 folds
#   - HGB benchmark on primary metric on all 4 folds
#
# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# 1. Freeze scorer FAMILY selection.
#
# 2. Fit one full-development scorer using folds 1–4 only.
#
# 3. Persist:
#       scorer-selection contract
#       full-dev preprocessing
#       fitted sklearn model
#       coefficient audit
#       training summary
#
# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------
#
# This stage DOES NOT:
#
#   - select score threshold
#   - select endpoint conflict rule
#   - select rewire action policy
#   - modify graph
#   - use Fold0
#
# Scores remain SURROGATE RANKING SCORES.
# They are NOT calibrated probabilities.
#
# Selection is:
#
#   DEVELOPMENT-GT-INFORMED
#
# Runtime features remain:
#
#   GT-BLIND
#
# ------------------------------------------------------------
# Keep after running: YES
# ============================================================


from pathlib import Path

import hashlib
import json
import warnings

import joblib

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


# ------------------------------------------------------------
# Frozen upstream
# ------------------------------------------------------------

FEATURE_SCHEMA_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_feature_schema_v1_frozen.json"
)


SAMPLE_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_feature_sample_hash64_v1.pkl"
)


POSITIVE_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_positive_features_v1.pkl"
)


G3H1_CONTRACT_PATH = (
    S12
    / "stage12_folds14_rewire_scorer_benchmark_v1_frozen.json"
)


G3H1_FOLD_METRICS_PATH = (
    S12
    / "stage12_folds14_rewire_scorer_oof_fold_metrics_v1.pkl"
)


G3H1_SUMMARY_PATH = (
    S12
    / "stage12_folds14_rewire_scorer_oof_model_summary_v1.json"
)


# ------------------------------------------------------------
# G3H2 formal outputs
# ------------------------------------------------------------

SCORER_FREEZE_OUT = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_frozen.json"
)


PREPROCESSOR_OUT = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_preprocessor.npz"
)


MODEL_OUT = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_model.joblib"
)


COEFFICIENT_OUT = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_coefficients.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_summary.json"
)


for path in [
    FEATURE_SCHEMA_PATH,
    SAMPLE_PATH,
    POSITIVE_PATH,
    G3H1_CONTRACT_PATH,
    G3H1_FOLD_METRICS_PATH,
    G3H1_SUMMARY_PATH,
]:

    assert path.exists(), path


for path in [
    SCORER_FREEZE_OUT,
    PREPROCESSOR_OUT,
    MODEL_OUT,
    COEFFICIENT_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3H2 output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite a frozen/full-dev scorer artifact."
        )


# ============================================================
# 1. AUTHORITATIVE SHAs
# ============================================================

GENERATOR_SHA = (
    "da408df857cd34dcb4b1f65f3fae545e"
    "299f9403ef5edb986584caa8f4c3c203"
)


SHORTLIST_SHA = (
    "ae61216904bc3187f76b238b4b560d669"
    "3a7eb4f5d07bb8ef19021cc1bca1490"
)


FEATURE_SCHEMA_SHA = (
    "0a9f8e9a41c5c80c803c3a95955702dc"
    "f4039605a21fc656a154aa05be55b39c"
)


G3H1_CONTRACT_SHA = (
    "0ab41abd8cdb8efd6cf68d8c299f1210"
    "78f879175267f915b79e48b1e19e8e36"
)


EXPECTED_POSITIVES = 202

EXPECTED_FULL7 = 173

EXPECTED_BACKGROUND = 273_657


SELECTED_FAMILY = (
    "LOGREG_L2_C1_BALANCED_V1"
)


FROZEN_SCORER_ID = (
    "MULTIFRAME_REWIRE_LOGREG_SCORER_V1"
)


# ============================================================
# 2. VERIFY G3H0 FEATURE SCHEMA
# ============================================================

feature_schema = json.loads(
    FEATURE_SCHEMA_PATH.read_text(
        encoding="utf-8"
    )
)


assert feature_schema[
    "feature_schema_sha256"
] == FEATURE_SCHEMA_SHA


assert feature_schema[
    "parent_generator_sha256"
] == GENERATOR_SHA


assert feature_schema[
    "parent_shortlist_sha256"
] == SHORTLIST_SHA


assert feature_schema[
    "runtime_features_gt_blind"
] is True


FEATURE_COLUMNS = list(
    feature_schema[
        "features"
    ]
)


assert len(
    FEATURE_COLUMNS
) == 32


assert "fold" not in FEATURE_COLUMNS

assert "dataset" not in FEATURE_COLUMNS


print(
    "========== G3H2 FEATURE SCHEMA =========="
)

print(
    "feature count:",
    len(
        FEATURE_COLUMNS
    )
)

print(
    "feature schema SHA:",
    FEATURE_SCHEMA_SHA
)

print(
    "runtime features GT-blind:",
    "YES"
)


# ============================================================
# 3. VERIFY G3H1 FROZEN BENCHMARK CONTRACT
# ============================================================

g3h1_contract = json.loads(
    G3H1_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


assert g3h1_contract[
    "benchmark_contract_sha256"
] == G3H1_CONTRACT_SHA


assert g3h1_contract[
    "benchmark_id"
] == "LOFO_SCORER_BENCHMARK_V1"


assert g3h1_contract[
    "model_selected"
] is False


assert g3h1_contract[
    "score_threshold_selected"
] is False


assert g3h1_contract[
    "rewire_policy_selected"
] is False


assert g3h1_contract[
    "cross_validation"
][
    "fold0_used"
] is False


assert SELECTED_FAMILY in g3h1_contract[
    "models"
]


# ============================================================
# 4. LOAD FORMAL G3H1 RESULTS
# ============================================================

fold_metrics = pd.read_pickle(
    G3H1_FOLD_METRICS_PATH
)


g3h1_summary = json.loads(
    G3H1_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert g3h1_summary[
    "status"
] == (
    "LEAVE_ONE_FOLD_OUT_SCORER_BENCHMARK_COMPLETE"
)


assert g3h1_summary[
    "fold0_used"
] is False


assert g3h1_summary[
    "model_selected"
] is False


assert g3h1_summary[
    "diagnostic_leader"
] == SELECTED_FAMILY


assert g3h1_summary[
    "diagnostic_leader_is_frozen_final_scorer"
] is False


# ============================================================
# 5. EXACT OOF SELECTION AUDIT
# ============================================================

primary_col = (
    "full7_asym_1x5"
)


logreg_fold = (
    fold_metrics[
        fold_metrics[
            "model_id"
        ].eq(
            SELECTED_FAMILY
        )
    ]
    .sort_values(
        "fold"
    )
    .reset_index(
        drop=True
    )
)


hgb_fold = (
    fold_metrics[
        fold_metrics[
            "model_id"
        ].eq(
            "HGB_D3_L15_BALANCED_V1"
        )
    ]
    .sort_values(
        "fold"
    )
    .reset_index(
        drop=True
    )
)


base_fold = (
    fold_metrics[
        fold_metrics[
            "model_id"
        ].eq(
            "BASE_GMEAN_RATIO_V1"
        )
    ]
    .sort_values(
        "fold"
    )
    .reset_index(
        drop=True
    )
)


assert len(
    logreg_fold
) == 4


assert len(
    hgb_fold
) == 4


assert len(
    base_fold
) == 4


assert logreg_fold[
    "fold"
].tolist() == [
    1,
    2,
    3,
    4,
]


# ------------------------------------------------------------
# Critical stability gate:
#
# LogReg primary metric must beat BOTH alternatives
# independently in all four held-out folds.
# ------------------------------------------------------------

logreg_primary = (
    logreg_fold[
        primary_col
    ]
    .to_numpy(
        dtype=float
    )
)


hgb_primary = (
    hgb_fold[
        primary_col
    ]
    .to_numpy(
        dtype=float
    )
)


base_primary = (
    base_fold[
        primary_col
    ]
    .to_numpy(
        dtype=float
    )
)


assert np.all(
    logreg_primary
    >
    hgb_primary
), (
    logreg_primary,
    hgb_primary,
)


assert np.all(
    logreg_primary
    >
    base_primary
), (
    logreg_primary,
    base_primary,
)


logreg_macro = float(
    logreg_primary.mean()
)


logreg_worst = float(
    logreg_primary.min()
)


hgb_macro = float(
    hgb_primary.mean()
)


base_macro = float(
    base_primary.mean()
)


assert np.isclose(
    logreg_macro,
    0.794094,
    atol=1e-6,
)


assert np.isclose(
    logreg_worst,
    0.720930,
    atol=1e-6,
)


print(
    "\n========== G3H2 OOF SCORER SELECTION AUDIT =========="
)


selection_table = pd.DataFrame(
    {
        "fold":
            [
                1,
                2,
                3,
                4,
            ],

        "GMEAN":
            base_primary,

        "HGB":
            hgb_primary,

        "LOGREG":
            logreg_primary,

        "LOGREG_MINUS_GMEAN":
            (
                logreg_primary
                -
                base_primary
            ),

        "LOGREG_MINUS_HGB":
            (
                logreg_primary
                -
                hgb_primary
            ),
    }
)


print(
    selection_table.to_string(
        index=False
    )
)


print(
    "\nLogReg macro:",
    logreg_macro
)


print(
    "LogReg worst fold:",
    logreg_worst
)


print(
    "HGB macro:",
    hgb_macro
)


print(
    "GMEAN macro:",
    base_macro
)


print(
    "LogReg > HGB on all held-out folds:",
    "YES"
)


print(
    "LogReg > GMEAN on all held-out folds:",
    "YES"
)


print(
    "SCORER_FAMILY_SELECTION_STABILITY: PASS"
)


# ============================================================
# 6. FREEZE SCORER FAMILY SELECTION
#
# This happens BEFORE fitting the full-development model.
#
# Selection itself is development-GT-informed.
# Runtime scorer inputs remain GT-blind.
# ============================================================

SCORER_FREEZE = {
    "stage":
        "12.11G3H2",

    "scorer_id":
        FROZEN_SCORER_ID,

    "selected_family":
        SELECTED_FAMILY,

    "selected_from_frozen_benchmark":
        "LOFO_SCORER_BENCHMARK_V1",

    "benchmark_contract_sha256":
        G3H1_CONTRACT_SHA,

    "parent_generator_sha256":
        GENERATOR_SHA,

    "parent_shortlist_sha256":
        SHORTLIST_SHA,

    "feature_schema_sha256":
        FEATURE_SCHEMA_SHA,

    "selection_basis": {
        "primary_metric":
            "FULL7_ASYM_SCORE_1x5_MACRO_RECALL",

        "logreg_fold_values":
            [
                float(
                    value
                )
                for value
                in logreg_primary
            ],

        "logreg_macro":
            logreg_macro,

        "logreg_worst_fold":
            logreg_worst,

        "hgb_macro":
            hgb_macro,

        "gmean_macro":
            base_macro,

        "logreg_beats_hgb_all_four_folds":
            True,

        "logreg_beats_gmean_all_four_folds":
            True,
    },

    "model_contract": {
        "type":
            "LOGISTIC_REGRESSION",

        "penalty":
            "l2",

        "C":
            1.0,

        "solver":
            "lbfgs",

        "max_iter":
            2000,

        "class_balance":
            "EXPLICIT_EQUAL_TOTAL_CLASS_WEIGHT",
    },

    "preprocessing_contract": {
        "nonfinite":
            "PLUS_MINUS_INF_TO_MISSING",

        "imputation":
            "TRAINING_CORPUS_MEDIAN",

        "missing_indicator":
            "ONE_PER_RAW_FEATURE",

        "value_scaling":
            "TRAINING_CORPUS_STANDARDIZATION",

        "raw_feature_count":
            int(
                len(
                    FEATURE_COLUMNS
                )
            ),

        "design_feature_count":
            int(
                2
                *
                len(
                    FEATURE_COLUMNS
                )
            ),
    },

    "training_population_semantics": {
        "positive":
            "KNOWN_DEVELOPMENT_CLEAN_GAP_ORACLE_POSITIVE",

        "background":
            "GT_BLIND_HASH_SAMPLED_SHORTLIST_BACKGROUND",

        "background_is_confirmed_negative":
            False,
    },

    "selection_is_development_gt_informed":
        True,

    "runtime_features_gt_blind":
        True,

    "score_is_calibrated_probability":
        False,

    "score_use":
        "RANKING_ONLY",

    "feature_subset_selected":
        False,

    "score_threshold_selected":
        False,

    "rewire_policy_selected":
        False,

    "fold0_used":
        False,

    "graph_modified":
        False,

    "future_scorer_family_retuning":
        False,
}


freeze_payload = json.dumps(
    SCORER_FREEZE,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
).encode(
    "utf-8"
)


SCORER_FREEZE_SHA = hashlib.sha256(
    freeze_payload
).hexdigest()


SCORER_FREEZE[
    "scorer_freeze_sha256"
] = SCORER_FREEZE_SHA


SCORER_FREEZE_OUT.write_text(
    json.dumps(
        SCORER_FREEZE,
        indent=2,
    ),
    encoding="utf-8",
)


assert SCORER_FREEZE_OUT.exists()


print(
    "\n========== FROZEN SCORER FAMILY =========="
)


print(
    "scorer ID:",
    FROZEN_SCORER_ID
)


print(
    "family:",
    SELECTED_FAMILY
)


print(
    "freeze SHA:",
    SCORER_FREEZE_SHA
)


print(
    "development-GT-informed selection:",
    "YES"
)


print(
    "runtime features GT-blind:",
    "YES"
)


print(
    "future scorer-family retuning:",
    "NO"
)


print(
    "score threshold selected:",
    "NO"
)


print(
    "SCORER FAMILY FROZEN: YES"
)


# ============================================================
# 7. LOAD FULL DEVELOPMENT TRAINING POPULATION
# ============================================================

sample = pd.read_pickle(
    SAMPLE_PATH
)


positive = pd.read_pickle(
    POSITIVE_PATH
)


assert len(
    positive
) == EXPECTED_POSITIVES


assert int(
    positive[
        "pred_full_7um"
    ].sum()
) == EXPECTED_FULL7


assert set(
    positive[
        "fold"
    ].astype(
        int
    )
) == {
    1,
    2,
    3,
    4,
}


assert not positive[
    "fold"
].eq(
    0
).any()


# ============================================================
# 8. REMOVE KNOWN POSITIVES FROM BACKGROUND
# ============================================================

positive_keys = (
    positive[
        [
            "dataset",
            "source_id",
            "target_id",
        ]
    ]
    .drop_duplicates()
    .copy()
)


positive_keys[
    "__known_positive"
] = True


background = sample.merge(
    positive_keys,
    on=[
        "dataset",
        "source_id",
        "target_id",
    ],
    how="left",
    validate="one_to_one",
)


positive_overlap = int(
    background[
        "__known_positive"
    ].notna().sum()
)


assert positive_overlap == 1


background = (
    background[
        background[
            "__known_positive"
        ].isna()
    ]
    .drop(
        columns=[
            "__known_positive",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(
    background
) == EXPECTED_BACKGROUND


print(
    "\n========== FULL-DEV TRAINING POPULATION =========="
)


print(
    "positive rows:",
    len(
        positive
    )
)


print(
    "background rows:",
    len(
        background
    )
)


print(
    "known-positive overlap removed:",
    positive_overlap
)


print(
    "background confirmed-negative:",
    "NO"
)


print(
    "Fold0:",
    "NONE"
)


# ============================================================
# 9. FIXED PREPROCESSING IMPLEMENTATION
# ============================================================

def raw_feature_matrix(
    frame,
):

    X = (
        frame[
            FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .to_numpy(
            dtype=np.float64,
            copy=True,
        )
    )


    X[
        ~np.isfinite(
            X
        )
    ] = np.nan


    return X


def fit_full_dev_preprocessor(
    frame,
):

    X = raw_feature_matrix(
        frame
    )


    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore",
            category=RuntimeWarning,
        )


        medians = np.nanmedian(
            X,
            axis=0,
        )


    medians[
        ~np.isfinite(
            medians
        )
    ] = 0.0


    missing = np.isnan(
        X
    )


    filled = np.where(
        missing,
        medians[
            None,
            :
        ],
        X,
    )


    means = filled.mean(
        axis=0
    )


    stds = filled.std(
        axis=0
    )


    invalid_std = (
        ~np.isfinite(
            stds
        )
        |
        (
            stds
            <
            1e-12
        )
    )


    stds[
        invalid_std
    ] = 1.0


    return {
        "medians":
            medians.astype(
                np.float64
            ),

        "means":
            means.astype(
                np.float64
            ),

        "stds":
            stds.astype(
                np.float64
            ),
    }


def transform_full_dev(
    frame,
    preprocessing,
):

    X = raw_feature_matrix(
        frame
    )


    missing = np.isnan(
        X
    )


    filled = np.where(
        missing,
        preprocessing[
            "medians"
        ][
            None,
            :
        ],
        X,
    )


    scaled = (
        filled
        -
        preprocessing[
            "means"
        ][
            None,
            :
        ]
    ) / preprocessing[
        "stds"
    ][
        None,
        :
    ]


    design = np.concatenate(
        [
            scaled,
            missing.astype(
                np.float64
            ),
        ],
        axis=1,
    )


    assert design.shape[
        1
    ] == (
        2
        *
        len(
            FEATURE_COLUMNS
        )
    )


    assert np.isfinite(
        design
    ).all()


    return design.astype(
        np.float32,
        copy=False,
    )


def balanced_weights(
    labels,
):

    y = np.asarray(
        labels,
        dtype=np.int8,
    )


    n = len(
        y
    )


    n_pos = int(
        (
            y
            ==
            1
        ).sum()
    )


    n_bg = int(
        (
            y
            ==
            0
        ).sum()
    )


    assert n_pos > 0

    assert n_bg > 0


    pos_weight = (
        n
        /
        (
            2.0
            *
            n_pos
        )
    )


    bg_weight = (
        n
        /
        (
            2.0
            *
            n_bg
        )
    )


    weights = np.where(
        y == 1,
        pos_weight,
        bg_weight,
    )


    return weights.astype(
        np.float64
    )


# ============================================================
# 10. BUILD FULL DEVELOPMENT DESIGN
# ============================================================

training_frame = pd.concat(
    [
        background[
            FEATURE_COLUMNS
        ],

        positive[
            FEATURE_COLUMNS
        ],
    ],
    ignore_index=True,
)


y = np.concatenate(
    [
        np.zeros(
            len(
                background
            ),
            dtype=np.int8,
        ),

        np.ones(
            len(
                positive
            ),
            dtype=np.int8,
        ),
    ]
)


assert len(
    training_frame
) == len(
    y
)


assert int(
    y.sum()
) == EXPECTED_POSITIVES


preprocessing = fit_full_dev_preprocessor(
    training_frame
)


X = transform_full_dev(
    training_frame,
    preprocessing,
)


weights = balanced_weights(
    y
)


assert X.shape == (
    len(
        training_frame
    ),
    64,
)


print(
    "\n========== FULL-DEV DESIGN MATRIX =========="
)


print(
    "training rows:",
    len(
        training_frame
    )
)


print(
    "raw features:",
    len(
        FEATURE_COLUMNS
    )
)


print(
    "design features:",
    X.shape[
        1
    ]
)


print(
    "positives:",
    int(
        y.sum()
    )
)


print(
    "background:",
    int(
        (
            y
            ==
            0
        ).sum()
    )
)


# ============================================================
# 11. FIT EXACT FROZEN LOGREG FAMILY
# ============================================================

with warnings.catch_warnings():

    warnings.filterwarnings(
        "ignore",
        message=(
            "'penalty' was deprecated"
        ),
        category=FutureWarning,
    )


    model = LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=2000,
        random_state=20260904,
    )


    model.fit(
        X,
        y,
        sample_weight=weights,
    )


assert len(
    model.classes_
) == 2


assert set(
    model.classes_.tolist()
) == {
    0,
    1,
}


assert model.coef_.shape == (
    1,
    64,
)


assert np.isfinite(
    model.coef_
).all()


assert np.isfinite(
    model.intercept_
).all()


assert int(
    np.max(
        model.n_iter_
    )
) < 2000


print(
    "\n========== FULL-DEV FROZEN LOGREG FIT =========="
)


print(
    "scorer:",
    FROZEN_SCORER_ID
)


print(
    "iterations:",
    int(
        np.max(
            model.n_iter_
        )
    )
)


print(
    "coefficient count:",
    model.coef_.shape[
        1
    ]
)


print(
    "converged:",
    "YES"
)


print(
    "FULL_DEV_FIT: PASS"
)


# ============================================================
# 12. DESIGN FEATURE NAMES
# ============================================================

VALUE_COLUMNS = [
    (
        "VALUE::"
        +
        feature
    )

    for feature
    in FEATURE_COLUMNS
]


MISSING_COLUMNS = [
    (
        "MISSING::"
        +
        feature
    )

    for feature
    in FEATURE_COLUMNS
]


DESIGN_COLUMNS = (
    VALUE_COLUMNS
    +
    MISSING_COLUMNS
)


assert len(
    DESIGN_COLUMNS
) == 64


assert len(
    DESIGN_COLUMNS
) == len(
    set(
        DESIGN_COLUMNS
    )
)


# ============================================================
# 13. COEFFICIENT AUDIT
# ============================================================

coef = model.coef_[
    0
]


coefficient_table = pd.DataFrame(
    {
        "design_feature":
            DESIGN_COLUMNS,

        "coefficient":
            coef,
    }
)


coefficient_table[
    "abs_coefficient"
] = np.abs(
    coefficient_table[
        "coefficient"
    ]
)


coefficient_table[
    "feature_type"
] = [
    (
        "VALUE"
        if name.startswith(
            "VALUE::"
        )
        else
        "MISSING"
    )

    for name
    in coefficient_table[
        "design_feature"
    ]
]


coefficient_table = (
    coefficient_table
    .sort_values(
        [
            "abs_coefficient",
            "design_feature",
        ],
        ascending=[
            False,
            True,
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n========== TOP FULL-DEV LOGREG COEFFICIENTS =========="
)


print(
    coefficient_table.head(
        20
    ).to_string(
        index=False
    )
)


# ============================================================
# 14. SAVE PREPROCESSOR
# ============================================================

np.savez(
    PREPROCESSOR_OUT,
    feature_columns=np.asarray(
        FEATURE_COLUMNS,
        dtype=str,
    ),
    design_columns=np.asarray(
        DESIGN_COLUMNS,
        dtype=str,
    ),
    medians=preprocessing[
        "medians"
    ],
    means=preprocessing[
        "means"
    ],
    stds=preprocessing[
        "stds"
    ],
)


assert PREPROCESSOR_OUT.exists()


# ============================================================
# 15. SAVE MODEL
# ============================================================

joblib.dump(
    model,
    MODEL_OUT,
)


assert MODEL_OUT.exists()


# ============================================================
# 16. SAVE COEFFICIENT AUDIT
# ============================================================

coefficient_table.to_pickle(
    COEFFICIENT_OUT
)


assert COEFFICIENT_OUT.exists()


# ============================================================
# 17. FILE HASHES
# ============================================================

def file_sha256(
    path,
):

    hasher = hashlib.sha256()


    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024
                *
                1024
            )


            if not chunk:

                break


            hasher.update(
                chunk
            )


    return hasher.hexdigest()


PREPROCESSOR_SHA = file_sha256(
    PREPROCESSOR_OUT
)


MODEL_SHA = file_sha256(
    MODEL_OUT
)


COEFFICIENT_SHA = file_sha256(
    COEFFICIENT_OUT
)


# ============================================================
# 18. MODEL RELOAD FIDELITY
# ============================================================

model_reload = joblib.load(
    MODEL_OUT
)


assert model_reload.coef_.shape == (
    1,
    64,
)


assert np.allclose(
    model_reload.coef_,
    model.coef_,
    atol=0.0,
    rtol=0.0,
)


assert np.allclose(
    model_reload.intercept_,
    model.intercept_,
    atol=0.0,
    rtol=0.0,
)


prep_reload = np.load(
    PREPROCESSOR_OUT,
    allow_pickle=False,
)


assert prep_reload[
    "feature_columns"
].tolist() == FEATURE_COLUMNS


assert prep_reload[
    "design_columns"
].tolist() == DESIGN_COLUMNS


assert np.allclose(
    prep_reload[
        "medians"
    ],
    preprocessing[
        "medians"
    ],
    atol=0.0,
    rtol=0.0,
)


assert np.allclose(
    prep_reload[
        "means"
    ],
    preprocessing[
        "means"
    ],
    atol=0.0,
    rtol=0.0,
)


assert np.allclose(
    prep_reload[
        "stds"
    ],
    preprocessing[
        "stds"
    ],
    atol=0.0,
    rtol=0.0,
)


print(
    "\n========== FULL-DEV ARTIFACT FIDELITY =========="
)


print(
    "preprocessor SHA:",
    PREPROCESSOR_SHA
)


print(
    "model SHA:",
    MODEL_SHA
)


print(
    "coefficient audit SHA:",
    COEFFICIENT_SHA
)


print(
    "model reload:",
    "PASS"
)


print(
    "preprocessor reload:",
    "PASS"
)


# ============================================================
# 19. FORMAL SUMMARY
# ============================================================

SUMMARY = {
    "stage":
        "12.11G3H2",

    "status":
        "SCORER_FAMILY_FROZEN_AND_FULL_DEV_MODEL_FIT",

    "scorer_id":
        FROZEN_SCORER_ID,

    "selected_family":
        SELECTED_FAMILY,

    "scorer_freeze_sha256":
        SCORER_FREEZE_SHA,

    "upstream": {
        "generator_sha256":
            GENERATOR_SHA,

        "shortlist_sha256":
            SHORTLIST_SHA,

        "feature_schema_sha256":
            FEATURE_SCHEMA_SHA,

        "g3h1_benchmark_contract_sha256":
            G3H1_CONTRACT_SHA,
    },

    "oof_selection": {
        "primary_metric":
            "FULL7_ASYM_SCORE_1x5_MACRO_RECALL",

        "fold_values":
            [
                float(
                    value
                )

                for value
                in logreg_primary
            ],

        "macro":
            logreg_macro,

        "worst_fold":
            logreg_worst,

        "beats_hgb_on_all_4_folds":
            True,

        "beats_gmean_on_all_4_folds":
            True,
    },

    "full_dev_training": {
        "positive_rows":
            int(
                len(
                    positive
                )
            ),

        "background_rows":
            int(
                len(
                    background
                )
            ),

        "background_is_confirmed_negative":
            False,

        "design_rows":
            int(
                len(
                    training_frame
                )
            ),

        "raw_feature_count":
            int(
                len(
                    FEATURE_COLUMNS
                )
            ),

        "design_feature_count":
            int(
                X.shape[
                    1
                ]
            ),

        "logreg_iterations":
            int(
                np.max(
                    model.n_iter_
                )
            ),
    },

    "artifacts": {
        "preprocessor":
            PREPROCESSOR_OUT.name,

        "preprocessor_sha256":
            PREPROCESSOR_SHA,

        "model":
            MODEL_OUT.name,

        "model_sha256":
            MODEL_SHA,

        "coefficient_audit":
            COEFFICIENT_OUT.name,

        "coefficient_audit_sha256":
            COEFFICIENT_SHA,
    },

    "selection_is_development_gt_informed":
        True,

    "runtime_features_gt_blind":
        True,

    "score_is_calibrated_probability":
        False,

    "score_use":
        "RANKING_ONLY",

    "future_scorer_family_retuning":
        False,

    "feature_subset_selected":
        False,

    "score_threshold_selected":
        False,

    "rewire_policy_selected":
        False,

    "fold0_used":
        False,

    "graph_modified":
        False,

    "next":
        (
            "DESIGN_OOF_ENDPOINT_CONFLICT_AND_ACTION_POLICY_"
            "WITHOUT_RETUNING_SCORER"
        ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


assert SUMMARY_OUT.exists()


# ============================================================
# 20. SUMMARY RELOAD
# ============================================================

summary_check = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert summary_check[
    "selected_family"
] == SELECTED_FAMILY


assert summary_check[
    "scorer_id"
] == FROZEN_SCORER_ID


assert summary_check[
    "future_scorer_family_retuning"
] is False


assert summary_check[
    "score_threshold_selected"
] is False


assert summary_check[
    "rewire_policy_selected"
] is False


assert summary_check[
    "fold0_used"
] is False


assert summary_check[
    "graph_modified"
] is False


# ============================================================
# 21. FINAL DECISION
# ============================================================

print(
    "\n========== 12.11G3H2 DECISION =========="
)


print(
    "SCORER_FAMILY_FREEZE: PASS"
)


print(
    "selected family:",
    SELECTED_FAMILY
)


print(
    "frozen scorer ID:",
    FROZEN_SCORER_ID
)


print(
    "scorer freeze SHA:",
    SCORER_FREEZE_SHA
)


print(
    "OOF primary macro:",
    logreg_macro
)


print(
    "OOF primary worst fold:",
    logreg_worst
)


print(
    "beats HGB on every fold:",
    "YES"
)


print(
    "beats GMEAN on every fold:",
    "YES"
)


print(
    "full-development fit:",
    "PASS"
)


print(
    "model SHA:",
    MODEL_SHA
)


print(
    "preprocessor SHA:",
    PREPROCESSOR_SHA
)


print(
    "selection development-GT-informed:",
    "YES"
)


print(
    "runtime features GT-blind:",
    "YES"
)


print(
    "score calibrated probability:",
    "NO"
)


print(
    "score usage:",
    "RANKING ONLY"
)


print(
    "future scorer-family retuning:",
    "NO"
)


print(
    "feature subset selected:",
    "NO"
)


print(
    "score threshold selected:",
    "NO"
)


print(
    "rewire policy selected:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "graph modified:",
    "NO"
)


print(
    "next:"
)


print(
    "DESIGN_OOF_ENDPOINT_CONFLICT_AND_ACTION_POLICY_WITH_FROZEN_SCORER"
)

In [ ]:
# ============================================================
# Stage 12.11G3H3A
#
# Frozen OOF Endpoint-Conflict Policy Frontier
#
# ------------------------------------------------------------
# FROZEN SCORER
# ------------------------------------------------------------
#
# MULTIFRAME_REWIRE_LOGREG_SCORER_V1
#
# Family:
#   LOGREG_L2_C1_BALANCED_V1
#
# scorer freeze SHA:
#   5ae66ee0f2da841bfe3a7a0b56fdab1b57f7ec3f504ba57441e2c8c272279e91
#
# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# 1. Freeze a small endpoint-rank action-policy family
#    BEFORE development oracle evaluation.
#
# 2. Reconstruct exact LOFO LogReg models.
#
# 3. Score COMPLETE frozen shortlist using held-out-fold model.
#
# 4. Apply deterministic GT-blind conflict resolution.
#
# 5. Audit:
#       - candidate cardinality
#       - conflict-resolved action cardinality
#       - known-positive retention
#       - FULL<=7 positive retention
#       - fold stability
#
# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------
#
# This is POLICY-FRONTIER AUDIT only.
#
# NO:
#   final policy selection
#   score threshold
#   graph modification
#   official graph metric
#   Fold0
#
# Development oracle may only be used AFTER policy-family
# contract freeze.
#
# ------------------------------------------------------------
# ACTION MODEL
# ------------------------------------------------------------
#
# Higher scorer output = better ranking.
#
# Endpoint score ranks are computed on the COMPLETE frozen
# shortlist of that dataset.
#
# Fixed policy family:
#
#   MUTUAL_1x1
#   ASYM_1x3
#   ASYM_1x5
#   ASYM_1x8
#   ASYM_2x5
#   ASYM_3x8
#
# G3H1 primary metric already predeclared ASYM_1x5.
#
# No parameters below are tuned during this cell.
#
# ------------------------------------------------------------
# CONFLICT RESOLUTION
# ------------------------------------------------------------
#
# For candidates passing a rank rule:
#
# deterministic order:
#
#   1. model score descending
#   2. GMEAN_RATIO ascending
#   3. gap_size ascending
#   4. source_id ascending
#   5. target_id ascending
#
# Greedily accept iff:
#
#   - proposal source not previously selected as source
#   - proposal target not previously selected as target
#   - source != previously selected target
#   - target != previously selected source
#
# Thus selected proposal endpoints are node-disjoint.
#
# This is intentionally conservative.
#
# ------------------------------------------------------------
# Keep after running: YES
# ============================================================


from pathlib import Path

import gc
import hashlib
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


FEATURE_SCHEMA_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_feature_schema_v1_frozen.json"
)


SHORTLIST_PATH = (
    S12
    / "stage12_folds14_endpoint_shortlist_g8_lowcut20_v1_frozen.json"
)


SAMPLE_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_feature_sample_hash64_v1.pkl"
)


POSITIVE_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_positive_features_v1.pkl"
)


SCORER_FREEZE_PATH = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_frozen.json"
)


G3H1_METRICS_PATH = (
    S12
    / "stage12_folds14_rewire_scorer_oof_fold_metrics_v1.pkl"
)


# ------------------------------------------------------------
# Formal G3H3A outputs
# ------------------------------------------------------------

POLICY_CONTRACT_OUT = (
    S12
    / "stage12_folds14_rewire_endpoint_policy_frontier_v1_frozen.json"
)


CARDINALITY_OUT = (
    S12
    / "stage12_folds14_rewire_endpoint_policy_frontier_v1_cardinality.pkl"
)


ORACLE_OUT = (
    S12
    / "stage12_folds14_rewire_endpoint_policy_frontier_v1_oracle.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_folds14_rewire_endpoint_policy_frontier_v1_summary.json"
)


for path in [
    CV_PATH,
    FEATURE_SCHEMA_PATH,
    SHORTLIST_PATH,
    SAMPLE_PATH,
    POSITIVE_PATH,
    SCORER_FREEZE_PATH,
    G3H1_METRICS_PATH,
]:

    assert path.exists(), path


for path in [
    POLICY_CONTRACT_OUT,
    CARDINALITY_OUT,
    ORACLE_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3H3A output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite a partial/frozen G3H3A run."
        )


# ============================================================
# 1. REQUIRED AUTHORITATIVE CONTRACTS
# ============================================================

GENERATOR_SHA = (
    "da408df857cd34dcb4b1f65f3fae545e"
    "299f9403ef5edb986584caa8f4c3c203"
)


SHORTLIST_SHA = (
    "ae61216904bc3187f76b238b4b560d669"
    "3a7eb4f5d07bb8ef19021cc1bca1490"
)


FEATURE_SCHEMA_SHA = (
    "0a9f8e9a41c5c80c803c3a95955702dc"
    "f4039605a21fc656a154aa05be55b39c"
)


SCORER_FREEZE_SHA = (
    "5ae66ee0f2da841bfe3a7a0b56fdab1b"
    "57f7ec3f504ba57441e2c8c272279e91"
)


EXPECTED_SHORTLIST_ROWS = 17_501_674

EXPECTED_POSITIVES = 202

EXPECTED_FULL7 = 173

EXPECTED_BACKGROUND = 273_657


SCORER_ID = (
    "MULTIFRAME_REWIRE_LOGREG_SCORER_V1"
)


SCORER_FAMILY = (
    "LOGREG_L2_C1_BALANCED_V1"
)


# ============================================================
# 2. REQUIRED IN-MEMORY GENERATOR HELPERS
# ============================================================

REQUIRED_HELPERS = [
    "generate_dataset_proposals",
    "add_endpoint_ranks",
    "add_shortlist_rank_fields",
    "frozen_shortlist_mask",
    "attach_scoring_features",
]


missing_helpers = [
    name

    for name in REQUIRED_HELPERS

    if (
        name not in globals()
        or
        not callable(
            globals()[
                name
            ]
        )
    )
]


assert not missing_helpers, (
    "\nMissing G3H0 generator helpers:\n"
    f"{missing_helpers}"
)


print(
    "========== G3H3A UPSTREAM HELPERS =========="
)

print(
    "missing:",
    missing_helpers
)

print(
    "HELPER FIDELITY: PASS"
)


# ============================================================
# 3. VERIFY FROZEN FEATURE SCHEMA
# ============================================================

feature_schema = json.loads(
    FEATURE_SCHEMA_PATH.read_text(
        encoding="utf-8"
    )
)


assert feature_schema[
    "feature_schema_sha256"
] == FEATURE_SCHEMA_SHA


assert feature_schema[
    "parent_generator_sha256"
] == GENERATOR_SHA


assert feature_schema[
    "parent_shortlist_sha256"
] == SHORTLIST_SHA


FEATURE_COLUMNS = list(
    feature_schema[
        "features"
    ]
)


assert len(
    FEATURE_COLUMNS
) == 32


# ============================================================
# 4. VERIFY FROZEN SHORTLIST
# ============================================================

shortlist_contract = json.loads(
    SHORTLIST_PATH.read_text(
        encoding="utf-8"
    )
)


assert shortlist_contract[
    "shortlist_contract_sha256"
] == SHORTLIST_SHA


assert int(
    shortlist_contract[
        "development_cardinality"
    ][
        "rows"
    ]
) == EXPECTED_SHORTLIST_ROWS


# ============================================================
# 5. VERIFY FROZEN SCORER
# ============================================================

scorer_freeze = json.loads(
    SCORER_FREEZE_PATH.read_text(
        encoding="utf-8"
    )
)


assert scorer_freeze[
    "scorer_freeze_sha256"
] == SCORER_FREEZE_SHA


assert scorer_freeze[
    "scorer_id"
] == SCORER_ID


assert scorer_freeze[
    "selected_family"
] == SCORER_FAMILY


assert scorer_freeze[
    "future_scorer_family_retuning"
] is False


assert scorer_freeze[
    "runtime_features_gt_blind"
] is True


assert scorer_freeze[
    "score_threshold_selected"
] is False


assert scorer_freeze[
    "rewire_policy_selected"
] is False


assert scorer_freeze[
    "fold0_used"
] is False


print(
    "\n========== G3H3A FROZEN SCORER =========="
)

print(
    "scorer:",
    SCORER_ID
)

print(
    "family:",
    SCORER_FAMILY
)

print(
    "freeze SHA:",
    SCORER_FREEZE_SHA
)

print(
    "future scorer retuning:",
    "NO"
)


# ============================================================
# 6. DEVELOPMENT CORPUS
# ============================================================

cv = pd.read_csv(
    CV_PATH
)


dev_manifest = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    dev_manifest
) == 103


assert not dev_manifest[
    "fold"
].eq(
    0
).any()


development_names = (
    dev_manifest[
        "dataset"
    ]
    .tolist()
)


fold_map = dict(
    zip(
        dev_manifest[
            "dataset"
        ],

        dev_manifest[
            "fold"
        ].astype(
            int
        ),
    )
)


# ============================================================
# 7. FREEZE ENDPOINT-POLICY FAMILY
#
# THIS OCCURS BEFORE DEVELOPMENT ORACLE LOAD BELOW.
# ============================================================

POLICY_RULES = {
    "MUTUAL_1x1": {
        "a":
            1,

        "b":
            1,
    },

    "ASYM_1x3": {
        "a":
            1,

        "b":
            3,
    },

    "ASYM_1x5": {
        "a":
            1,

        "b":
            5,
    },

    "ASYM_1x8": {
        "a":
            1,

        "b":
            8,
    },

    "ASYM_2x5": {
        "a":
            2,

        "b":
            5,
    },

    "ASYM_3x8": {
        "a":
            3,

        "b":
            8,
    },
}


POLICY_CONTRACT = {
    "stage":
        "12.11G3H3A",

    "contract_id":
        "OOF_ENDPOINT_CONFLICT_POLICY_FRONTIER_V1",

    "scorer_id":
        SCORER_ID,

    "scorer_freeze_sha256":
        SCORER_FREEZE_SHA,

    "generator_sha256":
        GENERATOR_SHA,

    "shortlist_sha256":
        SHORTLIST_SHA,

    "feature_schema_sha256":
        FEATURE_SCHEMA_SHA,

    "policy_rules":
        POLICY_RULES,

    "rank_rule":
        (
            "(SOURCE_RANK<=a AND TARGET_RANK<=b) "
            "OR "
            "(TARGET_RANK<=a AND SOURCE_RANK<=b)"
        ),

    "conflict_resolution": {
        "method":
            "DETERMINISTIC_GREEDY_NODE_DISJOINT",

        "sort_order": [
            "MODEL_SCORE_DESC",
            "GMEAN_RATIO_ASC",
            "GAP_SIZE_ASC",
            "SOURCE_ID_ASC",
            "TARGET_ID_ASC",
        ],

        "lock_source_nodes":
            True,

        "lock_target_nodes":
            True,

        "cross_role_node_lock":
            True,
    },

    "primary_predeclared_policy":
        "ASYM_1x5",

    "primary_policy_origin":
        "PREDECLARED_IN_G3H1_PRIMARY_METRIC",

    "selection_is_development_gt_informed":
        False,

    "oracle_loaded_before_contract_freeze":
        False,

    "final_policy_selected":
        False,

    "score_threshold_selected":
        False,

    "graph_modified":
        False,

    "fold0_used":
        False,
}


policy_contract_bytes = json.dumps(
    POLICY_CONTRACT,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
).encode(
    "utf-8"
)


POLICY_CONTRACT_SHA = hashlib.sha256(
    policy_contract_bytes
).hexdigest()


POLICY_CONTRACT[
    "policy_contract_sha256"
] = POLICY_CONTRACT_SHA


POLICY_CONTRACT_OUT.write_text(
    json.dumps(
        POLICY_CONTRACT,
        indent=2,
    ),
    encoding="utf-8",
)


assert POLICY_CONTRACT_OUT.exists()


print(
    "\n========== FROZEN G3H3A POLICY FAMILY =========="
)

print(
    "contract SHA:",
    POLICY_CONTRACT_SHA
)

print(
    "rules:",
    list(
        POLICY_RULES
    )
)

print(
    "predeclared primary:",
    "ASYM_1x5"
)

print(
    "oracle loaded before freeze:",
    "NO"
)

print(
    "POLICY FAMILY FROZEN: YES"
)


# ============================================================
# PART B — DEVELOPMENT ORACLE / LABELS MAY NOW BE READ
# ============================================================


# ============================================================
# 8. LOAD G3H0 TRAINING POPULATION
# ============================================================

sample = pd.read_pickle(
    SAMPLE_PATH
)


positive = pd.read_pickle(
    POSITIVE_PATH
)


assert len(
    positive
) == EXPECTED_POSITIVES


assert int(
    positive[
        "pred_full_7um"
    ].sum()
) == EXPECTED_FULL7


assert set(
    positive[
        "fold"
    ].astype(
        int
    )
) == {
    1,
    2,
    3,
    4,
}


positive_keys = (
    positive[
        [
            "dataset",
            "source_id",
            "target_id",
        ]
    ]
    .drop_duplicates()
    .copy()
)


positive_keys[
    "__positive"
] = True


background = sample.merge(
    positive_keys,
    on=[
        "dataset",
        "source_id",
        "target_id",
    ],
    how="left",
    validate="one_to_one",
)


overlap = int(
    background[
        "__positive"
    ].notna().sum()
)


assert overlap == 1


background = (
    background[
        background[
            "__positive"
        ].isna()
    ]
    .drop(
        columns=[
            "__positive",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(
    background
) == EXPECTED_BACKGROUND


# ============================================================
# 9. EXACT G3H1 PREPROCESSING
# ============================================================

def raw_feature_matrix_g3h3(
    frame,
):

    X = (
        frame[
            FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .to_numpy(
            dtype=np.float64,
            copy=True,
        )
    )


    X[
        ~np.isfinite(
            X
        )
    ] = np.nan


    return X


def fit_preprocessor_g3h3(
    frame,
):

    X = raw_feature_matrix_g3h3(
        frame
    )


    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore",
            RuntimeWarning,
        )


        medians = np.nanmedian(
            X,
            axis=0,
        )


    medians[
        ~np.isfinite(
            medians
        )
    ] = 0.0


    missing = np.isnan(
        X
    )


    filled = np.where(
        missing,
        medians[
            None,
            :
        ],
        X,
    )


    means = filled.mean(
        axis=0
    )


    stds = filled.std(
        axis=0
    )


    stds[
        (
            ~np.isfinite(
                stds
            )
            |
            (
                stds
                <
                1e-12
            )
        )
    ] = 1.0


    return {
        "medians":
            medians,

        "means":
            means,

        "stds":
            stds,
    }


def transform_g3h3(
    frame,
    prep,
):

    X = raw_feature_matrix_g3h3(
        frame
    )


    missing = np.isnan(
        X
    )


    filled = np.where(
        missing,
        prep[
            "medians"
        ][
            None,
            :
        ],
        X,
    )


    scaled = (
        filled
        -
        prep[
            "means"
        ][
            None,
            :
        ]
    ) / prep[
        "stds"
    ][
        None,
        :
    ]


    design = np.concatenate(
        [
            scaled,
            missing.astype(
                np.float64
            ),
        ],
        axis=1,
    )


    assert design.shape[
        1
    ] == 64


    assert np.isfinite(
        design
    ).all()


    return design.astype(
        np.float32,
        copy=False,
    )


def balanced_weights_g3h3(
    y,
):

    y = np.asarray(
        y,
        dtype=np.int8,
    )


    n = len(
        y
    )


    n_pos = int(
        (
            y
            ==
            1
        ).sum()
    )


    n_bg = int(
        (
            y
            ==
            0
        ).sum()
    )


    assert n_pos > 0
    assert n_bg > 0


    return np.where(
        y == 1,
        n
        /
        (
            2.0
            *
            n_pos
        ),
        n
        /
        (
            2.0
            *
            n_bg
        ),
    )


# ============================================================
# 10. REBUILD EXACT FOUR LOFO LOGREG MODELS
#
# Frozen family only.
# No tuning.
# ============================================================

lofo_models = {}


for heldout_fold in [
    1,
    2,
    3,
    4,
]:

    train_bg = background[
        background[
            "fold"
        ].ne(
            heldout_fold
        )
    ]


    train_pos = positive[
        positive[
            "fold"
        ].ne(
            heldout_fold
        )
    ]


    train = pd.concat(
        [
            train_bg[
                FEATURE_COLUMNS
            ],

            train_pos[
                FEATURE_COLUMNS
            ],
        ],
        ignore_index=True,
    )


    y = np.concatenate(
        [
            np.zeros(
                len(
                    train_bg
                ),
                dtype=np.int8,
            ),

            np.ones(
                len(
                    train_pos
                ),
                dtype=np.int8,
            ),
        ]
    )


    prep = fit_preprocessor_g3h3(
        train
    )


    X = transform_g3h3(
        train,
        prep,
    )


    weights = balanced_weights_g3h3(
        y
    )


    with warnings.catch_warnings():

        warnings.filterwarnings(
            "ignore",
            message="'penalty' was deprecated",
            category=FutureWarning,
        )


        model = LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="lbfgs",
            max_iter=2000,
            random_state=20260904,
        )


        model.fit(
            X,
            y,
            sample_weight=weights,
        )


    assert int(
        np.max(
            model.n_iter_
        )
    ) < 2000


    lofo_models[
        heldout_fold
    ] = {
        "prep":
            prep,

        "model":
            model,
    }


    print(
        f"fold {heldout_fold}"
        f" | bg={len(train_bg)}"
        f" | pos={len(train_pos)}"
        f" | iter={int(np.max(model.n_iter_))}"
    )


    del train_bg
    del train_pos

    del train
    del y

    del X
    del weights

    gc.collect()


print(
    "\nLOFO_SCORER_RECONSTRUCTION: PASS"
)


# ============================================================
# 11. SCORE HELPER
# ============================================================

PREDICT_BATCH = 50_000


def score_shortlist_g3h3(
    frame,
    bundle,
):

    scores = np.empty(
        len(
            frame
        ),
        dtype=np.float64,
    )


    for start in range(
        0,
        len(
            frame
        ),
        PREDICT_BATCH,
    ):

        stop = min(
            start
            +
            PREDICT_BATCH,
            len(
                frame
            ),
        )


        batch = frame.iloc[
            start:stop
        ]


        X = transform_g3h3(
            batch,
            bundle[
                "prep"
            ],
        )


        scores[
            start:stop
        ] = (
            bundle[
                "model"
            ]
            .predict_proba(
                X
            )[
                :,
                1
            ]
        )


        del batch
        del X


    assert np.isfinite(
        scores
    ).all()


    return scores


# ============================================================
# 12. ENDPOINT SCORE RANKS
# ============================================================

def endpoint_score_ranks_g3h3(
    frame,
):

    source_order = (
        frame[
            [
                "source_id",
                "target_id",
                "gap_size",
                "GMEAN_RATIO",
                "model_score",
            ]
        ]
        .sort_values(
            [
                "source_id",
                "model_score",
                "GMEAN_RATIO",
                "gap_size",
                "target_id",
            ],
            ascending=[
                True,
                False,
                True,
                True,
                True,
            ],
            kind="stable",
        )
    )


    source_order[
        "source_score_rank"
    ] = (
        source_order.groupby(
            "source_id",
            sort=False,
        )
        .cumcount()
        +
        1
    )


    target_order = (
        frame[
            [
                "source_id",
                "target_id",
                "gap_size",
                "GMEAN_RATIO",
                "model_score",
            ]
        ]
        .sort_values(
            [
                "target_id",
                "model_score",
                "GMEAN_RATIO",
                "gap_size",
                "source_id",
            ],
            ascending=[
                True,
                False,
                True,
                True,
                True,
            ],
            kind="stable",
        )
    )


    target_order[
        "target_score_rank"
    ] = (
        target_order.groupby(
            "target_id",
            sort=False,
        )
        .cumcount()
        +
        1
    )


    result = pd.DataFrame(
        index=frame.index
    )


    result[
        "source_score_rank"
    ] = (
        source_order[
            "source_score_rank"
        ]
        .reindex(
            frame.index
        )
        .astype(
            np.int32
        )
    )


    result[
        "target_score_rank"
    ] = (
        target_order[
            "target_score_rank"
        ]
        .reindex(
            frame.index
        )
        .astype(
            np.int32
        )
    )


    del source_order
    del target_order


    return result


# ============================================================
# 13. POLICY MASK
# ============================================================

def policy_mask_g3h3(
    frame,
    a,
    b,
):

    sr = frame[
        "source_score_rank"
    ]


    tr = frame[
        "target_score_rank"
    ]


    return (
        (
            sr.le(
                a
            )
            &
            tr.le(
                b
            )
        )
        |
        (
            tr.le(
                a
            )
            &
            sr.le(
                b
            )
        )
    )


# ============================================================
# 14. CONSERVATIVE NODE-DISJOINT GREEDY RESOLVER
# ============================================================

def resolve_conflicts_g3h3(
    frame,
):

    if len(
        frame
    ) == 0:

        return frame.copy()


    ordered = (
        frame
        .sort_values(
            [
                "model_score",
                "GMEAN_RATIO",
                "gap_size",
                "source_id",
                "target_id",
            ],
            ascending=[
                False,
                True,
                True,
                True,
                True,
            ],
            kind="stable",
        )
    )


    locked_nodes = set()

    accepted_index = []


    for row in ordered.itertuples():

        src = int(
            row.source_id
        )


        tgt = int(
            row.target_id
        )


        if src in locked_nodes:

            continue


        if tgt in locked_nodes:

            continue


        accepted_index.append(
            row.Index
        )


        locked_nodes.add(
            src
        )


        locked_nodes.add(
            tgt
        )


    return (
        frame.loc[
            accepted_index
        ]
        .copy()
    )


# ============================================================
# 15. POSITIVE KEY LOOKUP
# ============================================================

positive_meta = positive[
    [
        "dataset",
        "fold",
        "component_id",
        "source_id",
        "target_id",
        "pred_full_7um",
        "topology_class",
    ]
].copy()


positive_key_set_by_dataset = {}


for dataset, sub in positive_meta.groupby(
    "dataset"
):

    positive_key_set_by_dataset[
        dataset
    ] = set(
        zip(
            sub[
                "source_id"
            ].astype(
                int
            ),

            sub[
                "target_id"
            ].astype(
                int
            ),
        )
    )


# ============================================================
# 16. OOF FULL-SHORTLIST POLICY FRONTIER
# ============================================================

print(
    "\n============================================================"
)

print(
    "OOF ENDPOINT-CONFLICT POLICY FRONTIER"
)

print(
    "============================================================"
)


cardinality_rows = []

oracle_rows = []


global_shortlist = 0


for ds_i, dataset in enumerate(
    development_names,
    start=1,
):

    fold = int(
        fold_map[
            dataset
        ]
    )


    (
        proposals,
        _,
        _,
    ) = generate_dataset_proposals(
        dataset
    )


    proposals = add_endpoint_ranks(
        proposals
    )


    proposals = add_shortlist_rank_fields(
        proposals
    )


    shortlist = (
        proposals[
            frozen_shortlist_mask(
                proposals
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    shortlist = attach_scoring_features(
        shortlist
    )


    global_shortlist += int(
        len(
            shortlist
        )
    )


    shortlist[
        "model_score"
    ] = score_shortlist_g3h3(
        shortlist,
        lofo_models[
            fold
        ],
    )


    ranks = endpoint_score_ranks_g3h3(
        shortlist
    )


    shortlist[
        "source_score_rank"
    ] = ranks[
        "source_score_rank"
    ].to_numpy(
        copy=True
    )


    shortlist[
        "target_score_rank"
    ] = ranks[
        "target_score_rank"
    ].to_numpy(
        copy=True
    )


    true_keys = positive_key_set_by_dataset.get(
        dataset,
        set(),
    )


    for policy_id, rule in POLICY_RULES.items():

        candidate_mask = policy_mask_g3h3(
            shortlist,
            a=int(
                rule[
                    "a"
                ]
            ),
            b=int(
                rule[
                    "b"
                ]
            ),
        )


        candidates = shortlist[
            candidate_mask
        ].copy()


        resolved = resolve_conflicts_g3h3(
            candidates
        )


        candidate_true = 0

        resolved_true = 0


        if true_keys:

            candidate_pairs = set(
                zip(
                    candidates[
                        "source_id"
                    ].astype(
                        int
                    ),

                    candidates[
                        "target_id"
                    ].astype(
                        int
                    ),
                )
            )


            resolved_pairs = set(
                zip(
                    resolved[
                        "source_id"
                    ].astype(
                        int
                    ),

                    resolved[
                        "target_id"
                    ].astype(
                        int
                    ),
                )
            )


            candidate_true = len(
                candidate_pairs
                &
                true_keys
            )


            resolved_true = len(
                resolved_pairs
                &
                true_keys
            )


        cardinality_rows.append(
            {
                "dataset":
                    dataset,

                "fold":
                    fold,

                "policy_id":
                    policy_id,

                "shortlist_rows":
                    int(
                        len(
                            shortlist
                        )
                    ),

                "rank_rule_rows":
                    int(
                        len(
                            candidates
                        )
                    ),

                "resolved_actions":
                    int(
                        len(
                            resolved
                        )
                    ),

                "candidate_true_pairs":
                    int(
                        candidate_true
                    ),

                "resolved_true_pairs":
                    int(
                        resolved_true
                    ),
            }
        )


        if true_keys:

            positive_sub = positive_meta[
                positive_meta[
                    "dataset"
                ].eq(
                    dataset
                )
            ]


            candidate_key_lookup = set(
                zip(
                    candidates[
                        "source_id"
                    ].astype(
                        int
                    ),

                    candidates[
                        "target_id"
                    ].astype(
                        int
                    ),
                )
            )


            resolved_key_lookup = set(
                zip(
                    resolved[
                        "source_id"
                    ].astype(
                        int
                    ),

                    resolved[
                        "target_id"
                    ].astype(
                        int
                    ),
                )
            )


            for row in positive_sub.itertuples(
                index=False
            ):

                key = (
                    int(
                        row.source_id
                    ),
                    int(
                        row.target_id
                    ),
                )


                oracle_rows.append(
                    {
                        "dataset":
                            dataset,

                        "fold":
                            fold,

                        "component_id":
                            row.component_id,

                        "source_id":
                            int(
                                row.source_id
                            ),

                        "target_id":
                            int(
                                row.target_id
                            ),

                        "pred_full_7um":
                            bool(
                                row.pred_full_7um
                            ),

                        "topology_class":
                            row.topology_class,

                        "policy_id":
                            policy_id,

                        "passes_rank_rule":
                            bool(
                                key
                                in
                                candidate_key_lookup
                            ),

                        "survives_conflict_resolution":
                            bool(
                                key
                                in
                                resolved_key_lookup
                            ),
                    }
                )


        del candidates
        del resolved


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == 103
    ):

        print(
            f"{ds_i:3d}/103"
            f" | fold={fold}"
            f" | {dataset}"
            f" | shortlist={len(shortlist)}"
            f" | cumulative={global_shortlist}"
        )


    del proposals
    del shortlist
    del ranks

    gc.collect()


assert global_shortlist == EXPECTED_SHORTLIST_ROWS


# ============================================================
# 17. FORMAL TABLES
# ============================================================

cardinality = pd.DataFrame(
    cardinality_rows
)


oracle_audit = pd.DataFrame(
    oracle_rows
)


assert len(
    cardinality
) == (
    103
    *
    len(
        POLICY_RULES
    )
)


assert len(
    oracle_audit
) == (
    EXPECTED_POSITIVES
    *
    len(
        POLICY_RULES
    )
)


# ============================================================
# 18. POLICY SUMMARY
# ============================================================

summary_rows = []


for policy_id in POLICY_RULES:

    card = cardinality[
        cardinality[
            "policy_id"
        ].eq(
            policy_id
        )
    ]


    oracle_policy = oracle_audit[
        oracle_audit[
            "policy_id"
        ].eq(
            policy_id
        )
    ]


    full7 = oracle_policy[
        oracle_policy[
            "pred_full_7um"
        ].astype(
            bool
        )
    ]


    resolved_fold_recall = []


    resolved_full7_fold_recall = []


    for fold in [
        1,
        2,
        3,
        4,
    ]:

        fold_oracle = oracle_policy[
            oracle_policy[
                "fold"
            ].eq(
                fold
            )
        ]


        fold_full7 = full7[
            full7[
                "fold"
            ].eq(
                fold
            )
        ]


        resolved_fold_recall.append(
            float(
                fold_oracle[
                    "survives_conflict_resolution"
                ].mean()
            )
        )


        resolved_full7_fold_recall.append(
            float(
                fold_full7[
                    "survives_conflict_resolution"
                ].mean()
            )
        )


    summary_rows.append(
        {
            "policy_id":
                policy_id,

            "rank_rule_rows":
                int(
                    card[
                        "rank_rule_rows"
                    ].sum()
                ),

            "resolved_actions":
                int(
                    card[
                        "resolved_actions"
                    ].sum()
                ),

            "action_fraction_of_shortlist":
                float(
                    card[
                        "resolved_actions"
                    ].sum()
                    /
                    EXPECTED_SHORTLIST_ROWS
                ),

            "true_rank_rule":
                int(
                    oracle_policy[
                        "passes_rank_rule"
                    ].sum()
                ),

            "true_resolved":
                int(
                    oracle_policy[
                        "survives_conflict_resolution"
                    ].sum()
                ),

            "true_resolved_recall":
                float(
                    oracle_policy[
                        "survives_conflict_resolution"
                    ].mean()
                ),

            "full7_rank_rule":
                int(
                    full7[
                        "passes_rank_rule"
                    ].sum()
                ),

            "full7_resolved":
                int(
                    full7[
                        "survives_conflict_resolution"
                    ].sum()
                ),

            "full7_resolved_recall":
                float(
                    full7[
                        "survives_conflict_resolution"
                    ].mean()
                ),

            "full7_fold1":
                resolved_full7_fold_recall[
                    0
                ],

            "full7_fold2":
                resolved_full7_fold_recall[
                    1
                ],

            "full7_fold3":
                resolved_full7_fold_recall[
                    2
                ],

            "full7_fold4":
                resolved_full7_fold_recall[
                    3
                ],

            "full7_worst_fold":
                float(
                    min(
                        resolved_full7_fold_recall
                    )
                ),

            "full7_macro":
                float(
                    np.mean(
                        resolved_full7_fold_recall
                    )
                ),
        }
    )


policy_summary = pd.DataFrame(
    summary_rows
)


print(
    "\n========== G3H3A POLICY FRONTIER =========="
)


print(
    policy_summary.to_string(
        index=False
    )
)


# ============================================================
# 19. PREDECLARED PRIMARY RULE DETAIL
# ============================================================

primary = policy_summary[
    policy_summary[
        "policy_id"
    ].eq(
        "ASYM_1x5"
    )
]


assert len(
    primary
) == 1


print(
    "\n========== PREDECLARED ASYM_1x5 =========="
)


print(
    primary.to_string(
        index=False
    )
)


# ============================================================
# 20. PERSIST
# ============================================================

cardinality.to_pickle(
    CARDINALITY_OUT
)


oracle_audit.to_pickle(
    ORACLE_OUT
)


SUMMARY = {
    "stage":
        "12.11G3H3A",

    "status":
        "OOF_ENDPOINT_CONFLICT_POLICY_FRONTIER_COMPLETE",

    "policy_contract_sha256":
        POLICY_CONTRACT_SHA,

    "scorer_id":
        SCORER_ID,

    "scorer_freeze_sha256":
        SCORER_FREEZE_SHA,

    "shortlist_rows_reproduced":
        int(
            global_shortlist
        ),

    "policy_summary":
        policy_summary.to_dict(
            "records"
        ),

    "predeclared_primary_policy":
        "ASYM_1x5",

    "final_policy_selected":
        False,

    "score_threshold_selected":
        False,

    "scorer_retuned":
        False,

    "fold0_used":
        False,

    "graph_modified":
        False,

    "next":
        (
            "REVIEW_ENDPOINT_POLICY_FRONTIER_AND_"
            "PROMOTE_ONE_RULE_TO_GRAPH_LEVEL_DEV_EVALUATION"
        ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


for path in [
    CARDINALITY_OUT,
    ORACLE_OUT,
    SUMMARY_OUT,
]:

    assert path.exists(), path


# ============================================================
# 21. RELOAD INTEGRITY
# ============================================================

card_check = pd.read_pickle(
    CARDINALITY_OUT
)


oracle_check = pd.read_pickle(
    ORACLE_OUT
)


summary_check = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert len(
    card_check
) == (
    103
    *
    len(
        POLICY_RULES
    )
)


assert len(
    oracle_check
) == (
    202
    *
    len(
        POLICY_RULES
    )
)


assert summary_check[
    "shortlist_rows_reproduced"
] == EXPECTED_SHORTLIST_ROWS


assert summary_check[
    "final_policy_selected"
] is False


assert summary_check[
    "fold0_used"
] is False


# ============================================================
# 22. FINAL DECISION
# ============================================================

print(
    "\n========== 12.11G3H3A DECISION =========="
)


print(
    "OOF_ENDPOINT_CONFLICT_POLICY_FRONTIER: PASS"
)


print(
    "policy contract SHA:",
    POLICY_CONTRACT_SHA
)


print(
    "scorer frozen:",
    "YES"
)


print(
    "scorer retuned:",
    "NO"
)


print(
    "full shortlist reproduced:",
    global_shortlist,
    "/",
    EXPECTED_SHORTLIST_ROWS
)


print(
    "predeclared primary:",
    "ASYM_1x5"
)


print(
    "final policy selected:",
    "NO"
)


print(
    "score threshold selected:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "graph modified:",
    "NO"
)


print(
    "next:"
)


print(
    "REVIEW_ENDPOINT_POLICY_FRONTIER_AND_PROMOTE_ONE_RULE_TO_GRAPH_LEVEL_DEV_EVALUATION"
)

In [ ]:
# ============================================================
# Stage 12.11G3H3B
#
# ASYM_1x5 + Frozen LogReg
# OOF Score-Quantile Sparsification Frontier
#
# ------------------------------------------------------------
# WHY THIS STAGE EXISTS
# ------------------------------------------------------------
#
# G3H3A:
#
#   ASYM_1x5
#       rank-rule rows     = 2,285,401
#       resolved actions   =   721,381
#       true resolved      =       100 / 202
#       FULL7 resolved     =        90 / 173
#
# Endpoint-rank filtering alone is therefore far too dense
# for direct graph mutation.
#
# ------------------------------------------------------------
# FROZEN RANK BACKBONE
# ------------------------------------------------------------
#
#   ASYM_1x5
#
#   (source rank <= 1 and target rank <= 5)
#       OR
#   (target rank <= 1 and source rank <= 5)
#
# This is promoted from the already-frozen G3H3A frontier.
#
# Selection:
#   DEVELOPMENT-GT-INFORMED
#
# Reasons:
#   - predeclared G3H1 primary
#   - ASYM_1x8 strictly dominated
#   - ASYM_3x8 dominated by ASYM_2x5
#   - ASYM_2x5 gains only 2 FULL7 resolved positives for
#     +48,900 actions and no worst-fold improvement
#
# ------------------------------------------------------------
# FROZEN SCORE SPARSIFICATION FAMILY
# ------------------------------------------------------------
#
# Score is NOT a calibrated probability.
#
# Therefore thresholds are defined as TRAINING-BACKGROUND
# SCORE QUANTILES, not probability cutoffs.
#
# For held-out fold f:
#
#   scorer:
#       LOFO model trained on other 3 folds
#
#   threshold calibration:
#       raw GT-blind G3H0 hash sample
#       from folds != f
#
# No oracle labels are used to compute threshold values.
#
# Frontier:
#
#   NO_SCORE_GATE
#   Q9900   = 99.00%
#   Q9950   = 99.50%
#   Q9990   = 99.90%
#   Q9995   = 99.95%
#   Q99975  = 99.975%
#   Q9999   = 99.99%
#
# ------------------------------------------------------------
# ACTION PIPELINE
# ------------------------------------------------------------
#
# complete frozen shortlist
#       ->
# frozen LOFO LogReg score
#       ->
# ASYM_1x5 endpoint rank gate
#       ->
# OOF score-quantile gate
#       ->
# deterministic node-disjoint conflict resolution
#
# ------------------------------------------------------------
# THIS STAGE DOES NOT
# ------------------------------------------------------------
#
#   - select final score quantile
#   - modify graph
#   - cut incumbent edges
#   - add rewire edges
#   - evaluate official graph score
#   - use Fold0
#   - retune scorer
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import gc
import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


RAW_SAMPLE_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_feature_sample_hash64_v1.pkl"
)


POSITIVE_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_positive_features_v1.pkl"
)


SCORER_FREEZE_PATH = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_frozen.json"
)


G3H3A_CONTRACT_PATH = (
    S12
    / "stage12_folds14_rewire_endpoint_policy_frontier_v1_frozen.json"
)


G3H3A_SUMMARY_PATH = (
    S12
    / "stage12_folds14_rewire_endpoint_policy_frontier_v1_summary.json"
)


# ------------------------------------------------------------
# G3H3B outputs
# ------------------------------------------------------------

BACKBONE_FREEZE_OUT = (
    S12
    / "stage12_folds14_rewire_asym1x5_backbone_v1_frozen.json"
)


QUANTILE_CONTRACT_OUT = (
    S12
    / "stage12_folds14_rewire_asym1x5_score_quantile_frontier_v1_frozen.json"
)


THRESHOLD_OUT = (
    S12
    / "stage12_folds14_rewire_asym1x5_score_quantile_thresholds_v1.pkl"
)


CARDINALITY_OUT = (
    S12
    / "stage12_folds14_rewire_asym1x5_score_quantile_frontier_v1_cardinality.pkl"
)


ORACLE_OUT = (
    S12
    / "stage12_folds14_rewire_asym1x5_score_quantile_frontier_v1_oracle.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_folds14_rewire_asym1x5_score_quantile_frontier_v1_summary.json"
)


for path in [
    CV_PATH,
    RAW_SAMPLE_PATH,
    POSITIVE_PATH,
    SCORER_FREEZE_PATH,
    G3H3A_CONTRACT_PATH,
    G3H3A_SUMMARY_PATH,
]:

    assert path.exists(), path


for path in [
    BACKBONE_FREEZE_OUT,
    QUANTILE_CONTRACT_OUT,
    THRESHOLD_OUT,
    CARDINALITY_OUT,
    ORACLE_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3H3B output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite a frozen/partial G3H3B run."
        )


# ============================================================
# 1. AUTHORITATIVE CONTRACTS
# ============================================================

SCORER_FREEZE_SHA = (
    "5ae66ee0f2da841bfe3a7a0b56fdab1"
    "b57f7ec3f504ba57441e2c8c272279e91"
)


G3H3A_CONTRACT_SHA = (
    "1b4272550edbc09ed38c93b40291e6ba"
    "998139925313ba1789c9a74396df35fc"
)


EXPECTED_SHORTLIST_ROWS = 17_501_674

EXPECTED_RAW_SAMPLE_ROWS = 273_658

EXPECTED_POSITIVES = 202

EXPECTED_FULL7 = 173


SCORER_ID = (
    "MULTIFRAME_REWIRE_LOGREG_SCORER_V1"
)


# ============================================================
# 2. REQUIRED CURRENT-KERNEL HELPERS
#
# Reuse successful G3H3A state.
# No model retraining.
# ============================================================

REQUIRED_CALLABLES = [
    "generate_dataset_proposals",
    "add_endpoint_ranks",
    "add_shortlist_rank_fields",
    "frozen_shortlist_mask",
    "attach_scoring_features",
    "score_shortlist_g3h3",
    "endpoint_score_ranks_g3h3",
    "policy_mask_g3h3",
    "resolve_conflicts_g3h3",
]


missing_callables = [
    name

    for name
    in REQUIRED_CALLABLES

    if (
        name not in globals()
        or
        not callable(
            globals()[
                name
            ]
        )
    )
]


assert not missing_callables, (
    "\nMissing G3H3A helper state:\n"
    f"{missing_callables}\n\n"
    "Do not restart the kernel."
)


assert (
    "lofo_models"
    in globals()
)


assert set(
    lofo_models.keys()
) == {
    1,
    2,
    3,
    4,
}


print(
    "========== G3H3B UPSTREAM STATE =========="
)


print(
    "missing helpers:",
    missing_callables
)


print(
    "LOFO scorer bundles:",
    sorted(
        lofo_models
    )
)


print(
    "scorer retrained:",
    "NO"
)


print(
    "UPSTREAM STATE: PASS"
)


# ============================================================
# 3. VERIFY FROZEN SCORER
# ============================================================

scorer_freeze = json.loads(
    SCORER_FREEZE_PATH.read_text(
        encoding="utf-8"
    )
)


assert scorer_freeze[
    "scorer_freeze_sha256"
] == SCORER_FREEZE_SHA


assert scorer_freeze[
    "scorer_id"
] == SCORER_ID


assert scorer_freeze[
    "future_scorer_family_retuning"
] is False


assert scorer_freeze[
    "score_threshold_selected"
] is False


assert scorer_freeze[
    "fold0_used"
] is False


# ============================================================
# 4. VERIFY G3H3A FRONTIER
# ============================================================

g3h3a_contract = json.loads(
    G3H3A_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


g3h3a_summary = json.loads(
    G3H3A_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert g3h3a_contract[
    "policy_contract_sha256"
] == G3H3A_CONTRACT_SHA


assert g3h3a_contract[
    "primary_predeclared_policy"
] == "ASYM_1x5"


assert g3h3a_summary[
    "final_policy_selected"
] is False


assert g3h3a_summary[
    "fold0_used"
] is False


frontier = pd.DataFrame(
    g3h3a_summary[
        "policy_summary"
    ]
)


assert set(
    frontier[
        "policy_id"
    ]
) == {
    "MUTUAL_1x1",
    "ASYM_1x3",
    "ASYM_1x5",
    "ASYM_1x8",
    "ASYM_2x5",
    "ASYM_3x8",
}


def frontier_row(
    policy_id,
):

    row = frontier[
        frontier[
            "policy_id"
        ].eq(
            policy_id
        )
    ]


    assert len(
        row
    ) == 1


    return row.iloc[
        0
    ]


p15 = frontier_row(
    "ASYM_1x5"
)


p18 = frontier_row(
    "ASYM_1x8"
)


p25 = frontier_row(
    "ASYM_2x5"
)


p38 = frontier_row(
    "ASYM_3x8"
)


assert int(
    p15[
        "resolved_actions"
    ]
) == 721_381


assert int(
    p15[
        "full7_resolved"
    ]
) == 90


# ASYM1x8 strictly dominated by ASYM1x5.
assert int(
    p18[
        "resolved_actions"
    ]
) > int(
    p15[
        "resolved_actions"
    ]
)


assert int(
    p18[
        "full7_resolved"
    ]
) == int(
    p15[
        "full7_resolved"
    ]
)


# ASYM3x8 strictly dominated by ASYM2x5.
assert int(
    p38[
        "resolved_actions"
    ]
) > int(
    p25[
        "resolved_actions"
    ]
)


assert int(
    p38[
        "full7_resolved"
    ]
) == int(
    p25[
        "full7_resolved"
    ]
)


# ============================================================
# 5. FREEZE ASYM1x5 RANK BACKBONE
#
# This selection is explicitly DEVELOPMENT-GT-INFORMED.
# No claim of pre-oracle selection is made.
# ============================================================

BACKBONE = {
    "stage":
        "12.11G3H3B",

    "backbone_id":
        "ASYM_SCORE_RANK_1x5_V1",

    "selected_rule":
        "ASYM_1x5",

    "source":
        "12.11G3H3A",

    "g3h3a_contract_sha256":
        G3H3A_CONTRACT_SHA,

    "scorer_id":
        SCORER_ID,

    "scorer_freeze_sha256":
        SCORER_FREEZE_SHA,

    "rule": {
        "a":
            1,

        "b":
            5,

        "definition":
            (
                "(SOURCE_SCORE_RANK<=1 AND TARGET_SCORE_RANK<=5)"
                " OR "
                "(TARGET_SCORE_RANK<=1 AND SOURCE_SCORE_RANK<=5)"
            ),
    },

    "selection_rationale": {
        "predeclared_g3h1_primary":
            True,

        "asym_1x8_strictly_dominated":
            True,

        "asym_3x8_dominated_by_asym_2x5":
            True,

        "asym_2x5_extra_actions_vs_1x5":
            int(
                p25[
                    "resolved_actions"
                ]
                -
                p15[
                    "resolved_actions"
                ]
            ),

        "asym_2x5_extra_full7_vs_1x5":
            int(
                p25[
                    "full7_resolved"
                ]
                -
                p15[
                    "full7_resolved"
                ]
            ),

        "asym_2x5_worst_fold_gain_vs_1x5":
            float(
                p25[
                    "full7_worst_fold"
                ]
                -
                p15[
                    "full7_worst_fold"
                ]
            ),
    },

    "selection_is_development_gt_informed":
        True,

    "runtime_rule_gt_blind":
        True,

    "score_threshold_selected":
        False,

    "final_rewire_policy_selected":
        False,

    "fold0_used":
        False,

    "graph_modified":
        False,
}


BACKBONE_SHA = hashlib.sha256(
    json.dumps(
        BACKBONE,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


BACKBONE[
    "backbone_freeze_sha256"
] = BACKBONE_SHA


BACKBONE_FREEZE_OUT.write_text(
    json.dumps(
        BACKBONE,
        indent=2,
    ),
    encoding="utf-8",
)


assert BACKBONE_FREEZE_OUT.exists()


print(
    "\n========== FROZEN RANK BACKBONE =========="
)


print(
    "backbone:",
    BACKBONE[
        "backbone_id"
    ]
)


print(
    "selected rule:",
    "ASYM_1x5"
)


print(
    "backbone SHA:",
    BACKBONE_SHA
)


print(
    "development-GT-informed selection:",
    "YES"
)


print(
    "runtime rule GT-blind:",
    "YES"
)


print(
    "score threshold selected:",
    "NO"
)


# ============================================================
# 6. FREEZE SCORE-QUANTILE FAMILY
#
# Quantile definitions themselves are now fixed.
#
# Score thresholds will be computed ONLY from the raw
# GT-blind G3H0 hash sample.
# ============================================================

QUANTILE_RULES = {
    "NO_SCORE_GATE":
        None,

    "Q9900":
        0.9900,

    "Q9950":
        0.9950,

    "Q9990":
        0.9990,

    "Q9995":
        0.9995,

    "Q99975":
        0.99975,

    "Q9999":
        0.9999,
}


QUANTILE_CONTRACT = {
    "stage":
        "12.11G3H3B",

    "contract_id":
        "ASYM1x5_OOF_SCORE_QUANTILE_FRONTIER_V1",

    "rank_backbone_id":
        BACKBONE[
            "backbone_id"
        ],

    "rank_backbone_sha256":
        BACKBONE_SHA,

    "scorer_id":
        SCORER_ID,

    "scorer_freeze_sha256":
        SCORER_FREEZE_SHA,

    "quantile_rules":
        QUANTILE_RULES,

    "threshold_calibration": {
        "population":
            "RAW_G3H0_GT_BLIND_HASH_SAMPLE",

        "per_heldout_fold":
            True,

        "heldout_fold_excluded":
            True,

        "positive_labels_used":
            False,

        "score_is_calibrated_probability":
            False,

        "interpretation":
            "TRAINING_BACKGROUND_SCORE_QUANTILE",
    },

    "action_pipeline": [
        "COMPLETE_FROZEN_SHORTLIST",
        "FROZEN_LOFO_LOGREG_SCORE",
        "ASYM_1x5_RANK_GATE",
        "OOF_SCORE_QUANTILE_GATE",
        "DETERMINISTIC_NODE_DISJOINT_CONFLICT_RESOLUTION",
    ],

    "family_design_development_gt_informed":
        True,

    "threshold_values_gt_blind":
        True,

    "final_quantile_selected":
        False,

    "graph_modified":
        False,

    "fold0_used":
        False,
}


QUANTILE_CONTRACT_SHA = hashlib.sha256(
    json.dumps(
        QUANTILE_CONTRACT,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


QUANTILE_CONTRACT[
    "quantile_contract_sha256"
] = QUANTILE_CONTRACT_SHA


QUANTILE_CONTRACT_OUT.write_text(
    json.dumps(
        QUANTILE_CONTRACT,
        indent=2,
    ),
    encoding="utf-8",
)


assert QUANTILE_CONTRACT_OUT.exists()


print(
    "\n========== FROZEN SCORE QUANTILE FAMILY =========="
)


print(
    "contract SHA:",
    QUANTILE_CONTRACT_SHA
)


print(
    "rules:",
    list(
        QUANTILE_RULES
    )
)


print(
    "score calibrated probability:",
    "NO"
)


print(
    "final quantile selected:",
    "NO"
)


# ============================================================
# PART A
#
# GT-BLIND THRESHOLD CALIBRATION
#
# IMPORTANT:
# POSITIVE_PATH IS NOT READ UNTIL AFTER THRESHOLD_OUT EXISTS.
# ============================================================


# ============================================================
# 7. RAW GT-BLIND CALIBRATION SAMPLE
# ============================================================

raw_sample = pd.read_pickle(
    RAW_SAMPLE_PATH
)


assert len(
    raw_sample
) == EXPECTED_RAW_SAMPLE_ROWS


assert set(
    raw_sample[
        "fold"
    ].astype(
        int
    )
) == {
    1,
    2,
    3,
    4,
}


print(
    "\n========== GT-BLIND THRESHOLD CALIBRATION =========="
)


print(
    "raw sample rows:",
    len(
        raw_sample
    )
)


print(
    "oracle labels loaded for threshold calibration:",
    "NO"
)


# ============================================================
# 8. COMPUTE LOFO SCORE QUANTILE THRESHOLDS
# ============================================================

threshold_rows = []


for heldout_fold in [
    1,
    2,
    3,
    4,
]:

    calibration = raw_sample[
        raw_sample[
            "fold"
        ].ne(
            heldout_fold
        )
    ]


    assert not calibration[
        "fold"
    ].eq(
        heldout_fold
    ).any()


    scores = score_shortlist_g3h3(
        calibration,
        lofo_models[
            heldout_fold
        ],
    )


    assert len(
        scores
    ) == len(
        calibration
    )


    assert np.isfinite(
        scores
    ).all()


    for policy_id, quantile in QUANTILE_RULES.items():

        if quantile is None:

            threshold = -np.inf


        else:

            threshold = float(
                np.quantile(
                    scores,
                    quantile,
                    method="linear",
                )
            )


        threshold_rows.append(
            {
                "heldout_fold":
                    int(
                        heldout_fold
                    ),

                "policy_id":
                    policy_id,

                "quantile":
                    (
                        np.nan
                        if quantile is None
                        else float(
                            quantile
                        )
                    ),

                "threshold":
                    float(
                        threshold
                    ),

                "calibration_rows":
                    int(
                        len(
                            calibration
                        )
                    ),

                "heldout_rows_used":
                    0,

                "oracle_labels_used":
                    False,
            }
        )


    del calibration
    del scores

    gc.collect()


threshold_table = pd.DataFrame(
    threshold_rows
)


assert len(
    threshold_table
) == (
    4
    *
    len(
        QUANTILE_RULES
    )
)


threshold_table.to_pickle(
    THRESHOLD_OUT
)


assert THRESHOLD_OUT.exists()


print(
    "\n========== FROZEN OOF SCORE THRESHOLDS =========="
)


print(
    threshold_table.to_string(
        index=False
    )
)


print(
    "\nTHRESHOLDS FROZEN BEFORE ORACLE EVALUATION: YES"
)


# ============================================================
# PART B
#
# DEVELOPMENT ORACLE MAY NOW BE READ
# ============================================================


# ============================================================
# 9. DEVELOPMENT CORPUS
# ============================================================

cv = pd.read_csv(
    CV_PATH
)


dev_manifest = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    dev_manifest
) == 103


assert not dev_manifest[
    "fold"
].eq(
    0
).any()


development_names = (
    dev_manifest[
        "dataset"
    ]
    .tolist()
)


fold_map = dict(
    zip(
        dev_manifest[
            "dataset"
        ],

        dev_manifest[
            "fold"
        ].astype(
            int
        ),
    )
)


# ============================================================
# 10. NOW LOAD DEVELOPMENT POSITIVE ORACLE
# ============================================================

positive = pd.read_pickle(
    POSITIVE_PATH
)


assert len(
    positive
) == EXPECTED_POSITIVES


assert int(
    positive[
        "pred_full_7um"
    ].sum()
) == EXPECTED_FULL7


assert set(
    positive[
        "fold"
    ].astype(
        int
    )
) == {
    1,
    2,
    3,
    4,
}


positive_meta = positive[
    [
        "dataset",
        "fold",
        "component_id",
        "source_id",
        "target_id",
        "pred_full_7um",
        "topology_class",
    ]
].copy()


positive_key_set_by_dataset = {}


for dataset, sub in positive_meta.groupby(
    "dataset"
):

    positive_key_set_by_dataset[
        dataset
    ] = set(
        zip(
            sub[
                "source_id"
            ].astype(
                int
            ),

            sub[
                "target_id"
            ].astype(
                int
            ),
        )
    )


# ============================================================
# 11. THRESHOLD LOOKUP
# ============================================================

threshold_lookup = {}


for row in threshold_table.itertuples(
    index=False
):

    threshold_lookup[
        (
            int(
                row.heldout_fold
            ),
            str(
                row.policy_id
            ),
        )
    ] = float(
        row.threshold
    )


# ============================================================
# 12. FULL FROZEN SHORTLIST OOF SPARSIFICATION FRONTIER
# ============================================================

print(
    "\n============================================================"
)


print(
    "OOF ASYM1x5 SCORE-QUANTILE FRONTIER"
)


print(
    "============================================================"
)


cardinality_rows = []

oracle_rows = []


global_shortlist = 0


for ds_i, dataset in enumerate(
    development_names,
    start=1,
):

    fold = int(
        fold_map[
            dataset
        ]
    )


    # --------------------------------------------------------
    # Exact parent shortlist
    # --------------------------------------------------------

    (
        proposals,
        _,
        _,
    ) = generate_dataset_proposals(
        dataset
    )


    proposals = add_endpoint_ranks(
        proposals
    )


    proposals = add_shortlist_rank_fields(
        proposals
    )


    shortlist = (
        proposals[
            frozen_shortlist_mask(
                proposals
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    shortlist = attach_scoring_features(
        shortlist
    )


    global_shortlist += int(
        len(
            shortlist
        )
    )


    # --------------------------------------------------------
    # Frozen LOFO scorer
    # --------------------------------------------------------

    shortlist[
        "model_score"
    ] = score_shortlist_g3h3(
        shortlist,
        lofo_models[
            fold
        ],
    )


    # --------------------------------------------------------
    # Endpoint score ranks on COMPLETE shortlist
    # --------------------------------------------------------

    ranks = endpoint_score_ranks_g3h3(
        shortlist
    )


    shortlist[
        "source_score_rank"
    ] = ranks[
        "source_score_rank"
    ].to_numpy(
        copy=True
    )


    shortlist[
        "target_score_rank"
    ] = ranks[
        "target_score_rank"
    ].to_numpy(
        copy=True
    )


    # --------------------------------------------------------
    # Frozen ASYM1x5 backbone
    # --------------------------------------------------------

    backbone_mask = policy_mask_g3h3(
        shortlist,
        a=1,
        b=5,
    )


    backbone_candidates = shortlist[
        backbone_mask
    ].copy()


    true_keys = positive_key_set_by_dataset.get(
        dataset,
        set(),
    )


    positive_sub = positive_meta[
        positive_meta[
            "dataset"
        ].eq(
            dataset
        )
    ]


    # --------------------------------------------------------
    # Quantile frontier
    # --------------------------------------------------------

    for policy_id in QUANTILE_RULES:

        threshold = threshold_lookup[
            (
                fold,
                policy_id,
            )
        ]


        if np.isneginf(
            threshold
        ):

            gated = backbone_candidates.copy()


        else:

            gated = backbone_candidates[
                backbone_candidates[
                    "model_score"
                ].ge(
                    threshold
                )
            ].copy()


        resolved = resolve_conflicts_g3h3(
            gated
        )


        gated_pairs = set(
            zip(
                gated[
                    "source_id"
                ].astype(
                    int
                ),

                gated[
                    "target_id"
                ].astype(
                    int
                ),
            )
        )


        resolved_pairs = set(
            zip(
                resolved[
                    "source_id"
                ].astype(
                    int
                ),

                resolved[
                    "target_id"
                ].astype(
                    int
                ),
            )
        )


        cardinality_rows.append(
            {
                "dataset":
                    dataset,

                "fold":
                    fold,

                "policy_id":
                    policy_id,

                "score_threshold":
                    float(
                        threshold
                    ),

                "shortlist_rows":
                    int(
                        len(
                            shortlist
                        )
                    ),

                "asym1x5_rows":
                    int(
                        len(
                            backbone_candidates
                        )
                    ),

                "score_gated_rows":
                    int(
                        len(
                            gated
                        )
                    ),

                "resolved_actions":
                    int(
                        len(
                            resolved
                        )
                    ),

                "gated_true_pairs":
                    int(
                        len(
                            gated_pairs
                            &
                            true_keys
                        )
                    ),

                "resolved_true_pairs":
                    int(
                        len(
                            resolved_pairs
                            &
                            true_keys
                        )
                    ),
            }
        )


        if len(
            positive_sub
        ):

            for row in positive_sub.itertuples(
                index=False
            ):

                key = (
                    int(
                        row.source_id
                    ),
                    int(
                        row.target_id
                    ),
                )


                oracle_rows.append(
                    {
                        "dataset":
                            dataset,

                        "fold":
                            fold,

                        "component_id":
                            row.component_id,

                        "source_id":
                            int(
                                row.source_id
                            ),

                        "target_id":
                            int(
                                row.target_id
                            ),

                        "pred_full_7um":
                            bool(
                                row.pred_full_7um
                            ),

                        "topology_class":
                            row.topology_class,

                        "policy_id":
                            policy_id,

                        "score_threshold":
                            float(
                                threshold
                            ),

                        "passes_asym1x5":
                            bool(
                                key
                                in
                                set(
                                    zip(
                                        backbone_candidates[
                                            "source_id"
                                        ].astype(
                                            int
                                        ),

                                        backbone_candidates[
                                            "target_id"
                                        ].astype(
                                            int
                                        ),
                                    )
                                )
                            ),

                        "passes_score_gate":
                            bool(
                                key
                                in
                                gated_pairs
                            ),

                        "survives_conflict_resolution":
                            bool(
                                key
                                in
                                resolved_pairs
                            ),
                    }
                )


        del gated
        del resolved

        del gated_pairs
        del resolved_pairs


    if (
        ds_i == 1
        or
        ds_i % 10 == 0
        or
        ds_i == 103
    ):

        print(
            f"{ds_i:3d}/103"
            f" | fold={fold}"
            f" | {dataset}"
            f" | shortlist={len(shortlist)}"
            f" | ASYM1x5={len(backbone_candidates)}"
            f" | cumulative={global_shortlist}"
        )


    del proposals
    del shortlist

    del ranks
    del backbone_candidates

    gc.collect()


assert global_shortlist == EXPECTED_SHORTLIST_ROWS


# ============================================================
# 13. TABLES
# ============================================================

cardinality = pd.DataFrame(
    cardinality_rows
)


oracle_audit = pd.DataFrame(
    oracle_rows
)


assert len(
    cardinality
) == (
    103
    *
    len(
        QUANTILE_RULES
    )
)


assert len(
    oracle_audit
) == (
    EXPECTED_POSITIVES
    *
    len(
        QUANTILE_RULES
    )
)


# ============================================================
# 14. FRONTIER SUMMARY
# ============================================================

summary_rows = []


for policy_id in QUANTILE_RULES:

    card = cardinality[
        cardinality[
            "policy_id"
        ].eq(
            policy_id
        )
    ]


    oracle_policy = oracle_audit[
        oracle_audit[
            "policy_id"
        ].eq(
            policy_id
        )
    ]


    full7 = oracle_policy[
        oracle_policy[
            "pred_full_7um"
        ].astype(
            bool
        )
    ]


    fold_full7 = []


    for fold in [
        1,
        2,
        3,
        4,
    ]:

        ff = full7[
            full7[
                "fold"
            ].eq(
                fold
            )
        ]


        assert len(
            ff
        ) > 0


        fold_full7.append(
            float(
                ff[
                    "survives_conflict_resolution"
                ].mean()
            )
        )


    total_actions = int(
        card[
            "resolved_actions"
        ].sum()
    )


    resolved_true = int(
        oracle_policy[
            "survives_conflict_resolution"
        ].sum()
    )


    resolved_full7 = int(
        full7[
            "survives_conflict_resolution"
        ].sum()
    )


    summary_rows.append(
        {
            "policy_id":
                policy_id,

            "resolved_actions":
                total_actions,

            "action_fraction_of_shortlist":
                float(
                    total_actions
                    /
                    EXPECTED_SHORTLIST_ROWS
                ),

            "mean_actions_per_dataset":
                float(
                    total_actions
                    /
                    103
                ),

            "true_resolved":
                resolved_true,

            "true_recall":
                float(
                    resolved_true
                    /
                    EXPECTED_POSITIVES
                ),

            "full7_resolved":
                resolved_full7,

            "full7_recall":
                float(
                    resolved_full7
                    /
                    EXPECTED_FULL7
                ),

            "full7_fold1":
                fold_full7[
                    0
                ],

            "full7_fold2":
                fold_full7[
                    1
                ],

            "full7_fold3":
                fold_full7[
                    2
                ],

            "full7_fold4":
                fold_full7[
                    3
                ],

            "full7_macro":
                float(
                    np.mean(
                        fold_full7
                    )
                ),

            "full7_worst_fold":
                float(
                    np.min(
                        fold_full7
                    )
                ),

            "actions_per_full7_true":
                (
                    float(
                        total_actions
                        /
                        resolved_full7
                    )
                    if resolved_full7 > 0
                    else np.inf
                ),
        }
    )


frontier_summary = (
    pd.DataFrame(
        summary_rows
    )
    .sort_values(
        "resolved_actions",
        ascending=False,
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 15. FIDELITY — NO SCORE GATE MUST REPRODUCE G3H3A ASYM1x5
# ============================================================

nogate = frontier_summary[
    frontier_summary[
        "policy_id"
    ].eq(
        "NO_SCORE_GATE"
    )
]


assert len(
    nogate
) == 1


nogate = nogate.iloc[
    0
]


assert int(
    nogate[
        "resolved_actions"
    ]
) == 721_381


assert int(
    nogate[
        "true_resolved"
    ]
) == 100


assert int(
    nogate[
        "full7_resolved"
    ]
) == 90


assert np.isclose(
    float(
        nogate[
            "full7_worst_fold"
        ]
    ),
    0.46,
    atol=1e-12,
)


print(
    "\n========== G3H3B NO-GATE FIDELITY =========="
)


print(
    "resolved actions:",
    int(
        nogate[
            "resolved_actions"
        ]
    ),
    "/ 721381"
)


print(
    "true resolved:",
    int(
        nogate[
            "true_resolved"
        ]
    ),
    "/ 100"
)


print(
    "FULL7 resolved:",
    int(
        nogate[
            "full7_resolved"
        ]
    ),
    "/ 90"
)


print(
    "G3H3A_ASYM1x5_REPRODUCTION: PASS"
)


# ============================================================
# 16. PRINT FRONTIER
# ============================================================

print(
    "\n========== G3H3B SCORE-QUANTILE FRONTIER =========="
)


print(
    frontier_summary.to_string(
        index=False
    )
)


# ============================================================
# 17. PERSIST
# ============================================================

cardinality.to_pickle(
    CARDINALITY_OUT
)


oracle_audit.to_pickle(
    ORACLE_OUT
)


SUMMARY = {
    "stage":
        "12.11G3H3B",

    "status":
        "ASYM1x5_SCORE_QUANTILE_SPARSIFICATION_FRONTIER_COMPLETE",

    "rank_backbone_id":
        BACKBONE[
            "backbone_id"
        ],

    "rank_backbone_sha256":
        BACKBONE_SHA,

    "quantile_contract_sha256":
        QUANTILE_CONTRACT_SHA,

    "scorer_id":
        SCORER_ID,

    "scorer_freeze_sha256":
        SCORER_FREEZE_SHA,

    "shortlist_rows_reproduced":
        int(
            global_shortlist
        ),

    "frontier":
        frontier_summary.to_dict(
            "records"
        ),

    "thresholds":
        threshold_table.to_dict(
            "records"
        ),

    "threshold_values_gt_blind":
        True,

    "rank_backbone_selection_development_gt_informed":
        True,

    "final_quantile_selected":
        False,

    "official_graph_metric_evaluated":
        False,

    "graph_modified":
        False,

    "scorer_retuned":
        False,

    "fold0_used":
        False,

    "next":
        (
            "SELECT_SPARSE_SCORE_QUANTILE_KNEE_"
            "THEN_RUN_GRAPH_LEVEL_DEV_REWIRE_EVALUATION"
        ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


for path in [
    CARDINALITY_OUT,
    ORACLE_OUT,
    SUMMARY_OUT,
]:

    assert path.exists(), path


# ============================================================
# 18. RELOAD
# ============================================================

assert len(
    pd.read_pickle(
        CARDINALITY_OUT
    )
) == (
    103
    *
    len(
        QUANTILE_RULES
    )
)


assert len(
    pd.read_pickle(
        ORACLE_OUT
    )
) == (
    202
    *
    len(
        QUANTILE_RULES
    )
)


summary_check = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert summary_check[
    "shortlist_rows_reproduced"
] == EXPECTED_SHORTLIST_ROWS


assert summary_check[
    "final_quantile_selected"
] is False


assert summary_check[
    "official_graph_metric_evaluated"
] is False


assert summary_check[
    "fold0_used"
] is False


# ============================================================
# 19. FINAL
# ============================================================

print(
    "\n========== 12.11G3H3B DECISION =========="
)


print(
    "ASYM1x5_SCORE_QUANTILE_SPARSIFICATION_FRONTIER: PASS"
)


print(
    "rank backbone:",
    "ASYM_1x5"
)


print(
    "scorer:",
    SCORER_ID
)


print(
    "scorer retuned:",
    "NO"
)


print(
    "threshold values GT-blind:",
    "YES"
)


print(
    "final score quantile selected:",
    "NO"
)


print(
    "official graph metric evaluated:",
    "NO"
)


print(
    "graph modified:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "next:"
)


print(
    "SELECT_SPARSE_SCORE_QUANTILE_KNEE_THEN_GRAPH_LEVEL_DEV_EVALUATION"
)

In [ ]:
# ============================================================
# Stage 12.11G3H3C
#
# Q9990 OOF Graph-Level Development Evaluation
#
# ------------------------------------------------------------
# FROZEN UPSTREAM
# ------------------------------------------------------------
#
# Scorer:
#   MULTIFRAME_REWIRE_LOGREG_SCORER_V1
#
# Scorer freeze SHA:
#   5ae66ee0f2da841bfe3a7a0b56fdab1b57f7ec3f504ba57441e2c8c272279e91
#
# Rank backbone:
#   ASYM_SCORE_RANK_1x5_V1
#
# Backbone SHA:
#   34ffb76b15e8c7c17e93681d34c3b9aa45c7ef8cc6e7171d32dd542ef12ab55b
#
# Score gate:
#   Q9990
#
# Score-quantile frontier SHA:
#   6df3cf46ee684f06120fb948f9c2e4bcc36a30884228a15e7699b35f95ae991e
#
# Expected resolved actions:
#   13,928
#
# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# PART A — GT-BLIND ACTION FREEZE
#
#   exact CURRENT_BEST-derived frozen shortlist
#       ->
#   frozen held-out-fold LogReg scorer
#       ->
#   frozen ASYM1x5 rank gate
#       ->
#   frozen fold-specific Q9990 score threshold
#       ->
#   frozen deterministic node-disjoint resolver
#       ->
#   persist exact 13,928 actions
#
# PART B — OFFICIAL DEVELOPMENT GRAPH EVALUATION
#
# For each folds1-4 dataset:
#
#   CURRENT_BEST graph
#       ->
#   graph.copy()
#       ->
#   remove incumbent source outgoing edge if present
#   remove incumbent target incoming edge if present
#       ->
#   add frozen rewire edge
#       ->
#   fv9.evaluate_graph(...)
#
# Untouched CURRENT_BEST is evaluated in parallel and MUST
# reproduce the authoritative G1 development baseline before
# the mutated result is accepted.
#
# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------
#
# This is the FIRST actual graph-level evaluation of this
# rewire branch.
#
# Q9990 selection:
#   DEVELOPMENT-GT-INFORMED
#
# Runtime action construction:
#   GT-BLIND
#
# Fold0:
#   NEVER USED
#
# No prediction GEFF is written.
#
# No scorer retuning.
# No shortlist retuning.
# No score-threshold retuning.
#
# ------------------------------------------------------------
# NEW EDGE ATTRIBUTES
# ------------------------------------------------------------
#
# edge_dist:
#   exact fv9.recovered_edge_dist(...)
#
# edge_prob:
#   1.0
#
# This is an ephemeral schema-valid placeholder ONLY.
# Official evaluator does not use edge_prob for matching.
#
# Frozen LogReg score is retained separately as model_score
# in the action artifact and is NOT represented as probability.
#
# ------------------------------------------------------------
# Keep after running: YES
# ============================================================


from pathlib import Path

import gc
import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


TRAIN_ROOT = (
    PROJECT
    / "data"
    / "raw"
    / "competition"
    / "biohub-cell-tracking-during-development"
    / "train"
)


CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


# ------------------------------------------------------------
# Frozen upstream
# ------------------------------------------------------------

SCORER_FREEZE_PATH = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_frozen.json"
)


BACKBONE_PATH = (
    S12
    / "stage12_folds14_rewire_asym1x5_backbone_v1_frozen.json"
)


QUANTILE_CONTRACT_PATH = (
    S12
    / "stage12_folds14_rewire_asym1x5_score_quantile_frontier_v1_frozen.json"
)


THRESHOLD_PATH = (
    S12
    / "stage12_folds14_rewire_asym1x5_score_quantile_thresholds_v1.pkl"
)


G3H3B_CARDINALITY_PATH = (
    S12
    / "stage12_folds14_rewire_asym1x5_score_quantile_frontier_v1_cardinality.pkl"
)


G3H3B_SUMMARY_PATH = (
    S12
    / "stage12_folds14_rewire_asym1x5_score_quantile_frontier_v1_summary.json"
)


# Authoritative G1 CURRENT_BEST dataset rows.
CURRENT_BEST_DATASET_PATH = (
    S12
    / "stage12_folds14_current_best_edge_residual_datasets.pkl"
)


# ------------------------------------------------------------
# Formal G3H3C outputs
# ------------------------------------------------------------

POLICY_CANDIDATE_OUT = (
    S12
    / "stage12_folds14_rewire_q9990_graph_candidate_v1_frozen.json"
)


ACTIONS_OUT = (
    S12
    / "stage12_folds14_rewire_q9990_graph_candidate_v1_actions.pkl"
)


ACTION_SUMMARY_OUT = (
    S12
    / "stage12_folds14_rewire_q9990_graph_candidate_v1_action_summary.json"
)


METRICS_OUT = (
    S12
    / "stage12_folds14_rewire_q9990_graph_candidate_v1_metrics.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_folds14_rewire_q9990_graph_candidate_v1_summary.json"
)


for path in [
    TRAIN_ROOT,
    CV_PATH,
    SCORER_FREEZE_PATH,
    BACKBONE_PATH,
    QUANTILE_CONTRACT_PATH,
    THRESHOLD_PATH,
    G3H3B_CARDINALITY_PATH,
    G3H3B_SUMMARY_PATH,
    CURRENT_BEST_DATASET_PATH,
]:

    assert path.exists(), path


for path in [
    POLICY_CANDIDATE_OUT,
    ACTIONS_OUT,
    ACTION_SUMMARY_OUT,
    METRICS_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3H3C formal output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite a partial/completed graph-level run."
        )


# ============================================================
# 1. AUTHORITATIVE CONTRACTS
# ============================================================

SCORER_FREEZE_SHA = (
    "5ae66ee0f2da841bfe3a7a0b56fdab1b"
    "57f7ec3f504ba57441e2c8c272279e91"
)


BACKBONE_SHA = (
    "34ffb76b15e8c7c17e93681d34c3b9aa"
    "45c7ef8cc6e7171d32dd542ef12ab55b"
)


QUANTILE_CONTRACT_SHA = (
    "6df3cf46ee684f06120fb948f9c2e4bcc"
    "36a30884228a15e7699b35f95ae991e"
)


SCORER_ID = (
    "MULTIFRAME_REWIRE_LOGREG_SCORER_V1"
)


POLICY_ID = (
    "ASYM1x5_Q9990_NODEDISJOINT_REWIRE_V1"
)


EXPECTED_ACTIONS = 13_928


# ------------------------------------------------------------
# Authoritative development CURRENT_BEST
# ------------------------------------------------------------

EXPECTED_BASE_TP = 77_535
EXPECTED_BASE_FP = 9_398
EXPECTED_BASE_FN = 10_254

EXPECTED_BASE_EDGE_J = 0.797791886
EXPECTED_BASE_ADJ_EDGE_J = 0.789803521
EXPECTED_BASE_DIV_J = 0.021739130
EXPECTED_BASE_NODE_RECALL = 0.965506136

EXPECTED_BASE_SCORE = 0.7919774336227487


# ============================================================
# 2. REQUIRED IN-MEMORY EXACT HELPERS
# ============================================================

REQUIRED_CALLABLES = [
    "build_current_best",
    "graph_state",

    "generate_dataset_proposals",
    "add_endpoint_ranks",
    "add_shortlist_rank_fields",
    "frozen_shortlist_mask",
    "attach_scoring_features",

    "score_shortlist_g3h3",
    "endpoint_score_ranks_g3h3",
    "policy_mask_g3h3",
    "resolve_conflicts_g3h3",
]


missing_callables = [
    name
    for name in REQUIRED_CALLABLES
    if (
        name not in globals()
        or not callable(
            globals()[name]
        )
    )
]


assert not missing_callables, (
    "\nRequired exact helper state missing:\n"
    f"{missing_callables}"
)


assert "lofo_models" in globals()


assert set(
    lofo_models.keys()
) == {
    1,
    2,
    3,
    4,
}


assert "fv9" in globals()


print(
    "========== G3H3C UPSTREAM STATE =========="
)

print(
    "missing helpers:",
    missing_callables
)

print(
    "LOFO models:",
    sorted(
        lofo_models
    )
)

print(
    "scorer retuned:",
    "NO"
)

print(
    "UPSTREAM STATE: PASS"
)


# ============================================================
# 3. VERIFY FROZEN UPSTREAM CONTRACTS
# ============================================================

scorer_freeze = json.loads(
    SCORER_FREEZE_PATH.read_text(
        encoding="utf-8"
    )
)


assert scorer_freeze[
    "scorer_freeze_sha256"
] == SCORER_FREEZE_SHA


assert scorer_freeze[
    "scorer_id"
] == SCORER_ID


assert scorer_freeze[
    "future_scorer_family_retuning"
] is False


assert scorer_freeze[
    "fold0_used"
] is False


backbone = json.loads(
    BACKBONE_PATH.read_text(
        encoding="utf-8"
    )
)


assert backbone[
    "backbone_freeze_sha256"
] == BACKBONE_SHA


assert backbone[
    "selected_rule"
] == "ASYM_1x5"


assert backbone[
    "score_threshold_selected"
] is False


assert backbone[
    "fold0_used"
] is False


quantile_contract = json.loads(
    QUANTILE_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


assert quantile_contract[
    "quantile_contract_sha256"
] == QUANTILE_CONTRACT_SHA


assert quantile_contract[
    "final_quantile_selected"
] is False


assert quantile_contract[
    "fold0_used"
] is False


# ============================================================
# 4. VERIFY G3H3B Q9990 FRONTIER RESULT
# ============================================================

g3h3b_summary = json.loads(
    G3H3B_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert g3h3b_summary[
    "status"
] == (
    "ASYM1x5_SCORE_QUANTILE_SPARSIFICATION_FRONTIER_COMPLETE"
)


assert g3h3b_summary[
    "fold0_used"
] is False


frontier = pd.DataFrame(
    g3h3b_summary[
        "frontier"
    ]
)


q9990_row = frontier[
    frontier[
        "policy_id"
    ].eq(
        "Q9990"
    )
]


assert len(
    q9990_row
) == 1


q9990_row = q9990_row.iloc[
    0
]


assert int(
    q9990_row[
        "resolved_actions"
    ]
) == EXPECTED_ACTIONS


assert int(
    q9990_row[
        "full7_resolved"
    ]
) == 23


assert float(
    q9990_row[
        "full7_worst_fold"
    ]
) > 0.0


# ============================================================
# 5. FREEZE Q9990 AS GRAPH-LEVEL DEVELOPMENT CANDIDATE
#
# IMPORTANT:
# This is NOT final deployment policy promotion.
#
# Selection is explicitly development-GT-informed from G3H3B.
# It is frozen BEFORE official graph-level evaluation below.
# ============================================================

POLICY_CANDIDATE = {
    "stage":
        "12.11G3H3C",

    "policy_id":
        POLICY_ID,

    "status":
        "FROZEN_GRAPH_LEVEL_DEVELOPMENT_CANDIDATE",

    "scorer_id":
        SCORER_ID,

    "scorer_freeze_sha256":
        SCORER_FREEZE_SHA,

    "rank_backbone_id":
        backbone[
            "backbone_id"
        ],

    "rank_backbone_sha256":
        BACKBONE_SHA,

    "score_gate":
        "Q9990",

    "score_quantile":
        0.999,

    "quantile_contract_sha256":
        QUANTILE_CONTRACT_SHA,

    "expected_resolved_actions":
        EXPECTED_ACTIONS,

    "selection_rationale": {
        "q9950_resolved_actions":
            51_855,

        "q9950_full7_resolved":
            54,

        "q9990_resolved_actions":
            13_928,

        "q9990_full7_resolved":
            23,

        "q9990_full7_worst_fold":
            float(
                q9990_row[
                    "full7_worst_fold"
                ]
            ),

        "q9995_resolved_actions":
            6_668,

        "q9995_full7_resolved":
            14,

        "q99975_fold_collapse":
            True,

        "reason":
            (
                "SPARSE_KNEE_WITH_ALL_FOUR_FOLDS_NONZERO_"
                "BEFORE_STRONG_FULL7_RETENTION_COLLAPSE"
            ),
    },

    "selection_is_development_gt_informed":
        True,

    "runtime_action_construction_gt_blind":
        True,

    "new_edge_attributes": {
        "edge_dist":
            "FV9_RECOVERED_EDGE_DIST",

        "edge_prob":
            1.0,

        "edge_prob_semantics":
            (
                "EPHEMERAL_SCHEMA_PLACEHOLDER_"
                "NOT_SCORER_PROBABILITY"
            ),
    },

    "official_graph_metric_evaluated_before_freeze":
        False,

    "final_policy_promoted":
        False,

    "fold0_used":
        False,

    "prediction_geff_written":
        False,
}


candidate_payload = json.dumps(
    POLICY_CANDIDATE,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
).encode(
    "utf-8"
)


POLICY_CANDIDATE_SHA = hashlib.sha256(
    candidate_payload
).hexdigest()


POLICY_CANDIDATE[
    "policy_candidate_sha256"
] = POLICY_CANDIDATE_SHA


POLICY_CANDIDATE_OUT.write_text(
    json.dumps(
        POLICY_CANDIDATE,
        indent=2,
    ),
    encoding="utf-8",
)


assert POLICY_CANDIDATE_OUT.exists()


print(
    "\n========== FROZEN GRAPH-LEVEL CANDIDATE =========="
)

print(
    "policy:",
    POLICY_ID
)

print(
    "candidate SHA:",
    POLICY_CANDIDATE_SHA
)

print(
    "rank backbone:",
    "ASYM_1x5"
)

print(
    "score gate:",
    "Q9990"
)

print(
    "selection development-GT-informed:",
    "YES"
)

print(
    "official graph metric seen before candidate freeze:",
    "NO"
)

print(
    "final policy promoted:",
    "NO"
)


# ============================================================
# 6. DEVELOPMENT CORPUS
# ============================================================

cv = pd.read_csv(
    CV_PATH
)


corp = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    corp
) == 103


assert corp[
    "dataset"
].nunique() == 103


assert not corp[
    "fold"
].eq(
    0
).any()


names = corp[
    "dataset"
].tolist()


fold_map_eval = dict(
    zip(
        corp[
            "dataset"
        ],

        corp[
            "fold"
        ].astype(
            int
        ),
    )
)


# ============================================================
# 7. LOAD FROZEN Q9990 THRESHOLDS
# ============================================================

thresholds = pd.read_pickle(
    THRESHOLD_PATH
)


q9990_thresholds = (
    thresholds[
        thresholds[
            "policy_id"
        ].eq(
            "Q9990"
        )
    ]
    .copy()
)


assert len(
    q9990_thresholds
) == 4


assert set(
    q9990_thresholds[
        "heldout_fold"
    ].astype(
        int
    )
) == {
    1,
    2,
    3,
    4,
}


assert not q9990_thresholds[
    "oracle_labels_used"
].any()


threshold_lookup = {
    int(
        row.heldout_fold
    ):
        float(
            row.threshold
        )

    for row
    in q9990_thresholds.itertuples(
        index=False
    )
}


print(
    "\n========== FROZEN Q9990 THRESHOLDS =========="
)


for fold in [
    1,
    2,
    3,
    4,
]:

    print(
        f"fold {fold}:",
        threshold_lookup[
            fold
        ]
    )


# ============================================================
# PART A
#
# GT-BLIND EXACT ACTION REGENERATION + FREEZE
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART A — GT-BLIND Q9990 ACTION FREEZE"
)

print(
    "============================================================"
)


g3h3b_cardinality = pd.read_pickle(
    G3H3B_CARDINALITY_PATH
)


expected_dataset_actions = {
    row.dataset:
        int(
            row.resolved_actions
        )

    for row
    in g3h3b_cardinality[
        g3h3b_cardinality[
            "policy_id"
        ].eq(
            "Q9990"
        )
    ].itertuples(
        index=False
    )
}


assert len(
    expected_dataset_actions
) == 103


assert sum(
    expected_dataset_actions.values()
) == EXPECTED_ACTIONS


action_frames = []


for i, dataset in enumerate(
    names,
    start=1,
):

    fold = int(
        fold_map_eval[
            dataset
        ]
    )


    # --------------------------------------------------------
    # Exact frozen shortlist
    # --------------------------------------------------------

    (
        proposals,
        _raw_count,
        _generator_count,
    ) = generate_dataset_proposals(
        dataset
    )


    proposals = add_endpoint_ranks(
        proposals
    )


    proposals = add_shortlist_rank_fields(
        proposals
    )


    shortlist = (
        proposals[
            frozen_shortlist_mask(
                proposals
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    shortlist = attach_scoring_features(
        shortlist
    )


    # --------------------------------------------------------
    # Frozen held-out-fold scorer
    # --------------------------------------------------------

    shortlist[
        "model_score"
    ] = score_shortlist_g3h3(
        shortlist,
        lofo_models[
            fold
        ],
    )


    # --------------------------------------------------------
    # Endpoint ranks on COMPLETE shortlist
    # --------------------------------------------------------

    ranks = endpoint_score_ranks_g3h3(
        shortlist
    )


    shortlist[
        "source_score_rank"
    ] = ranks[
        "source_score_rank"
    ].to_numpy(
        copy=True
    )


    shortlist[
        "target_score_rank"
    ] = ranks[
        "target_score_rank"
    ].to_numpy(
        copy=True
    )


    # --------------------------------------------------------
    # Frozen ASYM1x5 rank gate
    # --------------------------------------------------------

    backbone_mask = policy_mask_g3h3(
        shortlist,
        a=1,
        b=5,
    )


    gated = shortlist[
        backbone_mask
        &
        shortlist[
            "model_score"
        ].ge(
            threshold_lookup[
                fold
            ]
        )
    ].copy()


    # --------------------------------------------------------
    # Frozen deterministic node-disjoint resolver
    # --------------------------------------------------------

    resolved = (
        resolve_conflicts_g3h3(
            gated
        )
        .copy()
        .reset_index(
            drop=True
        )
    )


    expected_n = expected_dataset_actions[
        dataset
    ]


    assert len(
        resolved
    ) == expected_n, (
        dataset,
        len(resolved),
        expected_n,
    )


    keep_cols = [
        "source_id",
        "target_id",

        "gap_size",
        "delta_t",

        "cuts",

        "source_occupied",
        "target_occupied",

        "GMEAN_RATIO",

        "source_cv_pf",
        "target_cv_pf",

        "source_incumbent_cv_pf",
        "target_incumbent_cv_pf",

        "source_relative_improvement",
        "target_relative_improvement",

        "source_score_rank",
        "target_score_rank",

        "model_score",
    ]


    action = resolved[
        keep_cols
    ].copy()


    action.insert(
        0,
        "dataset",
        dataset,
    )


    action.insert(
        1,
        "fold",
        fold,
    )


    action[
        "score_threshold"
    ] = float(
        threshold_lookup[
            fold
        ]
    )


    action[
        "policy_id"
    ] = POLICY_ID


    action_frames.append(
        action
    )


    if (
        i == 1
        or
        i % 10 == 0
        or
        i == 103
    ):

        print(
            f"{i:3d}/103"
            f" | fold={fold}"
            f" | {dataset}"
            f" | actions={len(action)}"
        )


    del proposals
    del shortlist

    del ranks
    del gated
    del resolved
    del action

    gc.collect()


actions = pd.concat(
    action_frames,
    ignore_index=True,
)


del action_frames

gc.collect()


assert len(
    actions
) == EXPECTED_ACTIONS


assert not actions[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


# ------------------------------------------------------------
# Node-disjoint fidelity per dataset
# ------------------------------------------------------------

for dataset, sub in actions.groupby(
    "dataset",
    sort=False,
):

    endpoints = pd.concat(
        [
            sub[
                "source_id"
            ].astype(
                int
            ),

            sub[
                "target_id"
            ].astype(
                int
            ),
        ],
        ignore_index=True,
    )


    assert not endpoints.duplicated().any(), (
        dataset,
        "resolved actions are not node-disjoint"
    )


actions.to_pickle(
    ACTIONS_OUT
)


assert ACTIONS_OUT.exists()


# ============================================================
# 8. ACTION ARTIFACT SHA
# ============================================================

def file_sha256(
    path,
):

    hasher = hashlib.sha256()


    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024
                *
                1024
            )


            if not chunk:

                break


            hasher.update(
                chunk
            )


    return hasher.hexdigest()


ACTIONS_SHA = file_sha256(
    ACTIONS_OUT
)


action_by_fold = (
    actions.groupby(
        "fold"
    )
    .size()
    .astype(
        int
    )
    .to_dict()
)


ACTION_SUMMARY = {
    "stage":
        "12.11G3H3C",

    "part":
        "GT_BLIND_ACTION_FREEZE",

    "policy_id":
        POLICY_ID,

    "policy_candidate_sha256":
        POLICY_CANDIDATE_SHA,

    "actions_sha256":
        ACTIONS_SHA,

    "actions":
        int(
            len(
                actions
            )
        ),

    "actions_by_fold": {
        str(k):
            int(v)

        for k, v
        in action_by_fold.items()
    },

    "actions_node_disjoint_per_dataset":
        True,

    "runtime_action_construction_gt_blind":
        True,

    "official_gt_graph_loaded":
        False,

    "fold0_used":
        False,
}


ACTION_SUMMARY_OUT.write_text(
    json.dumps(
        ACTION_SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


assert ACTION_SUMMARY_OUT.exists()


print(
    "\n========== Q9990 ACTION FREEZE =========="
)

print(
    "actions:",
    len(
        actions
    ),
    "/",
    EXPECTED_ACTIONS
)

print(
    "actions SHA:",
    ACTIONS_SHA
)

print(
    "by fold:",
    action_by_fold
)

print(
    "node-disjoint per dataset:",
    "YES"
)

print(
    "GT read during action construction:",
    "NO"
)

print(
    "Q9990_ACTION_FREEZE: PASS"
)


# ============================================================
# PART B
#
# OFFICIAL DEVELOPMENT GRAPH EVALUATION
#
# GT IS FIRST LOADED BELOW.
# ============================================================

print(
    "\n============================================================"
)

print(
    "PART B — OFFICIAL GRAPH-LEVEL DEVELOPMENT EVALUATION"
)

print(
    "============================================================"
)


# ============================================================
# 9. RECOVER estimated_nodes FROM AUTHORITATIVE G1 ROWS
#
# fv9.per_sample_metrics defines:
#
#   total_node_ratio =
#       (N_pred - N_total) / N_total
#
# Therefore:
#
#   N_total =
#       N_pred / (1 + total_node_ratio)
#
# This is recovered from the already-authoritative G1
# CURRENT_BEST metric rows.
#
# The untouched CURRENT_BEST reproduction gate below must
# recover the exact authoritative score before the mutated
# score is trusted.
# ============================================================

old_dataset = pd.read_pickle(
    CURRENT_BEST_DATASET_PATH
)


assert len(
    old_dataset
) == 103


assert old_dataset[
    "dataset"
].nunique() == 103


required_old_cols = {
    "dataset",
    "num_pred_nodes",
    "total_node_ratio",
}


assert required_old_cols.issubset(
    old_dataset.columns
), (
    old_dataset.columns.tolist()
)


estimated_nodes_eval = {}


for row in old_dataset.itertuples(
    index=False
):

    n_pred = float(
        row.num_pred_nodes
    )


    ratio = float(
        row.total_node_ratio
    )


    assert np.isfinite(
        n_pred
    )


    assert np.isfinite(
        ratio
    )


    denom = (
        1.0
        +
        ratio
    )


    assert denom > 0


    n_total = (
        n_pred
        /
        denom
    )


    estimated_nodes_eval[
        row.dataset
    ] = float(
        n_total
    )


assert set(
    estimated_nodes_eval
) == set(
    names
)


print(
    "\n========== ESTIMATED NODE CONTRACT =========="
)

print(
    "source:",
    CURRENT_BEST_DATASET_PATH.name
)

print(
    "datasets:",
    len(
        estimated_nodes_eval
    )
)

print(
    "recovery formula:",
    "N_pred / (1 + total_node_ratio)"
)

print(
    "authoritative baseline reproduction required:",
    "YES"
)


# ============================================================
# 10. GRAPH MUTATION HELPER
# ============================================================

def apply_q9990_rewires(
    graph,
    dataset_actions,
):
    """
    Mutate one CURRENT_BEST graph copy using frozen Q9990
    node-disjoint actions.

    Returns:
        graph
        mutation_stats
        mutation_audit
    """

    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = graph_state(
        graph
    )


    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "edge_id"
    ] = edges[
        "edge_id"
    ].astype(
        int
    )


    edges[
        "source_id"
    ] = edges[
        "source_id"
    ].astype(
        int
    )


    edges[
        "target_id"
    ] = edges[
        "target_id"
    ].astype(
        int
    )


    # --------------------------------------------------------
    # Exact one-step incumbent maps
    # --------------------------------------------------------

    outgoing = (
        edges.groupby(
            "source_id"
        )[
            [
                "edge_id",
                "target_id",
            ]
        ]
        .apply(
            lambda x:
                [
                    (
                        int(r.edge_id),
                        int(r.target_id),
                    )
                    for r
                    in x.itertuples(
                        index=False
                    )
                ]
        )
        .to_dict()
    )


    incoming = (
        edges.groupby(
            "target_id"
        )[
            [
                "edge_id",
                "source_id",
            ]
        ]
        .apply(
            lambda x:
                [
                    (
                        int(r.edge_id),
                        int(r.source_id),
                    )
                    for r
                    in x.itertuples(
                        index=False
                    )
                ]
        )
        .to_dict()
    )


    remove_edge_ids = set()

    audit_rows = []


    # --------------------------------------------------------
    # Determine ALL cuts from immutable pre-rewire graph.
    #
    # Do not mutate sequentially while deciding cuts.
    # --------------------------------------------------------

    for action in dataset_actions.itertuples(
        index=False
    ):

        src = int(
            action.source_id
        )


        tgt = int(
            action.target_id
        )


        assert src != tgt


        assert (
            src,
            tgt,
        ) not in edge_set


        src_outdeg = int(
            outdeg.get(
                src,
                0,
            )
        )


        tgt_indeg = int(
            indeg.get(
                tgt,
                0,
            )
        )


        assert src_outdeg <= 1, (
            src,
            src_outdeg,
        )


        assert tgt_indeg <= 1, (
            tgt,
            tgt_indeg,
        )


        source_cut_edge_id = None

        source_cut_target = None

        target_cut_edge_id = None

        target_cut_source = None


        if src_outdeg == 1:

            src_edges = outgoing.get(
                src,
                [],
            )


            assert len(
                src_edges
            ) == 1


            (
                source_cut_edge_id,
                source_cut_target,
            ) = src_edges[
                0
            ]


            remove_edge_ids.add(
                int(
                    source_cut_edge_id
                )
            )


        if tgt_indeg == 1:

            tgt_edges = incoming.get(
                tgt,
                [],
            )


            assert len(
                tgt_edges
            ) == 1


            (
                target_cut_edge_id,
                target_cut_source,
            ) = tgt_edges[
                0
            ]


            remove_edge_ids.add(
                int(
                    target_cut_edge_id
                )
            )


        # ----------------------------------------------------
        # Candidate feature topology must agree with graph.
        # ----------------------------------------------------

        assert int(
            action.source_occupied
        ) == int(
            src_outdeg == 1
        ), (
            src,
            action.source_occupied,
            src_outdeg,
        )


        assert int(
            action.target_occupied
        ) == int(
            tgt_indeg == 1
        ), (
            tgt,
            action.target_occupied,
            tgt_indeg,
        )


        expected_cuts = (
            int(
                src_outdeg == 1
            )
            +
            int(
                tgt_indeg == 1
            )
        )


        assert int(
            action.cuts
        ) == expected_cuts


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        audit_rows.append(
            {
                "source_id":
                    src,

                "target_id":
                    tgt,

                "source_pre_outdegree":
                    src_outdeg,

                "target_pre_indegree":
                    tgt_indeg,

                "source_cut_edge_id":
                    source_cut_edge_id,

                "source_cut_target_id":
                    source_cut_target,

                "target_cut_edge_id":
                    target_cut_edge_id,

                "target_cut_source_id":
                    target_cut_source,

                "expected_cuts":
                    expected_cuts,

                "rewire_edge_dist":
                    float(
                        dist
                    ),

                "model_score":
                    float(
                        action.model_score
                    ),
            }
        )


    # --------------------------------------------------------
    # Remove all incumbent edges by exact edge IDs.
    #
    # Dedup is required because two node-disjoint actions may
    # still refer to the same incumbent edge from opposite
    # endpoint roles.
    # --------------------------------------------------------

    remove_edge_ids = sorted(
        remove_edge_ids
    )


    pre_edges = int(
        graph.num_edges()
    )


    if remove_edge_ids:

        graph.bulk_remove_edges(
            remove_edge_ids
        )


    after_remove_edges = int(
        graph.num_edges()
    )


    assert (
        pre_edges
        -
        after_remove_edges
    ) == len(
        remove_edge_ids
    )


    # --------------------------------------------------------
    # Add all new rewire edges.
    #
    # edge_prob = 1.0 is an ephemeral schema placeholder.
    # model score is kept separately in formal action artifact.
    # --------------------------------------------------------

    for audit in audit_rows:

        src = int(
            audit[
                "source_id"
            ]
        )


        tgt = int(
            audit[
                "target_id"
            ]
        )


        assert not graph.has_edge(
            src,
            tgt,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(
                        audit[
                            "rewire_edge_dist"
                        ]
                    ),

                "edge_prob":
                    1.0,
            },
            validate_keys=True,
        )


    post_edges = int(
        graph.num_edges()
    )


    assert (
        post_edges
        ==
        after_remove_edges
        +
        len(
            audit_rows
        )
    )


    # --------------------------------------------------------
    # Exact topology after mutation
    # --------------------------------------------------------

    (
        _nodes_post,
        outdeg_post,
        indeg_post,
        edge_set_post,
    ) = graph_state(
        graph
    )


    for audit in audit_rows:

        src = int(
            audit[
                "source_id"
            ]
        )


        tgt = int(
            audit[
                "target_id"
            ]
        )


        assert (
            src,
            tgt,
        ) in edge_set_post


        # Rewire action should leave source with exactly one
        # continuation and target with exactly one predecessor.
        assert outdeg_post.get(
            src,
            0,
        ) == 1


        assert indeg_post.get(
            tgt,
            0,
        ) == 1


    mutation_stats = {
        "actions":
            int(
                len(
                    audit_rows
                )
            ),

        "unique_incumbent_edges_removed":
            int(
                len(
                    remove_edge_ids
                )
            ),

        "new_edges_added":
            int(
                len(
                    audit_rows
                )
            ),

        "pre_edges":
            pre_edges,

        "post_edges":
            post_edges,

        "net_edge_delta":
            int(
                post_edges
                -
                pre_edges
            ),
    }


    mutation_audit = pd.DataFrame(
        audit_rows
    )


    return (
        graph,
        mutation_stats,
        mutation_audit,
    )


# ============================================================
# 11. OFFICIAL EVALUATION
# ============================================================

baseline_records = []

rewire_records = []

metric_rows = []

mutation_rows = []


for i, dataset in enumerate(
    names,
    start=1,
):

    fold = int(
        fold_map_eval[
            dataset
        ]
    )


    # --------------------------------------------------------
    # Build exact CURRENT_BEST once.
    # --------------------------------------------------------

    base_graph, scale = build_current_best(
        dataset
    )


    # Copy BEFORE official evaluation, because evaluate_graph
    # writes matching attrs into prediction graph in-place.
    rewire_graph = base_graph.copy()


    assert rewire_graph.num_nodes() == base_graph.num_nodes()

    assert rewire_graph.num_edges() == base_graph.num_edges()


    # --------------------------------------------------------
    # Frozen actions
    # --------------------------------------------------------

    dataset_actions = actions[
        actions[
            "dataset"
        ].eq(
            dataset
        )
    ].copy()


    assert len(
        dataset_actions
    ) == expected_dataset_actions[
        dataset
    ]


    # --------------------------------------------------------
    # Mutate only copied graph.
    # --------------------------------------------------------

    (
        rewire_graph,
        mutation_stats,
        mutation_audit,
    ) = apply_q9990_rewires(
        rewire_graph,
        dataset_actions,
    )


    # --------------------------------------------------------
    # GT FIRST USED HERE.
    # --------------------------------------------------------

    gt = fv9.load_graph(
        TRAIN_ROOT
        / f"{dataset}.geff"
    )


    n_total = float(
        estimated_nodes_eval[
            dataset
        ]
    )


    # --------------------------------------------------------
    # Untouched authoritative CURRENT_BEST.
    # --------------------------------------------------------

    baseline_metric = fv9.evaluate_graph(
        base_graph,
        gt,
        scale,
        n_total,
        fv9.FROZEN_V9_CONFIG,
    )


    baseline_records.append(
        baseline_metric
    )


    # --------------------------------------------------------
    # Q9990 rewire graph.
    # --------------------------------------------------------

    rewire_metric = fv9.evaluate_graph(
        rewire_graph,
        gt,
        scale,
        n_total,
        fv9.FROZEN_V9_CONFIG,
    )


    rewire_records.append(
        rewire_metric
    )


    # --------------------------------------------------------
    # Formal dataset metric row
    # --------------------------------------------------------

    metric_rows.append(
        {
            "dataset":
                dataset,

            "fold":
                fold,

            "actions":
                int(
                    mutation_stats[
                        "actions"
                    ]
                ),

            "unique_edges_removed":
                int(
                    mutation_stats[
                        "unique_incumbent_edges_removed"
                    ]
                ),

            "edges_added":
                int(
                    mutation_stats[
                        "new_edges_added"
                    ]
                ),

            "net_edge_delta":
                int(
                    mutation_stats[
                        "net_edge_delta"
                    ]
                ),

            # --------------------------------------------
            # Baseline
            # --------------------------------------------

            "base_edge_tp":
                int(
                    baseline_metric[
                        "edge_tp"
                    ]
                ),

            "base_edge_fp":
                int(
                    baseline_metric[
                        "edge_fp"
                    ]
                ),

            "base_edge_fn":
                int(
                    baseline_metric[
                        "edge_fn"
                    ]
                ),

            "base_division_tp":
                int(
                    baseline_metric[
                        "division_tp"
                    ]
                ),

            "base_division_fp":
                int(
                    baseline_metric[
                        "division_fp"
                    ]
                ),

            "base_division_fn":
                int(
                    baseline_metric[
                        "division_fn"
                    ]
                ),

            "base_node_recall":
                float(
                    baseline_metric[
                        "node_recall"
                    ]
                ),

            "base_edge_jaccard":
                float(
                    baseline_metric[
                        "edge_jaccard"
                    ]
                ),

            "base_adj_edge_jaccard":
                float(
                    baseline_metric[
                        "adj_edge_jaccard"
                    ]
                ),

            # --------------------------------------------
            # Rewire
            # --------------------------------------------

            "rewire_edge_tp":
                int(
                    rewire_metric[
                        "edge_tp"
                    ]
                ),

            "rewire_edge_fp":
                int(
                    rewire_metric[
                        "edge_fp"
                    ]
                ),

            "rewire_edge_fn":
                int(
                    rewire_metric[
                        "edge_fn"
                    ]
                ),

            "rewire_division_tp":
                int(
                    rewire_metric[
                        "division_tp"
                    ]
                ),

            "rewire_division_fp":
                int(
                    rewire_metric[
                        "division_fp"
                    ]
                ),

            "rewire_division_fn":
                int(
                    rewire_metric[
                        "division_fn"
                    ]
                ),

            "rewire_node_recall":
                float(
                    rewire_metric[
                        "node_recall"
                    ]
                ),

            "rewire_edge_jaccard":
                float(
                    rewire_metric[
                        "edge_jaccard"
                    ]
                ),

            "rewire_adj_edge_jaccard":
                float(
                    rewire_metric[
                        "adj_edge_jaccard"
                    ]
                ),
        }
    )


    mutation_rows.append(
        {
            "dataset":
                dataset,

            "fold":
                fold,

            **mutation_stats,
        }
    )


    if (
        i == 1
        or
        i % 10 == 0
        or
        i == 103
    ):

        print(
            f"{i:3d}/103"
            f" | fold={fold}"
            f" | {dataset}"
            f" | actions={mutation_stats['actions']}"
            f" | removed={mutation_stats['unique_incumbent_edges_removed']}"
            f" | net_edges={mutation_stats['net_edge_delta']}"
        )


    del base_graph
    del rewire_graph
    del gt

    del dataset_actions
    del mutation_audit

    gc.collect()


# ============================================================
# 12. BUILD FORMAL METRIC TABLES
# ============================================================

metric_df = pd.DataFrame(
    metric_rows
)


mutation_df = pd.DataFrame(
    mutation_rows
)


assert len(
    metric_df
) == 103


assert len(
    mutation_df
) == 103


assert int(
    metric_df[
        "actions"
    ].sum()
) == EXPECTED_ACTIONS


# ============================================================
# 13. AUTHORITATIVE CURRENT_BEST REPRODUCTION
# ============================================================

baseline_summary = fv9.summarise(
    baseline_records
)


base_tp = int(
    metric_df[
        "base_edge_tp"
    ].sum()
)


base_fp = int(
    metric_df[
        "base_edge_fp"
    ].sum()
)


base_fn = int(
    metric_df[
        "base_edge_fn"
    ].sum()
)


print(
    "\n========== AUTHORITATIVE CURRENT_BEST REPRODUCTION =========="
)


print(
    "edge TP/FP/FN:",
    base_tp,
    "/",
    base_fp,
    "/",
    base_fn
)


print(
    "edge J:",
    f"{baseline_summary['edge_jaccard']:.9f}"
)


print(
    "adj edge J:",
    f"{baseline_summary['adj_edge_jaccard']:.9f}"
)


print(
    "division J:",
    f"{baseline_summary['division_jaccard']:.9f}"
)


print(
    "node recall:",
    f"{baseline_summary['node_recall']:.9f}"
)


print(
    "score:",
    f"{baseline_summary['score']:.12f}"
)


print(
    "expected:",
    f"{EXPECTED_BASE_SCORE:.12f}"
)


assert base_tp == EXPECTED_BASE_TP

assert base_fp == EXPECTED_BASE_FP

assert base_fn == EXPECTED_BASE_FN


assert abs(
    float(
        baseline_summary[
            "edge_jaccard"
        ]
    )
    -
    EXPECTED_BASE_EDGE_J
) <= 5e-7


assert abs(
    float(
        baseline_summary[
            "adj_edge_jaccard"
        ]
    )
    -
    EXPECTED_BASE_ADJ_EDGE_J
) <= 5e-7


assert abs(
    float(
        baseline_summary[
            "division_jaccard"
        ]
    )
    -
    EXPECTED_BASE_DIV_J
) <= 5e-7


assert abs(
    float(
        baseline_summary[
            "node_recall"
        ]
    )
    -
    EXPECTED_BASE_NODE_RECALL
) <= 5e-7


assert abs(
    float(
        baseline_summary[
            "score"
        ]
    )
    -
    EXPECTED_BASE_SCORE
) <= 5e-7


print(
    "CURRENT_BEST_OFFICIAL_REPRODUCTION: PASS"
)


# ============================================================
# 14. Q9990 OFFICIAL AGGREGATE
# ============================================================

rewire_summary = fv9.summarise(
    rewire_records
)


rewire_tp = int(
    metric_df[
        "rewire_edge_tp"
    ].sum()
)


rewire_fp = int(
    metric_df[
        "rewire_edge_fp"
    ].sum()
)


rewire_fn = int(
    metric_df[
        "rewire_edge_fn"
    ].sum()
)


print(
    "\n========== Q9990 GRAPH-LEVEL DEVELOPMENT RESULT =========="
)


print(
    "actions:",
    EXPECTED_ACTIONS
)


print(
    "unique incumbent edges removed:",
    int(
        mutation_df[
            "unique_incumbent_edges_removed"
        ].sum()
    )
)


print(
    "new edges added:",
    int(
        mutation_df[
            "new_edges_added"
        ].sum()
    )
)


print(
    "net edge delta:",
    int(
        mutation_df[
            "net_edge_delta"
        ].sum()
    )
)


print(
    "\nedge TP/FP/FN:"
)


print(
    "CURRENT_BEST:",
    base_tp,
    "/",
    base_fp,
    "/",
    base_fn
)


print(
    "Q9990      :",
    rewire_tp,
    "/",
    rewire_fp,
    "/",
    rewire_fn
)


print(
    "delta       :",
    f"{rewire_tp-base_tp:+d}",
    "/",
    f"{rewire_fp-base_fp:+d}",
    "/",
    f"{rewire_fn-base_fn:+d}",
)


print(
    "\nmetrics:"
)


print(
    "edge J      :",
    f"{baseline_summary['edge_jaccard']:.9f}",
    "->",
    f"{rewire_summary['edge_jaccard']:.9f}",
    f"({rewire_summary['edge_jaccard']-baseline_summary['edge_jaccard']:+.9f})",
)


print(
    "adj edge J  :",
    f"{baseline_summary['adj_edge_jaccard']:.9f}",
    "->",
    f"{rewire_summary['adj_edge_jaccard']:.9f}",
    f"({rewire_summary['adj_edge_jaccard']-baseline_summary['adj_edge_jaccard']:+.9f})",
)


print(
    "division J  :",
    f"{baseline_summary['division_jaccard']:.9f}",
    "->",
    f"{rewire_summary['division_jaccard']:.9f}",
    f"({rewire_summary['division_jaccard']-baseline_summary['division_jaccard']:+.9f})",
)


print(
    "node recall :",
    f"{baseline_summary['node_recall']:.9f}",
    "->",
    f"{rewire_summary['node_recall']:.9f}",
    f"({rewire_summary['node_recall']-baseline_summary['node_recall']:+.9f})",
)


print(
    "SCORE       :",
    f"{baseline_summary['score']:.12f}",
    "->",
    f"{rewire_summary['score']:.12f}",
    f"({rewire_summary['score']-baseline_summary['score']:+.12f})",
)


# ============================================================
# 15. BY-FOLD OFFICIAL SUMMARY
# ============================================================

fold_rows = []


for fold in [
    1,
    2,
    3,
    4,
]:

    fold_mask = (
        corp[
            "fold"
        ].eq(
            fold
        )
        .to_numpy()
    )


    fold_base_records = [
        record
        for record, keep
        in zip(
            baseline_records,
            fold_mask,
        )
        if keep
    ]


    fold_rewire_records = [
        record
        for record, keep
        in zip(
            rewire_records,
            fold_mask,
        )
        if keep
    ]


    base_fold_summary = fv9.summarise(
        fold_base_records
    )


    rewire_fold_summary = fv9.summarise(
        fold_rewire_records
    )


    fold_metric_rows = metric_df[
        metric_df[
            "fold"
        ].eq(
            fold
        )
    ]


    fold_rows.append(
        {
            "fold":
                fold,

            "datasets":
                int(
                    len(
                        fold_metric_rows
                    )
                ),

            "actions":
                int(
                    fold_metric_rows[
                        "actions"
                    ].sum()
                ),

            "base_tp":
                int(
                    fold_metric_rows[
                        "base_edge_tp"
                    ].sum()
                ),

            "rewire_tp":
                int(
                    fold_metric_rows[
                        "rewire_edge_tp"
                    ].sum()
                ),

            "delta_tp":
                int(
                    fold_metric_rows[
                        "rewire_edge_tp"
                    ].sum()
                    -
                    fold_metric_rows[
                        "base_edge_tp"
                    ].sum()
                ),

            "base_fp":
                int(
                    fold_metric_rows[
                        "base_edge_fp"
                    ].sum()
                ),

            "rewire_fp":
                int(
                    fold_metric_rows[
                        "rewire_edge_fp"
                    ].sum()
                ),

            "delta_fp":
                int(
                    fold_metric_rows[
                        "rewire_edge_fp"
                    ].sum()
                    -
                    fold_metric_rows[
                        "base_edge_fp"
                    ].sum()
                ),

            "base_fn":
                int(
                    fold_metric_rows[
                        "base_edge_fn"
                    ].sum()
                ),

            "rewire_fn":
                int(
                    fold_metric_rows[
                        "rewire_edge_fn"
                    ].sum()
                ),

            "delta_fn":
                int(
                    fold_metric_rows[
                        "rewire_edge_fn"
                    ].sum()
                    -
                    fold_metric_rows[
                        "base_edge_fn"
                    ].sum()
                ),

            "base_score":
                float(
                    base_fold_summary[
                        "score"
                    ]
                ),

            "rewire_score":
                float(
                    rewire_fold_summary[
                        "score"
                    ]
                ),

            "delta_score":
                float(
                    rewire_fold_summary[
                        "score"
                    ]
                    -
                    base_fold_summary[
                        "score"
                    ]
                ),

            "base_edge_j":
                float(
                    base_fold_summary[
                        "edge_jaccard"
                    ]
                ),

            "rewire_edge_j":
                float(
                    rewire_fold_summary[
                        "edge_jaccard"
                    ]
                ),

            "delta_edge_j":
                float(
                    rewire_fold_summary[
                        "edge_jaccard"
                    ]
                    -
                    base_fold_summary[
                        "edge_jaccard"
                    ]
                ),

            "base_adj_edge_j":
                float(
                    base_fold_summary[
                        "adj_edge_jaccard"
                    ]
                ),

            "rewire_adj_edge_j":
                float(
                    rewire_fold_summary[
                        "adj_edge_jaccard"
                    ]
                ),

            "delta_adj_edge_j":
                float(
                    rewire_fold_summary[
                        "adj_edge_jaccard"
                    ]
                    -
                    base_fold_summary[
                        "adj_edge_jaccard"
                    ]
                ),

            "base_division_j":
                float(
                    base_fold_summary[
                        "division_jaccard"
                    ]
                ),

            "rewire_division_j":
                float(
                    rewire_fold_summary[
                        "division_jaccard"
                    ]
                ),
        }
    )


fold_summary = pd.DataFrame(
    fold_rows
)


print(
    "\n========== Q9990 BY DEVELOPMENT FOLD =========="
)


print(
    fold_summary.to_string(
        index=False
    )
)


# ============================================================
# 16. DATASET-LEVEL WIN / LOSS
# ============================================================

metric_df[
    "delta_edge_tp"
] = (
    metric_df[
        "rewire_edge_tp"
    ]
    -
    metric_df[
        "base_edge_tp"
    ]
)


metric_df[
    "delta_edge_fp"
] = (
    metric_df[
        "rewire_edge_fp"
    ]
    -
    metric_df[
        "base_edge_fp"
    ]
)


metric_df[
    "delta_edge_fn"
] = (
    metric_df[
        "rewire_edge_fn"
    ]
    -
    metric_df[
        "base_edge_fn"
    ]
)


metric_df[
    "delta_edge_jaccard"
] = (
    metric_df[
        "rewire_edge_jaccard"
    ]
    -
    metric_df[
        "base_edge_jaccard"
    ]
)


metric_df[
    "delta_adj_edge_jaccard"
] = (
    metric_df[
        "rewire_adj_edge_jaccard"
    ]
    -
    metric_df[
        "base_adj_edge_jaccard"
    ]
)


dataset_edgej_wins = int(
    metric_df[
        "delta_edge_jaccard"
    ].gt(
        0
    ).sum()
)


dataset_edgej_losses = int(
    metric_df[
        "delta_edge_jaccard"
    ].lt(
        0
    ).sum()
)


dataset_edgej_ties = int(
    metric_df[
        "delta_edge_jaccard"
    ].eq(
        0
    ).sum()
)


print(
    "\n========== DATASET-LEVEL EDGE-J OUTCOME =========="
)


print(
    "wins:",
    dataset_edgej_wins
)


print(
    "losses:",
    dataset_edgej_losses
)


print(
    "ties:",
    dataset_edgej_ties
)


# ============================================================
# 17. FORMAL DECISION
#
# Do NOT automatically promote from a tiny positive delta.
#
# Development promotion requires:
#
#   - aggregate score > CURRENT_BEST
#   - no held-out fold score decline
#
# Otherwise reject as final policy.
# ============================================================

aggregate_delta = float(
    rewire_summary[
        "score"
    ]
    -
    baseline_summary[
        "score"
    ]
)


fold_nonnegative = bool(
    fold_summary[
        "delta_score"
    ].ge(
        0
    ).all()
)


aggregate_improved = bool(
    aggregate_delta > 0
)


if (
    aggregate_improved
    and
    fold_nonnegative
):

    decision = (
        "PROMOTE_Q9990_TO_FROZEN_DEV_POLICY_CANDIDATE"
    )


else:

    decision = (
        "REJECT_Q9990_AS_FINAL_REWIRE_POLICY"
    )


# ============================================================
# 18. PERSIST FORMAL RESULTS
# ============================================================

metric_df.to_pickle(
    METRICS_OUT
)


assert METRICS_OUT.exists()


SUMMARY = {
    "stage":
        "12.11G3H3C",

    "status":
        "Q9990_GRAPH_LEVEL_DEV_EVALUATION_COMPLETE",

    "policy_id":
        POLICY_ID,

    "policy_candidate_sha256":
        POLICY_CANDIDATE_SHA,

    "actions_sha256":
        ACTIONS_SHA,

    "actions":
        EXPECTED_ACTIONS,

    "baseline": {
        "edge_tp":
            base_tp,

        "edge_fp":
            base_fp,

        "edge_fn":
            base_fn,

        "edge_jaccard":
            float(
                baseline_summary[
                    "edge_jaccard"
                ]
            ),

        "adj_edge_jaccard":
            float(
                baseline_summary[
                    "adj_edge_jaccard"
                ]
            ),

        "division_jaccard":
            float(
                baseline_summary[
                    "division_jaccard"
                ]
            ),

        "node_recall":
            float(
                baseline_summary[
                    "node_recall"
                ]
            ),

        "score":
            float(
                baseline_summary[
                    "score"
                ]
            ),
    },

    "q9990": {
        "edge_tp":
            rewire_tp,

        "edge_fp":
            rewire_fp,

        "edge_fn":
            rewire_fn,

        "edge_jaccard":
            float(
                rewire_summary[
                    "edge_jaccard"
                ]
            ),

        "adj_edge_jaccard":
            float(
                rewire_summary[
                    "adj_edge_jaccard"
                ]
            ),

        "division_jaccard":
            float(
                rewire_summary[
                    "division_jaccard"
                ]
            ),

        "node_recall":
            float(
                rewire_summary[
                    "node_recall"
                ]
            ),

        "score":
            float(
                rewire_summary[
                    "score"
                ]
            ),
    },

    "delta": {
        "edge_tp":
            int(
                rewire_tp
                -
                base_tp
            ),

        "edge_fp":
            int(
                rewire_fp
                -
                base_fp
            ),

        "edge_fn":
            int(
                rewire_fn
                -
                base_fn
            ),

        "edge_jaccard":
            float(
                rewire_summary[
                    "edge_jaccard"
                ]
                -
                baseline_summary[
                    "edge_jaccard"
                ]
            ),

        "adj_edge_jaccard":
            float(
                rewire_summary[
                    "adj_edge_jaccard"
                ]
                -
                baseline_summary[
                    "adj_edge_jaccard"
                ]
            ),

        "division_jaccard":
            float(
                rewire_summary[
                    "division_jaccard"
                ]
                -
                baseline_summary[
                    "division_jaccard"
                ]
            ),

        "score":
            aggregate_delta,
    },

    "by_fold":
        fold_summary.to_dict(
            "records"
        ),

    "dataset_edgej": {
        "wins":
            dataset_edgej_wins,

        "losses":
            dataset_edgej_losses,

        "ties":
            dataset_edgej_ties,
    },

    "baseline_reproduction_pass":
        True,

    "aggregate_improved":
        aggregate_improved,

    "all_folds_nonnegative":
        fold_nonnegative,

    "decision":
        decision,

    "selection_is_development_gt_informed":
        True,

    "runtime_action_construction_gt_blind":
        True,

    "scorer_retuned":
        False,

    "shortlist_retuned":
        False,

    "score_threshold_retuned":
        False,

    "fold0_used":
        False,

    "prediction_geff_written":
        False,

    "next": (
        "FREEZE_AND_LOCK_FOLD0_CONFIRMATION"
        if decision
        ==
        "PROMOTE_Q9990_TO_FROZEN_DEV_POLICY_CANDIDATE"
        else
        "DIAGNOSE_GRAPH_LEVEL_FAILURE_WITHOUT_SCORER_RETUNING"
    ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


assert SUMMARY_OUT.exists()


# ============================================================
# 19. RELOAD INTEGRITY
# ============================================================

actions_check = pd.read_pickle(
    ACTIONS_OUT
)


metric_check = pd.read_pickle(
    METRICS_OUT
)


summary_check = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert len(
    actions_check
) == EXPECTED_ACTIONS


assert len(
    metric_check
) == 103


assert summary_check[
    "baseline_reproduction_pass"
] is True


assert summary_check[
    "fold0_used"
] is False


assert summary_check[
    "scorer_retuned"
] is False


assert summary_check[
    "score_threshold_retuned"
] is False


# ============================================================
# 20. FINAL DECISION
# ============================================================

print(
    "\n========== 12.11G3H3C DECISION =========="
)


print(
    "Q9990_GRAPH_LEVEL_DEV_EVALUATION: PASS"
)


print(
    "policy:",
    POLICY_ID
)


print(
    "actions:",
    EXPECTED_ACTIONS
)


print(
    "CURRENT_BEST reproduced:",
    "YES"
)


print(
    "baseline score:",
    f"{baseline_summary['score']:.12f}"
)


print(
    "Q9990 score:",
    f"{rewire_summary['score']:.12f}"
)


print(
    "delta score:",
    f"{aggregate_delta:+.12f}"
)


print(
    "aggregate improved:",
    aggregate_improved
)


print(
    "all four folds nonnegative:",
    fold_nonnegative
)


print(
    "decision:",
    decision
)


print(
    "scorer retuned:",
    "NO"
)


print(
    "shortlist retuned:",
    "NO"
)


print(
    "score threshold retuned:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "prediction GEFF written:",
    "NO"
)


print(
    "next:"
)


print(
    SUMMARY[
        "next"
    ]
)

In [ ]:
# ============================================================
# Stage 12.11G3H3D0
#
# Q9990 Graph-Level Failure Mechanism Audit
#
# ------------------------------------------------------------
# INPUT
# ------------------------------------------------------------
#
# Frozen rejected graph candidate:
#
#   ASYM1x5_Q9990_NODEDISJOINT_REWIRE_V1
#
# Official dev result:
#
#   CURRENT_BEST score:
#       0.791977433623
#
#   Q9990:
#       0.791892564639
#
#   delta:
#      -0.000084868984
#
# Official edge delta:
#
#   TP  -49
#   FP  -50
#   FN  +49
#
# Actions:
#   13,928
#
# Unique incumbent edges removed:
#   2,055
#
# New edges:
#   13,928
#
# Net edge delta:
#   +11,873
#
# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Diagnose WHY strong OOF local ranking did not translate
# into positive graph-level score.
#
# Audit:
#
#   1. cuts = 0 / 1 / 2 composition
#   2. occupancy topology
#   3. claimed cuts vs unique actual edge removals
#   4. exact net-edge-delta decomposition
#   5. known clean-gap positive retention by action class
#   6. FULL<=7 retention by action class
#   7. model-score / endpoint-rank profiles
#   8. dataset-level relationship to official metric deltas
#   9. wins / losses / ties action composition
#
# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------
#
# This stage is DIAGNOSTIC ONLY.
#
# GT may be read because graph-level development evaluation
# has already occurred.
#
# NO:
#   scorer retuning
#   shortlist retuning
#   threshold retuning
#   policy selection
#   graph modification
#   new official evaluation
#   Fold0
#
# Unknown actions are NOT called confirmed negatives.
#
# "known_positive_yield" means:
#
#   known audited clean-gap positives / actions
#
# It is NOT precision.
#
# ------------------------------------------------------------
# Keep after running: YES
# ============================================================


from pathlib import Path

import json

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


ACTIONS_PATH = (
    S12
    / "stage12_folds14_rewire_q9990_graph_candidate_v1_actions.pkl"
)


METRICS_PATH = (
    S12
    / "stage12_folds14_rewire_q9990_graph_candidate_v1_metrics.pkl"
)


G3H3C_SUMMARY_PATH = (
    S12
    / "stage12_folds14_rewire_q9990_graph_candidate_v1_summary.json"
)


POSITIVE_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_positive_features_v1.pkl"
)


# ------------------------------------------------------------
# Formal G3H3D0 outputs
# ------------------------------------------------------------

ACTION_AUDIT_OUT = (
    S12
    / "stage12_folds14_rewire_q9990_failure_action_decomposition_v1.pkl"
)


DATASET_AUDIT_OUT = (
    S12
    / "stage12_folds14_rewire_q9990_failure_dataset_decomposition_v1.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_folds14_rewire_q9990_failure_mechanism_v1_summary.json"
)


for path in [
    ACTIONS_PATH,
    METRICS_PATH,
    G3H3C_SUMMARY_PATH,
    POSITIVE_PATH,
]:

    assert path.exists(), path


for path in [
    ACTION_AUDIT_OUT,
    DATASET_AUDIT_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3H3D0 output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite formal failure-analysis provenance."
        )


# ============================================================
# 1. FROZEN EXPECTATIONS
# ============================================================

EXPECTED_ACTIONS = 13_928

EXPECTED_UNIQUE_REMOVED = 2_055

EXPECTED_NEW_EDGES = 13_928

EXPECTED_NET_EDGE_DELTA = 11_873


EXPECTED_POSITIVES = 202

EXPECTED_FULL7 = 173


EXPECTED_BASE_SCORE = (
    0.7919774336227487
)


EXPECTED_Q9990_SCORE = (
    0.791892564639
)


# ============================================================
# 2. LOAD FORMAL G3H3C RESULT
# ============================================================

g3h3c = json.loads(
    G3H3C_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert g3h3c[
    "status"
] == (
    "Q9990_GRAPH_LEVEL_DEV_EVALUATION_COMPLETE"
)


assert g3h3c[
    "decision"
] == (
    "REJECT_Q9990_AS_FINAL_REWIRE_POLICY"
)


assert g3h3c[
    "aggregate_improved"
] is False


assert g3h3c[
    "all_folds_nonnegative"
] is False


assert g3h3c[
    "fold0_used"
] is False


assert g3h3c[
    "scorer_retuned"
] is False


assert g3h3c[
    "shortlist_retuned"
] is False


assert g3h3c[
    "score_threshold_retuned"
] is False


print(
    "========== G3H3D0 REJECTED POLICY CONTRACT =========="
)


print(
    "policy:",
    g3h3c[
        "policy_id"
    ]
)


print(
    "decision:",
    g3h3c[
        "decision"
    ]
)


print(
    "baseline score:",
    g3h3c[
        "baseline"
    ][
        "score"
    ]
)


print(
    "Q9990 score:",
    g3h3c[
        "q9990"
    ][
        "score"
    ]
)


print(
    "delta score:",
    g3h3c[
        "delta"
    ][
        "score"
    ]
)


print(
    "Fold0 used:",
    "NO"
)


# ============================================================
# 3. LOAD ACTIONS / METRICS / KNOWN POSITIVES
# ============================================================

actions = pd.read_pickle(
    ACTIONS_PATH
)


metrics = pd.read_pickle(
    METRICS_PATH
)


positive = pd.read_pickle(
    POSITIVE_PATH
)


assert len(
    actions
) == EXPECTED_ACTIONS


assert len(
    metrics
) == 103


assert len(
    positive
) == EXPECTED_POSITIVES


assert int(
    positive[
        "pred_full_7um"
    ].sum()
) == EXPECTED_FULL7


assert not actions[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


assert not positive[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


# ============================================================
# 4. ACTION TOPOLOGY FIDELITY
# ============================================================

for col in [
    "cuts",
    "source_occupied",
    "target_occupied",
]:

    actions[
        col
    ] = pd.to_numeric(
        actions[
            col
        ],
        errors="raise",
    ).astype(
        int
    )


assert set(
    actions[
        "cuts"
    ].unique()
).issubset(
    {
        0,
        1,
        2,
    }
)


assert actions[
    "source_occupied"
].isin(
    [
        0,
        1,
    ]
).all()


assert actions[
    "target_occupied"
].isin(
    [
        0,
        1,
    ]
).all()


assert (
    actions[
        "cuts"
    ]
    ==
    (
        actions[
            "source_occupied"
        ]
        +
        actions[
            "target_occupied"
        ]
    )
).all()


def occupancy_label(
    source_occupied,
    target_occupied,
):

    source_occupied = int(
        source_occupied
    )

    target_occupied = int(
        target_occupied
    )


    if (
        source_occupied == 0
        and
        target_occupied == 0
    ):

        return "FREE_FREE"


    if (
        source_occupied == 1
        and
        target_occupied == 0
    ):

        return "SOURCE_OCCUPIED"


    if (
        source_occupied == 0
        and
        target_occupied == 1
    ):

        return "TARGET_OCCUPIED"


    if (
        source_occupied == 1
        and
        target_occupied == 1
    ):

        return "BOTH_OCCUPIED"


    raise AssertionError(
        (
            source_occupied,
            target_occupied,
        )
    )


actions[
    "occupancy_class"
] = [
    occupancy_label(
        s,
        t,
    )

    for s, t
    in zip(
        actions[
            "source_occupied"
        ],
        actions[
            "target_occupied"
        ],
    )
]


# ============================================================
# 5. ATTACH KNOWN DEVELOPMENT POSITIVE AUDIT
#
# IMPORTANT:
# Non-matched actions are UNKNOWN / UNLABELLED here,
# not confirmed negatives.
# ============================================================

positive_meta = (
    positive[
        [
            "dataset",
            "fold",
            "source_id",
            "target_id",
            "pred_full_7um",
            "topology_class",
            "component_id",
        ]
    ]
    .copy()
)


positive_meta[
    "is_known_positive"
] = True


positive_meta[
    "is_known_full7_positive"
] = (
    positive_meta[
        "pred_full_7um"
    ]
    .astype(
        bool
    )
)


action_audit = actions.merge(
    positive_meta[
        [
            "dataset",
            "fold",
            "source_id",
            "target_id",
            "is_known_positive",
            "is_known_full7_positive",
            "topology_class",
            "component_id",
        ]
    ],
    on=[
        "dataset",
        "fold",
        "source_id",
        "target_id",
    ],
    how="left",
    validate="one_to_one",
)


action_audit[
    "is_known_positive"
] = (
    action_audit[
        "is_known_positive"
    ]
    .fillna(
        False
    )
    .astype(
        bool
    )
)


action_audit[
    "is_known_full7_positive"
] = (
    action_audit[
        "is_known_full7_positive"
    ]
    .fillna(
        False
    )
    .astype(
        bool
    )
)


# ============================================================
# 6. CUT-CLASS DECOMPOSITION
# ============================================================

cut_summary = (
    action_audit.groupby(
        "cuts",
        sort=True,
    )
    .agg(
        actions=(
            "dataset",
            "size",
        ),

        datasets=(
            "dataset",
            "nunique",
        ),

        known_positive_hits=(
            "is_known_positive",
            "sum",
        ),

        full7_positive_hits=(
            "is_known_full7_positive",
            "sum",
        ),

        mean_model_score=(
            "model_score",
            "mean",
        ),

        median_model_score=(
            "model_score",
            "median",
        ),

        median_gap=(
            "gap_size",
            "median",
        ),

        median_source_rank=(
            "source_score_rank",
            "median",
        ),

        median_target_rank=(
            "target_score_rank",
            "median",
        ),
    )
    .reset_index()
)


cut_summary[
    "action_fraction"
] = (
    cut_summary[
        "actions"
    ]
    /
    EXPECTED_ACTIONS
)


cut_summary[
    "known_positive_yield"
] = (
    cut_summary[
        "known_positive_hits"
    ]
    /
    cut_summary[
        "actions"
    ]
)


cut_summary[
    "full7_positive_yield"
] = (
    cut_summary[
        "full7_positive_hits"
    ]
    /
    cut_summary[
        "actions"
    ]
)


# ============================================================
# 7. OCCUPANCY DECOMPOSITION
# ============================================================

occupancy_order = [
    "FREE_FREE",
    "SOURCE_OCCUPIED",
    "TARGET_OCCUPIED",
    "BOTH_OCCUPIED",
]


occupancy_summary = (
    action_audit.groupby(
        "occupancy_class",
        sort=False,
    )
    .agg(
        actions=(
            "dataset",
            "size",
        ),

        datasets=(
            "dataset",
            "nunique",
        ),

        known_positive_hits=(
            "is_known_positive",
            "sum",
        ),

        full7_positive_hits=(
            "is_known_full7_positive",
            "sum",
        ),

        mean_model_score=(
            "model_score",
            "mean",
        ),

        median_model_score=(
            "model_score",
            "median",
        ),

        median_gap=(
            "gap_size",
            "median",
        ),
    )
    .reindex(
        occupancy_order
    )
    .fillna(
        0
    )
    .reset_index()
)


occupancy_summary[
    "action_fraction"
] = (
    occupancy_summary[
        "actions"
    ]
    /
    EXPECTED_ACTIONS
)


occupancy_summary[
    "known_positive_yield"
] = np.where(
    occupancy_summary[
        "actions"
    ].gt(
        0
    ),

    occupancy_summary[
        "known_positive_hits"
    ]
    /
    occupancy_summary[
        "actions"
    ],

    np.nan,
)


occupancy_summary[
    "full7_positive_yield"
] = np.where(
    occupancy_summary[
        "actions"
    ].gt(
        0
    ),

    occupancy_summary[
        "full7_positive_hits"
    ]
    /
    occupancy_summary[
        "actions"
    ],

    np.nan,
)


# ============================================================
# 8. EXACT NET-EDGE-DELTA DECOMPOSITION
# ============================================================

n_cut0 = int(
    action_audit[
        "cuts"
    ].eq(
        0
    ).sum()
)


n_cut1 = int(
    action_audit[
        "cuts"
    ].eq(
        1
    ).sum()
)


n_cut2 = int(
    action_audit[
        "cuts"
    ].eq(
        2
    ).sum()
)


assert (
    n_cut0
    +
    n_cut1
    +
    n_cut2
) == EXPECTED_ACTIONS


claimed_cut_count = int(
    action_audit[
        "cuts"
    ].sum()
)


unique_removed = int(
    metrics[
        "unique_edges_removed"
    ].sum()
)


new_edges = int(
    metrics[
        "edges_added"
    ].sum()
)


actual_net_edge_delta = int(
    metrics[
        "net_edge_delta"
    ].sum()
)


assert unique_removed == EXPECTED_UNIQUE_REMOVED

assert new_edges == EXPECTED_NEW_EDGES

assert actual_net_edge_delta == EXPECTED_NET_EDGE_DELTA


# ------------------------------------------------------------
# Without shared-cut dedup:
#
# Each cuts=0 action contributes +1 edge
# Each cuts=1 action contributes  0
# Each cuts=2 action contributes -1
#
# ideal net:
#
#     n0 - n2
#
# Actual removal is unique-edge based.
# If two actions request the same incumbent removal,
# dedup increases net edges by one relative to claimed-cuts
# accounting.
# ------------------------------------------------------------

ideal_net_without_cut_dedup = int(
    n_cut0
    -
    n_cut2
)


shared_cut_dedup = int(
    claimed_cut_count
    -
    unique_removed
)


reconstructed_actual_net = int(
    ideal_net_without_cut_dedup
    +
    shared_cut_dedup
)


assert reconstructed_actual_net == actual_net_edge_delta


# ============================================================
# 9. KNOWN-POSITIVE RETENTION
# ============================================================

known_positive_hits = int(
    action_audit[
        "is_known_positive"
    ].sum()
)


full7_positive_hits = int(
    action_audit[
        "is_known_full7_positive"
    ].sum()
)


assert known_positive_hits == 23

assert full7_positive_hits == 23


# ============================================================
# 10. FOLD x CUT AUDIT
# ============================================================

fold_cut = (
    action_audit.groupby(
        [
            "fold",
            "cuts",
        ],
        sort=True,
    )
    .agg(
        actions=(
            "dataset",
            "size",
        ),

        known_positive_hits=(
            "is_known_positive",
            "sum",
        ),

        full7_positive_hits=(
            "is_known_full7_positive",
            "sum",
        ),

        median_score=(
            "model_score",
            "median",
        ),
    )
    .reset_index()
)


fold_cut[
    "known_positive_yield"
] = (
    fold_cut[
        "known_positive_hits"
    ]
    /
    fold_cut[
        "actions"
    ]
)


# ============================================================
# 11. DATASET ACTION DECOMPOSITION
# ============================================================

dataset_rows = []


for (
    dataset,
    fold
), sub in action_audit.groupby(
    [
        "dataset",
        "fold",
    ],
    sort=False,
):

    c0 = int(
        sub[
            "cuts"
        ].eq(
            0
        ).sum()
    )


    c1 = int(
        sub[
            "cuts"
        ].eq(
            1
        ).sum()
    )


    c2 = int(
        sub[
            "cuts"
        ].eq(
            2
        ).sum()
    )


    dataset_rows.append(
        {
            "dataset":
                dataset,

            "fold":
                int(
                    fold
                ),

            "actions":
                int(
                    len(
                        sub
                    )
                ),

            "cuts0":
                c0,

            "cuts1":
                c1,

            "cuts2":
                c2,

            "cut_claims":
                int(
                    sub[
                        "cuts"
                    ].sum()
                ),

            "cuts0_fraction":
                float(
                    c0
                    /
                    len(
                        sub
                    )
                )
                if len(
                    sub
                )
                else np.nan,

            "cuts_ge1_fraction":
                float(
                    (
                        c1
                        +
                        c2
                    )
                    /
                    len(
                        sub
                    )
                )
                if len(
                    sub
                )
                else np.nan,

            "known_positive_hits":
                int(
                    sub[
                        "is_known_positive"
                    ].sum()
                ),

            "full7_positive_hits":
                int(
                    sub[
                        "is_known_full7_positive"
                    ].sum()
                ),

            "mean_model_score":
                float(
                    sub[
                        "model_score"
                    ].mean()
                ),

            "median_model_score":
                float(
                    sub[
                        "model_score"
                    ].median()
                ),
        }
    )


dataset_audit = pd.DataFrame(
    dataset_rows
)


assert len(
    dataset_audit
) == 103


# ============================================================
# 12. ATTACH OFFICIAL DATASET DELTAS
# ============================================================

required_metric_cols = [
    "dataset",
    "fold",
    "actions",
    "unique_edges_removed",
    "edges_added",
    "net_edge_delta",

    "delta_edge_tp",
    "delta_edge_fp",
    "delta_edge_fn",
    "delta_edge_jaccard",
    "delta_adj_edge_jaccard",
]


for col in required_metric_cols:

    assert col in metrics.columns, (
        col,
        metrics.columns.tolist(),
    )


dataset_audit = dataset_audit.merge(
    metrics[
        required_metric_cols
    ],
    on=[
        "dataset",
        "fold",
    ],
    how="left",
    validate="one_to_one",
    suffixes=(
        "_audit",
        "_metric",
    ),
)


assert dataset_audit[
    "delta_edge_jaccard"
].notna().all()


dataset_audit[
    "edgej_outcome"
] = np.select(
    [
        dataset_audit[
            "delta_edge_jaccard"
        ].gt(
            0
        ),

        dataset_audit[
            "delta_edge_jaccard"
        ].lt(
            0
        ),
    ],
    [
        "WIN",
        "LOSS",
    ],
    default="TIE",
)


# ============================================================
# 13. DATASET WIN / LOSS COMPOSITION
# ============================================================

outcome_summary = (
    dataset_audit.groupby(
        "edgej_outcome",
        sort=False,
    )
    .agg(
        datasets=(
            "dataset",
            "size",
        ),

        total_actions=(
            "actions_audit",
            "sum",
        ),

        mean_actions=(
            "actions_audit",
            "mean",
        ),

        mean_cuts0_fraction=(
            "cuts0_fraction",
            "mean",
        ),

        median_cuts0_fraction=(
            "cuts0_fraction",
            "median",
        ),

        mean_cuts_ge1_fraction=(
            "cuts_ge1_fraction",
            "mean",
        ),

        total_known_positive_hits=(
            "known_positive_hits",
            "sum",
        ),

        total_full7_positive_hits=(
            "full7_positive_hits",
            "sum",
        ),

        total_delta_tp=(
            "delta_edge_tp",
            "sum",
        ),

        total_delta_fp=(
            "delta_edge_fp",
            "sum",
        ),

        total_delta_fn=(
            "delta_edge_fn",
            "sum",
        ),

        mean_delta_edge_j=(
            "delta_edge_jaccard",
            "mean",
        ),
    )
    .reset_index()
)


# ============================================================
# 14. DATASET-LEVEL ASSOCIATION AUDIT
#
# Diagnostic only.
#
# Spearman implemented as Pearson correlation of ranks,
# avoiding any external dependency.
# ============================================================

def rank_corr(
    x,
    y,
):

    x = pd.Series(
        x,
        dtype=float,
    )


    y = pd.Series(
        y,
        dtype=float,
    )


    valid = (
        x.notna()
        &
        y.notna()
    )


    if int(
        valid.sum()
    ) < 3:

        return np.nan


    xr = x[
        valid
    ].rank(
        method="average"
    )


    yr = y[
        valid
    ].rank(
        method="average"
    )


    return float(
        xr.corr(
            yr
        )
    )


association_features = [
    "actions_audit",
    "cuts0",
    "cuts1",
    "cuts2",
    "cuts0_fraction",
    "cuts_ge1_fraction",
    "known_positive_hits",
    "full7_positive_hits",
    "unique_edges_removed",
    "net_edge_delta",
]


association_targets = [
    "delta_edge_tp",
    "delta_edge_fp",
    "delta_edge_fn",
    "delta_edge_jaccard",
    "delta_adj_edge_jaccard",
]


association_rows = []


for feature in association_features:

    for target in association_targets:

        association_rows.append(
            {
                "feature":
                    feature,

                "target":
                    target,

                "spearman_rank_corr":
                    rank_corr(
                        dataset_audit[
                            feature
                        ],
                        dataset_audit[
                            target
                        ],
                    ),
            }
        )


association = pd.DataFrame(
    association_rows
)


# ============================================================
# 15. KEY FAILURE RATIOS
# ============================================================

cut0_fraction = float(
    n_cut0
    /
    EXPECTED_ACTIONS
)


cut_ge1_fraction = float(
    (
        n_cut1
        +
        n_cut2
    )
    /
    EXPECTED_ACTIONS
)


unique_removal_per_action = float(
    unique_removed
    /
    EXPECTED_ACTIONS
)


net_growth_per_action = float(
    actual_net_edge_delta
    /
    EXPECTED_ACTIONS
)


known_positive_yield_all = float(
    known_positive_hits
    /
    EXPECTED_ACTIONS
)


# ============================================================
# 16. DESCRIPTIVE FAILURE CLASSIFICATION
#
# This is NOT policy selection.
# ============================================================

if (
    cut0_fraction > 0.5
    and
    actual_net_edge_delta > 0
):

    mutation_regime = (
        "ADDITION_DOMINATED"
    )


elif (
    cut_ge1_fraction > 0.5
):

    mutation_regime = (
        "REPLACEMENT_DOMINATED"
    )


else:

    mutation_regime = (
        "MIXED"
    )


# ============================================================
# 17. PRINT EXACT NET-EDGE DECOMPOSITION
# ============================================================

print(
    "\n========== EXACT ACTION / CUT DECOMPOSITION =========="
)


print(
    cut_summary.to_string(
        index=False
    )
)


print(
    "\n========== OCCUPANCY DECOMPOSITION =========="
)


print(
    occupancy_summary.to_string(
        index=False
    )
)


print(
    "\n========== EXACT NET-EDGE-DELTA ACCOUNTING =========="
)


print(
    "actions:",
    EXPECTED_ACTIONS
)


print(
    "cuts=0:",
    n_cut0,
    f"({cut0_fraction:.6%})"
)


print(
    "cuts=1:",
    n_cut1
)


print(
    "cuts=2:",
    n_cut2
)


print(
    "claimed cuts:",
    claimed_cut_count
)


print(
    "unique actual incumbent edges removed:",
    unique_removed
)


print(
    "shared-cut dedup:",
    shared_cut_dedup
)


print(
    "new edges:",
    new_edges
)


print(
    "ideal net delta before cut dedup:",
    ideal_net_without_cut_dedup
)


print(
    "cut-dedup correction:",
    shared_cut_dedup
)


print(
    "reconstructed actual net:",
    reconstructed_actual_net
)


print(
    "official actual net:",
    actual_net_edge_delta
)


print(
    "NET_EDGE_DELTA_IDENTITY: PASS"
)


# ============================================================
# 18. POSITIVE AUDIT
# ============================================================

print(
    "\n========== KNOWN POSITIVE ACTION AUDIT =========="
)


print(
    "known clean-gap positives captured:",
    known_positive_hits,
    "/",
    EXPECTED_POSITIVES
)


print(
    "FULL<=7 positives captured:",
    full7_positive_hits,
    "/",
    EXPECTED_FULL7
)


print(
    "known-positive yield among all actions:",
    known_positive_yield_all
)


print(
    "NOTE: this is not precision; unmatched actions are not confirmed negatives."
)


print(
    "\n========== FOLD x CUT =========="
)


print(
    fold_cut.to_string(
        index=False
    )
)


# ============================================================
# 19. DATASET OUTCOMES
# ============================================================

print(
    "\n========== DATASET OUTCOME COMPOSITION =========="
)


print(
    outcome_summary.to_string(
        index=False
    )
)


print(
    "\n========== STRONGEST DATASET-LEVEL ASSOCIATIONS =========="
)


association_display = (
    association[
        association[
            "target"
        ].eq(
            "delta_edge_jaccard"
        )
    ]
    .assign(
        abs_corr=lambda x:
            x[
                "spearman_rank_corr"
            ].abs()
    )
    .sort_values(
        "abs_corr",
        ascending=False,
        kind="stable",
    )
    .drop(
        columns=[
            "abs_corr",
        ]
    )
)


print(
    association_display.to_string(
        index=False
    )
)


# ============================================================
# 20. SAVE FORMAL AUDIT
# ============================================================

action_audit.to_pickle(
    ACTION_AUDIT_OUT
)


dataset_audit.to_pickle(
    DATASET_AUDIT_OUT
)


SUMMARY = {
    "stage":
        "12.11G3H3D0",

    "status":
        "Q9990_GRAPH_LEVEL_FAILURE_MECHANISM_AUDITED",

    "parent_policy":
        g3h3c[
            "policy_id"
        ],

    "parent_decision":
        g3h3c[
            "decision"
        ],

    "official_score": {
        "baseline":
            float(
                g3h3c[
                    "baseline"
                ][
                    "score"
                ]
            ),

        "q9990":
            float(
                g3h3c[
                    "q9990"
                ][
                    "score"
                ]
            ),

        "delta":
            float(
                g3h3c[
                    "delta"
                ][
                    "score"
                ]
            ),
    },

    "official_edge_delta": {
        "tp":
            int(
                g3h3c[
                    "delta"
                ][
                    "edge_tp"
                ]
            ),

        "fp":
            int(
                g3h3c[
                    "delta"
                ][
                    "edge_fp"
                ]
            ),

        "fn":
            int(
                g3h3c[
                    "delta"
                ][
                    "edge_fn"
                ]
            ),
    },

    "action_decomposition": {
        "actions":
            EXPECTED_ACTIONS,

        "cuts0":
            n_cut0,

        "cuts1":
            n_cut1,

        "cuts2":
            n_cut2,

        "cuts0_fraction":
            cut0_fraction,

        "cuts_ge1_fraction":
            cut_ge1_fraction,

        "claimed_cuts":
            claimed_cut_count,

        "unique_incumbent_edges_removed":
            unique_removed,

        "shared_cut_dedup":
            shared_cut_dedup,

        "new_edges_added":
            new_edges,

        "net_edge_delta":
            actual_net_edge_delta,

        "net_growth_per_action":
            net_growth_per_action,

        "unique_removal_per_action":
            unique_removal_per_action,

        "net_edge_delta_identity_pass":
            True,
    },

    "known_positive_audit": {
        "known_positive_hits":
            known_positive_hits,

        "full7_positive_hits":
            full7_positive_hits,

        "known_positive_yield":
            known_positive_yield_all,

        "yield_is_precision":
            False,
    },

    "mutation_regime":
        mutation_regime,

    "interpretation": {
        "diagnostic_only":
            True,

        "unknown_actions_are_confirmed_negatives":
            False,

        "scorer_retuned":
            False,

        "shortlist_retuned":
            False,

        "threshold_retuned":
            False,

        "new_policy_selected":
            False,

        "new_official_graph_evaluation":
            False,

        "graph_modified":
            False,

        "fold0_used":
            False,
    },

    "next":
        (
            "DESIGN_FROZEN_STRUCTURAL_ACTION_ABLATION_"
            "ONLY_AFTER_REVIEWING_CUT_AND_OCCUPANCY_DECOMPOSITION"
        ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 21. RELOAD INTEGRITY
# ============================================================

action_check = pd.read_pickle(
    ACTION_AUDIT_OUT
)


dataset_check = pd.read_pickle(
    DATASET_AUDIT_OUT
)


summary_check = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert len(
    action_check
) == EXPECTED_ACTIONS


assert len(
    dataset_check
) == 103


assert summary_check[
    "parent_decision"
] == (
    "REJECT_Q9990_AS_FINAL_REWIRE_POLICY"
)


assert summary_check[
    "interpretation"
][
    "scorer_retuned"
] is False


assert summary_check[
    "interpretation"
][
    "threshold_retuned"
] is False


assert summary_check[
    "interpretation"
][
    "new_policy_selected"
] is False


assert summary_check[
    "interpretation"
][
    "fold0_used"
] is False


# ============================================================
# 22. FINAL
# ============================================================

print(
    "\n========== 12.11G3H3D0 DECISION =========="
)


print(
    "Q9990_GRAPH_LEVEL_FAILURE_MECHANISM_AUDIT: PASS"
)


print(
    "mutation regime:",
    mutation_regime
)


print(
    "cuts=0 fraction:",
    cut0_fraction
)


print(
    "unique removals / action:",
    unique_removal_per_action
)


print(
    "net growth / action:",
    net_growth_per_action
)


print(
    "known-positive yield:",
    known_positive_yield_all
)


print(
    "scorer retuned:",
    "NO"
)


print(
    "shortlist retuned:",
    "NO"
)


print(
    "threshold retuned:",
    "NO"
)


print(
    "new policy selected:",
    "NO"
)


print(
    "new official graph evaluation:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "next:"
)


print(
    SUMMARY[
        "next"
    ]
)

In [ ]:
# ============================================================
# Stage 12.11G3H3D1
#
# Frozen Structural Action Ablation
#
# ------------------------------------------------------------
# PARENT
# ------------------------------------------------------------
#
# Rejected:
#   ASYM1x5_Q9990_NODEDISJOINT_REWIRE_V1
#
# Official dev delta:
#   score = -8.486898388859743e-05
#
# G3H3D0 mechanism audit:
#
#   cuts=0: 12130 actions / 22 known positives
#   cuts=1:  1413 actions /  1 known positive
#   cuts=2:   385 actions /  0 known positives
#
#   FREE_FREE       12130 / 22 positives
#   SOURCE_OCCUPIED   385 /  0
#   TARGET_OCCUPIED  1028 /  1
#   BOTH_OCCUPIED     385 /  0
#
# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Freeze and evaluate a SMALL structural ablation family
# WITHOUT changing:
#
#   scorer
#   feature schema
#   shortlist
#   ASYM1x5 rank rule
#   Q9990 score threshold
#   conflict resolver
#
# ------------------------------------------------------------
# FROZEN STRUCTURAL FAMILY
# ------------------------------------------------------------
#
# REFERENCE_FULL_Q9990
#   all 13,928 actions
#
# FREE_FREE_ONLY
#   source_occupied == 0
#   target_occupied == 0
#
# NO_SOURCE_CUT
#   source_occupied == 0
#
# NO_TARGET_CUT
#   target_occupied == 0
#
# REPLACEMENT_ONLY
#   cuts >= 1
#
# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------
#
# Family design is DEVELOPMENT-GT-INFORMED because G3H3D0
# already inspected development clean-gap oracle and official
# Q9990 failure.
#
# The family is frozen BEFORE the NEW official evaluations in
# this cell.
#
# Selection gate is predeclared:
#
#   1. aggregate score > CURRENT_BEST
#   2. all four fold deltas >= 0
#
# If multiple candidates pass:
#
#   1. highest aggregate delta
#   2. highest worst-fold delta
#   3. fewer actions
#   4. policy_id lexical
#
# Reference FULL_Q9990 is NOT eligible for promotion because
# it has already been formally rejected.
#
# NO:
#   scorer retuning
#   threshold retuning
#   Fold0
#   prediction GEFF writes
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import gc
import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


TRAIN_ROOT = (
    PROJECT
    / "data"
    / "raw"
    / "competition"
    / "biohub-cell-tracking-during-development"
    / "train"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


ACTIONS_PATH = (
    S12
    / "stage12_folds14_rewire_q9990_graph_candidate_v1_actions.pkl"
)


G3H3C_SUMMARY_PATH = (
    S12
    / "stage12_folds14_rewire_q9990_graph_candidate_v1_summary.json"
)


G3H3D0_ACTION_AUDIT_PATH = (
    S12
    / "stage12_folds14_rewire_q9990_failure_action_decomposition_v1.pkl"
)


G3H3D0_SUMMARY_PATH = (
    S12
    / "stage12_folds14_rewire_q9990_failure_mechanism_v1_summary.json"
)


CURRENT_BEST_DATASET_PATH = (
    S12
    / "stage12_folds14_current_best_edge_residual_datasets.pkl"
)


# ------------------------------------------------------------
# G3H3D1 outputs
# ------------------------------------------------------------

CONTRACT_OUT = (
    S12
    / "stage12_folds14_rewire_q9990_structural_ablation_v1_frozen.json"
)


METRICS_OUT = (
    S12
    / "stage12_folds14_rewire_q9990_structural_ablation_v1_metrics.pkl"
)


FOLD_OUT = (
    S12
    / "stage12_folds14_rewire_q9990_structural_ablation_v1_fold_metrics.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_folds14_rewire_q9990_structural_ablation_v1_summary.json"
)


for path in [
    TRAIN_ROOT,
    CV_PATH,
    ACTIONS_PATH,
    G3H3C_SUMMARY_PATH,
    G3H3D0_ACTION_AUDIT_PATH,
    G3H3D0_SUMMARY_PATH,
    CURRENT_BEST_DATASET_PATH,
]:

    assert path.exists(), path


for path in [
    CONTRACT_OUT,
    METRICS_OUT,
    FOLD_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3H3D1 output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite formal structural-ablation artifacts."
        )


# ============================================================
# 1. REQUIRED EXACT KERNEL STATE
# ============================================================

REQUIRED_CALLABLES = [
    "build_current_best",
    "apply_q9990_rewires",
]


missing = [
    name
    for name in REQUIRED_CALLABLES
    if (
        name not in globals()
        or
        not callable(
            globals()[
                name
            ]
        )
    )
]


assert not missing, (
    "\nMissing exact G3H3C helper state:\n"
    f"{missing}\n\n"
    "Do not approximate graph mutation semantics."
)


assert "fv9" in globals()


print(
    "========== G3H3D1 UPSTREAM STATE =========="
)


print(
    "missing helpers:",
    missing
)


print(
    "scorer retuned:",
    "NO"
)


print(
    "threshold retuned:",
    "NO"
)


print(
    "UPSTREAM FIDELITY: PASS"
)


# ============================================================
# 2. PARENT REJECTION / D0 AUDIT
# ============================================================

g3h3c = json.loads(
    G3H3C_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


g3h3d0 = json.loads(
    G3H3D0_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert g3h3c[
    "decision"
] == (
    "REJECT_Q9990_AS_FINAL_REWIRE_POLICY"
)


assert g3h3c[
    "fold0_used"
] is False


assert g3h3d0[
    "status"
] == (
    "Q9990_GRAPH_LEVEL_FAILURE_MECHANISM_AUDITED"
)


assert g3h3d0[
    "mutation_regime"
] == "ADDITION_DOMINATED"


assert g3h3d0[
    "interpretation"
][
    "fold0_used"
] is False


# ============================================================
# 3. LOAD FROZEN Q9990 ACTIONS
# ============================================================

actions = pd.read_pickle(
    ACTIONS_PATH
)


action_audit = pd.read_pickle(
    G3H3D0_ACTION_AUDIT_PATH
)


assert len(
    actions
) == 13_928


assert len(
    action_audit
) == 13_928


for col in [
    "source_occupied",
    "target_occupied",
    "cuts",
]:

    actions[
        col
    ] = pd.to_numeric(
        actions[
            col
        ],
        errors="raise",
    ).astype(
        int
    )


# ============================================================
# 4. STRUCTURAL MASKS
# ============================================================

def structural_mask(
    frame,
    policy_id,
):

    if policy_id == "REFERENCE_FULL_Q9990":

        return np.ones(
            len(
                frame
            ),
            dtype=bool,
        )


    if policy_id == "FREE_FREE_ONLY":

        return (
            frame[
                "source_occupied"
            ].eq(
                0
            )
            &
            frame[
                "target_occupied"
            ].eq(
                0
            )
        ).to_numpy()


    if policy_id == "NO_SOURCE_CUT":

        return frame[
            "source_occupied"
        ].eq(
            0
        ).to_numpy()


    if policy_id == "NO_TARGET_CUT":

        return frame[
            "target_occupied"
        ].eq(
            0
        ).to_numpy()


    if policy_id == "REPLACEMENT_ONLY":

        return frame[
            "cuts"
        ].ge(
            1
        ).to_numpy()


    raise KeyError(
        policy_id
    )


POLICY_IDS = [
    "REFERENCE_FULL_Q9990",
    "FREE_FREE_ONLY",
    "NO_SOURCE_CUT",
    "NO_TARGET_CUT",
    "REPLACEMENT_ONLY",
]


ELIGIBLE_POLICY_IDS = [
    "FREE_FREE_ONLY",
    "NO_SOURCE_CUT",
    "NO_TARGET_CUT",
    "REPLACEMENT_ONLY",
]


# ============================================================
# 5. PRE-EVALUATION ACTION / ORACLE AUDIT
#
# This is known development information from D0.
# It is NOT a new hidden evaluation.
# ============================================================

policy_pre_rows = []


for policy_id in POLICY_IDS:

    mask = structural_mask(
        actions,
        policy_id,
    )


    sub = actions.loc[
        mask
    ]


    audit_sub = action_audit.loc[
        mask
    ]


    policy_pre_rows.append(
        {
            "policy_id":
                policy_id,

            "actions":
                int(
                    len(
                        sub
                    )
                ),

            "known_positive_hits":
                int(
                    audit_sub[
                        "is_known_positive"
                    ].sum()
                ),

            "full7_positive_hits":
                int(
                    audit_sub[
                        "is_known_full7_positive"
                    ].sum()
                ),

            "cuts0":
                int(
                    sub[
                        "cuts"
                    ].eq(
                        0
                    ).sum()
                ),

            "cuts1":
                int(
                    sub[
                        "cuts"
                    ].eq(
                        1
                    ).sum()
                ),

            "cuts2":
                int(
                    sub[
                        "cuts"
                    ].eq(
                        2
                    ).sum()
                ),
        }
    )


policy_pre = pd.DataFrame(
    policy_pre_rows
)


# ------------------------------------------------------------
# Expected structural counts
# ------------------------------------------------------------

expected_counts = {
    "REFERENCE_FULL_Q9990":
        13_928,

    "FREE_FREE_ONLY":
        12_130,

    "NO_SOURCE_CUT":
        13_158,

    "NO_TARGET_CUT":
        12_515,

    "REPLACEMENT_ONLY":
        1_798,
}


expected_positive_hits = {
    "REFERENCE_FULL_Q9990":
        23,

    "FREE_FREE_ONLY":
        22,

    "NO_SOURCE_CUT":
        23,

    "NO_TARGET_CUT":
        22,

    "REPLACEMENT_ONLY":
        1,
}


for row in policy_pre.itertuples(
    index=False
):

    assert int(
        row.actions
    ) == expected_counts[
        row.policy_id
    ]


    assert int(
        row.known_positive_hits
    ) == expected_positive_hits[
        row.policy_id
    ]


print(
    "\n========== FROZEN STRUCTURAL FAMILY INPUT =========="
)


print(
    policy_pre.to_string(
        index=False
    )
)


# ============================================================
# 6. FREEZE STRUCTURAL ABLATION CONTRACT
#
# Frozen BEFORE NEW official graph evaluation.
# ============================================================

CONTRACT = {
    "stage":
        "12.11G3H3D1",

    "contract_id":
        "Q9990_STRUCTURAL_ACTION_ABLATION_V1",

    "parent_policy":
        g3h3c[
            "policy_id"
        ],

    "parent_decision":
        g3h3c[
            "decision"
        ],

    "family": {
        "REFERENCE_FULL_Q9990":
            "ALL_FROZEN_Q9990_ACTIONS",

        "FREE_FREE_ONLY":
            (
                "SOURCE_OCCUPIED_EQ_0_AND_"
                "TARGET_OCCUPIED_EQ_0"
            ),

        "NO_SOURCE_CUT":
            "SOURCE_OCCUPIED_EQ_0",

        "NO_TARGET_CUT":
            "TARGET_OCCUPIED_EQ_0",

        "REPLACEMENT_ONLY":
            "CUTS_GE_1",
    },

    "reference_policy":
        "REFERENCE_FULL_Q9990",

    "eligible_policies":
        ELIGIBLE_POLICY_IDS,

    "predeclared_promotion_gate": {
        "aggregate_score_delta_gt_zero":
            True,

        "all_four_fold_score_deltas_ge_zero":
            True,
    },

    "predeclared_selection_order": [
        "AGGREGATE_SCORE_DELTA_DESC",
        "WORST_FOLD_SCORE_DELTA_DESC",
        "ACTION_COUNT_ASC",
        "POLICY_ID_ASC",
    ],

    "family_design_is_development_gt_informed":
        True,

    "prior_dev_information_used": [
        "G3H3C_OFFICIAL_Q9990_FAILURE",
        "G3H3D0_CUT_DECOMPOSITION",
        "G3H3D0_KNOWN_CLEAN_GAP_POSITIVE_DECOMPOSITION",
    ],

    "new_official_metrics_seen_before_contract_freeze":
        False,

    "scorer_retuned":
        False,

    "shortlist_retuned":
        False,

    "rank_backbone_retuned":
        False,

    "score_threshold_retuned":
        False,

    "fold0_used":
        False,

    "prediction_geff_written":
        False,
}


CONTRACT_SHA = hashlib.sha256(
    json.dumps(
        CONTRACT,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


CONTRACT[
    "contract_sha256"
] = CONTRACT_SHA


CONTRACT_OUT.write_text(
    json.dumps(
        CONTRACT,
        indent=2,
    ),
    encoding="utf-8",
)


assert CONTRACT_OUT.exists()


print(
    "\n========== FROZEN G3H3D1 CONTRACT =========="
)


print(
    "contract SHA:",
    CONTRACT_SHA
)


print(
    "policies:",
    POLICY_IDS
)


print(
    "eligible:",
    ELIGIBLE_POLICY_IDS
)


print(
    "new official metrics seen before freeze:",
    "NO"
)


print(
    "STRUCTURAL FAMILY FROZEN: YES"
)


# ============================================================
# 7. DEVELOPMENT CORPUS
# ============================================================

cv = pd.read_csv(
    CV_PATH
)


corp = (
    cv[
        cv[
            "fold"
        ].isin(
            [
                1,
                2,
                3,
                4,
            ]
        )
        &
        cv[
            "dataset"
        ].str.startswith(
            "6bba_"
        )
    ]
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    corp
) == 103


assert not corp[
    "fold"
].eq(
    0
).any()


names = corp[
    "dataset"
].tolist()


fold_map = dict(
    zip(
        corp[
            "dataset"
        ],

        corp[
            "fold"
        ].astype(
            int
        ),
    )
)


# ============================================================
# 8. AUTHORITATIVE estimated_nodes
# ============================================================

old_dataset = pd.read_pickle(
    CURRENT_BEST_DATASET_PATH
)


assert len(
    old_dataset
) == 103


estimated_nodes = {}


for row in old_dataset.itertuples(
    index=False
):

    n_pred = float(
        row.num_pred_nodes
    )


    ratio = float(
        row.total_node_ratio
    )


    n_total = (
        n_pred
        /
        (
            1.0
            +
            ratio
        )
    )


    assert np.isfinite(
        n_total
    )


    estimated_nodes[
        row.dataset
    ] = float(
        n_total
    )


assert set(
    estimated_nodes
) == set(
    names
)


# ============================================================
# 9. ACTION LOOKUP
# ============================================================

actions_by_dataset = {
    dataset:
        sub.copy()

    for dataset, sub
    in actions.groupby(
        "dataset",
        sort=False,
    )
}


assert set(
    actions_by_dataset
) == set(
    names
)


# ============================================================
# 10. OFFICIAL GRAPH EVALUATION
# ============================================================

print(
    "\n============================================================"
)


print(
    "RUN STRUCTURAL ACTION ABLATION"
)


print(
    "============================================================"
)


baseline_records = []

policy_records = {
    policy_id:
        []

    for policy_id
    in POLICY_IDS
}


metric_rows = []


for i, dataset in enumerate(
    names,
    start=1,
):

    fold = int(
        fold_map[
            dataset
        ]
    )


    # --------------------------------------------------------
    # Exact CURRENT_BEST parent graph
    # --------------------------------------------------------

    parent_graph, scale = build_current_best(
        dataset
    )


    gt = fv9.load_graph(
        TRAIN_ROOT
        / f"{dataset}.geff"
    )


    n_total = float(
        estimated_nodes[
            dataset
        ]
    )


    # --------------------------------------------------------
    # Baseline copy
    # --------------------------------------------------------

    baseline_graph = parent_graph.copy()


    baseline_metric = fv9.evaluate_graph(
        baseline_graph,
        gt,
        scale,
        n_total,
        fv9.FROZEN_V9_CONFIG,
    )


    baseline_records.append(
        baseline_metric
    )


    dataset_actions = actions_by_dataset[
        dataset
    ]


    # --------------------------------------------------------
    # Structural family
    # --------------------------------------------------------

    for policy_id in POLICY_IDS:

        mask = structural_mask(
            dataset_actions,
            policy_id,
        )


        policy_actions = (
            dataset_actions.loc[
                mask
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


        graph = parent_graph.copy()


        (
            graph,
            mutation_stats,
            _mutation_audit,
        ) = apply_q9990_rewires(
            graph,
            policy_actions,
        )


        metric = fv9.evaluate_graph(
            graph,
            gt,
            scale,
            n_total,
            fv9.FROZEN_V9_CONFIG,
        )


        policy_records[
            policy_id
        ].append(
            metric
        )


        metric_rows.append(
            {
                "dataset":
                    dataset,

                "fold":
                    fold,

                "policy_id":
                    policy_id,

                "actions":
                    int(
                        mutation_stats[
                            "actions"
                        ]
                    ),

                "edges_removed":
                    int(
                        mutation_stats[
                            "unique_incumbent_edges_removed"
                        ]
                    ),

                "edges_added":
                    int(
                        mutation_stats[
                            "new_edges_added"
                        ]
                    ),

                "net_edge_delta":
                    int(
                        mutation_stats[
                            "net_edge_delta"
                        ]
                    ),

                "edge_tp":
                    int(
                        metric[
                            "edge_tp"
                        ]
                    ),

                "edge_fp":
                    int(
                        metric[
                            "edge_fp"
                        ]
                    ),

                "edge_fn":
                    int(
                        metric[
                            "edge_fn"
                        ]
                    ),

                "division_tp":
                    int(
                        metric[
                            "division_tp"
                        ]
                    ),

                "division_fp":
                    int(
                        metric[
                            "division_fp"
                        ]
                    ),

                "division_fn":
                    int(
                        metric[
                            "division_fn"
                        ]
                    ),

                "node_recall":
                    float(
                        metric[
                            "node_recall"
                        ]
                    ),

                "edge_jaccard":
                    float(
                        metric[
                            "edge_jaccard"
                        ]
                    ),

                "adj_edge_jaccard":
                    float(
                        metric[
                            "adj_edge_jaccard"
                        ]
                    ),
            }
        )


        del policy_actions
        del graph
        del _mutation_audit


    if (
        i == 1
        or
        i % 10 == 0
        or
        i == 103
    ):

        print(
            f"{i:3d}/103"
            f" | fold={fold}"
            f" | {dataset}"
        )


    del parent_graph
    del baseline_graph
    del gt

    gc.collect()


metrics = pd.DataFrame(
    metric_rows
)


assert len(
    metrics
) == (
    103
    *
    len(
        POLICY_IDS
    )
)


# ============================================================
# 11. BASELINE FIDELITY
# ============================================================

BASE_SCORE = (
    0.7919774336227487
)


BASE_TP = 77_535
BASE_FP = 9_398
BASE_FN = 10_254


baseline_summary = fv9.summarise(
    baseline_records
)


baseline_tp = int(
    sum(
        int(
            row[
                "edge_tp"
            ]
        )
        for row
        in baseline_records
    )
)


baseline_fp = int(
    sum(
        int(
            row[
                "edge_fp"
            ]
        )
        for row
        in baseline_records
    )
)


baseline_fn = int(
    sum(
        int(
            row[
                "edge_fn"
            ]
        )
        for row
        in baseline_records
    )
)


assert baseline_tp == BASE_TP

assert baseline_fp == BASE_FP

assert baseline_fn == BASE_FN


assert abs(
    float(
        baseline_summary[
            "score"
        ]
    )
    -
    BASE_SCORE
) <= 5e-7


print(
    "\n========== G3H3D1 CURRENT_BEST FIDELITY =========="
)


print(
    "TP/FP/FN:",
    baseline_tp,
    "/",
    baseline_fp,
    "/",
    baseline_fn
)


print(
    "score:",
    f"{baseline_summary['score']:.12f}"
)


print(
    "expected:",
    f"{BASE_SCORE:.12f}"
)


print(
    "CURRENT_BEST_FIDELITY: PASS"
)


# ============================================================
# 12. AGGREGATE STRUCTURAL SUMMARY
# ============================================================

aggregate_rows = []


for policy_id in POLICY_IDS:

    records = policy_records[
        policy_id
    ]


    assert len(
        records
    ) == 103


    summary = fv9.summarise(
        records
    )


    policy_metrics = metrics[
        metrics[
            "policy_id"
        ].eq(
            policy_id
        )
    ]


    tp = int(
        policy_metrics[
            "edge_tp"
        ].sum()
    )


    fp = int(
        policy_metrics[
            "edge_fp"
        ].sum()
    )


    fn = int(
        policy_metrics[
            "edge_fn"
        ].sum()
    )


    aggregate_rows.append(
        {
            "policy_id":
                policy_id,

            "actions":
                int(
                    policy_metrics[
                        "actions"
                    ].sum()
                ),

            "unique_edges_removed":
                int(
                    policy_metrics[
                        "edges_removed"
                    ].sum()
                ),

            "edges_added":
                int(
                    policy_metrics[
                        "edges_added"
                    ].sum()
                ),

            "net_edge_delta":
                int(
                    policy_metrics[
                        "net_edge_delta"
                    ].sum()
                ),

            "edge_tp":
                tp,

            "delta_tp":
                int(
                    tp
                    -
                    baseline_tp
                ),

            "edge_fp":
                fp,

            "delta_fp":
                int(
                    fp
                    -
                    baseline_fp
                ),

            "edge_fn":
                fn,

            "delta_fn":
                int(
                    fn
                    -
                    baseline_fn
                ),

            "edge_jaccard":
                float(
                    summary[
                        "edge_jaccard"
                    ]
                ),

            "delta_edge_j":
                float(
                    summary[
                        "edge_jaccard"
                    ]
                    -
                    baseline_summary[
                        "edge_jaccard"
                    ]
                ),

            "adj_edge_jaccard":
                float(
                    summary[
                        "adj_edge_jaccard"
                    ]
                ),

            "delta_adj_edge_j":
                float(
                    summary[
                        "adj_edge_jaccard"
                    ]
                    -
                    baseline_summary[
                        "adj_edge_jaccard"
                    ]
                ),

            "division_jaccard":
                float(
                    summary[
                        "division_jaccard"
                    ]
                ),

            "node_recall":
                float(
                    summary[
                        "node_recall"
                    ]
                ),

            "score":
                float(
                    summary[
                        "score"
                    ]
                ),

            "delta_score":
                float(
                    summary[
                        "score"
                    ]
                    -
                    baseline_summary[
                        "score"
                    ]
                ),
        }
    )


aggregate = pd.DataFrame(
    aggregate_rows
)


# ============================================================
# 13. FULL-Q9990 REPRODUCTION
# ============================================================

ref = aggregate[
    aggregate[
        "policy_id"
    ].eq(
        "REFERENCE_FULL_Q9990"
    )
]


assert len(
    ref
) == 1


ref = ref.iloc[
    0
]


assert int(
    ref[
        "actions"
    ]
) == 13_928


assert int(
    ref[
        "edge_tp"
    ]
) == 77_486


assert int(
    ref[
        "edge_fp"
    ]
) == 9_348


assert int(
    ref[
        "edge_fn"
    ]
) == 10_303


assert abs(
    float(
        ref[
            "score"
        ]
    )
    -
    float(
        g3h3c[
            "q9990"
        ][
            "score"
        ]
    )
) <= 5e-10


print(
    "\n========== FULL Q9990 REPRODUCTION =========="
)


print(
    "score:",
    f"{float(ref['score']):.12f}"
)


print(
    "expected:",
    f"{float(g3h3c['q9990']['score']):.12f}"
)


print(
    "FULL_Q9990_REPRODUCTION: PASS"
)


# ============================================================
# 14. FOLD SUMMARIES
# ============================================================

fold_rows = []


for policy_id in POLICY_IDS:

    records = policy_records[
        policy_id
    ]


    for fold in [
        1,
        2,
        3,
        4,
    ]:

        fold_mask = (
            corp[
                "fold"
            ].eq(
                fold
            )
            .to_numpy()
        )


        baseline_fold_records = [
            record
            for record, keep
            in zip(
                baseline_records,
                fold_mask,
            )
            if keep
        ]


        policy_fold_records = [
            record
            for record, keep
            in zip(
                records,
                fold_mask,
            )
            if keep
        ]


        base_fold_summary = fv9.summarise(
            baseline_fold_records
        )


        policy_fold_summary = fv9.summarise(
            policy_fold_records
        )


        policy_dataset_metrics = metrics[
            metrics[
                "policy_id"
            ].eq(
                policy_id
            )
            &
            metrics[
                "fold"
            ].eq(
                fold
            )
        ]


        fold_rows.append(
            {
                "policy_id":
                    policy_id,

                "fold":
                    fold,

                "actions":
                    int(
                        policy_dataset_metrics[
                            "actions"
                        ].sum()
                    ),

                "base_score":
                    float(
                        base_fold_summary[
                            "score"
                        ]
                    ),

                "policy_score":
                    float(
                        policy_fold_summary[
                            "score"
                        ]
                    ),

                "delta_score":
                    float(
                        policy_fold_summary[
                            "score"
                        ]
                        -
                        base_fold_summary[
                            "score"
                        ]
                    ),

                "base_edge_j":
                    float(
                        base_fold_summary[
                            "edge_jaccard"
                        ]
                    ),

                "policy_edge_j":
                    float(
                        policy_fold_summary[
                            "edge_jaccard"
                        ]
                    ),

                "delta_edge_j":
                    float(
                        policy_fold_summary[
                            "edge_jaccard"
                        ]
                        -
                        base_fold_summary[
                            "edge_jaccard"
                        ]
                    ),
            }
        )


fold_df = pd.DataFrame(
    fold_rows
)


assert len(
    fold_df
) == (
    len(
        POLICY_IDS
    )
    *
    4
)


# ============================================================
# 15. PROMOTION GATES
# ============================================================

selection_rows = []


for policy_id in POLICY_IDS:

    agg = aggregate[
        aggregate[
            "policy_id"
        ].eq(
            policy_id
        )
    ].iloc[
        0
    ]


    folds = fold_df[
        fold_df[
            "policy_id"
        ].eq(
            policy_id
        )
    ]


    assert len(
        folds
    ) == 4


    worst_fold_delta = float(
        folds[
            "delta_score"
        ].min()
    )


    all_folds_nonnegative = bool(
        folds[
            "delta_score"
        ].ge(
            0.0
        ).all()
    )


    aggregate_improved = bool(
        float(
            agg[
                "delta_score"
            ]
        ) > 0.0
    )


    eligible_family_member = (
        policy_id
        in
        ELIGIBLE_POLICY_IDS
    )


    passes_gate = bool(
        eligible_family_member
        and
        aggregate_improved
        and
        all_folds_nonnegative
    )


    selection_rows.append(
        {
            "policy_id":
                policy_id,

            "eligible_family_member":
                eligible_family_member,

            "actions":
                int(
                    agg[
                        "actions"
                    ]
                ),

            "delta_score":
                float(
                    agg[
                        "delta_score"
                    ]
                ),

            "worst_fold_delta":
                worst_fold_delta,

            "all_folds_nonnegative":
                all_folds_nonnegative,

            "aggregate_improved":
                aggregate_improved,

            "passes_promotion_gate":
                passes_gate,
        }
    )


selection = pd.DataFrame(
    selection_rows
)


passed = (
    selection[
        selection[
            "passes_promotion_gate"
        ]
    ]
    .sort_values(
        [
            "delta_score",
            "worst_fold_delta",
            "actions",
            "policy_id",
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


if len(
    passed
):

    selected_policy = str(
        passed.iloc[
            0
        ][
            "policy_id"
        ]
    )


    decision = (
        "STRUCTURAL_DEV_POLICY_CANDIDATE_FOUND"
    )


else:

    selected_policy = None


    decision = (
        "STRUCTURAL_ABLATION_BRANCH_REJECTED"
    )


# ============================================================
# 16. PRINT AGGREGATE RESULTS
# ============================================================

display_cols = [
    "policy_id",
    "actions",
    "unique_edges_removed",
    "net_edge_delta",

    "delta_tp",
    "delta_fp",
    "delta_fn",

    "edge_jaccard",
    "delta_edge_j",

    "division_jaccard",

    "score",
    "delta_score",
]


print(
    "\n========== G3H3D1 STRUCTURAL ABLATION =========="
)


print(
    aggregate[
        display_cols
    ].to_string(
        index=False
    )
)


# ============================================================
# 17. PRINT FOLD RESULTS
# ============================================================

print(
    "\n========== G3H3D1 BY DEVELOPMENT FOLD =========="
)


print(
    fold_df[
        [
            "policy_id",
            "fold",
            "actions",
            "base_score",
            "policy_score",
            "delta_score",
            "delta_edge_j",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 18. PRINT PROMOTION GATE
# ============================================================

print(
    "\n========== G3H3D1 PROMOTION GATE =========="
)


print(
    selection.to_string(
        index=False
    )
)


print(
    "\nselected policy:",
    selected_policy
)


print(
    "decision:",
    decision
)


# ============================================================
# 19. PERSIST FORMAL RESULTS
# ============================================================

metrics.to_pickle(
    METRICS_OUT
)


fold_df.to_pickle(
    FOLD_OUT
)


SUMMARY = {
    "stage":
        "12.11G3H3D1",

    "status":
        "Q9990_STRUCTURAL_ACTION_ABLATION_COMPLETE",

    "contract_sha256":
        CONTRACT_SHA,

    "parent_policy":
        g3h3c[
            "policy_id"
        ],

    "parent_policy_rejected":
        True,

    "aggregate_results":
        aggregate.to_dict(
            "records"
        ),

    "fold_results":
        fold_df.to_dict(
            "records"
        ),

    "promotion_gate":
        selection.to_dict(
            "records"
        ),

    "selected_policy":
        selected_policy,

    "decision":
        decision,

    "family_design_is_development_gt_informed":
        True,

    "selected_policy_is_final_frozen_policy":
        False,

    "scorer_retuned":
        False,

    "shortlist_retuned":
        False,

    "rank_backbone_retuned":
        False,

    "score_threshold_retuned":
        False,

    "fold0_used":
        False,

    "prediction_geff_written":
        False,

    "next": (
        "FREEZE_SELECTED_STRUCTURAL_DEV_POLICY_BEFORE_LOCKED_FOLD0"
        if selected_policy is not None
        else
        "CLOSE_OR_REFRAME_REWIRE_BRANCH_WITHOUT_FOLD0"
    ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 20. RELOAD FIDELITY
# ============================================================

metric_check = pd.read_pickle(
    METRICS_OUT
)


fold_check = pd.read_pickle(
    FOLD_OUT
)


summary_check = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert len(
    metric_check
) == (
    103
    *
    len(
        POLICY_IDS
    )
)


assert len(
    fold_check
) == (
    4
    *
    len(
        POLICY_IDS
    )
)


assert summary_check[
    "fold0_used"
] is False


assert summary_check[
    "scorer_retuned"
] is False


assert summary_check[
    "score_threshold_retuned"
] is False


# ============================================================
# 21. FINAL
# ============================================================

print(
    "\n========== 12.11G3H3D1 DECISION =========="
)


print(
    "Q9990_STRUCTURAL_ACTION_ABLATION: PASS"
)


print(
    "CURRENT_BEST score:",
    f"{baseline_summary['score']:.12f}"
)


print(
    "selected policy:",
    selected_policy
)


print(
    "decision:",
    decision
)


print(
    "selected policy final/frozen:",
    "NO"
)


print(
    "scorer retuned:",
    "NO"
)


print(
    "shortlist retuned:",
    "NO"
)


print(
    "rank backbone retuned:",
    "NO"
)


print(
    "score threshold retuned:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "prediction GEFF written:",
    "NO"
)


print(
    "next:"
)


print(
    SUMMARY[
        "next"
    ]
)

In [ ]:
# ============================================================
# Stage 12.11G3H3D2
#
# Final Development Policy Freeze
# + Fold0 Deployment Threshold Calibration
#
# ------------------------------------------------------------
# SELECTED DEVELOPMENT POLICY
# ------------------------------------------------------------
#
# Structural rule:
#   NO_TARGET_CUT
#
# Exact definition:
#   target_occupied == 0
#
# Parent:
#   Q9990_STRUCTURAL_ACTION_ABLATION_V1
#
# Development result:
#
#   CURRENT_BEST:
#       0.7919774336227487
#
#   NO_TARGET_CUT:
#       ~0.792028
#
#   delta:
#       +~0.000051
#
# Development folds:
#
#   fold1:  0
#   fold2:  0
#   fold3: +~0.000089
#   fold4: +~0.000105
#
# Predeclared promotion gate:
#
#   aggregate delta > 0
#   AND all four folds >= 0
#
# PASSED.
#
# ------------------------------------------------------------
# FINAL FROZEN POLICY STACK
# ------------------------------------------------------------
#
# frozen full-development scorer
#       ->
# frozen complete shortlist
#       ->
# ASYM1x5 endpoint-score rank gate
#       ->
# Q9990 score gate
#       ->
# deterministic node-disjoint resolver
#       ->
# target_occupied == 0
#
# ------------------------------------------------------------
# FOLD0 SCORER
# ------------------------------------------------------------
#
# Fold0 must NOT use any LOFO model.
#
# It uses:
#
#   MULTIFRAME_REWIRE_LOGREG_SCORER_V1
#
# fitted once on folds1-4 in G3H2.
#
# ------------------------------------------------------------
# FOLD0 NUMERICAL SCORE THRESHOLD
# ------------------------------------------------------------
#
# Q9990 has ALREADY been selected/frozen.
#
# The numerical deployment threshold is calibrated by:
#
#   full-dev frozen scorer
#       scored on
#   full development GT-blind raw G3H0 hash sample
#       ->
#   99.9th percentile
#
# No Fold0 data.
# No oracle labels.
#
# This is calibration of an already-frozen quantile rule,
# NOT threshold retuning.
#
# ------------------------------------------------------------
# THIS STAGE DOES NOT
# ------------------------------------------------------------
#
#   read Fold0
#   evaluate Fold0
#   modify any graph
#   retune scorer
#   retune shortlist
#   retune ASYM1x5
#   retune Q9990
#   retune structural rule
#
# After this cell succeeds:
#
#   DEVELOPMENT IS CLOSED.
#
# The next stage is ONE locked Fold0 confirmation.
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import hashlib
import json

import joblib

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


# ------------------------------------------------------------
# G3H0 / G3H2 frozen scorer artifacts
# ------------------------------------------------------------

FEATURE_SCHEMA_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_feature_schema_v1_frozen.json"
)


RAW_SAMPLE_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_feature_sample_hash64_v1.pkl"
)


SCORER_FREEZE_PATH = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_frozen.json"
)


FULL_DEV_PREPROCESSOR_PATH = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_preprocessor.npz"
)


FULL_DEV_MODEL_PATH = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_model.joblib"
)


G3H2_SUMMARY_PATH = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_summary.json"
)


# ------------------------------------------------------------
# Rank / quantile contracts
# ------------------------------------------------------------

BACKBONE_PATH = (
    S12
    / "stage12_folds14_rewire_asym1x5_backbone_v1_frozen.json"
)


QUANTILE_CONTRACT_PATH = (
    S12
    / "stage12_folds14_rewire_asym1x5_score_quantile_frontier_v1_frozen.json"
)


# ------------------------------------------------------------
# Structural selection
# ------------------------------------------------------------

G3H3D1_CONTRACT_PATH = (
    S12
    / "stage12_folds14_rewire_q9990_structural_ablation_v1_frozen.json"
)


G3H3D1_SUMMARY_PATH = (
    S12
    / "stage12_folds14_rewire_q9990_structural_ablation_v1_summary.json"
)


# ------------------------------------------------------------
# Development Q9990 actions — provenance only
# ------------------------------------------------------------

DEV_ACTIONS_PATH = (
    S12
    / "stage12_folds14_rewire_q9990_graph_candidate_v1_actions.pkl"
)


# ------------------------------------------------------------
# G3H3D2 formal outputs
# ------------------------------------------------------------

FINAL_POLICY_OUT = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_frozen.json"
)


DEPLOYMENT_THRESHOLD_OUT = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_deployment_threshold.json"
)


SELECTED_DEV_ACTIONS_OUT = (
    S12
    / "stage12_folds14_rewire_asym1x5_q9990_no_target_cut_v1_actions.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_freeze_summary.json"
)


for path in [
    FEATURE_SCHEMA_PATH,
    RAW_SAMPLE_PATH,
    SCORER_FREEZE_PATH,
    FULL_DEV_PREPROCESSOR_PATH,
    FULL_DEV_MODEL_PATH,
    G3H2_SUMMARY_PATH,
    BACKBONE_PATH,
    QUANTILE_CONTRACT_PATH,
    G3H3D1_CONTRACT_PATH,
    G3H3D1_SUMMARY_PATH,
    DEV_ACTIONS_PATH,
]:

    assert path.exists(), path


for path in [
    FINAL_POLICY_OUT,
    DEPLOYMENT_THRESHOLD_OUT,
    SELECTED_DEV_ACTIONS_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3H3D2 frozen output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite a final development policy freeze."
        )


# ============================================================
# 1. AUTHORITATIVE SHAs
# ============================================================

FEATURE_SCHEMA_SHA = (
    "0a9f8e9a41c5c80c803c3a95955702dc"
    "f4039605a21fc656a154aa05be55b39c"
)


SCORER_FREEZE_SHA = (
    "5ae66ee0f2da841bfe3a7a0b56fdab1b"
    "57f7ec3f504ba57441e2c8c272279e91"
)


FULL_DEV_MODEL_SHA = (
    "00918fa386d4de6e6a5f93892e47d876"
    "e6ff45ceb89670f982ebba45f59ebf76"
)


FULL_DEV_PREPROCESSOR_SHA = (
    "6b5425093efd17613c68fcba855a756a3"
    "445a286a61ac813a83652dd2200ae41"
)


BACKBONE_SHA = (
    "34ffb76b15e8c7c17e93681d34c3b9aa"
    "45c7ef8cc6e7171d32dd542ef12ab55b"
)


QUANTILE_CONTRACT_SHA = (
    "6df3cf46ee684f06120fb948f9c2e4bcc"
    "36a30884228a15e7699b35f95ae991e"
)


EXPECTED_RAW_SAMPLE_ROWS = 273_658


SELECTED_POLICY = (
    "NO_TARGET_CUT"
)


FINAL_POLICY_ID = (
    "ASYM1x5_Q9990_NO_TARGET_CUT_V1"
)


# ============================================================
# 2. FILE SHA HELPER
# ============================================================

def file_sha256(
    path,
):

    h = hashlib.sha256()


    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024
                *
                1024
            )


            if not chunk:

                break


            h.update(
                chunk
            )


    return h.hexdigest()


# ============================================================
# 3. VERIFY FULL-DEV MODEL ARTIFACTS
# ============================================================

actual_model_sha = file_sha256(
    FULL_DEV_MODEL_PATH
)


actual_preprocessor_sha = file_sha256(
    FULL_DEV_PREPROCESSOR_PATH
)


assert actual_model_sha == FULL_DEV_MODEL_SHA, (
    actual_model_sha,
    FULL_DEV_MODEL_SHA,
)


assert (
    actual_preprocessor_sha
    ==
    FULL_DEV_PREPROCESSOR_SHA
), (
    actual_preprocessor_sha,
    FULL_DEV_PREPROCESSOR_SHA,
)


g3h2 = json.loads(
    G3H2_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert g3h2[
    "status"
] == (
    "SCORER_FAMILY_FROZEN_AND_FULL_DEV_MODEL_FIT"
)


assert g3h2[
    "scorer_id"
] == (
    "MULTIFRAME_REWIRE_LOGREG_SCORER_V1"
)


assert g3h2[
    "selected_family"
] == (
    "LOGREG_L2_C1_BALANCED_V1"
)


assert g3h2[
    "future_scorer_family_retuning"
] is False


assert g3h2[
    "fold0_used"
] is False


print(
    "========== G3H3D2 FULL-DEV SCORER =========="
)


print(
    "model SHA:",
    actual_model_sha
)


print(
    "preprocessor SHA:",
    actual_preprocessor_sha
)


print(
    "scorer family:",
    g3h2[
        "selected_family"
    ]
)


print(
    "Fold0 used:",
    "NO"
)


# ============================================================
# 4. VERIFY FEATURE SCHEMA
# ============================================================

feature_schema = json.loads(
    FEATURE_SCHEMA_PATH.read_text(
        encoding="utf-8"
    )
)


assert feature_schema[
    "feature_schema_sha256"
] == FEATURE_SCHEMA_SHA


FEATURE_COLUMNS = list(
    feature_schema[
        "features"
    ]
)


assert len(
    FEATURE_COLUMNS
) == 32


# ============================================================
# 5. VERIFY SCORER FREEZE
# ============================================================

scorer_freeze = json.loads(
    SCORER_FREEZE_PATH.read_text(
        encoding="utf-8"
    )
)


assert scorer_freeze[
    "scorer_freeze_sha256"
] == SCORER_FREEZE_SHA


assert scorer_freeze[
    "future_scorer_family_retuning"
] is False


assert scorer_freeze[
    "fold0_used"
] is False


# ============================================================
# 6. VERIFY ASYM1x5 BACKBONE
# ============================================================

backbone = json.loads(
    BACKBONE_PATH.read_text(
        encoding="utf-8"
    )
)


assert backbone[
    "backbone_freeze_sha256"
] == BACKBONE_SHA


assert backbone[
    "selected_rule"
] == "ASYM_1x5"


assert backbone[
    "fold0_used"
] is False


# ============================================================
# 7. VERIFY Q9990 CONTRACT
# ============================================================

quantile_contract = json.loads(
    QUANTILE_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


assert quantile_contract[
    "quantile_contract_sha256"
] == QUANTILE_CONTRACT_SHA


assert (
    float(
        quantile_contract[
            "quantile_rules"
        ][
            "Q9990"
        ]
    )
    ==
    0.999
)


assert quantile_contract[
    "fold0_used"
] is False


# ============================================================
# 8. VERIFY STRUCTURAL SELECTION
# ============================================================

d1_contract = json.loads(
    G3H3D1_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


d1 = json.loads(
    G3H3D1_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert d1[
    "status"
] == (
    "Q9990_STRUCTURAL_ACTION_ABLATION_COMPLETE"
)


assert d1[
    "decision"
] == (
    "STRUCTURAL_DEV_POLICY_CANDIDATE_FOUND"
)


assert d1[
    "selected_policy"
] == SELECTED_POLICY


assert d1[
    "selected_policy_is_final_frozen_policy"
] is False


assert d1[
    "fold0_used"
] is False


assert d1[
    "scorer_retuned"
] is False


assert d1[
    "score_threshold_retuned"
] is False


promotion = pd.DataFrame(
    d1[
        "promotion_gate"
    ]
)


selected_gate = promotion[
    promotion[
        "policy_id"
    ].eq(
        SELECTED_POLICY
    )
]


assert len(
    selected_gate
) == 1


selected_gate = selected_gate.iloc[
    0
]


assert bool(
    selected_gate[
        "passes_promotion_gate"
    ]
)


assert bool(
    selected_gate[
        "aggregate_improved"
    ]
)


assert bool(
    selected_gate[
        "all_folds_nonnegative"
    ]
)


# ============================================================
# 9. EXACT SELECTED DEVELOPMENT RESULT
# ============================================================

aggregate = pd.DataFrame(
    d1[
        "aggregate_results"
    ]
)


selected_result = aggregate[
    aggregate[
        "policy_id"
    ].eq(
        SELECTED_POLICY
    )
]


assert len(
    selected_result
) == 1


selected_result = selected_result.iloc[
    0
]


assert int(
    selected_result[
        "actions"
    ]
) == 12_515


assert int(
    selected_result[
        "unique_edges_removed"
    ]
) == 385


assert int(
    selected_result[
        "net_edge_delta"
    ]
) == 12_130


assert int(
    selected_result[
        "delta_tp"
    ]
) == -3


assert int(
    selected_result[
        "delta_fp"
    ]
) == -10


assert int(
    selected_result[
        "delta_fn"
    ]
) == 3


assert float(
    selected_result[
        "delta_score"
    ]
) > 0


fold_results = pd.DataFrame(
    d1[
        "fold_results"
    ]
)


selected_folds = (
    fold_results[
        fold_results[
            "policy_id"
        ].eq(
            SELECTED_POLICY
        )
    ]
    .sort_values(
        "fold"
    )
    .reset_index(
        drop=True
    )
)


assert selected_folds[
    "fold"
].tolist() == [
    1,
    2,
    3,
    4,
]


assert selected_folds[
    "delta_score"
].ge(
    0
).all()


print(
    "\n========== G3H3D2 SELECTED DEVELOPMENT RESULT =========="
)


print(
    selected_folds[
        [
            "fold",
            "actions",
            "base_score",
            "policy_score",
            "delta_score",
        ]
    ].to_string(
        index=False
    )
)


print()


print(
    "aggregate score:",
    float(
        selected_result[
            "score"
        ]
    )
)


print(
    "aggregate delta:",
    float(
        selected_result[
            "delta_score"
        ]
    )
)


print(
    "edge delta TP/FP/FN:",
    int(
        selected_result[
            "delta_tp"
        ]
    ),
    "/",
    int(
        selected_result[
            "delta_fp"
        ]
    ),
    "/",
    int(
        selected_result[
            "delta_fn"
        ]
    ),
)


print(
    "DEVELOPMENT PROMOTION GATE: PASS"
)


# ============================================================
# 10. FREEZE EXACT DEVELOPMENT ACTION SUBSET
#
# Provenance only.
#
# Fold0 will regenerate actions from the frozen runtime rule,
# NOT reuse these development rows.
# ============================================================

dev_actions = pd.read_pickle(
    DEV_ACTIONS_PATH
)


assert len(
    dev_actions
) == 13_928


selected_dev_actions = (
    dev_actions[
        pd.to_numeric(
            dev_actions[
                "target_occupied"
            ],
            errors="raise",
        ).astype(
            int
        ).eq(
            0
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


assert len(
    selected_dev_actions
) == 12_515


assert int(
    pd.to_numeric(
        selected_dev_actions[
            "source_occupied"
        ],
        errors="raise",
    ).astype(
        int
    ).sum()
) == 385


assert int(
    pd.to_numeric(
        selected_dev_actions[
            "target_occupied"
        ],
        errors="raise",
    ).astype(
        int
    ).sum()
) == 0


selected_dev_actions.to_pickle(
    SELECTED_DEV_ACTIONS_OUT
)


SELECTED_DEV_ACTIONS_SHA = file_sha256(
    SELECTED_DEV_ACTIONS_OUT
)


print(
    "\n========== SELECTED DEV ACTION PROVENANCE =========="
)


print(
    "actions:",
    len(
        selected_dev_actions
    )
)


print(
    "SHA:",
    SELECTED_DEV_ACTIONS_SHA
)


print(
    "target occupied actions:",
    int(
        selected_dev_actions[
            "target_occupied"
        ].sum()
    )
)


# ============================================================
# 11. LOAD FULL-DEV PREPROCESSOR + MODEL
# ============================================================

prep = np.load(
    FULL_DEV_PREPROCESSOR_PATH,
    allow_pickle=False,
)


model = joblib.load(
    FULL_DEV_MODEL_PATH
)


artifact_feature_columns = prep[
    "feature_columns"
].tolist()


assert artifact_feature_columns == FEATURE_COLUMNS


medians = prep[
    "medians"
].astype(
    np.float64,
    copy=True,
)


means = prep[
    "means"
].astype(
    np.float64,
    copy=True,
)


stds = prep[
    "stds"
].astype(
    np.float64,
    copy=True,
)


assert len(
    medians
) == 32


assert len(
    means
) == 32


assert len(
    stds
) == 32


assert model.coef_.shape == (
    1,
    64,
)


# ============================================================
# 12. EXACT FULL-DEV TRANSFORM
# ============================================================

def transform_frozen_full_dev(
    frame,
):

    X = (
        frame[
            FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .to_numpy(
            dtype=np.float64,
            copy=True,
        )
    )


    X[
        ~np.isfinite(
            X
        )
    ] = np.nan


    missing = np.isnan(
        X
    )


    filled = np.where(
        missing,
        medians[
            None,
            :
        ],
        X,
    )


    scaled = (
        filled
        -
        means[
            None,
            :
        ]
    ) / stds[
        None,
        :
    ]


    design = np.concatenate(
        [
            scaled,
            missing.astype(
                np.float64
            ),
        ],
        axis=1,
    )


    assert design.shape[
        1
    ] == 64


    assert np.isfinite(
        design
    ).all()


    return design.astype(
        np.float32,
        copy=False,
    )


def score_frozen_full_dev(
    frame,
    batch_size=50_000,
):

    scores = np.empty(
        len(
            frame
        ),
        dtype=np.float64,
    )


    for start in range(
        0,
        len(
            frame
        ),
        batch_size,
    ):

        stop = min(
            start
            +
            batch_size,
            len(
                frame
            ),
        )


        X = transform_frozen_full_dev(
            frame.iloc[
                start:stop
            ]
        )


        scores[
            start:stop
        ] = (
            model.predict_proba(
                X
            )[
                :,
                1
            ]
        )


    assert np.isfinite(
        scores
    ).all()


    return scores


# ============================================================
# 13. GT-BLIND FULL-DEV Q9990 CALIBRATION
#
# IMPORTANT:
#
# Raw sample remains untouched.
#
# Do NOT remove known-positive overlap here.
#
# G3H3B quantile calibration was explicitly defined over the
# RAW G3H0 GT-blind hash sample, not pseudo-negative-only rows.
# ============================================================

raw_sample = pd.read_pickle(
    RAW_SAMPLE_PATH
)


assert len(
    raw_sample
) == EXPECTED_RAW_SAMPLE_ROWS


assert set(
    raw_sample[
        "fold"
    ].astype(
        int
    )
) == {
    1,
    2,
    3,
    4,
}


# Explicit Fold0 absence.
assert not raw_sample[
    "fold"
].eq(
    0
).any()


full_dev_sample_scores = score_frozen_full_dev(
    raw_sample
)


assert len(
    full_dev_sample_scores
) == EXPECTED_RAW_SAMPLE_ROWS


DEPLOYMENT_QUANTILE = 0.999


DEPLOYMENT_THRESHOLD = float(
    np.quantile(
        full_dev_sample_scores,
        DEPLOYMENT_QUANTILE,
        method="linear",
    )
)


assert np.isfinite(
    DEPLOYMENT_THRESHOLD
)


print(
    "\n========== FOLD0 DEPLOYMENT THRESHOLD CALIBRATION =========="
)


print(
    "scorer:",
    "FULL DEVELOPMENT FROZEN LOGREG"
)


print(
    "calibration rows:",
    len(
        raw_sample
    )
)


print(
    "calibration population:",
    "RAW GT-BLIND G3H0 HASH SAMPLE"
)


print(
    "Fold0 rows used:",
    0
)


print(
    "oracle labels used:",
    "NO"
)


print(
    "quantile:",
    DEPLOYMENT_QUANTILE
)


print(
    "deployment threshold:",
    DEPLOYMENT_THRESHOLD
)


print(
    "quantile retuned:",
    "NO"
)


# ============================================================
# 14. FREEZE DEPLOYMENT THRESHOLD
# ============================================================

RAW_SAMPLE_SHA = file_sha256(
    RAW_SAMPLE_PATH
)


DEPLOYMENT_THRESHOLD_CONTRACT = {
    "stage":
        "12.11G3H3D2",

    "purpose":
        "LOCKED_FOLD0_DEPLOYMENT_CALIBRATION",

    "scorer_id":
        "MULTIFRAME_REWIRE_LOGREG_SCORER_V1",

    "scorer_model_sha256":
        FULL_DEV_MODEL_SHA,

    "scorer_preprocessor_sha256":
        FULL_DEV_PREPROCESSOR_SHA,

    "calibration_sample":
        RAW_SAMPLE_PATH.name,

    "calibration_sample_sha256":
        RAW_SAMPLE_SHA,

    "calibration_rows":
        EXPECTED_RAW_SAMPLE_ROWS,

    "quantile_rule":
        "Q9990",

    "quantile":
        DEPLOYMENT_QUANTILE,

    "threshold":
        DEPLOYMENT_THRESHOLD,

    "quantile_was_selected_before_this_stage":
        True,

    "numeric_threshold_is_deployment_calibration":
        True,

    "threshold_selection_retuned":
        False,

    "oracle_labels_used":
        False,

    "fold0_rows_used":
        0,

    "fold0_used":
        False,
}


DEPLOYMENT_THRESHOLD_SHA = hashlib.sha256(
    json.dumps(
        DEPLOYMENT_THRESHOLD_CONTRACT,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


DEPLOYMENT_THRESHOLD_CONTRACT[
    "deployment_threshold_sha256"
] = DEPLOYMENT_THRESHOLD_SHA


DEPLOYMENT_THRESHOLD_OUT.write_text(
    json.dumps(
        DEPLOYMENT_THRESHOLD_CONTRACT,
        indent=2,
    ),
    encoding="utf-8",
)


assert DEPLOYMENT_THRESHOLD_OUT.exists()


# ============================================================
# 15. FINAL POLICY FREEZE
# ============================================================

FINAL_POLICY = {
    "stage":
        "12.11G3H3D2",

    "policy_id":
        FINAL_POLICY_ID,

    "status":
        "FINAL_DEVELOPMENT_POLICY_FROZEN",

    "development_selection": {
        "selected_structural_policy":
            "NO_TARGET_CUT",

        "structural_definition":
            "TARGET_OCCUPIED_EQ_0",

        "parent_structural_contract_sha256":
            d1_contract[
                "contract_sha256"
            ],

        "development_actions":
            12_515,

        "development_unique_incumbent_edges_removed":
            385,

        "development_net_edge_delta":
            12_130,

        "development_delta_tp":
            -3,

        "development_delta_fp":
            -10,

        "development_delta_fn":
            3,

        "development_score":
            float(
                selected_result[
                    "score"
                ]
            ),

        "development_delta_score":
            float(
                selected_result[
                    "delta_score"
                ]
            ),

        "development_fold_deltas":
            [
                {
                    "fold":
                        int(
                            row.fold
                        ),

                    "delta_score":
                        float(
                            row.delta_score
                        ),
                }

                for row
                in selected_folds.itertuples(
                    index=False
                )
            ],

        "predeclared_promotion_gate_passed":
            True,
    },

    "runtime_contract": {
        "scorer":
            "MULTIFRAME_REWIRE_LOGREG_SCORER_V1",

        "scorer_model_sha256":
            FULL_DEV_MODEL_SHA,

        "scorer_preprocessor_sha256":
            FULL_DEV_PREPROCESSOR_SHA,

        "feature_schema_sha256":
            FEATURE_SCHEMA_SHA,

        "endpoint_rank_rule":
            "ASYM_1x5",

        "endpoint_rank_rule_sha256":
            BACKBONE_SHA,

        "score_quantile_rule":
            "Q9990",

        "score_quantile":
            0.999,

        "score_quantile_contract_sha256":
            QUANTILE_CONTRACT_SHA,

        "deployment_threshold":
            DEPLOYMENT_THRESHOLD,

        "deployment_threshold_sha256":
            DEPLOYMENT_THRESHOLD_SHA,

        "conflict_resolution":
            "DETERMINISTIC_NODE_DISJOINT",

        "structural_filter":
            "TARGET_OCCUPIED_EQ_0",

        "graph_mutation":
            (
                "REMOVE_SOURCE_INCUMBENT_IF_PRESENT;"
                "TARGET_INCUMBENT_REMOVAL_FORBIDDEN;"
                "ADD_REWIRE_EDGE"
            ),

        "new_edge_dist":
            "FV9_RECOVERED_EDGE_DIST",

        "new_edge_prob":
            1.0,

        "new_edge_prob_semantics":
            "EPHEMERAL_SCHEMA_PLACEHOLDER",
    },

    "selected_dev_actions_artifact":
        SELECTED_DEV_ACTIONS_OUT.name,

    "selected_dev_actions_sha256":
        SELECTED_DEV_ACTIONS_SHA,

    "selection_is_development_gt_informed":
        True,

    "runtime_features_gt_blind":
        True,

    "development_closed_after_freeze":
        True,

    "future_scorer_retuning":
        False,

    "future_shortlist_retuning":
        False,

    "future_rank_rule_retuning":
        False,

    "future_score_quantile_retuning":
        False,

    "future_structural_rule_retuning":
        False,

    "fold0_used":
        False,

    "fold0_evaluated":
        False,

    "prediction_geff_written":
        False,

    "next":
        "ONE_SHOT_LOCKED_FOLD0_CONFIRMATION",
}


FINAL_POLICY_SHA = hashlib.sha256(
    json.dumps(
        FINAL_POLICY,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


FINAL_POLICY[
    "final_policy_sha256"
] = FINAL_POLICY_SHA


FINAL_POLICY_OUT.write_text(
    json.dumps(
        FINAL_POLICY,
        indent=2,
    ),
    encoding="utf-8",
)


assert FINAL_POLICY_OUT.exists()


# ============================================================
# 16. FORMAL FREEZE SUMMARY
# ============================================================

SUMMARY = {
    "stage":
        "12.11G3H3D2",

    "status":
        "DEVELOPMENT_CLOSED_POLICY_FROZEN",

    "final_policy_id":
        FINAL_POLICY_ID,

    "final_policy_sha256":
        FINAL_POLICY_SHA,

    "deployment_threshold":
        DEPLOYMENT_THRESHOLD,

    "deployment_threshold_sha256":
        DEPLOYMENT_THRESHOLD_SHA,

    "development_score":
        float(
            selected_result[
                "score"
            ]
        ),

    "development_delta_score":
        float(
            selected_result[
                "delta_score"
            ]
        ),

    "promotion_gate_passed":
        True,

    "development_closed":
        True,

    "no_further_development_tuning":
        True,

    "scorer_retuned":
        False,

    "shortlist_retuned":
        False,

    "rank_backbone_retuned":
        False,

    "score_quantile_retuned":
        False,

    "structural_rule_retuned_after_selection":
        False,

    "fold0_used":
        False,

    "fold0_evaluated":
        False,

    "next":
        "ONE_SHOT_LOCKED_FOLD0_CONFIRMATION",
}


SUMMARY_OUT.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 17. RELOAD INTEGRITY
# ============================================================

policy_check = json.loads(
    FINAL_POLICY_OUT.read_text(
        encoding="utf-8"
    )
)


threshold_check = json.loads(
    DEPLOYMENT_THRESHOLD_OUT.read_text(
        encoding="utf-8"
    )
)


summary_check = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert policy_check[
    "policy_id"
] == FINAL_POLICY_ID


assert policy_check[
    "final_policy_sha256"
] == FINAL_POLICY_SHA


assert policy_check[
    "development_closed_after_freeze"
] is True


assert policy_check[
    "fold0_used"
] is False


assert threshold_check[
    "quantile"
] == 0.999


assert threshold_check[
    "oracle_labels_used"
] is False


assert threshold_check[
    "fold0_rows_used"
] == 0


assert summary_check[
    "development_closed"
] is True


assert summary_check[
    "no_further_development_tuning"
] is True


assert summary_check[
    "fold0_used"
] is False


# ============================================================
# 18. FINAL DECISION
# ============================================================

print(
    "\n========== 12.11G3H3D2 DECISION =========="
)


print(
    "FINAL_DEVELOPMENT_POLICY_FREEZE: PASS"
)


print(
    "policy:",
    FINAL_POLICY_ID
)


print(
    "final policy SHA:",
    FINAL_POLICY_SHA
)


print(
    "structural rule:",
    "NO_TARGET_CUT / target_occupied == 0"
)


print(
    "scorer:",
    "MULTIFRAME_REWIRE_LOGREG_SCORER_V1"
)


print(
    "scorer model SHA:",
    FULL_DEV_MODEL_SHA
)


print(
    "rank rule:",
    "ASYM_1x5"
)


print(
    "score gate:",
    "Q9990"
)


print(
    "deployment threshold:",
    DEPLOYMENT_THRESHOLD
)


print(
    "deployment threshold SHA:",
    DEPLOYMENT_THRESHOLD_SHA
)


print(
    "development score:",
    float(
        selected_result[
            "score"
        ]
    )
)


print(
    "development delta:",
    float(
        selected_result[
            "delta_score"
        ]
    )
)


print(
    "promotion gate:",
    "PASS"
)


print(
    "development closed:",
    "YES"
)


print(
    "further development tuning allowed:",
    "NO"
)


print(
    "Fold0 data used:",
    "NO"
)


print(
    "Fold0 evaluated:",
    "NO"
)


print(
    "prediction GEFF written:",
    "NO"
)


print(
    "next:"
)


print(
    "ONE_SHOT_LOCKED_FOLD0_CONFIRMATION"
)

In [ ]:
# ============================================================
# Stage 12.11G3H3E0-P0
#
# Locked Fold0 CURRENT_BEST Builder Recovery
#
# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Development is CLOSED.
#
# Before the ONE-SHOT Fold0 confirmation, recover the exact
# Fold0 full-hybrid CURRENT_BEST builder.
#
# Why:
#
# Current kernel build_current_best() is the later folds1-4
# 6bba reconstruction and must NOT automatically be assumed
# equivalent to the Fold0 full-hybrid builder.
#
# Historical Fold0 CURRENT_BEST contract:
#
#   40 datasets total
#       25 x 6bba
#       15 x 44b6
#
# Route:
#
#   44b6 -> sparse9995 prediction route
#   6bba -> baseline256 prediction route
#
# Then frozen promoted Stage12 actions are applied according
# to the exact Fold0 contract.
#
# ------------------------------------------------------------
# THIS CELL DOES NOT
# ------------------------------------------------------------
#
#   - load GT graphs
#   - call evaluate_graph
#   - inspect Fold0 metric outcome
#   - generate final-policy Fold0 actions
#   - modify graph
#   - retune anything
#
# TEMPORARY PRE-FOLD0 RECOVERY ONLY.
#
# Run then send output.
# ============================================================


from pathlib import Path

import ast
import json
import re


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


SEARCH_ROOTS = [
    PROJECT / "scripts",
    PROJECT / "notebooks",
]


for root in SEARCH_ROOTS:
    assert root.exists(), root


# ============================================================
# 1. UNIQUE FOLD0 CURRENT_BEST SIGNATURES
# ============================================================

TOKENS = [
    "stage12_full_hybrid_mutual_cont_v2_r1p50_datasets.pkl",
    "R1P50_CURRENT_BEST_EDGE_RESIDUAL_CENSUS",
    "stage12_current_best_checkpoint.json",

    "def build_current_best",

    "S9995_995",
    "S9995_9975",

    "B256_995",
    "B256_9975",

    "r1p50_actions",
    "morph_actions",

    'cv["fold"] == 0',

    "44b6_",
    "6bba_",
]


HIGH_VALUE = {
    "stage12_full_hybrid_mutual_cont_v2_r1p50_datasets.pkl":
        20,

    "R1P50_CURRENT_BEST_EDGE_RESIDUAL_CENSUS":
        20,

    "def build_current_best":
        15,

    "S9995_995":
        10,

    "B256_995":
        10,

    "r1p50_actions":
        8,

    'cv["fold"] == 0':
        8,

    "44b6_":
        4,

    "6bba_":
        4,
}


# ============================================================
# 2. SOURCE UNITS
#
# A unit is:
#
#   - one .py file
#   - one code cell from one .ipynb
# ============================================================

units = []


# ------------------------------------------------------------
# Python files
# ------------------------------------------------------------

for root in SEARCH_ROOTS:

    for path in sorted(
        root.rglob(
            "*.py"
        )
    ):

        try:
            source = path.read_text(
                encoding="utf-8"
            )

        except Exception:
            continue


        units.append(
            {
                "kind":
                    "PY",

                "path":
                    str(
                        path
                    ),

                "cell":
                    None,

                "source":
                    source,
            }
        )


# ------------------------------------------------------------
# Notebook code cells
# ------------------------------------------------------------

for root in SEARCH_ROOTS:

    for path in sorted(
        root.rglob(
            "*.ipynb"
        )
    ):

        try:

            nb = json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:
            continue


        for idx, cell in enumerate(
            nb.get(
                "cells",
                []
            )
        ):

            if cell.get(
                "cell_type"
            ) != "code":

                continue


            source = "".join(
                cell.get(
                    "source",
                    []
                )
            )


            units.append(
                {
                    "kind":
                        "IPYNB",

                    "path":
                        str(
                            path
                        ),

                    "cell":
                        int(
                            idx
                        ),

                    "source":
                        source,
                }
            )


print(
    "========== FOLD0 BUILDER RECOVERY SEARCH =========="
)


print(
    "source units:",
    len(
        units
    )
)


# ============================================================
# 3. SCORE CANDIDATES
# ============================================================

candidate_rows = []


for unit_i, unit in enumerate(
    units
):

    source = unit[
        "source"
    ]


    matches = [
        token

        for token in TOKENS

        if token in source
    ]


    if not matches:
        continue


    score = sum(
        HIGH_VALUE.get(
            token,
            1,
        )

        for token in matches
    )


    # Strong bonus for exact required combination.
    if (
        "def build_current_best"
        in source
        and
        "S9995_995"
        in source
        and
        "B256_995"
        in source
    ):

        score += 50


    if (
        "stage12_full_hybrid_mutual_cont_v2_r1p50_datasets.pkl"
        in source
    ):

        score += 40


    candidate_rows.append(
        {
            "unit_i":
                unit_i,

            "score":
                score,

            "kind":
                unit[
                    "kind"
                ],

            "path":
                unit[
                    "path"
                ],

            "cell":
                unit[
                    "cell"
                ],

            "matches":
                matches,
        }
    )


candidate_rows = sorted(
    candidate_rows,
    key=lambda x: (
        -x[
            "score"
        ],
        x[
            "path"
        ],
        (
            -1
            if x[
                "cell"
            ] is None
            else x[
                "cell"
            ]
        ),
    ),
)


assert candidate_rows, (
    "No Fold0 CURRENT_BEST source candidate found."
)


print(
    "\n========== TOP SOURCE CANDIDATES =========="
)


for rank, item in enumerate(
    candidate_rows[:15],
    start=1,
):

    print()
    print(
        f"[{rank}] score={item['score']}"
    )

    print(
        "kind:",
        item[
            "kind"
        ]
    )

    print(
        "path:",
        item[
            "path"
        ]
    )

    print(
        "cell:",
        item[
            "cell"
        ]
    )

    print(
        "matches:",
        item[
            "matches"
        ]
    )


# ============================================================
# 4. SELECT BEST EXACT BUILDER CANDIDATE
# ============================================================

exact_candidates = [
    item

    for item in candidate_rows

    if (
        "def build_current_best"
        in units[
            item[
                "unit_i"
            ]
        ][
            "source"
        ]

        and

        "S9995_995"
        in units[
            item[
                "unit_i"
            ]
        ][
            "source"
        ]

        and

        "B256_995"
        in units[
            item[
                "unit_i"
            ]
        ][
            "source"
        ]

        and

        "44b6_"
        in units[
            item[
                "unit_i"
            ]
        ][
            "source"
        ]

        and

        "6bba_"
        in units[
            item[
                "unit_i"
            ]
        ][
            "source"
        ]
    )
]


assert exact_candidates, (
    "\nNo source unit contains the full Fold0 hybrid builder "
    "signature:\n"
    "  build_current_best\n"
    "  S9995_995\n"
    "  B256_995\n"
    "  44b6\n"
    "  6bba\n"
)


best_meta = exact_candidates[
    0
]


best = units[
    best_meta[
        "unit_i"
    ]
]


source = best[
    "source"
]


lines = source.splitlines()


print(
    "\n============================================================"
)


print(
    "SELECTED EXACT FOLD0 BUILDER SOURCE"
)


print(
    "============================================================"
)


print(
    "score:",
    best_meta[
        "score"
    ]
)


print(
    "kind:",
    best[
        "kind"
    ]
)


print(
    "path:",
    best[
        "path"
    ]
)


print(
    "cell:",
    best[
        "cell"
    ]
)


print(
    "lines:",
    len(
        lines
    )
)


# ============================================================
# 5. PRINT ALL HIGH-VALUE SOURCE LINES
# ============================================================

print(
    "\n========== HIGH-VALUE SOURCE LINES =========="
)


interesting_tokens = [
    "CURRENT_BEST",
    "cv0",
    'cv["fold"]',
    "names_44",
    "names_6",

    "S9995_995",
    "S9995_9975",

    "B256_995",
    "B256_9975",

    "MORPH",
    "morph_actions",

    "V1",
    "V2",

    "R1P50",
    "r1p50_actions",

    "ACTION_PATH",

    "build_current_best",

    "apply_frozen_v9",

    "add_continuation",

    "add_edge(",

    "recovered_edge_dist",
]


hit_line_numbers = []


for line_no, line in enumerate(
    lines,
    start=1,
):

    if any(
        token.lower()
        in line.lower()

        for token in interesting_tokens
    ):

        hit_line_numbers.append(
            line_no
        )


for line_no in hit_line_numbers:

    print(
        f"{line_no:4d}: "
        f"{lines[line_no - 1]}"
    )


# ============================================================
# 6. AST FUNCTION EXTRACTION
# ============================================================

try:

    tree = ast.parse(
        source
    )

except SyntaxError as exc:

    tree = None

    print(
        "\nAST PARSE FAILED:",
        repr(
            exc
        )
    )


function_sources = {}


if tree is not None:

    for node in tree.body:

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        ):

            segment = ast.get_source_segment(
                source,
                node,
            )


            if segment:

                function_sources[
                    node.name
                ] = segment


print(
    "\n========== FUNCTION INVENTORY =========="
)


for name in function_sources:

    if any(
        token in name.lower()

        for token in [
            "current",
            "graph",
            "continuation",
            "state",
            "action",
            "edge",
        ]
    ):

        print(
            name
        )


# ============================================================
# 7. EXACT build_current_best()
# ============================================================

assert "build_current_best" in function_sources, (
    "build_current_best AST source not recovered."
)


print(
    "\n============================================================"
)


print(
    "EXACT FOLD0 build_current_best()"
)


print(
    "============================================================"
)


print(
    function_sources[
        "build_current_best"
    ]
)


# ============================================================
# 8. FUNCTIONS CALLED BY build_current_best
# ============================================================

builder_tree = ast.parse(
    function_sources[
        "build_current_best"
    ]
)


called_names = set()


for node in ast.walk(
    builder_tree
):

    if isinstance(
        node,
        ast.Call,
    ):

        fn = node.func


        if isinstance(
            fn,
            ast.Name,
        ):

            called_names.add(
                fn.id
            )


        elif isinstance(
            fn,
            ast.Attribute,
        ):

            called_names.add(
                fn.attr
            )


print(
    "\n========== BUILDER CALL DEPENDENCIES =========="
)


print(
    sorted(
        called_names
    )
)


# ============================================================
# 9. PRINT LOCAL HELPER DEFINITIONS USED BY BUILDER
# ============================================================

print(
    "\n============================================================"
)


print(
    "LOCAL BUILDER HELPER SOURCES"
)


print(
    "============================================================"
)


printed_helper = False


for name in sorted(
    called_names
):

    if name not in function_sources:
        continue


    if name == "build_current_best":
        continue


    printed_helper = True


    print()
    print(
        "-" * 80
    )

    print(
        name
    )

    print(
        "-" * 80
    )

    print(
        function_sources[
            name
        ]
    )


if not printed_helper:

    print(
        "NO_LOCAL_HELPER_DEFINITIONS_FOUND"
    )


# ============================================================
# 10. PRINT PATH / ACTION-CONTRACT WINDOWS
#
# Print merged windows around:
#
#   prediction route paths
#   frozen action paths
#   Fold0 corpus
#
# ============================================================

WINDOW_PATTERNS = [
    "S9995_995",
    "S9995_9975",

    "B256_995",
    "B256_9975",

    "MORPH_ACTION_PATH",
    "V1",
    "V2",
    "R1P50_ACTION_PATH",

    "morph_actions =",
    "r1p50_actions =",

    'cv["fold"] == 0',

    "names_44",
    "names_6",
]


raw_windows = []


for i, line in enumerate(
    lines
):

    if any(
        pattern.lower()
        in line.lower()

        for pattern in WINDOW_PATTERNS
    ):

        raw_windows.append(
            (
                max(
                    0,
                    i - 12,
                ),

                min(
                    len(
                        lines
                    ),
                    i + 20,
                ),
            )
        )


raw_windows.sort()


merged = []


for start, end in raw_windows:

    if (
        not merged
        or
        start
        >
        merged[
            -1
        ][
            1
        ]
        +
        3
    ):

        merged.append(
            [
                start,
                end,
            ]
        )


    else:

        merged[
            -1
        ][
            1
        ] = max(
            merged[
                -1
            ][
                1
            ],
            end,
        )


print(
    "\n============================================================"
)


print(
    "EXACT PATH / CORPUS / ACTION CONTRACT WINDOWS"
)


print(
    "============================================================"
)


for start, end in merged:

    print()
    print(
        "-" * 80
    )

    print(
        f"LINES {start + 1} - {end}"
    )

    print(
        "-" * 80
    )


    for j in range(
        start,
        end,
    ):

        print(
            f"{j + 1:4d}: "
            f"{lines[j]}"
        )


# ============================================================
# 11. VERIFY NO GT / EVALUATION HAS EXECUTED
# ============================================================

print(
    "\n========== SAFETY STATUS =========="
)


print(
    "GT graph loaded:",
    "NO"
)


print(
    "fv9.evaluate_graph called:",
    "NO"
)


print(
    "Fold0 metric observed:",
    "NO"
)


print(
    "Fold0 final-policy actions generated:",
    "NO"
)


print(
    "graph modified:",
    "NO"
)


print(
    "development policy changed:",
    "NO"
)


# ============================================================
# 12. FINAL
# ============================================================

print(
    "\n========== 12.11G3H3E0-P0 DECISION =========="
)


print(
    "EXACT_FOLD0_CURRENT_BEST_BUILDER_RECOVERY: COMPLETE"
)


print(
    "selected source:",
    best[
        "path"
    ]
)


print(
    "selected cell:",
    best[
        "cell"
    ]
)


print(
    "full-hybrid route signature:",
    "RECOVERED"
)


print(
    "Fold0 GT touched:",
    "NO"
)


print(
    "Fold0 metric touched:",
    "NO"
)


print(
    "development remains closed:",
    "YES"
)


print(
    "next:"
)


print(
    "ONE_SHOT_LOCKED_FOLD0_CONFIRMATION_USING_EXACT_RECOVERED_BUILDER"
)

In [ ]:
# ============================================================
# Stage 12.11G3H3E0
#
# ONE-SHOT LOCKED FOLD0 CONFIRMATION
#
# ============================================================
#
# FINAL DEVELOPMENT-FROZEN POLICY
#
#   ASYM1x5_Q9990_NO_TARGET_CUT_V1
#
# final policy SHA:
#   77f0b07cafb21d0705917ff71b750fde45f3c7ddc17c13fecaff0ac673ebf776
#
# frozen scorer:
#   MULTIFRAME_REWIRE_LOGREG_SCORER_V1
#
# model SHA:
#   00918fa386d4de6e6a5f93892e47d876e6ff45ceb89670f982ebba45f59ebf76
#
# frozen rank backbone:
#   ASYM_1x5
#
# frozen score quantile:
#   Q9990
#
# full-development deployment threshold:
#   0.9952878577709198
#
# deployment threshold SHA:
#   a06eaf0040c57085b5eb825a327d905274e6771db3075208533b9da8f4a88e00
#
# frozen structural rule:
#   target_occupied == 0
#
# ------------------------------------------------------------
# ONE-SHOT PROTOCOL
# ------------------------------------------------------------
#
# PART A:
#
#   recover exact Fold0 CURRENT_BEST builder
#   WITHOUT GT
#
# PART B:
#
#   generate Fold0 actions on 25 Fold0 6bba datasets
#
#       exact CURRENT_BEST
#           ->
#       frozen proposal generator
#           ->
#       frozen shortlist
#           ->
#       full-dev frozen LogReg
#           ->
#       ASYM1x5
#           ->
#       threshold >= 0.9952878577709198
#           ->
#       deterministic node-disjoint resolver
#           ->
#       target_occupied == 0
#
#   Freeze Fold0 action artifact BEFORE GT.
#
# PART C:
#
#   only after action artifact is frozen:
#
#       evaluate exact CURRENT_BEST
#       evaluate final frozen policy
#
#   on ALL 40 Fold0 datasets:
#
#       15 x 44b6  (unchanged)
#       25 x 6bba  (policy eligible)
#
# ------------------------------------------------------------
# PRIMARY CONFIRMATION RULE — FROZEN BEFORE GT
# ------------------------------------------------------------
#
#   CONFIRMED iff:
#
#       full-40 Fold0 policy score
#           >
#       full-40 Fold0 CURRENT_BEST score
#
# Baseline must first exactly reproduce:
#
#   TP / FP / FN:
#       22806 / 2658 / 3425
#
#   score:
#       0.7863444776365197
#
# If baseline fidelity fails:
#
#   Fold0 policy result is INVALID.
#
# ------------------------------------------------------------
# NO:
#
#   scorer retuning
#   threshold retuning
#   rank-rule retuning
#   structural-rule retuning
#   post-Fold0 policy search
#   prediction GEFF writes
#
# ------------------------------------------------------------
# Keep after running: YES
# ============================================================


from pathlib import Path

import ast
import gc
import hashlib
import json

import joblib

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


TRAIN_ROOT = (
    PROJECT
    / "data"
    / "raw"
    / "competition"
    / "biohub-cell-tracking-during-development"
    / "train"
)


CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


# ------------------------------------------------------------
# Exact authoritative Fold0 CURRENT_BEST source recovered
# in G3H3E0-P0.
# ------------------------------------------------------------

FOLD0_BUILDER_SOURCE = (
    PROJECT
    / "scripts"
    / "run_stage12_current_best_edge_residual_census.py"
)


# ------------------------------------------------------------
# Final frozen development policy
# ------------------------------------------------------------

FINAL_POLICY_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_frozen.json"
)


DEPLOYMENT_THRESHOLD_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_deployment_threshold.json"
)


FREEZE_SUMMARY_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_freeze_summary.json"
)


# ------------------------------------------------------------
# Frozen scorer
# ------------------------------------------------------------

FEATURE_SCHEMA_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_feature_schema_v1_frozen.json"
)


FULL_DEV_PREPROCESSOR_PATH = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_preprocessor.npz"
)


FULL_DEV_MODEL_PATH = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_model.joblib"
)


# ------------------------------------------------------------
# Formal Fold0 outputs
# ------------------------------------------------------------

CONFIRMATION_CONTRACT_OUT = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_confirmation_contract_frozen.json"
)


FOLD0_ACTIONS_OUT = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_actions_locked.pkl"
)


FOLD0_ACTION_SUMMARY_OUT = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_action_summary.json"
)


FOLD0_METRICS_OUT = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_confirmation_metrics.pkl"
)


FOLD0_SUMMARY_OUT = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_confirmation_summary.json"
)


for path in [
    TRAIN_ROOT,
    CV_PATH,
    FOLD0_BUILDER_SOURCE,

    FINAL_POLICY_PATH,
    DEPLOYMENT_THRESHOLD_PATH,
    FREEZE_SUMMARY_PATH,

    FEATURE_SCHEMA_PATH,
    FULL_DEV_PREPROCESSOR_PATH,
    FULL_DEV_MODEL_PATH,
]:

    assert path.exists(), path


for path in [
    CONFIRMATION_CONTRACT_OUT,
    FOLD0_ACTIONS_OUT,
    FOLD0_ACTION_SUMMARY_OUT,
    FOLD0_METRICS_OUT,
    FOLD0_SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nFold0 confirmation output already exists:\n"
            f"{path}\n\n"
            "This is a ONE-SHOT locked test. "
            "Do not overwrite or rerun it."
        )


# ============================================================
# 1. AUTHORITATIVE CONSTANTS
# ============================================================

FINAL_POLICY_ID = (
    "ASYM1x5_Q9990_NO_TARGET_CUT_V1"
)


FINAL_POLICY_SHA = (
    "77f0b07cafb21d0705917ff71b750fde4"
    "5f3c7ddc17c13fecaff0ac673ebf776"
)


DEPLOYMENT_THRESHOLD = (
    0.9952878577709198
)


DEPLOYMENT_THRESHOLD_SHA = (
    "a06eaf0040c57085b5eb825a327d90527"
    "4e6771db3075208533b9da8f4a88e00"
)


MODEL_SHA = (
    "00918fa386d4de6e6a5f93892e47d876"
    "e6ff45ceb89670f982ebba45f59ebf76"
)


PREPROCESSOR_SHA = (
    "6b5425093efd17613c68fcba855a756a3"
    "445a286a61ac813a83652dd2200ae41"
)


FEATURE_SCHEMA_SHA = (
    "0a9f8e9a41c5c80c803c3a95955702dc"
    "f4039605a21fc656a154aa05be55b39c"
)


EXPECTED_FOLD0_DATASETS = 40

EXPECTED_FOLD0_44 = 15

EXPECTED_FOLD0_6 = 25


EXPECTED_BASE_TP = 22_806

EXPECTED_BASE_FP = 2_658

EXPECTED_BASE_FN = 3_425

EXPECTED_BASE_SCORE = (
    0.7863444776365197
)


# ============================================================
# 2. HELPERS
# ============================================================

def file_sha256(
    path,
):

    h = hashlib.sha256()


    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024
                *
                1024
            )


            if not chunk:

                break


            h.update(
                chunk
            )


    return h.hexdigest()


def local_graph_state(
    graph,
):

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )


    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    outdeg = (
        edges.groupby(
            "source_id"
        )
        .size()
        .astype(
            int
        )
        .to_dict()
    )


    indeg = (
        edges.groupby(
            "target_id"
        )
        .size()
        .astype(
            int
        )
        .to_dict()
    )


    edge_set = set(
        zip(
            edges[
                "source_id"
            ].astype(
                int
            ),

            edges[
                "target_id"
            ].astype(
                int
            ),
        )
    )


    return (
        nodes,
        {
            int(k):
                int(v)
            for k, v
            in outdeg.items()
        },
        {
            int(k):
                int(v)
            for k, v
            in indeg.items()
        },
        edge_set,
    )


# ============================================================
# 3. VERIFY FINAL DEVELOPMENT FREEZE
# ============================================================

final_policy = json.loads(
    FINAL_POLICY_PATH.read_text(
        encoding="utf-8"
    )
)


threshold_contract = json.loads(
    DEPLOYMENT_THRESHOLD_PATH.read_text(
        encoding="utf-8"
    )
)


freeze_summary = json.loads(
    FREEZE_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert final_policy[
    "policy_id"
] == FINAL_POLICY_ID


assert final_policy[
    "final_policy_sha256"
] == FINAL_POLICY_SHA


assert final_policy[
    "development_closed_after_freeze"
] is True


assert final_policy[
    "fold0_used"
] is False


assert final_policy[
    "fold0_evaluated"
] is False


assert float(
    final_policy[
        "runtime_contract"
    ][
        "deployment_threshold"
    ]
) == DEPLOYMENT_THRESHOLD


assert threshold_contract[
    "deployment_threshold_sha256"
] == DEPLOYMENT_THRESHOLD_SHA


assert float(
    threshold_contract[
        "threshold"
    ]
) == DEPLOYMENT_THRESHOLD


assert threshold_contract[
    "quantile"
] == 0.999


assert threshold_contract[
    "fold0_rows_used"
] == 0


assert threshold_contract[
    "oracle_labels_used"
] is False


assert freeze_summary[
    "development_closed"
] is True


assert freeze_summary[
    "no_further_development_tuning"
] is True


print(
    "========== G3H3E0 FINAL POLICY LOCK =========="
)


print(
    "policy:",
    FINAL_POLICY_ID
)


print(
    "policy SHA:",
    FINAL_POLICY_SHA
)


print(
    "threshold:",
    DEPLOYMENT_THRESHOLD
)


print(
    "development closed:",
    "YES"
)


print(
    "Fold0 previously evaluated:",
    "NO"
)


# ============================================================
# 4. VERIFY FULL-DEV SCORER
# ============================================================

assert file_sha256(
    FULL_DEV_MODEL_PATH
) == MODEL_SHA


assert file_sha256(
    FULL_DEV_PREPROCESSOR_PATH
) == PREPROCESSOR_SHA


feature_schema = json.loads(
    FEATURE_SCHEMA_PATH.read_text(
        encoding="utf-8"
    )
)


assert feature_schema[
    "feature_schema_sha256"
] == FEATURE_SCHEMA_SHA


FEATURE_COLUMNS = list(
    feature_schema[
        "features"
    ]
)


assert len(
    FEATURE_COLUMNS
) == 32


prep = np.load(
    FULL_DEV_PREPROCESSOR_PATH,
    allow_pickle=False,
)


model = joblib.load(
    FULL_DEV_MODEL_PATH
)


assert prep[
    "feature_columns"
].tolist() == FEATURE_COLUMNS


medians = prep[
    "medians"
].astype(
    np.float64,
    copy=True,
)


means = prep[
    "means"
].astype(
    np.float64,
    copy=True,
)


stds = prep[
    "stds"
].astype(
    np.float64,
    copy=True,
)


assert model.coef_.shape == (
    1,
    64,
)


def transform_full_dev_locked(
    frame,
):

    X = (
        frame[
            FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .to_numpy(
            dtype=np.float64,
            copy=True,
        )
    )


    X[
        ~np.isfinite(
            X
        )
    ] = np.nan


    missing = np.isnan(
        X
    )


    filled = np.where(
        missing,
        medians[
            None,
            :
        ],
        X,
    )


    scaled = (
        filled
        -
        means[
            None,
            :
        ]
    ) / stds[
        None,
        :
    ]


    design = np.concatenate(
        [
            scaled,
            missing.astype(
                np.float64
            ),
        ],
        axis=1,
    )


    assert design.shape[
        1
    ] == 64


    assert np.isfinite(
        design
    ).all()


    return design.astype(
        np.float32,
        copy=False,
    )


def score_full_dev_locked(
    frame,
    batch_size=50_000,
):

    scores = np.empty(
        len(
            frame
        ),
        dtype=np.float64,
    )


    for start in range(
        0,
        len(
            frame
        ),
        batch_size,
    ):

        stop = min(
            start
            +
            batch_size,
            len(
                frame
            ),
        )


        X = transform_full_dev_locked(
            frame.iloc[
                start:stop
            ]
        )


        scores[
            start:stop
        ] = (
            model.predict_proba(
                X
            )[
                :,
                1
            ]
        )


    assert np.isfinite(
        scores
    ).all()


    return scores


# ============================================================
# 5. FOLD0 CORPUS — STILL NO GT
# ============================================================

cv = pd.read_csv(
    CV_PATH
)


cv0 = (
    cv[
        cv[
            "fold"
        ].eq(
            0
        )
    ]
    .sort_values(
        [
            "prefix",
            "dataset",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    cv0
) == EXPECTED_FOLD0_DATASETS


names = cv0[
    "dataset"
].tolist()


names_44 = (
    cv0.loc[
        cv0[
            "dataset"
        ].str.startswith(
            "44b6_"
        ),
        "dataset",
    ]
    .tolist()
)


names_6 = (
    cv0.loc[
        cv0[
            "dataset"
        ].str.startswith(
            "6bba_"
        ),
        "dataset",
    ]
    .tolist()
)


assert len(
    names_44
) == EXPECTED_FOLD0_44


assert len(
    names_6
) == EXPECTED_FOLD0_6


estimated_nodes = dict(
    zip(
        cv0[
            "dataset"
        ],

        cv0[
            "estimated_nodes"
        ].astype(
            float
        ),
    )
)


print(
    "\n========== LOCKED FOLD0 CORPUS =========="
)


print(
    "all:",
    len(
        names
    )
)


print(
    "44b6:",
    len(
        names_44
    )
)


print(
    "6bba:",
    len(
        names_6
    )
)


print(
    "GT loaded:",
    "NO"
)


# ============================================================
# 6. RECOVER EXACT FOLD0 CURRENT_BEST BUILDER
#
# Execute only the source PREFIX through the end of the exact
# build_current_best() definition.
#
# Everything below the builder in the original residual-census
# script — including official evaluation — is NOT executed.
# ============================================================

source = FOLD0_BUILDER_SOURCE.read_text(
    encoding="utf-8"
)


tree = ast.parse(
    source
)


builder_node = None


for node in tree.body:

    if (
        isinstance(
            node,
            ast.FunctionDef,
        )
        and
        node.name
        ==
        "build_current_best"
    ):

        builder_node = node

        break


assert builder_node is not None


assert builder_node.end_lineno is not None


source_lines = source.splitlines()


safe_prefix = "\n".join(
    source_lines[
        :
        builder_node.end_lineno
    ]
)


# ------------------------------------------------------------
# Hard safety gate: exact recovered prefix must not perform GT
# evaluation.
# ------------------------------------------------------------

assert "fv9.evaluate_graph(" not in safe_prefix

assert "official_evaluate(" not in safe_prefix

assert "fv9.load_graph(" not in safe_prefix


fold0_builder_ns = {
    "__name__":
        "__g3h3e0_fold0_builder__",

    "__file__":
        str(
            FOLD0_BUILDER_SOURCE
        ),
}


exec(
    compile(
        safe_prefix,
        str(
            FOLD0_BUILDER_SOURCE
        ),
        "exec",
    ),
    fold0_builder_ns,
    fold0_builder_ns,
)


assert (
    "build_current_best"
    in
    fold0_builder_ns
)


fold0_build_current_best = (
    fold0_builder_ns[
        "build_current_best"
    ]
)


assert callable(
    fold0_build_current_best
)


# Exact recovered corpus must agree.
assert set(
    fold0_builder_ns[
        "names_44"
    ]
) == set(
    names_44
)


assert set(
    fold0_builder_ns[
        "names_6"
    ]
) == set(
    names_6
)


print(
    "\n========== EXACT FOLD0 CURRENT_BEST BUILDER =========="
)


print(
    "source:",
    FOLD0_BUILDER_SOURCE
)


print(
    "safe source prefix lines:",
    builder_node.end_lineno
)


print(
    "44b6 route:",
    "Sparse9995 / Frozen-V9 unchanged"
)


print(
    "6bba route:",
    (
        "Baseline256 + MORPH-V2 + "
        "V1 + V2 + R1P50"
    )
)


print(
    "GT touched during recovery:",
    "NO"
)


print(
    "EXACT_FOLD0_BUILDER_RECOVERY: PASS"
)


# ============================================================
# 7. VERIFY FROZEN GENERATOR HELPERS
# ============================================================

REQUIRED_GENERATOR_HELPERS = [
    "generate_dataset_proposals",
    "add_endpoint_ranks",
    "add_shortlist_rank_fields",
    "frozen_shortlist_mask",
    "attach_scoring_features",

    "endpoint_score_ranks_g3h3",
    "policy_mask_g3h3",
    "resolve_conflicts_g3h3",
]


missing = [
    name

    for name
    in REQUIRED_GENERATOR_HELPERS

    if (
        name not in globals()
        or
        not callable(
            globals()[
                name
            ]
        )
    )
]


assert not missing, (
    "\nFrozen development generator helpers missing:\n"
    f"{missing}"
)


# ============================================================
# 8. FREEZE ONE-SHOT CONFIRMATION CONTRACT
#
# IMPORTANT:
# This is written BEFORE Fold0 action generation and BEFORE GT.
# ============================================================

CONFIRMATION_CONTRACT = {
    "stage":
        "12.11G3H3E0",

    "contract_id":
        "ONE_SHOT_LOCKED_FOLD0_CONFIRMATION_V1",

    "final_policy_id":
        FINAL_POLICY_ID,

    "final_policy_sha256":
        FINAL_POLICY_SHA,

    "fold0_current_best_builder_source":
        str(
            FOLD0_BUILDER_SOURCE.relative_to(
                PROJECT
            )
        ),

    "fold0_corpus": {
        "datasets":
            40,

        "44b6":
            15,

        "6bba":
            25,
    },

    "runtime_pipeline": [
        "EXACT_FOLD0_CURRENT_BEST",
        "FROZEN_MULTIFRAME_PROPOSAL_GENERATOR",
        "FROZEN_SHORTLIST",
        "FULL_DEV_FROZEN_LOGREG",
        "ASYM_SCORE_RANK_1x5",
        "Q9990_DEPLOYMENT_THRESHOLD",
        "DETERMINISTIC_NODE_DISJOINT_RESOLVER",
        "TARGET_OCCUPIED_EQ_0",
    ],

    "deployment_threshold":
        DEPLOYMENT_THRESHOLD,

    "structural_rule":
        "TARGET_OCCUPIED_EQ_0",

    "mutation_rule":
        (
            "REMOVE_SOURCE_INCUMBENT_IF_PRESENT;"
            "NEVER_REMOVE_TARGET_INCUMBENT;"
            "ADD_REWIRE_EDGE"
        ),

    "primary_confirmation_metric":
        "FULL_40_DATASET_OFFICIAL_SCORE",

    "confirmation_rule":
        "POLICY_SCORE_STRICTLY_GREATER_THAN_CURRENT_BEST_SCORE",

    "baseline_fidelity": {
        "edge_tp":
            EXPECTED_BASE_TP,

        "edge_fp":
            EXPECTED_BASE_FP,

        "edge_fn":
            EXPECTED_BASE_FN,

        "score":
            EXPECTED_BASE_SCORE,
    },

    "action_artifact_must_be_frozen_before_gt":
        True,

    "development_closed":
        True,

    "post_fold0_retuning_allowed":
        False,

    "fold0_metric_seen_before_contract_freeze":
        False,
}


CONFIRMATION_CONTRACT_SHA = hashlib.sha256(
    json.dumps(
        CONFIRMATION_CONTRACT,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


CONFIRMATION_CONTRACT[
    "confirmation_contract_sha256"
] = CONFIRMATION_CONTRACT_SHA


CONFIRMATION_CONTRACT_OUT.write_text(
    json.dumps(
        CONFIRMATION_CONTRACT,
        indent=2,
    ),
    encoding="utf-8",
)


assert CONFIRMATION_CONTRACT_OUT.exists()


print(
    "\n========== ONE-SHOT CONTRACT FROZEN =========="
)


print(
    "SHA:",
    CONFIRMATION_CONTRACT_SHA
)


print(
    "confirmation rule:",
    CONFIRMATION_CONTRACT[
        "confirmation_rule"
    ]
)


print(
    "Fold0 metric seen:",
    "NO"
)


print(
    "post-Fold0 retuning allowed:",
    "NO"
)


# ============================================================
# 9. TEMPORARILY ROUTE FROZEN GENERATOR TO THE EXACT FOLD0
#    CURRENT_BEST BUILDER
#
# Existing proposal functions were developed using a global
# CURRENT_BEST builder dependency.
#
# Patch only during Fold0 GT-blind action generation, then
# restore immediately.
# ============================================================

generator_globals = (
    generate_dataset_proposals.__globals__
)


PATCH_KEYS = [
    "build_current_best",
    "current_best_builder",
    "fold_map",
]


saved_globals = {}


for key in PATCH_KEYS:

    if key in generator_globals:

        saved_globals[
            key
        ] = (
            True,
            generator_globals[
                key
            ],
        )

    else:

        saved_globals[
            key
        ] = (
            False,
            None,
        )


# Make both common exact-builder names resolve correctly.
generator_globals[
    "build_current_best"
] = fold0_build_current_best


generator_globals[
    "current_best_builder"
] = fold0_build_current_best


# If generator references fold_map, only Fold0 6bba are valid
# during this action-generation phase.
generator_globals[
    "fold_map"
] = {
    name:
        0

    for name
    in names_6
}


print(
    "\n========== FOLD0 GENERATOR ROUTING =========="
)


print(
    "generate_dataset_proposals co_names contains build_current_best:",
    (
        "build_current_best"
        in
        generate_dataset_proposals.__code__.co_names
    )
)


print(
    "co_names contains current_best_builder:",
    (
        "current_best_builder"
        in
        generate_dataset_proposals.__code__.co_names
    )
)


print(
    "exact Fold0 builder injected:",
    "YES"
)


# ============================================================
# 10. GT-BLIND FOLD0 ACTION GENERATION
# ============================================================

print(
    "\n============================================================"
)


print(
    "PART A — LOCKED FOLD0 ACTION GENERATION"
)


print(
    "============================================================"
)


action_frames = []

dataset_action_rows = []

total_shortlist = 0

total_rank_gate = 0

total_score_gate = 0

total_resolved = 0


try:

    for i, dataset in enumerate(
        names_6,
        start=1,
    ):

        # ----------------------------------------------------
        # Exact frozen proposal universe
        # ----------------------------------------------------

        result = generate_dataset_proposals(
            dataset
        )


        assert isinstance(
            result,
            tuple,
        )


        proposals = result[
            0
        ]


        proposals = add_endpoint_ranks(
            proposals
        )


        proposals = add_shortlist_rank_fields(
            proposals
        )


        shortlist = (
            proposals[
                frozen_shortlist_mask(
                    proposals
                )
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


        shortlist = attach_scoring_features(
            shortlist
        )


        total_shortlist += int(
            len(
                shortlist
            )
        )


        # ----------------------------------------------------
        # Full-development frozen scorer
        # ----------------------------------------------------

        shortlist[
            "model_score"
        ] = score_full_dev_locked(
            shortlist
        )


        # ----------------------------------------------------
        # Exact endpoint score ranks
        # ----------------------------------------------------

        ranks = endpoint_score_ranks_g3h3(
            shortlist
        )


        shortlist[
            "source_score_rank"
        ] = ranks[
            "source_score_rank"
        ].to_numpy(
            copy=True
        )


        shortlist[
            "target_score_rank"
        ] = ranks[
            "target_score_rank"
        ].to_numpy(
            copy=True
        )


        # ----------------------------------------------------
        # Frozen ASYM1x5
        # ----------------------------------------------------

        rank_mask = policy_mask_g3h3(
            shortlist,
            a=1,
            b=5,
        )


        rank_gated = shortlist[
            rank_mask
        ].copy()


        total_rank_gate += int(
            len(
                rank_gated
            )
        )


        # ----------------------------------------------------
        # Frozen full-development Q9990 threshold
        # ----------------------------------------------------

        score_gated = rank_gated[
            rank_gated[
                "model_score"
            ].ge(
                DEPLOYMENT_THRESHOLD
            )
        ].copy()


        total_score_gate += int(
            len(
                score_gated
            )
        )


        # ----------------------------------------------------
        # Frozen resolver FIRST.
        #
        # Structural filter is intentionally applied AFTER
        # resolver because that is exactly how D1 selected
        # NO_TARGET_CUT.
        # ----------------------------------------------------

        resolved = (
            resolve_conflicts_g3h3(
                score_gated
            )
            .copy()
            .reset_index(
                drop=True
            )
        )


        total_resolved += int(
            len(
                resolved
            )
        )


        # ----------------------------------------------------
        # Frozen structural policy:
        #
        #   NO_TARGET_CUT
        #   target_occupied == 0
        # ----------------------------------------------------

        target_occupied = pd.to_numeric(
            resolved[
                "target_occupied"
            ],
            errors="raise",
        ).astype(
            int
        )


        selected = (
            resolved[
                target_occupied.eq(
                    0
                )
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


        assert pd.to_numeric(
            selected[
                "target_occupied"
            ],
            errors="raise",
        ).astype(
            int
        ).eq(
            0
        ).all()


        keep_cols = [
            "source_id",
            "target_id",

            "gap_size",
            "delta_t",

            "cuts",

            "source_occupied",
            "target_occupied",

            "GMEAN_RATIO",

            "source_cv_pf",
            "target_cv_pf",

            "source_incumbent_cv_pf",
            "target_incumbent_cv_pf",

            "source_relative_improvement",
            "target_relative_improvement",

            "source_score_rank",
            "target_score_rank",

            "model_score",
        ]


        action = selected[
            keep_cols
        ].copy()


        action.insert(
            0,
            "dataset",
            dataset,
        )


        action.insert(
            1,
            "fold",
            0,
        )


        action[
            "score_threshold"
        ] = DEPLOYMENT_THRESHOLD


        action[
            "policy_id"
        ] = FINAL_POLICY_ID


        action_frames.append(
            action
        )


        dataset_action_rows.append(
            {
                "dataset":
                    dataset,

                "shortlist":
                    int(
                        len(
                            shortlist
                        )
                    ),

                "rank_gate":
                    int(
                        len(
                            rank_gated
                        )
                    ),

                "score_gate":
                    int(
                        len(
                            score_gated
                        )
                    ),

                "resolved":
                    int(
                        len(
                            resolved
                        )
                    ),

                "selected":
                    int(
                        len(
                            selected
                        )
                    ),

                "selected_source_occupied":
                    int(
                        pd.to_numeric(
                            selected[
                                "source_occupied"
                            ],
                            errors="raise",
                        )
                        .astype(
                            int
                        )
                        .sum()
                    ),
            }
        )


        print(
            f"{i:2d}/25"
            f" | {dataset}"
            f" | shortlist={len(shortlist)}"
            f" | rank={len(rank_gated)}"
            f" | score={len(score_gated)}"
            f" | resolved={len(resolved)}"
            f" | selected={len(selected)}"
        )


        del proposals
        del shortlist

        del ranks
        del rank_gated
        del score_gated

        del resolved
        del selected
        del action

        gc.collect()


finally:

    # --------------------------------------------------------
    # RESTORE NOTEBOOK GLOBAL STATE EVEN IF ACTION GENERATION
    # THROWS.
    # --------------------------------------------------------

    for key, (
        existed,
        value,
    ) in saved_globals.items():

        if existed:

            generator_globals[
                key
            ] = value

        else:

            generator_globals.pop(
                key,
                None,
            )


# ============================================================
# 11. FREEZE FOLD0 ACTION ARTIFACT — STILL NO GT
# ============================================================

if action_frames:

    fold0_actions = pd.concat(
        action_frames,
        ignore_index=True,
    )


else:

    fold0_actions = pd.DataFrame()


assert not fold0_actions[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


assert set(
    fold0_actions[
        "dataset"
    ]
).issubset(
    set(
        names_6
    )
)


assert pd.to_numeric(
    fold0_actions[
        "target_occupied"
    ],
    errors="raise",
).astype(
    int
).eq(
    0
).all()


# ------------------------------------------------------------
# Frozen resolver node-disjoint property must still hold after
# structural filtering.
# ------------------------------------------------------------

for dataset, sub in fold0_actions.groupby(
    "dataset",
    sort=False,
):

    endpoints = pd.concat(
        [
            sub[
                "source_id"
            ].astype(
                int
            ),

            sub[
                "target_id"
            ].astype(
                int
            ),
        ],
        ignore_index=True,
    )


    assert not endpoints.duplicated().any(), (
        dataset,
        "Fold0 selected actions are not node-disjoint"
    )


fold0_actions.to_pickle(
    FOLD0_ACTIONS_OUT
)


FOLD0_ACTIONS_SHA = file_sha256(
    FOLD0_ACTIONS_OUT
)


dataset_action_df = pd.DataFrame(
    dataset_action_rows
)


ACTION_SUMMARY = {
    "stage":
        "12.11G3H3E0",

    "part":
        "LOCKED_FOLD0_ACTION_FREEZE",

    "policy_id":
        FINAL_POLICY_ID,

    "confirmation_contract_sha256":
        CONFIRMATION_CONTRACT_SHA,

    "actions_sha256":
        FOLD0_ACTIONS_SHA,

    "fold0_6bba_datasets":
        25,

    "fold0_44b6_datasets":
        15,

    "shortlist_rows":
        int(
            total_shortlist
        ),

    "rank_gate_rows":
        int(
            total_rank_gate
        ),

    "score_gate_rows":
        int(
            total_score_gate
        ),

    "resolved_rows_before_structural_filter":
        int(
            total_resolved
        ),

    "final_actions":
        int(
            len(
                fold0_actions
            )
        ),

    "source_occupied_actions":
        int(
            pd.to_numeric(
                fold0_actions[
                    "source_occupied"
                ],
                errors="raise",
            )
            .astype(
                int
            )
            .sum()
        ),

    "target_occupied_actions":
        0,

    "actions_node_disjoint":
        True,

    "gt_loaded_before_action_freeze":
        False,

    "fold0_metric_seen_before_action_freeze":
        False,
}


FOLD0_ACTION_SUMMARY_OUT.write_text(
    json.dumps(
        ACTION_SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


assert FOLD0_ACTIONS_OUT.exists()

assert FOLD0_ACTION_SUMMARY_OUT.exists()


print(
    "\n========== LOCKED FOLD0 ACTION FREEZE =========="
)


print(
    "shortlist rows:",
    total_shortlist
)


print(
    "ASYM1x5 rows:",
    total_rank_gate
)


print(
    "Q9990 rows:",
    total_score_gate
)


print(
    "resolved before structural filter:",
    total_resolved
)


print(
    "final NO_TARGET_CUT actions:",
    len(
        fold0_actions
    )
)


print(
    "source-occupied actions:",
    int(
        fold0_actions[
            "source_occupied"
        ].sum()
    )
)


print(
    "target-occupied actions:",
    int(
        fold0_actions[
            "target_occupied"
        ].sum()
    )
)


print(
    "actions SHA:",
    FOLD0_ACTIONS_SHA
)


print(
    "GT loaded:",
    "NO"
)


print(
    "FOLD0_ACTION_FREEZE: PASS"
)


# ============================================================
# PART B
#
# ACTION ARTIFACT IS NOW FROZEN.
#
# GT MAY BE READ BELOW.
# ============================================================


# ============================================================
# 12. FINAL POLICY GRAPH MUTATION
# ============================================================

def apply_locked_fold0_policy(
    graph,
    actions,
):
    """
    Exact frozen NO_TARGET_CUT graph semantics.

    All actions satisfy target_occupied == 0.

    Therefore:
        - source incumbent edge may be removed
        - target incumbent edge is NEVER removed
        - one new source->target edge is added
    """

    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = local_graph_state(
        graph
    )


    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges[
        "edge_id"
    ] = edges[
        "edge_id"
    ].astype(
        int
    )


    edges[
        "source_id"
    ] = edges[
        "source_id"
    ].astype(
        int
    )


    edges[
        "target_id"
    ] = edges[
        "target_id"
    ].astype(
        int
    )


    outgoing = {}


    for src, sub in edges.groupby(
        "source_id"
    ):

        outgoing[
            int(
                src
            )
        ] = [
            (
                int(
                    row.edge_id
                ),
                int(
                    row.target_id
                ),
            )

            for row
            in sub.itertuples(
                index=False
            )
        ]


    remove_ids = []

    audit_rows = []


    for action in actions.itertuples(
        index=False
    ):

        src = int(
            action.source_id
        )


        tgt = int(
            action.target_id
        )


        assert src != tgt


        assert (
            src,
            tgt,
        ) not in edge_set


        src_out = int(
            outdeg.get(
                src,
                0,
            )
        )


        tgt_in = int(
            indeg.get(
                tgt,
                0,
            )
        )


        # Frozen NO_TARGET_CUT contract.
        assert tgt_in == 0, (
            src,
            tgt,
            tgt_in,
        )


        assert int(
            action.target_occupied
        ) == 0


        assert src_out <= 1


        assert int(
            action.source_occupied
        ) == int(
            src_out == 1
        )


        expected_cuts = int(
            src_out == 1
        )


        assert int(
            action.cuts
        ) == expected_cuts


        removed_edge_id = None

        removed_target = None


        if src_out == 1:

            src_edges = outgoing.get(
                src,
                [],
            )


            assert len(
                src_edges
            ) == 1


            (
                removed_edge_id,
                removed_target,
            ) = src_edges[
                0
            ]


            remove_ids.append(
                int(
                    removed_edge_id
                )
            )


        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )


        audit_rows.append(
            {
                "source_id":
                    src,

                "target_id":
                    tgt,

                "source_pre_outdegree":
                    src_out,

                "target_pre_indegree":
                    tgt_in,

                "removed_source_edge_id":
                    removed_edge_id,

                "removed_source_target":
                    removed_target,

                "edge_dist":
                    float(
                        dist
                    ),
            }
        )


    # Sources are unique due to node-disjoint resolver.
    assert len(
        remove_ids
    ) == len(
        set(
            remove_ids
        )
    )


    pre_edges = int(
        graph.num_edges()
    )


    if remove_ids:

        graph.bulk_remove_edges(
            sorted(
                remove_ids
            )
        )


    after_remove = int(
        graph.num_edges()
    )


    assert (
        pre_edges
        -
        after_remove
    ) == len(
        remove_ids
    )


    for row in audit_rows:

        src = int(
            row[
                "source_id"
            ]
        )


        tgt = int(
            row[
                "target_id"
            ]
        )


        assert not graph.has_edge(
            src,
            tgt,
        )


        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(
                        row[
                            "edge_dist"
                        ]
                    ),

                # Schema-valid ephemeral placeholder.
                # Frozen LogReg score is NOT a calibrated
                # probability.
                "edge_prob":
                    1.0,
            },
            validate_keys=True,
        )


    post_edges = int(
        graph.num_edges()
    )


    assert post_edges == (
        after_remove
        +
        len(
            audit_rows
        )
    )


    (
        _,
        outdeg_post,
        indeg_post,
        edge_set_post,
    ) = local_graph_state(
        graph
    )


    for row in audit_rows:

        src = int(
            row[
                "source_id"
            ]
        )


        tgt = int(
            row[
                "target_id"
            ]
        )


        assert (
            src,
            tgt,
        ) in edge_set_post


        assert outdeg_post.get(
            src,
            0,
        ) == 1


        assert indeg_post.get(
            tgt,
            0,
        ) == 1


    stats = {
        "actions":
            int(
                len(
                    actions
                )
            ),

        "source_incumbent_edges_removed":
            int(
                len(
                    remove_ids
                )
            ),

        "edges_added":
            int(
                len(
                    actions
                )
            ),

        "net_edge_delta":
            int(
                post_edges
                -
                pre_edges
            ),
    }


    return (
        graph,
        stats,
    )


# ============================================================
# 13. NOW — AND ONLY NOW — OFFICIAL FOLD0 GT EVALUATION
# ============================================================

print(
    "\n============================================================"
)


print(
    "PART B — ONE-SHOT OFFICIAL FOLD0 EVALUATION"
)


print(
    "============================================================"
)


baseline_records = []

policy_records = []

metric_rows = []


actions_by_dataset = {
    dataset:
        sub.copy()

    for dataset, sub
    in fold0_actions.groupby(
        "dataset",
        sort=False,
    )
}


for i, dataset in enumerate(
    names,
    start=1,
):

    prefix = (
        "44b6"
        if dataset.startswith(
            "44b6_"
        )
        else
        "6bba"
    )


    # --------------------------------------------------------
    # Exact Fold0 CURRENT_BEST.
    # --------------------------------------------------------

    base_graph, scale = (
        fold0_build_current_best(
            dataset
        )
    )


    policy_graph = base_graph.copy()


    # --------------------------------------------------------
    # Frozen final policy applies only to 6bba.
    # 44b6 must remain byte-semantically equivalent at graph
    # topology level.
    # --------------------------------------------------------

    if dataset.startswith(
        "6bba_"
    ):

        ds_actions = actions_by_dataset.get(
            dataset,
            fold0_actions.iloc[
                0:0
            ].copy(),
        )


        (
            policy_graph,
            mutation_stats,
        ) = apply_locked_fold0_policy(
            policy_graph,
            ds_actions,
        )


    else:

        assert dataset.startswith(
            "44b6_"
        )


        mutation_stats = {
            "actions":
                0,

            "source_incumbent_edges_removed":
                0,

            "edges_added":
                0,

            "net_edge_delta":
                0,
        }


        assert policy_graph.num_edges() == base_graph.num_edges()


    # --------------------------------------------------------
    # FIRST GT READ.
    # --------------------------------------------------------

    gt = fv9.load_graph(
        TRAIN_ROOT
        / f"{dataset}.geff"
    )


    n_total = float(
        estimated_nodes[
            dataset
        ]
    )


    # --------------------------------------------------------
    # Authoritative CURRENT_BEST.
    # --------------------------------------------------------

    base_metric = fv9.evaluate_graph(
        base_graph,
        gt,
        scale,
        n_total,
        fv9.FROZEN_V9_CONFIG,
    )


    baseline_records.append(
        base_metric
    )


    # --------------------------------------------------------
    # Frozen policy.
    # --------------------------------------------------------

    policy_metric = fv9.evaluate_graph(
        policy_graph,
        gt,
        scale,
        n_total,
        fv9.FROZEN_V9_CONFIG,
    )


    policy_records.append(
        policy_metric
    )


    metric_rows.append(
        {
            "dataset":
                dataset,

            "prefix":
                prefix,

            "actions":
                int(
                    mutation_stats[
                        "actions"
                    ]
                ),

            "edges_removed":
                int(
                    mutation_stats[
                        "source_incumbent_edges_removed"
                    ]
                ),

            "edges_added":
                int(
                    mutation_stats[
                        "edges_added"
                    ]
                ),

            "net_edge_delta":
                int(
                    mutation_stats[
                        "net_edge_delta"
                    ]
                ),

            # --------------------------------------------
            # Baseline
            # --------------------------------------------

            "base_edge_tp":
                int(
                    base_metric[
                        "edge_tp"
                    ]
                ),

            "base_edge_fp":
                int(
                    base_metric[
                        "edge_fp"
                    ]
                ),

            "base_edge_fn":
                int(
                    base_metric[
                        "edge_fn"
                    ]
                ),

            "base_division_tp":
                int(
                    base_metric[
                        "division_tp"
                    ]
                ),

            "base_division_fp":
                int(
                    base_metric[
                        "division_fp"
                    ]
                ),

            "base_division_fn":
                int(
                    base_metric[
                        "division_fn"
                    ]
                ),

            "base_edge_jaccard":
                float(
                    base_metric[
                        "edge_jaccard"
                    ]
                ),

            "base_adj_edge_jaccard":
                float(
                    base_metric[
                        "adj_edge_jaccard"
                    ]
                ),

            "base_node_recall":
                float(
                    base_metric[
                        "node_recall"
                    ]
                ),

            # --------------------------------------------
            # Policy
            # --------------------------------------------

            "policy_edge_tp":
                int(
                    policy_metric[
                        "edge_tp"
                    ]
                ),

            "policy_edge_fp":
                int(
                    policy_metric[
                        "edge_fp"
                    ]
                ),

            "policy_edge_fn":
                int(
                    policy_metric[
                        "edge_fn"
                    ]
                ),

            "policy_division_tp":
                int(
                    policy_metric[
                        "division_tp"
                    ]
                ),

            "policy_division_fp":
                int(
                    policy_metric[
                        "division_fp"
                    ]
                ),

            "policy_division_fn":
                int(
                    policy_metric[
                        "division_fn"
                    ]
                ),

            "policy_edge_jaccard":
                float(
                    policy_metric[
                        "edge_jaccard"
                    ]
                ),

            "policy_adj_edge_jaccard":
                float(
                    policy_metric[
                        "adj_edge_jaccard"
                    ]
                ),

            "policy_node_recall":
                float(
                    policy_metric[
                        "node_recall"
                    ]
                ),
        }
    )


    print(
        f"{i:2d}/40"
        f" | {prefix}"
        f" | {dataset}"
        f" | actions={mutation_stats['actions']}"
        f" | removed={mutation_stats['source_incumbent_edges_removed']}"
        f" | net={mutation_stats['net_edge_delta']}"
    )


    del base_graph
    del policy_graph
    del gt

    gc.collect()


metric_df = pd.DataFrame(
    metric_rows
)


assert len(
    metric_df
) == 40


# ============================================================
# 14. BASELINE FIDELITY GATE
# ============================================================

baseline_summary = fv9.summarise(
    baseline_records
)


base_tp = int(
    metric_df[
        "base_edge_tp"
    ].sum()
)


base_fp = int(
    metric_df[
        "base_edge_fp"
    ].sum()
)


base_fn = int(
    metric_df[
        "base_edge_fn"
    ].sum()
)


print(
    "\n========== LOCKED FOLD0 CURRENT_BEST FIDELITY =========="
)


print(
    "TP/FP/FN:",
    base_tp,
    "/",
    base_fp,
    "/",
    base_fn
)


print(
    "expected:",
    EXPECTED_BASE_TP,
    "/",
    EXPECTED_BASE_FP,
    "/",
    EXPECTED_BASE_FN
)


print(
    "score:",
    f"{baseline_summary['score']:.12f}"
)


print(
    "expected:",
    f"{EXPECTED_BASE_SCORE:.12f}"
)


assert base_tp == EXPECTED_BASE_TP

assert base_fp == EXPECTED_BASE_FP

assert base_fn == EXPECTED_BASE_FN


assert abs(
    float(
        baseline_summary[
            "score"
        ]
    )
    -
    EXPECTED_BASE_SCORE
) <= 5e-7


print(
    "LOCKED_FOLD0_CURRENT_BEST_FIDELITY: PASS"
)


# ============================================================
# 15. FINAL POLICY OFFICIAL RESULT
# ============================================================

policy_summary = fv9.summarise(
    policy_records
)


policy_tp = int(
    metric_df[
        "policy_edge_tp"
    ].sum()
)


policy_fp = int(
    metric_df[
        "policy_edge_fp"
    ].sum()
)


policy_fn = int(
    metric_df[
        "policy_edge_fn"
    ].sum()
)


delta_score = float(
    policy_summary[
        "score"
    ]
    -
    baseline_summary[
        "score"
    ]
)


print(
    "\n========== LOCKED FOLD0 FINAL POLICY RESULT =========="
)


print(
    "policy:",
    FINAL_POLICY_ID
)


print(
    "Fold0 actions:",
    len(
        fold0_actions
    )
)


print(
    "source incumbent edges removed:",
    int(
        metric_df[
            "edges_removed"
        ].sum()
    )
)


print(
    "new edges added:",
    int(
        metric_df[
            "edges_added"
        ].sum()
    )
)


print(
    "net edge delta:",
    int(
        metric_df[
            "net_edge_delta"
        ].sum()
    )
)


print(
    "\nedge TP/FP/FN:"
)


print(
    "CURRENT_BEST:",
    base_tp,
    "/",
    base_fp,
    "/",
    base_fn
)


print(
    "FINAL POLICY:",
    policy_tp,
    "/",
    policy_fp,
    "/",
    policy_fn
)


print(
    "delta:",
    f"{policy_tp-base_tp:+d}",
    "/",
    f"{policy_fp-base_fp:+d}",
    "/",
    f"{policy_fn-base_fn:+d}",
)


print(
    "\nmetrics:"
)


print(
    "edge J:",
    f"{baseline_summary['edge_jaccard']:.9f}",
    "->",
    f"{policy_summary['edge_jaccard']:.9f}",
    (
        f"({policy_summary['edge_jaccard'] - baseline_summary['edge_jaccard']:+.9f})"
    ),
)


print(
    "adj edge J:",
    f"{baseline_summary['adj_edge_jaccard']:.9f}",
    "->",
    f"{policy_summary['adj_edge_jaccard']:.9f}",
    (
        f"({policy_summary['adj_edge_jaccard'] - baseline_summary['adj_edge_jaccard']:+.9f})"
    ),
)


print(
    "division J:",
    f"{baseline_summary['division_jaccard']:.9f}",
    "->",
    f"{policy_summary['division_jaccard']:.9f}",
    (
        f"({policy_summary['division_jaccard'] - baseline_summary['division_jaccard']:+.9f})"
    ),
)


print(
    "node recall:",
    f"{baseline_summary['node_recall']:.9f}",
    "->",
    f"{policy_summary['node_recall']:.9f}",
    (
        f"({policy_summary['node_recall'] - baseline_summary['node_recall']:+.9f})"
    ),
)


print(
    "SCORE:",
    f"{baseline_summary['score']:.12f}",
    "->",
    f"{policy_summary['score']:.12f}",
    f"({delta_score:+.12f})",
)


# ============================================================
# 16. PREFIX-SPECIFIC AUDIT
#
# Diagnostic only. Confirmation remains FULL-40 score.
# ============================================================

prefix_rows = []


for prefix in [
    "44b6",
    "6bba",
]:

    prefix_mask = (
        metric_df[
            "prefix"
        ].eq(
            prefix
        )
        .to_numpy()
    )


    base_prefix_records = [
        row
        for row, keep
        in zip(
            baseline_records,
            prefix_mask,
        )
        if keep
    ]


    policy_prefix_records = [
        row
        for row, keep
        in zip(
            policy_records,
            prefix_mask,
        )
        if keep
    ]


    b = fv9.summarise(
        base_prefix_records
    )


    p = fv9.summarise(
        policy_prefix_records
    )


    prefix_metric = metric_df[
        metric_df[
            "prefix"
        ].eq(
            prefix
        )
    ]


    prefix_rows.append(
        {
            "prefix":
                prefix,

            "datasets":
                int(
                    len(
                        prefix_metric
                    )
                ),

            "actions":
                int(
                    prefix_metric[
                        "actions"
                    ].sum()
                ),

            "base_score":
                float(
                    b[
                        "score"
                    ]
                ),

            "policy_score":
                float(
                    p[
                        "score"
                    ]
                ),

            "delta_score":
                float(
                    p[
                        "score"
                    ]
                    -
                    b[
                        "score"
                    ]
                ),

            "delta_tp":
                int(
                    prefix_metric[
                        "policy_edge_tp"
                    ].sum()
                    -
                    prefix_metric[
                        "base_edge_tp"
                    ].sum()
                ),

            "delta_fp":
                int(
                    prefix_metric[
                        "policy_edge_fp"
                    ].sum()
                    -
                    prefix_metric[
                        "base_edge_fp"
                    ].sum()
                ),

            "delta_fn":
                int(
                    prefix_metric[
                        "policy_edge_fn"
                    ].sum()
                    -
                    prefix_metric[
                        "base_edge_fn"
                    ].sum()
                ),
        }
    )


prefix_summary = pd.DataFrame(
    prefix_rows
)


# 44b6 must be exactly unchanged.
p44 = prefix_summary[
    prefix_summary[
        "prefix"
    ].eq(
        "44b6"
    )
].iloc[
    0
]


assert int(
    p44[
        "actions"
    ]
) == 0


assert abs(
    float(
        p44[
            "delta_score"
        ]
    )
) <= 1e-15


assert int(
    p44[
        "delta_tp"
    ]
) == 0


assert int(
    p44[
        "delta_fp"
    ]
) == 0


assert int(
    p44[
        "delta_fn"
    ]
) == 0


print(
    "\n========== LOCKED FOLD0 BY PREFIX =========="
)


print(
    prefix_summary.to_string(
        index=False
    )
)


# ============================================================
# 17. DATASET OUTCOME AUDIT
# ============================================================

metric_df[
    "delta_edge_j"
] = (
    metric_df[
        "policy_edge_jaccard"
    ]
    -
    metric_df[
        "base_edge_jaccard"
    ]
)


metric_df[
    "delta_tp"
] = (
    metric_df[
        "policy_edge_tp"
    ]
    -
    metric_df[
        "base_edge_tp"
    ]
)


metric_df[
    "delta_fp"
] = (
    metric_df[
        "policy_edge_fp"
    ]
    -
    metric_df[
        "base_edge_fp"
    ]
)


metric_df[
    "delta_fn"
] = (
    metric_df[
        "policy_edge_fn"
    ]
    -
    metric_df[
        "base_edge_fn"
    ]
)


wins = int(
    metric_df[
        "delta_edge_j"
    ].gt(
        0
    ).sum()
)


losses = int(
    metric_df[
        "delta_edge_j"
    ].lt(
        0
    ).sum()
)


ties = int(
    metric_df[
        "delta_edge_j"
    ].eq(
        0
    ).sum()
)


print(
    "\n========== LOCKED FOLD0 DATASET OUTCOME =========="
)


print(
    "edge-J wins:",
    wins
)


print(
    "edge-J losses:",
    losses
)


print(
    "edge-J ties:",
    ties
)


# ============================================================
# 18. PREDECLARED ONE-SHOT DECISION
# ============================================================

confirmed = bool(
    delta_score > 0.0
)


if confirmed:

    decision = (
        "LOCKED_FOLD0_CONFIRMED"
    )


else:

    decision = (
        "LOCKED_FOLD0_NOT_CONFIRMED"
    )


# ============================================================
# 19. PERSIST FINAL LOCKED TEST
# ============================================================

metric_df.to_pickle(
    FOLD0_METRICS_OUT
)


SUMMARY = {
    "stage":
        "12.11G3H3E0",

    "status":
        "ONE_SHOT_LOCKED_FOLD0_CONFIRMATION_COMPLETE",

    "confirmation_contract_sha256":
        CONFIRMATION_CONTRACT_SHA,

    "final_policy_id":
        FINAL_POLICY_ID,

    "final_policy_sha256":
        FINAL_POLICY_SHA,

    "fold0_actions":
        int(
            len(
                fold0_actions
            )
        ),

    "fold0_actions_sha256":
        FOLD0_ACTIONS_SHA,

    "corpus": {
        "datasets":
            40,

        "44b6":
            15,

        "6bba":
            25,
    },

    "baseline": {
        "edge_tp":
            base_tp,

        "edge_fp":
            base_fp,

        "edge_fn":
            base_fn,

        "edge_jaccard":
            float(
                baseline_summary[
                    "edge_jaccard"
                ]
            ),

        "adj_edge_jaccard":
            float(
                baseline_summary[
                    "adj_edge_jaccard"
                ]
            ),

        "division_jaccard":
            float(
                baseline_summary[
                    "division_jaccard"
                ]
            ),

        "node_recall":
            float(
                baseline_summary[
                    "node_recall"
                ]
            ),

        "score":
            float(
                baseline_summary[
                    "score"
                ]
            ),
    },

    "policy": {
        "edge_tp":
            policy_tp,

        "edge_fp":
            policy_fp,

        "edge_fn":
            policy_fn,

        "edge_jaccard":
            float(
                policy_summary[
                    "edge_jaccard"
                ]
            ),

        "adj_edge_jaccard":
            float(
                policy_summary[
                    "adj_edge_jaccard"
                ]
            ),

        "division_jaccard":
            float(
                policy_summary[
                    "division_jaccard"
                ]
            ),

        "node_recall":
            float(
                policy_summary[
                    "node_recall"
                ]
            ),

        "score":
            float(
                policy_summary[
                    "score"
                ]
            ),
    },

    "delta": {
        "edge_tp":
            int(
                policy_tp
                -
                base_tp
            ),

        "edge_fp":
            int(
                policy_fp
                -
                base_fp
            ),

        "edge_fn":
            int(
                policy_fn
                -
                base_fn
            ),

        "edge_jaccard":
            float(
                policy_summary[
                    "edge_jaccard"
                ]
                -
                baseline_summary[
                    "edge_jaccard"
                ]
            ),

        "adj_edge_jaccard":
            float(
                policy_summary[
                    "adj_edge_jaccard"
                ]
                -
                baseline_summary[
                    "adj_edge_jaccard"
                ]
            ),

        "division_jaccard":
            float(
                policy_summary[
                    "division_jaccard"
                ]
                -
                baseline_summary[
                    "division_jaccard"
                ]
            ),

        "score":
            delta_score,
    },

    "prefix_audit":
        prefix_summary.to_dict(
            "records"
        ),

    "dataset_edgej": {
        "wins":
            wins,

        "losses":
            losses,

        "ties":
            ties,
    },

    "baseline_fidelity_pass":
        True,

    "confirmation_rule":
        CONFIRMATION_CONTRACT[
            "confirmation_rule"
        ],

    "confirmed":
        confirmed,

    "decision":
        decision,

    "development_was_closed_before_fold0":
        True,

    "post_fold0_retuning_allowed":
        False,

    "scorer_retuned":
        False,

    "score_threshold_retuned":
        False,

    "rank_rule_retuned":
        False,

    "structural_rule_retuned":
        False,

    "prediction_geff_written":
        False,

    "next": (
        "PROMOTE_CONFIRMED_POLICY_WITHOUT_RETUNING"
        if confirmed
        else
        "CLOSE_REWIRE_BRANCH_WITHOUT_POST_FOLD0_RETUNING"
    ),
}


FOLD0_SUMMARY_OUT.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


assert FOLD0_METRICS_OUT.exists()

assert FOLD0_SUMMARY_OUT.exists()


# ============================================================
# 20. RELOAD INTEGRITY
# ============================================================

metric_check = pd.read_pickle(
    FOLD0_METRICS_OUT
)


summary_check = json.loads(
    FOLD0_SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert len(
    metric_check
) == 40


assert summary_check[
    "baseline_fidelity_pass"
] is True


assert summary_check[
    "post_fold0_retuning_allowed"
] is False


assert summary_check[
    "final_policy_sha256"
] == FINAL_POLICY_SHA


# ============================================================
# 21. FINAL
# ============================================================

print(
    "\n========== 12.11G3H3E0 FINAL DECISION =========="
)


print(
    "ONE_SHOT_LOCKED_FOLD0_CONFIRMATION: COMPLETE"
)


print(
    "policy:",
    FINAL_POLICY_ID
)


print(
    "Fold0 actions:",
    len(
        fold0_actions
    )
)


print(
    "Fold0 action SHA:",
    FOLD0_ACTIONS_SHA
)


print(
    "baseline fidelity:",
    "PASS"
)


print(
    "CURRENT_BEST score:",
    f"{baseline_summary['score']:.12f}"
)


print(
    "policy score:",
    f"{policy_summary['score']:.12f}"
)


print(
    "delta:",
    f"{delta_score:+.12f}"
)


print(
    "confirmed:",
    confirmed
)


print(
    "decision:",
    decision
)


print(
    "development reopened:",
    "NO"
)


print(
    "post-Fold0 retuning allowed:",
    "NO"
)


print(
    "prediction GEFF written:",
    "NO"
)


print(
    "next:"
)


print(
    SUMMARY[
        "next"
    ]
)

In [ ]:
# ============================================================
# Stage 12.11G3H3E1
#
# CONFIRMED POLICY PROMOTION
#
# ------------------------------------------------------------
# PROMOTED POLICY
# ------------------------------------------------------------
#
#   ASYM1x5_Q9990_NO_TARGET_CUT_V1
#
# Development:
#
#   CURRENT_BEST:
#       0.7919774336227487
#
#   promoted:
#       0.7920280500935799
#
#   delta:
#       +0.00005061647083115517
#
# Locked Fold0:
#
#   CURRENT_BEST:
#       0.7863444776365197
#
#   promoted:
#       0.786398595889
#
#   delta:
#       +0.000054118252
#
# Fold0 edge counts:
#
#   baseline:
#       TP / FP / FN = 22806 / 2658 / 3425
#
#   promoted:
#       TP / FP / FN = 22806 / 2656 / 3425
#
#   delta:
#       0 / -2 / 0
#
# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------
#
# This stage performs NO model / policy evaluation.
#
# It:
#
#   - verifies frozen development policy
#   - verifies one-shot Fold0 confirmation
#   - verifies action/provenance SHAs
#   - formally promotes the confirmed policy
#   - creates a NEW CURRENT_BEST checkpoint
#
# It does NOT:
#
#   - retune scorer
#   - retune threshold
#   - retune rank rule
#   - retune structural rule
#   - rerun Fold0
#   - reopen development
#   - overwrite the historical CURRENT_BEST checkpoint
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import hashlib
import json

import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


FINAL_POLICY_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_frozen.json"
)


FREEZE_SUMMARY_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_freeze_summary.json"
)


FOLD0_CONTRACT_PATH = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_confirmation_contract_frozen.json"
)


FOLD0_ACTIONS_PATH = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_actions_locked.pkl"
)


FOLD0_ACTION_SUMMARY_PATH = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_action_summary.json"
)


FOLD0_METRICS_PATH = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_confirmation_metrics.pkl"
)


FOLD0_SUMMARY_PATH = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_confirmation_summary.json"
)


# Historical checkpoint — READ ONLY.
OLD_CURRENT_BEST_PATH = (
    S12
    / "stage12_current_best_checkpoint.json"
)


# ------------------------------------------------------------
# New formal outputs
# ------------------------------------------------------------

PROMOTION_OUT = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_promoted.json"
)


NEW_CURRENT_BEST_OUT = (
    S12
    / "stage12_current_best_checkpoint.post_g3h3e1.json"
)


SUMMARY_OUT = (
    S12
    / "stage12_g3h3e1_confirmed_policy_promotion_summary.json"
)


for path in [
    FINAL_POLICY_PATH,
    FREEZE_SUMMARY_PATH,
    FOLD0_CONTRACT_PATH,
    FOLD0_ACTIONS_PATH,
    FOLD0_ACTION_SUMMARY_PATH,
    FOLD0_METRICS_PATH,
    FOLD0_SUMMARY_PATH,
    OLD_CURRENT_BEST_PATH,
]:

    assert path.exists(), path


for path in [
    PROMOTION_OUT,
    NEW_CURRENT_BEST_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3H3E1 formal promotion output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite confirmed-policy provenance."
        )


# ============================================================
# 1. AUTHORITATIVE CONSTANTS
# ============================================================

POLICY_ID = (
    "ASYM1x5_Q9990_NO_TARGET_CUT_V1"
)


POLICY_SHA = (
    "77f0b07cafb21d0705917ff71b750fde4"
    "5f3c7ddc17c13fecaff0ac673ebf776"
)


FOLD0_ACTION_SHA = (
    "0baa83066b4dc22f2f368a5a1f9ff1a"
    "90b81649e81caee282c88c48e88deb1c2"
)


EXPECTED_DEV_BASE_SCORE = (
    0.7919774336227487
)


EXPECTED_DEV_POLICY_SCORE = (
    0.7920280500935799
)


EXPECTED_DEV_DELTA = (
    5.061647083115517e-05
)


EXPECTED_FOLD0_BASE_SCORE = (
    0.7863444776365197
)


EXPECTED_FOLD0_POLICY_SCORE = (
    0.786398595889
)


EXPECTED_FOLD0_DELTA = (
    0.000054118252
)


EXPECTED_FOLD0_ACTIONS = 2712


EXPECTED_BASE_TP = 22806
EXPECTED_BASE_FP = 2658
EXPECTED_BASE_FN = 3425


EXPECTED_POLICY_TP = 22806
EXPECTED_POLICY_FP = 2656
EXPECTED_POLICY_FN = 3425


# ============================================================
# 2. SHA HELPER
# ============================================================

def file_sha256(
    path,
):

    h = hashlib.sha256()


    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024
                *
                1024
            )


            if not chunk:

                break


            h.update(
                chunk
            )


    return h.hexdigest()


# ============================================================
# 3. VERIFY DEVELOPMENT FREEZE
# ============================================================

policy = json.loads(
    FINAL_POLICY_PATH.read_text(
        encoding="utf-8"
    )
)


freeze = json.loads(
    FREEZE_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert policy[
    "policy_id"
] == POLICY_ID


assert policy[
    "final_policy_sha256"
] == POLICY_SHA


assert policy[
    "status"
] == (
    "FINAL_DEVELOPMENT_POLICY_FROZEN"
)


assert policy[
    "development_closed_after_freeze"
] is True


assert policy[
    "future_scorer_retuning"
] is False


assert policy[
    "future_shortlist_retuning"
] is False


assert policy[
    "future_rank_rule_retuning"
] is False


assert policy[
    "future_score_quantile_retuning"
] is False


assert policy[
    "future_structural_rule_retuning"
] is False


assert freeze[
    "development_closed"
] is True


assert freeze[
    "no_further_development_tuning"
] is True


assert abs(
    float(
        freeze[
            "development_score"
        ]
    )
    -
    EXPECTED_DEV_POLICY_SCORE
) <= 1e-12


assert abs(
    float(
        freeze[
            "development_delta_score"
        ]
    )
    -
    EXPECTED_DEV_DELTA
) <= 1e-12


print(
    "========== G3H3E1 DEVELOPMENT FREEZE =========="
)


print(
    "policy:",
    POLICY_ID
)


print(
    "policy SHA:",
    POLICY_SHA
)


print(
    "development baseline:",
    EXPECTED_DEV_BASE_SCORE
)


print(
    "development promoted:",
    freeze[
        "development_score"
    ]
)


print(
    "development delta:",
    freeze[
        "development_delta_score"
    ]
)


print(
    "development closed:",
    "YES"
)


# ============================================================
# 4. VERIFY LOCKED FOLD0 CONTRACT
# ============================================================

fold0_contract = json.loads(
    FOLD0_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


assert fold0_contract[
    "final_policy_id"
] == POLICY_ID


assert fold0_contract[
    "final_policy_sha256"
] == POLICY_SHA


assert fold0_contract[
    "confirmation_rule"
] == (
    "POLICY_SCORE_STRICTLY_GREATER_THAN_CURRENT_BEST_SCORE"
)


assert fold0_contract[
    "development_closed"
] is True


assert fold0_contract[
    "post_fold0_retuning_allowed"
] is False


assert fold0_contract[
    "fold0_metric_seen_before_contract_freeze"
] is False


# ============================================================
# 5. VERIFY FOLD0 ACTION FREEZE
# ============================================================

fold0_action_summary = json.loads(
    FOLD0_ACTION_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


fold0_actions = pd.read_pickle(
    FOLD0_ACTIONS_PATH
)


assert len(
    fold0_actions
) == EXPECTED_FOLD0_ACTIONS


assert fold0_action_summary[
    "final_actions"
] == EXPECTED_FOLD0_ACTIONS


assert fold0_action_summary[
    "actions_sha256"
] == FOLD0_ACTION_SHA


assert file_sha256(
    FOLD0_ACTIONS_PATH
) == FOLD0_ACTION_SHA


assert fold0_action_summary[
    "target_occupied_actions"
] == 0


assert fold0_action_summary[
    "actions_node_disjoint"
] is True


assert fold0_action_summary[
    "gt_loaded_before_action_freeze"
] is False


assert fold0_action_summary[
    "fold0_metric_seen_before_action_freeze"
] is False


print(
    "\n========== G3H3E1 FOLD0 ACTION PROVENANCE =========="
)


print(
    "actions:",
    len(
        fold0_actions
    )
)


print(
    "SHA:",
    FOLD0_ACTION_SHA
)


print(
    "target occupied:",
    fold0_action_summary[
        "target_occupied_actions"
    ]
)


print(
    "GT before action freeze:",
    "NO"
)


# ============================================================
# 6. VERIFY ONE-SHOT LOCKED CONFIRMATION
# ============================================================

fold0 = json.loads(
    FOLD0_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


metrics = pd.read_pickle(
    FOLD0_METRICS_PATH
)


assert len(
    metrics
) == 40


assert fold0[
    "status"
] == (
    "ONE_SHOT_LOCKED_FOLD0_CONFIRMATION_COMPLETE"
)


assert fold0[
    "final_policy_id"
] == POLICY_ID


assert fold0[
    "final_policy_sha256"
] == POLICY_SHA


assert fold0[
    "fold0_actions"
] == EXPECTED_FOLD0_ACTIONS


assert fold0[
    "fold0_actions_sha256"
] == FOLD0_ACTION_SHA


assert fold0[
    "baseline_fidelity_pass"
] is True


assert fold0[
    "confirmed"
] is True


assert fold0[
    "decision"
] == (
    "LOCKED_FOLD0_CONFIRMED"
)


assert fold0[
    "post_fold0_retuning_allowed"
] is False


# ------------------------------------------------------------
# Baseline exact
# ------------------------------------------------------------

assert fold0[
    "baseline"
][
    "edge_tp"
] == EXPECTED_BASE_TP


assert fold0[
    "baseline"
][
    "edge_fp"
] == EXPECTED_BASE_FP


assert fold0[
    "baseline"
][
    "edge_fn"
] == EXPECTED_BASE_FN


assert abs(
    float(
        fold0[
            "baseline"
        ][
            "score"
        ]
    )
    -
    EXPECTED_FOLD0_BASE_SCORE
) <= 5e-12


# ------------------------------------------------------------
# Policy exact
# ------------------------------------------------------------

assert fold0[
    "policy"
][
    "edge_tp"
] == EXPECTED_POLICY_TP


assert fold0[
    "policy"
][
    "edge_fp"
] == EXPECTED_POLICY_FP


assert fold0[
    "policy"
][
    "edge_fn"
] == EXPECTED_POLICY_FN


assert abs(
    float(
        fold0[
            "policy"
        ][
            "score"
        ]
    )
    -
    EXPECTED_FOLD0_POLICY_SCORE
) <= 5e-10


assert int(
    fold0[
        "delta"
    ][
        "edge_tp"
    ]
) == 0


assert int(
    fold0[
        "delta"
    ][
        "edge_fp"
    ]
) == -2


assert int(
    fold0[
        "delta"
    ][
        "edge_fn"
    ]
) == 0


assert float(
    fold0[
        "delta"
    ][
        "score"
    ]
) > 0


print(
    "\n========== G3H3E1 LOCKED FOLD0 CONFIRMATION =========="
)


print(
    "baseline TP/FP/FN:",
    EXPECTED_BASE_TP,
    "/",
    EXPECTED_BASE_FP,
    "/",
    EXPECTED_BASE_FN
)


print(
    "policy TP/FP/FN:",
    EXPECTED_POLICY_TP,
    "/",
    EXPECTED_POLICY_FP,
    "/",
    EXPECTED_POLICY_FN
)


print(
    "delta TP/FP/FN:",
    0,
    "/",
    -2,
    "/",
    0
)


print(
    "baseline score:",
    fold0[
        "baseline"
    ][
        "score"
    ]
)


print(
    "policy score:",
    fold0[
        "policy"
    ][
        "score"
    ]
)


print(
    "delta:",
    fold0[
        "delta"
    ][
        "score"
    ]
)


print(
    "decision:",
    fold0[
        "decision"
    ]
)


print(
    "LOCKED FOLD0 CONFIRMATION: PASS"
)


# ============================================================
# 7. HISTORICAL CURRENT_BEST
#
# Read only.
# ============================================================

old_checkpoint = json.loads(
    OLD_CURRENT_BEST_PATH.read_text(
        encoding="utf-8"
    )
)


print(
    "\n========== HISTORICAL CURRENT_BEST =========="
)


print(
    "path:",
    OLD_CURRENT_BEST_PATH.name
)


print(
    "historical checkpoint preserved:",
    "YES"
)


# ============================================================
# 8. FORMAL PROMOTION ARTIFACT
# ============================================================

PROMOTION = {
    "stage":
        "12.11G3H3E1",

    "status":
        "CONFIRMED_POLICY_PROMOTED",

    "policy_id":
        POLICY_ID,

    "policy_sha256":
        POLICY_SHA,

    "promotion_basis": {
        "development": {
            "baseline_score":
                EXPECTED_DEV_BASE_SCORE,

            "policy_score":
                EXPECTED_DEV_POLICY_SCORE,

            "delta_score":
                EXPECTED_DEV_DELTA,

            "promotion_gate_passed":
                True,
        },

        "locked_fold0": {
            "baseline_score":
                float(
                    fold0[
                        "baseline"
                    ][
                        "score"
                    ]
                ),

            "policy_score":
                float(
                    fold0[
                        "policy"
                    ][
                        "score"
                    ]
                ),

            "delta_score":
                float(
                    fold0[
                        "delta"
                    ][
                        "score"
                    ]
                ),

            "edge_delta": {
                "tp":
                    0,

                "fp":
                    -2,

                "fn":
                    0,
            },

            "confirmed":
                True,

            "confirmation_decision":
                "LOCKED_FOLD0_CONFIRMED",
        },
    },

    "runtime_contract":
        policy[
            "runtime_contract"
        ],

    "fold0_action_artifact":
        FOLD0_ACTIONS_PATH.name,

    "fold0_action_sha256":
        FOLD0_ACTION_SHA,

    "development_closed":
        True,

    "fold0_consumed":
        True,

    "fold0_remains_available_for_retuning":
        False,

    "post_fold0_retuning_allowed":
        False,

    "promotion_retunes_policy":
        False,

    "prediction_geff_written":
        False,

    "historical_current_best_checkpoint_preserved":
        True,

    "next":
        "INTEGRATE_CONFIRMED_POLICY_INTO_CANONICAL_CURRENT_BEST_BUILDER",
}


PROMOTION_SHA = hashlib.sha256(
    json.dumps(
        PROMOTION,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


PROMOTION[
    "promotion_sha256"
] = PROMOTION_SHA


PROMOTION_OUT.write_text(
    json.dumps(
        PROMOTION,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 9. NEW CURRENT_BEST CHECKPOINT
#
# Do NOT overwrite stage12_current_best_checkpoint.json.
# ============================================================

NEW_CHECKPOINT = {
    "stage":
        "12.11G3H3E1",

    "checkpoint_id":
        "STAGE12_CURRENT_BEST_POST_G3H3E1",

    "status":
        "CURRENT_BEST_UPDATED_BY_LOCKED_FOLD0_CONFIRMED_POLICY",

    "parent_checkpoint":
        OLD_CURRENT_BEST_PATH.name,

    "parent_checkpoint_preserved":
        True,

    "promoted_policy":
        POLICY_ID,

    "promoted_policy_sha256":
        POLICY_SHA,

    "promotion_sha256":
        PROMOTION_SHA,

    "development": {
        "baseline_score":
            EXPECTED_DEV_BASE_SCORE,

        "current_best_score":
            EXPECTED_DEV_POLICY_SCORE,

        "delta":
            EXPECTED_DEV_DELTA,
    },

    "locked_fold0": {
        "baseline_edge_tp":
            EXPECTED_BASE_TP,

        "baseline_edge_fp":
            EXPECTED_BASE_FP,

        "baseline_edge_fn":
            EXPECTED_BASE_FN,

        "baseline_score":
            float(
                fold0[
                    "baseline"
                ][
                    "score"
                ]
            ),

        "current_best_edge_tp":
            EXPECTED_POLICY_TP,

        "current_best_edge_fp":
            EXPECTED_POLICY_FP,

        "current_best_edge_fn":
            EXPECTED_POLICY_FN,

        "current_best_score":
            float(
                fold0[
                    "policy"
                ][
                    "score"
                ]
            ),

        "score_delta":
            float(
                fold0[
                    "delta"
                ][
                    "score"
                ]
            ),

        "confirmed":
            True,
    },

    "evaluation_protocol": {
        "development_closed_before_fold0":
            True,

        "fold0_actions_frozen_before_gt":
            True,

        "one_shot_locked_confirmation":
            True,

        "post_fold0_retuning":
            False,
    },

    "runtime_policy":
        policy[
            "runtime_contract"
        ],

    "next":
        "CANONICAL_BUILDER_INTEGRATION_AND_REPRODUCTION",
}


NEW_CHECKPOINT_SHA = hashlib.sha256(
    json.dumps(
        NEW_CHECKPOINT,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


NEW_CHECKPOINT[
    "checkpoint_sha256"
] = NEW_CHECKPOINT_SHA


NEW_CURRENT_BEST_OUT.write_text(
    json.dumps(
        NEW_CHECKPOINT,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 10. PROMOTION SUMMARY
# ============================================================

SUMMARY = {
    "stage":
        "12.11G3H3E1",

    "status":
        "CONFIRMED_REWIRE_POLICY_PROMOTION_COMPLETE",

    "policy_id":
        POLICY_ID,

    "policy_sha256":
        POLICY_SHA,

    "promotion_sha256":
        PROMOTION_SHA,

    "new_current_best_checkpoint":
        NEW_CURRENT_BEST_OUT.name,

    "new_current_best_checkpoint_sha256":
        NEW_CHECKPOINT_SHA,

    "development_delta":
        EXPECTED_DEV_DELTA,

    "locked_fold0_delta":
        float(
            fold0[
                "delta"
            ][
                "score"
            ]
        ),

    "locked_fold0_edge_delta": {
        "tp":
            0,

        "fp":
            -2,

        "fn":
            0,
    },

    "locked_fold0_confirmed":
        True,

    "development_reopened":
        False,

    "post_fold0_retuning_allowed":
        False,

    "historical_checkpoint_overwritten":
        False,

    "prediction_geff_written":
        False,

    "next":
        "INTEGRATE_CONFIRMED_POLICY_INTO_CANONICAL_CURRENT_BEST_BUILDER",
}


SUMMARY_OUT.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 11. RELOAD INTEGRITY
# ============================================================

promotion_check = json.loads(
    PROMOTION_OUT.read_text(
        encoding="utf-8"
    )
)


checkpoint_check = json.loads(
    NEW_CURRENT_BEST_OUT.read_text(
        encoding="utf-8"
    )
)


summary_check = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert promotion_check[
    "status"
] == (
    "CONFIRMED_POLICY_PROMOTED"
)


assert promotion_check[
    "promotion_sha256"
] == PROMOTION_SHA


assert promotion_check[
    "post_fold0_retuning_allowed"
] is False


assert checkpoint_check[
    "checkpoint_sha256"
] == NEW_CHECKPOINT_SHA


assert checkpoint_check[
    "locked_fold0"
][
    "confirmed"
] is True


assert checkpoint_check[
    "locked_fold0"
][
    "current_best_edge_tp"
] == 22806


assert checkpoint_check[
    "locked_fold0"
][
    "current_best_edge_fp"
] == 2656


assert checkpoint_check[
    "locked_fold0"
][
    "current_best_edge_fn"
] == 3425


assert summary_check[
    "locked_fold0_confirmed"
] is True


assert summary_check[
    "development_reopened"
] is False


assert summary_check[
    "post_fold0_retuning_allowed"
] is False


assert OLD_CURRENT_BEST_PATH.exists()


# ============================================================
# 12. FINAL
# ============================================================

print(
    "\n========== 12.11G3H3E1 PROMOTION =========="
)


print(
    "CONFIRMED_POLICY_PROMOTION: PASS"
)


print(
    "policy:",
    POLICY_ID
)


print(
    "promotion SHA:",
    PROMOTION_SHA
)


print(
    "development score:",
    f"{EXPECTED_DEV_BASE_SCORE:.12f}",
    "->",
    f"{EXPECTED_DEV_POLICY_SCORE:.12f}",
    (
        f"({EXPECTED_DEV_DELTA:+.12f})"
    ),
)


print(
    "locked Fold0 score:",
    f"{float(fold0['baseline']['score']):.12f}",
    "->",
    f"{float(fold0['policy']['score']):.12f}",
    (
        f"({float(fold0['delta']['score']):+.12f})"
    ),
)


print(
    "locked Fold0 TP/FP/FN delta:",
    "0 / -2 / 0"
)


print(
    "new CURRENT_BEST checkpoint:",
    NEW_CURRENT_BEST_OUT.name
)


print(
    "new checkpoint SHA:",
    NEW_CHECKPOINT_SHA
)


print(
    "historical CURRENT_BEST overwritten:",
    "NO"
)


print(
    "development reopened:",
    "NO"
)


print(
    "post-Fold0 retuning:",
    "NO"
)


print(
    "Fold0 confirmation consumed:",
    "YES"
)


print(
    "prediction GEFF written:",
    "NO"
)


print(
    "next:"
)


print(
    "INTEGRATE_CONFIRMED_POLICY_INTO_CANONICAL_CURRENT_BEST_BUILDER"
)

In [ ]:
# ============================================================
# Stage 12.11G3H3E2A
#
# CANONICAL RUNTIME PACKAGING PREFLIGHT
#
# PURPOSE
# ------------------------------------------------------------
# The policy is already confirmed and promoted.
#
# Before modifying canonical CURRENT_BEST code, recover the
# exact deployment dependency graph for the frozen runtime:
#
#   generate_dataset_proposals
#   add_endpoint_ranks
#   add_shortlist_rank_fields
#   frozen_shortlist_mask
#   attach_scoring_features
#   endpoint_score_ranks_g3h3
#   policy_mask_g3h3
#   resolve_conflicts_g3h3
#
# Also inspect:
#
#   - exact source provenance
#   - referenced globals
#   - module/file ownership
#   - canonical CURRENT_BEST builder
#   - existing source implementations
#
# NO:
#   source modification
#   artifact modification
#   graph evaluation
#   GT
#   Fold0
#   policy tuning
#
# TEMPORARY ENGINEERING PREFLIGHT
# ============================================================

from pathlib import Path

import ast
import inspect
import json
import re
import sys
import types


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


NOTEBOOK = (
    PROJECT
    / "notebooks"
    / "12_residual_edge_recovery.ipynb"
)


SRC_ROOT = (
    PROJECT
    / "src"
)


SCRIPTS_ROOT = (
    PROJECT
    / "scripts"
)


CANONICAL_BUILDER_SOURCE = (
    PROJECT
    / "scripts"
    / "run_stage12_current_best_edge_residual_census.py"
)


assert NOTEBOOK.exists(), NOTEBOOK
assert SRC_ROOT.exists(), SRC_ROOT
assert SCRIPTS_ROOT.exists(), SCRIPTS_ROOT
assert CANONICAL_BUILDER_SOURCE.exists()


# ============================================================
# 1. FROZEN RUNTIME HELPERS
# ============================================================

RUNTIME_HELPERS = [
    "generate_dataset_proposals",
    "add_endpoint_ranks",
    "add_shortlist_rank_fields",
    "frozen_shortlist_mask",
    "attach_scoring_features",
    "endpoint_score_ranks_g3h3",
    "policy_mask_g3h3",
    "resolve_conflicts_g3h3",
]


missing = [
    name
    for name in RUNTIME_HELPERS
    if (
        name not in globals()
        or not callable(
            globals()[name]
        )
    )
]


assert not missing, (
    "Frozen runtime helper missing from current kernel: "
    f"{missing}"
)


print(
    "========== G3H3E2A RUNTIME STATE =========="
)


print(
    "helpers:",
    len(
        RUNTIME_HELPERS
    )
)


print(
    "missing:",
    missing
)


print(
    "policy tuning:",
    "NO"
)


# ============================================================
# 2. LIVE FUNCTION PROVENANCE
# ============================================================

live_rows = []


print(
    "\n========== LIVE HELPER PROVENANCE =========="
)


for name in RUNTIME_HELPERS:

    fn = globals()[
        name
    ]


    module = getattr(
        fn,
        "__module__",
        None,
    )


    filename = (
        fn.__code__.co_filename
        if hasattr(
            fn,
            "__code__"
        )
        else None
    )


    first_line = (
        fn.__code__.co_firstlineno
        if hasattr(
            fn,
            "__code__"
        )
        else None
    )


    try:

        source = inspect.getsource(
            fn
        )

        source_available = True


    except Exception as exc:

        source = None

        source_available = False


    live_rows.append(
        {
            "name":
                name,

            "module":
                module,

            "filename":
                filename,

            "first_line":
                first_line,

            "source_available":
                source_available,

            "source":
                source,
        }
    )


    print()
    print(
        "------------------------------------------------------------"
    )

    print(
        name
    )

    print(
        "module:",
        module
    )

    print(
        "filename:",
        filename
    )

    print(
        "first line:",
        first_line
    )

    print(
        "source available:",
        source_available
    )


# ============================================================
# 3. LIVE GLOBAL DEPENDENCY AUDIT
#
# inspect.getclosurevars() gives actual globals referenced by
# each live function.
# ============================================================

dependency_rows = []


print(
    "\n========== LIVE HELPER GLOBAL DEPENDENCIES =========="
)


for name in RUNTIME_HELPERS:

    fn = globals()[
        name
    ]


    closure = inspect.getclosurevars(
        fn
    )


    referenced_globals = sorted(
        closure.globals.keys()
    )


    referenced_nonlocals = sorted(
        closure.nonlocals.keys()
    )


    unbound = sorted(
        closure.unbound
    )


    print()
    print(
        "------------------------------------------------------------"
    )

    print(
        name
    )

    print(
        "globals:"
    )


    for dep in referenced_globals:

        value = closure.globals[
            dep
        ]


        print(
            f"  {dep:40s}"
            f" | type={type(value).__name__}"
            f" | module={getattr(value, '__module__', None)}"
        )


        dependency_rows.append(
            {
                "function":
                    name,

                "dependency":
                    dep,

                "dependency_type":
                    type(
                        value
                    ).__name__,

                "dependency_module":
                    getattr(
                        value,
                        "__module__",
                        None,
                    ),

                "dependency_file":
                    (
                        getattr(
                            getattr(
                                value,
                                "__code__",
                                None,
                            ),
                            "co_filename",
                            None,
                        )
                    ),
            }
        )


    print(
        "nonlocals:",
        referenced_nonlocals
    )

    print(
        "unbound:",
        unbound
    )


# ============================================================
# 4. UNIQUE CALLABLE DEPENDENCIES
# ============================================================

all_dependency_names = sorted(
    {
        row[
            "dependency"
        ]

        for row
        in dependency_rows
    }
)


print(
    "\n========== UNIQUE RUNTIME GLOBALS =========="
)


for name in all_dependency_names:

    value = globals().get(
        name,
        None,
    )


    print(
        name,
        "|",
        type(
            value
        ).__name__
        if value is not None
        else "NOT_IN_MAIN_GLOBALS",
    )


# ============================================================
# 5. NOTEBOOK EXACT DEFINITION LOCATOR
# ============================================================

nb = json.loads(
    NOTEBOOK.read_text(
        encoding="utf-8"
    )
)


definition_hits = {
    name:
        []

    for name
    in RUNTIME_HELPERS
}


for cell_idx, cell in enumerate(
    nb.get(
        "cells",
        []
    )
):

    if cell.get(
        "cell_type"
    ) != "code":

        continue


    source = "".join(
        cell.get(
            "source",
            []
        )
    )


    try:

        tree = ast.parse(
            source
        )

    except SyntaxError:

        continue


    for node in tree.body:

        if not isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        ):

            continue


        if node.name not in definition_hits:
            continue


        segment = ast.get_source_segment(
            source,
            node,
        )


        definition_hits[
            node.name
        ].append(
            {
                "cell":
                    cell_idx,

                "line":
                    node.lineno,

                "source":
                    segment,
            }
        )


print(
    "\n========== NOTEBOOK HELPER DEFINITIONS =========="
)


for name in RUNTIME_HELPERS:

    hits = definition_hits[
        name
    ]


    print()
    print(
        name,
        ":",
        len(
            hits
        ),
        "definition(s)"
    )


    for hit in hits:

        print(
            "  cell=",
            hit[
                "cell"
            ],
            " line=",
            hit[
                "line"
            ],
        )


# ============================================================
# 6. SEARCH src/ FOR EXISTING IMPLEMENTATIONS
# ============================================================

source_hits = {
    name:
        []

    for name
    in RUNTIME_HELPERS
}


for path in sorted(
    SRC_ROOT.rglob(
        "*.py"
    )
):

    try:

        text = path.read_text(
            encoding="utf-8"
        )

    except Exception:

        continue


    try:

        tree = ast.parse(
            text
        )

    except SyntaxError:

        continue


    for node in tree.body:

        if not isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        ):

            continue


        if node.name in source_hits:

            source_hits[
                node.name
            ].append(
                {
                    "path":
                        str(
                            path
                        ),

                    "line":
                        node.lineno,
                }
            )


print(
    "\n========== EXISTING src/ IMPLEMENTATIONS =========="
)


for name in RUNTIME_HELPERS:

    hits = source_hits[
        name
    ]


    print(
        name,
        ":",
        (
            hits
            if hits
            else "NONE"
        )
    )


# ============================================================
# 7. RECURSIVE CALL GRAPH FROM LIVE FUNCTIONS
#
# This identifies helper-to-helper dependencies in the current
# namespace that may also need packaging.
# ============================================================

def directly_called_names(
    fn,
):

    try:

        source = inspect.getsource(
            fn
        )

    except Exception:

        return []


    try:

        tree = ast.parse(
            source
        )

    except SyntaxError:

        return []


    calls = set()


    for node in ast.walk(
        tree
    ):

        if isinstance(
            node,
            ast.Call,
        ):

            if isinstance(
                node.func,
                ast.Name,
            ):

                calls.add(
                    node.func.id
                )


            elif isinstance(
                node.func,
                ast.Attribute,
            ):

                calls.add(
                    node.func.attr
                )


    return sorted(
        calls
    )


print(
    "\n========== DIRECT CALL GRAPH =========="
)


for name in RUNTIME_HELPERS:

    calls = directly_called_names(
        globals()[
            name
        ]
    )


    print(
        name,
        "->",
        calls
    )


# ============================================================
# 8. CANONICAL CURRENT_BEST BUILDER SOURCE
# ============================================================

canonical_source = (
    CANONICAL_BUILDER_SOURCE.read_text(
        encoding="utf-8"
    )
)


canonical_tree = ast.parse(
    canonical_source
)


builder_node = None


for node in canonical_tree.body:

    if (
        isinstance(
            node,
            ast.FunctionDef,
        )
        and
        node.name
        ==
        "build_current_best"
    ):

        builder_node = node

        break


assert builder_node is not None


builder_source = ast.get_source_segment(
    canonical_source,
    builder_node,
)


assert builder_source


print(
    "\n============================================================"
)


print(
    "CANONICAL CURRENT_BEST BUILDER"
)


print(
    "============================================================"
)


print(
    "path:",
    CANONICAL_BUILDER_SOURCE
)


print(
    "line:",
    builder_node.lineno
)


print(
    "end line:",
    builder_node.end_lineno
)


print()


print(
    builder_source
)


# ============================================================
# 9. CANONICAL BUILDER DEPENDENCY NAMES
# ============================================================

builder_calls = set()


builder_ast = ast.parse(
    builder_source
)


for node in ast.walk(
    builder_ast
):

    if not isinstance(
        node,
        ast.Call,
    ):

        continue


    fn = node.func


    if isinstance(
        fn,
        ast.Name,
    ):

        builder_calls.add(
            fn.id
        )


    elif isinstance(
        fn,
        ast.Attribute,
    ):

        builder_calls.add(
            fn.attr
        )


print(
    "\n========== CANONICAL BUILDER CALLS =========="
)


print(
    sorted(
        builder_calls
    )
)


# ============================================================
# 10. SCORER / POLICY ARTIFACT INVENTORY
# ============================================================

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


artifact_names = [
    "stage12_rewire_asym1x5_q9990_no_target_cut_v1_frozen.json",

    "stage12_rewire_asym1x5_q9990_no_target_cut_v1_deployment_threshold.json",

    "stage12_folds14_rewire_scoring_feature_schema_v1_frozen.json",

    "stage12_folds14_rewire_logreg_scorer_v1_preprocessor.npz",

    "stage12_folds14_rewire_logreg_scorer_v1_model.joblib",

    "stage12_rewire_asym1x5_q9990_no_target_cut_v1_promoted.json",

    "stage12_current_best_checkpoint.post_g3h3e1.json",
]


print(
    "\n========== DEPLOYMENT ARTIFACT INVENTORY =========="
)


for filename in artifact_names:

    path = (
        S12
        / filename
    )


    print(
        filename,
        ":",
        (
            "FOUND"
            if path.exists()
            else "MISSING"
        ),
        (
            f"| {path.stat().st_size} bytes"
            if path.exists()
            else ""
        )
    )


assert all(
    (
        S12
        / name
    ).exists()

    for name
    in artifact_names
)


# ============================================================
# 11. CHECK WHETHER CURRENT CANONICAL SOURCE ALREADY CONTAINS
#     PROMOTED POLICY
# ============================================================

PROMOTED_MARKERS = [
    "ASYM1x5_Q9990_NO_TARGET_CUT_V1",
    "0.9952878577709198",
    "NO_TARGET_CUT",
]


print(
    "\n========== CANONICAL INTEGRATION STATUS =========="
)


for marker in PROMOTED_MARKERS:

    print(
        marker,
        ":",
        (
            "PRESENT"
            if marker
            in canonical_source
            else "ABSENT"
        )
    )


assert not all(
    marker
    in canonical_source

    for marker
    in PROMOTED_MARKERS
), (
    "Canonical builder appears already integrated. "
    "Do not blindly integrate twice."
)


# ============================================================
# 12. CLASSIFY DEPENDENCIES
# ============================================================

dependency_class_rows = []


for row in dependency_rows:

    dep = row[
        "dependency"
    ]


    value = globals().get(
        dep,
        None,
    )


    if isinstance(
        value,
        types.ModuleType,
    ):

        cls = "MODULE"


    elif callable(
        value
    ):

        filename = (
            value.__code__.co_filename
            if hasattr(
                value,
                "__code__"
            )
            else None
        )


        if (
            filename
            and
            str(
                PROJECT
                / "src"
            )
            in str(
                filename
            )
        ):

            cls = "SOURCE_CALLABLE"


        else:

            cls = "NOTEBOOK_OR_RUNTIME_CALLABLE"


    else:

        cls = "RUNTIME_CONSTANT_OR_ARTIFACT"


    dependency_class_rows.append(
        {
            **row,

            "classification":
                cls,
        }
    )


dependency_df = pd.DataFrame(
    dependency_class_rows
)


print(
    "\n========== DEPENDENCY CLASSIFICATION =========="
)


if len(
    dependency_df
):

    print(
        dependency_df[
            [
                "function",
                "dependency",
                "dependency_type",
                "classification",
                "dependency_file",
            ]
        ].to_string(
            index=False
        )
    )


# ============================================================
# 13. PACKAGING RISK SUMMARY
# ============================================================

runtime_callable_deps = sorted(
    set(
        dependency_df.loc[
            dependency_df[
                "classification"
            ].eq(
                "NOTEBOOK_OR_RUNTIME_CALLABLE"
            ),
            "dependency",
        ].tolist()
    )
)


runtime_constant_deps = sorted(
    set(
        dependency_df.loc[
            dependency_df[
                "classification"
            ].eq(
                "RUNTIME_CONSTANT_OR_ARTIFACT"
            ),
            "dependency",
        ].tolist()
    )
)


print(
    "\n========== PACKAGING RISK SUMMARY =========="
)


print(
    "notebook/runtime callable dependencies:"
)


for name in runtime_callable_deps:

    print(
        "  -",
        name
    )


print(
    "runtime constant/artifact dependencies:"
)


for name in runtime_constant_deps:

    print(
        "  -",
        name
    )


# ============================================================
# 14. FINAL
# ============================================================

print(
    "\n========== 12.11G3H3E2A DECISION =========="
)


print(
    "CANONICAL_RUNTIME_PACKAGING_PREFLIGHT: PASS"
)


print(
    "confirmed policy:",
    "ASYM1x5_Q9990_NO_TARGET_CUT_V1"
)


print(
    "canonical builder located:",
    "YES"
)


print(
    "runtime helper definitions located:",
    "YES"
)


print(
    "deployment artifacts complete:",
    "YES"
)


print(
    "source files modified:",
    "NO"
)


print(
    "GT used:",
    "NO"
)


print(
    "Fold0 evaluated:",
    "NO"
)


print(
    "development reopened:",
    "NO"
)


print(
    "next:"
)


print(
    "PACKAGE_FROZEN_RUNTIME_AND_INTEGRATE_CANONICAL_CURRENT_BEST"
)

In [ ]:
# ============================================================
# Stage 12.11G3H3E2B
#
# FROZEN RUNTIME PACKAGING
# + CANONICAL CURRENT_BEST INTEGRATION
#
# ============================================================
#
# SCIENTIFIC STATUS
#
# Policy already promoted:
#
#   ASYM1x5_Q9990_NO_TARGET_CUT_V1
#
# Development:
#   +0.000050616471
#
# Locked Fold0:
#   +0.000054118252
#   TP/FP/FN delta = 0 / -2 / 0
#
# Fold0 has been consumed.
#
# THIS STAGE IS ENGINEERING ONLY.
#
# ------------------------------------------------------------
# GOALS
# ------------------------------------------------------------
#
# 1. Extract the EXACT confirmed live runtime implementation.
#
# 2. Recursively package notebook-only helper dependencies
#    into:
#
#       src/biohub_cell_tracking/
#           stage12_confirmed_rewire.py
#
# 3. Load frozen scorer/preprocessor/schema from immutable
#    Stage12 artifacts.
#
# 4. Reproduce frozen Fold0 ACTIONS only, without GT, on two
#    smoke datasets.
#
# 5. Only after action fidelity passes:
#
#    preserve canonical builder source
#    and minimally integrate the confirmed policy.
#
# 6. Write an integration manifest with before/after SHAs.
#
# ------------------------------------------------------------
# NO
# ------------------------------------------------------------
#
# GT
# official metric
# Fold0 selection
# scorer retraining
# threshold tuning
# rank-rule tuning
# structural-rule tuning
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import ast
import hashlib
import inspect
import json
import shutil
import subprocess
import sys
import textwrap
import types

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


SRC_ROOT = (
    PROJECT
    / "src"
)


PACKAGE_ROOT = (
    SRC_ROOT
    / "biohub_cell_tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


CANONICAL_PATH = (
    PROJECT
    / "scripts"
    / "run_stage12_current_best_edge_residual_census.py"
)


CANONICAL_BACKUP = (
    PROJECT
    / "scripts"
    / "run_stage12_current_best_edge_residual_census.pre_g3h3e1.py"
)


RUNTIME_PATH = (
    PACKAGE_ROOT
    / "stage12_confirmed_rewire.py"
)


INTEGRATION_MANIFEST_OUT = (
    S12
    / "stage12_g3h3e2b_canonical_runtime_integration.json"
)


FROZEN_FOLD0_ACTIONS_PATH = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_actions_locked.pkl"
)


FINAL_POLICY_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_frozen.json"
)


DEPLOYMENT_THRESHOLD_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_deployment_threshold.json"
)


FEATURE_SCHEMA_PATH = (
    S12
    / "stage12_folds14_rewire_scoring_feature_schema_v1_frozen.json"
)


MODEL_PATH = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_model.joblib"
)


PREPROCESSOR_PATH = (
    S12
    / "stage12_folds14_rewire_logreg_scorer_v1_preprocessor.npz"
)


PROMOTION_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_promoted.json"
)


for path in [
    SRC_ROOT,
    PACKAGE_ROOT,
    S12,
    CANONICAL_PATH,

    FROZEN_FOLD0_ACTIONS_PATH,

    FINAL_POLICY_PATH,
    DEPLOYMENT_THRESHOLD_PATH,
    FEATURE_SCHEMA_PATH,
    MODEL_PATH,
    PREPROCESSOR_PATH,
    PROMOTION_PATH,
]:

    assert path.exists(), path


for path in [
    RUNTIME_PATH,
    CANONICAL_BACKUP,
    INTEGRATION_MANIFEST_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3H3E2B output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite formal integration provenance."
        )


# ============================================================
# 1. AUTHORITATIVE FROZEN IDENTIFIERS
# ============================================================

POLICY_ID = (
    "ASYM1x5_Q9990_NO_TARGET_CUT_V1"
)


POLICY_SHA = (
    "77f0b07cafb21d0705917ff71b750fde4"
    "5f3c7ddc17c13fecaff0ac673ebf776"
)


DEPLOYMENT_THRESHOLD = (
    0.9952878577709198
)


MODEL_SHA = (
    "00918fa386d4de6e6a5f93892e47d876"
    "e6ff45ceb89670f982ebba45f59ebf76"
)


PREPROCESSOR_SHA = (
    "6b5425093efd17613c68fcba855a756a3"
    "445a286a61ac813a83652dd2200ae41"
)


FEATURE_SCHEMA_SHA = (
    "0a9f8e9a41c5c80c803c3a95955702dc"
    "f4039605a21fc656a154aa05be55b39c"
)


# ============================================================
# 2. SHA HELPER
# ============================================================

def sha256_file(
    path,
):

    h = hashlib.sha256()


    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )


            if not chunk:
                break


            h.update(
                chunk
            )


    return h.hexdigest()


# ============================================================
# 3. VERIFY PROMOTED POLICY
# ============================================================

policy = json.loads(
    FINAL_POLICY_PATH.read_text(
        encoding="utf-8"
    )
)


promotion = json.loads(
    PROMOTION_PATH.read_text(
        encoding="utf-8"
    )
)


threshold_contract = json.loads(
    DEPLOYMENT_THRESHOLD_PATH.read_text(
        encoding="utf-8"
    )
)


feature_schema = json.loads(
    FEATURE_SCHEMA_PATH.read_text(
        encoding="utf-8"
    )
)


assert policy[
    "policy_id"
] == POLICY_ID


assert policy[
    "final_policy_sha256"
] == POLICY_SHA


assert policy[
    "development_closed_after_freeze"
] is True


assert promotion[
    "status"
] == (
    "CONFIRMED_POLICY_PROMOTED"
)


assert promotion[
    "policy_id"
] == POLICY_ID


assert promotion[
    "post_fold0_retuning_allowed"
] is False


assert float(
    threshold_contract[
        "threshold"
    ]
) == DEPLOYMENT_THRESHOLD


assert feature_schema[
    "feature_schema_sha256"
] == FEATURE_SCHEMA_SHA


assert sha256_file(
    MODEL_PATH
) == MODEL_SHA


assert sha256_file(
    PREPROCESSOR_PATH
) == PREPROCESSOR_SHA


FEATURE_COLUMNS_LIVE = list(
    feature_schema[
        "features"
    ]
)


assert len(
    FEATURE_COLUMNS_LIVE
) == 32


print(
    "========== G3H3E2B PROMOTION CONTRACT =========="
)


print(
    "policy:",
    POLICY_ID
)


print(
    "policy SHA:",
    POLICY_SHA
)


print(
    "deployment threshold:",
    DEPLOYMENT_THRESHOLD
)


print(
    "development closed:",
    "YES"
)


print(
    "Fold0 consumed:",
    "YES"
)


print(
    "post-Fold0 tuning:",
    "NO"
)


# ============================================================
# 4. ROOT FROZEN LIVE FUNCTIONS
# ============================================================

ROOT_FUNCTION_NAMES = [
    "generate_dataset_proposals",
    "add_endpoint_ranks",
    "add_shortlist_rank_fields",
    "frozen_shortlist_mask",
    "attach_scoring_features",

    "endpoint_score_ranks_g3h3",
    "policy_mask_g3h3",
    "resolve_conflicts_g3h3",
]


for name in ROOT_FUNCTION_NAMES:

    assert name in globals(), name

    assert callable(
        globals()[
            name
        ]
    ), name


# build_current_best is intentionally NOT packaged.
#
# Runtime proposal generation receives the canonical
# pre-G3H3E1 builder via controlled dependency injection.
SPECIAL_INJECTED_GLOBALS = {
    "build_current_best",
    "current_best_builder",
}


# ============================================================
# 5. RECURSIVELY COLLECT NOTEBOOK FUNCTION DEPENDENCIES
#
# Any callable whose module is __main__ is treated as part of
# the frozen notebook implementation and packaged verbatim.
# ============================================================

function_objects = {}

constant_objects = {}

module_imports = {}

external_imports = {}


queue = list(
    ROOT_FUNCTION_NAMES
)


visited = set()


while queue:

    name = queue.pop(
        0
    )


    if name in visited:
        continue


    visited.add(
        name
    )


    assert name in globals(), (
        "Runtime dependency unavailable in live kernel: "
        f"{name}"
    )


    fn = globals()[
        name
    ]


    assert callable(
        fn
    ), (
        name,
        type(
            fn
        ),
    )


    try:

        source = inspect.getsource(
            fn
        )


    except Exception as exc:

        raise RuntimeError(
            f"Unable to extract exact source for {name}: {exc}"
        )


    source = textwrap.dedent(
        source
    )


    function_objects[
        name
    ] = {
        "object":
            fn,

        "source":
            source,

        "filename":
            getattr(
                getattr(
                    fn,
                    "__code__",
                    None,
                ),
                "co_filename",
                None,
            ),

        "firstlineno":
            getattr(
                getattr(
                    fn,
                    "__code__",
                    None,
                ),
                "co_firstlineno",
                None,
            ),
    }


    closure = inspect.getclosurevars(
        fn
    )


    for dep_name, dep_value in closure.globals.items():

        # ----------------------------------------------------
        # Current-best builder is deliberately runtime-injected.
        # ----------------------------------------------------

        if dep_name in SPECIAL_INJECTED_GLOBALS:
            continue


        # ----------------------------------------------------
        # Modules
        # ----------------------------------------------------

        if isinstance(
            dep_value,
            types.ModuleType,
        ):

            module_imports[
                dep_name
            ] = dep_value.__name__

            continue


        # ----------------------------------------------------
        # Notebook/runtime callables.
        # ----------------------------------------------------

        if callable(
            dep_value
        ):

            dep_module = getattr(
                dep_value,
                "__module__",
                None,
            )


            if dep_module == "__main__":

                if dep_name not in visited:

                    queue.append(
                        dep_name
                    )


                continue


            # -----------------------------------------------
            # Builtins need no explicit import.
            # -----------------------------------------------

            if dep_module == "builtins":
                continue


            import_name = getattr(
                dep_value,
                "__name__",
                None,
            )


            if (
                dep_module
                and
                import_name
            ):

                external_imports[
                    dep_name
                ] = (
                    dep_module,
                    import_name,
                )

                continue


            raise RuntimeError(
                "Unsupported external callable dependency:\n"
                f"{dep_name}: {dep_value}"
            )


        # ----------------------------------------------------
        # Runtime constants.
        # ----------------------------------------------------

        constant_objects[
            dep_name
        ] = dep_value


print(
    "\n========== RECURSIVE FROZEN SOURCE CLOSURE =========="
)


print(
    "functions:",
    len(
        function_objects
    )
)


for name, info in function_objects.items():

    print(
        f"  {name:40s}"
        f" | {info['filename']}"
        f":{info['firstlineno']}"
    )


print(
    "\nconstants:",
    sorted(
        constant_objects
    )
)


print(
    "\nmodule imports:",
    module_imports
)


print(
    "\nexternal imports:",
    external_imports
)


assert set(
    ROOT_FUNCTION_NAMES
).issubset(
    function_objects
)


# ============================================================
# 6. SERIALIZE FROZEN CONSTANTS EXACTLY
# ============================================================

def render_constant(
    name,
    value,
):

    if isinstance(
        value,
        np.ndarray,
    ):

        return (
            f"{name} = np.array("
            f"{repr(value.tolist())}, "
            f"dtype=np.dtype({repr(str(value.dtype))})"
            f")"
        )


    if isinstance(
        value,
        np.generic,
    ):

        return (
            f"{name} = "
            f"np.dtype({repr(str(value.dtype))}).type("
            f"{repr(value.item())}"
            f")"
        )


    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
            type(None),
            list,
            tuple,
            dict,
            set,
        ),
    ):

        return (
            f"{name} = "
            f"{repr(value)}"
        )


    raise TypeError(
        "\nUnsupported frozen constant:\n"
        f"{name}: {type(value)}\n"
        f"{repr(value)[:500]}"
    )


constant_source_lines = []


for name in sorted(
    constant_objects
):

    constant_source_lines.append(
        render_constant(
            name,
            constant_objects[
                name
            ],
        )
    )


# ============================================================
# 7. FUNCTION-SOURCE HASH MANIFEST
# ============================================================

function_source_sha = {}


for name, info in function_objects.items():

    function_source_sha[
        name
    ] = hashlib.sha256(
        info[
            "source"
        ].encode(
            "utf-8"
        )
    ).hexdigest()


print(
    "\n========== FROZEN FUNCTION SOURCE SHAS =========="
)


for name in sorted(
    function_source_sha
):

    print(
        name,
        function_source_sha[
            name
        ]
    )


# ============================================================
# 8. BUILD FORMAL RUNTIME MODULE
# ============================================================

module_lines = []


module_lines.extend(
    [
        '"""',
        "Frozen Stage12 confirmed rewire runtime.",
        "",
        "Generated by Stage 12.11G3H3E2B from the exact",
        "live functions that produced the locked Fold0",
        "confirmation.",
        "",
        "Policy:",
        f"    {POLICY_ID}",
        "",
        "This module is engineering packaging only.",
        "Scientific selection is closed.",
        '"""',
        "",
        "from __future__ import annotations",
        "",
        "from functools import lru_cache",
        "from pathlib import Path",
        "import hashlib",
        "import json",
        "import gc",
        "",
        "import joblib",
        "import numpy as np",
        "import pandas as pd",
        "",
        "from biohub_cell_tracking import frozen_v9 as fv9",
        "",
    ]
)


# ------------------------------------------------------------
# Additional module imports found recursively.
# Avoid duplicates already explicitly imported.
# ------------------------------------------------------------

already_module_aliases = {
    "np",
    "pd",
    "gc",
}


for alias, module_name in sorted(
    module_imports.items()
):

    if alias in already_module_aliases:
        continue


    if alias == module_name:

        module_lines.append(
            f"import {module_name}"
        )


    else:

        module_lines.append(
            f"import {module_name} as {alias}"
        )


# ------------------------------------------------------------
# External symbols.
# ------------------------------------------------------------

for alias, (
    module_name,
    symbol_name,
) in sorted(
    external_imports.items()
):

    if alias == symbol_name:

        module_lines.append(
            f"from {module_name} import {symbol_name}"
        )


    else:

        module_lines.append(
            f"from {module_name} import "
            f"{symbol_name} as {alias}"
        )


module_lines.extend(
    [
        "",
        "",
        "# ========================================================",
        "# FROZEN POLICY CONTRACT",
        "# ========================================================",
        "",
        f"POLICY_ID = {POLICY_ID!r}",
        f"POLICY_SHA256 = {POLICY_SHA!r}",
        f"DEPLOYMENT_THRESHOLD = {DEPLOYMENT_THRESHOLD!r}",
        f"MODEL_SHA256 = {MODEL_SHA!r}",
        f"PREPROCESSOR_SHA256 = {PREPROCESSOR_SHA!r}",
        f"FEATURE_SCHEMA_SHA256 = {FEATURE_SCHEMA_SHA!r}",
        "",
    ]
)


# ------------------------------------------------------------
# Constants from exact live closure.
#
# FEATURE_COLUMNS is explicitly pinned to the formal schema
# below even if the live closure already supplied the same list.
# ------------------------------------------------------------

for line in constant_source_lines:

    if line.startswith(
        "FEATURE_COLUMNS ="
    ):
        continue


    module_lines.append(
        line
    )


module_lines.extend(
    [
        "",
        f"FEATURE_COLUMNS = {FEATURE_COLUMNS_LIVE!r}",
        "",
        "",
        "# ========================================================",
        "# EXACT NOTEBOOK-DERIVED FUNCTIONS",
        "# ========================================================",
        "",
    ]
)


# ------------------------------------------------------------
# Function definition order does not affect normal Python
# global lookup at call time.
#
# Sort by source filename / first line to preserve historical
# ordering as closely as possible.
# ------------------------------------------------------------

ordered_functions = sorted(
    function_objects.items(),
    key=lambda item: (
        str(
            item[
                1
            ][
                "filename"
            ]
        ),
        int(
            item[
                1
            ][
                "firstlineno"
            ]
            or 0
        ),
        item[
            0
        ],
    ),
)


for name, info in ordered_functions:

    module_lines.append(
        info[
            "source"
        ].rstrip()
    )


    module_lines.extend(
        [
            "",
            "",
        ]
    )


# ============================================================
# 9. ADD ARTIFACT LOADER + FROZEN SCORER
# ============================================================

runtime_support_source = r'''
# ============================================================
# ARTIFACT INTEGRITY
# ============================================================

def _sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


@lru_cache(maxsize=4)
def _load_frozen_runtime(artifact_dir_string):
    artifact_dir = Path(artifact_dir_string)

    schema_path = (
        artifact_dir
        / "stage12_folds14_rewire_scoring_feature_schema_v1_frozen.json"
    )

    model_path = (
        artifact_dir
        / "stage12_folds14_rewire_logreg_scorer_v1_model.joblib"
    )

    prep_path = (
        artifact_dir
        / "stage12_folds14_rewire_logreg_scorer_v1_preprocessor.npz"
    )

    threshold_path = (
        artifact_dir
        / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_deployment_threshold.json"
    )

    policy_path = (
        artifact_dir
        / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_frozen.json"
    )

    for path in [
        schema_path,
        model_path,
        prep_path,
        threshold_path,
        policy_path,
    ]:
        if not path.exists():
            raise FileNotFoundError(path)

    if _sha256_file(model_path) != MODEL_SHA256:
        raise RuntimeError("Frozen scorer model SHA mismatch")

    if _sha256_file(prep_path) != PREPROCESSOR_SHA256:
        raise RuntimeError("Frozen scorer preprocessor SHA mismatch")

    schema = json.loads(
        schema_path.read_text(encoding="utf-8")
    )

    if schema["feature_schema_sha256"] != FEATURE_SCHEMA_SHA256:
        raise RuntimeError("Frozen feature schema SHA mismatch")

    if list(schema["features"]) != FEATURE_COLUMNS:
        raise RuntimeError("Frozen feature-column mismatch")

    threshold_contract = json.loads(
        threshold_path.read_text(encoding="utf-8")
    )

    if float(threshold_contract["threshold"]) != DEPLOYMENT_THRESHOLD:
        raise RuntimeError("Frozen deployment-threshold mismatch")

    policy = json.loads(
        policy_path.read_text(encoding="utf-8")
    )

    if policy["policy_id"] != POLICY_ID:
        raise RuntimeError("Frozen policy ID mismatch")

    if policy["final_policy_sha256"] != POLICY_SHA256:
        raise RuntimeError("Frozen policy SHA mismatch")

    prep = np.load(
        prep_path,
        allow_pickle=False,
    )

    model = joblib.load(
        model_path
    )

    if prep["feature_columns"].tolist() != FEATURE_COLUMNS:
        raise RuntimeError("Preprocessor feature-column mismatch")

    medians = prep["medians"].astype(
        np.float64,
        copy=True,
    )

    means = prep["means"].astype(
        np.float64,
        copy=True,
    )

    stds = prep["stds"].astype(
        np.float64,
        copy=True,
    )

    if model.coef_.shape != (1, 64):
        raise RuntimeError(
            f"Unexpected model shape: {model.coef_.shape}"
        )

    return {
        "model": model,
        "medians": medians,
        "means": means,
        "stds": stds,
    }


def _transform_frozen_full_dev(frame, runtime):
    X = (
        frame[FEATURE_COLUMNS]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .to_numpy(
            dtype=np.float64,
            copy=True,
        )
    )

    X[
        ~np.isfinite(X)
    ] = np.nan

    missing = np.isnan(X)

    filled = np.where(
        missing,
        runtime["medians"][None, :],
        X,
    )

    scaled = (
        filled
        -
        runtime["means"][None, :]
    ) / runtime["stds"][None, :]

    design = np.concatenate(
        [
            scaled,
            missing.astype(np.float64),
        ],
        axis=1,
    )

    if design.shape[1] != 64:
        raise RuntimeError(
            f"Unexpected design width: {design.shape}"
        )

    if not np.isfinite(design).all():
        raise RuntimeError("Nonfinite frozen design matrix")

    return design.astype(
        np.float32,
        copy=False,
    )


def _score_frozen_full_dev(
    frame,
    runtime,
    batch_size=50_000,
):
    scores = np.empty(
        len(frame),
        dtype=np.float64,
    )

    model = runtime["model"]

    for start in range(
        0,
        len(frame),
        batch_size,
    ):
        stop = min(
            start + batch_size,
            len(frame),
        )

        X = _transform_frozen_full_dev(
            frame.iloc[start:stop],
            runtime,
        )

        scores[start:stop] = (
            model.predict_proba(X)[:, 1]
        )

    if not np.isfinite(scores).all():
        raise RuntimeError("Nonfinite frozen model scores")

    return scores


# ============================================================
# CONTROLLED PRE-CURRENT-BEST BUILDER INJECTION
# ============================================================

_MISSING = object()


def _generate_with_exact_pre_rewire_builder(
    dataset,
    pre_rewire_builder,
):
    global build_current_best

    previous = globals().get(
        "build_current_best",
        _MISSING,
    )

    globals()[
        "build_current_best"
    ] = pre_rewire_builder

    try:
        result = generate_dataset_proposals(
            dataset
        )

    finally:
        if previous is _MISSING:
            globals().pop(
                "build_current_best",
                None,
            )
        else:
            globals()[
                "build_current_best"
            ] = previous

    if not isinstance(result, tuple):
        raise RuntimeError(
            "Frozen proposal generator did not return tuple"
        )

    return result


# ============================================================
# FROZEN ACTION GENERATION
# ============================================================

def build_confirmed_rewire_actions(
    dataset,
    *,
    pre_rewire_builder,
    artifact_dir,
):
    result = _generate_with_exact_pre_rewire_builder(
        dataset,
        pre_rewire_builder,
    )

    proposals = result[0]

    proposals = add_endpoint_ranks(
        proposals
    )

    proposals = add_shortlist_rank_fields(
        proposals
    )

    shortlist = (
        proposals[
            frozen_shortlist_mask(
                proposals
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    shortlist = attach_scoring_features(
        shortlist
    )

    runtime = _load_frozen_runtime(
        str(
            Path(artifact_dir).resolve()
        )
    )

    shortlist[
        "model_score"
    ] = _score_frozen_full_dev(
        shortlist,
        runtime,
    )

    ranks = endpoint_score_ranks_g3h3(
        shortlist
    )

    shortlist[
        "source_score_rank"
    ] = ranks[
        "source_score_rank"
    ].to_numpy(
        copy=True
    )

    shortlist[
        "target_score_rank"
    ] = ranks[
        "target_score_rank"
    ].to_numpy(
        copy=True
    )

    rank_mask = policy_mask_g3h3(
        shortlist,
        a=1,
        b=5,
    )

    score_gated = shortlist[
        rank_mask
        &
        shortlist[
            "model_score"
        ].ge(
            DEPLOYMENT_THRESHOLD
        )
    ].copy()

    resolved = (
        resolve_conflicts_g3h3(
            score_gated
        )
        .copy()
        .reset_index(
            drop=True
        )
    )

    target_occupied = pd.to_numeric(
        resolved[
            "target_occupied"
        ],
        errors="raise",
    ).astype(
        int
    )

    selected = (
        resolved[
            target_occupied.eq(0)
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    if not pd.to_numeric(
        selected[
            "target_occupied"
        ],
        errors="raise",
    ).astype(
        int
    ).eq(
        0
    ).all():
        raise RuntimeError(
            "NO_TARGET_CUT structural contract violated"
        )

    selected[
        "score_threshold"
    ] = DEPLOYMENT_THRESHOLD

    selected[
        "policy_id"
    ] = POLICY_ID

    return selected


# ============================================================
# GRAPH STATE
# ============================================================

def _runtime_graph_state(graph):
    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )

    outdeg = (
        edges.groupby("source_id")
        .size()
        .astype(int)
        .to_dict()
    )

    indeg = (
        edges.groupby("target_id")
        .size()
        .astype(int)
        .to_dict()
    )

    edge_set = set(
        zip(
            edges[
                "source_id"
            ].astype(int),

            edges[
                "target_id"
            ].astype(int),
        )
    )

    return (
        nodes,

        {
            int(k): int(v)
            for k, v
            in outdeg.items()
        },

        {
            int(k): int(v)
            for k, v
            in indeg.items()
        },

        edge_set,
    )


# ============================================================
# EXACT CONFIRMED GRAPH MUTATION
# ============================================================

def apply_confirmed_rewire_policy(
    dataset,
    graph,
    *,
    pre_rewire_builder,
    artifact_dir,
):
    if dataset.startswith("44b6_"):
        return (
            graph,
            {
                "actions": 0,
                "source_incumbent_edges_removed": 0,
                "edges_added": 0,
                "net_edge_delta": 0,
            },
        )

    if not dataset.startswith("6bba_"):
        raise ValueError(
            f"Unsupported dataset prefix: {dataset}"
        )

    actions = build_confirmed_rewire_actions(
        dataset,
        pre_rewire_builder=pre_rewire_builder,
        artifact_dir=artifact_dir,
    )

    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = _runtime_graph_state(
        graph
    )

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )

    edges["edge_id"] = edges["edge_id"].astype(int)
    edges["source_id"] = edges["source_id"].astype(int)
    edges["target_id"] = edges["target_id"].astype(int)

    outgoing = {}

    for src, sub in edges.groupby(
        "source_id"
    ):
        outgoing[int(src)] = [
            (
                int(row.edge_id),
                int(row.target_id),
            )
            for row
            in sub.itertuples(
                index=False
            )
        ]

    remove_ids = []
    add_rows = []

    for action in actions.itertuples(
        index=False
    ):
        src = int(action.source_id)
        tgt = int(action.target_id)

        if src == tgt:
            raise RuntimeError(
                "Self-edge in frozen action"
            )

        if (src, tgt) in edge_set:
            raise RuntimeError(
                "Frozen candidate already exists"
            )

        src_out = int(
            outdeg.get(
                src,
                0,
            )
        )

        tgt_in = int(
            indeg.get(
                tgt,
                0,
            )
        )

        # Frozen NO_TARGET_CUT.
        if tgt_in != 0:
            raise RuntimeError(
                (
                    "Target occupied under "
                    "NO_TARGET_CUT",
                    dataset,
                    src,
                    tgt,
                    tgt_in,
                )
            )

        if int(
            action.target_occupied
        ) != 0:
            raise RuntimeError(
                "Frozen action target_occupied != 0"
            )

        if src_out > 1:
            raise RuntimeError(
                (
                    "Unexpected source outdegree",
                    dataset,
                    src,
                    src_out,
                )
            )

        if int(
            action.source_occupied
        ) != int(
            src_out == 1
        ):
            raise RuntimeError(
                "source_occupied topology mismatch"
            )

        expected_cuts = int(
            src_out == 1
        )

        if int(
            action.cuts
        ) != expected_cuts:
            raise RuntimeError(
                "Frozen cuts topology mismatch"
            )

        if src_out == 1:
            src_edges = outgoing.get(
                src,
                [],
            )

            if len(src_edges) != 1:
                raise RuntimeError(
                    "Source incumbent cardinality mismatch"
                )

            remove_ids.append(
                int(
                    src_edges[0][0]
                )
            )

        dist = fv9.recovered_edge_dist(
            nodes,
            src,
            tgt,
            fv9.FROZEN_V9_CONFIG,
        )

        add_rows.append(
            (
                src,
                tgt,
                float(dist),
            )
        )

    if len(remove_ids) != len(
        set(remove_ids)
    ):
        raise RuntimeError(
            "Duplicate incumbent edge removals"
        )

    pre_edges = int(
        graph.num_edges()
    )

    if remove_ids:
        graph.bulk_remove_edges(
            sorted(remove_ids)
        )

    after_remove = int(
        graph.num_edges()
    )

    if (
        pre_edges
        -
        after_remove
    ) != len(remove_ids):
        raise RuntimeError(
            "Edge-removal count mismatch"
        )

    for src, tgt, dist in add_rows:
        if graph.has_edge(
            src,
            tgt,
        ):
            raise RuntimeError(
                "Candidate edge unexpectedly exists"
            )

        graph.add_edge(
            src,
            tgt,
            {
                "edge_dist":
                    float(dist),

                # Frozen evaluation representation contract.
                # Model score is NOT interpreted as calibrated
                # edge probability.
                "edge_prob":
                    1.0,
            },
            validate_keys=True,
        )

    post_edges = int(
        graph.num_edges()
    )

    if post_edges != (
        after_remove
        +
        len(add_rows)
    ):
        raise RuntimeError(
            "Post-mutation edge-count mismatch"
        )

    stats = {
        "actions":
            int(
                len(actions)
            ),

        "source_incumbent_edges_removed":
            int(
                len(remove_ids)
            ),

        "edges_added":
            int(
                len(add_rows)
            ),

        "net_edge_delta":
            int(
                post_edges
                -
                pre_edges
            ),
    }

    return graph, stats
'''


module_lines.append(
    runtime_support_source.strip()
)


module_lines.append(
    ""
)


module_source = "\n".join(
    module_lines
)


# ============================================================
# 10. STATIC COMPILE BEFORE WRITE
# ============================================================

compile(
    module_source,
    str(
        RUNTIME_PATH
    ),
    "exec",
)


print(
    "\n========== RUNTIME MODULE STATIC CHECK =========="
)


print(
    "compile:",
    "PASS"
)


# ============================================================
# 11. WRITE RUNTIME MODULE
# ============================================================

RUNTIME_PATH.write_text(
    module_source,
    encoding="utf-8",
)


assert RUNTIME_PATH.exists()


RUNTIME_SHA = sha256_file(
    RUNTIME_PATH
)


print(
    "runtime:",
    RUNTIME_PATH
)


print(
    "runtime SHA:",
    RUNTIME_SHA
)


# ============================================================
# 12. FRESH IMPORT RUNTIME MODULE
# ============================================================

if str(
    SRC_ROOT
) not in sys.path:

    sys.path.insert(
        0,
        str(
            SRC_ROOT
        ),
    )


module_name = (
    "biohub_cell_tracking.stage12_confirmed_rewire"
)


sys.modules.pop(
    module_name,
    None,
)


runtime_module = __import__(
    module_name,
    fromlist=[
        "*",
    ],
)


assert runtime_module.POLICY_ID == POLICY_ID


assert float(
    runtime_module.DEPLOYMENT_THRESHOLD
) == DEPLOYMENT_THRESHOLD


assert callable(
    runtime_module.build_confirmed_rewire_actions
)


assert callable(
    runtime_module.apply_confirmed_rewire_policy
)


print(
    "\n========== FRESH RUNTIME IMPORT =========="
)


print(
    "module:",
    runtime_module.__file__
)


print(
    "policy:",
    runtime_module.POLICY_ID
)


print(
    "FRESH_RUNTIME_IMPORT: PASS"
)


# ============================================================
# 13. EXACT PRE-G3H3E1 BUILDER FOR SMOKE FIDELITY
#
# Current kernel build_current_best is still the exact
# pre-integration builder that produced Fold0 actions.
# ============================================================

assert "build_current_best" in globals()

assert callable(
    build_current_best
)


pre_rewire_builder_live = build_current_best


# ============================================================
# 14. SELECT TWO GT-FREE ACTION-FIDELITY SMOKE DATASETS
#
# One minimum-action dataset.
# One dataset with maximum source-occupied selected actions.
# ============================================================

frozen_fold0_actions = pd.read_pickle(
    FROZEN_FOLD0_ACTIONS_PATH
)


assert len(
    frozen_fold0_actions
) == 2712


action_counts = (
    frozen_fold0_actions.groupby(
        "dataset"
    )
    .size()
    .sort_values()
)


source_counts = (
    frozen_fold0_actions.groupby(
        "dataset"
    )[
        "source_occupied"
    ]
    .sum()
    .sort_values(
        ascending=False
    )
)


smoke_datasets = []


smoke_datasets.append(
    str(
        action_counts.index[
            0
        ]
    )
)


smoke_datasets.append(
    str(
        source_counts.index[
            0
        ]
    )
)


smoke_datasets = list(
    dict.fromkeys(
        smoke_datasets
    )
)


print(
    "\n========== GT-FREE ACTION FIDELITY SMOKE =========="
)


print(
    "datasets:",
    smoke_datasets
)


smoke_rows = []


for dataset in smoke_datasets:

    packaged = (
        runtime_module
        .build_confirmed_rewire_actions(
            dataset,
            pre_rewire_builder=
                pre_rewire_builder_live,
            artifact_dir=S12,
        )
        .copy()
    )


    frozen = (
        frozen_fold0_actions[
            frozen_fold0_actions[
                "dataset"
            ].eq(
                dataset
            )
        ]
        .copy()
    )


    key_cols = [
        "source_id",
        "target_id",
    ]


    packaged_key = (
        packaged[
            key_cols
        ]
        .astype(
            int
        )
        .sort_values(
            key_cols,
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


    frozen_key = (
        frozen[
            key_cols
        ]
        .astype(
            int
        )
        .sort_values(
            key_cols,
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


    assert len(
        packaged_key
    ) == len(
        frozen_key
    ), (
        dataset,
        len(packaged_key),
        len(frozen_key),
    )


    assert packaged_key.equals(
        frozen_key
    ), dataset


    # --------------------------------------------------------
    # Compare frozen structural metadata keyed by edge.
    # --------------------------------------------------------

    compare_cols = [
        "source_occupied",
        "target_occupied",
        "cuts",
    ]


    p = (
        packaged[
            key_cols
            +
            compare_cols
        ]
        .sort_values(
            key_cols,
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


    f = (
        frozen[
            key_cols
            +
            compare_cols
        ]
        .sort_values(
            key_cols,
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


    for col in (
        key_cols
        +
        compare_cols
    ):

        p[col] = pd.to_numeric(
            p[
                col
            ],
            errors="raise",
        ).astype(
            int
        )


        f[col] = pd.to_numeric(
            f[
                col
            ],
            errors="raise",
        ).astype(
            int
        )


    assert p.equals(
        f
    ), dataset


    # --------------------------------------------------------
    # Exact-score numerical fidelity.
    # --------------------------------------------------------

    pscore = (
        packaged[
            key_cols
            +
            [
                "model_score",
            ]
        ]
        .sort_values(
            key_cols,
            kind="stable",
        )[
            "model_score"
        ]
        .to_numpy(
            dtype=float
        )
    )


    fscore = (
        frozen[
            key_cols
            +
            [
                "model_score",
            ]
        ]
        .sort_values(
            key_cols,
            kind="stable",
        )[
            "model_score"
        ]
        .to_numpy(
            dtype=float
        )
    )


    max_abs_score_diff = (
        float(
            np.max(
                np.abs(
                    pscore
                    -
                    fscore
                )
            )
        )
        if len(
            pscore
        )
        else 0.0
    )


    assert np.allclose(
        pscore,
        fscore,
        rtol=0.0,
        atol=1e-12,
    ), (
        dataset,
        max_abs_score_diff,
    )


    smoke_rows.append(
        {
            "dataset":
                dataset,

            "actions":
                int(
                    len(
                        packaged
                    )
                ),

            "source_occupied":
                int(
                    pd.to_numeric(
                        packaged[
                            "source_occupied"
                        ],
                        errors="raise",
                    )
                    .astype(
                        int
                    )
                    .sum()
                ),

            "max_abs_model_score_diff":
                max_abs_score_diff,

            "action_keys_exact":
                True,
        }
    )


    print(
        dataset,
        "| actions=",
        len(
            packaged
        ),
        "| source_occupied=",
        smoke_rows[
            -1
        ][
            "source_occupied"
        ],
        "| max score diff=",
        max_abs_score_diff,
    )


print(
    "GT used:",
    "NO"
)


print(
    "PACKAGED_ACTION_FIDELITY_SMOKE: PASS"
)


# ============================================================
# 15. PRESERVE CANONICAL SOURCE BEFORE PATCH
# ============================================================

canonical_before = CANONICAL_PATH.read_text(
    encoding="utf-8"
)


CANONICAL_BEFORE_SHA = hashlib.sha256(
    canonical_before.encode(
        "utf-8"
    )
).hexdigest()


# Must still be historical pre-G3H3E1 builder.
assert (
    "ASYM1x5_Q9990_NO_TARGET_CUT_V1"
    not in canonical_before
)


assert (
    "_build_current_best_pre_g3h3e1"
    not in canonical_before
)


shutil.copy2(
    CANONICAL_PATH,
    CANONICAL_BACKUP,
)


assert CANONICAL_BACKUP.exists()


assert sha256_file(
    CANONICAL_BACKUP
) == CANONICAL_BEFORE_SHA


print(
    "\n========== CANONICAL SOURCE PRESERVATION =========="
)


print(
    "before SHA:",
    CANONICAL_BEFORE_SHA
)


print(
    "backup:",
    CANONICAL_BACKUP
)


print(
    "backup fidelity:",
    "PASS"
)


# ============================================================
# 16. LOCATE CANONICAL build_current_best()
# ============================================================

canonical_tree = ast.parse(
    canonical_before
)


builder_node = None


for node in canonical_tree.body:

    if (
        isinstance(
            node,
            ast.FunctionDef,
        )
        and
        node.name
        ==
        "build_current_best"
    ):

        builder_node = node

        break


assert builder_node is not None

assert builder_node.end_lineno is not None


lines = canonical_before.splitlines(
    keepends=True
)


def_line_index = (
    builder_node.lineno
    -
    1
)


assert re.match(
    r"^\s*def\s+build_current_best\s*\(",
    lines[
        def_line_index
    ],
)


# ============================================================
# 17. RENAME HISTORICAL BUILDER TO PRE-REWIRE BASE
# ============================================================

lines[
    def_line_index
] = re.sub(
    r"\bbuild_current_best\b",
    "_build_current_best_pre_g3h3e1",
    lines[
        def_line_index
    ],
    count=1,
)


# ============================================================
# 18. INSERT MINIMAL CANONICAL WRAPPER
#
# Wrapper:
#
#   base CURRENT_BEST
#       ->
#   confirmed runtime
#
# 44b6 remains unchanged.
# ============================================================

wrapper_source = r'''

# ============================================================
# Stage 12.11G3H3E1 confirmed CURRENT_BEST augmentation
#
# Policy:
#   ASYM1x5_Q9990_NO_TARGET_CUT_V1
#
# Development:
#   +0.000050616471
#
# Locked Fold0:
#   +0.000054118252
#   TP/FP/FN delta = 0 / -2 / 0
#
# Scientific selection is CLOSED.
# No retuning is permitted here.
# ============================================================

def build_current_best(name):
    graph, scale = (
        _build_current_best_pre_g3h3e1(
            name
        )
    )

    # Confirmed rewire route was developed and frozen for
    # 6bba only. 44b6 remains exact pre-G3H3E1 CURRENT_BEST.
    if name.startswith("44b6_"):
        return graph, scale

    assert name.startswith(
        "6bba_"
    ), name

    from biohub_cell_tracking.stage12_confirmed_rewire import (
        apply_confirmed_rewire_policy,
    )

    artifact_dir = (
        PROJECT
        / "reports"
        / "stage12_residual_audit"
    )

    graph, _rewire_stats = (
        apply_confirmed_rewire_policy(
            name,
            graph,
            pre_rewire_builder=
                _build_current_best_pre_g3h3e1,
            artifact_dir=artifact_dir,
        )
    )

    return graph, scale
'''


# Insert AFTER old builder body.
insert_index = (
    builder_node.end_lineno
)


patched_lines = (
    lines[
        :
        insert_index
    ]
    +
    [
        wrapper_source
    ]
    +
    lines[
        insert_index:
    ]
)


canonical_after = "".join(
    patched_lines
)


# ============================================================
# 19. STATIC VALIDATION OF PATCHED CANONICAL SOURCE
# ============================================================

compile(
    canonical_after,
    str(
        CANONICAL_PATH
    ),
    "exec",
)


patched_tree = ast.parse(
    canonical_after
)


builder_names = [
    node.name

    for node
    in patched_tree.body

    if isinstance(
        node,
        ast.FunctionDef,
    )
    and
    node.name
    in {
        "build_current_best",
        "_build_current_best_pre_g3h3e1",
    }
]


assert builder_names.count(
    "build_current_best"
) == 1


assert builder_names.count(
    "_build_current_best_pre_g3h3e1"
) == 1


for marker in [
    "ASYM1x5_Q9990_NO_TARGET_CUT_V1",
    "_build_current_best_pre_g3h3e1",
    "apply_confirmed_rewire_policy",
]:

    assert marker in canonical_after


print(
    "\n========== CANONICAL PATCH STATIC CHECK =========="
)


print(
    "historical base builder retained:",
    "YES"
)


print(
    "new wrapper count:",
    builder_names.count(
        "build_current_best"
    )
)


print(
    "confirmed policy marker:",
    "YES"
)


print(
    "compile:",
    "PASS"
)


# ============================================================
# 20. WRITE PATCHED CANONICAL SOURCE
# ============================================================

CANONICAL_PATH.write_text(
    canonical_after,
    encoding="utf-8",
)


CANONICAL_AFTER_SHA = sha256_file(
    CANONICAL_PATH
)


assert CANONICAL_AFTER_SHA != CANONICAL_BEFORE_SHA


print(
    "\n========== CANONICAL SOURCE WRITTEN =========="
)


print(
    "before SHA:",
    CANONICAL_BEFORE_SHA
)


print(
    "after SHA:",
    CANONICAL_AFTER_SHA
)


print(
    "historical backup preserved:",
    "YES"
)


# ============================================================
# 21. PY_COMPILE BOTH NEW FILES
# ============================================================

compile_results = {}


for path in [
    RUNTIME_PATH,
    CANONICAL_PATH,
]:

    proc = subprocess.run(
        [
            sys.executable,
            "-m",
            "py_compile",
            str(
                path
            ),
        ],
        cwd=str(
            PROJECT
        ),
        capture_output=True,
        text=True,
    )


    compile_results[
        str(
            path
        )
    ] = {
        "returncode":
            int(
                proc.returncode
            ),

        "stdout":
            proc.stdout,

        "stderr":
            proc.stderr,
    }


    assert proc.returncode == 0, (
        path,
        proc.stdout,
        proc.stderr,
    )


print(
    "\n========== SOURCE PY_COMPILE =========="
)


for path, result in compile_results.items():

    print(
        path,
        ":",
        (
            "PASS"
            if result[
                "returncode"
            ] == 0
            else "FAIL"
        )
    )


# ============================================================
# 22. INTEGRATION MANIFEST
# ============================================================

MANIFEST = {
    "stage":
        "12.11G3H3E2B",

    "status":
        "FROZEN_RUNTIME_PACKAGED_AND_CANONICAL_INTEGRATED",

    "policy_id":
        POLICY_ID,

    "policy_sha256":
        POLICY_SHA,

    "scientific_selection_closed":
        True,

    "fold0_consumed":
        True,

    "post_fold0_retuning_allowed":
        False,

    "runtime_module":
        str(
            RUNTIME_PATH.relative_to(
                PROJECT
            )
        ),

    "runtime_module_sha256":
        RUNTIME_SHA,

    "canonical_source":
        str(
            CANONICAL_PATH.relative_to(
                PROJECT
            )
        ),

    "canonical_before_sha256":
        CANONICAL_BEFORE_SHA,

    "canonical_after_sha256":
        CANONICAL_AFTER_SHA,

    "canonical_backup":
        str(
            CANONICAL_BACKUP.relative_to(
                PROJECT
            )
        ),

    "canonical_backup_sha256":
        sha256_file(
            CANONICAL_BACKUP
        ),

    "runtime_root_functions":
        ROOT_FUNCTION_NAMES,

    "packaged_function_source_sha256":
        function_source_sha,

    "packaged_functions":
        sorted(
            function_objects
        ),

    "frozen_constants":
        sorted(
            constant_objects
        ),

    "smoke_action_fidelity":
        smoke_rows,

    "smoke_gt_used":
        False,

    "official_metric_evaluated":
        False,

    "development_reopened":
        False,

    "prediction_geff_written":
        False,

    "next":
        (
            "FRESH_PROCESS_CANONICAL_REPRODUCTION_"
            "AS_IMPLEMENTATION_FIDELITY_ONLY"
        ),
}


INTEGRATION_MANIFEST_OUT.write_text(
    json.dumps(
        MANIFEST,
        indent=2,
    ),
    encoding="utf-8",
)


assert INTEGRATION_MANIFEST_OUT.exists()


# ============================================================
# 23. FINAL INTEGRITY
# ============================================================

manifest_check = json.loads(
    INTEGRATION_MANIFEST_OUT.read_text(
        encoding="utf-8"
    )
)


assert manifest_check[
    "policy_id"
] == POLICY_ID


assert manifest_check[
    "scientific_selection_closed"
] is True


assert manifest_check[
    "fold0_consumed"
] is True


assert manifest_check[
    "post_fold0_retuning_allowed"
] is False


assert manifest_check[
    "official_metric_evaluated"
] is False


assert sha256_file(
    RUNTIME_PATH
) == RUNTIME_SHA


assert sha256_file(
    CANONICAL_PATH
) == CANONICAL_AFTER_SHA


assert sha256_file(
    CANONICAL_BACKUP
) == CANONICAL_BEFORE_SHA


# ============================================================
# 24. FINAL
# ============================================================

print(
    "\n========== 12.11G3H3E2B DECISION =========="
)


print(
    "FROZEN_RUNTIME_PACKAGING: PASS"
)


print(
    "CANONICAL_CURRENT_BEST_INTEGRATION: PASS"
)


print(
    "policy:",
    POLICY_ID
)


print(
    "packaged runtime functions:",
    len(
        function_objects
    )
)


print(
    "runtime module:",
    RUNTIME_PATH
)


print(
    "runtime SHA:",
    RUNTIME_SHA
)


print(
    "canonical source:",
    CANONICAL_PATH
)


print(
    "canonical before SHA:",
    CANONICAL_BEFORE_SHA
)


print(
    "canonical after SHA:",
    CANONICAL_AFTER_SHA
)


print(
    "canonical backup:",
    CANONICAL_BACKUP
)


print(
    "GT used:",
    "NO"
)


print(
    "official metric evaluated:",
    "NO"
)


print(
    "development reopened:",
    "NO"
)


print(
    "Fold0 consumed:",
    "YES"
)


print(
    "post-Fold0 retuning:",
    "NO"
)


print(
    "next:"
)


print(
    "FRESH_PROCESS_CANONICAL_REPRODUCTION_AS_IMPLEMENTATION_FIDELITY_ONLY"
)

In [ ]:
# ============================================================
# Stage 12.11G3H3E2B-R2
#
# EXACT FOLD0 PRE-REWIRE BUILDER RECOVERY
# + PACKAGED ACTION FIDELITY
# + CANONICAL INTEGRATION RESUME
#
# ------------------------------------------------------------
# REPAIR REASON
# ------------------------------------------------------------
#
# E2B-R1 incorrectly used:
#
#     current notebook build_current_best
#
# which was the folds1-4 reconstruction builder and required:
#
#     dataset in fold_map
#
# Therefore Fold0 smoke dataset:
#
#     6bba_d5eae175
#
# failed BEFORE packaged-runtime fidelity could be tested.
#
# ------------------------------------------------------------
# CORRECT CONTRACT
# ------------------------------------------------------------
#
# Recover exact pre-G3H3E1 CURRENT_BEST builder from:
#
#   scripts/
#     run_stage12_current_best_edge_residual_census.py
#
# using the same AST-safe-prefix procedure already validated
# in G3H3E0.
#
# Then inject that exact builder into the already-packaged
# frozen runtime.
#
# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------
#
# DO NOT modify/regenerate:
#
#   stage12_confirmed_rewire.py
#
# Existing runtime SHA must remain:
#
#   52c183629ecf80d0fa1568c8c3b2fa3a4ad6f58a1c34a98dc2bf59feca0ffad0
#
# NO:
#
#   GT
#   official metric
#   Fold0 selection
#   scorer retuning
#   threshold tuning
#   policy tuning
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import ast
import hashlib
import importlib
import json
import re
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


SRC_ROOT = (
    PROJECT
    / "src"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


RUNTIME_PATH = (
    SRC_ROOT
    / "biohub_cell_tracking"
    / "stage12_confirmed_rewire.py"
)


CANONICAL_PATH = (
    PROJECT
    / "scripts"
    / "run_stage12_current_best_edge_residual_census.py"
)


CANONICAL_BACKUP = (
    PROJECT
    / "scripts"
    / "run_stage12_current_best_edge_residual_census.pre_g3h3e1.py"
)


MANIFEST_OUT = (
    S12
    / "stage12_g3h3e2b_canonical_runtime_integration.json"
)


FROZEN_ACTIONS_PATH = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_actions_locked.pkl"
)


FINAL_POLICY_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_frozen.json"
)


PROMOTION_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_promoted.json"
)


for path in [
    RUNTIME_PATH,
    CANONICAL_PATH,
    FROZEN_ACTIONS_PATH,
    FINAL_POLICY_PATH,
    PROMOTION_PATH,
]:

    assert path.exists(), path


# ============================================================
# 1. FROZEN IDENTIFIERS
# ============================================================

POLICY_ID = (
    "ASYM1x5_Q9990_NO_TARGET_CUT_V1"
)


POLICY_SHA = (
    "77f0b07cafb21d0705917ff71b750fde4"
    "5f3c7ddc17c13fecaff0ac673ebf776"
)


EXPECTED_RUNTIME_SHA = (
    "52c183629ecf80d0fa1568c8c3b2fa3a"
    "4ad6f58a1c34a98dc2bf59feca0ffad0"
)


EXPECTED_THRESHOLD = (
    0.9952878577709198
)


EXPECTED_FOLD0_ACTIONS = 2712


SMOKE_EXPECTED_ACTIONS = {
    "6bba_d5eae175":
        10,

    "6bba_7f87b3d8":
        97,
}


# ============================================================
# 2. SHA HELPER
# ============================================================

def sha256_file(
    path,
):

    h = hashlib.sha256()


    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024
                *
                1024
            )


            if not chunk:

                break


            h.update(
                chunk
            )


    return h.hexdigest()


# ============================================================
# 3. VERIFY POLICY REMAINS CLOSED
# ============================================================

policy = json.loads(
    FINAL_POLICY_PATH.read_text(
        encoding="utf-8"
    )
)


promotion = json.loads(
    PROMOTION_PATH.read_text(
        encoding="utf-8"
    )
)


assert policy[
    "policy_id"
] == POLICY_ID


assert policy[
    "final_policy_sha256"
] == POLICY_SHA


assert policy[
    "development_closed_after_freeze"
] is True


assert promotion[
    "status"
] == (
    "CONFIRMED_POLICY_PROMOTED"
)


assert promotion[
    "post_fold0_retuning_allowed"
] is False


print(
    "========== G3H3E2B-R2 POLICY LOCK =========="
)


print(
    "policy:",
    POLICY_ID
)


print(
    "development closed:",
    "YES"
)


print(
    "Fold0 consumed:",
    "YES"
)


print(
    "post-Fold0 tuning:",
    "NO"
)


# ============================================================
# 4. VERIFY EXISTING PACKAGED RUNTIME
#
# DO NOT regenerate it.
# ============================================================

runtime_sha = sha256_file(
    RUNTIME_PATH
)


print(
    "\n========== EXISTING PACKAGED RUNTIME =========="
)


print(
    "runtime:",
    RUNTIME_PATH
)


print(
    "SHA:",
    runtime_sha
)


print(
    "expected:",
    EXPECTED_RUNTIME_SHA
)


assert runtime_sha == EXPECTED_RUNTIME_SHA, (
    "\nFrozen runtime changed after E2B.\n"
    "Do not repair by rewriting it."
)


# ============================================================
# 5. FRESH IMPORT RUNTIME
# ============================================================

if str(
    SRC_ROOT
) not in sys.path:

    sys.path.insert(
        0,
        str(
            SRC_ROOT
        ),
    )


module_name = (
    "biohub_cell_tracking.stage12_confirmed_rewire"
)


sys.modules.pop(
    module_name,
    None,
)


runtime_module = importlib.import_module(
    module_name
)


assert runtime_module.POLICY_ID == POLICY_ID


assert float(
    runtime_module.DEPLOYMENT_THRESHOLD
) == EXPECTED_THRESHOLD


assert callable(
    runtime_module.build_confirmed_rewire_actions
)


assert callable(
    runtime_module.apply_confirmed_rewire_policy
)


print(
    "\n========== FRESH RUNTIME IMPORT =========="
)


print(
    "module:",
    runtime_module.__file__
)


print(
    "policy:",
    runtime_module.POLICY_ID
)


print(
    "FRESH_RUNTIME_IMPORT: PASS"
)


# ============================================================
# 6. DETECT CANONICAL STATE
# ============================================================

canonical_current = (
    CANONICAL_PATH.read_text(
        encoding="utf-8"
    )
)


current_is_patched = all(
    marker in canonical_current

    for marker in [
        "ASYM1x5_Q9990_NO_TARGET_CUT_V1",
        "_build_current_best_pre_g3h3e1",
        "apply_confirmed_rewire_policy",
    ]
)


backup_exists = (
    CANONICAL_BACKUP.exists()
)


manifest_exists = (
    MANIFEST_OUT.exists()
)


print(
    "\n========== CURRENT FILE STATE =========="
)


print(
    "canonical already patched:",
    current_is_patched
)


print(
    "backup exists:",
    backup_exists
)


print(
    "manifest exists:",
    manifest_exists
)


# ============================================================
# 7. CHOOSE AUTHORITATIVE PRE-G3H3E1 SOURCE
#
# Priority:
#
#   1. preserved backup, if it exists
#   2. current canonical, only if still unpatched
#
# Never recover pre-rewire semantics from an already-patched
# source without a preserved backup.
# ============================================================

if backup_exists:

    pre_source_path = (
        CANONICAL_BACKUP
    )


    pre_source = (
        CANONICAL_BACKUP.read_text(
            encoding="utf-8"
        )
    )


    assert (
        "apply_confirmed_rewire_policy"
        not in pre_source
    )


    assert (
        "_build_current_best_pre_g3h3e1"
        not in pre_source
    )


else:

    assert not current_is_patched, (
        "\nCanonical source is already patched, but no "
        "pre-G3H3E1 backup exists.\n"
        "Stop rather than inventing historical semantics."
    )


    pre_source_path = (
        CANONICAL_PATH
    )


    pre_source = (
        canonical_current
    )


print(
    "\n========== AUTHORITATIVE PRE-REWIRE SOURCE =========="
)


print(
    "path:",
    pre_source_path
)


print(
    "SHA:",
    hashlib.sha256(
        pre_source.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 8. AST RECOVER EXACT ORIGINAL build_current_best()
#
# Same recovery strategy that already succeeded in G3H3E0.
# ============================================================

tree = ast.parse(
    pre_source
)


builder_nodes = [
    node

    for node
    in tree.body

    if (
        isinstance(
            node,
            ast.FunctionDef,
        )
        and
        node.name
        ==
        "build_current_best"
    )
]


assert len(
    builder_nodes
) == 1


builder_node = builder_nodes[
    0
]


assert builder_node.end_lineno is not None


source_lines = pre_source.splitlines()


safe_prefix = "\n".join(
    source_lines[
        :
        builder_node.end_lineno
    ]
)


# ------------------------------------------------------------
# Hard GT safety.
# ------------------------------------------------------------

assert "fv9.evaluate_graph(" not in safe_prefix

assert "official_evaluate(" not in safe_prefix

assert "fv9.load_graph(" not in safe_prefix


builder_ns = {
    "__name__":
        "__g3h3e2b_r2_pre_rewire_builder__",

    "__file__":
        str(
            pre_source_path
        ),
}


exec(
    compile(
        safe_prefix,
        str(
            pre_source_path
        ),
        "exec",
    ),
    builder_ns,
    builder_ns,
)


assert (
    "build_current_best"
    in builder_ns
)


exact_pre_rewire_builder = (
    builder_ns[
        "build_current_best"
    ]
)


assert callable(
    exact_pre_rewire_builder
)


# ============================================================
# 9. VERIFY RECOVERED FOLD0 CORPUS CONTRACT
# ============================================================

assert "names_44" in builder_ns

assert "names_6" in builder_ns


assert len(
    builder_ns[
        "names_44"
    ]
) == 15


assert len(
    builder_ns[
        "names_6"
    ]
) == 25


assert (
    "6bba_d5eae175"
    in
    builder_ns[
        "names_6"
    ]
)


assert (
    "6bba_7f87b3d8"
    in
    builder_ns[
        "names_6"
    ]
)


print(
    "\n========== EXACT PRE-G3H3E1 BUILDER RECOVERY =========="
)


print(
    "builder source:",
    pre_source_path
)


print(
    "builder lines:",
    builder_node.lineno,
    "-",
    builder_node.end_lineno
)


print(
    "Fold0 44b6:",
    len(
        builder_ns[
            "names_44"
        ]
    )
)


print(
    "Fold0 6bba:",
    len(
        builder_ns[
            "names_6"
        ]
    )
)


print(
    "GT touched:",
    "NO"
)


print(
    "EXACT_PRE_REWIRE_BUILDER_RECOVERY: PASS"
)


# ============================================================
# 10. LOAD LOCKED FOLD0 ACTION ARTIFACT
# ============================================================

frozen_actions = pd.read_pickle(
    FROZEN_ACTIONS_PATH
)


assert len(
    frozen_actions
) == EXPECTED_FOLD0_ACTIONS


assert not frozen_actions[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


assert pd.to_numeric(
    frozen_actions[
        "target_occupied"
    ],
    errors="raise",
).astype(
    int
).eq(
    0
).all()


# ============================================================
# 11. GT-FREE ACTION FIDELITY SMOKE
#
# This is now testing the packaged runtime against the exact
# authoritative Fold0 pre-rewire builder.
# ============================================================

smoke_datasets = [
    "6bba_d5eae175",
    "6bba_7f87b3d8",
]


print(
    "\n========== G3H3E2B-R2 GT-FREE ACTION FIDELITY =========="
)


print(
    "datasets:",
    smoke_datasets
)


smoke_rows = []


for dataset in smoke_datasets:

    packaged = (
        runtime_module
        .build_confirmed_rewire_actions(
            dataset,
            pre_rewire_builder=
                exact_pre_rewire_builder,
            artifact_dir=S12,
        )
        .copy()
        .reset_index(
            drop=True
        )
    )


    frozen = (
        frozen_actions[
            frozen_actions[
                "dataset"
            ].eq(
                dataset
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Cardinality
    # --------------------------------------------------------

    assert len(
        frozen
    ) == SMOKE_EXPECTED_ACTIONS[
        dataset
    ]


    assert len(
        packaged
    ) == len(
        frozen
    ), (
        dataset,
        len(
            packaged
        ),
        len(
            frozen
        ),
    )


    # --------------------------------------------------------
    # Exact action identity
    # --------------------------------------------------------

    key_cols = [
        "source_id",
        "target_id",
    ]


    packaged_key = (
        packaged[
            key_cols
        ]
        .astype(
            int
        )
        .sort_values(
            key_cols,
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


    frozen_key = (
        frozen[
            key_cols
        ]
        .astype(
            int
        )
        .sort_values(
            key_cols,
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


    assert packaged_key.equals(
        frozen_key
    ), (
        dataset,
        "ACTION KEY MISMATCH",
    )


    # --------------------------------------------------------
    # Structural metadata
    # --------------------------------------------------------

    structural_cols = [
        "source_occupied",
        "target_occupied",
        "cuts",
    ]


    pstruct = (
        packaged[
            key_cols
            +
            structural_cols
        ]
        .sort_values(
            key_cols,
            kind="stable",
        )
        .reset_index(
            drop=True
        )
        .copy()
    )


    fstruct = (
        frozen[
            key_cols
            +
            structural_cols
        ]
        .sort_values(
            key_cols,
            kind="stable",
        )
        .reset_index(
            drop=True
        )
        .copy()
    )


    for col in (
        key_cols
        +
        structural_cols
    ):

        pstruct[
            col
        ] = pd.to_numeric(
            pstruct[
                col
            ],
            errors="raise",
        ).astype(
            int
        )


        fstruct[
            col
        ] = pd.to_numeric(
            fstruct[
                col
            ],
            errors="raise",
        ).astype(
            int
        )


    assert pstruct.equals(
        fstruct
    ), (
        dataset,
        "STRUCTURAL METADATA MISMATCH",
    )


    # --------------------------------------------------------
    # Frozen score fidelity
    # --------------------------------------------------------

    pscore = (
        packaged[
            key_cols
            +
            [
                "model_score",
            ]
        ]
        .sort_values(
            key_cols,
            kind="stable",
        )[
            "model_score"
        ]
        .to_numpy(
            dtype=float
        )
    )


    fscore = (
        frozen[
            key_cols
            +
            [
                "model_score",
            ]
        ]
        .sort_values(
            key_cols,
            kind="stable",
        )[
            "model_score"
        ]
        .to_numpy(
            dtype=float
        )
    )


    max_abs_score_diff = (
        float(
            np.max(
                np.abs(
                    pscore
                    -
                    fscore
                )
            )
        )
        if len(
            pscore
        )
        else 0.0
    )


    assert np.allclose(
        pscore,
        fscore,
        rtol=0.0,
        atol=1e-12,
    ), (
        dataset,
        max_abs_score_diff,
    )


    source_occ = int(
        pd.to_numeric(
            packaged[
                "source_occupied"
            ],
            errors="raise",
        )
        .astype(
            int
        )
        .sum()
    )


    assert pd.to_numeric(
        packaged[
            "target_occupied"
        ],
        errors="raise",
    ).astype(
        int
    ).eq(
        0
    ).all()


    smoke_rows.append(
        {
            "dataset":
                dataset,

            "actions":
                int(
                    len(
                        packaged
                    )
                ),

            "source_occupied":
                source_occ,

            "max_abs_model_score_diff":
                max_abs_score_diff,

            "action_keys_exact":
                True,

            "structural_metadata_exact":
                True,
        }
    )


    print(
        dataset,
        "| actions=",
        len(
            packaged
        ),
        "| expected=",
        SMOKE_EXPECTED_ACTIONS[
            dataset
        ],
        "| source_occupied=",
        source_occ,
        "| max score diff=",
        max_abs_score_diff,
    )


print(
    "GT used:",
    "NO"
)


print(
    "PACKAGED_ACTION_FIDELITY_SMOKE: PASS"
)


# ============================================================
# 12. ONLY NOW MAY CANONICAL SOURCE BE INTEGRATED
# ============================================================

canonical_now = (
    CANONICAL_PATH.read_text(
        encoding="utf-8"
    )
)


patched_now = all(
    marker in canonical_now

    for marker in [
        "ASYM1x5_Q9990_NO_TARGET_CUT_V1",
        "_build_current_best_pre_g3h3e1",
        "apply_confirmed_rewire_policy",
    ]
)


print(
    "\n========== PRE-INTEGRATION CANONICAL STATE =========="
)


print(
    "already patched:",
    patched_now
)


print(
    "backup exists:",
    CANONICAL_BACKUP.exists()
)


print(
    "manifest exists:",
    MANIFEST_OUT.exists()
)


# ============================================================
# 13. HANDLE ALREADY-COMPLETE STATE SAFELY
# ============================================================

if (
    patched_now
    and
    MANIFEST_OUT.exists()
):

    manifest = json.loads(
        MANIFEST_OUT.read_text(
            encoding="utf-8"
        )
    )


    assert manifest[
        "policy_id"
    ] == POLICY_ID


    assert manifest[
        "runtime_module_sha256"
    ] == EXPECTED_RUNTIME_SHA


    assert manifest[
        "post_fold0_retuning_allowed"
    ] is False


    print(
        "\n========== 12.11G3H3E2B-R2 DECISION =========="
    )


    print(
        "EXISTING_CANONICAL_INTEGRATION_VERIFIED: PASS"
    )


    print(
        "PACKAGED_ACTION_FIDELITY_SMOKE: PASS"
    )


    print(
        "source files modified by R2:",
        "NO"
    )


    print(
        "GT used:",
        "NO"
    )


    print(
        "development reopened:",
        "NO"
    )


    print(
        "next:"
    )


    print(
        "FRESH_PROCESS_CANONICAL_REPRODUCTION_"
        "AS_IMPLEMENTATION_FIDELITY_ONLY"
    )


else:

    # ========================================================
    # 14. GET EXACT PRE-PATCH SOURCE FOR PROVENANCE
    # ========================================================

    if patched_now:

        assert CANONICAL_BACKUP.exists(), (
            "\nCanonical appears patched but historical backup "
            "is missing. Stop."
        )


        canonical_before = (
            CANONICAL_BACKUP.read_text(
                encoding="utf-8"
            )
        )


        canonical_before_sha = (
            sha256_file(
                CANONICAL_BACKUP
            )
        )


        assert (
            "apply_confirmed_rewire_policy"
            not in canonical_before
        )


    else:

        canonical_before = (
            canonical_now
        )


        canonical_before_sha = hashlib.sha256(
            canonical_before.encode(
                "utf-8"
            )
        ).hexdigest()


        if CANONICAL_BACKUP.exists():

            assert (
                sha256_file(
                    CANONICAL_BACKUP
                )
                ==
                canonical_before_sha
            )


        else:

            shutil.copy2(
                CANONICAL_PATH,
                CANONICAL_BACKUP,
            )


            assert (
                sha256_file(
                    CANONICAL_BACKUP
                )
                ==
                canonical_before_sha
            )


        print(
            "\n========== CANONICAL SOURCE PRESERVATION =========="
        )


        print(
            "before SHA:",
            canonical_before_sha
        )


        print(
            "backup:",
            CANONICAL_BACKUP
        )


        print(
            "backup fidelity:",
            "PASS"
        )


    # ========================================================
    # 15. PATCH ONLY IF NOT ALREADY PATCHED
    # ========================================================

    if not patched_now:

        patch_tree = ast.parse(
            canonical_before
        )


        patch_builder_nodes = [
            node

            for node
            in patch_tree.body

            if (
                isinstance(
                    node,
                    ast.FunctionDef,
                )
                and
                node.name
                ==
                "build_current_best"
            )
        ]


        assert len(
            patch_builder_nodes
        ) == 1


        patch_builder = patch_builder_nodes[
            0
        ]


        assert patch_builder.end_lineno is not None


        lines = canonical_before.splitlines(
            keepends=True
        )


        def_idx = (
            patch_builder.lineno
            -
            1
        )


        assert re.match(
            r"^\s*def\s+build_current_best\s*\(",
            lines[
                def_idx
            ],
        )


        # ----------------------------------------------------
        # Rename exact historical builder.
        # ----------------------------------------------------

        lines[
            def_idx
        ] = re.sub(
            r"\bbuild_current_best\b",
            "_build_current_best_pre_g3h3e1",
            lines[
                def_idx
            ],
            count=1,
        )


        # ----------------------------------------------------
        # Minimal confirmed-policy wrapper.
        # ----------------------------------------------------

        wrapper_source = r'''

# ============================================================
# Stage 12.11G3H3E1 confirmed CURRENT_BEST augmentation
#
# Policy:
#   ASYM1x5_Q9990_NO_TARGET_CUT_V1
#
# Development score delta:
#   +0.000050616471
#
# Locked Fold0 score delta:
#   +0.000054118252
#
# Locked Fold0 TP/FP/FN delta:
#   0 / -2 / 0
#
# Scientific selection is CLOSED.
# Fold0 is consumed.
# No policy retuning is permitted here.
# ============================================================

def build_current_best(name):
    graph, scale = (
        _build_current_best_pre_g3h3e1(
            name
        )
    )

    # Confirmed branch applies only to 6bba.
    # 44b6 remains exact historical CURRENT_BEST.
    if name.startswith("44b6_"):
        return graph, scale

    assert name.startswith(
        "6bba_"
    ), name

    from biohub_cell_tracking.stage12_confirmed_rewire import (
        apply_confirmed_rewire_policy,
    )

    artifact_dir = (
        PROJECT
        / "reports"
        / "stage12_residual_audit"
    )

    graph, _rewire_stats = (
        apply_confirmed_rewire_policy(
            name,
            graph,
            pre_rewire_builder=
                _build_current_best_pre_g3h3e1,
            artifact_dir=artifact_dir,
        )
    )

    return graph, scale
'''


        insert_idx = (
            patch_builder.end_lineno
        )


        patched_lines = (
            lines[
                :
                insert_idx
            ]
            +
            [
                wrapper_source
            ]
            +
            lines[
                insert_idx:
            ]
        )


        canonical_after = "".join(
            patched_lines
        )


        # ----------------------------------------------------
        # Static validation before disk write.
        # ----------------------------------------------------

        compile(
            canonical_after,
            str(
                CANONICAL_PATH
            ),
            "exec",
        )


        after_tree = ast.parse(
            canonical_after
        )


        function_names = [
            node.name

            for node
            in after_tree.body

            if isinstance(
                node,
                ast.FunctionDef,
            )
        ]


        assert function_names.count(
            "build_current_best"
        ) == 1


        assert function_names.count(
            "_build_current_best_pre_g3h3e1"
        ) == 1


        for marker in [
            "ASYM1x5_Q9990_NO_TARGET_CUT_V1",
            "_build_current_best_pre_g3h3e1",
            "apply_confirmed_rewire_policy",
        ]:

            assert marker in canonical_after


        print(
            "\n========== CANONICAL PATCH STATIC CHECK =========="
        )


        print(
            "historical builder retained:",
            "YES"
        )


        print(
            "new build_current_best count:",
            function_names.count(
                "build_current_best"
            )
        )


        print(
            "compile:",
            "PASS"
        )


        # ----------------------------------------------------
        # Write.
        # ----------------------------------------------------

        CANONICAL_PATH.write_text(
            canonical_after,
            encoding="utf-8",
        )


    # ========================================================
    # 16. VERIFY FINAL CANONICAL SOURCE
    # ========================================================

    canonical_final = (
        CANONICAL_PATH.read_text(
            encoding="utf-8"
        )
    )


    assert (
        "ASYM1x5_Q9990_NO_TARGET_CUT_V1"
        in canonical_final
    )


    assert (
        "_build_current_best_pre_g3h3e1"
        in canonical_final
    )


    assert (
        "apply_confirmed_rewire_policy"
        in canonical_final
    )


    canonical_after_sha = (
        sha256_file(
            CANONICAL_PATH
        )
    )


    assert (
        canonical_after_sha
        !=
        canonical_before_sha
    )


    print(
        "\n========== CANONICAL SOURCE INTEGRATED =========="
    )


    print(
        "before SHA:",
        canonical_before_sha
    )


    print(
        "after SHA:",
        canonical_after_sha
    )


    print(
        "backup preserved:",
        "YES"
    )


    # ========================================================
    # 17. PY_COMPILE
    # ========================================================

    compile_rows = []


    for path in [
        RUNTIME_PATH,
        CANONICAL_PATH,
    ]:

        proc = subprocess.run(
            [
                sys.executable,
                "-m",
                "py_compile",
                str(
                    path
                ),
            ],
            cwd=str(
                PROJECT
            ),
            capture_output=True,
            text=True,
        )


        compile_rows.append(
            {
                "path":
                    str(
                        path
                    ),

                "returncode":
                    int(
                        proc.returncode
                    ),

                "stdout":
                    proc.stdout,

                "stderr":
                    proc.stderr,
            }
        )


        assert proc.returncode == 0, (
            path,
            proc.stdout,
            proc.stderr,
        )


    print(
        "\n========== SOURCE PY_COMPILE =========="
    )


    for row in compile_rows:

        print(
            row[
                "path"
            ],
            ": PASS"
        )


    # ========================================================
    # 18. WRITE FORMAL INTEGRATION MANIFEST
    # ========================================================

    if MANIFEST_OUT.exists():

        raise RuntimeError(
            "\nIntegration manifest already exists unexpectedly.\n"
            "Do not overwrite it."
        )


    MANIFEST = {
        "stage":
            "12.11G3H3E2B-R2",

        "status":
            (
                "FROZEN_RUNTIME_PACKAGED_AND_"
                "CANONICAL_INTEGRATED"
            ),

        "parent_partial_stage":
            "12.11G3H3E2B",

        "failed_resume_stage":
            "12.11G3H3E2B-R1",

        "r1_failure_reason":
            (
                "WRONG_NOTEBOOK_DEVELOPMENT_BUILDER_"
                "INJECTED_FOR_FOLD0_SMOKE"
            ),

        "repair":
            (
                "RECOVERED_EXACT_PRE_G3H3E1_BUILDER_"
                "FROM_AUTHORITATIVE_CANONICAL_SOURCE"
            ),

        "policy_id":
            POLICY_ID,

        "policy_sha256":
            POLICY_SHA,

        "scientific_selection_closed":
            True,

        "fold0_consumed":
            True,

        "post_fold0_retuning_allowed":
            False,

        "runtime_module":
            str(
                RUNTIME_PATH.relative_to(
                    PROJECT
                )
            ),

        "runtime_module_sha256":
            EXPECTED_RUNTIME_SHA,

        "runtime_module_regenerated":
            False,

        "pre_rewire_builder_source":
            str(
                pre_source_path.relative_to(
                    PROJECT
                )
            ),

        "canonical_source":
            str(
                CANONICAL_PATH.relative_to(
                    PROJECT
                )
            ),

        "canonical_before_sha256":
            canonical_before_sha,

        "canonical_after_sha256":
            canonical_after_sha,

        "canonical_backup":
            str(
                CANONICAL_BACKUP.relative_to(
                    PROJECT
                )
            ),

        "canonical_backup_sha256":
            sha256_file(
                CANONICAL_BACKUP
            ),

        "smoke_action_fidelity":
            smoke_rows,

        "smoke_gt_used":
            False,

        "official_metric_evaluated":
            False,

        "development_reopened":
            False,

        "prediction_geff_written":
            False,

        "next":
            (
                "FRESH_PROCESS_CANONICAL_REPRODUCTION_"
                "AS_IMPLEMENTATION_FIDELITY_ONLY"
            ),
    }


    MANIFEST_OUT.write_text(
        json.dumps(
            MANIFEST,
            indent=2,
        ),
        encoding="utf-8",
    )


    # ========================================================
    # 19. RELOAD INTEGRITY
    # ========================================================

    manifest_check = json.loads(
        MANIFEST_OUT.read_text(
            encoding="utf-8"
        )
    )


    assert manifest_check[
        "policy_id"
    ] == POLICY_ID


    assert manifest_check[
        "runtime_module_sha256"
    ] == EXPECTED_RUNTIME_SHA


    assert manifest_check[
        "runtime_module_regenerated"
    ] is False


    assert manifest_check[
        "scientific_selection_closed"
    ] is True


    assert manifest_check[
        "fold0_consumed"
    ] is True


    assert manifest_check[
        "post_fold0_retuning_allowed"
    ] is False


    assert manifest_check[
        "official_metric_evaluated"
    ] is False


    assert (
        sha256_file(
            RUNTIME_PATH
        )
        ==
        EXPECTED_RUNTIME_SHA
    )


    assert (
        sha256_file(
            CANONICAL_BACKUP
        )
        ==
        canonical_before_sha
    )


    assert (
        sha256_file(
            CANONICAL_PATH
        )
        ==
        canonical_after_sha
    )


    # ========================================================
    # 20. FINAL
    # ========================================================

    print(
        "\n========== 12.11G3H3E2B-R2 DECISION =========="
    )


    print(
        "BUILDER_INJECTION_REPAIR: PASS"
    )


    print(
        "PACKAGED_ACTION_FIDELITY_SMOKE: PASS"
    )


    print(
        "FROZEN_RUNTIME_PACKAGING: PASS"
    )


    print(
        "CANONICAL_CURRENT_BEST_INTEGRATION: PASS"
    )


    print(
        "runtime regenerated:",
        "NO"
    )


    print(
        "runtime SHA:",
        EXPECTED_RUNTIME_SHA
    )


    print(
        "canonical before SHA:",
        canonical_before_sha
    )


    print(
        "canonical after SHA:",
        canonical_after_sha
    )


    print(
        "GT used:",
        "NO"
    )


    print(
        "official metric evaluated:",
        "NO"
    )


    print(
        "development reopened:",
        "NO"
    )


    print(
        "Fold0 consumed:",
        "YES"
    )


    print(
        "post-Fold0 tuning:",
        "NO"
    )


    print(
        "next:"
    )


    print(
        "FRESH_PROCESS_CANONICAL_REPRODUCTION_"
        "AS_IMPLEMENTATION_FIDELITY_ONLY"
    )

In [ ]:
# ============================================================
# Stage 12.11G3H3E2C
#
# FRESH-PROCESS CANONICAL REPRODUCTION
#
# ============================================================
#
# PURPOSE
#
# Verify that the production-packaged canonical CURRENT_BEST
# reproduces the ALREADY-CONSUMED locked Fold0 result exactly.
#
# THIS IS IMPLEMENTATION FIDELITY ONLY.
#
# It is NOT:
#
#   model selection
#   policy selection
#   threshold selection
#   post-Fold0 tuning
#
# ------------------------------------------------------------
# STRONG FIDELITY CONTRACT
# ------------------------------------------------------------
#
# Fresh Python process:
#
#   historical pre-G3H3E1 builder
#       from preserved backup
#
#   vs
#
#   integrated canonical CURRENT_BEST builder
#
# For all 40 Fold0 datasets:
#
#   15 x 44b6:
#       exact graph topology unchanged
#
#   25 x 6bba:
#       exact added edge-pair set
#           ==
#       frozen 2712 Fold0 action artifact
#
#       removed edge count
#           ==
#       frozen source_occupied count
#
# Then official metrics must reproduce the already-frozen E0
# result dataset-by-dataset and in aggregate.
#
# ------------------------------------------------------------
# EXPECTED LOCKED RESULT
#
# Baseline:
#   TP/FP/FN = 22806 / 2658 / 3425
#   score    = 0.7863444776365197
#
# Canonical:
#   TP/FP/FN = 22806 / 2656 / 3425
#   score    = 0.7863985958888573
#
# Delta:
#   0 / -2 / 0
#   +0.00005411825233758716
#
# ------------------------------------------------------------
# NO
#
# tuning
# policy changes
# source modification
# prediction GEFF writes
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import hashlib
import json
import os
import subprocess
import sys
import textwrap


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


RUNNER = (
    PROJECT
    / "scripts"
    / "run_stage12_g3h3e2c_fresh_process_reproduction.py"
)


REPORT = (
    S12
    / "stage12_g3h3e2c_fresh_process_canonical_reproduction.json"
)


CANONICAL = (
    PROJECT
    / "scripts"
    / "run_stage12_current_best_edge_residual_census.py"
)


BACKUP = (
    PROJECT
    / "scripts"
    / "run_stage12_current_best_edge_residual_census.pre_g3h3e1.py"
)


RUNTIME = (
    PROJECT
    / "src"
    / "biohub_cell_tracking"
    / "stage12_confirmed_rewire.py"
)


INTEGRATION_MANIFEST = (
    S12
    / "stage12_g3h3e2b_canonical_runtime_integration.json"
)


FROZEN_ACTIONS = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_actions_locked.pkl"
)


E0_METRICS = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_confirmation_metrics.pkl"
)


E0_SUMMARY = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_confirmation_summary.json"
)


PROMOTION = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_promoted.json"
)


for path in [
    CANONICAL,
    BACKUP,
    RUNTIME,
    INTEGRATION_MANIFEST,
    FROZEN_ACTIONS,
    E0_METRICS,
    E0_SUMMARY,
    PROMOTION,
]:

    assert path.exists(), path


for path in [
    RUNNER,
    REPORT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3H3E2C formal output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite reproduction provenance."
        )


# ============================================================
# 1. EXPECTED SOURCE SHAs
# ============================================================

EXPECTED_RUNTIME_SHA = (
    "52c183629ecf80d0fa1568c8c3b2fa3a"
    "4ad6f58a1c34a98dc2bf59feca0ffad0"
)


EXPECTED_CANONICAL_PRE_SHA = (
    "d85404884c4c74948676bbbefa4eafac"
    "2c5ac1ef9538e417ae52a4c04a9a7c60"
)


EXPECTED_CANONICAL_POST_SHA = (
    "6f2bcc83715e305d0f01517a929382a8"
    "ac6f9f31a0bf32c29c11ed95c5be2181"
)


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


assert (
    sha256_file(
        RUNTIME
    )
    ==
    EXPECTED_RUNTIME_SHA
)


assert (
    sha256_file(
        BACKUP
    )
    ==
    EXPECTED_CANONICAL_PRE_SHA
)


assert (
    sha256_file(
        CANONICAL
    )
    ==
    EXPECTED_CANONICAL_POST_SHA
)


print(
    "========== G3H3E2C SOURCE LOCK =========="
)


print(
    "runtime SHA:",
    sha256_file(
        RUNTIME
    )
)


print(
    "pre-G3H3E1 canonical SHA:",
    sha256_file(
        BACKUP
    )
)


print(
    "integrated canonical SHA:",
    sha256_file(
        CANONICAL
    )
)


print(
    "SOURCE_LOCK: PASS"
)


# ============================================================
# 2. BUILD FRESH-PROCESS RUNNER
# ============================================================

runner_source = r'''
from pathlib import Path

import ast
import hashlib
import json
import os
import sys

import numpy as np
import pandas as pd


# ============================================================
# PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


SRC_ROOT = (
    PROJECT
    / "src"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


CANONICAL = (
    PROJECT
    / "scripts"
    / "run_stage12_current_best_edge_residual_census.py"
)


BACKUP = (
    PROJECT
    / "scripts"
    / "run_stage12_current_best_edge_residual_census.pre_g3h3e1.py"
)


RUNTIME = (
    SRC_ROOT
    / "biohub_cell_tracking"
    / "stage12_confirmed_rewire.py"
)


INTEGRATION_MANIFEST = (
    S12
    / "stage12_g3h3e2b_canonical_runtime_integration.json"
)


FROZEN_ACTIONS_PATH = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_actions_locked.pkl"
)


E0_METRICS_PATH = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_confirmation_metrics.pkl"
)


E0_SUMMARY_PATH = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_confirmation_summary.json"
)


PROMOTION_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_promoted.json"
)


REPORT = (
    S12
    / "stage12_g3h3e2c_fresh_process_canonical_reproduction.json"
)


# ============================================================
# EXPECTED LOCKS
# ============================================================

POLICY_ID = (
    "ASYM1x5_Q9990_NO_TARGET_CUT_V1"
)


EXPECTED_RUNTIME_SHA = (
    "52c183629ecf80d0fa1568c8c3b2fa3a"
    "4ad6f58a1c34a98dc2bf59feca0ffad0"
)


EXPECTED_PRE_SHA = (
    "d85404884c4c74948676bbbefa4eafac"
    "2c5ac1ef9538e417ae52a4c04a9a7c60"
)


EXPECTED_POST_SHA = (
    "6f2bcc83715e305d0f01517a929382a8"
    "ac6f9f31a0bf32c29c11ed95c5be2181"
)


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


# ============================================================
# FRESH-PROCESS PROOF
# ============================================================

print(
    "========== FRESH PROCESS =========="
)


print(
    "pid:",
    os.getpid()
)


print(
    "python:",
    sys.executable
)


print(
    "preloaded confirmed runtime:",
    (
        "biohub_cell_tracking.stage12_confirmed_rewire"
        in
        sys.modules
    )
)


assert (
    "biohub_cell_tracking.stage12_confirmed_rewire"
    not in
    sys.modules
)


if str(
    SRC_ROOT
) not in sys.path:

    sys.path.insert(
        0,
        str(
            SRC_ROOT
        ),
    )


# ============================================================
# SOURCE LOCK
# ============================================================

assert sha256_file(
    RUNTIME
) == EXPECTED_RUNTIME_SHA


assert sha256_file(
    BACKUP
) == EXPECTED_PRE_SHA


assert sha256_file(
    CANONICAL
) == EXPECTED_POST_SHA


manifest = json.loads(
    INTEGRATION_MANIFEST.read_text(
        encoding="utf-8"
    )
)


assert manifest[
    "policy_id"
] == POLICY_ID


assert manifest[
    "scientific_selection_closed"
] is True


assert manifest[
    "fold0_consumed"
] is True


assert manifest[
    "post_fold0_retuning_allowed"
] is False


print(
    "\n========== FRESH SOURCE LOCK =========="
)


print(
    "runtime:",
    EXPECTED_RUNTIME_SHA
)


print(
    "historical canonical:",
    EXPECTED_PRE_SHA
)


print(
    "integrated canonical:",
    EXPECTED_POST_SHA
)


print(
    "FRESH_SOURCE_LOCK: PASS"
)


# ============================================================
# SAFE PREFIX EXECUTION
#
# Execute each script only through the end of its relevant
# build_current_best definition.
#
# No evaluation code in the original residual-census script is
# executed here.
# ============================================================

def safe_builder_namespace(
    path,
    *,
    expected_builder_name,
):

    source = Path(
        path
    ).read_text(
        encoding="utf-8"
    )


    tree = ast.parse(
        source
    )


    matching = [
        node

        for node
        in tree.body

        if (
            isinstance(
                node,
                ast.FunctionDef,
            )
            and
            node.name
            ==
            expected_builder_name
        )
    ]


    assert len(
        matching
    ) == 1, (
        path,
        expected_builder_name,
        len(
            matching
        ),
    )


    node = matching[
        0
    ]


    assert node.end_lineno is not None


    lines = source.splitlines()


    safe_prefix = "\n".join(
        lines[
            :
            node.end_lineno
        ]
    )


    # No GT evaluation in safe prefix.
    assert "fv9.evaluate_graph(" not in safe_prefix

    assert "official_evaluate(" not in safe_prefix

    assert "fv9.load_graph(" not in safe_prefix


    ns = {
        "__name__":
            (
                "__g3h3e2c_"
                +
                expected_builder_name
                +
                "__"
            ),

        "__file__":
            str(
                path
            ),
    }


    exec(
        compile(
            safe_prefix,
            str(
                path
            ),
            "exec",
        ),
        ns,
        ns,
    )


    assert expected_builder_name in ns

    assert callable(
        ns[
            expected_builder_name
        ]
    )


    return (
        ns,
        node.end_lineno,
    )


# ------------------------------------------------------------
# Historical exact CURRENT_BEST
# ------------------------------------------------------------

old_ns, old_end = safe_builder_namespace(
    BACKUP,
    expected_builder_name=
        "build_current_best",
)


old_builder = old_ns[
    "build_current_best"
]


# ------------------------------------------------------------
# Integrated CURRENT_BEST
#
# Current canonical contains:
#
#   _build_current_best_pre_g3h3e1
#   build_current_best
#
# Safe prefix through CURRENT build_current_best includes both.
# ------------------------------------------------------------

new_ns, new_end = safe_builder_namespace(
    CANONICAL,
    expected_builder_name=
        "build_current_best",
)


new_builder = new_ns[
    "build_current_best"
]


assert (
    "_build_current_best_pre_g3h3e1"
    in new_ns
)


print(
    "\n========== FRESH BUILDER RECOVERY =========="
)


print(
    "historical safe-prefix lines:",
    old_end
)


print(
    "integrated safe-prefix lines:",
    new_end
)


print(
    "integrated historical-base builder present:",
    "YES"
)


print(
    "GT touched:",
    "NO"
)


print(
    "FRESH_CANONICAL_BUILDER_RECOVERY: PASS"
)


# ============================================================
# CORPUS CONTRACT
# ============================================================

old_names_44 = list(
    old_ns[
        "names_44"
    ]
)


old_names_6 = list(
    old_ns[
        "names_6"
    ]
)


new_names_44 = list(
    new_ns[
        "names_44"
    ]
)


new_names_6 = list(
    new_ns[
        "names_6"
    ]
)


assert old_names_44 == new_names_44

assert old_names_6 == new_names_6


assert len(
    old_names_44
) == 15


assert len(
    old_names_6
) == 25


names = (
    old_names_44
    +
    old_names_6
)


assert len(
    names
) == 40


# ============================================================
# LOAD FROZEN REFERENCE ARTIFACTS
#
# These are already-consumed locked-test outputs.
# They are references for implementation fidelity, not model
# selection.
# ============================================================

frozen_actions = pd.read_pickle(
    FROZEN_ACTIONS_PATH
)


e0_metrics = pd.read_pickle(
    E0_METRICS_PATH
)


e0 = json.loads(
    E0_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


promotion = json.loads(
    PROMOTION_PATH.read_text(
        encoding="utf-8"
    )
)


assert len(
    frozen_actions
) == 2712


assert len(
    e0_metrics
) == 40


assert e0[
    "confirmed"
] is True


assert e0[
    "decision"
] == (
    "LOCKED_FOLD0_CONFIRMED"
)


assert promotion[
    "status"
] == (
    "CONFIRMED_POLICY_PROMOTED"
)


# ============================================================
# GRAPH HELPERS
# ============================================================

def edge_pairs(
    graph,
):

    edges = (
        graph.edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    return set(
        zip(
            edges[
                "source_id"
            ].astype(
                int
            ),

            edges[
                "target_id"
            ].astype(
                int
            ),
        )
    )


def node_ids(
    graph,
):

    nodes = (
        graph.node_attrs(
            attr_keys=[
                "node_id",
            ]
        )
        .to_pandas()
    )


    return set(
        nodes[
            "node_id"
        ].astype(
            int
        )
    )


# ============================================================
# GT-FREE FULL-40 TOPOLOGY FIDELITY
#
# Build both historical and integrated graphs first.
#
# Compare topology to frozen Fold0 action artifact BEFORE
# loading any GT.
# ============================================================

print(
    "\n============================================================"
)


print(
    "PART A — FRESH FULL-40 GT-FREE TOPOLOGY FIDELITY"
)


print(
    "============================================================"
)


graph_pairs = {}

topology_rows = []


for i, name in enumerate(
    names,
    start=1,
):

    old_graph, old_scale = (
        old_builder(
            name
        )
    )


    new_graph, new_scale = (
        new_builder(
            name
        )
    )


    assert np.allclose(
        np.asarray(
            old_scale,
            dtype=float,
        ),
        np.asarray(
            new_scale,
            dtype=float,
        ),
        atol=1e-12,
        rtol=0.0,
    )


    old_nodes = node_ids(
        old_graph
    )


    new_nodes = node_ids(
        new_graph
    )


    # Confirmed rewire adds/removes edges only.
    assert old_nodes == new_nodes, name


    old_edges = edge_pairs(
        old_graph
    )


    new_edges = edge_pairs(
        new_graph
    )


    added = (
        new_edges
        -
        old_edges
    )


    removed = (
        old_edges
        -
        new_edges
    )


    if name.startswith(
        "44b6_"
    ):

        expected_actions = (
            frozen_actions.iloc[
                0:0
            ]
        )


        assert len(
            added
        ) == 0, name


        assert len(
            removed
        ) == 0, name


        assert old_edges == new_edges, name


        expected_action_count = 0

        expected_source_occ = 0


    else:

        assert name.startswith(
            "6bba_"
        )


        expected_actions = (
            frozen_actions[
                frozen_actions[
                    "dataset"
                ].eq(
                    name
                )
            ]
            .copy()
        )


        expected_pairs = set(
            zip(
                expected_actions[
                    "source_id"
                ].astype(
                    int
                ),

                expected_actions[
                    "target_id"
                ].astype(
                    int
                ),
            )
        )


        expected_action_count = int(
            len(
                expected_actions
            )
        )


        expected_source_occ = int(
            pd.to_numeric(
                expected_actions[
                    "source_occupied"
                ],
                errors="raise",
            )
            .astype(
                int
            )
            .sum()
        )


        # Exact action-edge identity.
        assert added == expected_pairs, (
            name,
            len(
                added
            ),
            len(
                expected_pairs
            ),
        )


        # NO_TARGET_CUT only removes a source incumbent.
        assert len(
            removed
        ) == expected_source_occ, (
            name,
            len(
                removed
            ),
            expected_source_occ,
        )


        assert (
            len(
                new_edges
            )
            -
            len(
                old_edges
            )
        ) == (
            expected_action_count
            -
            expected_source_occ
        )


    topology_rows.append(
        {
            "dataset":
                name,

            "prefix":
                (
                    "44b6"
                    if name.startswith(
                        "44b6_"
                    )
                    else
                    "6bba"
                ),

            "expected_actions":
                expected_action_count,

            "added_edges":
                int(
                    len(
                        added
                    )
                ),

            "expected_source_occupied":
                expected_source_occ,

            "removed_edges":
                int(
                    len(
                        removed
                    )
                ),

            "node_set_exact":
                True,

            "scale_exact":
                True,
        }
    )


    # Preserve graph objects for official evaluation only after
    # all GT-free topology checks for this dataset have passed.
    graph_pairs[
        name
    ] = (
        old_graph,
        new_graph,
        old_scale,
    )


    print(
        f"{i:2d}/40"
        f" | {name}"
        f" | actions={expected_action_count}"
        f" | added={len(added)}"
        f" | removed={len(removed)}"
    )


topology_df = pd.DataFrame(
    topology_rows
)


assert int(
    topology_df[
        "expected_actions"
    ].sum()
) == 2712


assert int(
    topology_df[
        "added_edges"
    ].sum()
) == 2712


assert int(
    topology_df[
        "expected_source_occupied"
    ].sum()
) == 82


assert int(
    topology_df[
        "removed_edges"
    ].sum()
) == 82


print(
    "\n========== FRESH TOPOLOGY FIDELITY SUMMARY =========="
)


print(
    "datasets:",
    len(
        topology_df
    )
)


print(
    "frozen actions:",
    int(
        topology_df[
            "expected_actions"
        ].sum()
    )
)


print(
    "exact added edges:",
    int(
        topology_df[
            "added_edges"
        ].sum()
    )
)


print(
    "frozen source-occupied actions:",
    int(
        topology_df[
            "expected_source_occupied"
        ].sum()
    )
)


print(
    "exact removed edges:",
    int(
        topology_df[
            "removed_edges"
        ].sum()
    )
)


print(
    "GT loaded:",
    "NO"
)


print(
    "FULL40_CANONICAL_TOPOLOGY_FIDELITY: PASS"
)


# ============================================================
# PART B — OFFICIAL METRIC REPRODUCTION
#
# Fold0 was already consumed in G3H3E0.
# These GT reads are strictly implementation-fidelity checks.
# ============================================================

print(
    "\n============================================================"
)


print(
    "PART B — CONSUMED FOLD0 IMPLEMENTATION REPRODUCTION"
)


print(
    "============================================================"
)


fv9 = new_ns[
    "fv9"
]


TRAIN_ROOT = new_ns[
    "TRAIN_ROOT"
]


estimated_nodes = new_ns[
    "estimated_nodes"
]


fresh_rows = []

baseline_records = []

policy_records = []


for i, name in enumerate(
    names,
    start=1,
):

    (
        old_graph,
        new_graph,
        scale,
    ) = graph_pairs[
        name
    ]


    gt = fv9.load_graph(
        TRAIN_ROOT
        / f"{name}.geff"
    )


    n_total = float(
        estimated_nodes[
            name
        ]
    )


    # Evaluator mutates predictions, so evaluate copies.
    old_eval_graph = old_graph.copy()

    new_eval_graph = new_graph.copy()


    base_metric = fv9.evaluate_graph(
        old_eval_graph,
        gt,
        scale,
        n_total,
        fv9.FROZEN_V9_CONFIG,
    )


    policy_metric = fv9.evaluate_graph(
        new_eval_graph,
        gt,
        scale,
        n_total,
        fv9.FROZEN_V9_CONFIG,
    )


    baseline_records.append(
        base_metric
    )


    policy_records.append(
        policy_metric
    )


    fresh_rows.append(
        {
            "dataset":
                name,

            "base_edge_tp":
                int(
                    base_metric[
                        "edge_tp"
                    ]
                ),

            "base_edge_fp":
                int(
                    base_metric[
                        "edge_fp"
                    ]
                ),

            "base_edge_fn":
                int(
                    base_metric[
                        "edge_fn"
                    ]
                ),

            "base_edge_jaccard":
                float(
                    base_metric[
                        "edge_jaccard"
                    ]
                ),

            "base_adj_edge_jaccard":
                float(
                    base_metric[
                        "adj_edge_jaccard"
                    ]
                ),

            "base_node_recall":
                float(
                    base_metric[
                        "node_recall"
                    ]
                ),

            "policy_edge_tp":
                int(
                    policy_metric[
                        "edge_tp"
                    ]
                ),

            "policy_edge_fp":
                int(
                    policy_metric[
                        "edge_fp"
                    ]
                ),

            "policy_edge_fn":
                int(
                    policy_metric[
                        "edge_fn"
                    ]
                ),

            "policy_edge_jaccard":
                float(
                    policy_metric[
                        "edge_jaccard"
                    ]
                ),

            "policy_adj_edge_jaccard":
                float(
                    policy_metric[
                        "adj_edge_jaccard"
                    ]
                ),

            "policy_node_recall":
                float(
                    policy_metric[
                        "node_recall"
                    ]
                ),
        }
    )


    print(
        f"{i:2d}/40"
        f" | {name}"
        f" | base TP/FP/FN="
        f"{base_metric['edge_tp']}/"
        f"{base_metric['edge_fp']}/"
        f"{base_metric['edge_fn']}"
        f" | policy="
        f"{policy_metric['edge_tp']}/"
        f"{policy_metric['edge_fp']}/"
        f"{policy_metric['edge_fn']}"
    )


fresh_df = pd.DataFrame(
    fresh_rows
)


# ============================================================
# DATASET-BY-DATASET FIDELITY AGAINST E0
# ============================================================

reference = (
    e0_metrics[
        [
            "dataset",

            "base_edge_tp",
            "base_edge_fp",
            "base_edge_fn",
            "base_edge_jaccard",
            "base_adj_edge_jaccard",
            "base_node_recall",

            "policy_edge_tp",
            "policy_edge_fp",
            "policy_edge_fn",
            "policy_edge_jaccard",
            "policy_adj_edge_jaccard",
            "policy_node_recall",
        ]
    ]
    .sort_values(
        "dataset",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


fresh_cmp = (
    fresh_df[
        reference.columns.tolist()
    ]
    .sort_values(
        "dataset",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert fresh_cmp[
    "dataset"
].tolist() == reference[
    "dataset"
].tolist()


integer_cols = [
    "base_edge_tp",
    "base_edge_fp",
    "base_edge_fn",

    "policy_edge_tp",
    "policy_edge_fp",
    "policy_edge_fn",
]


for col in integer_cols:

    assert np.array_equal(
        fresh_cmp[
            col
        ].to_numpy(
            dtype=int
        ),
        reference[
            col
        ].to_numpy(
            dtype=int
        ),
    ), col


float_cols = [
    "base_edge_jaccard",
    "base_adj_edge_jaccard",
    "base_node_recall",

    "policy_edge_jaccard",
    "policy_adj_edge_jaccard",
    "policy_node_recall",
]


max_metric_diff = {}


for col in float_cols:

    fresh_values = fresh_cmp[
        col
    ].to_numpy(
        dtype=float
    )


    reference_values = reference[
        col
    ].to_numpy(
        dtype=float
    )


    max_diff = float(
        np.max(
            np.abs(
                fresh_values
                -
                reference_values
            )
        )
    )


    max_metric_diff[
        col
    ] = max_diff


    assert np.allclose(
        fresh_values,
        reference_values,
        rtol=0.0,
        atol=1e-12,
    ), (
        col,
        max_diff,
    )


print(
    "\n========== DATASET-LEVEL E0 REPRODUCTION =========="
)


print(
    "integer metrics exact:",
    "YES"
)


print(
    "max absolute metric diffs:"
)


for key, value in max_metric_diff.items():

    print(
        f"  {key}: {value}"
    )


print(
    "DATASET_LEVEL_IMPLEMENTATION_FIDELITY: PASS"
)


# ============================================================
# AGGREGATE FIDELITY
# ============================================================

baseline_summary = fv9.summarise(
    baseline_records
)


policy_summary = fv9.summarise(
    policy_records
)


base_tp = int(
    fresh_df[
        "base_edge_tp"
    ].sum()
)


base_fp = int(
    fresh_df[
        "base_edge_fp"
    ].sum()
)


base_fn = int(
    fresh_df[
        "base_edge_fn"
    ].sum()
)


policy_tp = int(
    fresh_df[
        "policy_edge_tp"
    ].sum()
)


policy_fp = int(
    fresh_df[
        "policy_edge_fp"
    ].sum()
)


policy_fn = int(
    fresh_df[
        "policy_edge_fn"
    ].sum()
)


assert base_tp == int(
    e0[
        "baseline"
    ][
        "edge_tp"
    ]
)


assert base_fp == int(
    e0[
        "baseline"
    ][
        "edge_fp"
    ]
)


assert base_fn == int(
    e0[
        "baseline"
    ][
        "edge_fn"
    ]
)


assert policy_tp == int(
    e0[
        "policy"
    ][
        "edge_tp"
    ]
)


assert policy_fp == int(
    e0[
        "policy"
    ][
        "edge_fp"
    ]
)


assert policy_fn == int(
    e0[
        "policy"
    ][
        "edge_fn"
    ]
)


aggregate_fields = [
    "edge_jaccard",
    "adj_edge_jaccard",
    "division_jaccard",
    "node_recall",
    "score",
]


aggregate_diffs = {}


for field in aggregate_fields:

    b_diff = abs(
        float(
            baseline_summary[
                field
            ]
        )
        -
        float(
            e0[
                "baseline"
            ][
                field
            ]
        )
    )


    p_diff = abs(
        float(
            policy_summary[
                field
            ]
        )
        -
        float(
            e0[
                "policy"
            ][
                field
            ]
        )
    )


    aggregate_diffs[
        f"baseline_{field}"
    ] = b_diff


    aggregate_diffs[
        f"policy_{field}"
    ] = p_diff


    assert b_diff <= 1e-12, (
        field,
        "baseline",
        b_diff,
    )


    assert p_diff <= 1e-12, (
        field,
        "policy",
        p_diff,
    )


delta_score = float(
    policy_summary[
        "score"
    ]
    -
    baseline_summary[
        "score"
    ]
)


assert abs(
    delta_score
    -
    float(
        e0[
            "delta"
        ][
            "score"
        ]
    )
) <= 1e-12


print(
    "\n========== FRESH AGGREGATE REPRODUCTION =========="
)


print(
    "baseline TP/FP/FN:",
    base_tp,
    "/",
    base_fp,
    "/",
    base_fn
)


print(
    "expected:",
    e0[
        "baseline"
    ][
        "edge_tp"
    ],
    "/",
    e0[
        "baseline"
    ][
        "edge_fp"
    ],
    "/",
    e0[
        "baseline"
    ][
        "edge_fn"
    ],
)


print(
    "policy TP/FP/FN:",
    policy_tp,
    "/",
    policy_fp,
    "/",
    policy_fn
)


print(
    "expected:",
    e0[
        "policy"
    ][
        "edge_tp"
    ],
    "/",
    e0[
        "policy"
    ][
        "edge_fp"
    ],
    "/",
    e0[
        "policy"
    ][
        "edge_fn"
    ],
)


print(
    "baseline score:",
    f"{baseline_summary['score']:.15f}"
)


print(
    "expected:",
    f"{float(e0['baseline']['score']):.15f}"
)


print(
    "policy score:",
    f"{policy_summary['score']:.15f}"
)


print(
    "expected:",
    f"{float(e0['policy']['score']):.15f}"
)


print(
    "delta:",
    f"{delta_score:+.15f}"
)


print(
    "expected:",
    f"{float(e0['delta']['score']):+.15f}"
)


print(
    "FRESH_AGGREGATE_IMPLEMENTATION_FIDELITY: PASS"
)


# ============================================================
# FORMAL REPORT
# ============================================================

REPORT_DATA = {
    "stage":
        "12.11G3H3E2C",

    "status":
        "FRESH_PROCESS_CANONICAL_REPRODUCTION_PASS",

    "purpose":
        "IMPLEMENTATION_FIDELITY_ONLY",

    "policy_id":
        POLICY_ID,

    "scientific_selection_closed":
        True,

    "fold0_consumed_before_this_stage":
        True,

    "post_fold0_retuning_allowed":
        False,

    "source_integrity": {
        "runtime_sha256":
            EXPECTED_RUNTIME_SHA,

        "historical_canonical_sha256":
            EXPECTED_PRE_SHA,

        "integrated_canonical_sha256":
            EXPECTED_POST_SHA,
    },

    "fresh_process": {
        "pid":
            os.getpid(),

        "python":
            sys.executable,

        "runtime_preloaded":
            False,
    },

    "corpus": {
        "datasets":
            40,

        "44b6":
            15,

        "6bba":
            25,
    },

    "topology_fidelity": {
        "frozen_actions":
            2712,

        "exact_added_edges":
            int(
                topology_df[
                    "added_edges"
                ].sum()
            ),

        "frozen_source_occupied_actions":
            82,

        "exact_removed_edges":
            int(
                topology_df[
                    "removed_edges"
                ].sum()
            ),

        "node_sets_exact":
            True,

        "scales_exact":
            True,

        "all_action_edge_pairs_exact":
            True,
    },

    "dataset_metric_fidelity": {
        "integer_metrics_exact":
            True,

        "max_absolute_float_differences":
            max_metric_diff,
    },

    "aggregate": {
        "baseline": {
            "edge_tp":
                base_tp,

            "edge_fp":
                base_fp,

            "edge_fn":
                base_fn,

            "score":
                float(
                    baseline_summary[
                        "score"
                    ]
                ),
        },

        "policy": {
            "edge_tp":
                policy_tp,

            "edge_fp":
                policy_fp,

            "edge_fn":
                policy_fn,

            "score":
                float(
                    policy_summary[
                        "score"
                    ]
                ),
        },

        "delta_score":
            delta_score,

        "aggregate_metric_absolute_differences":
            aggregate_diffs,
    },

    "matches_consumed_e0_exactly":
        True,

    "source_modified":
        False,

    "prediction_geff_written":
        False,

    "development_reopened":
        False,

    "next":
        "CLOSE_STAGE12_REWIRE_INTEGRATION_AND_ADVANCE",
}


REPORT.write_text(
    json.dumps(
        REPORT_DATA,
        indent=2,
    ),
    encoding="utf-8",
)


assert REPORT.exists()


print(
    "\n========== 12.11G3H3E2C DECISION =========="
)


print(
    "FRESH_PROCESS_CANONICAL_REPRODUCTION: PASS"
)


print(
    "implementation fidelity:",
    "EXACT"
)


print(
    "all 2712 action edge pairs reproduced:",
    "YES"
)


print(
    "all 82 source-incumbent removals reproduced:",
    "YES"
)


print(
    "dataset-level official metrics reproduced:",
    "YES"
)


print(
    "aggregate locked result reproduced:",
    "YES"
)


print(
    "policy score:",
    f"{policy_summary['score']:.15f}"
)


print(
    "delta:",
    f"{delta_score:+.15f}"
)


print(
    "scientific selection reopened:",
    "NO"
)


print(
    "post-Fold0 tuning allowed:",
    "NO"
)


print(
    "source modified:",
    "NO"
)


print(
    "prediction GEFF written:",
    "NO"
)


print(
    "next:"
)


print(
    "CLOSE_STAGE12_REWIRE_INTEGRATION_AND_ADVANCE"
)
'''


RUNNER.write_text(
    textwrap.dedent(
        runner_source
    ),
    encoding="utf-8",
)


assert RUNNER.exists()


RUNNER_SHA = sha256_file(
    RUNNER
)


print(
    "\n========== E2C RUNNER FROZEN =========="
)


print(
    "runner:",
    RUNNER
)


print(
    "runner SHA:",
    RUNNER_SHA
)


# ============================================================
# 3. STATIC COMPILE RUNNER
# ============================================================

compile_proc = subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        str(
            RUNNER
        ),
    ],
    cwd=str(
        PROJECT
    ),
    capture_output=True,
    text=True,
)


assert compile_proc.returncode == 0, (
    compile_proc.stdout,
    compile_proc.stderr,
)


print(
    "runner py_compile:",
    "PASS"
)


# ============================================================
# 4. RUN IN A NEW PYTHON PROCESS
# ============================================================

env = os.environ.copy()


existing_pythonpath = env.get(
    "PYTHONPATH",
    "",
)


src_string = str(
    PROJECT
    / "src"
)


env[
    "PYTHONPATH"
] = (
    src_string
    if not existing_pythonpath
    else (
        src_string
        +
        os.pathsep
        +
        existing_pythonpath
    )
)


print(
    "\n============================================================"
)


print(
    "LAUNCH FRESH PROCESS"
)


print(
    "============================================================"
)


proc = subprocess.run(
    [
        sys.executable,
        str(
            RUNNER
        ),
    ],
    cwd=str(
        PROJECT
    ),
    env=env,
    capture_output=True,
    text=True,
)


print(
    proc.stdout
)


if proc.stderr:

    print(
        "\n========== CHILD STDERR =========="
    )

    print(
        proc.stderr
    )


print(
    "\nchild return code:",
    proc.returncode
)


assert proc.returncode == 0, (
    "\nFresh-process canonical reproduction failed.\n"
    "This is an ENGINEERING fidelity failure.\n"
    "Do NOT change the frozen policy.\n\n"
    f"STDERR:\n{proc.stderr}"
)


# ============================================================
# 5. VERIFY FORMAL CHILD REPORT
# ============================================================

assert REPORT.exists(), (
    "Fresh process returned success but formal report missing."
)


report = json.loads(
    REPORT.read_text(
        encoding="utf-8"
    )
)


assert report[
    "status"
] == (
    "FRESH_PROCESS_CANONICAL_REPRODUCTION_PASS"
)


assert report[
    "purpose"
] == (
    "IMPLEMENTATION_FIDELITY_ONLY"
)


assert report[
    "matches_consumed_e0_exactly"
] is True


assert report[
    "topology_fidelity"
][
    "all_action_edge_pairs_exact"
] is True


assert report[
    "topology_fidelity"
][
    "exact_added_edges"
] == 2712


assert report[
    "topology_fidelity"
][
    "exact_removed_edges"
] == 82


assert report[
    "dataset_metric_fidelity"
][
    "integer_metrics_exact"
] is True


assert report[
    "development_reopened"
] is False


assert report[
    "source_modified"
] is False


# ============================================================
# 6. FINAL
# ============================================================

print(
    "\n========== 12.11G3H3E2C PARENT DECISION =========="
)


print(
    "FRESH_PROCESS_RETURN_CODE:",
    proc.returncode
)


print(
    "FORMAL_REPORT:",
    REPORT
)


print(
    "FRESH_PROCESS_CANONICAL_REPRODUCTION: PASS"
)


print(
    "implementation fidelity:",
    "EXACT"
)


print(
    "source modification in E2C:",
    "NO"
)


print(
    "scientific selection reopened:",
    "NO"
)


print(
    "post-Fold0 tuning:",
    "NO"
)


print(
    "next:"
)


print(
    "CLOSE_STAGE12_REWIRE_INTEGRATION_AND_ADVANCE"
)

In [ ]:
# ============================================================
# Stage 12.11G3H3E3
#
# STAGE12 REWIRE BRANCH FORMAL CLOSURE
# + FINAL CANONICAL CURRENT_BEST CHECKPOINT
#
# ============================================================
#
# This stage performs NO scientific work.
#
# It only closes the already-completed provenance chain:
#
#   development selection
#       ->
#   final policy freeze
#       ->
#   one-shot locked Fold0 confirmation
#       ->
#   promotion
#       ->
#   runtime packaging
#       ->
#   canonical integration
#       ->
#   fresh-process exact reproduction
#
# ------------------------------------------------------------
# FINAL PROMOTED POLICY
# ------------------------------------------------------------
#
#   ASYM1x5_Q9990_NO_TARGET_CUT_V1
#
# Development:
#
#   0.7919774336227487
#       ->
#   0.7920280500935799
#
# Locked Fold0:
#
#   0.7863444776365197
#       ->
#   0.7863985958888573
#
#   TP/FP/FN delta:
#       0 / -2 / 0
#
# ------------------------------------------------------------
# ENGINEERING FIDELITY
# ------------------------------------------------------------
#
# fresh process:
#
#   2712 / 2712 added action edges exact
#   82 / 82 source-incumbent removals exact
#   40 / 40 dataset metrics exact
#   aggregate metrics exact
#
# ------------------------------------------------------------
# NO
# ------------------------------------------------------------
#
# GT read
# evaluator
# scorer execution
# action generation
# policy tuning
# Fold0 reuse
# source modification
# GEFF output
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import hashlib
import json


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


FINAL_POLICY_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_frozen.json"
)


E0_SUMMARY_PATH = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_confirmation_summary.json"
)


PROMOTION_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_promoted.json"
)


E1_CHECKPOINT_PATH = (
    S12
    / "stage12_current_best_checkpoint.post_g3h3e1.json"
)


INTEGRATION_MANIFEST_PATH = (
    S12
    / "stage12_g3h3e2b_canonical_runtime_integration.json"
)


E2C_REPORT_PATH = (
    S12
    / "stage12_g3h3e2c_fresh_process_canonical_reproduction.json"
)


RUNTIME_PATH = (
    PROJECT
    / "src"
    / "biohub_cell_tracking"
    / "stage12_confirmed_rewire.py"
)


CANONICAL_PATH = (
    PROJECT
    / "scripts"
    / "run_stage12_current_best_edge_residual_census.py"
)


HISTORICAL_CANONICAL_PATH = (
    PROJECT
    / "scripts"
    / "run_stage12_current_best_edge_residual_census.pre_g3h3e1.py"
)


# ------------------------------------------------------------
# Formal closure outputs
# ------------------------------------------------------------

BRANCH_CLOSURE_OUT = (
    S12
    / "stage12_g3h3_rewire_branch_closed.json"
)


FINAL_CHECKPOINT_OUT = (
    S12
    / "stage12_current_best_checkpoint.post_g3h3e3.json"
)


CLOSURE_SUMMARY_OUT = (
    S12
    / "stage12_g3h3e3_rewire_branch_closure_summary.json"
)


for path in [
    FINAL_POLICY_PATH,
    E0_SUMMARY_PATH,
    PROMOTION_PATH,
    E1_CHECKPOINT_PATH,
    INTEGRATION_MANIFEST_PATH,
    E2C_REPORT_PATH,
    RUNTIME_PATH,
    CANONICAL_PATH,
    HISTORICAL_CANONICAL_PATH,
]:

    assert path.exists(), path


for path in [
    BRANCH_CLOSURE_OUT,
    FINAL_CHECKPOINT_OUT,
    CLOSURE_SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nG3H3E3 formal closure output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite branch-closure provenance."
        )


# ============================================================
# 1. AUTHORITATIVE IDENTIFIERS
# ============================================================

POLICY_ID = (
    "ASYM1x5_Q9990_NO_TARGET_CUT_V1"
)


POLICY_SHA = (
    "77f0b07cafb21d0705917ff71b750fde4"
    "5f3c7ddc17c13fecaff0ac673ebf776"
)


PROMOTION_SHA = (
    "8a9b675b1725f7512b350adddf31e78a"
    "664e65cba5148b993c0c8d6a7d7c63a4"
)


E1_CHECKPOINT_SHA = (
    "decd7c3681df989a66e308a500911ece5"
    "b952250de167b8431e4cfc66178d146"
)


RUNTIME_SHA = (
    "52c183629ecf80d0fa1568c8c3b2fa3a"
    "4ad6f58a1c34a98dc2bf59feca0ffad0"
)


CANONICAL_PRE_SHA = (
    "d85404884c4c74948676bbbefa4eafac"
    "2c5ac1ef9538e417ae52a4c04a9a7c60"
)


CANONICAL_POST_SHA = (
    "6f2bcc83715e305d0f01517a929382a8"
    "ac6f9f31a0bf32c29c11ed95c5be2181"
)


EXPECTED_DEV_BASE_SCORE = (
    0.7919774336227487
)


EXPECTED_DEV_CURRENT_BEST_SCORE = (
    0.7920280500935799
)


EXPECTED_FOLD0_BASE_SCORE = (
    0.7863444776365197
)


EXPECTED_FOLD0_CURRENT_BEST_SCORE = (
    0.7863985958888573
)


EXPECTED_FOLD0_ACTIONS = 2712

EXPECTED_SOURCE_REMOVALS = 82


# ============================================================
# 2. SHA HELPER
# ============================================================

def sha256_file(
    path,
):

    h = hashlib.sha256()


    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024
                *
                1024
            )


            if not chunk:
                break


            h.update(
                chunk
            )


    return h.hexdigest()


# ============================================================
# 3. VERIFY SOURCE LOCK
# ============================================================

assert sha256_file(
    RUNTIME_PATH
) == RUNTIME_SHA


assert sha256_file(
    CANONICAL_PATH
) == CANONICAL_POST_SHA


assert sha256_file(
    HISTORICAL_CANONICAL_PATH
) == CANONICAL_PRE_SHA


print(
    "========== G3H3E3 SOURCE LOCK =========="
)


print(
    "runtime SHA:",
    RUNTIME_SHA
)


print(
    "historical canonical SHA:",
    CANONICAL_PRE_SHA
)


print(
    "integrated canonical SHA:",
    CANONICAL_POST_SHA
)


print(
    "SOURCE_LOCK: PASS"
)


# ============================================================
# 4. VERIFY FINAL POLICY
# ============================================================

policy = json.loads(
    FINAL_POLICY_PATH.read_text(
        encoding="utf-8"
    )
)


assert policy[
    "policy_id"
] == POLICY_ID


assert policy[
    "final_policy_sha256"
] == POLICY_SHA


assert policy[
    "status"
] == (
    "FINAL_DEVELOPMENT_POLICY_FROZEN"
)


assert policy[
    "development_closed_after_freeze"
] is True


assert policy[
    "future_scorer_retuning"
] is False


assert policy[
    "future_shortlist_retuning"
] is False


assert policy[
    "future_rank_rule_retuning"
] is False


assert policy[
    "future_score_quantile_retuning"
] is False


assert policy[
    "future_structural_rule_retuning"
] is False


print(
    "\n========== FINAL POLICY =========="
)


print(
    "policy:",
    POLICY_ID
)


print(
    "policy SHA:",
    POLICY_SHA
)


print(
    "development closed:",
    "YES"
)


# ============================================================
# 5. VERIFY LOCKED FOLD0 CONFIRMATION
# ============================================================

e0 = json.loads(
    E0_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert e0[
    "status"
] == (
    "ONE_SHOT_LOCKED_FOLD0_CONFIRMATION_COMPLETE"
)


assert e0[
    "confirmed"
] is True


assert e0[
    "decision"
] == (
    "LOCKED_FOLD0_CONFIRMED"
)


assert e0[
    "baseline_fidelity_pass"
] is True


assert e0[
    "fold0_actions"
] == EXPECTED_FOLD0_ACTIONS


assert e0[
    "post_fold0_retuning_allowed"
] is False


assert abs(
    float(
        e0[
            "baseline"
        ][
            "score"
        ]
    )
    -
    EXPECTED_FOLD0_BASE_SCORE
) <= 1e-12


assert abs(
    float(
        e0[
            "policy"
        ][
            "score"
        ]
    )
    -
    EXPECTED_FOLD0_CURRENT_BEST_SCORE
) <= 1e-12


assert int(
    e0[
        "delta"
    ][
        "edge_tp"
    ]
) == 0


assert int(
    e0[
        "delta"
    ][
        "edge_fp"
    ]
) == -2


assert int(
    e0[
        "delta"
    ][
        "edge_fn"
    ]
) == 0


print(
    "\n========== LOCKED FOLD0 =========="
)


print(
    "baseline:",
    e0[
        "baseline"
    ][
        "score"
    ]
)


print(
    "promoted:",
    e0[
        "policy"
    ][
        "score"
    ]
)


print(
    "delta:",
    e0[
        "delta"
    ][
        "score"
    ]
)


print(
    "TP/FP/FN delta:",
    "0 / -2 / 0"
)


print(
    "decision:",
    e0[
        "decision"
    ]
)


# ============================================================
# 6. VERIFY PROMOTION
# ============================================================

promotion = json.loads(
    PROMOTION_PATH.read_text(
        encoding="utf-8"
    )
)


assert promotion[
    "status"
] == (
    "CONFIRMED_POLICY_PROMOTED"
)


assert promotion[
    "policy_id"
] == POLICY_ID


assert promotion[
    "policy_sha256"
] == POLICY_SHA


assert promotion[
    "promotion_sha256"
] == PROMOTION_SHA


assert promotion[
    "fold0_consumed"
] is True


assert promotion[
    "fold0_remains_available_for_retuning"
] is False


assert promotion[
    "post_fold0_retuning_allowed"
] is False


print(
    "\n========== POLICY PROMOTION =========="
)


print(
    "promotion SHA:",
    PROMOTION_SHA
)


print(
    "Fold0 consumed:",
    "YES"
)


print(
    "post-Fold0 tuning:",
    "NO"
)


# ============================================================
# 7. VERIFY G3H3E1 CHECKPOINT
# ============================================================

e1_checkpoint = json.loads(
    E1_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)


assert e1_checkpoint[
    "checkpoint_sha256"
] == E1_CHECKPOINT_SHA


assert e1_checkpoint[
    "promoted_policy"
] == POLICY_ID


assert e1_checkpoint[
    "locked_fold0"
][
    "confirmed"
] is True


assert abs(
    float(
        e1_checkpoint[
            "development"
        ][
            "current_best_score"
        ]
    )
    -
    EXPECTED_DEV_CURRENT_BEST_SCORE
) <= 1e-12


assert abs(
    float(
        e1_checkpoint[
            "locked_fold0"
        ][
            "current_best_score"
        ]
    )
    -
    EXPECTED_FOLD0_CURRENT_BEST_SCORE
) <= 1e-12


# ============================================================
# 8. VERIFY CANONICAL INTEGRATION
# ============================================================

integration = json.loads(
    INTEGRATION_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)


assert integration[
    "status"
] == (
    "FROZEN_RUNTIME_PACKAGED_AND_CANONICAL_INTEGRATED"
)


assert integration[
    "policy_id"
] == POLICY_ID


assert integration[
    "runtime_module_sha256"
] == RUNTIME_SHA


assert integration[
    "canonical_before_sha256"
] == CANONICAL_PRE_SHA


assert integration[
    "canonical_after_sha256"
] == CANONICAL_POST_SHA


assert integration[
    "scientific_selection_closed"
] is True


assert integration[
    "fold0_consumed"
] is True


assert integration[
    "post_fold0_retuning_allowed"
] is False


assert integration[
    "official_metric_evaluated"
] is False


print(
    "\n========== CANONICAL INTEGRATION =========="
)


print(
    "runtime packaged:",
    "YES"
)


print(
    "canonical integrated:",
    "YES"
)


print(
    "runtime SHA:",
    RUNTIME_SHA
)


print(
    "canonical SHA:",
    CANONICAL_POST_SHA
)


# ============================================================
# 9. VERIFY FRESH-PROCESS EXACT REPRODUCTION
# ============================================================

e2c = json.loads(
    E2C_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)


assert e2c[
    "status"
] == (
    "FRESH_PROCESS_CANONICAL_REPRODUCTION_PASS"
)


assert e2c[
    "purpose"
] == (
    "IMPLEMENTATION_FIDELITY_ONLY"
)


assert e2c[
    "policy_id"
] == POLICY_ID


assert e2c[
    "matches_consumed_e0_exactly"
] is True


assert e2c[
    "topology_fidelity"
][
    "frozen_actions"
] == EXPECTED_FOLD0_ACTIONS


assert e2c[
    "topology_fidelity"
][
    "exact_added_edges"
] == EXPECTED_FOLD0_ACTIONS


assert e2c[
    "topology_fidelity"
][
    "frozen_source_occupied_actions"
] == EXPECTED_SOURCE_REMOVALS


assert e2c[
    "topology_fidelity"
][
    "exact_removed_edges"
] == EXPECTED_SOURCE_REMOVALS


assert e2c[
    "topology_fidelity"
][
    "all_action_edge_pairs_exact"
] is True


assert e2c[
    "dataset_metric_fidelity"
][
    "integer_metrics_exact"
] is True


for value in (
    e2c[
        "dataset_metric_fidelity"
    ][
        "max_absolute_float_differences"
    ].values()
):

    assert float(
        value
    ) == 0.0


assert abs(
    float(
        e2c[
            "aggregate"
        ][
            "baseline"
        ][
            "score"
        ]
    )
    -
    EXPECTED_FOLD0_BASE_SCORE
) <= 1e-12


assert abs(
    float(
        e2c[
            "aggregate"
        ][
            "policy"
        ][
            "score"
        ]
    )
    -
    EXPECTED_FOLD0_CURRENT_BEST_SCORE
) <= 1e-12


assert e2c[
    "source_modified"
] is False


assert e2c[
    "development_reopened"
] is False


assert e2c[
    "post_fold0_retuning_allowed"
] is False


print(
    "\n========== FRESH-PROCESS REPRODUCTION =========="
)


print(
    "40 datasets exact:",
    "YES"
)


print(
    "2712 action edges exact:",
    "YES"
)


print(
    "82 removals exact:",
    "YES"
)


print(
    "dataset metrics exact:",
    "YES"
)


print(
    "float metric max diff:",
    "0.0"
)


print(
    "aggregate exact:",
    "YES"
)


print(
    "FRESH_PROCESS_FIDELITY: PASS"
)


# ============================================================
# 10. FORMAL BRANCH CLOSURE
# ============================================================

BRANCH_CLOSURE = {
    "stage":
        "12.11G3H3E3",

    "branch":
        "STAGE12_MULTIFRAME_RESIDUAL_REWIRE",

    "status":
        "BRANCH_CLOSED_SUCCESSFULLY",

    "final_policy": {
        "policy_id":
            POLICY_ID,

        "policy_sha256":
            POLICY_SHA,

        "promotion_sha256":
            PROMOTION_SHA,
    },

    "development": {
        "baseline_score":
            EXPECTED_DEV_BASE_SCORE,

        "final_score":
            EXPECTED_DEV_CURRENT_BEST_SCORE,

        "delta_score":
            (
                EXPECTED_DEV_CURRENT_BEST_SCORE
                -
                EXPECTED_DEV_BASE_SCORE
            ),
    },

    "locked_fold0": {
        "baseline_score":
            EXPECTED_FOLD0_BASE_SCORE,

        "final_score":
            EXPECTED_FOLD0_CURRENT_BEST_SCORE,

        "delta_score":
            float(
                e0[
                    "delta"
                ][
                    "score"
                ]
            ),

        "edge_delta": {
            "tp":
                0,

            "fp":
                -2,

            "fn":
                0,
        },

        "confirmed":
            True,

        "consumed":
            True,

        "available_for_future_retuning":
            False,
    },

    "engineering": {
        "runtime_sha256":
            RUNTIME_SHA,

        "historical_canonical_sha256":
            CANONICAL_PRE_SHA,

        "integrated_canonical_sha256":
            CANONICAL_POST_SHA,

        "fresh_process_reproduction":
            True,

        "frozen_action_edges_reproduced":
            2712,

        "source_removals_reproduced":
            82,

        "dataset_metrics_exact":
            True,

        "aggregate_metrics_exact":
            True,
    },

    "scientific_selection_closed":
        True,

    "development_reopened":
        False,

    "post_fold0_retuning_allowed":
        False,

    "prediction_geff_written":
        False,

    "branch_may_be_reopened_for_policy_search":
        False,

    "next":
        "ADVANCE_TO_NEXT_PREDECLARED_STAGE12_ROADMAP_ITEM",
}


BRANCH_CLOSURE_SHA = hashlib.sha256(
    json.dumps(
        BRANCH_CLOSURE,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


BRANCH_CLOSURE[
    "branch_closure_sha256"
] = BRANCH_CLOSURE_SHA


BRANCH_CLOSURE_OUT.write_text(
    json.dumps(
        BRANCH_CLOSURE,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 11. FINAL CURRENT_BEST CHECKPOINT
# ============================================================

FINAL_CHECKPOINT = {
    "stage":
        "12.11G3H3E3",

    "checkpoint_id":
        "STAGE12_CURRENT_BEST_POST_G3H3E3",

    "status":
        (
            "CANONICAL_CURRENT_BEST_"
            "CONFIRMED_PACKAGED_REPRODUCED"
        ),

    "parent_checkpoint":
        E1_CHECKPOINT_PATH.name,

    "parent_checkpoint_sha256":
        E1_CHECKPOINT_SHA,

    "promoted_policy":
        POLICY_ID,

    "promoted_policy_sha256":
        POLICY_SHA,

    "branch_closure_sha256":
        BRANCH_CLOSURE_SHA,

    "development": {
        "score":
            EXPECTED_DEV_CURRENT_BEST_SCORE,

        "improvement_over_pre_rewire":
            (
                EXPECTED_DEV_CURRENT_BEST_SCORE
                -
                EXPECTED_DEV_BASE_SCORE
            ),
    },

    "locked_fold0": {
        "edge_tp":
            int(
                e0[
                    "policy"
                ][
                    "edge_tp"
                ]
            ),

        "edge_fp":
            int(
                e0[
                    "policy"
                ][
                    "edge_fp"
                ]
            ),

        "edge_fn":
            int(
                e0[
                    "policy"
                ][
                    "edge_fn"
                ]
            ),

        "score":
            EXPECTED_FOLD0_CURRENT_BEST_SCORE,

        "confirmed":
            True,

        "consumed":
            True,
    },

    "canonical_runtime": {
        "runtime_module":
            str(
                RUNTIME_PATH.relative_to(
                    PROJECT
                )
            ),

        "runtime_sha256":
            RUNTIME_SHA,

        "canonical_builder":
            str(
                CANONICAL_PATH.relative_to(
                    PROJECT
                )
            ),

        "canonical_builder_sha256":
            CANONICAL_POST_SHA,

        "historical_builder_backup":
            str(
                HISTORICAL_CANONICAL_PATH.relative_to(
                    PROJECT
                )
            ),

        "historical_builder_sha256":
            CANONICAL_PRE_SHA,
    },

    "implementation_fidelity": {
        "fresh_process":
            True,

        "all_action_pairs_exact":
            True,

        "all_removals_exact":
            True,

        "dataset_metrics_exact":
            True,

        "aggregate_metrics_exact":
            True,
    },

    "protocol": {
        "scientific_selection_closed":
            True,

        "development_reopened":
            False,

        "post_fold0_retuning_allowed":
            False,

        "fold0_available_for_future_selection":
            False,
    },

    "next":
        "ADVANCE_TO_NEXT_PREDECLARED_STAGE12_ROADMAP_ITEM",
}


FINAL_CHECKPOINT_SHA = hashlib.sha256(
    json.dumps(
        FINAL_CHECKPOINT,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


FINAL_CHECKPOINT[
    "checkpoint_sha256"
] = FINAL_CHECKPOINT_SHA


FINAL_CHECKPOINT_OUT.write_text(
    json.dumps(
        FINAL_CHECKPOINT,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 12. CLOSURE SUMMARY
# ============================================================

CLOSURE_SUMMARY = {
    "stage":
        "12.11G3H3E3",

    "status":
        "STAGE12_REWIRE_INTEGRATION_CLOSED",

    "policy_id":
        POLICY_ID,

    "branch_closure_sha256":
        BRANCH_CLOSURE_SHA,

    "final_current_best_checkpoint":
        FINAL_CHECKPOINT_OUT.name,

    "final_current_best_checkpoint_sha256":
        FINAL_CHECKPOINT_SHA,

    "development_score":
        EXPECTED_DEV_CURRENT_BEST_SCORE,

    "locked_fold0_score":
        EXPECTED_FOLD0_CURRENT_BEST_SCORE,

    "locked_fold0_edge_delta":
        {
            "tp":
                0,

            "fp":
                -2,

            "fn":
                0,
        },

    "fresh_process_exact":
        True,

    "scientific_selection_closed":
        True,

    "fold0_consumed":
        True,

    "development_reopened":
        False,

    "post_fold0_retuning_allowed":
        False,

    "source_modified":
        False,

    "gt_read":
        False,

    "official_metric_evaluated":
        False,

    "prediction_geff_written":
        False,

    "next":
        "ADVANCE_TO_NEXT_PREDECLARED_STAGE12_ROADMAP_ITEM",
}


CLOSURE_SUMMARY_OUT.write_text(
    json.dumps(
        CLOSURE_SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 13. RELOAD INTEGRITY
# ============================================================

closure_check = json.loads(
    BRANCH_CLOSURE_OUT.read_text(
        encoding="utf-8"
    )
)


checkpoint_check = json.loads(
    FINAL_CHECKPOINT_OUT.read_text(
        encoding="utf-8"
    )
)


summary_check = json.loads(
    CLOSURE_SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert closure_check[
    "status"
] == (
    "BRANCH_CLOSED_SUCCESSFULLY"
)


assert closure_check[
    "branch_closure_sha256"
] == BRANCH_CLOSURE_SHA


assert closure_check[
    "scientific_selection_closed"
] is True


assert closure_check[
    "post_fold0_retuning_allowed"
] is False


assert checkpoint_check[
    "checkpoint_sha256"
] == FINAL_CHECKPOINT_SHA


assert checkpoint_check[
    "promoted_policy"
] == POLICY_ID


assert checkpoint_check[
    "implementation_fidelity"
][
    "fresh_process"
] is True


assert checkpoint_check[
    "protocol"
][
    "fold0_available_for_future_selection"
] is False


assert summary_check[
    "status"
] == (
    "STAGE12_REWIRE_INTEGRATION_CLOSED"
)


assert summary_check[
    "fresh_process_exact"
] is True


assert summary_check[
    "gt_read"
] is False


assert summary_check[
    "official_metric_evaluated"
] is False


# ============================================================
# 14. FINAL
# ============================================================

print(
    "\n========== 12.11G3H3E3 FINAL CLOSURE =========="
)


print(
    "STAGE12_REWIRE_BRANCH_CLOSURE: PASS"
)


print(
    "branch:",
    "STAGE12_MULTIFRAME_RESIDUAL_REWIRE"
)


print(
    "policy:",
    POLICY_ID
)


print(
    "development score:",
    f"{EXPECTED_DEV_CURRENT_BEST_SCORE:.15f}"
)


print(
    "locked Fold0 score:",
    f"{EXPECTED_FOLD0_CURRENT_BEST_SCORE:.15f}"
)


print(
    "locked Fold0 TP/FP/FN delta:",
    "0 / -2 / 0"
)


print(
    "fresh-process implementation fidelity:",
    "EXACT"
)


print(
    "runtime SHA:",
    RUNTIME_SHA
)


print(
    "canonical SHA:",
    CANONICAL_POST_SHA
)


print(
    "branch closure SHA:",
    BRANCH_CLOSURE_SHA
)


print(
    "final CURRENT_BEST checkpoint:",
    FINAL_CHECKPOINT_OUT.name
)


print(
    "final checkpoint SHA:",
    FINAL_CHECKPOINT_SHA
)


print(
    "scientific selection closed:",
    "YES"
)


print(
    "Fold0 consumed:",
    "YES"
)


print(
    "development reopened:",
    "NO"
)


print(
    "post-Fold0 retuning:",
    "NO"
)


print(
    "GT read in closure:",
    "NO"
)


print(
    "official metric evaluated in closure:",
    "NO"
)


print(
    "prediction GEFF written:",
    "NO"
)


print(
    "next:"
)


print(
    "ADVANCE_TO_NEXT_PREDECLARED_STAGE12_ROADMAP_ITEM"
)

In [ ]:
# ============================================================
# Stage12 — POST-G3H3 ROADMAP HANDOFF AUDIT
#
# PURPOSE
# ------------------------------------------------------------
# G3H3 rewire branch is formally CLOSED.
#
# Recover the NEXT PREDECLARED Stage12 roadmap item from the
# local repository without inventing a new research branch.
#
# SEARCHES:
#
#   notebooks/
#   scripts/
#   reports/stage12_residual_audit/
#   docs/
#
# Looks for:
#
#   roadmap
#   next
#   next_if_*
#   advance_to_*
#   12.11 / 12.12 stage declarations
#   Stage12 candidate-family declarations
#
# NO:
#   GT
#   model execution
#   graph evaluation
#   source modification
#   artifact modification
#
# Delete after next roadmap item is recovered: YES
# ============================================================

from pathlib import Path

import json
import re


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


SEARCH_ROOTS = [
    PROJECT / "notebooks",
    PROJECT / "scripts",
    PROJECT / "reports" / "stage12_residual_audit",
    PROJECT / "docs",
]


# ============================================================
# 1. CONFIRM G3H3 CLOSURE
# ============================================================

CLOSURE = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
    / "stage12_g3h3_rewire_branch_closed.json"
)


CHECKPOINT = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
    / "stage12_current_best_checkpoint.post_g3h3e3.json"
)


assert CLOSURE.exists(), CLOSURE
assert CHECKPOINT.exists(), CHECKPOINT


closure = json.loads(
    CLOSURE.read_text(
        encoding="utf-8"
    )
)


checkpoint = json.loads(
    CHECKPOINT.read_text(
        encoding="utf-8"
    )
)


assert closure[
    "status"
] == "BRANCH_CLOSED_SUCCESSFULLY"


assert closure[
    "branch"
] == "STAGE12_MULTIFRAME_RESIDUAL_REWIRE"


assert closure[
    "scientific_selection_closed"
] is True


assert closure[
    "post_fold0_retuning_allowed"
] is False


print(
    "========== G3H3 CLOSURE LOCK =========="
)


print(
    "branch:",
    closure["branch"]
)


print(
    "policy:",
    closure["final_policy"]["policy_id"]
)


print(
    "scientific selection closed:",
    "YES"
)


print(
    "Fold0 reusable for tuning:",
    "NO"
)


# ============================================================
# 2. SEARCH PATTERNS
# ============================================================

PATTERNS = [
    r"\broadmap\b",
    r"\bnext\b",
    r"next_if_[A-Za-z0-9_]+",
    r"ADVANCE_TO_[A-Za-z0-9_]+",
    r"CLOSE_[A-Za-z0-9_]+",
    r"Stage\s*12(?:\.\d+)+[A-Za-z0-9\-]*",
    r"12\.11[A-Za-z0-9\-]*",
    r"12\.12[A-Za-z0-9\-]*",
    r"candidate family",
    r"research direction",
    r"next research direction",
    r"predeclared",
]


rx = re.compile(
    "|".join(
        f"(?:{p})"
        for p in PATTERNS
    ),
    flags=re.IGNORECASE,
)


# ============================================================
# 3. LOAD SEARCHABLE UNITS
# ============================================================

units = []


for root in SEARCH_ROOTS:

    if not root.exists():
        continue


    # --------------------------------------------------------
    # Text / Python / JSON / Markdown
    # --------------------------------------------------------

    for suffix in [
        "*.py",
        "*.md",
        "*.txt",
        "*.json",
    ]:

        for path in sorted(
            root.rglob(
                suffix
            )
        ):

            try:
                text = path.read_text(
                    encoding="utf-8"
                )
            except Exception:
                continue


            units.append(
                {
                    "path": path,
                    "kind": suffix,
                    "cell": None,
                    "text": text,
                }
            )


    # --------------------------------------------------------
    # Notebook cells
    # --------------------------------------------------------

    for path in sorted(
        root.rglob(
            "*.ipynb"
        )
    ):

        try:
            nb = json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )
        except Exception:
            continue


        for idx, cell in enumerate(
            nb.get(
                "cells",
                []
            )
        ):

            text = "".join(
                cell.get(
                    "source",
                    []
                )
            )


            if not text:
                continue


            units.append(
                {
                    "path": path,
                    "kind": cell.get(
                        "cell_type",
                        "unknown",
                    ),
                    "cell": idx,
                    "text": text,
                }
            )


print(
    "\n========== SEARCH CORPUS =========="
)


print(
    "units:",
    len(units)
)


# ============================================================
# 4. EXTRACT HIGH-VALUE WINDOWS
# ============================================================

hits = []


HIGH_VALUE_TERMS = {
    "ADVANCE_TO_NEXT_PREDECLARED_STAGE12_ROADMAP_ITEM": 100,

    "next_if_signal_exists": 60,

    "next_if_no_signal": 60,

    "next research direction": 40,

    "roadmap": 30,

    "predeclared": 30,

    "12.12": 25,

    "12.11H": 25,

    "12.11G4": 25,

    "G3H4": 25,

    "next": 5,
}


for unit in units:

    text = unit[
        "text"
    ]


    if not rx.search(
        text
    ):
        continue


    lines = text.splitlines()


    matched_lines = []


    score = 0


    for i, line in enumerate(
        lines
    ):

        line_lower = line.lower()


        local_score = 0


        for term, weight in HIGH_VALUE_TERMS.items():

            if term.lower() in line_lower:

                local_score += weight


        # Also prioritize statements near G3H3 / E3 closure.
        if (
            "g3h3" in line_lower
            or
            "rewire" in line_lower
        ):

            local_score += 8


        if local_score:

            score += local_score

            matched_lines.append(
                i
            )


    if not matched_lines:
        continue


    windows = []


    for i in matched_lines:

        start = max(
            0,
            i - 8,
        )

        end = min(
            len(lines),
            i + 12,
        )


        windows.append(
            (
                start,
                end,
            )
        )


    windows.sort()


    merged = []


    for start, end in windows:

        if (
            not merged
            or
            start > merged[-1][1] + 2
        ):

            merged.append(
                [
                    start,
                    end,
                ]
            )

        else:

            merged[-1][1] = max(
                merged[-1][1],
                end,
            )


    excerpts = []


    for start, end in merged:

        excerpt = "\n".join(
            f"{j+1:4d}: {lines[j]}"
            for j in range(
                start,
                end,
            )
        )


        excerpts.append(
            excerpt
        )


    hits.append(
        {
            "score": score,
            "path": unit["path"],
            "kind": unit["kind"],
            "cell": unit["cell"],
            "excerpts": excerpts,
        }
    )


hits.sort(
    key=lambda x: (
        -x["score"],
        str(x["path"]),
        -1 if x["cell"] is None else x["cell"],
    )
)


# ============================================================
# 5. PRINT TOP ROADMAP SOURCES
# ============================================================

print(
    "\n========== TOP ROADMAP CANDIDATES =========="
)


for rank, hit in enumerate(
    hits[:30],
    start=1,
):

    print()
    print(
        "=" * 80
    )

    print(
        f"[{rank}] score={hit['score']}"
    )

    print(
        "path:",
        hit["path"]
    )

    print(
        "kind:",
        hit["kind"]
    )

    print(
        "cell:",
        hit["cell"]
    )


    for excerpt in hit[
        "excerpts"
    ]:

        print(
            "\n"
            + excerpt
        )


# ============================================================
# 6. SPECIFIC SEARCH FOR DECLARATIONS AFTER G3H3
# ============================================================

POST_G3H3_TERMS = [
    "G3H4",
    "G3H3E4",
    "G3H4A",
    "12.11G4",
    "12.11H",
    "12.12",
    "NEXT_PREDECLARED_STAGE12",
    "ADVANCE_TO_NEXT_PREDECLARED_STAGE12_ROADMAP_ITEM",
]


print(
    "\n========== POST-G3H3 DECLARATION SEARCH =========="
)


declaration_hits = []


for unit in units:

    text = unit[
        "text"
    ]


    found = [
        term
        for term in POST_G3H3_TERMS
        if term.lower() in text.lower()
    ]


    if not found:
        continue


    declaration_hits.append(
        (
            unit,
            found,
        )
    )


for unit, found in declaration_hits:

    print()
    print(
        "path:",
        unit["path"]
    )

    print(
        "cell:",
        unit["cell"]
    )

    print(
        "terms:",
        found
    )


print(
    "\nnumber of declaration sources:",
    len(
        declaration_hits
    )
)


# ============================================================
# 7. SEARCH FOR EXPLICIT next: VALUES
# ============================================================

print(
    "\n========== EXPLICIT NEXT VALUES =========="
)


next_rows = []


next_regexes = [
    re.compile(
        r"""["']?next["']?\s*[:=]\s*["']([^"']+)["']""",
        re.IGNORECASE,
    ),

    re.compile(
        r"""print\(\s*["']([A-Z0-9_\-]*NEXT[A-Z0-9_\-]*)["']\s*\)""",
        re.IGNORECASE,
    ),

    re.compile(
        r"""next\s*:\s*([A-Z0-9_\- ]+)""",
        re.IGNORECASE,
    ),
]


for unit in units:

    text = unit[
        "text"
    ]


    for regex in next_regexes:

        for match in regex.finditer(
            text
        ):

            value = match.group(
                1
            ).strip()


            next_rows.append(
                {
                    "path":
                        str(
                            unit["path"]
                        ),

                    "cell":
                        unit["cell"],

                    "value":
                        value,
                }
            )


for row in next_rows[-100:]:

    print(
        row
    )


# ============================================================
# 8. FINAL
# ============================================================

print(
    "\n========== STAGE12 ROADMAP HANDOFF DECISION =========="
)


print(
    "G3H3 branch closed:",
    "YES"
)


print(
    "old G3H3 policy may be retuned:",
    "NO"
)


print(
    "roadmap candidates found:",
    len(
        hits
    )
)


print(
    "explicit post-G3H3 declaration sources:",
    len(
        declaration_hits
    )
)


if declaration_hits:

    print(
        "NEXT ACTION:"
    )

    print(
        "REVIEW_EXPLICIT_POST_G3H3_DECLARATION_AND_START_IT_UNCHANGED"
    )

else:

    print(
        "NEXT ACTION:"
    )

    print(
        "NO_EXPLICIT_POST_G3H3_STAGE_FOUND_IN_REPO"
    )

    print(
        "DO_NOT_INVENT_G3H4"
    )

    print(
        "RECOVER_MASTER_ROADMAP_OR_DECLARE_A_NEW_INDEPENDENT_BRANCH_BEFORE_GT"
    )


print(
    "GT used:",
    "NO"
)


print(
    "source modified:",
    "NO"
)

In [ ]:
# ============================================================
# Stage 12.12A0
#
# POST-R1P50 RESIDUAL SYNTHETIC-NODE RECOVERY
# — NEW INDEPENDENT BRANCH CONTRACT
#
# ============================================================
#
# WHY THIS BRANCH EXISTS
# ------------------------------------------------------------
#
# G3H3 has been formally closed:
#
#   STAGE12_MULTIFRAME_RESIDUAL_REWIRE
#
# Final promoted policy:
#
#   ASYM1x5_Q9990_NO_TARGET_CUT_V1
#
# Development:
#
#   0.7919774336227487
#       ->
#   0.7920280500935799
#
# Locked Fold0:
#
#   0.7863444776365197
#       ->
#   0.7863985958888573
#
# G3H3 may NOT be reopened.
#
# ------------------------------------------------------------
# ROADMAP RECOVERY RESULT
# ------------------------------------------------------------
#
# No concrete post-G3H3 predeclared stage was found.
#
# Historical Stage12 research identified two major mechanisms:
#
#   1. wrong-edge removal / rewiring
#   2. synthetic missing-node reconstruction
#
# Mechanism (1) has now been formally closed by G3H3.
#
# R1P50 already covers one conservative synthetic-node recovery
# family, but detection-limited residuals remain.
#
# This branch asks whether a DISTINCT, deployment-valid,
# GT-blind residual synthetic-node mechanism exists beyond R1P50.
#
# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------
#
# This is a NEW independent branch declaration.
#
# It is NOT claimed to have been predeclared before G3H3.
#
# This contract MUST be frozen before any new development-GT
# audit for this branch.
#
# ------------------------------------------------------------
# PROTOCOL
# ------------------------------------------------------------
#
#   A0  freeze branch contract                   <-- this cell
#
#   A1  exact current residual census on folds1-4
#
#   A2  GT-blind candidate-universe design
#       + freeze BEFORE oracle matching
#
#   A3  development oracle coverage audit
#
#   A4+ candidate scoring / structural policy
#       under explicitly frozen families
#
#   final development policy
#       ->
#   freeze
#       ->
#   at most one locked Fold0 confirmation
#
# Fold0 MUST NOT be used before a final development policy is
# frozen.
#
# ------------------------------------------------------------
# THIS CELL
# ------------------------------------------------------------
#
# GT:                         NO
# Fold0 GT:                   NO
# scorer training:            NO
# candidate generation:       NO
# graph modification:         NO
# source modification:        NO
# official metric:            NO
# GEFF write:                 NO
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import hashlib
import json


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


G3H3_CLOSURE_PATH = (
    S12
    / "stage12_g3h3_rewire_branch_closed.json"
)


CURRENT_BEST_PATH = (
    S12
    / "stage12_current_best_checkpoint.post_g3h3e3.json"
)


G3H3_CLOSURE_SUMMARY_PATH = (
    S12
    / "stage12_g3h3e3_rewire_branch_closure_summary.json"
)


CANONICAL_BUILDER_PATH = (
    PROJECT
    / "scripts"
    / "run_stage12_current_best_edge_residual_census.py"
)


RUNTIME_PATH = (
    PROJECT
    / "src"
    / "biohub_cell_tracking"
    / "stage12_confirmed_rewire.py"
)


# ------------------------------------------------------------
# New branch contract
# ------------------------------------------------------------

CONTRACT_OUT = (
    S12
    / "stage12_post_r1p50_residual_synthetic_node_recovery_v1_frozen.json"
)


SUMMARY_OUT = (
    S12
    / "stage12_post_r1p50_residual_synthetic_node_recovery_v1_branch_summary.json"
)


for path in [
    G3H3_CLOSURE_PATH,
    CURRENT_BEST_PATH,
    G3H3_CLOSURE_SUMMARY_PATH,
    CANONICAL_BUILDER_PATH,
    RUNTIME_PATH,
]:

    assert path.exists(), path


for path in [
    CONTRACT_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\n12.12A0 formal branch artifact already exists:\n"
            f"{path}\n\n"
            "Do not overwrite branch-declaration provenance."
        )


# ============================================================
# 1. EXPECTED LOCKS
# ============================================================

EXPECTED_G3H3_POLICY_ID = (
    "ASYM1x5_Q9990_NO_TARGET_CUT_V1"
)


EXPECTED_G3H3_POLICY_SHA = (
    "77f0b07cafb21d0705917ff71b750fde4"
    "5f3c7ddc17c13fecaff0ac673ebf776"
)


EXPECTED_G3H3_CLOSURE_SHA = (
    "528c8e83fbde518fe28185ad8ab79aa70"
    "cef157f9fe7d7f956f745c2bac5794b"
)


EXPECTED_CURRENT_BEST_CHECKPOINT_SHA = (
    "65d408c0a8c6089e1d008d0c2c69fb4a"
    "76914c0bc1138486e2845f2e835562a4"
)


EXPECTED_RUNTIME_SHA = (
    "52c183629ecf80d0fa1568c8c3b2fa3a"
    "4ad6f58a1c34a98dc2bf59feca0ffad0"
)


EXPECTED_CANONICAL_SHA = (
    "6f2bcc83715e305d0f01517a929382a8"
    "ac6f9f31a0bf32c29c11ed95c5be2181"
)


EXPECTED_DEV_SCORE = (
    0.7920280500935799
)


EXPECTED_FOLD0_SCORE = (
    0.7863985958888573
)


# ============================================================
# 2. FILE SHA
# ============================================================

def sha256_file(
    path,
):

    h = hashlib.sha256()


    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024
                *
                1024
            )


            if not chunk:
                break


            h.update(
                chunk
            )


    return h.hexdigest()


# ============================================================
# 3. VERIFY G3H3 IS IRREVOCABLY CLOSED
# ============================================================

closure = json.loads(
    G3H3_CLOSURE_PATH.read_text(
        encoding="utf-8"
    )
)


closure_summary = json.loads(
    G3H3_CLOSURE_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


current_best = json.loads(
    CURRENT_BEST_PATH.read_text(
        encoding="utf-8"
    )
)


assert closure[
    "status"
] == (
    "BRANCH_CLOSED_SUCCESSFULLY"
)


assert closure[
    "branch"
] == (
    "STAGE12_MULTIFRAME_RESIDUAL_REWIRE"
)


assert closure[
    "final_policy"
][
    "policy_id"
] == EXPECTED_G3H3_POLICY_ID


assert closure[
    "final_policy"
][
    "policy_sha256"
] == EXPECTED_G3H3_POLICY_SHA


assert closure[
    "branch_closure_sha256"
] == EXPECTED_G3H3_CLOSURE_SHA


assert closure[
    "scientific_selection_closed"
] is True


assert closure[
    "development_reopened"
] is False


assert closure[
    "post_fold0_retuning_allowed"
] is False


assert closure[
    "branch_may_be_reopened_for_policy_search"
] is False


assert closure_summary[
    "status"
] == (
    "STAGE12_REWIRE_INTEGRATION_CLOSED"
)


assert closure_summary[
    "fold0_consumed"
] is True


assert closure_summary[
    "post_fold0_retuning_allowed"
] is False


print(
    "========== 12.12A0 G3H3 BOUNDARY =========="
)


print(
    "G3H3 branch:",
    closure[
        "branch"
    ]
)


print(
    "G3H3 policy:",
    EXPECTED_G3H3_POLICY_ID
)


print(
    "scientific selection closed:",
    "YES"
)


print(
    "Fold0 consumed:",
    "YES"
)


print(
    "G3H3 may be retuned:",
    "NO"
)


# ============================================================
# 4. VERIFY CURRENT_BEST CHECKPOINT
# ============================================================

assert current_best[
    "status"
] == (
    "CANONICAL_CURRENT_BEST_"
    "CONFIRMED_PACKAGED_REPRODUCED"
)


assert current_best[
    "checkpoint_sha256"
] == EXPECTED_CURRENT_BEST_CHECKPOINT_SHA


assert current_best[
    "promoted_policy"
] == EXPECTED_G3H3_POLICY_ID


assert abs(
    float(
        current_best[
            "development"
        ][
            "score"
        ]
    )
    -
    EXPECTED_DEV_SCORE
) <= 1e-12


assert abs(
    float(
        current_best[
            "locked_fold0"
        ][
            "score"
        ]
    )
    -
    EXPECTED_FOLD0_SCORE
) <= 1e-12


assert current_best[
    "protocol"
][
    "scientific_selection_closed"
] is True


assert current_best[
    "protocol"
][
    "fold0_available_for_future_selection"
] is False


# ============================================================
# 5. VERIFY SOURCE LOCK
# ============================================================

runtime_sha = sha256_file(
    RUNTIME_PATH
)


canonical_sha = sha256_file(
    CANONICAL_BUILDER_PATH
)


assert runtime_sha == EXPECTED_RUNTIME_SHA


assert canonical_sha == EXPECTED_CANONICAL_SHA


print(
    "\n========== 12.12A0 CURRENT_BEST LOCK =========="
)


print(
    "development score:",
    EXPECTED_DEV_SCORE
)


print(
    "consumed Fold0 score:",
    EXPECTED_FOLD0_SCORE
)


print(
    "runtime SHA:",
    runtime_sha
)


print(
    "canonical SHA:",
    canonical_sha
)


print(
    "CURRENT_BEST_LOCK: PASS"
)


# ============================================================
# 6. FREEZE NEW INDEPENDENT BRANCH CONTRACT
# ============================================================

BRANCH_ID = (
    "POST_R1P50_RESIDUAL_SYNTHETIC_NODE_RECOVERY_V1"
)


CONTRACT = {
    "stage":
        "12.12A0",

    "branch_id":
        BRANCH_ID,

    "status":
        "NEW_INDEPENDENT_BRANCH_FROZEN_BEFORE_NEW_GT",

    # --------------------------------------------------------
    # Provenance boundary
    # --------------------------------------------------------

    "parent_current_best": {
        "checkpoint":
            CURRENT_BEST_PATH.name,

        "checkpoint_sha256":
            EXPECTED_CURRENT_BEST_CHECKPOINT_SHA,

        "development_score":
            EXPECTED_DEV_SCORE,

        "locked_fold0_score":
            EXPECTED_FOLD0_SCORE,
    },

    "closed_parent_branch": {
        "branch":
            "STAGE12_MULTIFRAME_RESIDUAL_REWIRE",

        "closure_sha256":
            EXPECTED_G3H3_CLOSURE_SHA,

        "policy_id":
            EXPECTED_G3H3_POLICY_ID,

        "policy_sha256":
            EXPECTED_G3H3_POLICY_SHA,

        "may_be_reopened":
            False,
    },

    # --------------------------------------------------------
    # Why this is new
    # --------------------------------------------------------

    "roadmap_handoff": {
        "concrete_post_g3h3_predeclared_stage_found":
            False,

        "new_branch_explicitly_declared":
            True,

        "new_branch_is_not_claimed_to_be_historically_predeclared":
            True,

        "motivation":
            (
                "Investigate residual detection-limited "
                "missing-node recovery beyond already-frozen "
                "R1P50, independently of the closed G3H3 "
                "rewire branch."
            ),
    },

    # --------------------------------------------------------
    # Scientific question
    # --------------------------------------------------------

    "scientific_question":
        (
            "After canonical CURRENT_BEST including R1P50 and "
            "the confirmed G3H3 rewire policy, do remaining "
            "development detection-limited residuals contain "
            "a distinct deployment-valid synthetic-node "
            "recovery mechanism?"
        ),

    # --------------------------------------------------------
    # Scope
    # --------------------------------------------------------

    "scope": {
        "development_corpus":
            "FOLDS_1_TO_4_ONLY",

        "fold0":
            "LOCKED_AND_UNAVAILABLE_FOR_RESEARCH_DESIGN",

        "target_error_class":
            (
                "DETECTION_LIMITED_EDGE_FN_WITH_AT_LEAST_ONE_"
                "UNMATCHED_GT_ENDPOINT"
            ),

        "focus":
            (
                "SYNTHETIC_NODE_RECOVERY_BEYOND_EXISTING_R1P50"
            ),

        "g3h3_rewire_policy":
            "FROZEN_AND_OUT_OF_SCOPE_FOR_RETUNING",
    },

    # --------------------------------------------------------
    # Deployment legality
    # --------------------------------------------------------

    "deployment_constraints": {
        "candidate_generation_must_be_gt_blind":
            True,

        "candidate_scoring_must_be_gt_blind":
            True,

        "runtime_may_use_gt":
            False,

        "runtime_may_use_official_fp_labels":
            False,

        "runtime_may_use_oracle_missing_node_coordinates":
            False,

        "runtime_may_use_fold_identity":
            False,

        "runtime_may_use_dataset_identity":
            False,

        "runtime_may_use_fold0_statistics":
            False,
    },

    # --------------------------------------------------------
    # Research sequencing
    # --------------------------------------------------------

    "protocol": {
        "A0":
            "FREEZE_BRANCH_CONTRACT",

        "A1":
            (
                "REPRODUCE_EXACT_CANONICAL_CURRENT_BEST_"
                "DEVELOPMENT_RESIDUAL_CENSUS"
            ),

        "A2":
            (
                "BUILD_AND_FREEZE_GT_BLIND_SYNTHETIC_NODE_"
                "CANDIDATE_UNIVERSE_BEFORE_ORACLE"
            ),

        "A3":
            (
                "DEVELOPMENT_ORACLE_COVERAGE_AND_FAILURE_"
                "MECHANISM_AUDIT"
            ),

        "A4_plus":
            (
                "ONLY_PREDECLARED_SCORER_STRUCTURAL_POLICY_"
                "FAMILIES_BEFORE_OFFICIAL_GRAPH_METRIC"
            ),

        "fold0_entry":
            (
                "ONLY_AFTER_FINAL_DEVELOPMENT_POLICY_IS_FROZEN"
            ),
    },

    # --------------------------------------------------------
    # Tuning boundaries
    # --------------------------------------------------------

    "selection_rules": {
        "fold0_may_select_features":
            False,

        "fold0_may_select_scorer":
            False,

        "fold0_may_select_threshold":
            False,

        "fold0_may_select_structural_rule":
            False,

        "fold0_posthoc_retuning":
            False,

        "g3h3_parameters_may_be_retuned":
            False,

        "r1p50_existing_policy_may_be_retuned_inside_this_branch":
            False,
    },

    # --------------------------------------------------------
    # Current state
    # --------------------------------------------------------

    "current_state": {
        "new_gt_read_for_this_branch":
            False,

        "candidate_universe_defined":
            False,

        "candidate_universe_frozen":
            False,

        "classifier_trained":
            False,

        "threshold_selected":
            False,

        "graph_modified":
            False,

        "official_metric_evaluated":
            False,

        "fold0_used":
            False,

        "prediction_geff_written":
            False,
    },

    "next":
        (
            "12.12A1_CANONICAL_CURRENT_BEST_"
            "DEVELOPMENT_DETECTION_RESIDUAL_CENSUS"
        ),
}


# ============================================================
# 7. CONTRACT SHA
# ============================================================

CONTRACT_SHA = hashlib.sha256(
    json.dumps(
        CONTRACT,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


CONTRACT[
    "branch_contract_sha256"
] = CONTRACT_SHA


# ============================================================
# 8. WRITE
# ============================================================

CONTRACT_OUT.write_text(
    json.dumps(
        CONTRACT,
        indent=2,
    ),
    encoding="utf-8",
)


SUMMARY = {
    "stage":
        "12.12A0",

    "status":
        "POST_R1P50_RESIDUAL_SYNTHETIC_NODE_BRANCH_DECLARED",

    "branch_id":
        BRANCH_ID,

    "branch_contract_sha256":
        CONTRACT_SHA,

    "parent_current_best_checkpoint_sha256":
        EXPECTED_CURRENT_BEST_CHECKPOINT_SHA,

    "closed_g3h3_branch_sha256":
        EXPECTED_G3H3_CLOSURE_SHA,

    "new_gt_read":
        False,

    "fold0_used":
        False,

    "source_modified":
        False,

    "graph_modified":
        False,

    "official_metric_evaluated":
        False,

    "g3h3_reopened":
        False,

    "next":
        (
            "12.12A1_CANONICAL_CURRENT_BEST_"
            "DEVELOPMENT_DETECTION_RESIDUAL_CENSUS"
        ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 9. RELOAD INTEGRITY
# ============================================================

contract_check = json.loads(
    CONTRACT_OUT.read_text(
        encoding="utf-8"
    )
)


summary_check = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert contract_check[
    "branch_contract_sha256"
] == CONTRACT_SHA


assert contract_check[
    "status"
] == (
    "NEW_INDEPENDENT_BRANCH_FROZEN_BEFORE_NEW_GT"
)


assert contract_check[
    "closed_parent_branch"
][
    "may_be_reopened"
] is False


assert contract_check[
    "deployment_constraints"
][
    "candidate_generation_must_be_gt_blind"
] is True


assert contract_check[
    "current_state"
][
    "new_gt_read_for_this_branch"
] is False


assert contract_check[
    "current_state"
][
    "fold0_used"
] is False


assert summary_check[
    "g3h3_reopened"
] is False


assert summary_check[
    "new_gt_read"
] is False


# ============================================================
# 10. FINAL
# ============================================================

print(
    "\n========== 12.12A0 BRANCH DECLARATION =========="
)


print(
    "NEW_INDEPENDENT_BRANCH_FREEZE: PASS"
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "contract SHA:",
    CONTRACT_SHA
)


print(
    "parent CURRENT_BEST:",
    EXPECTED_CURRENT_BEST_CHECKPOINT_SHA
)


print(
    "G3H3 closed:",
    "YES"
)


print(
    "G3H3 reopened:",
    "NO"
)


print(
    "new GT read:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "candidate universe selected:",
    "NO"
)


print(
    "scorer selected:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "graph modified:",
    "NO"
)


print(
    "official metric evaluated:",
    "NO"
)


print(
    "next:"
)


print(
    "12.12A1_CANONICAL_CURRENT_BEST_"
    "DEVELOPMENT_DETECTION_RESIDUAL_CENSUS"
)

In [ ]:
# ============================================================
# Stage 12.12A1-R1
#
# REPAIR / RESUME
#
# POST-G3H3 DEVELOPMENT DETECTION-RESIDUAL CENSUS
#
# ============================================================
#
# REPAIR REASON
# ------------------------------------------------------------
#
# Original A1 assumed:
#
#   final_policy["runtime_contract"]["structural_rule"]
#
# but the frozen policy runtime_contract does not expose that
# exact direct key.
#
# The structural rule is formally frozen elsewhere:
#
#   G3H3D1 structural-ablation contract
#       NO_TARGET_CUT = TARGET_OCCUPIED_EQ_0
#
#   G3H3E0 one-shot confirmation contract
#       structural_rule = TARGET_OCCUPIED_EQ_0
#
#       mutation_rule =
#         REMOVE_SOURCE_INCUMBENT_IF_PRESENT;
#         NEVER_REMOVE_TARGET_INCUMBENT;
#         ADD_REWIRE_EDGE
#
# This repair DOES NOT weaken the gate.
#
# It verifies:
#
#   1. D1 frozen structural contract
#   2. E0 frozen confirmation contract
#   3. packaged runtime source AST
#      contains edge mutation only and no node mutation
#   4. E2C exact canonical reproduction
#
# Only then is node-matching invariance accepted.
#
# ------------------------------------------------------------
# NO
# ------------------------------------------------------------
#
# new GT read
# Fold0 reuse
# evaluator
# candidate generation
# scorer training
# threshold selection
# graph modification
# source modification
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import ast
import hashlib
import json


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


# ------------------------------------------------------------
# New branch A0
# ------------------------------------------------------------

A0_CONTRACT_PATH = (
    S12
    / "stage12_post_r1p50_residual_synthetic_node_recovery_v1_frozen.json"
)


A0_SUMMARY_PATH = (
    S12
    / "stage12_post_r1p50_residual_synthetic_node_recovery_v1_branch_summary.json"
)


# ------------------------------------------------------------
# G3H3 closure / CURRENT_BEST
# ------------------------------------------------------------

G3H3_CLOSURE_PATH = (
    S12
    / "stage12_g3h3_rewire_branch_closed.json"
)


CURRENT_BEST_PATH = (
    S12
    / "stage12_current_best_checkpoint.post_g3h3e3.json"
)


FINAL_POLICY_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_frozen.json"
)


FREEZE_SUMMARY_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_freeze_summary.json"
)


# ------------------------------------------------------------
# Formal structural provenance
# ------------------------------------------------------------

STRUCTURAL_CONTRACT_PATH = (
    S12
    / "stage12_folds14_rewire_q9990_structural_ablation_v1_frozen.json"
)


FOLD0_CONFIRMATION_CONTRACT_PATH = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_confirmation_contract_frozen.json"
)


E2C_REPORT_PATH = (
    S12
    / "stage12_g3h3e2c_fresh_process_canonical_reproduction.json"
)


RUNTIME_PATH = (
    PROJECT
    / "src"
    / "biohub_cell_tracking"
    / "stage12_confirmed_rewire.py"
)


# ------------------------------------------------------------
# Pre-G3H3 exact development CURRENT_BEST reference
# ------------------------------------------------------------

PRE_G3H3_DEV_PATH = (
    S12
    / "stage12_folds14_rewire_q9990_graph_candidate_v1_summary.json"
)


# ------------------------------------------------------------
# A1 outputs
# ------------------------------------------------------------

INVARIANCE_OUT = (
    S12
    / "stage12_post_g3h3_detection_residual_invariance_v1_frozen.json"
)


CENSUS_OUT = (
    S12
    / "stage12_post_g3h3_development_detection_residual_census_v1.json"
)


SUMMARY_OUT = (
    S12
    / "stage12_post_g3h3_development_detection_residual_census_v1_summary.json"
)


for path in [
    A0_CONTRACT_PATH,
    A0_SUMMARY_PATH,

    G3H3_CLOSURE_PATH,
    CURRENT_BEST_PATH,

    FINAL_POLICY_PATH,
    FREEZE_SUMMARY_PATH,

    STRUCTURAL_CONTRACT_PATH,
    FOLD0_CONFIRMATION_CONTRACT_PATH,
    E2C_REPORT_PATH,

    RUNTIME_PATH,

    PRE_G3H3_DEV_PATH,
]:

    assert path.exists(), path


# Original failed A1 stopped before formal writes.
# If any formal output exists, preserve it and stop.
for path in [
    INVARIANCE_OUT,
    CENSUS_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nA1 formal output already exists:\n"
            f"{path}\n\n"
            "Do not overwrite. Inspect partial provenance first."
        )


# ============================================================
# 1. LOCKS
# ============================================================

BRANCH_ID = (
    "POST_R1P50_RESIDUAL_SYNTHETIC_NODE_RECOVERY_V1"
)


A0_SHA = (
    "03243381d2c6c33218fda9c94be0462f"
    "ad28733f8360fd37b1ea5e3d622ae760"
)


POLICY_ID = (
    "ASYM1x5_Q9990_NO_TARGET_CUT_V1"
)


POLICY_SHA = (
    "77f0b07cafb21d0705917ff71b750fde4"
    "5f3c7ddc17c13fecaff0ac673ebf776"
)


CURRENT_BEST_SHA = (
    "65d408c0a8c6089e1d008d0c2c69fb4a"
    "76914c0bc1138486e2845f2e835562a4"
)


RUNTIME_SHA = (
    "52c183629ecf80d0fa1568c8c3b2fa3a"
    "4ad6f58a1c34a98dc2bf59feca0ffad0"
)


# ------------------------------------------------------------
# Development metrics
# ------------------------------------------------------------

PRE_TP = 77_535
PRE_FP = 9_398
PRE_FN = 10_254

PRE_SCORE = (
    0.7919774336227487
)


DELTA_TP = -3
DELTA_FP = -10
DELTA_FN = +3


POST_TP = 77_532
POST_FP = 9_388
POST_FN = 10_257


POST_SCORE = (
    0.7920280500935799
)


# ------------------------------------------------------------
# Existing detection-limited census
# ------------------------------------------------------------

DETECTION_FN = 4_097

LINKING_FN_PRE = 6_157

LINKING_FN_POST = (
    POST_FN
    -
    DETECTION_FN
)


BOTH_MISSING = 1_390

SOURCE_MISSING = 1_365

TARGET_MISSING = 1_342


RAW_COMPONENTS = 1_595

MISSING_NODES = 2_985

MULTI_NODE_COMPONENTS = 591

MULTI_NODE_MISSING_NODES = 1_981


# ============================================================
# 2. SHA
# ============================================================

def sha256_file(
    path,
):

    h = hashlib.sha256()


    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )


            if not chunk:
                break


            h.update(
                chunk
            )


    return h.hexdigest()


# ============================================================
# 3. VERIFY A0
# ============================================================

a0 = json.loads(
    A0_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


a0_summary = json.loads(
    A0_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert a0[
    "branch_id"
] == BRANCH_ID


assert a0[
    "branch_contract_sha256"
] == A0_SHA


assert a0[
    "status"
] == (
    "NEW_INDEPENDENT_BRANCH_FROZEN_BEFORE_NEW_GT"
)


assert a0[
    "current_state"
][
    "new_gt_read_for_this_branch"
] is False


assert a0[
    "current_state"
][
    "fold0_used"
] is False


assert a0[
    "selection_rules"
][
    "g3h3_parameters_may_be_retuned"
] is False


assert a0_summary[
    "new_gt_read"
] is False


print(
    "========== 12.12A1-R1 BRANCH LOCK =========="
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "A0 SHA:",
    A0_SHA
)


print(
    "new GT previously read:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


# ============================================================
# 4. VERIFY G3H3 CLOSED
# ============================================================

closure = json.loads(
    G3H3_CLOSURE_PATH.read_text(
        encoding="utf-8"
    )
)


current_best = json.loads(
    CURRENT_BEST_PATH.read_text(
        encoding="utf-8"
    )
)


final_policy = json.loads(
    FINAL_POLICY_PATH.read_text(
        encoding="utf-8"
    )
)


freeze_summary = json.loads(
    FREEZE_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert closure[
    "status"
] == (
    "BRANCH_CLOSED_SUCCESSFULLY"
)


assert closure[
    "branch_may_be_reopened_for_policy_search"
] is False


assert closure[
    "post_fold0_retuning_allowed"
] is False


assert current_best[
    "checkpoint_sha256"
] == CURRENT_BEST_SHA


assert current_best[
    "promoted_policy"
] == POLICY_ID


assert final_policy[
    "policy_id"
] == POLICY_ID


assert final_policy[
    "final_policy_sha256"
] == POLICY_SHA


assert final_policy[
    "development_closed_after_freeze"
] is True


assert freeze_summary[
    "development_closed"
] is True


assert freeze_summary[
    "no_further_development_tuning"
] is True


print(
    "\n========== G3H3 CLOSED BOUNDARY =========="
)


print(
    "policy:",
    POLICY_ID
)


print(
    "development closed:",
    "YES"
)


print(
    "G3H3 retuning:",
    "NO"
)


print(
    "Fold0 reusable:",
    "NO"
)


# ============================================================
# 5. VERIFY PRE-G3H3 DEVELOPMENT BASELINE
# ============================================================

pre = json.loads(
    PRE_G3H3_DEV_PATH.read_text(
        encoding="utf-8"
    )
)


assert pre[
    "baseline_reproduction_pass"
] is True


assert int(
    pre[
        "baseline"
    ][
        "edge_tp"
    ]
) == PRE_TP


assert int(
    pre[
        "baseline"
    ][
        "edge_fp"
    ]
) == PRE_FP


assert int(
    pre[
        "baseline"
    ][
        "edge_fn"
    ]
) == PRE_FN


assert abs(
    float(
        pre[
            "baseline"
        ][
            "score"
        ]
    )
    -
    PRE_SCORE
) <= 1e-12


assert pre[
    "fold0_used"
] is False


print(
    "\n========== PRE-G3H3 DEVELOPMENT BASELINE =========="
)


print(
    "TP/FP/FN:",
    PRE_TP,
    "/",
    PRE_FP,
    "/",
    PRE_FN
)


print(
    "score:",
    f"{PRE_SCORE:.15f}"
)


print(
    "baseline reproduction:",
    "PASS"
)


# ============================================================
# 6. VERIFY POST-G3H3 DEVELOPMENT RESULT
# ============================================================

assert abs(
    float(
        current_best[
            "development"
        ][
            "score"
        ]
    )
    -
    POST_SCORE
) <= 1e-12


assert abs(
    float(
        freeze_summary[
            "development_score"
        ]
    )
    -
    POST_SCORE
) <= 1e-12


assert abs(
    float(
        freeze_summary[
            "development_delta_score"
        ]
    )
    -
    (
        POST_SCORE
        -
        PRE_SCORE
    )
) <= 1e-12


assert (
    PRE_TP
    +
    DELTA_TP
) == POST_TP


assert (
    PRE_FP
    +
    DELTA_FP
) == POST_FP


assert (
    PRE_FN
    +
    DELTA_FN
) == POST_FN


# ============================================================
# 7. REPAIR:
#    VERIFY STRUCTURAL RULE FROM THE ACTUAL FORMAL SOURCES
# ============================================================

structural = json.loads(
    STRUCTURAL_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


confirmation = json.loads(
    FOLD0_CONFIRMATION_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


# ------------------------------------------------------------
# D1 predeclared structural family
# ------------------------------------------------------------

assert structural[
    "contract_id"
] == (
    "Q9990_STRUCTURAL_ACTION_ABLATION_V1"
)


assert structural[
    "family"
][
    "NO_TARGET_CUT"
] == (
    "TARGET_OCCUPIED_EQ_0"
)


assert (
    "NO_TARGET_CUT"
    in
    structural[
        "eligible_policies"
    ]
)


# ------------------------------------------------------------
# E0 final frozen one-shot contract
# ------------------------------------------------------------

assert confirmation[
    "final_policy_id"
] == POLICY_ID


assert confirmation[
    "final_policy_sha256"
] == POLICY_SHA


assert confirmation[
    "structural_rule"
] == (
    "TARGET_OCCUPIED_EQ_0"
)


EXPECTED_MUTATION_RULE = (
    "REMOVE_SOURCE_INCUMBENT_IF_PRESENT;"
    "NEVER_REMOVE_TARGET_INCUMBENT;"
    "ADD_REWIRE_EDGE"
)


assert confirmation[
    "mutation_rule"
] == EXPECTED_MUTATION_RULE


assert confirmation[
    "runtime_pipeline"
][
    -1
] == (
    "TARGET_OCCUPIED_EQ_0"
)


assert confirmation[
    "action_artifact_must_be_frozen_before_gt"
] is True


assert confirmation[
    "post_fold0_retuning_allowed"
] is False


print(
    "\n========== REPAIRED STRUCTURAL PROVENANCE =========="
)


print(
    "D1 NO_TARGET_CUT:",
    structural[
        "family"
    ][
        "NO_TARGET_CUT"
    ]
)


print(
    "E0 structural rule:",
    confirmation[
        "structural_rule"
    ]
)


print(
    "E0 mutation rule:",
    confirmation[
        "mutation_rule"
    ]
)


print(
    "STRUCTURAL_PROVENANCE: PASS"
)


# ============================================================
# 8. VERIFY PACKAGED RUNTIME SHA
# ============================================================

assert sha256_file(
    RUNTIME_PATH
) == RUNTIME_SHA


# ============================================================
# 9. STATIC AST AUDIT:
#    CONFIRMED POLICY MUST NOT MUTATE NODES
# ============================================================

runtime_source = RUNTIME_PATH.read_text(
    encoding="utf-8"
)


runtime_tree = ast.parse(
    runtime_source
)


target_function_names = {
    "apply_confirmed_rewire_policy",
    "_runtime_graph_state",
}


function_nodes = {
    node.name:
        node

    for node in runtime_tree.body

    if (
        isinstance(
            node,
            ast.FunctionDef,
        )
        and
        node.name in target_function_names
    )
}


assert target_function_names.issubset(
    function_nodes
)


graph_method_calls = set()


for fn_name in target_function_names:

    fn_node = function_nodes[
        fn_name
    ]


    for node in ast.walk(
        fn_node
    ):

        if not isinstance(
            node,
            ast.Call,
        ):
            continue


        func = node.func


        if not isinstance(
            func,
            ast.Attribute,
        ):
            continue


        if (
            isinstance(
                func.value,
                ast.Name,
            )
            and
            func.value.id == "graph"
        ):

            graph_method_calls.add(
                func.attr
            )


PROHIBITED_NODE_MUTATION_METHODS = {
    "add_node",
    "add_nodes",

    "bulk_add_nodes",

    "remove_node",
    "remove_nodes",

    "bulk_remove_nodes",

    "set_node_attr",
    "set_node_attrs",

    "update_node_attr",
    "update_node_attrs",

    "replace_node",
    "move_node",
}


node_mutation_calls = sorted(
    graph_method_calls
    &
    PROHIBITED_NODE_MUTATION_METHODS
)


assert not node_mutation_calls, (
    node_mutation_calls
)


# Expected edge mutation must actually be present.
assert "add_edge" in graph_method_calls


assert (
    "bulk_remove_edges"
    in graph_method_calls
)


print(
    "\n========== PACKAGED RUNTIME AST AUDIT =========="
)


print(
    "runtime SHA:",
    RUNTIME_SHA
)


print(
    "graph methods:",
    sorted(
        graph_method_calls
    )
)


print(
    "node mutation methods:",
    node_mutation_calls
)


print(
    "edge add present:",
    "YES"
)


print(
    "edge remove present:",
    "YES"
)


print(
    "NODE_MUTATION_ABSENT: PASS"
)


# ============================================================
# 10. VERIFY E2C IMPLEMENTATION FIDELITY
# ============================================================

e2c = json.loads(
    E2C_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)


assert e2c[
    "status"
] == (
    "FRESH_PROCESS_CANONICAL_REPRODUCTION_PASS"
)


assert e2c[
    "purpose"
] == (
    "IMPLEMENTATION_FIDELITY_ONLY"
)


assert e2c[
    "matches_consumed_e0_exactly"
] is True


assert e2c[
    "topology_fidelity"
][
    "node_sets_exact"
] is True


assert e2c[
    "topology_fidelity"
][
    "all_action_edge_pairs_exact"
] is True


assert e2c[
    "topology_fidelity"
][
    "exact_added_edges"
] == 2712


assert e2c[
    "topology_fidelity"
][
    "exact_removed_edges"
] == 82


assert e2c[
    "post_fold0_retuning_allowed"
] is False


print(
    "\n========== E2C FIDELITY SUPPORT =========="
)


print(
    "fresh process exact:",
    "YES"
)


print(
    "node sets exact:",
    "YES"
)


print(
    "2712 action edges exact:",
    "YES"
)


print(
    "82 removals exact:",
    "YES"
)


# ============================================================
# 11. NODE-MATCHING INVARIANCE CONCLUSION
# ============================================================

# Formal provenance now jointly establishes:
#
#   - final mutation is edge-only
#   - no node mutation method exists in packaged runtime
#   - canonical fresh-process node sets are identical
#
# Therefore predicted node IDs and coordinates entering the
# matcher are unchanged by the G3H3 augmentation itself.
#
# Hence previously measured detection-limited missing-node
# identity remains invariant.


assert (
    DETECTION_FN
    +
    LINKING_FN_PRE
    ==
    PRE_FN
)


assert (
    BOTH_MISSING
    +
    SOURCE_MISSING
    +
    TARGET_MISSING
    ==
    DETECTION_FN
)


LINKING_FN_POST = (
    POST_FN
    -
    DETECTION_FN
)


assert LINKING_FN_POST == 6_160


print(
    "\n========== NODE-MATCHING INVARIANCE =========="
)


print(
    "G3H3 mutation type:",
    "EDGE_ONLY_REWIRE"
)


print(
    "predicted-node mutation:",
    "NO"
)


print(
    "detection-limited identities invariant:",
    "YES"
)


print(
    "old detection-limited FN:",
    DETECTION_FN
)


print(
    "new detection-limited FN:",
    DETECTION_FN
)


# ============================================================
# 12. POST-G3H3 DEVELOPMENT EDGE BUDGET
# ============================================================

print(
    "\n========== POST-G3H3 DEVELOPMENT EDGE BUDGET =========="
)


print(
    "pre TP/FP/FN :",
    PRE_TP,
    "/",
    PRE_FP,
    "/",
    PRE_FN
)


print(
    "G3H3 delta   :",
    f"{DELTA_TP:+d}",
    "/",
    f"{DELTA_FP:+d}",
    "/",
    f"{DELTA_FN:+d}",
)


print(
    "post TP/FP/FN:",
    POST_TP,
    "/",
    POST_FP,
    "/",
    POST_FN
)


print(
    "development score:",
    f"{POST_SCORE:.15f}"
)


# ============================================================
# 13. POST-G3H3 RESIDUAL ROOT CAUSE
# ============================================================

print(
    "\n========== POST-G3H3 RESIDUAL ROOT CAUSE =========="
)


print(
    "total edge FN      :",
    POST_FN
)


print(
    "detection-limited  :",
    DETECTION_FN,
    f"({DETECTION_FN / POST_FN:.3%})",
)


print(
    "linking-limited    :",
    LINKING_FN_POST,
    f"({LINKING_FN_POST / POST_FN:.3%})",
)


print(
    "\nDetection endpoint class:"
)


print(
    "BOTH_MISSING       :",
    BOTH_MISSING
)


print(
    "SOURCE_MISSING     :",
    SOURCE_MISSING
)


print(
    "TARGET_MISSING     :",
    TARGET_MISSING
)


# ============================================================
# 14. MISSING-NODE COMPONENT TOPOLOGY
# ============================================================

SINGLE_NODE_COMPONENTS = (
    RAW_COMPONENTS
    -
    MULTI_NODE_COMPONENTS
)


SINGLE_NODE_MISSING_NODES = (
    MISSING_NODES
    -
    MULTI_NODE_MISSING_NODES
)


assert (
    SINGLE_NODE_COMPONENTS
    ==
    SINGLE_NODE_MISSING_NODES
)


print(
    "\n========== DETECTION GAP TOPOLOGY =========="
)


print(
    "raw components:",
    RAW_COMPONENTS
)


print(
    "missing GT nodes:",
    MISSING_NODES
)


print(
    "single-node components:",
    SINGLE_NODE_COMPONENTS
)


print(
    "multi-node components:",
    MULTI_NODE_COMPONENTS
)


print(
    "missing nodes in multi-node components:",
    MULTI_NODE_MISSING_NODES
)


# ============================================================
# 15. FREEZE REPAIRED INVARIANCE CONTRACT
# ============================================================

INVARIANCE = {
    "stage":
        "12.12A1-R1",

    "contract_id":
        "POST_G3H3_DETECTION_RESIDUAL_INVARIANCE_V1",

    "branch_id":
        BRANCH_ID,

    "parent_branch_contract_sha256":
        A0_SHA,

    "repair_of":
        "12.12A1",

    "repair_reason":
        (
            "Original A1 assumed a nonexistent direct "
            "runtime_contract.structural_rule key. "
            "Structural semantics are instead verified from "
            "the frozen G3H3D1 and G3H3E0 contracts plus "
            "packaged-runtime AST."
        ),

    "g3h3_policy_id":
        POLICY_ID,

    "g3h3_policy_sha256":
        POLICY_SHA,

    "structural_provenance": {
        "d1_no_target_cut":
            structural[
                "family"
            ][
                "NO_TARGET_CUT"
            ],

        "e0_structural_rule":
            confirmation[
                "structural_rule"
            ],

        "e0_mutation_rule":
            confirmation[
                "mutation_rule"
            ],

        "runtime_sha256":
            RUNTIME_SHA,

        "runtime_node_mutation_calls":
            node_mutation_calls,

        "e2c_node_sets_exact":
            True,
    },

    "invariants": {
        "g3h3_is_edge_only":
            True,

        "predicted_node_set_unchanged":
            True,

        "predicted_node_geometry_unchanged_by_policy":
            True,

        "official_node_matching_unchanged":
            True,

        "detection_limited_edge_fn_identity_unchanged":
            True,

        "missing_gt_node_identity_unchanged":
            True,

        "missing_node_component_identity_unchanged":
            True,
    },

    "prohibited_inference": {
        "all_edge_fn_identity_unchanged":
            False,

        "linking_limited_edge_identity_unchanged":
            False,
    },

    "gt": {
        "new_gt_read":
            False,

        "reuses_preexisting_development_gt_derived_census":
            True,

        "fold0_used":
            False,
    },
}


INVARIANCE_SHA = hashlib.sha256(
    json.dumps(
        INVARIANCE,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


INVARIANCE[
    "contract_sha256"
] = INVARIANCE_SHA


INVARIANCE_OUT.write_text(
    json.dumps(
        INVARIANCE,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 16. FORMAL A1 CENSUS
# ============================================================

CENSUS = {
    "stage":
        "12.12A1-R1",

    "status":
        (
            "POST_G3H3_DEVELOPMENT_"
            "DETECTION_RESIDUAL_CENSUS_LOCKED"
        ),

    "branch_id":
        BRANCH_ID,

    "parent_branch_contract_sha256":
        A0_SHA,

    "invariance_contract_sha256":
        INVARIANCE_SHA,

    "development_current_best": {
        "score":
            POST_SCORE,

        "edge_tp":
            POST_TP,

        "edge_fp":
            POST_FP,

        "edge_fn":
            POST_FN,
    },

    "change_from_pre_g3h3": {
        "edge_tp":
            DELTA_TP,

        "edge_fp":
            DELTA_FP,

        "edge_fn":
            DELTA_FN,

        "score":
            (
                POST_SCORE
                -
                PRE_SCORE
            ),
    },

    "residual_root_cause": {
        "detection_limited_fn":
            DETECTION_FN,

        "linking_limited_fn":
            LINKING_FN_POST,

        "detection_limited_fraction":
            float(
                DETECTION_FN
                /
                POST_FN
            ),

        "linking_limited_fraction":
            float(
                LINKING_FN_POST
                /
                POST_FN
            ),
    },

    "detection_endpoint_class": {
        "BOTH_MISSING":
            BOTH_MISSING,

        "SOURCE_MISSING":
            SOURCE_MISSING,

        "TARGET_MISSING":
            TARGET_MISSING,
    },

    "detection_gap_topology": {
        "raw_components":
            RAW_COMPONENTS,

        "unique_missing_nodes":
            MISSING_NODES,

        "single_node_components":
            SINGLE_NODE_COMPONENTS,

        "multi_node_components":
            MULTI_NODE_COMPONENTS,

        "missing_nodes_in_multi_node_components":
            MULTI_NODE_MISSING_NODES,
    },

    "protocol": {
        "new_gt_read":
            False,

        "official_metric_evaluated":
            False,

        "node_matching_regenerated":
            False,

        "candidate_universe_generated":
            False,

        "scorer_selected":
            False,

        "threshold_selected":
            False,

        "graph_modified":
            False,

        "fold0_used":
            False,

        "g3h3_reopened":
            False,
    },

    "next":
        (
            "12.12A2_FREEZE_GT_BLIND_POST_R1P50_"
            "SYNTHETIC_NODE_CANDIDATE_UNIVERSE"
        ),
}


CENSUS_SHA = hashlib.sha256(
    json.dumps(
        CENSUS,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


CENSUS[
    "census_sha256"
] = CENSUS_SHA


CENSUS_OUT.write_text(
    json.dumps(
        CENSUS,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 17. SUMMARY
# ============================================================

SUMMARY = {
    "stage":
        "12.12A1-R1",

    "status":
        "CANONICAL_DEVELOPMENT_DETECTION_RESIDUAL_BASELINE_READY",

    "branch_id":
        BRANCH_ID,

    "repair_of":
        "12.12A1",

    "census_sha256":
        CENSUS_SHA,

    "invariance_contract_sha256":
        INVARIANCE_SHA,

    "development_score":
        POST_SCORE,

    "edge_tp":
        POST_TP,

    "edge_fp":
        POST_FP,

    "edge_fn":
        POST_FN,

    "detection_limited_fn":
        DETECTION_FN,

    "linking_limited_fn":
        LINKING_FN_POST,

    "missing_nodes":
        MISSING_NODES,

    "components":
        RAW_COMPONENTS,

    "new_gt_read":
        False,

    "fold0_used":
        False,

    "candidate_universe_generated":
        False,

    "g3h3_reopened":
        False,

    "next":
        (
            "12.12A2_FREEZE_GT_BLIND_POST_R1P50_"
            "SYNTHETIC_NODE_CANDIDATE_UNIVERSE"
        ),
}


SUMMARY_OUT.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 18. RELOAD INTEGRITY
# ============================================================

inv_check = json.loads(
    INVARIANCE_OUT.read_text(
        encoding="utf-8"
    )
)


census_check = json.loads(
    CENSUS_OUT.read_text(
        encoding="utf-8"
    )
)


summary_check = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert inv_check[
    "contract_sha256"
] == INVARIANCE_SHA


assert inv_check[
    "invariants"
][
    "g3h3_is_edge_only"
] is True


assert inv_check[
    "invariants"
][
    "detection_limited_edge_fn_identity_unchanged"
] is True


assert inv_check[
    "prohibited_inference"
][
    "all_edge_fn_identity_unchanged"
] is False


assert census_check[
    "census_sha256"
] == CENSUS_SHA


assert census_check[
    "development_current_best"
][
    "edge_fn"
] == 10_257


assert census_check[
    "residual_root_cause"
][
    "detection_limited_fn"
] == 4_097


assert census_check[
    "residual_root_cause"
][
    "linking_limited_fn"
] == 6_160


assert census_check[
    "protocol"
][
    "new_gt_read"
] is False


assert census_check[
    "protocol"
][
    "fold0_used"
] is False


assert census_check[
    "protocol"
][
    "candidate_universe_generated"
] is False


assert summary_check[
    "status"
] == (
    "CANONICAL_DEVELOPMENT_DETECTION_RESIDUAL_BASELINE_READY"
)


# ============================================================
# 19. FINAL
# ============================================================

print(
    "\n========== 12.12A1-R1 FINAL CENSUS =========="
)


print(
    "A1_SCHEMA_REPAIR: PASS"
)


print(
    "POST_G3H3_DEVELOPMENT_DETECTION_RESIDUAL_CENSUS: PASS"
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "development CURRENT_BEST score:",
    f"{POST_SCORE:.15f}"
)


print(
    "TP/FP/FN:",
    POST_TP,
    "/",
    POST_FP,
    "/",
    POST_FN
)


print(
    "detection-limited FN:",
    DETECTION_FN,
    f"({DETECTION_FN / POST_FN:.3%})",
)


print(
    "linking-limited FN:",
    LINKING_FN_POST,
    f"({LINKING_FN_POST / POST_FN:.3%})",
)


print(
    "BOTH_MISSING:",
    BOTH_MISSING
)


print(
    "SOURCE_MISSING:",
    SOURCE_MISSING
)


print(
    "TARGET_MISSING:",
    TARGET_MISSING
)


print(
    "missing GT nodes:",
    MISSING_NODES
)


print(
    "raw missing-node components:",
    RAW_COMPONENTS
)


print(
    "multi-node components:",
    MULTI_NODE_COMPONENTS
)


print(
    "new GT read:",
    "NO"
)


print(
    "official metric evaluated:",
    "NO"
)


print(
    "candidate universe generated:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "G3H3 reopened:",
    "NO"
)


print(
    "invariance SHA:",
    INVARIANCE_SHA
)


print(
    "census SHA:",
    CENSUS_SHA
)


print(
    "next:"
)


print(
    "12.12A2_FREEZE_GT_BLIND_POST_R1P50_"
    "SYNTHETIC_NODE_CANDIDATE_UNIVERSE"
)

In [ ]:
# ============================================================
# Stage 12.12A2
#
# FREEZE GT-BLIND POST-R1P50
# NODE-LEVEL SUPPRESSED CANDIDATE UNIVERSE
#
# ============================================================
#
# SCIENTIFIC QUESTION
# ------------------------------------------------------------
#
# Before asking:
#
#   "Can a suppressed candidate form a good bridge?"
#
# ask the more fundamental node-level question:
#
#   "Is this suppressed detector peak itself a plausible
#    missing predicted cell?"
#
# ------------------------------------------------------------
# DISTINCTNESS FROM R1P50
# ------------------------------------------------------------
#
# Frozen R1P50:
#
#   suppressed peak
#       +
#   free source
#       +
#   free target
#       +
#   B12 geometry
#       +
#   incoming rank == 1
#       +
#   outgoing rank == 1
#       +
#   bridge_min_prob >= 0.50
#
# New A2 universe:
#
#   suppressed detector peak ONLY
#
# before:
#
#   source/target bridge acceptance
#   association rank gates
#   bridge probability gates
#
# ------------------------------------------------------------
# EXACT CANDIDATE DEFINITION
# ------------------------------------------------------------
#
# Start from frozen Stage12.10C GT-blind universe:
#
#   production local maximum
#   sigmoid(det_logit) > 0.90
#   exact coordinate absent from persisted P995 detections
#
# Then remove exact coordinates already inserted by the
# frozen DET-BRIDGE-R1P50 policy.
#
# No additional probability threshold is introduced.
#
# ------------------------------------------------------------
# PROTOCOL
# ------------------------------------------------------------
#
# BEFORE this stage:
#   no new GT for this branch
#
# DURING this stage:
#   GT read                     NO
#   oracle missing-node coords  NO
#   official evaluator          NO
#   Fold0                       NO
#   graph mutation              NO
#   scorer                      NO
#   threshold tuning            NO
#
# AFTER successful completion:
#
#   candidate universe is FROZEN
#
# A3 may then evaluate development oracle coverage.
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


# ------------------------------------------------------------
# Parent branch
# ------------------------------------------------------------

A0_PATH = (
    S12
    / "stage12_post_r1p50_residual_synthetic_node_recovery_v1_frozen.json"
)


A1_PATH = (
    S12
    / "stage12_post_g3h3_development_detection_residual_census_v1.json"
)


A1_INV_PATH = (
    S12
    / "stage12_post_g3h3_detection_residual_invariance_v1_frozen.json"
)


CURRENT_BEST_PATH = (
    S12
    / "stage12_current_best_checkpoint.post_g3h3e3.json"
)


# ------------------------------------------------------------
# Exact historical GT-blind source universe
# ------------------------------------------------------------

SUPPRESSED_PATH = (
    S12
    / "stage12_folds14_suppressed_peak_universe.pkl"
)


SUPPRESSED_SUMMARY_PATH = (
    S12
    / "stage12_folds14_suppressed_peak_universe_summary.json"
)


SUPPRESSED_DATASET_SUMMARY_PATH = (
    S12
    / "stage12_folds14_suppressed_peak_dataset_summary.pkl"
)


# ------------------------------------------------------------
# Frozen R1 policy actions
# ------------------------------------------------------------

R1_ACTIONS_PATH = (
    S12
    / "stage12_detection_bridge_policy_actions.pkl"
)


R1_GRID_SUMMARY_PATH = (
    S12
    / "stage12_detection_bridge_policy_grid_summary.json"
)


# ------------------------------------------------------------
# Formal A2 outputs
# ------------------------------------------------------------

CONTRACT_OUT = (
    S12
    / "stage12_post_r1p50_node_candidate_universe_v1_contract_frozen.json"
)


UNIVERSE_OUT = (
    S12
    / "stage12_post_r1p50_node_candidate_universe_v1.pkl"
)


DATASET_SUMMARY_OUT = (
    S12
    / "stage12_post_r1p50_node_candidate_universe_v1_dataset_summary.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_post_r1p50_node_candidate_universe_v1_summary.json"
)


for path in [
    A0_PATH,
    A1_PATH,
    A1_INV_PATH,
    CURRENT_BEST_PATH,

    SUPPRESSED_PATH,
    SUPPRESSED_SUMMARY_PATH,
    SUPPRESSED_DATASET_SUMMARY_PATH,

    R1_ACTIONS_PATH,
    R1_GRID_SUMMARY_PATH,
]:

    assert path.exists(), path


for path in [
    CONTRACT_OUT,
    UNIVERSE_OUT,
    DATASET_SUMMARY_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\n12.12A2 formal artifact already exists:\n"
            f"{path}\n\n"
            "Do not overwrite frozen candidate-universe provenance."
        )


# ============================================================
# 1. LOCKED IDENTIFIERS
# ============================================================

BRANCH_ID = (
    "POST_R1P50_RESIDUAL_SYNTHETIC_NODE_RECOVERY_V1"
)


A0_SHA = (
    "03243381d2c6c33218fda9c94be0462f"
    "ad28733f8360fd37b1ea5e3d622ae760"
)


A1_CENSUS_SHA = (
    "b79ae5120680ff8c1fad7ff31039d0e0"
    "9dc26b7bffc6146f3c97867b1f80889a"
)


A1_INVARIANCE_SHA = (
    "524afa1a3d33bb83676bf0b721996f42"
    "91a969631fc755a4f21e0596fcd30e9b"
)


CURRENT_BEST_SHA = (
    "65d408c0a8c6089e1d008d0c2c69fb4a"
    "76914c0bc1138486e2845f2e835562a4"
)


FROZEN_R1_POLICY_ID = (
    "R1P50"
)


EXPECTED_SUPPRESSED_ROWS = (
    416_395
)


EXPECTED_R1P50_ACTIONS = (
    805
)


EXPECTED_FINAL_ROWS = (
    EXPECTED_SUPPRESSED_ROWS
    -
    EXPECTED_R1P50_ACTIONS
)


# ============================================================
# 2. HASH HELPERS
# ============================================================

def sha256_file(
    path,
):

    h = hashlib.sha256()


    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )


            if not chunk:
                break


            h.update(
                chunk
            )


    return h.hexdigest()


def dataframe_content_sha256(
    df,
    columns,
    *,
    chunk_size=50_000,
):

    """
    Stable content hash within the frozen schema/order contract.

    Data are first sorted deterministically.
    Hash uses canonical CSV serialization in chunks.
    """

    ordered = (
        df[
            columns
        ]
        .sort_values(
            [
                "dataset",
                "t",
                "z",
                "y",
                "x",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


    h = hashlib.sha256()


    for start in range(
        0,
        len(
            ordered
        ),
        chunk_size,
    ):

        stop = min(
            start
            +
            chunk_size,
            len(
                ordered
            ),
        )


        block = ordered.iloc[
            start:stop
        ]


        payload = block.to_csv(
            index=False,
            header=(
                start == 0
            ),
            float_format="%.17g",
            lineterminator="\n",
        ).encode(
            "utf-8"
        )


        h.update(
            payload
        )


    return h.hexdigest()


# ============================================================
# 3. VERIFY PARENT BRANCH
# ============================================================

a0 = json.loads(
    A0_PATH.read_text(
        encoding="utf-8"
    )
)


a1 = json.loads(
    A1_PATH.read_text(
        encoding="utf-8"
    )
)


a1_inv = json.loads(
    A1_INV_PATH.read_text(
        encoding="utf-8"
    )
)


current_best = json.loads(
    CURRENT_BEST_PATH.read_text(
        encoding="utf-8"
    )
)


assert a0[
    "branch_id"
] == BRANCH_ID


assert a0[
    "branch_contract_sha256"
] == A0_SHA


assert a0[
    "status"
] == (
    "NEW_INDEPENDENT_BRANCH_FROZEN_BEFORE_NEW_GT"
)


assert a0[
    "deployment_constraints"
][
    "candidate_generation_must_be_gt_blind"
] is True


assert a0[
    "selection_rules"
][
    "r1p50_existing_policy_may_be_retuned_inside_this_branch"
] is False


assert a0[
    "current_state"
][
    "candidate_universe_defined"
] is False


assert a0[
    "current_state"
][
    "candidate_universe_frozen"
] is False


assert a0[
    "current_state"
][
    "fold0_used"
] is False


assert a1[
    "census_sha256"
] == A1_CENSUS_SHA


assert a1[
    "residual_root_cause"
][
    "detection_limited_fn"
] == 4097


assert a1[
    "detection_gap_topology"
][
    "unique_missing_nodes"
] == 2985


assert a1[
    "protocol"
][
    "new_gt_read"
] is False


assert a1[
    "protocol"
][
    "candidate_universe_generated"
] is False


assert a1_inv[
    "contract_sha256"
] == A1_INVARIANCE_SHA


assert a1_inv[
    "gt"
][
    "new_gt_read"
] is False


assert current_best[
    "checkpoint_sha256"
] == CURRENT_BEST_SHA


assert current_best[
    "protocol"
][
    "fold0_available_for_future_selection"
] is False


print(
    "========== 12.12A2 BRANCH LOCK =========="
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "A0 SHA:",
    A0_SHA
)


print(
    "A1 census SHA:",
    A1_CENSUS_SHA
)


print(
    "development missing GT nodes:",
    2985
)


print(
    "new GT read before A2:",
    "NO"
)


print(
    "Fold0 available for selection:",
    "NO"
)


# ============================================================
# 4. VERIFY HISTORICAL SUPPRESSED-PEAK CONTRACT
# ============================================================

suppressed_summary = json.loads(
    SUPPRESSED_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert suppressed_summary[
    "stage"
] == "12.10C"


assert abs(
    float(
        suppressed_summary[
            "candidate_floor"
        ]
    )
    -
    0.90
) <= 1e-12


assert abs(
    float(
        suppressed_summary[
            "production_threshold"
        ]
    )
    -
    0.995
) <= 1e-12


assert int(
    suppressed_summary[
        "candidates"
    ]
) == EXPECTED_SUPPRESSED_ROWS


# The 12.10C source contract is explicitly GT-blind.
# Support either explicit summary representation if present.
if "gt_used" in suppressed_summary:

    assert (
        suppressed_summary[
            "gt_used"
        ]
        is False
    )


print(
    "\n========== HISTORICAL SUPPRESSED-PEAK CONTRACT =========="
)


print(
    "source stage:",
    suppressed_summary[
        "stage"
    ]
)


print(
    "candidate floor:",
    suppressed_summary[
        "candidate_floor"
    ]
)


print(
    "production threshold:",
    suppressed_summary[
        "production_threshold"
    ]
)


print(
    "historical candidates:",
    suppressed_summary[
        "candidates"
    ]
)


print(
    "source generation GT-blind:",
    "YES"
)


# ============================================================
# 5. LOAD SUPPRESSED UNIVERSE
# ============================================================

suppressed = pd.read_pickle(
    SUPPRESSED_PATH
)


EXPECTED_COLUMNS = [
    "dataset",
    "fold",

    "t",

    "z_ds",
    "y_ds",
    "x_ds",

    "z",
    "y",
    "x",

    "detector_prob",
    "detector_logit",

    "release_band",
]


assert list(
    suppressed.columns
) == EXPECTED_COLUMNS


assert len(
    suppressed
) == EXPECTED_SUPPRESSED_ROWS


assert suppressed[
    "dataset"
].nunique() == 103


assert sorted(
    pd.to_numeric(
        suppressed[
            "fold"
        ],
        errors="raise",
    )
    .astype(
        int
    )
    .unique()
    .tolist()
) == [
    1,
    2,
    3,
    4,
]


assert (
    pd.to_numeric(
        suppressed[
            "detector_prob"
        ],
        errors="raise",
    )
    .gt(
        0.90
    )
    .all()
)


assert (
    pd.to_numeric(
        suppressed[
            "detector_prob"
        ],
        errors="raise",
    )
    .le(
        0.995
        +
        1e-7
    )
    .all()
)


KEY_COLUMNS = [
    "dataset",
    "t",
    "z",
    "y",
    "x",
]


assert not suppressed[
    KEY_COLUMNS
].duplicated().any()


# ------------------------------------------------------------
# Hard safety:
# no GT/oracle/evaluation field may exist in the source universe
# ------------------------------------------------------------

FORBIDDEN_COLUMN_TOKENS = [
    "oracle",
    "truth",
    "ground_truth",
    "official_fp",
    "matched_gt",
    "gt_node",
    "gt_error",
    "is_true",
    "label",
]


for col in suppressed.columns:

    low = str(
        col
    ).lower()


    assert not any(
        token
        in
        low

        for token in
        FORBIDDEN_COLUMN_TOKENS
    ), col


print(
    "\n========== SOURCE NODE CANDIDATE UNIVERSE =========="
)


print(
    "rows:",
    len(
        suppressed
    )
)


print(
    "datasets:",
    suppressed[
        "dataset"
    ].nunique()
)


print(
    "folds:",
    sorted(
        suppressed[
            "fold"
        ]
        .astype(int)
        .unique()
        .tolist()
    )
)


print(
    "duplicate candidate keys:",
    int(
        suppressed[
            KEY_COLUMNS
        ]
        .duplicated()
        .sum()
    )
)


print(
    "GT/oracle columns:",
    "NONE"
)


print(
    "SOURCE_UNIVERSE_GT_BLIND_SCHEMA: PASS"
)


# ============================================================
# 6. VERIFY FROZEN R1P50 ACTIONS
# ============================================================

r1_actions = pd.read_pickle(
    R1_ACTIONS_PATH
)


required_r1_columns = {
    "policy_id",

    "dataset",
    "fold",

    "t",
    "z",
    "y",
    "x",

    "source_id",
    "target_id",

    "detector_prob",
}


assert required_r1_columns.issubset(
    set(
        r1_actions.columns
    )
)


r1p50 = (
    r1_actions[
        r1_actions[
            "policy_id"
        ].eq(
            FROZEN_R1_POLICY_ID
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


assert len(
    r1p50
) == EXPECTED_R1P50_ACTIONS


assert not r1p50[
    KEY_COLUMNS
].duplicated().any()


assert sorted(
    pd.to_numeric(
        r1p50[
            "fold"
        ],
        errors="raise",
    )
    .astype(int)
    .unique()
    .tolist()
) == [
    1,
    2,
    3,
    4,
]


print(
    "\n========== FROZEN R1P50 EXCLUSION =========="
)


print(
    "policy:",
    FROZEN_R1_POLICY_ID
)


print(
    "accepted synthetic nodes:",
    len(
        r1p50
    )
)


print(
    "R1P10 reused:",
    "NO"
)


print(
    "R1P25 reused:",
    "NO"
)


print(
    "R1P50 threshold changed:",
    "NO"
)


# ============================================================
# 7. VERIFY EVERY R1P50 NODE CAME FROM THE SUPPRESSED UNIVERSE
# ============================================================

source_keys = (
    suppressed[
        KEY_COLUMNS
    ]
    .copy()
)


source_keys[
    "_in_suppressed"
] = 1


r1_membership = (
    r1p50[
        KEY_COLUMNS
    ]
    .merge(
        source_keys,
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one",
    )
)


matched_r1p50 = int(
    r1_membership[
        "_in_suppressed"
    ]
    .fillna(
        0
    )
    .astype(int)
    .sum()
)


assert matched_r1p50 == EXPECTED_R1P50_ACTIONS, (
    matched_r1p50,
    EXPECTED_R1P50_ACTIONS,
)


print(
    "\nR1P50 coordinates found in 12.10C universe:",
    matched_r1p50,
    "/",
    EXPECTED_R1P50_ACTIONS
)


print(
    "R1P50_SOURCE_MEMBERSHIP: PASS"
)


# ============================================================
# 8. FREEZE CANDIDATE-DEFINITION CONTRACT
#
# IMPORTANT:
# write this BEFORE any oracle/GT operation.
# ============================================================

SOURCE_SUPPRESSED_FILE_SHA = (
    sha256_file(
        SUPPRESSED_PATH
    )
)


SOURCE_SUPPRESSED_SUMMARY_SHA = (
    sha256_file(
        SUPPRESSED_SUMMARY_PATH
    )
)


SOURCE_R1_ACTIONS_SHA = (
    sha256_file(
        R1_ACTIONS_PATH
    )
)


CONTRACT = {
    "stage":
        "12.12A2",

    "contract_id":
        (
            "POST_R1P50_NODE_LEVEL_"
            "SUPPRESSED_CANDIDATE_UNIVERSE_V1"
        ),

    "branch_id":
        BRANCH_ID,

    "status":
        "CANDIDATE_DEFINITION_FROZEN_BEFORE_ORACLE",

    "parent": {
        "a0_contract_sha256":
            A0_SHA,

        "a1_census_sha256":
            A1_CENSUS_SHA,

        "a1_invariance_sha256":
            A1_INVARIANCE_SHA,

        "current_best_checkpoint_sha256":
            CURRENT_BEST_SHA,
    },

    "source_universe": {
        "artifact":
            SUPPRESSED_PATH.name,

        "artifact_sha256":
            SOURCE_SUPPRESSED_FILE_SHA,

        "summary":
            SUPPRESSED_SUMMARY_PATH.name,

        "summary_sha256":
            SOURCE_SUPPRESSED_SUMMARY_SHA,

        "stage":
            "12.10C",

        "candidate_definition":
            (
                "PRODUCTION_LOCAL_MAXIMUM_AND_"
                "DETECTOR_PROB_GT_0P90_AND_"
                "EXACT_COORDINATE_ABSENT_FROM_P995"
            ),

        "candidate_floor":
            0.90,

        "p995_threshold":
            0.995,

        "rows":
            EXPECTED_SUPPRESSED_ROWS,

        "gt_blind":
            True,
    },

    "frozen_exclusion": {
        "reason":
            (
                "Nodes already inserted by frozen R1P50 "
                "are already present in CURRENT_BEST and "
                "must not remain residual node candidates."
            ),

        "artifact":
            R1_ACTIONS_PATH.name,

        "artifact_sha256":
            SOURCE_R1_ACTIONS_SHA,

        "policy_id":
            "R1P50",

        "actions":
            EXPECTED_R1P50_ACTIONS,

        "key":
            KEY_COLUMNS,

        "r1p50_retuned":
            False,
    },

    "candidate_rule": {
        "include":
            (
                "ALL_ROWS_IN_FROZEN_12P10C_"
                "SUPPRESSED_PEAK_UNIVERSE"
            ),

        "exclude":
            (
                "EXACT_DATASET_T_Z_Y_X_KEYS_ALREADY_"
                "INSERTED_BY_FROZEN_R1P50"
            ),

        "additional_detector_threshold":
            None,

        "bridge_geometry_required":
            False,

        "incoming_association_required":
            False,

        "outgoing_association_required":
            False,

        "association_rank_required":
            False,

        "bridge_probability_required":
            False,

        "source_endpoint_required":
            False,

        "target_endpoint_required":
            False,
    },

    "persisted_columns": (
        EXPECTED_COLUMNS
    ),

    "candidate_key": (
        KEY_COLUMNS
    ),

    "feature_legality": {
        "dataset":
            "PROVENANCE_ONLY_NOT_RUNTIME_SCORE_FEATURE",

        "fold":
            "PROVENANCE_ONLY_NEVER_RUNTIME_FEATURE",

        "t_z_y_x":
            "DEPLOYABLE_CANDIDATE_LOCATION",

        "z_ds_y_ds_x_ds":
            "DEPLOYABLE_DETECTOR_GRID_LOCATION",

        "detector_prob":
            "DEPLOYABLE_MODEL_SIGNAL",

        "detector_logit":
            "DEPLOYABLE_MODEL_SIGNAL",

        "release_band":
            "DERIVED_FROM_DETECTOR_PROB_AUDIT_ONLY",
    },

    "explicitly_forbidden_inputs": [
        "GT_NODE_COORDINATES",
        "GT_EDGE_LABELS",
        "OFFICIAL_FP",
        "ORACLE_COMPONENT_GEOMETRY",
        "PREDICTED_PATH_GT_SUPPORT_AUDIT",
        "FOLD0_STATISTICS",
        "R1P10_SELECTION",
        "R1P25_SELECTION",
        "R1P50_THRESHOLD_RETUNING",
        "BRIDGE_GEOMETRY_GATE",
        "BRIDGE_ASSOCIATION_GATE",
    ],

    "protocol": {
        "gt_read_before_freeze":
            False,

        "oracle_read_before_freeze":
            False,

        "fold0_used":
            False,

        "official_metric_evaluated":
            False,

        "scorer_selected":
            False,

        "threshold_selected":
            False,

        "graph_modified":
            False,
    },

    "next_after_materialization":
        (
            "12.12A3_DEVELOPMENT_ORACLE_"
            "NODE_CANDIDATE_COVERAGE_AUDIT"
        ),
}


CONTRACT_SHA = hashlib.sha256(
    json.dumps(
        CONTRACT,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


CONTRACT[
    "contract_sha256"
] = CONTRACT_SHA


CONTRACT_OUT.write_text(
    json.dumps(
        CONTRACT,
        indent=2,
    ),
    encoding="utf-8",
)


assert CONTRACT_OUT.exists()


print(
    "\n========== A2 CANDIDATE CONTRACT FREEZE =========="
)


print(
    "contract:",
    CONTRACT[
        "contract_id"
    ]
)


print(
    "contract SHA:",
    CONTRACT_SHA
)


print(
    "GT read before freeze:",
    "NO"
)


print(
    "oracle read before freeze:",
    "NO"
)


print(
    "CANDIDATE_DEFINITION_FREEZE: PASS"
)


# ============================================================
# 9. MATERIALIZE EXACT POST-R1P50 UNIVERSE
# ============================================================

r1_key_frame = (
    r1p50[
        KEY_COLUMNS
    ]
    .copy()
)


r1_key_frame[
    "_already_r1p50"
] = True


universe = (
    suppressed
    .merge(
        r1_key_frame,
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one",
    )
)


removed_mask = (
    universe[
        "_already_r1p50"
    ]
    .fillna(
        False
    )
    .astype(
        bool
    )
)


removed_count = int(
    removed_mask.sum()
)


assert removed_count == EXPECTED_R1P50_ACTIONS


universe = (
    universe[
        ~removed_mask
    ]
    .drop(
        columns=[
            "_already_r1p50",
        ]
    )
    .sort_values(
        KEY_COLUMNS,
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    universe
) == EXPECTED_FINAL_ROWS


assert universe[
    "dataset"
].nunique() == 103


assert not universe[
    KEY_COLUMNS
].duplicated().any()


# ------------------------------------------------------------
# No R1P50 coordinate may survive.
# ------------------------------------------------------------

remaining_overlap = (
    universe[
        KEY_COLUMNS
    ]
    .merge(
        r1p50[
            KEY_COLUMNS
        ],
        on=KEY_COLUMNS,
        how="inner",
    )
)


assert len(
    remaining_overlap
) == 0


# ------------------------------------------------------------
# Schema remains exactly node-level suppressed schema.
# ------------------------------------------------------------

assert list(
    universe.columns
) == EXPECTED_COLUMNS


# ============================================================
# 10. CONTENT HASH BEFORE PERSISTENCE
# ============================================================

CONTENT_HASH_COLUMNS = [
    "dataset",
    "fold",

    "t",

    "z_ds",
    "y_ds",
    "x_ds",

    "z",
    "y",
    "x",

    "detector_prob",
    "detector_logit",

    "release_band",
]


UNIVERSE_CONTENT_SHA = (
    dataframe_content_sha256(
        universe,
        CONTENT_HASH_COLUMNS,
    )
)


# ============================================================
# 11. DATASET SUMMARY
# ============================================================

source_counts = (
    suppressed
    .groupby(
        [
            "dataset",
            "fold",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size":
                "suppressed_source_candidates"
        }
    )
)


r1_counts = (
    r1p50
    .groupby(
        [
            "dataset",
            "fold",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size":
                "r1p50_nodes_excluded"
        }
    )
)


final_counts = (
    universe
    .groupby(
        [
            "dataset",
            "fold",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size":
                "a2_node_candidates"
        }
    )
)


dataset_summary = (
    source_counts
    .merge(
        r1_counts,
        on=[
            "dataset",
            "fold",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        final_counts,
        on=[
            "dataset",
            "fold",
        ],
        how="left",
        validate="one_to_one",
    )
)


dataset_summary[
    "r1p50_nodes_excluded"
] = (
    dataset_summary[
        "r1p50_nodes_excluded"
    ]
    .fillna(
        0
    )
    .astype(
        int
    )
)


dataset_summary[
    "a2_node_candidates"
] = (
    dataset_summary[
        "a2_node_candidates"
    ]
    .fillna(
        0
    )
    .astype(
        int
    )
)


assert len(
    dataset_summary
) == 103


assert (
    dataset_summary[
        "suppressed_source_candidates"
    ]
    -
    dataset_summary[
        "r1p50_nodes_excluded"
    ]
    ==
    dataset_summary[
        "a2_node_candidates"
    ]
).all()


assert int(
    dataset_summary[
        "suppressed_source_candidates"
    ].sum()
) == EXPECTED_SUPPRESSED_ROWS


assert int(
    dataset_summary[
        "r1p50_nodes_excluded"
    ].sum()
) == EXPECTED_R1P50_ACTIONS


assert int(
    dataset_summary[
        "a2_node_candidates"
    ].sum()
) == EXPECTED_FINAL_ROWS


# ============================================================
# 12. GT-BLIND DESCRIPTIVE AUDIT
#
# Descriptive only. No policy selection.
# ============================================================

prob = pd.to_numeric(
    universe[
        "detector_prob"
    ],
    errors="raise",
)


prob_quantiles = {
    str(
        q
    ):
        float(
            value
        )

    for q, value
    in prob.quantile(
        [
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    ).items()
}


nested_counts = {
    "gt_0p99":
        int(
            prob.gt(
                0.99
            ).sum()
        ),

    "gt_0p98":
        int(
            prob.gt(
                0.98
            ).sum()
        ),

    "gt_0p95":
        int(
            prob.gt(
                0.95
            ).sum()
        ),

    "gt_0p90":
        int(
            prob.gt(
                0.90
            ).sum()
        ),
}


fold_counts = (
    universe
    .groupby(
        "fold"
    )
    .size()
    .astype(int)
    .to_dict()
)


print(
    "\n========== A2 GT-BLIND UNIVERSE =========="
)


print(
    "historical suppressed candidates:",
    EXPECTED_SUPPRESSED_ROWS
)


print(
    "frozen R1P50 nodes excluded:",
    removed_count
)


print(
    "post-R1P50 node candidates:",
    len(
        universe
    )
)


print(
    "datasets:",
    universe[
        "dataset"
    ].nunique()
)


print(
    "fold counts:"
)


for fold in sorted(
    fold_counts
):

    print(
        f"  Fold {int(fold)}:",
        int(
            fold_counts[
                fold
            ]
        )
    )


print(
    "\ndetector probability quantiles:"
)


for q, value in prob_quantiles.items():

    print(
        f"  q={q}:",
        value
    )


print(
    "\nnested detector counts "
    "(AUDIT ONLY — NOT policy thresholds):"
)


for key, value in nested_counts.items():

    print(
        f"  {key}:",
        value
    )


# ============================================================
# 13. PERSIST FROZEN UNIVERSE
# ============================================================

universe.to_pickle(
    UNIVERSE_OUT
)


dataset_summary.to_pickle(
    DATASET_SUMMARY_OUT
)


assert UNIVERSE_OUT.exists()

assert DATASET_SUMMARY_OUT.exists()


UNIVERSE_FILE_SHA = (
    sha256_file(
        UNIVERSE_OUT
    )
)


DATASET_SUMMARY_FILE_SHA = (
    sha256_file(
        DATASET_SUMMARY_OUT
    )
)


# ------------------------------------------------------------
# Reload fidelity.
# ------------------------------------------------------------

reloaded = pd.read_pickle(
    UNIVERSE_OUT
)


assert len(
    reloaded
) == EXPECTED_FINAL_ROWS


assert list(
    reloaded.columns
) == EXPECTED_COLUMNS


RELOADED_CONTENT_SHA = (
    dataframe_content_sha256(
        reloaded,
        CONTENT_HASH_COLUMNS,
    )
)


assert (
    RELOADED_CONTENT_SHA
    ==
    UNIVERSE_CONTENT_SHA
)


# ============================================================
# 14. FORMAL SUMMARY
# ============================================================

SUMMARY = {
    "stage":
        "12.12A2",

    "status":
        "GT_BLIND_NODE_LEVEL_CANDIDATE_UNIVERSE_FROZEN",

    "branch_id":
        BRANCH_ID,

    "contract_id":
        CONTRACT[
            "contract_id"
        ],

    "contract_sha256":
        CONTRACT_SHA,

    "source": {
        "suppressed_universe":
            SUPPRESSED_PATH.name,

        "suppressed_universe_sha256":
            SOURCE_SUPPRESSED_FILE_SHA,

        "suppressed_summary_sha256":
            SOURCE_SUPPRESSED_SUMMARY_SHA,

        "source_candidates":
            EXPECTED_SUPPRESSED_ROWS,
    },

    "frozen_r1p50_exclusion": {
        "policy_id":
            "R1P50",

        "action_artifact_sha256":
            SOURCE_R1_ACTIONS_SHA,

        "nodes_excluded":
            EXPECTED_R1P50_ACTIONS,

        "all_r1p50_keys_found_in_source":
            True,

        "r1p50_retuned":
            False,
    },

    "universe": {
        "artifact":
            UNIVERSE_OUT.name,

        "artifact_file_sha256":
            UNIVERSE_FILE_SHA,

        "content_sha256":
            UNIVERSE_CONTENT_SHA,

        "rows":
            int(
                len(
                    universe
                )
            ),

        "datasets":
            int(
                universe[
                    "dataset"
                ].nunique()
            ),

        "folds":
            [
                1,
                2,
                3,
                4,
            ],

        "columns":
            EXPECTED_COLUMNS,

        "candidate_key":
            KEY_COLUMNS,

        "duplicate_keys":
            0,

        "r1p50_key_overlap":
            0,
    },

    "dataset_summary": {
        "artifact":
            DATASET_SUMMARY_OUT.name,

        "artifact_sha256":
            DATASET_SUMMARY_FILE_SHA,

        "datasets":
            int(
                len(
                    dataset_summary
                )
            ),
    },

    "gt_blind_audit": {
        "detector_probability_quantiles":
            prob_quantiles,

        "nested_probability_counts_audit_only":
            nested_counts,

        "fold_counts":
            {
                str(
                    int(
                        k
                    )
                ):
                    int(
                        v
                    )

                for k, v
                in fold_counts.items()
            },
    },

    "protocol": {
        "candidate_definition_frozen_before_oracle":
            True,

        "new_gt_read":
            False,

        "oracle_missing_node_coordinates_used":
            False,

        "official_metric_evaluated":
            False,

        "fold0_used":
            False,

        "scorer_selected":
            False,

        "threshold_selected":
            False,

        "association_policy_selected":
            False,

        "graph_modified":
            False,

        "g3h3_reopened":
            False,

        "r1p50_retuned":
            False,
    },

    "development_oracle_target_reserved_for_a3": {
        "missing_gt_nodes":
            2985,

        "detection_limited_edge_fn":
            4097,
    },

    "next":
        (
            "12.12A3_DEVELOPMENT_ORACLE_"
            "NODE_CANDIDATE_COVERAGE_AUDIT"
        ),
}


SUMMARY_SHA = hashlib.sha256(
    json.dumps(
        SUMMARY,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


SUMMARY[
    "summary_sha256"
] = SUMMARY_SHA


SUMMARY_OUT.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 15. FINAL RELOAD / FREEZE INTEGRITY
# ============================================================

contract_check = json.loads(
    CONTRACT_OUT.read_text(
        encoding="utf-8"
    )
)


summary_check = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert contract_check[
    "contract_sha256"
] == CONTRACT_SHA


assert contract_check[
    "status"
] == (
    "CANDIDATE_DEFINITION_FROZEN_BEFORE_ORACLE"
)


assert contract_check[
    "protocol"
][
    "gt_read_before_freeze"
] is False


assert contract_check[
    "protocol"
][
    "oracle_read_before_freeze"
] is False


assert summary_check[
    "summary_sha256"
] == SUMMARY_SHA


assert summary_check[
    "status"
] == (
    "GT_BLIND_NODE_LEVEL_CANDIDATE_UNIVERSE_FROZEN"
)


assert summary_check[
    "universe"
][
    "rows"
] == EXPECTED_FINAL_ROWS


assert summary_check[
    "universe"
][
    "r1p50_key_overlap"
] == 0


assert summary_check[
    "protocol"
][
    "new_gt_read"
] is False


assert summary_check[
    "protocol"
][
    "fold0_used"
] is False


assert summary_check[
    "protocol"
][
    "scorer_selected"
] is False


assert summary_check[
    "protocol"
][
    "threshold_selected"
] is False


# ============================================================
# 16. FINAL
# ============================================================

print(
    "\n========== 12.12A2 FINAL FREEZE =========="
)


print(
    "GT_BLIND_NODE_LEVEL_CANDIDATE_UNIVERSE_FREEZE: PASS"
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "candidate definition:",
    (
        "12.10C suppressed peaks "
        "minus frozen R1P50 inserted nodes"
    )
)


print(
    "source suppressed candidates:",
    EXPECTED_SUPPRESSED_ROWS
)


print(
    "R1P50 nodes excluded:",
    EXPECTED_R1P50_ACTIONS
)


print(
    "final candidate rows:",
    len(
        universe
    )
)


print(
    "expected:",
    EXPECTED_FINAL_ROWS
)


print(
    "datasets:",
    universe[
        "dataset"
    ].nunique()
)


print(
    "duplicate candidate keys:",
    0
)


print(
    "R1P50 coordinate overlap:",
    0
)


print(
    "contract SHA:",
    CONTRACT_SHA
)


print(
    "universe content SHA:",
    UNIVERSE_CONTENT_SHA
)


print(
    "universe file SHA:",
    UNIVERSE_FILE_SHA
)


print(
    "summary SHA:",
    SUMMARY_SHA
)


print(
    "GT read:",
    "NO"
)


print(
    "oracle touched:",
    "NO"
)


print(
    "official metric evaluated:",
    "NO"
)


print(
    "scorer selected:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "association policy selected:",
    "NO"
)


print(
    "R1P50 retuned:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "G3H3 reopened:",
    "NO"
)


print(
    "candidate universe now frozen:",
    "YES"
)


print(
    "next:"
)


print(
    "12.12A3_DEVELOPMENT_ORACLE_"
    "NODE_CANDIDATE_COVERAGE_AUDIT"
)

In [ ]:
# ============================================================
# Stage 12.12A3
#
# DEVELOPMENT ORACLE NODE-CANDIDATE COVERAGE AUDIT
#
# Parent:
#   POST_R1P50_NODE_LEVEL_SUPPRESSED_CANDIDATE_UNIVERSE_V1
#
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
#
# A2 froze, BEFORE oracle:
#
#   415,590 GT-blind node candidates
#
# generated as:
#
#   frozen Stage12.10C suppressed peaks
#   minus
#   frozen R1P50 inserted nodes
#
# A3 now asks, on folds1-4 DEVELOPMENT ONLY:
#
#   How much of the remaining detection-limited missing-node
#   population is geometrically covered?
#
# ------------------------------------------------------------
# TWO DIFFERENT CEILINGS
# ------------------------------------------------------------
#
# 1. ANY-CANDIDATE COVERAGE
#
#    A missing GT node is covered if at least one frozen
#    candidate exists within radius R in the SAME frame.
#
#    This is a loose geometric ceiling.
#
# 2. ONE-TO-ONE MAXIMUM MATCHING COVERAGE
#
#    Candidates and missing GT nodes are matched one-to-one
#    inside each dataset/frame under radius R.
#
#    This is a stronger, official-matching-like cardinality
#    ceiling and prevents one candidate from "covering"
#    multiple missing nodes simultaneously.
#
# ------------------------------------------------------------
# RADII
# ------------------------------------------------------------
#
#   2 um
#   3 um
#   5 um
#   7 um  <- official tolerance
#
# ------------------------------------------------------------
# ALSO AUDIT
# ------------------------------------------------------------
#
#   - missing-node nearest-candidate distance
#   - candidate multiplicity around each missing node
#   - missing-node multiplicity around each candidate
#   - SOURCE_MISSING / TARGET_MISSING / BOTH_MISSING edges
#   - single-node vs multi-node components
#   - per-fold / per-prefix coverage
#   - detector probability conditional on oracle proximity
#
# Detector-probability analysis is AUDIT ONLY.
#
# NO detector threshold is selected here.
#
# ------------------------------------------------------------
# PROTOCOL
# ------------------------------------------------------------
#
# Uses development GT:            YES
# New GT exposure for branch:     YES — first time after A2 freeze
# Fold0:                          NO
# Candidate universe modification:NO
# Candidate generation:           NO
# Scorer training:                NO
# Feature selection:              NO
# Threshold selection:            NO
# Association policy selection:   NO
# Graph mutation:                 NO
# Official graph metric:          NO
# G3H3 reopened:                  NO
# R1P50 retuned:                  NO
#
# Keep after running: YES
# ============================================================


from pathlib import Path

import hashlib
import json
import sys

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)


# ------------------------------------------------------------
# A0 / A1
# ------------------------------------------------------------

A0_PATH = (
    S12
    / "stage12_post_r1p50_residual_synthetic_node_recovery_v1_frozen.json"
)


A1_PATH = (
    S12
    / "stage12_post_g3h3_development_detection_residual_census_v1.json"
)


A1_INV_PATH = (
    S12
    / "stage12_post_g3h3_detection_residual_invariance_v1_frozen.json"
)


# ------------------------------------------------------------
# Frozen A2
# ------------------------------------------------------------

A2_CONTRACT_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_universe_v1_contract_frozen.json"
)


A2_UNIVERSE_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_universe_v1.pkl"
)


A2_SUMMARY_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_universe_v1_summary.json"
)


# ------------------------------------------------------------
# Frozen pre-G3H3 CURRENT_BEST FN identities
#
# A1-R1 formally established that detection-limited identities
# remain invariant under edge-only G3H3.
# ------------------------------------------------------------

FN_PATH = (
    S12
    / "stage12_folds14_current_best_edge_fn_residuals.pkl"
)


# ------------------------------------------------------------
# Formal A3 outputs
# ------------------------------------------------------------

MISSING_NODE_OUT = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_missing_nodes_v1.pkl"
)


CANDIDATE_ORACLE_OUT = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_candidates_v1.pkl"
)


EDGE_ORACLE_OUT = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_detection_edges_v1.pkl"
)


COMPONENT_OUT = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_components_v1.pkl"
)


MATCHING_OUT = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_matching_ceiling_v1.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_coverage_v1_summary.json"
)


for path in [
    A0_PATH,
    A1_PATH,
    A1_INV_PATH,

    A2_CONTRACT_PATH,
    A2_UNIVERSE_PATH,
    A2_SUMMARY_PATH,

    FN_PATH,
]:

    assert path.exists(), path


for path in [
    MISSING_NODE_OUT,
    CANDIDATE_ORACLE_OUT,
    EDGE_ORACLE_OUT,
    COMPONENT_OUT,
    MATCHING_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\n12.12A3 formal artifact already exists:\n"
            f"{path}\n\n"
            "Do not overwrite oracle-audit provenance."
        )


# ============================================================
# 1. EXACT LOCKS
# ============================================================

BRANCH_ID = (
    "POST_R1P50_RESIDUAL_SYNTHETIC_NODE_RECOVERY_V1"
)


A0_SHA = (
    "03243381d2c6c33218fda9c94be0462f"
    "ad28733f8360fd37b1ea5e3d622ae760"
)


A1_CENSUS_SHA = (
    "b79ae5120680ff8c1fad7ff31039d0e0"
    "9dc26b7bffc6146f3c97867b1f80889a"
)


A1_INVARIANCE_SHA = (
    "524afa1a3d33bb83676bf0b721996f42"
    "91a969631fc755a4f21e0596fcd30e9b"
)


A2_CONTRACT_SHA = (
    "fbf3f2bcc81d87180837a6170a598296"
    "a5276a62728c8a21d6c4fc0f9a57313b"
)


A2_CONTENT_SHA = (
    "3462a6cd20eee93e55ba7cca8f110654"
    "5c22cb5815ac1e05dde09e04d5cf8e91"
)


A2_FILE_SHA = (
    "a082babaeecda9e50a13a329abbc912f"
    "f4a16ad56ce3695301d266ca0c5e4147"
)


A2_SUMMARY_SHA = (
    "5b708ef71cb26bf7178db8b5b53663f7"
    "fbfb2a59cc23f2898b10f1b7c9f4062f"
)


EXPECTED_CANDIDATES = 415_590


EXPECTED_OLD_FN = 10_254
EXPECTED_DETECTION_FN = 4_097


EXPECTED_MISSING_NODES = 2_985
EXPECTED_COMPONENTS = 1_595

EXPECTED_SINGLE_COMPONENTS = 1_004
EXPECTED_MULTI_COMPONENTS = 591

EXPECTED_MULTI_NODES = 1_981


EXPECTED_EDGE_CLASSES = {
    "BOTH_MISSING":
        1_390,

    "SOURCE_MISSING":
        1_365,

    "TARGET_MISSING":
        1_342,
}


RADII = [
    2.0,
    3.0,
    5.0,
    7.0,
]


# Original image voxel dimensions in micrometers.
SCALE = np.asarray(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=np.float64,
)


# ============================================================
# 2. IMPORT EXACT GRAPH LOADER
# ============================================================

sys.path.insert(
    0,
    str(
        PROJECT
        / "src"
    ),
)


from biohub_cell_tracking import frozen_v9 as fv9


# ============================================================
# 3. HELPERS
# ============================================================

def sha256_file(
    path,
):

    h = hashlib.sha256()


    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )


            if not chunk:
                break


            h.update(
                chunk
            )


    return h.hexdigest()


def radius_name(
    radius,
):

    return (
        str(
            float(
                radius
            )
        )
        .replace(
            ".",
            "p",
        )
        +
        "um"
    )


def prefix_of(
    dataset,
):

    return str(
        dataset
    ).split(
        "_",
        1,
    )[0]


# ============================================================
# 4. VERIFY PARENT PROVENANCE
# ============================================================

a0 = json.loads(
    A0_PATH.read_text(
        encoding="utf-8"
    )
)


a1 = json.loads(
    A1_PATH.read_text(
        encoding="utf-8"
    )
)


a1_inv = json.loads(
    A1_INV_PATH.read_text(
        encoding="utf-8"
    )
)


a2_contract = json.loads(
    A2_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


a2_summary = json.loads(
    A2_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert a0[
    "branch_id"
] == BRANCH_ID


assert a0[
    "branch_contract_sha256"
] == A0_SHA


assert a1[
    "census_sha256"
] == A1_CENSUS_SHA


assert a1_inv[
    "contract_sha256"
] == A1_INVARIANCE_SHA


assert a1_inv[
    "invariants"
][
    "detection_limited_edge_fn_identity_unchanged"
] is True


assert a2_contract[
    "contract_sha256"
] == A2_CONTRACT_SHA


assert a2_contract[
    "status"
] == (
    "CANDIDATE_DEFINITION_FROZEN_BEFORE_ORACLE"
)


assert a2_contract[
    "protocol"
][
    "gt_read_before_freeze"
] is False


assert a2_contract[
    "protocol"
][
    "oracle_read_before_freeze"
] is False


assert a2_summary[
    "summary_sha256"
] == A2_SUMMARY_SHA


assert a2_summary[
    "status"
] == (
    "GT_BLIND_NODE_LEVEL_CANDIDATE_UNIVERSE_FROZEN"
)


assert a2_summary[
    "universe"
][
    "rows"
] == EXPECTED_CANDIDATES


assert a2_summary[
    "universe"
][
    "content_sha256"
] == A2_CONTENT_SHA


assert a2_summary[
    "universe"
][
    "artifact_file_sha256"
] == A2_FILE_SHA


assert sha256_file(
    A2_UNIVERSE_PATH
) == A2_FILE_SHA


assert a2_summary[
    "protocol"
][
    "new_gt_read"
] is False


assert a2_summary[
    "protocol"
][
    "fold0_used"
] is False


print(
    "========== 12.12A3 PARENT FREEZE =========="
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "A2 contract SHA:",
    A2_CONTRACT_SHA
)


print(
    "A2 content SHA:",
    A2_CONTENT_SHA
)


print(
    "A2 candidate rows:",
    EXPECTED_CANDIDATES
)


print(
    "candidate definition frozen before oracle:",
    "YES"
)


print(
    "Fold0 used:",
    "NO"
)


# ============================================================
# 5. LOAD FROZEN A2 CANDIDATES
# ============================================================

candidates = pd.read_pickle(
    A2_UNIVERSE_PATH
)


assert len(
    candidates
) == EXPECTED_CANDIDATES


CANDIDATE_KEY = [
    "dataset",
    "t",
    "z",
    "y",
    "x",
]


assert not candidates[
    CANDIDATE_KEY
].duplicated().any()


required_candidate_columns = {
    "dataset",
    "fold",

    "t",

    "z",
    "y",
    "x",

    "detector_prob",
    "detector_logit",
}


assert required_candidate_columns.issubset(
    candidates.columns
)


candidates[
    "prefix"
] = (
    candidates[
        "dataset"
    ]
    .map(
        prefix_of
    )
)


# Stable candidate identity.
candidates = (
    candidates
    .sort_values(
        CANDIDATE_KEY,
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


candidates[
    "candidate_id"
] = np.arange(
    len(
        candidates
    ),
    dtype=np.int64,
)


# ============================================================
# 6. LOAD FROZEN DETECTION-LIMITED FN IDENTITIES
# ============================================================

fn = pd.read_pickle(
    FN_PATH
)


assert len(
    fn
) == EXPECTED_OLD_FN


required_fn_columns = {
    "dataset",
    "fold",

    "root_cause",

    "gt_source_id",
    "gt_target_id",

    "pred_source_id",
    "pred_target_id",
}


assert required_fn_columns.issubset(
    fn.columns
), (
    "FN schema mismatch",
    list(
        fn.columns
    ),
)


det = (
    fn[
        fn[
            "root_cause"
        ].eq(
            "DETECTION_LIMITED"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


assert len(
    det
) == EXPECTED_DETECTION_FN


det[
    "source_missing"
] = (
    pd.to_numeric(
        det[
            "pred_source_id"
        ],
        errors="raise",
    )
    .lt(
        0
    )
)


det[
    "target_missing"
] = (
    pd.to_numeric(
        det[
            "pred_target_id"
        ],
        errors="raise",
    )
    .lt(
        0
    )
)


assert (
    det[
        "source_missing"
    ]
    |
    det[
        "target_missing"
    ]
).all()


det[
    "missing_class"
] = np.select(
    [
        (
            det[
                "source_missing"
            ]
            &
            det[
                "target_missing"
            ]
        ),

        det[
            "source_missing"
        ],

        det[
            "target_missing"
        ],
    ],
    [
        "BOTH_MISSING",
        "SOURCE_MISSING",
        "TARGET_MISSING",
    ],
    default="INVALID",
)


class_counts = (
    det[
        "missing_class"
    ]
    .value_counts()
    .to_dict()
)


assert class_counts == EXPECTED_EDGE_CLASSES, (
    class_counts,
    EXPECTED_EDGE_CLASSES,
)


det[
    "prefix"
] = (
    det[
        "dataset"
    ]
    .map(
        prefix_of
    )
)


print(
    "\n========== DETECTION-LIMITED ORACLE POPULATION =========="
)


print(
    "detection-limited edges:",
    len(
        det
    )
)


print(
    "classes:"
)


for key in [
    "BOTH_MISSING",
    "SOURCE_MISSING",
    "TARGET_MISSING",
]:

    print(
        f"  {key}:",
        class_counts[
            key
        ]
    )


# ============================================================
# 7. RECONSTRUCT UNIQUE MISSING NODE IDENTITIES
# ============================================================

records = []


for row in det.itertuples(
    index=False
):

    if bool(
        row.source_missing
    ):

        records.append(
            {
                "dataset":
                    row.dataset,

                "fold":
                    int(
                        row.fold
                    ),

                "prefix":
                    prefix_of(
                        row.dataset
                    ),

                "gt_node_id":
                    int(
                        row.gt_source_id
                    ),

                "role":
                    "SOURCE",
            }
        )


    if bool(
        row.target_missing
    ):

        records.append(
            {
                "dataset":
                    row.dataset,

                "fold":
                    int(
                        row.fold
                    ),

                "prefix":
                    prefix_of(
                        row.dataset
                    ),

                "gt_node_id":
                    int(
                        row.gt_target_id
                    ),

                "role":
                    "TARGET",
            }
        )


incidence = pd.DataFrame(
    records
)


node_base = (
    incidence[
        [
            "dataset",
            "fold",
            "prefix",
            "gt_node_id",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "dataset",
            "gt_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    node_base
) == EXPECTED_MISSING_NODES


# FN incidence per missing GT node.
fn_incidence = (
    incidence
    .groupby(
        [
            "dataset",
            "gt_node_id",
        ]
    )
    .size()
    .rename(
        "fn_incidence"
    )
    .reset_index()
)


node_base = node_base.merge(
    fn_incidence,
    on=[
        "dataset",
        "gt_node_id",
    ],
    how="left",
    validate="one_to_one",
)


# ============================================================
# 8. RECONSTRUCT MISSING-NODE COMPONENTS
# ============================================================

keys = [
    (
        str(
            row.dataset
        ),
        int(
            row.gt_node_id
        ),
    )

    for row
    in node_base.itertuples(
        index=False
    )
]


parent = {
    key:
        key

    for key
    in keys
}


def uf_find(
    x,
):

    while parent[
        x
    ] != x:

        parent[
            x
        ] = parent[
            parent[
                x
            ]
        ]


        x = parent[
            x
        ]


    return x


def uf_union(
    a,
    b,
):

    ra = uf_find(
        a
    )


    rb = uf_find(
        b
    )


    if ra == rb:

        return


    # deterministic root
    if ra <= rb:

        parent[
            rb
        ] = ra

    else:

        parent[
            ra
        ] = rb


# BOTH_MISSING edges connect two missing nodes in the same gap.
for row in det[
    det[
        "missing_class"
    ].eq(
        "BOTH_MISSING"
    )
].itertuples(
    index=False
):

    a = (
        str(
            row.dataset
        ),
        int(
            row.gt_source_id
        ),
    )


    b = (
        str(
            row.dataset
        ),
        int(
            row.gt_target_id
        ),
    )


    assert a in parent

    assert b in parent


    uf_union(
        a,
        b,
    )


groups = {}


for key in keys:

    root = uf_find(
        key
    )


    groups.setdefault(
        root,
        [],
    ).append(
        key
    )


assert len(
    groups
) == EXPECTED_COMPONENTS


component_rows = []


component_id_map = {}


for component_id, (
    root,
    members,
) in enumerate(
    sorted(
        groups.items(),
        key=lambda x: x[
            0
        ],
    )
):

    members = sorted(
        members
    )


    dataset = root[
        0
    ]


    component_key = (
        f"{dataset}::"
        f"{root[1]}"
    )


    for member in members:

        component_id_map[
            member
        ] = component_key


    component_rows.append(
        {
            "component_key":
                component_key,

            "dataset":
                dataset,

            "prefix":
                prefix_of(
                    dataset
                ),

            "missing_nodes":
                int(
                    len(
                        members
                    )
                ),

            "root_gt_node_id":
                int(
                    root[
                        1
                    ]
                ),
        }
    )


components = pd.DataFrame(
    component_rows
)


assert len(
    components
) == EXPECTED_COMPONENTS


assert int(
    components[
        "missing_nodes"
    ].sum()
) == EXPECTED_MISSING_NODES


assert int(
    components[
        "missing_nodes"
    ].eq(
        1
    ).sum()
) == EXPECTED_SINGLE_COMPONENTS


assert int(
    components[
        "missing_nodes"
    ].ge(
        2
    ).sum()
) == EXPECTED_MULTI_COMPONENTS


assert int(
    components.loc[
        components[
            "missing_nodes"
        ].ge(
            2
        ),
        "missing_nodes",
    ].sum()
) == EXPECTED_MULTI_NODES


node_base[
    "component_key"
] = [
    component_id_map[
        (
            str(
                row.dataset
            ),
            int(
                row.gt_node_id
            ),
        )
    ]

    for row
    in node_base.itertuples(
        index=False
    )
]


print(
    "\n========== RECONSTRUCTED MISSING-NODE TOPOLOGY =========="
)


print(
    "missing nodes:",
    len(
        node_base
    )
)


print(
    "components:",
    len(
        components
    )
)


print(
    "single-node components:",
    int(
        components[
            "missing_nodes"
        ].eq(
            1
        ).sum()
    )
)


print(
    "multi-node components:",
    int(
        components[
            "missing_nodes"
        ].ge(
            2
        ).sum()
    )
)


print(
    "nodes in multi-node components:",
    int(
        components.loc[
            components[
                "missing_nodes"
            ].ge(
                2
            ),
            "missing_nodes",
        ].sum()
    )
)


print(
    "MISSING_NODE_TOPOLOGY_REPRODUCTION: PASS"
)


# ============================================================
# 9. FIRST NEW GT READ FOR THIS BRANCH
#
# Candidate universe is ALREADY frozen.
# ============================================================

print(
    "\n========== FIRST DEVELOPMENT ORACLE READ =========="
)


print(
    "candidate contract already frozen:",
    "YES"
)


print(
    "A2 contract SHA:",
    A2_CONTRACT_SHA
)


print(
    "loading GT for missing-node coordinates:",
    "YES"
)


datasets_needed = sorted(
    node_base[
        "dataset"
    ].unique()
    .tolist()
)


gt_coordinate_parts = []


for i, dataset in enumerate(
    datasets_needed,
    start=1,
):

    gt_path = (
        TRAIN_ROOT
        / f"{dataset}.geff"
    )


    assert gt_path.exists(), gt_path


    gt = fv9.load_graph(
        gt_path
    )


    gt_nodes = (
        gt.node_attrs(
            attr_keys=[
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        )
        .to_pandas()
    )


    gt_nodes[
        "node_id"
    ] = pd.to_numeric(
        gt_nodes[
            "node_id"
        ],
        errors="raise",
    ).astype(
        np.int64
    )


    required_ids = (
        node_base.loc[
            node_base[
                "dataset"
            ].eq(
                dataset
            ),
            "gt_node_id",
        ]
        .astype(
            np.int64
        )
        .unique()
    )


    selected = (
        gt_nodes[
            gt_nodes[
                "node_id"
            ].isin(
                required_ids
            )
        ][
            [
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        ]
        .copy()
    )


    assert selected[
        "node_id"
    ].nunique() == len(
        required_ids
    ), (
        dataset,
        len(
            required_ids
        ),
        selected[
            "node_id"
        ].nunique(),
    )


    selected[
        "dataset"
    ] = dataset


    selected = selected.rename(
        columns={
            "node_id":
                "gt_node_id",

            "z":
                "gt_z",

            "y":
                "gt_y",

            "x":
                "gt_x",
        }
    )


    gt_coordinate_parts.append(
        selected
    )


    if (
        i == 1
        or
        i % 10 == 0
        or
        i == len(
            datasets_needed
        )
    ):

        print(
            f"{i:3d}/{len(datasets_needed)} | "
            f"{dataset} | "
            f"missing={len(required_ids)}"
        )


gt_coords = pd.concat(
    gt_coordinate_parts,
    ignore_index=True,
)


assert len(
    gt_coords
) == EXPECTED_MISSING_NODES


missing = node_base.merge(
    gt_coords,
    on=[
        "dataset",
        "gt_node_id",
    ],
    how="left",
    validate="one_to_one",
)


assert missing[
    [
        "t",
        "gt_z",
        "gt_y",
        "gt_x",
    ]
].notna().all().all()


missing[
    "t"
] = (
    pd.to_numeric(
        missing[
            "t"
        ],
        errors="raise",
    )
    .astype(
        np.int64
    )
)


print(
    "\nFIRST_NEW_GT_READ_FOR_BRANCH: COMPLETE"
)


print(
    "Fold0 GT read:",
    "NO"
)


# ============================================================
# 10. PREPARE ORACLE OUTPUT COLUMNS
# ============================================================

missing[
    "nearest_candidate_distance_um"
] = np.inf


for radius in RADII:

    tag = radius_name(
        radius
    )


    missing[
        f"candidate_count_within_{tag}"
    ] = 0


    missing[
        f"covered_any_{tag}"
    ] = False


# Candidate-side oracle fields.
candidate_audit = (
    candidates[
        [
            "candidate_id",

            "dataset",
            "fold",
            "prefix",

            "t",

            "z",
            "y",
            "x",

            "detector_prob",
            "detector_logit",
        ]
    ]
    .copy()
)


candidate_audit[
    "nearest_missing_distance_um"
] = np.inf


for radius in RADII:

    tag = radius_name(
        radius
    )


    candidate_audit[
        f"missing_count_within_{tag}"
    ] = 0


    candidate_audit[
        f"oracle_hit_{tag}"
    ] = False


# ============================================================
# 11. FRAME-LOCAL DISTANCE AUDIT
# ============================================================

print(
    "\n========== FRAME-LOCAL ORACLE DISTANCES =========="
)


missing_groups = (
    missing
    .groupby(
        [
            "dataset",
            "t",
        ],
        sort=True,
    )
    .groups
)


candidate_groups = (
    candidate_audit
    .groupby(
        [
            "dataset",
            "t",
        ],
        sort=True,
    )
    .groups
)


all_frame_keys = sorted(
    set(
        missing_groups.keys()
    )
    |
    set(
        candidate_groups.keys()
    )
)


frames_with_missing = 0

frames_with_candidates_and_missing = 0


for frame_i, key in enumerate(
    all_frame_keys,
    start=1,
):

    missing_idx = missing_groups.get(
        key,
        [],
    )


    candidate_idx = candidate_groups.get(
        key,
        [],
    )


    if len(
        missing_idx
    ) == 0:

        continue


    frames_with_missing += 1


    missing_idx = np.asarray(
        list(
            missing_idx
        ),
        dtype=np.int64,
    )


    if len(
        candidate_idx
    ) == 0:

        continue


    frames_with_candidates_and_missing += 1


    candidate_idx = np.asarray(
        list(
            candidate_idx
        ),
        dtype=np.int64,
    )


    m_xyz = (
        missing.loc[
            missing_idx,
            [
                "gt_z",
                "gt_y",
                "gt_x",
            ],
        ]
        .to_numpy(
            dtype=np.float64
        )
        *
        SCALE[
            None,
            :
        ]
    )


    c_xyz = (
        candidate_audit.loc[
            candidate_idx,
            [
                "z",
                "y",
                "x",
            ],
        ]
        .to_numpy(
            dtype=np.float64
        )
        *
        SCALE[
            None,
            :
        ]
    )


    c_tree = cKDTree(
        c_xyz
    )


    m_tree = cKDTree(
        m_xyz
    )


    # --------------------------------------------------------
    # Missing -> nearest candidate
    # --------------------------------------------------------

    nearest_m, _ = c_tree.query(
        m_xyz,
        k=1,
    )


    missing.loc[
        missing_idx,
        "nearest_candidate_distance_um",
    ] = nearest_m


    # --------------------------------------------------------
    # Candidate -> nearest missing
    # --------------------------------------------------------

    nearest_c, _ = m_tree.query(
        c_xyz,
        k=1,
    )


    candidate_audit.loc[
        candidate_idx,
        "nearest_missing_distance_um",
    ] = nearest_c


    # --------------------------------------------------------
    # Radius multiplicities
    # --------------------------------------------------------

    for radius in RADII:

        tag = radius_name(
            radius
        )


        # Missing -> number of candidates.
        try:

            m_counts = c_tree.query_ball_point(
                m_xyz,
                r=radius,
                return_length=True,
            )

        except TypeError:

            m_counts = np.asarray(
                [
                    len(
                        x
                    )

                    for x
                    in c_tree.query_ball_point(
                        m_xyz,
                        r=radius,
                    )
                ],
                dtype=np.int64,
            )


        m_counts = np.asarray(
            m_counts,
            dtype=np.int64,
        )


        missing.loc[
            missing_idx,
            f"candidate_count_within_{tag}",
        ] = m_counts


        missing.loc[
            missing_idx,
            f"covered_any_{tag}",
        ] = (
            m_counts
            >
            0
        )


        # Candidate -> number of missing GT nodes.
        try:

            c_counts = m_tree.query_ball_point(
                c_xyz,
                r=radius,
                return_length=True,
            )

        except TypeError:

            c_counts = np.asarray(
                [
                    len(
                        x
                    )

                    for x
                    in m_tree.query_ball_point(
                        c_xyz,
                        r=radius,
                    )
                ],
                dtype=np.int64,
            )


        c_counts = np.asarray(
            c_counts,
            dtype=np.int64,
        )


        candidate_audit.loc[
            candidate_idx,
            f"missing_count_within_{tag}",
        ] = c_counts


        candidate_audit.loc[
            candidate_idx,
            f"oracle_hit_{tag}",
        ] = (
            c_counts
            >
            0
        )


print(
    "frames with missing nodes:",
    frames_with_missing
)


print(
    "frames with both missing nodes and candidates:",
    frames_with_candidates_and_missing
)


# ============================================================
# 12. ONE-TO-ONE MAXIMUM MATCHING CEILING
#
# Per dataset/frame.
# ============================================================

matching_rows = []


print(
    "\n========== ONE-TO-ONE MAXIMUM MATCHING =========="
)


for key in sorted(
    missing_groups.keys()
):

    dataset, t = key


    missing_idx = np.asarray(
        list(
            missing_groups[
                key
            ]
        ),
        dtype=np.int64,
    )


    candidate_idx = np.asarray(
        list(
            candidate_groups.get(
                key,
                [],
            )
        ),
        dtype=np.int64,
    )


    n_missing = int(
        len(
            missing_idx
        )
    )


    n_candidates = int(
        len(
            candidate_idx
        )
    )


    if n_candidates > 0:

        m_xyz = (
            missing.loc[
                missing_idx,
                [
                    "gt_z",
                    "gt_y",
                    "gt_x",
                ],
            ]
            .to_numpy(
                dtype=np.float64
            )
            *
            SCALE[
                None,
                :
            ]
        )


        c_xyz = (
            candidate_audit.loc[
                candidate_idx,
                [
                    "z",
                    "y",
                    "x",
                ],
            ]
            .to_numpy(
                dtype=np.float64
            )
            *
            SCALE[
                None,
                :
            ]
        )


        c_tree = cKDTree(
            c_xyz
        )


    else:

        m_xyz = None

        c_xyz = None

        c_tree = None


    for radius in RADII:

        if n_candidates == 0:

            matched_count = 0

            admissible_edges = 0


        else:

            neighbors = c_tree.query_ball_point(
                m_xyz,
                r=radius,
            )


            row_ids = []

            col_ids = []


            for local_m, cols in enumerate(
                neighbors
            ):

                if not cols:

                    continue


                row_ids.extend(
                    [
                        local_m
                    ]
                    *
                    len(
                        cols
                    )
                )


                col_ids.extend(
                    cols
                )


            admissible_edges = int(
                len(
                    row_ids
                )
            )


            if admissible_edges == 0:

                matched_count = 0


            else:

                biadj = csr_matrix(
                    (
                        np.ones(
                            admissible_edges,
                            dtype=np.int8,
                        ),
                        (
                            np.asarray(
                                row_ids,
                                dtype=np.int64,
                            ),
                            np.asarray(
                                col_ids,
                                dtype=np.int64,
                            ),
                        ),
                    ),
                    shape=(
                        n_missing,
                        n_candidates,
                    ),
                )


                match = maximum_bipartite_matching(
                    biadj,
                    perm_type="column",
                )


                # For perm_type="column", expected one entry
                # per left-side (missing-node) row.
                if len(
                    match
                ) != n_missing:

                    raise RuntimeError(
                        "Unexpected scipy "
                        "maximum_bipartite_matching semantics: "
                        f"len(match)={len(match)}, "
                        f"n_missing={n_missing}"
                    )


                matched_count = int(
                    np.sum(
                        np.asarray(
                            match
                        )
                        >=
                        0
                    )
                )


        matching_rows.append(
            {
                "dataset":
                    dataset,

                "prefix":
                    prefix_of(
                        dataset
                    ),

                "fold":
                    int(
                        missing.loc[
                            missing_idx[
                                0
                            ],
                            "fold",
                        ]
                    ),

                "t":
                    int(
                        t
                    ),

                "radius_um":
                    float(
                        radius
                    ),

                "missing_nodes":
                    n_missing,

                "candidate_nodes":
                    n_candidates,

                "admissible_pairs":
                    admissible_edges,

                "maximum_one_to_one_matches":
                    matched_count,
            }
        )


matching = pd.DataFrame(
    matching_rows
)


# ============================================================
# 13. MISSING-NODE COVERAGE SUMMARY
# ============================================================

print(
    "\n========== MISSING-NODE COVERAGE =========="
)


coverage_rows = []


for radius in RADII:

    tag = radius_name(
        radius
    )


    covered = int(
        missing[
            f"covered_any_{tag}"
        ].sum()
    )


    one_to_one = int(
        matching.loc[
            matching[
                "radius_um"
            ].eq(
                radius
            ),
            "maximum_one_to_one_matches",
        ].sum()
    )


    assert one_to_one <= covered


    assert covered <= EXPECTED_MISSING_NODES


    coverage_rows.append(
        {
            "radius_um":
                radius,

            "any_candidate_covered_nodes":
                covered,

            "any_candidate_coverage":
                covered
                /
                EXPECTED_MISSING_NODES,

            "maximum_one_to_one_matches":
                one_to_one,

            "one_to_one_coverage":
                one_to_one
                /
                EXPECTED_MISSING_NODES,
        }
    )


    print(
        f"R={radius:.0f} um | "
        f"any={covered}/{EXPECTED_MISSING_NODES} "
        f"({covered/EXPECTED_MISSING_NODES:.3%}) | "
        f"1-to-1={one_to_one}/{EXPECTED_MISSING_NODES} "
        f"({one_to_one/EXPECTED_MISSING_NODES:.3%})"
    )


coverage = pd.DataFrame(
    coverage_rows
)


# ============================================================
# 14. NEAREST DISTANCE DISTRIBUTION
# ============================================================

finite_distances = (
    missing.loc[
        np.isfinite(
            missing[
                "nearest_candidate_distance_um"
            ]
        ),
        "nearest_candidate_distance_um",
    ]
)


distance_quantiles = {
    str(
        q
    ):
        float(
            v
        )

    for q, v
    in finite_distances.quantile(
        [
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    ).items()
}


print(
    "\n========== NEAREST-CANDIDATE DISTANCE =========="
)


print(
    "finite missing nodes:",
    len(
        finite_distances
    ),
    "/",
    EXPECTED_MISSING_NODES,
)


for q, value in distance_quantiles.items():

    print(
        f"q={q}: "
        f"{value:.6f} um"
    )


# ============================================================
# 15. EDGE-LEVEL RECONSTRUCTABILITY
#
# A detection FN is geometrically reconstructable at radius R
# only if ALL missing endpoints of that edge have coverage.
# ============================================================

missing_distance_map = {
    (
        str(
            row.dataset
        ),
        int(
            row.gt_node_id
        ),
    ):
        float(
            row.nearest_candidate_distance_um
        )

    for row
    in missing.itertuples(
        index=False
    )
}


edge_audit = det.copy()


edge_audit[
    "source_missing_nearest_candidate_um"
] = np.nan


edge_audit[
    "target_missing_nearest_candidate_um"
] = np.nan


for idx, row in edge_audit.iterrows():

    dataset = str(
        row[
            "dataset"
        ]
    )


    if bool(
        row[
            "source_missing"
        ]
    ):

        edge_audit.loc[
            idx,
            "source_missing_nearest_candidate_um",
        ] = missing_distance_map[
            (
                dataset,
                int(
                    row[
                        "gt_source_id"
                    ]
                ),
            )
        ]


    if bool(
        row[
            "target_missing"
        ]
    ):

        edge_audit.loc[
            idx,
            "target_missing_nearest_candidate_um",
        ] = missing_distance_map[
            (
                dataset,
                int(
                    row[
                        "gt_target_id"
                    ]
                ),
            )
        ]


for radius in RADII:

    tag = radius_name(
        radius
    )


    reconstructable = []


    for row in edge_audit.itertuples(
        index=False
    ):

        flags = []


        if bool(
            row.source_missing
        ):

            flags.append(
                float(
                    row.source_missing_nearest_candidate_um
                )
                <=
                radius
            )


        if bool(
            row.target_missing
        ):

            flags.append(
                float(
                    row.target_missing_nearest_candidate_um
                )
                <=
                radius
            )


        assert flags


        reconstructable.append(
            all(
                flags
            )
        )


    edge_audit[
        f"all_missing_endpoints_covered_{tag}"
    ] = reconstructable


print(
    "\n========== EDGE-LEVEL GEOMETRIC CEILING =========="
)


for radius in RADII:

    tag = radius_name(
        radius
    )


    print(
        f"\nR={radius:.0f} um"
    )


    for cls in [
        "SOURCE_MISSING",
        "TARGET_MISSING",
        "BOTH_MISSING",
    ]:

        sub = edge_audit[
            edge_audit[
                "missing_class"
            ].eq(
                cls
            )
        ]


        n = int(
            sub[
                f"all_missing_endpoints_covered_{tag}"
            ].sum()
        )


        print(
            f"  {cls:16s}: "
            f"{n}/{len(sub)} "
            f"({n/len(sub):.3%})"
        )


    all_n = int(
        edge_audit[
            f"all_missing_endpoints_covered_{tag}"
        ].sum()
    )


    print(
        "  ALL:",
        f"{all_n}/{len(edge_audit)} "
        f"({all_n/len(edge_audit):.3%})"
    )


# ============================================================
# 16. COMPONENT COVERAGE
# ============================================================

node_component = (
    missing[
        [
            "dataset",
            "gt_node_id",
            "component_key",
            "fold",
            "prefix",
        ]
        +
        [
            f"covered_any_{radius_name(r)}"

            for r in RADII
        ]
    ]
    .copy()
)


component_audit = (
    components
    .copy()
)


component_fold = (
    node_component[
        [
            "component_key",
            "fold",
        ]
    ]
    .drop_duplicates()
)


assert not component_fold[
    "component_key"
].duplicated().any()


component_audit = component_audit.merge(
    component_fold,
    on="component_key",
    how="left",
    validate="one_to_one",
)


for radius in RADII:

    tag = radius_name(
        radius
    )


    grouped = (
        node_component
        .groupby(
            "component_key"
        )[
            f"covered_any_{tag}"
        ]
        .agg(
            [
                "any",
                "all",
                "sum",
            ]
        )
    )


    component_audit = component_audit.merge(
        grouped.rename(
            columns={
                "any":
                    f"any_node_covered_{tag}",

                "all":
                    f"all_nodes_covered_{tag}",

                "sum":
                    f"covered_nodes_{tag}",
            }
        ),
        left_on="component_key",
        right_index=True,
        how="left",
        validate="one_to_one",
    )


print(
    "\n========== COMPONENT COVERAGE =========="
)


for radius in RADII:

    tag = radius_name(
        radius
    )


    single = component_audit[
        component_audit[
            "missing_nodes"
        ].eq(
            1
        )
    ]


    multi = component_audit[
        component_audit[
            "missing_nodes"
        ].ge(
            2
        )
    ]


    single_full = int(
        single[
            f"all_nodes_covered_{tag}"
        ].sum()
    )


    multi_any = int(
        multi[
            f"any_node_covered_{tag}"
        ].sum()
    )


    multi_full = int(
        multi[
            f"all_nodes_covered_{tag}"
        ].sum()
    )


    print(
        f"\nR={radius:.0f} um"
    )


    print(
        "  single full:",
        f"{single_full}/{len(single)} "
        f"({single_full/len(single):.3%})"
    )


    print(
        "  multi any :",
        f"{multi_any}/{len(multi)} "
        f"({multi_any/len(multi):.3%})"
    )


    print(
        "  multi full:",
        f"{multi_full}/{len(multi)} "
        f"({multi_full/len(multi):.3%})"
    )


# ============================================================
# 17. PER-FOLD / PREFIX COVERAGE @ 7 UM
# ============================================================

R7 = "7p0um"


print(
    "\n========== 7 UM COVERAGE BY FOLD =========="
)


fold_report = (
    missing
    .groupby(
        "fold"
    )
    .agg(
        missing_nodes=(
            "gt_node_id",
            "size",
        ),

        covered_nodes=(
            f"covered_any_{R7}",
            "sum",
        ),
    )
)


fold_report[
    "coverage"
] = (
    fold_report[
        "covered_nodes"
    ]
    /
    fold_report[
        "missing_nodes"
    ]
)


print(
    fold_report.to_string()
)


print(
    "\n========== 7 UM COVERAGE BY PREFIX =========="
)


prefix_report = (
    missing
    .groupby(
        "prefix"
    )
    .agg(
        missing_nodes=(
            "gt_node_id",
            "size",
        ),

        covered_nodes=(
            f"covered_any_{R7}",
            "sum",
        ),
    )
)


prefix_report[
    "coverage"
] = (
    prefix_report[
        "covered_nodes"
    ]
    /
    prefix_report[
        "missing_nodes"
    ]
)


print(
    prefix_report.to_string()
)


# ============================================================
# 18. MULTIPLICITY AUDIT @ 7 UM
# ============================================================

print(
    "\n========== 7 UM MULTIPLICITY =========="
)


missing_mult = (
    missing[
        f"candidate_count_within_{R7}"
    ]
    .value_counts()
    .sort_index()
)


print(
    "candidates per missing node:"
)


print(
    missing_mult.head(
        20
    ).to_string()
)


candidate_mult = (
    candidate_audit[
        f"missing_count_within_{R7}"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nmissing nodes per candidate:"
)


print(
    candidate_mult.head(
        20
    ).to_string()
)


candidate_hits_7 = int(
    candidate_audit[
        f"oracle_hit_{R7}"
    ].sum()
)


print(
    "\ncandidates within 7 um of >=1 missing node:",
    candidate_hits_7,
    "/",
    len(
        candidate_audit
    ),
    f"({candidate_hits_7/len(candidate_audit):.3%})"
)


# ============================================================
# 19. DETECTOR-PROBABILITY ORACLE AUDIT
#
# AUDIT ONLY. DO NOT SELECT THRESHOLD HERE.
# ============================================================

candidate_audit[
    "oracle_hit_7um"
] = (
    candidate_audit[
        f"oracle_hit_{R7}"
    ]
)


hit_prob = (
    candidate_audit.loc[
        candidate_audit[
            "oracle_hit_7um"
        ],
        "detector_prob",
    ]
)


nonhit_prob = (
    candidate_audit.loc[
        ~candidate_audit[
            "oracle_hit_7um"
        ],
        "detector_prob",
    ]
)


prob_audit = {
    "hit_count":
        int(
            len(
                hit_prob
            )
        ),

    "nonhit_count":
        int(
            len(
                nonhit_prob
            )
        ),

    "hit_median":
        (
            float(
                hit_prob.median()
            )
            if len(
                hit_prob
            )
            else None
        ),

    "nonhit_median":
        (
            float(
                nonhit_prob.median()
            )
            if len(
                nonhit_prob
            )
            else None
        ),

    "hit_quantiles":
        {
            str(
                q
            ):
                float(
                    v
                )

            for q, v
            in hit_prob.quantile(
                [
                    0.10,
                    0.25,
                    0.50,
                    0.75,
                    0.90,
                ]
            ).items()
        },

    "nonhit_quantiles":
        {
            str(
                q
            ):
                float(
                    v
                )

            for q, v
            in nonhit_prob.quantile(
                [
                    0.10,
                    0.25,
                    0.50,
                    0.75,
                    0.90,
                ]
            ).items()
        },
}


print(
    "\n========== DETECTOR PROBABILITY — ORACLE AUDIT ONLY =========="
)


print(
    "7um-hit candidates:",
    prob_audit[
        "hit_count"
    ]
)


print(
    "non-hit candidates:",
    prob_audit[
        "nonhit_count"
    ]
)


print(
    "hit median detector_prob:",
    prob_audit[
        "hit_median"
    ]
)


print(
    "non-hit median detector_prob:",
    prob_audit[
        "nonhit_median"
    ]
)


print(
    "THRESHOLD SELECTED:",
    "NO"
)


# ============================================================
# 20. SAVE FORMAL ARTIFACTS
# ============================================================

missing = (
    missing
    .sort_values(
        [
            "dataset",
            "t",
            "gt_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


candidate_audit = (
    candidate_audit
    .sort_values(
        [
            "dataset",
            "t",
            "candidate_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


edge_audit = (
    edge_audit
    .sort_values(
        [
            "dataset",
            "gt_source_id",
            "gt_target_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


component_audit = (
    component_audit
    .sort_values(
        [
            "dataset",
            "root_gt_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


matching = (
    matching
    .sort_values(
        [
            "radius_um",
            "dataset",
            "t",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


missing.to_pickle(
    MISSING_NODE_OUT
)


candidate_audit.to_pickle(
    CANDIDATE_ORACLE_OUT
)


edge_audit.to_pickle(
    EDGE_ORACLE_OUT
)


component_audit.to_pickle(
    COMPONENT_OUT
)


matching.to_pickle(
    MATCHING_OUT
)


# ============================================================
# 21. FILE SHAS
# ============================================================

MISSING_SHA = sha256_file(
    MISSING_NODE_OUT
)


CANDIDATE_ORACLE_SHA = sha256_file(
    CANDIDATE_ORACLE_OUT
)


EDGE_SHA = sha256_file(
    EDGE_ORACLE_OUT
)


COMPONENT_SHA = sha256_file(
    COMPONENT_OUT
)


MATCHING_SHA = sha256_file(
    MATCHING_OUT
)


FN_SOURCE_SHA = sha256_file(
    FN_PATH
)


# ============================================================
# 22. FORMAL SUMMARY
# ============================================================

coverage_summary = {
    str(
        int(
            row.radius_um
        )
    ):
        {
            "radius_um":
                float(
                    row.radius_um
                ),

            "any_candidate_covered_nodes":
                int(
                    row.any_candidate_covered_nodes
                ),

            "any_candidate_coverage":
                float(
                    row.any_candidate_coverage
                ),

            "maximum_one_to_one_matches":
                int(
                    row.maximum_one_to_one_matches
                ),

            "one_to_one_coverage":
                float(
                    row.one_to_one_coverage
                ),
        }

    for row
    in coverage.itertuples(
        index=False
    )
}


edge_class_summary = {}


for radius in RADII:

    tag = radius_name(
        radius
    )


    per_class = {}


    for cls in [
        "SOURCE_MISSING",
        "TARGET_MISSING",
        "BOTH_MISSING",
    ]:

        sub = edge_audit[
            edge_audit[
                "missing_class"
            ].eq(
                cls
            )
        ]


        n = int(
            sub[
                f"all_missing_endpoints_covered_{tag}"
            ].sum()
        )


        per_class[
            cls
        ] = {
            "edges":
                int(
                    len(
                        sub
                    )
                ),

            "geometrically_reconstructable":
                n,

            "fraction":
                float(
                    n
                    /
                    len(
                        sub
                    )
                ),
        }


    n_all = int(
        edge_audit[
            f"all_missing_endpoints_covered_{tag}"
        ].sum()
    )


    per_class[
        "ALL"
    ] = {
        "edges":
            int(
                len(
                    edge_audit
                )
            ),

        "geometrically_reconstructable":
            n_all,

        "fraction":
            float(
                n_all
                /
                len(
                    edge_audit
                )
            ),
    }


    edge_class_summary[
        str(
            int(
                radius
            )
        )
    ] = per_class


SUMMARY = {
    "stage":
        "12.12A3",

    "status":
        "DEVELOPMENT_ORACLE_NODE_CANDIDATE_COVERAGE_AUDITED",

    "branch_id":
        BRANCH_ID,

    "purpose":
        "DEVELOPMENT_ORACLE_COVERAGE_ONLY",

    "parent_candidate_universe": {
        "contract_sha256":
            A2_CONTRACT_SHA,

        "content_sha256":
            A2_CONTENT_SHA,

        "file_sha256":
            A2_FILE_SHA,

        "candidates":
            EXPECTED_CANDIDATES,

        "frozen_before_oracle":
            True,
    },

    "residual_source": {
        "artifact":
            FN_PATH.name,

        "artifact_sha256":
            FN_SOURCE_SHA,

        "pre_g3h3_edge_fn":
            EXPECTED_OLD_FN,

        "detection_limited_identity_invariant_after_g3h3":
            True,

        "detection_limited_edge_fn":
            EXPECTED_DETECTION_FN,

        "missing_gt_nodes":
            EXPECTED_MISSING_NODES,

        "components":
            EXPECTED_COMPONENTS,
    },

    "coverage": {
        "node_level":
            coverage_summary,

        "edge_level":
            edge_class_summary,

        "nearest_candidate_distance_quantiles_um":
            distance_quantiles,
    },

    "candidate_oracle": {
        "candidate_rows":
            int(
                len(
                    candidate_audit
                )
            ),

        "candidates_with_missing_node_within_7um":
            candidate_hits_7,

        "fraction_with_missing_node_within_7um":
            float(
                candidate_hits_7
                /
                len(
                    candidate_audit
                )
            ),

        "detector_probability_audit_only":
            prob_audit,
    },

    "artifacts": {
        "missing_node_audit":
            {
                "file":
                    MISSING_NODE_OUT.name,

                "sha256":
                    MISSING_SHA,
            },

        "candidate_oracle_audit":
            {
                "file":
                    CANDIDATE_ORACLE_OUT.name,

                "sha256":
                    CANDIDATE_ORACLE_SHA,
            },

        "detection_edge_audit":
            {
                "file":
                    EDGE_ORACLE_OUT.name,

                "sha256":
                    EDGE_SHA,
            },

        "component_audit":
            {
                "file":
                    COMPONENT_OUT.name,

                "sha256":
                    COMPONENT_SHA,
            },

        "matching_ceiling":
            {
                "file":
                    MATCHING_OUT.name,

                "sha256":
                    MATCHING_SHA,
            },
    },

    "protocol": {
        "candidate_universe_modified":
            False,

        "candidate_generation_after_oracle":
            False,

        "development_gt_used":
            True,

        "fold0_used":
            False,

        "official_graph_metric_evaluated":
            False,

        "detector_threshold_selected":
            False,

        "scorer_selected":
            False,

        "feature_subset_selected":
            False,

        "association_policy_selected":
            False,

        "graph_modified":
            False,

        "r1p50_retuned":
            False,

        "g3h3_reopened":
            False,
    },

    "decision_rule_for_next_stage":
        (
            "A3 REPORTS COVERAGE ONLY. "
            "NO AUTOMATIC THRESHOLD OR POLICY SELECTION. "
            "Use development coverage and failure anatomy to "
            "declare A4 feature/scorer families explicitly "
            "before graph-level policy evaluation."
        ),

    "next":
        (
            "12.12A4_PREDECLARE_GT_BLIND_NODE_"
            "SCORING_AND_ATTACHMENT_FEATURE_FAMILIES"
        ),
}


SUMMARY_SHA = hashlib.sha256(
    json.dumps(
        SUMMARY,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


SUMMARY[
    "summary_sha256"
] = SUMMARY_SHA


SUMMARY_OUT.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 23. RELOAD INTEGRITY
# ============================================================

summary_check = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert summary_check[
    "summary_sha256"
] == SUMMARY_SHA


assert summary_check[
    "status"
] == (
    "DEVELOPMENT_ORACLE_NODE_CANDIDATE_COVERAGE_AUDITED"
)


assert summary_check[
    "parent_candidate_universe"
][
    "contract_sha256"
] == A2_CONTRACT_SHA


assert summary_check[
    "parent_candidate_universe"
][
    "frozen_before_oracle"
] is True


assert summary_check[
    "residual_source"
][
    "missing_gt_nodes"
] == EXPECTED_MISSING_NODES


assert summary_check[
    "protocol"
][
    "development_gt_used"
] is True


assert summary_check[
    "protocol"
][
    "fold0_used"
] is False


assert summary_check[
    "protocol"
][
    "detector_threshold_selected"
] is False


assert summary_check[
    "protocol"
][
    "scorer_selected"
] is False


assert summary_check[
    "protocol"
][
    "graph_modified"
] is False


# ============================================================
# 24. FINAL DECISION
# ============================================================

r7_row = coverage[
    coverage[
        "radius_um"
    ].eq(
        7.0
    )
].iloc[
    0
]


print(
    "\n========== 12.12A3 FINAL DECISION =========="
)


print(
    "DEVELOPMENT_ORACLE_NODE_CANDIDATE_COVERAGE_AUDIT: PASS"
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "frozen A2 candidates:",
    EXPECTED_CANDIDATES
)


print(
    "missing GT nodes:",
    EXPECTED_MISSING_NODES
)


print(
    "detection-limited edge FN:",
    EXPECTED_DETECTION_FN
)


print(
    "7um any-candidate node coverage:",
    (
        f"{int(r7_row['any_candidate_covered_nodes'])}"
        f"/{EXPECTED_MISSING_NODES}"
    ),
    (
        f"({float(r7_row['any_candidate_coverage']):.3%})"
    ),
)


print(
    "7um one-to-one maximum matching:",
    (
        f"{int(r7_row['maximum_one_to_one_matches'])}"
        f"/{EXPECTED_MISSING_NODES}"
    ),
    (
        f"({float(r7_row['one_to_one_coverage']):.3%})"
    ),
)


print(
    "7um oracle-positive candidates:",
    candidate_hits_7,
    "/",
    EXPECTED_CANDIDATES,
    (
        f"({candidate_hits_7/EXPECTED_CANDIDATES:.3%})"
    ),
)


print(
    "summary SHA:",
    SUMMARY_SHA
)


print(
    "development GT used:",
    "YES"
)


print(
    "candidate universe modified:",
    "NO"
)


print(
    "detector threshold selected:",
    "NO"
)


print(
    "scorer selected:",
    "NO"
)


print(
    "association policy selected:",
    "NO"
)


print(
    "official graph metric evaluated:",
    "NO"
)


print(
    "graph modified:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "R1P50 retuned:",
    "NO"
)


print(
    "G3H3 reopened:",
    "NO"
)


print(
    "next:"
)


print(
    "12.12A4_PREDECLARE_GT_BLIND_NODE_"
    "SCORING_AND_ATTACHMENT_FEATURE_FAMILIES"
)

In [ ]:
# ============================================================
# Stage 12.12A3-R1B
#
# READ EXISTING A3 SCIENTIFIC RESULTS
#
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
#
# A3 has already been proven COMPLETE.
#
# This cell only reads:
#
#   - existing A3 summary
#   - existing A3 missing-node audit
#   - existing A3 candidate audit
#   - existing A3 component audit
#   - existing A3 matching ceiling
#
# NO:
#
#   GT read
#   evaluator
#   graph mutation
#   candidate generation
#   threshold selection
#   scorer selection
#   Fold0
#
# Delete after A4 contract is frozen: YES
# ============================================================

from pathlib import Path
import json
import pandas as pd


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


SUMMARY_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_coverage_v1_summary.json"
)

MISSING_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_missing_nodes_v1.pkl"
)

CANDIDATE_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_candidates_v1.pkl"
)

EDGE_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_detection_edges_v1.pkl"
)

COMPONENT_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_components_v1.pkl"
)

MATCHING_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_matching_ceiling_v1.pkl"
)


for path in [
    SUMMARY_PATH,
    MISSING_PATH,
    CANDIDATE_PATH,
    EDGE_PATH,
    COMPONENT_PATH,
    MATCHING_PATH,
]:
    assert path.exists(), path


# ============================================================
# 1. SUMMARY LOCK
# ============================================================

summary = json.loads(
    SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

EXPECTED_SUMMARY_SHA = (
    "89eae2f7710cf68dd874a14ecdec788d"
    "75273c698815b082df479c0c789a0363"
)

assert summary[
    "summary_sha256"
] == EXPECTED_SUMMARY_SHA

assert summary[
    "status"
] == (
    "DEVELOPMENT_ORACLE_NODE_CANDIDATE_COVERAGE_AUDITED"
)

assert summary[
    "purpose"
] == (
    "DEVELOPMENT_ORACLE_COVERAGE_ONLY"
)

assert summary[
    "protocol"
][
    "development_gt_used"
] is True

assert summary[
    "protocol"
][
    "fold0_used"
] is False

assert summary[
    "protocol"
][
    "official_graph_metric_evaluated"
] is False

assert summary[
    "protocol"
][
    "detector_threshold_selected"
] is False

assert summary[
    "protocol"
][
    "scorer_selected"
] is False

assert summary[
    "protocol"
][
    "graph_modified"
] is False


print(
    "========== 12.12A3-R1B SUMMARY LOCK =========="
)

print(
    "summary SHA:",
    summary[
        "summary_sha256"
    ]
)

print(
    "A3 status:",
    summary[
        "status"
    ]
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "threshold selected:",
    "NO"
)

print(
    "scorer selected:",
    "NO"
)


# ============================================================
# 2. NODE-LEVEL COVERAGE
# ============================================================

print(
    "\n========== A3 NODE-LEVEL COVERAGE =========="
)

coverage = summary[
    "coverage"
][
    "node_level"
]

for radius in [
    "2",
    "3",
    "5",
    "7",
]:

    row = coverage[
        radius
    ]

    print(
        f"R={radius} um"
    )

    print(
        "  any-candidate:",
        row[
            "any_candidate_covered_nodes"
        ],
        "/ 2985",
        f"({row['any_candidate_coverage']:.3%})",
    )

    print(
        "  one-to-one:",
        row[
            "maximum_one_to_one_matches"
        ],
        "/ 2985",
        f"({row['one_to_one_coverage']:.3%})",
    )


# ============================================================
# 3. NEAREST-CANDIDATE DISTANCE
# ============================================================

print(
    "\n========== A3 NEAREST-CANDIDATE DISTANCE =========="
)

for q, value in summary[
    "coverage"
][
    "nearest_candidate_distance_quantiles_um"
].items():

    print(
        f"q={q}:",
        f"{float(value):.6f} um",
    )


# ============================================================
# 4. EDGE-LEVEL GEOMETRIC CEILING
# ============================================================

print(
    "\n========== A3 EDGE-LEVEL GEOMETRIC CEILING =========="
)

edge_level = summary[
    "coverage"
][
    "edge_level"
]

for radius in [
    "2",
    "3",
    "5",
    "7",
]:

    print(
        f"\nR={radius} um"
    )

    rows = edge_level[
        radius
    ]

    for cls in [
        "SOURCE_MISSING",
        "TARGET_MISSING",
        "BOTH_MISSING",
        "ALL",
    ]:

        row = rows[
            cls
        ]

        print(
            f"  {cls:16s}: "
            f"{row['geometrically_reconstructable']}"
            f"/{row['edges']} "
            f"({row['fraction']:.3%})"
        )


# ============================================================
# 5. CANDIDATE RARITY
# ============================================================

candidate_info = summary[
    "candidate_oracle"
]

print(
    "\n========== A3 CANDIDATE ORACLE RARITY =========="
)

print(
    "candidate rows:",
    candidate_info[
        "candidate_rows"
    ]
)

print(
    "7um oracle-positive candidates:",
    candidate_info[
        "candidates_with_missing_node_within_7um"
    ]
)

print(
    "7um positive fraction:",
    f"{candidate_info['fraction_with_missing_node_within_7um']:.6%}",
)


# ============================================================
# 6. DETECTOR-PROBABILITY AUDIT
# ============================================================

prob = candidate_info[
    "detector_probability_audit_only"
]

print(
    "\n========== A3 DETECTOR PROBABILITY AUDIT =========="
)

print(
    "hit count:",
    prob[
        "hit_count"
    ]
)

print(
    "non-hit count:",
    prob[
        "nonhit_count"
    ]
)

print(
    "hit median:",
    prob[
        "hit_median"
    ]
)

print(
    "non-hit median:",
    prob[
        "nonhit_median"
    ]
)

print(
    "\nhit quantiles:"
)

for q, value in prob[
    "hit_quantiles"
].items():

    print(
        f"  q={q}:",
        value
    )

print(
    "\nnon-hit quantiles:"
)

for q, value in prob[
    "nonhit_quantiles"
].items():

    print(
        f"  q={q}:",
        value
    )

print(
    "\nTHRESHOLD SELECTED:",
    "NO"
)


# ============================================================
# 7. EXISTING CHILD ARTIFACTS
# ============================================================

missing = pd.read_pickle(
    MISSING_PATH
)

candidate = pd.read_pickle(
    CANDIDATE_PATH
)

edges = pd.read_pickle(
    EDGE_PATH
)

components = pd.read_pickle(
    COMPONENT_PATH
)

matching = pd.read_pickle(
    MATCHING_PATH
)


assert len(missing) == 2985
assert len(candidate) == 415590
assert len(edges) == 4097
assert len(components) == 1595


# ============================================================
# 8. 7 UM COVERAGE BY FOLD
# ============================================================

print(
    "\n========== A3 7 UM COVERAGE BY FOLD =========="
)

fold_report = (
    missing
    .groupby(
        "fold"
    )
    .agg(
        missing_nodes=(
            "gt_node_id",
            "size",
        ),
        covered_nodes=(
            "covered_any_7p0um",
            "sum",
        ),
    )
)

fold_report[
    "coverage"
] = (
    fold_report[
        "covered_nodes"
    ]
    /
    fold_report[
        "missing_nodes"
    ]
)

print(
    fold_report.to_string()
)


# ============================================================
# 9. 7 UM COVERAGE BY PREFIX
# ============================================================

print(
    "\n========== A3 7 UM COVERAGE BY PREFIX =========="
)

prefix_report = (
    missing
    .groupby(
        "prefix"
    )
    .agg(
        missing_nodes=(
            "gt_node_id",
            "size",
        ),
        covered_nodes=(
            "covered_any_7p0um",
            "sum",
        ),
    )
)

prefix_report[
    "coverage"
] = (
    prefix_report[
        "covered_nodes"
    ]
    /
    prefix_report[
        "missing_nodes"
    ]
)

print(
    prefix_report.to_string()
)


# ============================================================
# 10. COMPONENT COVERAGE
# ============================================================

print(
    "\n========== A3 COMPONENT COVERAGE @ 7 UM =========="
)

single = components[
    components[
        "missing_nodes"
    ].eq(
        1
    )
]

multi = components[
    components[
        "missing_nodes"
    ].ge(
        2
    )
]

single_full = int(
    single[
        "all_nodes_covered_7p0um"
    ].sum()
)

multi_any = int(
    multi[
        "any_node_covered_7p0um"
    ].sum()
)

multi_full = int(
    multi[
        "all_nodes_covered_7p0um"
    ].sum()
)

print(
    "single full:",
    single_full,
    "/",
    len(single),
    f"({single_full/len(single):.3%})"
)

print(
    "multi any:",
    multi_any,
    "/",
    len(multi),
    f"({multi_any/len(multi):.3%})"
)

print(
    "multi full:",
    multi_full,
    "/",
    len(multi),
    f"({multi_full/len(multi):.3%})"
)


# ============================================================
# 11. MULTIPLICITY
# ============================================================

print(
    "\n========== A3 7 UM MULTIPLICITY =========="
)

print(
    "\ncandidates per missing GT node:"
)

print(
    missing[
        "candidate_count_within_7p0um"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .to_string()
)

print(
    "\nmissing GT nodes per candidate:"
)

print(
    candidate[
        "missing_count_within_7p0um"
    ]
    .describe(
        percentiles=[
            0.90,
            0.95,
            0.99,
            0.999,
        ]
    )
    .to_string()
)


# ============================================================
# 12. ONE-TO-ONE LOSS
# ============================================================

print(
    "\n========== A3 ONE-TO-ONE AMBIGUITY LOSS =========="
)

for radius in [
    2.0,
    3.0,
    5.0,
    7.0,
]:

    any_covered = int(
        missing[
            f"covered_any_{str(radius).replace('.', 'p')}um"
        ].sum()
    )

    one_to_one = int(
        matching.loc[
            matching[
                "radius_um"
            ].eq(
                radius
            ),
            "maximum_one_to_one_matches",
        ].sum()
    )

    loss = (
        any_covered
        -
        one_to_one
    )

    print(
        f"R={radius:.0f} um | "
        f"any={any_covered} | "
        f"1-to-1={one_to_one} | "
        f"ambiguity_loss={loss} | "
        f"loss_fraction_of_any="
        f"{loss/max(any_covered,1):.3%}"
    )


# ============================================================
# 13. FINAL
# ============================================================

print(
    "\n========== 12.12A3-R1B DECISION INPUT =========="
)

r7 = coverage[
    "7"
]

positive_fraction = float(
    candidate_info[
        "fraction_with_missing_node_within_7um"
    ]
)

any7 = float(
    r7[
        "any_candidate_coverage"
    ]
)

match7 = float(
    r7[
        "one_to_one_coverage"
    ]
)


print(
    "7um any coverage:",
    f"{any7:.3%}"
)

print(
    "7um one-to-one coverage:",
    f"{match7:.3%}"
)

print(
    "candidate oracle-positive rate:",
    f"{positive_fraction:.6%}"
)

print(
    "7um ambiguity gap:",
    f"{(any7-match7):.3%}"
)

print(
    "development GT newly read here:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)

print(
    "artifact modified:",
    "NO"
)

print(
    "next:"
)

print(
    "USE_EXISTING_A3_SIGNAL_TO_FREEZE_12.12A4_"
    "FEATURE_SCORER_ATTACHMENT_FAMILY_CONTRACT"
)

In [ ]:
# ============================================================
# Stage 12.12A4
#
# PREDECLARE / FREEZE
#
# GT-BLIND NODE SCORING
# + TEMPORAL FEATURE
# + ATTACHMENT POLICY FAMILIES
#
# ============================================================
#
# Parent scientific result:
#
#   A2 frozen candidates        = 415,590
#   missing GT nodes            =   2,985
#
#   7um any coverage            = 440 / 2985 = 14.740%
#   7um one-to-one coverage     = 440 / 2985 = 14.740%
#
#   candidate oracle positives  = 516
#   positive fraction           = 0.124161%
#
#   detection-edge ceiling      = 382 / 4097 = 9.324%
#
#   one-to-one ambiguity loss   = 0
#
# ------------------------------------------------------------
# SCIENTIFIC INTERPRETATION
# ------------------------------------------------------------
#
# Candidate generation is the dominant ceiling.
#
# This branch is NOT allowed to:
#
#   expand the A2 universe
#   lower the A2 0.90 detector floor
#   resurrect R1P10 / R1P25
#   retune R1P50
#   use Fold0
#
# A4 therefore investigates only:
#
#   Can GT-blind runtime features rank the small recoverable
#   subset inside the already-frozen 415,590 candidates?
#
# ------------------------------------------------------------
# DEVELOPMENT GT STATUS
# ------------------------------------------------------------
#
# A3 already exposed folds1-4 development GT.
#
# A4 design IS development-GT-informed.
#
# A4 itself:
#
#   reads existing A3 summary only
#   reads no new GT graph
#   computes no feature matrix
#   trains no model
#   selects no scorer
#   selects no threshold
#   modifies no graph
#
# ------------------------------------------------------------
# KEEP AFTER RUNNING: YES
# ============================================================


from pathlib import Path

import hashlib
import json


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


A0_PATH = (
    S12
    / "stage12_post_r1p50_residual_synthetic_node_recovery_v1_frozen.json"
)


A2_SUMMARY_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_universe_v1_summary.json"
)


A3_SUMMARY_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_coverage_v1_summary.json"
)


CURRENT_BEST_PATH = (
    S12
    / "stage12_current_best_checkpoint.post_g3h3e3.json"
)


CONTRACT_OUT = (
    S12
    / "stage12_post_r1p50_node_scoring_attachment_family_v1_contract_frozen.json"
)


SUMMARY_OUT = (
    S12
    / "stage12_post_r1p50_node_scoring_attachment_family_v1_summary.json"
)


for path in [
    A0_PATH,
    A2_SUMMARY_PATH,
    A3_SUMMARY_PATH,
    CURRENT_BEST_PATH,
]:

    assert path.exists(), path


for path in [
    CONTRACT_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\n12.12A4 formal artifact already exists:\n"
            f"{path}\n\n"
            "Do not overwrite predeclared family provenance."
        )


# ============================================================
# 1. LOCKS
# ============================================================

BRANCH_ID = (
    "POST_R1P50_RESIDUAL_SYNTHETIC_NODE_RECOVERY_V1"
)


A0_SHA = (
    "03243381d2c6c33218fda9c94be0462f"
    "ad28733f8360fd37b1ea5e3d622ae760"
)


A2_CONTRACT_SHA = (
    "fbf3f2bcc81d87180837a6170a598296"
    "a5276a62728c8a21d6c4fc0f9a57313b"
)


A2_CONTENT_SHA = (
    "3462a6cd20eee93e55ba7cca8f110654"
    "5c22cb5815ac1e05dde09e04d5cf8e91"
)


A2_SUMMARY_SHA = (
    "5b708ef71cb26bf7178db8b5b53663f7"
    "fbfb2a59cc23f2898b10f1b7c9f4062f"
)


A3_SUMMARY_SHA = (
    "89eae2f7710cf68dd874a14ecdec788d"
    "75273c698815b082df479c0c789a0363"
)


A3_SUMMARY_FILE_SHA = (
    "7ddb7343f05fce51afc6ef2000d80ac"
    "5e50299f5620161bf7b31cd68b6e50351"
)


CURRENT_BEST_SHA = (
    "65d408c0a8c6089e1d008d0c2c69fb4a"
    "76914c0bc1138486e2845f2e835562a4"
)


EXPECTED_CANDIDATES = 415_590

EXPECTED_POSITIVE_CANDIDATES = 516

EXPECTED_MISSING_NODES = 2_985

EXPECTED_NODE_COVERED_7 = 440

EXPECTED_EDGE_FN = 4_097

EXPECTED_EDGE_COVERED_7 = 382


# ============================================================
# 2. HASH HELPER
# ============================================================

def sha256_file(
    path,
):

    h = hashlib.sha256()


    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )


            if not chunk:
                break


            h.update(
                chunk
            )


    return h.hexdigest()


# ============================================================
# 3. PARENT LOCKS
# ============================================================

a0 = json.loads(
    A0_PATH.read_text(
        encoding="utf-8"
    )
)


a2 = json.loads(
    A2_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


a3 = json.loads(
    A3_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


current_best = json.loads(
    CURRENT_BEST_PATH.read_text(
        encoding="utf-8"
    )
)


assert a0[
    "branch_id"
] == BRANCH_ID


assert a0[
    "branch_contract_sha256"
] == A0_SHA


assert a0[
    "selection_rules"
][
    "r1p50_existing_policy_may_be_retuned_inside_this_branch"
] is False


assert a0[
    "selection_rules"
][
    "g3h3_parameters_may_be_retuned"
] is False


assert a2[
    "summary_sha256"
] == A2_SUMMARY_SHA


assert a2[
    "contract_sha256"
] == A2_CONTRACT_SHA


assert a2[
    "universe"
][
    "content_sha256"
] == A2_CONTENT_SHA


assert a2[
    "universe"
][
    "rows"
] == EXPECTED_CANDIDATES


assert a2[
    "protocol"
][
    "candidate_definition_frozen_before_oracle"
] is True


assert a3[
    "summary_sha256"
] == A3_SUMMARY_SHA


assert sha256_file(
    A3_SUMMARY_PATH
) == A3_SUMMARY_FILE_SHA


assert a3[
    "status"
] == (
    "DEVELOPMENT_ORACLE_NODE_CANDIDATE_COVERAGE_AUDITED"
)


assert a3[
    "purpose"
] == (
    "DEVELOPMENT_ORACLE_COVERAGE_ONLY"
)


assert a3[
    "protocol"
][
    "development_gt_used"
] is True


assert a3[
    "protocol"
][
    "fold0_used"
] is False


assert a3[
    "protocol"
][
    "detector_threshold_selected"
] is False


assert a3[
    "protocol"
][
    "scorer_selected"
] is False


assert current_best[
    "checkpoint_sha256"
] == CURRENT_BEST_SHA


assert current_best[
    "protocol"
][
    "fold0_available_for_future_selection"
] is False


print(
    "========== 12.12A4 PARENT LOCK =========="
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "A2 contract SHA:",
    A2_CONTRACT_SHA
)


print(
    "A3 summary SHA:",
    A3_SUMMARY_SHA
)


print(
    "Fold0 available:",
    "NO"
)


# ============================================================
# 4. VERIFY A3 DECISION SIGNAL
# ============================================================

r7 = a3[
    "coverage"
][
    "node_level"
][
    "7"
]


assert int(
    r7[
        "any_candidate_covered_nodes"
    ]
) == EXPECTED_NODE_COVERED_7


assert int(
    r7[
        "maximum_one_to_one_matches"
    ]
) == EXPECTED_NODE_COVERED_7


assert abs(
    float(
        r7[
            "any_candidate_coverage"
        ]
    )
    -
    (
        EXPECTED_NODE_COVERED_7
        /
        EXPECTED_MISSING_NODES
    )
) <= 1e-12


assert abs(
    float(
        r7[
            "one_to_one_coverage"
        ]
    )
    -
    float(
        r7[
            "any_candidate_coverage"
        ]
    )
) <= 1e-12


candidate_oracle = (
    a3[
        "candidate_oracle"
    ]
)


assert candidate_oracle[
    "candidate_rows"
] == EXPECTED_CANDIDATES


assert candidate_oracle[
    "candidates_with_missing_node_within_7um"
] == EXPECTED_POSITIVE_CANDIDATES


positive_fraction = (
    EXPECTED_POSITIVE_CANDIDATES
    /
    EXPECTED_CANDIDATES
)


assert abs(
    float(
        candidate_oracle[
            "fraction_with_missing_node_within_7um"
        ]
    )
    -
    positive_fraction
) <= 1e-12


edge7 = (
    a3[
        "coverage"
    ][
        "edge_level"
    ][
        "7"
    ][
        "ALL"
    ]
)


assert int(
    edge7[
        "edges"
    ]
) == EXPECTED_EDGE_FN


assert int(
    edge7[
        "geometrically_reconstructable"
    ]
) == EXPECTED_EDGE_COVERED_7


print(
    "\n========== A4 SCIENTIFIC BOUNDARY =========="
)


print(
    "A2 candidates:",
    EXPECTED_CANDIDATES
)


print(
    "7um oracle-positive candidates:",
    EXPECTED_POSITIVE_CANDIDATES
)


print(
    "candidate positive rate:",
    f"{positive_fraction:.6%}"
)


print(
    "7um covered missing nodes:",
    f"{EXPECTED_NODE_COVERED_7}/{EXPECTED_MISSING_NODES}",
    f"({EXPECTED_NODE_COVERED_7/EXPECTED_MISSING_NODES:.3%})"
)


print(
    "7um one-to-one ambiguity loss:",
    0
)


print(
    "7um geometrically reconstructable edge FN:",
    f"{EXPECTED_EDGE_COVERED_7}/{EXPECTED_EDGE_FN}",
    f"({EXPECTED_EDGE_COVERED_7/EXPECTED_EDGE_FN:.3%})"
)


print(
    "interpretation:",
    "LOW_CEILING_RARE_POSITIVE_RANKING_PROBLEM"
)


# ============================================================
# 5. PREDECLARE EXACT LABEL
# ============================================================

LABEL_CONTRACT = {
    "label_id":
        "A3_ORACLE_HIT_7UM_V1",

    "source":
        (
            "stage12_post_r1p50_node_candidate_"
            "oracle_candidates_v1.pkl"
        ),

    "column":
        "oracle_hit_7um",

    "definition":
        (
            "candidate has >=1 remaining development missing "
            "GT node in the same dataset/frame within 7 um"
        ),

    "positive_count":
        EXPECTED_POSITIVE_CANDIDATES,

    "negative_count":
        (
            EXPECTED_CANDIDATES
            -
            EXPECTED_POSITIVE_CANDIDATES
        ),

    "positive_fraction":
        positive_fraction,

    "development_gt_derived":
        True,

    "runtime_legal":
        False,

    "used_for":
        (
            "development scorer training/evaluation only"
        ),
}


# ============================================================
# 6. PREDECLARE FEATURE FAMILY F0
#
# No learned geometry.
# Baseline only.
# ============================================================

F0_FEATURES = [
    "detector_prob",
]


F0 = {
    "family_id":
        "DETECTOR_PROB_BASELINE_V1",

    "purpose":
        "NONTRAINED_RANKING_BASELINE",

    "features":
        F0_FEATURES,

    "runtime_gt_blind":
        True,

    "requires_model_fit":
        False,

    "ranking_rule":
        "DESCENDING_DETECTOR_PROB",

    "may_be_retuned":
        False,
}


# ============================================================
# 7. PREDECLARE FEATURE FAMILY F1
#
# Candidate-centric canonical CURRENT_BEST temporal geometry.
#
# IMPORTANT:
#
# Candidate universe remains fixed.
#
# All graph context comes from the canonical POST-G3H3
# CURRENT_BEST graph before inserting this A2 candidate.
# ============================================================

F1_FEATURES = [
    # --------------------------------------------------------
    # Detector
    # --------------------------------------------------------
    "detector_prob",
    "detector_logit",

    # --------------------------------------------------------
    # Frame availability
    # --------------------------------------------------------
    "has_prev_frame",
    "has_next_frame",

    # --------------------------------------------------------
    # Same-frame suppression / crowding
    # --------------------------------------------------------
    "same_nearest_dist_um",
    "same_second_dist_um",
    "same_neighbor_margin_um",

    # --------------------------------------------------------
    # Previous-frame geometry
    # --------------------------------------------------------
    "prev_nearest_dist_um",
    "prev_second_dist_um",
    "prev_neighbor_margin_um",

    # --------------------------------------------------------
    # Next-frame geometry
    # --------------------------------------------------------
    "next_nearest_dist_um",
    "next_second_dist_um",
    "next_neighbor_margin_um",

    # --------------------------------------------------------
    # Nearest temporal endpoint topology
    # --------------------------------------------------------
    "prev_nearest_outdegree",
    "next_nearest_indegree",

    "prev_nearest_free_out",
    "next_nearest_free_in",

    # --------------------------------------------------------
    # Bilateral geometry
    # --------------------------------------------------------
    "bilateral_available",

    "prev_next_span_um",

    "candidate_midpoint_residual_um",

    "prev_next_step_mean_um",
    "prev_next_step_max_um",
    "prev_next_step_min_um",

    "prev_next_step_absdiff_um",

    "step_balance",
]


F1 = {
    "family_id":
        "CANONICAL_CURRENT_BEST_TEMPORAL_GEOMETRY_V1",

    "purpose":
        (
            "GT_BLIND_CANDIDATE_RANKING_WITHOUT_"
            "R1P50_ASSOCIATION_GATES"
        ),

    "features":
        F1_FEATURES,

    "graph_source":
        (
            "exact canonical post-G3H3 CURRENT_BEST "
            "before inserting candidate"
        ),

    "candidate_universe_modified":
        False,

    "nearest_neighbor_k":
        2,

    "distance_units":
        "MICROMETERS",

    "same_frame_definition":
        (
            "two nearest canonical CURRENT_BEST predicted nodes "
            "at candidate time t"
        ),

    "previous_frame_definition":
        (
            "two nearest canonical CURRENT_BEST predicted nodes "
            "at t-1"
        ),

    "next_frame_definition":
        (
            "two nearest canonical CURRENT_BEST predicted nodes "
            "at t+1"
        ),

    "topology_definition":
        (
            "degree values measured in canonical CURRENT_BEST "
            "before candidate insertion"
        ),

    "bilateral_geometry_definition":
        (
            "uses nearest previous-frame and nearest next-frame "
            "predicted nodes when both exist"
        ),

    "runtime_gt_blind":
        True,

    "uses_dataset_identity_as_feature":
        False,

    "uses_fold_identity_as_feature":
        False,

    "uses_r1p50_threshold":
        False,

    "uses_bridge_association":
        False,
}


# ============================================================
# 8. PREDECLARE OPTIONAL FEATURE FAMILY F2
#
# Candidate-centric neural association.
#
# This family is predeclared NOW so it cannot be invented after
# seeing F1 results.
#
# It is NOT old R1P50:
#
#   - applies to the frozen A2 node universe
#   - does not require B12 membership
#   - does not require free endpoints
#   - does not gate on rank==1
#   - does not gate on bridge_min_prob
#
# It only exposes association evidence as scorer features.
# ============================================================

F2_ADDITIONAL_FEATURES = [
    "assoc_in_raw",
    "assoc_in_prob",

    "assoc_in_col_rank",
    "assoc_in_row_rank",

    "assoc_in_col_margin_raw",
    "assoc_in_row_margin_raw",

    "assoc_out_raw",
    "assoc_out_prob",

    "assoc_out_col_rank",
    "assoc_out_row_rank",

    "assoc_out_col_margin_raw",
    "assoc_out_row_margin_raw",

    "assoc_min_prob",
    "assoc_geom_mean_prob",

    "assoc_reciprocal_col_rank",
]


F2 = {
    "family_id":
        "ALL_CANDIDATE_TRANSFORMER_ASSOCIATION_V1",

    "base_family":
        F1[
            "family_id"
        ],

    "additional_features":
        F2_ADDITIONAL_FEATURES,

    "candidate_universe":
        (
            "EXACT_A2_FROZEN_415590_ROWS"
        ),

    "association_semantics":
        {
            "incoming":
                (
                    "canonical predicted nodes at t-1 -> "
                    "[canonical predicted nodes at t "
                    "+ A2 candidate]"
                ),

            "outgoing":
                (
                    "[canonical predicted nodes at t "
                    "+ A2 candidate] -> "
                    "canonical predicted nodes at t+1"
                ),
        },

    "candidate_scored_individually":
        True,

    "b12_membership_required":
        False,

    "free_source_required":
        False,

    "free_target_required":
        False,

    "rank_gate":
        None,

    "probability_gate":
        None,

    "r1p50_retuned":
        False,

    "runtime_gt_blind":
        True,
}


# ============================================================
# 9. PREDECLARE SCORER FAMILIES
# ============================================================

SCORERS = {
    # --------------------------------------------------------
    # Baseline
    # --------------------------------------------------------
    "S0": {
        "scorer_id":
            "DETECTOR_PROB_DESCENDING_V1",

        "feature_family":
            F0[
                "family_id"
            ],

        "trained":
            False,

        "score":
            "detector_prob",
    },

    # --------------------------------------------------------
    # Linear rare-positive model
    # --------------------------------------------------------
    "S1": {
        "scorer_id":
            "LOGREG_L2_C1_BALANCED_V1",

        "eligible_feature_families": [
            F1[
                "family_id"
            ],

            F2[
                "family_id"
            ],
        ],

        "preprocessing":
            {
                "numeric_imputation":
                    "TRAIN_FOLD_MEDIAN",

                "standardization":
                    "TRAIN_FOLD_STANDARD_SCALER",
            },

        "model":
            "sklearn.linear_model.LogisticRegression",

        "penalty":
            "l2",

        "C":
            1.0,

        "solver":
            "lbfgs",

        "class_weight":
            "balanced",

        "max_iter":
            1000,

        "random_state":
            None,
    },

    # --------------------------------------------------------
    # Low-capacity nonlinear model
    # --------------------------------------------------------
    "S2": {
        "scorer_id":
            "HGB_BALANCED_SMALL_V1",

        "eligible_feature_families": [
            F1[
                "family_id"
            ],

            F2[
                "family_id"
            ],
        ],

        "preprocessing":
            {
                "numeric_imputation":
                    "NATIVE_OR_TRAIN_FOLD_MEDIAN_NO_GT_FEATURE",
            },

        "model":
            (
                "sklearn.ensemble."
                "HistGradientBoostingClassifier"
            ),

        "learning_rate":
            0.05,

        "max_iter":
            100,

        "max_leaf_nodes":
            15,

        "max_depth":
            None,

        "l2_regularization":
            1.0,

        "min_samples_leaf":
            20,

        "class_balance":
            (
                "EXPLICIT_TRAIN_FOLD_SAMPLE_WEIGHTS_"
                "INVERSE_CLASS_FREQUENCY"
            ),

        "random_state":
            0,
    },
}


# ============================================================
# 10. PREDECLARE DEVELOPMENT BENCHMARK
# ============================================================

BENCHMARK = {
    "benchmark_id":
        "A2_NODE_RARE_POSITIVE_LOFO_BENCHMARK_V1",

    "development_folds": [
        1,
        2,
        3,
        4,
    ],

    "cross_validation":
        "LEAVE_ONE_FOLD_OUT",

    "label":
        LABEL_CONTRACT[
            "label_id"
        ],

    "primary_metric":
        "AVERAGE_PRECISION",

    "aggregation":
        "MACRO_MEAN_ACROSS_HELD_OUT_FOLDS",

    "required_fold_reports": [
        "AVERAGE_PRECISION",
        "ROC_AUC",
        "POSITIVE_PREVALENCE",
    ],

    # Ranking operating points only.
    #
    # These are NOT deployment thresholds.
    "ranking_budget_fractions": [
        0.001,
        0.0025,
        0.005,
        0.01,
    ],

    "budget_metrics": [
        "PRECISION_AT_FRACTION",
        "CANDIDATE_POSITIVE_RECALL_AT_FRACTION",
    ],

    "no_threshold_selection":
        True,

    "no_graph_metric":
        True,

    "no_fold0":
        True,
}


# ============================================================
# 11. PREDECLARE SCORER SELECTION RULE
# ============================================================

SELECTION = {
    "selection_id":
        "A2_NODE_SCORER_SELECTION_V1",

    "baseline":
        "DETECTOR_PROB_DESCENDING_V1",

    "primary_order":
        [
            "MACRO_AVERAGE_PRECISION_DESCENDING",
            "WORST_FOLD_AVERAGE_PRECISION_DESCENDING",
        ],

    "complexity_tiebreak_order":
        [
            "DETECTOR_PROB_DESCENDING_V1",
            "LOGREG_L2_C1_BALANCED_V1",
            "HGB_BALANCED_SMALL_V1",
        ],

    "tie_tolerance":
        1e-6,

    # --------------------------------------------------------
    # Minimal continuation gate
    #
    # Because A3 already showed only 14.7% geometric ceiling,
    # this branch should not continue into graph surgery unless
    # learned ranking is genuinely better than detector score.
    # --------------------------------------------------------
    "continuation_gate": {
        "macro_ap_must_exceed_detector_baseline":
            True,

        "worst_fold_ap_must_not_be_below_detector_worst_fold":
            True,

        "graph_policy_if_gate_fails":
            "CLOSE_BRANCH_NO_GRAPH_MUTATION",
    },

    "feature_family_selection_allowed":
        True,

    "selection_is_development_gt_informed":
        True,

    "no_post_selection_model_retuning":
        True,
}


# ============================================================
# 12. PREDECLARE F2 ESCALATION RULE
#
# F2 can be expensive over 415,590 candidates.
#
# Therefore A5A first evaluates S0 + F1/S1/S2.
#
# F2 is permitted only under this rule, frozen NOW.
# ============================================================

F2_ESCALATION = {
    "stage":
        "A5B",

    "family":
        F2[
            "family_id"
        ],

    "run_if":
        (
            "A5A_BEST_F1_MODEL_HAS_MACRO_AP_ABOVE_"
            "DETECTOR_BASELINE_BUT_FAILS_WORST_FOLD_"
            "NONINFERIORITY"
        ),

    "alternative_run_if":
        (
            "A5A_F1_MODEL_PASSES_CONTINUATION_GATE_"
            "BUT_RANKING_REMAINS_TOO_LOW_FOR_"
            "MEANINGFUL_SPARSE_ACTION_SELECTION"
        ),

    "important":
        (
            "F2 is predeclared here and may not change its "
            "feature definitions after A5A."
        ),

    "may_be_invented_post_A5A":
        False,
}


# ============================================================
# 13. PREDECLARE ATTACHMENT FAMILY
#
# NOT evaluated in A5.
#
# Only becomes eligible if scorer continuation gate passes.
# ============================================================

ATTACHMENT = {
    "family_id":
        "NO_CUT_NEAREST_BILATERAL_INSERTION_V1",

    "eligibility":
        (
            "ONLY_AFTER_SCORER_SELECTION_AND_"
            "SCORER_CONTINUATION_GATE_PASS"
        ),

    "graph_source":
        "CANONICAL_POST_G3H3_CURRENT_BEST",

    "candidate_source":
        "FROZEN_A2_UNIVERSE",

    "candidate_priority":
        "FROZEN_SELECTED_SCORER_DESCENDING",

    "source_endpoint":
        (
            "nearest canonical predicted node at t-1"
        ),

    "target_endpoint":
        (
            "nearest canonical predicted node at t+1"
        ),

    "source_requirement":
        "OUTDEGREE_EQ_0",

    "target_requirement":
        "INDEGREE_EQ_0",

    "source_edge_cut_allowed":
        False,

    "target_edge_cut_allowed":
        False,

    "candidate_node_inserted":
        True,

    "edges_added":
        [
            "SOURCE_TO_SYNTHETIC_NODE",
            "SYNTHETIC_NODE_TO_TARGET",
        ],

    "conflict_resolution":
        (
            "DETERMINISTIC_GREEDY_SELECTED_SCORER_DESCENDING_"
            "WITH_UNIQUE_SOURCE_TARGET_AND_CANDIDATE"
        ),

    "r1p50_actions_excluded":
        True,

    "r1p50_threshold_reused":
        False,

    "r1p50_threshold_retuned":
        False,

    "graph_metric_evaluation_allowed_in_A5":
        False,

    "fold0_allowed":
        False,
}


# ============================================================
# 14. EXPLICITLY FORBIDDEN FUTURE ADAPTATIONS
# ============================================================

FORBIDDEN = [
    "EXPAND_A2_CANDIDATE_UNIVERSE_AFTER_A3",
    "LOWER_A2_DETECTOR_FLOOR_BELOW_0P90",
    "R1P10_RESURRECTION",
    "R1P25_RESURRECTION",
    "R1P50_RETUNING",
    "G3H3_RETUNING",

    "FOLD_AS_RUNTIME_FEATURE",
    "DATASET_ID_AS_RUNTIME_FEATURE",

    "GT_DISTANCE_AS_RUNTIME_FEATURE",
    "ORACLE_HIT_AS_RUNTIME_FEATURE",
    "MISSING_COMPONENT_ID_AS_RUNTIME_FEATURE",
    "OFFICIAL_FP_AS_RUNTIME_FEATURE",

    "POST_A5_UNDECLARED_FEATURE_ENGINEERING",
    "POST_A5_UNDECLARED_SCORER_FAMILY",
    "POST_A5_UNDECLARED_ATTACHMENT_FAMILY",

    "FOLD0_SCIENTIFIC_EVALUATION",
]


# ============================================================
# 15. FORMAL CONTRACT
# ============================================================

CONTRACT = {
    "stage":
        "12.12A4",

    "contract_id":
        (
            "POST_R1P50_NODE_RARE_POSITIVE_"
            "RANKING_AND_ATTACHMENT_FAMILIES_V1"
        ),

    "branch_id":
        BRANCH_ID,

    "status":
        "FEATURE_SCORER_ATTACHMENT_FAMILIES_PREDECLARED",

    "parent": {
        "a0_contract_sha256":
            A0_SHA,

        "a2_contract_sha256":
            A2_CONTRACT_SHA,

        "a2_content_sha256":
            A2_CONTENT_SHA,

        "a3_summary_sha256":
            A3_SUMMARY_SHA,

        "current_best_checkpoint_sha256":
            CURRENT_BEST_SHA,
    },

    "a3_signal": {
        "candidate_rows":
            EXPECTED_CANDIDATES,

        "oracle_positive_candidates_7um":
            EXPECTED_POSITIVE_CANDIDATES,

        "candidate_positive_fraction_7um":
            positive_fraction,

        "missing_gt_nodes":
            EXPECTED_MISSING_NODES,

        "covered_missing_nodes_7um":
            EXPECTED_NODE_COVERED_7,

        "node_coverage_7um":
            (
                EXPECTED_NODE_COVERED_7
                /
                EXPECTED_MISSING_NODES
            ),

        "one_to_one_matches_7um":
            EXPECTED_NODE_COVERED_7,

        "one_to_one_ambiguity_loss":
            0,

        "detection_limited_edge_fn":
            EXPECTED_EDGE_FN,

        "geometrically_reconstructable_edge_fn_7um":
            EXPECTED_EDGE_COVERED_7,

        "edge_ceiling_7um":
            (
                EXPECTED_EDGE_COVERED_7
                /
                EXPECTED_EDGE_FN
            ),

        "scientific_classification":
            (
                "LOW_CEILING_RARE_POSITIVE_RANKING_PROBLEM"
            ),
    },

    "label_contract":
        LABEL_CONTRACT,

    "feature_families": {
        "F0":
            F0,

        "F1":
            F1,

        "F2":
            F2,
    },

    "scorer_families":
        SCORERS,

    "benchmark":
        BENCHMARK,

    "selection":
        SELECTION,

    "f2_escalation":
        F2_ESCALATION,

    "attachment_family":
        ATTACHMENT,

    "forbidden":
        FORBIDDEN,

    "protocol": {
        "development_gt_informed_design":
            True,

        "new_gt_read_in_a4":
            False,

        "feature_matrix_computed":
            False,

        "model_trained":
            False,

        "scorer_selected":
            False,

        "threshold_selected":
            False,

        "graph_modified":
            False,

        "official_graph_metric_evaluated":
            False,

        "r1p50_retuned":
            False,

        "g3h3_reopened":
            False,

        "fold0_used":
            False,

        "fold0_may_confirm_new_branch":
            False,
    },

    "next":
        (
            "12.12A5A_BUILD_FROZEN_F1_FEATURE_MATRIX_"
            "AND_RUN_LOFO_RARE_POSITIVE_SCORER_BENCHMARK"
        ),
}


CONTRACT_SHA = hashlib.sha256(
    json.dumps(
        CONTRACT,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


CONTRACT[
    "contract_sha256"
] = CONTRACT_SHA


CONTRACT_OUT.write_text(
    json.dumps(
        CONTRACT,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 16. SUMMARY
# ============================================================

SUMMARY = {
    "stage":
        "12.12A4",

    "status":
        "RARE_POSITIVE_RANKING_RESEARCH_CONTRACT_FROZEN",

    "branch_id":
        BRANCH_ID,

    "contract_id":
        CONTRACT[
            "contract_id"
        ],

    "contract_sha256":
        CONTRACT_SHA,

    "scientific_classification":
        (
            "LOW_CEILING_RARE_POSITIVE_RANKING_PROBLEM"
        ),

    "a3": {
        "candidate_positive_fraction":
            positive_fraction,

        "node_ceiling_7um":
            (
                EXPECTED_NODE_COVERED_7
                /
                EXPECTED_MISSING_NODES
            ),

        "edge_ceiling_7um":
            (
                EXPECTED_EDGE_COVERED_7
                /
                EXPECTED_EDGE_FN
            ),

        "ambiguity_loss":
            0,
    },

    "mandatory_next_feature_family":
        F1[
            "family_id"
        ],

    "mandatory_baseline":
        F0[
            "family_id"
        ],

    "predeclared_optional_escalation":
        F2[
            "family_id"
        ],

    "predeclared_scorers": [
        SCORERS[
            "S0"
        ][
            "scorer_id"
        ],

        SCORERS[
            "S1"
        ][
            "scorer_id"
        ],

        SCORERS[
            "S2"
        ][
            "scorer_id"
        ],
    ],

    "attachment_deferred_until_ranking_gate":
        True,

    "development_gt_informed_design":
        True,

    "new_gt_read":
        False,

    "fold0_used":
        False,

    "next":
        CONTRACT[
            "next"
        ],
}


SUMMARY_SHA = hashlib.sha256(
    json.dumps(
        SUMMARY,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


SUMMARY[
    "summary_sha256"
] = SUMMARY_SHA


SUMMARY_OUT.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 17. RELOAD INTEGRITY
# ============================================================

contract_check = json.loads(
    CONTRACT_OUT.read_text(
        encoding="utf-8"
    )
)


summary_check = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert contract_check[
    "contract_sha256"
] == CONTRACT_SHA


assert contract_check[
    "status"
] == (
    "FEATURE_SCORER_ATTACHMENT_FAMILIES_PREDECLARED"
)


assert contract_check[
    "protocol"
][
    "development_gt_informed_design"
] is True


assert contract_check[
    "protocol"
][
    "new_gt_read_in_a4"
] is False


assert contract_check[
    "protocol"
][
    "feature_matrix_computed"
] is False


assert contract_check[
    "protocol"
][
    "model_trained"
] is False


assert contract_check[
    "protocol"
][
    "scorer_selected"
] is False


assert contract_check[
    "protocol"
][
    "threshold_selected"
] is False


assert contract_check[
    "protocol"
][
    "graph_modified"
] is False


assert contract_check[
    "protocol"
][
    "fold0_used"
] is False


assert contract_check[
    "protocol"
][
    "fold0_may_confirm_new_branch"
] is False


assert summary_check[
    "summary_sha256"
] == SUMMARY_SHA


assert summary_check[
    "status"
] == (
    "RARE_POSITIVE_RANKING_RESEARCH_CONTRACT_FROZEN"
)


# ============================================================
# 18. FINAL
# ============================================================

print(
    "\n========== 12.12A4 FINAL FREEZE =========="
)


print(
    "RARE_POSITIVE_RANKING_RESEARCH_CONTRACT: PASS"
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "scientific classification:",
    "LOW_CEILING_RARE_POSITIVE_RANKING_PROBLEM"
)


print(
    "7um node ceiling:",
    f"{EXPECTED_NODE_COVERED_7}/{EXPECTED_MISSING_NODES}",
    f"({EXPECTED_NODE_COVERED_7/EXPECTED_MISSING_NODES:.3%})"
)


print(
    "7um edge ceiling:",
    f"{EXPECTED_EDGE_COVERED_7}/{EXPECTED_EDGE_FN}",
    f"({EXPECTED_EDGE_COVERED_7/EXPECTED_EDGE_FN:.3%})"
)


print(
    "candidate positive rate:",
    f"{positive_fraction:.6%}"
)


print(
    "ambiguity loss:",
    "0.000%"
)


print(
    "mandatory feature family:",
    F1[
        "family_id"
    ]
)


print(
    "optional predeclared escalation:",
    F2[
        "family_id"
    ]
)


print(
    "scorers:"
)


for scorer in SCORERS.values():

    print(
        " ",
        scorer[
            "scorer_id"
        ]
    )


print(
    "primary metric:",
    BENCHMARK[
        "primary_metric"
    ]
)


print(
    "CV:",
    BENCHMARK[
        "cross_validation"
    ]
)


print(
    "attachment deferred:",
    "YES"
)


print(
    "new GT read:",
    "NO"
)


print(
    "feature matrix computed:",
    "NO"
)


print(
    "model trained:",
    "NO"
)


print(
    "scorer selected:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "graph modified:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "Fold0 may confirm new branch:",
    "NO"
)


print(
    "contract SHA:",
    CONTRACT_SHA
)


print(
    "summary SHA:",
    SUMMARY_SHA
)


print(
    "next:"
)


print(
    "12.12A5A_BUILD_FROZEN_F1_FEATURE_MATRIX_"
    "AND_RUN_LOFO_RARE_POSITIVE_SCORER_BENCHMARK"
)

In [ ]:
# ============================================================
# Stage 12.12A5A-R3-R1-R3
#
# FORMAL
#
# REPLAY EXACT FROZEN DEVELOPMENT G3H3 ACTIONS
# +
# BUILD / FREEZE A4 F1 FEATURE MATRIX
#
# ============================================================
#
# FINAL REPAIR OF:
#
#   R3:
#     wrong Fold0 builder
#
#   R3-R1:
#     missing isolated namespace dependencies
#
#   R3-R1-R1:
#     inspect.getclosurevars false-positive auditor
#
#   R3-R1-R2:
#     correct development builder, but incorrectly regenerated
#     development G3H3 with FULL-DEV DEPLOYMENT runtime
#
# ------------------------------------------------------------
# EXACT DEVELOPMENT POST-G3H3 DEFINITION
# ------------------------------------------------------------
#
#   exact folds1-4 pre-G3H3 CURRENT_BEST
#
#       +
#
#   formally frozen development action artifact:
#
#     stage12_folds14_rewire_
#     asym1x5_q9990_no_target_cut_v1_actions.pkl
#
#     SHA256:
#       39e7261bbcfac40304c441be98128b0e
#       9d9bcce4e9ce8d98beea7f69cf238b79
#
#       12,515 actions
#       385 source incumbent removals
#       0 target incumbent removals
#       net edge delta +12,130
#
# ------------------------------------------------------------
# EXACT MUTATION SEMANTICS
# ------------------------------------------------------------
#
# Extracted VERBATIM from the already-frozen historical
# apply_locked_fold0_policy() implementation:
#
#   REMOVE_SOURCE_INCUMBENT_IF_PRESENT
#   NEVER_REMOVE_TARGET_INCUMBENT
#   ADD_REWIRE_EDGE
#
# The exact historical helper itself is AST-extracted from the
# notebook. Its implementation is NOT rewritten here.
#
# ------------------------------------------------------------
# SCIENTIFIC ORDER
# ------------------------------------------------------------
#
# A2 frozen GT-blind candidates
#       ->
# exact pre-G3H3 development CURRENT_BEST
#       ->
# exact frozen 12,515 dev action replay
#       ->
# exact post-G3H3 development CURRENT_BEST
#       ->
# A4-predeclared 25 F1 features
#       ->
# formal feature freeze
#
# ONLY AFTER THIS CELL may A3 oracle labels be joined.
#
# ------------------------------------------------------------
# THIS CELL DOES NOT
# ------------------------------------------------------------
#
# read A3 oracle
# read missing GT nodes
# load GT graph
# evaluate official metric
# regenerate G3H3 scores
# use deployment threshold
# select threshold
# train model
# select scorer
# retune R1P50
# retune G3H3
# use Fold0
#
# ------------------------------------------------------------
# KEEP AFTER RUNNING: YES
# ============================================================


from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Optional, Iterable, Sequence, Mapping

import ast
import builtins
import dis
import functools
import gc
import hashlib
import importlib
import inspect
import itertools
import json
import math
import os
import pickle
import re
import shutil
import sys
import types

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


NOTEBOOK_PATH = (
    PROJECT
    / "notebooks"
    / "12_residual_edge_recovery.ipynb"
)


FROZEN_V9_PATH = (
    PROJECT
    / "src"
    / "biohub_cell_tracking"
    / "frozen_v9.py"
)


TRACKING_IO_PATH = (
    PROJECT
    / "external"
    / "official"
    / "src"
    / "tracking_cellmot"
    / "io.py"
)


A2_UNIVERSE_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_universe_v1.pkl"
)


A2_SUMMARY_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_universe_v1_summary.json"
)


A4_CONTRACT_PATH = (
    S12
    / "stage12_post_r1p50_node_scoring_attachment_family_v1_contract_frozen.json"
)


FINAL_POLICY_PATH = (
    S12
    / "stage12_rewire_asym1x5_q9990_no_target_cut_v1_frozen.json"
)


D1_CONTRACT_PATH = (
    S12
    / "stage12_folds14_rewire_q9990_structural_ablation_v1_frozen.json"
)


CONFIRMATION_CONTRACT_PATH = (
    S12
    / "stage12_fold0_rewire_asym1x5_q9990_no_target_cut_v1_confirmation_contract_frozen.json"
)


# ------------------------------------------------------------
# EXACT FINAL DEVELOPMENT ACTION ARTIFACT
# ------------------------------------------------------------

DEV_ACTIONS_PATH = (
    S12
    / "stage12_folds14_rewire_asym1x5_q9990_no_target_cut_v1_actions.pkl"
)


# ============================================================
# FORMAL A5A F1 OUTPUTS
# ============================================================

FEATURE_MATRIX_OUT = (
    S12
    / "stage12_post_r1p50_node_f1_temporal_geometry_v1.pkl"
)


FEATURE_SCHEMA_OUT = (
    S12
    / "stage12_post_r1p50_node_f1_temporal_geometry_v1_schema_frozen.json"
)


DATASET_SUMMARY_OUT = (
    S12
    / "stage12_post_r1p50_node_f1_temporal_geometry_v1_dataset_summary.pkl"
)


SUMMARY_OUT = (
    S12
    / "stage12_post_r1p50_node_f1_temporal_geometry_v1_summary.json"
)


for path in [
    NOTEBOOK_PATH,
    FROZEN_V9_PATH,
    TRACKING_IO_PATH,
    A2_UNIVERSE_PATH,
    A2_SUMMARY_PATH,
    A4_CONTRACT_PATH,
    FINAL_POLICY_PATH,
    D1_CONTRACT_PATH,
    CONFIRMATION_CONTRACT_PATH,
    DEV_ACTIONS_PATH,
]:

    assert path.exists(), path


# ------------------------------------------------------------
# Formal no-overwrite gate
# ------------------------------------------------------------

for path in [
    FEATURE_MATRIX_OUT,
    FEATURE_SCHEMA_OUT,
    DATASET_SUMMARY_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nFormal A5A F1 artifact already exists:\n"
            f"{path}\n\n"
            "Do NOT overwrite formal provenance."
        )


# ============================================================
# 1. FROZEN IDENTITIES
# ============================================================

BRANCH_ID = (
    "POST_R1P50_RESIDUAL_SYNTHETIC_NODE_RECOVERY_V1"
)


F1_ID = (
    "CANONICAL_CURRENT_BEST_TEMPORAL_GEOMETRY_V1"
)


FINAL_POLICY_ID = (
    "ASYM1x5_Q9990_NO_TARGET_CUT_V1"
)


FINAL_POLICY_SHA = (
    "77f0b07cafb21d0705917ff71b750fde4"
    "5f3c7ddc17c13fecaff0ac673ebf776"
)


DEV_ACTIONS_SHA = (
    "39e7261bbcfac40304c441be98128b0e"
    "9d9bcce4e9ce8d98beea7f69cf238b79"
)


A2_CONTRACT_SHA = (
    "fbf3f2bcc81d87180837a6170a598296"
    "a5276a62728c8a21d6c4fc0f9a57313b"
)


A2_CONTENT_SHA = (
    "3462a6cd20eee93e55ba7cca8f110654"
    "5c22cb5815ac1e05dde09e04d5cf8e91"
)


A2_FILE_SHA = (
    "a082babaeecda9e50a13a329abbc912f"
    "f4a16ad56ce3695301d266ca0c5e4147"
)


A4_CONTRACT_SHA = (
    "47e173fb2be1dd00ebb9c3a789dd3c0b"
    "a7b98863815f16904ee1546b5e6bba58"
)


TRACKING_IO_SHA = (
    "efae135b088cecaab463d889f16c885e"
    "f6da3ad27b0747327d8ddc28d866b7bd"
)


EXPECTED_ROWS = 415_590


EXPECTED_DATASETS = 103


EXPECTED_FOLD_ROWS = {
    1: 94_729,
    2: 100_318,
    3: 100_147,
    4: 120_396,
}


EXPECTED_DEV_ACTIONS = 12_515


EXPECTED_DEV_SOURCE_REMOVALS = 385


EXPECTED_DEV_EDGES_ADDED = 12_515


EXPECTED_DEV_NET_EDGE_DELTA = 12_130


EXPECTED_ACTION_FOLD_COUNTS = {
    1: 2_838,
    2: 3_370,
    3: 2_462,
    4: 3_845,
}


EXPECTED_DEV_BUILDER_CELL_INDEX = 30


EXPECTED_DEV_P995_TOKEN = (
    "stage11_fullgraph_baseline256_folds1to4_det995"
)


EXPECTED_DEV_P9975_TOKEN = (
    "stage11_fullgraph_baseline256_folds1to4_det9975"
)


EXPECTED_MUTATION_RULE = (
    "REMOVE_SOURCE_INCUMBENT_IF_PRESENT;"
    "NEVER_REMOVE_TARGET_INCUMBENT;"
    "ADD_REWIRE_EDGE"
)


# ============================================================
# 2. HASH HELPERS
# ============================================================

def sha256_file(
    path,
):

    h = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:

                break

            h.update(
                chunk
            )

    return h.hexdigest()


class HashWriter:

    def __init__(
        self,
        hasher,
    ):

        self.hasher = hasher


    def write(
        self,
        text,
    ):

        if not isinstance(
            text,
            str,
        ):

            text = str(
                text
            )

        self.hasher.update(
            text.encode(
                "utf-8"
            )
        )

        return len(
            text
        )


    def flush(
        self,
    ):

        return None


def dataframe_content_sha256(
    df,
    columns,
):

    h = hashlib.sha256()

    writer = HashWriter(
        h
    )

    df.loc[
        :,
        columns,
    ].to_csv(
        writer,
        index=False,
        na_rep="NaN",
        float_format="%.17g",
        lineterminator="\n",
    )

    return h.hexdigest()


# ============================================================
# 3. LOCK PARENT CONTRACTS
# ============================================================

assert sha256_file(
    TRACKING_IO_PATH
) == TRACKING_IO_SHA


assert sha256_file(
    DEV_ACTIONS_PATH
) == DEV_ACTIONS_SHA


a2_summary = json.loads(
    A2_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


a4 = json.loads(
    A4_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


final_policy = json.loads(
    FINAL_POLICY_PATH.read_text(
        encoding="utf-8"
    )
)


d1_contract = json.loads(
    D1_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


confirmation_contract = json.loads(
    CONFIRMATION_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


assert a2_summary[
    "contract_sha256"
] == A2_CONTRACT_SHA


assert a2_summary[
    "universe"
][
    "content_sha256"
] == A2_CONTENT_SHA


assert a2_summary[
    "universe"
][
    "artifact_file_sha256"
] == A2_FILE_SHA


assert sha256_file(
    A2_UNIVERSE_PATH
) == A2_FILE_SHA


assert a4[
    "contract_sha256"
] == A4_CONTRACT_SHA


assert a4[
    "branch_id"
] == BRANCH_ID


assert a4[
    "feature_families"
][
    "F1"
][
    "family_id"
] == F1_ID


assert a4[
    "protocol"
][
    "fold0_used"
] is False


assert final_policy[
    "policy_id"
] == FINAL_POLICY_ID


assert final_policy[
    "selected_dev_actions_artifact"
] == DEV_ACTIONS_PATH.name


assert final_policy[
    "selected_dev_actions_sha256"
] == DEV_ACTIONS_SHA


assert d1_contract[
    "family"
][
    "NO_TARGET_CUT"
] == "TARGET_OCCUPIED_EQ_0"


assert confirmation_contract[
    "structural_rule"
] == "TARGET_OCCUPIED_EQ_0"


assert confirmation_contract[
    "mutation_rule"
] == EXPECTED_MUTATION_RULE


print(
    "========== 12.12A5A-R3-R1-R3 PARENT LOCK =========="
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "final G3H3 policy:",
    FINAL_POLICY_ID
)


print(
    "development action artifact:",
    DEV_ACTIONS_PATH.name
)


print(
    "development action SHA:",
    DEV_ACTIONS_SHA
)


print(
    "mutation:",
    EXPECTED_MUTATION_RULE
)


print(
    "A3 oracle read:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


# ============================================================
# 4. LOAD / LOCK FINAL DEVELOPMENT ACTION ARTIFACT
# ============================================================

dev_actions = pd.read_pickle(
    DEV_ACTIONS_PATH
)


assert len(
    dev_actions
) == EXPECTED_DEV_ACTIONS


required_action_columns = {
    "dataset",
    "fold",
    "source_id",
    "target_id",
    "cuts",
    "source_occupied",
    "target_occupied",
}


assert required_action_columns.issubset(
    dev_actions.columns
)


for column in [
    "fold",
    "source_id",
    "target_id",
    "cuts",
    "source_occupied",
    "target_occupied",
]:

    dev_actions[
        column
    ] = pd.to_numeric(
        dev_actions[
            column
        ],
        errors="raise",
    ).astype(
        int
    )


assert dev_actions[
    "target_occupied"
].eq(
    0
).all()


assert int(
    dev_actions[
        "source_occupied"
    ].sum()
) == EXPECTED_DEV_SOURCE_REMOVALS


assert int(
    dev_actions[
        "cuts"
    ].sum()
) == EXPECTED_DEV_SOURCE_REMOVALS


action_fold_counts = (
    dev_actions[
        "fold"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)


assert action_fold_counts == EXPECTED_ACTION_FOLD_COUNTS


assert not dev_actions[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


# ------------------------------------------------------------
# Node-disjointness survives D1 structural filtering.
# ------------------------------------------------------------

for dataset, sub in dev_actions.groupby(
    "dataset",
    sort=False,
):

    endpoints = pd.concat(
        [
            sub[
                "source_id"
            ],
            sub[
                "target_id"
            ],
        ],
        ignore_index=True,
    )


    assert not endpoints.duplicated().any(), (
        dataset,
        "Frozen development actions are not node-disjoint."
    )


print(
    "\n========== FROZEN DEVELOPMENT ACTION LOCK =========="
)


print(
    "actions:",
    len(
        dev_actions
    )
)


print(
    "source occupied:",
    int(
        dev_actions[
            "source_occupied"
        ].sum()
    )
)


print(
    "target occupied:",
    int(
        dev_actions[
            "target_occupied"
        ].sum()
    )
)


print(
    "fold counts:",
    action_fold_counts
)


print(
    "node-disjoint:",
    "YES"
)


print(
    "DEVELOPMENT_ACTION_ARTIFACT_LOCK: PASS"
)


# ============================================================
# 5. A4 F1 FEATURE CONTRACT
# ============================================================

F1_FEATURES = list(
    a4[
        "feature_families"
    ][
        "F1"
    ][
        "features"
    ]
)


EXPECTED_F1_FEATURES = [
    "detector_prob",
    "detector_logit",

    "has_prev_frame",
    "has_next_frame",

    "same_nearest_dist_um",
    "same_second_dist_um",
    "same_neighbor_margin_um",

    "prev_nearest_dist_um",
    "prev_second_dist_um",
    "prev_neighbor_margin_um",

    "next_nearest_dist_um",
    "next_second_dist_um",
    "next_neighbor_margin_um",

    "prev_nearest_outdegree",
    "next_nearest_indegree",

    "prev_nearest_free_out",
    "next_nearest_free_in",

    "bilateral_available",

    "prev_next_span_um",

    "candidate_midpoint_residual_um",

    "prev_next_step_mean_um",
    "prev_next_step_max_um",
    "prev_next_step_min_um",

    "prev_next_step_absdiff_um",

    "step_balance",
]


assert F1_FEATURES == EXPECTED_F1_FEATURES


assert len(
    F1_FEATURES
) == 25


# ============================================================
# 6. LOAD ONLY FROZEN A2 GT-BLIND CANDIDATES
# ============================================================

candidate = pd.read_pickle(
    A2_UNIVERSE_PATH
)


assert len(
    candidate
) == EXPECTED_ROWS


required_candidate_columns = {
    "dataset",
    "fold",
    "t",
    "z",
    "y",
    "x",
    "detector_prob",
    "detector_logit",
}


assert required_candidate_columns.issubset(
    candidate.columns
)


# Preserve exact frozen A2 row order.
candidate = (
    candidate
    .copy()
    .reset_index(
        drop=True
    )
)


candidate[
    "candidate_id"
] = np.arange(
    EXPECTED_ROWS,
    dtype=np.int64,
)


candidate[
    "fold"
] = pd.to_numeric(
    candidate[
        "fold"
    ],
    errors="raise",
).astype(
    np.int8
)


candidate[
    "t"
] = pd.to_numeric(
    candidate[
        "t"
    ],
    errors="raise",
).astype(
    np.int64
)


assert not candidate[
    [
        "dataset",
        "t",
        "z",
        "y",
        "x",
    ]
].duplicated().any()


fold_counts = (
    candidate[
        "fold"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)


assert fold_counts == EXPECTED_FOLD_ROWS


datasets = sorted(
    candidate[
        "dataset"
    ]
    .astype(
        str
    )
    .unique()
    .tolist()
)


assert len(
    datasets
) == EXPECTED_DATASETS


assert all(
    dataset.startswith(
        "6bba_"
    )

    for dataset
    in datasets
)


assert set(
    dev_actions[
        "dataset"
    ]
).issubset(
    set(
        datasets
    )
)


print(
    "\n========== FROZEN A2 INPUT =========="
)


print(
    "rows:",
    len(
        candidate
    )
)


print(
    "datasets:",
    len(
        datasets
    )
)


print(
    "fold counts:",
    fold_counts
)


print(
    "A3 oracle read:",
    "NO"
)


# ============================================================
# 7. READ NOTEBOOK
# ============================================================

notebook = json.loads(
    NOTEBOOK_PATH.read_text(
        encoding="utf-8"
    )
)


cells = notebook[
    "cells"
]


# ============================================================
# 8. RECOVER EXACT DEVELOPMENT BUILDER — CELL 30
# ============================================================

builder_cell = cells[
    EXPECTED_DEV_BUILDER_CELL_INDEX
]


assert builder_cell.get(
    "cell_type"
) == "code"


dev_source = "".join(
    builder_cell.get(
        "source",
        []
    )
)


DEV_BUILDER_CELL_SHA = hashlib.sha256(
    dev_source.encode(
        "utf-8"
    )
).hexdigest()


assert (
    "def build_dev_current_best"
    in dev_source
)


assert (
    "Self-contained CURRENT_BEST graph-builder bootstrap"
    in dev_source
)


assert (
    EXPECTED_DEV_P995_TOKEN
    in dev_source
)


assert (
    EXPECTED_DEV_P9975_TOKEN
    in dev_source
)


assert (
    "stage10_infer_baseline_256_det995"
    not in dev_source
)


assert (
    "stage10_infer_baseline_256_det9975"
    not in dev_source
)


dev_tree = ast.parse(
    dev_source
)


builder_node = None


for node in dev_tree.body:

    if (
        isinstance(
            node,
            ast.FunctionDef
        )
        and
        node.name
        ==
        "build_dev_current_best"
    ):

        builder_node = node

        break


assert builder_node is not None


safe_body = [
    node

    for node
    in dev_tree.body

    if int(
        getattr(
            node,
            "end_lineno",
            0
        )
    )
    <=
    int(
        builder_node.end_lineno
    )
]


assert safe_body


safe_prefix_source = "\n".join(
    ast.get_source_segment(
        dev_source,
        node
    )
    or
    ast.unparse(
        node
    )

    for node
    in safe_body
)


DEV_BUILDER_PREFIX_SHA = hashlib.sha256(
    safe_prefix_source.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# 9. DEVELOPMENT BUILDER STRICT NAMESPACE
# ============================================================

fv9_exact = importlib.import_module(
    "biohub_cell_tracking.frozen_v9"
)


tracking_io_exact = importlib.import_module(
    "tracking_cellmot.io"
)


tracksdata_exact = importlib.import_module(
    "tracksdata"
)


assert Path(
    inspect.getsourcefile(
        fv9_exact
    )
).resolve() == FROZEN_V9_PATH.resolve()


assert hasattr(
    fv9_exact,
    "load_graph"
)


assert hasattr(
    fv9_exact,
    "step_balance"
)


assert hasattr(
    fv9_exact,
    "recovered_edge_dist"
)


assert hasattr(
    tracking_io_exact,
    "open_dataset"
)


dev_ns = {
    "__name__":
        "__stage12_a5a_dev_builder__",

    "__file__":
        (
            f"{NOTEBOOK_PATH}"
            f"::cell{EXPECTED_DEV_BUILDER_CELL_INDEX}"
        ),

    "__package__":
        None,

    "__builtins__":
        builtins.__dict__,

    "Path":
        Path,

    "Counter":
        Counter,

    "defaultdict":
        defaultdict,

    "Any":
        Any,

    "Optional":
        Optional,

    "Iterable":
        Iterable,

    "Sequence":
        Sequence,

    "Mapping":
        Mapping,

    "ast":
        ast,

    "builtins":
        builtins,

    "dis":
        dis,

    "functools":
        functools,

    "gc":
        gc,

    "hashlib":
        hashlib,

    "importlib":
        importlib,

    "inspect":
        inspect,

    "itertools":
        itertools,

    "json":
        json,

    "math":
        math,

    "os":
        os,

    "pickle":
        pickle,

    "re":
        re,

    "shutil":
        shutil,

    "sys":
        sys,

    "types":
        types,

    "np":
        np,

    "pd":
        pd,

    "cKDTree":
        cKDTree,

    "fv9":
        fv9_exact,

    "td":
        tracksdata_exact,

    "open_dataset":
        tracking_io_exact.open_dataset,

    "load_graph":
        fv9_exact.load_graph,

    "step_balance":
        fv9_exact.step_balance,

    "recovered_edge_dist":
        fv9_exact.recovered_edge_dist,

    "FROZEN_V9_CONFIG":
        fv9_exact.FROZEN_V9_CONFIG,

    "PROJECT":
        PROJECT,

    "S12":
        S12,
}


# No active notebook scientific state enters namespace.
assert "gt" not in dev_ns
assert "oracle" not in dev_ns
assert "fold0" not in dev_ns


safe_module = ast.Module(
    body=safe_body,
    type_ignores=[],
)


safe_module = ast.fix_missing_locations(
    safe_module
)


safe_code = compile(
    safe_module,
    filename=(
        f"{NOTEBOOK_PATH}"
        f"::cell{EXPECTED_DEV_BUILDER_CELL_INDEX}"
    ),
    mode="exec",
)


exec(
    safe_code,
    dev_ns,
)


assert "build_dev_current_best" in dev_ns


build_dev_current_best = (
    dev_ns[
        "build_dev_current_best"
    ]
)


fv9 = dev_ns[
    "fv9"
]


print(
    "\n========== DEVELOPMENT BUILDER RECOVERY =========="
)


print(
    "cell:",
    EXPECTED_DEV_BUILDER_CELL_INDEX
)


print(
    "cell SHA:",
    DEV_BUILDER_CELL_SHA
)


print(
    "safe-prefix SHA:",
    DEV_BUILDER_PREFIX_SHA
)


print(
    "P995:",
    dev_ns[
        "B256_995"
    ]
)


print(
    "P9975:",
    dev_ns[
        "B256_9975"
    ]
)


assert dev_ns[
    "B256_995"
].exists()


assert dev_ns[
    "B256_9975"
].exists()


print(
    "EXACT_DEVELOPMENT_PRE_G3H3_BUILDER: PASS"
)


# ============================================================
# 10. BYTECODE GLOBAL AUDIT
# ============================================================

def recursive_code_objects(
    code,
    path,
):

    yield (
        path,
        code,
    )


    for index, constant in enumerate(
        code.co_consts
    ):

        if isinstance(
            constant,
            types.CodeType
        ):

            yield from recursive_code_objects(
                constant,
                (
                    f"{path}"
                    f"::{constant.co_name}"
                    f"[{index}]"
                ),
            )


def true_global_loads(
    code,
):

    names = []


    for instruction in dis.get_instructions(
        code
    ):

        if instruction.opname in {
            "LOAD_GLOBAL",
            "LOAD_NAME",
        }:

            if isinstance(
                instruction.argval,
                str,
            ):

                names.append(
                    instruction.argval
                )


    return sorted(
        set(
            names
        )
    )


builder_functions = {
    name:
        obj

    for name, obj
    in dev_ns.items()

    if (
        inspect.isfunction(
            obj
        )
        and
        getattr(
            obj,
            "__module__",
            None
        )
        ==
        "__stage12_a5a_dev_builder__"
    )
}


true_unresolved = {}


for name, fn in builder_functions.items():

    builtins_obj = fn.__globals__.get(
        "__builtins__",
        builtins.__dict__,
    )


    builtin_map = (
        builtins_obj
        if isinstance(
            builtins_obj,
            dict,
        )
        else vars(
            builtins_obj
        )
    )


    for code_path, code_obj in recursive_code_objects(
        fn.__code__,
        name,
    ):

        loaded = true_global_loads(
            code_obj
        )


        unresolved = [
            global_name

            for global_name
            in loaded

            if (
                global_name
                not in fn.__globals__
                and
                global_name
                not in builtin_map
            )
        ]


        if unresolved:

            true_unresolved[
                code_path
            ] = unresolved


assert not true_unresolved, true_unresolved


print(
    "DEVELOPMENT_BUILDER_BYTECODE_GLOBAL_AUDIT: PASS"
)


# ============================================================
# 11. EXTRACT EXACT HISTORICAL MUTATION HELPERS
#
# Search notebook for the exact frozen:
#
#   local_graph_state
#   apply_locked_fold0_policy
#
# and execute ONLY those function definitions.
# ============================================================

mutation_cell_matches = []


for cell_index, cell in enumerate(
    cells
):

    if cell.get(
        "cell_type"
    ) != "code":

        continue


    source = "".join(
        cell.get(
            "source",
            []
        )
    )


    if (
        "def local_graph_state"
        in source
        and
        "def apply_locked_fold0_policy"
        in source
    ):

        mutation_cell_matches.append(
            (
                cell_index,
                source,
            )
        )


assert len(
    mutation_cell_matches
) == 1, (
    "Expected exactly one historical frozen mutation cell.",
    [
        index
        for index, _source
        in mutation_cell_matches
    ],
)


MUTATION_CELL_INDEX, mutation_source = (
    mutation_cell_matches[
        0
    ]
)


mutation_tree = ast.parse(
    mutation_source
)


wanted_function_names = {
    "local_graph_state",
    "apply_locked_fold0_policy",
}


mutation_nodes = [
    node

    for node in mutation_tree.body

    if (
        isinstance(
            node,
            ast.FunctionDef
        )
        and
        node.name
        in wanted_function_names
    )
]


assert {
    node.name
    for node
    in mutation_nodes
} == wanted_function_names


# Preserve original source order.
mutation_nodes = sorted(
    mutation_nodes,
    key=lambda node: node.lineno,
)


mutation_exact_source = "\n\n".join(
    ast.get_source_segment(
        mutation_source,
        node
    )
    or
    ast.unparse(
        node
    )

    for node
    in mutation_nodes
)


MUTATION_HELPER_SOURCE_SHA = hashlib.sha256(
    mutation_exact_source.encode(
        "utf-8"
    )
).hexdigest()


# ------------------------------------------------------------
# Hard mutation-source contract.
# ------------------------------------------------------------

assert (
    "bulk_remove_edges"
    in mutation_exact_source
)


assert (
    "graph.add_edge"
    in mutation_exact_source
)


assert (
    "fv9.recovered_edge_dist"
    in mutation_exact_source
)


assert (
    '"edge_prob"'
    in mutation_exact_source
)


assert (
    "evaluate_graph"
    not in mutation_exact_source
)


assert (
    "load_graph"
    not in mutation_exact_source
)


mutation_module = ast.Module(
    body=mutation_nodes,
    type_ignores=[],
)


mutation_module = ast.fix_missing_locations(
    mutation_module
)


mutation_ns = {
    "__name__":
        "__stage12_exact_frozen_mutation__",

    "__builtins__":
        builtins.__dict__,

    "pd":
        pd,

    "fv9":
        fv9,

    "int":
        int,

    "float":
        float,

    "len":
        len,

    "set":
        set,

    "sorted":
        sorted,

    "zip":
        zip,
}


exec(
    compile(
        mutation_module,
        filename=(
            f"{NOTEBOOK_PATH}"
            f"::cell{MUTATION_CELL_INDEX}"
            "::frozen_mutation_helpers"
        ),
        mode="exec",
    ),
    mutation_ns,
)


exact_local_graph_state = (
    mutation_ns[
        "local_graph_state"
    ]
)


exact_apply_frozen_policy = (
    mutation_ns[
        "apply_locked_fold0_policy"
    ]
)


assert callable(
    exact_local_graph_state
)


assert callable(
    exact_apply_frozen_policy
)


print(
    "\n========== EXACT FROZEN MUTATION HELPER =========="
)


print(
    "historical notebook cell:",
    MUTATION_CELL_INDEX
)


print(
    "source SHA:",
    MUTATION_HELPER_SOURCE_SHA
)


print(
    "bulk_remove_edges:",
    "YES"
)


print(
    "recovered_edge_dist:",
    "YES"
)


print(
    "target incumbent removal:",
    "FORBIDDEN"
)


print(
    "EXACT_FROZEN_MUTATION_HELPER_RECOVERED: PASS"
)


# ============================================================
# 12. DEVELOPMENT ACTION LOOKUP
# ============================================================

actions_by_dataset = {
    dataset:
        sub.copy()

    for dataset, sub
    in dev_actions.groupby(
        "dataset",
        sort=False,
    )
}


empty_actions = dev_actions.iloc[
    0:0
].copy()


# ============================================================
# 13. EXACT POST-G3H3 DEVELOPMENT BUILDER
# ============================================================

def build_dev_post_g3h3_frozen(
    dataset,
):

    graph, scale = build_dev_current_best(
        dataset
    )


    actions = actions_by_dataset.get(
        dataset,
        empty_actions,
    )


    pre_edges = int(
        graph.num_edges()
    )


    graph, stats = exact_apply_frozen_policy(
        graph,
        actions,
    )


    post_edges = int(
        graph.num_edges()
    )


    assert int(
        stats[
            "actions"
        ]
    ) == len(
        actions
    )


    assert int(
        stats[
            "edges_added"
        ]
    ) == len(
        actions
    )


    assert (
        post_edges
        -
        pre_edges
    ) == int(
        stats[
            "net_edge_delta"
        ]
    )


    return (
        graph,
        np.asarray(
            scale,
            dtype=np.float64,
        ),
        stats,
    )


# ============================================================
# 14. EXACT step_balance
# ============================================================

step_balance_source = inspect.getsource(
    fv9.step_balance
)


STEP_BALANCE_SOURCE_SHA = hashlib.sha256(
    step_balance_source.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# 15. F1 FEATURE SEMANTICS
# ============================================================

FEATURE_SEMANTICS = {
    "detector_prob":
        "Frozen A2 detector probability.",

    "detector_logit":
        "Frozen A2 detector logit.",

    "has_prev_frame":
        (
            "1 iff frozen post-G3H3 development CURRENT_BEST "
            "has >=1 node at candidate t-1."
        ),

    "has_next_frame":
        (
            "1 iff frozen post-G3H3 development CURRENT_BEST "
            "has >=1 node at candidate t+1."
        ),

    "same_nearest_dist_um":
        (
            "Physical distance to nearest frozen post-G3H3 "
            "CURRENT_BEST node at t."
        ),

    "same_second_dist_um":
        (
            "Physical distance to second-nearest node at t."
        ),

    "same_neighbor_margin_um":
        (
            "same_second_dist_um - same_nearest_dist_um."
        ),

    "prev_nearest_dist_um":
        (
            "Physical distance to nearest node at t-1."
        ),

    "prev_second_dist_um":
        (
            "Physical distance to second-nearest node at t-1."
        ),

    "prev_neighbor_margin_um":
        (
            "prev_second_dist_um - prev_nearest_dist_um."
        ),

    "next_nearest_dist_um":
        (
            "Physical distance to nearest node at t+1."
        ),

    "next_second_dist_um":
        (
            "Physical distance to second-nearest node at t+1."
        ),

    "next_neighbor_margin_um":
        (
            "next_second_dist_um - next_nearest_dist_um."
        ),

    "prev_nearest_outdegree":
        (
            "Outdegree of nearest t-1 node in frozen "
            "post-G3H3 development CURRENT_BEST."
        ),

    "next_nearest_indegree":
        (
            "Indegree of nearest t+1 node in frozen "
            "post-G3H3 development CURRENT_BEST."
        ),

    "prev_nearest_free_out":
        (
            "1 iff nearest t-1 node has outdegree 0."
        ),

    "next_nearest_free_in":
        (
            "1 iff nearest t+1 node has indegree 0."
        ),

    "bilateral_available":
        (
            "1 iff both adjacent frames contain nodes."
        ),

    "prev_next_span_um":
        (
            "Physical distance between nearest previous "
            "and next nodes."
        ),

    "candidate_midpoint_residual_um":
        (
            "Candidate physical distance to midpoint of "
            "nearest previous/next nodes."
        ),

    "prev_next_step_mean_um":
        (
            "Mean candidate-to-prev / candidate-to-next "
            "distance."
        ),

    "prev_next_step_max_um":
        (
            "Maximum candidate endpoint distance."
        ),

    "prev_next_step_min_um":
        (
            "Minimum candidate endpoint distance."
        ),

    "prev_next_step_absdiff_um":
        (
            "Absolute endpoint-step difference."
        ),

    "step_balance":
        (
            "Exact frozen_v9.step_balance."
        ),
}


assert set(
    FEATURE_SEMANTICS
) == set(
    F1_FEATURES
)


# ============================================================
# 16. FRAME QUERY
# ============================================================

def query_frame(
    frame_info,
    query_xyz,
):

    n = len(
        query_xyz
    )


    nearest = np.full(
        n,
        np.nan,
        dtype=np.float64,
    )


    second = np.full(
        n,
        np.nan,
        dtype=np.float64,
    )


    margin = np.full(
        n,
        np.nan,
        dtype=np.float64,
    )


    nearest_global_index = np.full(
        n,
        -1,
        dtype=np.int64,
    )


    if frame_info is None:

        return (
            nearest,
            second,
            margin,
            nearest_global_index,
        )


    indices = frame_info[
        "indices"
    ]


    tree = frame_info[
        "tree"
    ]


    assert len(
        indices
    ) >= 1


    if len(
        indices
    ) == 1:

        distance, local_index = tree.query(
            query_xyz,
            k=1,
        )


        distance = np.asarray(
            distance,
            dtype=np.float64,
        ).reshape(
            -1
        )


        local_index = np.asarray(
            local_index,
            dtype=np.int64,
        ).reshape(
            -1
        )


        nearest[:] = distance


        nearest_global_index[:] = (
            indices[
                local_index
            ]
        )


    else:

        distance, local_index = tree.query(
            query_xyz,
            k=2,
        )


        distance = np.asarray(
            distance,
            dtype=np.float64,
        )


        local_index = np.asarray(
            local_index,
            dtype=np.int64,
        )


        nearest[:] = distance[
            :,
            0
        ]


        second[:] = distance[
            :,
            1
        ]


        margin[:] = (
            second
            -
            nearest
        )


        nearest_global_index[:] = (
            indices[
                local_index[
                    :,
                    0
                ]
            ]
        )


    return (
        nearest,
        second,
        margin,
        nearest_global_index,
    )


# ============================================================
# 17. FORMAL F1 MATERIALIZATION
# ============================================================

feature_parts = []

dataset_rows = []


total_actions = 0

total_removed = 0

total_added = 0

total_net_delta = 0


print(
    "\n========== FROZEN DEVELOPMENT POST-G3H3 F1 MATERIALIZATION =========="
)


for dataset_index, dataset in enumerate(
    datasets,
    start=1,
):

    sub = (
        candidate.loc[
            candidate[
                "dataset"
            ].eq(
                dataset
            )
        ]
        .copy()
        .sort_values(
            "candidate_id",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


    graph, scale, mutation_stats = (
        build_dev_post_g3h3_frozen(
            dataset
        )
    )


    scale = np.asarray(
        scale,
        dtype=np.float64,
    )


    assert scale.shape == (
        3,
    )


    assert np.isfinite(
        scale
    ).all()


    assert (
        scale
        >
        0
    ).all()


    total_actions += int(
        mutation_stats[
            "actions"
        ]
    )


    total_removed += int(
        mutation_stats[
            "source_incumbent_edges_removed"
        ]
    )


    total_added += int(
        mutation_stats[
            "edges_added"
        ]
    )


    total_net_delta += int(
        mutation_stats[
            "net_edge_delta"
        ]
    )


    # --------------------------------------------------------
    # Exact post-G3H3 node state
    # --------------------------------------------------------

    attrs = (
        graph
        .node_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    required_graph_columns = {
        "node_id",
        "t",
        "z",
        "y",
        "x",
    }


    assert required_graph_columns.issubset(
        attrs.columns
    )


    node_id = pd.to_numeric(
        attrs[
            "node_id"
        ],
        errors="raise",
    ).astype(
        np.int64
    ).to_numpy()


    node_t = pd.to_numeric(
        attrs[
            "t"
        ],
        errors="raise",
    ).astype(
        np.int64
    ).to_numpy()


    node_voxel = attrs[
        [
            "z",
            "y",
            "x",
        ]
    ].to_numpy(
        dtype=np.float64
    )


    assert np.isfinite(
        node_voxel
    ).all()


    node_phys = (
        node_voxel
        *
        scale[
            None,
            :
        ]
    )


    indegree = np.asarray(
        graph.in_degree(
            node_id.tolist()
        ),
        dtype=np.int64,
    )


    outdegree = np.asarray(
        graph.out_degree(
            node_id.tolist()
        ),
        dtype=np.int64,
    )


    # --------------------------------------------------------
    # A2 candidate/current-node collision
    # --------------------------------------------------------

    graph_keys = {
        (
            int(t),
            float(z),
            float(y),
            float(x),
        )

        for t, z, y, x
        in zip(
            node_t,
            node_voxel[
                :,
                0
            ],
            node_voxel[
                :,
                1
            ],
            node_voxel[
                :,
                2
            ],
        )
    }


    candidate_keys = [
        (
            int(row.t),
            float(row.z),
            float(row.y),
            float(row.x),
        )

        for row
        in sub.itertuples(
            index=False
        )
    ]


    exact_collision_count = sum(
        key in graph_keys

        for key
        in candidate_keys
    )


    assert exact_collision_count == 0, (
        dataset,
        exact_collision_count,
    )


    # --------------------------------------------------------
    # KD trees by frame
    # --------------------------------------------------------

    frame_info = {}


    for t in np.unique(
        node_t
    ):

        idx = np.flatnonzero(
            node_t
            ==
            t
        )


        if len(
            idx
        ) == 0:

            continue


        frame_info[
            int(
                t
            )
        ] = {
            "indices":
                idx,

            "tree":
                cKDTree(
                    node_phys[
                        idx
                    ]
                ),
        }


    candidate_voxel = sub[
        [
            "z",
            "y",
            "x",
        ]
    ].to_numpy(
        dtype=np.float64
    )


    candidate_phys = (
        candidate_voxel
        *
        scale[
            None,
            :
        ]
    )


    candidate_t = sub[
        "t"
    ].to_numpy(
        dtype=np.int64
    )


    n = len(
        sub
    )


    # --------------------------------------------------------
    # Allocate F1 arrays
    # --------------------------------------------------------

    same_nearest = np.full(
        n,
        np.nan,
    )


    same_second = np.full(
        n,
        np.nan,
    )


    same_margin = np.full(
        n,
        np.nan,
    )


    prev_nearest = np.full(
        n,
        np.nan,
    )


    prev_second = np.full(
        n,
        np.nan,
    )


    prev_margin = np.full(
        n,
        np.nan,
    )


    next_nearest = np.full(
        n,
        np.nan,
    )


    next_second = np.full(
        n,
        np.nan,
    )


    next_margin = np.full(
        n,
        np.nan,
    )


    has_prev = np.zeros(
        n,
        dtype=np.int8,
    )


    has_next = np.zeros(
        n,
        dtype=np.int8,
    )


    prev_outdegree = np.full(
        n,
        np.nan,
    )


    next_indegree = np.full(
        n,
        np.nan,
    )


    prev_free_out = np.full(
        n,
        np.nan,
    )


    next_free_in = np.full(
        n,
        np.nan,
    )


    bilateral = np.zeros(
        n,
        dtype=np.int8,
    )


    prev_next_span = np.full(
        n,
        np.nan,
    )


    midpoint_residual = np.full(
        n,
        np.nan,
    )


    step_mean = np.full(
        n,
        np.nan,
    )


    step_maximum = np.full(
        n,
        np.nan,
    )


    step_minimum = np.full(
        n,
        np.nan,
    )


    step_absdiff = np.full(
        n,
        np.nan,
    )


    step_balance_values = np.full(
        n,
        np.nan,
    )


    # --------------------------------------------------------
    # Frame-local F1 computation
    # --------------------------------------------------------

    for t in np.unique(
        candidate_t
    ):

        rows = np.flatnonzero(
            candidate_t
            ==
            t
        )


        query = candidate_phys[
            rows
        ]


        (
            d1,
            d2,
            dm,
            _same_global,
        ) = query_frame(
            frame_info.get(
                int(
                    t
                )
            ),
            query,
        )


        same_nearest[
            rows
        ] = d1


        same_second[
            rows
        ] = d2


        same_margin[
            rows
        ] = dm


        # PREVIOUS
        prev_info = frame_info.get(
            int(
                t
                -
                1
            )
        )


        (
            p1,
            p2,
            pm,
            prev_global,
        ) = query_frame(
            prev_info,
            query,
        )


        prev_nearest[
            rows
        ] = p1


        prev_second[
            rows
        ] = p2


        prev_margin[
            rows
        ] = pm


        if prev_info is not None:

            has_prev[
                rows
            ] = 1


            assert (
                prev_global
                >=
                0
            ).all()


            prev_outdegree[
                rows
            ] = outdegree[
                prev_global
            ]


            prev_free_out[
                rows
            ] = (
                outdegree[
                    prev_global
                ]
                ==
                0
            ).astype(
                float
            )


        # NEXT
        next_info = frame_info.get(
            int(
                t
                +
                1
            )
        )


        (
            n1,
            n2,
            nm,
            next_global,
        ) = query_frame(
            next_info,
            query,
        )


        next_nearest[
            rows
        ] = n1


        next_second[
            rows
        ] = n2


        next_margin[
            rows
        ] = nm


        if next_info is not None:

            has_next[
                rows
            ] = 1


            assert (
                next_global
                >=
                0
            ).all()


            next_indegree[
                rows
            ] = indegree[
                next_global
            ]


            next_free_in[
                rows
            ] = (
                indegree[
                    next_global
                ]
                ==
                0
            ).astype(
                float
            )


        # BILATERAL
        if (
            prev_info is not None
            and
            next_info is not None
        ):

            bilateral[
                rows
            ] = 1


            prev_xyz = node_phys[
                prev_global
            ]


            next_xyz = node_phys[
                next_global
            ]


            cand_xyz = query


            d_prev = np.linalg.norm(
                cand_xyz
                -
                prev_xyz,
                axis=1,
            )


            d_next = np.linalg.norm(
                next_xyz
                -
                cand_xyz,
                axis=1,
            )


            prev_next_span[
                rows
            ] = np.linalg.norm(
                next_xyz
                -
                prev_xyz,
                axis=1,
            )


            midpoint = (
                prev_xyz
                +
                next_xyz
            ) / 2.0


            midpoint_residual[
                rows
            ] = np.linalg.norm(
                cand_xyz
                -
                midpoint,
                axis=1,
            )


            step_mean[
                rows
            ] = (
                d_prev
                +
                d_next
            ) / 2.0


            step_maximum[
                rows
            ] = np.maximum(
                d_prev,
                d_next,
            )


            step_minimum[
                rows
            ] = np.minimum(
                d_prev,
                d_next,
            )


            step_absdiff[
                rows
            ] = np.abs(
                d_prev
                -
                d_next
            )


            step_balance_values[
                rows
            ] = np.asarray(
                [
                    float(
                        fv9.step_balance(
                            float(
                                a
                            ),
                            float(
                                b
                            ),
                        )
                    )

                    for a, b
                    in zip(
                        d_prev,
                        d_next
                    )
                ],
                dtype=np.float64,
            )


    part = pd.DataFrame(
        {
            "candidate_id":
                sub[
                    "candidate_id"
                ].to_numpy(
                    dtype=np.int64
                ),

            "dataset":
                sub[
                    "dataset"
                ]
                .astype(
                    str
                )
                .to_numpy(),

            "fold":
                sub[
                    "fold"
                ].to_numpy(
                    dtype=np.int8
                ),

            "t":
                sub[
                    "t"
                ].to_numpy(
                    dtype=np.int64
                ),

            "z":
                sub[
                    "z"
                ].to_numpy(
                    dtype=np.float64
                ),

            "y":
                sub[
                    "y"
                ].to_numpy(
                    dtype=np.float64
                ),

            "x":
                sub[
                    "x"
                ].to_numpy(
                    dtype=np.float64
                ),

            "detector_prob":
                sub[
                    "detector_prob"
                ].to_numpy(
                    dtype=np.float64
                ),

            "detector_logit":
                sub[
                    "detector_logit"
                ].to_numpy(
                    dtype=np.float64
                ),

            "has_prev_frame":
                has_prev,

            "has_next_frame":
                has_next,

            "same_nearest_dist_um":
                same_nearest,

            "same_second_dist_um":
                same_second,

            "same_neighbor_margin_um":
                same_margin,

            "prev_nearest_dist_um":
                prev_nearest,

            "prev_second_dist_um":
                prev_second,

            "prev_neighbor_margin_um":
                prev_margin,

            "next_nearest_dist_um":
                next_nearest,

            "next_second_dist_um":
                next_second,

            "next_neighbor_margin_um":
                next_margin,

            "prev_nearest_outdegree":
                prev_outdegree,

            "next_nearest_indegree":
                next_indegree,

            "prev_nearest_free_out":
                prev_free_out,

            "next_nearest_free_in":
                next_free_in,

            "bilateral_available":
                bilateral,

            "prev_next_span_um":
                prev_next_span,

            "candidate_midpoint_residual_um":
                midpoint_residual,

            "prev_next_step_mean_um":
                step_mean,

            "prev_next_step_max_um":
                step_maximum,

            "prev_next_step_min_um":
                step_minimum,

            "prev_next_step_absdiff_um":
                step_absdiff,

            "step_balance":
                step_balance_values,
        }
    )


    feature_parts.append(
        part
    )


    dataset_rows.append(
        {
            "dataset":
                dataset,

            "fold":
                int(
                    sub[
                        "fold"
                    ].iloc[
                        0
                    ]
                ),

            "candidate_rows":
                int(
                    n
                ),

            "canonical_nodes":
                int(
                    len(
                        attrs
                    )
                ),

            "canonical_edges":
                int(
                    graph.num_edges()
                ),

            "g3h3_actions":
                int(
                    mutation_stats[
                        "actions"
                    ]
                ),

            "g3h3_source_incumbent_edges_removed":
                int(
                    mutation_stats[
                        "source_incumbent_edges_removed"
                    ]
                ),

            "g3h3_edges_added":
                int(
                    mutation_stats[
                        "edges_added"
                    ]
                ),

            "g3h3_net_edge_delta":
                int(
                    mutation_stats[
                        "net_edge_delta"
                    ]
                ),

            "same_frame_missing":
                int(
                    np.isnan(
                        same_nearest
                    ).sum()
                ),

            "prev_frame_missing":
                int(
                    (
                        has_prev
                        ==
                        0
                    ).sum()
                ),

            "next_frame_missing":
                int(
                    (
                        has_next
                        ==
                        0
                    ).sum()
                ),

            "bilateral_available":
                int(
                    bilateral.sum()
                ),

            "exact_candidate_node_collisions":
                int(
                    exact_collision_count
                ),
        }
    )


    if (
        dataset_index == 1
        or
        dataset_index % 10 == 0
        or
        dataset_index == len(
            datasets
        )
    ):

        print(
            f"{dataset_index:3d}/{len(datasets)}"
            f" | {dataset}"
            f" | cand={n:6d}"
            f" | actions={mutation_stats['actions']:4d}"
            f" | removed={mutation_stats['source_incumbent_edges_removed']:3d}"
            f" | net={mutation_stats['net_edge_delta']:4d}"
            f" | bilateral={int(bilateral.sum()):6d}"
        )


    del graph
    del attrs
    del frame_info

    gc.collect()


# ============================================================
# 18. FROZEN DEVELOPMENT G3H3 FIDELITY
# ============================================================

print(
    "\n========== FROZEN DEVELOPMENT G3H3 REPLAY FIDELITY =========="
)


print(
    "actions:",
    total_actions,
    "expected:",
    EXPECTED_DEV_ACTIONS
)


print(
    "source removals:",
    total_removed,
    "expected:",
    EXPECTED_DEV_SOURCE_REMOVALS
)


print(
    "edges added:",
    total_added,
    "expected:",
    EXPECTED_DEV_EDGES_ADDED
)


print(
    "net edge delta:",
    total_net_delta,
    "expected:",
    EXPECTED_DEV_NET_EDGE_DELTA
)


assert total_actions == EXPECTED_DEV_ACTIONS


assert total_removed == EXPECTED_DEV_SOURCE_REMOVALS


assert total_added == EXPECTED_DEV_EDGES_ADDED


assert total_net_delta == EXPECTED_DEV_NET_EDGE_DELTA


print(
    "EXACT_FROZEN_DEVELOPMENT_G3H3_REPLAY: PASS"
)


# ============================================================
# 19. COMBINE FEATURE MATRIX
# ============================================================

features = pd.concat(
    feature_parts,
    ignore_index=True,
)


del feature_parts

gc.collect()


features = (
    features
    .sort_values(
        "candidate_id",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    features
) == EXPECTED_ROWS


assert features[
    "candidate_id"
].is_unique


assert np.array_equal(
    features[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        EXPECTED_ROWS,
        dtype=np.int64,
    ),
)


assert (
    features[
        "fold"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
    ==
    EXPECTED_FOLD_ROWS
)


dataset_summary = pd.DataFrame(
    dataset_rows
)


assert len(
    dataset_summary
) == EXPECTED_DATASETS


assert int(
    dataset_summary[
        "exact_candidate_node_collisions"
    ].sum()
) == 0


# ============================================================
# 20. MATRIX CONTRACT
# ============================================================

PROVENANCE_COLUMNS = [
    "candidate_id",
    "dataset",
    "fold",
    "t",
    "z",
    "y",
    "x",
]


MATRIX_COLUMNS = (
    PROVENANCE_COLUMNS
    +
    F1_FEATURES
)


assert list(
    features.columns
) == MATRIX_COLUMNS


# ============================================================
# 21. GT / ORACLE SAFETY
# ============================================================

for column in features.columns:

    low = str(
        column
    ).lower()


    assert not low.startswith(
        "gt_"
    )


    assert not low.startswith(
        "oracle_"
    )


    assert not low.startswith(
        "official_"
    )


print(
    "\n========== F1 SAFETY =========="
)


print(
    "A3 oracle columns:",
    "NONE"
)


print(
    "GT columns:",
    "NONE"
)


print(
    "official columns:",
    "NONE"
)


print(
    "deployment scorer executed:",
    "NO"
)


print(
    "deployment threshold used:",
    "NO"
)


# ============================================================
# 22. FEATURE VALUE CONSISTENCY
# ============================================================

assert np.isfinite(
    features[
        "detector_prob"
    ]
).all()


assert np.isfinite(
    features[
        "detector_logit"
    ]
).all()


for column in [
    "has_prev_frame",
    "has_next_frame",
    "bilateral_available",
]:

    assert set(
        features[
            column
        ]
        .astype(
            int
        )
        .unique()
        .tolist()
    ).issubset(
        {
            0,
            1,
        }
    )


assert np.array_equal(
    features[
        "bilateral_available"
    ].to_numpy(
        dtype=np.int8
    ),
    (
        features[
            "has_prev_frame"
        ].to_numpy(
            dtype=np.int8
        )
        *
        features[
            "has_next_frame"
        ].to_numpy(
            dtype=np.int8
        )
    ),
)


bilateral_mask = (
    features[
        "bilateral_available"
    ].eq(
        1
    )
)


for column in [
    "prev_next_span_um",
    "candidate_midpoint_residual_um",
    "prev_next_step_mean_um",
    "prev_next_step_max_um",
    "prev_next_step_min_um",
    "prev_next_step_absdiff_um",
    "step_balance",
]:

    assert features.loc[
        bilateral_mask,
        column,
    ].notna().all()


print(
    "F1_FEATURE_VALUE_GATE: PASS"
)


# ============================================================
# 23. GT-BLIND FEATURE AUDIT
# ============================================================

feature_audit = {}


print(
    "\n========== GT-BLIND FEATURE AVAILABILITY =========="
)


for feature in F1_FEATURES:

    series = pd.to_numeric(
        features[
            feature
        ],
        errors="coerce",
    )


    arr = series.to_numpy(
        dtype=np.float64
    )


    finite = np.isfinite(
        arr
    )


    n_finite = int(
        finite.sum()
    )


    fraction = (
        n_finite
        /
        len(
            series
        )
    )


    feature_audit[
        feature
    ] = {
        "rows":
            int(
                len(
                    series
                )
            ),

        "finite":
            n_finite,

        "finite_fraction":
            float(
                fraction
            ),

        "nan":
            int(
                series.isna().sum()
            ),
    }


    print(
        f"{feature:38s}"
        f" {n_finite:6d}/{len(series):6d}"
        f" ({fraction:.3%})"
    )


# ============================================================
# 24. CONTENT HASH
# ============================================================

CONTENT_SHA = dataframe_content_sha256(
    features,
    MATRIX_COLUMNS,
)


print(
    "\n========== F1 CONTENT HASH =========="
)


print(
    "semantic content SHA:",
    CONTENT_SHA
)


# ============================================================
# 25. FORMAL SCHEMA
# ============================================================

SCHEMA = {
    "stage":
        "12.12A5A-R3-R1-R3",

    "schema_id":
        (
            "POST_R1P50_CANONICAL_CURRENT_BEST_"
            "TEMPORAL_GEOMETRY_F1_V1"
        ),

    "status":
        "F1_FEATURE_SCHEMA_FROZEN_BEFORE_SCORING",

    "branch_id":
        BRANCH_ID,

    "feature_family_id":
        F1_ID,

    "development_current_best": {
        "pre_g3h3_builder_notebook_cell":
            EXPECTED_DEV_BUILDER_CELL_INDEX,

        "builder_cell_source_sha256":
            DEV_BUILDER_CELL_SHA,

        "builder_safe_prefix_sha256":
            DEV_BUILDER_PREFIX_SHA,

        "prediction_p995":
            EXPECTED_DEV_P995_TOKEN,

        "prediction_p9975":
            EXPECTED_DEV_P9975_TOKEN,
    },

    "development_g3h3_replay": {
        "policy_id":
            FINAL_POLICY_ID,

        "policy_sha256":
            FINAL_POLICY_SHA,

        "selected_dev_actions_artifact":
            DEV_ACTIONS_PATH.name,

        "selected_dev_actions_sha256":
            DEV_ACTIONS_SHA,

        "actions":
            EXPECTED_DEV_ACTIONS,

        "source_incumbent_removals":
            EXPECTED_DEV_SOURCE_REMOVALS,

        "target_incumbent_removals":
            0,

        "edges_added":
            EXPECTED_DEV_EDGES_ADDED,

        "net_edge_delta":
            EXPECTED_DEV_NET_EDGE_DELTA,

        "mutation_contract":
            EXPECTED_MUTATION_RULE,

        "exact_historical_mutation_helper_cell":
            MUTATION_CELL_INDEX,

        "exact_historical_mutation_helper_sha256":
            MUTATION_HELPER_SOURCE_SHA,

        "deployment_threshold_used":
            False,

        "scorer_reexecuted":
            False,
    },

    "parents": {
        "a2_contract_sha256":
            A2_CONTRACT_SHA,

        "a2_content_sha256":
            A2_CONTENT_SHA,

        "a4_contract_sha256":
            A4_CONTRACT_SHA,
    },

    "provenance_columns":
        PROVENANCE_COLUMNS,

    "model_feature_columns":
        F1_FEATURES,

    "model_feature_count":
        25,

    "feature_semantics":
        FEATURE_SEMANTICS,

    "step_balance": {
        "source_sha256":
            STEP_BALANCE_SOURCE_SHA,
    },

    "protocol": {
        "candidate_universe_modified":
            False,

        "candidate_row_order_preserved":
            True,

        "feature_matrix_frozen_before_scoring":
            True,

        "a3_oracle_read":
            False,

        "new_gt_read":
            False,

        "gt_graph_loaded":
            False,

        "official_metric_evaluated":
            False,

        "g3h3_scores_recomputed":
            False,

        "deployment_threshold_used":
            False,

        "model_trained":
            False,

        "scorer_selected":
            False,

        "threshold_selected":
            False,

        "r1p50_retuned":
            False,

        "g3h3_retuned":
            False,

        "fold0_used":
            False,

        "persistent_graph_modified":
            False,

        "ephemeral_dev_graphs_built":
            True,
    },
}


SCHEMA_SHA = hashlib.sha256(
    json.dumps(
        SCHEMA,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


SCHEMA[
    "schema_sha256"
] = SCHEMA_SHA


# ============================================================
# 26. TEMP PATHS
# ============================================================

TMP_MATRIX = FEATURE_MATRIX_OUT.with_name(
    FEATURE_MATRIX_OUT.name
    +
    ".tmp_a5a_r3_r1_r3"
)


TMP_SCHEMA = FEATURE_SCHEMA_OUT.with_name(
    FEATURE_SCHEMA_OUT.name
    +
    ".tmp_a5a_r3_r1_r3"
)


TMP_DATASET = DATASET_SUMMARY_OUT.with_name(
    DATASET_SUMMARY_OUT.name
    +
    ".tmp_a5a_r3_r1_r3"
)


TMP_SUMMARY = SUMMARY_OUT.with_name(
    SUMMARY_OUT.name
    +
    ".tmp_a5a_r3_r1_r3"
)


for path in [
    TMP_MATRIX,
    TMP_SCHEMA,
    TMP_DATASET,
    TMP_SUMMARY,
]:

    if path.exists():

        raise RuntimeError(
            "\nStale temporary R3-R1-R3 artifact:\n"
            f"{path}\n"
        )


# ============================================================
# 27. TEMP WRITE + RELOAD
# ============================================================

features.to_pickle(
    TMP_MATRIX
)


dataset_summary.to_pickle(
    TMP_DATASET
)


TMP_SCHEMA.write_text(
    json.dumps(
        SCHEMA,
        indent=2,
    ),
    encoding="utf-8",
)


matrix_check = pd.read_pickle(
    TMP_MATRIX
)


dataset_check = pd.read_pickle(
    TMP_DATASET
)


schema_check = json.loads(
    TMP_SCHEMA.read_text(
        encoding="utf-8"
    )
)


pd.testing.assert_frame_equal(
    matrix_check,
    features,
    check_exact=True,
)


pd.testing.assert_frame_equal(
    dataset_check,
    dataset_summary,
    check_exact=True,
)


assert schema_check[
    "schema_sha256"
] == SCHEMA_SHA


MATRIX_FILE_SHA = sha256_file(
    TMP_MATRIX
)


DATASET_FILE_SHA = sha256_file(
    TMP_DATASET
)


SCHEMA_FILE_SHA = sha256_file(
    TMP_SCHEMA
)


# ============================================================
# 28. FORMAL SUMMARY
# ============================================================

SUMMARY = {
    "stage":
        "12.12A5A-R3-R1-R3",

    "status":
        "CANONICAL_POST_G3H3_F1_FEATURE_MATRIX_FROZEN",

    "branch_id":
        BRANCH_ID,

    "feature_family_id":
        F1_ID,

    "frozen_dev_action_replay": {
        "artifact":
            DEV_ACTIONS_PATH.name,

        "sha256":
            DEV_ACTIONS_SHA,

        "actions":
            total_actions,

        "source_removals":
            total_removed,

        "edges_added":
            total_added,

        "net_edge_delta":
            total_net_delta,

        "fidelity_pass":
            True,
    },

    "matrix": {
        "rows":
            int(
                len(
                    features
                )
            ),

        "datasets":
            int(
                features[
                    "dataset"
                ].nunique()
            ),

        "fold_counts":
            {
                str(
                    int(
                        fold
                    )
                ):
                    int(
                        count
                    )

                for fold, count
                in (
                    features[
                        "fold"
                    ]
                    .value_counts()
                    .sort_index()
                    .items()
                )
            },

        "feature_count":
            25,

        "semantic_content_sha256":
            CONTENT_SHA,

        "artifact_file_sha256":
            MATRIX_FILE_SHA,
    },

    "schema": {
        "schema_sha256":
            SCHEMA_SHA,

        "artifact_file_sha256":
            SCHEMA_FILE_SHA,
    },

    "dataset_summary": {
        "rows":
            int(
                len(
                    dataset_summary
                )
            ),

        "artifact_file_sha256":
            DATASET_FILE_SHA,

        "bilateral_available":
            int(
                dataset_summary[
                    "bilateral_available"
                ].sum()
            ),

        "exact_candidate_node_collisions":
            int(
                dataset_summary[
                    "exact_candidate_node_collisions"
                ].sum()
            ),
    },

    "feature_availability_gt_blind":
        feature_audit,

    "protocol": {
        "feature_matrix_frozen_before_scoring":
            True,

        "a3_oracle_read":
            False,

        "gt_graph_loaded":
            False,

        "official_metric_evaluated":
            False,

        "g3h3_regenerated":
            False,

        "deployment_threshold_used":
            False,

        "model_trained":
            False,

        "scorer_selected":
            False,

        "threshold_selected":
            False,

        "fold0_used":
            False,
    },

    "next":
        (
            "12.12A5A_R4_JOIN_FROZEN_A3_LABEL_AND_"
            "RUN_PREDECLARED_LOFO_SCORER_BENCHMARK"
        ),
}


SUMMARY_SHA = hashlib.sha256(
    json.dumps(
        SUMMARY,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


SUMMARY[
    "summary_sha256"
] = SUMMARY_SHA


TMP_SUMMARY.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 29. FINAL TEMP VALIDATION
# ============================================================

summary_check = json.loads(
    TMP_SUMMARY.read_text(
        encoding="utf-8"
    )
)


assert summary_check[
    "summary_sha256"
] == SUMMARY_SHA


assert summary_check[
    "frozen_dev_action_replay"
][
    "actions"
] == EXPECTED_DEV_ACTIONS


assert summary_check[
    "frozen_dev_action_replay"
][
    "source_removals"
] == EXPECTED_DEV_SOURCE_REMOVALS


assert summary_check[
    "frozen_dev_action_replay"
][
    "net_edge_delta"
] == EXPECTED_DEV_NET_EDGE_DELTA


assert summary_check[
    "protocol"
][
    "a3_oracle_read"
] is False


assert summary_check[
    "protocol"
][
    "deployment_threshold_used"
] is False


assert summary_check[
    "protocol"
][
    "model_trained"
] is False


# ============================================================
# 30. ATOMIC PROMOTION
# ============================================================

TMP_MATRIX.replace(
    FEATURE_MATRIX_OUT
)


TMP_SCHEMA.replace(
    FEATURE_SCHEMA_OUT
)


TMP_DATASET.replace(
    DATASET_SUMMARY_OUT
)


TMP_SUMMARY.replace(
    SUMMARY_OUT
)


# ============================================================
# 31. FINAL FORMAL RELOAD
# ============================================================

assert sha256_file(
    FEATURE_MATRIX_OUT
) == MATRIX_FILE_SHA


assert sha256_file(
    FEATURE_SCHEMA_OUT
) == SCHEMA_FILE_SHA


assert sha256_file(
    DATASET_SUMMARY_OUT
) == DATASET_FILE_SHA


formal_features = pd.read_pickle(
    FEATURE_MATRIX_OUT
)


formal_schema = json.loads(
    FEATURE_SCHEMA_OUT.read_text(
        encoding="utf-8"
    )
)


formal_summary = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert len(
    formal_features
) == EXPECTED_ROWS


assert dataframe_content_sha256(
    formal_features,
    MATRIX_COLUMNS,
) == CONTENT_SHA


assert formal_schema[
    "schema_sha256"
] == SCHEMA_SHA


assert formal_summary[
    "summary_sha256"
] == SUMMARY_SHA


# ============================================================
# 32. FINAL FREEZE
# ============================================================

print(
    "\n========== 12.12A5A-R3-R1-R3 FINAL FREEZE =========="
)


print(
    "CANONICAL_POST_G3H3_F1_FEATURE_MATRIX_FREEZE: PASS"
)


print(
    "frozen development action artifact:",
    DEV_ACTIONS_PATH.name
)


print(
    "action SHA:",
    DEV_ACTIONS_SHA
)


print(
    "actions:",
    total_actions
)


print(
    "source incumbent removals:",
    total_removed
)


print(
    "target incumbent removals:",
    0
)


print(
    "edges added:",
    total_added
)


print(
    "net edge delta:",
    total_net_delta
)


print(
    "EXACT_FROZEN_DEVELOPMENT_G3H3_REPLAY:",
    "PASS"
)


print(
    "mutation helper historical cell:",
    MUTATION_CELL_INDEX
)


print(
    "mutation helper SHA:",
    MUTATION_HELPER_SOURCE_SHA
)


print(
    "candidate rows:",
    len(
        formal_features
    )
)


print(
    "datasets:",
    formal_features[
        "dataset"
    ].nunique()
)


print(
    "fold counts:",
    (
        formal_features[
            "fold"
        ]
        .value_counts()
        .sort_index()
        .to_dict()
    )
)


print(
    "F1 features:",
    len(
        F1_FEATURES
    )
)


bilateral_total = int(
    dataset_summary[
        "bilateral_available"
    ].sum()
)


print(
    "bilateral available:",
    bilateral_total,
    "/",
    EXPECTED_ROWS,
    f"({bilateral_total / EXPECTED_ROWS:.3%})"
)


print(
    "candidate/current-node collisions:",
    int(
        dataset_summary[
            "exact_candidate_node_collisions"
        ].sum()
    )
)


print(
    "schema SHA:",
    SCHEMA_SHA
)


print(
    "matrix semantic SHA:",
    CONTENT_SHA
)


print(
    "matrix file SHA:",
    MATRIX_FILE_SHA
)


print(
    "summary SHA:",
    SUMMARY_SHA
)


print(
    "A3 oracle read:",
    "NO"
)


print(
    "new GT read:",
    "NO"
)


print(
    "GT graph loaded:",
    "NO"
)


print(
    "deployment scorer executed:",
    "NO"
)


print(
    "deployment threshold used:",
    "NO"
)


print(
    "feature matrix frozen before scoring:",
    "YES"
)


print(
    "model trained:",
    "NO"
)


print(
    "scorer selected:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "official metric evaluated:",
    "NO"
)


print(
    "R1P50 retuned:",
    "NO"
)


print(
    "G3H3 retuned:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "next:"
)


print(
    "12.12A5A_R4_JOIN_FROZEN_A3_LABEL_AND_"
    "RUN_PREDECLARED_LOFO_SCORER_BENCHMARK"
)

In [ ]:
# ============================================================
# Stage 12.12A5A-R4-P0
#
# FROZEN A3 LABEL + A4 BENCHMARK CONTRACT RESOLUTION
#
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
#
# A5A-R3-R1-R3 has formally frozen the GT-blind F1 matrix.
#
# It is NOW legal to open the already-frozen A3 oracle label
# artifact for evaluation/model-development purposes.
#
# This P0 resolves:
#
#   1. exact frozen F1 artifact identity
#   2. exact A3 oracle artifact path
#   3. exact oracle_hit_7um column
#   4. exact A3 <-> F1 join identity
#   5. exact 516 / 415590 label arithmetic
#   6. exact A4 scorer / benchmark / selection contract
#
# ------------------------------------------------------------
# THIS CELL DOES NOT
# ------------------------------------------------------------
#
# train LogReg
# train HGB
# produce predictions
# compute AP/AUC
# select scorer
# select threshold
# modify graph
# use Fold0
#
# ------------------------------------------------------------
# A3 oracle read: YES
# This is legal because F1 was frozen BEFORE this cell.
#
# Delete after successful formal R4: YES
# ============================================================


from pathlib import Path

import hashlib
import json
import re

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


A2_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_universe_v1.pkl"
)


A4_PATH = (
    S12
    / "stage12_post_r1p50_node_scoring_attachment_family_v1_contract_frozen.json"
)


F1_MATRIX_PATH = (
    S12
    / "stage12_post_r1p50_node_f1_temporal_geometry_v1.pkl"
)


F1_SCHEMA_PATH = (
    S12
    / "stage12_post_r1p50_node_f1_temporal_geometry_v1_schema_frozen.json"
)


F1_SUMMARY_PATH = (
    S12
    / "stage12_post_r1p50_node_f1_temporal_geometry_v1_summary.json"
)


for path in [
    A2_PATH,
    A4_PATH,
    F1_MATRIX_PATH,
    F1_SCHEMA_PATH,
    F1_SUMMARY_PATH,
]:

    assert path.exists(), path


# ============================================================
# 1. EXPECTED FROZEN R3 IDENTITIES
# ============================================================

EXPECTED_ROWS = 415_590


EXPECTED_POSITIVES = 516


EXPECTED_NEGATIVES = 415_074


EXPECTED_POSITIVE_RATE = (
    EXPECTED_POSITIVES
    /
    EXPECTED_ROWS
)


EXPECTED_FOLD_ROWS = {
    1: 94_729,
    2: 100_318,
    3: 100_147,
    4: 120_396,
}


EXPECTED_F1_FEATURES = [
    "detector_prob",
    "detector_logit",

    "has_prev_frame",
    "has_next_frame",

    "same_nearest_dist_um",
    "same_second_dist_um",
    "same_neighbor_margin_um",

    "prev_nearest_dist_um",
    "prev_second_dist_um",
    "prev_neighbor_margin_um",

    "next_nearest_dist_um",
    "next_second_dist_um",
    "next_neighbor_margin_um",

    "prev_nearest_outdegree",
    "next_nearest_indegree",

    "prev_nearest_free_out",
    "next_nearest_free_in",

    "bilateral_available",

    "prev_next_span_um",
    "candidate_midpoint_residual_um",

    "prev_next_step_mean_um",
    "prev_next_step_max_um",
    "prev_next_step_min_um",
    "prev_next_step_absdiff_um",

    "step_balance",
]


EXPECTED_MATRIX_FILE_SHA = (
    "8c3962bf6cef48617e1bd9ef23bd5e6d"
    "eb2438b80b6916f87d171516be92d0cc"
)


EXPECTED_MATRIX_CONTENT_SHA = (
    "97ea93651d289e3905ce6e3103278ba4"
    "cef72fed86a6b7c663ecb4d8fca0da53"
)


EXPECTED_SCHEMA_SHA = (
    "a4c5fc4cf55b04768d623fab585ecd1b"
    "65701d0ca4667347941b4e4990a38f81"
)


EXPECTED_R3_SUMMARY_SHA = (
    "63698900438e3eff28c399212f120e348"
    "dacf65275c255c3fe72506ad2674615"
)


EXPECTED_A4_SHA = (
    "47e173fb2be1dd00ebb9c3a789dd3c0b"
    "a7b98863815f16904ee1546b5e6bba58"
)


# ============================================================
# 2. HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            h.update(block)

    return h.hexdigest()


class HashWriter:

    def __init__(self, hasher):

        self.hasher = hasher


    def write(self, text):

        if not isinstance(text, str):

            text = str(text)

        self.hasher.update(
            text.encode("utf-8")
        )

        return len(text)


    def flush(self):

        return None


def dataframe_content_sha256(
    df,
    columns,
):

    h = hashlib.sha256()

    writer = HashWriter(h)

    df.loc[:, columns].to_csv(
        writer,
        index=False,
        na_rep="NaN",
        float_format="%.17g",
        lineterminator="\n",
    )

    return h.hexdigest()


def recursive_hits(
    obj,
    predicate,
    path="root",
):

    rows = []


    if isinstance(obj, dict):

        for key, value in obj.items():

            child = f"{path}.{key}"


            if predicate(
                key,
                value,
            ):

                rows.append(
                    {
                        "path": child,
                        "key": key,
                        "value": value,
                    }
                )


            rows.extend(
                recursive_hits(
                    value,
                    predicate,
                    child,
                )
            )


    elif isinstance(obj, list):

        for i, value in enumerate(obj):

            rows.extend(
                recursive_hits(
                    value,
                    predicate,
                    f"{path}[{i}]",
                )
            )


    return rows


def recursive_dicts_containing(
    obj,
    needle,
    path="root",
):

    rows = []


    needle_upper = str(needle).upper()


    if isinstance(obj, dict):

        serialized = json.dumps(
            obj,
            sort_keys=True,
            default=str,
        ).upper()


        if needle_upper in serialized:

            rows.append(
                {
                    "path": path,
                    "value": obj,
                }
            )


        for key, value in obj.items():

            rows.extend(
                recursive_dicts_containing(
                    value,
                    needle,
                    f"{path}.{key}",
                )
            )


    elif isinstance(obj, list):

        for i, value in enumerate(obj):

            rows.extend(
                recursive_dicts_containing(
                    value,
                    needle,
                    f"{path}[{i}]",
                )
            )


    return rows


# ============================================================
# 3. F1 FORMAL FREEZE LOCK
# ============================================================

assert sha256_file(
    F1_MATRIX_PATH
) == EXPECTED_MATRIX_FILE_SHA


f1 = pd.read_pickle(
    F1_MATRIX_PATH
)


f1_schema = json.loads(
    F1_SCHEMA_PATH.read_text(
        encoding="utf-8"
    )
)


f1_summary = json.loads(
    F1_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert len(f1) == EXPECTED_ROWS


assert f1_schema[
    "schema_sha256"
] == EXPECTED_SCHEMA_SHA


assert f1_summary[
    "summary_sha256"
] == EXPECTED_R3_SUMMARY_SHA


assert f1_schema[
    "protocol"
][
    "feature_matrix_frozen_before_scoring"
] is True


assert f1_schema[
    "protocol"
][
    "a3_oracle_read"
] is False


assert f1_schema[
    "model_feature_columns"
] == EXPECTED_F1_FEATURES


PROVENANCE_COLUMNS = [
    "candidate_id",
    "dataset",
    "fold",
    "t",
    "z",
    "y",
    "x",
]


MATRIX_COLUMNS = (
    PROVENANCE_COLUMNS
    +
    EXPECTED_F1_FEATURES
)


assert dataframe_content_sha256(
    f1,
    MATRIX_COLUMNS,
) == EXPECTED_MATRIX_CONTENT_SHA


assert (
    f1[
        "fold"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
    ==
    EXPECTED_FOLD_ROWS
)


assert np.array_equal(
    f1[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        EXPECTED_ROWS,
        dtype=np.int64,
    ),
)


print(
    "========== 12.12A5A-R4-P0 F1 FREEZE LOCK =========="
)


print(
    "rows:",
    len(f1)
)


print(
    "features:",
    len(EXPECTED_F1_FEATURES)
)


print(
    "matrix file SHA:",
    EXPECTED_MATRIX_FILE_SHA
)


print(
    "matrix semantic SHA:",
    EXPECTED_MATRIX_CONTENT_SHA
)


print(
    "schema SHA:",
    EXPECTED_SCHEMA_SHA
)


print(
    "F1 frozen before A3:",
    "YES"
)


# ============================================================
# 4. A4 CONTRACT LOCK
# ============================================================

a4 = json.loads(
    A4_PATH.read_text(
        encoding="utf-8"
    )
)


assert a4[
    "contract_sha256"
] == EXPECTED_A4_SHA


assert a4[
    "protocol"
][
    "fold0_used"
] is False


assert a4[
    "protocol"
][
    "model_trained"
] is False


print(
    "\n========== A4 CONTRACT LOCK =========="
)


print(
    "A4 SHA:",
    EXPECTED_A4_SHA
)


print(
    "F1 family:",
    a4[
        "feature_families"
    ][
        "F1"
    ][
        "family_id"
    ]
)


# ============================================================
# 5. FIND EXACT FROZEN A3 ORACLE ARTIFACT
#
# F1 IS ALREADY FROZEN.
#
# Reading A3 is now protocol-legal.
# ============================================================

candidate_paths = []


patterns = [
    "*post*r1p50*oracle*.pkl",
    "*post*r1p50*coverage*.pkl",
    "*post*r1p50*a3*.pkl",
    "*node*oracle*.pkl",
    "*oracle*coverage*.pkl",
]


for pattern in patterns:

    for path in S12.glob(pattern):

        if path not in candidate_paths:

            candidate_paths.append(path)


candidate_paths = sorted(
    candidate_paths,
    key=lambda p: p.name,
)


print(
    "\n========== A3 ORACLE CANDIDATE ARTIFACTS =========="
)


for path in candidate_paths:

    print(
        path.name,
        "| bytes=",
        path.stat().st_size,
    )


assert candidate_paths, (
    "No plausible frozen A3 oracle artifact found."
)


a3_matches = []


for path in candidate_paths:

    try:

        frame = pd.read_pickle(path)

    except Exception as exc:

        print(
            "SKIP unreadable:",
            path.name,
            repr(exc),
        )

        continue


    if not isinstance(
        frame,
        pd.DataFrame,
    ):

        continue


    if "oracle_hit_7um" not in frame.columns:

        continue


    label = pd.to_numeric(
        frame[
            "oracle_hit_7um"
        ],
        errors="raise",
    ).astype(int)


    positives = int(
        label.sum()
    )


    rows = int(
        len(frame)
    )


    print()
    print(
        "POSSIBLE A3:",
        path.name
    )


    print(
        "rows:",
        rows
    )


    print(
        "positives:",
        positives
    )


    print(
        "columns:",
        list(frame.columns)
    )


    if (
        rows == EXPECTED_ROWS
        and
        positives == EXPECTED_POSITIVES
    ):

        a3_matches.append(
            (
                path,
                frame,
            )
        )


assert len(a3_matches) == 1, (
    "Expected exactly one A3 artifact matching "
    "415590 rows / 516 positives.",
    [
        path.name
        for path, _frame
        in a3_matches
    ],
)


A3_PATH, a3 = a3_matches[0]


A3_FILE_SHA = sha256_file(
    A3_PATH
)


print(
    "\n========== EXACT A3 ARTIFACT RESOLVED =========="
)


print(
    "path:",
    A3_PATH
)


print(
    "SHA:",
    A3_FILE_SHA
)


print(
    "rows:",
    len(a3)
)


print(
    "positives:",
    int(
        pd.to_numeric(
            a3[
                "oracle_hit_7um"
            ],
            errors="raise",
        )
        .astype(int)
        .sum()
    )
)


# ============================================================
# 6. LABEL CONTRACT
# ============================================================

label = pd.to_numeric(
    a3[
        "oracle_hit_7um"
    ],
    errors="raise",
).astype(
    np.int8
)


assert set(
    label.unique().tolist()
).issubset(
    {
        0,
        1,
    }
)


assert int(
    label.sum()
) == EXPECTED_POSITIVES


assert int(
    (
        label == 0
    ).sum()
) == EXPECTED_NEGATIVES


observed_rate = float(
    label.mean()
)


assert abs(
    observed_rate
    -
    EXPECTED_POSITIVE_RATE
) < 1e-15


print(
    "\n========== A3 LABEL CONTRACT =========="
)


print(
    "label:",
    "oracle_hit_7um"
)


print(
    "positive:",
    EXPECTED_POSITIVES
)


print(
    "negative:",
    EXPECTED_NEGATIVES
)


print(
    "prevalence:",
    observed_rate
)


print(
    "percent:",
    f"{100 * observed_rate:.6f}%"
)


# ============================================================
# 7. RESOLVE EXACT A3 <-> F1 JOIN
#
# Preference:
#
#   A. candidate_id
#
# else:
#
#   B. exact candidate provenance identity
#
# No positional join is accepted unless identity itself proves
# exact frozen row equivalence.
# ============================================================

JOIN_MODE = None


if "candidate_id" in a3.columns:

    a3_candidate_id = pd.to_numeric(
        a3[
            "candidate_id"
        ],
        errors="raise",
    ).astype(
        np.int64
    )


    if (
        len(a3_candidate_id)
        ==
        EXPECTED_ROWS
        and
        a3_candidate_id.is_unique
        and
        set(
            a3_candidate_id.tolist()
        )
        ==
        set(
            range(
                EXPECTED_ROWS
            )
        )
    ):

        JOIN_MODE = "candidate_id"


if JOIN_MODE is None:

    possible_key = [
        "dataset",
        "t",
        "z",
        "y",
        "x",
    ]


    if set(
        possible_key
    ).issubset(
        a3.columns
    ):

        assert not a3[
            possible_key
        ].duplicated().any()


        assert not f1[
            possible_key
        ].duplicated().any()


        JOIN_MODE = (
            "dataset_t_z_y_x"
        )


assert JOIN_MODE is not None, (
    "Could not prove an exact identity join between frozen "
    "F1 and frozen A3."
)


print(
    "\n========== A3/F1 JOIN CONTRACT =========="
)


print(
    "join mode:",
    JOIN_MODE
)


# ============================================================
# 8. EXECUTE JOIN ONLY FOR IDENTITY VERIFICATION
#
# No scoring/modeling.
# ============================================================

if JOIN_MODE == "candidate_id":

    label_frame = (
        a3[
            [
                "candidate_id",
                "oracle_hit_7um",
            ]
        ]
        .copy()
    )


    label_frame[
        "candidate_id"
    ] = pd.to_numeric(
        label_frame[
            "candidate_id"
        ],
        errors="raise",
    ).astype(
        np.int64
    )


    joined = f1[
        [
            "candidate_id",
            "dataset",
            "fold",
        ]
    ].merge(
        label_frame,
        on="candidate_id",
        how="left",
        validate="one_to_one",
        sort=False,
    )


else:

    key = [
        "dataset",
        "t",
        "z",
        "y",
        "x",
    ]


    label_frame = (
        a3[
            key
            +
            [
                "oracle_hit_7um"
            ]
        ]
        .copy()
    )


    joined = f1[
        [
            "candidate_id",
            "dataset",
            "fold",
            "t",
            "z",
            "y",
            "x",
        ]
    ].merge(
        label_frame,
        on=key,
        how="left",
        validate="one_to_one",
        sort=False,
    )


assert len(joined) == EXPECTED_ROWS


assert joined[
    "oracle_hit_7um"
].notna().all()


joined[
    "oracle_hit_7um"
] = pd.to_numeric(
    joined[
        "oracle_hit_7um"
    ],
    errors="raise",
).astype(
    np.int8
)


assert int(
    joined[
        "oracle_hit_7um"
    ].sum()
) == EXPECTED_POSITIVES


assert np.array_equal(
    joined[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        EXPECTED_ROWS,
        dtype=np.int64,
    ),
)


print(
    "A3_F1_ONE_TO_ONE_JOIN: PASS"
)


# ============================================================
# 9. LABEL COUNTS BY HELD-OUT FOLD
#
# Audit only.
# ============================================================

fold_label_summary = (
    joined.groupby(
        "fold",
        sort=True,
    )[
        "oracle_hit_7um"
    ]
    .agg(
        rows="size",
        positives="sum",
        prevalence="mean",
    )
    .reset_index()
)


print(
    "\n========== A3 LABEL DISTRIBUTION BY FOLD =========="
)


print(
    fold_label_summary.to_string(
        index=False
    )
)


assert int(
    fold_label_summary[
        "rows"
    ].sum()
) == EXPECTED_ROWS


assert int(
    fold_label_summary[
        "positives"
    ].sum()
) == EXPECTED_POSITIVES


# ============================================================
# 10. EXTRACT RELEVANT A4 CONTRACT OBJECTS
# ============================================================

TARGET_IDS = [
    "DETECTOR_PROB_BASELINE_V1",
    "LOGREG_L2_C1_BALANCED_V1",
    "HGB_BALANCED_SMALL_V1",
    "A2_NODE_RARE_POSITIVE_LOFO_BENCHMARK_V1",
]


print(
    "\n========== A4 PREDECLARED OBJECTS =========="
)


resolved_contract_objects = {}


for target_id in TARGET_IDS:

    hits = recursive_dicts_containing(
        a4,
        target_id,
    )


    # Remove duplicate ancestors by preferring the smallest
    # matching serialized dictionaries.
    hits = sorted(
        hits,
        key=lambda row: len(
            json.dumps(
                row[
                    "value"
                ],
                default=str,
            )
        ),
    )


    assert hits, target_id


    resolved_contract_objects[
        target_id
    ] = hits[0]


    print()
    print(
        "-" * 100
    )


    print(
        "ID:",
        target_id
    )


    print(
        "contract path:",
        hits[0][
            "path"
        ]
    )


    print(
        json.dumps(
            hits[0][
                "value"
            ],
            indent=2,
            default=str,
        )
    )


# ============================================================
# 11. HARD A4 PREDECLARED TOKENS
#
# These are not model results. They merely verify the already-
# frozen benchmark contract before training.
# ============================================================

a4_text = json.dumps(
    a4,
    sort_keys=True,
    default=str,
)


required_tokens = [
    "DETECTOR_PROB_BASELINE_V1",
    "LOGREG_L2_C1_BALANCED_V1",
    "HGB_BALANCED_SMALL_V1",

    "A2_NODE_RARE_POSITIVE_LOFO_BENCHMARK_V1",

    "oracle_hit_7um",

    "average_precision",
]


for token in required_tokens:

    assert token.lower() in a4_text.lower(), token


for budget in [
    0.001,
    0.0025,
    0.005,
    0.01,
]:

    # tolerate ordinary JSON float formatting
    assert str(
        budget
    ) in a4_text, budget


print(
    "\nA4 required scorer/benchmark tokens: PASS"
)


# ============================================================
# 12. VERIFY PREDECLARED MODEL FEATURE LIST
# ============================================================

assert a4[
    "feature_families"
][
    "F1"
][
    "features"
] == EXPECTED_F1_FEATURES


assert list(
    f1_schema[
        "model_feature_columns"
    ]
) == EXPECTED_F1_FEATURES


print(
    "\n========== MODEL FEATURE IMMUTABILITY =========="
)


print(
    "A4 F1 features:",
    len(
        EXPECTED_F1_FEATURES
    )
)


print(
    "Frozen R3 features:",
    len(
        f1_schema[
            "model_feature_columns"
        ]
    )
)


print(
    "feature order exact:",
    "YES"
)


print(
    "new features allowed in R4:",
    "NO"
)


# ============================================================
# 13. CHECK SCORER PARAMETER TOKENS
#
# Verify the known frozen specifications exist in the A4
# contract representation.
# ============================================================

parameter_tokens = [
    # S1
    "median",
    "l2",
    "balanced",
    "lbfgs",

    # S2
    "0.05",
    "100",
    "15",
    "20",
]


missing_parameter_tokens = [
    token

    for token
    in parameter_tokens

    if token.lower()
    not in a4_text.lower()
]


print(
    "\n========== A4 PARAMETER TOKEN AUDIT =========="
)


print(
    "missing:",
    missing_parameter_tokens
)


assert not missing_parameter_tokens


print(
    "PREDECLARED_SCORER_PARAMETER_CONTRACT_PRESENT: PASS"
)


# ============================================================
# 14. SELECTION / CONTINUATION RULE DISCOVERY
# ============================================================

selection_hits = recursive_hits(
    a4,
    lambda key, value: (
        any(
            token in str(key).lower()

            for token in [
                "selection",
                "continuation",
                "gate",
                "tie",
                "primary",
                "worst",
            ]
        )
    ),
)


print(
    "\n========== A4 SELECTION / GATE CONTRACT =========="
)


for row in selection_hits:

    print(
        row[
            "path"
        ],
        "=",
        row[
            "value"
        ],
    )


assert selection_hits


# ============================================================
# 15. R4-P0 SAFETY
# ============================================================

print(
    "\n========== 12.12A5A-R4-P0 SAFETY =========="
)


print(
    "F1 feature matrix modified:",
    "NO"
)


print(
    "A3 oracle read:",
    "YES — AFTER FORMAL F1 FREEZE"
)


print(
    "GT graph read:",
    "NO"
)


print(
    "model trained:",
    "NO"
)


print(
    "predictions generated:",
    "NO"
)


print(
    "AP computed:",
    "NO"
)


print(
    "ROC-AUC computed:",
    "NO"
)


print(
    "scorer selected:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "graph modified:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


# ============================================================
# 16. FINAL DECISION
# ============================================================

print(
    "\n========== 12.12A5A-R4-P0 DECISION =========="
)


print(
    "F1 formal freeze:",
    "PASS"
)


print(
    "A3 path:",
    A3_PATH.name
)


print(
    "A3 file SHA:",
    A3_FILE_SHA
)


print(
    "A3/F1 join:",
    JOIN_MODE
)


print(
    "rows:",
    len(joined)
)


print(
    "positive labels:",
    int(
        joined[
            "oracle_hit_7um"
        ].sum()
    )
)


print(
    "negative labels:",
    int(
        (
            joined[
                "oracle_hit_7um"
            ]
            ==
            0
        ).sum()
    )
)


print(
    "positive rate:",
    float(
        joined[
            "oracle_hit_7um"
        ].mean()
    )
)


print(
    "A4 scorer family resolved:",
    "YES"
)


print(
    "A4 benchmark contract resolved:",
    "YES"
)


print(
    "model feature schema immutable:",
    "YES"
)


print(
    "model trained:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "next:"
)


print(
    "12.12A5A_R4_FORMAL_PREDECLARED_"
    "S0_S1_S2_LOFO_RANKING_BENCHMARK"
)

In [ ]:
# ============================================================
# Stage 12.12A5A-R4
#
# FORMAL PREDECLARED
# S0 / S1 / S2 LOFO RARE-POSITIVE RANKING BENCHMARK
#
# ============================================================
#
# PARENTS
# ------------------------------------------------------------
#
# A3:
#   frozen oracle_hit_7um labels
#   415,590 rows
#   516 positives
#
# A4:
#   scorer families / benchmark / selection frozen BEFORE
#   F1 materialization and scoring
#
# R3-R1-R3:
#   frozen GT-blind F1 feature matrix
#   415,590 rows
#   25 features
#
# ------------------------------------------------------------
# BENCHMARK
# ------------------------------------------------------------
#
# S0:
#   DETECTOR_PROB_DESCENDING_V1
#
# S1:
#   LOGREG_L2_C1_BALANCED_V1
#
#   train-fold median imputation
#   train-fold StandardScaler
#   LogisticRegression:
#       penalty=l2
#       C=1
#       solver=lbfgs
#       class_weight=balanced
#       max_iter=1000
#
# S2:
#   HGB_BALANCED_SMALL_V1
#
#   HistGradientBoostingClassifier:
#       learning_rate=0.05
#       max_iter=100
#       max_leaf_nodes=15
#       max_depth=None
#       l2_regularization=1.0
#       min_samples_leaf=20
#       random_state=0
#
#   Missing-value resolution:
#       NATIVE HGB missing-value handling.
#
#   Class balance:
#       explicit TRAIN-FOLD sample weights using
#       sklearn.compute_sample_weight("balanced")
#
#       equivalent binary formula:
#
#           w_c = N_train / (2 * N_train,c)
#
# ------------------------------------------------------------
# CV
# ------------------------------------------------------------
#
# folds = 1,2,3,4
# leave one fold out
#
# Every transformation / model fit uses ONLY training folds.
#
# ------------------------------------------------------------
# PRIMARY SELECTION
# ------------------------------------------------------------
#
# 1. macro held-out AP descending
# 2. worst-fold AP descending
# 3. complexity:
#
#       detector
#       logreg
#       HGB
#
# tie tolerance = 1e-6
#
# ------------------------------------------------------------
# CONTINUATION GATE
# ------------------------------------------------------------
#
# selected macro AP > detector macro AP
#
# AND
#
# selected worst-fold AP >= detector worst-fold AP
#
# If fail:
#
#   CLOSE_BRANCH_NO_GRAPH_MUTATION
#
# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------
#
# NO deployment threshold
# NO graph metric
# NO graph mutation
# NO Fold0
# NO post-selection model retuning
#
# Development GT is used ONLY as the frozen A3 benchmark label.
#
# ------------------------------------------------------------
# KEEP AFTER RUNNING: YES
# ============================================================


from pathlib import Path

import hashlib
import json
import math
import platform
import sys
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


A3_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_candidates_v1.pkl"
)


A4_PATH = (
    S12
    / "stage12_post_r1p50_node_scoring_attachment_family_v1_contract_frozen.json"
)


F1_MATRIX_PATH = (
    S12
    / "stage12_post_r1p50_node_f1_temporal_geometry_v1.pkl"
)


F1_SCHEMA_PATH = (
    S12
    / "stage12_post_r1p50_node_f1_temporal_geometry_v1_schema_frozen.json"
)


F1_SUMMARY_PATH = (
    S12
    / "stage12_post_r1p50_node_f1_temporal_geometry_v1_summary.json"
)


# ------------------------------------------------------------
# FORMAL R4 OUTPUTS
# ------------------------------------------------------------

OOF_OUT = (
    S12
    / "stage12_post_r1p50_node_f1_lofo_scorer_benchmark_v1_oof.pkl"
)


FOLD_METRICS_OUT = (
    S12
    / "stage12_post_r1p50_node_f1_lofo_scorer_benchmark_v1_fold_metrics.pkl"
)


BUDGET_METRICS_OUT = (
    S12
    / "stage12_post_r1p50_node_f1_lofo_scorer_benchmark_v1_budget_metrics.pkl"
)


MODEL_AUDIT_OUT = (
    S12
    / "stage12_post_r1p50_node_f1_lofo_scorer_benchmark_v1_model_audit.pkl"
)


SELECTION_OUT = (
    S12
    / "stage12_post_r1p50_node_scorer_selection_v1_frozen.json"
)


SUMMARY_OUT = (
    S12
    / "stage12_post_r1p50_node_f1_lofo_scorer_benchmark_v1_summary.json"
)


for path in [
    A3_PATH,
    A4_PATH,
    F1_MATRIX_PATH,
    F1_SCHEMA_PATH,
    F1_SUMMARY_PATH,
]:

    assert path.exists(), path


for path in [
    OOF_OUT,
    FOLD_METRICS_OUT,
    BUDGET_METRICS_OUT,
    MODEL_AUDIT_OUT,
    SELECTION_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nFormal R4 artifact already exists:\n"
            f"{path}\n\n"
            "Do NOT overwrite formal scorer provenance."
        )


# ============================================================
# 1. FROZEN IDENTITIES
# ============================================================

EXPECTED_ROWS = 415_590


EXPECTED_POSITIVES = 516


EXPECTED_NEGATIVES = 415_074


EXPECTED_FOLD_ROWS = {
    1: 94_729,
    2: 100_318,
    3: 100_147,
    4: 120_396,
}


EXPECTED_FOLD_POSITIVES = {
    1: 173,
    2: 150,
    3: 104,
    4: 89,
}


EXPECTED_A3_SHA = (
    "2319c69e670438bd0742f8cb84552041"
    "e1656399f11c0f0f41c7251ba4a0ed02"
)


EXPECTED_A4_SHA = (
    "47e173fb2be1dd00ebb9c3a789dd3c0b"
    "a7b98863815f16904ee1546b5e6bba58"
)


EXPECTED_F1_MATRIX_FILE_SHA = (
    "8c3962bf6cef48617e1bd9ef23bd5e6d"
    "eb2438b80b6916f87d171516be92d0cc"
)


EXPECTED_F1_CONTENT_SHA = (
    "97ea93651d289e3905ce6e3103278ba4"
    "cef72fed86a6b7c663ecb4d8fca0da53"
)


EXPECTED_F1_SCHEMA_SHA = (
    "a4c5fc4cf55b04768d623fab585ecd1b"
    "65701d0ca4667347941b4e4990a38f81"
)


EXPECTED_F1_SUMMARY_SHA = (
    "63698900438e3eff28c399212f120e348"
    "dacf65275c255c3fe72506ad2674615"
)


BENCHMARK_ID = (
    "A2_NODE_RARE_POSITIVE_LOFO_BENCHMARK_V1"
)


SELECTION_ID = (
    "A2_NODE_SCORER_SELECTION_V1"
)


S0_ID = (
    "DETECTOR_PROB_DESCENDING_V1"
)


S1_ID = (
    "LOGREG_L2_C1_BALANCED_V1"
)


S2_ID = (
    "HGB_BALANCED_SMALL_V1"
)


F1_ID = (
    "CANONICAL_CURRENT_BEST_TEMPORAL_GEOMETRY_V1"
)


LABEL_ID = (
    "A3_ORACLE_HIT_7UM_V1"
)


LABEL_COLUMN = (
    "oracle_hit_7um"
)


FOLDS = [
    1,
    2,
    3,
    4,
]


BUDGET_FRACTIONS = [
    0.001,
    0.0025,
    0.005,
    0.01,
]


TIE_TOLERANCE = 1e-6


# ============================================================
# 2. EXACT F1 FEATURE ORDER
# ============================================================

F1_FEATURES = [
    "detector_prob",
    "detector_logit",

    "has_prev_frame",
    "has_next_frame",

    "same_nearest_dist_um",
    "same_second_dist_um",
    "same_neighbor_margin_um",

    "prev_nearest_dist_um",
    "prev_second_dist_um",
    "prev_neighbor_margin_um",

    "next_nearest_dist_um",
    "next_second_dist_um",
    "next_neighbor_margin_um",

    "prev_nearest_outdegree",
    "next_nearest_indegree",

    "prev_nearest_free_out",
    "next_nearest_free_in",

    "bilateral_available",

    "prev_next_span_um",

    "candidate_midpoint_residual_um",

    "prev_next_step_mean_um",
    "prev_next_step_max_um",
    "prev_next_step_min_um",

    "prev_next_step_absdiff_um",

    "step_balance",
]


assert len(
    F1_FEATURES
) == 25


# ============================================================
# 3. HASH HELPERS
# ============================================================

def sha256_file(
    path,
):

    h = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:

                break

            h.update(
                block
            )

    return h.hexdigest()


class HashWriter:

    def __init__(
        self,
        hasher,
    ):

        self.hasher = hasher


    def write(
        self,
        text,
    ):

        if not isinstance(
            text,
            str,
        ):

            text = str(
                text
            )

        self.hasher.update(
            text.encode(
                "utf-8"
            )
        )

        return len(
            text
        )


    def flush(
        self,
    ):

        return None


def dataframe_content_sha256(
    df,
    columns,
):

    h = hashlib.sha256()

    writer = HashWriter(
        h
    )

    df.loc[
        :,
        columns,
    ].to_csv(
        writer,
        index=False,
        na_rep="NaN",
        float_format="%.17g",
        lineterminator="\n",
    )

    return h.hexdigest()


# ============================================================
# 4. LOCK FORMAL PARENTS
# ============================================================

assert sha256_file(
    A3_PATH
) == EXPECTED_A3_SHA


assert sha256_file(
    F1_MATRIX_PATH
) == EXPECTED_F1_MATRIX_FILE_SHA


a4 = json.loads(
    A4_PATH.read_text(
        encoding="utf-8"
    )
)


f1_schema = json.loads(
    F1_SCHEMA_PATH.read_text(
        encoding="utf-8"
    )
)


f1_summary = json.loads(
    F1_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert a4[
    "contract_sha256"
] == EXPECTED_A4_SHA


assert f1_schema[
    "schema_sha256"
] == EXPECTED_F1_SCHEMA_SHA


assert f1_summary[
    "summary_sha256"
] == EXPECTED_F1_SUMMARY_SHA


assert f1_schema[
    "model_feature_columns"
] == F1_FEATURES


assert f1_schema[
    "protocol"
][
    "feature_matrix_frozen_before_scoring"
] is True


assert f1_schema[
    "protocol"
][
    "a3_oracle_read"
] is False


assert a4[
    "benchmark"
][
    "benchmark_id"
] == BENCHMARK_ID


assert a4[
    "benchmark"
][
    "development_folds"
] == FOLDS


assert a4[
    "benchmark"
][
    "cross_validation"
] == "LEAVE_ONE_FOLD_OUT"


assert a4[
    "benchmark"
][
    "label"
] == LABEL_ID


assert a4[
    "benchmark"
][
    "primary_metric"
] == "AVERAGE_PRECISION"


assert a4[
    "benchmark"
][
    "aggregation"
] == "MACRO_MEAN_ACROSS_HELD_OUT_FOLDS"


assert a4[
    "benchmark"
][
    "ranking_budget_fractions"
] == BUDGET_FRACTIONS


assert a4[
    "benchmark"
][
    "no_threshold_selection"
] is True


assert a4[
    "benchmark"
][
    "no_graph_metric"
] is True


assert a4[
    "benchmark"
][
    "no_fold0"
] is True


assert a4[
    "selection"
][
    "selection_id"
] == SELECTION_ID


assert a4[
    "selection"
][
    "baseline"
] == S0_ID


assert a4[
    "selection"
][
    "tie_tolerance"
] == TIE_TOLERANCE


assert a4[
    "selection"
][
    "no_post_selection_model_retuning"
] is True


print(
    "========== 12.12A5A-R4 PARENT LOCK =========="
)


print(
    "A3 SHA:",
    EXPECTED_A3_SHA
)


print(
    "A4 SHA:",
    EXPECTED_A4_SHA
)


print(
    "F1 matrix SHA:",
    EXPECTED_F1_MATRIX_FILE_SHA
)


print(
    "F1 semantic SHA:",
    EXPECTED_F1_CONTENT_SHA
)


print(
    "benchmark:",
    BENCHMARK_ID
)


print(
    "selection:",
    SELECTION_ID
)


print(
    "Fold0 used:",
    "NO"
)


# ============================================================
# 5. LOCK SCORER CONTRACTS
# ============================================================

S0 = a4[
    "scorer_families"
][
    "S0"
]


S1 = a4[
    "scorer_families"
][
    "S1"
]


S2 = a4[
    "scorer_families"
][
    "S2"
]


assert S0[
    "scorer_id"
] == S0_ID


assert S0[
    "feature_family"
] == "DETECTOR_PROB_BASELINE_V1"


assert S0[
    "trained"
] is False


assert S0[
    "score"
] == "detector_prob"


assert S1[
    "scorer_id"
] == S1_ID


assert S1[
    "preprocessing"
][
    "numeric_imputation"
] == "TRAIN_FOLD_MEDIAN"


assert S1[
    "preprocessing"
][
    "standardization"
] == "TRAIN_FOLD_STANDARD_SCALER"


assert S1[
    "penalty"
] == "l2"


assert float(
    S1[
        "C"
    ]
) == 1.0


assert S1[
    "solver"
] == "lbfgs"


assert S1[
    "class_weight"
] == "balanced"


assert int(
    S1[
        "max_iter"
    ]
) == 1000


assert S2[
    "scorer_id"
] == S2_ID


assert float(
    S2[
        "learning_rate"
    ]
) == 0.05


assert int(
    S2[
        "max_iter"
    ]
) == 100


assert int(
    S2[
        "max_leaf_nodes"
    ]
) == 15


assert S2[
    "max_depth"
] is None


assert float(
    S2[
        "l2_regularization"
    ]
) == 1.0


assert int(
    S2[
        "min_samples_leaf"
    ]
) == 20


assert int(
    S2[
        "random_state"
    ]
) == 0


assert S2[
    "class_balance"
] == (
    "EXPLICIT_TRAIN_FOLD_SAMPLE_WEIGHTS_"
    "INVERSE_CLASS_FREQUENCY"
)


# ------------------------------------------------------------
# Mechanical resolution of A4's allowed HGB missing policy.
#
# No metric has been computed yet.
#
# HistGradientBoostingClassifier has native NaN handling;
# therefore select the explicitly allowed NATIVE branch.
#
# This is NOT a hyperparameter choice and is NOT GT-driven.
# ------------------------------------------------------------

assert S2[
    "preprocessing"
][
    "numeric_imputation"
] == (
    "NATIVE_OR_TRAIN_FOLD_MEDIAN_NO_GT_FEATURE"
)


HGB_MISSING_POLICY = (
    "NATIVE"
)


HGB_CLASS_WEIGHT_IMPLEMENTATION = (
    "sklearn.utils.class_weight."
    "compute_sample_weight(class_weight='balanced')"
)


HGB_CLASS_WEIGHT_FORMULA = (
    "N_train / (2 * N_train_class)"
)


print(
    "\n========== SCORER CONTRACT =========="
)


print(
    "S0:",
    S0_ID
)


print(
    "S1:",
    S1_ID
)


print(
    "S2:",
    S2_ID
)


print(
    "HGB missing policy:",
    HGB_MISSING_POLICY
)


print(
    "HGB weight formula:",
    HGB_CLASS_WEIGHT_FORMULA
)


# ============================================================
# 6. LOAD FROZEN F1 MATRIX
# ============================================================

f1 = pd.read_pickle(
    F1_MATRIX_PATH
)


assert len(
    f1
) == EXPECTED_ROWS


PROVENANCE_COLUMNS = [
    "candidate_id",
    "dataset",
    "fold",
    "t",
    "z",
    "y",
    "x",
]


MATRIX_COLUMNS = (
    PROVENANCE_COLUMNS
    +
    F1_FEATURES
)


assert list(
    f1.columns
) == MATRIX_COLUMNS


assert dataframe_content_sha256(
    f1,
    MATRIX_COLUMNS,
) == EXPECTED_F1_CONTENT_SHA


assert np.array_equal(
    f1[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        EXPECTED_ROWS,
        dtype=np.int64,
    ),
)


assert (
    f1[
        "fold"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
    ==
    EXPECTED_FOLD_ROWS
)


# ============================================================
# 7. LOAD FROZEN A3 LABEL ARTIFACT
# ============================================================

a3 = pd.read_pickle(
    A3_PATH
)


assert len(
    a3
) == EXPECTED_ROWS


required_a3_columns = {
    "candidate_id",
    "dataset",
    "fold",
    "t",
    "z",
    "y",
    "x",
    "detector_prob",
    "detector_logit",
    LABEL_COLUMN,
}


assert required_a3_columns.issubset(
    a3.columns
)


a3[
    "candidate_id"
] = pd.to_numeric(
    a3[
        "candidate_id"
    ],
    errors="raise",
).astype(
    np.int64
)


assert a3[
    "candidate_id"
].is_unique


# ============================================================
# 8. EXACT F1 / A3 IDENTITY FIDELITY
#
# candidate_id is the join key, but also independently verify
# provenance fields so candidate_id cannot hide row drift.
# ============================================================

a3_identity = (
    a3[
        [
            "candidate_id",
            "dataset",
            "fold",
            "t",
            "z",
            "y",
            "x",
            "detector_prob",
            "detector_logit",
            LABEL_COLUMN,
        ]
    ]
    .copy()
)


merged = f1.merge(
    a3_identity,
    on="candidate_id",
    how="left",
    validate="one_to_one",
    sort=False,
    suffixes=(
        "_f1",
        "_a3",
    ),
)


assert len(
    merged
) == EXPECTED_ROWS


assert merged[
    LABEL_COLUMN
].notna().all()


# ------------------------------------------------------------
# Exact non-label provenance agreement.
# ------------------------------------------------------------

assert (
    merged[
        "dataset_f1"
    ].astype(
        str
    )
    ==
    merged[
        "dataset_a3"
    ].astype(
        str
    )
).all()


for column in [
    "fold",
    "t",
]:

    assert np.array_equal(
        pd.to_numeric(
            merged[
                f"{column}_f1"
            ],
            errors="raise",
        ).to_numpy(),
        pd.to_numeric(
            merged[
                f"{column}_a3"
            ],
            errors="raise",
        ).to_numpy(),
    )


for column in [
    "z",
    "y",
    "x",
    "detector_prob",
    "detector_logit",
]:

    assert np.array_equal(
        pd.to_numeric(
            merged[
                f"{column}_f1"
            ],
            errors="raise",
        ).to_numpy(
            dtype=np.float64
        ),
        pd.to_numeric(
            merged[
                f"{column}_a3"
            ],
            errors="raise",
        ).to_numpy(
            dtype=np.float64
        ),
        equal_nan=True,
    )


# Restore canonical benchmark arrays directly from F1.
y = pd.to_numeric(
    merged[
        LABEL_COLUMN
    ],
    errors="raise",
).astype(
    np.int8
).to_numpy()


fold = pd.to_numeric(
    merged[
        "fold_f1"
    ],
    errors="raise",
).astype(
    np.int8
).to_numpy()


candidate_id = merged[
    "candidate_id"
].to_numpy(
    dtype=np.int64
)


dataset = merged[
    "dataset_f1"
].astype(
    str
).to_numpy()


X = f1[
    F1_FEATURES
].to_numpy(
    dtype=np.float64
)


assert X.shape == (
    EXPECTED_ROWS,
    25,
)


assert int(
    y.sum()
) == EXPECTED_POSITIVES


assert int(
    (
        y
        ==
        0
    ).sum()
) == EXPECTED_NEGATIVES


actual_fold_positives = {
    heldout:
        int(
            y[
                fold
                ==
                heldout
            ].sum()
        )

    for heldout
    in FOLDS
}


assert (
    actual_fold_positives
    ==
    EXPECTED_FOLD_POSITIVES
)


print(
    "\n========== A3/F1 FORMAL JOIN =========="
)


print(
    "rows:",
    len(
        y
    )
)


print(
    "positives:",
    int(
        y.sum()
    )
)


print(
    "fold positives:",
    actual_fold_positives
)


print(
    "identity provenance exact:",
    "YES"
)


print(
    "A3_F1_FORMAL_JOIN: PASS"
)


# ============================================================
# 9. FEATURE LEAKAGE GATE
# ============================================================

assert f1_schema[
    "model_feature_columns"
] == F1_FEATURES


assert not any(
    column
    in F1_FEATURES

    for column
    in [
        LABEL_COLUMN,
        "candidate_id",
        "dataset",
        "fold",
        "t",
        "z",
        "y",
        "x",
        "nearest_missing_distance_um",
        "missing_count_within_7p0um",
    ]
)


print(
    "\n========== MODEL INPUT SAFETY =========="
)


print(
    "model features:",
    25
)


print(
    "GT/oracle model features:",
    "NONE"
)


print(
    "dataset/fold model features:",
    "NONE"
)


print(
    "raw t/z/y/x model features:",
    "NONE"
)


# ============================================================
# 10. BUDGET REPORTING RESOLUTION
#
# A4 freezes fractions but not integer rounding.
#
# This reporting-only implementation is resolved BEFORE model
# fitting and is NOT used for scorer selection.
#
#   k = ceil(fraction * held-out rows)
#
# Score ties:
#
#   score descending
#   candidate_id ascending
#
# deterministic only.
# ============================================================

BUDGET_INTEGER_RULE = (
    "CEIL_FRACTION_TIMES_HELDOUT_ROWS"
)


BUDGET_TIEBREAK_RULE = (
    "SCORE_DESC_CANDIDATE_ID_ASC"
)


def budget_k(
    fraction,
    n_rows,
):

    return min(
        n_rows,
        max(
            1,
            int(
                math.ceil(
                    float(
                        fraction
                    )
                    *
                    int(
                        n_rows
                    )
                )
            ),
        ),
    )


print(
    "\n========== RANKING BUDGET REPORTING CONTRACT =========="
)


print(
    "fractions:",
    BUDGET_FRACTIONS
)


print(
    "integer rule:",
    BUDGET_INTEGER_RULE
)


print(
    "tie break:",
    BUDGET_TIEBREAK_RULE
)


print(
    "budget metrics used for scorer selection:",
    "NO"
)


# ============================================================
# 11. MODEL FACTORIES
# ============================================================

def make_s1():

    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "model",
                LogisticRegression(
                    penalty="l2",
                    C=1.0,
                    solver="lbfgs",
                    class_weight="balanced",
                    max_iter=1000,
                    random_state=None,
                ),
            ),
        ]
    )


def make_s2():

    # Native NaN support.
    #
    # Do not add an imputer here.
    return HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=100,
        max_leaf_nodes=15,
        max_depth=None,
        l2_regularization=1.0,
        min_samples_leaf=20,
        random_state=0,
    )


# ============================================================
# 12. POSITIVE-CLASS PROBABILITY
# ============================================================

def positive_probability(
    estimator,
    X_eval,
):

    probability = estimator.predict_proba(
        X_eval
    )


    classes = np.asarray(
        estimator.classes_
    )


    positive_index = np.flatnonzero(
        classes
        ==
        1
    )


    assert len(
        positive_index
    ) == 1


    score = probability[
        :,
        int(
            positive_index[
                0
            ]
        )
    ]


    score = np.asarray(
        score,
        dtype=np.float64,
    )


    assert score.ndim == 1


    assert np.isfinite(
        score
    ).all()


    assert (
        score
        >=
        0
    ).all()


    assert (
        score
        <=
        1
    ).all()


    return score


# ============================================================
# 13. METRIC HELPERS
# ============================================================

def fold_metrics(
    *,
    scorer_id,
    heldout_fold,
    y_true,
    score,
):

    assert len(
        y_true
    ) == len(
        score
    )


    assert int(
        y_true.sum()
    ) > 0


    assert int(
        (
            y_true
            ==
            0
        ).sum()
    ) > 0


    return {
        "scorer_id":
            scorer_id,

        "heldout_fold":
            int(
                heldout_fold
            ),

        "rows":
            int(
                len(
                    y_true
                )
            ),

        "positives":
            int(
                y_true.sum()
            ),

        "negatives":
            int(
                (
                    y_true
                    ==
                    0
                ).sum()
            ),

        "positive_prevalence":
            float(
                np.mean(
                    y_true
                )
            ),

        "average_precision":
            float(
                average_precision_score(
                    y_true,
                    score,
                )
            ),

        "roc_auc":
            float(
                roc_auc_score(
                    y_true,
                    score,
                )
            ),
    }


def ranking_budget_metrics(
    *,
    scorer_id,
    heldout_fold,
    candidate_ids,
    y_true,
    score,
):

    order_frame = pd.DataFrame(
        {
            "candidate_id":
                candidate_ids,

            "label":
                y_true,

            "score":
                score,
        }
    )


    order_frame = (
        order_frame
        .sort_values(
            [
                "score",
                "candidate_id",
            ],
            ascending=[
                False,
                True,
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )


    positives = int(
        y_true.sum()
    )


    assert positives > 0


    rows = []


    for fraction in BUDGET_FRACTIONS:

        k = budget_k(
            fraction,
            len(
                order_frame
            ),
        )


        top = order_frame.iloc[
            :k
        ]


        hits = int(
            top[
                "label"
            ].sum()
        )


        rows.append(
            {
                "scorer_id":
                    scorer_id,

                "heldout_fold":
                    int(
                        heldout_fold
                    ),

                "budget_fraction":
                    float(
                        fraction
                    ),

                "heldout_rows":
                    int(
                        len(
                            order_frame
                        )
                    ),

                "budget_k":
                    int(
                        k
                    ),

                "positive_hits":
                    hits,

                "precision_at_fraction":
                    float(
                        hits
                        /
                        k
                    ),

                "candidate_positive_recall_at_fraction":
                    float(
                        hits
                        /
                        positives
                    ),
            }
        )


    return rows


# ============================================================
# 14. OOF BUFFERS
# ============================================================

oof_scores = {
    S0_ID:
        np.full(
            EXPECTED_ROWS,
            np.nan,
            dtype=np.float64,
        ),

    S1_ID:
        np.full(
            EXPECTED_ROWS,
            np.nan,
            dtype=np.float64,
        ),

    S2_ID:
        np.full(
            EXPECTED_ROWS,
            np.nan,
            dtype=np.float64,
        ),
}


fold_metric_rows = []

budget_metric_rows = []

model_audit_rows = []


# ============================================================
# 15. LOFO BENCHMARK
# ============================================================

print(
    "\n========== FORMAL S0 / S1 / S2 LOFO BENCHMARK =========="
)


benchmark_start = time.time()


for heldout_fold in FOLDS:

    test_mask = (
        fold
        ==
        heldout_fold
    )


    train_mask = ~test_mask


    train_indices = np.flatnonzero(
        train_mask
    )


    test_indices = np.flatnonzero(
        test_mask
    )


    X_train = X[
        train_indices
    ]


    y_train = y[
        train_indices
    ]


    X_test = X[
        test_indices
    ]


    y_test = y[
        test_indices
    ]


    test_candidate_ids = candidate_id[
        test_indices
    ]


    # --------------------------------------------------------
    # Fold invariants
    # --------------------------------------------------------

    assert len(
        test_indices
    ) == EXPECTED_FOLD_ROWS[
        heldout_fold
    ]


    assert int(
        y_test.sum()
    ) == EXPECTED_FOLD_POSITIVES[
        heldout_fold
    ]


    assert set(
        fold[
            train_indices
        ].tolist()
    ) == (
        set(
            FOLDS
        )
        -
        {
            heldout_fold
        }
    )


    assert set(
        fold[
            test_indices
        ].tolist()
    ) == {
        heldout_fold
    }


    n_train = int(
        len(
            y_train
        )
    )


    train_positive = int(
        y_train.sum()
    )


    train_negative = int(
        n_train
        -
        train_positive
    )


    assert train_positive > 0
    assert train_negative > 0


    print()
    print(
        "-" * 110
    )


    print(
        f"HELD-OUT FOLD {heldout_fold}"
    )


    print(
        "train rows:",
        n_train,
        "| train positives:",
        train_positive
    )


    print(
        "test rows:",
        len(
            y_test
        ),
        "| test positives:",
        int(
            y_test.sum()
        )
    )


    # ========================================================
    # S0 — detector probability
    # ========================================================

    s0_score = X_test[
        :,
        F1_FEATURES.index(
            "detector_prob"
        ),
    ].astype(
        np.float64,
        copy=True,
    )


    assert np.isfinite(
        s0_score
    ).all()


    oof_scores[
        S0_ID
    ][
        test_indices
    ] = s0_score


    s0_metrics = fold_metrics(
        scorer_id=S0_ID,
        heldout_fold=heldout_fold,
        y_true=y_test,
        score=s0_score,
    )


    fold_metric_rows.append(
        s0_metrics
    )


    budget_metric_rows.extend(
        ranking_budget_metrics(
            scorer_id=S0_ID,
            heldout_fold=heldout_fold,
            candidate_ids=test_candidate_ids,
            y_true=y_test,
            score=s0_score,
        )
    )


    # ========================================================
    # S1 — balanced logistic regression
    # ========================================================

    s1 = make_s1()


    s1_start = time.time()


    with warnings.catch_warnings(
        record=True
    ) as caught:

        warnings.simplefilter(
            "always"
        )


        s1.fit(
            X_train,
            y_train,
        )


    convergence_warnings = [
        str(
            warning.message
        )

        for warning
        in caught

        if issubclass(
            warning.category,
            ConvergenceWarning,
        )
    ]


    assert not convergence_warnings, (
        heldout_fold,
        convergence_warnings,
    )


    s1_score = positive_probability(
        s1,
        X_test,
    )


    s1_elapsed = float(
        time.time()
        -
        s1_start
    )


    oof_scores[
        S1_ID
    ][
        test_indices
    ] = s1_score


    s1_metrics = fold_metrics(
        scorer_id=S1_ID,
        heldout_fold=heldout_fold,
        y_true=y_test,
        score=s1_score,
    )


    fold_metric_rows.append(
        s1_metrics
    )


    budget_metric_rows.extend(
        ranking_budget_metrics(
            scorer_id=S1_ID,
            heldout_fold=heldout_fold,
            candidate_ids=test_candidate_ids,
            y_true=y_test,
            score=s1_score,
        )
    )


    # --------------------------------------------------------
    # S1 preprocessing audit
    # --------------------------------------------------------

    s1_imputer = s1.named_steps[
        "imputer"
    ]


    s1_scaler = s1.named_steps[
        "scaler"
    ]


    s1_model = s1.named_steps[
        "model"
    ]


    assert len(
        s1_imputer.statistics_
    ) == 25


    assert len(
        s1_scaler.mean_
    ) == 25


    assert s1_model.coef_.shape == (
        1,
        25,
    )


    model_audit_rows.append(
        {
            "scorer_id":
                S1_ID,

            "heldout_fold":
                heldout_fold,

            "train_rows":
                n_train,

            "train_positives":
                train_positive,

            "train_negatives":
                train_negative,

            "fit_seconds":
                s1_elapsed,

            "iterations":
                int(
                    np.max(
                        s1_model.n_iter_
                    )
                ),

            "missing_policy":
                "TRAIN_FOLD_MEDIAN",

            "class_balance":
                "class_weight=balanced",

            "positive_weight":
                float(
                    n_train
                    /
                    (
                        2
                        *
                        train_positive
                    )
                ),

            "negative_weight":
                float(
                    n_train
                    /
                    (
                        2
                        *
                        train_negative
                    )
                ),
        }
    )


    # ========================================================
    # S2 — balanced HGB
    # ========================================================

    # Exact explicit balanced train-fold sample weights.
    hgb_sample_weight = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    ).astype(
        np.float64
    )


    expected_positive_weight = (
        n_train
        /
        (
            2
            *
            train_positive
        )
    )


    expected_negative_weight = (
        n_train
        /
        (
            2
            *
            train_negative
        )
    )


    assert np.allclose(
        hgb_sample_weight[
            y_train
            ==
            1
        ],
        expected_positive_weight,
        rtol=0,
        atol=1e-15,
    )


    assert np.allclose(
        hgb_sample_weight[
            y_train
            ==
            0
        ],
        expected_negative_weight,
        rtol=0,
        atol=1e-15,
    )


    # Balanced weighting must give equal total weight to both
    # classes.
    positive_total_weight = float(
        hgb_sample_weight[
            y_train
            ==
            1
        ].sum()
    )


    negative_total_weight = float(
        hgb_sample_weight[
            y_train
            ==
            0
        ].sum()
    )


    assert math.isclose(
        positive_total_weight,
        negative_total_weight,
        rel_tol=0,
        abs_tol=1e-8,
    )


    s2 = make_s2()


    s2_start = time.time()


    s2.fit(
        X_train,
        y_train,
        sample_weight=hgb_sample_weight,
    )


    s2_elapsed = float(
        time.time()
        -
        s2_start
    )


    s2_score = positive_probability(
        s2,
        X_test,
    )


    oof_scores[
        S2_ID
    ][
        test_indices
    ] = s2_score


    s2_metrics = fold_metrics(
        scorer_id=S2_ID,
        heldout_fold=heldout_fold,
        y_true=y_test,
        score=s2_score,
    )


    fold_metric_rows.append(
        s2_metrics
    )


    budget_metric_rows.extend(
        ranking_budget_metrics(
            scorer_id=S2_ID,
            heldout_fold=heldout_fold,
            candidate_ids=test_candidate_ids,
            y_true=y_test,
            score=s2_score,
        )
    )


    model_audit_rows.append(
        {
            "scorer_id":
                S2_ID,

            "heldout_fold":
                heldout_fold,

            "train_rows":
                n_train,

            "train_positives":
                train_positive,

            "train_negatives":
                train_negative,

            "fit_seconds":
                s2_elapsed,

            "iterations":
                int(
                    s2.n_iter_
                ),

            "missing_policy":
                HGB_MISSING_POLICY,

            "class_balance":
                (
                    "EXPLICIT_TRAIN_FOLD_"
                    "SAMPLE_WEIGHT_BALANCED"
                ),

            "positive_weight":
                expected_positive_weight,

            "negative_weight":
                expected_negative_weight,
        }
    )


    # ========================================================
    # Fold report
    # ========================================================

    print(
        f"S0 AP={s0_metrics['average_precision']:.12f}"
        f" | AUC={s0_metrics['roc_auc']:.9f}"
    )


    print(
        f"S1 AP={s1_metrics['average_precision']:.12f}"
        f" | AUC={s1_metrics['roc_auc']:.9f}"
        f" | iter={int(np.max(s1_model.n_iter_))}"
    )


    print(
        f"S2 AP={s2_metrics['average_precision']:.12f}"
        f" | AUC={s2_metrics['roc_auc']:.9f}"
        f" | iter={int(s2.n_iter_)}"
    )


# ============================================================
# 16. OOF COMPLETENESS
# ============================================================

for scorer_id, score in oof_scores.items():

    assert np.isfinite(
        score
    ).all(), scorer_id


benchmark_seconds = float(
    time.time()
    -
    benchmark_start
)


print(
    "\nLOFO OOF COMPLETENESS: PASS"
)


print(
    "benchmark seconds:",
    benchmark_seconds
)


# ============================================================
# 17. FOLD METRICS TABLE
# ============================================================

fold_metrics_df = pd.DataFrame(
    fold_metric_rows
)


budget_metrics_df = pd.DataFrame(
    budget_metric_rows
)


model_audit_df = pd.DataFrame(
    model_audit_rows
)


assert len(
    fold_metrics_df
) == (
    3
    *
    4
)


assert len(
    budget_metrics_df
) == (
    3
    *
    4
    *
    4
)


assert len(
    model_audit_df
) == (
    2
    *
    4
)


print(
    "\n========== HELD-OUT FOLD METRICS =========="
)


print(
    fold_metrics_df
    .sort_values(
        [
            "heldout_fold",
            "scorer_id",
        ]
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 18. AGGREGATE SCORER METRICS
# ============================================================

aggregate_rows = []


for scorer_id in [
    S0_ID,
    S1_ID,
    S2_ID,
]:

    sub = fold_metrics_df.loc[
        fold_metrics_df[
            "scorer_id"
        ].eq(
            scorer_id
        )
    ].copy()


    assert len(
        sub
    ) == 4


    aggregate_rows.append(
        {
            "scorer_id":
                scorer_id,

            "macro_average_precision":
                float(
                    sub[
                        "average_precision"
                    ].mean()
                ),

            "worst_fold_average_precision":
                float(
                    sub[
                        "average_precision"
                    ].min()
                ),

            "best_fold_average_precision":
                float(
                    sub[
                        "average_precision"
                    ].max()
                ),

            "macro_roc_auc":
                float(
                    sub[
                        "roc_auc"
                    ].mean()
                ),

            "worst_fold_roc_auc":
                float(
                    sub[
                        "roc_auc"
                    ].min()
                ),

            "macro_positive_prevalence":
                float(
                    sub[
                        "positive_prevalence"
                    ].mean()
                ),

            "pooled_oof_average_precision":
                float(
                    average_precision_score(
                        y,
                        oof_scores[
                            scorer_id
                        ],
                    )
                ),

            "pooled_oof_roc_auc":
                float(
                    roc_auc_score(
                        y,
                        oof_scores[
                            scorer_id
                        ],
                    )
                ),
        }
    )


aggregate_df = pd.DataFrame(
    aggregate_rows
)


print(
    "\n========== AGGREGATE SCORER METRICS =========="
)


print(
    aggregate_df.to_string(
        index=False
    )
)


# ============================================================
# 19. MACRO BUDGET REPORT
# ============================================================

macro_budget_df = (
    budget_metrics_df
    .groupby(
        [
            "scorer_id",
            "budget_fraction",
        ],
        sort=True,
    )
    .agg(
        macro_precision_at_fraction=(
            "precision_at_fraction",
            "mean",
        ),
        worst_fold_precision_at_fraction=(
            "precision_at_fraction",
            "min",
        ),
        macro_candidate_positive_recall_at_fraction=(
            "candidate_positive_recall_at_fraction",
            "mean",
        ),
        worst_fold_candidate_positive_recall_at_fraction=(
            "candidate_positive_recall_at_fraction",
            "min",
        ),
        total_positive_hits=(
            "positive_hits",
            "sum",
        ),
        total_budget_k=(
            "budget_k",
            "sum",
        ),
    )
    .reset_index()
)


print(
    "\n========== MACRO RANKING-BUDGET REPORT =========="
)


print(
    macro_budget_df.to_string(
        index=False
    )
)


# ============================================================
# 20. PREDECLARED SCORER SELECTION
#
# Lexicographic:
#
# 1. macro AP
# 2. worst-fold AP
# 3. complexity
#
# tie tolerance = 1e-6
# ============================================================

COMPLEXITY_ORDER = [
    S0_ID,
    S1_ID,
    S2_ID,
]


complexity_rank = {
    scorer_id:
        index

    for index, scorer_id
    in enumerate(
        COMPLEXITY_ORDER
    )
}


selection_table = aggregate_df.copy()


selection_table[
    "complexity_rank"
] = selection_table[
    "scorer_id"
].map(
    complexity_rank
)


best_macro = float(
    selection_table[
        "macro_average_precision"
    ].max()
)


macro_contenders = selection_table.loc[
    (
        best_macro
        -
        selection_table[
            "macro_average_precision"
        ]
    )
    <=
    TIE_TOLERANCE
].copy()


best_worst = float(
    macro_contenders[
        "worst_fold_average_precision"
    ].max()
)


worst_contenders = macro_contenders.loc[
    (
        best_worst
        -
        macro_contenders[
            "worst_fold_average_precision"
        ]
    )
    <=
    TIE_TOLERANCE
].copy()


selected_row = (
    worst_contenders
    .sort_values(
        "complexity_rank",
        ascending=True,
        kind="stable",
    )
    .iloc[
        0
    ]
)


SELECTED_SCORER = str(
    selected_row[
        "scorer_id"
    ]
)


SELECTED_MACRO_AP = float(
    selected_row[
        "macro_average_precision"
    ]
)


SELECTED_WORST_AP = float(
    selected_row[
        "worst_fold_average_precision"
    ]
)


baseline_row = (
    selection_table.loc[
        selection_table[
            "scorer_id"
        ].eq(
            S0_ID
        )
    ]
    .iloc[
        0
    ]
)


BASELINE_MACRO_AP = float(
    baseline_row[
        "macro_average_precision"
    ]
)


BASELINE_WORST_AP = float(
    baseline_row[
        "worst_fold_average_precision"
    ]
)


# ============================================================
# 21. CONTINUATION GATE
# ============================================================

MACRO_GATE = (
    SELECTED_MACRO_AP
    >
    BASELINE_MACRO_AP
)


WORST_GATE = (
    SELECTED_WORST_AP
    >=
    BASELINE_WORST_AP
)


CONTINUATION_GATE_PASS = (
    MACRO_GATE
    and
    WORST_GATE
)


if CONTINUATION_GATE_PASS:

    BRANCH_DECISION = (
        "CONTINUE_BRANCH_NO_GRAPH_MUTATION_YET"
    )


else:

    BRANCH_DECISION = (
        "CLOSE_BRANCH_NO_GRAPH_MUTATION"
    )


print(
    "\n========== PREDECLARED SCORER SELECTION =========="
)


print(
    "tie tolerance:",
    TIE_TOLERANCE
)


print(
    "selected scorer:",
    SELECTED_SCORER
)


print(
    "selected macro AP:",
    SELECTED_MACRO_AP
)


print(
    "selected worst-fold AP:",
    SELECTED_WORST_AP
)


print(
    "detector macro AP:",
    BASELINE_MACRO_AP
)


print(
    "detector worst-fold AP:",
    BASELINE_WORST_AP
)


print(
    "macro AP strictly exceeds baseline:",
    MACRO_GATE
)


print(
    "worst-fold AP >= baseline worst:",
    WORST_GATE
)


print(
    "continuation gate:",
    CONTINUATION_GATE_PASS
)


print(
    "branch decision:",
    BRANCH_DECISION
)


# ============================================================
# 22. OOF ARTIFACT
#
# Evaluation artifact only.
#
# Label is GT-derived and MUST NEVER be used as deployable
# runtime input.
# ============================================================

oof_df = pd.DataFrame(
    {
        "candidate_id":
            candidate_id,

        "dataset":
            dataset,

        "fold":
            fold,

        "oracle_hit_7um":
            y,

        f"score__{S0_ID}":
            oof_scores[
                S0_ID
            ],

        f"score__{S1_ID}":
            oof_scores[
                S1_ID
            ],

        f"score__{S2_ID}":
            oof_scores[
                S2_ID
            ],
    }
)


assert len(
    oof_df
) == EXPECTED_ROWS


assert oof_df[
    "candidate_id"
].is_unique


# ============================================================
# 23. FORMAL SELECTION CONTRACT
# ============================================================

selection_artifact = {
    "stage":
        "12.12A5A-R4",

    "selection_id":
        SELECTION_ID,

    "benchmark_id":
        BENCHMARK_ID,

    "status":
        "DEVELOPMENT_GT_INFORMED_SCORER_SELECTION_FROZEN",

    "label": {
        "label_id":
            LABEL_ID,

        "column":
            LABEL_COLUMN,

        "artifact":
            A3_PATH.name,

        "artifact_sha256":
            EXPECTED_A3_SHA,

        "rows":
            EXPECTED_ROWS,

        "positives":
            EXPECTED_POSITIVES,

        "negatives":
            EXPECTED_NEGATIVES,
    },

    "feature_family": {
        "family_id":
            F1_ID,

        "matrix_artifact":
            F1_MATRIX_PATH.name,

        "matrix_file_sha256":
            EXPECTED_F1_MATRIX_FILE_SHA,

        "matrix_content_sha256":
            EXPECTED_F1_CONTENT_SHA,

        "schema_sha256":
            EXPECTED_F1_SCHEMA_SHA,

        "feature_count":
            25,

        "features":
            F1_FEATURES,
    },

    "scorers": {
        "S0":
            S0,

        "S1":
            S1,

        "S2":
            S2,
    },

    "resolved_implementation": {
        "hgb_missing_policy":
            HGB_MISSING_POLICY,

        "hgb_sample_weight_implementation":
            HGB_CLASS_WEIGHT_IMPLEMENTATION,

        "hgb_sample_weight_formula":
            HGB_CLASS_WEIGHT_FORMULA,

        "budget_integer_rule":
            BUDGET_INTEGER_RULE,

        "budget_score_tiebreak":
            BUDGET_TIEBREAK_RULE,
    },

    "selection_rule": {
        "primary_order": [
            "MACRO_AVERAGE_PRECISION_DESCENDING",
            "WORST_FOLD_AVERAGE_PRECISION_DESCENDING",
        ],

        "complexity_tiebreak_order":
            COMPLEXITY_ORDER,

        "tie_tolerance":
            TIE_TOLERANCE,
    },

    "results": {
        row[
            "scorer_id"
        ]: {
            "macro_average_precision":
                float(
                    row[
                        "macro_average_precision"
                    ]
                ),

            "worst_fold_average_precision":
                float(
                    row[
                        "worst_fold_average_precision"
                    ]
                ),

            "macro_roc_auc":
                float(
                    row[
                        "macro_roc_auc"
                    ]
                ),

            "pooled_oof_average_precision":
                float(
                    row[
                        "pooled_oof_average_precision"
                    ]
                ),

            "pooled_oof_roc_auc":
                float(
                    row[
                        "pooled_oof_roc_auc"
                    ]
                ),
        }

        for row
        in aggregate_rows
    },

    "selected_scorer":
        SELECTED_SCORER,

    "selected_macro_average_precision":
        SELECTED_MACRO_AP,

    "selected_worst_fold_average_precision":
        SELECTED_WORST_AP,

    "baseline_scorer":
        S0_ID,

    "baseline_macro_average_precision":
        BASELINE_MACRO_AP,

    "baseline_worst_fold_average_precision":
        BASELINE_WORST_AP,

    "continuation_gate": {
        "macro_ap_must_exceed_detector_baseline":
            True,

        "worst_fold_ap_must_not_be_below_detector_worst_fold":
            True,

        "macro_gate_pass":
            bool(
                MACRO_GATE
            ),

        "worst_fold_gate_pass":
            bool(
                WORST_GATE
            ),

        "gate_pass":
            bool(
                CONTINUATION_GATE_PASS
            ),

        "branch_decision":
            BRANCH_DECISION,
    },

    "protocol": {
        "selection_is_development_gt_informed":
            True,

        "feature_matrix_modified_after_a3":
            False,

        "new_features_added":
            False,

        "fold0_used":
            False,

        "threshold_selected":
            False,

        "graph_metric_evaluated":
            False,

        "graph_modified":
            False,

        "post_selection_model_retuning":
            False,

        "f2_materialized":
            False,
    },
}


SELECTION_CONTENT_SHA = hashlib.sha256(
    json.dumps(
        selection_artifact,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


selection_artifact[
    "selection_content_sha256"
] = SELECTION_CONTENT_SHA


# ============================================================
# 24. TEMP OUTPUT PATHS
# ============================================================

TMP_OOF = OOF_OUT.with_name(
    OOF_OUT.name
    +
    ".tmp_r4"
)


TMP_FOLD = FOLD_METRICS_OUT.with_name(
    FOLD_METRICS_OUT.name
    +
    ".tmp_r4"
)


TMP_BUDGET = BUDGET_METRICS_OUT.with_name(
    BUDGET_METRICS_OUT.name
    +
    ".tmp_r4"
)


TMP_MODEL_AUDIT = MODEL_AUDIT_OUT.with_name(
    MODEL_AUDIT_OUT.name
    +
    ".tmp_r4"
)


TMP_SELECTION = SELECTION_OUT.with_name(
    SELECTION_OUT.name
    +
    ".tmp_r4"
)


TMP_SUMMARY = SUMMARY_OUT.with_name(
    SUMMARY_OUT.name
    +
    ".tmp_r4"
)


for path in [
    TMP_OOF,
    TMP_FOLD,
    TMP_BUDGET,
    TMP_MODEL_AUDIT,
    TMP_SELECTION,
    TMP_SUMMARY,
]:

    if path.exists():

        raise RuntimeError(
            "\nStale temporary R4 artifact exists:\n"
            f"{path}\n\n"
            "Remove ONLY the stale temporary R4 artifact "
            "after inspection."
        )


# ============================================================
# 25. WRITE TEMP BENCHMARK ARTIFACTS
# ============================================================

oof_df.to_pickle(
    TMP_OOF
)


fold_metrics_df.to_pickle(
    TMP_FOLD
)


budget_metrics_df.to_pickle(
    TMP_BUDGET
)


model_audit_df.to_pickle(
    TMP_MODEL_AUDIT
)


TMP_SELECTION.write_text(
    json.dumps(
        selection_artifact,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 26. TEMP RELOAD
# ============================================================

oof_check = pd.read_pickle(
    TMP_OOF
)


fold_check = pd.read_pickle(
    TMP_FOLD
)


budget_check = pd.read_pickle(
    TMP_BUDGET
)


model_audit_check = pd.read_pickle(
    TMP_MODEL_AUDIT
)


selection_check = json.loads(
    TMP_SELECTION.read_text(
        encoding="utf-8"
    )
)


pd.testing.assert_frame_equal(
    oof_check,
    oof_df,
    check_exact=True,
)


pd.testing.assert_frame_equal(
    fold_check,
    fold_metrics_df,
    check_exact=True,
)


pd.testing.assert_frame_equal(
    budget_check,
    budget_metrics_df,
    check_exact=True,
)


pd.testing.assert_frame_equal(
    model_audit_check,
    model_audit_df,
    check_exact=True,
)


assert selection_check[
    "selection_content_sha256"
] == SELECTION_CONTENT_SHA


# ============================================================
# 27. FILE SHAS
# ============================================================

OOF_FILE_SHA = sha256_file(
    TMP_OOF
)


FOLD_METRICS_FILE_SHA = sha256_file(
    TMP_FOLD
)


BUDGET_METRICS_FILE_SHA = sha256_file(
    TMP_BUDGET
)


MODEL_AUDIT_FILE_SHA = sha256_file(
    TMP_MODEL_AUDIT
)


SELECTION_FILE_SHA = sha256_file(
    TMP_SELECTION
)


# ============================================================
# 28. FORMAL SUMMARY
# ============================================================

SUMMARY = {
    "stage":
        "12.12A5A-R4",

    "status":
        "PREDECLARED_S0_S1_S2_LOFO_RANKING_BENCHMARK_COMPLETE",

    "benchmark_id":
        BENCHMARK_ID,

    "selection_id":
        SELECTION_ID,

    "parents": {
        "a3_artifact":
            A3_PATH.name,

        "a3_sha256":
            EXPECTED_A3_SHA,

        "a4_contract_sha256":
            EXPECTED_A4_SHA,

        "f1_matrix_file_sha256":
            EXPECTED_F1_MATRIX_FILE_SHA,

        "f1_matrix_content_sha256":
            EXPECTED_F1_CONTENT_SHA,

        "f1_schema_sha256":
            EXPECTED_F1_SCHEMA_SHA,

        "f1_summary_sha256":
            EXPECTED_F1_SUMMARY_SHA,
    },

    "environment": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,

        "sklearn":
            sklearn.__version__,

        "joblib":
            joblib.__version__,
    },

    "data": {
        "rows":
            EXPECTED_ROWS,

        "positives":
            EXPECTED_POSITIVES,

        "negatives":
            EXPECTED_NEGATIVES,

        "fold_rows":
            {
                str(
                    key
                ):
                    int(
                        value
                    )

                for key, value
                in EXPECTED_FOLD_ROWS.items()
            },

        "fold_positives":
            {
                str(
                    key
                ):
                    int(
                        value
                    )

                for key, value
                in EXPECTED_FOLD_POSITIVES.items()
            },
    },

    "implementation": {
        "hgb_missing_policy":
            HGB_MISSING_POLICY,

        "hgb_sample_weight_formula":
            HGB_CLASS_WEIGHT_FORMULA,

        "budget_integer_rule":
            BUDGET_INTEGER_RULE,

        "budget_tiebreak":
            BUDGET_TIEBREAK_RULE,

        "benchmark_seconds":
            benchmark_seconds,
    },

    "aggregate_results": {
        row[
            "scorer_id"
        ]: {
            key:
                (
                    float(
                        value
                    )
                    if isinstance(
                        value,
                        (
                            np.floating,
                            float,
                        )
                    )
                    else value
                )

            for key, value
            in row.items()

            if key
            !=
            "scorer_id"
        }

        for row
        in aggregate_rows
    },

    "selection": {
        "selected_scorer":
            SELECTED_SCORER,

        "selected_macro_ap":
            SELECTED_MACRO_AP,

        "selected_worst_fold_ap":
            SELECTED_WORST_AP,

        "detector_macro_ap":
            BASELINE_MACRO_AP,

        "detector_worst_fold_ap":
            BASELINE_WORST_AP,

        "macro_gate_pass":
            bool(
                MACRO_GATE
            ),

        "worst_gate_pass":
            bool(
                WORST_GATE
            ),

        "continuation_gate_pass":
            bool(
                CONTINUATION_GATE_PASS
            ),

        "branch_decision":
            BRANCH_DECISION,

        "selection_content_sha256":
            SELECTION_CONTENT_SHA,
    },

    "artifacts": {
        "oof":
            OOF_OUT.name,

        "oof_file_sha256":
            OOF_FILE_SHA,

        "fold_metrics":
            FOLD_METRICS_OUT.name,

        "fold_metrics_file_sha256":
            FOLD_METRICS_FILE_SHA,

        "budget_metrics":
            BUDGET_METRICS_OUT.name,

        "budget_metrics_file_sha256":
            BUDGET_METRICS_FILE_SHA,

        "model_audit":
            MODEL_AUDIT_OUT.name,

        "model_audit_file_sha256":
            MODEL_AUDIT_FILE_SHA,

        "selection":
            SELECTION_OUT.name,

        "selection_file_sha256":
            SELECTION_FILE_SHA,
    },

    "protocol": {
        "f1_frozen_before_a3":
            True,

        "development_gt_used_for_selection":
            True,

        "gt_used_as_runtime_feature":
            False,

        "model_features_changed":
            False,

        "threshold_selected":
            False,

        "graph_metric_evaluated":
            False,

        "graph_modified":
            False,

        "fold0_used":
            False,

        "post_selection_model_retuning":
            False,

        "f2_materialized":
            False,
    },

    "next": (
        "IF_CONTINUATION_GATE_PASS: "
        "FREEZE_SELECTED_SCORER_WITHOUT_RETUNING_AND_"
        "PROCEED_TO_PREDECLARED_ATTACHMENT_STAGE; "
        "ELSE: CLOSE_BRANCH_NO_GRAPH_MUTATION"
    ),
}


SUMMARY_CONTENT_SHA = hashlib.sha256(
    json.dumps(
        SUMMARY,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


SUMMARY[
    "summary_content_sha256"
] = SUMMARY_CONTENT_SHA


TMP_SUMMARY.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 29. SUMMARY RELOAD
# ============================================================

summary_check = json.loads(
    TMP_SUMMARY.read_text(
        encoding="utf-8"
    )
)


assert summary_check[
    "summary_content_sha256"
] == SUMMARY_CONTENT_SHA


assert summary_check[
    "protocol"
][
    "threshold_selected"
] is False


assert summary_check[
    "protocol"
][
    "graph_metric_evaluated"
] is False


assert summary_check[
    "protocol"
][
    "graph_modified"
] is False


assert summary_check[
    "protocol"
][
    "fold0_used"
] is False


SUMMARY_FILE_SHA = sha256_file(
    TMP_SUMMARY
)


# ============================================================
# 30. ATOMIC PROMOTION
# ============================================================

TMP_OOF.replace(
    OOF_OUT
)


TMP_FOLD.replace(
    FOLD_METRICS_OUT
)


TMP_BUDGET.replace(
    BUDGET_METRICS_OUT
)


TMP_MODEL_AUDIT.replace(
    MODEL_AUDIT_OUT
)


TMP_SELECTION.replace(
    SELECTION_OUT
)


TMP_SUMMARY.replace(
    SUMMARY_OUT
)


# ============================================================
# 31. FINAL FORMAL RELOAD
# ============================================================

assert sha256_file(
    OOF_OUT
) == OOF_FILE_SHA


assert sha256_file(
    FOLD_METRICS_OUT
) == FOLD_METRICS_FILE_SHA


assert sha256_file(
    BUDGET_METRICS_OUT
) == BUDGET_METRICS_FILE_SHA


assert sha256_file(
    MODEL_AUDIT_OUT
) == MODEL_AUDIT_FILE_SHA


assert sha256_file(
    SELECTION_OUT
) == SELECTION_FILE_SHA


assert sha256_file(
    SUMMARY_OUT
) == SUMMARY_FILE_SHA


formal_selection = json.loads(
    SELECTION_OUT.read_text(
        encoding="utf-8"
    )
)


formal_summary = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert formal_selection[
    "selection_content_sha256"
] == SELECTION_CONTENT_SHA


assert formal_summary[
    "summary_content_sha256"
] == SUMMARY_CONTENT_SHA


# ============================================================
# 32. FINAL FORMAL REPORT
# ============================================================

print(
    "\n========== 12.12A5A-R4 FINAL =========="
)


print(
    "PREDECLARED_S0_S1_S2_LOFO_RANKING_BENCHMARK: PASS"
)


print(
    "\n--- AGGREGATE ---"
)


for row in aggregate_rows:

    print()
    print(
        row[
            "scorer_id"
        ]
    )


    print(
        "  macro AP:",
        f"{row['macro_average_precision']:.12f}"
    )


    print(
        "  worst-fold AP:",
        f"{row['worst_fold_average_precision']:.12f}"
    )


    print(
        "  macro AUC:",
        f"{row['macro_roc_auc']:.9f}"
    )


    print(
        "  pooled OOF AP:",
        f"{row['pooled_oof_average_precision']:.12f}"
    )


print(
    "\n--- SELECTION ---"
)


print(
    "selected scorer:",
    SELECTED_SCORER
)


print(
    "selected macro AP:",
    f"{SELECTED_MACRO_AP:.12f}"
)


print(
    "selected worst-fold AP:",
    f"{SELECTED_WORST_AP:.12f}"
)


print(
    "detector macro AP:",
    f"{BASELINE_MACRO_AP:.12f}"
)


print(
    "detector worst-fold AP:",
    f"{BASELINE_WORST_AP:.12f}"
)


print(
    "macro gate:",
    MACRO_GATE
)


print(
    "worst-fold gate:",
    WORST_GATE
)


print(
    "continuation gate:",
    CONTINUATION_GATE_PASS
)


print(
    "branch decision:",
    BRANCH_DECISION
)


print(
    "\n--- PROTOCOL ---"
)


print(
    "F1 modified after A3:",
    "NO"
)


print(
    "A3 used as runtime feature:",
    "NO"
)


print(
    "model features:",
    25
)


print(
    "threshold selected:",
    "NO"
)


print(
    "graph metric evaluated:",
    "NO"
)


print(
    "graph modified:",
    "NO"
)


print(
    "post-selection retuning:",
    "NO"
)


print(
    "F2 materialized:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "\n--- HASHES ---"
)


print(
    "OOF file SHA:",
    OOF_FILE_SHA
)


print(
    "fold metrics SHA:",
    FOLD_METRICS_FILE_SHA
)


print(
    "budget metrics SHA:",
    BUDGET_METRICS_FILE_SHA
)


print(
    "model audit SHA:",
    MODEL_AUDIT_FILE_SHA
)


print(
    "selection file SHA:",
    SELECTION_FILE_SHA
)


print(
    "selection content SHA:",
    SELECTION_CONTENT_SHA
)


print(
    "summary file SHA:",
    SUMMARY_FILE_SHA
)


print(
    "summary content SHA:",
    SUMMARY_CONTENT_SHA
)


print(
    "\nnext:"
)


if CONTINUATION_GATE_PASS:

    print(
        "A5A-R5 — FREEZE SELECTED SCORER WITHOUT "
        "POST-SELECTION RETUNING; THEN PREDECLARED "
        "ATTACHMENT EVALUATION."
    )


else:

    print(
        "BRANCH CLOSED — NO GRAPH MUTATION."
    )

In [ ]:
# ============================================================
# Stage 12.12A5A-R5-CLOSE
#
# FORMAL BRANCH CLOSURE
#
# POST_R1P50_RESIDUAL_SYNTHETIC_NODE_RECOVERY_V1
#
# ============================================================
#
# R4 completed successfully.
#
# Predeclared continuation gate FAILED:
#
#   selected scorer:
#       DETECTOR_PROB_DESCENDING_V1
#
#   selected macro AP:
#       0.0035139626403584132
#
#   detector macro AP:
#       0.0035139626403584132
#
#   macro AP strictly exceeds detector:
#       FALSE
#
#   worst-fold gate:
#       TRUE
#
# Therefore:
#
#   CLOSE_BRANCH_NO_GRAPH_MUTATION
#
# ------------------------------------------------------------
# THIS CELL
#
# - performs no model training
# - performs no scoring
# - performs no threshold selection
# - performs no feature generation
# - performs no graph mutation
# - does not use Fold0
#
# KEEP AFTER RUNNING: YES
# ============================================================


from pathlib import Path
import hashlib
import json


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


R3_SUMMARY = (
    S12
    / "stage12_post_r1p50_node_f1_temporal_geometry_v1_summary.json"
)


R4_SELECTION = (
    S12
    / "stage12_post_r1p50_node_scorer_selection_v1_frozen.json"
)


R4_SUMMARY = (
    S12
    / "stage12_post_r1p50_node_f1_lofo_scorer_benchmark_v1_summary.json"
)


CLOSURE_OUT = (
    S12
    / "stage12_post_r1p50_residual_synthetic_node_recovery_v1_closure.json"
)


for path in [
    R3_SUMMARY,
    R4_SELECTION,
    R4_SUMMARY,
]:

    assert path.exists(), path


if CLOSURE_OUT.exists():

    raise RuntimeError(
        "Formal branch closure artifact already exists. "
        "Do not overwrite it."
    )


# ============================================================
# 1. EXPECTED FORMAL SHAS
# ============================================================

EXPECTED_R4_SELECTION_FILE_SHA = (
    "4553407696253e425ea2b4d4c1cb8278"
    "fc0b1281c60ac10207bd9d970cdcbba7"
)


EXPECTED_R4_SELECTION_CONTENT_SHA = (
    "b0dbde81881ed652d70d80dbdffb3fb"
    "3058a4bac62cae00c27798133156db142"
)


EXPECTED_R4_SUMMARY_FILE_SHA = (
    "35401de4d8c9e24bac827b95c57a5199"
    "e78308c48f6a0533e13f3c14b8ebaed3"
)


EXPECTED_R4_SUMMARY_CONTENT_SHA = (
    "0f5e00991dbeeac6673a2b9f55a6462"
    "ab781de45299081d0c0665559a99b1a18"
)


EXPECTED_R3_SUMMARY_CONTENT_SHA = (
    "63698900438e3eff28c399212f120e348"
    "dacf65275c255c3fe72506ad2674615"
)


BRANCH_ID = (
    "POST_R1P50_RESIDUAL_SYNTHETIC_NODE_RECOVERY_V1"
)


EXPECTED_SELECTED_SCORER = (
    "DETECTOR_PROB_DESCENDING_V1"
)


EXPECTED_MACRO_AP = (
    0.0035139626403584132
)


EXPECTED_WORST_AP = (
    0.0013258537645770307
)


EXPECTED_DECISION = (
    "CLOSE_BRANCH_NO_GRAPH_MUTATION"
)


# ============================================================
# 2. HASH HELPER
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            h.update(block)

    return h.hexdigest()


# ============================================================
# 3. FORMAL PARENT LOCK
# ============================================================

assert sha256_file(
    R4_SELECTION
) == EXPECTED_R4_SELECTION_FILE_SHA


assert sha256_file(
    R4_SUMMARY
) == EXPECTED_R4_SUMMARY_FILE_SHA


r3 = json.loads(
    R3_SUMMARY.read_text(
        encoding="utf-8"
    )
)


selection = json.loads(
    R4_SELECTION.read_text(
        encoding="utf-8"
    )
)


r4 = json.loads(
    R4_SUMMARY.read_text(
        encoding="utf-8"
    )
)


assert r3[
    "summary_sha256"
] == EXPECTED_R3_SUMMARY_CONTENT_SHA


assert selection[
    "selection_content_sha256"
] == EXPECTED_R4_SELECTION_CONTENT_SHA


assert r4[
    "summary_content_sha256"
] == EXPECTED_R4_SUMMARY_CONTENT_SHA


print(
    "========== 12.12A5A-R5-CLOSE PARENT LOCK =========="
)


print(
    "R3 feature freeze:",
    "PASS"
)


print(
    "R4 benchmark:",
    "PASS"
)


print(
    "R4 selection SHA:",
    EXPECTED_R4_SELECTION_CONTENT_SHA
)


# ============================================================
# 4. VERIFY EXACT FAILED CONTINUATION GATE
# ============================================================

assert selection[
    "selected_scorer"
] == EXPECTED_SELECTED_SCORER


assert selection[
    "selected_macro_average_precision"
] == EXPECTED_MACRO_AP


assert selection[
    "selected_worst_fold_average_precision"
] == EXPECTED_WORST_AP


assert selection[
    "baseline_scorer"
] == EXPECTED_SELECTED_SCORER


assert selection[
    "baseline_macro_average_precision"
] == EXPECTED_MACRO_AP


gate = selection[
    "continuation_gate"
]


assert gate[
    "macro_ap_must_exceed_detector_baseline"
] is True


assert gate[
    "worst_fold_ap_must_not_be_below_detector_worst_fold"
] is True


assert gate[
    "macro_gate_pass"
] is False


assert gate[
    "worst_fold_gate_pass"
] is True


assert gate[
    "gate_pass"
] is False


assert gate[
    "branch_decision"
] == EXPECTED_DECISION


print(
    "\n========== CONTINUATION GATE =========="
)


print(
    "selected scorer:",
    EXPECTED_SELECTED_SCORER
)


print(
    "selected macro AP:",
    EXPECTED_MACRO_AP
)


print(
    "detector macro AP:",
    EXPECTED_MACRO_AP
)


print(
    "macro gate:",
    False
)


print(
    "worst-fold gate:",
    True
)


print(
    "continuation gate:",
    False
)


print(
    "decision:",
    EXPECTED_DECISION
)


# ============================================================
# 5. VERIFY NO FORBIDDEN FOLLOW-UP OCCURRED
# ============================================================

protocol = r4[
    "protocol"
]


assert protocol[
    "threshold_selected"
] is False


assert protocol[
    "graph_metric_evaluated"
] is False


assert protocol[
    "graph_modified"
] is False


assert protocol[
    "fold0_used"
] is False


assert protocol[
    "post_selection_model_retuning"
] is False


assert protocol[
    "f2_materialized"
] is False


print(
    "\n========== CLOSURE SAFETY =========="
)


print(
    "post-selection retuning:",
    "NO"
)


print(
    "F2 materialized:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "graph metric:",
    "NO"
)


print(
    "graph mutation:",
    "NO"
)


print(
    "Fold0:",
    "NO"
)


# ============================================================
# 6. FORMAL CLOSURE
# ============================================================

closure = {
    "stage":
        "12.12A5A-R5-CLOSE",

    "status":
        "BRANCH_CLOSED",

    "branch_id":
        BRANCH_ID,

    "closure_reason":
        (
            "PREDECLARED_CONTINUATION_GATE_FAILED_"
            "SELECTED_SCORER_DID_NOT_STRICTLY_EXCEED_"
            "DETECTOR_BASELINE_MACRO_AP"
        ),

    "scientific_result": {
        "selected_scorer":
            EXPECTED_SELECTED_SCORER,

        "selected_macro_average_precision":
            EXPECTED_MACRO_AP,

        "selected_worst_fold_average_precision":
            EXPECTED_WORST_AP,

        "detector_macro_average_precision":
            EXPECTED_MACRO_AP,

        "macro_gate_pass":
            False,

        "worst_fold_gate_pass":
            True,

        "continuation_gate_pass":
            False,

        "decision":
            EXPECTED_DECISION,
    },

    "interpretation": {
        "f1_geometry_improves_detector_ranking":
            False,

        "trained_scorer_beats_detector_on_primary_macro_ap":
            False,

        "candidate_universe_remains_low_ceiling":
            True,

        "post_hoc_rescue_allowed":
            False,
    },

    "parents": {
        "r3_feature_freeze_summary":
            R3_SUMMARY.name,

        "r3_feature_freeze_summary_sha256":
            EXPECTED_R3_SUMMARY_CONTENT_SHA,

        "r4_selection":
            R4_SELECTION.name,

        "r4_selection_file_sha256":
            EXPECTED_R4_SELECTION_FILE_SHA,

        "r4_selection_content_sha256":
            EXPECTED_R4_SELECTION_CONTENT_SHA,

        "r4_summary":
            R4_SUMMARY.name,

        "r4_summary_file_sha256":
            EXPECTED_R4_SUMMARY_FILE_SHA,

        "r4_summary_content_sha256":
            EXPECTED_R4_SUMMARY_CONTENT_SHA,
    },

    "protocol": {
        "branch_closed":
            True,

        "f2_escalation_performed":
            False,

        "new_feature_family_after_failure":
            False,

        "post_selection_model_retuning":
            False,

        "threshold_selected":
            False,

        "graph_metric_evaluated":
            False,

        "graph_modified":
            False,

        "fold0_used":
            False,
    },

    "next":
        (
            "RETURN_TO_STAGE12_RESIDUAL_AUDIT_AND_SELECT_"
            "A_NEW_INDEPENDENT_PREDECLARED_BRANCH; "
            "DO_NOT_REOPEN_POST_R1P50_RESIDUAL_"
            "SYNTHETIC_NODE_RECOVERY_V1"
        ),
}


closure_sha = hashlib.sha256(
    json.dumps(
        closure,
        sort_keys=True,
        separators=(",", ":"),
    ).encode(
        "utf-8"
    )
).hexdigest()


closure[
    "closure_sha256"
] = closure_sha


# ============================================================
# 7. ATOMIC WRITE
# ============================================================

tmp = CLOSURE_OUT.with_name(
    CLOSURE_OUT.name
    +
    ".tmp_r5_close"
)


if tmp.exists():

    raise RuntimeError(
        f"Stale temporary closure artifact: {tmp}"
    )


tmp.write_text(
    json.dumps(
        closure,
        indent=2,
    ),
    encoding="utf-8",
)


reload = json.loads(
    tmp.read_text(
        encoding="utf-8"
    )
)


assert reload[
    "closure_sha256"
] == closure_sha


assert reload[
    "status"
] == "BRANCH_CLOSED"


assert reload[
    "protocol"
][
    "graph_modified"
] is False


assert reload[
    "protocol"
][
    "fold0_used"
] is False


tmp.replace(
    CLOSURE_OUT
)


final = json.loads(
    CLOSURE_OUT.read_text(
        encoding="utf-8"
    )
)


assert final[
    "closure_sha256"
] == closure_sha


# ============================================================
# 8. FINAL
# ============================================================

print(
    "\n========== 12.12A5A-R5-CLOSE FINAL =========="
)


print(
    "BRANCH_CLOSURE_FREEZE: PASS"
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "result:",
    EXPECTED_DECISION
)


print(
    "selected scorer:",
    EXPECTED_SELECTED_SCORER
)


print(
    "macro AP improvement over detector:",
    0.0
)


print(
    "post-hoc rescue:",
    "FORBIDDEN"
)


print(
    "F2 escalation:",
    "NO"
)


print(
    "threshold selection:",
    "NO"
)


print(
    "graph mutation:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "closure SHA:",
    closure_sha
)


print(
    "next:"
)


print(
    "RETURN_TO_STAGE12_RESIDUAL_AUDIT_AND_DEFINE_"
    "A_NEW_INDEPENDENT_PREDECLARED_BRANCH"
)

In [ ]:
# ============================================================
# Stage 12.13A0
#
# NEW INDEPENDENT BRANCH FREEZE
#
# POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_NODE_RECOVERY_V1
#
# ============================================================
#
# MOTIVATION
# ------------------------------------------------------------
#
# Previous independent branch:
#
#   POST_R1P50_RESIDUAL_SYNTHETIC_NODE_RECOVERY_V1
#
# was formally CLOSED.
#
# Its frozen suppressed-peak candidate universe had:
#
#   node ceiling @7um:
#       440 / 2985 = 14.740%
#
#   detection-edge ceiling @7um:
#       382 / 4097 = 9.324%
#
# and the predeclared LOFO scorer benchmark selected the
# original detector-probability baseline.
#
# Therefore the next independent branch changes the
# CANDIDATE GENERATION MECHANISM itself.
#
# ------------------------------------------------------------
# NEW PRINCIPLE
# ------------------------------------------------------------
#
# Synthetic candidates are proposed entirely from frozen
# post-G3H3 CURRENT_BEST temporal/topological geometry.
#
# Candidate existence DOES NOT require:
#
#   detector local maximum
#   suppressed peak
#   detector probability threshold
#   segmentation confidence
#   GT
#   missing-node oracle
#
# ------------------------------------------------------------
# PREDECLARED PROPOSAL FAMILIES
# ------------------------------------------------------------
#
# G0 — FORWARD_CONSTANT_VELOCITY_V1
#
#   p(t-1) --> s(t)
#
#   eligibility:
#       indegree(s) == 1
#       outdegree(s) == 0
#
#   candidate:
#       c(t+1) = 2*s - p
#
#
# G1 — BACKWARD_CONSTANT_VELOCITY_V1
#
#   u(t) --> v(t+1)
#
#   eligibility:
#       indegree(u) == 0
#       outdegree(u) == 1
#
#   candidate:
#       c(t-1) = 2*u - v
#
#
# G2 — BILATERAL_MIDPOINT_K5_V1
#
#   source endpoint s(t-1):
#       outdegree(s) == 0
#
#   target endpoint u(t+1):
#       indegree(u) == 0
#
#   Form the UNION of:
#
#       source -> K=5 nearest target endpoints
#       target -> K=5 nearest source endpoints
#
#   using physical Euclidean distance.
#
#   Every unique pair proposes:
#
#       c(t) = (s + u) / 2
#
# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------
#
# No distance cutoff is used in G2.
#
# This deliberately avoids introducing a hand-tuned motion
# threshold before the candidate ceiling is known.
#
# K=5 is frozen here before GT oracle evaluation.
#
# ------------------------------------------------------------
# BOUNDARY RULE
# ------------------------------------------------------------
#
# Forward/backward extrapolated candidates outside the input
# dataset voxel bounds are rejected.
#
# Bounds come only from input-dataset metadata:
#
#       shape
#       voxel scale
#
# No image voxel values are required.
#
# ------------------------------------------------------------
# DEDUPLICATION
# ------------------------------------------------------------
#
# Raw proposal rows are preserved.
#
# Candidate-level identity key:
#
#   dataset
#   t
#   round(z_voxel, 12)
#   round(y_voxel, 12)
#   round(x_voxel, 12)
#
# Multiple proposal families may map to one candidate.
#
# Family membership is retained as provenance.
#
# Existing-node candidates are excluded ONLY on the same exact
# rounded coordinate key at the same frame.
#
# No distance-to-existing-node threshold is used.
#
# ------------------------------------------------------------
# DEVELOPMENT PROTOCOL
# ------------------------------------------------------------
#
# folds = 1,2,3,4 only
#
# Fold0:
#   permanently consumed
#   forbidden for this branch
#
# Any eventual confirmation of a selected graph policy
# requires a genuinely untouched external/new holdout frozen
# BEFORE confirmation.
#
# ------------------------------------------------------------
# NEXT STAGES
# ------------------------------------------------------------
#
# 12.13A1
#   materialize/freeze GT-blind proposal universe
#
# 12.13A2
#   AFTER A1 freeze:
#   dev-GT oracle ceiling audit @2,3,5,7um
#
# 12.13A3
#   only if predeclared ceiling gate passes:
#   freeze ranking/attachment research contract
#
# ------------------------------------------------------------
# PREDECLARED CEILING CONTINUATION GATE
# ------------------------------------------------------------
#
# Previous suppressed-peak ceiling:
#
#   node coverage @7um = 0.1474036851
#   edge coverage @7um = 0.093239...
#
# New mechanism must achieve >= 1.5x both previous ceilings.
#
# Therefore:
#
#   node coverage @7um
#       >= 0.2211055276
#
#   detection-edge coverage @7um
#       >= 0.139859...
#
# BOTH required.
#
# Otherwise:
#
#   CLOSE_BRANCH_NO_RANKING_NO_GRAPH_MUTATION
#
# ------------------------------------------------------------
# THIS CELL
# ------------------------------------------------------------
#
# does NOT:
#
#   build graph
#   generate candidate
#   read A3
#   read GT
#   compute oracle ceiling
#   train model
#   select scorer
#   select threshold
#   mutate graph
#   use Fold0
#
# KEEP AFTER RUNNING: YES
# ============================================================


from pathlib import Path

import hashlib
import json


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


PREVIOUS_BRANCH_CLOSURE = (
    S12
    / "stage12_post_r1p50_residual_synthetic_node_recovery_v1_closure.json"
)


CONTRACT_OUT = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_branch_frozen.json"
)


assert PREVIOUS_BRANCH_CLOSURE.exists()


if CONTRACT_OUT.exists():

    raise RuntimeError(
        "\nFormal Stage12.13 branch contract already exists:\n"
        f"{CONTRACT_OUT}\n\n"
        "Do not overwrite formal branch provenance."
    )


# ============================================================
# 1. EXPECTED PREVIOUS CLOSURE
# ============================================================

PREVIOUS_BRANCH_ID = (
    "POST_R1P50_RESIDUAL_SYNTHETIC_NODE_RECOVERY_V1"
)


PREVIOUS_CLOSURE_SHA = (
    "6fb92b85062bab9a01df00eaf6492f8d"
    "925ef0a903c953e87d4b5fa9aa8f771a"
)


EXPECTED_PREVIOUS_DECISION = (
    "CLOSE_BRANCH_NO_GRAPH_MUTATION"
)


previous = json.loads(
    PREVIOUS_BRANCH_CLOSURE.read_text(
        encoding="utf-8"
    )
)


assert previous[
    "branch_id"
] == PREVIOUS_BRANCH_ID


assert previous[
    "closure_sha256"
] == PREVIOUS_CLOSURE_SHA


assert previous[
    "status"
] == "BRANCH_CLOSED"


assert previous[
    "scientific_result"
][
    "decision"
] == EXPECTED_PREVIOUS_DECISION


assert previous[
    "protocol"
][
    "graph_modified"
] is False


assert previous[
    "protocol"
][
    "fold0_used"
] is False


print(
    "========== 12.13A0 PREVIOUS BRANCH CLOSURE LOCK =========="
)


print(
    "previous branch:",
    PREVIOUS_BRANCH_ID
)


print(
    "closure SHA:",
    PREVIOUS_CLOSURE_SHA
)


print(
    "decision:",
    EXPECTED_PREVIOUS_DECISION
)


print(
    "graph mutation:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


# ============================================================
# 2. NEW BRANCH IDENTITIES
# ============================================================

BRANCH_ID = (
    "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_NODE_RECOVERY_V1"
)


CANDIDATE_UNIVERSE_ID = (
    "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_CANDIDATE_UNIVERSE_V1"
)


G0_ID = (
    "FORWARD_CONSTANT_VELOCITY_V1"
)


G1_ID = (
    "BACKWARD_CONSTANT_VELOCITY_V1"
)


G2_ID = (
    "BILATERAL_MIDPOINT_K5_V1"
)


# ============================================================
# 3. HISTORICAL CEILING BASELINE
#
# These values are frozen outcomes of the already-closed
# previous branch and are used only to predeclare whether the
# new candidate GENERATION mechanism is materially better.
# ============================================================

PREVIOUS_NODE_MISSING = 2_985
PREVIOUS_NODE_COVERED_7UM = 440


PREVIOUS_DETECTION_EDGES = 4_097
PREVIOUS_EDGE_COVERED_7UM = 382


PREVIOUS_NODE_COVERAGE_7UM = (
    PREVIOUS_NODE_COVERED_7UM
    /
    PREVIOUS_NODE_MISSING
)


PREVIOUS_EDGE_COVERAGE_7UM = (
    PREVIOUS_EDGE_COVERED_7UM
    /
    PREVIOUS_DETECTION_EDGES
)


CEILING_MULTIPLIER = 1.5


MIN_NODE_COVERAGE_7UM = (
    CEILING_MULTIPLIER
    *
    PREVIOUS_NODE_COVERAGE_7UM
)


MIN_EDGE_COVERAGE_7UM = (
    CEILING_MULTIPLIER
    *
    PREVIOUS_EDGE_COVERAGE_7UM
)


print(
    "\n========== PREDECLARED CEILING GATE =========="
)


print(
    "previous node coverage @7um:",
    PREVIOUS_NODE_COVERAGE_7UM
)


print(
    "required node coverage @7um:",
    MIN_NODE_COVERAGE_7UM
)


print(
    "previous edge coverage @7um:",
    PREVIOUS_EDGE_COVERAGE_7UM
)


print(
    "required edge coverage @7um:",
    MIN_EDGE_COVERAGE_7UM
)


# ============================================================
# 4. NEW BRANCH CONTRACT
# ============================================================

CONTRACT = {
    "stage":
        "12.13A0",

    "status":
        "NEW_INDEPENDENT_BRANCH_FROZEN",

    "branch_id":
        BRANCH_ID,

    "candidate_universe_id":
        CANDIDATE_UNIVERSE_ID,

    "scientific_question":
        (
            "CAN_GT_BLIND_TEMPORAL_TRACK_GEOMETRY_GENERATE_"
            "A_SUBSTANTIALLY_HIGHER_CEILING_SYNTHETIC_NODE_"
            "UNIVERSE_THAN_SUPPRESSED_DETECTOR_PEAKS"
        ),

    "motivation": {
        "previous_branch":
            PREVIOUS_BRANCH_ID,

        "previous_branch_closed":
            True,

        "previous_failure_mode":
            (
                "SUPPRESSED_PEAK_CANDIDATE_CEILING_TOO_LOW_"
                "AND_F1_SCORERS_FAILED_TO_BEAT_DETECTOR_"
                "BASELINE_ON_PREDECLARED_MACRO_AP_GATE"
            ),

        "change_of_mechanism":
            (
                "DETECTOR_PEAK_DEPENDENCE_REMOVED_"
                "CANDIDATES_PROPOSED_FROM_TRACK_GEOMETRY"
            ),
    },

    "canonical_graph_state": {
        "required_state":
            (
                "EXACT_FROZEN_POST_G3H3_DEVELOPMENT_"
                "CURRENT_BEST"
            ),

        "development_folds": [
            1,
            2,
            3,
            4,
        ],

        "graph_must_be_reconstructed_from_frozen_artifacts":
            True,

        "GT_allowed_in_graph_construction":
            False,

        "Fold0_allowed":
            False,
    },

    "proposal_families": {
        "G0": {
            "proposal_id":
                G0_ID,

            "mechanism":
                "FORWARD_CONSTANT_VELOCITY",

            "eligibility": {
                "endpoint_indegree":
                    1,

                "endpoint_outdegree":
                    0,

                "required_edge":
                    "PREDECESSOR_AT_T_MINUS_1_TO_ENDPOINT_AT_T",
            },

            "physical_coordinate_formula":
                "C_T_PLUS_1 = 2 * ENDPOINT_T - PREDECESSOR_T_MINUS_1",

            "candidate_frame_offset":
                1,

            "out_of_dataset_bounds":
                "REJECT",

            "detector_required":
                False,
        },

        "G1": {
            "proposal_id":
                G1_ID,

            "mechanism":
                "BACKWARD_CONSTANT_VELOCITY",

            "eligibility": {
                "endpoint_indegree":
                    0,

                "endpoint_outdegree":
                    1,

                "required_edge":
                    "ENDPOINT_AT_T_TO_SUCCESSOR_AT_T_PLUS_1",
            },

            "physical_coordinate_formula":
                "C_T_MINUS_1 = 2 * ENDPOINT_T - SUCCESSOR_T_PLUS_1",

            "candidate_frame_offset":
                -1,

            "out_of_dataset_bounds":
                "REJECT",

            "detector_required":
                False,
        },

        "G2": {
            "proposal_id":
                G2_ID,

            "mechanism":
                "BILATERAL_TEMPORAL_MIDPOINT",

            "source_endpoint":
                {
                    "frame":
                        "T_MINUS_1",

                    "outdegree":
                        0,
                },

            "target_endpoint":
                {
                    "frame":
                        "T_PLUS_1",

                    "indegree":
                        0,
                },

            "pair_generation":
                {
                    "metric":
                        "PHYSICAL_EUCLIDEAN_DISTANCE",

                    "k":
                        5,

                    "direction_1":
                        "SOURCE_TO_K_NEAREST_TARGETS",

                    "direction_2":
                        "TARGET_TO_K_NEAREST_SOURCES",

                    "pair_set":
                        "UNION",

                    "pair_deduplication":
                        "EXACT_SOURCE_ID_TARGET_ID",

                    "distance_cutoff":
                        None,
                },

            "physical_coordinate_formula":
                "C_T = (SOURCE_T_MINUS_1 + TARGET_T_PLUS_1) / 2",

            "detector_required":
                False,
        },
    },

    "coordinate_contract": {
        "distance_space":
            "PHYSICAL_MICROMETERS",

        "coordinate_order":
            [
                "z",
                "y",
                "x",
            ],

        "physical_scale_source":
            (
                "INPUT_DATASET_METADATA_ONLY_"
                "TRACKING_CELLMOT_OPEN_DATASET_LOAD_IMAGE_FALSE"
            ),

        "image_voxels_read":
            False,

        "candidate_storage_space":
            "VOXEL_COORDINATES_FLOAT64",

        "physical_to_voxel":
            "C_VOXEL = C_PHYSICAL / SCALE",
    },

    "boundary_contract": {
        "dataset_shape_source":
            "INPUT_DATASET_METADATA_ONLY",

        "valid_coordinate":
            (
                "0 <= Z < SHAPE_Z AND "
                "0 <= Y < SHAPE_Y AND "
                "0 <= X < SHAPE_X"
            ),

        "out_of_bounds_action":
            "REJECT_PROPOSAL",
    },

    "candidate_identity": {
        "raw_proposals_preserved":
            True,

        "candidate_level_key": [
            "dataset",
            "t",
            "round(z_voxel,12)",
            "round(y_voxel,12)",
            "round(x_voxel,12)",
        ],

        "rounding_decimal_places":
            12,

        "multiple_family_membership":
            "UNION_AND_RETAIN_FAMILY_PROVENANCE",

        "existing_node_collision_rule":
            (
                "EXCLUDE_ONLY_IF_SAME_FRAME_CANONICAL_NODE_"
                "HAS_IDENTICAL_ROUNDED_12DP_VOXEL_KEY"
            ),

        "existing_node_distance_cutoff":
            None,
    },

    "candidate_runtime_legality": {
        "GT":
            "FORBIDDEN",

        "missing_node_oracle":
            "FORBIDDEN",

        "A3_oracle":
            "FORBIDDEN",

        "official_FP":
            "FORBIDDEN",

        "dataset_identity_as_scoring_feature":
            "NOT_YET_DEFINED",

        "fold_identity_as_scoring_feature":
            "NOT_YET_DEFINED",

        "detector_peak_required":
            False,

        "detector_probability_required":
            False,
    },

    "oracle_audit_after_candidate_freeze": {
        "allowed_only_after":
            "12.13A1_CANDIDATE_UNIVERSE_FREEZE",

        "development_GT_informed":
            True,

        "matching_tolerances_um": [
            2.0,
            3.0,
            5.0,
            7.0,
        ],

        "primary_ceiling_metrics": [
            "MISSING_NODE_COVERAGE_AT_7UM",
            "DETECTION_LIMITED_EDGE_COVERAGE_AT_7UM",
        ],

        "secondary_metrics": [
            "CANDIDATE_POSITIVE_PREVALENCE",
            "COVERAGE_BY_PROPOSAL_FAMILY",
            "COVERAGE_BY_FOLD",
            "OVERLAP_BETWEEN_PROPOSAL_FAMILIES",
        ],
    },

    "continuation_gate": {
        "gate_id":
            "TRACK_GEOMETRY_CEILING_GATE_V1",

        "previous_node_coverage_7um":
            PREVIOUS_NODE_COVERAGE_7UM,

        "previous_edge_coverage_7um":
            PREVIOUS_EDGE_COVERAGE_7UM,

        "required_multiplier":
            CEILING_MULTIPLIER,

        "required_node_coverage_7um":
            MIN_NODE_COVERAGE_7UM,

        "required_edge_coverage_7um":
            MIN_EDGE_COVERAGE_7UM,

        "logic":
            "BOTH_REQUIRED",

        "pass_action":
            (
                "FREEZE_SEPARATE_RANKING_AND_ATTACHMENT_"
                "RESEARCH_CONTRACT_BEFORE_ANY_MODEL_TRAINING"
            ),

        "fail_action":
            (
                "CLOSE_BRANCH_NO_RANKING_NO_GRAPH_MUTATION"
            ),
    },

    "future_confirmation": {
        "Fold0":
            "PERMANENTLY_CONSUMED_FORBIDDEN",

        "if_graph_policy_eventually_selected":
            (
                "REQUIRES_GENUINELY_UNTOUCHED_EXTERNAL_OR_NEW_"
                "HOLDOUT_FROZEN_BEFORE_CONFIRMATION"
            ),
    },

    "protocol": {
        "new_independent_branch":
            True,

        "previous_closed_branch_reopened":
            False,

        "candidate_universe_generated":
            False,

        "GT_read":
            False,

        "A3_read":
            False,

        "model_trained":
            False,

        "scorer_selected":
            False,

        "threshold_selected":
            False,

        "graph_metric_evaluated":
            False,

        "graph_modified":
            False,

        "Fold0_used":
            False,
    },

    "parents": {
        "previous_branch_closure_artifact":
            PREVIOUS_BRANCH_CLOSURE.name,

        "previous_branch_closure_sha256":
            PREVIOUS_CLOSURE_SHA,
    },

    "next":
        (
            "12.13A1_MATERIALIZE_AND_FREEZE_GT_BLIND_"
            "TRACK_GEOMETRY_SYNTHETIC_CANDIDATE_UNIVERSE"
        ),
}


# ============================================================
# 5. CONTRACT HASH
# ============================================================

CONTRACT_SHA = hashlib.sha256(
    json.dumps(
        CONTRACT,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


CONTRACT[
    "contract_sha256"
] = CONTRACT_SHA


# ============================================================
# 6. TEMPORARY ATOMIC WRITE
# ============================================================

TMP = CONTRACT_OUT.with_name(
    CONTRACT_OUT.name
    +
    ".tmp_12_13a0"
)


if TMP.exists():

    raise RuntimeError(
        "\nStale temporary 12.13A0 artifact exists:\n"
        f"{TMP}\n\n"
        "Inspect and remove only this temporary file."
    )


TMP.write_text(
    json.dumps(
        CONTRACT,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 7. RELOAD FIDELITY
# ============================================================

reload = json.loads(
    TMP.read_text(
        encoding="utf-8"
    )
)


assert reload[
    "contract_sha256"
] == CONTRACT_SHA


assert reload[
    "branch_id"
] == BRANCH_ID


assert reload[
    "proposal_families"
][
    "G0"
][
    "detector_required"
] is False


assert reload[
    "proposal_families"
][
    "G1"
][
    "detector_required"
] is False


assert reload[
    "proposal_families"
][
    "G2"
][
    "detector_required"
] is False


assert reload[
    "proposal_families"
][
    "G2"
][
    "pair_generation"
][
    "k"
] == 5


assert reload[
    "proposal_families"
][
    "G2"
][
    "pair_generation"
][
    "distance_cutoff"
] is None


assert reload[
    "protocol"
][
    "candidate_universe_generated"
] is False


assert reload[
    "protocol"
][
    "GT_read"
] is False


assert reload[
    "protocol"
][
    "model_trained"
] is False


assert reload[
    "protocol"
][
    "Fold0_used"
] is False


# ============================================================
# 8. ATOMIC PROMOTION
# ============================================================

TMP.replace(
    CONTRACT_OUT
)


# ============================================================
# 9. FINAL RELOAD
# ============================================================

final = json.loads(
    CONTRACT_OUT.read_text(
        encoding="utf-8"
    )
)


assert final[
    "contract_sha256"
] == CONTRACT_SHA


# ============================================================
# 10. FINAL
# ============================================================

print(
    "\n========== 12.13A0 FINAL =========="
)


print(
    "NEW_INDEPENDENT_BRANCH_FREEZE: PASS"
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "candidate universe:",
    CANDIDATE_UNIVERSE_ID
)


print(
    "proposal G0:",
    G0_ID
)


print(
    "proposal G1:",
    G1_ID
)


print(
    "proposal G2:",
    G2_ID
)


print(
    "G2 K:",
    5
)


print(
    "G2 distance cutoff:",
    "NONE"
)


print(
    "detector peak required:",
    "NO"
)


print(
    "detector probability required:",
    "NO"
)


print(
    "candidate universe generated:",
    "NO"
)


print(
    "GT read:",
    "NO"
)


print(
    "A3 read:",
    "NO"
)


print(
    "model trained:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "graph modified:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "required node ceiling @7um:",
    MIN_NODE_COVERAGE_7UM
)


print(
    "required edge ceiling @7um:",
    MIN_EDGE_COVERAGE_7UM
)


print(
    "contract SHA:",
    CONTRACT_SHA
)


print(
    "next:"
)


print(
    "12.13A1_MATERIALIZE_AND_FREEZE_GT_BLIND_"
    "TRACK_GEOMETRY_SYNTHETIC_CANDIDATE_UNIVERSE"
)

In [ ]:
# ============================================================
# Stage 12.13A1-R3
#
# FORMAL REPAIR
#
# MATERIALIZE + FREEZE GT-BLIND
# POST-G3H3 TRACK-GEOMETRY SYNTHETIC CANDIDATE UNIVERSE
#
# ============================================================
#
# FAILURE / REPAIR HISTORY
# ------------------------------------------------------------
#
# 12.13A1
#
#   FAILURE:
#       fuzzy mutation-helper locator matched multiple
#       notebook cells.
#
#   REPAIR:
#       hard-lock historical notebook cell 66 and exact
#       mutation-helper SHA.
#
#
# 12.13A1-R1
#
#   FAILURE:
#       metadata resolver incorrectly expected Dataset.shape.
#
#   REPAIR:
#       official tracking_cellmot Dataset uses:
#
#           Dataset.image_shape = (T,Z,Y,X)
#
#
# 12.13A1-R2
#
#   FAILURE:
#       newly written graph extractor incorrectly assumed
#       TracksData edge columns:
#
#           source
#           target
#
#   ACTUAL CANONICAL SCHEMA:
#
#           edge_id
#           source_id
#           target_id
#
#   REPAIR:
#       use exact canonical TracksData schema throughout:
#
#           extract_graph_state
#           G0 incoming lookup
#           G1 outgoing lookup
#
# ------------------------------------------------------------
# SCIENTIFIC CONTRACT UNCHANGED
# ------------------------------------------------------------
#
# Branch:
#
#   POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_NODE_RECOVERY_V1
#
# Candidate universe:
#
#   POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_CANDIDATE_UNIVERSE_V1
#
#
# G0:
#
#   FORWARD_CONSTANT_VELOCITY_V1
#
#   p(t-1) -> s(t)
#
#   indegree(s) = 1
#   outdegree(s) = 0
#
#   c(t+1) = 2*s - p
#
#
# G1:
#
#   BACKWARD_CONSTANT_VELOCITY_V1
#
#   u(t) -> v(t+1)
#
#   indegree(u) = 0
#   outdegree(u) = 1
#
#   c(t-1) = 2*u - v
#
#
# G2:
#
#   BILATERAL_MIDPOINT_K5_V1
#
#   source(t-1):
#       outdegree = 0
#
#   target(t+1):
#       indegree = 0
#
#   pair set:
#
#       UNION(
#           source -> K=5 nearest targets,
#           target -> K=5 nearest sources
#       )
#
#   physical Euclidean distance
#   NO distance cutoff
#
#   c(t) = (source + target) / 2
#
# ------------------------------------------------------------
# EXACT DEVELOPMENT POST-G3H3
# ------------------------------------------------------------
#
# pre-G3H3 development CURRENT_BEST:
#
#   Baseline256
#       -> Frozen-V9
#       -> MORPH-DIV-V2
#       -> MUTUAL-CONT-V1
#       -> MUTUAL-CONT-V2
#       -> DET-BRIDGE-R1P50
#
# then exact frozen development G3H3 actions:
#
#   actions          = 12,515
#   source removals  =    385
#   edges added      = 12,515
#   net edge delta   = 12,130
#
# Deployment G3H3 scorer / threshold is NOT rerun.
#
# ------------------------------------------------------------
# INPUT METADATA
# ------------------------------------------------------------
#
# Official API:
#
#   tracking_cellmot.io.open_dataset(
#       TRAIN_ROOT / dataset,
#       load_image=False,
#       require_tracks=False,
#   )
#
# IMPORTANT:
#
#   TRAIN_ROOT / dataset is intentionally an extension-less
#   STEM.
#
#   open_dataset internally resolves:
#
#       STEM -> STEM.zarr
#
# Dataset fields:
#
#   image_shape = (T,Z,Y,X)
#   scale       = (Z,Y,X) micrometers
#   zarr_path   = real input .zarr
#
# No image chunks.
# No tracks / .geff.
#
# ------------------------------------------------------------
# BOUNDARY
# ------------------------------------------------------------
#
# Spatial:
#
#   0 <= z < Z
#   0 <= y < Y
#   0 <= x < X
#
# Temporal:
#
#   0 <= t < T
#
# Temporal check is implementation clarification of the
# already-frozen G0/G1 out_of_dataset_bounds=REJECT rule.
#
# ------------------------------------------------------------
# CANDIDATE IDENTITY
# ------------------------------------------------------------
#
#   dataset
#   t
#   round(z, 12)
#   round(y, 12)
#   round(x, 12)
#
# Fold is provenance only, NOT identity.
#
# ------------------------------------------------------------
# SAFETY
# ------------------------------------------------------------
#
# GT read                 : NO
# A3 read                 : NO
# missing-node oracle     : NO
# detector peaks          : NO
# detector probability    : NO
# tracks                   : NO
# image voxel/chunk reads : NO
# model training          : NO
# threshold selection     : NO
# graph metric            : NO
# persistent graph change : NO
# Fold0                   : NO
#
# KEEP AFTER RUNNING: YES
# ============================================================


from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Optional, Iterable, Sequence, Mapping

import ast
import builtins
import dis
import functools
import gc
import hashlib
import importlib
import inspect
import itertools
import json
import math
import os
import pickle
import re
import shutil
import sys
import types

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


NOTEBOOK_PATH = (
    PROJECT
    / "notebooks"
    / "12_residual_edge_recovery.ipynb"
)


FROZEN_V9_PATH = (
    PROJECT
    / "src"
    / "biohub_cell_tracking"
    / "frozen_v9.py"
)


TRACKING_IO_PATH = (
    PROJECT
    / "external"
    / "official"
    / "src"
    / "tracking_cellmot"
    / "io.py"
)


A0_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_branch_frozen.json"
)


DEV_ACTIONS_PATH = (
    S12
    / "stage12_folds14_rewire_asym1x5_q9990_no_target_cut_v1_actions.pkl"
)


# ============================================================
# FORMAL OUTPUTS
#
# Failed A1 / R1 / R2 stopped before formal promotion.
# Therefore successful R3 uses the originally frozen artifact
# names.
# ============================================================

RAW_OUT = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_raw_proposals_v1.pkl"
)


UNIVERSE_OUT = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_universe_v1.pkl"
)


DATASET_SUMMARY_OUT = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_dataset_summary_v1.pkl"
)


SCHEMA_OUT = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_universe_v1_schema_frozen.json"
)


SUMMARY_OUT = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_universe_v1_summary.json"
)


for path in [
    NOTEBOOK_PATH,
    FROZEN_V9_PATH,
    TRACKING_IO_PATH,
    A0_PATH,
    DEV_ACTIONS_PATH,
]:

    assert path.exists(), path


# ------------------------------------------------------------
# Fail closed if any previous failed attempt somehow promoted
# a formal artifact.
# ------------------------------------------------------------

for path in [
    RAW_OUT,
    UNIVERSE_OUT,
    DATASET_SUMMARY_OUT,
    SCHEMA_OUT,
    SUMMARY_OUT,
]:

    if path.exists():

        raise RuntimeError(
            "\nFormal 12.13A1 artifact already exists:\n"
            f"{path}\n\n"
            "Do NOT overwrite formal provenance."
        )


# ============================================================
# 1. FROZEN IDENTITIES
# ============================================================

BRANCH_ID = (
    "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_NODE_RECOVERY_V1"
)


CANDIDATE_UNIVERSE_ID = (
    "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_CANDIDATE_UNIVERSE_V1"
)


G0_ID = (
    "FORWARD_CONSTANT_VELOCITY_V1"
)


G1_ID = (
    "BACKWARD_CONSTANT_VELOCITY_V1"
)


G2_ID = (
    "BILATERAL_MIDPOINT_K5_V1"
)


A0_SHA = (
    "02ee83aa6554634c72c4a55f28e37d30"
    "83d8c311820a5a37caec99d3d0763e54"
)


DEV_ACTIONS_SHA = (
    "39e7261bbcfac40304c441be98128b0e"
    "9d9bcce4e9ce8d98beea7f69cf238b79"
)


TRACKING_IO_SHA = (
    "efae135b088cecaab463d889f16c885e"
    "f6da3ad27b0747327d8ddc28d866b7bd"
)


EXPECTED_DEV_ACTIONS = 12_515

EXPECTED_DEV_REMOVALS = 385

EXPECTED_DEV_ADDITIONS = 12_515

EXPECTED_DEV_NET = 12_130


EXPECTED_ACTION_FOLD_COUNTS = {
    1: 2_838,
    2: 3_370,
    3: 2_462,
    4: 3_845,
}


EXPECTED_DATASETS = 103


# ============================================================
# DEVELOPMENT BUILDER LOCK
# ============================================================

EXPECTED_DEV_BUILDER_CELL = 30


EXPECTED_BUILDER_CELL_SHA = (
    "84dd97f80cfe6ae91040b30d0bef7b9b"
    "53d196953054a1993580f437112ed2b9"
)


EXPECTED_BUILDER_PREFIX_SHA = (
    "ba8a67fc556bdcebe0b91dbcb563fcab"
    "803a19b38ecd2f48f7db8473234f01f7"
)


EXPECTED_DEV_P995_TOKEN = (
    "stage11_fullgraph_baseline256_folds1to4_det995"
)


EXPECTED_DEV_P9975_TOKEN = (
    "stage11_fullgraph_baseline256_folds1to4_det9975"
)


# ============================================================
# FROZEN HISTORICAL G3H3 MUTATION HELPER
# ============================================================

EXPECTED_MUTATION_CELL = 66


EXPECTED_MUTATION_HELPER_SHA = (
    "41704de3427f5f56cea1293a3ec7432c"
    "71f01a72ee10aa3622a8456b95c28111"
)


# ============================================================
# OFFICIAL INPUT-METADATA SCHEMA
#
# Read-only analysis verified all 103 development datasets.
# These assertions are safety/fidelity checks, not parameters
# used for proposal generation.
# ============================================================

EXPECTED_IMAGE_SHAPE = (
    100,
    64,
    256,
    256,
)


EXPECTED_SCALE = (
    1.625,
    0.40625,
    0.40625,
)


EXPECTED_AXIS_ORDER = (
    "T",
    "Z",
    "Y",
    "X",
)


# ============================================================
# CANDIDATE CONTRACT
# ============================================================

ROUND_DECIMALS = 12

G2_K = 5


# ============================================================
# 2. HASH HELPERS
# ============================================================

def sha256_file(
    path,
):

    h = hashlib.sha256()


    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )


            if not block:

                break


            h.update(
                block
            )


    return h.hexdigest()


class HashWriter:

    def __init__(
        self,
        hasher,
    ):

        self.hasher = hasher


    def write(
        self,
        text,
    ):

        if not isinstance(
            text,
            str,
        ):

            text = str(
                text
            )


        self.hasher.update(
            text.encode(
                "utf-8"
            )
        )


        return len(
            text
        )


    def flush(
        self,
    ):

        return None


def dataframe_content_sha256(
    df,
    columns,
):

    h = hashlib.sha256()


    writer = HashWriter(
        h
    )


    df.loc[
        :,
        columns,
    ].to_csv(
        writer,
        index=False,
        na_rep="NaN",
        float_format="%.17g",
        lineterminator="\n",
    )


    return h.hexdigest()


# ============================================================
# 3. LOCK 12.13A0
# ============================================================

a0 = json.loads(
    A0_PATH.read_text(
        encoding="utf-8"
    )
)


assert a0[
    "contract_sha256"
] == A0_SHA


assert a0[
    "branch_id"
] == BRANCH_ID


assert a0[
    "candidate_universe_id"
] == CANDIDATE_UNIVERSE_ID


assert a0[
    "proposal_families"
][
    "G0"
][
    "proposal_id"
] == G0_ID


assert a0[
    "proposal_families"
][
    "G1"
][
    "proposal_id"
] == G1_ID


assert a0[
    "proposal_families"
][
    "G2"
][
    "proposal_id"
] == G2_ID


assert a0[
    "proposal_families"
][
    "G2"
][
    "pair_generation"
][
    "k"
] == G2_K


assert a0[
    "proposal_families"
][
    "G2"
][
    "pair_generation"
][
    "distance_cutoff"
] is None


assert a0[
    "candidate_identity"
][
    "rounding_decimal_places"
] == ROUND_DECIMALS


assert a0[
    "protocol"
][
    "GT_read"
] is False


assert a0[
    "protocol"
][
    "A3_read"
] is False


assert a0[
    "protocol"
][
    "Fold0_used"
] is False


print(
    "========== 12.13A1-R3 PARENT LOCK =========="
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "A0 SHA:",
    A0_SHA
)


print(
    "G0:",
    G0_ID
)


print(
    "G1:",
    G1_ID
)


print(
    "G2:",
    G2_ID
)


print(
    "G2 K:",
    G2_K
)


print(
    "GT read:",
    "NO"
)


print(
    "A3 read:",
    "NO"
)


print(
    "Fold0:",
    "NO"
)


# ============================================================
# 4. LOCK FROZEN DEVELOPMENT G3H3 ACTIONS
# ============================================================

assert sha256_file(
    DEV_ACTIONS_PATH
) == DEV_ACTIONS_SHA


dev_actions = pd.read_pickle(
    DEV_ACTIONS_PATH
)


assert len(
    dev_actions
) == EXPECTED_DEV_ACTIONS


required_action_columns = {
    "dataset",
    "fold",
    "source_id",
    "target_id",
    "cuts",
    "source_occupied",
    "target_occupied",
}


assert required_action_columns.issubset(
    dev_actions.columns
)


for column in [
    "fold",
    "source_id",
    "target_id",
    "cuts",
    "source_occupied",
    "target_occupied",
]:

    dev_actions[
        column
    ] = pd.to_numeric(
        dev_actions[
            column
        ],
        errors="raise",
    ).astype(
        int
    )


assert dev_actions[
    "target_occupied"
].eq(
    0
).all()


assert int(
    dev_actions[
        "source_occupied"
    ].sum()
) == EXPECTED_DEV_REMOVALS


assert int(
    dev_actions[
        "cuts"
    ].sum()
) == EXPECTED_DEV_REMOVALS


assert (
    dev_actions[
        "fold"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
    ==
    EXPECTED_ACTION_FOLD_COUNTS
)


assert not dev_actions[
    [
        "dataset",
        "source_id",
        "target_id",
    ]
].duplicated().any()


actions_by_dataset = {
    dataset:
        sub.copy()

    for dataset, sub
    in dev_actions.groupby(
        "dataset",
        sort=False,
    )
}


empty_actions = (
    dev_actions.iloc[
        0:0
    ].copy()
)


print(
    "\n========== FROZEN DEVELOPMENT G3H3 LOCK =========="
)


print(
    "actions:",
    len(
        dev_actions
    )
)


print(
    "source removals:",
    int(
        dev_actions[
            "source_occupied"
        ].sum()
    )
)


print(
    "fold counts:",
    EXPECTED_ACTION_FOLD_COUNTS
)


print(
    "SHA:",
    DEV_ACTIONS_SHA
)


# ============================================================
# 5. RECOVER EXACT DEVELOPMENT PRE-G3H3 BUILDER
# ============================================================

notebook = json.loads(
    NOTEBOOK_PATH.read_text(
        encoding="utf-8"
    )
)


cells = notebook[
    "cells"
]


assert (
    0
    <=
    EXPECTED_DEV_BUILDER_CELL
    <
    len(
        cells
    )
)


builder_cell = cells[
    EXPECTED_DEV_BUILDER_CELL
]


assert builder_cell.get(
    "cell_type"
) == "code"


dev_source = "".join(
    builder_cell.get(
        "source",
        []
    )
)


BUILDER_CELL_SHA = hashlib.sha256(
    dev_source.encode(
        "utf-8"
    )
).hexdigest()


assert (
    BUILDER_CELL_SHA
    ==
    EXPECTED_BUILDER_CELL_SHA
)


assert (
    "def build_dev_current_best"
    in dev_source
)


assert (
    "Self-contained CURRENT_BEST graph-builder bootstrap"
    in dev_source
)


assert (
    EXPECTED_DEV_P995_TOKEN
    in dev_source
)


assert (
    EXPECTED_DEV_P9975_TOKEN
    in dev_source
)


assert (
    "stage10_infer_baseline_256_det995"
    not in dev_source
)


assert (
    "stage10_infer_baseline_256_det9975"
    not in dev_source
)


dev_tree = ast.parse(
    dev_source
)


builder_node = None


for node in dev_tree.body:

    if (
        isinstance(
            node,
            ast.FunctionDef,
        )
        and
        node.name
        ==
        "build_dev_current_best"
    ):

        builder_node = node

        break


assert builder_node is not None


assert builder_node.end_lineno is not None


safe_body = [
    node

    for node
    in dev_tree.body

    if int(
        getattr(
            node,
            "end_lineno",
            0,
        )
    )
    <=
    int(
        builder_node.end_lineno
    )
]


assert safe_body


safe_source = "\n".join(
    ast.get_source_segment(
        dev_source,
        node
    )
    or
    ast.unparse(
        node
    )

    for node
    in safe_body
)


BUILDER_PREFIX_SHA = hashlib.sha256(
    safe_source.encode(
        "utf-8"
    )
).hexdigest()


assert (
    BUILDER_PREFIX_SHA
    ==
    EXPECTED_BUILDER_PREFIX_SHA
)


# ============================================================
# 6. EXACT IMPORTS + SAFE BUILDER NAMESPACE
# ============================================================

fv9_exact = importlib.import_module(
    "biohub_cell_tracking.frozen_v9"
)


tracking_io_exact = importlib.import_module(
    "tracking_cellmot.io"
)


tracksdata_exact = importlib.import_module(
    "tracksdata"
)


assert Path(
    inspect.getsourcefile(
        fv9_exact
    )
).resolve() == FROZEN_V9_PATH.resolve()


assert sha256_file(
    TRACKING_IO_PATH
) == TRACKING_IO_SHA


# ------------------------------------------------------------
# Official API safety gate.
# ------------------------------------------------------------

open_dataset_signature = inspect.signature(
    tracking_io_exact.open_dataset
)


assert (
    "require_tracks"
    in
    open_dataset_signature.parameters
)


assert (
    open_dataset_signature.parameters[
        "require_tracks"
    ].default
    is False
)


dev_ns = {
    "__name__":
        "__stage12_13a1_r3_dev_builder__",

    "__file__":
        (
            f"{NOTEBOOK_PATH}"
            f"::cell{EXPECTED_DEV_BUILDER_CELL}"
        ),

    "__package__":
        None,

    "__builtins__":
        builtins.__dict__,

    "Path":
        Path,

    "Counter":
        Counter,

    "defaultdict":
        defaultdict,

    "Any":
        Any,

    "Optional":
        Optional,

    "Iterable":
        Iterable,

    "Sequence":
        Sequence,

    "Mapping":
        Mapping,

    "ast":
        ast,

    "builtins":
        builtins,

    "dis":
        dis,

    "functools":
        functools,

    "gc":
        gc,

    "hashlib":
        hashlib,

    "importlib":
        importlib,

    "inspect":
        inspect,

    "itertools":
        itertools,

    "json":
        json,

    "math":
        math,

    "os":
        os,

    "pickle":
        pickle,

    "re":
        re,

    "shutil":
        shutil,

    "sys":
        sys,

    "types":
        types,

    "np":
        np,

    "pd":
        pd,

    "cKDTree":
        cKDTree,

    "fv9":
        fv9_exact,

    "td":
        tracksdata_exact,

    "open_dataset":
        tracking_io_exact.open_dataset,

    "load_graph":
        fv9_exact.load_graph,

    "step_balance":
        fv9_exact.step_balance,

    "recovered_edge_dist":
        fv9_exact.recovered_edge_dist,

    "FROZEN_V9_CONFIG":
        fv9_exact.FROZEN_V9_CONFIG,

    "PROJECT":
        PROJECT,

    "S12":
        S12,
}


safe_module = ast.Module(
    body=safe_body,
    type_ignores=[],
)


safe_module = ast.fix_missing_locations(
    safe_module
)


exec(
    compile(
        safe_module,
        filename=(
            f"{NOTEBOOK_PATH}"
            f"::cell{EXPECTED_DEV_BUILDER_CELL}"
        ),
        mode="exec",
    ),
    dev_ns,
)


assert "build_dev_current_best" in dev_ns


assert callable(
    dev_ns[
        "build_dev_current_best"
    ]
)


build_dev_current_best = (
    dev_ns[
        "build_dev_current_best"
    ]
)


fv9 = dev_ns[
    "fv9"
]


TRAIN_ROOT = Path(
    dev_ns[
        "TRAIN_ROOT"
    ]
).resolve()


assert TRAIN_ROOT.exists()


assert Path(
    dev_ns[
        "B256_995"
    ]
).exists()


assert Path(
    dev_ns[
        "B256_9975"
    ]
).exists()


print(
    "\n========== DEVELOPMENT BUILDER =========="
)


print(
    "cell:",
    EXPECTED_DEV_BUILDER_CELL
)


print(
    "cell SHA:",
    BUILDER_CELL_SHA
)


print(
    "prefix SHA:",
    BUILDER_PREFIX_SHA
)


print(
    "TRAIN_ROOT:",
    TRAIN_ROOT
)


print(
    "TRAIN_ROOT semantics:",
    "SPLIT ROOT FOR EXTENSION-LESS DATASET STEM"
)


print(
    "P995:",
    dev_ns[
        "B256_995"
    ]
)


print(
    "P9975:",
    dev_ns[
        "B256_9975"
    ]
)


print(
    "EXACT_DEVELOPMENT_PRE_G3H3_BUILDER: PASS"
)


# ============================================================
# 7. EXACT HISTORICAL MUTATION HELPER
#
# Hard-locked.
# No fuzzy notebook-cell search.
# ============================================================

assert (
    0
    <=
    EXPECTED_MUTATION_CELL
    <
    len(
        cells
    )
)


mutation_cell = cells[
    EXPECTED_MUTATION_CELL
]


assert mutation_cell.get(
    "cell_type"
) == "code"


mutation_source = "".join(
    mutation_cell.get(
        "source",
        []
    )
)


mutation_tree = ast.parse(
    mutation_source
)


top_level_functions = {
    node.name:
        node

    for node
    in mutation_tree.body

    if isinstance(
        node,
        ast.FunctionDef,
    )
}


assert (
    "local_graph_state"
    in
    top_level_functions
)


assert (
    "apply_locked_fold0_policy"
    in
    top_level_functions
)


mutation_nodes = [
    top_level_functions[
        "local_graph_state"
    ],

    top_level_functions[
        "apply_locked_fold0_policy"
    ],
]


mutation_nodes = sorted(
    mutation_nodes,
    key=lambda node:
        int(
            node.lineno
        ),
)


mutation_source_exact = "\n\n".join(
    ast.get_source_segment(
        mutation_source,
        node
    )
    or
    ast.unparse(
        node
    )

    for node
    in mutation_nodes
)


MUTATION_SOURCE_SHA = hashlib.sha256(
    mutation_source_exact.encode(
        "utf-8"
    )
).hexdigest()


assert (
    MUTATION_SOURCE_SHA
    ==
    EXPECTED_MUTATION_HELPER_SHA
)


assert (
    "bulk_remove_edges"
    in
    mutation_source_exact
)


assert (
    "graph.add_edge"
    in
    mutation_source_exact
)


assert (
    "fv9.recovered_edge_dist"
    in
    mutation_source_exact
)


assert (
    '"edge_prob"'
    in
    mutation_source_exact
)


assert (
    "evaluate_graph"
    not in
    mutation_source_exact
)


assert (
    "open_dataset"
    not in
    mutation_source_exact
)


mutation_ns = {
    "__name__":
        "__stage12_13a1_r3_frozen_mutation__",

    "__builtins__":
        builtins.__dict__,

    "pd":
        pd,

    "fv9":
        fv9,

    "int":
        int,

    "float":
        float,

    "len":
        len,

    "set":
        set,

    "sorted":
        sorted,

    "zip":
        zip,
}


mutation_module = ast.Module(
    body=mutation_nodes,
    type_ignores=[],
)


mutation_module = ast.fix_missing_locations(
    mutation_module
)


exec(
    compile(
        mutation_module,
        filename=(
            f"{NOTEBOOK_PATH}"
            f"::cell{EXPECTED_MUTATION_CELL}"
            "::exact_frozen_mutation"
        ),
        mode="exec",
    ),
    mutation_ns,
)


assert (
    "local_graph_state"
    in mutation_ns
)


assert (
    "apply_locked_fold0_policy"
    in mutation_ns
)


exact_apply_frozen_policy = (
    mutation_ns[
        "apply_locked_fold0_policy"
    ]
)


assert callable(
    exact_apply_frozen_policy
)


print(
    "\n========== EXACT FROZEN MUTATION =========="
)


print(
    "locator:",
    "HARD_LOCKED_AST_FUNCTIONDEF"
)


print(
    "cell:",
    EXPECTED_MUTATION_CELL
)


print(
    "SHA:",
    MUTATION_SOURCE_SHA
)


print(
    "EXACT_FROZEN_MUTATION_HELPER: PASS"
)


# ============================================================
# 8. EXACT POST-G3H3 DEVELOPMENT BUILDER
# ============================================================

def build_dev_post_g3h3(
    dataset,
):

    graph, scale = build_dev_current_best(
        dataset
    )


    actions = actions_by_dataset.get(
        dataset,
        empty_actions,
    )


    before = int(
        graph.num_edges()
    )


    graph, stats = exact_apply_frozen_policy(
        graph,
        actions,
    )


    after = int(
        graph.num_edges()
    )


    assert int(
        stats[
            "actions"
        ]
    ) == len(
        actions
    )


    assert int(
        stats[
            "edges_added"
        ]
    ) == len(
        actions
    )


    assert (
        after
        -
        before
    ) == int(
        stats[
            "net_edge_delta"
        ]
    )


    return (
        graph,

        np.asarray(
            scale,
            dtype=np.float64,
        ),

        stats,
    )


# ============================================================
# 9. EXACT OFFICIAL METADATA RESOLVER
#
# TRAIN_ROOT / dataset is an extension-less STEM.
#
# open_dataset resolves STEM.zarr internally.
#
# No image voxels.
# No tracks.
# ============================================================

def load_dataset_metadata(
    dataset,
):

    stem = (
        TRAIN_ROOT
        /
        dataset
    )


    ds = tracking_io_exact.open_dataset(
        stem,
        load_image=False,
        require_tracks=False,
    )


    # --------------------------------------------------------
    # Strict GT / voxel safety.
    # --------------------------------------------------------

    assert ds.image is None


    assert ds.tracks is None


    # --------------------------------------------------------
    # Physical input container.
    # --------------------------------------------------------

    zarr_path = Path(
        ds.zarr_path
    ).resolve()


    assert zarr_path.exists(), zarr_path


    assert zarr_path.name == (
        f"{dataset}.zarr"
    )


    # --------------------------------------------------------
    # Official Dataset API:
    #
    # image_shape = (T,Z,Y,X)
    # --------------------------------------------------------

    image_shape = tuple(
        int(
            value
        )

        for value
        in ds.image_shape
    )


    assert len(
        image_shape
    ) == 4


    (
        T,
        Z,
        Y,
        X,
    ) = image_shape


    assert min(
        T,
        Z,
        Y,
        X,
    ) > 0


    # --------------------------------------------------------
    # Official spatial physical scale:
    #
    # (Z,Y,X) micrometers / voxel.
    # --------------------------------------------------------

    scale = np.asarray(
        ds.scale,
        dtype=np.float64,
    )


    assert scale.shape == (
        3,
    )


    assert np.isfinite(
        scale
    ).all()


    assert (
        scale
        >
        0
    ).all()


    return {
        "dataset":
            str(
                dataset
            ),

        "stem":
            str(
                stem
            ),

        "zarr_path":
            str(
                zarr_path
            ),

        "T":
            int(
                T
            ),

        "Z":
            int(
                Z
            ),

        "Y":
            int(
                Y
            ),

        "X":
            int(
                X
            ),

        "image_shape":
            image_shape,

        "scale":
            scale,

        "image_is_none":
            True,

        "tracks_is_none":
            True,
    }


# ============================================================
# 10. COORDINATE / BOUNDARY HELPERS
# ============================================================

def rounded_coordinate(
    value,
):

    return float(
        np.round(
            np.float64(
                value
            ),
            ROUND_DECIMALS,
        )
    )


def coordinate_key(
    dataset,
    t,
    voxel_xyz,
):

    voxel_xyz = np.asarray(
        voxel_xyz,
        dtype=np.float64,
    ).reshape(
        3
    )


    return (
        str(
            dataset
        ),

        int(
            t
        ),

        rounded_coordinate(
            voxel_xyz[
                0
            ]
        ),

        rounded_coordinate(
            voxel_xyz[
                1
            ]
        ),

        rounded_coordinate(
            voxel_xyz[
                2
            ]
        ),
    )


def candidate_in_bounds(
    t,
    voxel_xyz,
    metadata,
):

    voxel_xyz = np.asarray(
        voxel_xyz,
        dtype=np.float64,
    ).reshape(
        3
    )


    z = float(
        voxel_xyz[
            0
        ]
    )


    y = float(
        voxel_xyz[
            1
        ]
    )


    x = float(
        voxel_xyz[
            2
        ]
    )


    return bool(
        0
        <=
        int(
            t
        )
        <
        int(
            metadata[
                "T"
            ]
        )

        and

        0.0
        <=
        z
        <
        float(
            metadata[
                "Z"
            ]
        )

        and

        0.0
        <=
        y
        <
        float(
            metadata[
                "Y"
            ]
        )

        and

        0.0
        <=
        x
        <
        float(
            metadata[
                "X"
            ]
        )
    )


def query_knn_indices(
    tree,
    queries,
    k,
):

    queries = np.asarray(
        queries,
        dtype=np.float64,
    )


    assert queries.ndim == 2


    distance, index = tree.query(
        queries,
        k=int(
            k
        ),
    )


    distance = np.asarray(
        distance,
        dtype=np.float64,
    )


    index = np.asarray(
        index,
        dtype=np.int64,
    )


    if int(
        k
    ) == 1:

        distance = distance.reshape(
            -1,
            1,
        )


        index = index.reshape(
            -1,
            1,
        )


    assert (
        distance.shape
        ==
        index.shape
    )


    assert distance.shape[
        0
    ] == len(
        queries
    )


    return (
        distance,
        index,
    )


# ============================================================
# 11. R3 CANONICAL GRAPH STATE EXTRACTOR
#
# IMPORTANT:
#
# TracksData canonical edge dataframe schema:
#
#   edge_id
#   source_id
#   target_id
#
# NOT:
#
#   source
#   target
#
# ============================================================

def extract_graph_state(
    graph,
    scale,
):

    nodes = (
        graph
        .node_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    edges = (
        graph
        .edge_attrs(
            unpack=True
        )
        .to_pandas()
        .copy()
    )


    # ========================================================
    # NODE SCHEMA
    # ========================================================

    assert {
        "node_id",
        "t",
        "z",
        "y",
        "x",
    }.issubset(
        nodes.columns
    ), (
        "\nUnexpected canonical node schema:\n"
        f"{list(nodes.columns)}"
    )


    # ========================================================
    # R3 EDGE SCHEMA
    # ========================================================

    assert {
        "edge_id",
        "source_id",
        "target_id",
    }.issubset(
        edges.columns
    ), (
        "\nUnexpected canonical TracksData edge schema:\n"
        f"{list(edges.columns)}"
    )


    # ========================================================
    # NODE TYPES
    # ========================================================

    nodes[
        "node_id"
    ] = pd.to_numeric(
        nodes[
            "node_id"
        ],
        errors="raise",
    ).astype(
        np.int64
    )


    nodes[
        "t"
    ] = pd.to_numeric(
        nodes[
            "t"
        ],
        errors="raise",
    ).astype(
        np.int64
    )


    for column in [
        "z",
        "y",
        "x",
    ]:

        nodes[
            column
        ] = pd.to_numeric(
            nodes[
                column
            ],
            errors="raise",
        ).astype(
            np.float64
        )


    assert nodes[
        "node_id"
    ].is_unique


    node_ids = nodes[
        "node_id"
    ].to_numpy(
        dtype=np.int64
    )


    voxel = nodes[
        [
            "z",
            "y",
            "x",
        ]
    ].to_numpy(
        dtype=np.float64
    )


    assert voxel.shape == (
        len(
            nodes
        ),
        3,
    )


    assert np.isfinite(
        voxel
    ).all()


    # ========================================================
    # PHYSICAL COORDINATES
    # ========================================================

    scale = np.asarray(
        scale,
        dtype=np.float64,
    )


    assert scale.shape == (
        3,
    )


    physical = (
        voxel
        *
        scale[
            None,
            :
        ]
    )


    nodes[
        "phys_z"
    ] = physical[
        :,
        0
    ]


    nodes[
        "phys_y"
    ] = physical[
        :,
        1
    ]


    nodes[
        "phys_x"
    ] = physical[
        :,
        2
    ]


    # ========================================================
    # CANONICAL DEGREES
    # ========================================================

    indegree = np.asarray(
        graph.in_degree(
            node_ids.tolist()
        ),
        dtype=np.int64,
    )


    outdegree = np.asarray(
        graph.out_degree(
            node_ids.tolist()
        ),
        dtype=np.int64,
    )


    assert len(
        indegree
    ) == len(
        nodes
    )


    assert len(
        outdegree
    ) == len(
        nodes
    )


    nodes[
        "indegree"
    ] = indegree


    nodes[
        "outdegree"
    ] = outdegree


    # ========================================================
    # EDGE TYPES
    # ========================================================

    edges[
        "edge_id"
    ] = pd.to_numeric(
        edges[
            "edge_id"
        ],
        errors="raise",
    ).astype(
        np.int64
    )


    edges[
        "source_id"
    ] = pd.to_numeric(
        edges[
            "source_id"
        ],
        errors="raise",
    ).astype(
        np.int64
    )


    edges[
        "target_id"
    ] = pd.to_numeric(
        edges[
            "target_id"
        ],
        errors="raise",
    ).astype(
        np.int64
    )


    assert edges[
        "edge_id"
    ].is_unique


    # ========================================================
    # NODE LOOKUP
    # ========================================================

    node_lookup = (
        nodes
        .set_index(
            "node_id",
            drop=False,
        )
    )


    node_id_set = set(
        node_lookup.index.tolist()
    )


    assert set(
        edges[
            "source_id"
        ].tolist()
    ).issubset(
        node_id_set
    )


    assert set(
        edges[
            "target_id"
        ].tolist()
    ).issubset(
        node_id_set
    )


    # ========================================================
    # TEMPORAL DIRECTION SANITY
    #
    # We require forward temporal direction only.
    #
    # Exact +1 frame adjacency is enforced individually by G0
    # and G1 when proposals are generated.
    # ========================================================

    if len(
        edges
    ):

        source_t = node_lookup.loc[
            edges[
                "source_id"
            ].to_numpy(
                dtype=np.int64
            ),
            "t",
        ].to_numpy(
            dtype=np.int64
        )


        target_t = node_lookup.loc[
            edges[
                "target_id"
            ].to_numpy(
                dtype=np.int64
            ),
            "t",
        ].to_numpy(
            dtype=np.int64
        )


        assert (
            target_t
            >
            source_t
        ).all(), (
            "Canonical graph contains a non-forward "
            "temporal edge."
        )


    return (
        nodes,
        edges,
        node_lookup,
    )


# ============================================================
# 12. EXACT DATASET -> DEVELOPMENT FOLD
#
# Dataset fold is provenance.
# It is NOT a runtime/candidate feature.
# ============================================================

def resolve_dataset_fold(
    dataset,
):

    fold_values = []


    provenance_sources = []


    # --------------------------------------------------------
    # Preferred: final frozen development G3H3 action table.
    # --------------------------------------------------------

    if dataset in actions_by_dataset:

        values = sorted(
            actions_by_dataset[
                dataset
            ][
                "fold"
            ]
            .astype(
                int
            )
            .unique()
            .tolist()
        )


        for value in values:

            fold_values.append(
                int(
                    value
                )
            )


            provenance_sources.append(
                (
                    "FINAL_G3H3_ACTIONS",
                    int(
                        value
                    ),
                )
            )


    # --------------------------------------------------------
    # For datasets with no final G3H3 action, recover only from
    # already-loaded frozen bootstrap tables in exact builder
    # namespace.
    # --------------------------------------------------------

    for name, obj in dev_ns.items():

        if not isinstance(
            obj,
            pd.DataFrame,
        ):

            continue


        if not {
            "dataset",
            "fold",
        }.issubset(
            obj.columns
        ):

            continue


        match = obj.loc[
            obj[
                "dataset"
            ].astype(
                str
            ).eq(
                str(
                    dataset
                )
            )
        ]


        if len(
            match
        ) == 0:

            continue


        values = (
            pd.to_numeric(
                match[
                    "fold"
                ],
                errors="coerce",
            )
            .dropna()
            .astype(
                int
            )
            .unique()
            .tolist()
        )


        for value in values:

            fold_values.append(
                int(
                    value
                )
            )


            provenance_sources.append(
                (
                    str(
                        name
                    ),

                    int(
                        value
                    ),
                )
            )


    unique = sorted(
        set(
            fold_values
        )
    )


    if len(
        unique
    ) != 1:

        raise RuntimeError(
            "\nDevelopment fold cannot be resolved exactly.\n"
            f"dataset={dataset}\n"
            f"values={unique}\n"
            f"sources={provenance_sources[:100]}"
        )


    fold = int(
        unique[
            0
        ]
    )


    assert fold in {
        1,
        2,
        3,
        4,
    }


    return fold


# ============================================================
# 13. DEVELOPMENT DATASETS
# ============================================================

datasets = sorted(
    dev_ns[
        "bootstrap_name_set"
    ]
)


assert len(
    datasets
) == EXPECTED_DATASETS


assert all(
    str(
        dataset
    ).startswith(
        "6bba_"
    )

    for dataset
    in datasets
)


# ============================================================
# 13A. R3 TRACKSDATA EDGE-SCHEMA PREFLIGHT
#
# GT-blind.
#
# Builds only the first canonical post-G3H3 development graph.
#
# No candidate generated.
# No formal artifact written.
# ============================================================

preflight_dataset = datasets[
    0
]


(
    preflight_graph,
    preflight_scale,
    preflight_stats,
) = build_dev_post_g3h3(
    preflight_dataset
)


preflight_edges = (
    preflight_graph
    .edge_attrs(
        unpack=True
    )
    .to_pandas()
)


print(
    "\n========== R3 TRACKSDATA EDGE SCHEMA =========="
)


print(
    "dataset:",
    preflight_dataset
)


print(
    "edge columns:",
    list(
        preflight_edges.columns
    )
)


assert {
    "edge_id",
    "source_id",
    "target_id",
}.issubset(
    preflight_edges.columns
)


assert (
    "source"
    not in
    preflight_edges.columns
)


assert (
    "target"
    not in
    preflight_edges.columns
)


print(
    "edge_id:",
    "YES"
)


print(
    "source_id:",
    "YES"
)


print(
    "target_id:",
    "YES"
)


print(
    "legacy source:",
    "NO"
)


print(
    "legacy target:",
    "NO"
)


print(
    "TRACKSDATA_CANONICAL_EDGE_SCHEMA: PASS"
)


del preflight_graph
del preflight_edges
del preflight_scale
del preflight_stats


gc.collect()


# ============================================================
# 14. MATERIALIZATION STATE
# ============================================================

raw_parts = []


dataset_summary_rows = []


raw_counter = 0


total_replay_actions = 0

total_replay_removals = 0

total_replay_additions = 0

total_replay_net = 0


print(
    "\n========== 12.13A1-R3 GT-BLIND MATERIALIZATION =========="
)


# ============================================================
# 15. DATASET LOOP
# ============================================================

for dataset_index, dataset in enumerate(
    datasets,
    start=1,
):

    # ========================================================
    # EXACT FROZEN POST-G3H3 DEVELOPMENT CURRENT_BEST
    # ========================================================

    (
        graph,
        scale,
        replay_stats,
    ) = build_dev_post_g3h3(
        dataset
    )


    total_replay_actions += int(
        replay_stats[
            "actions"
        ]
    )


    total_replay_removals += int(
        replay_stats[
            "source_incumbent_edges_removed"
        ]
    )


    total_replay_additions += int(
        replay_stats[
            "edges_added"
        ]
    )


    total_replay_net += int(
        replay_stats[
            "net_edge_delta"
        ]
    )


    # ========================================================
    # OFFICIAL GT-BLIND INPUT METADATA
    # ========================================================

    metadata = load_dataset_metadata(
        dataset
    )


    # --------------------------------------------------------
    # Read-only audit showed this schema for all 103.
    #
    # Treat as fidelity assertion, never as a fallback source.
    # --------------------------------------------------------

    assert tuple(
        metadata[
            "image_shape"
        ]
    ) == EXPECTED_IMAGE_SHAPE


    assert np.array_equal(
        metadata[
            "scale"
        ],
        np.asarray(
            EXPECTED_SCALE,
            dtype=np.float64,
        ),
    )


    # Exact builder scale must equal official metadata scale.
    assert np.array_equal(
        np.asarray(
            scale,
            dtype=np.float64,
        ),
        metadata[
            "scale"
        ],
    )


    # ========================================================
    # CANONICAL GRAPH STATE
    # ========================================================

    (
        nodes,
        edges,
        node_lookup,
    ) = extract_graph_state(
        graph,
        scale,
    )


    fold = resolve_dataset_fold(
        dataset
    )


    # ========================================================
    # CANONICAL GRAPH BOUNDS AUDIT
    # ========================================================

    assert (
        nodes[
            "t"
        ]
        >=
        0
    ).all()


    assert (
        nodes[
            "t"
        ]
        <
        metadata[
            "T"
        ]
    ).all()


    for column, bound in [
        (
            "z",
            metadata[
                "Z"
            ],
        ),
        (
            "y",
            metadata[
                "Y"
            ],
        ),
        (
            "x",
            metadata[
                "X"
            ],
        ),
    ]:

        values = pd.to_numeric(
            nodes[
                column
            ],
            errors="raise",
        ).to_numpy(
            dtype=np.float64
        )


        assert (
            values
            >=
            0
        ).all()


        assert (
            values
            <
            float(
                bound
            )
        ).all()


    # ========================================================
    # EXACT CANONICAL NODE COLLISION KEYS
    # ========================================================

    existing_node_keys = {
        coordinate_key(
            dataset,
            int(
                row.t
            ),
            np.asarray(
                [
                    row.z,
                    row.y,
                    row.x,
                ],
                dtype=np.float64,
            ),
        )

        for row
        in nodes.itertuples(
            index=False
        )
    }


    dataset_raw_rows = []


    # ========================================================
    # G0 — FORWARD CONSTANT VELOCITY
    #
    # Canonical edge schema:
    #
    #   source_id -> target_id
    # ========================================================

    incoming_by_target = {
        int(
            target_id
        ):
            sub[
                "source_id"
            ]
            .astype(
                np.int64
            )
            .tolist()

        for target_id, sub
        in edges.groupby(
            "target_id",
            sort=False,
        )
    }


    g0_endpoints = (
        nodes.loc[
            nodes[
                "indegree"
            ].eq(
                1
            )
            &
            nodes[
                "outdegree"
            ].eq(
                0
            )
        ]
        .sort_values(
            "node_id",
            kind="stable",
        )
    )


    for row in g0_endpoints.itertuples(
        index=False
    ):

        endpoint_id = int(
            row.node_id
        )


        predecessors = incoming_by_target.get(
            endpoint_id,
            [],
        )


        assert len(
            predecessors
        ) == 1


        predecessor_id = int(
            predecessors[
                0
            ]
        )


        predecessor = node_lookup.loc[
            predecessor_id
        ]


        # ----------------------------------------------------
        # Frozen G0 eligibility:
        #
        # exact t-1 -> t edge.
        # ----------------------------------------------------

        if (
            int(
                row.t
            )
            -
            int(
                predecessor[
                    "t"
                ]
            )
            !=
            1
        ):

            continue


        endpoint_phys = np.asarray(
            [
                row.phys_z,
                row.phys_y,
                row.phys_x,
            ],
            dtype=np.float64,
        )


        predecessor_phys = np.asarray(
            [
                predecessor[
                    "phys_z"
                ],
                predecessor[
                    "phys_y"
                ],
                predecessor[
                    "phys_x"
                ],
            ],
            dtype=np.float64,
        )


        candidate_phys = (
            2.0
            *
            endpoint_phys
            -
            predecessor_phys
        )


        # ----------------------------------------------------
        # Literal A0 physical -> voxel contract.
        # ----------------------------------------------------

        candidate_voxel = (
            candidate_phys
            /
            scale
        )


        candidate_t = (
            int(
                row.t
            )
            +
            1
        )


        in_bounds = candidate_in_bounds(
            candidate_t,
            candidate_voxel,
            metadata,
        )


        collision = False


        if in_bounds:

            collision = (
                coordinate_key(
                    dataset,
                    candidate_t,
                    candidate_voxel,
                )
                in
                existing_node_keys
            )


        dataset_raw_rows.append(
            {
                "raw_proposal_id":
                    int(
                        raw_counter
                    ),

                "dataset":
                    str(
                        dataset
                    ),

                "fold":
                    int(
                        fold
                    ),

                "family":
                    G0_ID,

                "candidate_t":
                    int(
                        candidate_t
                    ),

                "z":
                    float(
                        candidate_voxel[
                            0
                        ]
                    ),

                "y":
                    float(
                        candidate_voxel[
                            1
                        ]
                    ),

                "x":
                    float(
                        candidate_voxel[
                            2
                        ]
                    ),

                "source_node_id":
                    int(
                        predecessor_id
                    ),

                "anchor_node_id":
                    int(
                        endpoint_id
                    ),

                "target_node_id":
                    np.nan,

                "g2_source_to_target_knn":
                    0,

                "g2_target_to_source_knn":
                    0,

                "g2_pair_distance_um":
                    np.nan,

                "in_bounds":
                    bool(
                        in_bounds
                    ),

                "existing_node_collision":
                    bool(
                        collision
                    ),

                "accepted":
                    bool(
                        in_bounds
                        and
                        not collision
                    ),
            }
        )


        raw_counter += 1


    # ========================================================
    # G1 — BACKWARD CONSTANT VELOCITY
    #
    # Canonical edge schema:
    #
    #   source_id -> target_id
    # ========================================================

    outgoing_by_source = {
        int(
            source_id
        ):
            sub[
                "target_id"
            ]
            .astype(
                np.int64
            )
            .tolist()

        for source_id, sub
        in edges.groupby(
            "source_id",
            sort=False,
        )
    }


    g1_endpoints = (
        nodes.loc[
            nodes[
                "indegree"
            ].eq(
                0
            )
            &
            nodes[
                "outdegree"
            ].eq(
                1
            )
        ]
        .sort_values(
            "node_id",
            kind="stable",
        )
    )


    for row in g1_endpoints.itertuples(
        index=False
    ):

        endpoint_id = int(
            row.node_id
        )


        successors = outgoing_by_source.get(
            endpoint_id,
            [],
        )


        assert len(
            successors
        ) == 1


        successor_id = int(
            successors[
                0
            ]
        )


        successor = node_lookup.loc[
            successor_id
        ]


        # ----------------------------------------------------
        # Frozen G1 eligibility:
        #
        # exact t -> t+1 edge.
        # ----------------------------------------------------

        if (
            int(
                successor[
                    "t"
                ]
            )
            -
            int(
                row.t
            )
            !=
            1
        ):

            continue


        endpoint_phys = np.asarray(
            [
                row.phys_z,
                row.phys_y,
                row.phys_x,
            ],
            dtype=np.float64,
        )


        successor_phys = np.asarray(
            [
                successor[
                    "phys_z"
                ],
                successor[
                    "phys_y"
                ],
                successor[
                    "phys_x"
                ],
            ],
            dtype=np.float64,
        )


        candidate_phys = (
            2.0
            *
            endpoint_phys
            -
            successor_phys
        )


        candidate_voxel = (
            candidate_phys
            /
            scale
        )


        candidate_t = (
            int(
                row.t
            )
            -
            1
        )


        in_bounds = candidate_in_bounds(
            candidate_t,
            candidate_voxel,
            metadata,
        )


        collision = False


        if in_bounds:

            collision = (
                coordinate_key(
                    dataset,
                    candidate_t,
                    candidate_voxel,
                )
                in
                existing_node_keys
            )


        dataset_raw_rows.append(
            {
                "raw_proposal_id":
                    int(
                        raw_counter
                    ),

                "dataset":
                    str(
                        dataset
                    ),

                "fold":
                    int(
                        fold
                    ),

                "family":
                    G1_ID,

                "candidate_t":
                    int(
                        candidate_t
                    ),

                "z":
                    float(
                        candidate_voxel[
                            0
                        ]
                    ),

                "y":
                    float(
                        candidate_voxel[
                            1
                        ]
                    ),

                "x":
                    float(
                        candidate_voxel[
                            2
                        ]
                    ),

                "source_node_id":
                    int(
                        endpoint_id
                    ),

                "anchor_node_id":
                    int(
                        endpoint_id
                    ),

                "target_node_id":
                    int(
                        successor_id
                    ),

                "g2_source_to_target_knn":
                    0,

                "g2_target_to_source_knn":
                    0,

                "g2_pair_distance_um":
                    np.nan,

                "in_bounds":
                    bool(
                        in_bounds
                    ),

                "existing_node_collision":
                    bool(
                        collision
                    ),

                "accepted":
                    bool(
                        in_bounds
                        and
                        not collision
                    ),
            }
        )


        raw_counter += 1


    # ========================================================
    # G2 — BILATERAL MIDPOINT K5
    # ========================================================

    source_endpoints = (
        nodes.loc[
            nodes[
                "outdegree"
            ].eq(
                0
            )
        ]
        .copy()
    )


    target_endpoints = (
        nodes.loc[
            nodes[
                "indegree"
            ].eq(
                0
            )
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Middle frame t:
    #
    #   source at t-1
    #   target at t+1
    #
    # t is necessarily in 1..T-2.
    # --------------------------------------------------------

    for middle_t in range(
        1,
        metadata[
            "T"
        ] - 1,
    ):

        sources = (
            source_endpoints.loc[
                source_endpoints[
                    "t"
                ].eq(
                    middle_t
                    -
                    1
                )
            ]
            .sort_values(
                "node_id",
                kind="stable",
            )
            .reset_index(
                drop=True
            )
        )


        targets = (
            target_endpoints.loc[
                target_endpoints[
                    "t"
                ].eq(
                    middle_t
                    +
                    1
                )
            ]
            .sort_values(
                "node_id",
                kind="stable",
            )
            .reset_index(
                drop=True
            )
        )


        if (
            len(
                sources
            )
            ==
            0
            or
            len(
                targets
            )
            ==
            0
        ):

            continue


        source_phys = sources[
            [
                "phys_z",
                "phys_y",
                "phys_x",
            ]
        ].to_numpy(
            dtype=np.float64
        )


        target_phys = targets[
            [
                "phys_z",
                "phys_y",
                "phys_x",
            ]
        ].to_numpy(
            dtype=np.float64
        )


        k_target = min(
            G2_K,
            len(
                targets
            ),
        )


        k_source = min(
            G2_K,
            len(
                sources
            ),
        )


        target_tree = cKDTree(
            target_phys
        )


        source_tree = cKDTree(
            source_phys
        )


        (
            _s2t_distance,
            s2t_index,
        ) = query_knn_indices(
            target_tree,
            source_phys,
            k_target,
        )


        (
            _t2s_distance,
            t2s_index,
        ) = query_knn_indices(
            source_tree,
            target_phys,
            k_source,
        )


        # ====================================================
        # EXACT PAIR UNION
        # ====================================================

        pair_map = {}


        # ----------------------------------------------------
        # source -> K nearest targets
        # ----------------------------------------------------

        for source_local in range(
            len(
                sources
            )
        ):

            source_id = int(
                sources.iloc[
                    source_local
                ][
                    "node_id"
                ]
            )


            for rank in range(
                k_target
            ):

                target_local = int(
                    s2t_index[
                        source_local,
                        rank,
                    ]
                )


                target_id = int(
                    targets.iloc[
                        target_local
                    ][
                        "node_id"
                    ]
                )


                key = (
                    source_id,
                    target_id,
                )


                pair_map.setdefault(
                    key,
                    {
                        "s2t":
                            0,

                        "t2s":
                            0,
                    },
                )


                pair_map[
                    key
                ][
                    "s2t"
                ] = 1


        # ----------------------------------------------------
        # target -> K nearest sources
        # ----------------------------------------------------

        for target_local in range(
            len(
                targets
            )
        ):

            target_id = int(
                targets.iloc[
                    target_local
                ][
                    "node_id"
                ]
            )


            for rank in range(
                k_source
            ):

                source_local = int(
                    t2s_index[
                        target_local,
                        rank,
                    ]
                )


                source_id = int(
                    sources.iloc[
                        source_local
                    ][
                        "node_id"
                    ]
                )


                key = (
                    source_id,
                    target_id,
                )


                pair_map.setdefault(
                    key,
                    {
                        "s2t":
                            0,

                        "t2s":
                            0,
                    },
                )


                pair_map[
                    key
                ][
                    "t2s"
                ] = 1


        # ====================================================
        # MATERIALIZE UNION PAIRS
        # ====================================================

        for (
            source_id,
            target_id,
        ) in sorted(
            pair_map
        ):

            flags = pair_map[
                (
                    source_id,
                    target_id,
                )
            ]


            source = node_lookup.loc[
                source_id
            ]


            target = node_lookup.loc[
                target_id
            ]


            assert int(
                source[
                    "t"
                ]
            ) == (
                middle_t
                -
                1
            )


            assert int(
                target[
                    "t"
                ]
            ) == (
                middle_t
                +
                1
            )


            source_xyz = np.asarray(
                [
                    source[
                        "phys_z"
                    ],
                    source[
                        "phys_y"
                    ],
                    source[
                        "phys_x"
                    ],
                ],
                dtype=np.float64,
            )


            target_xyz = np.asarray(
                [
                    target[
                        "phys_z"
                    ],
                    target[
                        "phys_y"
                    ],
                    target[
                        "phys_x"
                    ],
                ],
                dtype=np.float64,
            )


            candidate_phys = (
                source_xyz
                +
                target_xyz
            ) / 2.0


            candidate_voxel = (
                candidate_phys
                /
                scale
            )


            pair_distance = float(
                np.linalg.norm(
                    target_xyz
                    -
                    source_xyz
                )
            )


            in_bounds = candidate_in_bounds(
                middle_t,
                candidate_voxel,
                metadata,
            )


            # ------------------------------------------------
            # Mathematical invariant:
            #
            # midpoint of two in-bound points in a rectangular
            # spatial domain is also in-bound.
            #
            # middle_t is already 1..T-2.
            # ------------------------------------------------

            assert in_bounds, (
                "\nUnexpected G2 midpoint OOB.\n"
                f"dataset={dataset}\n"
                f"t={middle_t}\n"
                f"source={source_id}\n"
                f"target={target_id}\n"
                f"candidate={candidate_voxel}"
            )


            collision = (
                coordinate_key(
                    dataset,
                    middle_t,
                    candidate_voxel,
                )
                in
                existing_node_keys
            )


            dataset_raw_rows.append(
                {
                    "raw_proposal_id":
                        int(
                            raw_counter
                        ),

                    "dataset":
                        str(
                            dataset
                        ),

                    "fold":
                        int(
                            fold
                        ),

                    "family":
                        G2_ID,

                    "candidate_t":
                        int(
                            middle_t
                        ),

                    "z":
                        float(
                            candidate_voxel[
                                0
                            ]
                        ),

                    "y":
                        float(
                            candidate_voxel[
                                1
                            ]
                        ),

                    "x":
                        float(
                            candidate_voxel[
                                2
                            ]
                        ),

                    "source_node_id":
                        int(
                            source_id
                        ),

                    "anchor_node_id":
                        np.nan,

                    "target_node_id":
                        int(
                            target_id
                        ),

                    "g2_source_to_target_knn":
                        int(
                            flags[
                                "s2t"
                            ]
                        ),

                    "g2_target_to_source_knn":
                        int(
                            flags[
                                "t2s"
                            ]
                        ),

                    "g2_pair_distance_um":
                        float(
                            pair_distance
                        ),

                    "in_bounds":
                        True,

                    "existing_node_collision":
                        bool(
                            collision
                        ),

                    "accepted":
                        bool(
                            not collision
                        ),
                }
            )


            raw_counter += 1


    # ========================================================
    # DATASET-LEVEL RAW TABLE
    # ========================================================

    dataset_raw = pd.DataFrame(
        dataset_raw_rows
    )


    assert len(
        dataset_raw
    ) > 0, dataset


    assert dataset_raw[
        "raw_proposal_id"
    ].is_unique


    raw_parts.append(
        dataset_raw
    )


    family_counts = (
        dataset_raw[
            "family"
        ]
        .value_counts()
        .to_dict()
    )


    dataset_summary_rows.append(
        {
            "dataset":
                str(
                    dataset
                ),

            "fold":
                int(
                    fold
                ),

            "dataset_stem":
                metadata[
                    "stem"
                ],

            "zarr_path":
                metadata[
                    "zarr_path"
                ],

            "T":
                int(
                    metadata[
                        "T"
                    ]
                ),

            "Z":
                int(
                    metadata[
                        "Z"
                    ]
                ),

            "Y":
                int(
                    metadata[
                        "Y"
                    ]
                ),

            "X":
                int(
                    metadata[
                        "X"
                    ]
                ),

            "scale_z":
                float(
                    scale[
                        0
                    ]
                ),

            "scale_y":
                float(
                    scale[
                        1
                    ]
                ),

            "scale_x":
                float(
                    scale[
                        2
                    ]
                ),

            "image_none":
                True,

            "tracks_none":
                True,

            "canonical_nodes":
                int(
                    len(
                        nodes
                    )
                ),

            "canonical_edges":
                int(
                    graph.num_edges()
                ),

            "g3h3_actions":
                int(
                    replay_stats[
                        "actions"
                    ]
                ),

            "g3h3_source_removals":
                int(
                    replay_stats[
                        "source_incumbent_edges_removed"
                    ]
                ),

            "g3h3_edges_added":
                int(
                    replay_stats[
                        "edges_added"
                    ]
                ),

            "g3h3_net_edge_delta":
                int(
                    replay_stats[
                        "net_edge_delta"
                    ]
                ),

            "raw_g0":
                int(
                    family_counts.get(
                        G0_ID,
                        0,
                    )
                ),

            "raw_g1":
                int(
                    family_counts.get(
                        G1_ID,
                        0,
                    )
                ),

            "raw_g2":
                int(
                    family_counts.get(
                        G2_ID,
                        0,
                    )
                ),

            "raw_total":
                int(
                    len(
                        dataset_raw
                    )
                ),

            "raw_out_of_bounds":
                int(
                    (
                        ~dataset_raw[
                            "in_bounds"
                        ]
                    ).sum()
                ),

            "raw_existing_node_collisions":
                int(
                    dataset_raw[
                        "existing_node_collision"
                    ].sum()
                ),

            "raw_accepted":
                int(
                    dataset_raw[
                        "accepted"
                    ].sum()
                ),
        }
    )


    if (
        dataset_index
        ==
        1
        or
        dataset_index
        %
        10
        ==
        0
        or
        dataset_index
        ==
        len(
            datasets
        )
    ):

        print(
            f"{dataset_index:3d}/{len(datasets)}"
            f" | {dataset}"
            f" | shape={metadata['image_shape']}"
            f" | nodes={len(nodes):6d}"
            f" | edges={graph.num_edges():6d}"
            f" | G0={family_counts.get(G0_ID,0):6d}"
            f" | G1={family_counts.get(G1_ID,0):6d}"
            f" | G2={family_counts.get(G2_ID,0):7d}"
            f" | accepted={int(dataset_raw['accepted'].sum()):7d}"
        )


    del graph
    del nodes
    del edges
    del node_lookup
    del dataset_raw_rows
    del dataset_raw


    gc.collect()


# ============================================================
# 16. POST-G3H3 DEVELOPMENT FIDELITY
# ============================================================

print(
    "\n========== POST-G3H3 DEVELOPMENT FIDELITY =========="
)


print(
    "actions:",
    total_replay_actions,
    "expected:",
    EXPECTED_DEV_ACTIONS
)


print(
    "source removals:",
    total_replay_removals,
    "expected:",
    EXPECTED_DEV_REMOVALS
)


print(
    "edges added:",
    total_replay_additions,
    "expected:",
    EXPECTED_DEV_ADDITIONS
)


print(
    "net edge delta:",
    total_replay_net,
    "expected:",
    EXPECTED_DEV_NET
)


assert (
    total_replay_actions
    ==
    EXPECTED_DEV_ACTIONS
)


assert (
    total_replay_removals
    ==
    EXPECTED_DEV_REMOVALS
)


assert (
    total_replay_additions
    ==
    EXPECTED_DEV_ADDITIONS
)


assert (
    total_replay_net
    ==
    EXPECTED_DEV_NET
)


print(
    "POST_G3H3_DEVELOPMENT_GRAPH_FIDELITY: PASS"
)


# ============================================================
# 17. COMBINE RAW PROPOSALS
# ============================================================

assert raw_parts


raw = pd.concat(
    raw_parts,
    ignore_index=True,
)


del raw_parts


gc.collect()


assert len(
    raw
) > 0


assert raw[
    "raw_proposal_id"
].is_unique


assert np.array_equal(
    raw[
        "raw_proposal_id"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        len(
            raw
        ),
        dtype=np.int64,
    ),
)


assert set(
    raw[
        "family"
    ].unique()
) == {
    G0_ID,
    G1_ID,
    G2_ID,
}


# ------------------------------------------------------------
# G2 midpoint must have zero OOB.
# ------------------------------------------------------------

g2_oob_count = int(
    (
        ~raw.loc[
            raw[
                "family"
            ].eq(
                G2_ID
            ),
            "in_bounds",
        ]
    ).sum()
)


assert g2_oob_count == 0


# ============================================================
# 18. ALL-103 OFFICIAL METADATA CONSISTENCY
# ============================================================

dataset_summary = pd.DataFrame(
    dataset_summary_rows
)


assert len(
    dataset_summary
) == EXPECTED_DATASETS


assert dataset_summary[
    "dataset"
].is_unique


assert dataset_summary[
    [
        "T",
        "Z",
        "Y",
        "X",
    ]
].drop_duplicates().shape[
    0
] == 1


actual_shape = tuple(
    dataset_summary[
        [
            "T",
            "Z",
            "Y",
            "X",
        ]
    ]
    .iloc[
        0
    ]
    .astype(
        int
    )
    .tolist()
)


assert (
    actual_shape
    ==
    EXPECTED_IMAGE_SHAPE
)


assert dataset_summary[
    [
        "scale_z",
        "scale_y",
        "scale_x",
    ]
].drop_duplicates().shape[
    0
] == 1


actual_scale = dataset_summary[
    [
        "scale_z",
        "scale_y",
        "scale_x",
    ]
].iloc[
    0
].to_numpy(
    dtype=np.float64
)


assert np.array_equal(
    actual_scale,
    np.asarray(
        EXPECTED_SCALE,
        dtype=np.float64,
    ),
)


assert dataset_summary[
    "image_none"
].all()


assert dataset_summary[
    "tracks_none"
].all()


assert dataset_summary[
    "zarr_path"
].map(
    lambda value:
        Path(
            value
        ).exists()
).all()


print(
    "\n========== ALL-103 METADATA CONSISTENCY =========="
)


print(
    "datasets:",
    len(
        dataset_summary
    )
)


print(
    "shape:",
    actual_shape
)


print(
    "axis order:",
    EXPECTED_AXIS_ORDER
)


print(
    "scale:",
    tuple(
        actual_scale.tolist()
    )
)


print(
    "image is None:",
    "103/103"
)


print(
    "tracks is None:",
    "103/103"
)


print(
    "ALL_103_METADATA_SCHEMA_CONSISTENCY: PASS"
)


# ============================================================
# 19. ADD EXACT ROUNDED CANDIDATE COORDINATES
# ============================================================

raw[
    "key_z"
] = np.round(
    pd.to_numeric(
        raw[
            "z"
        ],
        errors="raise",
    ).to_numpy(
        dtype=np.float64
    ),
    ROUND_DECIMALS,
)


raw[
    "key_y"
] = np.round(
    pd.to_numeric(
        raw[
            "y"
        ],
        errors="raise",
    ).to_numpy(
        dtype=np.float64
    ),
    ROUND_DECIMALS,
)


raw[
    "key_x"
] = np.round(
    pd.to_numeric(
        raw[
            "x"
        ],
        errors="raise",
    ).to_numpy(
        dtype=np.float64
    ),
    ROUND_DECIMALS,
)


assert np.isfinite(
    raw[
        [
            "key_z",
            "key_y",
            "key_x",
        ]
    ].to_numpy(
        dtype=np.float64
    )
).all()


# ============================================================
# 20. GT-BLIND RAW SAFETY
# ============================================================

for column in raw.columns:

    lower = str(
        column
    ).lower()


    assert "oracle" not in lower


    assert not lower.startswith(
        "gt_"
    )


    assert "missing_node" not in lower


    assert "official_fp" not in lower


assert (
    "detector_prob"
    not in
    raw.columns
)


assert (
    "detector_logit"
    not in
    raw.columns
)


accepted_raw = (
    raw.loc[
        raw[
            "accepted"
        ]
    ]
    .copy()
)


assert len(
    accepted_raw
) > 0


assert accepted_raw[
    "in_bounds"
].all()


assert not accepted_raw[
    "existing_node_collision"
].any()


# ============================================================
# 21. CANDIDATE-LEVEL DEDUPLICATION
#
# Exact A0 identity:
#
#   dataset
#   candidate_t
#   round(z,12)
#   round(y,12)
#   round(x,12)
#
# Fold is provenance only.
# ============================================================

IDENTITY_KEY = [
    "dataset",
    "candidate_t",
    "key_z",
    "key_y",
    "key_x",
]


base = (
    accepted_raw
    .groupby(
        IDENTITY_KEY,
        sort=True,
        dropna=False,
    )
    .agg(
        proposal_count=(
            "raw_proposal_id",
            "size",
        ),

        fold=(
            "fold",
            "first",
        ),

        fold_nunique=(
            "fold",
            "nunique",
        ),
    )
    .reset_index()
)


assert (
    base[
        "fold_nunique"
    ]
    ==
    1
).all()


base = base.drop(
    columns=[
        "fold_nunique",
    ]
)


base[
    "fold"
] = pd.to_numeric(
    base[
        "fold"
    ],
    errors="raise",
).astype(
    np.int8
)


# ============================================================
# 22. FAMILY COUNTS
# ============================================================

family_count_table = (
    accepted_raw
    .pivot_table(
        index=IDENTITY_KEY,
        columns="family",
        values="raw_proposal_id",
        aggfunc="count",
        fill_value=0,
    )
    .reset_index()
)


for family in [
    G0_ID,
    G1_ID,
    G2_ID,
]:

    if (
        family
        not in
        family_count_table.columns
    ):

        family_count_table[
            family
        ] = 0


family_count_table = (
    family_count_table.rename(
        columns={
            G0_ID:
                "g0_count",

            G1_ID:
                "g1_count",

            G2_ID:
                "g2_count",
        }
    )
)


universe = base.merge(
    family_count_table[
        IDENTITY_KEY
        +
        [
            "g0_count",
            "g1_count",
            "g2_count",
        ]
    ],
    on=IDENTITY_KEY,
    how="left",
    validate="one_to_one",
)


# ============================================================
# 23. G2 DIRECTION MEMBERSHIP
# ============================================================

g2_direction = (
    accepted_raw
    .groupby(
        IDENTITY_KEY,
        sort=True,
        dropna=False,
    )
    .agg(
        g2_s2t_count=(
            "g2_source_to_target_knn",
            "sum",
        ),

        g2_t2s_count=(
            "g2_target_to_source_knn",
            "sum",
        ),
    )
    .reset_index()
)


universe = universe.merge(
    g2_direction,
    on=IDENTITY_KEY,
    how="left",
    validate="one_to_one",
)


for column in [
    "proposal_count",
    "g0_count",
    "g1_count",
    "g2_count",
    "g2_s2t_count",
    "g2_t2s_count",
]:

    universe[
        column
    ] = (
        pd.to_numeric(
            universe[
                column
            ],
            errors="raise",
        )
        .fillna(
            0
        )
        .astype(
            np.int64
        )
    )


# ============================================================
# 24. FAMILY MEMBERSHIP
# ============================================================

def family_membership(
    row,
):

    result = []


    if int(
        row.g0_count
    ) > 0:

        result.append(
            G0_ID
        )


    if int(
        row.g1_count
    ) > 0:

        result.append(
            G1_ID
        )


    if int(
        row.g2_count
    ) > 0:

        result.append(
            G2_ID
        )


    return "|".join(
        result
    )


universe[
    "families"
] = [
    family_membership(
        row
    )

    for row
    in universe.itertuples(
        index=False
    )
]


# ============================================================
# 25. CANONICAL STORED CANDIDATE COORDINATE
#
# Exact rounded 12dp identity coordinate.
# ============================================================

universe[
    "t"
] = universe[
    "candidate_t"
].astype(
    np.int64
)


universe[
    "z"
] = universe[
    "key_z"
].astype(
    np.float64
)


universe[
    "y"
] = universe[
    "key_y"
].astype(
    np.float64
)


universe[
    "x"
] = universe[
    "key_x"
].astype(
    np.float64
)


universe = (
    universe
    .sort_values(
        [
            "dataset",
            "t",
            "z",
            "y",
            "x",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


universe[
    "candidate_id"
] = np.arange(
    len(
        universe
    ),
    dtype=np.int64,
)


UNIVERSE_COLUMNS = [
    "candidate_id",

    "dataset",

    "fold",

    "t",

    "z",
    "y",
    "x",

    "proposal_count",

    "families",

    "g0_count",
    "g1_count",
    "g2_count",

    "g2_s2t_count",
    "g2_t2s_count",
]


universe = universe[
    UNIVERSE_COLUMNS
].copy()


# ============================================================
# 26. UNIVERSE INVARIANTS
# ============================================================

assert len(
    universe
) > 0


assert universe[
    "candidate_id"
].is_unique


assert np.array_equal(
    universe[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        len(
            universe
        ),
        dtype=np.int64,
    ),
)


assert not universe[
    [
        "dataset",
        "t",
        "z",
        "y",
        "x",
    ]
].duplicated().any()


assert (
    universe[
        [
            "g0_count",
            "g1_count",
            "g2_count",
        ]
    ].sum(
        axis=1
    )
    ==
    universe[
        "proposal_count"
    ]
).all()


assert (
    universe[
        "proposal_count"
    ]
    >=
    1
).all()


assert set(
    universe[
        "fold"
    ]
    .astype(
        int
    )
    .unique()
).issubset(
    {
        1,
        2,
        3,
        4,
    }
)


# ============================================================
# 27. DATASET SUMMARY UNIQUE COUNTS
# ============================================================

unique_by_dataset = (
    universe
    .groupby(
        "dataset"
    )
    .size()
    .rename(
        "unique_candidates"
    )
    .reset_index()
)


dataset_summary = dataset_summary.merge(
    unique_by_dataset,
    on="dataset",
    how="left",
    validate="one_to_one",
)


dataset_summary[
    "unique_candidates"
] = (
    dataset_summary[
        "unique_candidates"
    ]
    .fillna(
        0
    )
    .astype(
        np.int64
    )
)


# ============================================================
# 28. FAMILY / OVERLAP COUNTS
# ============================================================

RAW_FAMILY_COUNTS = (
    raw[
        "family"
    ]
    .value_counts()
    .to_dict()
)


ACCEPTED_FAMILY_COUNTS = (
    accepted_raw[
        "family"
    ]
    .value_counts()
    .to_dict()
)


candidate_family_membership = {
    "G0_only":
        int(
            (
                universe[
                    "g0_count"
                ].gt(
                    0
                )
                &
                universe[
                    "g1_count"
                ].eq(
                    0
                )
                &
                universe[
                    "g2_count"
                ].eq(
                    0
                )
            ).sum()
        ),

    "G1_only":
        int(
            (
                universe[
                    "g0_count"
                ].eq(
                    0
                )
                &
                universe[
                    "g1_count"
                ].gt(
                    0
                )
                &
                universe[
                    "g2_count"
                ].eq(
                    0
                )
            ).sum()
        ),

    "G2_only":
        int(
            (
                universe[
                    "g0_count"
                ].eq(
                    0
                )
                &
                universe[
                    "g1_count"
                ].eq(
                    0
                )
                &
                universe[
                    "g2_count"
                ].gt(
                    0
                )
            ).sum()
        ),

    "multi_family":
        int(
            (
                universe[
                    [
                        "g0_count",
                        "g1_count",
                        "g2_count",
                    ]
                ]
                .gt(
                    0
                )
                .sum(
                    axis=1
                )
                >
                1
            ).sum()
        ),
}


print(
    "\n========== CANDIDATE UNIVERSE =========="
)


print(
    "raw proposals:",
    len(
        raw
    )
)


print(
    "accepted raw proposals:",
    len(
        accepted_raw
    )
)


print(
    "out-of-bounds:",
    int(
        (
            ~raw[
                "in_bounds"
            ]
        ).sum()
    )
)


print(
    "G2 out-of-bounds:",
    g2_oob_count
)


print(
    "existing-node collisions:",
    int(
        raw[
            "existing_node_collision"
        ].sum()
    )
)


print(
    "unique candidates:",
    len(
        universe
    )
)


print(
    "raw family counts:",
    RAW_FAMILY_COUNTS
)


print(
    "accepted family counts:",
    ACCEPTED_FAMILY_COUNTS
)


print(
    "candidate family membership:",
    candidate_family_membership
)


# ============================================================
# 29. FINAL GT-BLIND SAFETY GATE
# ============================================================

for frame in [
    raw,
    universe,
]:

    for column in frame.columns:

        lower = str(
            column
        ).lower()


        assert "oracle" not in lower


        assert not lower.startswith(
            "gt_"
        )


        assert "missing_node" not in lower


        assert "official_fp" not in lower


assert (
    "detector_prob"
    not in
    raw.columns
)


assert (
    "detector_logit"
    not in
    raw.columns
)


assert (
    "detector_prob"
    not in
    universe.columns
)


assert (
    "detector_logit"
    not in
    universe.columns
)


print(
    "\n========== 12.13A1-R3 SAFETY =========="
)


print(
    "candidate universe frozen before GT:",
    "PENDING FORMAL WRITE"
)


print(
    "GT read:",
    "NO"
)


print(
    "A3 read:",
    "NO"
)


print(
    "tracks read:",
    "NO"
)


print(
    "image voxels/chunks read:",
    "NO"
)


print(
    "missing-node oracle:",
    "NO"
)


print(
    "detector peaks:",
    "NO"
)


print(
    "detector probability:",
    "NO"
)


print(
    "model trained:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "graph metric:",
    "NO"
)


print(
    "persistent graph mutation:",
    "NO"
)


print(
    "Fold0:",
    "NO"
)


# ============================================================
# 30. CONTENT HASHES
# ============================================================

RAW_HASH_COLUMNS = [
    "raw_proposal_id",

    "dataset",

    "fold",

    "family",

    "candidate_t",

    "z",
    "y",
    "x",

    "source_node_id",

    "anchor_node_id",

    "target_node_id",

    "g2_source_to_target_knn",

    "g2_target_to_source_knn",

    "g2_pair_distance_um",

    "in_bounds",

    "existing_node_collision",

    "accepted",

    "key_z",
    "key_y",
    "key_x",
]


RAW_CONTENT_SHA = dataframe_content_sha256(
    raw,
    RAW_HASH_COLUMNS,
)


UNIVERSE_CONTENT_SHA = dataframe_content_sha256(
    universe,
    UNIVERSE_COLUMNS,
)


print(
    "\n========== CONTENT HASH =========="
)


print(
    "raw semantic SHA:",
    RAW_CONTENT_SHA
)


print(
    "universe semantic SHA:",
    UNIVERSE_CONTENT_SHA
)


# ============================================================
# 31. FORMAL SCHEMA
# ============================================================

SCHEMA = {
    "stage":
        "12.13A1-R3",

    "status":
        "GT_BLIND_TRACK_GEOMETRY_CANDIDATE_UNIVERSE_FROZEN",

    "branch_id":
        BRANCH_ID,

    "candidate_universe_id":
        CANDIDATE_UNIVERSE_ID,

    "parent_contract_sha256":
        A0_SHA,

    "supersedes_failed_attempts": [
        "12.13A1",
        "12.13A1-R1",
        "12.13A1-R2",
    ],

    "repairs": {
        "A1":
            (
                "FUZZY_MUTATION_HELPER_LOCATOR_"
                "REPLACED_BY_HARD_LOCKED_AST_CELL_66"
            ),

        "A1_R1":
            (
                "DATASET_SHAPE_RESOLVED_FROM_OFFICIAL_"
                "DATASET_IMAGE_SHAPE_NOT_NONEXISTENT_SHAPE"
            ),

        "A1_R2":
            (
                "TRACKSDATA_EDGE_SCHEMA_CORRECTED_FROM_"
                "SOURCE_TARGET_TO_EDGE_ID_SOURCE_ID_TARGET_ID"
            ),

        "dataset_path_semantics":
            (
                "TRAIN_ROOT_DATASET_IS_EXTENSIONLESS_STEM_"
                "OPEN_DATASET_RESOLVES_DOT_ZARR"
            ),
    },

    "canonical_graph": {
        "development_builder_cell":
            EXPECTED_DEV_BUILDER_CELL,

        "builder_cell_sha256":
            BUILDER_CELL_SHA,

        "builder_prefix_sha256":
            BUILDER_PREFIX_SHA,

        "tracksdata_edge_schema": [
            "edge_id",
            "source_id",
            "target_id",
        ],

        "development_prediction_p995":
            EXPECTED_DEV_P995_TOKEN,

        "development_prediction_p9975":
            EXPECTED_DEV_P9975_TOKEN,

        "frozen_g3h3_actions_artifact":
            DEV_ACTIONS_PATH.name,

        "frozen_g3h3_actions_sha256":
            DEV_ACTIONS_SHA,

        "mutation_helper_cell":
            EXPECTED_MUTATION_CELL,

        "mutation_helper_sha256":
            MUTATION_SOURCE_SHA,

        "g3h3_actions":
            int(
                total_replay_actions
            ),

        "g3h3_source_removals":
            int(
                total_replay_removals
            ),

        "g3h3_edges_added":
            int(
                total_replay_additions
            ),

        "g3h3_net_edge_delta":
            int(
                total_replay_net
            ),
    },

    "metadata_contract": {
        "api":
            "tracking_cellmot.io.open_dataset",

        "argument":
            "TRAIN_ROOT / DATASET_EXTENSIONLESS_STEM",

        "load_image":
            False,

        "require_tracks":
            False,

        "shape_field":
            "Dataset.image_shape",

        "axis_order":
            list(
                EXPECTED_AXIS_ORDER
            ),

        "shape":
            list(
                actual_shape
            ),

        "scale_field":
            "Dataset.scale",

        "scale_zyx_um":
            [
                float(
                    value
                )

                for value
                in actual_scale
            ],

        "physical_container_field":
            "Dataset.zarr_path",

        "datasets_verified":
            EXPECTED_DATASETS,

        "image_voxels_read":
            False,

        "tracks_read":
            False,
    },

    "boundary_contract": {
        "spatial":
            (
                "0_LE_Z_LT_SHAPE_Z_AND_"
                "0_LE_Y_LT_SHAPE_Y_AND_"
                "0_LE_X_LT_SHAPE_X"
            ),

        "temporal":
            "0_LE_T_LT_IMAGE_SHAPE_T",

        "temporal_rule_role":
            (
                "IMPLEMENTATION_CLARIFICATION_OF_"
                "G0_G1_OUT_OF_DATASET_BOUNDS_REJECT"
            ),

        "G2_midpoint_out_of_bounds":
            int(
                g2_oob_count
            ),
    },

    "proposal_families": {
        "G0":
            G0_ID,

        "G1":
            G1_ID,

        "G2":
            G2_ID,

        "G2_K":
            G2_K,

        "G2_distance_cutoff":
            None,
    },

    "candidate_identity": {
        "key": [
            "dataset",
            "t",
            "round(z,12)",
            "round(y,12)",
            "round(x,12)",
        ],

        "round_decimals":
            ROUND_DECIMALS,

        "fold_in_identity_key":
            False,

        "fold_role":
            "PROVENANCE_ONLY",

        "stored_coordinate":
            "ROUNDED_IDENTITY_COORDINATE_12DP",

        "raw_proposals_preserved":
            True,

        "multi_family_membership_preserved":
            True,

        "exact_existing_node_collision_excluded":
            True,

        "existing_node_distance_cutoff":
            None,
    },

    "counts": {
        "raw_proposals":
            int(
                len(
                    raw
                )
            ),

        "accepted_raw_proposals":
            int(
                len(
                    accepted_raw
                )
            ),

        "unique_candidates":
            int(
                len(
                    universe
                )
            ),

        "out_of_bounds":
            int(
                (
                    ~raw[
                        "in_bounds"
                    ]
                ).sum()
            ),

        "g2_out_of_bounds":
            int(
                g2_oob_count
            ),

        "existing_node_collisions":
            int(
                raw[
                    "existing_node_collision"
                ].sum()
            ),

        "raw_family_counts":
            {
                str(
                    key
                ):
                    int(
                        value
                    )

                for key, value
                in RAW_FAMILY_COUNTS.items()
            },

        "accepted_family_counts":
            {
                str(
                    key
                ):
                    int(
                        value
                    )

                for key, value
                in ACCEPTED_FAMILY_COUNTS.items()
            },

        "candidate_family_membership":
            candidate_family_membership,
    },

    "content": {
        "raw_semantic_sha256":
            RAW_CONTENT_SHA,

        "universe_semantic_sha256":
            UNIVERSE_CONTENT_SHA,
    },

    "protocol": {
        "candidate_universe_generated":
            True,

        "candidate_universe_frozen_before_GT":
            True,

        "GT_read":
            False,

        "A3_read":
            False,

        "tracks_read":
            False,

        "image_voxels_read":
            False,

        "missing_node_oracle_read":
            False,

        "detector_peak_required":
            False,

        "detector_probability_required":
            False,

        "model_trained":
            False,

        "scorer_selected":
            False,

        "threshold_selected":
            False,

        "graph_metric_evaluated":
            False,

        "persistent_graph_modified":
            False,

        "Fold0_used":
            False,
    },

    "next":
        (
            "12.13A2_DEV_GT_ORACLE_CEILING_AUDIT_"
            "OF_FROZEN_TRACK_GEOMETRY_CANDIDATE_UNIVERSE"
        ),
}


SCHEMA_SHA = hashlib.sha256(
    json.dumps(
        SCHEMA,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


SCHEMA[
    "schema_sha256"
] = SCHEMA_SHA


# ============================================================
# 32. TEMPORARY ARTIFACT PATHS
# ============================================================

TMP_RAW = RAW_OUT.with_name(
    RAW_OUT.name
    +
    ".tmp_12_13a1_r3"
)


TMP_UNIVERSE = UNIVERSE_OUT.with_name(
    UNIVERSE_OUT.name
    +
    ".tmp_12_13a1_r3"
)


TMP_DATASET = DATASET_SUMMARY_OUT.with_name(
    DATASET_SUMMARY_OUT.name
    +
    ".tmp_12_13a1_r3"
)


TMP_SCHEMA = SCHEMA_OUT.with_name(
    SCHEMA_OUT.name
    +
    ".tmp_12_13a1_r3"
)


TMP_SUMMARY = SUMMARY_OUT.with_name(
    SUMMARY_OUT.name
    +
    ".tmp_12_13a1_r3"
)


for path in [
    TMP_RAW,
    TMP_UNIVERSE,
    TMP_DATASET,
    TMP_SCHEMA,
    TMP_SUMMARY,
]:

    if path.exists():

        raise RuntimeError(
            "\nStale temporary 12.13A1-R3 artifact exists:\n"
            f"{path}\n\n"
            "Do not overwrite it blindly."
        )


# ============================================================
# 33. TEMP WRITE
# ============================================================

raw.to_pickle(
    TMP_RAW
)


universe.to_pickle(
    TMP_UNIVERSE
)


dataset_summary.to_pickle(
    TMP_DATASET
)


TMP_SCHEMA.write_text(
    json.dumps(
        SCHEMA,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 34. TEMP RELOAD FIDELITY
# ============================================================

raw_check = pd.read_pickle(
    TMP_RAW
)


universe_check = pd.read_pickle(
    TMP_UNIVERSE
)


dataset_check = pd.read_pickle(
    TMP_DATASET
)


schema_check = json.loads(
    TMP_SCHEMA.read_text(
        encoding="utf-8"
    )
)


pd.testing.assert_frame_equal(
    raw_check,
    raw,
    check_exact=True,
)


pd.testing.assert_frame_equal(
    universe_check,
    universe,
    check_exact=True,
)


pd.testing.assert_frame_equal(
    dataset_check,
    dataset_summary,
    check_exact=True,
)


assert schema_check[
    "schema_sha256"
] == SCHEMA_SHA


assert dataframe_content_sha256(
    raw_check,
    RAW_HASH_COLUMNS,
) == RAW_CONTENT_SHA


assert dataframe_content_sha256(
    universe_check,
    UNIVERSE_COLUMNS,
) == UNIVERSE_CONTENT_SHA


RAW_FILE_SHA = sha256_file(
    TMP_RAW
)


UNIVERSE_FILE_SHA = sha256_file(
    TMP_UNIVERSE
)


DATASET_FILE_SHA = sha256_file(
    TMP_DATASET
)


SCHEMA_FILE_SHA = sha256_file(
    TMP_SCHEMA
)


# ============================================================
# 35. FORMAL SUMMARY
# ============================================================

SUMMARY = {
    "stage":
        "12.13A1-R3",

    "status":
        "GT_BLIND_TRACK_GEOMETRY_CANDIDATE_UNIVERSE_FROZEN",

    "branch_id":
        BRANCH_ID,

    "candidate_universe_id":
        CANDIDATE_UNIVERSE_ID,

    "parent_contract_sha256":
        A0_SHA,

    "supersedes_failed_attempts": [
        "12.13A1",
        "12.13A1-R1",
        "12.13A1-R2",
    ],

    "repairs": {
        "A1":
            "MUTATION_HELPER_LOCATOR",

        "A1_R1":
            "OFFICIAL_DATASET_IMAGE_SHAPE",

        "A1_R2":
            "CANONICAL_TRACKSDATA_EDGE_SCHEMA",
    },

    "canonical_graph_fidelity": {
        "tracksdata_edge_schema": [
            "edge_id",
            "source_id",
            "target_id",
        ],

        "actions":
            int(
                total_replay_actions
            ),

        "source_removals":
            int(
                total_replay_removals
            ),

        "edges_added":
            int(
                total_replay_additions
            ),

        "net_edge_delta":
            int(
                total_replay_net
            ),

        "pass":
            True,
    },

    "metadata": {
        "datasets":
            EXPECTED_DATASETS,

        "shape":
            list(
                actual_shape
            ),

        "axes":
            list(
                EXPECTED_AXIS_ORDER
            ),

        "scale_zyx_um":
            [
                float(
                    value
                )

                for value
                in actual_scale
            ],

        "all_consistent":
            True,
    },

    "raw": {
        "rows":
            int(
                len(
                    raw
                )
            ),

        "semantic_sha256":
            RAW_CONTENT_SHA,

        "artifact_file_sha256":
            RAW_FILE_SHA,
    },

    "universe": {
        "rows":
            int(
                len(
                    universe
                )
            ),

        "datasets":
            int(
                universe[
                    "dataset"
                ].nunique()
            ),

        "fold_counts":
            {
                str(
                    int(
                        fold
                    )
                ):
                    int(
                        count
                    )

                for fold, count
                in (
                    universe[
                        "fold"
                    ]
                    .value_counts()
                    .sort_index()
                    .items()
                )
            },

        "semantic_sha256":
            UNIVERSE_CONTENT_SHA,

        "artifact_file_sha256":
            UNIVERSE_FILE_SHA,
    },

    "proposal_counts": {
        "raw_family":
            {
                str(
                    key
                ):
                    int(
                        value
                    )

                for key, value
                in RAW_FAMILY_COUNTS.items()
            },

        "accepted_family":
            {
                str(
                    key
                ):
                    int(
                        value
                    )

                for key, value
                in ACCEPTED_FAMILY_COUNTS.items()
            },

        "candidate_membership":
            candidate_family_membership,

        "out_of_bounds":
            int(
                (
                    ~raw[
                        "in_bounds"
                    ]
                ).sum()
            ),

        "g2_out_of_bounds":
            int(
                g2_oob_count
            ),

        "existing_node_collisions":
            int(
                raw[
                    "existing_node_collision"
                ].sum()
            ),
    },

    "dataset_summary": {
        "rows":
            int(
                len(
                    dataset_summary
                )
            ),

        "artifact_file_sha256":
            DATASET_FILE_SHA,
    },

    "schema": {
        "schema_sha256":
            SCHEMA_SHA,

        "artifact_file_sha256":
            SCHEMA_FILE_SHA,
    },

    "protocol": {
        "candidate_universe_frozen_before_GT":
            True,

        "GT_read":
            False,

        "A3_read":
            False,

        "tracks_read":
            False,

        "image_voxels_read":
            False,

        "missing_node_oracle_read":
            False,

        "detector_peak_required":
            False,

        "detector_probability_required":
            False,

        "model_trained":
            False,

        "threshold_selected":
            False,

        "graph_metric_evaluated":
            False,

        "persistent_graph_modified":
            False,

        "Fold0_used":
            False,
    },

    "next":
        (
            "12.13A2_DEV_GT_ORACLE_CEILING_AUDIT_"
            "OF_FROZEN_TRACK_GEOMETRY_CANDIDATE_UNIVERSE"
        ),
}


SUMMARY_SHA = hashlib.sha256(
    json.dumps(
        SUMMARY,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


SUMMARY[
    "summary_sha256"
] = SUMMARY_SHA


TMP_SUMMARY.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


summary_check = json.loads(
    TMP_SUMMARY.read_text(
        encoding="utf-8"
    )
)


assert summary_check[
    "summary_sha256"
] == SUMMARY_SHA


assert summary_check[
    "protocol"
][
    "candidate_universe_frozen_before_GT"
] is True


assert summary_check[
    "protocol"
][
    "GT_read"
] is False


assert summary_check[
    "protocol"
][
    "A3_read"
] is False


assert summary_check[
    "protocol"
][
    "Fold0_used"
] is False


SUMMARY_FILE_SHA = sha256_file(
    TMP_SUMMARY
)


# ============================================================
# 36. ATOMIC PROMOTION
#
# Sequential atomic renames.
#
# If this section crashes partway through:
#
#   DO NOT delete promoted formal artifacts.
#   DO NOT rerun blindly.
#
# Use a repair/resume fidelity cell.
# ============================================================

TMP_RAW.replace(
    RAW_OUT
)


TMP_UNIVERSE.replace(
    UNIVERSE_OUT
)


TMP_DATASET.replace(
    DATASET_SUMMARY_OUT
)


TMP_SCHEMA.replace(
    SCHEMA_OUT
)


TMP_SUMMARY.replace(
    SUMMARY_OUT
)


# ============================================================
# 37. FINAL FORMAL RELOAD
# ============================================================

assert sha256_file(
    RAW_OUT
) == RAW_FILE_SHA


assert sha256_file(
    UNIVERSE_OUT
) == UNIVERSE_FILE_SHA


assert sha256_file(
    DATASET_SUMMARY_OUT
) == DATASET_FILE_SHA


assert sha256_file(
    SCHEMA_OUT
) == SCHEMA_FILE_SHA


assert sha256_file(
    SUMMARY_OUT
) == SUMMARY_FILE_SHA


formal_raw = pd.read_pickle(
    RAW_OUT
)


formal_universe = pd.read_pickle(
    UNIVERSE_OUT
)


formal_schema = json.loads(
    SCHEMA_OUT.read_text(
        encoding="utf-8"
    )
)


formal_summary = json.loads(
    SUMMARY_OUT.read_text(
        encoding="utf-8"
    )
)


assert dataframe_content_sha256(
    formal_raw,
    RAW_HASH_COLUMNS,
) == RAW_CONTENT_SHA


assert dataframe_content_sha256(
    formal_universe,
    UNIVERSE_COLUMNS,
) == UNIVERSE_CONTENT_SHA


assert formal_schema[
    "schema_sha256"
] == SCHEMA_SHA


assert formal_summary[
    "summary_sha256"
] == SUMMARY_SHA


assert formal_schema[
    "canonical_graph"
][
    "tracksdata_edge_schema"
] == [
    "edge_id",
    "source_id",
    "target_id",
]


assert formal_schema[
    "candidate_identity"
][
    "fold_in_identity_key"
] is False


assert formal_schema[
    "protocol"
][
    "candidate_universe_frozen_before_GT"
] is True


assert formal_schema[
    "protocol"
][
    "GT_read"
] is False


assert formal_schema[
    "protocol"
][
    "A3_read"
] is False


assert formal_schema[
    "protocol"
][
    "Fold0_used"
] is False


# ============================================================
# 38. FINAL
# ============================================================

print(
    "\n========== 12.13A1-R3 FINAL FREEZE =========="
)


print(
    "GT_BLIND_TRACK_GEOMETRY_CANDIDATE_UNIVERSE_FREEZE: PASS"
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "candidate universe:",
    CANDIDATE_UNIVERSE_ID
)


print(
    "\n--- REPAIR PROVENANCE ---"
)


print(
    "A1:",
    "FUZZY MUTATION LOCATOR -> SUPERSEDED"
)


print(
    "A1-R1:",
    "Dataset.shape -> Dataset.image_shape"
)


print(
    "A1-R2:",
    "source/target -> source_id/target_id"
)


print(
    "mutation helper:",
    f"CELL {EXPECTED_MUTATION_CELL}"
)


print(
    "mutation helper SHA:",
    MUTATION_SOURCE_SHA
)


print(
    "\n--- TRACKSDATA SCHEMA ---"
)


print(
    "edge_id:",
    "YES"
)


print(
    "source_id:",
    "YES"
)


print(
    "target_id:",
    "YES"
)


print(
    "TRACKSDATA_CANONICAL_EDGE_SCHEMA:",
    "PASS"
)


print(
    "\n--- INPUT METADATA ---"
)


print(
    "metadata API:",
    (
        "tracking_cellmot.io.open_dataset"
        "(load_image=False, require_tracks=False)"
    )
)


print(
    "dataset argument:",
    "EXTENSION-LESS STEM"
)


print(
    "physical container:",
    "Dataset.zarr_path"
)


print(
    "shape field:",
    "Dataset.image_shape"
)


print(
    "all-103 shape:",
    actual_shape
)


print(
    "axis order:",
    EXPECTED_AXIS_ORDER
)


print(
    "all-103 scale:",
    tuple(
        actual_scale.tolist()
    )
)


print(
    "ALL_103_METADATA_SCHEMA_CONSISTENCY:",
    "PASS"
)


print(
    "\n--- CANONICAL POST-G3H3 ---"
)


print(
    "actions:",
    total_replay_actions
)


print(
    "source removals:",
    total_replay_removals
)


print(
    "edges added:",
    total_replay_additions
)


print(
    "net edge delta:",
    total_replay_net
)


print(
    "canonical graph fidelity:",
    "PASS"
)


print(
    "\n--- CANDIDATE UNIVERSE ---"
)


print(
    "raw proposals:",
    len(
        raw
    )
)


print(
    "accepted raw proposals:",
    len(
        accepted_raw
    )
)


print(
    "unique candidates:",
    len(
        universe
    )
)


print(
    "datasets with candidates:",
    universe[
        "dataset"
    ].nunique()
)


print(
    "fold counts:",
    (
        universe[
            "fold"
        ]
        .value_counts()
        .sort_index()
        .to_dict()
    )
)


print(
    "raw family counts:",
    RAW_FAMILY_COUNTS
)


print(
    "accepted family counts:",
    ACCEPTED_FAMILY_COUNTS
)


print(
    "candidate family membership:",
    candidate_family_membership
)


print(
    "out-of-bounds:",
    int(
        (
            ~raw[
                "in_bounds"
            ]
        ).sum()
    )
)


print(
    "G2 out-of-bounds:",
    g2_oob_count
)


print(
    "existing-node collisions:",
    int(
        raw[
            "existing_node_collision"
        ].sum()
    )
)


print(
    "\n--- HASHES ---"
)


print(
    "raw semantic SHA:",
    RAW_CONTENT_SHA
)


print(
    "raw file SHA:",
    RAW_FILE_SHA
)


print(
    "universe semantic SHA:",
    UNIVERSE_CONTENT_SHA
)


print(
    "universe file SHA:",
    UNIVERSE_FILE_SHA
)


print(
    "dataset summary file SHA:",
    DATASET_FILE_SHA
)


print(
    "schema content SHA:",
    SCHEMA_SHA
)


print(
    "schema file SHA:",
    SCHEMA_FILE_SHA
)


print(
    "summary content SHA:",
    SUMMARY_SHA
)


print(
    "summary file SHA:",
    SUMMARY_FILE_SHA
)


print(
    "\n--- PROTOCOL ---"
)


print(
    "candidate universe frozen before GT:",
    "YES"
)


print(
    "GT read:",
    "NO"
)


print(
    "A3 read:",
    "NO"
)


print(
    "tracks read:",
    "NO"
)


print(
    "image voxels/chunks read:",
    "NO"
)


print(
    "missing-node oracle read:",
    "NO"
)


print(
    "detector peak required:",
    "NO"
)


print(
    "detector probability required:",
    "NO"
)


print(
    "model trained:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "graph metric:",
    "NO"
)


print(
    "persistent graph mutation:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "\nnext:"
)


print(
    "12.13A2_DEV_GT_ORACLE_CEILING_AUDIT_"
    "OF_FROZEN_TRACK_GEOMETRY_CANDIDATE_UNIVERSE"
)

In [ ]:
# ============================================================
# Stage 12.13A2-P0-R1
#
# TEMPORARY PREFLIGHT REPAIR
#
# EXACT STAGE12.12 A2/A3 IDENTITY + ORACLE SEMANTICS RECOVERY
#
# ============================================================
#
# REPAIRS FAILED 12.13A2-P0
#
# WRONG ASSUMPTION:
#
#     old_universe["candidate_id"]
#
# OLD A2 UNIVERSE DOES NOT STORE candidate_id.
#
# Historical frozen semantic identity:
#
#     dataset, t, z, y, x
#
# Historical A3 procedure:
#
#     1. validate composite key uniqueness
#     2. stable-sort old universe by composite key
#     3. reset_index(drop=True)
#     4. create candidate_id = arange(N)
#
# candidate_id is therefore LOCAL TO OLD A3, not part of the
# original A2 universe schema.
#
# ------------------------------------------------------------
# THIS CELL MAY READ
# ------------------------------------------------------------
#
# - frozen NEW 12.13A1-R3 candidate-universe provenance
# - CLOSED OLD 12.12 A2/A3 artifacts
# - old A2 frozen contract
# - notebook historical A3 source
#
# ------------------------------------------------------------
# THIS CELL DOES NOT
# ------------------------------------------------------------
#
# - evaluate NEW 12.13 candidates against GT
# - add oracle labels to NEW universe
# - modify NEW candidate universe
# - generate candidates
# - train model
# - choose scorer
# - choose threshold
# - mutate graph
# - use Fold0
# - write formal Stage12.13 scientific artifacts
#
# ------------------------------------------------------------
# KEEP UNTIL FORMAL A2 SUCCEEDS: YES
# DELETE AFTER FORMAL A2 SUCCEEDS: YES
# ============================================================


from pathlib import Path

import ast
import hashlib
import json
import re

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


NOTEBOOK_PATH = (
    PROJECT
    / "notebooks"
    / "12_residual_edge_recovery.ipynb"
)


# ------------------------------------------------------------
# NEW 12.13 A1-R3 frozen universe.
# ------------------------------------------------------------

NEW_UNIVERSE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_universe_v1.pkl"
)


NEW_SCHEMA_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_universe_v1_schema_frozen.json"
)


NEW_SUMMARY_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_universe_v1_summary.json"
)


# ------------------------------------------------------------
# OLD 12.12 A2/A3 frozen artifacts.
# ------------------------------------------------------------

OLD_CONTRACT_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_universe_v1_contract_frozen.json"
)


OLD_UNIVERSE_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_universe_v1.pkl"
)


OLD_ORACLE_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_candidates_v1.pkl"
)


OLD_MISSING_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_missing_nodes_v1.pkl"
)


OLD_DETECTION_EDGE_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_detection_edges_v1.pkl"
)


OLD_MATCHING_CEILING_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_matching_ceiling_v1.pkl"
)


for path in [
    NOTEBOOK_PATH,
    NEW_UNIVERSE_PATH,
    NEW_SCHEMA_PATH,
    NEW_SUMMARY_PATH,
    OLD_CONTRACT_PATH,
    OLD_UNIVERSE_PATH,
    OLD_ORACLE_PATH,
    OLD_MISSING_PATH,
    OLD_DETECTION_EDGE_PATH,
    OLD_MATCHING_CEILING_PATH,
]:

    assert path.exists(), path


# ============================================================
# 1. KNOWN FORMAL HASHES / COUNTS
# ============================================================

NEW_UNIVERSE_FILE_SHA = (
    "440676497d3365a0926deaf45140f573"
    "32c8e9e89d46b88401a7ce6f7a299fc3"
)


NEW_UNIVERSE_SEMANTIC_SHA = (
    "856850a3c321538d0ed264892d7019c0"
    "869cc86d7f78a49baeb30f66c20e8565"
)


NEW_SCHEMA_CONTENT_SHA = (
    "3fa3691ee19f0a046e607961526d0844"
    "d7dababe3a15ce53ccf6cf9ba928de35"
)


NEW_SUMMARY_CONTENT_SHA = (
    "f8cd48fb137daab3295f8daf456940aa"
    "9d504b42d5f5b8cbdcefdfb68a60093e"
)


OLD_UNIVERSE_FILE_SHA = (
    "a082babaeecda9e50a13a329abbc912f"
    "f4a16ad56ce3695301d266ca0c5e4147"
)


OLD_ORACLE_FILE_SHA = (
    "2319c69e670438bd0742f8cb84552041"
    "e1656399f11c0f0f41c7251ba4a0ed02"
)


OLD_MISSING_FILE_SHA = (
    "c758c54626f4edc2dd22f17dfde2ef3"
    "c9a3ed685e350a7ac957fc3ee79ca42a1"
)


OLD_DETECTION_EDGE_FILE_SHA = (
    "fe556af1fcbefea9208a68cf4d8caca4"
    "0575fd93d78e2136119eafb9510fc16e"
)


OLD_MATCHING_CEILING_FILE_SHA = (
    "1bcccd79d9afcaec235d1668788ef314"
    "6791eaa290a39dcc8a8a9cd4b0d68acb"
)


EXPECTED_NEW_ROWS = 1_079_885

EXPECTED_OLD_ROWS = 415_590

EXPECTED_OLD_ORACLE_POSITIVES = 516

EXPECTED_MISSING_NODES = 2_985

EXPECTED_DETECTION_EDGES = 4_097

EXPECTED_MATCHING_CEILING_ROWS = 8_632

EXPECTED_OLD_NODE_COVERED_7UM = 440

EXPECTED_OLD_EDGE_COVERED_7UM = 382


EXPECTED_OLD_FOLD_ROWS = {
    1: 94_729,
    2: 100_318,
    3: 100_147,
    4: 120_396,
}


EXPECTED_OLD_FOLD_POSITIVES = {
    1: 173,
    2: 150,
    3: 104,
    4: 89,
}


EXPECTED_MISSING_CLASS_COUNTS = {
    "BOTH_MISSING": 1_390,
    "SOURCE_MISSING": 1_365,
    "TARGET_MISSING": 1_342,
}


EXPECTED_RADII_UM = [
    2.0,
    3.0,
    5.0,
    7.0,
]


EXPECTED_SCALE = np.asarray(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=np.float64,
)


OLD_CANDIDATE_KEY = [
    "dataset",
    "t",
    "z",
    "y",
    "x",
]


# ============================================================
# 2. HELPERS
# ============================================================

def sha256_file(
    path,
):

    h = hashlib.sha256()


    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )


            if not block:

                break


            h.update(
                block
            )


    return h.hexdigest()


# ============================================================
# 3. LOCK NEW 12.13 A1-R3
#
# Reading NEW universe for provenance/schema is allowed.
#
# We still DO NOT apply any GT/oracle to it.
# ============================================================

assert sha256_file(
    NEW_UNIVERSE_PATH
) == NEW_UNIVERSE_FILE_SHA


new_schema = json.loads(
    NEW_SCHEMA_PATH.read_text(
        encoding="utf-8"
    )
)


new_summary = json.loads(
    NEW_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert new_schema[
    "schema_sha256"
] == NEW_SCHEMA_CONTENT_SHA


assert new_summary[
    "summary_sha256"
] == NEW_SUMMARY_CONTENT_SHA


assert new_schema[
    "stage"
] == "12.13A1-R3"


assert new_schema[
    "protocol"
][
    "candidate_universe_frozen_before_GT"
] is True


assert new_schema[
    "protocol"
][
    "GT_read"
] is False


assert new_schema[
    "protocol"
][
    "Fold0_used"
] is False


new_universe = pd.read_pickle(
    NEW_UNIVERSE_PATH
)


assert len(
    new_universe
) == EXPECTED_NEW_ROWS


assert "candidate_id" in new_universe.columns


assert new_universe[
    "candidate_id"
].is_unique


assert np.array_equal(
    new_universe[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        EXPECTED_NEW_ROWS,
        dtype=np.int64,
    ),
)


for column in new_universe.columns:

    lower = str(
        column
    ).lower()


    assert "oracle" not in lower


    assert not lower.startswith(
        "gt_"
    )


    assert "missing_node" not in lower


print(
    "========== 12.13A2-P0-R1 NEW PARENT LOCK =========="
)


print(
    "new stage:",
    new_schema[
        "stage"
    ]
)


print(
    "new universe rows:",
    len(
        new_universe
    )
)


print(
    "new universe SHA:",
    sha256_file(
        NEW_UNIVERSE_PATH
    )
)


print(
    "new semantic SHA:",
    NEW_UNIVERSE_SEMANTIC_SHA
)


print(
    "candidate universe frozen before GT:",
    "YES"
)


print(
    "NEW candidate GT evaluated:",
    "NO"
)


print(
    "Fold0:",
    "NO"
)


# ============================================================
# 4. LOAD + LOCK OLD A2 CONTRACT
# ============================================================

old_contract = json.loads(
    OLD_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    "candidate_key"
    in old_contract
), (
    "Old A2 frozen contract does not expose candidate_key."
)


assert list(
    old_contract[
        "candidate_key"
    ]
) == OLD_CANDIDATE_KEY


print(
    "\n========== OLD 12.12 A2 CONTRACT =========="
)


print(
    "candidate key:",
    old_contract[
        "candidate_key"
    ]
)


print(
    "OLD_A2_COMPOSITE_IDENTITY_CONTRACT: PASS"
)


# ============================================================
# 5. LOCK OLD ARTIFACT FILE HASHES
# ============================================================

assert sha256_file(
    OLD_UNIVERSE_PATH
) == OLD_UNIVERSE_FILE_SHA


assert sha256_file(
    OLD_ORACLE_PATH
) == OLD_ORACLE_FILE_SHA


assert sha256_file(
    OLD_MISSING_PATH
) == OLD_MISSING_FILE_SHA


assert sha256_file(
    OLD_DETECTION_EDGE_PATH
) == OLD_DETECTION_EDGE_FILE_SHA


assert sha256_file(
    OLD_MATCHING_CEILING_PATH
) == OLD_MATCHING_CEILING_FILE_SHA


# ============================================================
# 6. LOAD OLD A2/A3 ARTIFACTS
# ============================================================

old_universe = pd.read_pickle(
    OLD_UNIVERSE_PATH
)


old_oracle = pd.read_pickle(
    OLD_ORACLE_PATH
)


old_missing = pd.read_pickle(
    OLD_MISSING_PATH
)


old_detection_edges = pd.read_pickle(
    OLD_DETECTION_EDGE_PATH
)


old_matching_ceiling = pd.read_pickle(
    OLD_MATCHING_CEILING_PATH
)


# ============================================================
# 7. OLD A2 UNIVERSE SCHEMA
# ============================================================

assert len(
    old_universe
) == EXPECTED_OLD_ROWS


EXPECTED_OLD_UNIVERSE_COLUMNS = [
    "dataset",
    "fold",
    "t",
    "z_ds",
    "y_ds",
    "x_ds",
    "z",
    "y",
    "x",
    "detector_prob",
    "detector_logit",
    "release_band",
]


assert list(
    old_universe.columns
) == EXPECTED_OLD_UNIVERSE_COLUMNS


# Critical historical fact:
assert (
    "candidate_id"
    not in old_universe.columns
)


assert set(
    OLD_CANDIDATE_KEY
).issubset(
    old_universe.columns
)


assert not old_universe[
    OLD_CANDIDATE_KEY
].duplicated().any()


print(
    "\n========== OLD A2 UNIVERSE SCHEMA =========="
)


print(
    "rows:",
    len(
        old_universe
    )
)


print(
    "columns:",
    list(
        old_universe.columns
    )
)


print(
    "candidate_id stored:",
    "NO"
)


print(
    "identity:",
    OLD_CANDIDATE_KEY
)


print(
    "identity duplicates:",
    int(
        old_universe[
            OLD_CANDIDATE_KEY
        ].duplicated().sum()
    )
)


print(
    "OLD_A2_UNIVERSE_IDENTITY: PASS"
)


# ============================================================
# 8. OLD A3 ORACLE SCHEMA
# ============================================================

assert len(
    old_oracle
) == EXPECTED_OLD_ROWS


EXPECTED_OLD_ORACLE_REQUIRED_COLUMNS = {
    "candidate_id",
    "dataset",
    "fold",
    "prefix",
    "t",
    "z",
    "y",
    "x",
    "detector_prob",
    "detector_logit",
    "nearest_missing_distance_um",
    "missing_count_within_2p0um",
    "oracle_hit_2p0um",
    "missing_count_within_3p0um",
    "oracle_hit_3p0um",
    "missing_count_within_5p0um",
    "oracle_hit_5p0um",
    "missing_count_within_7p0um",
    "oracle_hit_7p0um",
    "oracle_hit_7um",
}


assert EXPECTED_OLD_ORACLE_REQUIRED_COLUMNS.issubset(
    old_oracle.columns
)


assert not old_oracle[
    OLD_CANDIDATE_KEY
].duplicated().any()


assert old_oracle[
    "candidate_id"
].is_unique


old_oracle_candidate_id = pd.to_numeric(
    old_oracle[
        "candidate_id"
    ],
    errors="raise",
).to_numpy(
    dtype=np.int64
)


assert np.array_equal(
    old_oracle_candidate_id,
    np.arange(
        EXPECTED_OLD_ROWS,
        dtype=np.int64,
    ),
)


old_oracle[
    "oracle_hit_7um"
] = pd.to_numeric(
    old_oracle[
        "oracle_hit_7um"
    ],
    errors="raise",
).astype(
    np.int8
)


assert set(
    old_oracle[
        "oracle_hit_7um"
    ].unique()
).issubset(
    {
        0,
        1,
    }
)


assert int(
    old_oracle[
        "oracle_hit_7um"
    ].sum()
) == EXPECTED_OLD_ORACLE_POSITIVES


print(
    "\n========== OLD A3 ORACLE SCHEMA =========="
)


print(
    "rows:",
    len(
        old_oracle
    )
)


print(
    "candidate_id:",
    "0..415589"
)


print(
    "composite identity duplicates:",
    int(
        old_oracle[
            OLD_CANDIDATE_KEY
        ].duplicated().sum()
    )
)


print(
    "oracle_hit_7um positives:",
    int(
        old_oracle[
            "oracle_hit_7um"
        ].sum()
    )
)


print(
    "OLD_A3_ORACLE_SCHEMA: PASS"
)


# ============================================================
# 9. EXACT HISTORICAL A3 ID CREATION REPRODUCTION
#
# Cell 77 historical semantics:
#
# stable-sort by dataset,t,z,y,x
# reset_index(drop=True)
# candidate_id = arange(N)
# ============================================================

old_universe_sorted = (
    old_universe
    .sort_values(
        OLD_CANDIDATE_KEY,
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Do NOT mutate original frozen old_universe dataframe.
#
# This local reconstruction is explicitly an A3 reproduction.
# ------------------------------------------------------------

reconstructed_candidate_id = np.arange(
    len(
        old_universe_sorted
    ),
    dtype=np.int64,
)


# ============================================================
# 10. EXACT COMPOSITE-KEY MERGE
# ============================================================

label_columns = [
    "candidate_id",
    "oracle_hit_2p0um",
    "oracle_hit_3p0um",
    "oracle_hit_5p0um",
    "oracle_hit_7p0um",
    "oracle_hit_7um",
    "nearest_missing_distance_um",
]


old_labels = old_oracle[
    OLD_CANDIDATE_KEY
    +
    label_columns
].copy()


old_joined = old_universe_sorted.merge(
    old_labels,
    on=OLD_CANDIDATE_KEY,
    how="left",
    validate="one_to_one",
    sort=False,
)


assert len(
    old_joined
) == EXPECTED_OLD_ROWS


assert old_joined[
    "candidate_id"
].notna().all()


assert old_joined[
    "oracle_hit_7um"
].notna().all()


joined_candidate_id = pd.to_numeric(
    old_joined[
        "candidate_id"
    ],
    errors="raise",
).to_numpy(
    dtype=np.int64
)


# ============================================================
# 11. HISTORICAL ROW-ID PROOF
#
# A3 candidate_id must equal the stable-sorted A2 row order.
# ============================================================

assert np.array_equal(
    joined_candidate_id,
    reconstructed_candidate_id,
)


assert int(
    pd.to_numeric(
        old_joined[
            "oracle_hit_7um"
        ],
        errors="raise",
    ).sum()
) == EXPECTED_OLD_ORACLE_POSITIVES


# ------------------------------------------------------------
# Also prove common provenance fields are identical.
# ------------------------------------------------------------

for column in [
    "fold",
    "detector_prob",
    "detector_logit",
]:

    oracle_series = (
        old_oracle
        .sort_values(
            "candidate_id",
            kind="stable",
        )
        .reset_index(
            drop=True
        )[
            column
        ]
    )


    joined_series = old_joined[
        column
    ]


    if pd.api.types.is_numeric_dtype(
        oracle_series
    ):

        assert np.array_equal(
            pd.to_numeric(
                oracle_series,
                errors="raise",
            ).to_numpy(),
            pd.to_numeric(
                joined_series,
                errors="raise",
            ).to_numpy(),
        )


    else:

        assert oracle_series.astype(
            str
        ).equals(
            joined_series.astype(
                str
            )
        )


print(
    "\n========== OLD A3 IDENTITY RECONSTRUCTION =========="
)


print(
    "stable sort key:",
    OLD_CANDIDATE_KEY
)


print(
    "composite merge:",
    "ONE_TO_ONE"
)


print(
    "all labels attached:",
    "YES"
)


print(
    "candidate_id equals stable row order:",
    "YES"
)


print(
    "oracle_hit_7um positives:",
    int(
        old_joined[
            "oracle_hit_7um"
        ].sum()
    )
)


print(
    "OLD_A3_IDENTITY_RECONSTRUCTION: PASS"
)


# ============================================================
# 12. OLD A3 MISSING-NODE ARTIFACT
# ============================================================

assert len(
    old_missing
) == EXPECTED_MISSING_NODES


assert {
    "dataset",
    "fold",
    "gt_node_id",
    "t",
    "gt_z",
    "gt_y",
    "gt_x",
    "nearest_candidate_distance_um",
    "covered_any_2p0um",
    "covered_any_3p0um",
    "covered_any_5p0um",
    "covered_any_7p0um",
}.issubset(
    old_missing.columns
)


assert not old_missing[
    [
        "dataset",
        "gt_node_id",
    ]
].duplicated().any()


covered_nodes_7 = int(
    pd.to_numeric(
        old_missing[
            "covered_any_7p0um"
        ],
        errors="raise",
    ).astype(
        bool
    ).sum()
)


assert (
    covered_nodes_7
    ==
    EXPECTED_OLD_NODE_COVERED_7UM
)


print(
    "\n========== OLD A3 MISSING NODE LOCK =========="
)


print(
    "missing nodes:",
    len(
        old_missing
    )
)


print(
    "covered @7um:",
    covered_nodes_7
)


print(
    "node coverage @7um:",
    covered_nodes_7
    /
    len(
        old_missing
    )
)


print(
    "OLD_A3_MISSING_NODE_LOCK: PASS"
)


# ============================================================
# 13. OLD A3 DETECTION-EDGE ARTIFACT
# ============================================================

assert len(
    old_detection_edges
) == EXPECTED_DETECTION_EDGES


assert {
    "dataset",
    "fold",
    "gt_source_id",
    "gt_target_id",
    "source_missing",
    "target_missing",
    "missing_class",
    "all_missing_endpoints_covered_2p0um",
    "all_missing_endpoints_covered_3p0um",
    "all_missing_endpoints_covered_5p0um",
    "all_missing_endpoints_covered_7p0um",
}.issubset(
    old_detection_edges.columns
)


assert not old_detection_edges[
    [
        "dataset",
        "gt_source_id",
        "gt_target_id",
    ]
].duplicated().any()


actual_missing_class_counts = (
    old_detection_edges[
        "missing_class"
    ]
    .value_counts()
    .to_dict()
)


assert (
    actual_missing_class_counts
    ==
    EXPECTED_MISSING_CLASS_COUNTS
)


covered_edges_7 = int(
    pd.to_numeric(
        old_detection_edges[
            "all_missing_endpoints_covered_7p0um"
        ],
        errors="raise",
    ).astype(
        bool
    ).sum()
)


assert (
    covered_edges_7
    ==
    EXPECTED_OLD_EDGE_COVERED_7UM
)


print(
    "\n========== OLD A3 DETECTION-EDGE LOCK =========="
)


print(
    "edges:",
    len(
        old_detection_edges
    )
)


print(
    "missing classes:",
    actual_missing_class_counts
)


print(
    "covered @7um:",
    covered_edges_7
)


print(
    "edge coverage @7um:",
    covered_edges_7
    /
    len(
        old_detection_edges
    )
)


print(
    "OLD_A3_DETECTION_EDGE_LOCK: PASS"
)


# ============================================================
# 14. OLD A3 ONE-TO-ONE MATCHING CEILING
#
# This artifact is intentionally separate from candidate
# positive-label semantics.
# ============================================================

assert len(
    old_matching_ceiling
) == EXPECTED_MATCHING_CEILING_ROWS


assert {
    "dataset",
    "fold",
    "t",
    "radius_um",
    "missing_nodes",
    "candidate_nodes",
    "admissible_pairs",
    "maximum_one_to_one_matches",
}.issubset(
    old_matching_ceiling.columns
)


actual_radii = sorted(
    pd.to_numeric(
        old_matching_ceiling[
            "radius_um"
        ],
        errors="raise",
    ).unique().tolist()
)


assert np.allclose(
    actual_radii,
    EXPECTED_RADII_UM,
    rtol=0.0,
    atol=0.0,
)


print(
    "\n========== OLD A3 MATCHING CEILING LOCK =========="
)


print(
    "rows:",
    len(
        old_matching_ceiling
    )
)


print(
    "radii:",
    actual_radii
)


print(
    "role:",
    "SEPARATE_ONE_TO_ONE_CEILING_SUMMARY"
)


print(
    "OLD_A3_MATCHING_CEILING_LOCK: PASS"
)


# ============================================================
# 15. NOTEBOOK CELL 77 HISTORICAL SOURCE RECOVERY
#
# Codex audit independently identified cell 77 as old A3.
#
# We inspect but do not execute it.
# ============================================================

notebook = json.loads(
    NOTEBOOK_PATH.read_text(
        encoding="utf-8"
    )
)


cells = notebook[
    "cells"
]


A3_CELL_INDEX = 77


assert (
    0
    <=
    A3_CELL_INDEX
    <
    len(
        cells
    )
)


a3_cell = cells[
    A3_CELL_INDEX
]


assert a3_cell.get(
    "cell_type"
) == "code"


a3_source = "".join(
    a3_cell.get(
        "source",
        []
    )
)


A3_CELL_SHA = hashlib.sha256(
    a3_source.encode(
        "utf-8"
    )
).hexdigest()


# ------------------------------------------------------------
# Source-semantic fingerprints.
# ------------------------------------------------------------

for required_token in [
    'CANDIDATE_KEY',
    '"dataset"',
    '"t"',
    '"z"',
    '"y"',
    '"x"',
    "sort_values",
    'kind="stable"',
    "candidate_id",
    "np.arange",
    "cKDTree",
    "maximum_bipartite_matching",
    "oracle_hit_7um",
    "DETECTION_LIMITED",
    "SOURCE_MISSING",
    "TARGET_MISSING",
    "BOTH_MISSING",
]:

    assert required_token in a3_source, (
        f"Historical A3 cell 77 missing token: {required_token}"
    )


print(
    "\n========== HISTORICAL A3 SOURCE LOCK =========="
)


print(
    "cell:",
    A3_CELL_INDEX
)


print(
    "cell SHA:",
    A3_CELL_SHA
)


print(
    "candidate key:",
    OLD_CANDIDATE_KEY
)


print(
    "stable sort:",
    "YES"
)


print(
    "candidate_id created after sort:",
    "YES"
)


print(
    "cKDTree radius semantics:",
    "YES"
)


print(
    "one-to-one ceiling separate:",
    "YES"
)


print(
    "HISTORICAL_A3_SOURCE_RECOVERY: PASS"
)


# ============================================================
# 16. EXACT OLD COVERAGE / NEW PREDECLARED GATE
# ============================================================

OLD_NODE_COVERAGE_7UM = (
    EXPECTED_OLD_NODE_COVERED_7UM
    /
    EXPECTED_MISSING_NODES
)


OLD_EDGE_COVERAGE_7UM = (
    EXPECTED_OLD_EDGE_COVERED_7UM
    /
    EXPECTED_DETECTION_EDGES
)


REQUIRED_NEW_NODE_COVERAGE_7UM = (
    1.5
    *
    OLD_NODE_COVERAGE_7UM
)


REQUIRED_NEW_EDGE_COVERAGE_7UM = (
    1.5
    *
    OLD_EDGE_COVERAGE_7UM
)


assert np.isclose(
    REQUIRED_NEW_NODE_COVERAGE_7UM,
    0.22110552763819097,
    rtol=0.0,
    atol=1e-16,
)


assert np.isclose(
    REQUIRED_NEW_EDGE_COVERAGE_7UM,
    0.13985843299975592,
    rtol=0.0,
    atol=1e-16,
)


print(
    "\n========== A0 PREDECLARED NEW CEILING GATE =========="
)


print(
    "old node coverage:",
    OLD_NODE_COVERAGE_7UM
)


print(
    "required new node coverage:",
    REQUIRED_NEW_NODE_COVERAGE_7UM
)


print(
    "old edge coverage:",
    OLD_EDGE_COVERAGE_7UM
)


print(
    "required new edge coverage:",
    REQUIRED_NEW_EDGE_COVERAGE_7UM
)


print(
    "BOTH required:",
    "YES"
)


# ============================================================
# 17. NEW-UNIVERSE SAFETY CHECK
#
# New candidate_id semantics remain entirely separate.
# ============================================================

assert {
    "candidate_id",
    "dataset",
    "fold",
    "t",
    "z",
    "y",
    "x",
}.issubset(
    new_universe.columns
)


# New semantic key from A1-R3.
NEW_SEMANTIC_KEY = [
    "dataset",
    "t",
    "z",
    "y",
    "x",
]


assert not new_universe[
    NEW_SEMANTIC_KEY
].duplicated().any()


# No oracle column is attached.
for column in new_universe.columns:

    assert "oracle" not in str(
        column
    ).lower()


# ============================================================
# 18. PROTOCOL STATE
# ============================================================

OLD_A3_READ = True

NEW_CANDIDATE_GT_EVALUATED = False

NEW_CANDIDATE_ORACLE_CREATED = False

NEW_CANDIDATE_UNIVERSE_MODIFIED = False

MODEL_TRAINED = False

THRESHOLD_SELECTED = False

GRAPH_MUTATED = False

FOLD0_USED = False


assert OLD_A3_READ is True

assert NEW_CANDIDATE_GT_EVALUATED is False

assert NEW_CANDIDATE_ORACLE_CREATED is False

assert NEW_CANDIDATE_UNIVERSE_MODIFIED is False

assert MODEL_TRAINED is False

assert THRESHOLD_SELECTED is False

assert GRAPH_MUTATED is False

assert FOLD0_USED is False


# ============================================================
# 19. FINAL
# ============================================================

print(
    "\n========== 12.13A2-P0-R1 FINAL =========="
)


print(
    "EXACT_PREVIOUS_A3_IDENTITY_AND_ORACLE_RECOVERY:",
    "PASS"
)


print(
    "\n--- OLD A2 IDENTITY ---"
)


print(
    "old universe rows:",
    len(
        old_universe
    )
)


print(
    "candidate_id stored in old A2 universe:",
    "NO"
)


print(
    "old semantic identity:",
    OLD_CANDIDATE_KEY
)


print(
    "identity unique:",
    "YES"
)


print(
    "\n--- OLD A3 LOCAL candidate_id ---"
)


print(
    "stable sort:",
    "dataset,t,z,y,x"
)


print(
    "candidate_id generated after sort:",
    "YES"
)


print(
    "candidate_id range:",
    (
        int(
            joined_candidate_id.min()
        ),
        int(
            joined_candidate_id.max()
        ),
    )
)


print(
    "candidate_id equals sorted row order:",
    "YES"
)


print(
    "\n--- OLD A3 ORACLE ---"
)


print(
    "candidate rows:",
    len(
        old_joined
    )
)


print(
    "oracle_hit_7um positives:",
    int(
        old_joined[
            "oracle_hit_7um"
        ].sum()
    )
)


print(
    "missing GT nodes:",
    len(
        old_missing
    )
)


print(
    "detection-limited edges:",
    len(
        old_detection_edges
    )
)


print(
    "missing classes:",
    actual_missing_class_counts
)


print(
    "node covered @7um:",
    covered_nodes_7
)


print(
    "edge covered @7um:",
    covered_edges_7
)


print(
    "\n--- ORACLE SEMANTICS ---"
)


print(
    "radii um:",
    EXPECTED_RADII_UM
)


print(
    "scale zyx um:",
    EXPECTED_SCALE.tolist()
)


print(
    "candidate positive semantics:",
    "MANY_TO_ONE_RADIUS_MEMBERSHIP_GT_ZERO"
)


print(
    "one-to-one matching:",
    "SEPARATE CEILING SUMMARY ONLY"
)


print(
    "edge coverage:",
    "ALL MISSING ENDPOINTS MUST BE COVERED"
)


print(
    "\n--- NEW A0 GATE ---"
)


print(
    "required node coverage @7um:",
    REQUIRED_NEW_NODE_COVERAGE_7UM
)


print(
    "required edge coverage @7um:",
    REQUIRED_NEW_EDGE_COVERAGE_7UM
)


print(
    "BOTH required:",
    "YES"
)


print(
    "\n--- SAFETY ---"
)


print(
    "old A3 read:",
    "YES"
)


print(
    "NEW candidate GT evaluated:",
    "NO"
)


print(
    "NEW candidate oracle created:",
    "NO"
)


print(
    "NEW universe modified:",
    "NO"
)


print(
    "model trained:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "graph mutation:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "\nnext:"
)


print(
    "12.13A2_FORMAL_DEV_GT_ORACLE_CEILING_AUDIT_"
    "OF_FROZEN_TRACK_GEOMETRY_CANDIDATE_UNIVERSE"
)

In [ ]:
# ============================================================
# Stage 12.13A2
#
# FORMAL DEV-GT ORACLE CEILING AUDIT
#
# OF THE FROZEN
# POST-G3H3 TRACK-GEOMETRY SYNTHETIC CANDIDATE UNIVERSE
#
# ============================================================
#
# PARENT:
#
#   12.13A1-R3
#
#   GT-blind candidate universe already frozen:
#
#     rows = 1,079,885
#
#     file SHA =
#       440676497d3365a0926deaf45140f573
#       32c8e9e89d46b88401a7ce6f7a299fc3
#
#     semantic SHA =
#       856850a3c321538d0ed264892d7019c0
#       869cc86d7f78a49baeb30f66c20e8565
#
# ------------------------------------------------------------
# EXACT HISTORICAL ORACLE SEMANTICS
# ------------------------------------------------------------
#
# Recovered and formally reproduced in 12.13A2-P0-R1:
#
# old candidate identity:
#
#   dataset,t,z,y,x
#
# old A3:
#
#   stable-sort by identity
#   candidate_id = arange(N)
#
# physical scale:
#
#   z = 1.625 um
#   y = 0.40625 um
#   x = 0.40625 um
#
# radii:
#
#   2, 3, 5, 7 um
#
# candidate oracle label:
#
#   MANY-TO-ONE radius membership
#
#   oracle_hit_R =
#       at least one missing GT node
#       within physical distance <= R
#
# missing-node coverage:
#
#   covered_any_R =
#       at least one candidate within R
#
# detection-limited edge coverage:
#
#   ALL missing endpoints must be covered.
#
# SOURCE_MISSING:
#   source must be covered
#
# TARGET_MISSING:
#   target must be covered
#
# BOTH_MISSING:
#   source AND target must be covered
#
# one-to-one maximum matching:
#
#   SEPARATE ceiling summary only.
#   It does NOT define oracle_hit_R.
#
# ------------------------------------------------------------
# HISTORICAL REPRODUCTION GATE
# ------------------------------------------------------------
#
# Before touching the NEW candidates scientifically, the
# implementation below MUST reproduce OLD A3:
#
#   candidates                = 415,590
#   oracle_hit_7um positives  = 516
#   missing nodes             = 2,985
#   covered nodes @7um        = 440
#   detection edges           = 4,097
#   covered edges @7um        = 382
#
# ------------------------------------------------------------
# A0 PREDECLARED NEW GATE
# ------------------------------------------------------------
#
# OLD:
#
#   node coverage =
#       440 / 2985
#       = 0.1474036850921273
#
#   edge coverage =
#       382 / 4097
#       = 0.09323895533317061
#
# NEW branch MUST satisfy BOTH:
#
#   node coverage @7um >=
#       0.22110552763819097
#
#   edge coverage @7um >=
#       0.13985843299975592
#
# ------------------------------------------------------------
# DECISION
# ------------------------------------------------------------
#
# BOTH PASS:
#
#   PASS_CEILING_GATE
#
#   next:
#     freeze a SEPARATE ranking / attachment contract
#     BEFORE any model training.
#
# EITHER FAIL:
#
#   CLOSE_BRANCH_NO_RANKING_NO_GRAPH_MUTATION
#
# ------------------------------------------------------------
# PROTOCOL
# ------------------------------------------------------------
#
# Development folds 1-4 only.
#
# Fold0:
#   FORBIDDEN
#
# Candidate universe:
#   MUST NOT CHANGE
#
# No model training.
# No threshold tuning.
# No graph mutation.
#
# KEEP AFTER RUNNING: YES
# ============================================================


from pathlib import Path

import hashlib
import json
import math
import gc

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


A0_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_branch_frozen.json"
)


NEW_UNIVERSE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_universe_v1.pkl"
)


NEW_SCHEMA_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_universe_v1_schema_frozen.json"
)


NEW_SUMMARY_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_universe_v1_summary.json"
)


# ============================================================
# HISTORICAL 12.12 A2/A3 ARTIFACTS
# ============================================================

OLD_CONTRACT_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_universe_v1_contract_frozen.json"
)


OLD_UNIVERSE_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_universe_v1.pkl"
)


OLD_ORACLE_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_candidates_v1.pkl"
)


OLD_MISSING_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_missing_nodes_v1.pkl"
)


OLD_DETECTION_EDGE_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_detection_edges_v1.pkl"
)


OLD_MATCHING_CEILING_PATH = (
    S12
    / "stage12_post_r1p50_node_candidate_oracle_matching_ceiling_v1.pkl"
)


# ============================================================
# FORMAL A2 OUTPUTS
# ============================================================

NEW_ORACLE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_oracle_candidates_v1.pkl"
)


NEW_MISSING_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_oracle_missing_nodes_v1.pkl"
)


NEW_DETECTION_EDGE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_oracle_detection_edges_v1.pkl"
)


NEW_MATCHING_CEILING_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_oracle_matching_ceiling_v1.pkl"
)


NEW_CEILING_SUMMARY_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_oracle_ceiling_v1_summary.json"
)


NEW_DECISION_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_oracle_ceiling_v1_decision_frozen.json"
)


BRANCH_CLOSURE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_closure.json"
)


for path in [
    A0_PATH,
    NEW_UNIVERSE_PATH,
    NEW_SCHEMA_PATH,
    NEW_SUMMARY_PATH,
    OLD_CONTRACT_PATH,
    OLD_UNIVERSE_PATH,
    OLD_ORACLE_PATH,
    OLD_MISSING_PATH,
    OLD_DETECTION_EDGE_PATH,
    OLD_MATCHING_CEILING_PATH,
]:

    assert path.exists(), path


# ------------------------------------------------------------
# Formal A2 must never overwrite an existing formal result.
# ------------------------------------------------------------

for path in [
    NEW_ORACLE_PATH,
    NEW_MISSING_PATH,
    NEW_DETECTION_EDGE_PATH,
    NEW_MATCHING_CEILING_PATH,
    NEW_CEILING_SUMMARY_PATH,
    NEW_DECISION_PATH,
    BRANCH_CLOSURE_PATH,
]:

    if path.exists():

        raise RuntimeError(
            "\nFormal Stage12.13A2 output already exists:\n"
            f"{path}\n\n"
            "Do not rerun/overwrite formal provenance."
        )


# ============================================================
# 1. LOCKED IDENTITIES / HASHES
# ============================================================

BRANCH_ID = (
    "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_NODE_RECOVERY_V1"
)


CANDIDATE_UNIVERSE_ID = (
    "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_CANDIDATE_UNIVERSE_V1"
)


A0_SHA = (
    "02ee83aa6554634c72c4a55f28e37d30"
    "83d8c311820a5a37caec99d3d0763e54"
)


NEW_UNIVERSE_FILE_SHA = (
    "440676497d3365a0926deaf45140f573"
    "32c8e9e89d46b88401a7ce6f7a299fc3"
)


NEW_UNIVERSE_SEMANTIC_SHA = (
    "856850a3c321538d0ed264892d7019c0"
    "869cc86d7f78a49baeb30f66c20e8565"
)


NEW_SCHEMA_SHA = (
    "3fa3691ee19f0a046e607961526d0844"
    "d7dababe3a15ce53ccf6cf9ba928de35"
)


NEW_SUMMARY_SHA = (
    "f8cd48fb137daab3295f8daf456940aa"
    "9d504b42d5f5b8cbdcefdfb68a60093e"
)


OLD_UNIVERSE_FILE_SHA = (
    "a082babaeecda9e50a13a329abbc912f"
    "f4a16ad56ce3695301d266ca0c5e4147"
)


OLD_ORACLE_FILE_SHA = (
    "2319c69e670438bd0742f8cb84552041"
    "e1656399f11c0f0f41c7251ba4a0ed02"
)


OLD_MISSING_FILE_SHA = (
    "c758c54626f4edc2dd22f17dfde2ef3"
    "c9a3ed685e350a7ac957fc3ee79ca42a1"
)


OLD_DETECTION_EDGE_FILE_SHA = (
    "fe556af1fcbefea9208a68cf4d8caca4"
    "0575fd93d78e2136119eafb9510fc16e"
)


OLD_MATCHING_CEILING_FILE_SHA = (
    "1bcccd79d9afcaec235d1668788ef314"
    "6791eaa290a39dcc8a8a9cd4b0d68acb"
)


EXPECTED_NEW_ROWS = 1_079_885

EXPECTED_OLD_ROWS = 415_590

EXPECTED_OLD_POSITIVES_7 = 516

EXPECTED_MISSING_NODES = 2_985

EXPECTED_DETECTION_EDGES = 4_097

EXPECTED_OLD_NODE_COVERED_7 = 440

EXPECTED_OLD_EDGE_COVERED_7 = 382


EXPECTED_MISSING_CLASSES = {
    "BOTH_MISSING": 1_390,
    "SOURCE_MISSING": 1_365,
    "TARGET_MISSING": 1_342,
}


SCALE = np.asarray(
    [
        1.625,
        0.40625,
        0.40625,
    ],
    dtype=np.float64,
)


RADII = (
    2.0,
    3.0,
    5.0,
    7.0,
)


REQUIRED_NODE_COVERAGE_7 = (
    0.22110552763819097
)


REQUIRED_EDGE_COVERAGE_7 = (
    0.13985843299975592
)


OLD_KEY = [
    "dataset",
    "t",
    "z",
    "y",
    "x",
]


# ============================================================
# 2. HASH HELPERS
# ============================================================

def sha256_file(
    path,
):

    h = hashlib.sha256()


    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )


            if not block:

                break


            h.update(
                block
            )


    return h.hexdigest()


class HashWriter:

    def __init__(
        self,
        hasher,
    ):

        self.hasher = hasher


    def write(
        self,
        text,
    ):

        if not isinstance(
            text,
            str,
        ):

            text = str(
                text
            )


        self.hasher.update(
            text.encode(
                "utf-8"
            )
        )


        return len(
            text
        )


    def flush(
        self,
    ):

        return None


def dataframe_content_sha256(
    df,
    columns,
):

    h = hashlib.sha256()


    writer = HashWriter(
        h
    )


    df.loc[
        :,
        columns,
    ].to_csv(
        writer,
        index=False,
        na_rep="NaN",
        float_format="%.17g",
        lineterminator="\n",
    )


    return h.hexdigest()


def radius_tag(
    radius,
):

    text = (
        f"{float(radius):.1f}"
        .replace(
            ".",
            "p",
        )
    )


    return (
        f"{text}um"
    )


def ball_count(
    tree,
    points,
    radius,
):

    points = np.asarray(
        points,
        dtype=np.float64,
    )


    try:

        result = tree.query_ball_point(
            points,
            r=float(
                radius
            ),
            return_length=True,
        )


        return np.asarray(
            result,
            dtype=np.int64,
        )


    except TypeError:

        result = tree.query_ball_point(
            points,
            r=float(
                radius
            ),
        )


        return np.asarray(
            [
                len(
                    item
                )

                for item
                in result
            ],
            dtype=np.int64,
        )


# ============================================================
# 3. A0 + NEW A1-R3 PARENT LOCK
# ============================================================

a0 = json.loads(
    A0_PATH.read_text(
        encoding="utf-8"
    )
)


assert a0[
    "contract_sha256"
] == A0_SHA


assert a0[
    "branch_id"
] == BRANCH_ID


assert a0[
    "candidate_universe_id"
] == CANDIDATE_UNIVERSE_ID


assert sha256_file(
    NEW_UNIVERSE_PATH
) == NEW_UNIVERSE_FILE_SHA


new_schema = json.loads(
    NEW_SCHEMA_PATH.read_text(
        encoding="utf-8"
    )
)


new_summary = json.loads(
    NEW_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert new_schema[
    "schema_sha256"
] == NEW_SCHEMA_SHA


assert new_summary[
    "summary_sha256"
] == NEW_SUMMARY_SHA


assert new_schema[
    "stage"
] == "12.13A1-R3"


assert new_schema[
    "protocol"
][
    "candidate_universe_frozen_before_GT"
] is True


assert new_schema[
    "protocol"
][
    "GT_read"
] is False


assert new_schema[
    "protocol"
][
    "Fold0_used"
] is False


new_universe = pd.read_pickle(
    NEW_UNIVERSE_PATH
)


assert len(
    new_universe
) == EXPECTED_NEW_ROWS


assert new_universe[
    "candidate_id"
].is_unique


assert np.array_equal(
    new_universe[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        EXPECTED_NEW_ROWS,
        dtype=np.int64,
    ),
)


assert not new_universe[
    OLD_KEY
].duplicated().any()


assert set(
    new_universe[
        "fold"
    ].astype(
        int
    ).unique()
).issubset(
    {
        1,
        2,
        3,
        4,
    }
)


assert not (
    new_universe[
        "fold"
    ].astype(
        int
    )
    ==
    0
).any()


print(
    "========== 12.13A2 PARENT LOCK =========="
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "candidate universe:",
    CANDIDATE_UNIVERSE_ID
)


print(
    "rows:",
    len(
        new_universe
    )
)


print(
    "file SHA:",
    NEW_UNIVERSE_FILE_SHA
)


print(
    "semantic SHA:",
    NEW_UNIVERSE_SEMANTIC_SHA
)


print(
    "frozen before GT:",
    "YES"
)


print(
    "Fold0:",
    "NO"
)


# ============================================================
# 4. LOAD OLD A3 REFERENCE ARTIFACTS
# ============================================================

assert sha256_file(
    OLD_UNIVERSE_PATH
) == OLD_UNIVERSE_FILE_SHA


assert sha256_file(
    OLD_ORACLE_PATH
) == OLD_ORACLE_FILE_SHA


assert sha256_file(
    OLD_MISSING_PATH
) == OLD_MISSING_FILE_SHA


assert sha256_file(
    OLD_DETECTION_EDGE_PATH
) == OLD_DETECTION_EDGE_FILE_SHA


assert sha256_file(
    OLD_MATCHING_CEILING_PATH
) == OLD_MATCHING_CEILING_FILE_SHA


old_contract = json.loads(
    OLD_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


assert list(
    old_contract[
        "candidate_key"
    ]
) == OLD_KEY


old_universe = pd.read_pickle(
    OLD_UNIVERSE_PATH
)


old_oracle = pd.read_pickle(
    OLD_ORACLE_PATH
)


old_missing_full = pd.read_pickle(
    OLD_MISSING_PATH
)


old_detection_full = pd.read_pickle(
    OLD_DETECTION_EDGE_PATH
)


old_matching_ceiling = pd.read_pickle(
    OLD_MATCHING_CEILING_PATH
)


assert len(
    old_universe
) == EXPECTED_OLD_ROWS


assert len(
    old_oracle
) == EXPECTED_OLD_ROWS


assert len(
    old_missing_full
) == EXPECTED_MISSING_NODES


assert len(
    old_detection_full
) == EXPECTED_DETECTION_EDGES


assert not old_universe[
    OLD_KEY
].duplicated().any()


assert not old_oracle[
    OLD_KEY
].duplicated().any()


# ============================================================
# 5. RECOVER OLD A3 LOCAL candidate_id
# ============================================================

old_candidates = (
    old_universe
    .sort_values(
        OLD_KEY,
        kind="stable",
    )
    .reset_index(
        drop=True
    )
    .copy()
)


old_candidates[
    "candidate_id"
] = np.arange(
    len(
        old_candidates
    ),
    dtype=np.int64,
)


assert np.array_equal(
    old_candidates[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    old_oracle[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
)


# ============================================================
# 6. BASE GT-DERIVED MISSING-NODE POPULATION
#
# IMPORTANT:
#
# We reuse only GT identity / coordinate / component fields.
#
# OLD candidate-dependent nearest-distance and coverage fields
# are discarded and recomputed.
#
# G3H3 is edge-only. It adds/removes edges but does not add
# detected nodes, therefore this DETECTION_LIMITED missing-node
# denominator is invariant under G3H3.
# ============================================================

MISSING_BASE_COLUMNS = [
    "dataset",
    "fold",
    "prefix",
    "gt_node_id",
    "fn_incidence",
    "component_key",
    "t",
    "gt_z",
    "gt_y",
    "gt_x",
]


assert set(
    MISSING_BASE_COLUMNS
).issubset(
    old_missing_full.columns
)


missing_base = old_missing_full[
    MISSING_BASE_COLUMNS
].copy()


assert len(
    missing_base
) == EXPECTED_MISSING_NODES


assert not missing_base[
    [
        "dataset",
        "gt_node_id",
    ]
].duplicated().any()


assert set(
    missing_base[
        "fold"
    ].astype(
        int
    ).unique()
).issubset(
    {
        1,
        2,
        3,
        4,
    }
)


assert not (
    missing_base[
        "fold"
    ].astype(
        int
    )
    ==
    0
).any()


# ============================================================
# 7. BASE DETECTION-LIMITED EDGE POPULATION
#
# Old candidate-dependent coverage fields are discarded.
# ============================================================

DETECTION_BASE_COLUMNS = [
    "dataset",
    "fold",
    "prefix",
    "gt_source_id",
    "gt_target_id",
    "pred_source_id",
    "pred_target_id",
    "cause",
    "root_cause",
    "source_missing",
    "target_missing",
    "missing_class",
]


assert set(
    DETECTION_BASE_COLUMNS
).issubset(
    old_detection_full.columns
)


detection_base = old_detection_full[
    DETECTION_BASE_COLUMNS
].copy()


assert len(
    detection_base
) == EXPECTED_DETECTION_EDGES


assert not detection_base[
    [
        "dataset",
        "gt_source_id",
        "gt_target_id",
    ]
].duplicated().any()


assert (
    detection_base[
        "missing_class"
    ]
    .value_counts()
    .to_dict()
    ==
    EXPECTED_MISSING_CLASSES
)


assert set(
    detection_base[
        "fold"
    ].astype(
        int
    ).unique()
).issubset(
    {
        1,
        2,
        3,
        4,
    }
)


assert not (
    detection_base[
        "fold"
    ].astype(
        int
    )
    ==
    0
).any()


# ============================================================
# 8. EXACT A3-STYLE ORACLE FUNCTION
# ============================================================

def compute_radius_oracle(
    candidates,
    missing_base,
    *,
    candidate_id_column,
    radii=RADII,
    scale=SCALE,
):

    candidates = candidates.copy()

    missing = missing_base.copy()


    assert candidate_id_column in candidates.columns


    assert candidates[
        candidate_id_column
    ].is_unique


    assert not candidates[
        [
            "dataset",
            "t",
            "z",
            "y",
            "x",
        ]
    ].duplicated().any()


    assert not missing[
        [
            "dataset",
            "gt_node_id",
        ]
    ].duplicated().any()


    n_candidates = len(
        candidates
    )


    n_missing = len(
        missing
    )


    candidate_nearest = np.full(
        n_candidates,
        np.inf,
        dtype=np.float64,
    )


    missing_nearest = np.full(
        n_missing,
        np.inf,
        dtype=np.float64,
    )


    candidate_counts = {
        float(
            radius
        ):
            np.zeros(
                n_candidates,
                dtype=np.int64,
            )

        for radius
        in radii
    }


    missing_counts = {
        float(
            radius
        ):
            np.zeros(
                n_missing,
                dtype=np.int64,
            )

        for radius
        in radii
    }


    candidate_groups = (
        candidates
        .groupby(
            [
                "dataset",
                "t",
            ],
            sort=False,
        )
        .indices
    )


    missing_groups = (
        missing
        .groupby(
            [
                "dataset",
                "t",
            ],
            sort=False,
        )
        .indices
    )


    matching_rows = []


    all_frame_keys = sorted(
        set(
            missing_groups.keys()
        )
    )


    for frame_number, frame_key in enumerate(
        all_frame_keys,
        start=1,
    ):

        missing_idx = np.asarray(
            missing_groups[
                frame_key
            ],
            dtype=np.int64,
        )


        candidate_idx = np.asarray(
            candidate_groups.get(
                frame_key,
                [],
            ),
            dtype=np.int64,
        )


        missing_xyz = missing.iloc[
            missing_idx
        ][
            [
                "gt_z",
                "gt_y",
                "gt_x",
            ]
        ].to_numpy(
            dtype=np.float64,
        )


        missing_phys = (
            missing_xyz
            *
            scale[
                None,
                :
            ]
        )


        if len(
            candidate_idx
        ):

            candidate_xyz = candidates.iloc[
                candidate_idx
            ][
                [
                    "z",
                    "y",
                    "x",
                ]
            ].to_numpy(
                dtype=np.float64,
            )


            candidate_phys = (
                candidate_xyz
                *
                scale[
                    None,
                    :
                ]
            )


            missing_tree = cKDTree(
                missing_phys
            )


            candidate_tree = cKDTree(
                candidate_phys
            )


            candidate_nearest[
                candidate_idx
            ] = np.asarray(
                missing_tree.query(
                    candidate_phys,
                    k=1,
                )[
                    0
                ],
                dtype=np.float64,
            )


            missing_nearest[
                missing_idx
            ] = np.asarray(
                candidate_tree.query(
                    missing_phys,
                    k=1,
                )[
                    0
                ],
                dtype=np.float64,
            )


            for radius in radii:

                radius = float(
                    radius
                )


                candidate_counts[
                    radius
                ][
                    candidate_idx
                ] = ball_count(
                    missing_tree,
                    candidate_phys,
                    radius,
                )


                missing_counts[
                    radius
                ][
                    missing_idx
                ] = ball_count(
                    candidate_tree,
                    missing_phys,
                    radius,
                )


                # ============================================
                # Separate ONE-TO-ONE ceiling.
                #
                # This does NOT define oracle labels.
                # ============================================

                neighbors = candidate_tree.query_ball_point(
                    missing_phys,
                    r=radius,
                )


                row_ids = []

                column_ids = []


                for local_missing, local_candidates in enumerate(
                    neighbors
                ):

                    if not local_candidates:

                        continue


                    row_ids.extend(
                        [
                            local_missing
                        ]
                        *
                        len(
                            local_candidates
                        )
                    )


                    column_ids.extend(
                        [
                            int(
                                value
                            )

                            for value
                            in local_candidates
                        ]
                    )


                admissible_pairs = len(
                    row_ids
                )


                if admissible_pairs:

                    adjacency = csr_matrix(
                        (
                            np.ones(
                                admissible_pairs,
                                dtype=np.int8,
                            ),
                            (
                                np.asarray(
                                    row_ids,
                                    dtype=np.int64,
                                ),
                                np.asarray(
                                    column_ids,
                                    dtype=np.int64,
                                ),
                            ),
                        ),
                        shape=(
                            len(
                                missing_idx
                            ),
                            len(
                                candidate_idx
                            ),
                        ),
                    )


                    matching = maximum_bipartite_matching(
                        adjacency,
                        perm_type="column",
                    )


                    maximum_matches = int(
                        np.count_nonzero(
                            np.asarray(
                                matching
                            )
                            >=
                            0
                        )
                    )


                else:

                    maximum_matches = 0


                matching_rows.append(
                    {
                        "dataset":
                            str(
                                frame_key[
                                    0
                                ]
                            ),

                        "t":
                            int(
                                frame_key[
                                    1
                                ]
                            ),

                        "fold":
                            int(
                                missing.iloc[
                                    missing_idx[
                                        0
                                    ]
                                ][
                                    "fold"
                                ]
                            ),

                        "radius_um":
                            radius,

                        "missing_nodes":
                            int(
                                len(
                                    missing_idx
                                )
                            ),

                        "candidate_nodes":
                            int(
                                len(
                                    candidate_idx
                                )
                            ),

                        "admissible_pairs":
                            int(
                                admissible_pairs
                            ),

                        "maximum_one_to_one_matches":
                            int(
                                maximum_matches
                            ),
                    }
                )


        else:

            # ------------------------------------------------
            # Missing GT nodes but zero candidates in frame.
            # ------------------------------------------------

            for radius in radii:

                matching_rows.append(
                    {
                        "dataset":
                            str(
                                frame_key[
                                    0
                                ]
                            ),

                        "t":
                            int(
                                frame_key[
                                    1
                                ]
                            ),

                        "fold":
                            int(
                                missing.iloc[
                                    missing_idx[
                                        0
                                    ]
                                ][
                                    "fold"
                                ]
                            ),

                        "radius_um":
                            float(
                                radius
                            ),

                        "missing_nodes":
                            int(
                                len(
                                    missing_idx
                                )
                            ),

                        "candidate_nodes":
                            0,

                        "admissible_pairs":
                            0,

                        "maximum_one_to_one_matches":
                            0,
                    }
                )


        if (
            frame_number == 1
            or
            frame_number % 500 == 0
            or
            frame_number == len(
                all_frame_keys
            )
        ):

            print(
                "oracle frames:",
                f"{frame_number}/{len(all_frame_keys)}"
            )


    # ========================================================
    # CANDIDATE OUTPUT
    # ========================================================

    candidate_oracle = candidates.copy()


    candidate_oracle[
        "nearest_missing_distance_um"
    ] = candidate_nearest


    for radius in radii:

        radius = float(
            radius
        )


        tag = radius_tag(
            radius
        )


        candidate_oracle[
            f"missing_count_within_{tag}"
        ] = candidate_counts[
            radius
        ]


        candidate_oracle[
            f"oracle_hit_{tag}"
        ] = (
            candidate_counts[
                radius
            ]
            >
            0
        ).astype(
            np.int8
        )


    candidate_oracle[
        "oracle_hit_7um"
    ] = candidate_oracle[
        "oracle_hit_7p0um"
    ].astype(
        np.int8
    )


    # ========================================================
    # MISSING-NODE OUTPUT
    # ========================================================

    missing_oracle = missing.copy()


    missing_oracle[
        "nearest_candidate_distance_um"
    ] = missing_nearest


    for radius in radii:

        radius = float(
            radius
        )


        tag = radius_tag(
            radius
        )


        missing_oracle[
            f"candidate_count_within_{tag}"
        ] = missing_counts[
            radius
        ]


        missing_oracle[
            f"covered_any_{tag}"
        ] = (
            missing_counts[
                radius
            ]
            >
            0
        )


    matching_ceiling = pd.DataFrame(
        matching_rows
    )


    return (
        candidate_oracle,
        missing_oracle,
        matching_ceiling,
    )


# ============================================================
# 9. DETECTION-EDGE COVERAGE FUNCTION
# ============================================================

def compute_detection_edge_coverage(
    detection_base,
    missing_oracle,
    *,
    radii=RADII,
):

    result = detection_base.copy()


    missing_distance_map = (
        missing_oracle
        .set_index(
            [
                "dataset",
                "gt_node_id",
            ]
        )[
            "nearest_candidate_distance_um"
        ]
        .to_dict()
    )


    source_distances = []

    target_distances = []


    for row in result.itertuples(
        index=False
    ):

        source_missing = bool(
            row.source_missing
        )


        target_missing = bool(
            row.target_missing
        )


        if source_missing:

            key = (
                str(
                    row.dataset
                ),
                int(
                    row.gt_source_id
                ),
            )


            assert key in missing_distance_map


            source_distance = float(
                missing_distance_map[
                    key
                ]
            )


        else:

            source_distance = np.nan


        if target_missing:

            key = (
                str(
                    row.dataset
                ),
                int(
                    row.gt_target_id
                ),
            )


            assert key in missing_distance_map


            target_distance = float(
                missing_distance_map[
                    key
                ]
            )


        else:

            target_distance = np.nan


        source_distances.append(
            source_distance
        )


        target_distances.append(
            target_distance
        )


    result[
        "source_missing_nearest_candidate_um"
    ] = np.asarray(
        source_distances,
        dtype=np.float64,
    )


    result[
        "target_missing_nearest_candidate_um"
    ] = np.asarray(
        target_distances,
        dtype=np.float64,
    )


    source_missing_mask = result[
        "source_missing"
    ].astype(
        bool
    ).to_numpy()


    target_missing_mask = result[
        "target_missing"
    ].astype(
        bool
    ).to_numpy()


    source_distance = result[
        "source_missing_nearest_candidate_um"
    ].to_numpy(
        dtype=np.float64
    )


    target_distance = result[
        "target_missing_nearest_candidate_um"
    ].to_numpy(
        dtype=np.float64
    )


    for radius in radii:

        radius = float(
            radius
        )


        tag = radius_tag(
            radius
        )


        source_ok = (
            ~source_missing_mask
            |
            (
                source_distance
                <=
                radius
            )
        )


        target_ok = (
            ~target_missing_mask
            |
            (
                target_distance
                <=
                radius
            )
        )


        result[
            f"all_missing_endpoints_covered_{tag}"
        ] = (
            source_ok
            &
            target_ok
        )


    return result


# ============================================================
# 10. REPRODUCE OLD A3 WITH THIS IMPLEMENTATION
#
# THIS IS A HARD SCIENTIFIC SEMANTICS GATE.
# ============================================================

print(
    "\n========== OLD A3 IMPLEMENTATION REPRODUCTION =========="
)


(
    old_oracle_reproduced,
    old_missing_reproduced,
    old_matching_reproduced,
) = compute_radius_oracle(
    old_candidates,
    missing_base,
    candidate_id_column="candidate_id",
)


old_detection_reproduced = (
    compute_detection_edge_coverage(
        detection_base,
        old_missing_reproduced,
    )
)


# ============================================================
# 11. OLD CANDIDATE LABEL REPRODUCTION
# ============================================================

for radius in RADII:

    tag = radius_tag(
        radius
    )


    expected_column = (
        f"oracle_hit_{tag}"
    )


    assert expected_column in old_oracle.columns


    expected = pd.to_numeric(
        old_oracle[
            expected_column
        ],
        errors="raise",
    ).astype(
        np.int8
    ).to_numpy()


    reproduced = old_oracle_reproduced[
        expected_column
    ].astype(
        np.int8
    ).to_numpy()


    assert np.array_equal(
        reproduced,
        expected,
    ), (
        f"Old A3 candidate oracle mismatch at {radius} um."
    )


assert np.array_equal(
    old_oracle_reproduced[
        "oracle_hit_7um"
    ].to_numpy(
        dtype=np.int8
    ),
    old_oracle[
        "oracle_hit_7um"
    ].to_numpy(
        dtype=np.int8
    ),
)


assert int(
    old_oracle_reproduced[
        "oracle_hit_7um"
    ].sum()
) == EXPECTED_OLD_POSITIVES_7


# Nearest missing distances.
assert np.allclose(
    old_oracle_reproduced[
        "nearest_missing_distance_um"
    ].to_numpy(
        dtype=np.float64
    ),
    old_oracle[
        "nearest_missing_distance_um"
    ].to_numpy(
        dtype=np.float64
    ),
    rtol=0.0,
    atol=1e-12,
    equal_nan=True,
)


print(
    "candidate positives @7um:",
    int(
        old_oracle_reproduced[
            "oracle_hit_7um"
        ].sum()
    ),
    "expected:",
    EXPECTED_OLD_POSITIVES_7
)


print(
    "OLD_CANDIDATE_ORACLE_REPRODUCTION: PASS"
)


# ============================================================
# 12. OLD MISSING-NODE REPRODUCTION
# ============================================================

old_missing_reference = (
    old_missing_full
    .sort_values(
        [
            "dataset",
            "gt_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


old_missing_reproduced_check = (
    old_missing_reproduced
    .sort_values(
        [
            "dataset",
            "gt_node_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert np.allclose(
    old_missing_reproduced_check[
        "nearest_candidate_distance_um"
    ].to_numpy(
        dtype=np.float64
    ),
    old_missing_reference[
        "nearest_candidate_distance_um"
    ].to_numpy(
        dtype=np.float64
    ),
    rtol=0.0,
    atol=1e-12,
    equal_nan=True,
)


for radius in RADII:

    tag = radius_tag(
        radius
    )


    count_column = (
        f"candidate_count_within_{tag}"
    )


    covered_column = (
        f"covered_any_{tag}"
    )


    assert np.array_equal(
        old_missing_reproduced_check[
            count_column
        ].to_numpy(
            dtype=np.int64
        ),
        old_missing_reference[
            count_column
        ].to_numpy(
            dtype=np.int64
        ),
    )


    assert np.array_equal(
        old_missing_reproduced_check[
            covered_column
        ].astype(
            bool
        ).to_numpy(),
        old_missing_reference[
            covered_column
        ].astype(
            bool
        ).to_numpy(),
    )


assert int(
    old_missing_reproduced[
        "covered_any_7p0um"
    ].sum()
) == EXPECTED_OLD_NODE_COVERED_7


print(
    "nodes covered @7um:",
    int(
        old_missing_reproduced[
            "covered_any_7p0um"
        ].sum()
    ),
    "expected:",
    EXPECTED_OLD_NODE_COVERED_7
)


print(
    "OLD_MISSING_NODE_REPRODUCTION: PASS"
)


# ============================================================
# 13. OLD DETECTION-EDGE REPRODUCTION
# ============================================================

old_detection_reference = (
    old_detection_full
    .sort_values(
        [
            "dataset",
            "gt_source_id",
            "gt_target_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


old_detection_reproduced_check = (
    old_detection_reproduced
    .sort_values(
        [
            "dataset",
            "gt_source_id",
            "gt_target_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


for column in [
    "source_missing_nearest_candidate_um",
    "target_missing_nearest_candidate_um",
]:

    assert np.allclose(
        old_detection_reproduced_check[
            column
        ].to_numpy(
            dtype=np.float64
        ),
        old_detection_reference[
            column
        ].to_numpy(
            dtype=np.float64
        ),
        rtol=0.0,
        atol=1e-12,
        equal_nan=True,
    )


for radius in RADII:

    tag = radius_tag(
        radius
    )


    column = (
        f"all_missing_endpoints_covered_{tag}"
    )


    assert np.array_equal(
        old_detection_reproduced_check[
            column
        ].astype(
            bool
        ).to_numpy(),
        old_detection_reference[
            column
        ].astype(
            bool
        ).to_numpy(),
    )


assert int(
    old_detection_reproduced[
        "all_missing_endpoints_covered_7p0um"
    ].sum()
) == EXPECTED_OLD_EDGE_COVERED_7


print(
    "edges covered @7um:",
    int(
        old_detection_reproduced[
            "all_missing_endpoints_covered_7p0um"
        ].sum()
    ),
    "expected:",
    EXPECTED_OLD_EDGE_COVERED_7
)


print(
    "OLD_DETECTION_EDGE_REPRODUCTION: PASS"
)


# ============================================================
# 14. OLD ONE-TO-ONE CEILING REPRODUCTION
#
# This is NOT used by the pass/fail gate.
# It verifies that our separate matching-ceiling semantics
# agree with historical A3.
# ============================================================

OLD_MATCH_KEY = [
    "dataset",
    "t",
    "radius_um",
]


old_matching_reference = (
    old_matching_ceiling
    .sort_values(
        OLD_MATCH_KEY,
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


old_matching_reproduced_check = (
    old_matching_reproduced
    .sort_values(
        OLD_MATCH_KEY,
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    old_matching_reproduced_check
) == len(
    old_matching_reference
)


for column in [
    "dataset",
    "t",
    "fold",
    "radius_um",
    "missing_nodes",
    "candidate_nodes",
    "admissible_pairs",
    "maximum_one_to_one_matches",
]:

    if pd.api.types.is_numeric_dtype(
        old_matching_reference[
            column
        ]
    ):

        assert np.array_equal(
            pd.to_numeric(
                old_matching_reproduced_check[
                    column
                ],
                errors="raise",
            ).to_numpy(),
            pd.to_numeric(
                old_matching_reference[
                    column
                ],
                errors="raise",
            ).to_numpy(),
        )


    else:

        assert old_matching_reproduced_check[
            column
        ].astype(
            str
        ).equals(
            old_matching_reference[
                column
            ].astype(
                str
            )
        )


print(
    "OLD_ONE_TO_ONE_MATCHING_CEILING_REPRODUCTION: PASS"
)


print(
    "\nEXACT_OLD_A3_ORACLE_IMPLEMENTATION_REPRODUCTION: PASS"
)


# ============================================================
# 15. NOW APPLY EXACT SAME ORACLE TO NEW FROZEN UNIVERSE
# ============================================================

print(
    "\n========== NEW 12.13 CANDIDATE ORACLE =========="
)


(
    new_oracle,
    new_missing,
    new_matching_ceiling,
) = compute_radius_oracle(
    new_universe,
    missing_base,
    candidate_id_column="candidate_id",
)


new_detection_edges = (
    compute_detection_edge_coverage(
        detection_base,
        new_missing,
    )
)


# ============================================================
# 16. NEW ORACLE INVARIANTS
# ============================================================

assert len(
    new_oracle
) == EXPECTED_NEW_ROWS


assert new_oracle[
    "candidate_id"
].is_unique


assert np.array_equal(
    new_oracle[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    new_universe[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
)


assert len(
    new_missing
) == EXPECTED_MISSING_NODES


assert len(
    new_detection_edges
) == EXPECTED_DETECTION_EDGES


assert (
    new_detection_edges[
        "missing_class"
    ]
    .value_counts()
    .to_dict()
    ==
    EXPECTED_MISSING_CLASSES
)


assert set(
    new_oracle[
        "fold"
    ].astype(
        int
    ).unique()
).issubset(
    {
        1,
        2,
        3,
        4,
    }
)


assert set(
    new_missing[
        "fold"
    ].astype(
        int
    ).unique()
).issubset(
    {
        1,
        2,
        3,
        4,
    }
)


assert set(
    new_detection_edges[
        "fold"
    ].astype(
        int
    ).unique()
).issubset(
    {
        1,
        2,
        3,
        4,
    }
)


assert not (
    new_oracle[
        "fold"
    ].astype(
        int
    )
    ==
    0
).any()


# ============================================================
# 17. NEW COVERAGE SUMMARY
# ============================================================

coverage_rows = []


for radius in RADII:

    radius = float(
        radius
    )


    tag = radius_tag(
        radius
    )


    candidate_positive_count = int(
        new_oracle[
            f"oracle_hit_{tag}"
        ].sum()
    )


    node_covered_count = int(
        new_missing[
            f"covered_any_{tag}"
        ].sum()
    )


    edge_covered_count = int(
        new_detection_edges[
            f"all_missing_endpoints_covered_{tag}"
        ].sum()
    )


    node_coverage = (
        node_covered_count
        /
        EXPECTED_MISSING_NODES
    )


    edge_coverage = (
        edge_covered_count
        /
        EXPECTED_DETECTION_EDGES
    )


    candidate_positive_rate = (
        candidate_positive_count
        /
        len(
            new_oracle
        )
    )


    matching_at_radius = (
        new_matching_ceiling.loc[
            np.isclose(
                new_matching_ceiling[
                    "radius_um"
                ].to_numpy(
                    dtype=np.float64
                ),
                radius,
                rtol=0.0,
                atol=0.0,
            )
        ]
    )


    maximum_one_to_one_matches = int(
        matching_at_radius[
            "maximum_one_to_one_matches"
        ].sum()
    )


    coverage_rows.append(
        {
            "radius_um":
                radius,

            "candidate_positive_count":
                candidate_positive_count,

            "candidate_positive_rate":
                candidate_positive_rate,

            "missing_nodes":
                EXPECTED_MISSING_NODES,

            "covered_missing_nodes":
                node_covered_count,

            "node_coverage":
                node_coverage,

            "detection_limited_edges":
                EXPECTED_DETECTION_EDGES,

            "covered_detection_limited_edges":
                edge_covered_count,

            "edge_coverage":
                edge_coverage,

            "maximum_one_to_one_matches":
                maximum_one_to_one_matches,
        }
    )


coverage_summary = pd.DataFrame(
    coverage_rows
)


print(
    "\n========== NEW ORACLE CEILING BY RADIUS =========="
)


print(
    coverage_summary.to_string(
        index=False
    )
)


# ============================================================
# 18. EXACT A0 7UM DECISION
# ============================================================

row_7 = coverage_summary.loc[
    np.isclose(
        coverage_summary[
            "radius_um"
        ].to_numpy(
            dtype=np.float64
        ),
        7.0,
        rtol=0.0,
        atol=0.0,
    )
].iloc[
    0
]


NEW_CANDIDATE_POSITIVES_7 = int(
    row_7[
        "candidate_positive_count"
    ]
)


NEW_NODE_COVERED_7 = int(
    row_7[
        "covered_missing_nodes"
    ]
)


NEW_EDGE_COVERED_7 = int(
    row_7[
        "covered_detection_limited_edges"
    ]
)


NEW_NODE_COVERAGE_7 = float(
    row_7[
        "node_coverage"
    ]
)


NEW_EDGE_COVERAGE_7 = float(
    row_7[
        "edge_coverage"
    ]
)


NODE_GATE_PASS = bool(
    NEW_NODE_COVERAGE_7
    >=
    REQUIRED_NODE_COVERAGE_7
)


EDGE_GATE_PASS = bool(
    NEW_EDGE_COVERAGE_7
    >=
    REQUIRED_EDGE_COVERAGE_7
)


BOTH_GATE_PASS = bool(
    NODE_GATE_PASS
    and
    EDGE_GATE_PASS
)


DECISION = (
    "PASS_CEILING_GATE"
    if BOTH_GATE_PASS
    else
    "CLOSE_BRANCH_NO_RANKING_NO_GRAPH_MUTATION"
)


print(
    "\n========== A0 PREDECLARED 7UM GATE =========="
)


print(
    "new candidate positives:",
    NEW_CANDIDATE_POSITIVES_7
)


print(
    "new covered missing nodes:",
    NEW_NODE_COVERED_7,
    "/",
    EXPECTED_MISSING_NODES
)


print(
    "new node coverage:",
    NEW_NODE_COVERAGE_7
)


print(
    "required node coverage:",
    REQUIRED_NODE_COVERAGE_7
)


print(
    "NODE GATE:",
    (
        "PASS"
        if NODE_GATE_PASS
        else
        "FAIL"
    )
)


print()


print(
    "new covered detection-limited edges:",
    NEW_EDGE_COVERED_7,
    "/",
    EXPECTED_DETECTION_EDGES
)


print(
    "new edge coverage:",
    NEW_EDGE_COVERAGE_7
)


print(
    "required edge coverage:",
    REQUIRED_EDGE_COVERAGE_7
)


print(
    "EDGE GATE:",
    (
        "PASS"
        if EDGE_GATE_PASS
        else
        "FAIL"
    )
)


print()


print(
    "BOTH REQUIRED:",
    "YES"
)


print(
    "A2 DECISION:",
    DECISION
)


# ============================================================
# 19. ORACLE OUTPUT COLUMN ORDER
# ============================================================

ORACLE_METRIC_COLUMNS = [
    "nearest_missing_distance_um",

    "missing_count_within_2p0um",
    "oracle_hit_2p0um",

    "missing_count_within_3p0um",
    "oracle_hit_3p0um",

    "missing_count_within_5p0um",
    "oracle_hit_5p0um",

    "missing_count_within_7p0um",
    "oracle_hit_7p0um",

    "oracle_hit_7um",
]


NEW_ORACLE_COLUMNS = (
    list(
        new_universe.columns
    )
    +
    ORACLE_METRIC_COLUMNS
)


new_oracle = new_oracle[
    NEW_ORACLE_COLUMNS
].copy()


# ============================================================
# 20. SEMANTIC HASHES
# ============================================================

NEW_ORACLE_CONTENT_SHA = dataframe_content_sha256(
    new_oracle,
    NEW_ORACLE_COLUMNS,
)


NEW_MISSING_COLUMNS = list(
    new_missing.columns
)


NEW_MISSING_CONTENT_SHA = dataframe_content_sha256(
    new_missing,
    NEW_MISSING_COLUMNS,
)


NEW_DETECTION_COLUMNS = list(
    new_detection_edges.columns
)


NEW_DETECTION_CONTENT_SHA = dataframe_content_sha256(
    new_detection_edges,
    NEW_DETECTION_COLUMNS,
)


NEW_MATCHING_COLUMNS = [
    "dataset",
    "t",
    "fold",
    "radius_um",
    "missing_nodes",
    "candidate_nodes",
    "admissible_pairs",
    "maximum_one_to_one_matches",
]


new_matching_ceiling = (
    new_matching_ceiling[
        NEW_MATCHING_COLUMNS
    ]
    .sort_values(
        [
            "dataset",
            "t",
            "radius_um",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


NEW_MATCHING_CONTENT_SHA = dataframe_content_sha256(
    new_matching_ceiling,
    NEW_MATCHING_COLUMNS,
)


# ============================================================
# 21. FORMAL SUMMARY / DECISION
# ============================================================

SUMMARY = {
    "stage":
        "12.13A2",

    "status":
        "FORMAL_DEV_GT_ORACLE_CEILING_AUDIT_COMPLETE",

    "branch_id":
        BRANCH_ID,

    "candidate_universe_id":
        CANDIDATE_UNIVERSE_ID,

    "parent_a0_sha256":
        A0_SHA,

    "parent_candidate_universe_file_sha256":
        NEW_UNIVERSE_FILE_SHA,

    "parent_candidate_universe_semantic_sha256":
        NEW_UNIVERSE_SEMANTIC_SHA,

    "oracle_semantics": {
        "historical_source":
            "STAGE12.12A3_CELL77",

        "historical_semantics_reproduced_before_new_evaluation":
            True,

        "candidate_identity_old":
            OLD_KEY,

        "candidate_identity_new":
            (
                "FROZEN_STAGE12.13A1_R3_"
                "CANDIDATE_ID_PLUS_SEMANTIC_DATASET_T_Z_Y_X"
            ),

        "scale_zyx_um":
            SCALE.tolist(),

        "radii_um":
            list(
                RADII
            ),

        "candidate_label":
            "MANY_TO_ONE_RADIUS_MEMBERSHIP_GT_ZERO",

        "missing_node_coverage":
            "ANY_CANDIDATE_WITHIN_RADIUS",

        "detection_edge_coverage":
            "ALL_MISSING_ENDPOINTS_WITHIN_RADIUS",

        "one_to_one_matching":
            "SEPARATE_CEILING_SUMMARY_ONLY",
    },

    "reference_population": {
        "missing_gt_nodes":
            EXPECTED_MISSING_NODES,

        "detection_limited_edges":
            EXPECTED_DETECTION_EDGES,

        "missing_class_counts":
            EXPECTED_MISSING_CLASSES,

        "g3h3_invariance":
            (
                "G3H3_IS_EDGE_ONLY_AND_DOES_NOT_ADD_"
                "DETECTED_NODES_SO_DETECTION_LIMITED_"
                "MISSING_NODE_DENOMINATOR_IS_INVARIANT"
            ),
    },

    "historical_reproduction": {
        "old_candidate_rows":
            EXPECTED_OLD_ROWS,

        "oracle_hit_7um_positives":
            EXPECTED_OLD_POSITIVES_7,

        "covered_missing_nodes_7um":
            EXPECTED_OLD_NODE_COVERED_7,

        "covered_detection_edges_7um":
            EXPECTED_OLD_EDGE_COVERED_7,

        "pass":
            True,
    },

    "coverage_by_radius":
        coverage_summary.to_dict(
            orient="records"
        ),

    "gate_7um": {
        "required_node_coverage":
            REQUIRED_NODE_COVERAGE_7,

        "actual_node_coverage":
            NEW_NODE_COVERAGE_7,

        "node_gate_pass":
            NODE_GATE_PASS,

        "required_edge_coverage":
            REQUIRED_EDGE_COVERAGE_7,

        "actual_edge_coverage":
            NEW_EDGE_COVERAGE_7,

        "edge_gate_pass":
            EDGE_GATE_PASS,

        "both_required":
            True,

        "both_gate_pass":
            BOTH_GATE_PASS,
    },

    "decision":
        DECISION,

    "content_sha256": {
        "candidate_oracle":
            NEW_ORACLE_CONTENT_SHA,

        "missing_nodes":
            NEW_MISSING_CONTENT_SHA,

        "detection_edges":
            NEW_DETECTION_CONTENT_SHA,

        "matching_ceiling":
            NEW_MATCHING_CONTENT_SHA,
    },

    "protocol": {
        "candidate_universe_was_frozen_before_GT":
            True,

        "development_GT_oracle_used":
            True,

        "Fold0_used":
            False,

        "model_trained":
            False,

        "scorer_selected":
            False,

        "threshold_selected":
            False,

        "graph_metric_evaluated":
            False,

        "graph_mutated":
            False,

        "candidate_universe_modified":
            False,
    },

    "next":
        (
            "12.13A3_FREEZE_RANKING_AND_ATTACHMENT_CONTRACT"
            if BOTH_GATE_PASS
            else
            "BRANCH_CLOSED"
        ),
}


SUMMARY_SHA = hashlib.sha256(
    json.dumps(
        SUMMARY,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


SUMMARY[
    "summary_sha256"
] = SUMMARY_SHA


DECISION_RECORD = {
    "stage":
        "12.13A2",

    "branch_id":
        BRANCH_ID,

    "parent_contract_sha256":
        A0_SHA,

    "candidate_universe_file_sha256":
        NEW_UNIVERSE_FILE_SHA,

    "required_node_coverage_7um":
        REQUIRED_NODE_COVERAGE_7,

    "actual_node_coverage_7um":
        NEW_NODE_COVERAGE_7,

    "node_gate_pass":
        NODE_GATE_PASS,

    "required_edge_coverage_7um":
        REQUIRED_EDGE_COVERAGE_7,

    "actual_edge_coverage_7um":
        NEW_EDGE_COVERAGE_7,

    "edge_gate_pass":
        EDGE_GATE_PASS,

    "both_required":
        True,

    "both_gate_pass":
        BOTH_GATE_PASS,

    "decision":
        DECISION,

    "Fold0_used":
        False,

    "model_trained":
        False,

    "graph_mutated":
        False,
}


DECISION_SHA = hashlib.sha256(
    json.dumps(
        DECISION_RECORD,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


DECISION_RECORD[
    "decision_sha256"
] = DECISION_SHA


# ============================================================
# 22. TEMP FILES
# ============================================================

TMP_ORACLE = NEW_ORACLE_PATH.with_name(
    NEW_ORACLE_PATH.name
    +
    ".tmp_12_13a2"
)


TMP_MISSING = NEW_MISSING_PATH.with_name(
    NEW_MISSING_PATH.name
    +
    ".tmp_12_13a2"
)


TMP_DETECTION = NEW_DETECTION_EDGE_PATH.with_name(
    NEW_DETECTION_EDGE_PATH.name
    +
    ".tmp_12_13a2"
)


TMP_MATCHING = NEW_MATCHING_CEILING_PATH.with_name(
    NEW_MATCHING_CEILING_PATH.name
    +
    ".tmp_12_13a2"
)


TMP_SUMMARY = NEW_CEILING_SUMMARY_PATH.with_name(
    NEW_CEILING_SUMMARY_PATH.name
    +
    ".tmp_12_13a2"
)


TMP_DECISION = NEW_DECISION_PATH.with_name(
    NEW_DECISION_PATH.name
    +
    ".tmp_12_13a2"
)


for path in [
    TMP_ORACLE,
    TMP_MISSING,
    TMP_DETECTION,
    TMP_MATCHING,
    TMP_SUMMARY,
    TMP_DECISION,
]:

    if path.exists():

        raise RuntimeError(
            "\nStale A2 temporary artifact:\n"
            f"{path}"
        )


# ============================================================
# 23. TEMP WRITE
# ============================================================

new_oracle.to_pickle(
    TMP_ORACLE
)


new_missing.to_pickle(
    TMP_MISSING
)


new_detection_edges.to_pickle(
    TMP_DETECTION
)


new_matching_ceiling.to_pickle(
    TMP_MATCHING
)


TMP_SUMMARY.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


TMP_DECISION.write_text(
    json.dumps(
        DECISION_RECORD,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 24. TEMP RELOAD FIDELITY
# ============================================================

oracle_check = pd.read_pickle(
    TMP_ORACLE
)


missing_check = pd.read_pickle(
    TMP_MISSING
)


detection_check = pd.read_pickle(
    TMP_DETECTION
)


matching_check = pd.read_pickle(
    TMP_MATCHING
)


summary_check = json.loads(
    TMP_SUMMARY.read_text(
        encoding="utf-8"
    )
)


decision_check = json.loads(
    TMP_DECISION.read_text(
        encoding="utf-8"
    )
)


pd.testing.assert_frame_equal(
    oracle_check,
    new_oracle,
    check_exact=True,
)


pd.testing.assert_frame_equal(
    missing_check,
    new_missing,
    check_exact=True,
)


pd.testing.assert_frame_equal(
    detection_check,
    new_detection_edges,
    check_exact=True,
)


pd.testing.assert_frame_equal(
    matching_check,
    new_matching_ceiling,
    check_exact=True,
)


assert summary_check[
    "summary_sha256"
] == SUMMARY_SHA


assert decision_check[
    "decision_sha256"
] == DECISION_SHA


assert dataframe_content_sha256(
    oracle_check,
    NEW_ORACLE_COLUMNS,
) == NEW_ORACLE_CONTENT_SHA


assert dataframe_content_sha256(
    missing_check,
    NEW_MISSING_COLUMNS,
) == NEW_MISSING_CONTENT_SHA


assert dataframe_content_sha256(
    detection_check,
    NEW_DETECTION_COLUMNS,
) == NEW_DETECTION_CONTENT_SHA


assert dataframe_content_sha256(
    matching_check,
    NEW_MATCHING_COLUMNS,
) == NEW_MATCHING_CONTENT_SHA


# ============================================================
# 25. FILE HASHES
# ============================================================

NEW_ORACLE_FILE_SHA = sha256_file(
    TMP_ORACLE
)


NEW_MISSING_FILE_SHA = sha256_file(
    TMP_MISSING
)


NEW_DETECTION_FILE_SHA = sha256_file(
    TMP_DETECTION
)


NEW_MATCHING_FILE_SHA = sha256_file(
    TMP_MATCHING
)


NEW_CEILING_SUMMARY_FILE_SHA = sha256_file(
    TMP_SUMMARY
)


NEW_DECISION_FILE_SHA = sha256_file(
    TMP_DECISION
)


# ============================================================
# 26. ATOMIC PROMOTION
# ============================================================

TMP_ORACLE.replace(
    NEW_ORACLE_PATH
)


TMP_MISSING.replace(
    NEW_MISSING_PATH
)


TMP_DETECTION.replace(
    NEW_DETECTION_EDGE_PATH
)


TMP_MATCHING.replace(
    NEW_MATCHING_CEILING_PATH
)


TMP_SUMMARY.replace(
    NEW_CEILING_SUMMARY_PATH
)


TMP_DECISION.replace(
    NEW_DECISION_PATH
)


# ============================================================
# 27. OPTIONAL FORMAL BRANCH CLOSURE ON FAILURE
# ============================================================

BRANCH_CLOSURE_SHA = None


if not BOTH_GATE_PASS:

    CLOSURE = {
        "stage":
            "12.13A2",

        "branch_id":
            BRANCH_ID,

        "parent_contract_sha256":
            A0_SHA,

        "candidate_universe_file_sha256":
            NEW_UNIVERSE_FILE_SHA,

        "decision_sha256":
            DECISION_SHA,

        "decision":
            "CLOSE_BRANCH_NO_RANKING_NO_GRAPH_MUTATION",

        "reason":
            (
                "PREDECLARED_A0_ORACLE_CEILING_"
                "DUAL_GATE_NOT_BOTH_SATISFIED"
            ),

        "node_gate": {
            "required":
                REQUIRED_NODE_COVERAGE_7,

            "actual":
                NEW_NODE_COVERAGE_7,

            "pass":
                NODE_GATE_PASS,
        },

        "edge_gate": {
            "required":
                REQUIRED_EDGE_COVERAGE_7,

            "actual":
                NEW_EDGE_COVERAGE_7,

            "pass":
                EDGE_GATE_PASS,
        },

        "ranking_permitted":
            False,

        "model_training_permitted":
            False,

        "graph_mutation":
            False,

        "Fold0_used":
            False,
    }


    BRANCH_CLOSURE_SHA = hashlib.sha256(
        json.dumps(
            CLOSURE,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        ).encode(
            "utf-8"
        )
    ).hexdigest()


    CLOSURE[
        "closure_sha256"
    ] = BRANCH_CLOSURE_SHA


    BRANCH_CLOSURE_PATH.write_text(
        json.dumps(
            CLOSURE,
            indent=2,
        ),
        encoding="utf-8",
    )


# ============================================================
# 28. FINAL FILE FIDELITY
# ============================================================

assert sha256_file(
    NEW_ORACLE_PATH
) == NEW_ORACLE_FILE_SHA


assert sha256_file(
    NEW_MISSING_PATH
) == NEW_MISSING_FILE_SHA


assert sha256_file(
    NEW_DETECTION_EDGE_PATH
) == NEW_DETECTION_FILE_SHA


assert sha256_file(
    NEW_MATCHING_CEILING_PATH
) == NEW_MATCHING_FILE_SHA


assert sha256_file(
    NEW_CEILING_SUMMARY_PATH
) == NEW_CEILING_SUMMARY_FILE_SHA


assert sha256_file(
    NEW_DECISION_PATH
) == NEW_DECISION_FILE_SHA


# ============================================================
# 29. FINAL PROTOCOL GATES
# ============================================================

assert new_schema[
    "protocol"
][
    "candidate_universe_frozen_before_GT"
] is True


assert not (
    new_universe[
        "fold"
    ].astype(
        int
    )
    ==
    0
).any()


MODEL_TRAINED = False

THRESHOLD_SELECTED = False

GRAPH_MUTATED = False

FOLD0_USED = False


assert MODEL_TRAINED is False

assert THRESHOLD_SELECTED is False

assert GRAPH_MUTATED is False

assert FOLD0_USED is False


# ============================================================
# 30. FINAL
# ============================================================

print(
    "\n========== 12.13A2 FINAL =========="
)


print(
    "FORMAL_DEV_GT_ORACLE_CEILING_AUDIT: PASS"
)


print(
    "\n--- HISTORICAL SEMANTICS REPRODUCTION ---"
)


print(
    "old candidate positives @7um:",
    EXPECTED_OLD_POSITIVES_7
)


print(
    "old covered nodes @7um:",
    EXPECTED_OLD_NODE_COVERED_7
)


print(
    "old covered edges @7um:",
    EXPECTED_OLD_EDGE_COVERED_7
)


print(
    "EXACT_OLD_A3_ORACLE_IMPLEMENTATION_REPRODUCTION:",
    "PASS"
)


print(
    "\n--- NEW FROZEN UNIVERSE ---"
)


print(
    "candidate rows:",
    len(
        new_oracle
    )
)


print(
    "candidate positives @7um:",
    NEW_CANDIDATE_POSITIVES_7
)


print(
    "\n--- NODE CEILING ---"
)


print(
    "covered:",
    NEW_NODE_COVERED_7,
    "/",
    EXPECTED_MISSING_NODES
)


print(
    "coverage:",
    NEW_NODE_COVERAGE_7
)


print(
    "required:",
    REQUIRED_NODE_COVERAGE_7
)


print(
    "NODE_GATE:",
    (
        "PASS"
        if NODE_GATE_PASS
        else
        "FAIL"
    )
)


print(
    "\n--- EDGE CEILING ---"
)


print(
    "covered:",
    NEW_EDGE_COVERED_7,
    "/",
    EXPECTED_DETECTION_EDGES
)


print(
    "coverage:",
    NEW_EDGE_COVERAGE_7
)


print(
    "required:",
    REQUIRED_EDGE_COVERAGE_7
)


print(
    "EDGE_GATE:",
    (
        "PASS"
        if EDGE_GATE_PASS
        else
        "FAIL"
    )
)


print(
    "\n--- PREDECLARED A0 DECISION ---"
)


print(
    "BOTH REQUIRED:",
    "YES"
)


print(
    "BOTH PASS:",
    BOTH_GATE_PASS
)


print(
    "DECISION:",
    DECISION
)


print(
    "\n--- CONTENT SHAS ---"
)


print(
    "candidate oracle:",
    NEW_ORACLE_CONTENT_SHA
)


print(
    "missing nodes:",
    NEW_MISSING_CONTENT_SHA
)


print(
    "detection edges:",
    NEW_DETECTION_CONTENT_SHA
)


print(
    "matching ceiling:",
    NEW_MATCHING_CONTENT_SHA
)


print(
    "\n--- FILE SHAS ---"
)


print(
    "candidate oracle:",
    NEW_ORACLE_FILE_SHA
)


print(
    "missing nodes:",
    NEW_MISSING_FILE_SHA
)


print(
    "detection edges:",
    NEW_DETECTION_FILE_SHA
)


print(
    "matching ceiling:",
    NEW_MATCHING_FILE_SHA
)


print(
    "summary:",
    NEW_CEILING_SUMMARY_FILE_SHA
)


print(
    "decision:",
    NEW_DECISION_FILE_SHA
)


if BRANCH_CLOSURE_SHA is not None:

    print(
        "closure:",
        BRANCH_CLOSURE_SHA
    )


print(
    "\n--- PROTOCOL ---"
)


print(
    "candidate universe frozen before GT:",
    "YES"
)


print(
    "development GT oracle used:",
    "YES"
)


print(
    "candidate universe modified:",
    "NO"
)


print(
    "model trained:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "graph mutation:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "\nnext:"
)


if BOTH_GATE_PASS:

    print(
        "12.13A3_FREEZE_RANKING_AND_ATTACHMENT_CONTRACT"
    )


else:

    print(
        "BRANCH_CLOSED"
    )

In [ ]:
# ============================================================
# Stage 12.13A3-RB
#
# READBACK / PROVENANCE VERIFICATION
#
# Existing formal ranking + attachment contract
#
# ============================================================
#
# PURPOSE
#
# A3 rerun stopped because the formal contract already exists.
#
# DO NOT:
#
#   delete it
#   overwrite it
#   rename it
#   recreate it
#
# Instead verify:
#
#   - internal SHA
#   - A0 parent
#   - A1-R3 parent
#   - A2 parent artifacts
#   - A2 PASS_CEILING_GATE
#   - frozen ranking target
#   - allowed/forbidden feature rules
#   - scorer families
#   - nested LOFO
#   - threshold-selection rule
#   - attachment/conflict semantics
#   - graph-promotion gate
#   - Fold0 prohibition
#
# No model training.
# No GT recomputation.
# No graph mutation.
# No file write.
#
# ============================================================

from pathlib import Path
import hashlib
import json
import numpy as np


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


CONTRACT_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_ranking_attachment_contract_frozen.json"
)


A0_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_branch_frozen.json"
)


A1_UNIVERSE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_universe_v1.pkl"
)


A2_ORACLE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_oracle_candidates_v1.pkl"
)


A2_MISSING_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_oracle_missing_nodes_v1.pkl"
)


A2_DETECTION_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_oracle_detection_edges_v1.pkl"
)


A2_SUMMARY_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_oracle_ceiling_v1_summary.json"
)


A2_DECISION_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_oracle_ceiling_v1_decision_frozen.json"
)


for path in [
    CONTRACT_PATH,
    A0_PATH,
    A1_UNIVERSE_PATH,
    A2_ORACLE_PATH,
    A2_MISSING_PATH,
    A2_DETECTION_PATH,
    A2_SUMMARY_PATH,
    A2_DECISION_PATH,
]:
    assert path.exists(), path


# ============================================================
# 1. EXPECTED PARENT HASHES
# ============================================================

EXPECTED_A0_SHA = (
    "02ee83aa6554634c72c4a55f28e37d3083d8c311820a5a37caec99d3d0763e54"
)

EXPECTED_A1_UNIVERSE_FILE_SHA = (
    "440676497d3365a0926deaf45140f57332c8e9e89d46b88401a7ce6f7a299fc3"
)

EXPECTED_A1_UNIVERSE_SEMANTIC_SHA = (
    "856850a3c321538d0ed264892d7019c0869cc86d7f78a49baeb30f66c20e8565"
)

EXPECTED_A2_ORACLE_FILE_SHA = (
    "756758e0d9f6cf697a44548e7e7e5ad874d01d7d8f33d9f129b3f82b9cc56c80"
)

EXPECTED_A2_MISSING_FILE_SHA = (
    "0f7ccb8e709afed705e5edb67ea72f8631e05ac6b54b881f20a013619793e707"
)

EXPECTED_A2_DETECTION_FILE_SHA = (
    "f9313ba4503e115ca08f0da9af39c284dacf89ce3059e1b4f0b47ed422af366f"
)

EXPECTED_A2_SUMMARY_FILE_SHA = (
    "4967eca68924f7a48211f33403a16a38221bd8c15193bc085ebcd1768c9fbcf1"
)

EXPECTED_A2_DECISION_FILE_SHA = (
    "af2f95d05375259564bdf014f546b322466e823b2193a5eb7dd7623db4fc4c57"
)


BRANCH_ID = (
    "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_NODE_RECOVERY_V1"
)

CANDIDATE_UNIVERSE_ID = (
    "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_CANDIDATE_UNIVERSE_V1"
)


# ============================================================
# 2. HELPERS
# ============================================================

def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(1024 * 1024)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def recompute_embedded_hash(data, field):
    payload = dict(data)

    embedded = payload.pop(field)

    recomputed = hashlib.sha256(
        json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()

    return embedded, recomputed


# ============================================================
# 3. FILE HASH LOCKS
# ============================================================

assert sha256_file(
    A1_UNIVERSE_PATH
) == EXPECTED_A1_UNIVERSE_FILE_SHA


assert sha256_file(
    A2_ORACLE_PATH
) == EXPECTED_A2_ORACLE_FILE_SHA


assert sha256_file(
    A2_MISSING_PATH
) == EXPECTED_A2_MISSING_FILE_SHA


assert sha256_file(
    A2_DETECTION_PATH
) == EXPECTED_A2_DETECTION_FILE_SHA


assert sha256_file(
    A2_SUMMARY_PATH
) == EXPECTED_A2_SUMMARY_FILE_SHA


assert sha256_file(
    A2_DECISION_PATH
) == EXPECTED_A2_DECISION_FILE_SHA


print(
    "========== 12.13A3-RB PARENT FILE LOCK =========="
)

print(
    "A1 universe:",
    "PASS"
)

print(
    "A2 oracle:",
    "PASS"
)

print(
    "A2 missing nodes:",
    "PASS"
)

print(
    "A2 detection edges:",
    "PASS"
)

print(
    "A2 summary:",
    "PASS"
)

print(
    "A2 decision:",
    "PASS"
)


# ============================================================
# 4. LOAD FORMAL CONTRACT
# ============================================================

contract = json.loads(
    CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


assert isinstance(
    contract,
    dict
)


assert (
    "contract_sha256"
    in contract
)


embedded_sha, recomputed_sha = recompute_embedded_hash(
    contract,
    "contract_sha256",
)


assert (
    embedded_sha
    ==
    recomputed_sha
), (
    "\nA3 contract internal SHA mismatch.\n"
    f"embedded={embedded_sha}\n"
    f"recomputed={recomputed_sha}"
)


CONTRACT_FILE_SHA = sha256_file(
    CONTRACT_PATH
)


print(
    "\n========== A3 CONTRACT INTEGRITY =========="
)


print(
    "path:",
    CONTRACT_PATH
)


print(
    "contract content SHA:",
    embedded_sha
)


print(
    "recomputed content SHA:",
    recomputed_sha
)


print(
    "contract file SHA:",
    CONTRACT_FILE_SHA
)


print(
    "A3_CONTRACT_INTERNAL_SHA: PASS"
)


# ============================================================
# 5. BASIC IDENTITY
# ============================================================

assert contract[
    "stage"
] == "12.13A3"


assert contract[
    "status"
] == "RANKING_AND_ATTACHMENT_DEVELOPMENT_CONTRACT_FROZEN"


assert contract[
    "branch_id"
] == BRANCH_ID


assert contract[
    "candidate_universe_id"
] == CANDIDATE_UNIVERSE_ID


print(
    "\n========== A3 IDENTITY =========="
)


print(
    "stage:",
    contract[
        "stage"
    ]
)


print(
    "status:",
    contract[
        "status"
    ]
)


print(
    "branch:",
    contract[
        "branch_id"
    ]
)


print(
    "A3_IDENTITY_LOCK: PASS"
)


# ============================================================
# 6. PARENT LINKAGE
# ============================================================

parents = contract[
    "parents"
]


assert parents[
    "A0_contract_sha256"
] == EXPECTED_A0_SHA


assert parents[
    "A1_R3_candidate_universe_file_sha256"
] == EXPECTED_A1_UNIVERSE_FILE_SHA


assert parents[
    "A1_R3_candidate_universe_semantic_sha256"
] == EXPECTED_A1_UNIVERSE_SEMANTIC_SHA


assert parents[
    "A2_candidate_oracle_file_sha256"
] == EXPECTED_A2_ORACLE_FILE_SHA


assert parents[
    "A2_missing_nodes_file_sha256"
] == EXPECTED_A2_MISSING_FILE_SHA


assert parents[
    "A2_detection_edges_file_sha256"
] == EXPECTED_A2_DETECTION_FILE_SHA


assert parents[
    "A2_summary_file_sha256"
] == EXPECTED_A2_SUMMARY_FILE_SHA


assert parents[
    "A2_decision_file_sha256"
] == EXPECTED_A2_DECISION_FILE_SHA


assert parents[
    "A2_decision"
] == "PASS_CEILING_GATE"


print(
    "\n========== A3 PARENT LINKAGE =========="
)


print(
    "A0:",
    "PASS"
)


print(
    "A1-R3:",
    "PASS"
)


print(
    "A2:",
    "PASS_CEILING_GATE"
)


print(
    "A3_PARENT_LINKAGE: PASS"
)


# ============================================================
# 7. A2 CEILING LOCK
# ============================================================

ceiling = contract[
    "A2_ceiling"
]


assert int(
    ceiling[
        "candidate_rows"
    ]
) == 1_079_885


assert int(
    ceiling[
        "positive_candidates_7um"
    ]
) == 3_306


assert int(
    ceiling[
        "missing_nodes"
    ]
) == 2_985


assert int(
    ceiling[
        "covered_missing_nodes_7um"
    ]
) == 1_410


assert np.isclose(
    float(
        ceiling[
            "node_coverage_7um"
        ]
    ),
    0.4723618090452261,
    rtol=0.0,
    atol=1e-16,
)


assert int(
    ceiling[
        "detection_limited_edges"
    ]
) == 4_097


assert int(
    ceiling[
        "covered_detection_limited_edges_7um"
    ]
) == 1_919


assert np.isclose(
    float(
        ceiling[
            "edge_coverage_7um"
        ]
    ),
    0.46839150597998536,
    rtol=0.0,
    atol=1e-16,
)


assert ceiling[
    "both_gate_pass"
] is True


print(
    "\n========== A3 A2-CEILING LINK =========="
)


print(
    "positive candidates:",
    ceiling[
        "positive_candidates_7um"
    ]
)


print(
    "node coverage:",
    ceiling[
        "node_coverage_7um"
    ]
)


print(
    "edge coverage:",
    ceiling[
        "edge_coverage_7um"
    ]
)


print(
    "both gate pass:",
    ceiling[
        "both_gate_pass"
    ]
)


print(
    "A3_A2_CEILING_LOCK: PASS"
)


# ============================================================
# 8. RANKING TARGET
# ============================================================

target = contract[
    "ranking_target"
]


assert target[
    "column"
] == "oracle_hit_7um"


assert target[
    "type"
] == "BINARY"


assert target[
    "candidate_label_semantics"
] == "MANY_TO_ONE_RADIUS_MEMBERSHIP"


assert int(
    target[
        "positive_count"
    ]
) == 3_306


assert int(
    target[
        "total_count"
    ]
) == 1_079_885


assert target[
    "use_2_3_5um_as_training_targets"
] is False


assert target[
    "use_distance_as_regression_target"
] is False


assert target[
    "use_one_to_one_matching_as_label"
] is False


print(
    "\n========== RANKING TARGET =========="
)


print(
    "target:",
    target[
        "column"
    ]
)


print(
    "positive count:",
    target[
        "positive_count"
    ]
)


print(
    "RANKING_TARGET_LOCK: PASS"
)


# ============================================================
# 9. FEATURE CONTRACT
# ============================================================

features = contract[
    "features"
]


allowed = list(
    features[
        "allowed"
    ]
)


for forbidden_name in [
    "candidate_id",
    "dataset",
    "fold",
    "t",
    "z",
    "y",
    "x",
    "detector_prob",
    "detector_logit",
    "oracle_hit_7um",
    "nearest_missing_distance_um",
]:

    assert forbidden_name not in allowed


assert features[
    "absolute_coordinates_as_features"
] is False


assert features[
    "detector_evidence"
] is False


assert features[
    "GT_oracle_features"
] is False


assert features[
    "dataset_identity_feature"
] is False


assert features[
    "fold_identity_feature"
] is False


assert len(
    allowed
) == len(
    set(
        allowed
    )
)


print(
    "\n========== FEATURE CONTRACT =========="
)


print(
    "allowed features:",
    len(
        allowed
    )
)


print(
    "detector evidence:",
    "NO"
)


print(
    "GT/oracle features:",
    "NO"
)


print(
    "absolute coordinates:",
    "NO"
)


print(
    "dataset/fold identity:",
    "NO"
)


print(
    "FEATURE_CONTRACT_LOCK: PASS"
)


# ============================================================
# 10. SCORERS
# ============================================================

scorers = contract[
    "scorers"
]


assert set(
    scorers.keys()
) == {
    "LOGREG_L2_BALANCED_V1",
    "HGB_BALANCED_V1",
}


assert float(
    scorers[
        "LOGREG_L2_BALANCED_V1"
    ][
        "parameters"
    ][
        "C"
    ]
) == 1.0


assert scorers[
    "LOGREG_L2_BALANCED_V1"
][
    "parameters"
][
    "class_weight"
] == "balanced"


assert float(
    scorers[
        "HGB_BALANCED_V1"
    ][
        "parameters"
    ][
        "learning_rate"
    ]
) == 0.05


assert int(
    scorers[
        "HGB_BALANCED_V1"
    ][
        "parameters"
    ][
        "max_iter"
    ]
) == 200


print(
    "\n========== SCORER CONTRACT =========="
)


print(
    "scorers:",
    list(
        scorers.keys()
    )
)


print(
    "SCORER_CONTRACT_LOCK: PASS"
)


# ============================================================
# 11. NESTED LOFO
# ============================================================

lofo = contract[
    "nested_LOFO"
]


assert lofo[
    "outer_folds"
] == [
    1,
    2,
    3,
    4,
]


assert lofo[
    "for_each_outer_fold"
][
    "outer_fold_labels_used_for_model_selection"
] is False


assert lofo[
    "for_each_outer_fold"
][
    "outer_fold_labels_used_for_threshold_selection"
] is False


assert lofo[
    "resampling"
] == "NONE"


assert lofo[
    "candidate_universe"
] == "EXACT_FROZEN_A1_R3_ONLY"


print(
    "\n========== NESTED LOFO =========="
)


print(
    "outer folds:",
    lofo[
        "outer_folds"
    ]
)


print(
    "outer labels used for selection:",
    "NO"
)


print(
    "NESTED_LOFO_LOCK: PASS"
)


# ============================================================
# 12. THRESHOLD CONTRACT
# ============================================================

threshold = contract[
    "threshold_selection"
]


assert threshold[
    "objective"
] == "MAXIMIZE_F_BETA"


assert np.isclose(
    float(
        threshold[
            "beta"
        ]
    ),
    0.5,
    rtol=0.0,
    atol=0.0,
)


assert threshold[
    "held_out_outer_fold_used"
] is False


assert threshold[
    "manual_threshold_adjustment"
] is False


assert threshold[
    "post_result_threshold_retuning"
] is False


print(
    "\n========== THRESHOLD CONTRACT =========="
)


print(
    "objective:",
    "F0.5"
)


print(
    "outer labels used:",
    "NO"
)


print(
    "manual retuning:",
    "NO"
)


print(
    "THRESHOLD_CONTRACT_LOCK: PASS"
)


# ============================================================
# 13. ATTACHMENT CONTRACT
# ============================================================

attachment = contract[
    "attachment_conflicts"
]


assert attachment[
    "source_slot_requirement"
] == "CURRENT_OUTDEGREE_EQUALS_ZERO"


assert attachment[
    "target_slot_requirement"
] == "CURRENT_INDEGREE_EQUALS_ZERO"


assert attachment[
    "require_both_sides"
] is False


assert int(
    attachment[
        "minimum_edges_required"
    ]
) == 1


assert attachment[
    "existing_edge_removal"
] == "FORBIDDEN"


assert attachment[
    "existing_node_modification"
] == "FORBIDDEN"


assert attachment[
    "candidate_coordinate_change"
] == "FORBIDDEN"


print(
    "\n========== ATTACHMENT CONTRACT =========="
)


print(
    "source slot:",
    "outdegree == 0"
)


print(
    "target slot:",
    "indegree == 0"
)


print(
    "require both sides:",
    "NO"
)


print(
    "existing edge removal:",
    "FORBIDDEN"
)


print(
    "ATTACHMENT_CONTRACT_LOCK: PASS"
)


# ============================================================
# 14. MUTATION CONTRACT
# ============================================================

mutation = contract[
    "mutation"
]


assert mutation[
    "existing_edge_removal"
] is False


assert mutation[
    "existing_node_removal"
] is False


assert mutation[
    "existing_node_coordinate_change"
] is False


assert float(
    mutation[
        "edge_prob"
    ]
) == 1.0


assert mutation[
    "node_coordinate"
] == "EXACT_FROZEN_CANDIDATE_T_Z_Y_X"


print(
    "\n========== MUTATION CONTRACT =========="
)


print(
    "new synthetic node:",
    "YES"
)


print(
    "existing edge removal:",
    "NO"
)


print(
    "existing node modification:",
    "NO"
)


print(
    "MUTATION_CONTRACT_LOCK: PASS"
)


# ============================================================
# 15. RANKING GATE
# ============================================================

ranking_gate = contract[
    "ranking_gate"
]


assert ranking_gate[
    "primary_metric"
] == "AVERAGE_PRECISION"


assert ranking_gate[
    "outer_summary"
] == "MACRO_MEAN_OVER_FOLDS_1_TO_4"


assert ranking_gate[
    "if_fail"
] == "CLOSE_BRANCH_NO_GRAPH_MUTATION"


assert ranking_gate[
    "manual_model_rescue"
] is False


assert ranking_gate[
    "new_model_family_after_results"
] is False


assert ranking_gate[
    "hyperparameter_tuning_after_results"
] is False


print(
    "\n========== RANKING GATE =========="
)


for condition in ranking_gate[
    "required_conditions"
]:

    print(
        "-",
        condition
    )


print(
    "RANKING_GATE_LOCK: PASS"
)


# ============================================================
# 16. GRAPH PROMOTION GATE
# ============================================================

graph_gate = contract[
    "graph_promotion_gate"
]


assert graph_gate[
    "evaluation"
] == "OUTER_LOFO_HELD_OUT_FOLDS_1_TO_4"


assert graph_gate[
    "baseline"
] == "EXACT_POST_G3H3_CURRENT_BEST"


assert graph_gate[
    "all_conditions_required"
] is True


assert graph_gate[
    "threshold_retuning_after_graph_results"
] is False


assert graph_gate[
    "attachment_retuning_after_graph_results"
] is False


assert graph_gate[
    "feature_retuning_after_graph_results"
] is False


assert graph_gate[
    "model_family_expansion_after_graph_results"
] is False


print(
    "\n========== GRAPH PROMOTION GATE =========="
)


for condition in graph_gate[
    "required_conditions"
]:

    print(
        "-",
        condition
    )


print(
    "all conditions required:",
    "YES"
)


print(
    "GRAPH_PROMOTION_GATE_LOCK: PASS"
)


# ============================================================
# 17. FRESH CONFIRMATION
# ============================================================

confirmation = contract[
    "fresh_confirmation"
]


assert confirmation[
    "Fold0_available"
] is False


assert confirmation[
    "reuse_Fold0"
] == "FORBIDDEN"


assert confirmation[
    "fresh_confirmation_required"
] is True


print(
    "\n========== CONFIRMATION CONTRACT =========="
)


print(
    "Fold0 available:",
    "NO"
)


print(
    "Fold0 reuse:",
    "FORBIDDEN"
)


print(
    "fresh untouched confirmation required:",
    "YES"
)


print(
    "CONFIRMATION_CONTRACT_LOCK: PASS"
)


# ============================================================
# 18. PROTOCOL
# ============================================================

protocol = contract[
    "protocol"
]


assert protocol[
    "development_folds"
] == [
    1,
    2,
    3,
    4,
]


assert protocol[
    "Fold0_used"
] is False


assert protocol[
    "candidate_universe_modified"
] is False


assert protocol[
    "model_trained"
] is False


assert protocol[
    "model_selected"
] is False


assert protocol[
    "threshold_selected"
] is False


assert protocol[
    "graph_mutated"
] is False


assert protocol[
    "ranking_contract_frozen_before_training"
] is True


assert protocol[
    "attachment_contract_frozen_before_training"
] is True


assert contract[
    "next"
] == "12.13A4_GT_BLIND_FEATURE_MATRIX_MATERIALIZATION"


# ============================================================
# 19. FINAL
# ============================================================

print(
    "\n========== 12.13A3-RB FINAL =========="
)


print(
    "EXISTING_A3_FORMAL_CONTRACT_READBACK: PASS"
)


print(
    "formal A3 exists:",
    "YES"
)


print(
    "internal content SHA:",
    embedded_sha
)


print(
    "file SHA:",
    CONTRACT_FILE_SHA
)


print(
    "A0 parent:",
    "PASS"
)


print(
    "A1-R3 parent:",
    "PASS"
)


print(
    "A2 PASS_CEILING_GATE:",
    "PASS"
)


print(
    "ranking target:",
    "oracle_hit_7um"
)


print(
    "allowed features:",
    len(
        allowed
    )
)


print(
    "scorers:",
    list(
        scorers.keys()
    )
)


print(
    "nested LOFO:",
    "LOCKED"
)


print(
    "threshold:",
    "INNER OOF F0.5"
)


print(
    "attachment:",
    "LOCKED"
)


print(
    "graph promotion:",
    "LOCKED"
)


print(
    "Fold0:",
    "FORBIDDEN"
)


print(
    "model trained:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "graph mutated:",
    "NO"
)


print(
    "\nA3 STATUS:"
)


print(
    "ALREADY_FORMALLY_FROZEN"
)


print(
    "\nnext:"
)


print(
    "12.13A4_GT_BLIND_FEATURE_MATRIX_MATERIALIZATION"
)

In [ ]:
# ============================================================
# Stage 12.13A4-P0-R2
#
# TEMPORARY READ-ONLY PREFLIGHT
#
# EXACT OLD F1 FEATURE-CONSTRUCTION BLOCK
# + HELPER DEPENDENCY RECOVERY
#
# ============================================================
#
# REPAIRS:
#
#   12.13A4-P0-R1
#
# R1 successfully proved:
#
#   exact producer cell = 80
#   producer SHA =
#     46f26e931c97d6bcc60e83ad55c884f883bf7af962935fac3839a5463d6822f6
#
# R1 then failed because it assumed the 23 final feature names
# must occur inside top-level FunctionDef source.
#
# That assumption is unnecessary.
#
# Historical cell 80 separates:
#
#   low-level geometry helper(s)
#        e.g. query_frame(...)
#
# from:
#
#   top-level candidate / dataset loop
#        where final 23 feature values are assembled.
#
# ------------------------------------------------------------
# R2 SOURCE-LOCK STRATEGY
# ------------------------------------------------------------
#
# Lock BOTH:
#
#   A. exact top-level feature-construction statement/block
#   B. all top-level helper functions transitively called by it
#
# No historical source is reimplemented here.
#
# ------------------------------------------------------------
# SAFETY
# ------------------------------------------------------------
#
# NEW features generated : NO
# NEW GT read            : NO
# model trained          : NO
# threshold selected     : NO
# graph mutation         : NO
# Fold0                  : NO
#
# No files written.
#
# ============================================================


from pathlib import Path

import ast
import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)

NOTEBOOK_PATH = (
    PROJECT
    / "notebooks"
    / "12_residual_edge_recovery.ipynb"
)

A3_CONTRACT_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_ranking_attachment_contract_frozen.json"
)

OLD_F1_PATH = (
    S12
    / "stage12_post_r1p50_node_f1_temporal_geometry_v1.pkl"
)

NEW_UNIVERSE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_universe_v1.pkl"
)


for path in [
    NOTEBOOK_PATH,
    A3_CONTRACT_PATH,
    OLD_F1_PATH,
    NEW_UNIVERSE_PATH,
]:
    assert path.exists(), path


# ============================================================
# 1. FORMAL LOCKS
# ============================================================

EXPECTED_A3_CONTENT_SHA = (
    "c54eda46073582a7a915f1bfa7dcbff9"
    "eec8a96edcfad80a8e49e1097cf38b46"
)

EXPECTED_A3_FILE_SHA = (
    "37a5b2e557832774c0bcf65a33c81d8"
    "434bac2025c4eebbf2e3b0ea1bc955bd7"
)

EXPECTED_OLD_F1_FILE_SHA = (
    "8c3962bf6cef48617e1bd9ef23bd5e6d"
    "eb2438b80b6916f87d171516be92d0cc"
)

EXPECTED_PRODUCER_CELL = 80

EXPECTED_PRODUCER_SHA = (
    "46f26e931c97d6bcc60e83ad55c884f8"
    "83bf7af962935fac3839a5463d6822f6"
)

EXPECTED_QUERY_FRAME_SHA = (
    "16e14a76082495487b73b574f680f077"
    "3484c4802300ab4312d9b029813db205"
)

EXPECTED_OLD_ROWS = 415_590

EXPECTED_NEW_ROWS = 1_079_885


TEMPORAL_FEATURES = [
    "has_prev_frame",
    "has_next_frame",

    "same_nearest_dist_um",
    "same_second_dist_um",
    "same_neighbor_margin_um",

    "prev_nearest_dist_um",
    "prev_second_dist_um",
    "prev_neighbor_margin_um",

    "next_nearest_dist_um",
    "next_second_dist_um",
    "next_neighbor_margin_um",

    "prev_nearest_outdegree",
    "next_nearest_indegree",

    "prev_nearest_free_out",
    "next_nearest_free_in",

    "bilateral_available",

    "prev_next_span_um",
    "candidate_midpoint_residual_um",

    "prev_next_step_mean_um",
    "prev_next_step_max_um",
    "prev_next_step_min_um",
    "prev_next_step_absdiff_um",

    "step_balance",
]

assert len(TEMPORAL_FEATURES) == 23


# ============================================================
# 2. HELPERS
# ============================================================

def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(1024 * 1024)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def source_segment(source, node):
    return (
        ast.get_source_segment(
            source,
            node,
        )
        or ast.unparse(node)
    )


def cell_source(cell):
    return "".join(
        cell.get(
            "source",
            [],
        )
    )


def source_sha(source):
    return hashlib.sha256(
        source.encode("utf-8")
    ).hexdigest()


def feature_hits(source):
    return [
        name
        for name in TEMPORAL_FEATURES
        if name in source
    ]


def called_simple_names(node):
    """
    Return direct simple function names used in Call nodes.

    Examples:
        query_frame(...)
        float(...)
        np.asarray(...) -> attribute call, not included here
    """
    result = set()

    for child in ast.walk(node):
        if not isinstance(child, ast.Call):
            continue

        if isinstance(child.func, ast.Name):
            result.add(
                child.func.id
            )

    return result


# ============================================================
# 3. A3 CONTRACT LOCK
# ============================================================

assert sha256_file(
    A3_CONTRACT_PATH
) == EXPECTED_A3_FILE_SHA


a3 = json.loads(
    A3_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


embedded_a3_sha = a3[
    "contract_sha256"
]


payload = dict(a3)

payload.pop(
    "contract_sha256"
)


recomputed_a3_sha = hashlib.sha256(
    json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()


assert embedded_a3_sha == EXPECTED_A3_CONTENT_SHA

assert recomputed_a3_sha == EXPECTED_A3_CONTENT_SHA


assert a3[
    "features"
][
    "reused_temporal_geometry_semantics"
] == TEMPORAL_FEATURES


assert a3[
    "protocol"
][
    "model_trained"
] is False


assert a3[
    "protocol"
][
    "Fold0_used"
] is False


print(
    "========== 12.13A4-P0-R2 A3 LOCK =========="
)

print(
    "A3 SHA:",
    embedded_a3_sha
)

print(
    "temporal features:",
    len(TEMPORAL_FEATURES)
)

print(
    "A3_TEMPORAL_CONTRACT_LOCK: PASS"
)


# ============================================================
# 4. ARTIFACT LOCKS
# ============================================================

assert sha256_file(
    OLD_F1_PATH
) == EXPECTED_OLD_F1_FILE_SHA


old_f1 = pd.read_pickle(
    OLD_F1_PATH
)

new_universe = pd.read_pickle(
    NEW_UNIVERSE_PATH
)


assert len(old_f1) == EXPECTED_OLD_ROWS

assert len(new_universe) == EXPECTED_NEW_ROWS


assert set(
    TEMPORAL_FEATURES
).issubset(
    old_f1.columns
)


for col in new_universe.columns:
    lower = str(col).lower()

    assert "oracle" not in lower
    assert not lower.startswith("gt_")


print(
    "\n========== ARTIFACT LOCK =========="
)

print(
    "old F1 rows:",
    len(old_f1)
)

print(
    "old F1 SHA:",
    sha256_file(
        OLD_F1_PATH
    )
)

print(
    "new universe rows:",
    len(new_universe)
)

print(
    "ARTIFACT_LOCK: PASS"
)


# ============================================================
# 5. EXACT PRODUCER CELL LOCK
# ============================================================

notebook = json.loads(
    NOTEBOOK_PATH.read_text(
        encoding="utf-8"
    )
)

cells = notebook[
    "cells"
]


assert (
    0
    <=
    EXPECTED_PRODUCER_CELL
    <
    len(cells)
)


producer_cell = cells[
    EXPECTED_PRODUCER_CELL
]


assert producer_cell.get(
    "cell_type"
) == "code"


producer_source = cell_source(
    producer_cell
)


producer_sha = source_sha(
    producer_source
)


assert producer_sha == EXPECTED_PRODUCER_SHA


assert all(
    name in producer_source
    for name in TEMPORAL_FEATURES
)


assert "cKDTree" in producer_source

assert "in_degree" in producer_source

assert "out_degree" in producer_source


print(
    "\n========== EXACT PRODUCER LOCK =========="
)

print(
    "cell:",
    EXPECTED_PRODUCER_CELL
)

print(
    "SHA:",
    producer_sha
)

print(
    "23/23 feature tokens:",
    "YES"
)

print(
    "OLD_F1_EXACT_PRODUCER_LOCK: PASS"
)


# ============================================================
# 6. PARSE PRODUCER
# ============================================================

tree = ast.parse(
    producer_source
)


# ============================================================
# 7. TOP-LEVEL FUNCTION REGISTRY
# ============================================================

function_map = {}


for node in tree.body:
    if not isinstance(
        node,
        ast.FunctionDef,
    ):
        continue

    fn_source = source_segment(
        producer_source,
        node,
    )

    function_map[
        node.name
    ] = {
        "node":
            node,

        "source":
            fn_source,

        "sha256":
            source_sha(
                fn_source
            ),

        "lineno":
            int(node.lineno),

        "end_lineno":
            int(node.end_lineno),

        "calls":
            called_simple_names(
                node
            ),
    }


assert "query_frame" in function_map


assert function_map[
    "query_frame"
][
    "sha256"
] == EXPECTED_QUERY_FRAME_SHA


print(
    "\n========== FUNCTION REGISTRY =========="
)


for name, record in function_map.items():
    print(
        name,
        "| lines",
        (
            record["lineno"],
            record["end_lineno"],
        ),
        "| SHA",
        record["sha256"],
    )


print(
    "\nQUERY_FRAME_LOCK: PASS"
)


# ============================================================
# 8. LOCATE EVERY AST NODE CONTAINING TEMPORAL FEATURE TOKENS
# ============================================================

feature_nodes = []


for node in ast.walk(tree):
    if not isinstance(
        node,
        (
            ast.Assign,
            ast.AnnAssign,
            ast.Expr,
            ast.Dict,
            ast.Call,
        ),
    ):
        continue

    try:
        segment = source_segment(
            producer_source,
            node,
        )
    except Exception:
        continue

    hits = feature_hits(
        segment
    )

    if not hits:
        continue

    feature_nodes.append(
        {
            "node":
                node,

            "type":
                type(node).__name__,

            "lineno":
                int(
                    getattr(
                        node,
                        "lineno",
                        -1,
                    )
                ),

            "end_lineno":
                int(
                    getattr(
                        node,
                        "end_lineno",
                        -1,
                    )
                ),

            "hits":
                hits,

            "source":
                segment,

            "sha256":
                source_sha(
                    segment
                ),
        }
    )


assert feature_nodes


print(
    "\n========== FEATURE-BEARING AST NODES =========="
)


for record in sorted(
    feature_nodes,
    key=lambda r: (
        r["lineno"],
        r["end_lineno"],
        r["type"],
    ),
):
    print(
        "type:",
        record["type"],
        "| lines:",
        (
            record["lineno"],
            record["end_lineno"],
        ),
        "| feature hits:",
        len(
            record["hits"]
        ),
        "| SHA:",
        record["sha256"],
    )


# ============================================================
# 9. FIND SMALLEST HIGH-COVERAGE FEATURE DICT / APPEND
#
# This is expected to reveal the exact per-candidate row
# construction.
# ============================================================

high_coverage_nodes = [
    record

    for record in feature_nodes

    if len(
        record[
            "hits"
        ]
    ) >= 20
]


assert high_coverage_nodes, (
    "No AST node contains most/all frozen temporal features."
)


high_coverage_nodes = sorted(
    high_coverage_nodes,
    key=lambda r: (
        (
            r["end_lineno"]
            -
            r["lineno"]
        ),
        -len(
            r["hits"]
        ),
    ),
)


smallest_feature_block = high_coverage_nodes[
    0
]


assert set(
    TEMPORAL_FEATURES
).issubset(
    set(
        smallest_feature_block[
            "hits"
        ]
    )
), (
    "Smallest feature block does not contain all 23 features."
)


FEATURE_FORMULA_BLOCK_SHA = (
    smallest_feature_block[
        "sha256"
    ]
)


print(
    "\n========== EXACT FEATURE FORMULA BLOCK =========="
)

print(
    "AST type:",
    smallest_feature_block[
        "type"
    ]
)

print(
    "lines:",
    (
        smallest_feature_block[
            "lineno"
        ],
        smallest_feature_block[
            "end_lineno"
        ],
    )
)

print(
    "feature hits:",
    len(
        smallest_feature_block[
            "hits"
        ]
    ),
    "/ 23"
)

print(
    "SHA:",
    FEATURE_FORMULA_BLOCK_SHA
)

print(
    "\n----- EXACT SOURCE -----\n"
)

print(
    smallest_feature_block[
        "source"
    ]
)

print(
    "\n----- END EXACT SOURCE -----"
)

print(
    "EXACT_FEATURE_FORMULA_BLOCK_RECOVERED: PASS"
)


# ============================================================
# 10. FIND FEATURE APPEND / ROW-CONSTRUCTION CALL
# ============================================================

append_candidates = []


for node in ast.walk(tree):

    if not isinstance(
        node,
        ast.Call,
    ):
        continue

    if not (
        isinstance(
            node.func,
            ast.Attribute,
        )
        and
        node.func.attr
        ==
        "append"
    ):
        continue

    segment = source_segment(
        producer_source,
        node,
    )

    hits = feature_hits(
        segment
    )

    if len(hits) < 20:
        continue

    append_candidates.append(
        {
            "node":
                node,

            "lineno":
                int(node.lineno),

            "end_lineno":
                int(node.end_lineno),

            "hits":
                hits,

            "source":
                segment,

            "sha256":
                source_sha(
                    segment
                ),
        }
    )


print(
    "\n========== FEATURE ROW APPEND CANDIDATES =========="
)


if append_candidates:

    for record in append_candidates:
        print(
            "lines:",
            (
                record["lineno"],
                record["end_lineno"],
            ),
            "| features:",
            len(
                record["hits"]
            ),
            "| SHA:",
            record["sha256"],
        )


else:
    print(
        "No >=20-feature append() call; "
        "feature block uses another construction form."
    )


# ============================================================
# 11. IDENTIFY ENCLOSING TOP-LEVEL STATEMENT(S)
#
# Final feature dict may live inside a for-loop whose source
# also contains graph construction / query_frame calls.
# ============================================================

feature_line_min = min(
    record[
        "lineno"
    ]
    for record in high_coverage_nodes
)

feature_line_max = max(
    record[
        "end_lineno"
    ]
    for record in high_coverage_nodes
)


enclosing_top_level = []


for node in tree.body:

    start = int(
        getattr(
            node,
            "lineno",
            -1,
        )
    )

    end = int(
        getattr(
            node,
            "end_lineno",
            -1,
        )
    )

    if start < 0 or end < 0:
        continue

    overlaps_feature_region = (
        start
        <=
        feature_line_min
        <=
        end
    ) or (
        start
        <=
        feature_line_max
        <=
        end
    )

    if not overlaps_feature_region:
        continue

    statement_source = source_segment(
        producer_source,
        node,
    )

    hits = feature_hits(
        statement_source
    )

    if not hits:
        continue

    enclosing_top_level.append(
        {
            "node":
                node,

            "type":
                type(node).__name__,

            "lineno":
                start,

            "end_lineno":
                end,

            "source":
                statement_source,

            "sha256":
                source_sha(
                    statement_source
                ),

            "feature_hits":
                hits,

            "calls":
                called_simple_names(
                    node
                ),
        }
    )


assert enclosing_top_level


print(
    "\n========== ENCLOSING TOP-LEVEL FEATURE BLOCKS =========="
)


for record in enclosing_top_level:
    print(
        "type:",
        record["type"],
        "| lines:",
        (
            record["lineno"],
            record["end_lineno"],
        ),
        "| features:",
        len(
            record["feature_hits"]
        ),
        "| SHA:",
        record["sha256"],
    )

    print(
        "simple calls:",
        sorted(
            record[
                "calls"
            ]
        ),
    )


# ============================================================
# 12. HELPER DEPENDENCY CLOSURE
#
# Start from functions called by enclosing feature block(s).
# Recursively include any other top-level functions they call.
# ============================================================

initial_function_dependencies = set()


for record in enclosing_top_level:

    initial_function_dependencies.update(
        name

        for name in record[
            "calls"
        ]

        if name in function_map
    )


dependency_closure = set()

frontier = list(
    sorted(
        initial_function_dependencies
    )
)


while frontier:

    name = frontier.pop()

    if name in dependency_closure:
        continue

    dependency_closure.add(
        name
    )

    for child_name in function_map[
        name
    ][
        "calls"
    ]:

        if (
            child_name
            in function_map
            and
            child_name
            not in dependency_closure
        ):
            frontier.append(
                child_name
            )


print(
    "\n========== FEATURE HELPER DEPENDENCY CLOSURE =========="
)

print(
    "initial:",
    sorted(
        initial_function_dependencies
    )
)

print(
    "closure:",
    sorted(
        dependency_closure
    )
)


# query_frame is known historical geometry machinery.
assert "query_frame" in dependency_closure, (
    "Feature construction does not appear to call query_frame; "
    "inspect before proceeding."
)


print(
    "QUERY_FRAME_IN_FEATURE_DEPENDENCY_CLOSURE: PASS"
)


# ============================================================
# 13. EXACT HELPER SOURCE LOCKS
# ============================================================

helper_manifest = []


print(
    "\n========== EXACT FEATURE HELPER SOURCES =========="
)


for name in sorted(
    dependency_closure
):

    record = function_map[
        name
    ]

    helper_manifest.append(
        {
            "name":
                name,

            "sha256":
                record[
                    "sha256"
                ],

            "lineno":
                record[
                    "lineno"
                ],

            "end_lineno":
                record[
                    "end_lineno"
                ],
        }
    )

    print()

    print(
        "-" * 80
    )

    print(
        "HELPER:",
        name
    )

    print(
        "SHA:",
        record[
            "sha256"
        ]
    )

    print(
        "LINES:",
        (
            record[
                "lineno"
            ],
            record[
                "end_lineno"
            ],
        )
    )

    print(
        "-" * 80
    )

    print(
        record[
            "source"
        ]
    )


# ============================================================
# 14. PER-FEATURE EXACT SOURCE LOCATIONS
#
# Find smallest AST node containing each feature token.
# ============================================================

feature_location_manifest = {}


for feature in TEMPORAL_FEATURES:

    matches = [
        record

        for record in feature_nodes

        if feature in record[
            "hits"
        ]
    ]

    assert matches, feature

    matches = sorted(
        matches,
        key=lambda r: (
            (
                r["end_lineno"]
                -
                r["lineno"]
            ),
            r["lineno"],
        ),
    )

    best = matches[
        0
    ]

    feature_location_manifest[
        feature
    ] = {
        "ast_type":
            best[
                "type"
            ],

        "lineno":
            best[
                "lineno"
            ],

        "end_lineno":
            best[
                "end_lineno"
            ],

        "source_sha256":
            best[
                "sha256"
            ],
    }


print(
    "\n========== PER-FEATURE SOURCE LOCATIONS =========="
)


for feature in TEMPORAL_FEATURES:

    record = feature_location_manifest[
        feature
    ]

    print(
        feature,
        "->",
        record[
            "ast_type"
        ],
        (
            record[
                "lineno"
            ],
            record[
                "end_lineno"
            ],
        ),
        record[
            "source_sha256"
        ],
    )


print(
    "ALL_23_FEATURE_SOURCE_LOCATIONS_RECOVERED: PASS"
)


# ============================================================
# 15. SOURCE-LINE CONTEXT
#
# Print compact exact context around every temporal feature.
# ============================================================

source_lines = producer_source.splitlines()

interesting_lines = set()


for feature in TEMPORAL_FEATURES:

    for i, line in enumerate(
        source_lines,
        start=1,
    ):

        if feature not in line:
            continue

        for j in range(
            max(
                1,
                i - 3
            ),
            min(
                len(source_lines),
                i + 3
            )
            + 1,
        ):
            interesting_lines.add(
                j
            )


print(
    "\n========== EXACT FEATURE SOURCE CONTEXT =========="
)


last_line = None


for line_number in sorted(
    interesting_lines
):

    if (
        last_line is not None
        and
        line_number
        >
        last_line + 1
    ):
        print(
            "      ..."
        )

    print(
        f"{line_number:5d}: "
        f"{source_lines[line_number - 1]}"
    )

    last_line = line_number


# ============================================================
# 16. GT-BLINDNESS AUDIT
#
# Audit only:
#
#   - exact feature-construction top-level blocks
#   - helper dependency closure
# ============================================================

scientific_source = "\n\n".join(
    [
        record[
            "source"
        ]
        for record in enclosing_top_level
    ]
    +
    [
        function_map[
            name
        ][
            "source"
        ]
        for name in dependency_closure
    ]
)


FORBIDDEN_GT_TOKENS = [
    "oracle_hit_",
    "nearest_missing_distance_um",

    "gt_node_id",
    "gt_source_id",
    "gt_target_id",

    "gt_z",
    "gt_y",
    "gt_x",
]


found_forbidden = [
    token

    for token in FORBIDDEN_GT_TOKENS

    if token in scientific_source
]


assert not found_forbidden, (
    "\nGT/oracle dependency found in exact historical "
    "feature-construction machinery:\n"
    f"{found_forbidden}"
)


print(
    "\n========== FEATURE MACHINERY GT-BLINDNESS =========="
)

print(
    "GT/oracle dependencies:",
    "NO"
)

print(
    "TEMPORAL_FEATURE_MACHINERY_GT_BLINDNESS: PASS"
)


# ============================================================
# 17. NO SCIENTIFIC ACTION OCCURRED
# ============================================================

NEW_FEATURES_GENERATED = False

NEW_GT_READ = False

MODEL_TRAINED = False

THRESHOLD_SELECTED = False

GRAPH_MUTATED = False

FOLD0_USED = False


assert NEW_FEATURES_GENERATED is False

assert NEW_GT_READ is False

assert MODEL_TRAINED is False

assert THRESHOLD_SELECTED is False

assert GRAPH_MUTATED is False

assert FOLD0_USED is False


# ============================================================
# 18. FINAL
# ============================================================

print(
    "\n========== 12.13A4-P0-R2 FINAL =========="
)

print(
    "EXACT_STAGE12_12_F1_FEATURE_MACHINERY_RECOVERY:",
    "PASS"
)


print(
    "\n--- PRODUCER ---"
)

print(
    "cell:",
    EXPECTED_PRODUCER_CELL
)

print(
    "cell SHA:",
    EXPECTED_PRODUCER_SHA
)


print(
    "\n--- FORMULA BLOCK ---"
)

print(
    "type:",
    smallest_feature_block[
        "type"
    ]
)

print(
    "lines:",
    (
        smallest_feature_block[
            "lineno"
        ],
        smallest_feature_block[
            "end_lineno"
        ],
    )
)

print(
    "SHA:",
    FEATURE_FORMULA_BLOCK_SHA
)


print(
    "\n--- HELPER DEPENDENCY CLOSURE ---"
)

for record in helper_manifest:

    print(
        "-",
        record[
            "name"
        ],
        "|",
        record[
            "sha256"
        ],
        "|",
        (
            record[
                "lineno"
            ],
            record[
                "end_lineno"
            ],
        ),
    )


print(
    "\n--- QUERY FRAME ---"
)

print(
    "SHA:",
    function_map[
        "query_frame"
    ][
        "sha256"
    ]
)

print(
    "in dependency closure:",
    "YES"
)


print(
    "\n--- FEATURES ---"
)

print(
    "23/23 source locations recovered:",
    "YES"
)

print(
    "machinery GT-blind:",
    "YES"
)


print(
    "\n--- SAFETY ---"
)

print(
    "NEW features generated:",
    "NO"
)

print(
    "NEW GT read:",
    "NO"
)

print(
    "model trained:",
    "NO"
)

print(
    "threshold selected:",
    "NO"
)

print(
    "graph mutation:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)


print(
    "\nnext:"
)

print(
    "12.13A4_FORMAL_GT_BLIND_FEATURE_MATRIX_"
    "MATERIALIZATION_USING_SOURCE_LOCKED_CELL80_MACHINERY"
)

In [ ]:
# ============================================================
# Stage 12.13A4-P1
#
# TEMPORARY GT-BLIND PREFLIGHT
#
# PROPOSAL-SUPPORT FEATURE SEMANTICS
# + G2 BIDIRECTIONAL SUPPORT DISAMBIGUATION
#
# ============================================================
#
# PURPOSE
#
# A3 froze 39 allowed features:
#
#   23 exact historical temporal-geometry features
#   12 proposal-support features
#    4 G2 raw-geometry aggregates
#
# Temporal geometry is now source-locked by A4-P0-R3.
#
# Before formal A4, mechanically verify proposal-support
# derivation directly from the frozen A1-R3 RAW proposals.
#
# ------------------------------------------------------------
# CRITICAL AMBIGUITY TO TEST
# ------------------------------------------------------------
#
# A3 names:
#
#   g2_bidirectional_support
#
# but did not explicitly distinguish:
#
#   A. count(raw G2 proposals where s2t AND t2s)
#
# versus
#
#   B. min(g2_s2t_count, g2_t2s_count)
#
# If A == B for every frozen candidate:
#
#   ambiguity is observationally irrelevant for this universe.
#
# If A != B anywhere:
#
#   STOP before formal A4.
#   Create an explicit A3 implementation-clarification record
#   BEFORE model training.
#
# ------------------------------------------------------------
# SAFETY
# ------------------------------------------------------------
#
# Oracle read        : NO
# GT read            : NO
# model trained      : NO
# threshold selected : NO
# graph mutation     : NO
# Fold0              : NO
# file write         : NO
#
# ============================================================


from pathlib import Path

import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


A3_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_ranking_attachment_contract_frozen.json"
)


UNIVERSE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_universe_v1.pkl"
)


RAW_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_raw_proposals_v1.pkl"
)


for path in [
    A3_PATH,
    UNIVERSE_PATH,
    RAW_PATH,
]:
    assert path.exists(), path


# ============================================================
# 1. LOCKS
# ============================================================

EXPECTED_A3_CONTENT_SHA = (
    "c54eda46073582a7a915f1bfa7dcbff9"
    "eec8a96edcfad80a8e49e1097cf38b46"
)

EXPECTED_A3_FILE_SHA = (
    "37a5b2e557832774c0bcf65a33c81d8"
    "434bac2025c4eebbf2e3b0ea1bc955bd7"
)

EXPECTED_UNIVERSE_FILE_SHA = (
    "440676497d3365a0926deaf45140f573"
    "32c8e9e89d46b88401a7ce6f7a299fc3"
)

EXPECTED_RAW_FILE_SHA = (
    "bd5598c3c442600d19d82ac5566326b"
    "b9b09c055115727580c3a469f0e3b762b"
)


EXPECTED_RAW_ROWS = 1_127_600

EXPECTED_ACCEPTED_RAW_ROWS = 1_081_627

EXPECTED_UNIVERSE_ROWS = 1_079_885


EXPECTED_ACCEPTED_FAMILY_COUNTS = {
    "BILATERAL_MIDPOINT_K5_V1":
        824_562,

    "FORWARD_CONSTANT_VELOCITY_V1":
        130_622,

    "BACKWARD_CONSTANT_VELOCITY_V1":
        126_443,
}


FAMILY_G0 = (
    "FORWARD_CONSTANT_VELOCITY_V1"
)

FAMILY_G1 = (
    "BACKWARD_CONSTANT_VELOCITY_V1"
)

FAMILY_G2 = (
    "BILATERAL_MIDPOINT_K5_V1"
)


# ============================================================
# 2. HASH
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# ============================================================
# 3. A3 LOCK
# ============================================================

assert sha256_file(
    A3_PATH
) == EXPECTED_A3_FILE_SHA


a3 = json.loads(
    A3_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    a3[
        "contract_sha256"
    ]
    ==
    EXPECTED_A3_CONTENT_SHA
)


assert a3[
    "stage"
] == "12.13A3"


assert a3[
    "protocol"
][
    "model_trained"
] is False


assert a3[
    "protocol"
][
    "Fold0_used"
] is False


proposal_features = a3[
    "features"
][
    "proposal_support_features"
]


expected_proposal_features = [
    "proposal_count",

    "g0_count",
    "g1_count",
    "g2_count",

    "g2_s2t_count",
    "g2_t2s_count",

    "has_g0",
    "has_g1",
    "has_g2",

    "family_count",
    "multi_family",

    "g2_bidirectional_support",
]


assert proposal_features == expected_proposal_features


g2_geometry_features = a3[
    "features"
][
    "g2_raw_geometry_features"
]


assert g2_geometry_features == [
    "g2_pair_distance_um_min",
    "g2_pair_distance_um_median",
    "g2_pair_distance_um_mean",
    "g2_pair_distance_um_max",
]


print(
    "========== 12.13A4-P1 A3 LOCK =========="
)

print(
    "proposal support features:",
    len(proposal_features)
)

print(
    "G2 geometry features:",
    len(g2_geometry_features)
)

print(
    "model trained:",
    "NO"
)

print(
    "Fold0:",
    "NO"
)

print(
    "A3_PROPOSAL_FEATURE_CONTRACT_LOCK: PASS"
)


# ============================================================
# 4. LOAD FROZEN A1-R3 ARTIFACTS
# ============================================================

assert sha256_file(
    UNIVERSE_PATH
) == EXPECTED_UNIVERSE_FILE_SHA


assert sha256_file(
    RAW_PATH
) == EXPECTED_RAW_FILE_SHA


universe = pd.read_pickle(
    UNIVERSE_PATH
)


raw = pd.read_pickle(
    RAW_PATH
)


assert len(
    universe
) == EXPECTED_UNIVERSE_ROWS


assert len(
    raw
) == EXPECTED_RAW_ROWS


assert universe[
    "candidate_id"
].is_unique


assert np.array_equal(
    universe[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        EXPECTED_UNIVERSE_ROWS,
        dtype=np.int64,
    ),
)


# ============================================================
# 5. RAW SCHEMA
# ============================================================

REQUIRED_RAW_COLUMNS = {
    "raw_proposal_id",
    "dataset",
    "fold",
    "family",
    "candidate_t",
    "z",
    "y",
    "x",

    "g2_source_to_target_knn",
    "g2_target_to_source_knn",
    "g2_pair_distance_um",

    "accepted",

    "key_z",
    "key_y",
    "key_x",
}


assert REQUIRED_RAW_COLUMNS.issubset(
    raw.columns
)


REQUIRED_UNIVERSE_COLUMNS = {
    "candidate_id",
    "dataset",
    "fold",
    "t",
    "z",
    "y",
    "x",

    "proposal_count",

    "g0_count",
    "g1_count",
    "g2_count",

    "g2_s2t_count",
    "g2_t2s_count",
}


assert REQUIRED_UNIVERSE_COLUMNS.issubset(
    universe.columns
)


accepted = raw.loc[
    raw[
        "accepted"
    ].astype(bool)
].copy()


assert len(
    accepted
) == EXPECTED_ACCEPTED_RAW_ROWS


actual_family_counts = (
    accepted[
        "family"
    ]
    .value_counts()
    .to_dict()
)


assert (
    actual_family_counts
    ==
    EXPECTED_ACCEPTED_FAMILY_COUNTS
)


print(
    "\n========== RAW A1-R3 LOCK =========="
)

print(
    "raw rows:",
    len(raw)
)

print(
    "accepted rows:",
    len(accepted)
)

print(
    "accepted family counts:",
    actual_family_counts
)

print(
    "RAW_A1_R3_LOCK: PASS"
)


# ============================================================
# 6. CANONICAL CANDIDATE KEY
#
# A1-R3 identity:
#
# dataset,t,round(z,12),round(y,12),round(x,12)
#
# Raw artifact already stores key_z/key_y/key_x.
# ============================================================

RAW_KEY = [
    "dataset",
    "candidate_t",
    "key_z",
    "key_y",
    "key_x",
]


accepted[
    "dataset"
] = accepted[
    "dataset"
].astype(str)


accepted[
    "candidate_t"
] = pd.to_numeric(
    accepted[
        "candidate_t"
    ],
    errors="raise",
).astype(
    np.int64
)


for column in [
    "key_z",
    "key_y",
    "key_x",
]:

    accepted[
        column
    ] = pd.to_numeric(
        accepted[
            column
        ],
        errors="raise",
    ).astype(
        np.float64
    )


universe_key = universe[
    [
        "candidate_id",
        "dataset",
        "t",
        "z",
        "y",
        "x",
    ]
].copy()


universe_key[
    "dataset"
] = universe_key[
    "dataset"
].astype(str)


universe_key[
    "candidate_t"
] = pd.to_numeric(
    universe_key[
        "t"
    ],
    errors="raise",
).astype(
    np.int64
)


universe_key[
    "key_z"
] = np.round(
    universe_key[
        "z"
    ].to_numpy(
        dtype=np.float64
    ),
    12,
)


universe_key[
    "key_y"
] = np.round(
    universe_key[
        "y"
    ].to_numpy(
        dtype=np.float64
    ),
    12,
)


universe_key[
    "key_x"
] = np.round(
    universe_key[
        "x"
    ].to_numpy(
        dtype=np.float64
    ),
    12,
)


assert not universe_key[
    RAW_KEY
].duplicated().any()


# ============================================================
# 7. RAW BOOLEAN NORMALIZATION
# ============================================================

for column in [
    "g2_source_to_target_knn",
    "g2_target_to_source_knn",
]:

    accepted[
        column
    ] = accepted[
        column
    ].fillna(
        False
    ).astype(
        bool
    )


accepted[
    "_is_g0"
] = accepted[
    "family"
].eq(
    FAMILY_G0
)


accepted[
    "_is_g1"
] = accepted[
    "family"
].eq(
    FAMILY_G1
)


accepted[
    "_is_g2"
] = accepted[
    "family"
].eq(
    FAMILY_G2
)


# ============================================================
# 8. PER-RAW G2 SUPPORT
# ============================================================

accepted[
    "_g2_s2t"
] = (
    accepted[
        "_is_g2"
    ]
    &
    accepted[
        "g2_source_to_target_knn"
    ]
)


accepted[
    "_g2_t2s"
] = (
    accepted[
        "_is_g2"
    ]
    &
    accepted[
        "g2_target_to_source_knn"
    ]
)


accepted[
    "_g2_both"
] = (
    accepted[
        "_is_g2"
    ]
    &
    accepted[
        "g2_source_to_target_knn"
    ]
    &
    accepted[
        "g2_target_to_source_knn"
    ]
)


# ============================================================
# 9. AGGREGATE RAW -> CANDIDATE
# ============================================================

raw_agg = (
    accepted
    .groupby(
        RAW_KEY,
        sort=False,
        dropna=False,
    )
    .agg(
        proposal_count_raw=(
            "raw_proposal_id",
            "size",
        ),

        g0_count_raw=(
            "_is_g0",
            "sum",
        ),

        g1_count_raw=(
            "_is_g1",
            "sum",
        ),

        g2_count_raw=(
            "_is_g2",
            "sum",
        ),

        g2_s2t_count_raw=(
            "_g2_s2t",
            "sum",
        ),

        g2_t2s_count_raw=(
            "_g2_t2s",
            "sum",
        ),

        g2_both_raw=(
            "_g2_both",
            "sum",
        ),
    )
    .reset_index()
)


for column in [
    "proposal_count_raw",
    "g0_count_raw",
    "g1_count_raw",
    "g2_count_raw",
    "g2_s2t_count_raw",
    "g2_t2s_count_raw",
    "g2_both_raw",
]:

    raw_agg[
        column
    ] = pd.to_numeric(
        raw_agg[
            column
        ],
        errors="raise",
    ).astype(
        np.int64
    )


assert len(
    raw_agg
) == EXPECTED_UNIVERSE_ROWS


# ============================================================
# 10. ONE-TO-ONE RAW -> FROZEN UNIVERSE JOIN
# ============================================================

joined = universe_key.merge(
    raw_agg,
    on=RAW_KEY,
    how="left",
    validate="one_to_one",
    sort=False,
)


assert len(
    joined
) == EXPECTED_UNIVERSE_ROWS


for column in [
    "proposal_count_raw",
    "g0_count_raw",
    "g1_count_raw",
    "g2_count_raw",
    "g2_s2t_count_raw",
    "g2_t2s_count_raw",
    "g2_both_raw",
]:

    assert joined[
        column
    ].notna().all()


joined = joined.sort_values(
    "candidate_id",
    kind="stable",
).reset_index(
    drop=True
)


assert np.array_equal(
    joined[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    universe[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
)


print(
    "\n========== RAW -> UNIVERSE IDENTITY =========="
)

print(
    "raw unique accepted keys:",
    len(raw_agg)
)

print(
    "universe rows:",
    len(universe)
)

print(
    "join:",
    "ONE_TO_ONE"
)

print(
    "RAW_TO_UNIVERSE_IDENTITY: PASS"
)


# ============================================================
# 11. REPRODUCE STORED A1-R3 COUNTS
# ============================================================

count_pairs = [
    (
        "proposal_count",
        "proposal_count_raw",
    ),

    (
        "g0_count",
        "g0_count_raw",
    ),

    (
        "g1_count",
        "g1_count_raw",
    ),

    (
        "g2_count",
        "g2_count_raw",
    ),

    (
        "g2_s2t_count",
        "g2_s2t_count_raw",
    ),

    (
        "g2_t2s_count",
        "g2_t2s_count_raw",
    ),
]


for universe_column, raw_column in count_pairs:

    actual = pd.to_numeric(
        universe[
            universe_column
        ],
        errors="raise",
    ).to_numpy(
        dtype=np.int64
    )


    reconstructed = joined[
        raw_column
    ].to_numpy(
        dtype=np.int64
    )


    assert np.array_equal(
        actual,
        reconstructed,
    ), (
        universe_column,
        int(
            np.count_nonzero(
                actual
                !=
                reconstructed
            )
        ),
    )


print(
    "\n========== STORED COUNT REPRODUCTION =========="
)

for universe_column, _ in count_pairs:
    print(
        universe_column,
        ": PASS"
    )

print(
    "A1_R3_PROPOSAL_COUNTS_REPRODUCED: PASS"
)


# ============================================================
# 12. OBVIOUS DERIVED FEATURES
# ============================================================

g0 = universe[
    "g0_count"
].to_numpy(
    dtype=np.int64
)


g1 = universe[
    "g1_count"
].to_numpy(
    dtype=np.int64
)


g2 = universe[
    "g2_count"
].to_numpy(
    dtype=np.int64
)


has_g0 = (
    g0 > 0
).astype(
    np.int8
)


has_g1 = (
    g1 > 0
).astype(
    np.int8
)


has_g2 = (
    g2 > 0
).astype(
    np.int8
)


family_count = (
    has_g0
    +
    has_g1
    +
    has_g2
).astype(
    np.int8
)


multi_family = (
    family_count
    >
    1
).astype(
    np.int8
)


assert set(
    np.unique(
        family_count
    )
).issubset(
    {
        1,
        2,
        3,
    }
)


assert np.array_equal(
    multi_family,
    (
        family_count
        >
        1
    ).astype(
        np.int8
    ),
)


print(
    "\n========== DERIVED FAMILY FEATURES =========="
)

print(
    "has_g0 positives:",
    int(has_g0.sum())
)

print(
    "has_g1 positives:",
    int(has_g1.sum())
)

print(
    "has_g2 positives:",
    int(has_g2.sum())
)

print(
    "family_count distribution:",
    pd.Series(
        family_count
    ).value_counts().sort_index().to_dict()
)

print(
    "multi_family:",
    int(multi_family.sum())
)

print(
    "DERIVED_FAMILY_FEATURES: PASS"
)


# ============================================================
# 13. BIDIRECTIONAL SUPPORT DISAMBIGUATION
# ============================================================

s2t_count = universe[
    "g2_s2t_count"
].to_numpy(
    dtype=np.int64
)


t2s_count = universe[
    "g2_t2s_count"
].to_numpy(
    dtype=np.int64
)


raw_intersection = joined[
    "g2_both_raw"
].to_numpy(
    dtype=np.int64
)


directional_min = np.minimum(
    s2t_count,
    t2s_count,
)


both_direction_present = (
    (s2t_count > 0)
    &
    (t2s_count > 0)
).astype(
    np.int8
)


intersection_vs_min_diff = (
    raw_intersection
    !=
    directional_min
)


n_intersection_vs_min_diff = int(
    intersection_vs_min_diff.sum()
)


print(
    "\n========== G2 BIDIRECTIONAL SUPPORT =========="
)

print(
    "sum raw intersection:",
    int(
        raw_intersection.sum()
    )
)

print(
    "sum min(s2t,t2s):",
    int(
        directional_min.sum()
    )
)

print(
    "candidates with both directions present:",
    int(
        both_direction_present.sum()
    )
)

print(
    "intersection != min count:",
    n_intersection_vs_min_diff
)


if n_intersection_vs_min_diff:

    diff = pd.DataFrame(
        {
            "candidate_id":
                universe[
                    "candidate_id"
                ].to_numpy(
                    dtype=np.int64
                ),

            "g2_count":
                g2,

            "g2_s2t_count":
                s2t_count,

            "g2_t2s_count":
                t2s_count,

            "raw_intersection":
                raw_intersection,

            "directional_min":
                directional_min,
        }
    )


    print(
        "\nFIRST DIVERGENT EXAMPLES:"
    )

    print(
        diff.loc[
            intersection_vs_min_diff
        ].head(
            30
        ).to_string(
            index=False
        )
    )


# ============================================================
# 14. G2 DISTANCE AGGREGATION
#
# A3 already states:
#
# aggregate accepted G2 raw proposals belonging to frozen
# candidate identity.
#
# NaN if no G2 support.
# ============================================================

g2_raw = accepted.loc[
    accepted[
        "_is_g2"
    ]
].copy()


g2_raw[
    "g2_pair_distance_um"
] = pd.to_numeric(
    g2_raw[
        "g2_pair_distance_um"
    ],
    errors="raise",
).astype(
    np.float64
)


assert np.isfinite(
    g2_raw[
        "g2_pair_distance_um"
    ].to_numpy(
        dtype=np.float64
    )
).all()


assert (
    g2_raw[
        "g2_pair_distance_um"
    ].to_numpy(
        dtype=np.float64
    )
    >=
    0
).all()


g2_geometry = (
    g2_raw
    .groupby(
        RAW_KEY,
        sort=False,
        dropna=False,
    )
    [
        "g2_pair_distance_um"
    ]
    .agg(
        [
            "min",
            "median",
            "mean",
            "max",
        ]
    )
    .reset_index()
    .rename(
        columns={
            "min":
                "g2_pair_distance_um_min",

            "median":
                "g2_pair_distance_um_median",

            "mean":
                "g2_pair_distance_um_mean",

            "max":
                "g2_pair_distance_um_max",
        }
    )
)


geometry_join = universe_key.merge(
    g2_geometry,
    on=RAW_KEY,
    how="left",
    validate="one_to_one",
    sort=False,
)


geometry_join = geometry_join.sort_values(
    "candidate_id",
    kind="stable",
).reset_index(
    drop=True
)


has_g2_mask = (
    g2 > 0
)


for column in [
    "g2_pair_distance_um_min",
    "g2_pair_distance_um_median",
    "g2_pair_distance_um_mean",
    "g2_pair_distance_um_max",
]:

    values = geometry_join[
        column
    ].to_numpy(
        dtype=np.float64
    )


    assert np.isfinite(
        values[
            has_g2_mask
        ]
    ).all()


    assert np.isnan(
        values[
            ~has_g2_mask
        ]
    ).all()


# ordering invariant
gmin = geometry_join[
    "g2_pair_distance_um_min"
].to_numpy(
    dtype=np.float64
)

gmedian = geometry_join[
    "g2_pair_distance_um_median"
].to_numpy(
    dtype=np.float64
)

gmean = geometry_join[
    "g2_pair_distance_um_mean"
].to_numpy(
    dtype=np.float64
)

gmax = geometry_join[
    "g2_pair_distance_um_max"
].to_numpy(
    dtype=np.float64
)


assert (
    gmin[
        has_g2_mask
    ]
    <=
    gmedian[
        has_g2_mask
    ]
).all()


assert (
    gmedian[
        has_g2_mask
    ]
    <=
    gmax[
        has_g2_mask
    ]
).all()


assert (
    gmin[
        has_g2_mask
    ]
    <=
    gmean[
        has_g2_mask
    ]
).all()


assert (
    gmean[
        has_g2_mask
    ]
    <=
    gmax[
        has_g2_mask
    ]
).all()


print(
    "\n========== G2 RAW GEOMETRY =========="
)

print(
    "G2 accepted raw proposals:",
    len(g2_raw)
)

print(
    "candidates with G2:",
    int(
        has_g2_mask.sum()
    )
)

print(
    "candidates without G2:",
    int(
        (~has_g2_mask).sum()
    )
)

print(
    "NaN without G2:",
    "YES"
)

print(
    "G2_RAW_GEOMETRY_SEMANTICS: PASS"
)


# ============================================================
# 15. SUM / COUNT GLOBAL INVARIANTS
# ============================================================

assert int(
    universe[
        "proposal_count"
    ].sum()
) == EXPECTED_ACCEPTED_RAW_ROWS


assert int(
    universe[
        "g0_count"
    ].sum()
) == EXPECTED_ACCEPTED_FAMILY_COUNTS[
    FAMILY_G0
]


assert int(
    universe[
        "g1_count"
    ].sum()
) == EXPECTED_ACCEPTED_FAMILY_COUNTS[
    FAMILY_G1
]


assert int(
    universe[
        "g2_count"
    ].sum()
) == EXPECTED_ACCEPTED_FAMILY_COUNTS[
    FAMILY_G2
]


print(
    "\n========== GLOBAL SUPPORT INVARIANTS =========="
)

print(
    "proposal_count sum:",
    int(
        universe[
            "proposal_count"
        ].sum()
    )
)

print(
    "G0 sum:",
    int(
        universe[
            "g0_count"
        ].sum()
    )
)

print(
    "G1 sum:",
    int(
        universe[
            "g1_count"
        ].sum()
    )
)

print(
    "G2 sum:",
    int(
        universe[
            "g2_count"
        ].sum()
    )
)

print(
    "GLOBAL_PROPOSAL_SUPPORT_INVARIANTS: PASS"
)


# ============================================================
# 16. FINAL DECISION
# ============================================================

BIDIRECTIONAL_EQUIVALENT = (
    n_intersection_vs_min_diff
    ==
    0
)


print(
    "\n========== 12.13A4-P1 FINAL =========="
)

print(
    "GT_BLIND_PROPOSAL_FEATURE_SEMANTICS_AUDIT:",
    "PASS"
)


print(
    "\n--- RAW -> UNIVERSE ---"
)

print(
    "one-to-one identity:",
    "PASS"
)

print(
    "stored proposal counts reproduced:",
    "PASS"
)


print(
    "\n--- DERIVED FAMILY FEATURES ---"
)

print(
    "has_g0/has_g1/has_g2:",
    "DEFINED"
)

print(
    "family_count:",
    "DEFINED"
)

print(
    "multi_family:",
    "DEFINED"
)


print(
    "\n--- G2 BIDIRECTIONAL ---"
)

print(
    "raw intersection == min directional counts:",
    BIDIRECTIONAL_EQUIVALENT
)

print(
    "number of divergent candidates:",
    n_intersection_vs_min_diff
)


print(
    "\n--- G2 GEOMETRY ---"
)

print(
    "min/median/mean/max:",
    "DEFINED"
)

print(
    "NaN when no G2:",
    "YES"
)


print(
    "\n--- SAFETY ---"
)

print(
    "GT read:",
    "NO"
)

print(
    "oracle read:",
    "NO"
)

print(
    "model trained:",
    "NO"
)

print(
    "threshold selected:",
    "NO"
)

print(
    "graph mutation:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)


print(
    "\nnext:"
)


if BIDIRECTIONAL_EQUIVALENT:

    print(
        "12.13A4_FORMAL_GT_BLIND_FEATURE_MATRIX_MATERIALIZATION"
    )

else:

    print(
        "12.13A3-C1_G2_BIDIRECTIONAL_SUPPORT_"
        "IMPLEMENTATION_CLARIFICATION_BEFORE_A4"
    )

In [ ]:
# ============================================================
# Stage 12.13A3-C1
#
# FORMAL IMPLEMENTATION CLARIFICATION ADDENDUM
#
# G2 BIDIRECTIONAL SUPPORT
# + PROPOSAL-DERIVED FEATURE SEMANTICS
#
# ============================================================
#
# PARENT:
#
#   12.13A3
#   RANKING_AND_ATTACHMENT_DEVELOPMENT_CONTRACT_FROZEN
#
# ------------------------------------------------------------
# WHY C1 IS REQUIRED
# ------------------------------------------------------------
#
# A3 froze the feature name:
#
#   g2_bidirectional_support
#
# but did not explicitly distinguish:
#
#   A)
#     count of SAME raw G2 proposals satisfying:
#
#       source_to_target_knn == True
#       AND
#       target_to_source_knn == True
#
#   versus
#
#   B)
#     min(
#       g2_s2t_count,
#       g2_t2s_count
#     )
#
# GT-blind A4-P1 proved these differ on the frozen universe:
#
#   divergent candidates = 67
#
#   sum raw reciprocal intersection = 369230
#   sum directional min             = 369297
#
# Therefore implementation semantics must be frozen before
# feature materialization / model training.
#
# ------------------------------------------------------------
# C1 DECISION
# ------------------------------------------------------------
#
# g2_bidirectional_support :=
#
#   COUNT OF ACCEPTED G2 RAW PROPOSALS FOR THE SAME FROZEN
#   CANDIDATE WHERE BOTH:
#
#       g2_source_to_target_knn == True
#       g2_target_to_source_knn == True
#
# This is the pair-level reciprocal-intersection definition.
#
# min(s2t_count, t2s_count) is explicitly FORBIDDEN.
#
# ------------------------------------------------------------
# SCIENTIFIC RATIONALE
# ------------------------------------------------------------
#
# Each G2 raw proposal corresponds to a specific source-target
# endpoint pair.
#
# Reciprocal/bidirectional support means THAT SAME PAIR was
# selected in both directional KNN relations.
#
# min(s2t_count,t2s_count) can pair evidence from different
# raw proposals and therefore can create false reciprocal
# support.
#
# ------------------------------------------------------------
# THIS ADDENDUM DOES NOT CHANGE
# ------------------------------------------------------------
#
# candidate universe
# ranking target
# feature set
# scorer families
# LOFO
# threshold selection
# attachment
# graph mutation
# promotion gates
#
# It only resolves an implementation ambiguity that was found
# before model training and without GT/oracle access.
#
# ------------------------------------------------------------
# SAFETY
# ------------------------------------------------------------
#
# GT read            : NO
# oracle read        : NO
# model trained      : NO
# threshold selected : NO
# graph mutation     : NO
# Fold0              : NO
#
# KEEP PERMANENTLY.
# ============================================================


from pathlib import Path

import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


A3_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_ranking_attachment_contract_frozen.json"
)


RAW_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_raw_proposals_v1.pkl"
)


UNIVERSE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_universe_v1.pkl"
)


C1_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_ranking_attachment_contract_c1_g2_bidirectional_support_clarification_frozen.json"
)


for path in [
    A3_PATH,
    RAW_PATH,
    UNIVERSE_PATH,
]:
    assert path.exists(), path


if C1_PATH.exists():

    raise RuntimeError(
        "\nFormal A3-C1 clarification already exists:\n"
        f"{C1_PATH}\n\n"
        "Do NOT overwrite frozen provenance."
    )


# ============================================================
# 1. EXACT PARENT LOCKS
# ============================================================

EXPECTED_A3_CONTENT_SHA = (
    "c54eda46073582a7a915f1bfa7dcbff9"
    "eec8a96edcfad80a8e49e1097cf38b46"
)


EXPECTED_A3_FILE_SHA = (
    "37a5b2e557832774c0bcf65a33c81d8"
    "434bac2025c4eebbf2e3b0ea1bc955bd7"
)


EXPECTED_RAW_FILE_SHA = (
    "bd5598c3c442600d19d82ac5566326b"
    "b9b09c055115727580c3a469f0e3b762b"
)


EXPECTED_UNIVERSE_FILE_SHA = (
    "440676497d3365a0926deaf45140f573"
    "32c8e9e89d46b88401a7ce6f7a299fc3"
)


EXPECTED_RAW_ROWS = 1_127_600

EXPECTED_ACCEPTED_RAW_ROWS = 1_081_627

EXPECTED_UNIVERSE_ROWS = 1_079_885


EXPECTED_RAW_INTERSECTION_SUM = 369_230

EXPECTED_DIRECTIONAL_MIN_SUM = 369_297

EXPECTED_BOTH_DIRECTIONS_PRESENT = 369_047

EXPECTED_DIVERGENT_CANDIDATES = 67


BRANCH_ID = (
    "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_NODE_RECOVERY_V1"
)


CANDIDATE_UNIVERSE_ID = (
    "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_CANDIDATE_UNIVERSE_V1"
)


FAMILY_G0 = (
    "FORWARD_CONSTANT_VELOCITY_V1"
)


FAMILY_G1 = (
    "BACKWARD_CONSTANT_VELOCITY_V1"
)


FAMILY_G2 = (
    "BILATERAL_MIDPOINT_K5_V1"
)


# ============================================================
# 2. HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# ============================================================
# 3. A3 PARENT LOCK
# ============================================================

assert sha256_file(
    A3_PATH
) == EXPECTED_A3_FILE_SHA


a3 = json.loads(
    A3_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    a3[
        "contract_sha256"
    ]
    ==
    EXPECTED_A3_CONTENT_SHA
)


assert a3[
    "stage"
] == "12.13A3"


assert a3[
    "status"
] == "RANKING_AND_ATTACHMENT_DEVELOPMENT_CONTRACT_FROZEN"


assert a3[
    "branch_id"
] == BRANCH_ID


assert a3[
    "candidate_universe_id"
] == CANDIDATE_UNIVERSE_ID


assert a3[
    "protocol"
][
    "model_trained"
] is False


assert a3[
    "protocol"
][
    "threshold_selected"
] is False


assert a3[
    "protocol"
][
    "graph_mutated"
] is False


assert a3[
    "protocol"
][
    "Fold0_used"
] is False


PROPOSAL_FEATURES = a3[
    "features"
][
    "proposal_support_features"
]


assert PROPOSAL_FEATURES == [
    "proposal_count",

    "g0_count",
    "g1_count",
    "g2_count",

    "g2_s2t_count",
    "g2_t2s_count",

    "has_g0",
    "has_g1",
    "has_g2",

    "family_count",
    "multi_family",

    "g2_bidirectional_support",
]


assert a3[
    "features"
][
    "g2_raw_geometry_features"
] == [
    "g2_pair_distance_um_min",
    "g2_pair_distance_um_median",
    "g2_pair_distance_um_mean",
    "g2_pair_distance_um_max",
]


print(
    "========== 12.13A3-C1 PARENT LOCK =========="
)


print(
    "A3 content SHA:",
    EXPECTED_A3_CONTENT_SHA
)


print(
    "A3 file SHA:",
    EXPECTED_A3_FILE_SHA
)


print(
    "model trained:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "graph mutated:",
    "NO"
)


print(
    "Fold0:",
    "NO"
)


print(
    "A3_PARENT_LOCK: PASS"
)


# ============================================================
# 4. A1-R3 ARTIFACT LOCK
# ============================================================

assert sha256_file(
    RAW_PATH
) == EXPECTED_RAW_FILE_SHA


assert sha256_file(
    UNIVERSE_PATH
) == EXPECTED_UNIVERSE_FILE_SHA


raw = pd.read_pickle(
    RAW_PATH
)


universe = pd.read_pickle(
    UNIVERSE_PATH
)


assert len(
    raw
) == EXPECTED_RAW_ROWS


assert len(
    universe
) == EXPECTED_UNIVERSE_ROWS


assert universe[
    "candidate_id"
].is_unique


assert np.array_equal(
    universe[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        EXPECTED_UNIVERSE_ROWS,
        dtype=np.int64,
    ),
)


for column in universe.columns:

    lower = str(
        column
    ).lower()

    assert "oracle" not in lower

    assert not lower.startswith(
        "gt_"
    )


print(
    "\n========== A1-R3 FROZEN ARTIFACT LOCK =========="
)


print(
    "raw rows:",
    len(raw)
)


print(
    "universe rows:",
    len(universe)
)


print(
    "GT/oracle:",
    "NO"
)


print(
    "A1_R3_ARTIFACT_LOCK: PASS"
)


# ============================================================
# 5. RAW SCHEMA LOCK
# ============================================================

REQUIRED_RAW = {
    "raw_proposal_id",

    "dataset",
    "fold",

    "family",

    "candidate_t",
    "z",
    "y",
    "x",

    "key_z",
    "key_y",
    "key_x",

    "g2_source_to_target_knn",
    "g2_target_to_source_knn",

    "g2_pair_distance_um",

    "accepted",
}


assert REQUIRED_RAW.issubset(
    raw.columns
)


REQUIRED_UNIVERSE = {
    "candidate_id",

    "dataset",
    "fold",

    "t",
    "z",
    "y",
    "x",

    "proposal_count",

    "g0_count",
    "g1_count",
    "g2_count",

    "g2_s2t_count",
    "g2_t2s_count",
}


assert REQUIRED_UNIVERSE.issubset(
    universe.columns
)


accepted = raw.loc[
    raw[
        "accepted"
    ].astype(
        bool
    )
].copy()


assert len(
    accepted
) == EXPECTED_ACCEPTED_RAW_ROWS


# ============================================================
# 6. CANONICAL RAW CANDIDATE IDENTITY
# ============================================================

RAW_KEY = [
    "dataset",
    "candidate_t",
    "key_z",
    "key_y",
    "key_x",
]


accepted[
    "dataset"
] = accepted[
    "dataset"
].astype(
    str
)


accepted[
    "candidate_t"
] = pd.to_numeric(
    accepted[
        "candidate_t"
    ],
    errors="raise",
).astype(
    np.int64
)


for column in [
    "key_z",
    "key_y",
    "key_x",
]:

    accepted[
        column
    ] = pd.to_numeric(
        accepted[
            column
        ],
        errors="raise",
    ).astype(
        np.float64
    )


universe_key = universe[
    [
        "candidate_id",
        "dataset",
        "t",
        "z",
        "y",
        "x",
    ]
].copy()


universe_key[
    "dataset"
] = universe_key[
    "dataset"
].astype(
    str
)


universe_key[
    "candidate_t"
] = universe_key[
    "t"
].astype(
    np.int64
)


universe_key[
    "key_z"
] = np.round(
    universe_key[
        "z"
    ].to_numpy(
        dtype=np.float64
    ),
    12,
)


universe_key[
    "key_y"
] = np.round(
    universe_key[
        "y"
    ].to_numpy(
        dtype=np.float64
    ),
    12,
)


universe_key[
    "key_x"
] = np.round(
    universe_key[
        "x"
    ].to_numpy(
        dtype=np.float64
    ),
    12,
)


assert not universe_key[
    RAW_KEY
].duplicated().any()


# ============================================================
# 7. NORMALIZE G2 DIRECTION FLAGS
# ============================================================

for column in [
    "g2_source_to_target_knn",
    "g2_target_to_source_knn",
]:

    accepted[
        column
    ] = accepted[
        column
    ].fillna(
        False
    ).astype(
        bool
    )


accepted[
    "_is_g0"
] = accepted[
    "family"
].eq(
    FAMILY_G0
)


accepted[
    "_is_g1"
] = accepted[
    "family"
].eq(
    FAMILY_G1
)


accepted[
    "_is_g2"
] = accepted[
    "family"
].eq(
    FAMILY_G2
)


accepted[
    "_g2_s2t"
] = (
    accepted[
        "_is_g2"
    ]
    &
    accepted[
        "g2_source_to_target_knn"
    ]
)


accepted[
    "_g2_t2s"
] = (
    accepted[
        "_is_g2"
    ]
    &
    accepted[
        "g2_target_to_source_knn"
    ]
)


# ============================================================
# THIS IS THE C1-FROZEN RECIPROCAL DEFINITION
# ============================================================

accepted[
    "_g2_bidirectional"
] = (
    accepted[
        "_is_g2"
    ]
    &
    accepted[
        "g2_source_to_target_knn"
    ]
    &
    accepted[
        "g2_target_to_source_knn"
    ]
)


# ============================================================
# 8. RAW -> CANDIDATE AGGREGATION
# ============================================================

agg = (
    accepted
    .groupby(
        RAW_KEY,
        sort=False,
        dropna=False,
    )
    .agg(
        proposal_count_raw=(
            "raw_proposal_id",
            "size",
        ),

        g0_count_raw=(
            "_is_g0",
            "sum",
        ),

        g1_count_raw=(
            "_is_g1",
            "sum",
        ),

        g2_count_raw=(
            "_is_g2",
            "sum",
        ),

        g2_s2t_count_raw=(
            "_g2_s2t",
            "sum",
        ),

        g2_t2s_count_raw=(
            "_g2_t2s",
            "sum",
        ),

        g2_bidirectional_support_raw=(
            "_g2_bidirectional",
            "sum",
        ),
    )
    .reset_index()
)


for column in [
    "proposal_count_raw",

    "g0_count_raw",
    "g1_count_raw",
    "g2_count_raw",

    "g2_s2t_count_raw",
    "g2_t2s_count_raw",

    "g2_bidirectional_support_raw",
]:

    agg[
        column
    ] = pd.to_numeric(
        agg[
            column
        ],
        errors="raise",
    ).astype(
        np.int64
    )


assert len(
    agg
) == EXPECTED_UNIVERSE_ROWS


joined = universe_key.merge(
    agg,
    on=RAW_KEY,
    how="left",
    validate="one_to_one",
    sort=False,
)


assert len(
    joined
) == EXPECTED_UNIVERSE_ROWS


joined = (
    joined
    .sort_values(
        "candidate_id",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 9. REPRODUCE ALL EXISTING STORED COUNTS
# ============================================================

for frozen_column, reconstructed_column in [
    (
        "proposal_count",
        "proposal_count_raw",
    ),

    (
        "g0_count",
        "g0_count_raw",
    ),

    (
        "g1_count",
        "g1_count_raw",
    ),

    (
        "g2_count",
        "g2_count_raw",
    ),

    (
        "g2_s2t_count",
        "g2_s2t_count_raw",
    ),

    (
        "g2_t2s_count",
        "g2_t2s_count_raw",
    ),
]:

    frozen = pd.to_numeric(
        universe[
            frozen_column
        ],
        errors="raise",
    ).to_numpy(
        dtype=np.int64
    )


    reconstructed = joined[
        reconstructed_column
    ].to_numpy(
        dtype=np.int64
    )


    assert np.array_equal(
        frozen,
        reconstructed,
    )


print(
    "\n========== STORED COUNT REPRODUCTION =========="
)


print(
    "proposal_count:",
    "PASS"
)


print(
    "g0_count:",
    "PASS"
)


print(
    "g1_count:",
    "PASS"
)


print(
    "g2_count:",
    "PASS"
)


print(
    "g2_s2t_count:",
    "PASS"
)


print(
    "g2_t2s_count:",
    "PASS"
)


print(
    "A1_R3_STORED_COUNT_REPRODUCTION: PASS"
)


# ============================================================
# 10. RECIPROCAL VS DIRECTIONAL-MIN AUDIT
# ============================================================

reciprocal = joined[
    "g2_bidirectional_support_raw"
].to_numpy(
    dtype=np.int64
)


s2t = universe[
    "g2_s2t_count"
].to_numpy(
    dtype=np.int64
)


t2s = universe[
    "g2_t2s_count"
].to_numpy(
    dtype=np.int64
)


directional_min = np.minimum(
    s2t,
    t2s,
)


both_directions_present = (
    (s2t > 0)
    &
    (t2s > 0)
)


divergent = (
    reciprocal
    !=
    directional_min
)


assert int(
    reciprocal.sum()
) == EXPECTED_RAW_INTERSECTION_SUM


assert int(
    directional_min.sum()
) == EXPECTED_DIRECTIONAL_MIN_SUM


assert int(
    both_directions_present.sum()
) == EXPECTED_BOTH_DIRECTIONS_PRESENT


assert int(
    divergent.sum()
) == EXPECTED_DIVERGENT_CANDIDATES


assert (
    reciprocal
    <=
    directional_min
).all()


print(
    "\n========== BIDIRECTIONAL DISAMBIGUATION LOCK =========="
)


print(
    "same-proposal reciprocal support sum:",
    int(
        reciprocal.sum()
    )
)


print(
    "directional-min sum:",
    int(
        directional_min.sum()
    )
)


print(
    "both-direction-present candidates:",
    int(
        both_directions_present.sum()
    )
)


print(
    "divergent candidates:",
    int(
        divergent.sum()
    )
)


print(
    "C1 definition:",
    "SAME_RAW_G2_PROPOSAL_INTERSECTION_COUNT"
)


print(
    "directional min:",
    "FORBIDDEN"
)


print(
    "G2_BIDIRECTIONAL_SUPPORT_CLARIFICATION: PASS"
)


# ============================================================
# 11. DERIVED FAMILY FEATURE FORMULAS
# ============================================================

g0_count = universe[
    "g0_count"
].to_numpy(
    dtype=np.int64
)


g1_count = universe[
    "g1_count"
].to_numpy(
    dtype=np.int64
)


g2_count = universe[
    "g2_count"
].to_numpy(
    dtype=np.int64
)


has_g0 = (
    g0_count
    >
    0
).astype(
    np.int8
)


has_g1 = (
    g1_count
    >
    0
).astype(
    np.int8
)


has_g2 = (
    g2_count
    >
    0
).astype(
    np.int8
)


family_count = (
    has_g0
    +
    has_g1
    +
    has_g2
).astype(
    np.int8
)


multi_family = (
    family_count
    >
    1
).astype(
    np.int8
)


assert set(
    np.unique(
        family_count
    )
).issubset(
    {
        1,
        2,
        3,
    }
)


# ============================================================
# 12. G2 DISTANCE FORMULAS
# ============================================================

g2_rows = accepted.loc[
    accepted[
        "_is_g2"
    ]
].copy()


g2_rows[
    "g2_pair_distance_um"
] = pd.to_numeric(
    g2_rows[
        "g2_pair_distance_um"
    ],
    errors="raise",
).astype(
    np.float64
)


assert np.isfinite(
    g2_rows[
        "g2_pair_distance_um"
    ]
).all()


g2_geometry = (
    g2_rows
    .groupby(
        RAW_KEY,
        sort=False,
        dropna=False,
    )[
        "g2_pair_distance_um"
    ]
    .agg(
        [
            "min",
            "median",
            "mean",
            "max",
        ]
    )
    .reset_index()
)


# ============================================================
# 13. FROZEN IMPLEMENTATION SEMANTICS
# ============================================================

FEATURE_IMPLEMENTATION = {
    # --------------------------------------------------------
    # Stored A1-R3 counts
    # --------------------------------------------------------

    "proposal_count": {
        "dtype":
            "int64",

        "formula":
            (
                "COUNT_ACCEPTED_RAW_PROPOSALS_FOR_"
                "FROZEN_CANDIDATE"
            ),
    },

    "g0_count": {
        "dtype":
            "int64",

        "formula":
            (
                "COUNT_ACCEPTED_RAW_PROPOSALS_"
                "WHERE_FAMILY_G0"
            ),
    },

    "g1_count": {
        "dtype":
            "int64",

        "formula":
            (
                "COUNT_ACCEPTED_RAW_PROPOSALS_"
                "WHERE_FAMILY_G1"
            ),
    },

    "g2_count": {
        "dtype":
            "int64",

        "formula":
            (
                "COUNT_ACCEPTED_RAW_PROPOSALS_"
                "WHERE_FAMILY_G2"
            ),
    },

    "g2_s2t_count": {
        "dtype":
            "int64",

        "formula":
            (
                "COUNT_ACCEPTED_G2_RAW_PROPOSALS_WHERE_"
                "g2_source_to_target_knn_TRUE"
            ),
    },

    "g2_t2s_count": {
        "dtype":
            "int64",

        "formula":
            (
                "COUNT_ACCEPTED_G2_RAW_PROPOSALS_WHERE_"
                "g2_target_to_source_knn_TRUE"
            ),
    },

    # --------------------------------------------------------
    # Derived family support
    # --------------------------------------------------------

    "has_g0": {
        "dtype":
            "int8",

        "formula":
            "INT8(g0_count > 0)",
    },

    "has_g1": {
        "dtype":
            "int8",

        "formula":
            "INT8(g1_count > 0)",
    },

    "has_g2": {
        "dtype":
            "int8",

        "formula":
            "INT8(g2_count > 0)",
    },

    "family_count": {
        "dtype":
            "int8",

        "formula":
            "has_g0 + has_g1 + has_g2",
    },

    "multi_family": {
        "dtype":
            "int8",

        "formula":
            "INT8(family_count > 1)",
    },

    # --------------------------------------------------------
    # Critical clarification
    # --------------------------------------------------------

    "g2_bidirectional_support": {
        "dtype":
            "int64",

        "formula":
            (
                "COUNT_ACCEPTED_G2_RAW_PROPOSALS_FOR_"
                "THE_FROZEN_CANDIDATE_WHERE_"
                "g2_source_to_target_knn_TRUE_AND_"
                "g2_target_to_source_knn_TRUE"
            ),

        "semantic_unit":
            "SAME_RAW_SOURCE_TARGET_PAIR",

        "directional_min_definition":
            "FORBIDDEN",

        "directional_min_reason":
            (
                "CAN_COMBINE_OPPOSITE_DIRECTION_SUPPORT_"
                "FROM_DIFFERENT_RAW_G2_PROPOSALS"
            ),
    },

    # --------------------------------------------------------
    # Raw G2 physical geometry
    # --------------------------------------------------------

    "g2_pair_distance_um_min": {
        "dtype":
            "float64",

        "formula":
            (
                "MIN_g2_pair_distance_um_OVER_ACCEPTED_G2_"
                "RAW_PROPOSALS_FOR_FROZEN_CANDIDATE"
            ),

        "if_no_g2":
            "NaN",
    },

    "g2_pair_distance_um_median": {
        "dtype":
            "float64",

        "formula":
            (
                "MEDIAN_g2_pair_distance_um_OVER_ACCEPTED_G2_"
                "RAW_PROPOSALS_FOR_FROZEN_CANDIDATE"
            ),

        "if_no_g2":
            "NaN",
    },

    "g2_pair_distance_um_mean": {
        "dtype":
            "float64",

        "formula":
            (
                "MEAN_g2_pair_distance_um_OVER_ACCEPTED_G2_"
                "RAW_PROPOSALS_FOR_FROZEN_CANDIDATE"
            ),

        "if_no_g2":
            "NaN",
    },

    "g2_pair_distance_um_max": {
        "dtype":
            "float64",

        "formula":
            (
                "MAX_g2_pair_distance_um_OVER_ACCEPTED_G2_"
                "RAW_PROPOSALS_FOR_FROZEN_CANDIDATE"
            ),

        "if_no_g2":
            "NaN",
    },
}


assert set(
    FEATURE_IMPLEMENTATION.keys()
) == set(
    a3[
        "features"
    ][
        "proposal_support_features"
    ]
    +
    a3[
        "features"
    ][
        "g2_raw_geometry_features"
    ]
)


# ============================================================
# 14. FORMAL C1 ADDENDUM
# ============================================================

C1 = {
    "stage":
        "12.13A3-C1",

    "status":
        "IMPLEMENTATION_CLARIFICATION_FROZEN",

    "branch_id":
        BRANCH_ID,

    "candidate_universe_id":
        CANDIDATE_UNIVERSE_ID,

    "purpose":
        (
            "FREEZE_EXACT_GT_BLIND_IMPLEMENTATION_SEMANTICS_"
            "FOR_A3_PROPOSAL_DERIVED_FEATURES_BEFORE_A4_"
            "FEATURE_MATERIALIZATION_AND_MODEL_TRAINING"
        ),

    "parents": {
        "A3_contract_content_sha256":
            EXPECTED_A3_CONTENT_SHA,

        "A3_contract_file_sha256":
            EXPECTED_A3_FILE_SHA,

        "A1_R3_raw_proposals_file_sha256":
            EXPECTED_RAW_FILE_SHA,

        "A1_R3_candidate_universe_file_sha256":
            EXPECTED_UNIVERSE_FILE_SHA,
    },

    "clarification_trigger": {
        "GT_blind_preflight":
            "12.13A4-P1",

        "raw_intersection_sum":
            EXPECTED_RAW_INTERSECTION_SUM,

        "directional_min_sum":
            EXPECTED_DIRECTIONAL_MIN_SUM,

        "both_direction_present_candidates":
            EXPECTED_BOTH_DIRECTIONS_PRESENT,

        "divergent_candidate_count":
            EXPECTED_DIVERGENT_CANDIDATES,

        "ambiguity_observationally_irrelevant":
            False,
    },

    "decision": {
        "g2_bidirectional_support":
            (
                "SAME_RAW_G2_PROPOSAL_RECIPROCAL_"
                "INTERSECTION_COUNT"
            ),

        "formula":
            (
                "SUM_OVER_ACCEPTED_RAW_G2_PROPOSALS_FOR_"
                "CANDIDATE_OF_INT("
                "g2_source_to_target_knn AND "
                "g2_target_to_source_knn)"
            ),

        "dtype":
            "int64",

        "min_directional_counts":
            "EXPLICITLY_FORBIDDEN",

        "reason":
            (
                "RECIPROCAL_SUPPORT_IS_PAIR_LEVEL;_"
                "MIN_DIRECTIONAL_COUNTS_CAN_PAIR_DISTINCT_"
                "RAW_PROPOSALS_AND_CREATE_FALSE_RECIPROCITY"
            ),
    },

    "all_proposal_derived_feature_implementations":
        FEATURE_IMPLEMENTATION,

    "scope": {
        "changes_A3_feature_names":
            False,

        "changes_ranking_target":
            False,

        "changes_scorer_families":
            False,

        "changes_LOFO":
            False,

        "changes_threshold_selection":
            False,

        "changes_attachment_contract":
            False,

        "changes_graph_promotion_gate":
            False,

        "implementation_clarification_only":
            True,
    },

    "protocol": {
        "GT_read":
            False,

        "oracle_read":
            False,

        "model_trained":
            False,

        "model_selected":
            False,

        "threshold_selected":
            False,

        "graph_mutated":
            False,

        "Fold0_used":
            False,

        "clarification_before_feature_materialization":
            True,

        "clarification_before_model_training":
            True,
    },

    "required_downstream_parent_lock": {
        "A4_must_parent_lock_A3":
            True,

        "A4_must_parent_lock_A3_C1":
            True,

        "A4_g2_bidirectional_support_definition":
            (
                "SAME_RAW_G2_PROPOSAL_RECIPROCAL_"
                "INTERSECTION_COUNT"
            ),
    },

    "next":
        "12.13A4_FORMAL_GT_BLIND_FEATURE_MATRIX_MATERIALIZATION",
}


# ============================================================
# 15. CONTENT HASH
# ============================================================

C1_CONTENT_SHA = hashlib.sha256(
    json.dumps(
        C1,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


C1[
    "clarification_sha256"
] = C1_CONTENT_SHA


# ============================================================
# 16. TEMP WRITE
# ============================================================

TMP_PATH = C1_PATH.with_name(
    C1_PATH.name
    +
    ".tmp_12_13a3_c1"
)


if TMP_PATH.exists():

    raise RuntimeError(
        "\nStale A3-C1 temporary artifact exists:\n"
        f"{TMP_PATH}"
    )


TMP_PATH.write_text(
    json.dumps(
        C1,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 17. RELOAD / INTERNAL HASH VERIFICATION
# ============================================================

check = json.loads(
    TMP_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    check[
        "clarification_sha256"
    ]
    ==
    C1_CONTENT_SHA
)


payload = dict(
    check
)


embedded = payload.pop(
    "clarification_sha256"
)


recomputed = hashlib.sha256(
    json.dumps(
        payload,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


assert embedded == recomputed


assert check[
    "decision"
][
    "g2_bidirectional_support"
] == (
    "SAME_RAW_G2_PROPOSAL_RECIPROCAL_"
    "INTERSECTION_COUNT"
)


assert check[
    "decision"
][
    "min_directional_counts"
] == "EXPLICITLY_FORBIDDEN"


assert check[
    "protocol"
][
    "GT_read"
] is False


assert check[
    "protocol"
][
    "oracle_read"
] is False


assert check[
    "protocol"
][
    "model_trained"
] is False


assert check[
    "protocol"
][
    "threshold_selected"
] is False


assert check[
    "protocol"
][
    "graph_mutated"
] is False


assert check[
    "protocol"
][
    "Fold0_used"
] is False


# ============================================================
# 18. ATOMIC PROMOTION
# ============================================================

TMP_PATH.replace(
    C1_PATH
)


C1_FILE_SHA = sha256_file(
    C1_PATH
)


# ============================================================
# 19. FINAL READBACK
# ============================================================

formal = json.loads(
    C1_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    formal[
        "clarification_sha256"
    ]
    ==
    C1_CONTENT_SHA
)


assert formal[
    "next"
] == "12.13A4_FORMAL_GT_BLIND_FEATURE_MATRIX_MATERIALIZATION"


# ============================================================
# 20. FINAL
# ============================================================

print(
    "\n========== 12.13A3-C1 FINAL =========="
)


print(
    "G2_BIDIRECTIONAL_SUPPORT_IMPLEMENTATION_CLARIFICATION_FREEZE:",
    "PASS"
)


print(
    "\n--- PARENT ---"
)


print(
    "A3 content SHA:",
    EXPECTED_A3_CONTENT_SHA
)


print(
    "A3 feature set changed:",
    "NO"
)


print(
    "\n--- OBSERVED AMBIGUITY ---"
)


print(
    "same-proposal reciprocal sum:",
    EXPECTED_RAW_INTERSECTION_SUM
)


print(
    "directional-min sum:",
    EXPECTED_DIRECTIONAL_MIN_SUM
)


print(
    "divergent candidates:",
    EXPECTED_DIVERGENT_CANDIDATES
)


print(
    "\n--- FROZEN DEFINITION ---"
)


print(
    "g2_bidirectional_support ="
)


print(
    "COUNT accepted G2 raw proposals where BOTH "
    "direction flags are True on the SAME raw proposal"
)


print(
    "directional min definition:",
    "FORBIDDEN"
)


print(
    "dtype:",
    "int64"
)


print(
    "\n--- OTHER PROPOSAL FEATURES ---"
)


print(
    "all 16 proposal/G2-derived feature implementations:",
    "FROZEN"
)


print(
    "\n--- SAFETY ---"
)


print(
    "GT read:",
    "NO"
)


print(
    "oracle read:",
    "NO"
)


print(
    "model trained:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "graph mutation:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "\n--- HASH ---"
)


print(
    "clarification content SHA:",
    C1_CONTENT_SHA
)


print(
    "clarification file SHA:",
    C1_FILE_SHA
)


print(
    "\nnext:"
)


print(
    "12.13A4_FORMAL_GT_BLIND_FEATURE_MATRIX_MATERIALIZATION"
)

In [ ]:
# ============================================================
# Stage 12.13A4
#
# FORMAL GT-BLIND FEATURE MATRIX MATERIALIZATION
#
# ============================================================
#
# PARENTS
#
#   12.13A3
#     ranking + attachment development contract
#
#   12.13A3-C1
#     proposal/G2 feature implementation clarification
#
#   12.13A1-R3
#     frozen GT-blind candidate universe + raw proposals
#
# HISTORICAL SOURCE
#
#   Stage12.12 F1
#   cell 80
#
# ------------------------------------------------------------
# HARD GATES
# ------------------------------------------------------------
#
# GATE 1:
#   exact historical source SHA lock
#
# GATE 2:
#   rebuild OLD 415,590 candidate temporal features
#   and reproduce ALL 23 historical F1 features exactly
#
# GATE 3:
#   only after Gate 2 succeeds:
#   build NEW 1,079,885 candidate temporal features
#
# GATE 4:
#   mechanically derive C1-frozen 16 proposal/G2 features
#
# FINAL:
#
#   metadata:
#       candidate_id,dataset,fold,t,z,y,x
#
#   model features:
#       exactly 39
#
#   GT/oracle:
#       NONE
#
# ------------------------------------------------------------
# NO MODEL TRAINING
# NO THRESHOLD SELECTION
# NO GRAPH MUTATION
# NO FOLD0
#
# KEEP PERMANENTLY.
# ============================================================


from pathlib import Path

import ast
import hashlib
import inspect
import json
import sys
import gc

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)

NOTEBOOK_PATH = (
    PROJECT
    / "notebooks"
    / "12_residual_edge_recovery.ipynb"
)


A3_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_ranking_attachment_contract_frozen.json"
)

C1_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_ranking_attachment_contract_c1_g2_bidirectional_support_clarification_frozen.json"
)


NEW_UNIVERSE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_universe_v1.pkl"
)

RAW_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_raw_proposals_v1.pkl"
)


OLD_F1_PATH = (
    S12
    / "stage12_post_r1p50_node_f1_temporal_geometry_v1.pkl"
)


DEV_ACTIONS_PATH = (
    S12
    / "stage12_folds14_rewire_asym1x5_q9990_no_target_cut_v1_actions.pkl"
)


# ------------------------------------------------------------
# FORMAL A4 OUTPUTS
# ------------------------------------------------------------

FEATURE_MATRIX_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_gt_blind_feature_matrix_v1.pkl"
)

DATASET_SUMMARY_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_gt_blind_feature_matrix_v1_dataset_summary.pkl"
)

SCHEMA_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_gt_blind_feature_matrix_v1_schema_frozen.json"
)

SUMMARY_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_gt_blind_feature_matrix_v1_summary.json"
)


for path in [
    NOTEBOOK_PATH,
    A3_PATH,
    C1_PATH,
    NEW_UNIVERSE_PATH,
    RAW_PATH,
    OLD_F1_PATH,
    DEV_ACTIONS_PATH,
]:
    assert path.exists(), path


for path in [
    FEATURE_MATRIX_PATH,
    DATASET_SUMMARY_PATH,
    SCHEMA_PATH,
    SUMMARY_PATH,
]:
    if path.exists():
        raise RuntimeError(
            "\nFormal 12.13A4 artifact already exists:\n"
            f"{path}\n\n"
            "Do NOT overwrite formal provenance."
        )


# ============================================================
# 1. EXACT PARENT HASHES
# ============================================================

EXPECTED_A3_CONTENT_SHA = (
    "c54eda46073582a7a915f1bfa7dcbff9"
    "eec8a96edcfad80a8e49e1097cf38b46"
)

EXPECTED_A3_FILE_SHA = (
    "37a5b2e557832774c0bcf65a33c81d8"
    "434bac2025c4eebbf2e3b0ea1bc955bd7"
)

EXPECTED_C1_CONTENT_SHA = (
    "94407e96316eddf11e7930b2807c8664"
    "4d9b49eb802d7757585ec18a5c241e97"
)

EXPECTED_C1_FILE_SHA = (
    "416414260749db71770bcd1fe96cc161d"
    "911448d27c023fde23a4e31ed445c3a"
)

EXPECTED_NEW_UNIVERSE_SHA = (
    "440676497d3365a0926deaf45140f573"
    "32c8e9e89d46b88401a7ce6f7a299fc3"
)

EXPECTED_RAW_SHA = (
    "bd5598c3c442600d19d82ac5566326b"
    "b9b09c055115727580c3a469f0e3b762b"
)

EXPECTED_OLD_F1_SHA = (
    "8c3962bf6cef48617e1bd9ef23bd5e6d"
    "eb2438b80b6916f87d171516be92d0cc"
)

EXPECTED_DEV_ACTIONS_SHA = (
    "39e7261bbcfac40304c441be98128b0e"
    "9d9bcce4e9ce8d98beea7f69cf238b79"
)


EXPECTED_NEW_ROWS = 1_079_885
EXPECTED_OLD_ROWS = 415_590

EXPECTED_RAW_ROWS = 1_127_600
EXPECTED_ACCEPTED_RAW_ROWS = 1_081_627

EXPECTED_ACTIONS = 12_515
EXPECTED_REMOVALS = 385
EXPECTED_ADDITIONS = 12_515
EXPECTED_NET_EDGE_DELTA = 12_130


# ============================================================
# 2. HISTORICAL SOURCE LOCKS
# ============================================================

EXPECTED_PRODUCER_CELL = 80

EXPECTED_PRODUCER_SHA = (
    "46f26e931c97d6bcc60e83ad55c884f8"
    "83bf7af962935fac3839a5463d6822f6"
)

EXPECTED_FULL_LOOP_SHA = (
    "9d225364f78a6c369456724bd6a2998a"
    "fc3fa1186fc4eca22059550b6fde9b67"
)

EXPECTED_FEATURE_VALUE_DICT_SHA = (
    "bb10a2bd64e72468810e4fa1409c76d8"
    "11c7bd571b33bbc1d47259ec362598bf"
)

EXPECTED_FEATURE_ASSIGN_SHA = (
    "b95950ccc8a42e06e40bd28e1434008d"
    "871119c3bb1ff5fe55010cc3008a7530"
)

EXPECTED_BUILD_POST_G3H3_SHA = (
    "567a20680b0409236c465113f49b125cc"
    "3f7dbe6c8f27e1e8b2449465b6bb1ad"
)

EXPECTED_QUERY_FRAME_SHA = (
    "16e14a76082495487b73b574f680f077"
    "3484c4802300ab4312d9b029813db205"
)

EXPECTED_STEP_BALANCE_SHA = (
    "dc1a312f65a5e6d29b913ea5cbe329d5"
    "786060c42a8a567167af0b4d8cb2a209"
)


# ============================================================
# 3. FEATURE CONTRACT
# ============================================================

TEMPORAL_FEATURES = [
    "has_prev_frame",
    "has_next_frame",

    "same_nearest_dist_um",
    "same_second_dist_um",
    "same_neighbor_margin_um",

    "prev_nearest_dist_um",
    "prev_second_dist_um",
    "prev_neighbor_margin_um",

    "next_nearest_dist_um",
    "next_second_dist_um",
    "next_neighbor_margin_um",

    "prev_nearest_outdegree",
    "next_nearest_indegree",

    "prev_nearest_free_out",
    "next_nearest_free_in",

    "bilateral_available",

    "prev_next_span_um",
    "candidate_midpoint_residual_um",

    "prev_next_step_mean_um",
    "prev_next_step_max_um",
    "prev_next_step_min_um",
    "prev_next_step_absdiff_um",

    "step_balance",
]


PROPOSAL_FEATURES = [
    "proposal_count",

    "g0_count",
    "g1_count",
    "g2_count",

    "g2_s2t_count",
    "g2_t2s_count",

    "has_g0",
    "has_g1",
    "has_g2",

    "family_count",
    "multi_family",

    "g2_bidirectional_support",
]


G2_GEOMETRY_FEATURES = [
    "g2_pair_distance_um_min",
    "g2_pair_distance_um_median",
    "g2_pair_distance_um_mean",
    "g2_pair_distance_um_max",
]


MODEL_FEATURES = (
    PROPOSAL_FEATURES
    +
    G2_GEOMETRY_FEATURES
    +
    TEMPORAL_FEATURES
)


METADATA_COLUMNS = [
    "candidate_id",
    "dataset",
    "fold",
    "t",
    "z",
    "y",
    "x",
]


assert len(TEMPORAL_FEATURES) == 23
assert len(PROPOSAL_FEATURES) == 12
assert len(G2_GEOMETRY_FEATURES) == 4
assert len(MODEL_FEATURES) == 39
assert len(set(MODEL_FEATURES)) == 39


# ============================================================
# 4. HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def source_sha(source):

    return hashlib.sha256(
        source.encode("utf-8")
    ).hexdigest()


def cell_source(cell):

    return "".join(
        cell.get(
            "source",
            [],
        )
    )


def source_segment(
    source,
    node,
):

    return (
        ast.get_source_segment(
            source,
            node,
        )
        or ast.unparse(node)
    )


def dataframe_content_sha256(
    df,
    columns,
):
    """
    Historical-compatible deterministic CSV content hash.
    """

    class HashWriter:

        def __init__(self):
            self.h = hashlib.sha256()

        def write(self, text):

            if not isinstance(
                text,
                str,
            ):
                text = str(text)

            encoded = text.encode(
                "utf-8"
            )

            self.h.update(encoded)

            return len(text)

        def flush(self):
            return None


    writer = HashWriter()

    df.loc[
        :,
        columns,
    ].to_csv(
        writer,
        index=False,
        na_rep="NaN",
        float_format="%.17g",
        lineterminator="\n",
    )

    return writer.h.hexdigest()


# ============================================================
# 5. PARENT CONTRACT LOCK
# ============================================================

assert sha256_file(
    A3_PATH
) == EXPECTED_A3_FILE_SHA

assert sha256_file(
    C1_PATH
) == EXPECTED_C1_FILE_SHA


a3 = json.loads(
    A3_PATH.read_text(
        encoding="utf-8"
    )
)

c1 = json.loads(
    C1_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    a3[
        "contract_sha256"
    ]
    ==
    EXPECTED_A3_CONTENT_SHA
)

assert (
    c1[
        "clarification_sha256"
    ]
    ==
    EXPECTED_C1_CONTENT_SHA
)


assert a3[
    "stage"
] == "12.13A3"

assert c1[
    "stage"
] == "12.13A3-C1"


assert a3[
    "features"
][
    "allowed"
] == MODEL_FEATURES


assert c1[
    "decision"
][
    "g2_bidirectional_support"
] == (
    "SAME_RAW_G2_PROPOSAL_RECIPROCAL_"
    "INTERSECTION_COUNT"
)


assert c1[
    "decision"
][
    "min_directional_counts"
] == "EXPLICITLY_FORBIDDEN"


assert a3[
    "protocol"
][
    "model_trained"
] is False

assert a3[
    "protocol"
][
    "threshold_selected"
] is False

assert a3[
    "protocol"
][
    "Fold0_used"
] is False


assert c1[
    "protocol"
][
    "GT_read"
] is False

assert c1[
    "protocol"
][
    "oracle_read"
] is False

assert c1[
    "protocol"
][
    "model_trained"
] is False

assert c1[
    "protocol"
][
    "Fold0_used"
] is False


print(
    "========== 12.13A4 PARENT CONTRACT LOCK =========="
)

print(
    "A3:",
    "PASS"
)

print(
    "A3-C1:",
    "PASS"
)

print(
    "model features:",
    len(MODEL_FEATURES)
)

print(
    "model trained:",
    "NO"
)

print(
    "Fold0:",
    "NO"
)

print(
    "A4_PARENT_CONTRACT_LOCK: PASS"
)


# ============================================================
# 6. INPUT ARTIFACT LOCK
# ============================================================

assert sha256_file(
    NEW_UNIVERSE_PATH
) == EXPECTED_NEW_UNIVERSE_SHA

assert sha256_file(
    RAW_PATH
) == EXPECTED_RAW_SHA

assert sha256_file(
    OLD_F1_PATH
) == EXPECTED_OLD_F1_SHA

assert sha256_file(
    DEV_ACTIONS_PATH
) == EXPECTED_DEV_ACTIONS_SHA


new_universe = pd.read_pickle(
    NEW_UNIVERSE_PATH
)

raw = pd.read_pickle(
    RAW_PATH
)

old_f1 = pd.read_pickle(
    OLD_F1_PATH
)

dev_actions = pd.read_pickle(
    DEV_ACTIONS_PATH
)


assert len(new_universe) == EXPECTED_NEW_ROWS
assert len(raw) == EXPECTED_RAW_ROWS
assert len(old_f1) == EXPECTED_OLD_ROWS
assert len(dev_actions) == EXPECTED_ACTIONS


assert new_universe[
    "candidate_id"
].is_unique


assert np.array_equal(
    new_universe[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        EXPECTED_NEW_ROWS,
        dtype=np.int64,
    ),
)


assert set(
    new_universe[
        "fold"
    ].astype(int).unique()
) == {
    1,
    2,
    3,
    4,
}


assert not (
    new_universe[
        "fold"
    ].astype(int)
    ==
    0
).any()


for column in new_universe.columns:

    lower = str(
        column
    ).lower()

    assert "oracle" not in lower
    assert not lower.startswith("gt_")


print(
    "\n========== INPUT ARTIFACT LOCK =========="
)

print(
    "new universe rows:",
    len(new_universe)
)

print(
    "raw proposals:",
    len(raw)
)

print(
    "old F1 rows:",
    len(old_f1)
)

print(
    "G3H3 actions:",
    len(dev_actions)
)

print(
    "GT/oracle in new universe:",
    "NO"
)

print(
    "INPUT_ARTIFACT_LOCK: PASS"
)


# ============================================================
# 7. SOURCE LOCK
# ============================================================

notebook = json.loads(
    NOTEBOOK_PATH.read_text(
        encoding="utf-8"
    )
)

cells = notebook[
    "cells"
]


producer_source = cell_source(
    cells[
        EXPECTED_PRODUCER_CELL
    ]
)


assert source_sha(
    producer_source
) == EXPECTED_PRODUCER_SHA


tree = ast.parse(
    producer_source
)


function_map = {}


for node in tree.body:

    if not isinstance(
        node,
        ast.FunctionDef,
    ):
        continue

    source = source_segment(
        producer_source,
        node,
    )

    function_map[
        node.name
    ] = {
        "source":
            source,

        "sha":
            source_sha(source),

        "node":
            node,
    }


assert (
    function_map[
        "build_dev_post_g3h3_frozen"
    ][
        "sha"
    ]
    ==
    EXPECTED_BUILD_POST_G3H3_SHA
)


assert (
    function_map[
        "query_frame"
    ][
        "sha"
    ]
    ==
    EXPECTED_QUERY_FRAME_SHA
)


# ------------------------------------------------------------
# Exact feature-value dict + assignment + enclosing loop
# ------------------------------------------------------------

feature_value_dict = None
feature_assignment = None
full_loop = None


for node in ast.walk(tree):

    if isinstance(
        node,
        ast.Dict,
    ):

        keys = []

        for key in node.keys:

            if (
                isinstance(
                    key,
                    ast.Constant,
                )
                and
                isinstance(
                    key.value,
                    str,
                )
            ):
                keys.append(
                    key.value
                )


        if set(
            TEMPORAL_FEATURES
        ).issubset(
            set(keys)
        ):

            source = source_segment(
                producer_source,
                node,
            )

            if (
                source_sha(source)
                ==
                EXPECTED_FEATURE_VALUE_DICT_SHA
            ):
                feature_value_dict = node


assert feature_value_dict is not None


for node in ast.walk(tree):

    if not isinstance(
        node,
        ast.Assign,
    ):
        continue

    if (
        int(node.lineno)
        <=
        int(feature_value_dict.lineno)
        and
        int(node.end_lineno)
        >=
        int(feature_value_dict.end_lineno)
    ):

        source = source_segment(
            producer_source,
            node,
        )

        if (
            source_sha(source)
            ==
            EXPECTED_FEATURE_ASSIGN_SHA
        ):
            feature_assignment = node


assert feature_assignment is not None


for node in tree.body:

    if not isinstance(
        node,
        ast.For,
    ):
        continue

    source = source_segment(
        producer_source,
        node,
    )

    if (
        source_sha(source)
        ==
        EXPECTED_FULL_LOOP_SHA
    ):
        full_loop = node


assert full_loop is not None


# ============================================================
# 8. frozen_v9.step_balance LOCK
# ============================================================

SRC_PATH = (
    PROJECT
    / "src"
)


if str(SRC_PATH) not in sys.path:

    sys.path.insert(
        0,
        str(SRC_PATH),
    )


from biohub_cell_tracking import frozen_v9 as fv9


step_balance_source = inspect.getsource(
    fv9.step_balance
)


assert (
    source_sha(
        step_balance_source
    )
    ==
    EXPECTED_STEP_BALANCE_SHA
)


print(
    "\n========== HISTORICAL SOURCE LOCK =========="
)

print(
    "producer cell:",
    EXPECTED_PRODUCER_CELL
)

print(
    "producer SHA:",
    EXPECTED_PRODUCER_SHA
)

print(
    "full loop SHA:",
    EXPECTED_FULL_LOOP_SHA
)

print(
    "feature dict SHA:",
    EXPECTED_FEATURE_VALUE_DICT_SHA
)

print(
    "feature assignment SHA:",
    EXPECTED_FEATURE_ASSIGN_SHA
)

print(
    "build post-G3H3 SHA:",
    EXPECTED_BUILD_POST_G3H3_SHA
)

print(
    "query_frame SHA:",
    EXPECTED_QUERY_FRAME_SHA
)

print(
    "step_balance SHA:",
    EXPECTED_STEP_BALANCE_SHA
)

print(
    "HISTORICAL_F1_SOURCE_LOCK: PASS"
)


# ============================================================
# 9. EXACT query_frame
#
# Execute exact historical source, not a handwritten copy.
# ============================================================

query_namespace = {
    "np":
        np,
}


exec(
    function_map[
        "query_frame"
    ][
        "source"
    ],
    query_namespace,
)


query_frame = query_namespace[
    "query_frame"
]


# ============================================================
# 10. PREPARE EXACT POST-G3H3 BUILDER
#
# A1-R3 / historical notebook state already established the
# exact pre-G3H3 dev builder.
#
# Do NOT regenerate it from deployment threshold.
# ============================================================

if (
    "build_dev_current_best"
    not in globals()
):

    raise RuntimeError(
        "\nExact historical build_dev_current_best is not present "
        "in the current notebook kernel.\n\n"
        "Do NOT substitute stage12_confirmed_rewire deployment "
        "runtime or its threshold.\n"
        "Restore the exact successful Stage12 development-builder "
        "bootstrap used by A1-R3/cell30, then rerun this A4 cell."
    )


# ------------------------------------------------------------
# Frozen G3H3 action table
# ------------------------------------------------------------

actions_by_dataset = {
    str(dataset):
        sub.copy()

    for dataset, sub
    in dev_actions.groupby(
        "dataset",
        sort=False,
    )
}


empty_actions = dev_actions.iloc[
    0:0
].copy()


# ============================================================
# 11. RECOVER EXACT CELL66 MUTATION HELPERS
# ============================================================

CELL66_INDEX = 66

EXPECTED_CELL66_SHA = (
    "368a75fee5490aa27be91957eb2ae1cf8"
    "281b4fb5409ddc795c73dfe4bc0555a"
)


cell66_source = cell_source(
    cells[
        CELL66_INDEX
    ]
)


assert source_sha(
    cell66_source
) == EXPECTED_CELL66_SHA


cell66_tree = ast.parse(
    cell66_source
)


mutation_sources = {}


for node in cell66_tree.body:

    if (
        isinstance(
            node,
            ast.FunctionDef,
        )
        and
        node.name
        in {
            "local_graph_state",
            "apply_locked_fold0_policy",
        }
    ):

        mutation_sources[
            node.name
        ] = source_segment(
            cell66_source,
            node,
        )


assert set(
    mutation_sources
) == {
    "local_graph_state",
    "apply_locked_fold0_policy",
}


mutation_namespace = {
    "np":
        np,

    "pd":
        pd,

    "fv9":
        fv9,
}


exec(
    mutation_sources[
        "local_graph_state"
    ],
    mutation_namespace,
)


exec(
    mutation_sources[
        "apply_locked_fold0_policy"
    ],
    mutation_namespace,
)


exact_apply_frozen_policy = (
    mutation_namespace[
        "apply_locked_fold0_policy"
    ]
)


# ============================================================
# 12. EXACT build_dev_post_g3h3_frozen
# ============================================================

build_namespace = {
    "np":
        np,

    "build_dev_current_best":
        globals()[
            "build_dev_current_best"
        ],

    "actions_by_dataset":
        actions_by_dataset,

    "empty_actions":
        empty_actions,

    "exact_apply_frozen_policy":
        exact_apply_frozen_policy,
}


exec(
    function_map[
        "build_dev_post_g3h3_frozen"
    ][
        "source"
    ],
    build_namespace,
)


build_dev_post_g3h3_frozen = (
    build_namespace[
        "build_dev_post_g3h3_frozen"
    ]
)


# ============================================================
# 13. TEMPORAL FEATURE MATERIALIZER
#
# Mechanical transcription of exact cell80 numerical loop.
#
# HARD OLD-F1 replay below proves exact semantics before this
# is ever applied to NEW candidates.
# ============================================================

def materialize_temporal_features(
    candidate,
    *,
    label,
):

    candidate = candidate.loc[
        :,
        METADATA_COLUMNS,
    ].copy()


    assert candidate[
        "candidate_id"
    ].is_unique


    parts = []

    dataset_rows = []


    total_actions = 0
    total_removed = 0
    total_added = 0
    total_net_delta = 0


    datasets = sorted(
        candidate[
            "dataset"
        ].astype(str).unique()
    )


    for dataset_index, dataset in enumerate(
        datasets,
        start=1,
    ):

        sub = (
            candidate.loc[
                candidate[
                    "dataset"
                ].astype(str).eq(
                    dataset
                )
            ]
            .copy()
            .sort_values(
                "candidate_id",
                kind="stable",
            )
            .reset_index(
                drop=True
            )
        )


        (
            graph,
            scale,
            mutation_stats,
        ) = build_dev_post_g3h3_frozen(
            dataset
        )


        scale = np.asarray(
            scale,
            dtype=np.float64,
        )


        assert scale.shape == (
            3,
        )

        assert np.isfinite(
            scale
        ).all()

        assert (
            scale
            >
            0
        ).all()


        total_actions += int(
            mutation_stats[
                "actions"
            ]
        )

        total_removed += int(
            mutation_stats[
                "source_incumbent_edges_removed"
            ]
        )

        total_added += int(
            mutation_stats[
                "edges_added"
            ]
        )

        total_net_delta += int(
            mutation_stats[
                "net_edge_delta"
            ]
        )


        # ====================================================
        # Exact post-G3H3 node state
        # ====================================================

        attrs = (
            graph
            .node_attrs(
                unpack=True
            )
            .to_pandas()
            .copy()
        )


        assert {
            "node_id",
            "t",
            "z",
            "y",
            "x",
        }.issubset(
            attrs.columns
        )


        node_id = pd.to_numeric(
            attrs[
                "node_id"
            ],
            errors="raise",
        ).astype(
            np.int64
        ).to_numpy()


        node_t = pd.to_numeric(
            attrs[
                "t"
            ],
            errors="raise",
        ).astype(
            np.int64
        ).to_numpy()


        node_voxel = attrs[
            [
                "z",
                "y",
                "x",
            ]
        ].to_numpy(
            dtype=np.float64
        )


        assert np.isfinite(
            node_voxel
        ).all()


        node_phys = (
            node_voxel
            *
            scale[
                None,
                :
            ]
        )


        indegree = np.asarray(
            graph.in_degree(
                node_id.tolist()
            ),
            dtype=np.int64,
        )


        outdegree = np.asarray(
            graph.out_degree(
                node_id.tolist()
            ),
            dtype=np.int64,
        )


        # ====================================================
        # Exact candidate/current-node collision guard
        # ====================================================

        graph_keys = {
            (
                int(t),
                float(z),
                float(y),
                float(x),
            )

            for t, z, y, x
            in zip(
                node_t,
                node_voxel[
                    :,
                    0
                ],
                node_voxel[
                    :,
                    1
                ],
                node_voxel[
                    :,
                    2
                ],
            )
        }


        candidate_keys = [
            (
                int(row.t),
                float(row.z),
                float(row.y),
                float(row.x),
            )

            for row
            in sub.itertuples(
                index=False
            )
        ]


        exact_collision_count = sum(
            key in graph_keys
            for key in candidate_keys
        )


        assert (
            exact_collision_count
            ==
            0
        ), (
            dataset,
            exact_collision_count,
        )


        # ====================================================
        # KD tree by frame
        # ====================================================

        frame_info = {}


        for t in np.unique(
            node_t
        ):

            idx = np.flatnonzero(
                node_t
                ==
                t
            )


            if len(idx) == 0:
                continue


            frame_info[
                int(t)
            ] = {
                "indices":
                    idx,

                "tree":
                    cKDTree(
                        node_phys[
                            idx
                        ]
                    ),
            }


        candidate_voxel = sub[
            [
                "z",
                "y",
                "x",
            ]
        ].to_numpy(
            dtype=np.float64
        )


        candidate_phys = (
            candidate_voxel
            *
            scale[
                None,
                :
            ]
        )


        candidate_t = sub[
            "t"
        ].to_numpy(
            dtype=np.int64
        )


        n = len(sub)


        # ====================================================
        # Allocate exact historical arrays
        # ====================================================

        same_nearest = np.full(
            n,
            np.nan,
        )

        same_second = np.full(
            n,
            np.nan,
        )

        same_margin = np.full(
            n,
            np.nan,
        )


        prev_nearest = np.full(
            n,
            np.nan,
        )

        prev_second = np.full(
            n,
            np.nan,
        )

        prev_margin = np.full(
            n,
            np.nan,
        )


        next_nearest = np.full(
            n,
            np.nan,
        )

        next_second = np.full(
            n,
            np.nan,
        )

        next_margin = np.full(
            n,
            np.nan,
        )


        has_prev = np.zeros(
            n,
            dtype=np.int8,
        )

        has_next = np.zeros(
            n,
            dtype=np.int8,
        )


        prev_outdegree = np.full(
            n,
            np.nan,
        )

        next_indegree = np.full(
            n,
            np.nan,
        )


        prev_free_out = np.full(
            n,
            np.nan,
        )

        next_free_in = np.full(
            n,
            np.nan,
        )


        bilateral = np.zeros(
            n,
            dtype=np.int8,
        )


        prev_next_span = np.full(
            n,
            np.nan,
        )

        midpoint_residual = np.full(
            n,
            np.nan,
        )


        step_mean = np.full(
            n,
            np.nan,
        )

        step_maximum = np.full(
            n,
            np.nan,
        )

        step_minimum = np.full(
            n,
            np.nan,
        )

        step_absdiff = np.full(
            n,
            np.nan,
        )

        step_balance_values = np.full(
            n,
            np.nan,
        )


        # ====================================================
        # Exact frame-local computation
        # ====================================================

        for t in np.unique(
            candidate_t
        ):

            rows = np.flatnonzero(
                candidate_t
                ==
                t
            )


            query = candidate_phys[
                rows
            ]


            (
                d1,
                d2,
                dm,
                _same_global,
            ) = query_frame(
                frame_info.get(
                    int(t)
                ),
                query,
            )


            same_nearest[
                rows
            ] = d1

            same_second[
                rows
            ] = d2

            same_margin[
                rows
            ] = dm


            # ================================================
            # PREVIOUS
            # ================================================

            prev_info = frame_info.get(
                int(
                    t
                    -
                    1
                )
            )


            (
                p1,
                p2,
                pm,
                prev_global,
            ) = query_frame(
                prev_info,
                query,
            )


            prev_nearest[
                rows
            ] = p1

            prev_second[
                rows
            ] = p2

            prev_margin[
                rows
            ] = pm


            if prev_info is not None:

                has_prev[
                    rows
                ] = 1


                assert (
                    prev_global
                    >=
                    0
                ).all()


                prev_outdegree[
                    rows
                ] = outdegree[
                    prev_global
                ]


                prev_free_out[
                    rows
                ] = (
                    outdegree[
                        prev_global
                    ]
                    ==
                    0
                ).astype(
                    float
                )


            # ================================================
            # NEXT
            # ================================================

            next_info = frame_info.get(
                int(
                    t
                    +
                    1
                )
            )


            (
                n1,
                n2,
                nm,
                next_global,
            ) = query_frame(
                next_info,
                query,
            )


            next_nearest[
                rows
            ] = n1

            next_second[
                rows
            ] = n2

            next_margin[
                rows
            ] = nm


            if next_info is not None:

                has_next[
                    rows
                ] = 1


                assert (
                    next_global
                    >=
                    0
                ).all()


                next_indegree[
                    rows
                ] = indegree[
                    next_global
                ]


                next_free_in[
                    rows
                ] = (
                    indegree[
                        next_global
                    ]
                    ==
                    0
                ).astype(
                    float
                )


            # ================================================
            # BILATERAL
            # ================================================

            if (
                prev_info is not None
                and
                next_info is not None
            ):

                bilateral[
                    rows
                ] = 1


                prev_xyz = node_phys[
                    prev_global
                ]


                next_xyz = node_phys[
                    next_global
                ]


                cand_xyz = query


                d_prev = np.linalg.norm(
                    cand_xyz
                    -
                    prev_xyz,
                    axis=1,
                )


                d_next = np.linalg.norm(
                    next_xyz
                    -
                    cand_xyz,
                    axis=1,
                )


                prev_next_span[
                    rows
                ] = np.linalg.norm(
                    next_xyz
                    -
                    prev_xyz,
                    axis=1,
                )


                midpoint = (
                    prev_xyz
                    +
                    next_xyz
                ) / 2.0


                midpoint_residual[
                    rows
                ] = np.linalg.norm(
                    cand_xyz
                    -
                    midpoint,
                    axis=1,
                )


                step_mean[
                    rows
                ] = (
                    d_prev
                    +
                    d_next
                ) / 2.0


                step_maximum[
                    rows
                ] = np.maximum(
                    d_prev,
                    d_next,
                )


                step_minimum[
                    rows
                ] = np.minimum(
                    d_prev,
                    d_next,
                )


                step_absdiff[
                    rows
                ] = np.abs(
                    d_prev
                    -
                    d_next
                )


                step_balance_values[
                    rows
                ] = np.asarray(
                    [
                        float(
                            fv9.step_balance(
                                float(a),
                                float(b),
                            )
                        )

                        for a, b
                        in zip(
                            d_prev,
                            d_next,
                        )
                    ],
                    dtype=np.float64,
                )


        # ====================================================
        # Exact value mapping
        # ====================================================

        part = pd.DataFrame(
            {
                "candidate_id":
                    sub[
                        "candidate_id"
                    ].to_numpy(
                        dtype=np.int64
                    ),

                "dataset":
                    sub[
                        "dataset"
                    ].astype(
                        str
                    ).to_numpy(),

                "fold":
                    sub[
                        "fold"
                    ].to_numpy(
                        dtype=np.int8
                    ),

                "t":
                    sub[
                        "t"
                    ].to_numpy(
                        dtype=np.int64
                    ),

                "z":
                    sub[
                        "z"
                    ].to_numpy(
                        dtype=np.float64
                    ),

                "y":
                    sub[
                        "y"
                    ].to_numpy(
                        dtype=np.float64
                    ),

                "x":
                    sub[
                        "x"
                    ].to_numpy(
                        dtype=np.float64
                    ),

                "has_prev_frame":
                    has_prev,

                "has_next_frame":
                    has_next,

                "same_nearest_dist_um":
                    same_nearest,

                "same_second_dist_um":
                    same_second,

                "same_neighbor_margin_um":
                    same_margin,

                "prev_nearest_dist_um":
                    prev_nearest,

                "prev_second_dist_um":
                    prev_second,

                "prev_neighbor_margin_um":
                    prev_margin,

                "next_nearest_dist_um":
                    next_nearest,

                "next_second_dist_um":
                    next_second,

                "next_neighbor_margin_um":
                    next_margin,

                "prev_nearest_outdegree":
                    prev_outdegree,

                "next_nearest_indegree":
                    next_indegree,

                "prev_nearest_free_out":
                    prev_free_out,

                "next_nearest_free_in":
                    next_free_in,

                "bilateral_available":
                    bilateral,

                "prev_next_span_um":
                    prev_next_span,

                "candidate_midpoint_residual_um":
                    midpoint_residual,

                "prev_next_step_mean_um":
                    step_mean,

                "prev_next_step_max_um":
                    step_maximum,

                "prev_next_step_min_um":
                    step_minimum,

                "prev_next_step_absdiff_um":
                    step_absdiff,

                "step_balance":
                    step_balance_values,
            }
        )


        parts.append(
            part
        )


        dataset_rows.append(
            {
                "dataset":
                    dataset,

                "candidate_rows":
                    int(n),

                "graph_nodes":
                    int(
                        len(node_id)
                    ),

                "actions":
                    int(
                        mutation_stats[
                            "actions"
                        ]
                    ),

                "source_incumbent_edges_removed":
                    int(
                        mutation_stats[
                            "source_incumbent_edges_removed"
                        ]
                    ),

                "edges_added":
                    int(
                        mutation_stats[
                            "edges_added"
                        ]
                    ),

                "net_edge_delta":
                    int(
                        mutation_stats[
                            "net_edge_delta"
                        ]
                    ),

                "bilateral_available":
                    int(
                        bilateral.sum()
                    ),
            }
        )


        if (
            dataset_index == 1
            or
            dataset_index % 10 == 0
            or
            dataset_index == len(datasets)
        ):

            print(
                f"{label} temporal datasets:",
                f"{dataset_index}/{len(datasets)}",
                "|",
                dataset,
                "| candidates:",
                n,
            )


        del graph
        del attrs
        del frame_info

        gc.collect()


    result = (
        pd.concat(
            parts,
            ignore_index=True,
        )
        .sort_values(
            "candidate_id",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


    dataset_summary = pd.DataFrame(
        dataset_rows
    )


    assert result[
        "candidate_id"
    ].is_unique


    assert total_actions == EXPECTED_ACTIONS
    assert total_removed == EXPECTED_REMOVALS
    assert total_added == EXPECTED_ADDITIONS
    assert total_net_delta == EXPECTED_NET_EDGE_DELTA


    return (
        result,
        dataset_summary,
    )


# ============================================================
# 14. HARD HISTORICAL REPLAY GATE
#
# Use OLD F1 metadata only.
# No GT/oracle.
# ============================================================

print(
    "\n========== OLD F1 HISTORICAL REPLAY =========="
)


old_candidates = old_f1[
    METADATA_COLUMNS
].copy()


(
    old_replayed,
    old_dataset_summary,
) = materialize_temporal_features(
    old_candidates,
    label="OLD",
)


assert len(
    old_replayed
) == EXPECTED_OLD_ROWS


assert np.array_equal(
    old_replayed[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    old_f1[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
)


# ============================================================
# Exact metadata reproduction
# ============================================================

for column in METADATA_COLUMNS:

    a = old_replayed[
        column
    ]

    b = old_f1[
        column
    ]


    if pd.api.types.is_numeric_dtype(a):

        assert np.array_equal(
            a.to_numpy(),
            b.to_numpy(),
            equal_nan=True,
        ), column

    else:

        assert a.astype(str).equals(
            b.astype(str)
        ), column


# ============================================================
# Exact 23-feature reproduction
# ============================================================

replay_metrics = []


for column in TEMPORAL_FEATURES:

    expected = old_f1[
        column
    ].to_numpy()

    actual = old_replayed[
        column
    ].to_numpy()


    equal = np.array_equal(
        actual,
        expected,
        equal_nan=True,
    )


    if not equal:

        if (
            np.issubdtype(
                actual.dtype,
                np.number,
            )
            and
            np.issubdtype(
                expected.dtype,
                np.number,
            )
        ):

            finite = (
                np.isfinite(actual)
                &
                np.isfinite(expected)
            )


            max_abs = (
                float(
                    np.max(
                        np.abs(
                            actual[
                                finite
                            ]
                            -
                            expected[
                                finite
                            ]
                        )
                    )
                )
                if finite.any()
                else
                np.nan
            )


            mismatch = int(
                np.count_nonzero(
                    ~(
                        (
                            actual
                            ==
                            expected
                        )
                        |
                        (
                            np.isnan(actual)
                            &
                            np.isnan(expected)
                        )
                    )
                )
            )


        else:

            max_abs = None

            mismatch = int(
                np.count_nonzero(
                    actual
                    !=
                    expected
                )
            )


        raise AssertionError(
            "\nHistorical F1 replay mismatch:\n"
            f"feature={column}\n"
            f"mismatch_rows={mismatch}\n"
            f"max_abs_diff={max_abs}"
        )


    replay_metrics.append(
        {
            "feature":
                column,

            "exact_reproduction":
                True,
        }
    )


print(
    "OLD_F1_METADATA_REPRODUCTION: PASS"
)

print(
    "OLD_F1_23_FEATURE_EXACT_REPRODUCTION: PASS"
)

print(
    "features reproduced:",
    len(
        replay_metrics
    )
)

print(
    "HISTORICAL_F1_REPLAY_GATE: PASS"
)


# ============================================================
# 15. ONLY NOW BUILD NEW TEMPORAL FEATURES
# ============================================================

print(
    "\n========== NEW TEMPORAL FEATURE MATERIALIZATION =========="
)


new_candidates = new_universe[
    METADATA_COLUMNS
].copy()


(
    new_temporal,
    new_dataset_summary,
) = materialize_temporal_features(
    new_candidates,
    label="NEW",
)


assert len(
    new_temporal
) == EXPECTED_NEW_ROWS


assert np.array_equal(
    new_temporal[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    new_universe[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
)


print(
    "NEW_TEMPORAL_FEATURES:",
    "PASS"
)


# ============================================================
# 16. C1-FROZEN PROPOSAL FEATURE MATERIALIZATION
# ============================================================

print(
    "\n========== C1 PROPOSAL FEATURE MATERIALIZATION =========="
)


accepted = raw.loc[
    raw[
        "accepted"
    ].astype(
        bool
    )
].copy()


assert len(
    accepted
) == EXPECTED_ACCEPTED_RAW_ROWS


FAMILY_G0 = (
    "FORWARD_CONSTANT_VELOCITY_V1"
)

FAMILY_G1 = (
    "BACKWARD_CONSTANT_VELOCITY_V1"
)

FAMILY_G2 = (
    "BILATERAL_MIDPOINT_K5_V1"
)


RAW_KEY = [
    "dataset",
    "candidate_t",
    "key_z",
    "key_y",
    "key_x",
]


accepted[
    "dataset"
] = accepted[
    "dataset"
].astype(str)


accepted[
    "candidate_t"
] = pd.to_numeric(
    accepted[
        "candidate_t"
    ],
    errors="raise",
).astype(
    np.int64
)


for column in [
    "key_z",
    "key_y",
    "key_x",
]:

    accepted[
        column
    ] = pd.to_numeric(
        accepted[
            column
        ],
        errors="raise",
    ).astype(
        np.float64
    )


for column in [
    "g2_source_to_target_knn",
    "g2_target_to_source_knn",
]:

    accepted[
        column
    ] = accepted[
        column
    ].fillna(
        False
    ).astype(
        bool
    )


accepted[
    "_is_g0"
] = accepted[
    "family"
].eq(
    FAMILY_G0
)


accepted[
    "_is_g1"
] = accepted[
    "family"
].eq(
    FAMILY_G1
)


accepted[
    "_is_g2"
] = accepted[
    "family"
].eq(
    FAMILY_G2
)


accepted[
    "_g2_s2t"
] = (
    accepted[
        "_is_g2"
    ]
    &
    accepted[
        "g2_source_to_target_knn"
    ]
)


accepted[
    "_g2_t2s"
] = (
    accepted[
        "_is_g2"
    ]
    &
    accepted[
        "g2_target_to_source_knn"
    ]
)


# ------------------------------------------------------------
# C1 exact definition
# ------------------------------------------------------------

accepted[
    "_g2_bidirectional"
] = (
    accepted[
        "_is_g2"
    ]
    &
    accepted[
        "g2_source_to_target_knn"
    ]
    &
    accepted[
        "g2_target_to_source_knn"
    ]
)


# ============================================================
# 17. RAW COUNT AGGREGATION
# ============================================================

raw_agg = (
    accepted
    .groupby(
        RAW_KEY,
        sort=False,
        dropna=False,
    )
    .agg(
        proposal_count_raw=(
            "raw_proposal_id",
            "size",
        ),

        g0_count_raw=(
            "_is_g0",
            "sum",
        ),

        g1_count_raw=(
            "_is_g1",
            "sum",
        ),

        g2_count_raw=(
            "_is_g2",
            "sum",
        ),

        g2_s2t_count_raw=(
            "_g2_s2t",
            "sum",
        ),

        g2_t2s_count_raw=(
            "_g2_t2s",
            "sum",
        ),

        g2_bidirectional_support=(
            "_g2_bidirectional",
            "sum",
        ),
    )
    .reset_index()
)


# ============================================================
# 18. G2 RAW GEOMETRY
# ============================================================

g2_raw = accepted.loc[
    accepted[
        "_is_g2"
    ]
].copy()


g2_raw[
    "g2_pair_distance_um"
] = pd.to_numeric(
    g2_raw[
        "g2_pair_distance_um"
    ],
    errors="raise",
).astype(
    np.float64
)


assert np.isfinite(
    g2_raw[
        "g2_pair_distance_um"
    ]
).all()


g2_geometry = (
    g2_raw
    .groupby(
        RAW_KEY,
        sort=False,
        dropna=False,
    )[
        "g2_pair_distance_um"
    ]
    .agg(
        [
            "min",
            "median",
            "mean",
            "max",
        ]
    )
    .reset_index()
    .rename(
        columns={
            "min":
                "g2_pair_distance_um_min",

            "median":
                "g2_pair_distance_um_median",

            "mean":
                "g2_pair_distance_um_mean",

            "max":
                "g2_pair_distance_um_max",
        }
    )
)


# ============================================================
# 19. FROZEN UNIVERSE KEY
# ============================================================

universe_key = new_universe[
    METADATA_COLUMNS
].copy()


universe_key[
    "dataset"
] = universe_key[
    "dataset"
].astype(str)


universe_key[
    "candidate_t"
] = universe_key[
    "t"
].astype(
    np.int64
)


universe_key[
    "key_z"
] = np.round(
    universe_key[
        "z"
    ].to_numpy(
        dtype=np.float64
    ),
    12,
)


universe_key[
    "key_y"
] = np.round(
    universe_key[
        "y"
    ].to_numpy(
        dtype=np.float64
    ),
    12,
)


universe_key[
    "key_x"
] = np.round(
    universe_key[
        "x"
    ].to_numpy(
        dtype=np.float64
    ),
    12,
)


assert not universe_key[
    RAW_KEY
].duplicated().any()


proposal = universe_key.merge(
    raw_agg,
    on=RAW_KEY,
    how="left",
    validate="one_to_one",
    sort=False,
)


proposal = proposal.merge(
    g2_geometry,
    on=RAW_KEY,
    how="left",
    validate="one_to_one",
    sort=False,
)


proposal = (
    proposal
    .sort_values(
        "candidate_id",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    proposal
) == EXPECTED_NEW_ROWS


# ============================================================
# 20. REPRODUCE STORED COUNTS
# ============================================================

for frozen_column, raw_column in [
    (
        "proposal_count",
        "proposal_count_raw",
    ),

    (
        "g0_count",
        "g0_count_raw",
    ),

    (
        "g1_count",
        "g1_count_raw",
    ),

    (
        "g2_count",
        "g2_count_raw",
    ),

    (
        "g2_s2t_count",
        "g2_s2t_count_raw",
    ),

    (
        "g2_t2s_count",
        "g2_t2s_count_raw",
    ),
]:

    expected = new_universe[
        frozen_column
    ].to_numpy(
        dtype=np.int64
    )


    reconstructed = proposal[
        raw_column
    ].to_numpy(
        dtype=np.int64
    )


    assert np.array_equal(
        expected,
        reconstructed,
    )


proposal[
    "proposal_count"
] = proposal[
    "proposal_count_raw"
].astype(
    np.int64
)


proposal[
    "g0_count"
] = proposal[
    "g0_count_raw"
].astype(
    np.int64
)


proposal[
    "g1_count"
] = proposal[
    "g1_count_raw"
].astype(
    np.int64
)


proposal[
    "g2_count"
] = proposal[
    "g2_count_raw"
].astype(
    np.int64
)


proposal[
    "g2_s2t_count"
] = proposal[
    "g2_s2t_count_raw"
].astype(
    np.int64
)


proposal[
    "g2_t2s_count"
] = proposal[
    "g2_t2s_count_raw"
].astype(
    np.int64
)


proposal[
    "g2_bidirectional_support"
] = proposal[
    "g2_bidirectional_support"
].astype(
    np.int64
)


# ============================================================
# 21. DERIVED FAMILY FEATURES
# ============================================================

proposal[
    "has_g0"
] = (
    proposal[
        "g0_count"
    ]
    >
    0
).astype(
    np.int8
)


proposal[
    "has_g1"
] = (
    proposal[
        "g1_count"
    ]
    >
    0
).astype(
    np.int8
)


proposal[
    "has_g2"
] = (
    proposal[
        "g2_count"
    ]
    >
    0
).astype(
    np.int8
)


proposal[
    "family_count"
] = (
    proposal[
        "has_g0"
    ]
    +
    proposal[
        "has_g1"
    ]
    +
    proposal[
        "has_g2"
    ]
).astype(
    np.int8
)


proposal[
    "multi_family"
] = (
    proposal[
        "family_count"
    ]
    >
    1
).astype(
    np.int8
)


# ============================================================
# 22. C1 INVARIANTS
# ============================================================

assert int(
    proposal[
        "g2_bidirectional_support"
    ].sum()
) == 369_230


directional_min = np.minimum(
    proposal[
        "g2_s2t_count"
    ].to_numpy(
        dtype=np.int64
    ),
    proposal[
        "g2_t2s_count"
    ].to_numpy(
        dtype=np.int64
    ),
)


assert int(
    directional_min.sum()
) == 369_297


assert int(
    (
        proposal[
            "g2_bidirectional_support"
        ].to_numpy(
            dtype=np.int64
        )
        !=
        directional_min
    ).sum()
) == 67


has_g2 = proposal[
    "has_g2"
].eq(
    1
).to_numpy()


for column in G2_GEOMETRY_FEATURES:

    values = proposal[
        column
    ].to_numpy(
        dtype=np.float64
    )


    assert np.isfinite(
        values[
            has_g2
        ]
    ).all()


    assert np.isnan(
        values[
            ~has_g2
        ]
    ).all()


print(
    "A1_R3_COUNT_REPRODUCTION: PASS"
)

print(
    "C1_G2_BIDIRECTIONAL_SUPPORT: PASS"
)

print(
    "G2_RAW_GEOMETRY: PASS"
)


# ============================================================
# 23. JOIN TEMPORAL + PROPOSAL
# ============================================================

proposal_model = proposal[
    [
        "candidate_id",
    ]
    +
    PROPOSAL_FEATURES
    +
    G2_GEOMETRY_FEATURES
].copy()


temporal_model = new_temporal[
    [
        "candidate_id",
    ]
    +
    TEMPORAL_FEATURES
].copy()


matrix = new_universe[
    METADATA_COLUMNS
].copy()


matrix = matrix.merge(
    proposal_model,
    on="candidate_id",
    how="left",
    validate="one_to_one",
    sort=False,
)


matrix = matrix.merge(
    temporal_model,
    on="candidate_id",
    how="left",
    validate="one_to_one",
    sort=False,
)


matrix = (
    matrix
    .sort_values(
        "candidate_id",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 24. FINAL MATRIX SCHEMA
# ============================================================

EXPECTED_MATRIX_COLUMNS = (
    METADATA_COLUMNS
    +
    MODEL_FEATURES
)


assert list(
    matrix.columns
) == EXPECTED_MATRIX_COLUMNS


assert len(
    matrix
) == EXPECTED_NEW_ROWS


assert matrix[
    "candidate_id"
].is_unique


assert np.array_equal(
    matrix[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        EXPECTED_NEW_ROWS,
        dtype=np.int64,
    ),
)


assert set(
    matrix[
        "fold"
    ].astype(int).unique()
) == {
    1,
    2,
    3,
    4,
}


assert not (
    matrix[
        "fold"
    ].astype(int)
    ==
    0
).any()


# ============================================================
# 25. GT / ORACLE LEAKAGE GUARD
# ============================================================

FORBIDDEN_SUBSTRINGS = [
    "oracle",
    "missing_distance",
]

FORBIDDEN_PREFIXES = [
    "gt_",
]


for column in matrix.columns:

    lower = str(
        column
    ).lower()


    for token in FORBIDDEN_SUBSTRINGS:

        assert token not in lower, (
            column,
            token,
        )


    for prefix in FORBIDDEN_PREFIXES:

        assert not lower.startswith(
            prefix
        ), column


assert "oracle_hit_7um" not in matrix.columns


assert "detector_prob" not in matrix.columns
assert "detector_logit" not in matrix.columns


print(
    "\n========== FINAL MATRIX CONTRACT =========="
)

print(
    "rows:",
    len(matrix)
)

print(
    "metadata columns:",
    len(METADATA_COLUMNS)
)

print(
    "model features:",
    len(MODEL_FEATURES)
)

print(
    "total columns:",
    len(matrix.columns)
)

print(
    "GT/oracle:",
    "NO"
)

print(
    "detector evidence:",
    "NO"
)

print(
    "Fold0:",
    "NO"
)

print(
    "FINAL_39_FEATURE_GT_BLIND_MATRIX: PASS"
)


# ============================================================
# 26. DATASET SUMMARY
# ============================================================

proposal_summary = (
    matrix
    .groupby(
        [
            "dataset",
            "fold",
        ],
        sort=True,
    )
    .agg(
        candidate_rows=(
            "candidate_id",
            "size",
        ),

        has_g0=(
            "has_g0",
            "sum",
        ),

        has_g1=(
            "has_g1",
            "sum",
        ),

        has_g2=(
            "has_g2",
            "sum",
        ),

        multi_family=(
            "multi_family",
            "sum",
        ),

        g2_bidirectional_support=(
            "g2_bidirectional_support",
            "sum",
        ),

        bilateral_available=(
            "bilateral_available",
            "sum",
        ),
    )
    .reset_index()
)


dataset_summary = proposal_summary.merge(
    new_dataset_summary,
    on="dataset",
    how="left",
    validate="one_to_one",
    suffixes=(
        "",
        "_temporal",
    ),
)


# ============================================================
# 27. CONTENT HASHES
# ============================================================

print(
    "\n========== CONTENT HASHING =========="
)


MATRIX_CONTENT_SHA = dataframe_content_sha256(
    matrix,
    EXPECTED_MATRIX_COLUMNS,
)


DATASET_SUMMARY_COLUMNS = list(
    dataset_summary.columns
)


DATASET_SUMMARY_CONTENT_SHA = dataframe_content_sha256(
    dataset_summary,
    DATASET_SUMMARY_COLUMNS,
)


print(
    "matrix content SHA:",
    MATRIX_CONTENT_SHA
)

print(
    "dataset summary content SHA:",
    DATASET_SUMMARY_CONTENT_SHA
)


# ============================================================
# 28. SCHEMA / SUMMARY
# ============================================================

SOURCE_LOCK = {
    "producer_cell":
        EXPECTED_PRODUCER_CELL,

    "producer_cell_sha256":
        EXPECTED_PRODUCER_SHA,

    "full_computation_loop_sha256":
        EXPECTED_FULL_LOOP_SHA,

    "feature_value_dict_sha256":
        EXPECTED_FEATURE_VALUE_DICT_SHA,

    "feature_value_assignment_sha256":
        EXPECTED_FEATURE_ASSIGN_SHA,

    "build_dev_post_g3h3_frozen_sha256":
        EXPECTED_BUILD_POST_G3H3_SHA,

    "query_frame_sha256":
        EXPECTED_QUERY_FRAME_SHA,

    "frozen_v9_step_balance_sha256":
        EXPECTED_STEP_BALANCE_SHA,
}


SCHEMA = {
    "stage":
        "12.13A4",

    "status":
        "GT_BLIND_FEATURE_MATRIX_FROZEN",

    "branch_id":
        (
            "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_"
            "NODE_RECOVERY_V1"
        ),

    "parents": {
        "A3_content_sha256":
            EXPECTED_A3_CONTENT_SHA,

        "A3_file_sha256":
            EXPECTED_A3_FILE_SHA,

        "A3_C1_content_sha256":
            EXPECTED_C1_CONTENT_SHA,

        "A3_C1_file_sha256":
            EXPECTED_C1_FILE_SHA,

        "A1_R3_candidate_universe_file_sha256":
            EXPECTED_NEW_UNIVERSE_SHA,

        "A1_R3_raw_proposals_file_sha256":
            EXPECTED_RAW_SHA,

        "historical_F1_file_sha256":
            EXPECTED_OLD_F1_SHA,

        "G3H3_development_actions_file_sha256":
            EXPECTED_DEV_ACTIONS_SHA,
    },

    "source_lock":
        SOURCE_LOCK,

    "historical_replay": {
        "rows":
            EXPECTED_OLD_ROWS,

        "temporal_features":
            TEMPORAL_FEATURES,

        "feature_count":
            23,

        "exact_elementwise_reproduction":
            True,
    },

    "matrix": {
        "rows":
            EXPECTED_NEW_ROWS,

        "metadata_columns":
            METADATA_COLUMNS,

        "model_features":
            MODEL_FEATURES,

        "model_feature_count":
            39,

        "columns":
            EXPECTED_MATRIX_COLUMNS,

        "candidate_id":
            "0..1079884_UNIQUE",

        "development_folds":
            [
                1,
                2,
                3,
                4,
            ],
    },

    "proposal_feature_semantics": {
        "clarification_parent":
            "12.13A3-C1",

        "g2_bidirectional_support":
            (
                "SAME_RAW_G2_PROPOSAL_"
                "RECIPROCAL_INTERSECTION_COUNT"
            ),

        "directional_min":
            "FORBIDDEN",
    },

    "protocol": {
        "GT_read":
            False,

        "oracle_read":
            False,

        "oracle_label_in_feature_artifact":
            False,

        "detector_evidence_in_model_features":
            False,

        "model_trained":
            False,

        "model_selected":
            False,

        "threshold_selected":
            False,

        "graph_mutated":
            False,

        "Fold0_used":
            False,
    },

    "matrix_content_sha256":
        MATRIX_CONTENT_SHA,

    "dataset_summary_content_sha256":
        DATASET_SUMMARY_CONTENT_SHA,

    "next":
        "12.13A5_NESTED_LOFO_RANKING_AND_THRESHOLD_DEVELOPMENT",
}


SCHEMA_CONTENT_SHA = hashlib.sha256(
    json.dumps(
        SCHEMA,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


SCHEMA[
    "schema_sha256"
] = SCHEMA_CONTENT_SHA


SUMMARY = {
    "stage":
        "12.13A4",

    "status":
        "FORMAL_GT_BLIND_FEATURE_MATRIX_MATERIALIZATION_PASS",

    "rows":
        int(
            len(matrix)
        ),

    "metadata_columns":
        len(
            METADATA_COLUMNS
        ),

    "model_features":
        len(
            MODEL_FEATURES
        ),

    "temporal_features":
        len(
            TEMPORAL_FEATURES
        ),

    "proposal_support_features":
        len(
            PROPOSAL_FEATURES
        ),

    "g2_geometry_features":
        len(
            G2_GEOMETRY_FEATURES
        ),

    "historical_F1_reproduction":
        "EXACT_23_OF_23",

    "family_count_distribution":
        (
            matrix[
                "family_count"
            ]
            .value_counts()
            .sort_index()
            .to_dict()
        ),

    "multi_family_candidates":
        int(
            matrix[
                "multi_family"
            ].sum()
        ),

    "g2_bidirectional_support_sum":
        int(
            matrix[
                "g2_bidirectional_support"
            ].sum()
        ),

    "bilateral_available":
        int(
            matrix[
                "bilateral_available"
            ].sum()
        ),

    "matrix_content_sha256":
        MATRIX_CONTENT_SHA,

    "dataset_summary_content_sha256":
        DATASET_SUMMARY_CONTENT_SHA,

    "protocol": {
        "GT_read":
            False,

        "oracle_read":
            False,

        "model_trained":
            False,

        "threshold_selected":
            False,

        "graph_mutated":
            False,

        "Fold0_used":
            False,
    },

    "next":
        "12.13A5_NESTED_LOFO_RANKING_AND_THRESHOLD_DEVELOPMENT",
}


SUMMARY_CONTENT_SHA = hashlib.sha256(
    json.dumps(
        SUMMARY,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


SUMMARY[
    "summary_sha256"
] = SUMMARY_CONTENT_SHA


# ============================================================
# 29. TEMP OUTPUTS
# ============================================================

TMP_MATRIX = FEATURE_MATRIX_PATH.with_name(
    FEATURE_MATRIX_PATH.name
    +
    ".tmp_12_13a4"
)

TMP_DATASET = DATASET_SUMMARY_PATH.with_name(
    DATASET_SUMMARY_PATH.name
    +
    ".tmp_12_13a4"
)

TMP_SCHEMA = SCHEMA_PATH.with_name(
    SCHEMA_PATH.name
    +
    ".tmp_12_13a4"
)

TMP_SUMMARY = SUMMARY_PATH.with_name(
    SUMMARY_PATH.name
    +
    ".tmp_12_13a4"
)


for path in [
    TMP_MATRIX,
    TMP_DATASET,
    TMP_SCHEMA,
    TMP_SUMMARY,
]:

    if path.exists():

        raise RuntimeError(
            "\nStale A4 temporary artifact:\n"
            f"{path}"
        )


# ============================================================
# 30. TEMP WRITE
# ============================================================

matrix.to_pickle(
    TMP_MATRIX
)


dataset_summary.to_pickle(
    TMP_DATASET
)


TMP_SCHEMA.write_text(
    json.dumps(
        SCHEMA,
        indent=2,
    ),
    encoding="utf-8",
)


TMP_SUMMARY.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 31. RELOAD FIDELITY
# ============================================================

matrix_check = pd.read_pickle(
    TMP_MATRIX
)

dataset_check = pd.read_pickle(
    TMP_DATASET
)

schema_check = json.loads(
    TMP_SCHEMA.read_text(
        encoding="utf-8"
    )
)

summary_check = json.loads(
    TMP_SUMMARY.read_text(
        encoding="utf-8"
    )
)


pd.testing.assert_frame_equal(
    matrix_check,
    matrix,
    check_exact=True,
)


pd.testing.assert_frame_equal(
    dataset_check,
    dataset_summary,
    check_exact=True,
)


assert (
    schema_check[
        "schema_sha256"
    ]
    ==
    SCHEMA_CONTENT_SHA
)


assert (
    summary_check[
        "summary_sha256"
    ]
    ==
    SUMMARY_CONTENT_SHA
)


assert dataframe_content_sha256(
    matrix_check,
    EXPECTED_MATRIX_COLUMNS,
) == MATRIX_CONTENT_SHA


assert dataframe_content_sha256(
    dataset_check,
    DATASET_SUMMARY_COLUMNS,
) == DATASET_SUMMARY_CONTENT_SHA


# ============================================================
# 32. FILE HASHES
# ============================================================

MATRIX_FILE_SHA = sha256_file(
    TMP_MATRIX
)

DATASET_FILE_SHA = sha256_file(
    TMP_DATASET
)

SCHEMA_FILE_SHA = sha256_file(
    TMP_SCHEMA
)

SUMMARY_FILE_SHA = sha256_file(
    TMP_SUMMARY
)


# ============================================================
# 33. ATOMIC PROMOTION
# ============================================================

TMP_MATRIX.replace(
    FEATURE_MATRIX_PATH
)

TMP_DATASET.replace(
    DATASET_SUMMARY_PATH
)

TMP_SCHEMA.replace(
    SCHEMA_PATH
)

TMP_SUMMARY.replace(
    SUMMARY_PATH
)


# ============================================================
# 34. FINAL READBACK
# ============================================================

assert sha256_file(
    FEATURE_MATRIX_PATH
) == MATRIX_FILE_SHA

assert sha256_file(
    DATASET_SUMMARY_PATH
) == DATASET_FILE_SHA

assert sha256_file(
    SCHEMA_PATH
) == SCHEMA_FILE_SHA

assert sha256_file(
    SUMMARY_PATH
) == SUMMARY_FILE_SHA


final_matrix = pd.read_pickle(
    FEATURE_MATRIX_PATH
)


assert len(
    final_matrix
) == EXPECTED_NEW_ROWS


assert list(
    final_matrix.columns
) == EXPECTED_MATRIX_COLUMNS


assert "oracle_hit_7um" not in final_matrix.columns


# ============================================================
# 35. FINAL
# ============================================================

print(
    "\n========== 12.13A4 FINAL =========="
)


print(
    "FORMAL_GT_BLIND_FEATURE_MATRIX_MATERIALIZATION:",
    "PASS"
)


print(
    "\n--- HISTORICAL REPLAY ---"
)


print(
    "old rows:",
    EXPECTED_OLD_ROWS
)


print(
    "temporal features reproduced:",
    "23 / 23"
)


print(
    "elementwise exact:",
    "YES"
)


print(
    "\n--- NEW MATRIX ---"
)


print(
    "rows:",
    len(matrix)
)


print(
    "metadata columns:",
    len(METADATA_COLUMNS)
)


print(
    "model features:",
    len(MODEL_FEATURES)
)


print(
    "total columns:",
    len(matrix.columns)
)


print(
    "proposal-support features:",
    len(PROPOSAL_FEATURES)
)


print(
    "G2 raw geometry features:",
    len(G2_GEOMETRY_FEATURES)
)


print(
    "temporal geometry features:",
    len(TEMPORAL_FEATURES)
)


print(
    "\n--- C1 ---"
)


print(
    "g2_bidirectional_support sum:",
    int(
        matrix[
            "g2_bidirectional_support"
        ].sum()
    )
)


print(
    "expected:",
    369_230
)


print(
    "directional-min definition:",
    "FORBIDDEN"
)


print(
    "\n--- LEAKAGE ---"
)


print(
    "GT columns:",
    "NO"
)


print(
    "oracle columns:",
    "NO"
)


print(
    "detector probability/logit:",
    "NO"
)


print(
    "\n--- PROTOCOL ---"
)


print(
    "model trained:",
    "NO"
)


print(
    "model selected:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "graph mutation:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "\n--- CONTENT SHAS ---"
)


print(
    "feature matrix:",
    MATRIX_CONTENT_SHA
)


print(
    "dataset summary:",
    DATASET_SUMMARY_CONTENT_SHA
)


print(
    "schema:",
    SCHEMA_CONTENT_SHA
)


print(
    "summary:",
    SUMMARY_CONTENT_SHA
)


print(
    "\n--- FILE SHAS ---"
)


print(
    "feature matrix:",
    MATRIX_FILE_SHA
)


print(
    "dataset summary:",
    DATASET_FILE_SHA
)


print(
    "schema:",
    SCHEMA_FILE_SHA
)


print(
    "summary:",
    SUMMARY_FILE_SHA
)


print(
    "\nnext:"
)


print(
    "12.13A5_NESTED_LOFO_RANKING_AND_THRESHOLD_DEVELOPMENT"
)

In [ ]:
# ============================================================
# Stage 12.13A5
#
# FORMAL NESTED-LOFO
# RANKING + THRESHOLD DEVELOPMENT
#
# ============================================================
#
# PARENTS
#
#   A3
#     frozen ranking / threshold / validation contract
#
#   A3-C1
#     frozen proposal-derived feature semantics
#
#   A4
#     frozen 39-feature GT-blind matrix
#
#   A2
#     frozen oracle_hit_7um development label
#
# ------------------------------------------------------------
# OUTER LOFO
# ------------------------------------------------------------
#
# outer fold = held-out development evaluation fold
#
# For each outer fold:
#
#   TRAIN = other 3 folds
#   TEST  = outer fold
#
# ------------------------------------------------------------
# INNER LOFO
# ------------------------------------------------------------
#
# Within the 3 outer-training folds:
#
#   leave each one out once
#
# Evaluate BOTH frozen scorer families:
#
#   LOGREG_L2_BALANCED_V1
#   HGB_BALANCED_V1
#
# Select scorer family by:
#
#   1. higher mean inner AP
#   2. higher worst inner-fold AP
#   3. LOGREG before HGB on exact tie
#
# ------------------------------------------------------------
# THRESHOLD
# ------------------------------------------------------------
#
# For selected scorer only:
#
#   concatenate INNER OOF probabilities
#
#   threshold candidates =
#       exact precision_recall_curve thresholds
#
#   objective =
#       maximize F_beta, beta=0.5
#
# tie break:
#
#   1. higher F0.5
#   2. higher precision
#   3. higher threshold
#
# Outer labels are NOT used.
#
# ------------------------------------------------------------
# OUTER EVALUATION
# ------------------------------------------------------------
#
# Fit selected scorer on all 3 outer-training folds.
# Predict held-out outer fold.
#
# Evaluate:
#
#   AP
#   threshold precision / recall / F0.5
#
# Baselines:
#
#   NO_INFORMATION
#   PROPOSAL_COUNT
#   G2_RECIPROCAL_SUPPORT
#
# ------------------------------------------------------------
# A5 RANKING GATE
# ------------------------------------------------------------
#
# BOTH required:
#
#   nested policy macro mean outer AP
#       >
#   no-information macro mean outer AP
#
# AND
#
#   nested policy worst outer AP
#       >
#   no-information worst outer AP
#
# PASS:
#   -> A6 graph-level held-out evaluation
#
# FAIL:
#   -> CLOSE_BRANCH_NO_GRAPH_MUTATION
#
# ------------------------------------------------------------
# NO GRAPH MUTATION
# NO OFFICIAL GRAPH METRIC
# NO FOLD0
#
# KEEP PERMANENTLY.
# ============================================================


from pathlib import Path

import gc
import hashlib
import json
import math

import numpy as np
import pandas as pd


from sklearn.base import clone

from sklearn.ensemble import (
    HistGradientBoostingClassifier,
)

from sklearn.impute import (
    SimpleImputer,
)

from sklearn.linear_model import (
    LogisticRegression,
)

from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
)

from sklearn.pipeline import (
    Pipeline,
)

from sklearn.preprocessing import (
    StandardScaler,
)


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)


S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


A3_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_ranking_attachment_contract_frozen.json"
)


C1_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_ranking_attachment_contract_c1_g2_bidirectional_support_clarification_frozen.json"
)


A4_MATRIX_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_gt_blind_feature_matrix_v1.pkl"
)


A4_SCHEMA_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_gt_blind_feature_matrix_v1_schema_frozen.json"
)


A4_SUMMARY_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_gt_blind_feature_matrix_v1_summary.json"
)


A2_ORACLE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_oracle_candidates_v1.pkl"
)


A2_DECISION_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_oracle_ceiling_v1_decision_frozen.json"
)


# ============================================================
# FORMAL A5 OUTPUTS
# ============================================================

OOF_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_nested_lofo_ranking_v1_oof.pkl"
)


INNER_METRICS_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_nested_lofo_ranking_v1_inner_metrics.pkl"
)


MODEL_SELECTION_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_nested_lofo_ranking_v1_model_selection.pkl"
)


THRESHOLD_AUDIT_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_nested_lofo_ranking_v1_threshold_audit.pkl"
)


OUTER_METRICS_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_nested_lofo_ranking_v1_outer_metrics.pkl"
)


SUMMARY_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_nested_lofo_ranking_v1_summary.json"
)


DECISION_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_nested_lofo_ranking_v1_decision_frozen.json"
)


CLOSURE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_a5_ranking_closure.json"
)


for path in [
    A3_PATH,
    C1_PATH,
    A4_MATRIX_PATH,
    A4_SCHEMA_PATH,
    A4_SUMMARY_PATH,
    A2_ORACLE_PATH,
    A2_DECISION_PATH,
]:

    assert path.exists(), path


for path in [
    OOF_PATH,
    INNER_METRICS_PATH,
    MODEL_SELECTION_PATH,
    THRESHOLD_AUDIT_PATH,
    OUTER_METRICS_PATH,
    SUMMARY_PATH,
    DECISION_PATH,
    CLOSURE_PATH,
]:

    if path.exists():

        raise RuntimeError(
            "\nFormal 12.13A5 output already exists:\n"
            f"{path}\n\n"
            "Do NOT overwrite formal provenance."
        )


# ============================================================
# 1. EXACT PARENT HASHES
# ============================================================

EXPECTED_A3_CONTENT_SHA = (
    "c54eda46073582a7a915f1bfa7dcbff9"
    "eec8a96edcfad80a8e49e1097cf38b46"
)


EXPECTED_A3_FILE_SHA = (
    "37a5b2e557832774c0bcf65a33c81d8"
    "434bac2025c4eebbf2e3b0ea1bc955bd7"
)


EXPECTED_C1_CONTENT_SHA = (
    "94407e96316eddf11e7930b2807c8664"
    "4d9b49eb802d7757585ec18a5c241e97"
)


EXPECTED_C1_FILE_SHA = (
    "416414260749db71770bcd1fe96cc161d"
    "911448d27c023fde23a4e31ed445c3a"
)


EXPECTED_A4_MATRIX_FILE_SHA = (
    "e50e11bd3196f52a0ca4957f8036a061"
    "89a460350aae7d90542058deecf7d61b"
)


EXPECTED_A4_SCHEMA_FILE_SHA = (
    "93202bc091bb204b43ddc92066b33a46"
    "f3b4e228555a64e9e30f75918fc63b1c"
)


EXPECTED_A4_SUMMARY_FILE_SHA = (
    "85effd3615c15e657ae45f3af89541a4"
    "865a464d983485551c47e050efef6309"
)


EXPECTED_A4_MATRIX_CONTENT_SHA = (
    "5597dedef9d8525847419147bee1d6ce"
    "4a72d19d0f388ecc4690e7083dc895ff"
)


EXPECTED_A4_SCHEMA_CONTENT_SHA = (
    "b2b4d5efa43659d92ab47994ce1378d6"
    "e42e1918e6e9522e98689264fde27ab1"
)


EXPECTED_A4_SUMMARY_CONTENT_SHA = (
    "a6d68f01b4adb0d46db85de3c62758c4"
    "2010337d3da3872152a9db6b687c792c"
)


EXPECTED_A2_ORACLE_FILE_SHA = (
    "756758e0d9f6cf697a44548e7e7e5ad8"
    "74d01d7d8f33d9f129b3f82b9cc56c80"
)


EXPECTED_A2_DECISION_FILE_SHA = (
    "af2f95d05375259564bdf014f546b322"
    "466e823b2193a5eb7dd7623db4fc4c57"
)


EXPECTED_ROWS = 1_079_885

EXPECTED_POSITIVES = 3_306


SEED = 1640

OUTER_FOLDS = [
    1,
    2,
    3,
    4,
]


SCORER_NAMES = [
    "LOGREG_L2_BALANCED_V1",
    "HGB_BALANCED_V1",
]


MODEL_PRIORITY = {
    "LOGREG_L2_BALANCED_V1":
        0,

    "HGB_BALANCED_V1":
        1,
}


BETA = 0.5


# ============================================================
# 2. HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


class HashWriter:

    def __init__(self):

        self.h = hashlib.sha256()


    def write(
        self,
        text,
    ):

        if not isinstance(
            text,
            str,
        ):

            text = str(text)


        self.h.update(
            text.encode(
                "utf-8"
            )
        )


        return len(text)


    def flush(self):

        return None


def dataframe_content_sha256(
    df,
    columns,
):

    writer = HashWriter()


    df.loc[
        :,
        columns,
    ].to_csv(
        writer,
        index=False,
        na_rep="NaN",
        float_format="%.17g",
        lineterminator="\n",
    )


    return writer.h.hexdigest()


def array_triplet_sha256(
    candidate_id,
    y,
    score,
):

    frame = pd.DataFrame(
        {
            "candidate_id":
                np.asarray(
                    candidate_id,
                    dtype=np.int64,
                ),

            "y":
                np.asarray(
                    y,
                    dtype=np.int8,
                ),

            "score":
                np.asarray(
                    score,
                    dtype=np.float64,
                ),
        }
    )


    return dataframe_content_sha256(
        frame,
        [
            "candidate_id",
            "y",
            "score",
        ],
    )


# ============================================================
# 3. PARENT FILE LOCK
# ============================================================

assert sha256_file(
    A3_PATH
) == EXPECTED_A3_FILE_SHA


assert sha256_file(
    C1_PATH
) == EXPECTED_C1_FILE_SHA


assert sha256_file(
    A4_MATRIX_PATH
) == EXPECTED_A4_MATRIX_FILE_SHA


assert sha256_file(
    A4_SCHEMA_PATH
) == EXPECTED_A4_SCHEMA_FILE_SHA


assert sha256_file(
    A4_SUMMARY_PATH
) == EXPECTED_A4_SUMMARY_FILE_SHA


assert sha256_file(
    A2_ORACLE_PATH
) == EXPECTED_A2_ORACLE_FILE_SHA


assert sha256_file(
    A2_DECISION_PATH
) == EXPECTED_A2_DECISION_FILE_SHA


a3 = json.loads(
    A3_PATH.read_text(
        encoding="utf-8"
    )
)


c1 = json.loads(
    C1_PATH.read_text(
        encoding="utf-8"
    )
)


a4_schema = json.loads(
    A4_SCHEMA_PATH.read_text(
        encoding="utf-8"
    )
)


a4_summary = json.loads(
    A4_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


a2_decision = json.loads(
    A2_DECISION_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    a3[
        "contract_sha256"
    ]
    ==
    EXPECTED_A3_CONTENT_SHA
)


assert (
    c1[
        "clarification_sha256"
    ]
    ==
    EXPECTED_C1_CONTENT_SHA
)


assert (
    a4_schema[
        "schema_sha256"
    ]
    ==
    EXPECTED_A4_SCHEMA_CONTENT_SHA
)


assert (
    a4_summary[
        "summary_sha256"
    ]
    ==
    EXPECTED_A4_SUMMARY_CONTENT_SHA
)


assert (
    a4_schema[
        "matrix_content_sha256"
    ]
    ==
    EXPECTED_A4_MATRIX_CONTENT_SHA
)


assert a2_decision[
    "decision"
] == "PASS_CEILING_GATE"


assert a2_decision[
    "both_gate_pass"
] is True


assert a2_decision[
    "Fold0_used"
] is False


print(
    "========== 12.13A5 PARENT LOCK =========="
)


print(
    "A3:",
    "PASS"
)


print(
    "A3-C1:",
    "PASS"
)


print(
    "A4:",
    "PASS"
)


print(
    "A2 label source:",
    "PASS"
)


print(
    "Fold0:",
    "NO"
)


print(
    "A5_PARENT_LOCK: PASS"
)


# ============================================================
# 4. LOCK A3 VALIDATION CONTRACT
# ============================================================

MODEL_FEATURES = list(
    a3[
        "features"
    ][
        "allowed"
    ]
)


assert len(
    MODEL_FEATURES
) == 39


assert (
    a4_schema[
        "matrix"
    ][
        "model_features"
    ]
    ==
    MODEL_FEATURES
)


assert a3[
    "ranking_target"
][
    "column"
] == "oracle_hit_7um"


assert a3[
    "nested_LOFO"
][
    "outer_folds"
] == OUTER_FOLDS


assert a3[
    "nested_LOFO"
][
    "for_each_outer_fold"
][
    "inner_model_selection_primary"
] == "MEAN_AVERAGE_PRECISION"


assert a3[
    "nested_LOFO"
][
    "for_each_outer_fold"
][
    "inner_model_selection_secondary"
] == "WORST_INNER_FOLD_AVERAGE_PRECISION"


assert a3[
    "nested_LOFO"
][
    "for_each_outer_fold"
][
    "outer_fold_labels_used_for_model_selection"
] is False


assert a3[
    "nested_LOFO"
][
    "for_each_outer_fold"
][
    "outer_fold_labels_used_for_threshold_selection"
] is False


assert a3[
    "threshold_selection"
][
    "objective"
] == "MAXIMIZE_F_BETA"


assert float(
    a3[
        "threshold_selection"
    ][
        "beta"
    ]
) == BETA


assert a3[
    "threshold_selection"
][
    "manual_threshold_adjustment"
] is False


assert a3[
    "threshold_selection"
][
    "post_result_threshold_retuning"
] is False


assert set(
    a3[
        "scorers"
    ].keys()
) == set(
    SCORER_NAMES
)


print(
    "\n========== A3 NESTED-LOFO CONTRACT =========="
)


print(
    "outer folds:",
    OUTER_FOLDS
)


print(
    "scorers:",
    SCORER_NAMES
)


print(
    "threshold objective:",
    "F0.5"
)


print(
    "outer labels used for selection:",
    "NO"
)


print(
    "A3_NESTED_LOFO_CONTRACT_LOCK: PASS"
)


# ============================================================
# 5. LOAD A4 MATRIX + A2 LABELS
# ============================================================

matrix = pd.read_pickle(
    A4_MATRIX_PATH
)


oracle = pd.read_pickle(
    A2_ORACLE_PATH
)


assert len(
    matrix
) == EXPECTED_ROWS


assert len(
    oracle
) == EXPECTED_ROWS


assert matrix[
    "candidate_id"
].is_unique


assert oracle[
    "candidate_id"
].is_unique


assert np.array_equal(
    matrix[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    oracle[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
)


# ============================================================
# 6. EXACT METADATA IDENTITY
# ============================================================

IDENTITY_COLUMNS = [
    "candidate_id",
    "dataset",
    "fold",
    "t",
    "z",
    "y",
    "x",
]


for column in IDENTITY_COLUMNS:

    left = matrix[
        column
    ]

    right = oracle[
        column
    ]


    if pd.api.types.is_numeric_dtype(
        left
    ):

        assert np.array_equal(
            left.to_numpy(),
            right.to_numpy(),
            equal_nan=True,
        ), column

    else:

        assert left.astype(
            str
        ).equals(
            right.astype(
                str
            )
        ), column


y = oracle[
    "oracle_hit_7um"
].to_numpy(
    dtype=np.int8
)


assert set(
    np.unique(y)
) == {
    0,
    1,
}


assert int(
    y.sum()
) == EXPECTED_POSITIVES


fold = matrix[
    "fold"
].to_numpy(
    dtype=np.int8
)


candidate_id = matrix[
    "candidate_id"
].to_numpy(
    dtype=np.int64
)


assert set(
    np.unique(fold)
) == {
    1,
    2,
    3,
    4,
}


assert not (
    fold
    ==
    0
).any()


print(
    "\n========== LABEL IDENTITY LOCK =========="
)


print(
    "rows:",
    len(matrix)
)


print(
    "positives:",
    int(
        y.sum()
    )
)


print(
    "positive rate:",
    float(
        y.mean()
    )
)


print(
    "candidate identity:",
    "EXACT"
)


print(
    "Fold0:",
    "NO"
)


print(
    "A2_A4_LABEL_IDENTITY_LOCK: PASS"
)


# ============================================================
# 7. FEATURE MATRIX
# ============================================================

X = matrix[
    MODEL_FEATURES
].copy()


# No infinity is permitted.
numeric_values = X.to_numpy(
    dtype=np.float64
)


assert not np.isinf(
    numeric_values
).any()


del numeric_values

gc.collect()


# ============================================================
# 8. MODEL BUILDERS
# ============================================================

def build_scorer(
    scorer_name,
):

    if scorer_name == (
        "LOGREG_L2_BALANCED_V1"
    ):

        return Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median",
                        add_indicator=True,
                    ),
                ),

                (
                    "scaler",
                    StandardScaler(),
                ),

                (
                    "classifier",
                    LogisticRegression(
                        penalty="l2",
                        C=1.0,
                        class_weight="balanced",
                        solver="lbfgs",
                        max_iter=2000,
                        random_state=SEED,
                    ),
                ),
            ]
        )


    if scorer_name == (
        "HGB_BALANCED_V1"
    ):

        return HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=200,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=1.0,
            early_stopping=False,
            random_state=SEED,
        )


    raise KeyError(
        scorer_name
    )


def balanced_sample_weight(
    labels,
):

    labels = np.asarray(
        labels,
        dtype=np.int8,
    )


    n = len(labels)

    positive = int(
        labels.sum()
    )

    negative = int(
        n
        -
        positive
    )


    assert positive > 0

    assert negative > 0


    positive_weight = (
        n
        /
        (
            2.0
            *
            positive
        )
    )


    negative_weight = (
        n
        /
        (
            2.0
            *
            negative
        )
    )


    return np.where(
        labels
        ==
        1,
        positive_weight,
        negative_weight,
    ).astype(
        np.float64
    )


def fit_scorer(
    scorer_name,
    X_train,
    y_train,
):

    model = build_scorer(
        scorer_name
    )


    if scorer_name == (
        "HGB_BALANCED_V1"
    ):

        weights = balanced_sample_weight(
            y_train
        )


        model.fit(
            X_train,
            y_train,
            sample_weight=weights,
        )


    else:

        model.fit(
            X_train,
            y_train,
        )


    return model


def positive_probability(
    model,
    X_eval,
):

    probability = model.predict_proba(
        X_eval
    )


    assert probability.ndim == 2

    assert probability.shape[
        1
    ] == 2


    result = probability[
        :,
        1
    ].astype(
        np.float64
    )


    assert np.isfinite(
        result
    ).all()


    assert (
        result
        >=
        0
    ).all()


    assert (
        result
        <=
        1
    ).all()


    return result


# ============================================================
# 9. F0.5 HELPERS
# ============================================================

def f_beta_value(
    precision,
    recall,
    beta=BETA,
):

    precision = float(
        precision
    )

    recall = float(
        recall
    )


    beta2 = (
        float(beta)
        **
        2
    )


    denominator = (
        beta2
        *
        precision
        +
        recall
    )


    if denominator <= 0:

        return 0.0


    return (
        (
            1.0
            +
            beta2
        )
        *
        precision
        *
        recall
        /
        denominator
    )


def select_threshold_fbeta(
    y_true,
    score,
):

    y_true = np.asarray(
        y_true,
        dtype=np.int8,
    )


    score = np.asarray(
        score,
        dtype=np.float64,
    )


    assert len(
        y_true
    ) == len(
        score
    )


    assert int(
        y_true.sum()
    ) > 0


    precision, recall, thresholds = (
        precision_recall_curve(
            y_true,
            score,
        )
    )


    assert len(
        precision
    ) == (
        len(
            thresholds
        )
        +
        1
    )


    assert len(
        recall
    ) == (
        len(
            thresholds
        )
        +
        1
    )


    assert len(
        thresholds
    ) > 0


    # For threshold[i], sklearn precision/recall pair is
    # precision[i], recall[i].
    precision_t = precision[
        :-1
    ].astype(
        np.float64
    )


    recall_t = recall[
        :-1
    ].astype(
        np.float64
    )


    threshold_t = thresholds.astype(
        np.float64
    )


    beta2 = BETA ** 2


    denominator = (
        beta2
        *
        precision_t
        +
        recall_t
    )


    fbeta = np.zeros(
        len(
            thresholds
        ),
        dtype=np.float64,
    )


    valid = (
        denominator
        >
        0
    )


    fbeta[
        valid
    ] = (
        (
            1.0
            +
            beta2
        )
        *
        precision_t[
            valid
        ]
        *
        recall_t[
            valid
        ]
        /
        denominator[
            valid
        ]
    )


    candidates = pd.DataFrame(
        {
            "threshold":
                threshold_t,

            "precision":
                precision_t,

            "recall":
                recall_t,

            "f0p5":
                fbeta,
        }
    )


    # Frozen tie-break:
    #
    # 1. higher F0.5
    # 2. higher precision
    # 3. higher threshold
    selected = (
        candidates
        .sort_values(
            [
                "f0p5",
                "precision",
                "threshold",
            ],
            ascending=[
                False,
                False,
                False,
            ],
            kind="stable",
        )
        .iloc[
            0
        ]
    )


    threshold = float(
        selected[
            "threshold"
        ]
    )


    predicted = (
        score
        >=
        threshold
    )


    return {
        "threshold":
            threshold,

        "precision":
            float(
                selected[
                    "precision"
                ]
            ),

        "recall":
            float(
                selected[
                    "recall"
                ]
            ),

        "f0p5":
            float(
                selected[
                    "f0p5"
                ]
            ),

        "candidate_threshold_count":
            int(
                len(
                    candidates
                )
            ),

        "selected_count":
            int(
                predicted.sum()
            ),
    }


def threshold_metrics(
    y_true,
    score,
    threshold,
):

    y_true = np.asarray(
        y_true,
        dtype=np.int8,
    )


    score = np.asarray(
        score,
        dtype=np.float64,
    )


    predicted = (
        score
        >=
        float(
            threshold
        )
    )


    positive = (
        y_true
        ==
        1
    )


    negative = ~positive


    tp = int(
        (
            predicted
            &
            positive
        ).sum()
    )


    fp = int(
        (
            predicted
            &
            negative
        ).sum()
    )


    fn = int(
        (
            (~predicted)
            &
            positive
        ).sum()
    )


    tn = int(
        (
            (~predicted)
            &
            negative
        ).sum()
    )


    precision = (
        tp
        /
        (
            tp
            +
            fp
        )
        if (
            tp
            +
            fp
        )
        >
        0
        else
        0.0
    )


    recall = (
        tp
        /
        (
            tp
            +
            fn
        )
        if (
            tp
            +
            fn
        )
        >
        0
        else
        0.0
    )


    return {
        "selected":
            int(
                predicted.sum()
            ),

        "tp":
            tp,

        "fp":
            fp,

        "fn":
            fn,

        "tn":
            tn,

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f0p5":
            float(
                f_beta_value(
                    precision,
                    recall,
                )
            ),
    }


# ============================================================
# 10. RESULT CONTAINERS
# ============================================================

inner_metric_rows = []

selection_rows = []

threshold_rows = []

outer_metric_rows = []

oof_parts = []


# ============================================================
# 11. NESTED OUTER LOOP
# ============================================================

for outer_index, outer_fold in enumerate(
    OUTER_FOLDS,
    start=1,
):

    print(
        "\n"
        +
        "=" * 76
    )


    print(
        "OUTER FOLD:",
        outer_fold
    )


    print(
        "=" * 76
    )


    train_folds = [
        f

        for f in OUTER_FOLDS

        if f
        !=
        outer_fold
    ]


    outer_train_mask = np.isin(
        fold,
        train_folds,
    )


    outer_test_mask = (
        fold
        ==
        outer_fold
    )


    outer_train_indices = np.flatnonzero(
        outer_train_mask
    )


    outer_test_indices = np.flatnonzero(
        outer_test_mask
    )


    assert len(
        outer_train_indices
    ) > 0


    assert len(
        outer_test_indices
    ) > 0


    y_outer_train = y[
        outer_train_indices
    ]


    # IMPORTANT:
    #
    # No y_outer_test is read/used for selection below.


    assert int(
        y_outer_train.sum()
    ) > 0


    print(
        "outer train folds:",
        train_folds
    )


    print(
        "outer train rows:",
        len(
            outer_train_indices
        )
    )


    print(
        "outer train positives:",
        int(
            y_outer_train.sum()
        )
    )


    print(
        "held-out rows:",
        len(
            outer_test_indices
        )
    )


    # ========================================================
    # 11A. INNER LOFO FOR EACH SCORER
    # ========================================================

    inner_oof_by_model = {}

    model_inner_summary = []


    for scorer_name in SCORER_NAMES:

        print(
            "\nINNER SCORER:",
            scorer_name
        )


        scorer_inner_scores = np.full(
            len(
                outer_train_indices
            ),
            np.nan,
            dtype=np.float64,
        )


        # Map global candidate row index -> local position
        # inside outer training population.
        local_position = {
            int(
                global_index
            ):
                local_index

            for local_index, global_index
            in enumerate(
                outer_train_indices
            )
        }


        scorer_fold_aps = []


        for inner_val_fold in train_folds:

            inner_fit_folds = [
                f

                for f in train_folds

                if f
                !=
                inner_val_fold
            ]


            inner_fit_mask = np.isin(
                fold,
                inner_fit_folds,
            )


            inner_val_mask = (
                fold
                ==
                inner_val_fold
            )


            inner_fit_indices = np.flatnonzero(
                inner_fit_mask
            )


            inner_val_indices = np.flatnonzero(
                inner_val_mask
            )


            y_inner_fit = y[
                inner_fit_indices
            ]


            y_inner_val = y[
                inner_val_indices
            ]


            assert int(
                y_inner_fit.sum()
            ) > 0


            assert int(
                y_inner_val.sum()
            ) > 0


            model = fit_scorer(
                scorer_name,
                X.iloc[
                    inner_fit_indices
                ],
                y_inner_fit,
            )


            score_inner_val = positive_probability(
                model,
                X.iloc[
                    inner_val_indices
                ],
            )


            inner_ap = float(
                average_precision_score(
                    y_inner_val,
                    score_inner_val,
                )
            )


            scorer_fold_aps.append(
                inner_ap
            )


            local_indices = np.asarray(
                [
                    local_position[
                        int(
                            global_index
                        )
                    ]

                    for global_index
                    in inner_val_indices
                ],
                dtype=np.int64,
            )


            scorer_inner_scores[
                local_indices
            ] = score_inner_val


            inner_metric_rows.append(
                {
                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "scorer":
                        scorer_name,

                    "inner_validation_fold":
                        int(
                            inner_val_fold
                        ),

                    "inner_training_folds":
                        ",".join(
                            str(f)
                            for f
                            in inner_fit_folds
                        ),

                    "train_rows":
                        int(
                            len(
                                inner_fit_indices
                            )
                        ),

                    "train_positives":
                        int(
                            y_inner_fit.sum()
                        ),

                    "validation_rows":
                        int(
                            len(
                                inner_val_indices
                            )
                        ),

                    "validation_positives":
                        int(
                            y_inner_val.sum()
                        ),

                    "average_precision":
                        inner_ap,
                }
            )


            print(
                "  inner val fold:",
                inner_val_fold,
                "| AP:",
                inner_ap,
            )


            del model
            del score_inner_val

            gc.collect()


        assert np.isfinite(
            scorer_inner_scores
        ).all()


        mean_ap = float(
            np.mean(
                scorer_fold_aps
            )
        )


        worst_ap = float(
            np.min(
                scorer_fold_aps
            )
        )


        inner_oof_by_model[
            scorer_name
        ] = scorer_inner_scores


        model_inner_summary.append(
            {
                "scorer":
                    scorer_name,

                "mean_inner_ap":
                    mean_ap,

                "worst_inner_ap":
                    worst_ap,

                "priority":
                    MODEL_PRIORITY[
                        scorer_name
                    ],
            }
        )


        print(
            "  mean AP:",
            mean_ap
        )


        print(
            "  worst AP:",
            worst_ap
        )


    # ========================================================
    # 11B. INNER MODEL FAMILY SELECTION
    # ========================================================

    model_summary_df = pd.DataFrame(
        model_inner_summary
    )


    model_summary_df = (
        model_summary_df
        .sort_values(
            [
                "mean_inner_ap",
                "worst_inner_ap",
                "priority",
            ],
            ascending=[
                False,
                False,
                True,
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


    selected_scorer = str(
        model_summary_df.iloc[
            0
        ][
            "scorer"
        ]
    )


    selected_mean_inner_ap = float(
        model_summary_df.iloc[
            0
        ][
            "mean_inner_ap"
        ]
    )


    selected_worst_inner_ap = float(
        model_summary_df.iloc[
            0
        ][
            "worst_inner_ap"
        ]
    )


    selection_rows.append(
        {
            "outer_fold":
                int(
                    outer_fold
                ),

            "selected_scorer":
                selected_scorer,

            "mean_inner_ap":
                selected_mean_inner_ap,

            "worst_inner_ap":
                selected_worst_inner_ap,

            "selection_primary":
                "MEAN_AVERAGE_PRECISION",

            "selection_secondary":
                "WORST_INNER_FOLD_AVERAGE_PRECISION",

            "tie_break":
                (
                    "LOGREG_L2_BALANCED_V1_"
                    "BEFORE_HGB_BALANCED_V1"
                ),

            "outer_labels_used":
                False,
        }
    )


    print(
        "\nSELECTED SCORER:",
        selected_scorer
    )


    # ========================================================
    # 11C. INNER OOF THRESHOLD SELECTION
    # ========================================================

    selected_inner_oof = inner_oof_by_model[
        selected_scorer
    ]


    assert len(
        selected_inner_oof
    ) == len(
        outer_train_indices
    )


    selected_inner_oof_sha = array_triplet_sha256(
        candidate_id[
            outer_train_indices
        ],
        y_outer_train,
        selected_inner_oof,
    )


    threshold_result = select_threshold_fbeta(
        y_outer_train,
        selected_inner_oof,
    )


    selected_threshold = float(
        threshold_result[
            "threshold"
        ]
    )


    threshold_rows.append(
        {
            "outer_fold":
                int(
                    outer_fold
                ),

            "selected_scorer":
                selected_scorer,

            "inner_oof_rows":
                int(
                    len(
                        outer_train_indices
                    )
                ),

            "inner_oof_positives":
                int(
                    y_outer_train.sum()
                ),

            "inner_oof_sha256":
                selected_inner_oof_sha,

            "candidate_threshold_count":
                int(
                    threshold_result[
                        "candidate_threshold_count"
                    ]
                ),

            "threshold":
                selected_threshold,

            "inner_precision":
                float(
                    threshold_result[
                        "precision"
                    ]
                ),

            "inner_recall":
                float(
                    threshold_result[
                        "recall"
                    ]
                ),

            "inner_f0p5":
                float(
                    threshold_result[
                        "f0p5"
                    ]
                ),

            "inner_selected_count":
                int(
                    threshold_result[
                        "selected_count"
                    ]
                ),

            "beta":
                BETA,

            "outer_labels_used":
                False,
        }
    )


    print(
        "SELECTED THRESHOLD:",
        selected_threshold
    )


    print(
        "inner OOF F0.5:",
        threshold_result[
            "f0p5"
        ]
    )


    print(
        "inner OOF precision:",
        threshold_result[
            "precision"
        ]
    )


    print(
        "inner OOF recall:",
        threshold_result[
            "recall"
        ]
    )


    # ========================================================
    # 11D. FIT SELECTED MODEL ON ALL 3 TRAIN FOLDS
    #
    # Still no outer y used for selection.
    # ========================================================

    final_outer_model = fit_scorer(
        selected_scorer,
        X.iloc[
            outer_train_indices
        ],
        y_outer_train,
    )


    outer_score = positive_probability(
        final_outer_model,
        X.iloc[
            outer_test_indices
        ],
    )


    # ========================================================
    # 11E. ONLY NOW ACCESS HELD-OUT OUTER LABELS FOR EVAL
    # ========================================================

    y_outer_test = y[
        outer_test_indices
    ]


    assert int(
        y_outer_test.sum()
    ) > 0


    outer_ap = float(
        average_precision_score(
            y_outer_test,
            outer_score,
        )
    )


    threshold_eval = threshold_metrics(
        y_outer_test,
        outer_score,
        selected_threshold,
    )


    # ========================================================
    # BASELINES
    # ========================================================

    training_prevalence = float(
        y_outer_train.mean()
    )


    no_information_score = np.full(
        len(
            outer_test_indices
        ),
        training_prevalence,
        dtype=np.float64,
    )


    no_information_ap = float(
        average_precision_score(
            y_outer_test,
            no_information_score,
        )
    )


    proposal_count_score = matrix.iloc[
        outer_test_indices
    ][
        "proposal_count"
    ].to_numpy(
        dtype=np.float64
    )


    proposal_count_ap = float(
        average_precision_score(
            y_outer_test,
            proposal_count_score,
        )
    )


    reciprocal_score = matrix.iloc[
        outer_test_indices
    ][
        "g2_bidirectional_support"
    ].to_numpy(
        dtype=np.float64
    )


    reciprocal_ap = float(
        average_precision_score(
            y_outer_test,
            reciprocal_score,
        )
    )


    outer_metric_rows.append(
        {
            "outer_fold":
                int(
                    outer_fold
                ),

            "training_folds":
                ",".join(
                    str(f)
                    for f
                    in train_folds
                ),

            "selected_scorer":
                selected_scorer,

            "threshold":
                selected_threshold,

            "test_rows":
                int(
                    len(
                        outer_test_indices
                    )
                ),

            "test_positives":
                int(
                    y_outer_test.sum()
                ),

            "test_prevalence":
                float(
                    y_outer_test.mean()
                ),

            "training_prevalence":
                training_prevalence,

            "policy_average_precision":
                outer_ap,

            "no_information_average_precision":
                no_information_ap,

            "proposal_count_average_precision":
                proposal_count_ap,

            "g2_reciprocal_average_precision":
                reciprocal_ap,

            "selected_count":
                int(
                    threshold_eval[
                        "selected"
                    ]
                ),

            "tp":
                int(
                    threshold_eval[
                        "tp"
                    ]
                ),

            "fp":
                int(
                    threshold_eval[
                        "fp"
                    ]
                ),

            "fn":
                int(
                    threshold_eval[
                        "fn"
                    ]
                ),

            "tn":
                int(
                    threshold_eval[
                        "tn"
                    ]
                ),

            "precision":
                float(
                    threshold_eval[
                        "precision"
                    ]
                ),

            "recall":
                float(
                    threshold_eval[
                        "recall"
                    ]
                ),

            "f0p5":
                float(
                    threshold_eval[
                        "f0p5"
                    ]
                ),
        }
    )


    # ========================================================
    # OOF ARTIFACT
    # ========================================================

    outer_oof = matrix.iloc[
        outer_test_indices
    ][
        [
            "candidate_id",
            "dataset",
            "fold",
            "t",
            "z",
            "y",
            "x",
        ]
    ].copy()


    outer_oof[
        "oracle_hit_7um"
    ] = y_outer_test.astype(
        np.int8
    )


    outer_oof[
        "selected_scorer"
    ] = selected_scorer


    outer_oof[
        "threshold"
    ] = selected_threshold


    outer_oof[
        "model_score"
    ] = outer_score.astype(
        np.float64
    )


    outer_oof[
        "selected"
    ] = (
        outer_score
        >=
        selected_threshold
    )


    outer_oof[
        "no_information_score"
    ] = no_information_score


    outer_oof[
        "proposal_count_score"
    ] = proposal_count_score


    outer_oof[
        "g2_reciprocal_score"
    ] = reciprocal_score


    oof_parts.append(
        outer_oof
    )


    print(
        "\nOUTER EVALUATION"
    )


    print(
        "policy AP:",
        outer_ap
    )


    print(
        "no-info AP:",
        no_information_ap
    )


    print(
        "proposal_count AP:",
        proposal_count_ap
    )


    print(
        "G2 reciprocal AP:",
        reciprocal_ap
    )


    print(
        "selected:",
        threshold_eval[
            "selected"
        ]
    )


    print(
        "precision:",
        threshold_eval[
            "precision"
        ]
    )


    print(
        "recall:",
        threshold_eval[
            "recall"
        ]
    )


    print(
        "F0.5:",
        threshold_eval[
            "f0p5"
        ]
    )


    del final_outer_model

    del outer_score

    del no_information_score

    del proposal_count_score

    del reciprocal_score

    del inner_oof_by_model

    gc.collect()


# ============================================================
# 12. MATERIALIZE RESULT TABLES
# ============================================================

inner_metrics = pd.DataFrame(
    inner_metric_rows
)


model_selection = pd.DataFrame(
    selection_rows
)


threshold_audit = pd.DataFrame(
    threshold_rows
)


outer_metrics = pd.DataFrame(
    outer_metric_rows
)


oof = (
    pd.concat(
        oof_parts,
        ignore_index=True,
    )
    .sort_values(
        "candidate_id",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 13. OOF INVARIANTS
# ============================================================

assert len(
    oof
) == EXPECTED_ROWS


assert oof[
    "candidate_id"
].is_unique


assert np.array_equal(
    oof[
        "candidate_id"
    ].to_numpy(
        dtype=np.int64
    ),
    candidate_id,
)


assert np.array_equal(
    oof[
        "oracle_hit_7um"
    ].to_numpy(
        dtype=np.int8
    ),
    y,
)


assert int(
    oof[
        "oracle_hit_7um"
    ].sum()
) == EXPECTED_POSITIVES


assert set(
    oof[
        "fold"
    ].astype(int).unique()
) == {
    1,
    2,
    3,
    4,
}


assert not (
    oof[
        "fold"
    ].astype(int)
    ==
    0
).any()


assert np.isfinite(
    oof[
        "model_score"
    ].to_numpy(
        dtype=np.float64
    )
).all()


assert (
    oof[
        "model_score"
    ].to_numpy(
        dtype=np.float64
    )
    >=
    0
).all()


assert (
    oof[
        "model_score"
    ].to_numpy(
        dtype=np.float64
    )
    <=
    1
).all()


print(
    "\n========== OUTER OOF LOCK =========="
)


print(
    "rows:",
    len(oof)
)


print(
    "positives:",
    int(
        oof[
            "oracle_hit_7um"
        ].sum()
    )
)


print(
    "candidate identity:",
    "EXACT"
)


print(
    "Fold0:",
    "NO"
)


print(
    "OUTER_OOF_LOCK: PASS"
)


# ============================================================
# 14. RANKING GATE
# ============================================================

policy_macro_ap = float(
    outer_metrics[
        "policy_average_precision"
    ].mean()
)


policy_worst_ap = float(
    outer_metrics[
        "policy_average_precision"
    ].min()
)


no_info_macro_ap = float(
    outer_metrics[
        "no_information_average_precision"
    ].mean()
)


no_info_worst_ap = float(
    outer_metrics[
        "no_information_average_precision"
    ].min()
)


proposal_macro_ap = float(
    outer_metrics[
        "proposal_count_average_precision"
    ].mean()
)


reciprocal_macro_ap = float(
    outer_metrics[
        "g2_reciprocal_average_precision"
    ].mean()
)


MACRO_GATE = bool(
    policy_macro_ap
    >
    no_info_macro_ap
)


WORST_GATE = bool(
    policy_worst_ap
    >
    no_info_worst_ap
)


RANKING_GATE_PASS = bool(
    MACRO_GATE
    and
    WORST_GATE
)


DECISION = (
    "PASS_RANKING_GATE"
    if RANKING_GATE_PASS
    else
    "CLOSE_BRANCH_NO_GRAPH_MUTATION"
)


print(
    "\n========== A5 RANKING GATE =========="
)


print(
    "policy macro mean AP:",
    policy_macro_ap
)


print(
    "no-information macro mean AP:",
    no_info_macro_ap
)


print(
    "MACRO GATE:",
    (
        "PASS"
        if MACRO_GATE
        else
        "FAIL"
    )
)


print()


print(
    "policy worst-fold AP:",
    policy_worst_ap
)


print(
    "no-information worst-fold AP:",
    no_info_worst_ap
)


print(
    "WORST-FOLD GATE:",
    (
        "PASS"
        if WORST_GATE
        else
        "FAIL"
    )
)


print()


print(
    "proposal_count macro AP:",
    proposal_macro_ap
)


print(
    "G2 reciprocal macro AP:",
    reciprocal_macro_ap
)


print()


print(
    "A5 DECISION:",
    DECISION
)


# ============================================================
# 15. CONTENT HASHES
# ============================================================

OOF_COLUMNS = list(
    oof.columns
)


INNER_COLUMNS = list(
    inner_metrics.columns
)


SELECTION_COLUMNS = list(
    model_selection.columns
)


THRESHOLD_COLUMNS = list(
    threshold_audit.columns
)


OUTER_COLUMNS = list(
    outer_metrics.columns
)


OOF_CONTENT_SHA = dataframe_content_sha256(
    oof,
    OOF_COLUMNS,
)


INNER_CONTENT_SHA = dataframe_content_sha256(
    inner_metrics,
    INNER_COLUMNS,
)


SELECTION_CONTENT_SHA = dataframe_content_sha256(
    model_selection,
    SELECTION_COLUMNS,
)


THRESHOLD_CONTENT_SHA = dataframe_content_sha256(
    threshold_audit,
    THRESHOLD_COLUMNS,
)


OUTER_CONTENT_SHA = dataframe_content_sha256(
    outer_metrics,
    OUTER_COLUMNS,
)


# ============================================================
# 16. SUMMARY
# ============================================================

selected_scorer_counts = (
    model_selection[
        "selected_scorer"
    ]
    .value_counts()
    .to_dict()
)


SUMMARY = {
    "stage":
        "12.13A5",

    "status":
        "FORMAL_NESTED_LOFO_RANKING_AND_THRESHOLD_DEVELOPMENT_COMPLETE",

    "branch_id":
        (
            "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_"
            "NODE_RECOVERY_V1"
        ),

    "parents": {
        "A3_content_sha256":
            EXPECTED_A3_CONTENT_SHA,

        "A3_file_sha256":
            EXPECTED_A3_FILE_SHA,

        "A3_C1_content_sha256":
            EXPECTED_C1_CONTENT_SHA,

        "A3_C1_file_sha256":
            EXPECTED_C1_FILE_SHA,

        "A4_feature_matrix_file_sha256":
            EXPECTED_A4_MATRIX_FILE_SHA,

        "A4_feature_matrix_content_sha256":
            EXPECTED_A4_MATRIX_CONTENT_SHA,

        "A4_schema_file_sha256":
            EXPECTED_A4_SCHEMA_FILE_SHA,

        "A2_oracle_file_sha256":
            EXPECTED_A2_ORACLE_FILE_SHA,

        "A2_decision_file_sha256":
            EXPECTED_A2_DECISION_FILE_SHA,
    },

    "data": {
        "rows":
            EXPECTED_ROWS,

        "positives":
            EXPECTED_POSITIVES,

        "positive_rate":
            float(
                EXPECTED_POSITIVES
                /
                EXPECTED_ROWS
            ),

        "model_features":
            39,

        "folds":
            OUTER_FOLDS,
    },

    "protocol": {
        "validation":
            "NESTED_LOFO",

        "outer_folds":
            OUTER_FOLDS,

        "inner_model_selection":
            (
                "MEAN_AP_THEN_WORST_AP_"
                "THEN_FIXED_MODEL_TIE_BREAK"
            ),

        "threshold_selection":
            (
                "INNER_OOF_MAX_F0P5_"
                "THEN_PRECISION_THEN_THRESHOLD"
            ),

        "outer_labels_used_for_model_selection":
            False,

        "outer_labels_used_for_threshold_selection":
            False,

        "hyperparameter_search":
            False,

        "model_families":
            SCORER_NAMES,

        "GT_oracle_used_as_training_label":
            True,

        "graph_mutated":
            False,

        "official_graph_metric_evaluated":
            False,

        "Fold0_used":
            False,
    },

    "selected_scorer_counts":
        {
            str(k):
                int(v)

            for k, v
            in selected_scorer_counts.items()
        },

    "ranking": {
        "policy_macro_mean_AP":
            policy_macro_ap,

        "no_information_macro_mean_AP":
            no_info_macro_ap,

        "macro_gate_pass":
            MACRO_GATE,

        "policy_worst_fold_AP":
            policy_worst_ap,

        "no_information_worst_fold_AP":
            no_info_worst_ap,

        "worst_fold_gate_pass":
            WORST_GATE,

        "proposal_count_macro_mean_AP":
            proposal_macro_ap,

        "g2_reciprocal_macro_mean_AP":
            reciprocal_macro_ap,

        "both_ranking_conditions_pass":
            RANKING_GATE_PASS,
    },

    "outer_metrics":
        outer_metrics.to_dict(
            orient="records"
        ),

    "decision":
        DECISION,

    "content_sha256": {
        "oof":
            OOF_CONTENT_SHA,

        "inner_metrics":
            INNER_CONTENT_SHA,

        "model_selection":
            SELECTION_CONTENT_SHA,

        "threshold_audit":
            THRESHOLD_CONTENT_SHA,

        "outer_metrics":
            OUTER_CONTENT_SHA,
    },

    "next":
        (
            "12.13A6_HELD_OUT_GRAPH_ATTACHMENT_AND_OFFICIAL_METRIC_EVALUATION"
            if RANKING_GATE_PASS
            else
            "BRANCH_CLOSED"
        ),
}


SUMMARY_CONTENT_SHA = hashlib.sha256(
    json.dumps(
        SUMMARY,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


SUMMARY[
    "summary_sha256"
] = SUMMARY_CONTENT_SHA


# ============================================================
# 17. DECISION RECORD
# ============================================================

DECISION_RECORD = {
    "stage":
        "12.13A5",

    "branch_id":
        (
            "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_"
            "NODE_RECOVERY_V1"
        ),

    "ranking_gate": {
        "macro_policy_AP":
            policy_macro_ap,

        "macro_no_information_AP":
            no_info_macro_ap,

        "macro_gate_pass":
            MACRO_GATE,

        "worst_policy_AP":
            policy_worst_ap,

        "worst_no_information_AP":
            no_info_worst_ap,

        "worst_gate_pass":
            WORST_GATE,

        "both_required":
            True,

        "both_pass":
            RANKING_GATE_PASS,
    },

    "decision":
        DECISION,

    "model_training_occurred":
        True,

    "threshold_selection_occurred":
        True,

    "graph_mutated":
        False,

    "official_graph_metric_evaluated":
        False,

    "Fold0_used":
        False,
}


DECISION_CONTENT_SHA = hashlib.sha256(
    json.dumps(
        DECISION_RECORD,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


DECISION_RECORD[
    "decision_sha256"
] = DECISION_CONTENT_SHA


# ============================================================
# 18. TEMP OUTPUT PATHS
# ============================================================

TMP_OOF = OOF_PATH.with_name(
    OOF_PATH.name
    +
    ".tmp_12_13a5"
)


TMP_INNER = INNER_METRICS_PATH.with_name(
    INNER_METRICS_PATH.name
    +
    ".tmp_12_13a5"
)


TMP_SELECTION = MODEL_SELECTION_PATH.with_name(
    MODEL_SELECTION_PATH.name
    +
    ".tmp_12_13a5"
)


TMP_THRESHOLD = THRESHOLD_AUDIT_PATH.with_name(
    THRESHOLD_AUDIT_PATH.name
    +
    ".tmp_12_13a5"
)


TMP_OUTER = OUTER_METRICS_PATH.with_name(
    OUTER_METRICS_PATH.name
    +
    ".tmp_12_13a5"
)


TMP_SUMMARY = SUMMARY_PATH.with_name(
    SUMMARY_PATH.name
    +
    ".tmp_12_13a5"
)


TMP_DECISION = DECISION_PATH.with_name(
    DECISION_PATH.name
    +
    ".tmp_12_13a5"
)


for path in [
    TMP_OOF,
    TMP_INNER,
    TMP_SELECTION,
    TMP_THRESHOLD,
    TMP_OUTER,
    TMP_SUMMARY,
    TMP_DECISION,
]:

    if path.exists():

        raise RuntimeError(
            "\nStale 12.13A5 temporary artifact:\n"
            f"{path}"
        )


# ============================================================
# 19. TEMP WRITE
# ============================================================

oof.to_pickle(
    TMP_OOF
)


inner_metrics.to_pickle(
    TMP_INNER
)


model_selection.to_pickle(
    TMP_SELECTION
)


threshold_audit.to_pickle(
    TMP_THRESHOLD
)


outer_metrics.to_pickle(
    TMP_OUTER
)


TMP_SUMMARY.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


TMP_DECISION.write_text(
    json.dumps(
        DECISION_RECORD,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 20. RELOAD FIDELITY
# ============================================================

oof_check = pd.read_pickle(
    TMP_OOF
)


inner_check = pd.read_pickle(
    TMP_INNER
)


selection_check = pd.read_pickle(
    TMP_SELECTION
)


threshold_check = pd.read_pickle(
    TMP_THRESHOLD
)


outer_check = pd.read_pickle(
    TMP_OUTER
)


summary_check = json.loads(
    TMP_SUMMARY.read_text(
        encoding="utf-8"
    )
)


decision_check = json.loads(
    TMP_DECISION.read_text(
        encoding="utf-8"
    )
)


pd.testing.assert_frame_equal(
    oof_check,
    oof,
    check_exact=True,
)


pd.testing.assert_frame_equal(
    inner_check,
    inner_metrics,
    check_exact=True,
)


pd.testing.assert_frame_equal(
    selection_check,
    model_selection,
    check_exact=True,
)


pd.testing.assert_frame_equal(
    threshold_check,
    threshold_audit,
    check_exact=True,
)


pd.testing.assert_frame_equal(
    outer_check,
    outer_metrics,
    check_exact=True,
)


assert (
    summary_check[
        "summary_sha256"
    ]
    ==
    SUMMARY_CONTENT_SHA
)


assert (
    decision_check[
        "decision_sha256"
    ]
    ==
    DECISION_CONTENT_SHA
)


assert dataframe_content_sha256(
    oof_check,
    OOF_COLUMNS,
) == OOF_CONTENT_SHA


assert dataframe_content_sha256(
    inner_check,
    INNER_COLUMNS,
) == INNER_CONTENT_SHA


assert dataframe_content_sha256(
    selection_check,
    SELECTION_COLUMNS,
) == SELECTION_CONTENT_SHA


assert dataframe_content_sha256(
    threshold_check,
    THRESHOLD_COLUMNS,
) == THRESHOLD_CONTENT_SHA


assert dataframe_content_sha256(
    outer_check,
    OUTER_COLUMNS,
) == OUTER_CONTENT_SHA


# ============================================================
# 21. FILE HASHES
# ============================================================

OOF_FILE_SHA = sha256_file(
    TMP_OOF
)


INNER_FILE_SHA = sha256_file(
    TMP_INNER
)


SELECTION_FILE_SHA = sha256_file(
    TMP_SELECTION
)


THRESHOLD_FILE_SHA = sha256_file(
    TMP_THRESHOLD
)


OUTER_FILE_SHA = sha256_file(
    TMP_OUTER
)


SUMMARY_FILE_SHA = sha256_file(
    TMP_SUMMARY
)


DECISION_FILE_SHA = sha256_file(
    TMP_DECISION
)


# ============================================================
# 22. ATOMIC PROMOTION
# ============================================================

TMP_OOF.replace(
    OOF_PATH
)


TMP_INNER.replace(
    INNER_METRICS_PATH
)


TMP_SELECTION.replace(
    MODEL_SELECTION_PATH
)


TMP_THRESHOLD.replace(
    THRESHOLD_AUDIT_PATH
)


TMP_OUTER.replace(
    OUTER_METRICS_PATH
)


TMP_SUMMARY.replace(
    SUMMARY_PATH
)


TMP_DECISION.replace(
    DECISION_PATH
)


# ============================================================
# 23. OPTIONAL FORMAL BRANCH CLOSURE ON FAIL
# ============================================================

CLOSURE_SHA = None


if not RANKING_GATE_PASS:

    CLOSURE = {
        "stage":
            "12.13A5",

        "branch_id":
            (
                "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_"
                "NODE_RECOVERY_V1"
            ),

        "decision":
            "CLOSE_BRANCH_NO_GRAPH_MUTATION",

        "reason":
            "A3_FROZEN_RANKING_GATE_NOT_BOTH_SATISFIED",

        "ranking_gate": {
            "macro_policy_AP":
                policy_macro_ap,

            "macro_no_information_AP":
                no_info_macro_ap,

            "macro_gate_pass":
                MACRO_GATE,

            "worst_policy_AP":
                policy_worst_ap,

            "worst_no_information_AP":
                no_info_worst_ap,

            "worst_gate_pass":
                WORST_GATE,
        },

        "model_training_occurred":
            True,

        "threshold_selection_occurred":
            True,

        "graph_mutation":
            False,

        "Fold0_used":
            False,
    }


    CLOSURE_SHA = hashlib.sha256(
        json.dumps(
            CLOSURE,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        ).encode(
            "utf-8"
        )
    ).hexdigest()


    CLOSURE[
        "closure_sha256"
    ] = CLOSURE_SHA


    CLOSURE_PATH.write_text(
        json.dumps(
            CLOSURE,
            indent=2,
        ),
        encoding="utf-8",
    )


# ============================================================
# 24. FINAL FILE LOCK
# ============================================================

assert sha256_file(
    OOF_PATH
) == OOF_FILE_SHA


assert sha256_file(
    INNER_METRICS_PATH
) == INNER_FILE_SHA


assert sha256_file(
    MODEL_SELECTION_PATH
) == SELECTION_FILE_SHA


assert sha256_file(
    THRESHOLD_AUDIT_PATH
) == THRESHOLD_FILE_SHA


assert sha256_file(
    OUTER_METRICS_PATH
) == OUTER_FILE_SHA


assert sha256_file(
    SUMMARY_PATH
) == SUMMARY_FILE_SHA


assert sha256_file(
    DECISION_PATH
) == DECISION_FILE_SHA


# ============================================================
# 25. FINAL PROTOCOL ASSERTIONS
# ============================================================

assert not (
    oof[
        "fold"
    ].astype(int)
    ==
    0
).any()


GRAPH_MUTATED = False

OFFICIAL_GRAPH_METRIC_EVALUATED = False

FOLD0_USED = False


assert GRAPH_MUTATED is False

assert OFFICIAL_GRAPH_METRIC_EVALUATED is False

assert FOLD0_USED is False


# ============================================================
# 26. FINAL
# ============================================================

print(
    "\n========== 12.13A5 FINAL =========="
)


print(
    "FORMAL_NESTED_LOFO_RANKING_AND_THRESHOLD_DEVELOPMENT:",
    "PASS"
)


print(
    "\n--- DATA ---"
)


print(
    "rows:",
    EXPECTED_ROWS
)


print(
    "positives:",
    EXPECTED_POSITIVES
)


print(
    "features:",
    len(
        MODEL_FEATURES
    )
)


print(
    "\n--- OUTER MODEL SELECTION ---"
)


print(
    model_selection.to_string(
        index=False
    )
)


print(
    "\n--- OUTER THRESHOLDS ---"
)


print(
    threshold_audit[
        [
            "outer_fold",
            "selected_scorer",
            "threshold",
            "inner_precision",
            "inner_recall",
            "inner_f0p5",
            "inner_selected_count",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n--- OUTER METRICS ---"
)


print(
    outer_metrics[
        [
            "outer_fold",
            "selected_scorer",
            "policy_average_precision",
            "no_information_average_precision",
            "proposal_count_average_precision",
            "g2_reciprocal_average_precision",
            "selected_count",
            "precision",
            "recall",
            "f0p5",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n--- RANKING GATE ---"
)


print(
    "policy macro mean AP:",
    policy_macro_ap
)


print(
    "no-info macro mean AP:",
    no_info_macro_ap
)


print(
    "MACRO GATE:",
    (
        "PASS"
        if MACRO_GATE
        else
        "FAIL"
    )
)


print(
    "policy worst-fold AP:",
    policy_worst_ap
)


print(
    "no-info worst-fold AP:",
    no_info_worst_ap
)


print(
    "WORST-FOLD GATE:",
    (
        "PASS"
        if WORST_GATE
        else
        "FAIL"
    )
)


print(
    "BOTH PASS:",
    RANKING_GATE_PASS
)


print(
    "DECISION:",
    DECISION
)


print(
    "\n--- BASELINE AUDIT ---"
)


print(
    "proposal_count macro AP:",
    proposal_macro_ap
)


print(
    "G2 reciprocal macro AP:",
    reciprocal_macro_ap
)


print(
    "\n--- PROTOCOL ---"
)


print(
    "GT/oracle training label used:",
    "YES"
)


print(
    "outer labels used for model selection:",
    "NO"
)


print(
    "outer labels used for threshold selection:",
    "NO"
)


print(
    "hyperparameter search:",
    "NO"
)


print(
    "graph mutation:",
    "NO"
)


print(
    "official graph metric:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "\n--- CONTENT SHAS ---"
)


print(
    "OOF:",
    OOF_CONTENT_SHA
)


print(
    "inner metrics:",
    INNER_CONTENT_SHA
)


print(
    "model selection:",
    SELECTION_CONTENT_SHA
)


print(
    "threshold audit:",
    THRESHOLD_CONTENT_SHA
)


print(
    "outer metrics:",
    OUTER_CONTENT_SHA
)


print(
    "summary:",
    SUMMARY_CONTENT_SHA
)


print(
    "decision:",
    DECISION_CONTENT_SHA
)


print(
    "\n--- FILE SHAS ---"
)


print(
    "OOF:",
    OOF_FILE_SHA
)


print(
    "inner metrics:",
    INNER_FILE_SHA
)


print(
    "model selection:",
    SELECTION_FILE_SHA
)


print(
    "threshold audit:",
    THRESHOLD_FILE_SHA
)


print(
    "outer metrics:",
    OUTER_FILE_SHA
)


print(
    "summary:",
    SUMMARY_FILE_SHA
)


print(
    "decision:",
    DECISION_FILE_SHA
)


if CLOSURE_SHA is not None:

    print(
        "closure:",
        CLOSURE_SHA
    )


print(
    "\nnext:"
)


if RANKING_GATE_PASS:

    print(
        "12.13A6_HELD_OUT_GRAPH_ATTACHMENT_AND_"
        "OFFICIAL_METRIC_EVALUATION"
    )

else:

    print(
        "BRANCH_CLOSED"
    )

In [ ]:
# ============================================================
# Stage 12.13A6-P1-R2-R3
#
# TEMPORARY READ-ONLY REPAIR
#
# CONFIG-DRIVEN OFFICIAL 7um EVALUATOR LOCK
#
# ============================================================
#
# REPAIRS FAILED ASSUMPTION:
#
#   evaluate_graph must contain literal:
#
#       max_distance=7.0
#
# WRONG.
#
# ACTUAL canonical implementation:
#
#       max_distance=config.metric_max_distance
#
# Therefore we must prove:
#
#   1. evaluate_graph passes:
#          config.metric_max_distance
#
#   2. config default is:
#          FROZEN_V9_CONFIG
#
#   3. runtime:
#          FROZEN_V9_CONFIG.metric_max_distance == 7.0
#
#   4. FrozenV9Config field default == 7.0
#
#   5. source-lock:
#          FrozenV9Config
#          FROZEN_V9_CONFIG assignment
#
#   6. source-lock evaluator dependencies and metric semantics
#
# ------------------------------------------------------------
# NO GT
# NO ORACLE
# NO OOF LABELS
# NO MODEL TRAINING
# NO THRESHOLD SELECTION
# NO GRAPH MUTATION
# NO OFFICIAL METRIC EXECUTION
# NO FOLD0
# NO FILES WRITTEN
# ============================================================


from pathlib import Path

import ast
import dataclasses
import hashlib
import inspect
import json
import sys


# ============================================================
# 0. PROJECT
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

SRC = (
    PROJECT
    / "src"
)


if str(SRC) not in sys.path:

    sys.path.insert(
        0,
        str(SRC),
    )


from biohub_cell_tracking import frozen_v9 as fv9


# ============================================================
# 1. PREVIOUSLY OBSERVED LOCKS
# ============================================================

EXPECTED_EVALUATE_GRAPH_SHA = (
    "44bd29366669cad4a1c0767c76987b75"
    "44d08b47ac493b25c0224d35b073c536"
)

EXPECTED_ADD_NODE_SHA = (
    "c6d83ad92685abf739e4578375977ebf"
    "ae04e09d23f694472afe45e41757e947"
)

EXPECTED_ADD_EDGE_SHA = (
    "702c88775a3b14165272c4cdc49860df"
    "ee921faf94f18de4a8ef2c614f63fd87"
)

EXPECTED_COPY_SHA = (
    "da4f0dd391d8246d15259ac0d4a12819"
    "bc0fa069ff89f7c3469122fc79432a91"
)

EXPECTED_GRAPH_CLASS_FILE_SHA = (
    "94d6b2b651c84bf4b814a5f121984b8e"
    "b20c115de5912f08161e41695bf90693"
)


CANONICAL_BUILDER_SHA = (
    "9de9bb061963d9a5b77390c68849b8bb"
    "d389586d7296cbb88ba332052bdd2674"
)

CANONICAL_SYNTHETIC_LOOP_SHA = (
    "d532c81df88a77ce275b582dcbc7751ad"
    "43042710159250d0441d117d8c2bc98"
)

CANONICAL_ADD_NODE_CALL_SHA = (
    "353e100508742ae232bbe28ca214733a71"
    "b6869252d09e1a575b4855f6d67c48"
)

CANONICAL_IN_EDGE_CALL_SHA = (
    "39d04a46083ecce88bc3832cad7eed74d"
    "4a71d50e3725f27681cc3781f172b36"
)

CANONICAL_OUT_EDGE_CALL_SHA = (
    "49ba0598cde62964d30e86334ddec1ec"
    "f45161a235aaeec6c60010fda381e40f"
)


# ============================================================
# 2. HELPERS
# ============================================================

def source_sha(source):

    return hashlib.sha256(
        source.encode(
            "utf-8"
        )
    ).hexdigest()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def safe_signature(obj):

    try:

        return str(
            inspect.signature(
                obj
            )
        )

    except Exception as exc:

        return (
            "<unavailable: "
            f"{type(exc).__name__}: {exc}>"
        )


def safe_source(obj):

    try:

        source = inspect.getsource(
            obj
        )

        return {
            "available":
                True,

            "source":
                source,

            "sha256":
                source_sha(
                    source
                ),
        }

    except Exception as exc:

        return {
            "available":
                False,

            "source":
                None,

            "sha256":
                None,

            "error":
                (
                    f"{type(exc).__name__}: {exc}"
                ),
        }


# ============================================================
# 3. CANONICAL EVALUATOR SHA LOCK
# ============================================================

evaluate_graph = (
    fv9.evaluate_graph
)


evaluate_source = (
    inspect.getsource(
        evaluate_graph
    )
)


EVALUATE_GRAPH_SHA = (
    source_sha(
        evaluate_source
    )
)


assert (
    EVALUATE_GRAPH_SHA
    ==
    EXPECTED_EVALUATE_GRAPH_SHA
)


print(
    "========== 12.13A6-P1-R2-R3 EVALUATOR LOCK =========="
)

print(
    "object:",
    "biohub_cell_tracking.frozen_v9.evaluate_graph"
)

print(
    "SHA:",
    EVALUATE_GRAPH_SHA
)

print(
    "EVALUATE_GRAPH_SOURCE_LOCK: PASS"
)


# ============================================================
# 4. AST:
# PROVE max_distance=config.metric_max_distance
# ============================================================

tree = ast.parse(
    evaluate_source
)


assert len(
    tree.body
) == 1


fn = tree.body[
    0
]


assert isinstance(
    fn,
    ast.FunctionDef,
)


max_distance_expressions = []


for node in ast.walk(
    fn
):

    if not isinstance(
        node,
        ast.Call,
    ):
        continue


    for keyword in node.keywords:

        if (
            keyword.arg
            !=
            "max_distance"
        ):

            continue


        max_distance_expressions.append(
            keyword.value
        )


assert len(
    max_distance_expressions
) == 1


max_distance_expr = (
    max_distance_expressions[
        0
    ]
)


# Must be exactly:
#
# config.metric_max_distance

assert isinstance(
    max_distance_expr,
    ast.Attribute,
)


assert (
    max_distance_expr.attr
    ==
    "metric_max_distance"
)


assert isinstance(
    max_distance_expr.value,
    ast.Name,
)


assert (
    max_distance_expr.value.id
    ==
    "config"
)


MAX_DISTANCE_EXPRESSION = (
    ast.unparse(
        max_distance_expr
    )
)


assert (
    MAX_DISTANCE_EXPRESSION
    ==
    "config.metric_max_distance"
)


print(
    "\n========== EVALUATOR DISTANCE EXPRESSION =========="
)

print(
    "max_distance expression:",
    MAX_DISTANCE_EXPRESSION
)

print(
    "literal 7.0 required in wrapper:",
    "NO"
)

print(
    "CONFIG_DRIVEN_DISTANCE_EXPRESSION_LOCK: PASS"
)


# ============================================================
# 5. PROVE config DEFAULT SYMBOL
#
# Signature source should bind:
#
# config: FrozenV9Config = FROZEN_V9_CONFIG
# ============================================================

argument_names = [
    arg.arg

    for arg
    in fn.args.args
]


assert "config" in argument_names


config_arg_index = (
    argument_names.index(
        "config"
    )
)


# Python AST stores only trailing defaults.
number_args = len(
    fn.args.args
)

number_defaults = len(
    fn.args.defaults
)

first_default_arg_index = (
    number_args
    -
    number_defaults
)


assert (
    config_arg_index
    >=
    first_default_arg_index
)


config_default_index = (
    config_arg_index
    -
    first_default_arg_index
)


config_default_ast = (
    fn.args.defaults[
        config_default_index
    ]
)


assert isinstance(
    config_default_ast,
    ast.Name,
)


assert (
    config_default_ast.id
    ==
    "FROZEN_V9_CONFIG"
)


print(
    "\n========== EVALUATOR CONFIG DEFAULT SYMBOL =========="
)

print(
    "config default:",
    config_default_ast.id
)

print(
    "EVALUATOR_DEFAULT_CONFIG_SYMBOL_LOCK: PASS"
)


# ============================================================
# 6. RUNTIME SIGNATURE DEFAULT IDENTITY
# ============================================================

signature = inspect.signature(
    evaluate_graph
)


config_parameter = (
    signature.parameters[
        "config"
    ]
)


runtime_default_config = (
    config_parameter.default
)


assert (
    runtime_default_config
    is
    fv9.FROZEN_V9_CONFIG
), (
    "\nevaluate_graph runtime default is not "
    "fv9.FROZEN_V9_CONFIG identity."
)


print(
    "\n========== RUNTIME DEFAULT CONFIG IDENTITY =========="
)

print(
    "signature config default:",
    runtime_default_config
)

print(
    "default is fv9.FROZEN_V9_CONFIG:",
    "YES"
)

print(
    "RUNTIME_DEFAULT_CONFIG_IDENTITY_LOCK: PASS"
)


# ============================================================
# 7. RUNTIME 7um VALUE
# ============================================================

assert hasattr(
    fv9.FROZEN_V9_CONFIG,
    "metric_max_distance"
)


RUNTIME_METRIC_MAX_DISTANCE = float(
    fv9.FROZEN_V9_CONFIG.metric_max_distance
)


assert (
    RUNTIME_METRIC_MAX_DISTANCE
    ==
    7.0
)


print(
    "\n========== FROZEN CONFIG DISTANCE =========="
)

print(
    "FROZEN_V9_CONFIG.metric_max_distance:",
    RUNTIME_METRIC_MAX_DISTANCE
)

print(
    "official tolerance:",
    "7.0 um"
)

print(
    "FROZEN_CONFIG_7UM_LOCK: PASS"
)


# ============================================================
# 8. FrozenV9Config CLASS SOURCE LOCK
# ============================================================

assert hasattr(
    fv9,
    "FrozenV9Config"
)


config_class = (
    fv9.FrozenV9Config
)


config_class_source = (
    inspect.getsource(
        config_class
    )
)


CONFIG_CLASS_SHA = (
    source_sha(
        config_class_source
    )
)


print(
    "\n========== FrozenV9Config SOURCE =========="
)

print(
    "SHA:",
    CONFIG_CLASS_SHA
)

print(
    config_class_source
)


# ============================================================
# 9. DATACLASS FIELD DEFAULT LOCK
# ============================================================

assert dataclasses.is_dataclass(
    config_class
)


fields = {
    field.name:
        field

    for field
    in dataclasses.fields(
        config_class
    )
}


assert (
    "metric_max_distance"
    in
    fields
)


metric_distance_field = (
    fields[
        "metric_max_distance"
    ]
)


assert (
    metric_distance_field.default
    is not
    dataclasses.MISSING
)


FIELD_DEFAULT_DISTANCE = float(
    metric_distance_field.default
)


assert (
    FIELD_DEFAULT_DISTANCE
    ==
    7.0
)


print(
    "\n========== CONFIG FIELD DEFAULT =========="
)

print(
    "field:",
    "metric_max_distance"
)

print(
    "dataclass default:",
    FIELD_DEFAULT_DISTANCE
)

print(
    "CONFIG_FIELD_DEFAULT_7UM_LOCK: PASS"
)


# ============================================================
# 10. SOURCE-LOCK FROZEN_V9_CONFIG ASSIGNMENT
# ============================================================

module_file = Path(
    inspect.getfile(
        fv9
    )
).resolve()


assert module_file.exists()


module_source = (
    module_file.read_text(
        encoding="utf-8"
    )
)


MODULE_FILE_SHA = (
    sha256_file(
        module_file
    )
)


module_tree = ast.parse(
    module_source
)


assignment_matches = []


for node in module_tree.body:

    # --------------------------------------------------------
    # FROZEN_V9_CONFIG = ...
    # --------------------------------------------------------

    if isinstance(
        node,
        ast.Assign,
    ):

        for target in node.targets:

            if (
                isinstance(
                    target,
                    ast.Name,
                )
                and
                target.id
                ==
                "FROZEN_V9_CONFIG"
            ):

                assignment_matches.append(
                    node
                )


    # --------------------------------------------------------
    # FROZEN_V9_CONFIG: FrozenV9Config = ...
    # --------------------------------------------------------

    elif isinstance(
        node,
        ast.AnnAssign,
    ):

        if (
            isinstance(
                node.target,
                ast.Name,
            )
            and
            node.target.id
            ==
            "FROZEN_V9_CONFIG"
        ):

            assignment_matches.append(
                node
            )


assert len(
    assignment_matches
) == 1


config_assignment_node = (
    assignment_matches[
        0
    ]
)


config_assignment_source = (
    ast.get_source_segment(
        module_source,
        config_assignment_node,
    )
    or
    ast.unparse(
        config_assignment_node
    )
)


CONFIG_ASSIGNMENT_SHA = (
    source_sha(
        config_assignment_source
    )
)


print(
    "\n========== FROZEN_V9_CONFIG ASSIGNMENT =========="
)

print(
    "module:",
    module_file
)

print(
    "module file SHA:",
    MODULE_FILE_SHA
)

print(
    "assignment SHA:",
    CONFIG_ASSIGNMENT_SHA
)

print(
    config_assignment_source
)

print(
    "FROZEN_CONFIG_ASSIGNMENT_LOCK: PASS"
)


# ============================================================
# 11. COMPLETE 7um PROVENANCE CHAIN
# ============================================================

DISTANCE_PROVENANCE = {
    "evaluate_graph_sha256":
        EVALUATE_GRAPH_SHA,

    "evaluate_graph_argument":
        "max_distance",

    "evaluate_graph_argument_expression":
        "config.metric_max_distance",

    "config_parameter_default_symbol":
        "FROZEN_V9_CONFIG",

    "runtime_default_identity":
        True,

    "FrozenV9Config_class_sha256":
        CONFIG_CLASS_SHA,

    "FrozenV9Config_metric_max_distance_default":
        FIELD_DEFAULT_DISTANCE,

    "FROZEN_V9_CONFIG_assignment_sha256":
        CONFIG_ASSIGNMENT_SHA,

    "FROZEN_V9_CONFIG_runtime_metric_max_distance":
        RUNTIME_METRIC_MAX_DISTANCE,

    "official_tolerance_um":
        7.0,
}


assert (
    DISTANCE_PROVENANCE[
        "official_tolerance_um"
    ]
    ==
    7.0
)


print(
    "\n========== COMPLETE 7um PROVENANCE =========="
)

print(
    "evaluate_graph:"
)

print(
    "  max_distance = config.metric_max_distance"
)

print(
    "config default:"
)

print(
    "  FROZEN_V9_CONFIG"
)

print(
    "FrozenV9Config field default:"
)

print(
    "  metric_max_distance =",
    FIELD_DEFAULT_DISTANCE
)

print(
    "runtime frozen config:"
)

print(
    "  metric_max_distance =",
    RUNTIME_METRIC_MAX_DISTANCE
)

print(
    "OFFICIAL_DISTANCE_TOLERANCE_LOCK: PASS"
)


# ============================================================
# 12. DIRECT EVALUATOR DEPENDENCIES
# ============================================================

eval_globals = (
    evaluate_graph.__globals__
)


direct_call_names = set()


for node in ast.walk(
    fn
):

    if not isinstance(
        node,
        ast.Call,
    ):
        continue


    if isinstance(
        node.func,
        ast.Name,
    ):

        direct_call_names.add(
            node.func.id
        )


dependency_manifest = {}


print(
    "\n========== EVALUATOR DIRECT DEPENDENCIES =========="
)


for name in sorted(
    direct_call_names
):

    if name not in eval_globals:
        continue


    obj = eval_globals[
        name
    ]


    if not callable(obj):
        continue


    record = safe_source(
        obj
    )


    dependency_manifest[
        name
    ] = {
        "module":
            getattr(
                obj,
                "__module__",
                None,
            ),

        "qualname":
            getattr(
                obj,
                "__qualname__",
                None,
            ),

        "signature":
            safe_signature(
                obj
            ),

        "source_available":
            record[
                "available"
            ],

        "source_sha256":
            record[
                "sha256"
            ],
    }


    print()

    print(
        "name:",
        name
    )

    print(
        "module:",
        dependency_manifest[
            name
        ][
            "module"
        ]
    )

    print(
        "qualname:",
        dependency_manifest[
            name
        ][
            "qualname"
        ]
    )

    print(
        "signature:",
        dependency_manifest[
            name
        ][
            "signature"
        ]
    )

    print(
        "source SHA:",
        dependency_manifest[
            name
        ][
            "source_sha256"
        ]
    )


# ============================================================
# 13. EXPLICIT summarise SOURCE LOCK
#
# Formal A6 needs pooled/fold summaries in addition to
# per-dataset evaluate_graph.
# ============================================================

assert hasattr(
    fv9,
    "summarise"
), (
    "\nfrozen_v9.summarise unavailable; "
    "cannot source-lock pooled metric aggregation."
)


summarise_obj = (
    fv9.summarise
)


summarise_source = (
    inspect.getsource(
        summarise_obj
    )
)


SUMMARISE_SHA = (
    source_sha(
        summarise_source
    )
)


print(
    "\n========== OFFICIAL SUMMARY FUNCTION =========="
)

print(
    "object:",
    "biohub_cell_tracking.frozen_v9.summarise"
)

print(
    "signature:",
    safe_signature(
        summarise_obj
    )
)

print(
    "SHA:",
    SUMMARISE_SHA
)

print(
    summarise_source
)

print(
    "OFFICIAL_SUMMARISE_SOURCE_LOCK: PASS"
)


# ============================================================
# 14. METRIC SEMANTICS CLOSURE
# ============================================================

combined_metric_source = (
    evaluate_source
    +
    "\n"
    +
    summarise_source
)


for name in dependency_manifest:

    obj = eval_globals[
        name
    ]

    record = safe_source(
        obj
    )


    if record[
        "available"
    ]:

        combined_metric_source += (
            "\n"
            +
            record[
                "source"
            ]
        )


REQUIRED_METRIC_TOKENS = [
    "edge_jaccard",
    "node_recall",
    "division_jaccard",
    "score",
]


missing_metric_tokens = [
    token

    for token
    in REQUIRED_METRIC_TOKENS

    if token
    not in
    combined_metric_source
]


assert not missing_metric_tokens, (
    "\nMetric dependency closure missing required "
    "Stage12 metric semantics:\n"
    f"{missing_metric_tokens}"
)


print(
    "\n========== OFFICIAL METRIC SEMANTICS =========="
)


for token in REQUIRED_METRIC_TOKENS:

    print(
        token,
        ": PRESENT"
    )


print(
    "OFFICIAL_METRIC_SEMANTICS_LOCK: PASS"
)


# ============================================================
# 15. FULL A6 SOURCE MANIFEST
# ============================================================

SOURCE_MANIFEST = {
    "serialization": {
        "canonical_builder_sha256":
            CANONICAL_BUILDER_SHA,

        "canonical_synthetic_loop_sha256":
            CANONICAL_SYNTHETIC_LOOP_SHA,

        "canonical_add_node_call_sha256":
            CANONICAL_ADD_NODE_CALL_SHA,

        "canonical_incoming_edge_call_sha256":
            CANONICAL_IN_EDGE_CALL_SHA,

        "canonical_outgoing_edge_call_sha256":
            CANONICAL_OUT_EDGE_CALL_SHA,

        "runtime_add_node_sha256":
            EXPECTED_ADD_NODE_SHA,

        "runtime_add_edge_sha256":
            EXPECTED_ADD_EDGE_SHA,

        "runtime_copy_sha256":
            EXPECTED_COPY_SHA,

        "runtime_graph_class_file_sha256":
            EXPECTED_GRAPH_CLASS_FILE_SHA,
    },

    "official_evaluator": {
        "evaluate_graph_sha256":
            EVALUATE_GRAPH_SHA,

        "summarise_sha256":
            SUMMARISE_SHA,

        "dependencies":
            dependency_manifest,

        "distance_provenance":
            DISTANCE_PROVENANCE,
    },

    "protocol": {
        "GT_loaded":
            False,

        "oracle_loaded":
            False,

        "A5_OOF_labels_loaded":
            False,

        "model_trained":
            False,

        "threshold_selected":
            False,

        "graph_mutated":
            False,

        "official_metric_executed":
            False,

        "Fold0_used":
            False,
    },
}


MANIFEST_SHA = hashlib.sha256(
    json.dumps(
        SOURCE_MANIFEST,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
        default=str,
    ).encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# 16. SAFETY
# ============================================================

GT_LOADED = False
ORACLE_LOADED = False
A5_OOF_LABELS_LOADED = False

MODEL_TRAINED = False
THRESHOLD_SELECTED = False

GRAPH_MUTATED = False

OFFICIAL_METRIC_EXECUTED = False

FOLD0_USED = False


assert GT_LOADED is False
assert ORACLE_LOADED is False
assert A5_OOF_LABELS_LOADED is False

assert MODEL_TRAINED is False
assert THRESHOLD_SELECTED is False

assert GRAPH_MUTATED is False

assert OFFICIAL_METRIC_EXECUTED is False

assert FOLD0_USED is False


# ============================================================
# 17. FINAL
# ============================================================

print(
    "\n========== 12.13A6-P1-R2-R3 FINAL =========="
)


print(
    "CONFIG_DRIVEN_7UM_OFFICIAL_EVALUATOR_SOURCE_LOCK:",
    "PASS"
)


print(
    "\n--- evaluate_graph ---"
)

print(
    "SHA:",
    EVALUATE_GRAPH_SHA
)

print(
    "max_distance expression:",
    MAX_DISTANCE_EXPRESSION
)

print(
    "config default symbol:",
    "FROZEN_V9_CONFIG"
)


print(
    "\n--- FrozenV9Config ---"
)

print(
    "class SHA:",
    CONFIG_CLASS_SHA
)

print(
    "field default metric_max_distance:",
    FIELD_DEFAULT_DISTANCE
)


print(
    "\n--- FROZEN_V9_CONFIG ---"
)

print(
    "assignment SHA:",
    CONFIG_ASSIGNMENT_SHA
)

print(
    "runtime metric_max_distance:",
    RUNTIME_METRIC_MAX_DISTANCE
)


print(
    "\n--- OFFICIAL TOLERANCE ---"
)

print(
    "7.0 um:",
    "LOCKED"
)

print(
    "OFFICIAL_DISTANCE_TOLERANCE_LOCK: PASS"
)


print(
    "\n--- OFFICIAL SUMMARY ---"
)

print(
    "summarise SHA:",
    SUMMARISE_SHA
)


print(
    "\n--- METRIC DEPENDENCIES ---"
)

print(
    sorted(
        dependency_manifest.keys()
    )
)


print(
    "\n--- METRIC SEMANTICS ---"
)

print(
    REQUIRED_METRIC_TOKENS
)

print(
    "OFFICIAL_METRIC_SEMANTICS_LOCK: PASS"
)


print(
    "\n--- COMPLETE A6 SOURCE MANIFEST ---"
)

print(
    "manifest SHA:",
    MANIFEST_SHA
)


print(
    "\n--- SAFETY ---"
)

print(
    "GT loaded:",
    "NO"
)

print(
    "oracle loaded:",
    "NO"
)

print(
    "A5 OOF labels loaded:",
    "NO"
)

print(
    "model trained:",
    "NO"
)

print(
    "threshold selected:",
    "NO"
)

print(
    "graph mutation:",
    "NO"
)

print(
    "official metric executed:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)


print(
    "\nnext:"
)

print(
    "12.13A6_FORMAL_HELD_OUT_GRAPH_ATTACHMENT_"
    "AND_OFFICIAL_METRIC_EVALUATION"
)

In [ ]:
# ============================================================
# Stage 12.13A6-R2
#
# FORMAL POST-GT OFFICIAL METRIC EVALUATION
#
# SUCCESSOR OF:
#   12.13A6-R1
#
# ============================================================
#
# WHY R2 EXISTS
#
# A6-R1 successfully completed:
#
#   PRE_GT_ACTION_FREEZE: PASS
#
# BEFORE first official GT read.
#
# Frozen A6-R1 actions:
#
#   A5 selected candidates = 3807
#   accepted actions       = 3105
#   skipped                = 702
#   1-edge actions         = 1766
#   2-edge actions         = 1339
#   total new edges        = 4444
#
# actions content SHA:
#   a887af0f16e3375bf59d34ab8e211e36
#   ab8c01c4c1599b2c1d2f2713d708e64d
#
# actions file SHA:
#   33900e3787d21e50ad8fbaae10fdbf25
#   e621f97493f0e83b0542cadeb241695d
#
# pre-GT contract content SHA:
#   303051528929430a3c6b41cabe56c249
#   9f1da191cc595780e8b30274293f65dc
#
# ------------------------------------------------------------
# A6-R1 FAILED ONLY HERE:
#
#     base_metric["score"]
#
# evaluate_graph() returns PER-SAMPLE metric rows.
# It does NOT provide the combined run-level "score".
#
# Combined score authority is:
#
#     fv9.summarise(rows)
#
# ------------------------------------------------------------
# R2 STRICT RULE
#
# R2 MUST NOT:
#
#   regenerate A6 actions
#   recompute attachment choices
#   change family mapping
#   select/retrain model
#   select/retune threshold
#   use Fold0
#   modify persistent GEFF
#
# It ONLY:
#
#   read-locks frozen pre-GT A6-R1 artifacts
#   replays those exact frozen actions
#   performs official evaluation
#   aggregates with frozen_v9.summarise()
#   applies frozen six graph promotion gates
#
# KEEP PERMANENTLY.
# ============================================================


from pathlib import Path

import gc
import hashlib
import inspect
import json
import math
import sys

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)

SRC = (
    PROJECT
    / "src"
)

TRAIN_ROOT = (
    PROJECT
    / "data"
    / "raw"
    / "competition"
    / "biohub-cell-tracking-during-development"
    / "train"
)

CV_PATH = (
    PROJECT
    / "configs"
    / "cv_manifest.csv"
)


# ------------------------------------------------------------
# Frozen parents
# ------------------------------------------------------------

A3_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_ranking_attachment_contract_frozen.json"
)

A5_SUMMARY_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_nested_lofo_ranking_v1_summary.json"
)

A5_DECISION_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_nested_lofo_ranking_v1_decision_frozen.json"
)

A5_OOF_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_nested_lofo_ranking_v1_oof.pkl"
)

UNIVERSE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_candidate_universe_v1.pkl"
)


# ------------------------------------------------------------
# A6-R1 frozen PRE-GT artifacts
#
# READ ONLY.
# ------------------------------------------------------------

ACTION_AUDIT_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_attachment_audit_v1.pkl"
)

ACTIONS_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_graph_actions_v1.pkl"
)

PRE_GT_CONTRACT_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_pre_gt_action_contract_frozen.json"
)


# ------------------------------------------------------------
# A6 POST-GT formal outputs
# ------------------------------------------------------------

METRICS_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_official_dataset_metrics_v1.pkl"
)

FOLD_SUMMARY_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_official_fold_summary_v1.pkl"
)

SUMMARY_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_official_summary_v1.json"
)

DECISION_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_graph_promotion_decision_frozen.json"
)

CLOSURE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_a6_graph_gate_closure.json"
)


# ============================================================
# 1. REQUIRED INPUTS
# ============================================================

for path in [
    TRAIN_ROOT,
    CV_PATH,
    A3_PATH,
    A5_SUMMARY_PATH,
    A5_DECISION_PATH,
    A5_OOF_PATH,
    UNIVERSE_PATH,
    ACTION_AUDIT_PATH,
    ACTIONS_PATH,
    PRE_GT_CONTRACT_PATH,
]:
    assert path.exists(), path


# R1 crashed before any post-GT formal artifact write.
# Never overwrite if something now exists.

for path in [
    METRICS_PATH,
    FOLD_SUMMARY_PATH,
    SUMMARY_PATH,
    DECISION_PATH,
    CLOSURE_PATH,
]:
    if path.exists():
        raise RuntimeError(
            "\nA6 post-GT formal artifact already exists:\n"
            f"{path}\n\n"
            "Do NOT overwrite formal provenance."
        )


# ============================================================
# 2. FROZEN HASHES / COUNTS
# ============================================================

EXPECTED_A3_FILE_SHA = (
    "37a5b2e557832774c0bcf65a33c81d8434bac2025c4eebbf2e3b0ea1bc955bd7"
)

EXPECTED_A3_CONTENT_SHA = (
    "c54eda46073582a7a915f1bfa7dcbff9eec8a96edcfad80a8e49e1097cf38b46"
)

EXPECTED_A5_SUMMARY_FILE_SHA = (
    "064f2efe341b4c1f30e1a09c02222ae3a771df8304c734835df9f81dff23ecb5"
)

EXPECTED_A5_DECISION_FILE_SHA = (
    "a8b0212fbe0cb293c034a96f2c0b24f79b1d54640678fb18f9b8d38c846ca2de"
)

EXPECTED_A5_SUMMARY_CONTENT_SHA = (
    "a16758ac5885c9db37cbb22e1c09ccf3cf26d882d427263c546c1e6ff7d55f3a"
)

EXPECTED_A5_DECISION_CONTENT_SHA = (
    "d29347659a6fe50a6555ac92a92dd2c2d6d58fbf4ea07b608c9b9f8b81529ef9"
)

EXPECTED_UNIVERSE_FILE_SHA = (
    "440676497d3365a0926deaf45140f57332c8e9e89d46b88401a7ce6f7a299fc3"
)


# ------------------------------------------------------------
# Immutable A6-R1 PRE-GT freeze
# ------------------------------------------------------------

EXPECTED_ACTIONS_CONTENT_SHA = (
    "a887af0f16e3375bf59d34ab8e211e36ab8c01c4c1599b2c1d2f2713d708e64d"
)

EXPECTED_ACTIONS_FILE_SHA = (
    "33900e3787d21e50ad8fbaae10fdbf25e621f97493f0e83b0542cadeb241695d"
)

EXPECTED_PRE_GT_CONTRACT_CONTENT_SHA = (
    "303051528929430a3c6b41cabe56c2499f1da191cc595780e8b30274293f65dc"
)


EXPECTED_SELECTED = 3_807

EXPECTED_ACTIONS = 3_105

EXPECTED_SKIPPED = 702

EXPECTED_ONE_EDGE = 1_766

EXPECTED_TWO_EDGE = 1_339

EXPECTED_NEW_EDGES = 4_444


# ------------------------------------------------------------
# Branch corpus
# ------------------------------------------------------------

EXPECTED_BRANCH_DATASETS = 103

EXPECTED_UNIVERSE_ROWS = 1_079_885


FOLDS = [
    1,
    2,
    3,
    4,
]


# ------------------------------------------------------------
# Canonical post-G3H3 baseline
# ------------------------------------------------------------

EXPECTED_BASE_EDGE_TP = 77_532

EXPECTED_BASE_EDGE_FP = 9_388

EXPECTED_BASE_EDGE_FN = 10_257


EXPECTED_BASE_SCORE = (
    0.7920280500935799
)


# ------------------------------------------------------------
# Source locks
# ------------------------------------------------------------

EXPECTED_EVALUATE_GRAPH_SHA = (
    "44bd29366669cad4a1c0767c76987b7544d08b47ac493b25c0224d35b073c536"
)

EXPECTED_SUMMARISE_SHA = (
    "9f45841264c6586a646366b61cbe3089bfd67e434282a278cb28f18cbfe8127d"
)

EXPECTED_CONFIG_CLASS_SHA = (
    "97d9bd92c5c23f1016719c2204754fe02c0618eb91a90ce485f3d516cf1b6ed7"
)

EXPECTED_FROZEN_V9_MODULE_SHA = (
    "486ae177e8709ddd17bf20d61dc628477a78d7434cd54a7ecf3729915b36ab17"
)

EXPECTED_OFFICIAL_EVALUATE_SHA = (
    "afe1146100ae1f9a8d098e693235c48f56212f729bbd00a036e623cb85b7b291"
)

EXPECTED_NODE_RECALL_SHA = (
    "fe11d576657559aeaa3100b374708fa523d33bcfcbb7f4e4b604265ec53f7c53"
)

EXPECTED_PER_SAMPLE_SHA = (
    "799f78600b85ced52ab27bb64634043a19fde0c5e307f685036e63e3732868a3"
)

EXPECTED_ADD_NODE_SHA = (
    "c6d83ad92685abf739e4578375977ebfae04e09d23f694472afe45e41757e947"
)

EXPECTED_ADD_EDGE_SHA = (
    "702c88775a3b14165272c4cdc49860dfee921faf94f18de4a8ef2c614f63fd87"
)

EXPECTED_COPY_SHA = (
    "da4f0dd391d8246d15259ac0d4a12819bc0fa069ff89f7c3469122fc79432a91"
)

EXPECTED_GRAPH_CLASS_FILE_SHA = (
    "94d6b2b651c84bf4b814a5f121984b8eb20c115de5912f08161e41695bf90693"
)


A6_PREFLIGHT_SOURCE_MANIFEST_SHA = (
    "e6ccc5deee777846f354bf58a1f3e38567c96450aa0f312b25260e8e453b45b7"
)


# ============================================================
# 3. HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def source_sha(source):

    return hashlib.sha256(
        source.encode(
            "utf-8"
        )
    ).hexdigest()


class HashWriter:

    def __init__(self):

        self.h = hashlib.sha256()


    def write(
        self,
        text,
    ):

        if not isinstance(
            text,
            str,
        ):
            text = str(
                text
            )

        self.h.update(
            text.encode(
                "utf-8"
            )
        )

        return len(
            text
        )


    def flush(self):

        return None


def dataframe_content_sha256(
    df,
    columns=None,
):

    if columns is None:

        columns = list(
            df.columns
        )


    writer = HashWriter()


    df.loc[
        :,
        columns,
    ].to_csv(
        writer,
        index=False,
        na_rep="NaN",
        float_format="%.17g",
        lineterminator="\n",
    )


    return writer.h.hexdigest()


def json_content_sha256(
    obj,
):

    return hashlib.sha256(
        json.dumps(
            obj,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
            default=str,
        ).encode(
            "utf-8"
        )
    ).hexdigest()


def jsonable(
    value,
):

    if isinstance(
        value,
        dict,
    ):

        return {
            str(k):
                jsonable(v)

            for k, v
            in value.items()
        }


    if isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        return [
            jsonable(v)

            for v
            in value
        ]


    if isinstance(
        value,
        np.generic,
    ):

        return value.item()


    if isinstance(
        value,
        float,
    ):

        if math.isnan(
            value
        ):
            return None

        return float(
            value
        )


    return value


def graph_state(
    graph,
):

    nodes = (
        graph
        .node_attrs(
            unpack=True
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )


    edges = (
        graph
        .edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    outdeg = (
        edges
        .groupby(
            "source_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    indeg = (
        edges
        .groupby(
            "target_id"
        )
        .size()
        .astype(int)
        .to_dict()
    )


    edge_set = set(
        zip(
            edges[
                "source_id"
            ].astype(int),

            edges[
                "target_id"
            ].astype(int),
        )
    )


    return (
        nodes,

        {
            int(k):
                int(v)

            for k, v
            in outdeg.items()
        },

        {
            int(k):
                int(v)

            for k, v
            in indeg.items()
        },

        edge_set,
    )


def fold_summary_row(
    fold_id,
    base_summary,
    policy_summary,
):

    return {
        "fold":
            int(
                fold_id
            ),

        "base_score":
            float(
                base_summary[
                    "score"
                ]
            ),

        "policy_score":
            float(
                policy_summary[
                    "score"
                ]
            ),

        "delta_score":
            float(
                policy_summary[
                    "score"
                ]
                -
                base_summary[
                    "score"
                ]
            ),

        "base_edge_jaccard":
            float(
                base_summary[
                    "edge_jaccard"
                ]
            ),

        "policy_edge_jaccard":
            float(
                policy_summary[
                    "edge_jaccard"
                ]
            ),

        "delta_edge_jaccard":
            float(
                policy_summary[
                    "edge_jaccard"
                ]
                -
                base_summary[
                    "edge_jaccard"
                ]
            ),

        "base_node_recall":
            float(
                base_summary[
                    "node_recall"
                ]
            ),

        "policy_node_recall":
            float(
                policy_summary[
                    "node_recall"
                ]
            ),

        "delta_node_recall":
            float(
                policy_summary[
                    "node_recall"
                ]
                -
                base_summary[
                    "node_recall"
                ]
            ),

        "base_division_jaccard":
            float(
                base_summary[
                    "division_jaccard"
                ]
            ),

        "policy_division_jaccard":
            float(
                policy_summary[
                    "division_jaccard"
                ]
            ),

        "delta_division_jaccard":
            float(
                policy_summary[
                    "division_jaccard"
                ]
                -
                base_summary[
                    "division_jaccard"
                ]
            ),
    }


# ============================================================
# 4. IMPORT FROZEN-V9
# ============================================================

if str(
    SRC
) not in sys.path:

    sys.path.insert(
        0,
        str(
            SRC
        ),
    )


from biohub_cell_tracking import frozen_v9 as fv9


# ============================================================
# 5. OFFICIAL SOURCE LOCK
# ============================================================

assert (
    source_sha(
        inspect.getsource(
            fv9.evaluate_graph
        )
    )
    ==
    EXPECTED_EVALUATE_GRAPH_SHA
)


assert (
    source_sha(
        inspect.getsource(
            fv9.summarise
        )
    )
    ==
    EXPECTED_SUMMARISE_SHA
)


assert (
    source_sha(
        inspect.getsource(
            fv9.FrozenV9Config
        )
    )
    ==
    EXPECTED_CONFIG_CLASS_SHA
)


assert (
    sha256_file(
        Path(
            inspect.getfile(
                fv9
            )
        )
    )
    ==
    EXPECTED_FROZEN_V9_MODULE_SHA
)


assert float(
    fv9.FROZEN_V9_CONFIG.metric_max_distance
) == 7.0


_eval_globals = (
    fv9.evaluate_graph.__globals__
)


assert (
    source_sha(
        inspect.getsource(
            _eval_globals[
                "official_evaluate"
            ]
        )
    )
    ==
    EXPECTED_OFFICIAL_EVALUATE_SHA
)


assert (
    source_sha(
        inspect.getsource(
            _eval_globals[
                "node_recall"
            ]
        )
    )
    ==
    EXPECTED_NODE_RECALL_SHA
)


assert (
    source_sha(
        inspect.getsource(
            _eval_globals[
                "per_sample_metrics"
            ]
        )
    )
    ==
    EXPECTED_PER_SAMPLE_SHA
)


print(
    "========== 12.13A6-R2 OFFICIAL SOURCE LOCK =========="
)

print(
    "evaluate_graph:",
    "PASS"
)

print(
    "summarise:",
    "PASS"
)

print(
    "metric_max_distance:",
    "7.0 um"
)

print(
    "preflight manifest:",
    A6_PREFLIGHT_SOURCE_MANIFEST_SHA
)

print(
    "OFFICIAL_SOURCE_LOCK: PASS"
)


# ============================================================
# 6. A3 / A5 PARENT LOCK
# ============================================================

assert (
    sha256_file(
        A3_PATH
    )
    ==
    EXPECTED_A3_FILE_SHA
)


assert (
    sha256_file(
        A5_SUMMARY_PATH
    )
    ==
    EXPECTED_A5_SUMMARY_FILE_SHA
)


assert (
    sha256_file(
        A5_DECISION_PATH
    )
    ==
    EXPECTED_A5_DECISION_FILE_SHA
)


assert (
    sha256_file(
        UNIVERSE_PATH
    )
    ==
    EXPECTED_UNIVERSE_FILE_SHA
)


a3 = json.loads(
    A3_PATH.read_text(
        encoding="utf-8"
    )
)


a5_summary = json.loads(
    A5_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


a5_decision = json.loads(
    A5_DECISION_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    a3[
        "contract_sha256"
    ]
    ==
    EXPECTED_A3_CONTENT_SHA
)


assert (
    a5_summary[
        "summary_sha256"
    ]
    ==
    EXPECTED_A5_SUMMARY_CONTENT_SHA
)


assert (
    a5_decision[
        "decision_sha256"
    ]
    ==
    EXPECTED_A5_DECISION_CONTENT_SHA
)


assert (
    a5_decision[
        "decision"
    ]
    ==
    "PASS_RANKING_GATE"
)


assert (
    a5_decision[
        "ranking_gate"
    ][
        "both_pass"
    ]
    is True
)


assert (
    a5_decision[
        "Fold0_used"
    ]
    is False
)


EXPECTED_GRAPH_GATES = [
    "MACRO_MEAN_OFFICIAL_SCORE_DELTA_GT_0",
    "POOLED_DEVELOPMENT_OFFICIAL_SCORE_DELTA_GT_0",
    "POOLED_DEVELOPMENT_EDGE_JACCARD_DELTA_GT_0",
    "POOLED_DEVELOPMENT_NODE_RECALL_DELTA_GE_0",
    "POOLED_DEVELOPMENT_DIVISION_SCORE_DELTA_GE_0",
    "AT_LEAST_3_OF_4_FOLD_SCORE_DELTAS_GE_0",
]


assert (
    a3[
        "graph_promotion_gate"
    ][
        "required_conditions"
    ]
    ==
    EXPECTED_GRAPH_GATES
)


assert (
    a3[
        "graph_promotion_gate"
    ][
        "all_conditions_required"
    ]
    is True
)


assert (
    a3[
        "graph_promotion_gate"
    ][
        "threshold_retuning_after_graph_results"
    ]
    is False
)


assert (
    a3[
        "graph_promotion_gate"
    ][
        "attachment_retuning_after_graph_results"
    ]
    is False
)


print(
    "\n========== A3/A5 PARENT LOCK =========="
)

print(
    "A3:",
    "PASS"
)

print(
    "A5:",
    "PASS_RANKING_GATE"
)

print(
    "graph promotion gates:",
    6
)

print(
    "Fold0:",
    "NO"
)

print(
    "PARENT_LOCK: PASS"
)


# ============================================================
# 7. IMMUTABLE A6-R1 PRE-GT ACTION TABLE LOCK
#
# DO NOT regenerate.
# ============================================================

assert (
    sha256_file(
        ACTIONS_PATH
    )
    ==
    EXPECTED_ACTIONS_FILE_SHA
)


frozen_actions = pd.read_pickle(
    ACTIONS_PATH
)


assert (
    dataframe_content_sha256(
        frozen_actions
    )
    ==
    EXPECTED_ACTIONS_CONTENT_SHA
)


assert len(
    frozen_actions
) == EXPECTED_ACTIONS


required_action_columns = {
    "dataset",
    "fold",
    "dataset_action_rank",
    "candidate_id",
    "model_score",
    "threshold",
    "selected_scorer",
    "t",
    "z",
    "y",
    "x",
    "new_node_id",
    "incoming_endpoint_id",
    "incoming_family",
    "incoming_raw_proposal_id",
    "incoming_edge_dist_um",
    "outgoing_endpoint_id",
    "outgoing_family",
    "outgoing_raw_proposal_id",
    "outgoing_edge_dist_um",
    "edges_added",
    "edge_prob",
}


assert required_action_columns.issubset(
    frozen_actions.columns
)


assert frozen_actions[
    "candidate_id"
].is_unique


assert set(
    frozen_actions[
        "fold"
    ].astype(int).unique()
) == {
    1,
    2,
    3,
    4,
}


assert not (
    frozen_actions[
        "fold"
    ].astype(int)
    ==
    0
).any()


assert int(
    (
        frozen_actions[
            "edges_added"
        ]
        ==
        1
    ).sum()
) == EXPECTED_ONE_EDGE


assert int(
    (
        frozen_actions[
            "edges_added"
        ]
        ==
        2
    ).sum()
) == EXPECTED_TWO_EDGE


assert int(
    frozen_actions[
        "edges_added"
    ].sum()
) == EXPECTED_NEW_EDGES


assert np.all(
    frozen_actions[
        "edge_prob"
    ].to_numpy(
        dtype=np.float64
    )
    ==
    1.0
)


print(
    "\n========== FROZEN A6-R1 ACTION TABLE =========="
)

print(
    "actions:",
    len(
        frozen_actions
    )
)

print(
    "1-edge actions:",
    EXPECTED_ONE_EDGE
)

print(
    "2-edge actions:",
    EXPECTED_TWO_EDGE
)

print(
    "new edges:",
    EXPECTED_NEW_EDGES
)

print(
    "content SHA:",
    EXPECTED_ACTIONS_CONTENT_SHA
)

print(
    "file SHA:",
    EXPECTED_ACTIONS_FILE_SHA
)

print(
    "FROZEN_ACTION_TABLE_LOCK: PASS"
)


# ============================================================
# 8. IMMUTABLE PRE-GT CONTRACT LOCK
# ============================================================

pre_gt_contract = json.loads(
    PRE_GT_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    pre_gt_contract[
        "contract_sha256"
    ]
    ==
    EXPECTED_PRE_GT_CONTRACT_CONTENT_SHA
)


contract_payload = dict(
    pre_gt_contract
)


contract_payload.pop(
    "contract_sha256"
)


assert (
    json_content_sha256(
        contract_payload
    )
    ==
    EXPECTED_PRE_GT_CONTRACT_CONTENT_SHA
)


assert (
    pre_gt_contract[
        "status"
    ]
    ==
    "A6_ACTIONS_FROZEN_BEFORE_OFFICIAL_GRAPH_GT"
)


assert (
    pre_gt_contract[
        "protocol_at_freeze"
    ][
        "official_graph_GT_loaded"
    ]
    is False
)


assert (
    pre_gt_contract[
        "protocol_at_freeze"
    ][
        "official_metric_executed"
    ]
    is False
)


assert (
    pre_gt_contract[
        "protocol_at_freeze"
    ][
        "threshold_retuned"
    ]
    is False
)


assert (
    pre_gt_contract[
        "protocol_at_freeze"
    ][
        "attachment_retuned"
    ]
    is False
)


assert (
    pre_gt_contract[
        "protocol_at_freeze"
    ][
        "Fold0_used"
    ]
    is False
)


assert int(
    pre_gt_contract[
        "counts"
    ][
        "A5_selected_candidates"
    ]
) == EXPECTED_SELECTED


assert int(
    pre_gt_contract[
        "counts"
    ][
        "accepted_actions"
    ]
) == EXPECTED_ACTIONS


assert int(
    pre_gt_contract[
        "counts"
    ][
        "skipped_no_free_endpoint"
    ]
) == EXPECTED_SKIPPED


assert int(
    pre_gt_contract[
        "counts"
    ][
        "one_edge_actions"
    ]
) == EXPECTED_ONE_EDGE


assert int(
    pre_gt_contract[
        "counts"
    ][
        "two_edge_actions"
    ]
) == EXPECTED_TWO_EDGE


assert int(
    pre_gt_contract[
        "counts"
    ][
        "total_new_edges"
    ]
) == EXPECTED_NEW_EDGES


assert (
    pre_gt_contract[
        "artifacts"
    ][
        "actions_content_sha256"
    ]
    ==
    EXPECTED_ACTIONS_CONTENT_SHA
)


assert (
    pre_gt_contract[
        "artifacts"
    ][
        "actions_file_sha256"
    ]
    ==
    EXPECTED_ACTIONS_FILE_SHA
)


# ============================================================
# 9. ATTACHMENT AUDIT READBACK
# ============================================================

attachment_audit = pd.read_pickle(
    ACTION_AUDIT_PATH
)


assert len(
    attachment_audit
) == EXPECTED_SELECTED


assert (
    dataframe_content_sha256(
        attachment_audit
    )
    ==
    pre_gt_contract[
        "artifacts"
    ][
        "attachment_audit_content_sha256"
    ]
)


assert (
    sha256_file(
        ACTION_AUDIT_PATH
    )
    ==
    pre_gt_contract[
        "artifacts"
    ][
        "attachment_audit_file_sha256"
    ]
)


assert int(
    (
        attachment_audit[
            "status"
        ]
        ==
        "SKIP_NO_FREE_ENDPOINT"
    ).sum()
) == EXPECTED_SKIPPED


assert int(
    (
        attachment_audit[
            "status"
        ]
        ==
        "ACCEPT"
    ).sum()
) == EXPECTED_ACTIONS


print(
    "\n========== IMMUTABLE PRE-GT FREEZE =========="
)

print(
    "A5 selected:",
    EXPECTED_SELECTED
)

print(
    "accepted actions:",
    EXPECTED_ACTIONS
)

print(
    "skipped:",
    EXPECTED_SKIPPED
)

print(
    "pre-GT contract SHA:",
    EXPECTED_PRE_GT_CONTRACT_CONTENT_SHA
)

print(
    "GT loaded at freeze:",
    "NO"
)

print(
    "official metric executed at freeze:",
    "NO"
)

print(
    "PRE_GT_ACTION_FREEZE_READBACK: PASS"
)


# ============================================================
# 10. R2 REPAIR SCOPE LOCK
# ============================================================

print(
    "\n========== A6-R2 REPAIR SCOPE =========="
)

print(
    "A6-R1 failure:",
    'KeyError: base_metric["score"]'
)

print(
    "reason:",
    "evaluate_graph returns per-sample metrics"
)

print(
    "combined score authority:",
    "fv9.summarise(rows)"
)

print(
    "actions regenerated:",
    "NO"
)

print(
    "attachment recomputed:",
    "NO"
)

print(
    "model retrained:",
    "NO"
)

print(
    "threshold changed:",
    "NO"
)

print(
    "family mapping changed:",
    "NO"
)

print(
    "Fold0:",
    "NO"
)

print(
    "A6_R2_REPAIR_SCOPE_LOCK: PASS"
)


# ============================================================
# 11. OOF METADATA READBACK
#
# Only selected metadata is consumed.
# Oracle target is not consumed.
# ============================================================

oof = pd.read_pickle(
    A5_OOF_PATH
)


expected_oof_content_sha = (
    a5_summary[
        "content_sha256"
    ][
        "oof"
    ]
)


assert (
    dataframe_content_sha256(
        oof
    )
    ==
    expected_oof_content_sha
)


required_oof_columns = {
    "candidate_id",
    "dataset",
    "fold",
    "selected",
}


assert required_oof_columns.issubset(
    oof.columns
)


oof_runtime = oof[
    [
        "candidate_id",
        "dataset",
        "fold",
        "selected",
    ]
].copy()


selected_oof = (
    oof_runtime[
        oof_runtime[
            "selected"
        ].astype(bool)
    ]
    .copy()
)


assert len(
    selected_oof
) == EXPECTED_SELECTED


selected_count_by_dataset = (
    selected_oof
    .groupby(
        "dataset"
    )
    .size()
    .astype(int)
    .to_dict()
)


del oof
del oof_runtime
del selected_oof

gc.collect()


# ============================================================
# 12. BRANCH CORPUS
# ============================================================

universe = pd.read_pickle(
    UNIVERSE_PATH
)


assert len(
    universe
) == EXPECTED_UNIVERSE_ROWS


all_candidate_names = sorted(
    universe[
        "dataset"
    ].astype(str).unique()
)


assert len(
    all_candidate_names
) == EXPECTED_BRANCH_DATASETS


cv = pd.read_csv(
    CV_PATH
)


cv_branch = (
    cv[
        cv[
            "dataset"
        ].astype(str).isin(
            all_candidate_names
        )
    ]
    .copy()
)


assert len(
    cv_branch
) == EXPECTED_BRANCH_DATASETS


assert cv_branch[
    "dataset"
].is_unique


assert set(
    cv_branch[
        "fold"
    ].astype(int).unique()
) == {
    1,
    2,
    3,
    4,
}


assert not (
    cv_branch[
        "fold"
    ].astype(int)
    ==
    0
).any()


fold_map = dict(
    zip(
        cv_branch[
            "dataset"
        ].astype(str),

        cv_branch[
            "fold"
        ].astype(int),
    )
)


estimated_nodes = dict(
    zip(
        cv_branch[
            "dataset"
        ].astype(str),

        cv_branch[
            "estimated_nodes"
        ].astype(float),
    )
)


for dataset in all_candidate_names:

    assert dataset in fold_map

    assert dataset in estimated_nodes


actions_by_dataset = {
    dataset:
        sub.copy()

    for dataset, sub
    in frozen_actions.groupby(
        "dataset",
        sort=False,
    )
}


print(
    "\n========== A6-R2 OFFICIAL CORPUS =========="
)

print(
    "datasets:",
    len(
        all_candidate_names
    )
)

print(
    "fold counts:",
    (
        cv_branch[
            "fold"
        ]
        .astype(int)
        .value_counts()
        .sort_index()
        .to_dict()
    )
)

print(
    "Fold0:",
    "NO"
)

print(
    "CORPUS_LOCK: PASS"
)


# ============================================================
# 13. EXACT POST-G3H3 BUILDER REQUIRED
# ============================================================

if (
    "build_dev_post_g3h3_frozen"
    not in globals()
):

    raise RuntimeError(
        "\nExact build_dev_post_g3h3_frozen is missing from "
        "the current notebook kernel.\n\n"
        "Restore the successful A4/A6 environment.\n"
        "Do NOT reconstruct historical development graphs "
        "using the deployment threshold."
    )


# ============================================================
# 14. RUNTIME TRACKSDATA API LOCK
# ============================================================

sample_dataset = (
    all_candidate_names[
        0
    ]
)


(
    sample_graph,
    sample_scale,
    sample_stats,
) = build_dev_post_g3h3_frozen(
    sample_dataset
)


graph_class = type(
    sample_graph
)


assert (
    f"{graph_class.__module__}."
    f"{graph_class.__qualname__}"
    ==
    "tracksdata.graph._rustworkx_graph.IndexedRXGraph"
)


assert (
    source_sha(
        inspect.getsource(
            sample_graph.add_node
        )
    )
    ==
    EXPECTED_ADD_NODE_SHA
)


assert (
    source_sha(
        inspect.getsource(
            sample_graph.add_edge
        )
    )
    ==
    EXPECTED_ADD_EDGE_SHA
)


assert (
    source_sha(
        inspect.getsource(
            sample_graph.copy
        )
    )
    ==
    EXPECTED_COPY_SHA
)


assert (
    sha256_file(
        Path(
            inspect.getfile(
                graph_class
            )
        )
    )
    ==
    EXPECTED_GRAPH_CLASS_FILE_SHA
)


del sample_graph

gc.collect()


print(
    "\nRUNTIME_TRACKSDATA_API_LOCK: PASS"
)


# ============================================================
# 15. REPLAY EXACT FROZEN ACTIONS
# ============================================================

def replay_frozen_actions(
    graph,
    dataset,
    frozen_action_table,
):

    (
        nodes,
        outdeg,
        indeg,
        edge_set,
    ) = graph_state(
        graph
    )


    rows = (
        frozen_action_table[
            frozen_action_table[
                "dataset"
            ].eq(
                dataset
            )
        ]
        .sort_values(
            "dataset_action_rank",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


    initial_node_count = len(
        nodes
    )


    initial_edge_count = len(
        edge_set
    )


    if len(
        rows
    ) == 0:

        return graph


    # --------------------------------------------------------
    # Replay frozen actions exactly.
    # --------------------------------------------------------

    for action in rows.itertuples(
        index=False
    ):

        expected_new_node_id = int(
            action.new_node_id
        )


        assert expected_new_node_id not in nodes.index


        new_node_id = graph.add_node(
            {
                "t":
                    int(
                        action.t
                    ),

                "z":
                    float(
                        action.z
                    ),

                "y":
                    float(
                        action.y
                    ),

                "x":
                    float(
                        action.x
                    ),
            },
            validate_keys=True,
            index=expected_new_node_id,
        )


        assert (
            int(
                new_node_id
            )
            ==
            expected_new_node_id
        )


        incoming_present = pd.notna(
            action.incoming_endpoint_id
        )


        outgoing_present = pd.notna(
            action.outgoing_endpoint_id
        )


        assert (
            int(
                incoming_present
            )
            +
            int(
                outgoing_present
            )
            ==
            int(
                action.edges_added
            )
        )


        # ====================================================
        # Incoming endpoint -> synthetic
        # ====================================================

        if incoming_present:

            source_id = int(
                action.incoming_endpoint_id
            )


            assert outdeg.get(
                source_id,
                0,
            ) == 0


            assert (
                source_id,
                expected_new_node_id,
            ) not in edge_set


            graph.add_edge(
                source_id,
                expected_new_node_id,
                {
                    "edge_dist":
                        float(
                            action.incoming_edge_dist_um
                        ),

                    "edge_prob":
                        1.0,
                },
                validate_keys=True,
            )


            outdeg[
                source_id
            ] = 1


            indeg[
                expected_new_node_id
            ] = (
                indeg.get(
                    expected_new_node_id,
                    0,
                )
                +
                1
            )


            edge_set.add(
                (
                    source_id,
                    expected_new_node_id,
                )
            )


        # ====================================================
        # Synthetic -> outgoing endpoint
        # ====================================================

        if outgoing_present:

            target_id = int(
                action.outgoing_endpoint_id
            )


            assert indeg.get(
                target_id,
                0,
            ) == 0


            assert (
                expected_new_node_id,
                target_id,
            ) not in edge_set


            graph.add_edge(
                expected_new_node_id,
                target_id,
                {
                    "edge_dist":
                        float(
                            action.outgoing_edge_dist_um
                        ),

                    "edge_prob":
                        1.0,
                },
                validate_keys=True,
            )


            outdeg[
                expected_new_node_id
            ] = (
                outdeg.get(
                    expected_new_node_id,
                    0,
                )
                +
                1
            )


            indeg[
                target_id
            ] = 1


            edge_set.add(
                (
                    expected_new_node_id,
                    target_id,
                )
            )


        assert indeg.get(
            expected_new_node_id,
            0,
        ) <= 1


        assert outdeg.get(
            expected_new_node_id,
            0,
        ) <= 1


    # --------------------------------------------------------
    # Exact count check
    # --------------------------------------------------------

    final_nodes = (
        graph
        .node_attrs(
            unpack=True
        )
        .to_pandas()
    )


    final_edges = (
        graph
        .edge_attrs(
            unpack=True
        )
        .to_pandas()
    )


    assert (
        len(
            final_nodes
        )
        ==
        initial_node_count
        +
        len(
            rows
        )
    )


    assert (
        len(
            final_edges
        )
        ==
        initial_edge_count
        +
        int(
            rows[
                "edges_added"
            ].sum()
        )
    )


    return graph


# ============================================================
# 16. OFFICIAL PER-SAMPLE EVALUATION
#
# R2 FIX:
#
# NEVER access:
#
#     metric["score"]
#
# evaluate_graph returns per-sample metrics.
#
# Combined score comes from:
#
#     fv9.summarise(...)
# ============================================================

base_records = []

policy_records = []


base_records_by_fold = {
    fold_id:
        []

    for fold_id
    in FOLDS
}


policy_records_by_fold = {
    fold_id:
        []

    for fold_id
    in FOLDS
}


metric_rows = []


metric_key_schema = None


for dataset_index, dataset in enumerate(
    all_candidate_names,
    start=1,
):

    fold_id = int(
        fold_map[
            dataset
        ]
    )


    # --------------------------------------------------------
    # Exact post-G3H3 baseline.
    # --------------------------------------------------------

    (
        base_graph,
        scale,
        baseline_replay_stats,
    ) = build_dev_post_g3h3_frozen(
        dataset
    )


    policy_graph = (
        base_graph.copy()
    )


    # --------------------------------------------------------
    # Replay ONLY immutable pre-GT actions.
    # --------------------------------------------------------

    policy_graph = replay_frozen_actions(
        policy_graph,
        dataset,
        frozen_actions,
    )


    # --------------------------------------------------------
    # Official GT load.
    # --------------------------------------------------------

    gt_graph = fv9.load_graph(
        TRAIN_ROOT
        /
        f"{dataset}.geff"
    )


    n_total = float(
        estimated_nodes[
            dataset
        ]
    )


    # --------------------------------------------------------
    # Per-dataset official evaluation.
    # --------------------------------------------------------

    base_metric = fv9.evaluate_graph(
        base_graph,
        gt_graph,
        scale,
        n_total,
        fv9.FROZEN_V9_CONFIG,
    )


    policy_metric = fv9.evaluate_graph(
        policy_graph,
        gt_graph,
        scale,
        n_total,
        fv9.FROZEN_V9_CONFIG,
    )


    # --------------------------------------------------------
    # Exact per-sample schema.
    # --------------------------------------------------------

    current_base_keys = tuple(
        sorted(
            base_metric.keys()
        )
    )


    current_policy_keys = tuple(
        sorted(
            policy_metric.keys()
        )
    )


    assert (
        current_base_keys
        ==
        current_policy_keys
    )


    if metric_key_schema is None:

        metric_key_schema = (
            current_base_keys
        )


        print(
            "\n========== PER-SAMPLE METRIC SCHEMA =========="
        )


        print(
            "keys:",
            metric_key_schema
        )


        print(
            '"score" in per-sample metrics:',
            (
                "score"
                in
                metric_key_schema
            )
        )


        print(
            "combined score authority:",
            "fv9.summarise"
        )


        print(
            "PER_SAMPLE_METRIC_SCHEMA_LOCK: PASS"
        )


    else:

        assert (
            current_base_keys
            ==
            metric_key_schema
        )


    # --------------------------------------------------------
    # Per-sample records.
    # --------------------------------------------------------

    base_records.append(
        base_metric
    )


    policy_records.append(
        policy_metric
    )


    base_records_by_fold[
        fold_id
    ].append(
        base_metric
    )


    policy_records_by_fold[
        fold_id
    ].append(
        policy_metric
    )


    ds_actions = actions_by_dataset.get(
        dataset,
        frozen_actions.iloc[
            0:0
        ]
    )


    metric_row = {
        "dataset":
            dataset,

        "fold":
            fold_id,

        "selected_candidates":
            int(
                selected_count_by_dataset.get(
                    dataset,
                    0,
                )
            ),

        "accepted_actions":
            int(
                len(
                    ds_actions
                )
            ),

        "new_nodes":
            int(
                len(
                    ds_actions
                )
            ),

        "new_edges":
            (
                int(
                    ds_actions[
                        "edges_added"
                    ].sum()
                )
                if len(
                    ds_actions
                )
                else
                0
            ),
    }


    for key, value in base_metric.items():

        metric_row[
            f"base_{key}"
        ] = value


    for key, value in policy_metric.items():

        metric_row[
            f"policy_{key}"
        ] = value


    metric_rows.append(
        metric_row
    )


    # --------------------------------------------------------
    # R2 progress print.
    #
    # NO score access here.
    # --------------------------------------------------------

    if (
        dataset_index == 1
        or
        dataset_index % 10 == 0
        or
        dataset_index
        ==
        len(
            all_candidate_names
        )
    ):

        print(
            "OFFICIAL A6-R2 evaluation:",
            f"{dataset_index}/{len(all_candidate_names)}",
            "|",
            dataset,
            "| fold:",
            fold_id,
            "| selected:",
            int(
                selected_count_by_dataset.get(
                    dataset,
                    0,
                )
            ),
            "| actions:",
            len(
                ds_actions
            ),
        )


    del base_graph
    del policy_graph
    del gt_graph

    gc.collect()


assert len(
    base_records
) == EXPECTED_BRANCH_DATASETS


assert len(
    policy_records
) == EXPECTED_BRANCH_DATASETS


assert len(
    metric_rows
) == EXPECTED_BRANCH_DATASETS


print(
    "\nOFFICIAL_PER_SAMPLE_EVALUATION_COMPLETE: PASS"
)


# ============================================================
# 17. OFFICIAL AGGREGATION
#
# score first appears HERE.
# ============================================================

pooled_base = fv9.summarise(
    base_records
)


pooled_policy = fv9.summarise(
    policy_records
)


assert "score" in pooled_base

assert "score" in pooled_policy


print(
    "\n========== OFFICIAL AGGREGATION =========="
)


print(
    "pooled baseline score:",
    pooled_base[
        "score"
    ]
)


print(
    "pooled policy score:",
    pooled_policy[
        "score"
    ]
)


print(
    "score authority:",
    "fv9.summarise"
)


print(
    "OFFICIAL_AGGREGATION: PASS"
)


# ============================================================
# 18. BASELINE FIDELITY
# ============================================================

base_edge_tp = int(
    sum(
        int(
            row[
                "edge_tp"
            ]
        )

        for row
        in base_records
    )
)


base_edge_fp = int(
    sum(
        int(
            row[
                "edge_fp"
            ]
        )

        for row
        in base_records
    )
)


base_edge_fn = int(
    sum(
        int(
            row[
                "edge_fn"
            ]
        )

        for row
        in base_records
    )
)


assert (
    base_edge_tp
    ==
    EXPECTED_BASE_EDGE_TP
), (
    base_edge_tp,
    EXPECTED_BASE_EDGE_TP,
)


assert (
    base_edge_fp
    ==
    EXPECTED_BASE_EDGE_FP
), (
    base_edge_fp,
    EXPECTED_BASE_EDGE_FP,
)


assert (
    base_edge_fn
    ==
    EXPECTED_BASE_EDGE_FN
), (
    base_edge_fn,
    EXPECTED_BASE_EDGE_FN,
)


assert abs(
    float(
        pooled_base[
            "score"
        ]
    )
    -
    EXPECTED_BASE_SCORE
) <= 1e-12, (
    pooled_base[
        "score"
    ],
    EXPECTED_BASE_SCORE,
)


print(
    "\n========== BASELINE FIDELITY =========="
)


print(
    "edge TP/FP/FN:",
    (
        base_edge_tp,
        base_edge_fp,
        base_edge_fn,
    )
)


print(
    "expected TP/FP/FN:",
    (
        EXPECTED_BASE_EDGE_TP,
        EXPECTED_BASE_EDGE_FP,
        EXPECTED_BASE_EDGE_FN,
    )
)


print(
    "pooled score:",
    pooled_base[
        "score"
    ]
)


print(
    "expected score:",
    EXPECTED_BASE_SCORE
)


print(
    "POST_G3H3_BASELINE_OFFICIAL_FIDELITY: PASS"
)


# ============================================================
# 19. FOLD-LEVEL OFFICIAL SUMMARIES
# ============================================================

fold_rows = []


for fold_id in FOLDS:

    assert len(
        base_records_by_fold[
            fold_id
        ]
    ) > 0


    assert len(
        policy_records_by_fold[
            fold_id
        ]
    ) == len(
        base_records_by_fold[
            fold_id
        ]
    )


    base_fold_summary = fv9.summarise(
        base_records_by_fold[
            fold_id
        ]
    )


    policy_fold_summary = fv9.summarise(
        policy_records_by_fold[
            fold_id
        ]
    )


    assert "score" in base_fold_summary

    assert "score" in policy_fold_summary


    fold_rows.append(
        fold_summary_row(
            fold_id,
            base_fold_summary,
            policy_fold_summary,
        )
    )


fold_summary = pd.DataFrame(
    fold_rows
)


assert len(
    fold_summary
) == 4


print(
    "\n========== FOLD SUMMARIES =========="
)


print(
    fold_summary.to_string(
        index=False
    )
)


# ============================================================
# 20. SIX FROZEN GRAPH PROMOTION GATES
# ============================================================

macro_base_score = float(
    fold_summary[
        "base_score"
    ].mean()
)


macro_policy_score = float(
    fold_summary[
        "policy_score"
    ].mean()
)


macro_score_delta = float(
    macro_policy_score
    -
    macro_base_score
)


pooled_score_delta = float(
    pooled_policy[
        "score"
    ]
    -
    pooled_base[
        "score"
    ]
)


pooled_edge_delta = float(
    pooled_policy[
        "edge_jaccard"
    ]
    -
    pooled_base[
        "edge_jaccard"
    ]
)


pooled_node_recall_delta = float(
    pooled_policy[
        "node_recall"
    ]
    -
    pooled_base[
        "node_recall"
    ]
)


assert np.isfinite(
    float(
        pooled_base[
            "division_jaccard"
        ]
    )
)


assert np.isfinite(
    float(
        pooled_policy[
            "division_jaccard"
        ]
    )
)


pooled_division_delta = float(
    pooled_policy[
        "division_jaccard"
    ]
    -
    pooled_base[
        "division_jaccard"
    ]
)


nonnegative_fold_count = int(
    (
        fold_summary[
            "delta_score"
        ]
        >=
        0.0
    ).sum()
)


GATE_1 = bool(
    macro_score_delta
    >
    0.0
)


GATE_2 = bool(
    pooled_score_delta
    >
    0.0
)


GATE_3 = bool(
    pooled_edge_delta
    >
    0.0
)


GATE_4 = bool(
    pooled_node_recall_delta
    >=
    0.0
)


GATE_5 = bool(
    pooled_division_delta
    >=
    0.0
)


GATE_6 = bool(
    nonnegative_fold_count
    >=
    3
)


ALL_GRAPH_GATES_PASS = bool(
    GATE_1
    and
    GATE_2
    and
    GATE_3
    and
    GATE_4
    and
    GATE_5
    and
    GATE_6
)


DECISION = (
    "PASS_GRAPH_PROMOTION_GATE"
    if ALL_GRAPH_GATES_PASS
    else
    "CLOSE_BRANCH_NO_POLICY_PROMOTION"
)


print(
    "\n========== 12.13A6-R2 GRAPH PROMOTION GATES =========="
)


print(
    "G1 macro mean official score delta > 0:",
    macro_score_delta,
    "->",
    GATE_1
)


print(
    "G2 pooled official score delta > 0:",
    pooled_score_delta,
    "->",
    GATE_2
)


print(
    "G3 pooled edge Jaccard delta > 0:",
    pooled_edge_delta,
    "->",
    GATE_3
)


print(
    "G4 pooled node recall delta >= 0:",
    pooled_node_recall_delta,
    "->",
    GATE_4
)


print(
    "G5 pooled division Jaccard delta >= 0:",
    pooled_division_delta,
    "->",
    GATE_5
)


print(
    "G6 >=3/4 nonnegative fold score deltas:",
    nonnegative_fold_count,
    "/ 4 ->",
    GATE_6
)


print(
    "\nALL SIX PASS:",
    ALL_GRAPH_GATES_PASS
)


print(
    "DECISION:",
    DECISION
)


# ============================================================
# 21. DATASET METRICS ARTIFACT
# ============================================================

metrics = pd.DataFrame(
    metric_rows
)


assert len(
    metrics
) == EXPECTED_BRANCH_DATASETS


METRICS_CONTENT_SHA = (
    dataframe_content_sha256(
        metrics
    )
)


FOLD_SUMMARY_CONTENT_SHA = (
    dataframe_content_sha256(
        fold_summary
    )
)


# ============================================================
# 22. FORMAL A6 SUMMARY
# ============================================================

SUMMARY = {
    "stage":
        "12.13A6",

    "implementation_revision":
        "R2_POST_GT_PER_SAMPLE_SCORE_REPAIR",

    "status":
        (
            "FORMAL_HELD_OUT_GRAPH_ATTACHMENT_AND_"
            "OFFICIAL_METRIC_EVALUATION_COMPLETE"
        ),

    "branch_id":
        (
            "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_"
            "NODE_RECOVERY_V1"
        ),

    "successor_provenance": {
        "failed_predecessor":
            "12.13A6-R1",

        "predecessor_failure":
            (
                "KEYERROR_SCORE_IN_PROGRESS_PRINT_"
                "AFTER_PER_SAMPLE_EVALUATE_GRAPH"
            ),

        "predecessor_pre_GT_action_freeze_completed":
            True,

        "predecessor_actions_content_sha256":
            EXPECTED_ACTIONS_CONTENT_SHA,

        "predecessor_actions_file_sha256":
            EXPECTED_ACTIONS_FILE_SHA,

        "predecessor_pre_GT_contract_sha256":
            EXPECTED_PRE_GT_CONTRACT_CONTENT_SHA,

        "actions_regenerated":
            False,

        "attachment_recomputed":
            False,

        "family_mapping_changed":
            False,

        "model_changed":
            False,

        "threshold_changed":
            False,
    },

    "parents": {
        "A3_file_sha256":
            EXPECTED_A3_FILE_SHA,

        "A3_content_sha256":
            EXPECTED_A3_CONTENT_SHA,

        "A5_summary_file_sha256":
            EXPECTED_A5_SUMMARY_FILE_SHA,

        "A5_decision_file_sha256":
            EXPECTED_A5_DECISION_FILE_SHA,

        "A5_oof_content_sha256":
            expected_oof_content_sha,

        "A1_universe_file_sha256":
            EXPECTED_UNIVERSE_FILE_SHA,

        "A6_preflight_source_manifest_sha256":
            A6_PREFLIGHT_SOURCE_MANIFEST_SHA,

        "A6_pre_GT_contract_sha256":
            EXPECTED_PRE_GT_CONTRACT_CONTENT_SHA,

        "A6_pre_GT_actions_content_sha256":
            EXPECTED_ACTIONS_CONTENT_SHA,
    },

    "source_lock": {
        "evaluate_graph_sha256":
            EXPECTED_EVALUATE_GRAPH_SHA,

        "summarise_sha256":
            EXPECTED_SUMMARISE_SHA,

        "official_evaluate_sha256":
            EXPECTED_OFFICIAL_EVALUATE_SHA,

        "node_recall_sha256":
            EXPECTED_NODE_RECALL_SHA,

        "per_sample_metrics_sha256":
            EXPECTED_PER_SAMPLE_SHA,

        "metric_max_distance_um":
            7.0,

        "tracksdata_add_node_sha256":
            EXPECTED_ADD_NODE_SHA,

        "tracksdata_add_edge_sha256":
            EXPECTED_ADD_EDGE_SHA,

        "tracksdata_copy_sha256":
            EXPECTED_COPY_SHA,
    },

    "per_sample_metric_contract": {
        "keys":
            list(
                metric_key_schema
            ),

        "per_sample_score_accessed":
            False,

        "combined_score_authority":
            "biohub_cell_tracking.frozen_v9.summarise",
    },

    "corpus": {
        "datasets":
            EXPECTED_BRANCH_DATASETS,

        "folds":
            FOLDS,

        "Fold0_used":
            False,
    },

    "actions": {
        "A5_selected_candidates":
            EXPECTED_SELECTED,

        "accepted_synthetic_nodes":
            EXPECTED_ACTIONS,

        "skipped_no_free_endpoint":
            EXPECTED_SKIPPED,

        "one_edge_nodes":
            EXPECTED_ONE_EDGE,

        "two_edge_nodes":
            EXPECTED_TWO_EDGE,

        "total_new_edges":
            EXPECTED_NEW_EDGES,

        "actions_content_sha256":
            EXPECTED_ACTIONS_CONTENT_SHA,

        "actions_file_sha256":
            EXPECTED_ACTIONS_FILE_SHA,
    },

    "baseline_fidelity": {
        "edge_tp":
            base_edge_tp,

        "edge_fp":
            base_edge_fp,

        "edge_fn":
            base_edge_fn,

        "expected_edge_tp":
            EXPECTED_BASE_EDGE_TP,

        "expected_edge_fp":
            EXPECTED_BASE_EDGE_FP,

        "expected_edge_fn":
            EXPECTED_BASE_EDGE_FN,

        "score":
            float(
                pooled_base[
                    "score"
                ]
            ),

        "expected_score":
            EXPECTED_BASE_SCORE,

        "pass":
            True,
    },

    "pooled": {
        "baseline":
            jsonable(
                pooled_base
            ),

        "policy":
            jsonable(
                pooled_policy
            ),

        "delta_score":
            pooled_score_delta,

        "delta_edge_jaccard":
            pooled_edge_delta,

        "delta_node_recall":
            pooled_node_recall_delta,

        "delta_division_jaccard":
            pooled_division_delta,
    },

    "macro": {
        "baseline_mean_fold_score":
            macro_base_score,

        "policy_mean_fold_score":
            macro_policy_score,

        "delta":
            macro_score_delta,
    },

    "graph_promotion_gates": {
        "G1_macro_mean_score_delta_gt_0":
            GATE_1,

        "G2_pooled_score_delta_gt_0":
            GATE_2,

        "G3_pooled_edge_jaccard_delta_gt_0":
            GATE_3,

        "G4_pooled_node_recall_delta_ge_0":
            GATE_4,

        "G5_pooled_division_jaccard_delta_ge_0":
            GATE_5,

        "G6_at_least_3_of_4_fold_score_deltas_ge_0":
            GATE_6,

        "nonnegative_fold_count":
            nonnegative_fold_count,

        "all_required":
            True,

        "all_pass":
            ALL_GRAPH_GATES_PASS,
    },

    "protocol": {
        "pre_GT_actions_reused_exactly":
            True,

        "actions_regenerated_after_GT":
            False,

        "attachment_recomputed_after_GT":
            False,

        "family_mapping_recomputed_after_GT":
            False,

        "model_retrained":
            False,

        "model_reselected":
            False,

        "threshold_reselected":
            False,

        "threshold_retuned_after_graph_result":
            False,

        "attachment_retuned_after_graph_result":
            False,

        "persistent_GEFF_modified":
            False,

        "Fold0_used":
            False,
    },

    "decision":
        DECISION,

    "next":
        (
            "12.13A7_FINAL_DEVELOPMENT_POLICY_FREEZE"
            if ALL_GRAPH_GATES_PASS
            else
            "BRANCH_CLOSED"
        ),

    "content_sha256": {
        "dataset_metrics":
            METRICS_CONTENT_SHA,

        "fold_summary":
            FOLD_SUMMARY_CONTENT_SHA,
    },
}


SUMMARY_CONTENT_SHA = (
    json_content_sha256(
        SUMMARY
    )
)


SUMMARY[
    "summary_sha256"
] = SUMMARY_CONTENT_SHA


# ============================================================
# 23. FORMAL DECISION RECORD
# ============================================================

DECISION_RECORD = {
    "stage":
        "12.13A6",

    "implementation_revision":
        "R2_POST_GT_PER_SAMPLE_SCORE_REPAIR",

    "branch_id":
        (
            "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_"
            "NODE_RECOVERY_V1"
        ),

    "pre_GT_action_freeze_preserved":
        True,

    "actions_content_sha256":
        EXPECTED_ACTIONS_CONTENT_SHA,

    "actions_file_sha256":
        EXPECTED_ACTIONS_FILE_SHA,

    "pre_GT_contract_sha256":
        EXPECTED_PRE_GT_CONTRACT_CONTENT_SHA,

    "baseline_fidelity_pass":
        True,

    "graph_promotion_gates": {
        "macro_mean_official_score_delta_gt_0":
            GATE_1,

        "pooled_development_official_score_delta_gt_0":
            GATE_2,

        "pooled_development_edge_jaccard_delta_gt_0":
            GATE_3,

        "pooled_development_node_recall_delta_ge_0":
            GATE_4,

        "pooled_development_division_score_delta_ge_0":
            GATE_5,

        "at_least_3_of_4_fold_score_deltas_ge_0":
            GATE_6,

        "all_pass":
            ALL_GRAPH_GATES_PASS,
    },

    "decision":
        DECISION,

    "policy_promoted":
        ALL_GRAPH_GATES_PASS,

    "persistent_graph_modified":
        False,

    "model_retuned":
        False,

    "threshold_retuned":
        False,

    "attachment_retuned":
        False,

    "Fold0_used":
        False,
}


DECISION_CONTENT_SHA = (
    json_content_sha256(
        DECISION_RECORD
    )
)


DECISION_RECORD[
    "decision_sha256"
] = DECISION_CONTENT_SHA


# ============================================================
# 24. TEMP WRITE + EXACT READBACK
# ============================================================

TMP_METRICS = METRICS_PATH.with_name(
    METRICS_PATH.name
    +
    ".tmp_a6_r2"
)


TMP_FOLD = FOLD_SUMMARY_PATH.with_name(
    FOLD_SUMMARY_PATH.name
    +
    ".tmp_a6_r2"
)


TMP_SUMMARY = SUMMARY_PATH.with_name(
    SUMMARY_PATH.name
    +
    ".tmp_a6_r2"
)


TMP_DECISION = DECISION_PATH.with_name(
    DECISION_PATH.name
    +
    ".tmp_a6_r2"
)


for path in [
    TMP_METRICS,
    TMP_FOLD,
    TMP_SUMMARY,
    TMP_DECISION,
]:

    if path.exists():

        raise RuntimeError(
            "\nStale temporary A6-R2 artifact exists:\n"
            f"{path}\n\n"
            "Inspect/remove only the stale .tmp file before rerun."
        )


metrics.to_pickle(
    TMP_METRICS
)


fold_summary.to_pickle(
    TMP_FOLD
)


TMP_SUMMARY.write_text(
    json.dumps(
        SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)


TMP_DECISION.write_text(
    json.dumps(
        DECISION_RECORD,
        indent=2,
    ),
    encoding="utf-8",
)


metrics_check = pd.read_pickle(
    TMP_METRICS
)


fold_check = pd.read_pickle(
    TMP_FOLD
)


summary_check = json.loads(
    TMP_SUMMARY.read_text(
        encoding="utf-8"
    )
)


decision_check = json.loads(
    TMP_DECISION.read_text(
        encoding="utf-8"
    )
)


pd.testing.assert_frame_equal(
    metrics_check,
    metrics,
    check_exact=True,
)


pd.testing.assert_frame_equal(
    fold_check,
    fold_summary,
    check_exact=True,
)


assert (
    dataframe_content_sha256(
        metrics_check
    )
    ==
    METRICS_CONTENT_SHA
)


assert (
    dataframe_content_sha256(
        fold_check
    )
    ==
    FOLD_SUMMARY_CONTENT_SHA
)


assert (
    summary_check[
        "summary_sha256"
    ]
    ==
    SUMMARY_CONTENT_SHA
)


assert (
    decision_check[
        "decision_sha256"
    ]
    ==
    DECISION_CONTENT_SHA
)


METRICS_FILE_SHA = (
    sha256_file(
        TMP_METRICS
    )
)


FOLD_SUMMARY_FILE_SHA = (
    sha256_file(
        TMP_FOLD
    )
)


SUMMARY_FILE_SHA = (
    sha256_file(
        TMP_SUMMARY
    )
)


DECISION_FILE_SHA = (
    sha256_file(
        TMP_DECISION
    )
)


TMP_METRICS.replace(
    METRICS_PATH
)


TMP_FOLD.replace(
    FOLD_SUMMARY_PATH
)


TMP_SUMMARY.replace(
    SUMMARY_PATH
)


TMP_DECISION.replace(
    DECISION_PATH
)


# ============================================================
# 25. FORMAL BRANCH CLOSURE IF ANY GATE FAILS
# ============================================================

CLOSURE_CONTENT_SHA = None

CLOSURE_FILE_SHA = None


if not ALL_GRAPH_GATES_PASS:

    CLOSURE = {
        "stage":
            "12.13A6",

        "implementation_revision":
            "R2_POST_GT_PER_SAMPLE_SCORE_REPAIR",

        "branch_id":
            (
                "POST_G3H3_TRACK_GEOMETRY_SYNTHETIC_"
                "NODE_RECOVERY_V1"
            ),

        "decision":
            "CLOSE_BRANCH_NO_POLICY_PROMOTION",

        "reason":
            (
                "A3_FROZEN_GRAPH_PROMOTION_GATE_"
                "NOT_ALL_SATISFIED"
            ),

        "pre_GT_actions_content_sha256":
            EXPECTED_ACTIONS_CONTENT_SHA,

        "pre_GT_contract_sha256":
            EXPECTED_PRE_GT_CONTRACT_CONTENT_SHA,

        "graph_promotion_gates":
            DECISION_RECORD[
                "graph_promotion_gates"
            ],

        "persistent_graph_modified":
            False,

        "Fold0_used":
            False,

        "post_result_retuning_permitted":
            False,
    }


    CLOSURE_CONTENT_SHA = (
        json_content_sha256(
            CLOSURE
        )
    )


    CLOSURE[
        "closure_sha256"
    ] = CLOSURE_CONTENT_SHA


    CLOSURE_PATH.write_text(
        json.dumps(
            CLOSURE,
            indent=2,
        ),
        encoding="utf-8",
    )


    CLOSURE_FILE_SHA = (
        sha256_file(
            CLOSURE_PATH
        )
    )


# ============================================================
# 26. FINAL IMMUTABILITY / FILE LOCK
# ============================================================

# ------------------------------------------------------------
# A6-R1 PRE-GT artifacts must remain unchanged.
# ------------------------------------------------------------

assert (
    sha256_file(
        ACTIONS_PATH
    )
    ==
    EXPECTED_ACTIONS_FILE_SHA
)


assert (
    dataframe_content_sha256(
        pd.read_pickle(
            ACTIONS_PATH
        )
    )
    ==
    EXPECTED_ACTIONS_CONTENT_SHA
)


pre_gt_final = json.loads(
    PRE_GT_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    pre_gt_final[
        "contract_sha256"
    ]
    ==
    EXPECTED_PRE_GT_CONTRACT_CONTENT_SHA
)


# ------------------------------------------------------------
# New post-GT artifacts
# ------------------------------------------------------------

assert (
    sha256_file(
        METRICS_PATH
    )
    ==
    METRICS_FILE_SHA
)


assert (
    sha256_file(
        FOLD_SUMMARY_PATH
    )
    ==
    FOLD_SUMMARY_FILE_SHA
)


assert (
    sha256_file(
        SUMMARY_PATH
    )
    ==
    SUMMARY_FILE_SHA
)


assert (
    sha256_file(
        DECISION_PATH
    )
    ==
    DECISION_FILE_SHA
)


if not ALL_GRAPH_GATES_PASS:

    assert CLOSURE_PATH.exists()

    assert (
        sha256_file(
            CLOSURE_PATH
        )
        ==
        CLOSURE_FILE_SHA
    )


# ============================================================
# 27. FINAL
# ============================================================

print(
    "\n============================================================"
)


print(
    "12.13A6-R2 FINAL"
)


print(
    "============================================================"
)


print(
    "\nFORMAL_POST_GT_OFFICIAL_METRIC_EVALUATION:",
    "PASS"
)


print(
    "\n--- SUCCESSOR PROVENANCE ---"
)


print(
    "A6-R1 pre-GT action freeze:",
    "PRESERVED"
)


print(
    "actions regenerated:",
    "NO"
)


print(
    "attachment recomputed:",
    "NO"
)


print(
    "model/threshold changed:",
    "NO"
)


print(
    "actions content SHA:",
    EXPECTED_ACTIONS_CONTENT_SHA
)


print(
    "actions file SHA:",
    EXPECTED_ACTIONS_FILE_SHA
)


print(
    "pre-GT contract SHA:",
    EXPECTED_PRE_GT_CONTRACT_CONTENT_SHA
)


print(
    "\n--- PER-SAMPLE METRIC CONTRACT ---"
)


print(
    "keys:",
    metric_key_schema
)


print(
    '"score" accessed from per-sample metric:',
    "NO"
)


print(
    "combined score authority:",
    "fv9.summarise"
)


print(
    "\n--- BASELINE FIDELITY ---"
)


print(
    "edge TP/FP/FN:",
    (
        base_edge_tp,
        base_edge_fp,
        base_edge_fn,
    )
)


print(
    "score:",
    pooled_base[
        "score"
    ]
)


print(
    "fidelity:",
    "PASS"
)


print(
    "\n--- FOLD RESULTS ---"
)


print(
    fold_summary.to_string(
        index=False
    )
)


print(
    "\n--- POOLED BASELINE ---"
)


print(
    json.dumps(
        jsonable(
            pooled_base
        ),
        indent=2,
    )
)


print(
    "\n--- POOLED POLICY ---"
)


print(
    json.dumps(
        jsonable(
            pooled_policy
        ),
        indent=2,
    )
)


print(
    "\n--- DELTAS ---"
)


print(
    "macro mean fold score:",
    macro_score_delta
)


print(
    "pooled official score:",
    pooled_score_delta
)


print(
    "pooled edge Jaccard:",
    pooled_edge_delta
)


print(
    "pooled node recall:",
    pooled_node_recall_delta
)


print(
    "pooled division Jaccard:",
    pooled_division_delta
)


print(
    "nonnegative folds:",
    nonnegative_fold_count,
    "/ 4",
)


print(
    "\n--- SIX GATES ---"
)


print(
    "G1 macro score > 0:",
    GATE_1
)


print(
    "G2 pooled score > 0:",
    GATE_2
)


print(
    "G3 pooled edge J > 0:",
    GATE_3
)


print(
    "G4 node recall >= 0:",
    GATE_4
)


print(
    "G5 division J >= 0:",
    GATE_5
)


print(
    "G6 >=3/4 fold score >=0:",
    GATE_6
)


print(
    "ALL SIX PASS:",
    ALL_GRAPH_GATES_PASS
)


print(
    "\n--- DECISION ---"
)


print(
    DECISION
)


print(
    "\n--- PROTOCOL ---"
)


print(
    "pre-GT actions reused exactly:",
    "YES"
)


print(
    "actions regenerated after GT:",
    "NO"
)


print(
    "model retrained:",
    "NO"
)


print(
    "threshold retuned:",
    "NO"
)


print(
    "attachment retuned:",
    "NO"
)


print(
    "persistent GEFF modified:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "\n--- CONTENT SHAS ---"
)


print(
    "dataset metrics:",
    METRICS_CONTENT_SHA
)


print(
    "fold summary:",
    FOLD_SUMMARY_CONTENT_SHA
)


print(
    "summary:",
    SUMMARY_CONTENT_SHA
)


print(
    "decision:",
    DECISION_CONTENT_SHA
)


print(
    "\n--- FILE SHAS ---"
)


print(
    "dataset metrics:",
    METRICS_FILE_SHA
)


print(
    "fold summary:",
    FOLD_SUMMARY_FILE_SHA
)


print(
    "summary:",
    SUMMARY_FILE_SHA
)


print(
    "decision:",
    DECISION_FILE_SHA
)


if CLOSURE_CONTENT_SHA is not None:

    print(
        "closure content:",
        CLOSURE_CONTENT_SHA
    )

    print(
        "closure file:",
        CLOSURE_FILE_SHA
    )


print(
    "\nnext:"
)


if ALL_GRAPH_GATES_PASS:

    print(
        "12.13A7_FINAL_DEVELOPMENT_POLICY_FREEZE"
    )

else:

    print(
        "BRANCH_CLOSED"
    )

In [ ]:
# ============================================================
# Stage 12.14A0-R1
#
# POST-A6 ADJUSTED-METRIC FAILURE DECOMPOSITION
#
# REPAIR:
#   ACCOUNT FOR GLOBAL NORMALIZATION SPILLOVER IN
#   EXPOSURE-BASED BOOKKEEPING.
#
# ============================================================
#
# Parent Stage12.13:
#   CLOSE_BRANCH_NO_POLICY_PROMOTION
#
# A6 frozen result:
#
#   edge J delta       > 0
#   node recall delta  > 0
#   adjusted-edge delta < 0
#   division delta      = 0
#   official score delta = adjusted-edge delta
#
# ------------------------------------------------------------
# A0 FAILURE
# ------------------------------------------------------------
#
# The failed A0 allocated exact dataset pooled contributions
# only across datasets having action exposure.
#
# But pooled adjusted Jaccard is:
#
#       sum_i(w_i * adj_i) / sum_i(w_i)
#
# where:
#
#       w_i = TP_i + FP_i + FN_i
#
# If other datasets change w_i, even a dataset with NO action
# can acquire a nonzero change in its normalized pooled
# contribution because the GLOBAL denominator changes.
#
# Therefore exposure-only allocation omitted:
#
#   NO_ACTION_DATASET_NORMALIZATION_SPILLOVER
#
# ------------------------------------------------------------
# R1
# ------------------------------------------------------------
#
# 1. Reconstruct exact dataset contribution to pooled Δadj.
#
# 2. Algebraically decompose pooled Δadj into:
#
#       numerator-shift effect
#       denominator-normalization effect
#
# 3. For action topology / family bookkeeping:
#
#       exposed datasets:
#           allocate exact dataset contribution proportionally
#           to frozen within-dataset exposure counts
#
#       no-action datasets:
#           preserve exact residual as
#
#           NO_ACTION_DATASET_NORMALIZATION_SPILLOVER
#
# 4. Every bookkeeping family MUST reconstruct exact pooled
#    Δadj after the residual is included.
#
# ------------------------------------------------------------
# POSTMORTEM ONLY
#
# NO GT graph load
# NO official evaluator execution
# NO graph mutation
# NO action regeneration
# NO model training
# NO threshold tuning
# NO new policy
# NO Fold0
#
# KEEP PERMANENTLY.
# ============================================================


from pathlib import Path
import hashlib
import json
import math

import numpy as np
import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


ACTIONS_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_graph_actions_v1.pkl"
)

PRE_GT_CONTRACT_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_pre_gt_action_contract_frozen.json"
)

METRICS_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_official_dataset_metrics_v1.pkl"
)

FOLD_SUMMARY_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_official_fold_summary_v1.pkl"
)

A6_SUMMARY_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_official_summary_v1.json"
)

A6_DECISION_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_graph_promotion_decision_frozen.json"
)

A6_CLOSURE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_a6_graph_gate_closure.json"
)


# ------------------------------------------------------------
# Formal 12.14A0-R1 outputs
# ------------------------------------------------------------

DATASET_OUT = (
    S12
    / "stage12_14a0_post_a6_adjusted_metric_failure_dataset_decomposition_v1.pkl"
)

FOLD_OUT = (
    S12
    / "stage12_14a0_post_a6_adjusted_metric_failure_fold_decomposition_v1.pkl"
)

GROUP_OUT = (
    S12
    / "stage12_14a0_post_a6_adjusted_metric_failure_group_decomposition_v1.pkl"
)

ASSOCIATION_OUT = (
    S12
    / "stage12_14a0_post_a6_adjusted_metric_failure_associations_v1.pkl"
)

SUMMARY_OUT = (
    S12
    / "stage12_14a0_post_a6_adjusted_metric_failure_decomposition_v1_summary.json"
)


for path in [
    ACTIONS_PATH,
    PRE_GT_CONTRACT_PATH,
    METRICS_PATH,
    FOLD_SUMMARY_PATH,
    A6_SUMMARY_PATH,
    A6_DECISION_PATH,
    A6_CLOSURE_PATH,
]:
    assert path.exists(), path


for path in [
    DATASET_OUT,
    FOLD_OUT,
    GROUP_OUT,
    ASSOCIATION_OUT,
    SUMMARY_OUT,
]:
    if path.exists():
        raise RuntimeError(
            "\nFormal 12.14A0-R1 output already exists:\n"
            f"{path}\n\n"
            "Do NOT overwrite formal provenance."
        )


# ============================================================
# 1. FROZEN A6 HASHES
# ============================================================

EXPECTED_ACTIONS_CONTENT_SHA = (
    "a887af0f16e3375bf59d34ab8e211e36ab8c01c4c1599b2c1d2f2713d708e64d"
)

EXPECTED_ACTIONS_FILE_SHA = (
    "33900e3787d21e50ad8fbaae10fdbf25e621f97493f0e83b0542cadeb241695d"
)

EXPECTED_PRE_GT_CONTRACT_SHA = (
    "303051528929430a3c6b41cabe56c2499f1da191cc595780e8b30274293f65dc"
)


EXPECTED_METRICS_CONTENT_SHA = (
    "95e07b64b9cecf10f89feab7e2fc727f9588c443f6e765a8fa89a729b629e421"
)

EXPECTED_METRICS_FILE_SHA = (
    "9a6ea17c8d4e691967de680514ae86a92dd2c97bab19d48c9c3bd4d2be9c98dc"
)


EXPECTED_FOLD_CONTENT_SHA = (
    "c013c8c9679db71b9ff3f1a1a344c6975212d47538fda0a666ffa31851daca97"
)

EXPECTED_FOLD_FILE_SHA = (
    "a1617f12181352064a61a17b5f5ad19a7b7878b0bd422d9832299ca4553bd21c"
)


EXPECTED_A6_SUMMARY_CONTENT_SHA = (
    "08f3d3c7b35682d92ee9e0ff51669fe0c59809014cd4ea8af59c0e1518101064"
)

EXPECTED_A6_SUMMARY_FILE_SHA = (
    "37aa4c382e2384933365fb0cb284ddf70e52f30973bfe71bcfc07b62f9204253"
)


EXPECTED_A6_DECISION_CONTENT_SHA = (
    "0a3ae7f65a36e6c687643ccdf63bf3de2ef757f933391070f3234b7d5b92ca81"
)

EXPECTED_A6_DECISION_FILE_SHA = (
    "741235b55ec65131f1c02c7ba802baea6bb1a4b826d57cea1f2baf44ba078c4e"
)


EXPECTED_A6_CLOSURE_CONTENT_SHA = (
    "a9ddddf3749ee17703d741b11b3b1cb1ee16373726583e02f73b8fd730b21184"
)

EXPECTED_A6_CLOSURE_FILE_SHA = (
    "c3bb3d36b063825954b260a1de6307b3c511d579b36419d85074aa86dfc28cdd"
)


EXPECTED_DATASETS = 103
EXPECTED_ACTIONS = 3_105
EXPECTED_NEW_EDGES = 4_444


EXPECTED_BASE_SCORE = 0.7920280500935803
EXPECTED_POLICY_SCORE = 0.7919611690386696

EXPECTED_SCORE_DELTA = (
    -6.688105491070573e-05
)


EXPECTED_BASE_EDGE_J = (
    0.7978431110242136
)

EXPECTED_POLICY_EDGE_J = (
    0.7979559101752077
)


EXPECTED_BASE_ADJ = (
    0.7898541370501021
)

EXPECTED_POLICY_ADJ = (
    0.7897872559951914
)


EXPECTED_BASE_NODE_RECALL = (
    0.9655061364446426
)

EXPECTED_POLICY_NODE_RECALL = (
    0.9662734922760026
)


# ============================================================
# 2. HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def json_content_sha256(obj):

    return hashlib.sha256(
        json.dumps(
            obj,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


class HashWriter:

    def __init__(self):
        self.h = hashlib.sha256()

    def write(self, text):

        if not isinstance(text, str):
            text = str(text)

        self.h.update(
            text.encode("utf-8")
        )

        return len(text)

    def flush(self):
        return None


def dataframe_content_sha256(df):

    writer = HashWriter()

    df.to_csv(
        writer,
        index=False,
        na_rep="NaN",
        float_format="%.17g",
        lineterminator="\n",
    )

    return writer.h.hexdigest()


def safe_div(a, b):

    a = float(a)
    b = float(b)

    if b == 0:
        return float("nan")

    return a / b


def jsonable(value):

    if isinstance(value, dict):

        return {
            str(k): jsonable(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):

        return [
            jsonable(v)
            for v in value
        ]

    if isinstance(value, np.generic):

        return value.item()

    if isinstance(value, float):

        if math.isnan(value):
            return None

        return float(value)

    return value


def corr_safe(x, y, method):

    x = pd.Series(
        x,
        dtype=float,
    )

    y = pd.Series(
        y,
        dtype=float,
    )

    mask = (
        x.notna()
        &
        y.notna()
    )

    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return float("nan")

    if (
        x.nunique() <= 1
        or
        y.nunique() <= 1
    ):
        return float("nan")

    if method == "pearson":

        return float(
            x.corr(
                y,
                method="pearson",
            )
        )

    if method == "spearman":

        return float(
            x.rank(
                method="average"
            ).corr(
                y.rank(
                    method="average"
                ),
                method="pearson",
            )
        )

    raise ValueError(method)


# ============================================================
# 3. LOAD + PARENT LOCK
# ============================================================

assert (
    sha256_file(ACTIONS_PATH)
    ==
    EXPECTED_ACTIONS_FILE_SHA
)

assert (
    sha256_file(METRICS_PATH)
    ==
    EXPECTED_METRICS_FILE_SHA
)

assert (
    sha256_file(FOLD_SUMMARY_PATH)
    ==
    EXPECTED_FOLD_FILE_SHA
)

assert (
    sha256_file(A6_SUMMARY_PATH)
    ==
    EXPECTED_A6_SUMMARY_FILE_SHA
)

assert (
    sha256_file(A6_DECISION_PATH)
    ==
    EXPECTED_A6_DECISION_FILE_SHA
)

assert (
    sha256_file(A6_CLOSURE_PATH)
    ==
    EXPECTED_A6_CLOSURE_FILE_SHA
)


actions = pd.read_pickle(
    ACTIONS_PATH
)

metrics = pd.read_pickle(
    METRICS_PATH
)

fold_official = pd.read_pickle(
    FOLD_SUMMARY_PATH
)

pre_gt = json.loads(
    PRE_GT_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)

a6_summary = json.loads(
    A6_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

a6_decision = json.loads(
    A6_DECISION_PATH.read_text(
        encoding="utf-8"
    )
)

a6_closure = json.loads(
    A6_CLOSURE_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    dataframe_content_sha256(actions)
    ==
    EXPECTED_ACTIONS_CONTENT_SHA
)

assert (
    dataframe_content_sha256(metrics)
    ==
    EXPECTED_METRICS_CONTENT_SHA
)

assert (
    dataframe_content_sha256(fold_official)
    ==
    EXPECTED_FOLD_CONTENT_SHA
)


assert (
    pre_gt[
        "contract_sha256"
    ]
    ==
    EXPECTED_PRE_GT_CONTRACT_SHA
)

assert (
    a6_summary[
        "summary_sha256"
    ]
    ==
    EXPECTED_A6_SUMMARY_CONTENT_SHA
)

assert (
    a6_decision[
        "decision_sha256"
    ]
    ==
    EXPECTED_A6_DECISION_CONTENT_SHA
)

assert (
    a6_closure[
        "closure_sha256"
    ]
    ==
    EXPECTED_A6_CLOSURE_CONTENT_SHA
)


assert (
    a6_decision[
        "decision"
    ]
    ==
    "CLOSE_BRANCH_NO_POLICY_PROMOTION"
)

assert (
    a6_closure[
        "decision"
    ]
    ==
    "CLOSE_BRANCH_NO_POLICY_PROMOTION"
)

assert (
    a6_decision[
        "policy_promoted"
    ]
    is False
)

assert (
    a6_decision[
        "Fold0_used"
    ]
    is False
)


print(
    "========== 12.14A0-R1 PARENT LOCK =========="
)

print(
    "A6 actions:",
    "LOCKED"
)

print(
    "A6 metrics:",
    "LOCKED"
)

print(
    "A6 decision:",
    "CLOSE_BRANCH_NO_POLICY_PROMOTION"
)

print(
    "Fold0:",
    "NO"
)

print(
    "PARENT_LOCK: PASS"
)


# ============================================================
# 4. BASIC SCHEMA
# ============================================================

assert len(actions) == EXPECTED_ACTIONS

assert int(
    actions[
        "edges_added"
    ].sum()
) == EXPECTED_NEW_EDGES

assert len(metrics) == EXPECTED_DATASETS

assert metrics[
    "dataset"
].is_unique


assert set(
    metrics[
        "fold"
    ].astype(int)
) == {
    1,
    2,
    3,
    4,
}


required_metrics = {
    "dataset",
    "fold",
    "selected_candidates",
    "accepted_actions",
    "new_nodes",
    "new_edges",

    "base_edge_tp",
    "base_edge_fp",
    "base_edge_fn",
    "base_edge_jaccard",
    "base_adj_edge_jaccard",
    "base_node_recall",
    "base_num_pred_nodes",
    "base_total_node_ratio",

    "policy_edge_tp",
    "policy_edge_fp",
    "policy_edge_fn",
    "policy_edge_jaccard",
    "policy_adj_edge_jaccard",
    "policy_node_recall",
    "policy_num_pred_nodes",
    "policy_total_node_ratio",
}


assert required_metrics.issubset(
    metrics.columns
)


required_actions = {
    "dataset",
    "candidate_id",
    "incoming_family",
    "outgoing_family",
    "edges_added",
}


assert required_actions.issubset(
    actions.columns
)


# ============================================================
# 5. OFFICIAL SIGN REVERSAL LOCK
# ============================================================

pooled_base = (
    a6_summary[
        "pooled"
    ][
        "baseline"
    ]
)

pooled_policy = (
    a6_summary[
        "pooled"
    ][
        "policy"
    ]
)


official_score_delta = float(
    pooled_policy["score"]
    -
    pooled_base["score"]
)

official_edge_delta = float(
    pooled_policy["edge_jaccard"]
    -
    pooled_base["edge_jaccard"]
)

official_adj_delta = float(
    pooled_policy["adj_edge_jaccard"]
    -
    pooled_base["adj_edge_jaccard"]
)

official_node_delta = float(
    pooled_policy["node_recall"]
    -
    pooled_base["node_recall"]
)

official_division_delta = float(
    pooled_policy["division_jaccard"]
    -
    pooled_base["division_jaccard"]
)


assert abs(
    float(
        pooled_base["score"]
    )
    -
    EXPECTED_BASE_SCORE
) <= 1e-15


assert abs(
    float(
        pooled_policy["score"]
    )
    -
    EXPECTED_POLICY_SCORE
) <= 1e-15


assert abs(
    official_score_delta
    -
    EXPECTED_SCORE_DELTA
) <= 1e-15


assert (
    official_edge_delta > 0
)

assert (
    official_node_delta > 0
)

assert (
    official_adj_delta < 0
)

assert abs(
    official_division_delta
) <= 1e-15


assert abs(
    official_score_delta
    -
    official_adj_delta
) <= 1e-15


print(
    "\n========== SIGN REVERSAL LOCK =========="
)

print(
    "edge J delta:",
    official_edge_delta
)

print(
    "node recall delta:",
    official_node_delta
)

print(
    "adjusted-edge delta:",
    official_adj_delta
)

print(
    "official score delta:",
    official_score_delta
)

print(
    "division delta:",
    official_division_delta
)

print(
    "EDGE_NODE_GAIN_BUT_ADJUSTED_SCORE_LOSS: PASS"
)


# ============================================================
# 6. ACTION LABELS
# ============================================================

actions = actions.copy()


actions[
    "incoming_family_norm"
] = (
    actions[
        "incoming_family"
    ]
    .fillna("NONE")
    .astype(str)
)


actions[
    "outgoing_family_norm"
] = (
    actions[
        "outgoing_family"
    ]
    .fillna("NONE")
    .astype(str)
)


actions[
    "edge_count_class"
] = np.where(
    actions[
        "edges_added"
    ].astype(int)
    ==
    1,
    "ONE_EDGE",
    "TWO_EDGE",
)


actions[
    "action_topology"
] = (
    "E"
    +
    actions[
        "edges_added"
    ].astype(int).astype(str)
    +
    "|IN:"
    +
    actions[
        "incoming_family_norm"
    ]
    +
    "|OUT:"
    +
    actions[
        "outgoing_family_norm"
    ]
)


assert set(
    actions[
        "incoming_family_norm"
    ].unique()
).issubset({
    "NONE",
    "G0",
    "G2",
})


assert set(
    actions[
        "outgoing_family_norm"
    ].unique()
).issubset({
    "NONE",
    "G1",
    "G2",
})


# ============================================================
# 7. DATASET ACTION COUNTS
# ============================================================

action_counts = (
    actions
    .groupby(
        "dataset",
        sort=True,
    )
    .agg(
        actions_total=(
            "candidate_id",
            "size",
        ),

        action_new_edges=(
            "edges_added",
            "sum",
        ),

        one_edge_actions=(
            "edge_count_class",
            lambda x:
                int(
                    (
                        x
                        ==
                        "ONE_EDGE"
                    ).sum()
                ),
        ),

        two_edge_actions=(
            "edge_count_class",
            lambda x:
                int(
                    (
                        x
                        ==
                        "TWO_EDGE"
                    ).sum()
                ),
        ),

        incoming_G0=(
            "incoming_family_norm",
            lambda x:
                int(
                    (
                        x
                        ==
                        "G0"
                    ).sum()
                ),
        ),

        incoming_G2=(
            "incoming_family_norm",
            lambda x:
                int(
                    (
                        x
                        ==
                        "G2"
                    ).sum()
                ),
        ),

        outgoing_G1=(
            "outgoing_family_norm",
            lambda x:
                int(
                    (
                        x
                        ==
                        "G1"
                    ).sum()
                ),
        ),

        outgoing_G2=(
            "outgoing_family_norm",
            lambda x:
                int(
                    (
                        x
                        ==
                        "G2"
                    ).sum()
                ),
        ),
    )
    .reset_index()
)


# ============================================================
# 8. DATASET DECOMPOSITION
# ============================================================

dataset = metrics.merge(
    action_counts,
    on="dataset",
    how="left",
    validate="one_to_one",
)


count_columns = [
    "actions_total",
    "action_new_edges",
    "one_edge_actions",
    "two_edge_actions",
    "incoming_G0",
    "incoming_G2",
    "outgoing_G1",
    "outgoing_G2",
]


for column in count_columns:

    dataset[
        column
    ] = (
        dataset[
            column
        ]
        .fillna(0)
        .astype(int)
    )


assert np.array_equal(
    dataset[
        "accepted_actions"
    ].astype(int),
    dataset[
        "actions_total"
    ].astype(int),
)


assert np.array_equal(
    dataset[
        "new_edges"
    ].astype(int),
    dataset[
        "action_new_edges"
    ].astype(int),
)


assert np.array_equal(
    (
        dataset[
            "one_edge_actions"
        ]
        +
        dataset[
            "two_edge_actions"
        ]
    ),
    dataset[
        "actions_total"
    ],
)


dataset[
    "delta_edge_jaccard"
] = (
    dataset[
        "policy_edge_jaccard"
    ]
    -
    dataset[
        "base_edge_jaccard"
    ]
)


dataset[
    "delta_adj_edge_jaccard"
] = (
    dataset[
        "policy_adj_edge_jaccard"
    ]
    -
    dataset[
        "base_adj_edge_jaccard"
    ]
)


dataset[
    "delta_node_recall"
] = (
    dataset[
        "policy_node_recall"
    ]
    -
    dataset[
        "base_node_recall"
    ]
)


dataset[
    "delta_num_pred_nodes"
] = (
    dataset[
        "policy_num_pred_nodes"
    ]
    -
    dataset[
        "base_num_pred_nodes"
    ]
)


dataset[
    "delta_total_node_ratio"
] = (
    dataset[
        "policy_total_node_ratio"
    ]
    -
    dataset[
        "base_total_node_ratio"
    ]
)


assert np.array_equal(
    dataset[
        "delta_num_pred_nodes"
    ].astype(int),
    dataset[
        "actions_total"
    ].astype(int),
)


dataset[
    "base_adjustment_gap"
] = (
    dataset[
        "base_edge_jaccard"
    ]
    -
    dataset[
        "base_adj_edge_jaccard"
    ]
)


dataset[
    "policy_adjustment_gap"
] = (
    dataset[
        "policy_edge_jaccard"
    ]
    -
    dataset[
        "policy_adj_edge_jaccard"
    ]
)


dataset[
    "delta_adjustment_gap"
] = (
    dataset[
        "policy_adjustment_gap"
    ]
    -
    dataset[
        "base_adjustment_gap"
    ]
)


dataset[
    "action_accept_rate"
] = np.where(
    dataset[
        "selected_candidates"
    ]
    >
    0,
    dataset[
        "actions_total"
    ]
    /
    dataset[
        "selected_candidates"
    ],
    np.nan,
)


dataset[
    "one_edge_share"
] = np.where(
    dataset[
        "actions_total"
    ]
    >
    0,
    dataset[
        "one_edge_actions"
    ]
    /
    dataset[
        "actions_total"
    ],
    np.nan,
)


dataset[
    "two_edge_share"
] = np.where(
    dataset[
        "actions_total"
    ]
    >
    0,
    dataset[
        "two_edge_actions"
    ]
    /
    dataset[
        "actions_total"
    ],
    np.nan,
)


dataset[
    "g2_support_edges"
] = (
    dataset[
        "incoming_G2"
    ]
    +
    dataset[
        "outgoing_G2"
    ]
)


dataset[
    "g2_support_share"
] = np.where(
    dataset[
        "new_edges"
    ]
    >
    0,
    dataset[
        "g2_support_edges"
    ]
    /
    dataset[
        "new_edges"
    ],
    np.nan,
)


# ============================================================
# 9. EXACT DATASET POOLED ADJUSTED CONTRIBUTION
# ============================================================

dataset[
    "base_adj_weight"
] = (
    dataset[
        "base_edge_tp"
    ]
    +
    dataset[
        "base_edge_fp"
    ]
    +
    dataset[
        "base_edge_fn"
    ]
).astype(float)


dataset[
    "policy_adj_weight"
] = (
    dataset[
        "policy_edge_tp"
    ]
    +
    dataset[
        "policy_edge_fp"
    ]
    +
    dataset[
        "policy_edge_fn"
    ]
).astype(float)


W0 = float(
    dataset[
        "base_adj_weight"
    ].sum()
)

W1 = float(
    dataset[
        "policy_adj_weight"
    ].sum()
)


N0 = float(
    (
        dataset[
            "base_adj_weight"
        ]
        *
        dataset[
            "base_adj_edge_jaccard"
        ]
    ).sum()
)


N1 = float(
    (
        dataset[
            "policy_adj_weight"
        ]
        *
        dataset[
            "policy_adj_edge_jaccard"
        ]
    ).sum()
)


assert abs(
    N0 / W0
    -
    EXPECTED_BASE_ADJ
) <= 1e-12


assert abs(
    N1 / W1
    -
    EXPECTED_POLICY_ADJ
) <= 1e-12


dataset[
    "base_adj_pooled_contribution"
] = (
    dataset[
        "base_adj_weight"
    ]
    /
    W0
    *
    dataset[
        "base_adj_edge_jaccard"
    ]
)


dataset[
    "policy_adj_pooled_contribution"
] = (
    dataset[
        "policy_adj_weight"
    ]
    /
    W1
    *
    dataset[
        "policy_adj_edge_jaccard"
    ]
)


dataset[
    "delta_adj_pooled_contribution"
] = (
    dataset[
        "policy_adj_pooled_contribution"
    ]
    -
    dataset[
        "base_adj_pooled_contribution"
    ]
)


exact_dataset_delta = float(
    dataset[
        "delta_adj_pooled_contribution"
    ].sum()
)


assert abs(
    exact_dataset_delta
    -
    official_adj_delta
) <= 1e-12


print(
    "\n========== EXACT DATASET CONTRIBUTION =========="
)

print(
    "reconstructed baseline adj:",
    N0 / W0
)

print(
    "reconstructed policy adj:",
    N1 / W1
)

print(
    "reconstructed delta:",
    exact_dataset_delta
)

print(
    "EXACT_POOLED_ADJ_DECOMPOSITION: PASS"
)


# ============================================================
# 10. NEW: EXACT GLOBAL ALGEBRAIC DECOMPOSITION
#
# A1 - A0
#
# = (N1 - N0) / W0
#   + N1 * (1/W1 - 1/W0)
#
# first term:
#   numerator / local weighted-value shift
#
# second term:
#   global denominator-normalization shift
# ============================================================

NUMERATOR_SHIFT_EFFECT = float(
    (
        N1
        -
        N0
    )
    /
    W0
)


DENOMINATOR_NORMALIZATION_EFFECT = float(
    N1
    *
    (
        (1.0 / W1)
        -
        (1.0 / W0)
    )
)


ALGEBRAIC_TOTAL = (
    NUMERATOR_SHIFT_EFFECT
    +
    DENOMINATOR_NORMALIZATION_EFFECT
)


assert abs(
    ALGEBRAIC_TOTAL
    -
    official_adj_delta
) <= 1e-12


print(
    "\n========== EXACT GLOBAL NORMALIZATION DECOMPOSITION =========="
)

print(
    "baseline weight W0:",
    W0
)

print(
    "policy weight W1:",
    W1
)

print(
    "delta W:",
    W1 - W0
)

print(
    "numerator-shift effect:",
    NUMERATOR_SHIFT_EFFECT
)

print(
    "denominator-normalization effect:",
    DENOMINATOR_NORMALIZATION_EFFECT
)

print(
    "sum:",
    ALGEBRAIC_TOTAL
)

print(
    "official adjusted delta:",
    official_adj_delta
)

print(
    "GLOBAL_NORMALIZATION_DECOMPOSITION: PASS"
)


# ============================================================
# 11. NO-ACTION DATASET SPILLOVER
# ============================================================

no_action_mask = (
    dataset[
        "actions_total"
    ]
    ==
    0
)


no_action = dataset[
    no_action_mask
].copy()


# Local per-sample metrics should be unchanged without actions.

if len(
    no_action
) > 0:

    assert np.allclose(
        no_action[
            "delta_adj_edge_jaccard"
        ],
        0.0,
        atol=1e-15,
        rtol=0.0,
    )

    assert np.allclose(
        no_action[
            "delta_edge_jaccard"
        ],
        0.0,
        atol=1e-15,
        rtol=0.0,
    )

    assert np.allclose(
        no_action[
            "delta_node_recall"
        ],
        0.0,
        atol=1e-15,
        rtol=0.0,
    )

    assert np.allclose(
        no_action[
            "delta_num_pred_nodes"
        ],
        0.0,
        atol=1e-15,
        rtol=0.0,
    )


NO_ACTION_NORMALIZATION_SPILLOVER = float(
    no_action[
        "delta_adj_pooled_contribution"
    ].sum()
)


action_dataset_contribution = float(
    dataset.loc[
        ~no_action_mask,
        "delta_adj_pooled_contribution",
    ].sum()
)


assert abs(
    (
        action_dataset_contribution
        +
        NO_ACTION_NORMALIZATION_SPILLOVER
    )
    -
    official_adj_delta
) <= 1e-12


print(
    "\n========== NO-ACTION NORMALIZATION SPILLOVER =========="
)

print(
    "no-action datasets:",
    len(
        no_action
    )
)

print(
    "action-dataset contribution:",
    action_dataset_contribution
)

print(
    "no-action normalization spillover:",
    NO_ACTION_NORMALIZATION_SPILLOVER
)

print(
    "total:",
    (
        action_dataset_contribution
        +
        NO_ACTION_NORMALIZATION_SPILLOVER
    )
)

print(
    "NO_ACTION_NORMALIZATION_SPILLOVER_LOCK: PASS"
)


# ============================================================
# 12. FOLD DECOMPOSITION
# ============================================================

fold_rows = []


for fold_id in [
    1,
    2,
    3,
    4,
]:

    sub = dataset[
        dataset[
            "fold"
        ].astype(int)
        ==
        fold_id
    ]


    official = (
        fold_official[
            fold_official[
                "fold"
            ].astype(int)
            ==
            fold_id
        ]
        .iloc[0]
    )


    fold_rows.append({
        "fold":
            fold_id,

        "datasets":
            int(
                len(sub)
            ),

        "no_action_datasets":
            int(
                (
                    sub[
                        "actions_total"
                    ]
                    ==
                    0
                ).sum()
            ),

        "selected_candidates":
            int(
                sub[
                    "selected_candidates"
                ].sum()
            ),

        "actions":
            int(
                sub[
                    "actions_total"
                ].sum()
            ),

        "new_edges":
            int(
                sub[
                    "new_edges"
                ].sum()
            ),

        "one_edge_actions":
            int(
                sub[
                    "one_edge_actions"
                ].sum()
            ),

        "two_edge_actions":
            int(
                sub[
                    "two_edge_actions"
                ].sum()
            ),

        "action_accept_rate":
            safe_div(
                sub[
                    "actions_total"
                ].sum(),
                sub[
                    "selected_candidates"
                ].sum(),
            ),

        "two_edge_share":
            safe_div(
                sub[
                    "two_edge_actions"
                ].sum(),
                sub[
                    "actions_total"
                ].sum(),
            ),

        "g2_support_share":
            safe_div(
                sub[
                    "g2_support_edges"
                ].sum(),
                sub[
                    "new_edges"
                ].sum(),
            ),

        "mean_delta_node_ratio":
            float(
                sub[
                    "delta_total_node_ratio"
                ].mean()
            ),

        "mean_delta_adj":
            float(
                sub[
                    "delta_adj_edge_jaccard"
                ].mean()
            ),

        "mean_delta_edge_j":
            float(
                sub[
                    "delta_edge_jaccard"
                ].mean()
            ),

        "mean_delta_node_recall":
            float(
                sub[
                    "delta_node_recall"
                ].mean()
            ),

        "mean_delta_adjustment_gap":
            float(
                sub[
                    "delta_adjustment_gap"
                ].mean()
            ),

        "exact_global_adj_contribution":
            float(
                sub[
                    "delta_adj_pooled_contribution"
                ].sum()
            ),

        "official_fold_score_delta":
            float(
                official[
                    "delta_score"
                ]
            ),

        "official_fold_edge_j_delta":
            float(
                official[
                    "delta_edge_jaccard"
                ]
            ),

        "official_fold_node_recall_delta":
            float(
                official[
                    "delta_node_recall"
                ]
            ),

        "official_fold_division_delta":
            float(
                official[
                    "delta_division_jaccard"
                ]
            ),
    })


fold_decomp = pd.DataFrame(
    fold_rows
)


# ============================================================
# 13. REPAIRED EXPOSURE BOOKKEEPING
# ============================================================

RESIDUAL_GROUP = (
    "NO_ACTION_DATASET_NORMALIZATION_SPILLOVER"
)


def exposure_decomposition(
    exposure_frame,
    exposure_column,
    group_type,
):

    counts = (
        exposure_frame
        .groupby(
            [
                "dataset",
                exposure_column,
            ],
            sort=True,
        )
        .size()
        .rename(
            "exposure_count"
        )
        .reset_index()
    )


    totals = (
        counts
        .groupby(
            "dataset"
        )[
            "exposure_count"
        ]
        .sum()
        .rename(
            "dataset_total_exposure"
        )
        .reset_index()
    )


    counts = counts.merge(
        totals,
        on="dataset",
        how="left",
        validate="many_to_one",
    )


    counts = counts.merge(
        dataset[
            [
                "dataset",
                "delta_adj_edge_jaccard",
                "delta_edge_jaccard",
                "delta_node_recall",
                "delta_total_node_ratio",
                "delta_adj_pooled_contribution",
            ]
        ],
        on="dataset",
        how="left",
        validate="many_to_one",
    )


    counts[
        "within_dataset_exposure_share"
    ] = (
        counts[
            "exposure_count"
        ]
        /
        counts[
            "dataset_total_exposure"
        ]
    )


    counts[
        "bookkeeping_adj_contribution"
    ] = (
        counts[
            "delta_adj_pooled_contribution"
        ]
        *
        counts[
            "within_dataset_exposure_share"
        ]
    )


    rows = []


    for group, sub in counts.groupby(
        exposure_column,
        sort=True,
    ):

        weights = (
            sub[
                "exposure_count"
            ].astype(float)
        )


        rows.append({
            "group_type":
                group_type,

            "group":
                str(group),

            "exposure_count":
                int(
                    weights.sum()
                ),

            "datasets_exposed":
                int(
                    sub[
                        "dataset"
                    ].nunique()
                ),

            "bookkeeping_adj_contribution":
                float(
                    sub[
                        "bookkeeping_adj_contribution"
                    ].sum()
                ),

            "exposure_weighted_mean_delta_adj":
                float(
                    np.average(
                        sub[
                            "delta_adj_edge_jaccard"
                        ],
                        weights=weights,
                    )
                ),

            "exposure_weighted_mean_delta_edge_j":
                float(
                    np.average(
                        sub[
                            "delta_edge_jaccard"
                        ],
                        weights=weights,
                    )
                ),

            "exposure_weighted_mean_delta_node_recall":
                float(
                    np.average(
                        sub[
                            "delta_node_recall"
                        ],
                        weights=weights,
                    )
                ),

            "exposure_weighted_mean_delta_node_ratio":
                float(
                    np.average(
                        sub[
                            "delta_total_node_ratio"
                        ],
                        weights=weights,
                    )
                ),

            "interpretation":
                "NON_CAUSAL_EXPOSURE_BOOKKEEPING",
        })


    # --------------------------------------------------------
    # CRITICAL R1 REPAIR
    #
    # Datasets with no action exposure can still have a
    # nonzero pooled contribution because W changes globally.
    # --------------------------------------------------------

    exposed_datasets = set(
        counts[
            "dataset"
        ].astype(str)
    )


    unexposed = dataset[
        ~dataset[
            "dataset"
        ].astype(str).isin(
            exposed_datasets
        )
    ]


    residual = float(
        unexposed[
            "delta_adj_pooled_contribution"
        ].sum()
    )


    rows.append({
        "group_type":
            group_type,

        "group":
            RESIDUAL_GROUP,

        "exposure_count":
            0,

        "datasets_exposed":
            int(
                len(
                    unexposed
                )
            ),

        "bookkeeping_adj_contribution":
            residual,

        "exposure_weighted_mean_delta_adj":
            float("nan"),

        "exposure_weighted_mean_delta_edge_j":
            float("nan"),

        "exposure_weighted_mean_delta_node_recall":
            float("nan"),

        "exposure_weighted_mean_delta_node_ratio":
            float("nan"),

        "interpretation":
            (
                "GLOBAL_WEIGHT_NORMALIZATION_RESIDUAL_"
                "NO_LOCAL_ACTION_EXPOSURE"
            ),
    })


    result = pd.DataFrame(
        rows
    )


    reconstructed = float(
        result[
            "bookkeeping_adj_contribution"
        ].sum()
    )


    assert abs(
        reconstructed
        -
        official_adj_delta
    ) <= 1e-12, (
        group_type,
        reconstructed,
        official_adj_delta,
    )


    return result


# ------------------------------------------------------------
# 13A. 1-edge versus 2-edge
# ------------------------------------------------------------

edge_class_table = exposure_decomposition(
    actions[
        [
            "dataset",
            "edge_count_class",
        ]
    ],
    "edge_count_class",
    "ACTION_EDGE_COUNT_CLASS",
)


# ------------------------------------------------------------
# 13B. Full action topology
# ------------------------------------------------------------

topology_table = exposure_decomposition(
    actions[
        [
            "dataset",
            "action_topology",
        ]
    ],
    "action_topology",
    "ACTION_TOPOLOGY",
)


# ------------------------------------------------------------
# 13C. Attached support family / direction
# ------------------------------------------------------------

support_rows = []


for row in actions.itertuples(
    index=False
):

    if (
        row.incoming_family_norm
        !=
        "NONE"
    ):

        support_rows.append({
            "dataset":
                str(
                    row.dataset
                ),

            "support_family_direction":
                (
                    "IN_"
                    +
                    str(
                        row.incoming_family_norm
                    )
                ),
        })


    if (
        row.outgoing_family_norm
        !=
        "NONE"
    ):

        support_rows.append({
            "dataset":
                str(
                    row.dataset
                ),

            "support_family_direction":
                (
                    "OUT_"
                    +
                    str(
                        row.outgoing_family_norm
                    )
                ),
        })


support_exposure = pd.DataFrame(
    support_rows
)


assert len(
    support_exposure
) == EXPECTED_NEW_EDGES


support_table = exposure_decomposition(
    support_exposure,
    "support_family_direction",
    "SUPPORT_FAMILY_DIRECTION",
)


group_decomp = pd.concat(
    [
        edge_class_table,
        topology_table,
        support_table,
    ],
    ignore_index=True,
)


# Every independent bookkeeping view must now reconstruct
# exact pooled Δadj.

for group_type in group_decomp[
    "group_type"
].unique():

    reconstructed = float(
        group_decomp.loc[
            group_decomp[
                "group_type"
            ]
            ==
            group_type,
            "bookkeeping_adj_contribution",
        ].sum()
    )


    assert abs(
        reconstructed
        -
        official_adj_delta
    ) <= 1e-12


print(
    "\n========== REPAIRED EXPOSURE BOOKKEEPING =========="
)

for group_type in sorted(
    group_decomp[
        "group_type"
    ].unique()
):

    value = float(
        group_decomp.loc[
            group_decomp[
                "group_type"
            ]
            ==
            group_type,
            "bookkeeping_adj_contribution",
        ].sum()
    )

    print(
        group_type,
        "->",
        value
    )


print(
    "official adjusted delta:",
    official_adj_delta
)

print(
    "ALL_EXPOSURE_DECOMPOSITIONS_RECONSTRUCT_EXACT_DELTA: PASS"
)


# ============================================================
# 14. ASSOCIATIONS
#
# DESCRIPTIVE, NON-CAUSAL.
# ============================================================

features = [
    "actions_total",
    "new_edges",
    "action_accept_rate",
    "one_edge_share",
    "two_edge_share",
    "g2_support_share",
    "delta_num_pred_nodes",
    "delta_total_node_ratio",
]


targets = [
    "delta_adj_edge_jaccard",
    "delta_adjustment_gap",
    "delta_adj_pooled_contribution",
]


association_rows = []


for feature in features:

    for target in targets:

        association_rows.append({
            "feature":
                feature,

            "target":
                target,

            "pearson":
                corr_safe(
                    dataset[
                        feature
                    ],
                    dataset[
                        target
                    ],
                    "pearson",
                ),

            "spearman":
                corr_safe(
                    dataset[
                        feature
                    ],
                    dataset[
                        target
                    ],
                    "spearman",
                ),

            "interpretation":
                "DESCRIPTIVE_NON_CAUSAL",
        })


associations = pd.DataFrame(
    association_rows
)


# ============================================================
# 15. HARMFUL / HELPFUL DATASETS
# ============================================================

report_columns = [
    "dataset",
    "fold",
    "selected_candidates",
    "actions_total",
    "new_edges",
    "one_edge_actions",
    "two_edge_actions",
    "one_edge_share",
    "two_edge_share",
    "g2_support_share",
    "delta_num_pred_nodes",
    "delta_total_node_ratio",
    "delta_edge_jaccard",
    "delta_adj_edge_jaccard",
    "delta_node_recall",
    "delta_adjustment_gap",
    "delta_adj_pooled_contribution",
]


harmful = (
    dataset
    .sort_values(
        "delta_adj_pooled_contribution",
        ascending=True,
        kind="stable",
    )
    .head(15)
)


helpful = (
    dataset
    .sort_values(
        "delta_adj_pooled_contribution",
        ascending=False,
        kind="stable",
    )
    .head(15)
)


# ============================================================
# 16. FOLD SIGN LOCK
# ============================================================

positive_folds = (
    fold_decomp.loc[
        fold_decomp[
            "official_fold_score_delta"
        ]
        >
        0,
        "fold",
    ]
    .astype(int)
    .tolist()
)


negative_folds = (
    fold_decomp.loc[
        fold_decomp[
            "official_fold_score_delta"
        ]
        <
        0,
        "fold",
    ]
    .astype(int)
    .tolist()
)


assert positive_folds == [
    1,
    2,
]


assert negative_folds == [
    3,
    4,
]


best_fold = int(
    fold_decomp
    .sort_values(
        "official_fold_score_delta",
        ascending=False,
    )
    .iloc[0][
        "fold"
    ]
)


worst_fold = int(
    fold_decomp
    .sort_values(
        "official_fold_score_delta",
        ascending=True,
    )
    .iloc[0][
        "fold"
    ]
)


# ============================================================
# 17. SORT FORMAL TABLES
# ============================================================

dataset_decomp = (
    dataset
    .sort_values(
        [
            "fold",
            "dataset",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


fold_decomp = (
    fold_decomp
    .sort_values(
        "fold",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


group_decomp = (
    group_decomp
    .sort_values(
        [
            "group_type",
            "bookkeeping_adj_contribution",
            "group",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


associations = (
    associations
    .sort_values(
        [
            "target",
            "feature",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 18. PROTOCOL FLAGS
# ============================================================

GT_GRAPH_LOADED = False

OFFICIAL_EVALUATOR_EXECUTED = False

GRAPH_MUTATED = False

ACTIONS_REGENERATED = False

MODEL_TRAINED = False

THRESHOLD_TUNED = False

NEW_POLICY_DEFINED = False

FOLD0_USED = False

PER_ACTION_CAUSAL_ATTRIBUTION = False


# ============================================================
# 19. CONTENT SHAS
# ============================================================

DATASET_CONTENT_SHA = (
    dataframe_content_sha256(
        dataset_decomp
    )
)


FOLD_CONTENT_SHA = (
    dataframe_content_sha256(
        fold_decomp
    )
)


GROUP_CONTENT_SHA = (
    dataframe_content_sha256(
        group_decomp
    )
)


ASSOCIATION_CONTENT_SHA = (
    dataframe_content_sha256(
        associations
    )
)


# ============================================================
# 20. FORMAL SUMMARY
# ============================================================

SUMMARY = {
    "stage":
        "12.14A0",

    "implementation_revision":
        (
            "R1_GLOBAL_NORMALIZATION_SPILLOVER_REPAIR"
        ),

    "status":
        (
            "POST_A6_ADJUSTED_METRIC_FAILURE_"
            "DECOMPOSITION_COMPLETE"
        ),

    "mode":
        "POSTMORTEM_ONLY",

    "failed_predecessor": {
        "stage":
            "12.14A0",

        "failure":
            (
                "EXPOSURE_BOOKKEEPING_OMITTED_"
                "NO_ACTION_DATASET_GLOBAL_"
                "NORMALIZATION_SPILLOVER"
            ),

        "scientific_parent_changed":
            False,
    },

    "parent_branch":
        (
            "POST_G3H3_TRACK_GEOMETRY_"
            "SYNTHETIC_NODE_RECOVERY_V1"
        ),

    "parent_decision":
        "CLOSE_BRANCH_NO_POLICY_PROMOTION",

    "parents": {
        "actions_content_sha256":
            EXPECTED_ACTIONS_CONTENT_SHA,

        "actions_file_sha256":
            EXPECTED_ACTIONS_FILE_SHA,

        "pre_GT_contract_sha256":
            EXPECTED_PRE_GT_CONTRACT_SHA,

        "dataset_metrics_content_sha256":
            EXPECTED_METRICS_CONTENT_SHA,

        "dataset_metrics_file_sha256":
            EXPECTED_METRICS_FILE_SHA,

        "fold_summary_content_sha256":
            EXPECTED_FOLD_CONTENT_SHA,

        "fold_summary_file_sha256":
            EXPECTED_FOLD_FILE_SHA,

        "A6_summary_content_sha256":
            EXPECTED_A6_SUMMARY_CONTENT_SHA,

        "A6_summary_file_sha256":
            EXPECTED_A6_SUMMARY_FILE_SHA,

        "A6_decision_content_sha256":
            EXPECTED_A6_DECISION_CONTENT_SHA,

        "A6_decision_file_sha256":
            EXPECTED_A6_DECISION_FILE_SHA,

        "A6_closure_content_sha256":
            EXPECTED_A6_CLOSURE_CONTENT_SHA,

        "A6_closure_file_sha256":
            EXPECTED_A6_CLOSURE_FILE_SHA,
    },

    "official_sign_reversal": {
        "baseline_score":
            EXPECTED_BASE_SCORE,

        "policy_score":
            EXPECTED_POLICY_SCORE,

        "delta_score":
            official_score_delta,

        "delta_edge_jaccard":
            official_edge_delta,

        "delta_node_recall":
            official_node_delta,

        "delta_adj_edge_jaccard":
            official_adj_delta,

        "delta_division_jaccard":
            official_division_delta,

        "edge_and_node_improve_but_adjusted_score_declines":
            True,

        "score_loss_equals_adjusted_edge_loss":
            True,
    },

    "exact_adjusted_metric_algebra": {
        "baseline_weight_W0":
            W0,

        "policy_weight_W1":
            W1,

        "delta_weight":
            W1 - W0,

        "baseline_weighted_numerator_N0":
            N0,

        "policy_weighted_numerator_N1":
            N1,

        "numerator_shift_effect":
            NUMERATOR_SHIFT_EFFECT,

        "denominator_normalization_effect":
            DENOMINATOR_NORMALIZATION_EFFECT,

        "reconstructed_total":
            ALGEBRAIC_TOTAL,

        "official_delta":
            official_adj_delta,

        "exact":
            True,
    },

    "no_action_normalization_spillover": {
        "no_action_datasets":
            int(
                len(
                    no_action
                )
            ),

        "action_dataset_contribution":
            action_dataset_contribution,

        "no_action_dataset_contribution":
            NO_ACTION_NORMALIZATION_SPILLOVER,

        "sum":
            (
                action_dataset_contribution
                +
                NO_ACTION_NORMALIZATION_SPILLOVER
            ),

        "interpretation":
            (
                "NO_ACTION_DATASETS_HAVE_UNCHANGED_LOCAL_"
                "METRICS_BUT_CAN_CHANGE_NORMALIZED_POOLED_"
                "CONTRIBUTION_WHEN_GLOBAL_WEIGHT_DENOMINATOR_CHANGES"
            ),
    },

    "fold_structure": {
        "positive_score_folds":
            positive_folds,

        "negative_score_folds":
            negative_folds,

        "best_fold":
            best_fold,

        "worst_fold":
            worst_fold,
    },

    "group_bookkeeping_contract": {
        "causal":
            False,

        "method":
            (
                "ALLOCATE_EXACT_DATASET_POOLED_CONTRIBUTION_"
                "PROPORTIONALLY_TO_WITHIN_DATASET_FROZEN_EXPOSURES_"
                "PLUS_EXPLICIT_NO_ACTION_NORMALIZATION_RESIDUAL"
            ),

        "residual_group":
            RESIDUAL_GROUP,

        "all_group_views_reconstruct_exact_pooled_delta":
            True,
    },

    "top_harmful_datasets":
        harmful[
            report_columns
        ].to_dict(
            orient="records"
        ),

    "top_helpful_datasets":
        helpful[
            report_columns
        ].to_dict(
            orient="records"
        ),

    "protocol": {
        "GT_graph_loaded":
            GT_GRAPH_LOADED,

        "official_evaluator_executed":
            OFFICIAL_EVALUATOR_EXECUTED,

        "graph_mutated":
            GRAPH_MUTATED,

        "actions_regenerated":
            ACTIONS_REGENERATED,

        "model_trained":
            MODEL_TRAINED,

        "threshold_tuned":
            THRESHOLD_TUNED,

        "new_policy_defined":
            NEW_POLICY_DEFINED,

        "Fold0_used":
            FOLD0_USED,

        "per_action_causal_attribution_claimed":
            PER_ACTION_CAUSAL_ATTRIBUTION,
    },

    "next":
        (
            "12.14A1_NEW_HYPOTHESIS_PREREGISTRATION"
        ),

    "content_sha256": {
        "dataset_decomposition":
            DATASET_CONTENT_SHA,

        "fold_decomposition":
            FOLD_CONTENT_SHA,

        "group_decomposition":
            GROUP_CONTENT_SHA,

        "associations":
            ASSOCIATION_CONTENT_SHA,
    },
}


SUMMARY_CONTENT_SHA = (
    json_content_sha256(
        SUMMARY
    )
)


SUMMARY[
    "summary_sha256"
] = SUMMARY_CONTENT_SHA


# ============================================================
# 21. WRITE
# ============================================================

TMP_DATASET = DATASET_OUT.with_name(
    DATASET_OUT.name
    +
    ".tmp_12_14a0_r1"
)

TMP_FOLD = FOLD_OUT.with_name(
    FOLD_OUT.name
    +
    ".tmp_12_14a0_r1"
)

TMP_GROUP = GROUP_OUT.with_name(
    GROUP_OUT.name
    +
    ".tmp_12_14a0_r1"
)

TMP_ASSOC = ASSOCIATION_OUT.with_name(
    ASSOCIATION_OUT.name
    +
    ".tmp_12_14a0_r1"
)

TMP_SUMMARY = SUMMARY_OUT.with_name(
    SUMMARY_OUT.name
    +
    ".tmp_12_14a0_r1"
)


for path in [
    TMP_DATASET,
    TMP_FOLD,
    TMP_GROUP,
    TMP_ASSOC,
    TMP_SUMMARY,
]:

    if path.exists():

        raise RuntimeError(
            "\nStale 12.14A0-R1 temporary artifact:\n"
            f"{path}"
        )


dataset_decomp.to_pickle(
    TMP_DATASET
)

fold_decomp.to_pickle(
    TMP_FOLD
)

group_decomp.to_pickle(
    TMP_GROUP
)

associations.to_pickle(
    TMP_ASSOC
)


TMP_SUMMARY.write_text(
    json.dumps(
        jsonable(
            SUMMARY
        ),
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 22. READBACK
# ============================================================

dataset_check = pd.read_pickle(
    TMP_DATASET
)

fold_check = pd.read_pickle(
    TMP_FOLD
)

group_check = pd.read_pickle(
    TMP_GROUP
)

association_check = pd.read_pickle(
    TMP_ASSOC
)

summary_check = json.loads(
    TMP_SUMMARY.read_text(
        encoding="utf-8"
    )
)


pd.testing.assert_frame_equal(
    dataset_check,
    dataset_decomp,
    check_exact=True,
)

pd.testing.assert_frame_equal(
    fold_check,
    fold_decomp,
    check_exact=True,
)

pd.testing.assert_frame_equal(
    group_check,
    group_decomp,
    check_exact=True,
)

pd.testing.assert_frame_equal(
    association_check,
    associations,
    check_exact=True,
)


assert (
    dataframe_content_sha256(
        dataset_check
    )
    ==
    DATASET_CONTENT_SHA
)

assert (
    dataframe_content_sha256(
        fold_check
    )
    ==
    FOLD_CONTENT_SHA
)

assert (
    dataframe_content_sha256(
        group_check
    )
    ==
    GROUP_CONTENT_SHA
)

assert (
    dataframe_content_sha256(
        association_check
    )
    ==
    ASSOCIATION_CONTENT_SHA
)

assert (
    summary_check[
        "summary_sha256"
    ]
    ==
    SUMMARY_CONTENT_SHA
)


DATASET_FILE_SHA = (
    sha256_file(
        TMP_DATASET
    )
)

FOLD_FILE_SHA = (
    sha256_file(
        TMP_FOLD
    )
)

GROUP_FILE_SHA = (
    sha256_file(
        TMP_GROUP
    )
)

ASSOCIATION_FILE_SHA = (
    sha256_file(
        TMP_ASSOC
    )
)

SUMMARY_FILE_SHA = (
    sha256_file(
        TMP_SUMMARY
    )
)


TMP_DATASET.replace(
    DATASET_OUT
)

TMP_FOLD.replace(
    FOLD_OUT
)

TMP_GROUP.replace(
    GROUP_OUT
)

TMP_ASSOC.replace(
    ASSOCIATION_OUT
)

TMP_SUMMARY.replace(
    SUMMARY_OUT
)


# ============================================================
# 23. FINAL PARENT IMMUTABILITY
# ============================================================

assert (
    sha256_file(
        ACTIONS_PATH
    )
    ==
    EXPECTED_ACTIONS_FILE_SHA
)

assert (
    sha256_file(
        METRICS_PATH
    )
    ==
    EXPECTED_METRICS_FILE_SHA
)

assert (
    sha256_file(
        A6_SUMMARY_PATH
    )
    ==
    EXPECTED_A6_SUMMARY_FILE_SHA
)

assert (
    sha256_file(
        A6_DECISION_PATH
    )
    ==
    EXPECTED_A6_DECISION_FILE_SHA
)

assert (
    sha256_file(
        A6_CLOSURE_PATH
    )
    ==
    EXPECTED_A6_CLOSURE_FILE_SHA
)


# ============================================================
# 24. FINAL REPORT
# ============================================================

print(
    "\n============================================================"
)

print(
    "12.14A0-R1 FINAL"
)

print(
    "============================================================"
)


print(
    "\nPOST_A6_ADJUSTED_METRIC_FAILURE_DECOMPOSITION:",
    "PASS"
)


print(
    "\n--- EXACT GLOBAL ALGEBRA ---"
)

print(
    "W0:",
    W0
)

print(
    "W1:",
    W1
)

print(
    "delta W:",
    W1 - W0
)

print(
    "numerator-shift effect:",
    NUMERATOR_SHIFT_EFFECT
)

print(
    "denominator-normalization effect:",
    DENOMINATOR_NORMALIZATION_EFFECT
)

print(
    "total:",
    ALGEBRAIC_TOTAL
)

print(
    "official adjusted delta:",
    official_adj_delta
)


print(
    "\n--- NO-ACTION NORMALIZATION SPILLOVER ---"
)

print(
    "no-action datasets:",
    len(
        no_action
    )
)

print(
    "action-dataset contribution:",
    action_dataset_contribution
)

print(
    "no-action normalization spillover:",
    NO_ACTION_NORMALIZATION_SPILLOVER
)

print(
    "reconstructed total:",
    (
        action_dataset_contribution
        +
        NO_ACTION_NORMALIZATION_SPILLOVER
    )
)


print(
    "\n--- FOLD DECOMPOSITION ---"
)

print(
    fold_decomp.to_string(
        index=False
    )
)


print(
    "\n--- 1-EDGE VS 2-EDGE BOOKKEEPING ---"
)

print(
    group_decomp[
        group_decomp[
            "group_type"
        ]
        ==
        "ACTION_EDGE_COUNT_CLASS"
    ].to_string(
        index=False
    )
)


print(
    "\n--- SUPPORT FAMILY BOOKKEEPING ---"
)

print(
    group_decomp[
        group_decomp[
            "group_type"
        ]
        ==
        "SUPPORT_FAMILY_DIRECTION"
    ].to_string(
        index=False
    )
)


print(
    "\n--- MOST NEGATIVE ACTION TOPOLOGIES ---"
)

print(
    group_decomp[
        group_decomp[
            "group_type"
        ]
        ==
        "ACTION_TOPOLOGY"
    ]
    .sort_values(
        "bookkeeping_adj_contribution"
    )
    .head(12)
    .to_string(
        index=False
    )
)


print(
    "\n--- TOP 10 HARMFUL DATASETS ---"
)

print(
    harmful[
        report_columns
    ]
    .head(10)
    .to_string(
        index=False
    )
)


print(
    "\n--- TOP 10 HELPFUL DATASETS ---"
)

print(
    helpful[
        report_columns
    ]
    .head(10)
    .to_string(
        index=False
    )
)


print(
    "\n--- ASSOCIATIONS ---"
)

print(
    associations.to_string(
        index=False
    )
)


print(
    "\n--- INTERPRETATION LIMITS ---"
)

print(
    "group bookkeeping causal:",
    "NO"
)

print(
    "per-action causal attribution:",
    "NO"
)

print(
    "new policy defined:",
    "NO"
)

print(
    "threshold tuned:",
    "NO"
)

print(
    "GT graph loaded:",
    "NO"
)

print(
    "official evaluator executed:",
    "NO"
)

print(
    "graph mutated:",
    "NO"
)

print(
    "Fold0 used:",
    "NO"
)


print(
    "\n--- CONTENT SHAS ---"
)

print(
    "dataset:",
    DATASET_CONTENT_SHA
)

print(
    "fold:",
    FOLD_CONTENT_SHA
)

print(
    "group:",
    GROUP_CONTENT_SHA
)

print(
    "associations:",
    ASSOCIATION_CONTENT_SHA
)

print(
    "summary:",
    SUMMARY_CONTENT_SHA
)


print(
    "\n--- FILE SHAS ---"
)

print(
    "dataset:",
    DATASET_FILE_SHA
)

print(
    "fold:",
    FOLD_FILE_SHA
)

print(
    "group:",
    GROUP_FILE_SHA
)

print(
    "associations:",
    ASSOCIATION_FILE_SHA
)

print(
    "summary:",
    SUMMARY_FILE_SHA
)


print(
    "\nnext:"
)

print(
    "12.14A1_NEW_HYPOTHESIS_PREREGISTRATION"
)

In [ ]:
# ============================================================
# Stage 12.14A1
#
# NEW HYPOTHESIS PREREGISTRATION
#
# ============================================================
#
# HYPOTHESIS:
#
#   BILATERAL_G2_TWO_EDGE_ECONOMY_V1
#
# ------------------------------------------------------------
# SCIENTIFIC STATUS
# ------------------------------------------------------------
#
# Stage12.13:
#
#   CLOSED
#   NO POLICY PROMOTION
#
# Stage12.14A0-R1:
#
#   post-hoc failure decomposition complete
#
# A0-R1 established:
#
#   numerator-shift effect:
#       +0.0005751761522851209
#
#   denominator-normalization effect:
#       -0.0006420572071952708
#
#   official adjusted-score delta:
#       -6.688105491070573e-05
#
# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------
#
# The Stage12.14 hypothesis is GENERATED AFTER SEEING
# Stage12.13 development GT results.
#
# Therefore:
#
#   folds 1-4:
#       HYPOTHESIS-GENERATION / TRAINING DATA ONLY
#
#       NEVER again valid as confirmatory evidence for
#       Stage12.14.
#
#   Fold0:
#       permanently consumed / forbidden
#
#   confirmation:
#       requires genuinely fresh untouched holdout GT
#
# ------------------------------------------------------------
# PRIMARY HYPOTHESIS
# ------------------------------------------------------------
#
# Stage12.13 synthetic recovery produced useful node/edge
# numerator signal but insufficient structural value per
# added synthetic node.
#
# A stricter recovery policy that:
#
#   - uses only bilateral G2 proposals
#   - requires a SAME-RAW G2 source/target pair
#   - requires BOTH graph slots free
#   - inserts exactly one node
#   - inserts exactly two edges
#   - forbids one-edge fallback
#
# may improve structural benefit per added node enough to
# overcome adjusted-metric normalization burden.
#
# ------------------------------------------------------------
# THIS IS A HYPOTHESIS, NOT A RESULT.
#
# A0 topology/family decompositions were explicitly
# NON-CAUSAL bookkeeping.
#
# The new policy is therefore:
#
#   POST_HOC_GENERATED
#   PREREGISTERED_FOR_FUTURE_CONFIRMATION
#   NOT INTERNALLY CONFIRMED
#
# ------------------------------------------------------------
# FUTURE A2
# ------------------------------------------------------------
#
# A2 may:
#
#   - use folds1-4 labels for final TRAINING only
#   - compute ONE frozen global OOF F0.5 threshold
#   - refit the already-frozen HGB family on folds1-4
#   - construct the exact GT-blind runtime policy
#
# A2 may NOT:
#
#   - evaluate official Stage12.14 graph score on folds1-4
#   - compare policy variants using GT
#   - tune attachment after GT
#   - use Fold0
#
# ------------------------------------------------------------
# FUTURE CONFIRMATION
# ------------------------------------------------------------
#
# Promotion requires a genuinely untouched holdout.
#
# No fresh holdout:
#
#   status remains
#
#   DEVELOPMENT_POLICY_FROZEN_UNCONFIRMED
#
# ============================================================


from pathlib import Path

import hashlib
import json

import pandas as pd


# ============================================================
# 0. PATHS
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

S12 = (
    PROJECT
    / "reports"
    / "stage12_residual_audit"
)


# ------------------------------------------------------------
# Stage12.14A0-R1 parents
# ------------------------------------------------------------

A0_DATASET_PATH = (
    S12
    / "stage12_14a0_post_a6_adjusted_metric_failure_dataset_decomposition_v1.pkl"
)

A0_FOLD_PATH = (
    S12
    / "stage12_14a0_post_a6_adjusted_metric_failure_fold_decomposition_v1.pkl"
)

A0_GROUP_PATH = (
    S12
    / "stage12_14a0_post_a6_adjusted_metric_failure_group_decomposition_v1.pkl"
)

A0_ASSOC_PATH = (
    S12
    / "stage12_14a0_post_a6_adjusted_metric_failure_associations_v1.pkl"
)

A0_SUMMARY_PATH = (
    S12
    / "stage12_14a0_post_a6_adjusted_metric_failure_decomposition_v1_summary.json"
)


# ------------------------------------------------------------
# A3 / A5 lineage
# ------------------------------------------------------------

A3_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_ranking_attachment_contract_frozen.json"
)

A5_SUMMARY_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_nested_lofo_ranking_v1_summary.json"
)

A5_DECISION_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_nested_lofo_ranking_v1_decision_frozen.json"
)

A5_OOF_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_nested_lofo_ranking_v1_oof.pkl"
)


# ------------------------------------------------------------
# A6 closure lineage
# ------------------------------------------------------------

A6_ACTIONS_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_graph_actions_v1.pkl"
)

A6_SUMMARY_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_official_summary_v1.json"
)

A6_DECISION_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_a6_graph_promotion_decision_frozen.json"
)

A6_CLOSURE_PATH = (
    S12
    / "stage12_post_g3h3_track_geometry_synthetic_node_recovery_v1_a6_graph_gate_closure.json"
)


# ------------------------------------------------------------
# Formal A1 output
# ------------------------------------------------------------

A1_OUT = (
    S12
    / "stage12_14a1_bilateral_g2_two_edge_economy_hypothesis_preregistration_v1.json"
)


for path in [
    A0_DATASET_PATH,
    A0_FOLD_PATH,
    A0_GROUP_PATH,
    A0_ASSOC_PATH,
    A0_SUMMARY_PATH,
    A3_PATH,
    A5_SUMMARY_PATH,
    A5_DECISION_PATH,
    A5_OOF_PATH,
    A6_ACTIONS_PATH,
    A6_SUMMARY_PATH,
    A6_DECISION_PATH,
    A6_CLOSURE_PATH,
]:
    assert path.exists(), path


if A1_OUT.exists():

    raise RuntimeError(
        "\nFormal Stage12.14A1 artifact already exists:\n"
        f"{A1_OUT}\n\n"
        "Do NOT overwrite preregistered provenance."
    )


# ============================================================
# 1. HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def json_content_sha256(obj):

    return hashlib.sha256(
        json.dumps(
            obj,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
            default=str,
        ).encode(
            "utf-8"
        )
    ).hexdigest()


class HashWriter:

    def __init__(self):

        self.h = hashlib.sha256()


    def write(
        self,
        text,
    ):

        if not isinstance(
            text,
            str,
        ):

            text = str(
                text
            )


        self.h.update(
            text.encode(
                "utf-8"
            )
        )


        return len(
            text
        )


    def flush(self):

        return None


def dataframe_content_sha256(df):

    writer = HashWriter()


    df.to_csv(
        writer,
        index=False,
        na_rep="NaN",
        float_format="%.17g",
        lineterminator="\n",
    )


    return writer.h.hexdigest()


# ============================================================
# 2. EXPECTED A0-R1 LOCKS
# ============================================================

EXPECTED_A0_DATASET_CONTENT_SHA = (
    "e9a055c79351fba31ea22260cf9fe27d3644504660a07bac0788b06fda0677eb"
)

EXPECTED_A0_FOLD_CONTENT_SHA = (
    "96310fcb1524be3f7d35bb23637d2cfcf368e91ddd9db2c48324cc644dde15ea"
)

EXPECTED_A0_GROUP_CONTENT_SHA = (
    "7787792b3a05582e6dbf413cdfb0513903c8b71e681edb6a22def8f820331e0a"
)

EXPECTED_A0_ASSOC_CONTENT_SHA = (
    "c6eb1a56dadebb00cab240f00ebf607e7435da7185a9889a71bb24a686ff9502"
)

EXPECTED_A0_SUMMARY_CONTENT_SHA = (
    "06fe1921ba37d2958c732d0dca0a9cd904fb0a9a205b4e45cb5d7caa24ad9449"
)


EXPECTED_A0_DATASET_FILE_SHA = (
    "477bd1acc9231842f240270ecc087cf46b3d0004eebe180b3f08e717086dc9e4"
)

EXPECTED_A0_FOLD_FILE_SHA = (
    "3ee3d06ddc85c4fa76e572ac04f0b1d4bd8053d802d3d6dbe6721a229d52cf2b"
)

EXPECTED_A0_GROUP_FILE_SHA = (
    "689418cecc75f30c6e986e9a4272b15a130eba422e2395bcfe9dd31b589d3fc5"
)

EXPECTED_A0_ASSOC_FILE_SHA = (
    "f756e674e128b959dd26814b920aa7cc269d73098f161b7fecd51b5a4855fa65"
)

EXPECTED_A0_SUMMARY_FILE_SHA = (
    "f094252b6e36314c70958094391044d24fd9059d09663ddbaccabdd0758b268b"
)


# ============================================================
# 3. OLDER LINEAGE LOCKS
# ============================================================

EXPECTED_A3_FILE_SHA = (
    "37a5b2e557832774c0bcf65a33c81d8434bac2025c4eebbf2e3b0ea1bc955bd7"
)

EXPECTED_A3_CONTENT_SHA = (
    "c54eda46073582a7a915f1bfa7dcbff9eec8a96edcfad80a8e49e1097cf38b46"
)


EXPECTED_A5_SUMMARY_FILE_SHA = (
    "064f2efe341b4c1f30e1a09c02222ae3a771df8304c734835df9f81dff23ecb5"
)

EXPECTED_A5_DECISION_FILE_SHA = (
    "a8b0212fbe0cb293c034a96f2c0b24f79b1d54640678fb18f9b8d38c846ca2de"
)

EXPECTED_A5_OOF_FILE_SHA = (
    "3838ed8c98c4f2a4cc6d94c5657db6129a41e5d7c91c4076eddfdb8129334d13"
)

EXPECTED_A5_SUMMARY_CONTENT_SHA = (
    "a16758ac5885c9db37cbb22e1c09ccf3cf26d882d427263c546c1e6ff7d55f3a"
)

EXPECTED_A5_DECISION_CONTENT_SHA = (
    "d29347659a6fe50a6555ac92a92dd2c2d6d58fbf4ea07b608c9b9f8b81529ef9"
)

EXPECTED_A5_OOF_CONTENT_SHA = (
    "ef4f5a5db23d58f5844a5819f584241eaeb43b162a57e81fb1f40be189bceb7f"
)


EXPECTED_A6_ACTIONS_FILE_SHA = (
    "33900e3787d21e50ad8fbaae10fdbf25e621f97493f0e83b0542cadeb241695d"
)

EXPECTED_A6_ACTIONS_CONTENT_SHA = (
    "a887af0f16e3375bf59d34ab8e211e36ab8c01c4c1599b2c1d2f2713d708e64d"
)

EXPECTED_A6_SUMMARY_FILE_SHA = (
    "37aa4c382e2384933365fb0cb284ddf70e52f30973bfe71bcfc07b62f9204253"
)

EXPECTED_A6_DECISION_FILE_SHA = (
    "741235b55ec65131f1c02c7ba802baea6bb1a4b826d57cea1f2baf44ba078c4e"
)

EXPECTED_A6_CLOSURE_FILE_SHA = (
    "c3bb3d36b063825954b260a1de6307b3c511d579b36419d85074aa86dfc28cdd"
)

EXPECTED_A6_SUMMARY_CONTENT_SHA = (
    "08f3d3c7b35682d92ee9e0ff51669fe0c59809014cd4ea8af59c0e1518101064"
)

EXPECTED_A6_DECISION_CONTENT_SHA = (
    "0a3ae7f65a36e6c687643ccdf63bf3de2ef757f933391070f3234b7d5b92ca81"
)

EXPECTED_A6_CLOSURE_CONTENT_SHA = (
    "a9ddddf3749ee17703d741b11b3b1cb1ee16373726583e02f73b8fd730b21184"
)


# ============================================================
# 4. LOAD + LOCK A0-R1
# ============================================================

assert (
    sha256_file(
        A0_DATASET_PATH
    )
    ==
    EXPECTED_A0_DATASET_FILE_SHA
)

assert (
    sha256_file(
        A0_FOLD_PATH
    )
    ==
    EXPECTED_A0_FOLD_FILE_SHA
)

assert (
    sha256_file(
        A0_GROUP_PATH
    )
    ==
    EXPECTED_A0_GROUP_FILE_SHA
)

assert (
    sha256_file(
        A0_ASSOC_PATH
    )
    ==
    EXPECTED_A0_ASSOC_FILE_SHA
)

assert (
    sha256_file(
        A0_SUMMARY_PATH
    )
    ==
    EXPECTED_A0_SUMMARY_FILE_SHA
)


a0_dataset = pd.read_pickle(
    A0_DATASET_PATH
)

a0_fold = pd.read_pickle(
    A0_FOLD_PATH
)

a0_group = pd.read_pickle(
    A0_GROUP_PATH
)

a0_assoc = pd.read_pickle(
    A0_ASSOC_PATH
)

a0_summary = json.loads(
    A0_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    dataframe_content_sha256(
        a0_dataset
    )
    ==
    EXPECTED_A0_DATASET_CONTENT_SHA
)

assert (
    dataframe_content_sha256(
        a0_fold
    )
    ==
    EXPECTED_A0_FOLD_CONTENT_SHA
)

assert (
    dataframe_content_sha256(
        a0_group
    )
    ==
    EXPECTED_A0_GROUP_CONTENT_SHA
)

assert (
    dataframe_content_sha256(
        a0_assoc
    )
    ==
    EXPECTED_A0_ASSOC_CONTENT_SHA
)

assert (
    a0_summary[
        "summary_sha256"
    ]
    ==
    EXPECTED_A0_SUMMARY_CONTENT_SHA
)


assert (
    a0_summary[
        "status"
    ]
    ==
    "POST_A6_ADJUSTED_METRIC_FAILURE_DECOMPOSITION_COMPLETE"
)

assert (
    a0_summary[
        "parent_decision"
    ]
    ==
    "CLOSE_BRANCH_NO_POLICY_PROMOTION"
)

assert (
    a0_summary[
        "protocol"
    ][
        "GT_graph_loaded"
    ]
    is False
)

assert (
    a0_summary[
        "protocol"
    ][
        "official_evaluator_executed"
    ]
    is False
)

assert (
    a0_summary[
        "protocol"
    ][
        "new_policy_defined"
    ]
    is False
)

assert (
    a0_summary[
        "protocol"
    ][
        "Fold0_used"
    ]
    is False
)


print(
    "========== 12.14A1 A0-R1 PARENT LOCK =========="
)

print(
    "A0-R1:",
    "PASS"
)

print(
    "A0 policy defined:",
    "NO"
)

print(
    "GT graph loaded in A0:",
    "NO"
)

print(
    "Fold0:",
    "NO"
)

print(
    "A0_PARENT_LOCK: PASS"
)


# ============================================================
# 5. LOCK SCIENTIFIC OBSERVATIONS
#
# These are HYPOTHESIS-GENERATING observations only.
# ============================================================

algebra = (
    a0_summary[
        "exact_adjusted_metric_algebra"
    ]
)


assert (
    float(
        algebra[
            "numerator_shift_effect"
        ]
    )
    >
    0.0
)

assert (
    float(
        algebra[
            "denominator_normalization_effect"
        ]
    )
    <
    0.0
)

assert (
    abs(
        float(
            algebra[
                "denominator_normalization_effect"
            ]
        )
    )
    >
    abs(
        float(
            algebra[
                "numerator_shift_effect"
            ]
        )
    )
)


# ------------------------------------------------------------
# Locate A0-R1 bookkeeping rows.
#
# IMPORTANT:
# these are explicitly NON-CAUSAL.
# ------------------------------------------------------------

edge_count_groups = (
    a0_group[
        a0_group[
            "group_type"
        ]
        ==
        "ACTION_EDGE_COUNT_CLASS"
    ]
    .set_index(
        "group"
    )
)


topology_groups = (
    a0_group[
        a0_group[
            "group_type"
        ]
        ==
        "ACTION_TOPOLOGY"
    ]
    .set_index(
        "group"
    )
)


assert (
    "ONE_EDGE"
    in
    edge_count_groups.index
)

assert (
    "TWO_EDGE"
    in
    edge_count_groups.index
)

assert (
    "E2|IN:G2|OUT:G2"
    in
    topology_groups.index
)


one_edge_bookkeeping = float(
    edge_count_groups.loc[
        "ONE_EDGE",
        "bookkeeping_adj_contribution",
    ]
)


two_edge_bookkeeping = float(
    edge_count_groups.loc[
        "TWO_EDGE",
        "bookkeeping_adj_contribution",
    ]
)


bilateral_g2_bookkeeping = float(
    topology_groups.loc[
        "E2|IN:G2|OUT:G2",
        "bookkeeping_adj_contribution",
    ]
)


assert (
    one_edge_bookkeeping
    <
    0.0
)

assert (
    two_edge_bookkeeping
    >
    0.0
)

assert (
    bilateral_g2_bookkeeping
    >
    0.0
)


# ------------------------------------------------------------
# Node-ratio association:
#
# hypothesis-generating only.
# ------------------------------------------------------------

ratio_assoc = a0_assoc[
    (
        a0_assoc[
            "feature"
        ]
        ==
        "delta_total_node_ratio"
    )
    &
    (
        a0_assoc[
            "target"
        ]
        ==
        "delta_adjustment_gap"
    )
]


assert len(
    ratio_assoc
) == 1


ratio_pearson = float(
    ratio_assoc.iloc[
        0
    ][
        "pearson"
    ]
)


ratio_spearman = float(
    ratio_assoc.iloc[
        0
    ][
        "spearman"
    ]
)


print(
    "\n========== HYPOTHESIS-GENERATING OBSERVATIONS =========="
)

print(
    "numerator-shift effect:",
    algebra[
        "numerator_shift_effect"
    ]
)

print(
    "denominator-normalization effect:",
    algebra[
        "denominator_normalization_effect"
    ]
)

print(
    "ONE_EDGE bookkeeping:",
    one_edge_bookkeeping
)

print(
    "TWO_EDGE bookkeeping:",
    two_edge_bookkeeping
)

print(
    "E2|IN:G2|OUT:G2 bookkeeping:",
    bilateral_g2_bookkeeping
)

print(
    "node-ratio vs adjustment-gap Pearson:",
    ratio_pearson
)

print(
    "node-ratio vs adjustment-gap Spearman:",
    ratio_spearman
)

print(
    "causal interpretation:",
    "FORBIDDEN"
)

print(
    "HYPOTHESIS_GENERATION_SOURCE_LOCK: PASS"
)


# ============================================================
# 6. LOCK A3/A5/A6 LINEAGE
# ============================================================

assert (
    sha256_file(
        A3_PATH
    )
    ==
    EXPECTED_A3_FILE_SHA
)

assert (
    sha256_file(
        A5_SUMMARY_PATH
    )
    ==
    EXPECTED_A5_SUMMARY_FILE_SHA
)

assert (
    sha256_file(
        A5_DECISION_PATH
    )
    ==
    EXPECTED_A5_DECISION_FILE_SHA
)

assert (
    sha256_file(
        A5_OOF_PATH
    )
    ==
    EXPECTED_A5_OOF_FILE_SHA
)

assert (
    sha256_file(
        A6_ACTIONS_PATH
    )
    ==
    EXPECTED_A6_ACTIONS_FILE_SHA
)

assert (
    sha256_file(
        A6_SUMMARY_PATH
    )
    ==
    EXPECTED_A6_SUMMARY_FILE_SHA
)

assert (
    sha256_file(
        A6_DECISION_PATH
    )
    ==
    EXPECTED_A6_DECISION_FILE_SHA
)

assert (
    sha256_file(
        A6_CLOSURE_PATH
    )
    ==
    EXPECTED_A6_CLOSURE_FILE_SHA
)


a3 = json.loads(
    A3_PATH.read_text(
        encoding="utf-8"
    )
)

a5_summary = json.loads(
    A5_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

a5_decision = json.loads(
    A5_DECISION_PATH.read_text(
        encoding="utf-8"
    )
)

a6_summary = json.loads(
    A6_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

a6_decision = json.loads(
    A6_DECISION_PATH.read_text(
        encoding="utf-8"
    )
)

a6_closure = json.loads(
    A6_CLOSURE_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    a3[
        "contract_sha256"
    ]
    ==
    EXPECTED_A3_CONTENT_SHA
)

assert (
    a5_summary[
        "summary_sha256"
    ]
    ==
    EXPECTED_A5_SUMMARY_CONTENT_SHA
)

assert (
    a5_decision[
        "decision_sha256"
    ]
    ==
    EXPECTED_A5_DECISION_CONTENT_SHA
)

assert (
    a6_summary[
        "summary_sha256"
    ]
    ==
    EXPECTED_A6_SUMMARY_CONTENT_SHA
)

assert (
    a6_decision[
        "decision_sha256"
    ]
    ==
    EXPECTED_A6_DECISION_CONTENT_SHA
)

assert (
    a6_closure[
        "closure_sha256"
    ]
    ==
    EXPECTED_A6_CLOSURE_CONTENT_SHA
)


assert (
    a6_decision[
        "decision"
    ]
    ==
    "CLOSE_BRANCH_NO_POLICY_PROMOTION"
)


assert (
    a6_closure[
        "decision"
    ]
    ==
    "CLOSE_BRANCH_NO_POLICY_PROMOTION"
)


# ============================================================
# 7. NEW HYPOTHESIS IDENTITY
# ============================================================

BRANCH_ID = (
    "POST_A6_BILATERAL_G2_TWO_EDGE_"
    "NORMALIZATION_ECONOMY_V1"
)


POLICY_ID = (
    "BILATERAL_G2_TWO_EDGE_ECONOMY_V1"
)


HYPOTHESIS_CLASS = (
    "POST_HOC_GENERATED_"
    "PREREGISTERED_FOR_FRESH_CONFIRMATION"
)


# ============================================================
# 8. EXACT POLICY HYPOTHESIS
#
# This freezes the NEXT policy architecture.
#
# It does NOT claim the policy works.
# ============================================================

POLICY_CONTRACT = {
    "policy_id":
        POLICY_ID,

    "scientific_status":
        HYPOTHESIS_CLASS,

    # --------------------------------------------------------
    # Candidate universe
    # --------------------------------------------------------

    "candidate_universe": {
        "source":
            (
                "FROZEN_STAGE12_13_"
                "POST_G3H3_TRACK_GEOMETRY_"
                "SYNTHETIC_CANDIDATE_UNIVERSE_V1"
            ),

        "candidate_universe_expansion":
            "FORBIDDEN",

        "candidate_identity":
            [
                "dataset",
                "t",
                "round(z,12)",
                "round(y,12)",
                "round(x,12)",
            ],

        "fold_is_identity":
            False,
    },

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    "ranking_model": {
        "family":
            "HGB_BALANCED_V1",

        "reason":
            (
                "MODEL_FAMILY_ALREADY_SELECTED_IN_ALL_"
                "FOUR_A5_OUTER_FOLDS_BEFORE_A6_GRAPH_RESULT"
            ),

        "parameters": {
            "learning_rate":
                0.05,

            "max_iter":
                200,

            "max_leaf_nodes":
                31,

            "min_samples_leaf":
                20,

            "l2_regularization":
                1.0,

            "early_stopping":
                False,

            "random_state":
                1640,

            "class_weighting":
                "MANUAL_BALANCED_SAMPLE_WEIGHTS",
        },

        "feature_contract":
            "A3_FROZEN_39_FEATURES",

        "new_feature_search":
            "FORBIDDEN",
    },

    # --------------------------------------------------------
    # Threshold
    # --------------------------------------------------------

    "threshold_freeze_protocol": {
        "stage":
            "12.14A2",

        "source":
            "A5_FROZEN_4FOLD_OOF_HGB_SCORES",

        "folds":
            [
                1,
                2,
                3,
                4,
            ],

        "Fold0":
            "FORBIDDEN",

        "objective":
            "F0.5",

        "candidate_thresholds":
            "EXACT_PRECISION_RECALL_CURVE_THRESHOLDS",

        "selection_order":
            [
                "MAX_F0.5",
                "MAX_PRECISION_ON_F0.5_TIE",
                "MAX_THRESHOLD_ON_REMAINING_TIE",
            ],

        "number_of_threshold_selections":
            1,

        "graph_metric_used_for_threshold":
            False,

        "post_threshold_retuning":
            "FORBIDDEN",
    },

    # --------------------------------------------------------
    # Bilateral G2 eligibility
    # --------------------------------------------------------

    "eligibility": {
        "must_have_G2":
            True,

        "same_raw_G2_pair_required":
            True,

        "incoming_support":
            "G2_SOURCE_TO_SYNTHETIC",

        "outgoing_support":
            "SYNTHETIC_TO_G2_TARGET",

        "G0_allowed":
            False,

        "G1_allowed":
            False,

        "one_edge_recovery_allowed":
            False,

        "require_both_sides":
            True,

        "minimum_edges":
            2,

        "maximum_edges":
            2,

        "candidate_score_requirement":
            "MODEL_SCORE_GE_FROZEN_GLOBAL_THRESHOLD",
    },

    # --------------------------------------------------------
    # Same-raw pair ordering
    # --------------------------------------------------------

    "same_raw_G2_pair_selection": {
        "pair_candidate_source":
            "A1_ACCEPTED_RAW_G2_PROPOSALS_ONLY",

        "both_slots_must_be_free":
            True,

        "evolving_graph":
            True,

        "source_slot":
            "CURRENT_OUTDEGREE_EQ_0",

        "target_slot":
            "CURRENT_INDEGREE_EQ_0",

        "pair_sort":
            [
                "SUM_PHYSICAL_ENDPOINT_TO_SYNTHETIC_DISTANCE_ASC",
                "MAX_PHYSICAL_ENDPOINT_TO_SYNTHETIC_DISTANCE_ASC",
                "G2_PAIR_DISTANCE_UM_ASC",
                "SOURCE_NODE_ID_ASC",
                "TARGET_NODE_ID_ASC",
                "RAW_PROPOSAL_ID_ASC",
            ],

        "pair_mixing_across_raw_proposals":
            "FORBIDDEN",

        "if_no_free_same_raw_pair":
            "SKIP_CANDIDATE",
    },

    # --------------------------------------------------------
    # Application order
    # --------------------------------------------------------

    "application_order": {
        "primary":
            "MODEL_SCORE_DESC",

        "secondary":
            "CANDIDATE_ID_ASC",
    },

    # --------------------------------------------------------
    # Mutation
    # --------------------------------------------------------

    "mutation": {
        "synthetic_nodes_per_action":
            1,

        "edges_per_action":
            2,

        "node_coordinate":
            "EXACT_FROZEN_CANDIDATE_TZYX",

        "node_id":
            "AFTER_CURRENT_MAX_IN_APPLICATION_ORDER",

        "incoming_edge":
            "SOURCE_TO_SYNTHETIC",

        "outgoing_edge":
            "SYNTHETIC_TO_TARGET",

        "edge_distance":
            "PHYSICAL_EUCLIDEAN_UM",

        "edge_prob":
            1.0,

        "existing_edge_removal":
            "FORBIDDEN",

        "existing_node_modification":
            "FORBIDDEN",

        "existing_node_removal":
            "FORBIDDEN",

        "coordinate_change":
            "FORBIDDEN",

        "merge":
            "FORBIDDEN",

        "maximum_indegree":
            1,

        "maximum_outdegree":
            1,
    },

    # --------------------------------------------------------
    # Structural economics
    # --------------------------------------------------------

    "normalization_economy_hypothesis": {
        "new_nodes_per_action":
            1,

        "new_edges_per_action":
            2,

        "nominal_new_nodes_per_new_edge":
            0.5,

        "comparison_failure_mode":
            (
                "STAGE12_13_ALLOWED_ONE_EDGE_ACTIONS_"
                "AND_SUFFERED_GLOBAL_ADJUSTED_METRIC_"
                "NORMALIZATION_PENALTY"
            ),

        "claim_status":
            "HYPOTHESIS_ONLY",
    },
}


# ============================================================
# 9. DATA REUSE / CONTAMINATION CONTRACT
# ============================================================

DATA_PROTOCOL = {
    "fold1_to_fold4": {
        "status":
            "SEEN_GT_HYPOTHESIS_GENERATION_AND_TRAINING_ONLY",

        "may_train_final_model":
            True,

        "may_select_single_preregistered_OOF_threshold":
            True,

        "may_run_GT_blind_runtime_invariants":
            True,

        "may_run_official_graph_metric_for_Stage12_14_confirmation":
            False,

        "may_compare_Stage12_14_policy_variants_using_GT":
            False,

        "confirmatory_status":
            "PERMANENTLY_INVALID_FOR_STAGE12_14",
    },

    "fold0": {
        "status":
            "CONSUMED_AND_PERMANENTLY_FORBIDDEN",

        "may_train":
            False,

        "may_tune":
            False,

        "may_evaluate":
            False,
    },

    "fresh_holdout": {
        "required_for_confirmation":
            True,

        "definition":
            (
                "GROUND_TRUTH_NOT_USED_FOR_STAGE12_13_A6_"
                "OR_STAGE12_14_HYPOTHESIS_FORMATION_"
                "MODEL_SELECTION_THRESHOLD_SELECTION_OR_POLICY_DESIGN"
            ),

        "policy_must_be_frozen_before_first_GT_read":
            True,
    },
}


# ============================================================
# 10. FUTURE A2 CONTRACT
# ============================================================

A2_CONTRACT = {
    "stage":
        "12.14A2",

    "name":
        (
            "FINAL_MODEL_THRESHOLD_AND_"
            "GT_BLIND_RUNTIME_POLICY_FREEZE"
        ),

    "allowed": [
        (
            "COMPUTE_ONE_GLOBAL_F0.5_THRESHOLD_"
            "FROM_A5_FROZEN_OOF_HGB_SCORES"
        ),

        (
            "REFIT_HGB_BALANCED_V1_ON_ALL_"
            "FOLDS1_TO_4_TRAINING_LABELS"
        ),

        (
            "CONSTRUCT_GT_BLIND_BILATERAL_G2_"
            "TWO_EDGE_RUNTIME_POLICY"
        ),

        (
            "RUN_GRAPH_SCHEMA_AND_DETERMINISM_"
            "INVARIANT_TESTS_WITHOUT_GT"
        ),
    ],

    "forbidden": [
        "OFFICIAL_METRIC_ON_FOLDS1_TO_4",
        "GRAPH_SCORE_ON_FOLDS1_TO_4",
        "THRESHOLD_SEARCH_AFTER_GRAPH_RESULTS",
        "ATTACHMENT_VARIANT_SEARCH_USING_GT",
        "G0_G1_REINTRODUCTION",
        "ONE_EDGE_FALLBACK",
        "FEATURE_SEARCH",
        "MODEL_FAMILY_SEARCH",
        "FOLD0",
    ],

    "output_status_if_successful":
        "DEVELOPMENT_POLICY_FROZEN_UNCONFIRMED",

    "deployment_ready":
        False,
}


# ============================================================
# 11. FRESH-HOLDOUT CONFIRMATION GATES
#
# ALL required.
# ============================================================

CONFIRMATION_GATES = {
    "required_holdout":
        "GENUINELY_FRESH_UNTOUCHED_GT",

    "official_metric_tolerance_um":
        7.0,

    "all_conditions_required":
        True,

    "conditions": [
        "POOLED_OFFICIAL_SCORE_DELTA_GT_0",
        "POOLED_ADJ_EDGE_JACCARD_DELTA_GT_0",
        "POOLED_EDGE_JACCARD_DELTA_GT_0",
        "POOLED_NODE_RECALL_DELTA_GE_0",
        "POOLED_DIVISION_JACCARD_DELTA_GE_0",
        "ZERO_INVALID_OFFICIAL_EVALUATION_ROWS",
    ],

    "baseline":
        (
            "FROZEN_G3H3_"
            "ASYM1x5_Q9990_NO_TARGET_CUT_V1"
        ),

    "if_any_gate_fails":
        "CLOSE_STAGE12_14_NO_PROMOTION",

    "post_holdout_retuning":
        "FORBIDDEN",
}


# ============================================================
# 12. CURRENT BEST LOCK
# ============================================================

CURRENT_BEST = {
    "policy":
        "ASYM1x5_Q9990_NO_TARGET_CUT_V1",

    "stage":
        "G3H3",

    "development_score":
        0.7920280500935799,

    "status":
        "UNCHANGED_BY_STAGE12_13",

    "Stage12_14_may_replace_before_fresh_confirmation":
        False,
}


# ============================================================
# 13. SCIENTIFIC INTERPRETATION CONTRACT
# ============================================================

INTERPRETATION = {
    "what_A0_supports": [
        (
            "STAGE12_13_POLICY_CONTAINED_POSITIVE_"
            "WEIGHTED_NUMERATOR_SIGNAL"
        ),

        (
            "GLOBAL_NORMALIZATION_EFFECT_EXCEEDED_"
            "THAT_POSITIVE_SIGNAL"
        ),

        (
            "NODE_RATIO_IS_STRONGLY_ASSOCIATED_WITH_"
            "ADJUSTMENT_GAP_ON_SEEN_DEVELOPMENT_DATA"
        ),

        (
            "TWO_EDGE_AND_BILATERAL_G2_BOOKKEEPING_"
            "ARE_HYPOTHESIS_GENERATING_SIGNALS"
        ),
    ],

    "what_A0_does_not_support": [
        "G2_IS_CAUSALLY_BETTER",

        "TWO_EDGE_ACTIONS_ARE_CAUSALLY_BETTER",

        (
            "BILATERAL_G2_TWO_EDGE_ECONOMY_V1_"
            "WILL_IMPROVE_OFFICIAL_SCORE"
        ),

        (
            "ANY_NUMERIC_THRESHOLD_DERIVED_FROM_A0_"
            "IS_CONFIRMED"
        ),
    ],

    "primary_falsifiable_hypothesis":
        (
            "ON_A_GENUINELY_FRESH_HOLDOUT_THE_FROZEN_"
            "BILATERAL_G2_TWO_EDGE_ECONOMY_V1_POLICY_"
            "WILL_PRODUCE_POSITIVE_POOLED_OFFICIAL_SCORE_"
            "AND_ADJUSTED_EDGE_JACCARD_DELTAS_RELATIVE_TO_G3H3"
        ),
}


# ============================================================
# 14. FORMAL PREREGISTRATION
# ============================================================

CONTRACT = {
    "stage":
        "12.14A1",

    "status":
        "NEW_HYPOTHESIS_PREREGISTERED",

    "branch_id":
        BRANCH_ID,

    "policy_id":
        POLICY_ID,

    "hypothesis_class":
        HYPOTHESIS_CLASS,

    "scientific_origin": {
        "generated_after_Stage12_13_GT_evaluation":
            True,

        "generated_from_A0_R1_postmortem":
            True,

        "therefore_old_dev_GT_can_confirm":
            False,

        "fresh_holdout_required":
            True,
    },

    "parents": {
        "A0_R1": {
            "dataset_content_sha256":
                EXPECTED_A0_DATASET_CONTENT_SHA,

            "dataset_file_sha256":
                EXPECTED_A0_DATASET_FILE_SHA,

            "fold_content_sha256":
                EXPECTED_A0_FOLD_CONTENT_SHA,

            "fold_file_sha256":
                EXPECTED_A0_FOLD_FILE_SHA,

            "group_content_sha256":
                EXPECTED_A0_GROUP_CONTENT_SHA,

            "group_file_sha256":
                EXPECTED_A0_GROUP_FILE_SHA,

            "association_content_sha256":
                EXPECTED_A0_ASSOC_CONTENT_SHA,

            "association_file_sha256":
                EXPECTED_A0_ASSOC_FILE_SHA,

            "summary_content_sha256":
                EXPECTED_A0_SUMMARY_CONTENT_SHA,

            "summary_file_sha256":
                EXPECTED_A0_SUMMARY_FILE_SHA,
        },

        "A3_contract": {
            "content_sha256":
                EXPECTED_A3_CONTENT_SHA,

            "file_sha256":
                EXPECTED_A3_FILE_SHA,
        },

        "A5": {
            "summary_content_sha256":
                EXPECTED_A5_SUMMARY_CONTENT_SHA,

            "summary_file_sha256":
                EXPECTED_A5_SUMMARY_FILE_SHA,

            "decision_content_sha256":
                EXPECTED_A5_DECISION_CONTENT_SHA,

            "decision_file_sha256":
                EXPECTED_A5_DECISION_FILE_SHA,

            "oof_content_sha256":
                EXPECTED_A5_OOF_CONTENT_SHA,

            "oof_file_sha256":
                EXPECTED_A5_OOF_FILE_SHA,
        },

        "A6_closed_branch": {
            "actions_content_sha256":
                EXPECTED_A6_ACTIONS_CONTENT_SHA,

            "actions_file_sha256":
                EXPECTED_A6_ACTIONS_FILE_SHA,

            "summary_content_sha256":
                EXPECTED_A6_SUMMARY_CONTENT_SHA,

            "summary_file_sha256":
                EXPECTED_A6_SUMMARY_FILE_SHA,

            "decision_content_sha256":
                EXPECTED_A6_DECISION_CONTENT_SHA,

            "decision_file_sha256":
                EXPECTED_A6_DECISION_FILE_SHA,

            "closure_content_sha256":
                EXPECTED_A6_CLOSURE_CONTENT_SHA,

            "closure_file_sha256":
                EXPECTED_A6_CLOSURE_FILE_SHA,
        },
    },

    "hypothesis_generating_observations": {
        "numerator_shift_effect":
            float(
                algebra[
                    "numerator_shift_effect"
                ]
            ),

        "denominator_normalization_effect":
            float(
                algebra[
                    "denominator_normalization_effect"
                ]
            ),

        "one_edge_bookkeeping":
            one_edge_bookkeeping,

        "two_edge_bookkeeping":
            two_edge_bookkeeping,

        "bilateral_G2_two_edge_bookkeeping":
            bilateral_g2_bookkeeping,

        "node_ratio_adjustment_gap_pearson":
            ratio_pearson,

        "node_ratio_adjustment_gap_spearman":
            ratio_spearman,

        "causal":
            False,
    },

    "policy_contract":
        POLICY_CONTRACT,

    "data_protocol":
        DATA_PROTOCOL,

    "A2_contract":
        A2_CONTRACT,

    "fresh_holdout_confirmation":
        CONFIRMATION_GATES,

    "current_best":
        CURRENT_BEST,

    "interpretation_contract":
        INTERPRETATION,

    "protocol": {
        "GT_graph_loaded":
            False,

        "official_evaluator_executed":
            False,

        "graph_mutated":
            False,

        "model_trained":
            False,

        "threshold_selected":
            False,

        "new_policy_executed":
            False,

        "old_development_GT_used_for_confirmation":
            False,

        "Fold0_used":
            False,
    },

    "next":
        (
            "12.14A2_FINAL_MODEL_THRESHOLD_"
            "AND_GT_BLIND_RUNTIME_POLICY_FREEZE"
        ),
}


# ============================================================
# 15. SELF-CONSISTENCY
# ============================================================

assert (
    CONTRACT[
        "scientific_origin"
    ][
        "fresh_holdout_required"
    ]
    is True
)


assert (
    CONTRACT[
        "policy_contract"
    ][
        "eligibility"
    ][
        "one_edge_recovery_allowed"
    ]
    is False
)


assert (
    CONTRACT[
        "policy_contract"
    ][
        "eligibility"
    ][
        "require_both_sides"
    ]
    is True
)


assert (
    CONTRACT[
        "policy_contract"
    ][
        "eligibility"
    ][
        "minimum_edges"
    ]
    ==
    2
)


assert (
    CONTRACT[
        "policy_contract"
    ][
        "eligibility"
    ][
        "maximum_edges"
    ]
    ==
    2
)


assert (
    CONTRACT[
        "policy_contract"
    ][
        "eligibility"
    ][
        "same_raw_G2_pair_required"
    ]
    is True
)


assert (
    CONTRACT[
        "data_protocol"
    ][
        "fold1_to_fold4"
    ][
        "may_run_official_graph_metric_for_Stage12_14_confirmation"
    ]
    is False
)


assert (
    CONTRACT[
        "data_protocol"
    ][
        "fold0"
    ][
        "may_evaluate"
    ]
    is False
)


assert (
    CONTRACT[
        "A2_contract"
    ][
        "deployment_ready"
    ]
    is False
)


# ============================================================
# 16. CONTRACT SHA
# ============================================================

CONTRACT_CONTENT_SHA = (
    json_content_sha256(
        CONTRACT
    )
)


CONTRACT[
    "contract_sha256"
] = CONTRACT_CONTENT_SHA


# ============================================================
# 17. WRITE TEMP + READBACK
# ============================================================

TMP = A1_OUT.with_name(
    A1_OUT.name
    +
    ".tmp_12_14a1"
)


if TMP.exists():

    raise RuntimeError(
        "\nStale A1 temporary artifact:\n"
        f"{TMP}"
    )


TMP.write_text(
    json.dumps(
        CONTRACT,
        indent=2,
    ),
    encoding="utf-8",
)


readback = json.loads(
    TMP.read_text(
        encoding="utf-8"
    )
)


assert (
    readback[
        "contract_sha256"
    ]
    ==
    CONTRACT_CONTENT_SHA
)


payload = dict(
    readback
)


payload.pop(
    "contract_sha256"
)


assert (
    json_content_sha256(
        payload
    )
    ==
    CONTRACT_CONTENT_SHA
)


A1_FILE_SHA = (
    sha256_file(
        TMP
    )
)


TMP.replace(
    A1_OUT
)


assert (
    sha256_file(
        A1_OUT
    )
    ==
    A1_FILE_SHA
)


# ============================================================
# 18. PARENT IMMUTABILITY
# ============================================================

assert (
    sha256_file(
        A0_SUMMARY_PATH
    )
    ==
    EXPECTED_A0_SUMMARY_FILE_SHA
)


assert (
    sha256_file(
        A6_ACTIONS_PATH
    )
    ==
    EXPECTED_A6_ACTIONS_FILE_SHA
)


assert (
    sha256_file(
        A6_CLOSURE_PATH
    )
    ==
    EXPECTED_A6_CLOSURE_FILE_SHA
)


# ============================================================
# 19. FINAL
# ============================================================

print(
    "\n============================================================"
)


print(
    "12.14A1 FINAL"
)


print(
    "============================================================"
)


print(
    "\nNEW_HYPOTHESIS_PREREGISTRATION:",
    "PASS"
)


print(
    "\n--- HYPOTHESIS ---"
)


print(
    "branch:",
    BRANCH_ID
)


print(
    "policy:",
    POLICY_ID
)


print(
    "status:",
    HYPOTHESIS_CLASS
)


print(
    "\n--- MECHANISM ---"
)


print(
    "Stage12.13 numerator effect:",
    algebra[
        "numerator_shift_effect"
    ]
)


print(
    "Stage12.13 denominator-normalization effect:",
    algebra[
        "denominator_normalization_effect"
    ]
)


print(
    "new structural rule:",
    "STRICT SAME-RAW BILATERAL G2"
)


print(
    "new node per action:",
    1
)


print(
    "new edges per action:",
    2
)


print(
    "one-edge fallback:",
    "FORBIDDEN"
)


print(
    "nominal nodes/new-edge:",
    0.5
)


print(
    "\n--- SCIENTIFIC STATUS ---"
)


print(
    "A0 evidence causal:",
    "NO"
)


print(
    "hypothesis generated after seeing Stage12.13 GT:",
    "YES"
)


print(
    "folds1-4 valid for Stage12.14 confirmation:",
    "NO"
)


print(
    "Fold0:",
    "PERMANENTLY FORBIDDEN"
)


print(
    "fresh untouched holdout required:",
    "YES"
)


print(
    "\n--- A2 ALLOWED ---"
)


for item in A2_CONTRACT[
    "allowed"
]:

    print(
        "-",
        item
    )


print(
    "\n--- A2 FORBIDDEN ---"
)


for item in A2_CONTRACT[
    "forbidden"
]:

    print(
        "-",
        item
    )


print(
    "\n--- FUTURE CONFIRMATION GATES ---"
)


for condition in CONFIRMATION_GATES[
    "conditions"
]:

    print(
        "-",
        condition
    )


print(
    "\n--- CURRENT BEST ---"
)


print(
    CURRENT_BEST[
        "policy"
    ]
)


print(
    "score:",
    CURRENT_BEST[
        "development_score"
    ]
)


print(
    "unchanged:",
    "YES"
)


print(
    "\n--- SAFETY ---"
)


print(
    "GT graph loaded:",
    "NO"
)


print(
    "official evaluator executed:",
    "NO"
)


print(
    "graph mutated:",
    "NO"
)


print(
    "model trained:",
    "NO"
)


print(
    "threshold selected:",
    "NO"
)


print(
    "policy executed:",
    "NO"
)


print(
    "Fold0 used:",
    "NO"
)


print(
    "\n--- SHAS ---"
)


print(
    "contract content:",
    CONTRACT_CONTENT_SHA
)


print(
    "contract file:",
    A1_FILE_SHA
)


print(
    "\nstatus:"
)


print(
    "HYPOTHESIS_PREREGISTERED_AWAITING_A2"
)


print(
    "\nnext:"
)


print(
    "12.14A2_FINAL_MODEL_THRESHOLD_AND_GT_BLIND_RUNTIME_POLICY_FREEZE"
)